<a href="https://colab.research.google.com/github/ErickJLA/CoMeta/blob/main/CoMeta_1_filters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ⚙️ 1. Environment Setup & Core Functions
# --- Standard Library ---
import base64
import datetime
import importlib.metadata
import importlib.util
import io
import os
import sys
import warnings

# --- ENVIRONMENT DETECTION (standard library only, so it cannot fail) ---
try:
    IS_COLAB = importlib.util.find_spec('google.colab') is not None
except (ImportError, ValueError):     # no 'google' package at all
    IS_COLAB = False

# Versions CoMeta was tested with (Google Colab defaults where possible).
REQUIRED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scipy": "1.16.3",
    "matplotlib": "3.10.0",
    "seaborn": "0.13.2",
    "ipywidgets": "7.7.1",
    "gspread": "6.2.1",
    "google-auth": "2.47.0",
    "patsy": "1.0.2",
    "statsmodels": "0.14.6",
    "plotly": "5.24.1",
    "tqdm": "4.67.3",
    "xlsxwriter": "3.2.9",
    "rpy2": "3.5.17"
}

def check_and_lockdown():
    """Install MISSING packages at the tested version; report, but do not
    change, packages whose installed version differs.

    Upgrading or downgrading a package that the session has already imported
    (numpy, pandas, scipy, ...) needs a runtime restart, and a restart wipes
    everything this cell defines -- later cells then fail (e.g. NameError:
    mark_stale). Missing packages are not imported yet, so installing them is
    safe. Runs before the third-party imports below.
    """
    missing, different = [], []
    for pkg, ver in REQUIRED_PACKAGES.items():
        try:
            current = importlib.metadata.version(pkg)
            if current != ver:
                different.append((pkg, current, ver))
        except importlib.metadata.PackageNotFoundError:
            missing.append((pkg, ver))

    if missing:
        spec = ' '.join(f"{p}=={v}" for p, v in missing)
        if IS_COLAB:
            print(f"Installing missing packages: {spec}")
            os.system(f"pip install -q {spec}")
        else:
            print(f"⚠️ Missing packages: {spec}\n"
                  f"   Install them (pip install {spec}) and re-run this cell.")
    return {'missing': missing, 'different': different}

# Run the check (results are shown at the end of this cell)
COMETA_ENV_CHECK = check_and_lockdown()

# --- Data Science & Scientific Computing ---
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.optimize import minimize, minimize_scalar
from scipy.special import gamma, gammaln
from scipy.stats import chi2, norm, t
try:
    import gspread              # Google Sheets import (Colab only)
except ImportError:
    gspread = None

# --- Plotting & Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Jupyter Notebook & UI Components ---
from IPython.display import HTML, Javascript, clear_output, display
import ipywidgets as widgets

# --- Colab-only services ---
if IS_COLAB:
    from google.colab import auth
    from google.colab import output
    from google.auth import default
    try:
        output.no_vertical_scroll()
    except Exception:
        pass

# --- DISPLAY LOADING SCREEN ---
display(HTML("""
<style>
  @keyframes fly-by {
    0% { transform: translate(-100px, -50px) rotate(35deg); opacity: 0; }
    10% { opacity: 1; }
    30% { transform: translate(600px, 400px) rotate(35deg); opacity: 0; }
    100% { transform: translate(600px, 400px) rotate(35deg); opacity: 0; }
  }
  .loading-comet {
    position: absolute;
    width: 120px;
    height: 2px;
    background: linear-gradient(90deg, transparent, #ffffff, transparent);
    box-shadow: 0 0 10px 2px rgba(255,255,255,0.8);
    animation: fly-by 2.5s infinite linear;
  }
</style>
<div style="position: relative; overflow: hidden; background: linear-gradient(135deg, #0f172a 0%, #1e3a8a 100%);
            padding: 40px; border-radius: 12px; text-align: center; font-family: sans-serif;
            max-width: 900px; margin: 20px auto; box-shadow: 0 10px 25px -5px rgba(0,0,0,0.2);">
   <!-- Comets -->
   <div class="loading-comet" style="top: 10%; left: 10%; animation-delay: 0s;"></div>
   <div class="loading-comet" style="top: 40%; left: -10%; animation-delay: 0.8s;"></div>
   <div class="loading-comet" style="top: -10%; left: 30%; animation-delay: 1.6s;"></div>

   <!-- Text -->
   <h2 style="margin:0; color: #ffffff; display: flex; align-items: center; justify-content: center;">
      <span style="margin-right: 10px;"></span> Initializing CoMeta
   </h2>
   <p style="margin-top:12px; color: #bfdbfe; font-size: 15px;">
      Loading... Please wait.
   </p>
</div>
"""))

# =============================================================================
# CELL 1: ENVIRONMENT SETUP
# Purpose: Import required libraries and authenticate Google Sheets access
# Dependencies: None
# Outputs: Authentication status, library versions, system info
# =============================================================================



# Suppress unnecessary warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Configuration Constants ---
REQUIRED_COLUMNS = {
    'effect_data': ['xe', 'sde', 'ne', 'xc', 'sdc', 'nc'],
    'metadata': ['id']
}

# Binary outcome columns for log Odds Ratio
BINARY_REQUIRED_COLUMNS = {
    'effect_data': ['events_e', 'nonevents_e', 'events_c', 'nonevents_c'],
    'optional': ['ne', 'nc'],  # recommended for VCV matrix
    'metadata': ['id']
}

SUPPORTED_EFFECT_SIZES = {
    'lnRR': 'Log Response Ratio',
    'hedges_g': "Hedges' g (corrected SMD)",
    'cohen_d': "Cohen's d (uncorrected SMD)",
    'log_OR': 'Log Odds Ratio',
    'log_RR': 'Log Risk Ratio'
}

# --- Authentication ---

# =============================================================================
# GLOBAL STATE MANAGER
# =============================================================================
import ipywidgets as widgets

STALE_BANNERS = {}

def register_banner(module_id: str, banner_widget: widgets.HTML):
    """Registers a module's banner widget for global access."""
    STALE_BANNERS[module_id] = banner_widget

def mark_stale(module_ids: list, trigger_source: str):
    """Activates the warning banner for specific downstream modules."""
    html_content = f"""
    <div style='background-color: rgba(255, 193, 7, 0.15); padding: 12px;
                border-left: 4px solid #ffc107; margin-bottom: 15px; border-radius: 4px;
                font-family: sans-serif;'>
        <h4 style='margin: 0 0 8px 0; color: #d39e00; display: flex; align-items: center;'>
            <span style='margin-right: 8px;'>⚠️</span> Stale Results Detected
        </h4>
        <p style='margin: 0; font-size: 13px; color: var(--colab-primary-text, #333);'>
            Upstream settings for <b>{trigger_source}</b> have been modified.
            Please click the <b>Run</b> button below to update this module.
        </p>
    </div>
    """
    for mod_id in module_ids:
        if mod_id in STALE_BANNERS:
            STALE_BANNERS[mod_id].value = html_content

def clear_stale(module_id: str):
    """Clears the warning banner once the module is successfully re-run."""
    if module_id in STALE_BANNERS:
        STALE_BANNERS[module_id].value = ""

ALL_DOWNSTREAM = ['overall', 'subgroup', 'regression', 'spline', 'pub_bias', 'pet_peese', 'loo', 'cumulative', 'plots']


# =============================================================================
# ESTIMATORS
# Provides multiple methods for estimating between-study variance
# =============================================================================



# --- RESTRICTED MAXIMUM LIKELIHOOD (REML) ---

def calculate_tau_squared_REML(df, effect_col, var_col, max_iter=100, tol=1e-8):
    """
    REML estimator for tau-squared

    Advantages:
    - Unbiased for tau²
    - Accounts for uncertainty in estimating mu
    - Better performance in small samples
    - Generally preferred in literature

    Disadvantages:
    - Iterative (slightly slower)
    - Can fail to converge in extreme cases

    Reference:
    Viechtbauer, W. (2005). Bias and efficiency of meta-analytic variance
    estimators in the random-effects model. Journal of Educational and
    Behavioral Statistics, 30(3), 261-293.

    Parameters:
    -----------
    df : DataFrame
        Data with effect sizes and variances
    effect_col : str
        Name of effect size column
    var_col : str
        Name of variance column
    max_iter, tol :
        Unused (kept for compatibility); fit_mv sets its own tolerances.

    Returns:
    --------
    float : tau-squared estimate
    """
    k = len(df)
    if k < 2:
        return 0.0

    try:
        # Extract data
        yi = df[effect_col].values
        vi = df[var_col].values

        # Remove any infinite or negative variances
        valid_mask = np.isfinite(vi) & (vi > 0)
        if not valid_mask.all():
            warnings.warn(f"Removed {(~valid_mask).sum()} observations with invalid variances")
            yi = yi[valid_mask]
            vi = vi[valid_mask]
            k = len(yi)

        if k < 2:
            return 0.0

        # Two-level REML (rma(method = 'REML')) with the shared engine, Cell 1b
        return max(0.0, tau2_univariate(yi, vi, method='REML'))

    except Exception as e:
        warnings.warn(f"Error in REML estimator: {e}, using DL fallback")
        return calculate_tau_squared_DL(df, effect_col, var_col)


# --- 3. MAXIMUM LIKELIHOOD (ML) ---

def calculate_tau_squared_ML(df, effect_col, var_col, max_iter=100, tol=1e-8):
    """
    Maximum Likelihood estimator for tau-squared

    Advantages:
    - Efficient asymptotically
    - Produces valid estimates

    Disadvantages:
    - Biased downward (underestimates tau²)
    - Less preferred than REML
    - REML is generally recommended instead

    Parameters:
    -----------
    df : DataFrame
        Data with effect sizes and variances
    effect_col : str
        Name of effect size column
    var_col : str
        Name of variance column
    max_iter, tol :
        Unused (kept for compatibility); fit_mv sets its own tolerances.

    Returns:
    --------
    float : tau-squared estimate
    """
    k = len(df)
    if k < 2:
        return 0.0

    try:
        yi = df[effect_col].values
        vi = df[var_col].values

        valid_mask = np.isfinite(vi) & (vi > 0)
        if not valid_mask.all():
            yi = yi[valid_mask]
            vi = vi[valid_mask]
            k = len(yi)

        if k < 2:
            return 0.0

        # Two-level ML (rma(method = 'ML')) with the shared engine, Cell 1b
        return max(0.0, tau2_univariate(yi, vi, method='ML'))

    except Exception as e:
        warnings.warn(f"Error in ML estimator: {e}, using DL fallback")
        return calculate_tau_squared_DL(df, effect_col, var_col)


# --- 4. PAULE-MANDEL (PM) ---

def calculate_tau_squared_PM(df, effect_col, var_col, max_iter=100, tol=1e-8):
    """
    Paule-Mandel estimator for tau-squared

    Solves Q(tau^2) = k - 1 by root finding (as metafor::rma(method =
    "PM")); tau^2 = 0 when Q(0) <= k - 1.

    Advantages:
    - Exact solution to Q = k-1 equation
    - Good performance

    Disadvantages:
    - Can be unstable with few studies
    - Requires iterative solution in practice

    Reference:
    Paule, R. C., & Mandel, J. (1982). Consensus values and weighting factors.
    Journal of Research of the National Bureau of Standards, 87(5), 377-385.

    Parameters:
    -----------
    df : DataFrame
        Data with effect sizes and variances
    effect_col : str
        Name of effect size column
    var_col : str
        Name of variance column
    max_iter : int
        Maximum iterations
    tol : float
        Convergence tolerance

    Returns:
    --------
    float : tau-squared estimate
    """
    k = len(df)
    if k < 2:
        return 0.0

    try:
        yi = df[effect_col].values
        vi = df[var_col].values

        valid_mask = np.isfinite(vi) & (vi > 0)
        if not valid_mask.all():
            yi = yi[valid_mask]
            vi = vi[valid_mask]
            k = len(yi)

        if k < 2:
            return 0.0

        df_Q = k - 1

        # PM: the root of Q(tau2) = k - 1, where Q is the generalized
        # Q statistic with weights 1/(vi + tau2). Q decreases in tau2, so the
        # root is unique; tau2 = 0 when Q(0) <= k - 1 (as metafor).
        def q_minus_df(tau2):
            wi = 1 / (vi + tau2)
            mu = (wi * yi).sum() / wi.sum()
            return (wi * (yi - mu)**2).sum() - df_Q

        if q_minus_df(0.0) <= 0:
            return 0.0

        # Bracket the root: Q -> 0 as tau2 -> infinity
        upper = max(np.var(yi), vi.max(), 1e-8)
        n_expand = 0
        while q_minus_df(upper) > 0:
            upper *= 10
            n_expand += 1
            if n_expand > 60:
                raise RuntimeError("could not bracket the PM root")

        from scipy.optimize import brentq
        tau_sq = brentq(q_minus_df, 0.0, upper, xtol=tol * 1e-4, rtol=1e-12,
                        maxiter=max(max_iter, 200))

        return max(0.0, tau_sq)

    except Exception as e:
        warnings.warn(f"Error in PM estimator: {e}, using DL fallback")
        return calculate_tau_squared_DL(df, effect_col, var_col)


# --- 5. SIDIK-JONKMAN (SJ) ---

def calculate_tau_squared_SJ(df, effect_col, var_col):
    """
    Sidik-Jonkman estimator for tau-squared

    Starts from the unweighted estimate tau0^2 = sum((y - mean(y))^2) / k
    and rescales it with the weighted residual sum of squares:

        w_i   = 1 / (v_i + tau0^2)
        tau^2 = tau0^2 * sum(w_i * (y_i - y_w)^2) / (k - 1)

    where y_w is the w-weighted mean. Same as metafor::rma(method = "SJ")
    with the default initial value. The estimate is never negative and is
    zero only when all y_i are equal.

    Advantages:
    - Simple, non-iterative
    - Good performance with few studies
    - Conservative (tends to produce larger estimates)

    Disadvantages:
    - Can be overly conservative
    - Less commonly used

    Reference:
    Sidik, K., & Jonkman, J. N. (2005). Simple heterogeneity variance
    estimation for meta-analysis. Journal of the Royal Statistical Society,
    Series C, 54(2), 367-384.

    Parameters:
    -----------
    df : DataFrame
        Data with effect sizes and variances
    effect_col : str
        Name of effect size column
    var_col : str
        Name of variance column

    Returns:
    --------
    float : tau-squared estimate
    """
    k = len(df)
    if k < 2:  # SJ needs at least 2 estimates (as metafor)
        return calculate_tau_squared_DL(df, effect_col, var_col)

    try:
        yi = df[effect_col].values
        vi = df[var_col].values

        valid_mask = np.isfinite(vi) & (vi > 0)
        if not valid_mask.all():
            yi = yi[valid_mask]
            vi = vi[valid_mask]
            k = len(yi)

        if k < 2:
            return calculate_tau_squared_DL(df, effect_col, var_col)

        # Initial (unweighted) estimate, as metafor: var(yi) * (k-1)/k
        tau2_0 = ((yi - yi.mean())**2).sum() / k
        if tau2_0 <= 0:
            return 0.0

        # Weighted residual sum of squares with weights 1/(vi + tau2_0)
        wi = 1 / (vi + tau2_0)
        y_w = (wi * yi).sum() / wi.sum()
        rss = (wi * (yi - y_w)**2).sum()

        tau_sq = tau2_0 * rss / (k - 1)

        return max(0.0, tau_sq)

    except Exception as e:
        warnings.warn(f"Error in SJ estimator: {e}, using DL fallback")
        return calculate_tau_squared_DL(df, effect_col, var_col)


def calculate_tau_squared_DL(df, effect_col, var_col):
    """
    DerSimonian-Laird estimator for tau-squared

    Advantages:
    - Simple, fast
    - Non-iterative
    - Always converges

    Disadvantages:
    - Can underestimate tau² in small samples
    - Negative values truncated to 0
    - Less efficient than ML methods

    Parameters:
    -----------
    df : DataFrame
        Data with effect sizes and variances
    effect_col : str
        Name of effect size column
    var_col : str
        Name of variance column

    Returns:
    --------
    float : tau-squared estimate
    """
    k = len(df)
    if k < 2:
        return 0.0

    try:
        # Fixed-effects weights
        w = 1 / df[var_col]
        sum_w = w.sum()

        if sum_w <= 0:
            return 0.0

        # Fixed-effects pooled estimate
        pooled_effect = (w * df[effect_col]).sum() / sum_w

        # Q statistic
        Q = (w * (df[effect_col] - pooled_effect)**2).sum()
        df_Q = k - 1

        # C constant
        sum_w_sq = (w**2).sum()
        C = sum_w - (sum_w_sq / sum_w)

        # Tau-squared
        if C > 0 and Q > df_Q:
            tau_sq = (Q - df_Q) / C
        else:
            tau_sq = 0.0

        return max(0.0, tau_sq)

    except Exception as e:
        warnings.warn(f"Error in DL estimator: {e}")
        return 0.0



def calculate_tau_squared(df, effect_col, var_col, method='REML', **kwargs):
    """
    Unified function to calculate tau-squared using specified method

    Parameters:
    -----------
    df : DataFrame
        Data with effect sizes and variances
    effect_col : str
        Name of effect size column
    var_col : str
        Name of variance column
    method : str
        Estimation method: 'DL', 'REML', 'ML', 'PM', 'SJ'
        Default: 'REML' (recommended)
    **kwargs : dict
        Additional arguments passed to estimator

    Returns:
    --------
    float : tau-squared estimate
    dict : additional information (method used, convergence, etc.)
    """
    method = method.upper()

    estimators = {
        'DL': calculate_tau_squared_DL,
        'REML': calculate_tau_squared_REML,
        'ML': calculate_tau_squared_ML,
        'PM': calculate_tau_squared_PM,
        'SJ': calculate_tau_squared_SJ
    }

    if method not in estimators:
        warnings.warn(f"Unknown method '{method}', using REML")
        method = 'REML'

    try:
        tau_sq = estimators[method](df, effect_col, var_col, **kwargs)

        info = {
            'method': method,
            'tau_squared': tau_sq,
            'tau': np.sqrt(tau_sq),
            'success': True
        }

        return tau_sq, info

    except Exception as e:
        warnings.warn(f"Error with {method}, falling back to DL: {e}")
        tau_sq = calculate_tau_squared_DL(df, effect_col, var_col)

        info = {
            'method': 'DL',
            'tau_squared': tau_sq,
            'tau': np.sqrt(tau_sq),
            'success': False,
            'fallback': True,
            'error': str(e)
        }

        return tau_sq, info

def calculate_re_pooled(df, tau_squared, effect_col, var_col, alpha=0.05):
    """Calculate Random-Effects pooled estimate with CI"""
    k = len(df)
    if k < 1: return np.nan, np.nan, np.nan, np.nan, np.nan
    try:
        w_re = 1 / (df[var_col] + tau_squared)
        sum_w_re = w_re.sum()
        if sum_w_re <= 0: return np.nan, np.nan, np.nan, np.nan, np.nan

        pooled_effect = (w_re * df[effect_col]).sum() / sum_w_re
        pooled_var = 1 / sum_w_re
        pooled_se = np.sqrt(pooled_var)

        z_crit = norm.ppf(1 - alpha / 2)
        ci_lower = pooled_effect - z_crit * pooled_se
        ci_upper = pooled_effect + z_crit * pooled_se

        # Calculate I-squared
        w_fixed = 1 / df[var_col]
        sum_w_fixed = w_fixed.sum()
        pooled_effect_fe = (w_fixed * df[effect_col]).sum() / sum_w_fixed
        Q = (w_fixed * (df[effect_col] - pooled_effect_fe)**2).sum()
        df_Q = k - 1
        I_sq = max(0, ((Q - df_Q) / Q) * 100) if Q > 0 else 0

        return pooled_effect, pooled_se, ci_lower, ci_upper, I_sq
    except Exception:
        return np.nan, np.nan, np.nan, np.nan, np.nan


# Citation for this tool, used in every reference list the notebook writes.
# Fill in once (authors, year, title, DOI) when the release is archived.
COMETA_CITATION = ("<b>[CITATION FOR THIS TOOL]:</b> Author, A. A. (202X). <i>CoMeta: A no-code, "
                   "glass-box platform for multilevel meta-analysis</i>. [Repository/DOI].")


def _alpha() -> float:
    """Significance level chosen in the Overall Meta-Analysis settings
    (default 0.05). Used wherever generated text calls a result
    'significant', so the text always follows the chosen alpha."""
    try:
        return float(ANALYSIS_CONFIG.get('global_settings', {}).get('alpha', 0.05))
    except Exception:
        return 0.05


def calculate_knapp_hartung_ci(yi, vi, tau_sq, pooled_effect, alpha=0.05, modified=True):
    """
    Calculate Knapp-Hartung adjusted confidence interval.
    If modified=True, applies the ad hoc truncation of Knapp & Hartung
    (2003): the scale factor Q/df is not allowed below 1, so the CI is
    never narrower than the standard random-effects CI (metafor's
    test="adhoc"; modified=False is metafor's test="knha").
    """
    yi = np.asarray(yi, dtype=np.float64)
    vi = np.asarray(vi, dtype=np.float64)
    df = len(yi) - 1
    if df < 1:
        return None  # KH needs at least two estimates

    # Random-effects weights
    wi_star = 1.0 / (vi + tau_sq)
    sum_wi_star = np.sum(wi_star)

    # Calculate Q statistic (residual heterogeneity)
    Q = np.sum(wi_star * (yi - pooled_effect)**2)

    # Standard random-effects variance
    var_standard = 1 / sum_wi_star

    # Knapp-Hartung adjusted variance
    var_KH = (Q / df) * var_standard

    # Ad hoc truncation (Knapp & Hartung 2003): variance not smaller than the standard RE variance
    if modified:
        var_KH = max(var_standard, var_KH)

    se_KH = np.sqrt(var_KH)

    # t-distribution critical value
    t_crit = t.ppf(1 - alpha/2, df)

    # Confidence interval
    ci_lower = pooled_effect - t_crit * se_KH
    ci_upper = pooled_effect + t_crit * se_KH

    # Test statistic and p-value
    t_stat = pooled_effect / se_KH
    p_value = 2 * (1 - t.cdf(abs(t_stat), df))

    return {
        'se_KH': se_KH,
        'var_KH': var_KH,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        't_stat': t_stat,
        't_crit': t_crit,
        'df': df,
        'p_value': p_value,
        'Q': Q
    }



def compare_tau_estimators(df, effect_col, var_col):
    """
    Compare all tau-squared estimators on the same dataset

    Useful for sensitivity analysis and understanding which method
    is most appropriate for your data.

    Parameters:
    -----------
    df : DataFrame
        Data with effect sizes and variances
    effect_col : str
        Name of effect size column
    var_col : str
        Name of variance column

    Returns:
    --------
    DataFrame : Comparison of all methods
    """
    methods = ['DL', 'REML', 'ML', 'PM', 'SJ']
    results = []

    for method in methods:
        try:
            tau_sq, info = calculate_tau_squared(df, effect_col, var_col, method=method)

            results.append({
                'Method': method,
                'τ²': tau_sq,
                'τ': np.sqrt(tau_sq),
                'Success': info['success']
            })
        except Exception as e:
            results.append({
                'Method': method,
                'τ²': np.nan,
                'τ': np.nan,
                'Success': False
            })

    comparison_df = pd.DataFrame(results)

    return comparison_df



def calculate_hedges_g_python(df):
    """Calculate Hedges' g using EXACT Gamma correction via log-gamma."""
    df = df.copy()

    # Pooled SD
    n_e, n_c = df['ne'], df['nc']
    sd_e, sd_c = df['sde'], df['sdc']
    mean_e, mean_c = df['xe'], df['xc']

    df_d = n_e + n_c - 2
    sd_pooled = np.sqrt(((n_e - 1)*sd_e**2 + (n_c - 1)*sd_c**2) / df_d)

    # Cohen's d
    d = (mean_e - mean_c) / sd_pooled

    # Hedges' correction (J) - via Log-Gamma to prevent overflow
    m = df_d
    log_J = gammaln(m / 2) - np.log(np.sqrt(m / 2)) - gammaln((m - 1) / 2)
    J = np.exp(log_J)

    g = d * J

    # Large-sample variance of g, as metafor::escalc(measure = "SMD") and
    # the Effect Size Calculation cell compute it (no J² factor)
    vg = (n_e + n_c) / (n_e * n_c) + (g**2 / (2 * (n_e + n_c)))

    return g, vg


#==============================================================================
# Meta regression function
#==============================================================================
import scipy.linalg


def _run_three_level_reml_regression_v2(analysis_data, moderator_col, effect_col, var_col,
                                        categorical_cols=None, reference_levels=None,
                                        center_continuous=True):
    """
    Three-level REML meta-regression with CR2 / Satterthwaite inference.

    Fitted with the shared engine (fit_blocks / fit_mv, Cell 1b) on the full
    sampling VCV, with the diagonal V as fallback. When no study has more
    than one effect size, sigma² is fixed at 0 (two-level model).
    """
    # =================================================================
    # 0. RESOLVE MODERATOR SPEC (backward-compatible overload)
    # =================================================================
    # `moderator_col` may be a single string (legacy: every existing caller)
    # or a list of column names (new multi-moderator path). A single string
    # resolves to a one-element list and builds [intercept, x] exactly as the
    # original code did, so legacy behavior is byte-identical.
    if isinstance(moderator_col, (list, tuple)):
        _moderator_cols = list(moderator_col)
    else:
        _moderator_cols = [moderator_col]

    _categorical_cols = set(categorical_cols or [])
    _continuous_cols = [m for m in _moderator_cols if m not in _categorical_cols]
    _reference_levels = dict(reference_levels or {})

    # Precompute treatment-coding reference levels and dummy level orders for
    # any categorical moderators, so each per-study slice codes consistently.
    _cat_levels = {}
    for m in _categorical_cols:
        levels = sorted(analysis_data[m].astype(str).unique().tolist())
        ref = _reference_levels.get(m, levels[0]) if levels else None
        if ref is not None:
            ref = str(ref)
        _cat_levels[m] = {'levels': levels,
                          'ref': ref,
                          'non_ref': [lv for lv in levels if lv != ref]}

    # NOTE: a duplicate of the layout builder used to sit INSIDE the loop
    # above, indented one level too far. It crashed with KeyError on the
    # second categorical moderator, because on the loop's first pass
    # _cat_levels holds only the first one while the inner loop already
    # walks every moderator -- so any model with two or more categorical
    # moderators failed before it could fit. (One categorical worked by
    # accident: a single pass fills _cat_levels before the inner loop reads
    # it.) It also shadowed the loop variable `m`. The block was dead code
    # regardless: _compute_layout() below is the canonical builder and its
    # result overwrites _layout/_term_groups on the very next statement, so
    # it is simply removed rather than dedented.

    def _compute_layout():
        term_groups = {}
        layout = [('__intercept__', None)]
        col = 1
        for m in _moderator_cols:
            if m in _categorical_cols:
                idxs = []
                for lv in _cat_levels[m]['non_ref']:
                    layout.append((m, lv))
                    idxs.append(col)
                    col += 1
                term_groups[m] = idxs
            else:
                layout.append((m, None))
                term_groups[m] = [col]
                col += 1
        return layout, term_groups

    _layout, _term_groups = _compute_layout()

    def _build_Xi_for_group(group, k):
        """Build the per-study design block from the canonical _layout.
        Continuous mods enter as their numeric column; categorical mods as
        treatment-coded dummies (reference dropped). Driven by the same
        _layout that defines _term_groups, guaranteeing column alignment."""
        cols = [np.ones(k, dtype=np.float64)]
        for src, lv in _layout[1:]:   # skip intercept
            if lv is not None:        # categorical dummy
                ser = group[src].astype(str).values
                cols.append((ser == lv).astype(np.float64))
            else:                      # continuous
                cols.append(group[src].values.astype(np.float64))
        return np.column_stack(cols)


    # =================================================================
    # 1. DATA PREPARATION
    # =================================================================
    # Keep each row's sampling-VCV identity through the sort/reset below.
    if 'attach_vcv_row_pos' in globals():
        analysis_data = attach_vcv_row_pos(analysis_data)
    analysis_data = analysis_data.sort_values(['id'], kind='mergesort').reset_index(drop=True)

    # Mean-center continuous moderators (the intercept is then the effect at
    # the moderator means). center_continuous=False keeps the intercept at
    # moderator = 0, which PET/PEESE need: their intercept IS the
    # bias-corrected effect (SE = 0 / variance = 0).
    for m in (_continuous_cols if center_continuous else []):
        col = pd.to_numeric(analysis_data[m], errors='coerce').astype(np.float64)
        analysis_data[m] = col - float(col.mean())

    vcv_dict = ANALYSIS_CONFIG.get('vcv_matrices', {})

    grouped = analysis_data.groupby('id', sort=False)

    y_all = []
    vcv_all_matrix = []
    vcv_all_diag = []
    X_all = []
    study_ids = []

    for study_id, group in grouped:
        k = len(group)
        study_ids.append(study_id)

        y_i = group[effect_col].values.astype(np.float64)
        y_all.append(y_i)

        vi = group[var_col].values.astype(np.float64)
        vcv_all_diag.append(np.diag(vi))

        # Sliced by row identity (not by shape), so a study that lost some
        # rows keeps the shared-control covariances of the rows it kept.
        if 'cluster_vcv' in globals():
            V_i, _src = cluster_vcv(vcv_dict, study_id, group, var_col)
            if _src == 'diagonal (VCV mismatch)':
                warnings.warn(f"Study {study_id}: cached sampling VCV could not be aligned "
                              f"to the rows used; a diagonal V is used for this study.",
                              RuntimeWarning)
        else:
            V_i = np.diag(vi)
        vcv_all_matrix.append(V_i)

        X_i = _build_Xi_for_group(group, k)
        X_all.append(X_i)

    N_total = len(analysis_data)
    M_studies = len(y_all)

    # --- Determine parameter count from the design just built ---
    # X_all entries are k_i x p; p is constant across studies.
    p_params = X_all[0].shape[1] if X_all else 2

    if M_studies < 2 or N_total < 3:
        return None, None, None

    # Within-study constancy: for multi-moderator, ALL moderator columns
    # must be constant within study. _moderator_cols is the resolved list
    # (single-string callers -> length-1 list; see the X-build block above).
    # Reported with the results (study-level vs within-study moderators);
    # it does NOT decide whether sigma^2 is estimated (see
    # needs_variance_regularization below).
    is_constant_within = all(
        analysis_data.groupby('id')[m].nunique().max() == 1
        for m in _moderator_cols
    )

    # sigma^2 (within-study, between-effect variance) is only weakly
    # identified when NO study contributes more than one effect size --
    # i.e. there is no within-study replication anywhere to separate
    # sigma^2 from tau^2. This is a DIFFERENT condition from
    # is_constant_within: a study-level moderator (the common case) makes
    # is_constant_within True even when many studies DO have multiple
    # effect sizes, which fully identifies sigma^2 without any
    # regularization. Gating the constrained optimizer on
    # is_constant_within (as before) pulled tau^2 toward a crude,
    # unweighted method-of-moments prior in that common case, biasing
    # sigma^2 (and downstream CR2 SEs) away from the unconstrained REML
    # optimum that metafor/clubSandwich compute.
    needs_variance_regularization = not any(
        X_j.shape[0] > 1 for X_j in X_all
    )

    # Variation guard: at least one moderator must vary across the dataset.
    mod_range_ok = any(
        (pd.to_numeric(analysis_data[m], errors='coerce').max()
         - pd.to_numeric(analysis_data[m], errors='coerce').min()) > 1e-10
        if m in _continuous_cols else
        analysis_data[m].astype(str).nunique() > 1
        for m in _moderator_cols
    )
    if not mod_range_ok:
        return None, None, None


    # =================================================================
    # 2. REML FIT with the shared engine (fit_blocks, Cell 1b): the full
    #    sampling VCV when any study has shared-control covariances, the
    #    diagonal V as fallback; sigma² fixed at 0 when no study has more
    #    than one effect size (tau² and sigma² not separately identified).
    # =================================================================
    has_off_diag = any(not np.allclose(m, np.diag(np.diag(m))) for m in vcv_all_matrix)
    try:
        fit = fit_blocks(y_all, vcv_all_matrix if has_off_diag else vcv_all_diag, X_all,
                         two_level=needs_variance_regularization,
                         V_fallback_blocks=vcv_all_diag if has_off_diag else None,
                         cluster_ids=study_ids)
    except (ValueError, RuntimeError, np.linalg.LinAlgError):
        return None, None, None

    if has_off_diag and not fit.v_fallback:
        final_vcv = vcv_all_matrix
        model_type = ("3-Level VCV" if not needs_variance_regularization
                      else "2-Level VCV (one effect per study; σ² = 0)")
    else:
        final_vcv = vcv_all_diag
        model_type = ("3-Level Diagonal" if not needs_variance_regularization
                      else "2-Level (one effect per study; σ² = 0)")
    best_res = fit_as_optimize_result(fit)

    # =================================================================
    # 3. INFERENCE
    # =================================================================

    try:
        final_est = fit_estimates(fit)

        betas = final_est['betas']
        se_betas = final_est['se_betas']

        _wald = None
        try:
            _rv_out = _compute_robust_var_betas(
                betas, y_all, final_vcv, X_all,
                final_est['tau_sq'], final_est['sigma_sq'],
                with_wald_helper=True
            )
            # with_wald_helper=True returns a 3-tuple.
            var_betas_robust, dfs_robust, _wald = _rv_out
            se_betas_robust = np.sqrt(np.diag(var_betas_robust))
            if np.all(np.isfinite(se_betas_robust)) and np.all(se_betas_robust > 0):
                se_betas = se_betas_robust
                df = dfs_robust  # Use the CR2 fractional DFs array
            else:
                var_betas_robust = final_est['cov_beta']
                df = np.full(p_params, max(1, M_studies - p_params))
                _wald = None  # SEs unreliable -> don't trust the Wald helper
        except Exception as e:
            warnings.warn(f"Meta-regression: CR2 step failed ({e}); using model-based SEs "
                          f"with df = studies - p.")
            var_betas_robust = final_est['cov_beta']
            df = np.full(p_params, max(1, M_studies - p_params))
            _wald = None

        # --- Omnibus tests via the AHZ (HTZ) Wald helper -----------------
        # Build per-factor column groups from the resolved design. Continuous
        # moderators occupy one column each; categorical factors occupy their
        # contiguous block of dummy columns. Column 0 is the intercept.
        omnibus = None
        if _wald is not None:
            try:
                # Use the canonical grouping built alongside the design (Step
                # 1), not a reconstruction — guarantees the contrasts target
                # the exact columns the design actually used.
                per_factor = {}
                # Joint test per categorical factor (>1 dummy column).
                for m in _categorical_cols:
                    idxs = _term_groups.get(m, [])

                    if not idxs:
                        continue
                    C = np.zeros((len(idxs), p_params))
                    for r, ci in enumerate(idxs):
                        C[r, ci] = 1.0
                    per_factor[m] = _wald(C, test='AHZ')

                # Overall test of ALL non-intercept coefficients jointly.
                overall = None
                non_int = list(range(1, p_params))
                if len(non_int) > 1:
                    C_all = np.zeros((len(non_int), p_params))


                    for r, ci in enumerate(non_int):
                        C_all[r, ci] = 1.0
                    overall = _wald(C_all, test='AHZ')

                omnibus = {'overall': overall, 'per_factor': per_factor}
            except Exception:
                omnibus = None

        # Inference (Arrays broadcast perfectly here)
        t_stats = betas / se_betas
        p_values = 2 * (1 - t.cdf(np.abs(t_stats), df))

        # Confidence intervals
        alpha = ANALYSIS_CONFIG.get('global_settings', {}).get('alpha', 0.05)
        crit_val = t.ppf(1 - alpha/2, df)
        ci_lower = betas - crit_val * se_betas
        ci_upper = betas + crit_val * se_betas

        final_est.update({
            'df': df,
            't_stats': t_stats,
            'p_values': p_values,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper,
            'var_betas_robust': var_betas_robust,
            'se_betas': se_betas,
            'model_type': model_type,
            'is_constant_within': is_constant_within,
            'variance_regularized': needs_variance_regularization,
            'n_studies': M_studies,
            'n_obs': N_total,
            'optimizer_converged': getattr(best_res, 'success', False),
            'omnibus': omnibus,
            # AHZ closure (Tipton & Pustejovsky 2015). Exposed so callers can
            # get a CONTRAST-specific small-sample DF -- at q = 1 it is the
            # Satterthwaite DF clubSandwich's coef_test reports, which is what
            # a predicted effect x0 @ beta needs. Without it a prediction would
            # fall back to min(df) over the coefficients it loads on, the same
            # degenerate rule that made every subgroup report an identical DF.
            # None when the robust fit was unusable; never serialisable.
            'htz_wald': _wald,
            # The exact inputs this fit used. The full sampling VCV or its
            # diagonal fallback is chosen at run time, and the design is
            # built internally from _layout on id-sorted data, so a caller
            # that wants to reproduce the fit elsewhere (the validation cell)
            # must be handed these rather than rebuild them and hope they
            # agree -- the failure mode that made Co-Meta and metafor fit
            # different Sigma matrices in the subgroup path.
            'vcv_used': final_vcv,
            'X_stacked': (np.vstack(X_all) if X_all else None),
            'y_stacked': (np.concatenate(y_all) if y_all else None),
            'row_ids': ([sid for sid, y_i in zip(study_ids, y_all)
                         for _ in range(len(y_i))] if y_all else None),
        })


        return final_est, (N_total, M_studies, p_params), best_res

    except Exception:
        return None, None, None


# =============================================================================
# CR2 CLUSTER-ROBUST INFERENCE (Satterthwaite df, HTZ Wald tests)
# =============================================================================

def _compute_robust_var_betas(betas, y_all, vcv_all, X_all, tau_sq, sigma_sq,
                                absorb_cluster_effects='auto',
                                with_wald_helper=False):
    """
    CR2 cluster-robust covariance of the fixed effects with Satterthwaite
    degrees of freedom (Pustejovsky & Tipton 2018), clustered by study,
    with the fitted working model (V_i + sigma² I + tau² J) as target:
    clubSandwich::vcovCR(rma.mv fit, type = "CR2") / coef_test().

    Returns (V_robust, df) -- df per coefficient -- or, with
    with_wald_helper=True, (V_robust, df, wald) where wald(C, test='AHZ')
    computes the HTZ Wald test of C @ beta = 0 (clubSandwich::Wald_test,
    test = "HTZ"); at q = 1 its denominator df is the Satterthwaite df.

    absorb_cluster_effects is accepted for compatibility and ignored.
    """
    M_studies = len(y_all)
    p_params = len(betas)
    betas = np.asarray(betas, dtype=np.float64).ravel()
    PINV_TOL = 1e-12

    if absorb_cluster_effects not in (False, 'auto'):
        warnings.warn(
            "absorb_cluster_effects is ignored: cluster-intercept absorption "
            "does not match clubSandwich/metafor for random-effects rma.mv and "
            "has been disabled.", stacklevel=2)
    absorb = False
    # ---------- Pass 1 (unchanged): build per-cluster blocks + bread ----
    sum_XtWX = np.zeros((p_params, p_params))
    studies = []
    for i in range(M_studies):
        y_i = np.asarray(y_all[i], dtype=np.float64).ravel()
        X_i = np.asarray(X_all[i], dtype=np.float64)
        if X_i.ndim == 1:
            X_i = X_i.reshape(1, -1)
        V_i = np.asarray(vcv_all[i], dtype=np.float64)
        n_i = len(y_i)
        if V_i.ndim == 0:
            V_i = np.array([[float(V_i)]])
        elif V_i.ndim == 1:
            V_i = np.diag(V_i)

        Phi_i = V_i + sigma_sq * np.eye(n_i) + tau_sq * np.ones((n_i, n_i))
        Phi_i = 0.5 * (Phi_i + Phi_i.T)

        eigvals, eigvecs = np.linalg.eigh(Phi_i)
        floor = max(1e-12,
                    1e-10 * float(eigvals.max()) if eigvals.size else 1e-12)
        eigvals_safe = np.maximum(eigvals, floor)
        Phi_i_pd = (eigvecs * eigvals_safe) @ eigvecs.T
        Phi_i_pd = 0.5 * (Phi_i_pd + Phi_i_pd.T)

        try:
            L_i = np.linalg.cholesky(Phi_i_pd)
        except np.linalg.LinAlgError:
            L_i = eigvecs * np.sqrt(eigvals_safe)

        W_i = (eigvecs * (1.0 / eigvals_safe)) @ eigvecs.T
        W_i = 0.5 * (W_i + W_i.T)

        XW_i = X_i.T @ W_i
        sum_XtWX += XW_i @ X_i

        sc = {
            'X_i':   X_i,
            'Phi_i': Phi_i_pd,
            'L_i':   L_i,
            'W_i':   W_i,
            'XW_i':  XW_i,
            'e_i':   y_i - X_i @ betas,
            'n_i':   n_i,
        }
        studies.append(sc)

    try:
        M_bread = np.linalg.inv(sum_XtWX)
    except np.linalg.LinAlgError:
        warnings.warn(
            "X'WX is singular — design matrix is rank-deficient. "
            "Cannot compute robust variance. Returning NaN.",
            stacklevel=2
        )
        nan_var = np.full((p_params, p_params), np.nan)
        nan_df  = np.full(p_params, np.nan)
        if with_wald_helper:
            return nan_var, nan_df, None
        return nan_var, nan_df

    M_bread = 0.5 * (M_bread + M_bread.T)

    # ---------- Pre-compute augmented bread if absorbing -----------------
    # When absorbing per-cluster intercepts, the augmented design is
    # X_aug = [X | Z] where Z is the J-column matrix of cluster indicators.
    # By the block-inverse formula, computing the augmented hat matrix
    # per cluster reduces to solving a small system PER CLUSTER:
    #   H_tilde_j = X_aug_j @ M_aug @ X_aug_j' @ W_j
    # We compute M_aug implicitly via a per-cluster augmentation.
    #
    # The clean approach: for each cluster j, form
    #   X_tilde_j = [X_j | 1_j]    (n_j x (p+1))
    # and compute the augmented bread JUST FOR THAT CLUSTER's H matrix.
    # The full augmented bread (X_aug' W X_aug)^{-1} requires inverting
    # a (p + J) x (p + J) matrix, but block-inverse formulas let us
    # extract only what we need per cluster.

    if absorb:
        # Build the per-cluster auxiliary needed for absorbed H_j.
        # We use the Frisch-Waugh-Lovell representation:
        # the augmented hat matrix H_tilde_j satisfies
        #   I - H_tilde_j = M_j - M_j X_j (X' M X)_resid^{-1} X_j' M_j
        # where M_j = I - P_{1_j} is the within-cluster demeaning operator
        # in the W_j metric.
        #
        # Practically, this means: demean X_j and y_j within each cluster
        # using W_j-weighted means, build the bread on demeaned data, and
        # use the demeaned residual operator for adjustment.
        #
        # Per-cluster W-weighted projector onto 1_j:
        #   P_1j = 1_j (1_j' W_j 1_j)^{-1} 1_j' W_j
        # Demeaning operator:
        #   Q_j = I - P_1j
        # Demeaned design:
        #   X_tilde_j = Q_j @ X_j   (note: removes cluster-specific weighted mean)
        # The "absorbed bread" is then:
        #   M_abs = (sum_j X_tilde_j' W_j X_tilde_j)^{-1}
        # But — and this is the key point — we want to KEEP the original
        # bread for the score and the sandwich, and only use M_abs for
        # the H matrix in the CR2 adjustment.

        sum_XtWX_abs = np.zeros((p_params, p_params))
        for sc in studies:
            X_j = sc['X_i']
            W_j = sc['W_i']
            n_j = sc['n_i']
            if n_j > 1:
                ones_j = np.ones(n_j)
                W_ones = W_j @ ones_j
                denom = float(ones_j @ W_ones)
                if denom > 0:
                    # Q_j @ X_j  (n_j x p)
                    weighted_mean = (W_ones @ X_j) / denom   # (p,)
                    X_demeaned = X_j - np.outer(ones_j, weighted_mean)
                else:
                    X_demeaned = X_j
            else:
                # Singleton clusters: absorption removes them entirely
                X_demeaned = np.zeros_like(X_j)

            sum_XtWX_abs += X_demeaned.T @ W_j @ X_demeaned
            sc['X_demeaned'] = X_demeaned

        try:
            M_bread_abs = np.linalg.inv(sum_XtWX_abs)
            M_bread_abs = 0.5 * (M_bread_abs + M_bread_abs.T)
            abs_failed = False
        except np.linalg.LinAlgError:
            warnings.warn(
                "Absorbed-design bread is singular. Falling back to "
                "non-absorbed CR2 (target=NULL behaviour).",
                stacklevel=2
            )
            M_bread_abs = M_bread
            abs_failed = True
    else:
        M_bread_abs = M_bread
        abs_failed = False

    # Cholesky of (the bread we'll use in H_array for Satterthwaite)
    try:
        L_M = np.linalg.cholesky(M_bread)
    except np.linalg.LinAlgError:
        eigvals_M, eigvecs_M = np.linalg.eigh(M_bread)
        floor_M = max(1e-12, 1e-10 * float(eigvals_M.max()))
        eigvals_M = np.maximum(eigvals_M, floor_M)
        M_psd = (eigvecs_M * eigvals_M) @ eigvecs_M.T
        try:
            L_M = np.linalg.cholesky(0.5 * (M_psd + M_psd.T))
        except np.linalg.LinAlgError:
            L_M = eigvecs_M * np.sqrt(eigvals_M)

    # ---------- Pass 2: CR2 adjustment and meat ------------------------
    meat = np.zeros((p_params, p_params))

    for sc in studies:
        X_i   = sc['X_i']
        L_i   = sc['L_i']
        XW_i  = sc['XW_i']
        e_i   = sc['e_i']
        n_i   = sc['n_i']
        Phi_i = sc['Phi_i']
        W_i   = sc['W_i']

        if absorb and not abs_failed and n_i > 1:
            # Augmented hat: use the demeaned X and the absorbed bread.
            # H_tilde_j[a,b] is computed via the augmented design's
            # contribution. The Frisch-Waugh result tells us:
            #   I - H_tilde_j = Q_j - Q_j X_j M_abs X_j' Q_j' W_j
            # where Q_j is the within-cluster W-demeaner.
            X_dem_i = sc['X_demeaned']
            # Within-cluster W-demean operator applied to identity:
            ones_i = np.ones(n_i)
            W_ones = W_i @ ones_i
            denom = float(ones_i @ W_ones)
            # Q_i = I - 1 (1' W 1)^{-1} 1' W
            Q_i = np.eye(n_i) - np.outer(ones_i, W_ones) / denom
            # I - H_tilde = Q - Q X M_abs X' Q' W
            IH_i = Q_i - X_dem_i @ M_bread_abs @ X_dem_i.T @ W_i
        else:
            # Standard (non-absorbed) operator
            IH_i = np.eye(n_i) - X_i @ M_bread @ XW_i

        G_i = L_i.T @ IH_i @ Phi_i @ L_i
        G_i = 0.5 * (G_i + G_i.T)

        eigvals, eigvecs = np.linalg.eigh(G_i)
        val_inv_sqrt = np.where(eigvals > PINV_TOL,
                                1.0 / np.sqrt(np.maximum(eigvals, PINV_TOL)),
                                0.0)
        G_inv_sqrt = (eigvecs * val_inv_sqrt) @ eigvecs.T

        A_i = L_i @ G_inv_sqrt @ L_i.T

        # Score uses the ORIGINAL XW_i (original parameter space)
        EA_i = XW_i @ A_i
        g_i  = EA_i @ e_i
        meat += np.outer(g_i, g_i)

        sc['EA_i'] = EA_i

        if with_wald_helper:
            sc['A_i']   = A_i
            sc['IH_i']  = IH_i

    var_robust = M_bread @ meat @ M_bread
    var_robust = 0.5 * (var_robust + var_robust.T)

    eigvals_vr = np.linalg.eigvalsh(var_robust)
    if eigvals_vr.size and eigvals_vr.min() < -1e-8 * max(abs(eigvals_vr.max()), 1.0):
        warnings.warn(
            f"Robust variance has negative eigenvalue ({eigvals_vr.min():.2e}). "
            f"J={M_studies}, p={p_params}; Tipton (2015) recommends J >= 4p.",
            stacklevel=2
        )

    # ---------- Pass 3: Satterthwaite df (unchanged) -------------------
    H_stack = np.zeros((M_studies, p_params, p_params))
    G_norms = np.zeros((M_studies, p_params))

    for j_idx, sc in enumerate(studies):
        EA_j = sc['EA_i']
        X_j  = sc['X_i']
        L_j  = sc['L_i']

        ME_j    = M_bread @ EA_j
        H_per_j = ME_j @ X_j @ L_M
        G_per_j = ME_j @ L_j

        H_stack[j_idx] = H_per_j
        G_norms[j_idx] = np.sum(G_per_j * G_per_j, axis=1)

    df_vec = np.zeros(p_params)
    # df_upper: absorption reduces df by # of absorbed dimensions (≤ J)
    n_absorbed = (M_studies if absorb and not abs_failed else 0)
    df_fallback   = max(1.0, float(M_studies - p_params))

    for i in range(p_params):
        H_mat  = H_stack[:, i, :]
        Gram   = H_mat @ H_mat.T
        diag_G = np.diag(Gram).copy()
        gN     = G_norms[:, i]

        trace_P  = float(np.sum(gN - diag_G))
        sum_P_sq = float(np.sum(Gram * Gram)
                         - 2.0 * np.sum(diag_G * gN)
                         + np.sum(gN * gN))

        if sum_P_sq > 0.0 and trace_P > 0.0:
            # Reported as is, like clubSandwich (no clipping to [1, J - 1]):
            # a df below 1 is a real warning sign and must not be hidden.
            df_vec[i] = float(trace_P * trace_P / sum_P_sq)
        else:
            df_vec[i] = df_fallback

    # If the caller didn't ask for the Wald helper, return the original
    # 2-tuple — call sites that pre-date `with_wald_helper` are byte-identical.
    if not with_wald_helper:
        return var_robust, df_vec

    # ------------------------------------------------------------------
    # OPTIONAL: Hotelling-T^2 Wald helper for testing H0: C @ betas == 0.
    #
    # Implements the Approximate Hotelling-Zhang (AHZ) test of Tipton &
    # Pustejovsky (2015), J. Educ. Behav. Stat. 40(6):604-634. AHZ is the
    # variant clubSandwich exposes as test="HTZ" and is the level-alpha
    # test the paper's simulations recommend (Findings 2-3).
    #
    # Specifically we implement:
    #   - statistic:    Eq. 10  F = T^2 * (eta - q + 1) / (eta * q)
    #                          referred to F(q, eta - q + 1)
    #   - df:           Eq. 14  eta_Z = q*(q+1) / sum_{s,t} Var(d_{st})
    #   - moments:      Thm 1 / Eq. 8 with u1=u3=e_s, u2=u4=e_t
    #   - B_j matrices: Eq. 9   B_j = Omega^{-1/2} C M X_j' W_j A_j
    #                                 (I_K - X M X' W)_j
    #     with Omega = E[C V_CR2 C'] = sum_j (un-normalised B_j B_j'),
    #     as clubSandwich (not C M C', which differs when CR2 needs a
    #     pseudo-inverse)
    #
    # AHA/AHB variants from the paper share the per-cluster machinery
    # but differ in how Var(d_{st}) is aggregated into eta. We leave them
    # as NotImplementedError stubs: the dispatch is in place for later,
    # the AHZ default matches clubSandwich, and routing is done by the
    # GUI layer so an end-user can't trigger an unimplemented branch.
    #
    # ALGORITHM (cluster count J, model dim p, total effects K):
    #
    #   For each cluster j, B_j is q x K. Its block of columns at cluster k is
    #     B_j[:, cluster_k] = F_j @ IH_j                  if k == j
    #                       = - R_j @ (M @ XW_k)          if k != j
    #   where F_j = Omega^{-1/2} C M XW_j A_j  (q x n_j)
    #         R_j = F_j @ X_j                  (q x p)
    #         M @ XW_k is the global "back loading" for cluster k.
    #
    # Theorem 1 + Eq. 14, after summing Var(d_{st}) over (s,t)=1..q, gives:
    #     sum_{s,t} Var(d_{st})
    #         = sum_{i,j} [ ||P_{ij}||_F^2  +  tr(P_{ij})^2 ]
    #     with P_{ij} = B_i B_j'.
    # Two clean O(J) collapses:
    #     sum_{i,j} ||B_i B_j'||_F^2  =  tr(S^2),  S = sum_j B_j' B_j   (K x K)
    #     sum_{i,j} tr(B_i B_j')^2    =  || G G' ||_F^2,  G = stack_vec(B_j)
    # so eta_Z costs one K x K accumulation and one J x J Gram, both cheap
    # at typical meta-analytic sizes.
    # ------------------------------------------------------------------

    # Capture only what the helper needs (no closure over `studies`).
    _EA_list = [sc['EA_i'] for sc in studies]
    _X_list  = [sc['X_i']  for sc in studies]
    _L_list  = [sc['L_i']  for sc in studies]
    _A_list  = [sc['A_i']  for sc in studies]
    _IH_list = [sc['IH_i'] for sc in studies]
    _XW_list = [sc['XW_i'] for sc in studies]
    _n_list  = [int(sc['n_i']) for sc in studies]
    _betas   = np.asarray(betas, dtype=np.float64).ravel()
    _Vrob    = var_robust
    _M       = M_bread
    _p       = p_params
    _J       = M_studies

    # Column offsets so we can write each cluster's slice of B_j into a
    # single q x K row vector. K = sum of n_j across clusters.
    _K        = int(sum(_n_list))
    _col_ofs  = np.concatenate(([0], np.cumsum(_n_list))).astype(int)  # length J+1

    # Precompute the global "back loadings" M @ XW_k for each cluster k.
    # These are p x n_k; they're what couples B_j to clusters k != j.
    _MXW_list = [_M @ XW for XW in _XW_list]

    from scipy.stats import f as _f_dist

    def _htz_wald(C, test='AHZ'):
        """Hotelling-type small-sample F-test of H0: C @ betas == 0.

        Parameters
        ----------
        C    : array-like, shape (q, p) or (p,). 1-D inputs are treated
               as a single-row constraint.
        test : {'AHZ'} (default). 'AHA' and 'AHB' are recognised but
               not yet implemented; calling them raises NotImplementedError.
               The GUI is the safety net for routing.

        Returns
        -------
        dict with keys: 'F', 'df_num', 'df_denom', 'p_val', 'T2', 'eta',
                        'test'.
        """
        Cm = np.atleast_2d(np.asarray(C, dtype=np.float64))
        q = Cm.shape[0]
        if Cm.shape[1] != _p:
            raise ValueError(
                f"Constraint matrix has {Cm.shape[1]} columns but model "
                f"has {_p} parameters."
            )

        # --- Dispatch -------------------------------------------------
        if test == 'AHA':
            raise NotImplementedError(
                "AHA variant (Pan & Wall 2002 aggregation) is not "
                "implemented; use test='AHZ' (Tipton & Pustejovsky 2015 "
                "Eq. 12 vs Eq. 14).")
        if test == 'AHB':
            raise NotImplementedError(
                "AHB variant (lower-triangle aggregation, Eq. 13) is not "
                "implemented; use test='AHZ'.")
        if test != 'AHZ':
            raise ValueError(f"Unknown test='{test}'. Supported: 'AHZ'.")

        # ============================================================
        # AHZ implementation
        # ============================================================

        # --- Test statistic T^2 ---------------------------------------
        Cb   = Cm @ _betas
        CVCt = Cm @ _Vrob @ Cm.T
        CVCt = 0.5 * (CVCt + CVCt.T)
        try:
            T2 = float(Cb @ np.linalg.solve(CVCt, Cb))
        except np.linalg.LinAlgError:
            ev, evec = np.linalg.eigh(CVCt)
            floor = max(1e-12, 1e-10 * float(ev.max()) if ev.size else 1e-12)
            ev = np.maximum(ev, floor)
            CVCt_inv = (evec * (1.0 / ev)) @ evec.T
            T2 = float(Cb @ CVCt_inv @ Cb)

        # --- Un-normalised B_j (q x K): C M X_j' W_j A_j e_j as a linear
        # map of the standardised data (Eq. 9 without Omega^{-1/2}).
        CM_raw = Cm @ _M
        B_stack = np.empty((_J, q, _K))
        for j in range(_J):
            F_j = CM_raw @ _XW_list[j] @ _A_list[j]    # q x n_j
            R_j = F_j @ _X_list[j]                     # q x p
            B_j = np.zeros((q, _K))
            for k in range(_J):
                c0, c1 = _col_ofs[k], _col_ofs[k + 1]
                if k == j:
                    B_j[:, c0:c1] = (F_j @ _IH_list[j]) @ _L_list[j]
                else:
                    B_j[:, c0:c1] = (-R_j @ _MXW_list[k]) @ _L_list[k]
            B_stack[j] = B_j

        # --- Omega = E[C V_CR2 C'] under the working model = sum_j B_j B_j'
        # (clubSandwich: Omega <- apply(P_array, 1:2, function(x) sum(diag(x)))).
        # It equals C M C' only when the CR2 adjustment is exact; when a
        # coefficient rests on a single study the adjustment matrices need a
        # pseudo-inverse and C M C' overstates eta (e.g. 535 vs 3.85 df).
        Omega = np.einsum('jsk,jtk->st', B_stack, B_stack)
        Omega = 0.5 * (Omega + Omega.T)
        ev_O, evec_O = np.linalg.eigh(Omega)
        floor_O = max(1e-12, 1e-10 * float(ev_O.max()) if ev_O.size else 1e-12)
        ev_O = np.maximum(ev_O, floor_O)
        Omega_isqrt = (evec_O * (1.0 / np.sqrt(ev_O))) @ evec_O.T  # q x q
        B_stack = np.einsum('st,jtk->jsk', Omega_isqrt, B_stack)

        # --- sum_{s,t} Var(d_st) = sum_{i,j} [ tr(M_ij^2) + tr(M_ij)^2 ],
        #     M_ij = B_i B_j'.  Mirrors clubSandwich total_variance_mat.
        #     FIRST term is tr(M_ij @ M_ij) -- the trace of the SQUARE,
        #     NOT ||M_ij||_F^2 = tr(M_ij @ M_ij').  They agree only for the
        #     symmetric self-blocks (i==j) or q==1; the non-symmetric
        #     cross-cluster blocks (i!=j) are where the Frobenius form
        #     over-counts and deflates eta.
        M = np.einsum('isk,jtk->ijst', B_stack, B_stack, optimize=True)
        term_sq   = float(np.einsum('ijst,ijts->', M, M, optimize=True))  # Σ tr(M_ij²)
        tr_M      = np.einsum('ijss->ij', M)                              # tr(M_ij)
        term_trsq = float(np.sum(tr_M * tr_M))                            # Σ tr(M_ij)²
        denom = term_sq + term_trsq


        if denom <= 0.0:
              eta = float(max(1.0, _J - q))
        else:
              eta = float(q * (q + 1)) / denom

        # --- Statistic + reference distribution (Eq. 10) -------------
        # F = T^2 (eta - q + 1) / (eta * q),  df = (q, eta - q + 1).
        # eta - q + 1 must stay positive; clip with a soft floor.
        df_denom = max(1.0, eta - q + 1.0)
        scale    = df_denom / (eta * q) if eta > 0 else 1.0 / q
        F_stat   = T2 * scale
        p_val    = float(_f_dist.sf(F_stat, q, df_denom))

        return {
            'F':        F_stat,
            'df_num':   int(q),
            'df_denom': df_denom,
            'p_val':    p_val,
            'T2':       T2,
            'eta':      eta,
            'test':     'AHZ',
        }

    return var_robust, df_vec, _htz_wald


# =============================================================================
# SPLINE META-REGRESSION
# =============================================================================


def spline_prediction_band(res, x_grid, alpha=0.05, dist_type='t'):
    """Fitted spline curve and pointwise (1 - alpha) confidence band on x_grid.

    Uses the model's coefficient covariance (CR2 when available) and, for
    dist_type 't', df = studies - coefficients, in line with study-level
    cluster-robust inference. Validation 9 checks the band's coverage.
    Returns (y_pred, se, lower, upper, df_t); df_t is None for the normal.
    """
    import patsy
    betas = np.asarray(res['betas'], float)
    cov = res.get('cov_beta', res.get('var_betas'))
    formula = res['formula']
    reg = res['reg_df']
    mod_col = res['moderator_col']
    x_grid = np.asarray(x_grid, float)
    # The fallback engine fits on the standardised moderator without patsy's intercept
    if "- 1" in formula:
        orig_x = (reg[mod_col].values - res['mod_mean']) / res['mod_std']
        grid_x = (x_grid - res['mod_mean']) / res['mod_std']
    else:
        orig_x, grid_x = reg[mod_col].values, x_grid
    # Rebuild the original design so the grid uses the same knots
    design_info = patsy.dmatrix(formula, {"x": orig_x}, return_type='matrix').design_info
    X_pred = np.asarray(patsy.build_design_matrices([design_info], {"x": grid_x})[0])
    if X_pred.shape[1] == len(betas) - 1:
        X_pred = np.column_stack([np.ones(len(x_grid)), X_pred])
    elif X_pred.shape[1] != len(betas):
        raise ValueError(f"Matrix shape mismatch: X_pred has {X_pred.shape[1]} cols, betas has {len(betas)}.")
    y_pred = X_pred @ betas
    se = np.sqrt(np.maximum(np.sum((X_pred @ np.asarray(cov, float)) * X_pred, axis=1), 0.0))
    if dist_type == 't':
        df_t = max(1, int(reg['id'].nunique()) - len(betas))
        crit = stats.t.ppf(1 - alpha / 2, df_t)
    else:
        df_t, crit = None, stats.norm.ppf(1 - alpha / 2)
    return y_pred, se, y_pred - crit * se, y_pred + crit * se, df_t


def _run_robust_spline_analysis(df, moderator_col, effect_col, var_col,
                                df_spline=3, joint_reml=True):
    """
    Robust 3-Level Non-Linear Meta-Regression using Natural Cubic Splines.

    WHAT IT DOES:
    -------------
    Tests for a relationship between a continuous moderator and effect sizes
    while accounting for the hierarchical/nested structure of the data
    (multiple effect sizes per study).

    VARIANCE COMPONENTS (joint REML, plug-in fallback):
      The between-study (tau^2) and within-study (sigma^2) variance components
      are estimated by REML *under the spline model itself*: the fixed effects
      (spline coefficients) are profiled out by GLS, leaving a stable 2-D
      optimization over (tau^2, sigma^2). The linear-model fit is used for
      starting values and as a fallback if the joint optimizer fails to improve
      on it. The path actually used is recorded in result['variance_estimation'].

      Why this matters: estimating the components from the LINEAR model and
      freezing them (the old "plug-in" approach) inflates tau^2 when the true
      relationship is curved, because the linear model dumps unmodelled
      curvature into between-study heterogeneity. Joint estimation removes that
      bias.

    EXECUTION:
      1. Generate a collinearity-free, centered Natural Cubic Spline basis.
      2. Estimate (tau^2, sigma^2) by REML under the spline model (or fall back
         to the linear-model plug-in values).
      3. Solve spline coefficients via GLS at those components.
      4. Robust CR2 + Satterthwaite inference on the coefficients.
      5. Wald-type omnibus test on the non-intercept coefficients.

    NOTE ON THE OMNIBUS TEST: it tests betas[1:] == 0 against a basis that
    INCLUDES the linear trend, so it answers "does the moderator matter?", not
    "is the relationship non-linear?" (a purely linear effect will reject). It
    is the HTZ Wald test (clubSandwich::Wald_test, test = "HTZ"; checked in
    Validations 6, 8 and 10). Non-linearity is tested separately in the spline
    cell by an ML likelihood-ratio test of the spline against the linear model.

    ARGUMENTS:
    ----------
    df            : pd.DataFrame : Dataset with effect sizes and moderators.
    moderator_col : str          : Continuous moderator column to test.
    effect_col    : str          : Effect-size column (e.g. 'lnRR', 'hedges_g').
    var_col       : str          : Sampling-variance column.
    df_spline     : int          : Spline degrees of freedom (Default: 3).
    joint_reml    : bool         : Estimate (tau^2, sigma^2) by REML under the
                                   spline model (Default: True; shared engine,
                                   Cell 1b). If False, use the linear-model
                                   plug-in values directly.

    RETURNS:
    --------
    tuple : (spline_est_dict, error_message_str)
    """

    # -----------------------------------------------------------------
    # 0. Input validation + missing-data cleaning
    # -----------------------------------------------------------------
    required_cols = [moderator_col, effect_col, var_col, 'id']
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        return None, f"Missing required column(s): {missing_cols}"

    n_before = len(df)
    if 'attach_vcv_row_pos' in globals():
        df = attach_vcv_row_pos(df)      # keeps VCV row identity through reset_index
    df_clean = df.dropna(subset=required_cols).reset_index(drop=True)
    n_dropped = n_before - len(df_clean)
    if n_dropped > 0:
        warnings.warn(
            f"_run_robust_spline_analysis: dropped {n_dropped} row(s) with "
            f"missing values in {required_cols}. The sampling VCV of each affected "
            f"study is sliced to the rows kept.",
            RuntimeWarning,
        )
    if len(df_clean) == 0:
        return None, "No complete rows remain after dropping missing values."

    # -----------------------------------------------------------------
    # 1. Sort + raw moderator + basis (centered, full rank)
    # -----------------------------------------------------------------
    df_sorted = df_clean.sort_values('id', kind='mergesort').reset_index(drop=True)
    mod_vals = df_sorted[moderator_col].values.astype(np.float64)

    if not np.isfinite(mod_vals).all():
        return None, "Moderator contains non-finite values (inf) after NaN removal."

    n_unique_mod = len(np.unique(mod_vals))
    if n_unique_mod <= df_spline:
        return None, (
            f"Insufficient moderator support: {n_unique_mod} unique value(s) "
            f"for a spline with df={df_spline}. Reduce df_spline."
        )

    mod_mean = np.mean(mod_vals)   # reporting only
    mod_std = np.std(mod_vals)     # reporting only (ddof=0, kept for back-compat)

    try:
        import patsy
        # constraints='center': patsy's cr() basis columns sum to 1 (partition
        # of unity), so without centering the basis spans the constant and is
        # collinear with patsy's intercept -> identical exploded SEs. Centering
        # orthogonalises the basis against the constant. Column count/ordering
        # preserved: [Intercept, df_spline spline cols], full rank; betas[0] is
        # the intercept, betas[1:] the curve terms.
        formula = f"cr(x, df={df_spline}, constraints='center')"
        basis_matrix = patsy.dmatrix(
            formula, {"x": mod_vals}, return_type='matrix', NA_action="raise"
        )
        basis_matrix = np.asarray(basis_matrix)
    except Exception as e:
        return None, f"Patsy matrix generation error: {e}"

    if basis_matrix.shape[0] != len(df_sorted):
        return None, (
            f"Basis/row count mismatch: basis has {basis_matrix.shape[0]} rows "
            f"vs {len(df_sorted)} data rows. Aborting to avoid misalignment."
        )

    basis_cond = float(np.linalg.cond(basis_matrix))
    if basis_cond > 1e12:
        return None, (
            f"Spline basis is numerically singular (condition number "
            f"{basis_cond:.2e}). Check constraints='center' / reduce df_spline."
        )
    elif basis_cond > 1e8:
        warnings.warn(
            f"Spline basis is ill-conditioned (condition number "
            f"{basis_cond:.2e}); standard errors may be unstable.",
            RuntimeWarning,
        )

    # -----------------------------------------------------------------
    # 2. Group data into per-study (y, V, X) blocks
    # -----------------------------------------------------------------
    grouped = df_sorted.groupby('id', sort=False)
    vcv_dict = ANALYSIS_CONFIG.get('vcv_matrices', {})

    y_all, vcv_all, X_all = [], [], []
    vcv_fallback_studies = []

    global_idx = 0
    for study_id, group in grouped:
        k = len(group)
        y_all.append(group[effect_col].values.astype(np.float64))

        if 'cluster_vcv' in globals():
            V_i, _src = cluster_vcv(vcv_dict, study_id, group, var_col)
            if _src == 'diagonal (VCV mismatch)':
                vcv_fallback_studies.append(study_id)
        else:
            V_i = np.diag(group[var_col].values.astype(np.float64))
        vcv_all.append(V_i)

        X_all.append(basis_matrix[global_idx: global_idx + k, :])
        global_idx += k

    if vcv_fallback_studies:
        warnings.warn(
            f"VCV shape mismatch for study/studies {vcv_fallback_studies}; "
            f"fell back to diagonal sampling variances (within-study "
            f"correlation discarded).",
            RuntimeWarning,
        )

    N_total = len(df_sorted)
    M_studies = len(y_all)
    p_params = X_all[0].shape[1]

    if M_studies <= p_params:
        warnings.warn(
            f"Cluster count ({M_studies}) <= parameters ({p_params}); "
            f"small-sample inference is unreliable.",
            RuntimeWarning,
        )

    # -----------------------------------------------------------------
    # 3. Linear-model fit: fallback (plug-in) components
    # -----------------------------------------------------------------
    lin_est, lin_info, _ = _run_three_level_reml_regression_v2(
        df_clean, moderator_col, effect_col, var_col
    )
    if lin_est is not None:
        tau2_plugin = float(lin_est['tau_sq'])
        sigma2_plugin = float(lin_est['sigma_sq'])
    else:
        tau2_plugin = sigma2_plugin = None

    # -----------------------------------------------------------------
    # 4. REML estimation of (tau^2, sigma^2) under the spline model with
    #    the shared engine (fit_blocks, Cell 1b). The linear-model
    #    components are the fallback (plug-in).
    # -----------------------------------------------------------------
    variance_estimation = None
    var_estimation_reason = None
    fit = None

    if joint_reml:
        try:
            fit = fit_blocks(y_all, vcv_all, X_all)
            variance_estimation = "Joint REML"
            if not fit.converged:
                var_estimation_reason = ("optimizer success flag False; the REML maximum "
                                         "over the interior and the boundary was accepted")
        except Exception as e:
            var_estimation_reason = f"joint REML unavailable: {e}"

    # Fallback to plug-in if joint was disabled or failed.
    if variance_estimation is None:
        if lin_est is None:
            return None, (
                "Variance components unavailable: joint REML did not succeed "
                "and the linear base-model fallback failed to converge."
            )
        try:
            fit = fit_blocks(y_all, vcv_all, X_all, tau2=tau2_plugin, sigma2=sigma2_plugin)
        except Exception:
            return None, "GLS Matrix inversion failed at the chosen variance components."
        variance_estimation = "Plug-in (linear model)"
        if not joint_reml:
            var_estimation_reason = "joint_reml=False; plug-in used by request"

    # REML log-likelihood of the spline model at the linear-model components
    ll_start = None
    if lin_est is not None:
        try:
            ll_start = fit_blocks(y_all, vcv_all, X_all,
                                  tau2=tau2_plugin, sigma2=sigma2_plugin).loglik
        except Exception:
            ll_start = None

    # -----------------------------------------------------------------
    # 5. GLS estimates at the chosen variance components
    # -----------------------------------------------------------------
    tau2_used, sigma2_used = float(fit.tau2), float(fit.sigma2)
    spline_est = fit_estimates(fit)
    betas = spline_est['betas']
    cov_beta = spline_est['cov_beta']

    # -----------------------------------------------------------------
    # 5b. CR2 + Satterthwaite robust inference (with provenance)
    # -----------------------------------------------------------------
    inference_se_method = "Naive-GLS"
    robust_fallback_reason = None
    htz_fn = None

    _robust_fn = globals().get('_compute_robust_var_betas', None)
    if _robust_fn is None:
        robust_fallback_reason = "_compute_robust_var_betas not found in scope."
        spline_est['se_betas'] = np.sqrt(np.maximum(np.diag(cov_beta), 0.0))
    else:
        try:
            robust_out = _robust_fn(
                betas, y_all, vcv_all, X_all, tau2_used, sigma2_used,
                with_wald_helper=True
            )
            # _compute_robust_var_betas returns (var, df, htz_fn)  when the
            # helper is requested; keep the 2-tuple fallback in mind for
            # older implementations that don't accept the kwarg.
            if isinstance(robust_out, tuple) and len(robust_out) == 3:
                var_robust, df_robust, htz_fn = robust_out
            else:
                var_robust, df_robust = robust_out
                htz_fn = None

            se_robust = np.sqrt(np.maximum(np.diag(var_robust), 0.0))
            if np.any(se_robust == 0) or np.isnan(se_robust).any():
                robust_fallback_reason = (
                    "CR2 produced zero/NaN SEs (likely non-PSD CRVE); "
                    "used naive GLS SEs.")
                warnings.warn(robust_fallback_reason, RuntimeWarning)
                spline_est['se_betas'] = np.sqrt(np.maximum(np.diag(cov_beta), 0.0))
            else:
                cov_beta = var_robust
                spline_est['cov_beta'] = var_robust
                spline_est['se_betas'] = se_robust
                spline_est['dfs'] = df_robust
                inference_se_method = "CR2-CRVE"
        except Exception as e:
            robust_fallback_reason = f"CR2 computation raised: {e}"
            warnings.warn(robust_fallback_reason, RuntimeWarning)
            spline_est['se_betas'] = np.sqrt(np.maximum(np.diag(cov_beta), 0.0))

    # -----------------------------------------------------------------
    # 5c. Residuals for diagnostics
    # -----------------------------------------------------------------
    X_stacked = np.vstack(X_all)
    y_stacked = np.concatenate(y_all)
    fitted_vals = X_stacked @ betas
    spline_est['fitted'] = fitted_vals
    spline_est['resid'] = y_stacked - fitted_vals

    # -----------------------------------------------------------------
    # 6. Omnibus Test (AHZ / Hotelling-Zhang via the CR2 helper)
    #
    # Tests H0: betas[1:] == 0 (i.e. "the moderator has any non-intercept
    # effect"). NOTE the same semantic caveat as before: this is *not* a
    # curvature-only test; a purely linear effect will reject it. To test
    # non-linearity specifically, compare the spline against the linear
    # model via an ML-refit likelihood-ratio test (REML log-likelihoods
    # are not comparable across different fixed-effect designs).
    #
    # Inference path priority:
    #   1. AHZ via htz_fn(C)  — Tipton & Pustejovsky 2015 Eq. 10 + 14,
    #      matches clubSandwich's Wald_test(test="HTZ") to within ~1%
    #      on df_denom (well inside the method's own approximation error).
    #   2. F-approx with heuristic min(dfs) denom df — legacy path, used
    #      only when CR2 robust inference was unavailable (naive GLS) or
    #      the helper itself raised.
    #   3. Large-sample chi-square — last-resort fallback.
    # -----------------------------------------------------------------
    C_omni = np.eye(p_params)[1:]            # (p-1) x p, selects betas[1:]
    q_omni = C_omni.shape[0]

    omnibus_F        = None
    omnibus_T2       = None
    omnibus_eta      = None
    omnibus_df_num   = q_omni
    omnibus_df_denom = None
    p_val            = None
    omnibus_p_method = None

    if htz_fn is not None:
        try:
            ah = htz_fn(C_omni, test='AHZ')
            omnibus_F        = float(ah['F'])
            omnibus_T2       = float(ah['T2'])
            omnibus_eta      = float(ah['eta'])
            omnibus_df_denom = float(ah['df_denom'])
            p_val            = float(ah['p_val'])
            omnibus_p_method = "AHZ (Tipton & Pustejovsky 2015, Eq. 10 + 14)"
        except Exception as e:
            warnings.warn(
                f"AHZ omnibus via htz_fn raised: {e}. Falling back to "
                f"heuristic F-approximation.", RuntimeWarning)
            htz_fn_used = False
        else:
            htz_fn_used = True
    else:
        htz_fn_used = False

    if not htz_fn_used:
        # Legacy fallback: T^2 with min(dfs) heuristic OR large-sample chi^2.
        b_test   = betas[1:]
        cov_test = cov_beta[1:, 1:]
        try:
            T2 = float(b_test.T @ np.linalg.solve(cov_test, b_test))
            omnibus_T2 = T2

            if 'dfs' in spline_est and len(spline_est['dfs']) > 1:
                from scipy.stats import f as _f_dist
                df_den = float(np.min(spline_est['dfs'][1:]))
                F_stat = T2 / q_omni
                p_val  = float(_f_dist.sf(F_stat, q_omni, df_den))
                omnibus_F        = F_stat
                omnibus_df_denom = df_den
                omnibus_p_method = (
                    "F-approx (heuristic Satterthwaite-type denom df) "
                    "[fallback: AHZ helper unavailable]")
            else:
                p_val = float(chi2.sf(T2, q_omni))
                omnibus_F        = T2 / q_omni
                omnibus_df_denom = float(max(1.0, M_studies - p_params))
                omnibus_p_method = (
                    "Large-sample chi-square "
                    "[fallback: AHZ helper unavailable, no robust dfs]")
        except np.linalg.LinAlgError as e:
            return None, (f"Omnibus test failed — cov_beta[1:,1:] is "
                          f"singular: {e}")
    # -----------------------------------------------------------------
    # 7. Package metadata (model_type reflects BOTH the variance-estimation
    #    path and the inference path actually taken)
    # -----------------------------------------------------------------
    se_label = "CR2 Robust" if inference_se_method == "CR2-CRVE" else "Naive GLS"
    model_type = f"3-Level Spline ({variance_estimation}, {se_label})"

    spline_est.update({
        'mod_mean': mod_mean,
        'mod_std': mod_std,
        'formula': formula,
        'reg_df': df_sorted,
        'moderator_col': moderator_col,
        'omnibus_chi2':     omnibus_T2,
        'omnibus_df':       omnibus_df_num,
        'omnibus_p':        p_val,
        'omnibus_F': omnibus_F,
        'omnibus_T2': omnibus_T2,
        'omnibus_df_num': omnibus_df_num,
        'omnibus_df_denom': omnibus_df_denom,
        'omnibus_eta': omnibus_eta,
        'df_spline': df_spline,
        'basis_cond': basis_cond,
        'model_type': model_type,
        # --- variance-component provenance ---
        'tau_sq': tau2_used,
        'sigma_sq': sigma2_used,
        'tau_sq_plugin': tau2_plugin,
        'sigma_sq_plugin': sigma2_plugin,
        'variance_estimation': variance_estimation,
        'var_estimation_reason': var_estimation_reason,
        'log_lik_reml_at_plugin': (ll_start if lin_est is not None else None),
        # --- inference provenance ---
        'inference_se_method': inference_se_method,
        'robust_fallback_reason': robust_fallback_reason,
        'omnibus_p_method': omnibus_p_method,
        'omnibus_tests': 'any-association (includes linear term); NOT curvature-only',
        'vcv_fallback_studies': vcv_fallback_studies,
        'n_dropped_missing': n_dropped,
        'n_obs_used': N_total,
        'n_clusters_used': M_studies,
    })

    return spline_est, None

# --- 6. EXPORT EXCEL ---

import xlsxwriter

# --- 1. HELPER: EXCLUSION LOG ---
def _clean_html_tags(html_text):
    """Plain text from generated HTML for the 'Report Text' sheet: block-level
    tags become line breaks, other tags are removed, entities decoded."""
    import re as _re
    import html as _html
    t = _re.sub(r'(?i)<br\s*/?>|</p>|</h[1-6]>|</li>|</tr>|</div>', '\n', str(html_text))
    t = _re.sub(r'<[^>]+>', '', t)
    t = _html.unescape(t)
    t = _re.sub(r'[ \t\r\f\v]+', ' ', t)
    t = _re.sub(r'\n\s*\n+', '\n\n', t)
    return t.strip()


def _get_exclusion_log():
    if 'ANALYSIS_CONFIG' in globals() and 'removed_records' in ANALYSIS_CONFIG:
        return ANALYSIS_CONFIG['removed_records']
    return pd.DataFrame([{'ID': 'All Data Kept', 'Reason': 'No records were removed.'}])

# --- 2. HELPER: EXCEL FORMATTER ---
def _apply_excel_formatting(writer, sheet_name, df):
    workbook = writer.book
    worksheet = writer.sheets[sheet_name]
    header_fmt = workbook.add_format({'bold': True, 'valign': 'top', 'fg_color': '#1F4E78', 'font_color': '#FFFFFF', 'border': 1})
    for col_num, value in enumerate(df.columns.values):
        worksheet.write(0, col_num, value, header_fmt)
    for i, col in enumerate(df.columns):
        col_len = len(str(col))
        # Drop NAs first so we don't format empty cells as the string "nan"
        sample_vals = df[col].head(20).dropna()
        max_val_len = sample_vals.astype(str).map(len).max() if not sample_vals.empty else 0
        worksheet.set_column(i, i, min(max(col_len, max_val_len) + 2, 50))

# --- 3. HELPER: PROTOCOL SHEET FORMATTER ---
def _apply_protocol_sheet_formatting(writer, sheet_name, df):
    """
    Custom formatting for the Protocol & Settings sheet.
    - Bold headers
    - Bold Category column
    - Auto-adjust column widths for readability
    """
    workbook = writer.book
    worksheet = writer.sheets[sheet_name]

    # Define formats
    header_fmt = workbook.add_format({
        'bold': True,
        'valign': 'top',
        'fg_color': '#1F4E78',
        'font_color': '#FFFFFF',
        'border': 1
    })

    category_fmt = workbook.add_format({
        'bold': True,
        'valign': 'top'
    })

    # Apply header formatting
    for col_num, value in enumerate(df.columns.values):
        worksheet.write(0, col_num, value, header_fmt)

    # Apply bold formatting to Category column (column 0)
    for row_num in range(1, len(df) + 1):
        cell_value = df.iloc[row_num - 1, 0]  # Category column
        worksheet.write(row_num, 0, cell_value, category_fmt)

    # Auto-adjust column widths
    for i, col in enumerate(df.columns):
        col_len = len(str(col))
        # Drop NAs first to handle completely empty columns safely
        sample_vals = df[col].head(50).dropna()
        max_val_len = sample_vals.astype(str).map(len).max() if not sample_vals.empty else 0
        # Make columns wider for better readability
        col_width = min(max(col_len, max_val_len) + 3, 60)
        worksheet.set_column(i, i, col_width)


# --- 4. HELPER: SMART METADATA COLLECTOR ---
def _get_protocol_metadata(report_type):
    """
    Captures comprehensive protocol settings and configuration.
    Returns a DataFrame with columns: [Category, Parameter, Value]
    """
    meta = []

    # System Information
    meta.append({'Category': 'System', 'Parameter': 'Timestamp', 'Value': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")})

    if 'ANALYSIS_CONFIG' not in globals():
        return pd.DataFrame(meta)

    ac = ANALYSIS_CONFIG

    try:
        # === ANALYSIS SETTINGS ===
        global_settings = ac.get('global_settings', {})

        # Confidence Level (convert alpha to percentage)
        alpha = global_settings.get('alpha', 0.05)
        confidence_pct = (1 - alpha) * 100
        meta.append({'Category': 'Settings', 'Parameter': 'Confidence Level', 'Value': f"{confidence_pct:.0f}%"})

        # Inference Distribution
        dist_type = global_settings.get('dist_type', 'norm')
        dist_label = 't-distribution' if dist_type == 't' else 'Normal distribution'
        meta.append({'Category': 'Settings', 'Parameter': 'Inference Distribution', 'Value': dist_label})

        # Model Selection
        overall_results = ac.get('overall_results', {})
        model_choice = overall_results.get('model_choice', 'Unknown')
        meta.append({'Category': 'Settings', 'Parameter': 'Model', 'Value': model_choice})

        # Heterogeneity Estimator (tau method)
        tau_method = overall_results.get('tau_method', 'REML')
        meta.append({'Category': 'Settings', 'Parameter': 'Heterogeneity Estimator', 'Value': tau_method})

        # Knapp-Hartung Adjustment
        kh_info = overall_results.get('knapp_hartung', {})
        use_kh = kh_info.get('used', False)
        kh_label = 'Yes' if use_kh else 'No'
        meta.append({'Category': 'Settings', 'Parameter': 'Knapp-Hartung Adjustment', 'Value': kh_label})

        # === DATA PROVENANCE ===

        # Input Mode
        data_type = ac.get('data_type', 'Unknown')
        input_mode = 'Pre-Calculated' if data_type == 'pre_calculated' else 'Raw Data'
        meta.append({'Category': 'Data', 'Parameter': 'Input Mode', 'Value': input_mode})

        # Effect Size Information
        es_config = ac.get('es_config', {})
        effect_label = es_config.get('effect_label', 'Unknown')
        effect_label_short = es_config.get('effect_label_short', 'Unknown')
        meta.append({'Category': 'Data', 'Parameter': 'Effect Size Type', 'Value': f"{effect_label} ({effect_label_short})"})

        # Effect Size Column Names
        effect_col = ac.get('effect_col', es_config.get('effect_col', 'Unknown'))
        var_col = ac.get('var_col', es_config.get('var_col', 'Unknown'))
        meta.append({'Category': 'Data', 'Parameter': 'Effect Size Column', 'Value': effect_col})
        meta.append({'Category': 'Data', 'Parameter': 'Variance Column', 'Value': var_col})

        # Sample Size Information
        analysis_data = ac.get('analysis_data')
        if analysis_data is not None:
            k_total = len(analysis_data)
            meta.append({'Category': 'Data', 'Parameter': 'Total Observations (k)', 'Value': k_total})

            # Count unique studies
            if 'id' in analysis_data.columns:
                n_studies = analysis_data['id'].nunique()
                meta.append({'Category': 'Data', 'Parameter': 'Unique Studies', 'Value': n_studies})

        # === REPORT-SPECIFIC METADATA ===

        if report_type == 'cumulative' and 'cumulative_results' in ac:
            cum_df = ac['cumulative_results']
            if not cum_df.empty:
                meta.append({'Category': 'Cumulative', 'Parameter': 'Total Steps', 'Value': len(cum_df)})
                meta.append({'Category': 'Cumulative', 'Parameter': 'Year Range', 'Value': f"{int(cum_df['year'].min())} - {int(cum_df['year'].max())}"})
                meta.append({'Category': 'Cumulative', 'Parameter': 'Initial Effect (Start)', 'Value': f"{cum_df.iloc[0]['pooled_effect']:.4f}"})
                meta.append({'Category': 'Cumulative', 'Parameter': 'Final Effect (End)', 'Value': f"{cum_df.iloc[-1]['pooled_effect']:.4f}"})
                meta.append({'Category': 'Cumulative', 'Parameter': 'Method', 'Value': 'Iterative REML'})

        elif report_type == 'loo' and 'loo_3level_results' in ac:
            loo = ac['loo_3level_results']
            meta.append({'Category': 'Sensitivity', 'Parameter': 'Method', 'Value': 'Leave-One-Out (3-Level)'})
            meta.append({'Category': 'Sensitivity', 'Parameter': 'Original Effect', 'Value': f"{loo.get('original_effect', 0):.4f}"})
            meta.append({'Category': 'Sensitivity', 'Parameter': 'Significance Changers', 'Value': loo.get('n_sig_changers', 0)})

        elif report_type == 'subgroup' and 'subgroup_config' in ac:
            sub_config = ac['subgroup_config']
            analysis_type = sub_config.get('analysis_type', 'Unknown')
            moderator1 = sub_config.get('moderator1', 'Unknown')
            meta.append({'Category': 'Subgroup', 'Parameter': 'Analysis Type', 'Value': analysis_type})
            meta.append({'Category': 'Subgroup', 'Parameter': 'Moderator', 'Value': moderator1})
            if 'moderator2' in sub_config and sub_config['moderator2']:
                moderator2 = sub_config['moderator2']
                meta.append({'Category': 'Subgroup', 'Parameter': 'Second Moderator', 'Value': moderator2})

        elif report_type == 'regression' and 'meta_regression_RVE_results' in ac:
            reg = ac['meta_regression_RVE_results']
            moderator_col = reg.get('moderator_col_name', 'Unknown')
            meta.append({'Category': 'Regression', 'Parameter': 'Moderator Variable', 'Value': moderator_col})
            meta.append({'Category': 'Regression', 'Parameter': 'Model Type', 'Value': '3-Level REML'})

        elif report_type == 'spline' and 'spline_model_results' in ac:
            spline = ac['spline_model_results']
            moderator = spline.get('moderator_col', 'Unknown')
            n_knots = spline.get('n_knots', 'Unknown')
            meta.append({'Category': 'Spline', 'Parameter': 'Moderator Variable', 'Value': moderator})
            meta.append({'Category': 'Spline', 'Parameter': 'Number of Knots', 'Value': n_knots})

        elif report_type == 'publication_bias':
            meta.append({'Category': 'Publication Bias', 'Parameter': 'Tests Performed', 'Value': 'Egger Test, Trim & Fill'})

    except Exception as e:
        meta.append({'Category': 'Error', 'Parameter': 'Metadata Error', 'Value': str(e)})

    return pd.DataFrame(meta)

# --- 4. MAIN ORCHESTRATOR ---
def export_analysis_report(report_type='overall', filename_prefix="MetaAnalysis"):
    if 'ANALYSIS_CONFIG' not in globals(): return

    buffer = io.BytesIO()
    with pd.ExcelWriter(buffer, engine='xlsxwriter') as writer:
        try:
            text_key = 'latest_text'


            if report_type == 'overall':
                text_key = 'overall_text'
                res = ANALYSIS_CONFIG['overall_results']
                simple_res = {k:v for k,v in res.items() if not isinstance(v, dict)}
                pd.DataFrame([simple_res]).T.reset_index().to_excel(writer, sheet_name='Overall Results', index=False)

            elif report_type == 'subgroup':
                text_key = 'subgroup_text'
                if 'subgroup_results' in ANALYSIS_CONFIG:
                    sg_res = ANALYSIS_CONFIG['subgroup_results']
                    # Export main dataframe
                    sg_res['results_df'].to_excel(writer, sheet_name='Subgroup Results', index=False)

                    # Export heterogeneity stats
                    het_stats = {
                        'Q_Model (QM)': sg_res.get('QM'),
                        'df_QM': sg_res.get('df_QM'),
                        'p_value_QM': sg_res.get('p_value_QM'),
                        'Q_Residual (QE)': sg_res.get('Qe'),
                        'df_QE': sg_res.get('df_Qe'),
                        'R_squared (%)': sg_res.get('R_squared')
                    }
                    pd.DataFrame([het_stats]).to_excel(writer, sheet_name='Heterogeneity', index=False)


            elif report_type == 'regression':
                text_key = 'regression_text'
                if 'meta_regression_RVE_results' in ANALYSIS_CONFIG:
                    reg = ANALYSIS_CONFIG['meta_regression_RVE_results']
                    pd.DataFrame({'Term': ['Intercept', reg['moderator_col_name']], 'Beta': reg['betas'], 'SE': reg['std_errors_robust'], 'P-Value': [reg['p_intercept'], reg['p_slope']]}).to_excel(writer, sheet_name='Regression Results', index=False)

            elif report_type == 'regression2':
                text_key = 'regression_text'
                if 'meta_regression_RVE_results' in ANALYSIS_CONFIG:
                    reg = ANALYSIS_CONFIG['meta_regression_RVE_results']
                    # Prefer the full multi-moderator arrays when present;
                    # fall back to the legacy 2-row form for old result dicts.
                    if 'term_names' in reg and 'betas_all' in reg:
                        terms = list(reg['term_names'])
                        betas = list(np.asarray(reg['betas_all']).ravel())
                        ses = list(np.asarray(reg['std_errors_all']).ravel())
                        pvs = list(np.asarray(reg['p_values_all']).ravel())
                        cis = reg.get('cis_all', [(None, None)] * len(terms))
                        dfs = list(np.asarray(reg.get('df_all', [None] * len(terms))).ravel())
                        out = pd.DataFrame({
                            'Term': terms,
                            'Beta': betas,
                            'SE': ses,
                            'df': dfs,
                            'P-Value': pvs,
                            'CI_Lower': [c[0] for c in cis],
                            'CI_Upper': [c[1] for c in cis],
                        })
                    else:
                        out = pd.DataFrame({
                            'Term': ['Intercept', reg['moderator_col_name']],
                            'Beta': reg['betas'],
                            'SE': reg['std_errors_robust'],
                            'P-Value': [reg['p_intercept'], reg['p_slope']],
                        })
                    out.to_excel(writer, sheet_name='Regression Results', index=False)

                    # Optional omnibus sheet, when available.
                    om = reg.get('omnibus')
                    if om and om.get('overall'):
                        ov = om['overall']
                        pd.DataFrame([{
                            'Test': 'Omnibus (all moderators)',
                            'F': ov.get('F'), 'df_num': ov.get('df_num'),
                            'df_denom': ov.get('df_denom'), 'p_value': ov.get('p_val'),
                        }]).to_excel(writer, sheet_name='Omnibus Test', index=False)

            elif report_type == 'spline':
                text_key = 'spline_text'
                if 'spline_model_results' in ANALYSIS_CONFIG:
                    res = ANALYSIS_CONFIG['spline_model_results']
                    _rows = [{'Metric': 'Omnibus (any association) statistic', 'Value': res.get('omnibus_chi2')},
                             {'Metric': 'Omnibus P-Value', 'Value': res.get('omnibus_p')},
                             {'Metric': 'Variance estimation', 'Value': res.get('variance_estimation')}]
                    _mc = res.get('model_comparison') or {}
                    if _mc and not _mc.get('error'):
                        _rows += [{'Metric': 'Linear logLik (ML)', 'Value': _mc['linear']['loglik']},
                                  {'Metric': 'Spline logLik (ML)', 'Value': _mc['spline']['loglik']},
                                  {'Metric': 'Linear AIC (ML)', 'Value': _mc['linear']['aic']},
                                  {'Metric': 'Spline AIC (ML)', 'Value': _mc['spline']['aic']},
                                  {'Metric': 'Delta AIC (linear - spline)', 'Value': _mc['delta_aic']},
                                  {'Metric': 'Non-linearity LRT', 'Value': _mc.get('lrt')},
                                  {'Metric': 'Non-linearity LRT df', 'Value': _mc.get('df')},
                                  {'Metric': 'Non-linearity p-value', 'Value': _mc.get('p_value')}]
                    pd.DataFrame(_rows).to_excel(writer, sheet_name='Spline Results', index=False)

            elif report_type == 'publication_bias':
                text_key = 'bias_text'
                if 'funnel_results' in ANALYSIS_CONFIG:
                    egg = ANALYSIS_CONFIG['funnel_results']
                    pd.DataFrame([{'Test': 'Egger (slope on SE)', 'Slope': egg.get('beta_slope'), 'SE': egg.get('se'),
                                   'P-Value': egg['p_value'], 'Intercept (SE = 0)': egg['intercept']}]).to_excel(writer, sheet_name='Eggers Test', index=False)
                if 'trimfill_results' in ANALYSIS_CONFIG:
                    tf = ANALYSIS_CONFIG['trimfill_results']
                    pd.DataFrame([{'Missing': tf['k0'], 'Original': tf['pooled_original'], 'Adjusted': tf['pooled_filled']}]).to_excel(writer, sheet_name='Trim Fill', index=False)

            # === CUMULATIVE RESULTS ===
            elif report_type == 'cumulative':
                text_key = 'cumulative_text'
                if 'cumulative_results' in ANALYSIS_CONFIG:
                    # 1. Full Step-by-Step Table
                    ANALYSIS_CONFIG['cumulative_results'].to_excel(writer, sheet_name='Cumulative Results', index=False)

                    # 2. Summary Table (Start vs End)
                    df = ANALYSIS_CONFIG['cumulative_results']
                    if not df.empty:
                        summary = df.iloc[[0, -1]].copy()
                        summary.insert(0, 'Stage', ['Start', 'End'])
                        summary.to_excel(writer, sheet_name='Cumulative Summary', index=False)

            # === LOO RESULTS ===
            elif report_type == 'loo':
                text_key = 'loo_text'
                if 'loo_3level_results' in ANALYSIS_CONFIG:
                    # 1. Full Sensitivity Table
                    res_df = ANALYSIS_CONFIG['loo_3level_results']['results_df']
                    res_df.to_excel(writer, sheet_name='Sensitivity Results', index=False)

                    # 2. Influential Studies (if any)
                    influencers = res_df[res_df['changes_sig'] == True]
                    if not influencers.empty:
                        influencers.to_excel(writer, sheet_name='Influential Studies', index=False)
                    else:
                        pd.DataFrame([{'Result': 'Robust', 'Note': 'No single study changed significance.'}]).to_excel(writer, sheet_name='Influential Studies', index=False)

            # --- COMMON SHEETS ---

            if 'analysis_data' in ANALYSIS_CONFIG:
                ANALYSIS_CONFIG['analysis_data'].to_excel(writer, sheet_name='Processed Data', index=False)
                _apply_excel_formatting(writer, 'Processed Data', ANALYSIS_CONFIG['analysis_data'])

            # --- VCV MATRICES ---
            if 'vcv_matrices' in ANALYSIS_CONFIG and ANALYSIS_CONFIG['vcv_matrices']:
                vcv_rows = []
                for study_id, matrix in ANALYSIS_CONFIG['vcv_matrices'].items():
                    mat = np.atleast_2d(matrix)
                    for i, row_vals in enumerate(mat):
                        row_data = {'Study_ID': study_id, 'Effect_Index': i + 1}
                        for j, val in enumerate(row_vals):
                            row_data[f'Cov_with_Effect_{j + 1}'] = val
                        vcv_rows.append(row_data)

                if vcv_rows:
                    df_vcv = pd.DataFrame(vcv_rows)
                    df_vcv.to_excel(writer, sheet_name='VCV Matrices', index=False)
                    _apply_excel_formatting(writer, 'VCV Matrices', df_vcv)

            df_excl = _get_exclusion_log()
            df_excl.to_excel(writer, sheet_name='Data Exclusions', index=False)
            _apply_excel_formatting(writer, 'Data Exclusions', df_excl)

            # --- DATA USED BY THIS ANALYSIS (rows used / dropped and why) ---
            _acc_src = {'overall': 'overall_results', 'subgroup': 'subgroup_results',
                        'regression': 'meta_regression_RVE_results',
                        'regression2': 'meta_regression_RVE_results',
                        'spline': 'spline_model_results'}.get(report_type)
            _acc = (ANALYSIS_CONFIG.get(_acc_src) or {}).get('data_accounting') if _acc_src else None
            if _acc:
                _rows = [{'Item': 'Effect sizes used', 'Count': _acc['n_used_effects'],
                          'Of': _acc['n_input_effects'], 'Studies lost entirely': '', 'Rows': ''},
                         {'Item': 'Studies used', 'Count': _acc['n_used_clusters'],
                          'Of': _acc['n_input_clusters'], 'Studies lost entirely': '', 'Rows': ''}]
                for _d in _acc.get('dropped', []):
                    _rows.append({'Item': f"Dropped: {_d['reason']}", 'Count': _d['n_effects'], 'Of': '',
                                  'Studies lost entirely': ', '.join(_d.get('clusters_lost', [])),
                                  'Rows': ', '.join(_d.get('row_index', []))})
                if _acc.get('v_downgraded_clusters'):
                    _rows.append({'Item': 'Studies fitted with a diagonal V (VCV could not be aligned)',
                                  'Count': len(_acc['v_downgraded_clusters']), 'Of': '',
                                  'Studies lost entirely': ', '.join(_acc['v_downgraded_clusters']), 'Rows': ''})
                if _acc.get('common_row_set'):
                    _rows.append({'Item': _acc['common_row_set'], 'Count': '', 'Of': '',
                                  'Studies lost entirely': '', 'Rows': ''})
                df_used = pd.DataFrame(_rows)
                df_used.to_excel(writer, sheet_name='Data Used', index=False)
                _apply_excel_formatting(writer, 'Data Used', df_used)

            # --- GENERATED TEXT ---
            final_text = ANALYSIS_CONFIG.get(text_key, "")
            if final_text:
                clean_text = _clean_html_tags(final_text)
                df_text = pd.DataFrame({'Generated Interpretation': [clean_text]})
                df_text.to_excel(writer, sheet_name='Report Text', index=False)
                writer.sheets['Report Text'].set_column(0, 0, 100)
                writer.sheets['Report Text'].set_row(1, 300, writer.book.add_format({'text_wrap': True, 'valign': 'top'}))

            # --- PROTOCOL & SETTINGS (LAST SHEET) ---
            # This comprehensive sheet captures all configuration settings used for this analysis
            df_proto = _get_protocol_metadata(report_type)
            df_proto.to_excel(writer, sheet_name='Protocol & Settings', index=False)
            _apply_protocol_sheet_formatting(writer, 'Protocol & Settings', df_proto)

        except Exception as e:
            pd.DataFrame([{'Error': str(e)}]).to_excel(writer, sheet_name='Error_Log')

    buffer.seek(0)
    b64 = base64.b64encode(buffer.read()).decode()
    filename = f"{filename_prefix}_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
    payload = f"var link = document.createElement('a'); link.href = 'data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}'; link.download = '{filename}'; document.body.appendChild(link); link.click(); document.body.removeChild(link);"
    display(Javascript(payload))


# --- 7. PRESET FOR PLOTS ---

PRESETS = {
    'Custom': {},

    # 1 Column: 8.5 cm -> 3.35 inches
    'Cell Press 1-Col (85mm)': {
        'width': 3.35, 'height': 3.35,
        'title': 9, 'label': 8, 'tick': 7  # Fonts must be small to fit
    },
    # 1.5 Column: 11.4 cm -> 4.49 inches
    'Cell Press 1.5-Col (114mm)': {
        'width': 4.49, 'height': 4.00,
        'title': 10, 'label': 9, 'tick': 8
    },
    # Full Width: 17.4 cm -> 6.85 inches
    'Cell Press Full (174mm)': {
        'width': 6.85, 'height': 5.50,
        'title': 11, 'label': 10, 'tick': 9
    },

    # 1 Column: 13.4 cm -> 5.28 inches
    'STAR Protocols 1-Col (134mm)': {
        'width': 5.28, 'height': 5.00,
        'title': 11, 'label': 10, 'tick': 9
    },
    # Full Width: 17.2 cm -> 6.77 inches
    'STAR Protocols Full (172mm)': {
        'width': 6.77, 'height': 6.00,
        'title': 12, 'label': 11, 'tick': 10
    },

    # 1 Column: 5.5 cm -> 2.17 inches (Very narrow!)
    'Cell 3-Col Layout (Narrow/55mm)': {
        'width': 2.17, 'height': 2.50,
        'title': 8, 'label': 7, 'tick': 6
    },

    # --- General / Thesis ---
    'Thesis (A4 Portrait)':  {'width': 6.30, 'height': 8.00, 'title': 12, 'label': 11, 'tick': 10},
    'Presentation (16:9)':   {'width': 13.3, 'height': 7.50, 'title': 18, 'label': 14, 'tick': 12}
}

import datetime as _dt_mod
import html as _html_mod
import json as _json_mod
import re as _re_mod



def is_text_dtype(x) -> bool:
    """True for text columns: object dtype (pandas 2) or the string dtype (pandas 3)."""
    dt = getattr(x, 'dtype', x)
    return pd.api.types.is_object_dtype(dt) or pd.api.types.is_string_dtype(dt)



def preset_dropdown(targets: dict, description: str = '📏 Preset:'):
    """Journal size preset for one plot cell.

    `targets` maps 'width', 'height', 'title', 'label', 'tick' to that cell's
    widgets (any subset). Choosing a preset sets those widgets, widening a
    slider's range when the preset value lies outside it.
    """
    dd = widgets.Dropdown(options=list(PRESETS.keys()), value='Custom', description=description,
                          style={'description_width': '130px'}, layout=widgets.Layout(width='450px'))

    def _apply(change):
        values = PRESETS.get(change['new'], {})
        for key, w in targets.items():
            if key not in values or w is None:
                continue
            v = values[key]
            if hasattr(w, 'min') and v < w.min:
                w.min = v
            if hasattr(w, 'max') and v > w.max:
                w.max = v
            w.value = v

    dd.observe(_apply, names='value')
    return dd


# =============================================================================
# PLOT HELPERS & PLOT RECIPES
# A plot recipe is one JSON file per figure: the numbers the figure draws
# (already computed, no refit), the value of every setting, and a note of
# the analysis it came from. Loading it in the same plot cell redraws the
# figure exactly, without re-running the analysis.
# =============================================================================

PLOT_RECIPE_VERSION = 1
COMETA_PLOT_PANELS = {}   # plot_type -> {'save': fn, 'load': fn}; used by the recipe bar


def ci_pct_label(alpha=None) -> str:
    """'95%' for alpha 0.05; follows the CI level chosen in Step 7."""
    a = _alpha() if alpha is None else float(alpha)
    pct = 100.0 * (1.0 - a)
    return f"{pct:.0f}%" if abs(pct - round(pct)) < 1e-9 else f"{pct:g}%"


def es_null_value() -> float:
    """Value of 'no effect' on the analysis scale (0 for every effect size CoMeta computes)."""
    try:
        return float((ANALYSIS_CONFIG.get('es_config') or {}).get('null_value', 0) or 0)
    except Exception:
        return 0.0


def cometa_timestamp() -> str:
    return _dt_mod.datetime.now().strftime('%Y%m%d_%H%M%S')


def _safe_filename(text: str) -> str:
    text = _re_mod.sub(r'[^\w\-.]+', '_', str(text).strip()).strip('_')
    return text or 'plot'


def cometa_save_figure(fig, prefix, pdf=True, png=True, svg=False, dpi=300,
                       transparent=False, timestamp=True, stamp=None):
    """Save a matplotlib figure as PDF/PNG/SVG in the working directory
    (the Colab Files panel). Returns the list of file names."""
    base = _safe_filename(prefix)
    if timestamp:
        base = f"{base}_{stamp or cometa_timestamp()}"
    saved = []
    for ext, on in (('pdf', pdf), ('png', png), ('svg', svg)):
        if not on:
            continue
        kw = {'bbox_inches': 'tight', 'transparent': bool(transparent)}
        if ext == 'png':
            kw['dpi'] = int(dpi)
        fig.savefig(f"{base}.{ext}", **kw)
        saved.append(f"{base}.{ext}")
    return saved


def saved_files_html(paths) -> str:
    if not paths:
        return ""
    return ("<div style='margin-top:6px; font-size:12px; color:#155724;'>💾 Saved in the Files panel: "
            + ", ".join(f"<code>{_html_mod.escape(p)}</code>" for p in paths) + "</div>")


def caption_html(text: str, title: str = 'Figure caption') -> str:
    """Standard box for an auto-generated figure caption."""
    return (f"<div style='background:#f8f9fa; border-left:4px solid #2E86AB; padding:12px; "
            f"font-family:Georgia, serif; font-size:13px; line-height:1.5; max-width:900px;'>"
            f"<b>{_html_mod.escape(title)}.</b> {text}</div>")


def show_figure(fig):
    """Display a figure inside the current output area and release it."""
    display(fig)
    plt.close(fig)


# --- Settings <-> widgets ---

def widget_values(controls: dict) -> dict:
    """Current value of every widget in `controls` (name -> widget), JSON-ready."""
    out = {}
    for name, w in controls.items():
        v = w.value
        out[name] = list(v) if isinstance(v, tuple) else v
    return make_json_safe(out)


def apply_widget_values(controls: dict, values: dict) -> list:
    """Set widgets from a settings dict. Unknown names are ignored, values a
    widget does not accept are skipped. Returns the skipped names."""
    skipped = []
    for name, v in (values or {}).items():
        w = controls.get(name)
        if w is None:
            continue
        try:
            if isinstance(w, widgets.SelectMultiple):
                v = tuple(v)
            if hasattr(w, 'options'):
                allowed = [o[1] if isinstance(o, tuple) else o for o in w.options]
                if isinstance(v, tuple):
                    if not all(x in allowed for x in v):
                        skipped.append(name)
                        continue
                elif v not in allowed:
                    skipped.append(name)
                    continue
            if hasattr(w, 'min') and v is not None and isinstance(v, (int, float)) and v < w.min:
                w.min = v
            if hasattr(w, 'max') and v is not None and isinstance(v, (int, float)) and v > w.max:
                w.max = v
            w.value = v
        except Exception:
            skipped.append(name)
    return skipped


# --- Data snapshot encoding (DataFrames and arrays survive the JSON round trip) ---

def recipe_encode(obj):
    if isinstance(obj, pd.DataFrame):
        return {'__dataframe__': True, 'columns': [str(c) for c in obj.columns],
                'dtypes': {str(c): str(t) for c, t in obj.dtypes.items()},
                'data': {str(c): recipe_encode(obj[c].tolist()) for c in obj.columns}}
    if isinstance(obj, pd.Series):
        return recipe_encode(obj.tolist())
    if isinstance(obj, np.ndarray):
        return recipe_encode(obj.tolist())
    if isinstance(obj, dict):
        return {str(k): recipe_encode(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [recipe_encode(v) for v in obj]
    if isinstance(obj, (np.floating, float)):
        return float(obj)
    if obj is pd.NaT:
        return None
    return make_json_safe(obj)


def recipe_decode(obj):
    if isinstance(obj, dict):
        if obj.get('__dataframe__'):
            df = pd.DataFrame({c: obj['data'][c] for c in obj['columns']}, columns=obj['columns'])
            for c, t in (obj.get('dtypes') or {}).items():
                if c in df.columns and t in ('object', 'str', 'string', 'category', 'bool'):
                    df[c] = df[c].astype(object if t in ('str', 'string', 'category') else t)
            return df
        return {k: recipe_decode(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [recipe_decode(v) for v in obj]
    return obj


def build_plot_recipe(plot_type: str, data: dict, settings: dict, source: dict) -> dict:
    return {
        'cometa_plot_recipe': PLOT_RECIPE_VERSION,
        'plot_type': plot_type,
        'saved_at': _dt_mod.datetime.now().isoformat(timespec='seconds'),
        'source': make_json_safe(source or {}),
        'settings': make_json_safe(settings or {}),
        'data': recipe_encode(data),
    }


def parse_plot_recipe(content, plot_type: str) -> dict:
    """Read an uploaded recipe (bytes or str); check it belongs to this plot."""
    if isinstance(content, (bytes, bytearray, memoryview)):
        content = bytes(content).decode('utf-8')
    rec = _json_mod.loads(content)
    if not isinstance(rec, dict) or 'cometa_plot_recipe' not in rec:
        raise ValueError("This is not a CoMeta plot recipe file. (Session files are loaded in Step 2, "
                         "tab Restore Session.)")
    if rec.get('plot_type') != plot_type:
        raise ValueError(f"This recipe is for a {rec.get('plot_type')!r} plot; this cell draws "
                         f"{plot_type!r} plots. Load it in the matching plot cell.")
    if int(rec['cometa_plot_recipe']) > PLOT_RECIPE_VERSION:
        raise ValueError("This recipe was saved by a newer version of CoMeta.")
    rec['data'] = recipe_decode(rec.get('data'))
    return rec


def source_html(source: dict, from_recipe: str = None) -> str:
    """One line saying which analysis a plot shows."""
    if not source:
        return ("<div style='padding:8px; background:#fff3cd; border-left:4px solid #ffc107; font-size:13px;'>"
                "No results in this session yet. Run the analysis step first, or load a plot recipe below.</div>")
    parts = [f"<b>{_html_mod.escape(str(source.get('analysis', '')))}</b>"]
    for key, lab in (('effect_label', 'effect size'), ('model', 'model'), ('ci_level', 'CI'),
                     ('analysis_time', 'run')):
        if source.get(key):
            parts.append(f"{lab}: {_html_mod.escape(str(source[key]))}")
    origin = (f"📂 From recipe file <code>{_html_mod.escape(from_recipe)}</code>" if from_recipe
              else "📊 From the results in this session")
    return (f"<div style='padding:8px; background:#e8f4f8; border-left:4px solid #2E86AB; font-size:13px;'>"
            f"{origin}: " + " · ".join(parts) + "</div>")


def results_token(key: str):
    """Identity of a results object, to notice when an analysis was re-run."""
    obj = (globals().get('ANALYSIS_CONFIG') or {}).get(key)
    return None if obj is None else id(obj)


def stale_warning_html(step_label: str) -> str:
    return ("<div style='padding:8px; background:#fff3cd; border-left:4px solid #ffc107; font-size:13px; "
            f"margin-bottom:6px;'>⚠️ {_html_mod.escape(step_label)} was run again after this cell. This plot still "
            "shows the earlier results; re-run this cell to plot the new ones.</div>")


def plot_recipe_bar(plot_type: str, get_recipe, on_load, filename_prefix=lambda: 'plot'):
    """Save / load buttons for plot recipes.

    get_recipe() -> recipe dict (build_plot_recipe) or None if nothing to save;
    on_load(recipe, style_only) applies a parsed recipe and redraws.
    """
    save_btn = widgets.Button(description='💾 Save Plot Recipe', button_style='info',
                              layout=widgets.Layout(width='200px'),
                              tooltip='One file with this figure\'s data and settings')
    upload = widgets.FileUpload(accept='.json', multiple=False, description='📂 Load Plot Recipe',
                                layout=widgets.Layout(width='200px'))
    style_only = widgets.Checkbox(value=False, indent=False,
                                  description='Apply style only (keep the current data)')
    msg = widgets.Output()

    def _save(_b=None):
        with msg:
            clear_output()
            rec = get_recipe()
            if rec is None:
                display(HTML("<div style='color:#721c24;'>❌ Nothing to save yet: generate the plot first "
                             "or load a recipe.</div>"))
                return None
            payload = _json_mod.dumps(rec, indent=1, allow_nan=True)
            fname = f"{_safe_filename(filename_prefix())}_recipe_{cometa_timestamp()}.json"
            with open(fname, 'w') as f:
                f.write(payload)
            b64 = base64.b64encode(payload.encode()).decode()
            display(Javascript(f"""
                var link = document.createElement('a');
                link.href = 'data:application/json;base64,{b64}';
                link.download = '{fname}';
                document.body.appendChild(link); link.click(); document.body.removeChild(link);
            """))
            display(HTML(f"<div style='color:#155724;'>✅ Recipe saved: <code>{_html_mod.escape(fname)}</code> "
                         f"({len(payload) / 1024:.0f} KB). Load it here later to redraw this figure.</div>"))
            return fname

    def _load(content, name='recipe', style=None):
        with msg:
            clear_output()
            try:
                rec = parse_plot_recipe(content, plot_type)
            except Exception as e:
                display(HTML(f"<div style='color:#721c24;'>❌ {_html_mod.escape(str(e))}</div>"))
                return False
            skipped = on_load(rec, style_only.value if style is None else style, name)
            note = (f" Settings not applied (not available here): {', '.join(skipped)}." if skipped else "")
            display(HTML(f"<div style='color:#155724;'>✅ Loaded <code>{_html_mod.escape(name)}</code> "
                         f"(saved {_html_mod.escape(str(rec.get('saved_at', '')))}).{_html_mod.escape(note)}</div>"))
            return True

    def _on_upload(change):
        files = change['new']
        if not files:
            return
        info = files[0] if isinstance(files, (list, tuple)) else list(files.values())[0]
        content = info['content'] if isinstance(info, dict) else info
        name = (info.get('name') or info.get('metadata', {}).get('name', 'recipe')) if isinstance(info, dict) else 'recipe'
        if hasattr(content, 'tobytes'):
            content = content.tobytes()
        _load(content, name)
        try:
            upload.value.clear() if isinstance(upload.value, dict) else setattr(upload, 'value', ())
        except Exception:
            pass

    save_btn.on_click(_save)
    upload.observe(_on_upload, names='value')
    COMETA_PLOT_PANELS[plot_type] = {'save': _save, 'load': _load}
    return widgets.VBox([
        widgets.HTML("<b>Plot recipe</b> <span style='color:#666; font-size:12px;'>(one file per figure: "
                     "its data and settings, to redraw it later without re-running the analysis)</span>"),
        widgets.HBox([save_btn, upload, style_only]), msg])


# =============================================================================
# REPRODUCIBILITY INFRASTRUCTURE
# =============================================================================

def _safe_set_widget(widget, value, observer_fn=None, observer_name='value'):
    """
    Set a widget's value without triggering its observer.

    Args:
        widget: ipywidgets widget instance
        value: Value to set
        observer_fn: Observer function to temporarily detach (optional)
        observer_name: Name of the trait being observed (default 'value')
    """
    if value is None:
        return
    try:
        if observer_fn is not None:
            widget.unobserve(observer_fn, names=observer_name)
        widget.value = value
        if observer_fn is not None:
            widget.observe(observer_fn, names=observer_name)
    except Exception:
        pass


def make_json_safe(obj):
    """Recursively convert an object to JSON-serializable types."""
    import json as _json
    import datetime as _dt
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, _dt.datetime):
        return obj.isoformat()
    if isinstance(obj, pd.DataFrame):
        return None
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(item) for item in obj]
    if isinstance(obj, set):
        return sorted([str(x) for x in obj])
    try:
        _json.dumps(obj)
        return obj
    except (TypeError, ValueError):
        return str(obj)


def export_config_for_reproducibility(analysis_config):
    """
    Create a JSON-serializable dictionary from ANALYSIS_CONFIG.
    Prioritizes human-readable settings at the top and embeds clean_dataframe
    as CSV string under 'embedded_data' at the very bottom.
    """
    import json as _json
    import datetime as _dt

    RESULT_KEYS = {
        'analysis_data', 'removed_records', 'vcv_matrices',
        'overall_results', 'three_level_results', 'subgroup_results',
        'meta_regression_RVE_results', 'spline_model_results',
        'funnel_results', 'trimfill_results', 'pet_peese_results',
        'cumulative_results', 'loo_results', 'loo_3level_results',
        'overall_text', 'subgroup_text', 'regression_text',
        'bias_text', 'cumulative_text', 'cumulative_metadata',
        'meaningful_results', 'meaningful_text',
    }

    clean = {}

    # 1. Metadata (Top of JSON file)
    clean['_reproducibility'] = {
        'is_reproducing': True,
        'exported_at': _dt.datetime.now().isoformat(),
        'co_met_version': '8.0',
        'strategy': 'sequential_auto_populate',
    }

    SKIP_KEYS = RESULT_KEYS | {'clean_dataframe'}

    # 2. Extract primary user settings to the top for easy reading
    gs = analysis_config.get('global_settings', {})
    clean['global_settings'] = make_json_safe(gs)

    if 'subgroup_config' in analysis_config:
         clean['subgroup_config'] = make_json_safe(analysis_config['subgroup_config'])

    # 3. Copy remaining INPUT keys
    for key, value in analysis_config.items():
        if key in SKIP_KEYS or key in clean:
            continue
        safe_value = make_json_safe(value)
        if safe_value is not None:
            clean[key] = safe_value

    # 4. Embed clean_dataframe as CSV at the VERY END
    df = analysis_config.get('clean_dataframe')
    if df is not None and isinstance(df, pd.DataFrame):
        # column types, so text columns (e.g. IDs like "007") stay text on reload
        clean['embedded_dtypes'] = {str(c): str(t) for c, t in df.dtypes.items()}
        csv_buffer = io.StringIO()
        df.to_csv(csv_buffer, index=False)
        clean['embedded_data'] = csv_buffer.getvalue()

    return clean


def load_reproducibility_config(json_path_or_bytes):
    """
    Load a previously exported analysis_settings.json.
    Hydrates embedded_data back into a DataFrame.
    """
    import json as _json

    if isinstance(json_path_or_bytes, bytes):
        config = _json.loads(json_path_or_bytes.decode('utf-8'))
    elif isinstance(json_path_or_bytes, str):
        with open(json_path_or_bytes, 'r') as f:
            config = _json.load(f)
    else:
        raise TypeError(f"Expected str or bytes, got {type(json_path_or_bytes)}")

    if 'embedded_data' in config:
        csv_string = config.pop('embedded_data')
        dtypes = config.pop('embedded_dtypes', None) or {}
        text_cols = {c: str for c, t in dtypes.items()
                     if t in ('object', 'str', 'string') or t.startswith('string')}
        df = pd.read_csv(io.StringIO(csv_string), dtype=text_cols or None)
        config['clean_dataframe'] = df
        globals()['raw_data_standardized'] = df.copy()
        globals()['DATA_TYPE_SELECTED'] = config.get('data_type', 'raw')

    config['is_reproducing'] = True
    return config

# --- 8. DISPLAY MASTER DASHBOARD ---

clear_output(wait=True) # Clear the loading screen


display(HTML("""
<style>
  /* Subtle Comet Animation */
  @keyframes comet-sweep {
    0% { transform: translate(-100px, -50px) rotate(35deg); opacity: 0; }
    5% { opacity: 1; }
    15% { transform: translate(900px, 500px) rotate(35deg); opacity: 0; }
    100% { opacity: 0; }
  }
</style>

<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; max-width: 900px; border: 1px solid #e2e8f0; border-radius: 12px; overflow: hidden; box-shadow: 0 10px 25px -5px rgba(0,0,0,0.05), 0 8px 10px -6px rgba(0,0,0,0.01); margin: 20px auto; background-color: #ffffff;">

  <!-- Header with Comet Effect -->
  <div style="position: relative; background: linear-gradient(135deg, #0f172a 0%, #1e3a8a 60%, #3b82f6 100%); padding: 32px 36px 26px 36px; overflow: hidden;">
    <!-- The Comet Tail -->
    <div style="position: absolute; top: 0; left: 0; width: 200px; height: 2px; background: linear-gradient(90deg, transparent, rgba(255,255,255,0.9), transparent); box-shadow: 0 0 15px 2px rgba(255,255,255,0.7); animation: comet-sweep 8s infinite cubic-bezier(0.25, 1, 0.5, 1);"></div>

    <!-- Explicitly set color: #ffffff to prevent Colab theme overrides -->
    <h1 style="position: relative; z-index: 1; margin: 0 0 6px 0; font-size: 28px; font-weight: 800; letter-spacing: 0.5px; display: flex; align-items: center; color: #ffffff;">
      <!-- Abstract Comet Icon -->
    <svg width="26" height="26" viewBox="0 0 26 26" fill="none" xmlns="http://www.w3.org/2000/svg" style="margin-right: 12px;">
  <defs>
    <radialGradient id="ng" cx="50%" cy="50%" r="50%">
      <stop offset="0%" stop-color="#ffffff"/>
      <stop offset="35%" stop-color="#ffffff"/>
      <stop offset="65%" stop-color="#60a5fa"/>
      <stop offset="100%" stop-color="#3b82f6" stop-opacity="0"/>
    </radialGradient>
    <linearGradient id="fan" x1="20" y1="6" x2="2" y2="22" gradientUnits="userSpaceOnUse">
      <stop offset="0%" stop-color="#bfdbfe" stop-opacity="0.95"/>
      <stop offset="100%" stop-color="#93c5fd" stop-opacity="0"/>
    </linearGradient>
    <linearGradient id="fan2" x1="20" y1="6" x2="2" y2="22" gradientUnits="userSpaceOnUse">
      <stop offset="0%" stop-color="#ffffff" stop-opacity="0.7"/>
      <stop offset="100%" stop-color="#ffffff" stop-opacity="0"/>
    </linearGradient>
  </defs>
  <path d="M19 7 Q14 10 10 14 Q7 17 3 22 Q6 19 9 16 Q13 12 17 8 Z" fill="url(#fan)"/>
  <path d="M19 7 Q15 11 12 14 Q10 16 7 20 Q9 17 11 15 Q15 11 18 8 Z" fill="url(#fan2)"/>
  <path d="M18 8 Q14 12 11 15 L9 18 Q11 15 13 13 Q16 10 18 8 Z" fill="white" fill-opacity="0.25"/>
  <path d="M17.5 8.5 L8 19 L9 18 L18 8.5 Z" fill="white" fill-opacity="0.5"/>
  <circle cx="20" cy="6" r="4.5" fill="#3b82f6" opacity="0.3"/>
  <circle cx="20" cy="6" r="3.2" fill="#60a5fa" opacity="0.5"/>
  <circle cx="20" cy="6" r="2.2" fill="url(#ng)"/>
  <circle cx="20" cy="6" r="1.1" fill="white"/>
</svg>
      CoMeta
      <span style="font-size: 12px; background: rgba(255,255,255,0.15); backdrop-filter: blur(4px); padding: 4px 10px; border-radius: 20px; margin-left: 14px; font-weight: 600; letter-spacing: 0.5px; border: 1px solid rgba(255,255,255,0.2); color: #ffffff;">v1.0</span>
    </h1>
    <p style="position: relative; z-index: 1; margin: 0; font-size: 15px; font-weight: 400; color: #bfdbfe; letter-spacing: 0.3px;">From data to report a no-code platform for rigorous three-level meta-analysis in ecology
</p>
  </div>

  <!-- Description Band -->
  <div style="background: #f8fafc; padding: 26px 36px; border-bottom: 1px solid #e2e8f0;">
    <p style="margin: 0 0 18px 0; font-size: 14.5px; color: #334155; line-height: 1.7;">
      <strong>CoMeta</strong> is an interactive, end-to-end meta-analytic pipeline designed to address the statistical complexities of ecological data. Its core contribution is the rigorous handling of <em>non-independent effect sizes</em>: it reconstructs exact Variance-Covariance matrices (Gleser &amp; Olkin, 2009) and partitions total heterogeneity into explicit between-study (&#964;&#178;) and within-study (&#963;&#178;) components via three-level mixed-effects models.
    </p>

    <div style="background: #fffbeb; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 0 6px 6px 0;">
      <p style="margin: 0; font-size: 13.5px; color: #92400e; line-height: 1.5;">
        <strong>Instruction:</strong> Execute cells sequentially. This notebook fully reproduces all analyses reported in the associated manuscript. Alternatively, provide your own dataset in Step 2 to apply the pipeline to an independent analysis.
      </p>
    </div>
  </div>

  <!-- Citation Row -->
  <div style="padding: 18px 36px; border-bottom: 1px solid #e2e8f0; background: #ffffff; display: flex; flex-direction: column; gap: 4px;">
    <div style="font-size: 11px; text-transform: uppercase; font-weight: 700; letter-spacing: 1px; color: #64748b;">Citation Reference</div>
    <div style="font-size: 14px; color: #0f172a; font-weight: 600;">Journal, Year, Vol(Issue)</div>
    <div style="font-size: 13px; color: #3b82f6; font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace;">DOI: 10.1111/xxxx.xxxxx</div>
  </div>

  <!-- Status Bar -->
  <div style="padding: 16px 36px; background: #f8fafc; display: flex; align-items: center; justify-content: space-between; flex-wrap: wrap; gap: 12px; border-top: 1px solid #f1f5f9;">
    <div style="display: flex; align-items: center; gap: 16px;">
      <span style="display: flex; align-items: center; gap: 8px; font-size: 13.5px; color: #166534; font-weight: 600;">
        <span style="width: 10px; height: 10px; background: #22c55e; border-radius: 50%; display: inline-block; box-shadow: 0 0 0 3px #dcfce7; flex-shrink: 0;"></span>
        Environment Verified
      </span>
      <span style="font-size: 13.5px; color: #64748b; border-left: 2px solid #e2e8f0; padding-left: 16px;">Core engine initialised</span>
    </div>
  </div>

</div>
"""))

# --- Package check (from check_and_lockdown at the top of this cell) ---
if COMETA_ENV_CHECK['missing'] or COMETA_ENV_CHECK['different']:
    _rows = ''.join(f"<li><code>{p}</code>: installed {cur}, tested with {ver}</li>"
                    for p, cur, ver in COMETA_ENV_CHECK['different'])
    _rows += ''.join(f"<li><code>{p}</code>: was missing, installed {ver}</li>" if IS_COLAB
                     else f"<li><code>{p}</code>: missing (install {ver})</li>"
                     for p, ver in COMETA_ENV_CHECK['missing'])
    display(HTML(
        "<div style='max-width:900px; margin:0 auto 16px auto; padding:10px 14px; background:#fff8e1; "
        "border-left:4px solid #ffc107; border-radius:4px; font-family:sans-serif; font-size:13px; color:#5d4600;'>"
        "<b>Package versions differ from the tested set.</b> CoMeta continues with the installed versions; "
        "they are recorded in the session and validation exports. Do not restart the runtime to change them "
        "unless you re-run Cells 1 and 1b afterwards.<ul style='margin:6px 0 0 0;'>" + _rows + "</ul></div>"))

# Later cells check this flag and stop with a clear message when Cell 1 has
# not run in the current session (e.g. after a runtime restart).
COMETA_CELL1_DONE = True


In [ ]:
#@title ⚙️ 1b. Unified Model Engine (FitSpec · fit_mv · compare_fits)
# Requires Cell 1 in this session.
if not globals().get('COMETA_CELL1_DONE'):
    raise RuntimeError("Cell 1 (Environment Setup) has not run in this session. Run Cell 1, then this cell. "
                       "This happens after a runtime restart, e.g. when Colab asks to restart after installing packages.")

# =============================================================================
# Purpose: ONE place that decides which rows, which sampling V and which
#          random structure a model is fitted on, so that every comparison
#          statistic (AIC, LRT, pseudo-R²) is computed between models that
#          really are nested.
#
#   FitSpec        - frozen description of the data a model sees: the exact
#                    rows (original index labels), cluster blocks, the
#                    sampling-VCV blocks sliced by ROW IDENTITY, the design
#                    matrix, and a row-accounting record (used / dropped and
#                    why).
#   build_fit_spec - builds a FitSpec from a DataFrame + ANALYSIS_CONFIG.
#   fit_mv         - REML/ML fit of the multilevel model
#                        y = X b + u_cluster + e_effect + sampling error,
#                        Var = V + tau2 * J (within cluster) + sigma2 * I
#                    where either variance component can be FIXED (e.g. at 0)
#                    on the same rows, V and X. Equivalent to
#                    metafor::rma.mv(yi, V, mods = X, random = ~ 1 | id/es_id,
#                                    sigma2 = c(tau2, sigma2)).
#   compare_fits   - AIC / LRT between two fits. Refuses (ComparisonError)
#                    when the fits use different rows or different V, and for
#                    different fixed-effect designs either refits under ML or
#                    refuses. Boundary-corrected (chi-bar-squared) p-values.
#
#   meaningful_proportions - share of true effects beyond thresholds of
#                    importance (section 9; Step 9b), parametric and calibrated,
#                    with cluster-bootstrap BCa CIs (Validation 25).
#   fit_blocks     - the entry point every analysis cell uses (section 8):
#                    overall model, subgroups, meta-regression, splines,
#                    Egger / trim-and-fill, leave-one-out, cumulative and the
#                    two-level REML/ML tau² estimators all fit through
#                    fit_mv, so there is one likelihood and one optimiser to
#                    validate (Validation 15 checks fit_mv against rma.mv).
#
# Naming: tau2  = between-cluster variance  (metafor sigma2[1], ~ 1 | id)
#         sigma2 = within-cluster variance  (metafor sigma2[2], ~ 1 | id/es)
# =============================================================================

import hashlib
import warnings
from dataclasses import dataclass, field, replace
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.stats import chi2

COMETA_ENGINE_VERSION = "1.0"

_LOG_2PI = float(np.log(2.0 * np.pi))


class ComparisonError(ValueError):
    """Raised when two fits cannot be validly compared (different rows,
    different sampling V, different estimation method, ...)."""


# =============================================================================
# 1. SAMPLING-VCV SLICING BY ROW IDENTITY
# =============================================================================

def vcv_row_positions(df_subset: pd.DataFrame,
                      reference_df: Optional[pd.DataFrame] = None,
                      cluster_col: str = 'id') -> Optional[np.ndarray]:
    """Within-cluster position of each row of `df_subset` in the dataset the
    VCV matrices were built from (Step 6 builds each cluster's matrix with
    df.groupby('id'), i.e. in the cluster's original row order).

    Uses a '_vcv_row_pos' column when present; otherwise maps the subset's
    index labels onto `reference_df` (default: ANALYSIS_CONFIG['analysis_data']).
    Returns None when the identity cannot be established (index labels not
    found in the reference); callers then fall back to the shape-only path,
    which is guarded by the diagonal check in slice_vcv_block().
    """
    if '_vcv_row_pos' in df_subset.columns:
        return df_subset['_vcv_row_pos'].to_numpy(dtype=int)

    if reference_df is None:
        cfg = globals().get('ANALYSIS_CONFIG', {}) or {}
        reference_df = cfg.get('analysis_data')
    if reference_df is None or cluster_col not in reference_df.columns:
        return None
    if not reference_df.index.is_unique:
        return None

    pos = reference_df.groupby(cluster_col, sort=False).cumcount()
    if not df_subset.index.isin(pos.index).all():
        return None
    # The labels must denote the SAME rows: after a reset_index the labels
    # 0..n-1 also exist in the reference but point at other rows. Require the
    # cluster id to agree on every row; otherwise identity is unknown.
    if cluster_col in df_subset.columns:
        ref_ids = reference_df.loc[df_subset.index, cluster_col].astype(str).to_numpy()
        if not np.array_equal(ref_ids, df_subset[cluster_col].astype(str).to_numpy()):
            return None
    return pos.loc[df_subset.index].to_numpy(dtype=int)


def attach_vcv_row_pos(df: pd.DataFrame,
                       reference_df: Optional[pd.DataFrame] = None,
                       cluster_col: str = 'id') -> pd.DataFrame:
    """Return `df` with a '_vcv_row_pos' column (each row's within-cluster
    position in the data the VCV matrices were built from), so the identity
    survives later sort_values / reset_index. Unchanged when it is already
    present or cannot be established."""
    if '_vcv_row_pos' in df.columns:
        return df
    pos = vcv_row_positions(df, reference_df, cluster_col)
    if pos is None:
        return df
    out = df.copy()
    out['_vcv_row_pos'] = pos
    return out


def cluster_vcv(vcv_dict: Optional[Dict[Any, np.ndarray]], cluster_id: Any,
                group: pd.DataFrame, var_col: str) -> Tuple[np.ndarray, str]:
    """Sampling-VCV block for one cluster's rows (`group`), sliced by row
    identity when the group carries '_vcv_row_pos' (see attach_vcv_row_pos).
    Returns (V, source) as slice_vcv_block does."""
    v = group[var_col].to_numpy(dtype=np.float64)
    pos = group['_vcv_row_pos'].to_numpy() if '_vcv_row_pos' in group.columns else None
    return slice_vcv_block(vcv_dict, cluster_id, v, pos)


def slice_vcv_block(vcv_dict: Optional[Dict[Any, np.ndarray]],
                    cluster_id: Any,
                    variances: np.ndarray,
                    row_positions: Optional[np.ndarray] = None
                    ) -> Tuple[np.ndarray, str]:
    """Sampling-VCV block for one cluster, aligned to the rows actually used.

        V_sub = cached[np.ix_(row_positions, row_positions)]

    Keeps the shared-control covariances when only SOME of a cluster's rows
    are dropped, and never applies the cached matrix to the wrong rows: the
    slice is accepted only if its diagonal equals `variances` (the cached
    diagonal IS the sampling-variance column).

    Returns (V, source) with source one of
        'full'                       cached matrix (or slice of it) used
        'diagonal (no cached VCV)'   nothing cached for this cluster
        'diagonal (VCV mismatch)'    cached matrix could not be aligned
    """
    variances = np.asarray(variances, dtype=np.float64)
    k = variances.size
    diag = np.diag(variances)

    cached = None
    if vcv_dict:
        for key in (str(cluster_id), cluster_id):
            try:
                if key in vcv_dict:
                    cached = np.asarray(vcv_dict[key], dtype=np.float64)
                    break
            except TypeError:  # unhashable id
                continue
    if cached is None:
        return diag, 'diagonal (no cached VCV)'

    candidate = None
    if cached.ndim == 2 and cached.shape[0] == cached.shape[1]:
        if row_positions is not None:
            pos = np.asarray(row_positions, dtype=int)
            if (pos.size == k and k > 0 and pos.min() >= 0
                    and pos.max() < cached.shape[0]
                    and np.unique(pos).size == k):
                candidate = cached[np.ix_(pos, pos)]
        elif cached.shape[0] == k:
            candidate = cached

    if candidate is not None and np.allclose(
            np.diag(candidate), variances, rtol=1e-8, atol=1e-12):
        return candidate, 'full'
    return diag, 'diagonal (VCV mismatch)'


# =============================================================================
# 2. ROW ACCOUNTING
# =============================================================================

@dataclass
class RowAccounting:
    """Which rows a model used, which it dropped, and why."""
    n_input_effects: int
    n_input_clusters: int
    n_used_effects: int
    n_used_clusters: int
    dropped: List[Dict[str, Any]] = field(default_factory=list)
    # each entry: {'reason': str, 'n_effects': int, 'row_index': list,
    #              'clusters_lost': list}
    v_downgraded_clusters: List[Any] = field(default_factory=list)

    @property
    def n_dropped_effects(self) -> int:
        return self.n_input_effects - self.n_used_effects

    @property
    def n_dropped_clusters(self) -> int:
        return self.n_input_clusters - self.n_used_clusters

    def summary_text(self) -> str:
        s = (f"Used {self.n_used_effects} of {self.n_input_effects} effect sizes "
             f"in {self.n_used_clusters} of {self.n_input_clusters} clusters")
        if self.dropped:
            parts = [f"{d['n_effects']} – {d['reason']}" for d in self.dropped]
            s += "; dropped: " + "; ".join(parts)
        if self.v_downgraded_clusters:
            s += (f"; WARNING: {len(self.v_downgraded_clusters)} cluster(s) "
                  f"used a diagonal V because the cached VCV could not be aligned")
        return s + "."

    def to_frame(self) -> pd.DataFrame:
        rows = [{'Reason': d['reason'], 'Effects dropped': d['n_effects'],
                 'Clusters lost entirely': len(d['clusters_lost'])}
                for d in self.dropped]
        rows.append({'Reason': 'USED IN MODEL', 'Effects dropped': self.n_used_effects,
                     'Clusters lost entirely': self.n_used_clusters})
        return pd.DataFrame(rows)

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> 'RowAccounting':
        return cls(n_input_effects=d['n_input_effects'], n_input_clusters=d['n_input_clusters'],
                   n_used_effects=d['n_used_effects'], n_used_clusters=d['n_used_clusters'],
                   dropped=[{'reason': x['reason'], 'n_effects': x['n_effects'],
                             'row_index': list(x.get('row_index', [])),
                             'clusters_lost': list(x.get('clusters_lost', []))}
                            for x in d.get('dropped', [])],
                   v_downgraded_clusters=list(d.get('v_downgraded_clusters', [])))

    def to_dict(self) -> Dict[str, Any]:
        """JSON-safe record (row labels as strings) for storage and export."""
        return {
            'n_input_effects': int(self.n_input_effects),
            'n_input_clusters': int(self.n_input_clusters),
            'n_used_effects': int(self.n_used_effects),
            'n_used_clusters': int(self.n_used_clusters),
            'dropped': [{'reason': d['reason'], 'n_effects': int(d['n_effects']),
                         'clusters_lost': [str(c) for c in d['clusters_lost']],
                         'row_index': [str(r) for r in d['row_index']]}
                        for d in self.dropped],
            'v_downgraded_clusters': [str(c) for c in self.v_downgraded_clusters],
            'common_row_set': common_row_set_note(),
            'summary': self.summary_text(),
        }

    def to_html(self) -> str:
        colour = '#856404' if (self.dropped or self.v_downgraded_clusters) else '#155724'
        bg = '#fff3cd' if (self.dropped or self.v_downgraded_clusters) else '#d4edda'
        items = ''.join(
            f"<li>{d['n_effects']} effect size(s): {d['reason']}"
            + (f" ({len(d['clusters_lost'])} cluster(s) lost entirely)"
               if d['clusters_lost'] else '') + "</li>"
            for d in self.dropped)
        if self.v_downgraded_clusters:
            items += (f"<li><b>{len(self.v_downgraded_clusters)} cluster(s) fitted "
                      f"with a diagonal V</b> (cached shared-control VCV could not be "
                      f"aligned to the rows used).</li>")
        note = common_row_set_note()
        if note:
            items += f"<li>{note}</li>"
        body = f"<ul style='margin:4px 0 0 0;'>{items}</ul>" if items else ''
        return (f"<div style='padding:8px 12px; background:{bg}; color:{colour}; "
                f"border-radius:4px; font-size:0.9em; margin:6px 0;'><b>Data used:</b> "
                f"{self.n_used_effects}/{self.n_input_effects} effect sizes, "
                f"{self.n_used_clusters}/{self.n_input_clusters} studies.{body}</div>")


def common_row_set_note() -> str:
    """Describe the global complete-case restriction (Step 3), if active."""
    cfg = globals().get('ANALYSIS_CONFIG', {}) or {}
    cols = cfg.get('complete_case_moderators') or []
    if not cols:
        return ''
    log = cfg.get('complete_case_log') or {}
    n = log.get('n_dropped')
    extra = f"; {n} record(s) removed there" if n is not None else ''
    return ("Common row set: all analyses restricted (Step 3) to complete cases on "
            + ", ".join(f"'{c}'" for c in cols) + extra + ".")


def _missing_value_mask(col: pd.Series) -> pd.Series:
    return col.isna() | col.astype(str).str.strip().isin(['', 'nan', 'NaN', 'None', '<NA>'])


def select_rows(df: pd.DataFrame,
                effect_col: str,
                var_col: str,
                cluster_col: str = 'id',
                complete_case_cols: Optional[Sequence[str]] = None,
                extra: Optional[Sequence[Tuple[pd.Series, str]]] = None
                ) -> Tuple[pd.DataFrame, RowAccounting]:
    """Apply the standard row exclusions in a fixed order and record every
    dropped row with its reason (the first failing reason wins):

        missing cluster id -> missing/non-finite effect -> missing/non-positive
        variance -> missing value in each complete-case column (in order) ->
        each (mask, reason) in `extra` (masks aligned to df.index).

    Returns (rows kept, in their original order, with original index labels;
    RowAccounting). Raises ValueError when no rows are left.
    """
    for c in (effect_col, var_col, cluster_col):
        if c not in df.columns:
            raise ValueError(f"select_rows: column '{c}' not found.")
    n_input = len(df)
    clusters_in = set(df[cluster_col].dropna().unique())
    keep = pd.Series(True, index=df.index)
    dropped: List[Dict[str, Any]] = []

    def _drop(mask: pd.Series, reason: str):
        mask = pd.Series(np.asarray(mask, dtype=bool), index=df.index) & keep
        if mask.any():
            dropped.append({'reason': reason, 'n_effects': int(mask.sum()),
                            'row_index': list(df.index[mask]), 'clusters_lost': []})
            keep[mask] = False

    yv = pd.to_numeric(df[effect_col], errors='coerce')
    vv = pd.to_numeric(df[var_col], errors='coerce')
    _drop(df[cluster_col].isna(), f"missing cluster id ('{cluster_col}')")
    _drop(~np.isfinite(yv), f"missing/non-finite effect size ('{effect_col}')")
    _drop(~np.isfinite(vv) | (vv <= 0), f"missing/non-positive variance ('{var_col}')")
    for c in (complete_case_cols or []):
        if c not in df.columns:
            raise ValueError(f"select_rows: complete-case column '{c}' not found.")
        _drop(_missing_value_mask(df[c]), f"missing value in '{c}'")
    for mask, reason in (extra or []):
        _drop(mask, reason)

    used = df.loc[keep]
    if len(used) == 0:
        raise ValueError("select_rows: no rows left after exclusions ("
                         + "; ".join(f"{d['n_effects']} {d['reason']}" for d in dropped) + ").")
    clusters_used = set(used[cluster_col].unique())
    for d in dropped:
        lost = {df.at[i, cluster_col] for i in d['row_index']} - clusters_used
        d['clusters_lost'] = sorted(lost, key=str)
    acc = RowAccounting(n_input_effects=n_input, n_input_clusters=len(clusters_in),
                        n_used_effects=len(used), n_used_clusters=len(clusters_used),
                        dropped=dropped)
    return used, acc


def accounting_from_log(original: pd.DataFrame, final: pd.DataFrame,
                        drop_log: Sequence[Tuple[Sequence[Any], str]],
                        cluster_col: str = 'id') -> RowAccounting:
    """RowAccounting for a mode that drops rows in its own steps:
    `drop_log` = [(index labels dropped at that step, reason), ...]."""
    clusters_used = set(final[cluster_col].unique())
    dropped = []
    for labels, reason in drop_log:
        labels = list(labels)
        if not labels:
            continue
        lost = {original.at[i, cluster_col] for i in labels} - clusters_used
        dropped.append({'reason': reason, 'n_effects': len(labels), 'row_index': labels,
                        'clusters_lost': sorted(lost, key=str)})
    return RowAccounting(n_input_effects=len(original),
                         n_input_clusters=int(original[cluster_col].nunique()),
                         n_used_effects=len(final), n_used_clusters=len(clusters_used),
                         dropped=dropped)


# =============================================================================
# 3. FIT SPECIFICATION
# =============================================================================

def _hash_arrays(*arrays) -> str:
    h = hashlib.sha1()
    for a in arrays:
        a = np.ascontiguousarray(np.asarray(a, dtype=np.float64))
        h.update(str(a.shape).encode())
        h.update(np.round(a, 12).tobytes())
    return h.hexdigest()[:16]


@dataclass
class FitSpec:
    """Everything a model needs except its variance structure.

    Two fits are comparable iff their specs share `data_fingerprint`
    (same rows, same y, same V). Their fixed-effect designs may differ
    (`design_fingerprint`), which is only allowed under ML.
    """
    row_index: np.ndarray            # original index labels, in fit order
    cluster_ids: List[Any]           # one per block
    y_blocks: List[np.ndarray]
    V_blocks: List[np.ndarray]
    X_blocks: List[np.ndarray]
    term_names: List[str]
    v_source: Dict[Any, str]
    accounting: RowAccounting
    effect_col: str = 'yi'
    var_col: str = 'vi'
    cluster_col: str = 'id'
    label: str = ''

    # ---- sizes --------------------------------------------------------------
    @property
    def n_effects(self) -> int:
        return int(sum(len(y) for y in self.y_blocks))

    @property
    def n_clusters(self) -> int:
        return len(self.y_blocks)

    @property
    def n_coef(self) -> int:
        return int(self.X_blocks[0].shape[1]) if self.X_blocks else 0

    @property
    def y(self) -> np.ndarray:
        return np.concatenate(self.y_blocks)

    @property
    def X(self) -> np.ndarray:
        return np.vstack(self.X_blocks)

    @property
    def uses_full_vcv(self) -> bool:
        return any(not np.allclose(V, np.diag(np.diag(V))) for V in self.V_blocks)

    # ---- identity -----------------------------------------------------------
    @property
    def data_fingerprint(self) -> str:
        """Rows + y + V. Must match for ANY comparison statistic."""
        rows = pd.Index(self.row_index).astype(str)
        h = hashlib.sha1('|'.join(rows).encode()).hexdigest()[:16]
        return h + _hash_arrays(self.y, *self.V_blocks)

    @property
    def design_fingerprint(self) -> str:
        return _hash_arrays(self.X)

    # ---- derived specs ------------------------------------------------------
    def with_design(self, X_blocks: Sequence[np.ndarray],
                    term_names: Sequence[str], label: str = '') -> 'FitSpec':
        """Same rows, same y, same V, different fixed effects."""
        X_blocks = [np.asarray(X, dtype=np.float64) for X in X_blocks]
        if len(X_blocks) != self.n_clusters or any(
                X.shape[0] != len(y) for X, y in zip(X_blocks, self.y_blocks)):
            raise ValueError("with_design: X blocks are not row-aligned with this spec.")
        return replace(self, X_blocks=X_blocks, term_names=list(term_names),
                       label=label or self.label)

    def intercept_only(self) -> 'FitSpec':
        """The matching null model: same rows, y and V; intercept-only."""
        return self.with_design([np.ones((len(y), 1)) for y in self.y_blocks],
                                ['intrcpt'], label=(self.label + ' [null]').strip())

    def describe(self) -> str:
        v = ('full shared-control V' if self.uses_full_vcv else 'diagonal V')
        return (f"{self.n_effects} effects in {self.n_clusters} clusters, "
                f"{self.n_coef} fixed-effect coefficient(s), {v}")


def build_fit_spec(df: pd.DataFrame,
                   effect_col: Optional[str] = None,
                   var_col: Optional[str] = None,
                   cluster_col: str = 'id',
                   X: Optional[Union[pd.DataFrame, np.ndarray]] = None,
                   term_names: Optional[Sequence[str]] = None,
                   complete_case_cols: Optional[Sequence[str]] = None,
                   vcv_dict: Optional[Dict[Any, np.ndarray]] = None,
                   vcv_reference_df: Optional[pd.DataFrame] = None,
                   label: str = '') -> FitSpec:
    """Build a FitSpec from `df`, recording every dropped row and why.

    df                 Subset of ANALYSIS_CONFIG['analysis_data'] that KEEPS
                       its original index labels (do not reset_index), so
                       the sampling-VCV can be sliced by row identity.
    X                  Fixed-effect design aligned with df's rows (DataFrame
                       with the same index, or array with len(df) rows).
                       None -> intercept only.
    complete_case_cols Rows with a missing value in any of these columns are
                       dropped (reason recorded per column) even if X does not
                       use them -- this is how several models are restricted
                       to one common row set.
    """
    cfg = globals().get('ANALYSIS_CONFIG', {}) or {}
    effect_col = effect_col or cfg.get('effect_col', 'yi')
    var_col = var_col or cfg.get('var_col', 'vi')
    if vcv_dict is None:
        vcv_dict = cfg.get('vcv_matrices', {}) or {}

    for c in (effect_col, var_col, cluster_col):
        if c not in df.columns:
            raise ValueError(f"build_fit_spec: column '{c}' not found.")
    if not df.index.is_unique:
        raise ValueError("build_fit_spec: df index must be unique (row identity).")

    # ---- design matrix aligned with df -------------------------------------
    if X is None:
        X_df = pd.DataFrame({'intrcpt': np.ones(len(df))}, index=df.index)
    elif isinstance(X, pd.DataFrame):
        if not X.index.equals(df.index):
            X = X.reindex(df.index)
        X_df = X.astype(float)
    else:
        X_arr = np.asarray(X, dtype=np.float64)
        if X_arr.ndim == 1:
            X_arr = X_arr[:, None]
        if X_arr.shape[0] != len(df):
            raise ValueError("build_fit_spec: X must have one row per row of df.")
        names = list(term_names) if term_names else [f"x{j}" for j in range(X_arr.shape[1])]
        X_df = pd.DataFrame(X_arr, index=df.index, columns=names)
    if term_names is not None and len(term_names) == X_df.shape[1]:
        X_df.columns = list(term_names)

    # ---- row exclusions, each recorded with its reason ---------------------
    used, acc = select_rows(
        df, effect_col, var_col, cluster_col, complete_case_cols,
        extra=[(~np.isfinite(X_df.to_numpy()).all(axis=1),
                "missing/non-finite design-matrix value")])

    # ---- block structure: clusters sorted, original order within cluster --
    order = used.sort_values(cluster_col, kind='mergesort').index
    used = used.loc[order]
    positions = vcv_row_positions(used, vcv_reference_df, cluster_col)
    pos_s = pd.Series(positions, index=used.index) if positions is not None else None

    y_blocks, V_blocks, X_blocks, cids = [], [], [], []
    v_source: Dict[Any, str] = {}
    for cid, g in used.groupby(cluster_col, sort=False):
        v_i = g[var_col].to_numpy(dtype=np.float64)
        V_i, src = slice_vcv_block(
            vcv_dict, cid, v_i,
            pos_s.loc[g.index].to_numpy() if pos_s is not None else None)
        cids.append(cid)
        y_blocks.append(g[effect_col].to_numpy(dtype=np.float64))
        V_blocks.append(V_i)
        X_blocks.append(X_df.loc[g.index].to_numpy(dtype=np.float64))
        v_source[cid] = src

    acc.v_downgraded_clusters = [c for c, s in v_source.items() if s == 'diagonal (VCV mismatch)']
    if acc.v_downgraded_clusters:
        warnings.warn(acc.summary_text(), stacklevel=2)

    return FitSpec(row_index=np.asarray(used.index), cluster_ids=cids,
                   y_blocks=y_blocks, V_blocks=V_blocks, X_blocks=X_blocks,
                   term_names=list(X_df.columns), v_source=v_source,
                   accounting=acc, effect_col=effect_col, var_col=var_col,
                   cluster_col=cluster_col, label=label)


# =============================================================================
# 4. LIKELIHOOD
# =============================================================================

def _mv_loglik(spec: FitSpec, tau2: float, sigma2: float, method: str = 'REML',
               reml_full: bool = True, return_estimates: bool = False):
    """Log-likelihood of the multilevel model at (tau2, sigma2), beta profiled.

    Sigma_i = V_i + sigma2 * I + tau2 * J
    ML  : -1/2 [ N log 2pi + log|Sigma| + r'Sigma^-1 r ]
    REML: -1/2 [ (N-p) log 2pi + log|Sigma| + log|X'Sigma^-1 X| + r'Sigma^-1 r ]
          + 1/2 log|X'X|  (reml_full=True: metafor's default REMLf=TRUE; this
                           makes it the exact likelihood of the error contrasts)
    """
    p = spec.n_coef
    d = _diag_stats(spec)
    if d is not None:
        # Diagonal sampling V: Sigma_i = diag(v + sigma2) + tau2 * J, inverted
        # in closed form (Sherman-Morrison) for all clusters at once.
        a = d['v'] + sigma2
        if np.any(a <= 0):
            return (-np.inf, None) if return_estimates else -np.inf
        w = 1.0 / a
        s_i = np.add.reduceat(w, d['starts'])
        c_i = tau2 / (1.0 + tau2 * s_i)
        wX = d['X'] * w[:, None]
        u = np.add.reduceat(wX, d['starts'], axis=0)          # X_i' A^-1 1
        r = np.add.reduceat(w * d['y'], d['starts'])           # 1' A^-1 y_i
        XtWX = d['X'].T @ wX - (u.T * c_i) @ u
        XtWy = wX.T @ d['y'] - u.T @ (c_i * r)
        ytWy = float(np.sum(w * d['y'] ** 2) - np.sum(c_i * r ** 2))
        logdet = float(np.sum(np.log(a)) + np.sum(np.log1p(tau2 * s_i)))
        return _mv_loglik_finish(spec, XtWX, XtWy, ytWy, logdet, method, reml_full,
                                 return_estimates)

    # Full sampling V: clusters of the same size are processed together
    # (stacked Cholesky factorisations), the same algebra as one at a time.
    XtWX = np.zeros((p, p))
    XtWy = np.zeros(p)
    ytWy = 0.0
    logdet = 0.0
    for k, (Y, Xg, Vg) in _block_groups(spec).items():
        S = Vg + sigma2 * np.eye(k) + tau2 * np.ones((k, k))
        try:
            L = np.linalg.cholesky(S)
        except np.linalg.LinAlgError:
            return (-np.inf, None) if return_estimates else -np.inf
        logdet += 2.0 * float(np.sum(np.log(np.diagonal(L, axis1=1, axis2=2))))
        Ly = np.linalg.solve(L, Y[..., None])[..., 0]
        LX = np.linalg.solve(L, Xg)
        XtWX += np.einsum('gkp,gkq->pq', LX, LX)
        XtWy += np.einsum('gkp,gk->p', LX, Ly)
        ytWy += float(np.sum(Ly * Ly))
    return _mv_loglik_finish(spec, XtWX, XtWy, ytWy, logdet, method, reml_full,
                             return_estimates)


def _diag_stats(spec: FitSpec) -> Optional[Dict[str, np.ndarray]]:
    """Stacked y, X, sampling variances and cluster starts when every V block
    is diagonal (cached on the spec); None when any block has covariances."""
    cached = getattr(spec, '_diag_cache', 0)
    if cached != 0:
        return cached
    out = None
    if all(np.count_nonzero(V - np.diag(np.diag(V))) == 0 for V in spec.V_blocks):
        sizes = np.array([len(y) for y in spec.y_blocks])
        out = {'y': spec.y, 'X': spec.X,
               'v': np.concatenate([np.diag(V) for V in spec.V_blocks]),
               'starts': np.concatenate([[0], np.cumsum(sizes)[:-1]]).astype(np.intp)}
    spec._diag_cache = out
    return out


def _block_groups(spec: FitSpec) -> Dict[int, Tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """Clusters grouped by size k (cached on the spec): stacked y (g, k),
    X (g, k, p) and sampling V (g, k, k) for each k."""
    cached = getattr(spec, '_block_groups_cache', None)
    if cached is None:
        by_size: Dict[int, List[int]] = {}
        for i, y in enumerate(spec.y_blocks):
            by_size.setdefault(len(y), []).append(i)
        cached = {k: (np.stack([spec.y_blocks[i] for i in idx]),
                      np.stack([spec.X_blocks[i] for i in idx]),
                      np.stack([spec.V_blocks[i] for i in idx]))
                  for k, idx in sorted(by_size.items())}
        spec._block_groups_cache = cached
    return cached


def _xtx_logdet(spec: FitSpec) -> float:
    """log|X'X| of the spec's design (cached): the REMLf term of metafor."""
    cached = getattr(spec, '_xtx_logdet_cache', None)
    if cached is None:
        X = spec.X
        cached = float(np.linalg.slogdet(X.T @ X)[1])
        spec._xtx_logdet_cache = cached
    return cached


def _mv_loglik_finish(spec, XtWX, XtWy, ytWy, logdet, method, reml_full, return_estimates):
    """Profile out beta and assemble the ML / REML log-likelihood."""
    p = spec.n_coef
    sign, logdet_xwx = np.linalg.slogdet(XtWX)
    if sign <= 0:
        return (-np.inf, None) if return_estimates else -np.inf
    beta = np.linalg.solve(XtWX, XtWy)
    rss = float(ytWy - beta @ XtWy)
    N = spec.n_effects

    if method == 'ML':
        ll = -0.5 * (N * _LOG_2PI + logdet + rss)
    elif method == 'REML':
        ll = -0.5 * ((N - p) * _LOG_2PI + logdet + logdet_xwx + rss)
        if reml_full:
            ll += 0.5 * _xtx_logdet(spec)
    else:
        raise ValueError("method must be 'REML' or 'ML'")

    if not return_estimates:
        return float(ll)
    return float(ll), {'beta': beta, 'cov_beta': np.linalg.inv(XtWX), 'rss': rss}


# =============================================================================
# 5. FIT
# =============================================================================

@dataclass
class MVFit:
    spec: FitSpec
    method: str
    reml_full: bool
    tau2: float
    sigma2: float
    tau2_fixed: Optional[float]      # None -> estimated
    sigma2_fixed: Optional[float]
    betas: np.ndarray
    cov_beta: np.ndarray              # model-based (not robust)
    loglik: float
    converged: bool
    structure: str                    # human-readable label of what was fitted
    notes: List[str] = field(default_factory=list)
    v_fallback: bool = False          # True: fitted with the fallback (diagonal) V

    @property
    def se(self) -> np.ndarray:
        return np.sqrt(np.diag(self.cov_beta))

    @property
    def n_var_params(self) -> int:
        return int(self.tau2_fixed is None) + int(self.sigma2_fixed is None)

    @property
    def n_params(self) -> int:
        """Parameters counted for AIC: fixed effects + estimated variance
        components (metafor convention, also under REML)."""
        return self.spec.n_coef + self.n_var_params

    @property
    def deviance(self) -> float:
        return -2.0 * self.loglik

    @property
    def aic(self) -> float:
        return self.deviance + 2.0 * self.n_params

    @property
    def bic(self) -> float:
        n = self.spec.n_effects - (self.spec.n_coef if self.method == 'REML' else 0)
        return self.deviance + self.n_params * np.log(n)

    @property
    def aicc(self) -> float:
        n = self.spec.n_effects - (self.spec.n_coef if self.method == 'REML' else 0)
        k = self.n_params
        return self.aic + (2.0 * k * (k + 1) / (n - k - 1) if n - k - 1 > 0 else np.nan)

    @property
    def total_heterogeneity(self) -> float:
        return self.tau2 + self.sigma2

    def summary(self) -> Dict[str, Any]:
        return {
            'structure': self.structure, 'method': self.method,
            'tau2': self.tau2, 'sigma2': self.sigma2,
            'tau2_fixed': self.tau2_fixed, 'sigma2_fixed': self.sigma2_fixed,
            'betas': dict(zip(self.spec.term_names, map(float, self.betas))),
            'se': dict(zip(self.spec.term_names, map(float, self.se))),
            'loglik': self.loglik, 'aic': self.aic, 'bic': self.bic, 'aicc': self.aicc,
            'n_params': self.n_params, 'k_effects': self.spec.n_effects,
            'k_clusters': self.spec.n_clusters, 'converged': self.converged,
            'data': self.spec.accounting.summary_text(),
        }


def structure_label(tau2_fixed: Optional[float], sigma2_fixed: Optional[float]) -> str:
    """Name a model by what was actually fitted."""
    t_free, s_free = tau2_fixed is None, sigma2_fixed is None
    if t_free and s_free:
        return "Three-level (between- and within-cluster components)"
    if t_free and sigma2_fixed == 0:
        return "Between-cluster component only (within-cluster σ² fixed at 0)"
    if s_free and tau2_fixed == 0:
        return "Within-cluster component only (between-cluster τ² fixed at 0)"
    if tau2_fixed == 0 and sigma2_fixed == 0:
        return "Common-effect (no random effects)"
    return (f"Multilevel (τ² {'estimated' if t_free else f'fixed at {tau2_fixed:g}'}, "
            f"σ² {'estimated' if s_free else f'fixed at {sigma2_fixed:g}'})")


def fit_mv(spec: FitSpec,
           tau2: Optional[float] = None,
           sigma2: Optional[float] = None,
           method: str = 'REML',
           reml_full: bool = True,
           upper: Optional[float] = None,
           starts: Optional[Sequence[Tuple[float, float]]] = None) -> MVFit:
    """Fit the multilevel model on `spec`.

    tau2 / sigma2 : None -> estimated (bounded at 0); a number -> held fixed.
        fit_mv(spec)              ~ rma.mv(..., sigma2 = c(NA, NA))
        fit_mv(spec, sigma2=0)    ~ rma.mv(..., sigma2 = c(NA, 0))
        fit_mv(spec, tau2=0)      ~ rma.mv(..., sigma2 = c(0, NA))

    The optimum is searched over the interior (multi-start L-BFGS-B) AND
    over every boundary face (component = 0), so the returned log-likelihood
    is the true constrained maximum. This is what guarantees
    AIC(reduced) - AIC(full) >= -2 for nested variance structures.

    starts : interior starting points (tau2, sigma2) used instead of the
        default multi-start set, e.g. the estimates of a closely related fit
        (bootstrap refits). The boundary faces are always searched.
    """
    method = method.upper()
    if method not in ('REML', 'ML'):
        raise ValueError("method must be 'REML' or 'ML'")
    if spec.n_effects - spec.n_coef < 1:
        raise ValueError("fit_mv: not enough effect sizes for the number of coefficients.")
    if np.linalg.matrix_rank(spec.X) < spec.n_coef:
        raise ValueError("fit_mv: design matrix is rank-deficient on these rows.")

    t_free, s_free = tau2 is None, sigma2 is None
    if upper is None:
        y = spec.y
        mean_v = float(np.mean(np.concatenate([np.diag(V) for V in spec.V_blocks])))
        upper = max(10.0 * (float(np.var(y)) + mean_v), 1.0)

    def ll_at(t, s):
        return _mv_loglik(spec, t, s, method, reml_full)

    def full(params):
        it = iter(params)
        t = next(it) if t_free else float(tau2)
        s = next(it) if s_free else float(sigma2)
        return t, s

    notes: List[str] = []
    candidates: List[Tuple[float, Tuple[float, float]]] = []
    converged = True

    n_free = int(t_free) + int(s_free)
    if n_free == 0:
        candidates.append((ll_at(float(tau2), float(sigma2)), (float(tau2), float(sigma2))))
    elif n_free == 1:
        f = lambda x: -ll_at(*full([x]))
        r = minimize_scalar(f, bounds=(0.0, upper), method='bounded',
                            options={'xatol': 1e-10})
        candidates.append((-r.fun, full([r.x])))
        candidates.append((ll_at(*full([0.0])), full([0.0])))          # boundary
        converged = bool(r.success)
    else:
        f = lambda x: -ll_at(*full(x))
        if starts is None:
            starts = [(0.01, 0.01), (0.1, 0.1), (0.5, 0.5), (1.0, 0.1), (0.1, 1.0),
                      (upper / 20, upper / 20)]
        starts = [(min(max(a, 1e-8), upper), min(max(b, 1e-8), upper)) for a, b in starts]
        any_ok = False
        for st in starts:
            try:
                r = minimize(f, np.asarray(st), method='L-BFGS-B',
                             bounds=[(0.0, upper)] * 2,
                             options={'ftol': 1e-13, 'gtol': 1e-10, 'maxiter': 5000})
            except (ValueError, np.linalg.LinAlgError, FloatingPointError):
                continue
            if np.isfinite(r.fun):
                candidates.append((-r.fun, full(r.x)))
                any_ok = any_ok or bool(r.success)
        # boundary faces: profile each component with the other at 0
        for which in ('tau', 'sigma'):
            g = (lambda x: -ll_at(x, 0.0)) if which == 'tau' else (lambda x: -ll_at(0.0, x))
            r = minimize_scalar(g, bounds=(0.0, upper), method='bounded',
                                options={'xatol': 1e-10})
            pt = (r.x, 0.0) if which == 'tau' else (0.0, r.x)
            candidates.append((-r.fun, pt))
        candidates.append((ll_at(0.0, 0.0), (0.0, 0.0)))
        converged = any_ok

    candidates = [c for c in candidates if np.isfinite(c[0])]
    if not candidates:
        raise RuntimeError("fit_mv: likelihood could not be evaluated (Sigma not positive definite).")
    ll_best, (t_hat, s_hat) = max(candidates, key=lambda c: c[0])
    t_hat, s_hat = max(float(t_hat), 0.0), max(float(s_hat), 0.0)

    if (t_free and t_hat > 0.95 * upper) or (s_free and s_hat > 0.95 * upper):
        notes.append(f"variance estimate near search bound ({upper:.3g}); check effect-size scale")

    ll_final, est = _mv_loglik(spec, t_hat, s_hat, method, reml_full, return_estimates=True)
    return MVFit(spec=spec, method=method, reml_full=reml_full,
                 tau2=t_hat, sigma2=s_hat,
                 tau2_fixed=None if t_free else float(tau2),
                 sigma2_fixed=None if s_free else float(sigma2),
                 betas=est['beta'], cov_beta=est['cov_beta'], loglik=ll_final,
                 converged=converged,
                 structure=structure_label(None if t_free else float(tau2),
                                           None if s_free else float(sigma2)),
                 notes=notes)


def refit(fit: MVFit, method: Optional[str] = None, spec: Optional[FitSpec] = None) -> MVFit:
    """Refit the same variance structure under another method and/or spec."""
    return fit_mv(spec or fit.spec, tau2=fit.tau2_fixed, sigma2=fit.sigma2_fixed,
                  method=method or fit.method, reml_full=fit.reml_full)


# =============================================================================
# 6. COMPARISON
# =============================================================================

def chibar_pvalue(lrt: float, df_fixed: int, n_boundary: int) -> float:
    """p-value of an LRT where `n_boundary` variance components are tested at
    their boundary (0) and `df_fixed` other constraints are interior.

    Mixture sum_{j=0..b} C(b,j) / 2^b * chi2_{df_fixed + j}  (Self & Liang 1987;
    Stram & Lee 1994). Exact for b = 1 (the 50:50 chi2_0 : chi2_1 mixture);
    a standard conservative-ish approximation for b = 2.
    """
    lrt = max(float(lrt), 0.0)
    if n_boundary == 0:
        return float(chi2.sf(lrt, df_fixed)) if df_fixed > 0 else 1.0
    from math import comb
    p = 0.0
    for j in range(n_boundary + 1):
        w = comb(n_boundary, j) / 2.0 ** n_boundary
        d = df_fixed + j
        p += w * (float(chi2.sf(lrt, d)) if d > 0 else float(lrt <= 0))
    return float(min(max(p, 0.0), 1.0))


@dataclass
class FitComparison:
    full: MVFit
    reduced: MVFit
    method: str
    delta_aic: float              # AIC(reduced) - AIC(full); >0 favours full
    delta_bic: float
    lrt: Optional[float]
    df: Optional[int]
    n_boundary: Optional[int]
    p_value: Optional[float]
    nested: bool
    refitted_ml: bool
    warnings: List[str] = field(default_factory=list)

    def summary(self) -> Dict[str, Any]:
        return {'full': self.full.structure, 'reduced': self.reduced.structure,
                'method': self.method, 'AIC_full': self.full.aic,
                'AIC_reduced': self.reduced.aic, 'delta_AIC': self.delta_aic,
                'delta_BIC': self.delta_bic, 'LRT': self.lrt, 'df': self.df,
                'boundary_components': self.n_boundary, 'p_value': self.p_value,
                'nested': self.nested, 'refitted_ML': self.refitted_ml,
                'k_effects': self.full.spec.n_effects,
                'k_clusters': self.full.spec.n_clusters, 'warnings': self.warnings}


def _col_space_contains(X_big: np.ndarray, X_small: np.ndarray, tol: float = 1e-8) -> bool:
    r_big = np.linalg.matrix_rank(X_big, tol=tol)
    return np.linalg.matrix_rank(np.hstack([X_big, X_small]), tol=tol) == r_big


def assert_comparable(a: MVFit, b: MVFit) -> None:
    """Raise ComparisonError unless a and b were fitted on the same rows,
    same effect sizes and the same sampling V."""
    if a.spec.data_fingerprint != b.spec.data_fingerprint:
        ra, rb = set(map(str, a.spec.row_index)), set(map(str, b.spec.row_index))
        if ra != rb:
            raise ComparisonError(
                f"Models were fitted on different rows ({len(ra)} vs {len(rb)} effect "
                f"sizes; {len(ra ^ rb)} differ). Refit both on a common row set "
                f"(build_fit_spec(..., complete_case_cols=[...])).")
        raise ComparisonError(
            "Models use the same rows but different effect sizes or sampling V "
            "(e.g. full shared-control V vs diagonal V); they are not nested.")


def compare_fits(full: MVFit, reduced: MVFit, refit_ml: bool = True) -> FitComparison:
    """AIC/BIC/LRT comparison of two fits on the same rows and V.

    * Different rows or V            -> ComparisonError (never compared).
    * Same fixed effects             -> REML (or ML) compared as fitted.
    * Different fixed effects        -> REML log-likelihoods are not
      comparable; both are refitted under ML when refit_ml=True, otherwise
      ComparisonError.
    """
    assert_comparable(full, reduced)
    msgs: List[str] = []
    refitted = False

    if full.method != reduced.method:
        if not refit_ml:
            raise ComparisonError("Fits use different estimation methods (REML vs ML).")
        full, reduced, refitted = refit(full, 'ML'), refit(reduced, 'ML'), True

    same_design = full.spec.design_fingerprint == reduced.spec.design_fingerprint
    if not same_design and full.method == 'REML':
        if not refit_ml:
            raise ComparisonError(
                "REML log-likelihoods are not comparable across different fixed-effect "
                "designs; refit with method='ML'.")
        full, reduced, refitted = refit(full, 'ML'), refit(reduced, 'ML'), True
        msgs.append("Fixed effects differ: both models refitted by ML for the comparison.")
    if full.reml_full != reduced.reml_full:
        raise ComparisonError("Fits use different REML constants (reml_full).")

    # ---- nesting ------------------------------------------------------------
    nested_fixed = _col_space_contains(full.spec.X, reduced.spec.X)
    n_boundary, nested_var = 0, True
    for f_fix, r_fix in ((full.tau2_fixed, reduced.tau2_fixed),
                         (full.sigma2_fixed, reduced.sigma2_fixed)):
        if f_fix is None and r_fix is None:
            continue
        if f_fix is None and r_fix == 0:
            n_boundary += 1
        elif f_fix is None and r_fix is not None:
            pass  # fixed at an interior value: ordinary constraint
        elif f_fix is not None and r_fix is not None and f_fix == r_fix:
            continue
        else:
            nested_var = False
    nested = nested_fixed and nested_var
    df_diff = full.n_params - reduced.n_params

    lrt = p = None
    if nested and df_diff > 0:
        lrt = 2.0 * (full.loglik - reduced.loglik)
        if lrt < -1e-6:
            msgs.append(f"Full model log-likelihood is below the reduced model's "
                        f"(LRT = {lrt:.4f}): optimiser did not reach the maximum.")
        lrt = max(lrt, 0.0)
        p = chibar_pvalue(lrt, df_diff - n_boundary, n_boundary)
    elif not nested:
        msgs.append("Models are not nested: AIC/BIC shown, no LRT.")

    d_aic = reduced.aic - full.aic
    if nested and df_diff > 0 and d_aic < -2.0 * df_diff - 1e-6:
        msgs.append(f"ΔAIC = {d_aic:.3f} < -2·df: impossible for nested models; "
                    f"check optimiser convergence.")

    return FitComparison(full=full, reduced=reduced, method=full.method,
                         delta_aic=d_aic, delta_bic=reduced.bic - full.bic,
                         lrt=lrt, df=df_diff if nested else None,
                         n_boundary=n_boundary if nested else None, p_value=p,
                         nested=nested, refitted_ml=refitted, warnings=msgs)


# =============================================================================
# 7. PSEUDO-R² AGAINST THE MATCHING NULL
# =============================================================================

def pseudo_r2(model: MVFit, null: Optional[MVFit] = None) -> Dict[str, Any]:
    """Raudenbush-type pseudo-R² with a null that has the same rows, the same
    V, the same variance structure and the same method, intercept-only.
    The null is fitted automatically when not supplied.

    Returns total and per-level proportions (each truncated at 0 and
    reported untruncated as well), plus the raw components.
    """
    if null is None:
        null = fit_mv(model.spec.intercept_only(), tau2=model.tau2_fixed,
                      sigma2=model.sigma2_fixed, method=model.method,
                      reml_full=model.reml_full)
    assert_comparable(model, null)
    if (model.tau2_fixed, model.sigma2_fixed, model.method) != \
            (null.tau2_fixed, null.sigma2_fixed, null.method):
        raise ComparisonError("Null model must use the same variance structure and method.")

    def _r(n, m):
        if n <= 0:
            return np.nan
        return (n - m) / n

    raw_total = _r(null.total_heterogeneity, model.total_heterogeneity)
    raw_tau = _r(null.tau2, model.tau2)
    raw_sigma = _r(null.sigma2, model.sigma2)
    clip = lambda r: (max(0.0, r) if np.isfinite(r) else np.nan)
    return {
        'R2': clip(raw_total), 'R2_raw': raw_total,
        'R2_tau2': clip(raw_tau), 'R2_tau2_raw': raw_tau,
        'R2_sigma2': clip(raw_sigma), 'R2_sigma2_raw': raw_sigma,
        'tau2_null': null.tau2, 'sigma2_null': null.sigma2,
        'tau2_model': model.tau2, 'sigma2_model': model.sigma2,
        'null_fit': null,
    }


def fit_spec_from_blocks(y_blocks: Sequence[np.ndarray],
                         V_blocks: Sequence[np.ndarray],
                         X_blocks: Sequence[np.ndarray],
                         term_names: Optional[Sequence[str]] = None,
                         cluster_ids: Optional[Sequence[Any]] = None,
                         label: str = '') -> FitSpec:
    """FitSpec from per-cluster blocks that an analysis cell has already
    assembled (rows selected, V sliced by row identity). Row identity is the
    position in the stacked blocks, so specs built from the same blocks are
    comparable with each other."""
    y_blocks = [np.asarray(y, dtype=np.float64).ravel() for y in y_blocks]
    V_blocks = [np.asarray(V, dtype=np.float64) for V in V_blocks]
    X_blocks = [np.asarray(X, dtype=np.float64).reshape(len(y), -1)
                for X, y in zip(X_blocks, y_blocks)]
    if not (len(y_blocks) == len(V_blocks) == len(X_blocks)) or any(
            V.shape != (len(y), len(y)) for V, y in zip(V_blocks, y_blocks)):
        raise ValueError("fit_spec_from_blocks: y, V and X blocks are not aligned.")
    p = X_blocks[0].shape[1]
    names = list(term_names) if term_names is not None and len(term_names) == p \
        else [f"x{j}" for j in range(p)]
    cids = list(cluster_ids) if cluster_ids is not None else list(range(len(y_blocks)))
    n = int(sum(len(y) for y in y_blocks))
    acc = RowAccounting(n_input_effects=n, n_input_clusters=len(cids),
                        n_used_effects=n, n_used_clusters=len(cids))
    return FitSpec(row_index=np.arange(n), cluster_ids=cids, y_blocks=y_blocks,
                   V_blocks=V_blocks, X_blocks=X_blocks, term_names=names,
                   v_source={c: 'as supplied' for c in cids}, accounting=acc,
                   label=label)


def pseudo_r2_report(spec: FitSpec, three_level: bool,
                     reported: Optional[Tuple[float, float, str]] = None) -> Dict[str, Any]:
    """Pseudo-R² of the model in `spec` against its matching null, as a plain
    (JSON-safe) dict with percentages.

    Both models are fitted by REML on `spec`'s rows and V with the same random
    structure: three_level=True estimates tau2 and sigma2 in both; False (one
    effect per cluster) uses a single between-cluster component (sigma2 = 0).
    R² = (Σσ²_null − Σσ²_model) / Σσ²_null, truncated at 0 overall and per
    level; raw values kept. `reported` = (tau2, sigma2, model_type) of the
    model shown to the user: a warning is added if it differs from the REML
    fit used here.
    """
    model = fit_mv(spec, **({} if three_level else {'sigma2': 0.0}))
    r = pseudo_r2(model)
    pct = lambda x: float(100.0 * x) if np.isfinite(x) else float('nan')
    out = {
        'R2': pct(r['R2']), 'R2_raw': float(r['R2_raw']),
        'R2_tau2': pct(r['R2_tau2']), 'R2_tau2_raw': float(r['R2_tau2_raw']),
        'R2_sigma2': pct(r['R2_sigma2']) if three_level else float('nan'),
        'R2_sigma2_raw': float(r['R2_sigma2_raw']) if three_level else float('nan'),
        'tau2_null': float(r['tau2_null']), 'sigma2_null': float(r['sigma2_null']),
        'tau2_model': float(r['tau2_model']), 'sigma2_model': float(r['sigma2_model']),
        'three_level': bool(three_level),
        'structure': model.structure,
        'k_effects': int(spec.n_effects), 'k_clusters': int(spec.n_clusters),
        'v_type': 'full shared-control V' if spec.uses_full_vcv else 'diagonal V',
        'data_summary': spec.accounting.summary_text(),
        'warnings': [],
    }
    if reported is not None:
        t_rep, s_rep, mtype = reported
        scale = max(1e-4, model.tau2 + model.sigma2)
        if (abs(float(t_rep) - model.tau2) > 1e-3 * scale + 1e-6
                or abs(float(s_rep) - model.sigma2) > 1e-3 * scale + 1e-6):
            out['warnings'].append(
                f"The reported model ({mtype}: τ² = {float(t_rep):.4f}, σ² = "
                f"{float(s_rep):.4f}) differs from the REML fit on the same rows and "
                f"V used for the pseudo-R² (τ² = {model.tau2:.4f}, σ² = "
                f"{model.sigma2:.4f}); the pseudo-R² compares the matching REML pair.")
    return out


def fmt_r2(r2) -> str:
    """Pseudo-R² (in %) for display; 'n/a' when it could not be computed."""
    try:
        return f"{float(r2):.1f}%" if np.isfinite(r2) else "n/a"
    except (TypeError, ValueError):
        return "n/a"


def level_change_text(pct, raw) -> str:
    """Per-level reduction for display: the truncated value, plus the actual
    increase when the component went UP (raw < 0), so a truncated 0% is never
    mistaken for 'unchanged'."""
    try:
        raw = float(raw)
    except (TypeError, ValueError):
        return "n/a"
    if not np.isfinite(raw):
        return "n/a"
    if raw < 0:
        return f"0% (increased by {-100 * raw:.1f}%)"
    return fmt_r2(pct)


def r2_breakdown_html(d: Optional[Dict[str, Any]], cluster_word: str = 'studies') -> str:
    """Total and per-level pseudo-R², the null/model components, warnings."""
    if not d:
        return ""
    if d.get('error'):
        return (f"<p style='margin: 5px 0;'><b>Pseudo-R²:</b> not computable "
                f"({d['error']})</p>")
    parts = [f"<b>Pseudo-R² (total):</b> {fmt_r2(d['R2'])}"]
    if d.get('three_level'):
        parts.append(f"between-study τ²: {level_change_text(d['R2_tau2'], d['R2_tau2_raw'])}")
        parts.append(f"within-study σ²: {level_change_text(d['R2_sigma2'], d['R2_sigma2_raw'])}")
    html = f"<p style='margin: 5px 0;'>{' &nbsp;·&nbsp; '.join(parts)}</p>"
    null_s = f"τ² = {d['tau2_null']:.4f}"
    mod_s = f"τ² = {d['tau2_model']:.4f}"
    if d.get('three_level'):
        null_s += f", σ² = {d['sigma2_null']:.4f}"
        mod_s += f", σ² = {d['sigma2_model']:.4f}"
    html += (f"<p style='margin: 2px 0 5px 0; font-size: 0.85em; color: #6c757d;'>"
             f"Intercept-only model: {null_s}; moderator model: {mod_s} "
             f"(both REML, same {d['k_effects']} effects in {d['k_clusters']} {cluster_word}, "
             f"same {d['v_type']}). Negative reductions are truncated at 0"
             + (f" (untruncated total: {100 * d['R2_raw']:.1f}%)" if d['R2_raw'] < 0 else "")
             + ".</p>")
    for w in d.get('warnings', []):
        html += f"<p style='margin: 2px 0; font-size: 0.85em; color: #856404;'>⚠️ {w}</p>"
    return html


def accounting_html(d: Optional[Dict[str, Any]]) -> str:
    """'Data used' box from a stored accounting dict ('' when absent)."""
    if not d:
        return ''
    try:
        return RowAccounting.from_dict(d).to_html()
    except Exception:
        return ''


# =============================================================================
# 8. ONE ENGINE FOR EVERY MODULE
# =============================================================================

def has_replication(y_blocks: Sequence[np.ndarray]) -> bool:
    """True when at least one cluster contributes more than one effect size,
    i.e. the within-cluster component sigma² is identified."""
    return any(len(y) > 1 for y in y_blocks)


def fit_blocks(y_blocks: Sequence[np.ndarray],
               V_blocks: Sequence[np.ndarray],
               X_blocks: Optional[Sequence[np.ndarray]] = None,
               *,
               tau2: Optional[float] = None,
               sigma2: Optional[float] = None,
               method: str = 'REML',
               two_level: Optional[bool] = None,
               V_fallback_blocks: Optional[Sequence[np.ndarray]] = None,
               cluster_ids: Optional[Sequence[Any]] = None,
               term_names: Optional[Sequence[str]] = None,
               label: str = '',
               starts: Optional[Sequence[Tuple[float, float]]] = None) -> MVFit:
    """Fit the multilevel model on per-cluster blocks (y, sampling V, X).

    X_blocks          None -> intercept only.
    tau2 / sigma2     None -> estimated; a number -> held fixed (as fit_mv).
    two_level         None -> decided from the data: sigma² is fixed at 0 when
                      no cluster has more than one effect size, because tau²
                      and sigma² are then not separately identified (the
                      two-level model metafor's rma() fits). True / False
                      forces the choice.
    V_fallback_blocks Used instead of V_blocks when a block of V_blocks is not
                      positive semi-definite (not a valid sampling covariance)
                      or the fit with V_blocks cannot be evaluated; the
                      returned fit then has v_fallback = True and a note.
    """
    if X_blocks is None:
        X_blocks = [np.ones((len(y), 1)) for y in y_blocks]
    if two_level is None:
        two_level = not has_replication(y_blocks)
    if two_level and sigma2 is None:
        sigma2 = 0.0

    def _fit(Vb):
        spec = fit_spec_from_blocks(y_blocks, Vb, X_blocks, term_names=term_names,
                                    cluster_ids=cluster_ids, label=label)
        return fit_mv(spec, tau2=tau2, sigma2=sigma2, method=method, starts=starts)

    def _not_psd(V):
        V = np.asarray(V, dtype=np.float64)
        return V.shape[0] > 1 and np.linalg.eigvalsh(V).min() < -1e-10 * max(np.abs(np.diag(V)).max(), 1e-300)

    reason = None
    if V_fallback_blocks is not None:
        bad = [i for i, V in enumerate(V_blocks) if _not_psd(V)]
        if bad:
            reason = (f"sampling V of {len(bad)} cluster(s) is not positive semi-definite; "
                      f"diagonal V used")
    if reason is None:
        try:
            return _fit(V_blocks)
        except (RuntimeError, np.linalg.LinAlgError) as e:
            if V_fallback_blocks is None:
                raise
            reason = f"fit with the supplied sampling V failed ({e}); diagonal V used"
    fit = _fit(V_fallback_blocks)
    fit.v_fallback = True
    fit.notes.append(reason)
    warnings.warn(reason, RuntimeWarning, stacklevel=2)
    return fit


def fit_estimates(fit: MVFit) -> Dict[str, Any]:
    """The fit as the estimates dictionary the analysis cells pass around
    (model-based covariance; CR2 is added by the caller via fit_cr2)."""
    betas = np.asarray(fit.betas, dtype=np.float64)
    cov = np.asarray(fit.cov_beta, dtype=np.float64)
    se = np.sqrt(np.maximum(np.diag(cov), 0.0))
    out = {'betas': betas, 'cov_beta': cov, 'var_betas': cov, 'se_betas': se,
           'tau_sq': float(fit.tau2), 'sigma_sq': float(fit.sigma2),
           'n_obs': fit.spec.n_effects, 'k_studies': fit.spec.n_clusters,
           'optimizer_converged': bool(fit.converged)}
    out['log_lik_reml' if fit.method == 'REML' else 'log_lik_ml'] = float(fit.loglik)
    if fit.spec.n_coef == 1:      # intercept-only: the pooled effect
        out.update({'mu': float(betas[0]), 'se_mu': float(se[0]), 'var_mu': float(cov[0, 0])})
    return out


def fit_cr2(fit: MVFit, with_wald_helper: bool = False):
    """CR2 cluster-robust covariance and Satterthwaite df for `fit`, computed
    on exactly the blocks (y, V, X) and variance components it used
    (_compute_robust_var_betas, Cell 1)."""
    s = fit.spec
    return _compute_robust_var_betas(fit.betas, s.y_blocks, s.V_blocks, s.X_blocks,
                                     fit.tau2, fit.sigma2, with_wald_helper=with_wald_helper)


def fit_as_optimize_result(fit: MVFit):
    """scipy-style result (x = [tau², sigma²]) for callers that expect one."""
    from scipy.optimize import OptimizeResult
    return OptimizeResult(x=np.array([fit.tau2, fit.sigma2]), fun=-fit.loglik,
                          success=bool(fit.converged), message='fit_mv')


def profile_ci(fit: MVFit, component: str, alpha: float = 0.05) -> Tuple[float, float]:
    """Profile-likelihood CI for 'tau2' or 'sigma2', as metafor's confint():
    the values where the log-likelihood, with the other component
    re-estimated (or kept fixed if it was fixed in `fit`), falls
    chi2_{1-alpha}(1) / 2 below its maximum. The lower bound is 0 when the
    drop is not reached above 0; the upper bound is inf when the likelihood
    never falls far enough."""
    from scipy.optimize import brentq
    if component not in ('tau2', 'sigma2'):
        raise ValueError("component must be 'tau2' or 'sigma2'")
    est = fit.tau2 if component == 'tau2' else fit.sigma2
    target = fit.loglik - 0.5 * float(chi2.ppf(1 - alpha, df=1))

    def prof(v):
        if component == 'tau2':
            f = fit_mv(fit.spec, tau2=float(v), sigma2=fit.sigma2_fixed,
                       method=fit.method, reml_full=fit.reml_full)
        else:
            f = fit_mv(fit.spec, tau2=fit.tau2_fixed, sigma2=float(v),
                       method=fit.method, reml_full=fit.reml_full)
        return f.loglik - target

    try:
        lower = 0.0 if (est < 1e-6 or prof(0.0) >= 0) else \
            brentq(prof, 0.0, est, xtol=1e-10, maxiter=200)
    except (ValueError, RuntimeError):
        lower = 0.0
    try:
        start = max(est, 1e-6)
        hi = max(est * 100 + 2.0, 10.0)
        while prof(hi) > 0 and hi < 1e6:
            hi *= 10
        if prof(start) < 0:
            upper = start
        elif prof(hi) > 0:
            upper = np.inf
        else:
            upper = brentq(prof, start, hi, xtol=1e-10, maxiter=200)
    except (ValueError, RuntimeError):
        upper = np.inf
    return max(0.0, float(lower)), float(upper)


def tau2_univariate(yi: np.ndarray, vi: np.ndarray, method: str = 'REML') -> float:
    """Two-level (one effect per unit) REML or ML estimate of tau², the
    model metafor's rma(method = 'REML' / 'ML') fits."""
    yi = np.asarray(yi, dtype=np.float64)
    vi = np.asarray(vi, dtype=np.float64)
    fit = fit_blocks([np.array([y]) for y in yi], [np.array([[v]]) for v in vi],
                     two_level=True, method=method)
    return float(fit.tau2)


# =============================================================================
# 9. PROPORTION OF MEANINGFUL EFFECTS
# Mathur & VanderWeele (2019, Statistics in Medicine 38: 1336-1342; 2020,
# Epidemiology 31: 356-358): the share of TRUE effects beyond a threshold of
# scientific importance, as a complement to the mean and its CI.
#
# Multilevel version used here: the true effect of an effect size is
# distributed N(x'b, tau2 + sigma2) under the fitted model ('effect' level),
# and the true mean effect of a study N(x'b, tau2) ('study' level). With one
# effect per study (two-level model) both reduce to N(mu, tau2), the case of
# the original papers and of MetaUtility::prop_stronger.
#
#   parametric  P = 1 - Phi((q - mu) / T)  (share above q), with the
#               delta-method SE of Mathur & VanderWeele (2019, eq. in
#               MetaUtility::prop_stronger) from SE(mu) and SE(T^2).
#   calibrated  the share of calibrated estimates beyond q (Wang & Lee 2019;
#               Mathur & VanderWeele 2020): each effect size shrunk toward its
#               model mean by sqrt(T^2 / (T^2 + v)). No normality assumption.
#   CIs         BCa bootstrap resampling whole clusters (studies), the model
#               refitted in each resample, acceleration from the
#               leave-one-cluster-out jackknife; MetaUtility resamples
#               clusters the same way.
# =============================================================================

from scipy.stats import norm as _norm


def _num_hessian(f, x: np.ndarray, rel: float = 1e-3) -> np.ndarray:
    """Hessian of f at x by central differences with one Richardson step.
    Steps are relative to |x| and never cross 0 (variance components)."""
    x = np.asarray(x, dtype=np.float64)
    n = x.size

    def hess(h):
        H = np.empty((n, n))
        f0 = f(x)
        for i in range(n):
            ei = np.zeros(n)
            ei[i] = h[i]
            H[i, i] = (f(x + ei) - 2.0 * f0 + f(x - ei)) / h[i] ** 2
            for j in range(i + 1, n):
                ej = np.zeros(n)
                ej[j] = h[j]
                H[i, j] = H[j, i] = (f(x + ei + ej) - f(x + ei - ej) - f(x - ei + ej)
                                     + f(x - ei - ej)) / (4.0 * h[i] * h[j])
        return H

    h = np.minimum(rel * np.maximum(np.abs(x), 1e-6), np.abs(x) / 4.0)
    return (4.0 * hess(h / 2.0) - hess(h)) / 3.0


def var_components_vcov(fit: MVFit) -> np.ndarray:
    """Covariance of the estimated variance components (tau2, sigma2): the
    inverse of the negative Hessian of the (restricted) log-likelihood at the
    estimates, as metafor's rma.mv(..., cvvc = TRUE). A component that was
    held fixed or estimated at the boundary 0 is not a free parameter there
    and gets variance 0. NaN when the Hessian is not negative definite."""
    est = np.array([fit.tau2, fit.sigma2], dtype=np.float64)
    free = [i for i, (fixed, val) in enumerate(((fit.tau2_fixed, fit.tau2),
                                                 (fit.sigma2_fixed, fit.sigma2)))
            if fixed is None and val > 1e-10]
    out = np.zeros((2, 2))
    if not free:
        return out

    def ll(theta):
        th = est.copy()
        th[free] = theta
        return _mv_loglik(fit.spec, th[0], th[1], fit.method, fit.reml_full)

    H = _num_hessian(ll, est[free])
    try:
        cov = np.linalg.inv(-H)
    except np.linalg.LinAlgError:
        return np.full((2, 2), np.nan)
    if np.any(np.diag(cov) <= 0) or not np.all(np.isfinite(cov)):
        return np.full((2, 2), np.nan)
    for a, i in enumerate(free):
        for b, j in enumerate(free):
            out[i, j] = cov[a, b]
    return out


def true_effect_variance(tau2: float, sigma2: float, level: str = 'effect',
                         vc_vcov: Optional[np.ndarray] = None) -> Tuple[float, float]:
    """(T^2, SE(T^2)) of the true-effect distribution at the chosen level:
    'effect' -> tau2 + sigma2 (true effects of individual effect sizes),
    'study'  -> tau2 (true mean effects of studies)."""
    if level == 'study':
        t2, g = float(tau2), np.array([1.0, 0.0])
    else:
        t2, g = float(tau2) + float(sigma2), np.array([1.0, 1.0])
    if vc_vcov is None:
        return t2, np.nan
    var = float(g @ np.asarray(vc_vcov) @ g)
    return t2, (np.sqrt(var) if var >= 0 else np.nan)


def prop_beyond(mu: float, t2: float, threshold: float, tail: str) -> float:
    """Parametric share of true effects above ('above') or below ('below')
    `threshold` when true effects are N(mu, t2)."""
    if not t2 > 0:
        return float(mu > threshold) if tail == 'above' else float(mu < threshold)
    z = (threshold - mu) / np.sqrt(t2)
    return float(_norm.sf(z)) if tail == 'above' else float(_norm.cdf(z))


def prop_beyond_se(mu: float, t2: float, threshold: float, se_mu: float, se_t2: float) -> float:
    """Delta-method SE of prop_beyond (the same for both tails), as in
    MetaUtility::prop_stronger(ci.method = 'parametric')."""
    if not (t2 > 0 and np.isfinite(se_mu) and np.isfinite(se_t2)):
        return np.nan
    z = (threshold - mu) / np.sqrt(t2)
    term = np.sqrt(se_mu ** 2 / t2 + se_t2 ** 2 * (threshold - mu) ** 2 / (4.0 * t2 ** 3))
    return float(term * _norm.pdf(z))


def calibrated_effects(y: np.ndarray, v: np.ndarray, fitted: np.ndarray, t2: float) -> np.ndarray:
    """Calibrated estimates of the true effects (Wang & Lee 2019): each effect
    size shrunk toward its model mean by sqrt(T^2 / (T^2 + v))."""
    y, v, fitted = (np.asarray(a, dtype=np.float64) for a in (y, v, fitted))
    if not t2 > 0:
        return fitted.copy()
    return fitted + np.sqrt(t2 / (t2 + v)) * (y - fitted)


def _norm_inter(t: np.ndarray, alpha: Sequence[float]) -> np.ndarray:
    """Bootstrap quantiles as boot::norm.inter (Davison & Hinkley 1997, eq.
    5.8): order statistic (R + 1) * alpha, interpolated on the normal scale
    between neighbouring order statistics; the extremes when out of range."""
    t = np.sort(np.asarray(t, dtype=np.float64)[np.isfinite(t)])
    R = t.size
    out = []
    for a in np.atleast_1d(alpha):
        rk = (R + 1) * a
        k = int(np.trunc(rk))
        if k == rk and 1 <= k <= R:
            out.append(t[k - 1])
        elif k <= 0:
            out.append(t[0])
        elif k >= R:
            out.append(t[-1])
        else:
            z, z1, z2 = _norm.ppf(a), _norm.ppf(k / (R + 1)), _norm.ppf((k + 1) / (R + 1))
            out.append(t[k - 1] + (z - z1) / (z2 - z1) * (t[k] - t[k - 1]))
    return np.asarray(out)


def bca_interval(est: float, boot: np.ndarray, jack: Optional[np.ndarray],
                 alpha: float = 0.05) -> Tuple[float, float, str]:
    """BCa bootstrap interval (Efron 1987), as boot::boot.ci(type = 'bca')
    with the jackknife influence values as L. `boot`: replicates; `jack`:
    leave-one-cluster-out estimates for the acceleration (None -> 0).
    Ties with the estimate count half (proportions of calibrated estimates
    are discrete; boot counts only replicates below the estimate). Falls back
    to the percentile interval when the bias correction is undefined (every
    replicate on one side of the estimate)."""
    boot = np.asarray(boot, dtype=np.float64)
    boot = boot[np.isfinite(boot)]
    if boot.size < 20 or not np.isfinite(est):
        return np.nan, np.nan, 'not available'
    if np.all(boot == boot[0]):
        return float(boot[0]), float(boot[0]), 'no bootstrap variation'
    below = np.mean(boot < est) + 0.5 * np.mean(boot == est)
    zl, zu = _norm.ppf(alpha / 2.0), _norm.ppf(1.0 - alpha / 2.0)
    if below <= 0.0 or below >= 1.0:
        lo, hi = _norm_inter(boot, [alpha / 2.0, 1.0 - alpha / 2.0])
        return float(lo), float(hi), 'percentile'
    z0 = _norm.ppf(below)
    a = 0.0
    if jack is not None:
        jack = np.asarray(jack, dtype=np.float64)
        jack = jack[np.isfinite(jack)]
        if jack.size > 2:
            d = jack.mean() - jack
            den = 6.0 * np.sum(d ** 2) ** 1.5
            a = float(np.sum(d ** 3) / den) if den > 0 else 0.0
    a1 = _norm.cdf(z0 + (z0 + zl) / (1.0 - a * (z0 + zl)))
    a2 = _norm.cdf(z0 + (z0 + zu) / (1.0 - a * (z0 + zu)))
    lo, hi = _norm_inter(boot, [a1, a2])
    return float(lo), float(hi), 'BCa'


@dataclass
class MeaningfulModel:
    """What the proportion of meaningful effects needs from a fitted model.

    blocks      per-cluster (y, V, X) the model was fitted on.
    betas, tau2, sigma2   the fitted model.
    targets     label -> contrast row c (target mean = c @ betas): 'Overall'
                for the intercept-only model, one per subgroup level.
    se_mu       label -> SE of the target mean as reported (CR2 or model-based).
    vc_vcov     covariance of (tau2, sigma2) (var_components_vcov) or None
                (no delta-method CI, e.g. moment estimators of tau2).
    refit       function(y_blocks, V_blocks, X_blocks) -> (betas, tau2, sigma2)
                fitting the SAME model on resampled clusters (None on failure).
    """
    y_blocks: List[np.ndarray]
    V_blocks: List[np.ndarray]
    X_blocks: List[np.ndarray]
    betas: np.ndarray
    tau2: float
    sigma2: float
    targets: Dict[str, np.ndarray]
    se_mu: Dict[str, float]
    vc_vcov: Optional[np.ndarray]
    refit: Any
    model_label: str = ''

    def row_targets(self, X_blocks: Sequence[np.ndarray]) -> List[np.ndarray]:
        """Target label of every row (the target whose contrast equals the
        row's design), per block; '' when no target matches."""
        labels = list(self.targets)
        C = np.vstack([self.targets[k] for k in labels])
        out = []
        for X in X_blocks:
            lab = np.full(X.shape[0], '', dtype=object)
            for r in range(X.shape[0]):
                hit = np.where(np.all(np.isclose(C, X[r][None, :], atol=1e-12), axis=1))[0]
                if hit.size:
                    lab[r] = labels[hit[0]]
            out.append(lab)
        return out


def _meaningful_point(model: MeaningfulModel, betas, tau2, sigma2, y_blocks, V_blocks,
                      X_blocks, row_targets, thresholds, level) -> np.ndarray:
    """Vector of [param_above, param_below, calib_above, calib_below] per target."""
    q_lo, q_hi = thresholds
    t2, _ = true_effect_variance(tau2, sigma2, level)
    y = np.concatenate(y_blocks)
    v = np.concatenate([np.diag(V) for V in V_blocks])
    fitted = np.vstack(X_blocks) @ np.asarray(betas)
    rt = np.concatenate(row_targets)
    cal = calibrated_effects(y, v, fitted, t2) if level == 'effect' else None
    out = []
    for lab, c in model.targets.items():
        mu = float(np.asarray(c) @ np.asarray(betas))
        out += [prop_beyond(mu, t2, q_hi, 'above'), prop_beyond(mu, t2, q_lo, 'below')]
        sel = rt == lab
        if cal is not None and sel.any():
            out += [float(np.mean(cal[sel] > q_hi)), float(np.mean(cal[sel] < q_lo))]
        else:
            out += [np.nan, np.nan]
    return np.asarray(out, dtype=np.float64)


def meaningful_proportions(model: MeaningfulModel,
                           thresholds: Tuple[float, float],
                           level: str = 'effect',
                           alpha: float = 0.05,
                           n_boot: int = 0,
                           seed: int = 2024,
                           progress=None,
                           acceleration: bool = True) -> pd.DataFrame:
    """Share of true effects that are meaningfully large in either direction.

    thresholds  (q_lo, q_hi): a true effect above q_hi is a meaningful
                increase, below q_lo a meaningful decrease, in between
                negligible (q_lo < q_hi on the analysis scale).
    level       'effect' or 'study' (see true_effect_variance).
    n_boot      bootstrap resamples of clusters for BCa CIs (0 -> none; the
                parametric estimate then has the delta-method CI only).
    progress    optional callable(done, total).
    acceleration  False skips the leave-one-cluster-out jackknife (one refit
                per cluster): bias-corrected percentile intervals instead of
                BCa, for quick exploratory runs.

    One row per target and method ('parametric', 'calibrated'), with the
    three shares and their CIs.
    """
    q_lo, q_hi = map(float, thresholds)
    if not q_lo < q_hi:
        raise ValueError("The decrease threshold must be below the increase threshold.")
    if level not in ('effect', 'study'):
        raise ValueError("level must be 'effect' or 'study'")
    y_b, V_b, X_b = model.y_blocks, model.V_blocks, model.X_blocks
    rt = model.row_targets(X_b)
    est = _meaningful_point(model, model.betas, model.tau2, model.sigma2, y_b, V_b, X_b, rt,
                            (q_lo, q_hi), level)
    t2, se_t2 = true_effect_variance(model.tau2, model.sigma2, level, model.vc_vcov)
    labels = list(model.targets)
    n_stat = est.size
    M = len(y_b)

    boot = jack = None
    n_ok = 0
    if n_boot and n_boot > 0:
        rng = np.random.default_rng(seed)

        def stat(idx):
            yy, VV, XX = [y_b[i] for i in idx], [V_b[i] for i in idx], [X_b[i] for i in idx]
            r = model.refit(yy, VV, XX)
            if r is None:
                return None
            b, tt, ss = r
            return _meaningful_point(model, b, tt, ss, yy, VV, XX, [rt[i] for i in idx],
                                     (q_lo, q_hi), level)

        total = n_boot + (M if acceleration else 0)
        boot = np.full((n_boot, n_stat), np.nan)
        for r_i in range(n_boot):
            s = stat(rng.integers(0, M, M))
            if s is not None:
                boot[r_i] = s
                n_ok += 1
            if progress is not None:
                progress(r_i + 1, total)
        jack = np.full((M if acceleration else 0, n_stat), np.nan)
        for j in range(jack.shape[0]):
            s = stat(np.delete(np.arange(M), j))
            if s is not None:
                jack[j] = s
            if progress is not None:
                progress(n_boot + j + 1, total)

    z = _norm.ppf(1.0 - alpha / 2.0)
    rows = []
    for t_i, lab in enumerate(labels):
        mu = float(np.asarray(model.targets[lab]) @ np.asarray(model.betas))
        se_mu = float(model.se_mu.get(lab, np.nan))
        for m_i, method in enumerate(('parametric', 'calibrated')):
            base = 4 * t_i + 2 * m_i
            p_up, p_dn = est[base], est[base + 1]
            row = {'target': lab, 'method': method, 'level': level, 'mu': mu,
                   'T2': t2, 'q_lo': q_lo, 'q_hi': q_hi,
                   'p_increase': p_up, 'p_decrease': p_dn,
                   'p_negligible': (1.0 - p_up - p_dn) if np.isfinite(p_up + p_dn) else np.nan,
                   'n_boot': n_ok, 'note': ''}
            notes = []
            if method == 'calibrated' and level == 'study':
                notes.append('calibrated estimates are defined for individual effect sizes only')
            if method == 'parametric':
                for key, p, q in (('increase', p_up, q_hi), ('decrease', p_dn, q_lo)):
                    se = prop_beyond_se(mu, t2, q, se_mu, se_t2)
                    row[f'se_{key}_delta'] = se
                    row[f'{key}_lo_delta'] = max(0.0, p - z * se) if np.isfinite(se) else np.nan
                    row[f'{key}_hi_delta'] = min(1.0, p + z * se) if np.isfinite(se) else np.nan
                if not t2 > 0:
                    notes.append('no heterogeneity estimated (T² = 0): the shares are 0 or 1')
                elif any(np.isfinite(p) and (p < 0.15 or p > 0.85) for p in (p_up, p_dn)):
                    notes.append('share below 0.15 or above 0.85: the delta-method CI can be poor, '
                                 'use the bootstrap CI')
            if boot is not None:
                neg_b = 1.0 - boot[:, base] - boot[:, base + 1]
                neg_j = 1.0 - jack[:, base] - jack[:, base + 1]
                for key, b, j, e in (('increase', boot[:, base], jack[:, base], p_up),
                                     ('decrease', boot[:, base + 1], jack[:, base + 1], p_dn),
                                     ('negligible', neg_b, neg_j, row['p_negligible'])):
                    lo, hi, how = bca_interval(e, b, j if acceleration else None, alpha)
                    if how == 'BCa' and not acceleration:
                        how = 'bias-corrected percentile'
                    row[f'{key}_lo_boot'], row[f'{key}_hi_boot'] = lo, hi
                    row['boot_ci_method'] = how
                if n_ok < n_boot:
                    notes.append(f'{n_boot - n_ok} of {n_boot} resamples could not be fitted '
                                 f'(e.g. a subgroup missing from the resample)')
            row['note'] = '; '.join(notes)
            rows.append(row)
    return pd.DataFrame(rows)

# Later cells check this flag and stop with a clear message when Cells 1 and
# 1b have not run in the current session (e.g. after a runtime restart).
COMETA_SETUP_DONE = True


In [ ]:
#@title ⚙️ 2. Data Ingestion
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
#  DATA INGESTION & COLUMN MAPPING
# =============================================================================
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd
import numpy as np
import io
import html
try:
    import gspread              # Google Sheets import (Colab only)
except ImportError:
    gspread = None


# --- Configuration: Required Columns & Synonyms ---
RAW_COLUMN_SPECS = {
    'id':  ['id', 'study', 'study_id', 'paper', 'author'],
    'xe':  ['xe', 'mean_e', 'mean_exp', 'x_e', 'treatment_mean'],
    'sde': ['sde', 'sd_e', 'sd_exp', 'sigma_e'],
    'ne':  ['ne', 'n_e', 'n_exp', 'sample_e'],
    'xc':  ['xc', 'mean_c', 'mean_ctrl', 'x_c', 'control_mean'],
    'sdc': ['sdc', 'sd_c', 'sd_ctrl', 'sigma_c'],
    'nc':  ['nc', 'n_c', 'n_ctrl', 'sample_c']
}

BINARY_COLUMN_SPECS = {
    'id':          ['id', 'study', 'study_id', 'paper', 'author'],
    'events_e':    ['events_e', 'events_exp', 'cases_e', 'cases_exp', 'a', 'treatment_events'],
    'nonevents_e': ['nonevents_e', 'nonevents_exp', 'control_e', 'b', 'treatment_nonevents'],
    'events_c':    ['events_c', 'events_ctrl', 'cases_c', 'cases_ctrl', 'c', 'control_events'],
    'nonevents_c': ['nonevents_c', 'nonevents_ctrl', 'control_c', 'd', 'control_nonevents']
}

PRECALC_COLUMN_SPECS = {
    'id':       ['id', 'study', 'study_id', 'paper', 'author'],
    'yi':       ['yi', 'effect_size', 'es', 'hedges_g', 'lnrr', 'smd', 'effect', 'g', 'd'],
    'variance': ['variance', 'vi', 'var', 'v'],
    'se':       ['se', 'standard_error', 'stderr', 'se_es'],
    'n_total':  ['n_total', 'n', 'sample_size', 'total_n', 'sample_n']
}

# --- Geographic Column Synonyms ---
GEO_COLUMN_SPECS = {
    'latitude':  ['latitude', 'lat'],
    'longitude': ['longitude', 'lon', 'long', 'lng'],
    'country':   ['country', 'nation', 'region']
}

# Global placeholders
temp_raw_df = None
_data_type_widget = None
_mapping_container = None
# Ensure ANALYSIS_CONFIG always exists at module level so downstream cells
if 'ANALYSIS_CONFIG' not in globals():
    ANALYSIS_CONFIG = {}

# =============================================================================
# SAMPLE DATASETS
# ==================================================
BCG_DATA = {np.str_('trial'): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13], 'id': ['Aronson', 'Ferguson & Simes', 'Rosenthal et al', 'Hart & Sutherland', 'Frimodt-Moller et al', 'Stein & Aronson', 'Vandiviere et al', 'TPT Madras', 'Coetzee & Berjak', 'Rosenthal et al', 'Comstock et al', 'Comstock & Webster', 'Comstock et al'], np.str_('year'): [1948, 1949, 1960, 1977, 1973, 1953, 1973, 1980, 1968, 1961, 1974, 1969, 1976], 'events_e': [4, 6, 3, 62, 33, 180, 8, 505, 29, 17, 186, 5, 27], 'nonevents_e': [119, 300, 228, 13536, 5036, 1361, 2537, 87886, 7470, 1699, 50448, 2493, 16886], 'events_c': [11, 29, 11, 248, 47, 372, 10, 499, 45, 65, 141, 3, 29], 'nonevents_c': [128, 274, 209, 12619, 5761, 1079, 619, 87892, 7232, 1600, 27197, 2338, 17825], np.str_('ablat'): [44, 55, 42, 52, 13, 44, 19, 13, 27, 42, 18, 33, 33], 'allocation': ['random', 'random', 'random', 'random', 'alternate', 'alternate', 'random', 'random', 'random', 'systematic', 'systematic', 'systematic', 'systematic']}
NORMAND_DATA = {np.str_('study'): [1, 2, 3, 4, 5, 6, 7, 8, 9], 'id': ['Edinburgh', 'Orpington-Mild', 'Orpington-Moderate', 'Orpington-Severe', 'Montreal-Home', 'Montreal-Transfer', 'Newcastle', 'Umea', 'Uppsala'], 'ne': [155, 31, 75, 18, 8, 57, 34, 110, 60], 'xe': [55, 27, 64, 66, 14, 19, 52, 21, 30], 'sde': [47, 7, 17, 20, 8, 7, 45, 16, 27], 'nc': [156, 32, 71, 18, 13, 52, 33, 183, 52], 'xc': [75, 29, 119, 137, 18, 18, 41, 31, 23], 'sdc': [64, 4, 29, 48, 11, 4, 34, 27, 20]}
KONST_DATA = {'id': [11, 11, 11, 11, 12, 12, 12, 12, 18, 18, 18, 27, 27, 27, 27, 56, 56, 56, 56, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 58, 71, 71, 71, 86, 86, 86, 86, 86, 86, 86, 86, 91, 91, 91, 91, 91, 91, 108, 108, 108, 108, 108, 644, 644, 644, 644], 'school_id': [1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 1, 2, 3, 1, 2, 3, 4, 5, 6, 7, 8, 1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 1, 2, 3, 4], np.str_('study'): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56], np.str_('year'): [1976, 1976, 1976, 1976, 1989, 1989, 1989, 1989, 1994, 1994, 1994, 1976, 1976, 1976, 1976, 1997, 1997, 1997, 1997, 1976, 1976, 1976, 1976, 1976, 1976, 1976, 1976, 1976, 1976, 1976, 1997, 1997, 1997, 1997, 1997, 1997, 1997, 1997, 1997, 1997, 1997, 2000, 2000, 2000, 2000, 2000, 2000, 2000, 2000, 2000, 2000, 2000, 1995, 1995, 1994, 1994], np.str_('yi'): [-0.18, -0.22, 0.23, -0.3, 0.13, -0.26, 0.19, 0.32, 0.45, 0.38, 0.29, 0.16, 0.65, 0.36, 0.6, 0.08, 0.04, 0.19, -0.06, -0.18, 0.0, 0.0, -0.28, -0.04, -0.3, 0.07, 0.0, 0.05, -0.08, -0.09, 0.3, 0.98, 1.19, -0.07, -0.05, -0.01, 0.02, -0.03, 0.0, 0.01, -0.1, 0.5, 0.66, 0.2, 0.0, 0.05, 0.07, -0.52, 0.7, -0.03, 0.27, -0.34, 0.12, 0.61, 0.04, -0.05], np.str_('vi'): [0.118, 0.118, 0.144, 0.144, 0.014, 0.014, 0.015, 0.024, 0.023, 0.043, 0.012, 0.02, 0.004, 0.004, 0.007, 0.019, 0.007, 0.005, 0.004, 0.02, 0.018, 0.019, 0.022, 0.02, 0.021, 0.006, 0.007, 0.007, 0.007, 0.007, 0.015, 0.011, 0.01, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.01, 0.011, 0.01, 0.009, 0.013, 0.013, 0.031, 0.031, 0.03, 0.03, 0.03, 0.087, 0.082, 0.067, 0.067]}
RAUDENBUSH_DATA = {'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19], np.str_('author'): ['Rosenthal et al.', 'Conn et al.', 'Jose & Cody', 'Pellegrini & Hicks', 'Pellegrini & Hicks', 'Evans & Rosenthal', 'Fielder et al.', 'Claiborn', 'Kester', 'Maxwell', 'Carter', 'Flowers', 'Keshock', 'Henrikson', 'Fine', 'Grieger', 'Rosenthal & Jacobson', 'Fleming & Anttonen', 'Ginsburg'], np.str_('year'): [1974, 1968, 1971, 1972, 1972, 1969, 1971, 1969, 1969, 1970, 1970, 1966, 1970, 1970, 1972, 1970, 1968, 1971, 1970], np.str_('weeks'): [2, 21, 19, 0, 0, 3, 17, 24, 0, 1, 0, 0, 1, 2, 17, 5, 1, 2, 7], np.str_('setting'): ['group', 'group', 'group', 'group', 'group', 'group', 'group', 'group', 'group', 'indiv', 'group', 'group', 'indiv', 'indiv', 'group', 'group', 'group', 'group', 'group'], np.str_('tester'): ['aware', 'aware', 'aware', 'aware', 'blind', 'aware', 'blind', 'aware', 'aware', 'blind', 'blind', 'blind', 'blind', 'blind', 'aware', 'blind', 'aware', 'blind', 'aware'], np.str_('n1i'): [77, 60, 72, 11, 11, 129, 110, 26, 75, 32, 22, 43, 24, 19, 80, 72, 65, 233, 65], np.str_('n2i'): [339, 198, 72, 22, 22, 348, 636, 99, 74, 32, 22, 38, 24, 32, 79, 72, 255, 224, 67], np.str_('yi'): [0.03, 0.12, -0.14, 1.18, 0.26, -0.06, -0.02, -0.32, 0.27, 0.8, 0.54, 0.18, -0.02, 0.23, -0.18, -0.06, 0.3, 0.07, -0.07], np.str_('vi'): [0.0156, 0.0216, 0.0279, 0.1391, 0.1362, 0.0106, 0.0106, 0.0484, 0.0269, 0.063, 0.0912, 0.0497, 0.0835, 0.0841, 0.0253, 0.0279, 0.0193, 0.0088, 0.0303]}
CURTIS_DATA = {'row_id': [21, 22, 27, 32, 35, 38, 44, 63, 86, 87, 95, 96, 120, 125, 130, 163, 170, 182, 208, 209, 230, 231, 236, 242, 254, 256, 257, 267, 277, 287, 296, 306, 313, 320, 327, 334, 343, 344, 392, 393, 394, 395, 410, 411, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 479, 480, 523, 524, 525, 526, 547, 548, 561, 562, 575, 576, 614, 615, 623, 624, 636, 637, 721, 725, 726, 727, 731, 732, 733, 737, 738, 739, 743, 744, 745, 749, 750, 751, 755, 756, 757, 765, 774, 783], 'id': [44, 44, 121, 121, 121, 121, 159, 183, 209, 209, 210, 210, 290, 290, 290, 468, 470, 503, 505, 505, 506, 506, 510, 510, 550, 553, 553, 582, 582, 582, 582, 582, 596, 596, 596, 596, 666, 666, 2003, 2003, 2003, 2003, 2026, 2026, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2037, 2039, 2039, 2045, 2045, 2045, 2045, 2048, 2048, 2048, 2048, 2048, 2048, 2110, 2110, 2117, 2117, 2117, 2117, 2217, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2223, 2224, 2224, 2224], np.str_('genus'): ['ALNUS', 'ALNUS', 'ACER', 'QUERCUS', 'MALUS', 'ACER', 'CASTANEA', 'CITRUS', 'CASTANEA', 'CASTANEA', 'CASTANEA', 'CASTANEA', 'PINUS', 'NOTHOFAGUS', 'PSEUDOTSUGA', 'CASTANEA', 'CASTANEA', 'PINUS', 'QUERCUS', 'QUERCUS', 'LIRIODENDRON', 'LIRIODENDRON', 'QUERCUS', 'PINUS', 'BETULA', 'PICEA', 'PICEA', 'PIPER', 'SENNA', 'MYRIOCARPA', 'CECROPIA', 'TRICHOSPERMUM', 'BETULA', 'BETULA', 'BETULA', 'BETULA', 'PINUS', 'PINUS', 'BETULA', 'BETULA', 'BETULA', 'BETULA', 'PINUS', 'PINUS', 'ACER', 'ACER', 'ACER', 'QUERCUS', 'QUERCUS', 'QUERCUS', 'ACER', 'ACER', 'ACER', 'BETULA', 'BETULA', 'BETULA', 'FRAXINUS', 'FRAXINUS', 'FRAXINUS', 'BETULA', 'BETULA', 'BETULA', 'POPULUSX', 'POPULUSX', 'PICEA', 'PICEA', 'PICEA', 'PICEA', 'PICEA', 'PICEA', 'PICEA', 'PICEA', 'PINUS', 'PINUS', 'POPULUS', 'POPULUS', 'PICEA', 'PICEA', 'BETULA', 'BETULA', 'MARANTHES', 'ACER', 'ACER', 'ACER', 'QUERCUS', 'QUERCUS', 'QUERCUS', 'ACER', 'ACER', 'ACER', 'BETULA', 'BETULA', 'BETULA', 'FRAXINUS', 'FRAXINUS', 'FRAXINUS', 'BETULA', 'BETULA', 'BETULA', 'QUERCUS', 'ACER', 'POPULUS'], np.str_('species'): ['RUBRA', 'RUBRA', 'RUBRUM', 'PRINUS', 'DOMESTICA', 'SACCHARINUM', 'SATIVA', 'SINENSIS', 'SATIVA', 'SATIVA', 'SATIVA', 'SATIVA', 'RADIATA', 'FUSCA', 'MENZIESII', 'SATIVA', 'SATIVA', 'ECHINATA', 'ALBA', 'ALBA', 'TULIPIFERA', 'TULIPIFERA', 'ALBA', 'ECHINATA', 'PENDULA', 'ABIES', 'ABIES', 'AURITUM', 'MULTIJUGA', 'LONGIPES', 'OBTUSIFOLIA', 'MEXICANUM', 'LENTA', 'PAPYRIFERA', 'POPULIFOLIA', 'ALLEGHANIENSIS', 'BANKSIANA', 'BANKSIANA', 'PUBESCENS', 'PUBESCENS', 'PUBESCENS', 'PUBESCENS', 'PONDEROSA', 'PONDEROSA', 'RUBRUM', 'RUBRUM', 'RUBRUM', 'RUBRA', 'RUBRA', 'RUBRA', 'PENSYLVANICUM', 'PENSYLVANICUM', 'PENSYLVANICUM', 'POPULIFOLIA', 'POPULIFOLIA', 'POPULIFOLIA', 'AMERICANA', 'AMERICANA', 'AMERICANA', 'ALLEGHANIENSIS', 'ALLEGHANIENSIS', 'ALLEGHANIENSIS', 'EURAMERICANA', 'EURAMERICANA', 'MARIANA', 'MARIANA', 'MARIANA', 'MARIANA', 'GLAUCA', 'GLAUCA', 'MARIANA', 'MARIANA', 'BANKSIANA', 'BANKSIANA', 'EURAMERICANA', 'EURAMERICANA', 'ABIES', 'ABIES', 'PENDULA', 'PENDULA', 'CORYMBOSA', 'RUBRUM', 'RUBRUM', 'RUBRUM', 'RUBRA', 'RUBRA', 'RUBRA', 'PENSYLVANICUM', 'PENSYLVANICUM', 'PENSYLVANICUM', 'POPULIFOLIA', 'POPULIFOLIA', 'POPULIFOLIA', 'AMERICANA', 'AMERICANA', 'AMERICANA', 'ALLEGHANIENSIS', 'ALLEGHANIENSIS', 'ALLEGHANIENSIS', 'RUBRA', 'SACCHARUM', 'TREMULOIDES'], np.str_('fungrp'): ['N2FIX', 'N2FIX', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'GYMNO', 'ANGIO', 'GYMNO', 'ANGIO', 'ANGIO', 'GYMNO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'GYMNO', 'ANGIO', 'GYMNO', 'GYMNO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'GYMNO', 'GYMNO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'GYMNO', 'GYMNO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'GYMNO', 'ANGIO', 'ANGIO', 'GYMNO', 'GYMNO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO', 'ANGIO'], np.str_('co2.ambi'): [350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 395.0, 350.0, 350.0, 350.0, 350.0, 340.0, 340.0, 340.0, 350.0, 350.0, 368.0, 389.0, 389.0, 371.0, 371.0, 360.0, 360.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 34.5, 34.5, 340.0, 340.0, 340.0, 340.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 34.5, 34.5, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 350.0, 385.0, 385.0, 385.0], np.str_('co2.elev'): [650.0, 650.0, 700.0, 700.0, 700.0, 700.0, 700.0, 795.0, 700.0, 700.0, 700.0, 700.0, 640.0, 640.0, 640.0, 700.0, 700.0, 695.0, 793.0, 793.0, 787.0, 787.0, 700.0, 700.0, 700.0, 750.0, 750.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 750.0, 750.0, 700.0, 700.0, 560.0, 560.0, 650.0, 650.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 69.3, 69.3, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 69.3, 69.3, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 700.0, 642.0, 642.0, 642.0], np.str_('units'): ['ul/l', 'ul/l', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ul/l', 'ul/l', 'ul/l', 'umol/mol', 'ppm', 'ul/l', 'cm3/m3', 'cm3/m3', 'ppm', 'ppm', 'ul/l', 'ul/l', 'umol/mol', 'cm3/m3', 'cm3/m3', 'ppm', 'ppm', 'ppm', 'ppm', 'ppm', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'ubar', 'ubar', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'Pa', 'Pa', 'ppm', 'ppm', 'ppm', 'ppm', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'Pa', 'Pa', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'umol/mol', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l', 'ul/l'], np.str_('time'): [47, 47, 59, 70, 64, 50, 730, 365, 365, 365, 120, 120, 120, 120, 120, 1095, 180, 287, 168, 168, 168, 168, 210, 168, 40, 180, 180, 111, 111, 111, 111, 111, 90, 90, 90, 90, 270, 270, 34, 34, 35, 35, 60, 60, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 1095, 148, 148, 155, 155, 155, 155, 112, 112, 112, 112, 112, 112, 158, 158, 47, 47, 35, 35, 210, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 165, 60, 60, 60], np.str_('pot'): ['0.5', '0.5', '2.6', '2.6', '2.6', '2.6', 'GRND', '9', '24', '24', '12', '12', '4', '4', '4', '24', '24', '0.95', '2.6', '2.6', '3.5', '3.5', '0.95', '0.95', 'HYDRO', '50', '50', '0.67', '0.67', '0.67', '0.67', '0.67', '0.66', '0.66', '0.66', '0.66', '0.04', '0.04', '0.2', '0.2', '0.3', '0.3', '1.8', '1.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', '2.8', 'GRND', 'GRND', '0.164', '0.164', '0.164', '0.164', '0.2', '0.2', '0.2', '0.2', '0.2', '0.2', 'GRND', 'GRND', '0.2', '0.2', '0.2', '0.2', '10', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '1.25', '6', '6', '6'], np.str_('method'): ['GC', 'GC', 'GH', 'GH', 'GH', 'GH', 'GC', 'GH', 'GH', 'GH', 'GH', 'GH', 'GC', 'GC', 'GC', 'OTC', 'OTC', 'GC', 'GH', 'GH', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'OTC', 'OTC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'GC', 'OTC', 'OTC', 'GH', 'GH', 'GH', 'GH', 'OTC', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GH', 'GC', 'GC', 'GC'], np.str_('stock'): ['SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SAP', 'SAP', 'SEED', 'SEED', 'SAP', 'SAP', 'SAP', 'SAP', 'SAP', 'SAP', 'SAP', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SAP', 'SAP', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SAP', 'SAP', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SAP', 'SAP', 'SEED', 'SEED', 'SAP', 'SAP', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED', 'SEED'], np.str_('xtrt'): ['FERT', 'FERT', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'FERT', 'FERT', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'FERT', 'FERT', 'FERT', 'FERT', 'NONE', 'NONE', 'NONE', 'OZONE', 'OZONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'NONE', 'UVB', 'UVB', 'TEMP', 'TEMP', 'OZONE', 'OZONE', 'TEMP', 'TEMP', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'FERT', 'H2O', 'H2O', 'FERT', 'FERT', 'UVB', 'UVB', 'UVB', 'UVB', 'UVB', 'UVB', 'FERT', 'FERT', 'TEMP', 'TEMP', 'TEMP', 'TEMP', 'NONE', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'FERT', 'LIGHT', 'FERT+L', 'NONE', 'NONE', 'NONE'], np.str_('level'): ['HIGH', 'CONTROL', '.', '.', '.', '.', '.', '.', 'HIGH', 'CONTROL', '.', '.', '.', '.', '.', '.', '.', '.', 'HIGH', 'CONTROL', 'HIGH', 'CONTROL', '.', '.', '.', 'HIGH', 'LOW', '.', '.', '.', '.', '.', '.', '.', '.', '.', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'HIGH', 'LOW', 'WW', 'DRT', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', 'HIGH', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', 'LOW', 'LOW', '.', '.', '.', '.'], 'xe': [6.8168999999999995, 2.5961, 2.99, 5.91, 4.61, 10.78, 153.5, 1439.0, 183.6, 71.72, 45.8, 72.5, 59.1, 4.9, 19.0, 146.0, 129.0, 6.91, 17.89, 14.68, 63.97, 6.01, 14.15, 2.2, 23.11, 262.1, 273.3, 10.14, 15.72, 9.01, 21.33, 14.68, 8.36, 7.91, 12.45, 7.84, 0.09315999999999999, 0.59046, 4.77, 3.74, 3.15, 3.78, 2.15, 1.76, 47.38, 258.53, 482.04, 30.59, 172.4, 277.46, 23.08, 196.8, 254.39, 31.1, 186.0, 232.37, 25.19, 120.8, 262.77, 38.36, 178.33, 186.26, 113.0, 75.0, 4.773, 3.272, 5.249, 2.798, 0.333, 0.64, 0.498, 0.824, 0.843, 1.533, 562.8, 374.4, 1.092, 1.12, 6.91, 6.94, 16.959, 2.4, 16.98, 28.18, 3.09, 13.8, 18.2, 1.9, 10.23, 16.59, 2.34, 16.22, 24.55, 1.51, 9.12, 18.62, 1.9, 14.45, 16.98, 16.9, 7.2, 102.6], 'sde': [1.769982, 0.6674662, 0.856, 1.742, 1.407, 1.163, 27.1932, 142.028, 39.06, 14.3, 15.8, 15.0, 12.9, 1.9, 5.4, 10.0, 59.0, 3.22, 10.185, 5.013, 7.66, 1.02, 8.6467, 0.4111, 0.8443, 16.4, 14.0, 2.52, 2.76, 2.58, 3.06, 2.82, 3.915, 1.809, 6.944, 3.324, 0.071404, 0.41036900000000004, 0.198, 0.0424, 0.297, 0.424, 0.36, 0.28, 5.015, 50.315, 14.801, 2.152, 23.678, 14.834, 4.294, 26.614, 32.624, 1.073, 17.729, 8.867, 2.151, 20.773, 17.832, 3.215, 11.833, 11.834, 4.472, 8.944, 2.1477, 0.9215, 1.6628, 0.5958, 0.147, 0.154, 0.163, 0.234, 0.234, 0.395, 62.3863, 109.5673, 0.0297, 0.0141, 0.3111, 0.4243, 2.186, 0.5634, 1.9596, 3.2333, 0.7348, 2.425, 3.1843, 0.1225, 0.6124, 2.9149, 0.2694, 2.8414, 1.3962, 0.1715, 1.5922, 2.1556, 0.2205, 1.6657, 0.5389, 1.7321, 1.7321, 6.2354], 'ne': [3, 5, 5, 5, 4, 5, 3, 3, 20, 16, 20, 24, 8, 8, 8, 5, 12, 5, 5, 5, 5, 5, 6, 10, 22, 6, 6, 4, 4, 4, 4, 4, 10, 10, 10, 10, 10, 10, 2, 2, 2, 2, 16, 16, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 5, 5, 48, 48, 48, 48, 10, 10, 10, 10, 10, 10, 5, 5, 2, 2, 2, 2, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 3, 3, 3], 'xc': [3.945, 2.2512, 1.93, 6.62, 4.1, 6.42, 127.3, 1140.6, 144.1, 59.94, 28.2, 60.5, 46.4, 4.2, 18.5, 136.0, 90.4, 6.81, 14.62, 11.18, 54.03, 4.93, 8.2, 1.33, 14.94, 235.5, 236.4, 8.81, 13.22, 9.8, 18.6, 12.18, 5.3, 3.6, 4.98, 4.01, 0.07382, 0.41117000000000004, 3.83, 3.4, 2.4, 3.17, 1.7, 1.66, 32.99, 221.0, 396.31, 44.8, 44.93, 187.67, 18.8, 109.07, 168.88, 29.36, 161.05, 232.69, 23.2, 114.68, 168.74, 26.49, 161.74, 180.48, 76.0, 60.0, 3.862, 2.49, 4.031, 2.35, 0.243, 0.443, 0.328, 0.55, 0.599, 0.842, 381.6, 298.2, 0.971, 0.975, 5.95, 5.78, 10.578, 2.24, 10.96, 19.5, 2.04, 4.36, 14.79, 1.51, 5.89, 13.8, 2.24, 11.48, 19.95, 2.14, 5.49, 13.49, 1.48, 8.71, 16.59, 7.2, 4.6, 69.7], 'sdc': [1.115797, 0.32758390000000004, 0.552, 1.631, 1.257, 2.026, 47.4582, 82.965, 25.7, 14.28, 11.9, 14.0, 13.1, 1.0, 8.1, 35.0, 36.0, 1.57, 2.294, 5.749, 1.62, 1.35, 3.3558, 0.1897, 0.8443, 15.0, 7.5, 2.06, 1.94, 2.06, 2.34, 2.46, 6.625, 3.026, 3.937, 2.106, 0.08329399999999999, 0.255702, 0.1414, 0.0566, 0.014, 0.424, 0.32, 0.28, 5.375, 32.542, 41.697, 7.879, 11.845, 26.635, 4.668, 29.574, 20.734, 3.943, 8.876, 17.897, 1.078, 26.625, 26.83, 1.79, 14.856, 11.867, 6.708, 8.944, 1.4272, 0.6582, 1.2956, 0.582, 0.118, 0.13, 0.127, 0.112, 0.132, 0.151, 64.3988, 96.3745, 0.0255, 0.0127, 0.198, 0.099, 1.21, 0.3919, 1.9351, 2.2535, 0.3674, 2.474, 3.5028, 0.1715, 1.3962, 1.5922, 0.3919, 2.0086, 2.3025, 0.4899, 1.3227, 0.7593, 0.1715, 1.5187, 0.9553, 2.5981, 1.7321, 3.6373], 'nc': [5, 5, 5, 5, 4, 3, 3, 3, 20, 16, 23, 24, 8, 8, 8, 5, 12, 5, 5, 5, 5, 5, 6, 10, 22, 6, 6, 4, 4, 4, 4, 4, 10, 10, 10, 10, 10, 10, 2, 2, 2, 2, 16, 16, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 5, 5, 48, 48, 48, 48, 10, 10, 10, 10, 10, 10, 5, 5, 2, 2, 2, 2, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 3, 3, 3]}

BUILT_IN_DATASETS = {
    # Existing Datasets
    'Binary (BCG Vaccine - Tuberculosis)': {'data': BCG_DATA, 'type': 'raw_binary'},
    'Continuous (Normand 1999 - Stroke Rehab)': {'data': NORMAND_DATA, 'type': 'raw_continuous'},
    '3-Level Pre-Calculated (Konstantopoulos 2011)': {'data': KONST_DATA, 'type': 'pre_calculated'},
    'Meta-Regression / Splines (Raudenbush 1985)': {'data': RAUDENBUSH_DATA, 'type': 'pre_calculated'},
    'Ecology Continuous (Curtis 1998 - Plant CO2)': {'data': CURTIS_DATA, 'type': 'raw_continuous'}
}
# =============================================================================
# HELPER: UNIVERSAL FILE EXTRACTOR
# =============================================================================
def get_uploaded_file_data(uploader_widget):
    """Robustly extracts filename and binary content from ipywidgets.FileUpload."""
    val = uploader_widget.value
    if not val:
        return None, None
    try:
        if isinstance(val, (tuple, list)):
            file_obj = val[0]
            fname = file_obj['name']
            content = file_obj['content']
        elif isinstance(val, dict):
            keys = list(val.keys())
            if not keys:
                return None, None
            fname = keys[0]
            content = val[fname]['content']
        else:
            raise ValueError(f"Unknown widget format: {type(val)}")

        if hasattr(content, 'tobytes'):
            content = content.tobytes()
        return fname, content

    except Exception as e:
        print(f"Debug Info - Raw uploader value type: {type(val)}")
        raise e


# =============================================================================
# HELPER: DUPLICATE COLUMN
# =============================================================================
def _check_duplicate_columns(df, source_label="file"):
    """
    Raises ValueError if the DataFrame has duplicate column names, listing
    the offenders so the user knows exactly what to fix.
    """
    if df.columns.duplicated().any():
        dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
        raise ValueError(
            f"The {source_label} contains duplicate column names: "
            f"{dupes}. "
            f"Please rename them in the source file so each column is unique, "
            f"then re-upload."
        )


# =============================================================================
# HELPER: NUMERIC COLUMN VALIDATOR FOR FINALIZE
# =============================================================================

# Columns that MUST be coercible to numeric for each data type.
# Any NaN produced by coercion on these columns = a dropped row.
_NUMERIC_REQUIRED = {
    'raw_continuous': ['xe', 'sde', 'ne', 'xc', 'sdc', 'nc'],
    'raw_binary':     ['events_e', 'nonevents_e', 'events_c', 'nonevents_c'],
    'pre_calculated': ['yi'],           # variance/se handled separately below
}
_NUMERIC_SOFT = {
    # soft = coerce + warn but do NOT drop rows, downstream can decide
    'pre_calculated': ['variance', 'se', 'n_total'],
}

def _validate_and_coerce_mapped_numerics(df, col_map, data_type):
    """
    For each mapped column that is required to be numeric:
      1. Coerces the column to numeric (errors → NaN).
      2. Counts how many rows became NaN.
      3. Collects per-column warnings.
      4. Raises ValueError if any HARD-required column has > 0 NaN rows
         (after giving the user a clear breakdown).

    For soft-required columns (variance, se, n_total in pre_calculated mode)
    it warns but does not raise.

    Parameters
    ----------
    df       : DataFrame already renamed to standard column names
    col_map  : {std_name: original_col_name} — used only for the error message
               so the user sees their own column name, not the internal one
    data_type: 'raw_continuous' | 'raw_binary' | 'pre_calculated'

    Returns
    -------
    df       : DataFrame with numeric columns coerced
    warn_html: HTML string (empty string if no warnings)
    """
    df = df.copy()
    hard_fields = _NUMERIC_REQUIRED.get(data_type, [])
    soft_fields = _NUMERIC_SOFT.get(data_type, [])

    hard_errors = []   # will block finalize
    soft_warnings = [] # will display but allow through

    for std_name in hard_fields:
        if std_name not in df.columns:
            continue
        original_name = col_map.get(std_name, std_name)
        before = df[std_name].notna().sum()
        df[std_name] = pd.to_numeric(df[std_name], errors='coerce')
        after  = df[std_name].notna().sum()
        lost   = int(before - after)
        if lost > 0:
            # Identify which rows so the user can fix the source
            bad_idx = df.index[df[std_name].isna()].tolist()
            # Try to show study IDs if available, otherwise row numbers
            if 'id' in df.columns:
                bad_labels = df.loc[bad_idx, 'id'].astype(str).tolist()
                label_str  = ", ".join(bad_labels[:6])
                if len(bad_labels) > 6:
                    label_str += f" … +{len(bad_labels)-6} more"
            else:
                label_str = ", ".join(f"row {i+2}" for i in bad_idx[:6])  # +2: 1-indexed + header
                if len(bad_idx) > 6:
                    label_str += f" … +{len(bad_idx)-6} more"

            hard_errors.append(
                f"<li style='margin-bottom:6px;'>"
                f"<b>{original_name}</b> → <code>{std_name}</code>: "
                f"<b>{lost}</b> non-numeric value(s) found. "
                f"Affected studies/rows: <i>{label_str}</i>."
                f"</li>"
            )

    for std_name in soft_fields:
        if std_name not in df.columns:
            continue
        original_name = col_map.get(std_name, std_name)
        before = df[std_name].notna().sum()
        df[std_name] = pd.to_numeric(df[std_name], errors='coerce')
        after  = df[std_name].notna().sum()
        lost   = int(before - after)
        if lost > 0:
            soft_warnings.append(
                f"<li style='margin-bottom:6px;'>"
                f"<b>{original_name}</b> → <code>{std_name}</code>: "
                f"<b>{lost}</b> non-numeric value(s) coerced to NaN. "
                f"These rows will have missing variance/SE — check downstream diagnostics."
                f"</li>"
            )

    # --- Build HTML output ---
    warn_html = ""

    if hard_errors:
        items = "".join(hard_errors)
        # Raise with HTML so the caller can display it properly
        err_html = (
            f"<div style='background-color:#f8d7da; color:#721c24; padding:12px 15px; "
            f"border-radius:6px; border:1px solid #f5c6cb; margin-top:10px;'>"
            f"<b>❌ Non-numeric values in required columns</b><br>"
            f"<small>The following mapped columns contain text or missing values where numbers "
            f"are required. Please fix the source data and re-upload, or remove those rows.</small>"
            f"<ul style='margin:8px 0 0 0; padding-left:18px;'>{items}</ul>"
            f"</div>"
        )
        raise ValueError(err_html)   # caller displays this as HTML, not plain text

    if soft_warnings:
        items = "".join(soft_warnings)
        warn_html = (
            f"<div style='background-color:#fff3cd; color:#856404; padding:12px 15px; "
            f"border-radius:6px; border:1px solid #ffc107; margin-top:10px;'>"
            f"<b>⚠️ Non-numeric values in optional columns</b>"
            f"<ul style='margin:8px 0 0 0; padding-left:18px;'>{items}</ul>"
            f"</div>"
        )

    return df, warn_html



# =============================================================================
# HELPER: COERCE NUMERIC COLUMNS
# =============================================================================
def _coerce_numeric_columns(df):
    """
    Google Sheets returns everything as strings. Converts columns that are
    predominantly numeric to numeric types.

    Defensive changes vs original:
      - Skips columns whose name contains 'id', 'name', 'label', 'author',
        'study', 'paper' (case-insensitive) to avoid corrupting ID fields.
      - Raises threshold to 0.8 to reduce false-positive coercions.
    """
    _ID_LIKE = {'id', 'name', 'label', 'author', 'study', 'paper', 'title'}
    df = df.copy()
    for col in df.columns:
        # Skip columns that are already numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            continue
        # Skip obvious identifier columns
        col_lower = str(col).lower().strip()
        if any(tok in col_lower for tok in _ID_LIKE):
            continue
        try:
            converted = pd.to_numeric(df[col], errors='coerce')
            non_empty = df[col].replace('', np.nan).dropna()
            if len(non_empty) > 0:
                valid_ratio = converted.notna().sum() / len(non_empty)
                if valid_ratio >= 0.8:
                    df[col] = converted
        except Exception:
            pass
    return df

# -----------------------------------------------------------------------------
# HELPER: Error display
# -----------------------------------------------------------------------------

def _show_finalize_error(e):
    """Displays a user-friendly error and prints the full traceback to stdout."""
    import traceback
    tb = traceback.format_exc()
    display(HTML(
        f"<div style='padding:10px; background-color:#f8d7da; border-left:4px solid #dc3545; "
        f"color:#721c24; border-radius:4px; margin-top:10px;'>"
        f"❌ <b>Mapping Error:</b> {e}"
        f"<br><small style='color:#999;'>See cell output below for full traceback.</small></div>"
    ))
    print(tb)

# -----------------------------------------------------------------------------
# HELPER: Safe CSV reader with encoding fallback# and duplicate-column detection.
# -----------------------------------------------------------------------------
def _safe_read_csv(content_bytes):
    """
    Attempts to parse CSV bytes with UTF-8, then falls back to common
    Western encodings.  Also detects duplicate column names and warns.
    Returns a DataFrame.
    """
    _ENCODINGS = ['utf-8', 'latin-1', 'windows-1252', 'utf-8-sig']
    last_exc = None
    for enc in _ENCODINGS:
        try:
            df = pd.read_csv(io.BytesIO(content_bytes), encoding=enc)
            # P8 — duplicate column check
            _check_duplicate_columns(df, 'CSV file')
            return df
        except UnicodeDecodeError as e:
            last_exc = e
            continue
    raise ValueError(
        f"Could not decode the CSV file with any of the attempted encodings "
        f"({', '.join(_ENCODINGS)}). Last error: {last_exc}"
    )

# =============================================================================
# HELPER: VALIDATE GEOGRAPHIC DATA & BUILD WARNINGS
# =============================================================================
def _validate_geo_data(df, geo_map):
    """
    Validates mapped geographic columns and returns:
      - warnings_html: HTML string with per-row warnings
      - summary_html:  HTML string with overall geo summary
      - geo_type:      'coordinates' | 'country' | 'both' | None
    """
    warnings = []
    has_coords = ('latitude' in geo_map and 'latitude' in df.columns
                  and 'longitude' in geo_map and 'longitude' in df.columns)
    has_country = ('country' in geo_map and 'country' in df.columns)

    if not has_coords and not has_country:
        return "", "", None

    geo_type_parts = []
    if has_coords:
        geo_type_parts.append('coordinates')
    if has_country:
        geo_type_parts.append('country')
    geo_type = '+'.join(geo_type_parts) if len(geo_type_parts) == 2 else geo_type_parts[0]

    coord_missing = 0
    country_missing = 0

    if has_coords:
        lat_s = pd.to_numeric(df['latitude'], errors='coerce')
        lon_s = pd.to_numeric(df['longitude'], errors='coerce')
        coord_missing = (lat_s.isna() | lon_s.isna()).sum()

        if coord_missing > 0:
            missing_ids = df.loc[lat_s.isna() | lon_s.isna(), 'id'].tolist() if 'id' in df.columns else []
            id_hint = f" (studies: {', '.join(str(s) for s in missing_ids[:5])}{'…' if len(missing_ids)>5 else ''})" if missing_ids else ""
            warnings.append(f"⚠️ <b>{coord_missing}</b> row(s) have missing or non-numeric coordinates{id_hint}. These will be excluded from the geographic map.")

        lat_oob = ((lat_s < -90) | (lat_s > 90)).sum()
        lon_oob = ((lon_s < -180) | (lon_s > 180)).sum()
        if lat_oob > 0:
            warnings.append(f"⚠️ <b>{lat_oob}</b> latitude value(s) are outside the valid range (−90 to 90).")
        if lon_oob > 0:
            warnings.append(f"⚠️ <b>{lon_oob}</b> longitude value(s) are outside the valid range (−180 to 180).")

    if has_country:
        country_s = df['country'].astype(str).str.strip().replace('', np.nan)
        country_missing = country_s.isna().sum() + (country_s == 'nan').sum()
        if country_missing > 0:
            warnings.append(f"⚠️ <b>{country_missing}</b> row(s) have missing country/region values. These will be excluded from country-level map layers.")

    warnings_html = ""
    if warnings:
        items = "".join(f"<li style='margin-bottom:4px;'>{w}</li>" for w in warnings)
        warnings_html = f"""
        <div style='background-color:#fff3cd; color:#856404; padding:10px 15px; border-radius:6px;
                    border:1px solid #ffc107; margin-top:10px; margin-bottom:10px;'>
            <b>🌍 Geographic Data Warnings</b>
            <ul style='margin:6px 0 0 0; padding-left:18px;'>{items}</ul>
        </div>
        """

    parts = []
    if has_coords:
        valid_coords = len(df) - (coord_missing if has_coords else 0)
        parts.append(f"Coordinates: <b>{valid_coords}</b>/{len(df)} valid")
    if has_country:
        valid_countries = len(df) - country_missing
        n_unique = country_s.nunique()
        parts.append(f"Countries: <b>{valid_countries}</b>/{len(df)} valid ({n_unique} unique)")
    summary_html = "<br>🌍 Geographic data: " + " &nbsp;·&nbsp; ".join(parts) if parts else ""

    return warnings_html, summary_html, geo_type


# =============================================================================
# HELPER: RICH MODERATOR SUMMARY
# =============================================================================
def _build_moderator_summary_html(df, mapped_cols, extra_cols, mode_label,
                                   extra_info="", geo_warnings_html="",
                                   geo_summary_html="", geo_type=None):
    """Builds a rich HTML summary showing moderators, their unique values, and record counts."""
    n_rows = len(df)
    n_mapped = len(mapped_cols)
    geo_badge = f" &nbsp;·&nbsp; 🌍 Geo: <b>{geo_type}</b>" if geo_type else ""

    html = f"""
    <div style='background-color:#d4edda; color:#155724; padding:15px; border-radius:8px;
                border:1px solid #c3e6cb; margin-bottom:15px;'>
        <h4 style='margin:0 0 8px 0;'>✅ Data Ready ({mode_label})</h4>
        <b>{n_rows}</b> rows loaded &nbsp;·&nbsp;
        <b>{n_mapped}</b> mapped columns &nbsp;·&nbsp;
        <b>{len(extra_cols)}</b> moderator columns detected{geo_badge}
        {extra_info}
        {geo_summary_html}
    </div>
    """

    html += geo_warnings_html

    if extra_cols:
        html += """
        <div style='background-color:#f8f9fa; padding:12px; border-radius:8px;
                    border:1px solid #dee2e6; margin-bottom:15px;'>
            <h4 style='color:#2E86AB; margin:0 0 10px 0;'>📊 Moderator Variables Overview</h4>
            <table style='width:100%; border-collapse:collapse; font-size:13px;'>
                <thead>
                    <tr style='border-bottom:2px solid #dee2e6; text-align:left;'>
                        <th style='padding:6px 10px;'>Column</th>
                        <th style='padding:6px 10px;'>Type</th>
                        <th style='padding:6px 10px;'>Unique</th>
                        <th style='padding:6px 10px;'>Missing</th>
                        <th style='padding:6px 10px;'>Values (count)</th>
                    </tr>
                </thead>
                <tbody>
        """
        MAX_VALUES_SHOWN = 8

        for col in extra_cols:
            series = df[col]
            n_unique = series.nunique()
            n_missing = series.isna().sum() + (series.astype(str).str.strip() == '').sum()
            is_numeric = pd.api.types.is_numeric_dtype(series)
            col_type = "Numeric" if is_numeric else "Categorical"

            if is_numeric:
                clean = pd.to_numeric(series, errors='coerce').dropna()
                if len(clean) > 0:
                    val_summary = f"Range: {clean.min():.3g} – {clean.max():.3g}, Mean: {clean.mean():.3g}"
                else:
                    val_summary = "<i>No valid numeric data</i>"
            else:
                counts = series.value_counts(dropna=True)
                parts = []
                for val, cnt in counts.head(MAX_VALUES_SHOWN).items():
                    val_str = str(val).strip()
                    if len(val_str) > 30:
                        val_str = val_str[:27] + "..."
                    parts.append(f"<b>{val_str}</b>&nbsp;({cnt})")
                val_summary = ", &nbsp;".join(parts)
                if len(counts) > MAX_VALUES_SHOWN:
                    val_summary += f", &nbsp;<i>… +{len(counts) - MAX_VALUES_SHOWN} more</i>"

            miss_style = "color:#c0392b; font-weight:bold;" if n_missing > 0 else ""

            html += f"""
                <tr style='border-bottom:1px solid #eee;'>
                    <td style='padding:6px 10px; font-weight:600;'>{col}</td>
                    <td style='padding:6px 10px;'>{col_type}</td>
                    <td style='padding:6px 10px;'>{n_unique}</td>
                    <td style='padding:6px 10px; {miss_style}'>{n_missing}</td>
                    <td style='padding:6px 10px; font-size:12px;'>{val_summary}</td>
                </tr>
            """

        html += """
                </tbody>
            </table>
        </div>
        """
    else:
        html += """
        <div style='background-color:#fff3cd; color:#856404; padding:10px; border-radius:5px;
                    border:1px solid #ffc107; margin-bottom:15px;'>
            ⚠️ No moderator columns detected beyond the mapped variables.
            If you expected moderator variables, verify that your spreadsheet includes them.
        </div>
        """

    html += "<p style='color:#555; font-size:12px;'>Please proceed to the next cell to configure analysis filters.</p>"
    return html


# =============================================================================
# HELPER: BUILD GEO MAPPING UI SECTION
# =============================================================================
def _build_geo_mapping_widgets(df):
    cols_lower = [str(c).lower().strip() for c in df.columns]
    options_with_none = ['None'] + list(df.columns)
    geo_widgets = {}

    GEO_FIELD_INFO = {
        'latitude':  {'label': 'Latitude:',  'desc': 'Decimal latitude (−90 to 90). Optional.'},
        'longitude': {'label': 'Longitude:', 'desc': 'Decimal longitude (−180 to 180). Optional.'},
        'country':   {'label': 'Country / Region:', 'desc': 'Country or region name. Optional.'}
    }

    ui_rows = [widgets.HTML("""
    <hr>
    <h4 style='color:#2E86AB; margin-top:15px;'>🌍 Geographic Columns <span style='font-size:12px; color:#888; font-weight:normal;'>(Optional)</span></h4>
    <div style='background-color:#e8f5e9; padding:10px; border-radius:5px; margin-bottom:10px; color:#2e7d32; font-size:13px;'>
        <b>Optional:</b> Map geographic columns to enable a publication-quality study location map in a later cell.<br>
        You can map <b>coordinates</b> (Lat + Lon), <b>country/region</b>, or <b>both</b>. Leave as "None" to skip.
    </div>
    """)]

    for std_name, synonyms in GEO_COLUMN_SPECS.items():
        info = GEO_FIELD_INFO[std_name]
        guessed_val = 'None'
        for syn in synonyms:
            if syn in cols_lower:
                guessed_val = df.columns[cols_lower.index(syn)]
                break
        w = widgets.Dropdown(
            options=options_with_none,
            value=guessed_val,
            description=info['label'],
            style={'description_width': '180px'},
            layout=widgets.Layout(width='600px')
        )
        geo_widgets[std_name] = w
        ui_rows.append(widgets.VBox([
            w,
            widgets.HTML(f"<div style='margin-left:185px; font-size:11px; color:#666; margin-bottom:8px'>"
                         f"<i>{info['desc']}</i></div>")
        ]))

    return widgets.VBox(ui_rows), geo_widgets

def _finalize_geo_columns(df, geo_widgets, col_map_values):
    geo_map = {}
    for std_name, w in geo_widgets.items():
        val = w.value
        if val != 'None':
            geo_map[std_name] = val

    has_lat = 'latitude' in geo_map
    has_lon = 'longitude' in geo_map
    if has_lat != has_lon:
        mapped_one = 'Latitude' if has_lat else 'Longitude'
        missing_one = 'Longitude' if has_lat else 'Latitude'
        raise ValueError(f"You mapped {mapped_one} but not {missing_one}. Please map both coordinates or neither.")

    if not geo_map:
        return geo_map, "", "", None

    for std_name, orig_col in geo_map.items():
        if orig_col in col_map_values:
            raise ValueError(f"Geographic column '{orig_col}' is already mapped to a core analysis field. Please choose a different column or set it to 'None'.")

    rename_map = {}
    for std_name, orig_col in geo_map.items():
        if orig_col != std_name:
            rename_map[orig_col] = std_name
    if rename_map:
        df.rename(columns=rename_map, inplace=True)

    resolved_geo_map = {std_name: std_name for std_name in geo_map}
    geo_warnings_html, geo_summary_html, geo_type = _validate_geo_data(df, resolved_geo_map)

    return geo_map, geo_warnings_html, geo_summary_html, geo_type


# =============================================================================
# UI PART 1: DATA LOADING (Tabs)
# =============================================================================
# --- Tab 1: Google Sheets ---
btn_auth = widgets.Button(description="1. Connect Google Account", button_style='warning', icon='google')
txt_sheet_name = widgets.Text(value='tesis', description='Sheet Name:', layout=widgets.Layout(width='300px'))
btn_fetch_ws = widgets.Button(description="2. Find Worksheets", button_style='primary', disabled=True)
dd_worksheets = widgets.Dropdown(description='Worksheet:', layout=widgets.Layout(width='300px'), disabled=True)
btn_load_sheet = widgets.Button(description="3. Load Data", button_style='success', disabled=True)

if globals().get('IS_COLAB', False):
    gs_vbox = widgets.VBox([
        widgets.HTML("<b>Step A: Load from Google Sheets</b>"),
        btn_auth,
        widgets.HBox([txt_sheet_name, btn_fetch_ws]),
        widgets.HBox([dd_worksheets, btn_load_sheet])
    ])
else:
    gs_vbox = widgets.VBox([
        widgets.HTML("""
        <div style='padding: 20px; background-color: #fff3cd; border-left: 5px solid #ffc107; border-radius: 4px; font-family: sans-serif;'>
            <h3 style='margin-top: 0; color: #856404;'>⚠️ Cloud Feature Only</h3>
            <p style='color: #856404; font-size: 14px;'>Google Sheets integration is only supported when running this tool online in Google Colaboratory.</p>
            <p style='color: #856404; font-size: 14px; margin-bottom: 0;'><b>Please click the "Upload Excel/CSV" tab above to load your local data.</b></p>
        </div>
        """)
    ])

# --- Tab 2: Local File ---
uploader = widgets.FileUpload(accept='.csv,.xlsx,.xls', multiple=False, description='Select Data')
dd_local_sheets = widgets.Dropdown(
    description='Select Sheet:',
    layout=widgets.Layout(width='300px', display='none'),
    style={'description_width': 'initial'}
)
btn_process_file = widgets.Button(description="Load Data File", button_style='success', disabled=True)
local_vbox = widgets.VBox([
    widgets.HTML(
        "<b>Step B: Upload Local File</b><br>"
        "1. Choose a .csv or .xlsx file.<br>"
        "2. Click 'Load Data File' to proceed."
    ),
    uploader,
    dd_local_sheets,
    btn_process_file
])

# --- Output Areas ---
log_output = widgets.Output()
step2_header_output = widgets.Output()
step2_body_output = widgets.Output()

# =============================================================================
# LOGIC PART 1: LOADERS
# =============================================================================
def on_auth_clicked(b):
    with log_output:
        clear_output(wait=True)
        display(HTML("<div style='padding: 10px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; margin-bottom: 10px;'>⏳ <b>Authenticating with Google...</b> Please follow the popup instructions.</div>"))
        try:
            auth.authenticate_user()
            creds, _ = default()
            global gc
            gc = gspread.authorize(creds)
            btn_auth.button_style = 'success'
            btn_auth.description = "Connected ✓"
            btn_auth.disabled = True
            btn_fetch_ws.disabled = False
            clear_output(wait=True)
            display(HTML("<div style='padding: 10px; background-color: #d4edda; border-left: 4px solid #28a745; color: #155724; margin-bottom: 10px;'>✅ <b>Authentication successful.</b> You may now find your worksheets.</div>"))
        except Exception as e:
            clear_output(wait=True)
            display(HTML(f"<div style='padding: 10px; background-color: #f8d7da; border-left: 4px solid #dc3545; color: #721c24; margin-bottom: 10px;'>❌ <b>Authentication failed:</b> {e}</div>"))

def on_fetch_ws_clicked(b):
    with log_output:
        clear_output(wait=True)
        if 'gc' not in globals():
            display(HTML("<div style='padding: 10px; background-color: #fff3cd; border-left: 4px solid #ffc107; color: #856404; margin-bottom: 10px;'>⚠️ <b>Please connect your Google Account first.</b></div>"))
            return
        name = txt_sheet_name.value.strip()
        if not name:
            display(HTML("<div style='padding: 10px; background-color: #fff3cd; border-left: 4px solid #ffc107; color: #856404; margin-bottom: 10px;'>⚠️ <b>Please enter a valid Google Sheet name.</b></div>"))
            return
        display(HTML(f"<div style='padding: 10px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; margin-bottom: 10px;'>🔎 <b>Searching for spreadsheet:</b> '{name}'...</div>"))
        try:
            global spreadsheet
            spreadsheet = gc.open(name)
            titles = [ws.title for ws in spreadsheet.worksheets()]
            dd_worksheets.options = titles
            dd_worksheets.disabled = False
            btn_load_sheet.disabled = False
            clear_output(wait=True)
            display(HTML(f"<div style='padding: 10px; background-color: #d1ecf1; border-left: 4px solid #17a2b8; color: #0c5460; margin-bottom: 10px;'>✓ <b>Found {len(titles)} worksheet(s).</b> Please select one and click 'Load Data'.</div>"))
        except Exception as e:
            clear_output(wait=True)
            display(HTML(f"<div style='padding: 10px; background-color: #f8d7da; border-left: 4px solid #dc3545; color: #721c24; margin-bottom: 10px;'>❌ <b>Error locating spreadsheet:</b> Ensure the name is exact and the account has access. ({e})</div>"))

def on_load_sheet_clicked(b):
    df = None
    with log_output:
        clear_output(wait=True)
        display(HTML(
            f"<div style='padding:10px; background-color:#e2e3e5; border-left:4px solid #6c757d; "
            f"color:#383d41; margin-bottom:10px;'>📥 <b>Downloading worksheet:</b> "
            f"'{dd_worksheets.value}'...</div>"
        ))
        try:
            ws = spreadsheet.worksheet(dd_worksheets.value)
            rows = ws.get_all_values()
            if len(rows) < 2:
                raise ValueError("Worksheet is empty or lacks data rows below the header.")

            df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
            df.columns = [str(c).strip() for c in df.columns]

            # P6 — empty DataFrame guard
            if df.empty:
                raise ValueError(
                    "The worksheet has a header row but no data rows. "
                    "Please check the sheet content."
                )

            df = df.replace('', np.nan).dropna(how='all').dropna(axis=1, how='all').fillna('')
            df = _coerce_numeric_columns(df)

            global temp_raw_df
            temp_raw_df = df
            clear_output(wait=True)
            display(HTML(
                f"<div style='padding:10px; background-color:#d4edda; border-left:4px solid #28a745; "
                f"color:#155724; margin-bottom:10px;'>✅ <b>Data successfully loaded!</b> "
                f"Analyzed {len(df)} rows and {len(df.columns)} columns. "
                f"Proceed to Part 2 below.</div>"
            ))

        except Exception as e:
            import traceback                    # P5
            tb = traceback.format_exc()
            clear_output(wait=True)
            display(HTML(
                f"<div style='padding:10px; background-color:#f8d7da; border-left:4px solid #dc3545; "
                f"color:#721c24; margin-bottom:10px;'>❌ <b>Error loading data:</b> {e}</div>"
            ))
            print(tb)
            return

    # called outside the with-block; guarded so df is always bound
    if df is not None:
        _initiate_mapping_interface(df)

# --- LOCAL FILE LOGIC ---
def on_file_upload(change):
    with log_output:
        clear_output(wait=True)
        try:
            dd_local_sheets.layout.display = 'none'
            btn_process_file.disabled = True
            fname, content_bytes = get_uploaded_file_data(uploader)
            if not fname: return

            display(HTML(f"<div style='padding: 10px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; margin-bottom: 10px;'>🔎 <b>Analyzing file structure:</b> '{fname}'...</div>"))

            if fname.endswith(('.xls', '.xlsx')):
                excel_file = pd.ExcelFile(io.BytesIO(content_bytes))
                sheets = excel_file.sheet_names
                dd_local_sheets.options = sheets
                dd_local_sheets.value = sheets[0]
                dd_local_sheets.layout.display = 'block'
                clear_output(wait=True)
                if len(sheets) > 1:
                    display(HTML(f"<div style='padding: 10px; background-color: #d1ecf1; border-left: 4px solid #17a2b8; color: #0c5460; margin-bottom: 10px;'>✓ <b>Excel file recognized.</b> Found {len(sheets)} sheets. Please select the target sheet below.</div>"))
                else:
                    display(HTML("<div style='padding: 10px; background-color: #d1ecf1; border-left: 4px solid #17a2b8; color: #0c5460; margin-bottom: 10px;'>✓ <b>Excel file ready.</b> 1 sheet found.</div>"))
            else:
                clear_output(wait=True)
                display(HTML("<div style='padding: 10px; background-color: #d1ecf1; border-left: 4px solid #17a2b8; color: #0c5460; margin-bottom: 10px;'>✓ <b>CSV file structure verified.</b> Ready to load.</div>"))
            btn_process_file.disabled = False
        except Exception as e:
            clear_output(wait=True)
            display(HTML(f"<div style='padding: 10px; background-color: #f8d7da; border-left: 4px solid #dc3545; color: #721c24; margin-bottom: 10px;'>❌ <b>Error reading file structure:</b> {e}</div>"))

def on_process_file_clicked(b):
    with log_output:
        clear_output(wait=True)
        df = None                               # P1: always initialise df
        try:
            fname, content_bytes = get_uploaded_file_data(uploader)

            # P4: explicit None / empty guard before any use of content_bytes
            if not fname or content_bytes is None:
                display(HTML(
                    "<div style='padding:10px; background-color:#fff3cd; border-left:4px solid #ffc107; "
                    "color:#856404; margin-bottom:10px;'>"
                    "⚠️ <b>No file detected.</b> Please select a file before clicking Load.</div>"
                ))
                return

            if fname.endswith('.csv'):
                display(HTML(
                    "<div style='padding:10px; background-color:#e2e3e5; border-left:4px solid #6c757d; "
                    "color:#383d41; margin-bottom:10px;'>📥 <b>Parsing CSV data...</b></div>"
                ))
                df = _safe_read_csv(content_bytes)

            else:
                sheet_target = dd_local_sheets.value
                display(HTML(
                    f"<div style='padding:10px; background-color:#e2e3e5; border-left:4px solid #6c757d; "
                    f"color:#383d41; margin-bottom:10px;'>📥 <b>Parsing Excel sheet:</b> '{sheet_target}'...</div>"
                ))
                df = pd.read_excel(io.BytesIO(content_bytes), sheet_name=sheet_target)
                _check_duplicate_columns(df, fname)

            df.columns = [str(c).strip() for c in df.columns]

            # P6 — empty DataFrame guard
            if df.empty:
                display(HTML(
                    "<div style='padding:10px; background-color:#f8d7da; border-left:4px solid #dc3545; "
                    "color:#721c24; margin-bottom:10px;'>"
                    "❌ <b>The file contains no data rows.</b> "
                    "Please check that the file has at least one row below the header.</div>"
                ))
                return

            global temp_raw_df
            temp_raw_df = df
            clear_output(wait=True)
            display(HTML(
                f"<div style='padding:10px; background-color:#d4edda; border-left:4px solid #28a745; "
                f"color:#155724; margin-bottom:10px;'>✅ <b>Data successfully loaded!</b> "
                f"Analyzed {len(df)} rows and {len(df.columns)} columns. "
                f"Proceed to Part 2 below.</div>"
            ))

        except Exception as e:
            clear_output(wait=True)
            import traceback                    # P5: preserve traceback
            tb = traceback.format_exc()
            display(HTML(
                f"<div style='padding:10px; background-color:#f8d7da; border-left:4px solid #dc3545; "
                f"color:#721c24; margin-bottom:10px;'>❌ <b>File processing error:</b> {e}</div>"
            ))
            print(tb)                           # visible in cell output / logs
            return

    # P1: only call if df was successfully assigned
    if df is not None:
        _initiate_mapping_interface(df)


# =============================================================================
# LOGIC PART 2: COLUMN MAPPING (The "Bridge")
# =============================================================================
def _initiate_mapping_interface(df):
    global _data_type_widget

    with step2_header_output:
        clear_output(wait=True)

        display(HTML("""
        <h3 style='color:#2E86AB; margin-bottom:10px; border-bottom: 2px solid #3498db; padding-bottom: 5px;'>Part 2: Select Data Type & Map Columns</h3>
        <div style='background-color:#e7f2fa; padding:15px; border-radius:6px; color:#2c3e50; margin-bottom:15px; border-left: 4px solid #3498db;'>
            <b>Please select your input data format:</b><br>
            <ul style='margin-top: 8px; margin-bottom: 0;'>
                <li><b>Raw Data:</b> You have means, standard deviations, and sample sizes for your Treatment and Control groups.</li>
                <li><b>Pre-calculated:</b> You already have standardized effect sizes (e.g., Hedges' g) and variances calculated.</li>
            </ul>
        </div>
        """))

        _data_type_widget = widgets.RadioButtons(
            options=[
                ('Raw Data - Continuous (Means/SDs/N)', 'raw_continuous'),
                ('Raw Data - Binary (Events/Non-Events)', 'raw_binary'),
                ('Pre-calculated (Effect/SE)', 'pre_calculated')
            ],
            value='raw_continuous',
            description='Data Type:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='600px')
        )
        display(_data_type_widget)

        def _on_data_type_change(change):
            _render_column_mapping(df, change['new'])

        _data_type_widget.observe(_on_data_type_change, names='value')

    # Trigger the first render outside the with block to ensure reliable layout rendering
    _render_column_mapping(df, _data_type_widget.value)


def _render_column_mapping(df, data_type):
    with step2_body_output:
        clear_output(wait=True)
        if data_type == 'raw_continuous':
            _render_raw_mapping(df)
        elif data_type == 'raw_binary':
            _render_binary_mapping(df)
        else:
            _render_precalc_mapping(df)

def _render_binary_mapping(df):
    """Column mapping for binary raw data (Events/Non-Events) — includes geographic section."""
    FIELD_INFO = {
        'id':          {'label': 'Study ID / Label:', 'desc': 'Unique name for the study or paper (e.g., "Smith 2020").'},
        'events_e':    {'label': 'Treatment Events (a):', 'desc': 'Number of events (successes/cases) in the Treatment group.'},
        'nonevents_e': {'label': 'Treatment Non-Events (b):', 'desc': 'Number of non-events (failures/controls) in the Treatment group.'},
        'events_c':    {'label': 'Control Events (c):', 'desc': 'Number of events (successes/cases) in the Control group.'},
        'nonevents_c': {'label': 'Control Non-Events (d):', 'desc': 'Number of non-events (failures/controls) in the Control group.'}
    }
    cols_lower = [str(c).lower().strip() for c in df.columns]
    mapping_widgets = {}
    ui_rows = [widgets.HTML("<hr><h4 style='color:#2E86AB; margin-top:15px;'>Map Your Columns (Binary Data Mode)</h4>")]

    for std_name, synonyms in BINARY_COLUMN_SPECS.items():
        guessed_val = None
        for syn in synonyms:
            if syn in cols_lower:
                guessed_val = df.columns[cols_lower.index(syn)]
                break
        w = widgets.Dropdown(
            options=list(df.columns),
            value=guessed_val,
            description=FIELD_INFO[std_name]['label'],
            style={'description_width': '220px'},
            layout=widgets.Layout(width='600px')
        )
        mapping_widgets[std_name] = w
        ui_rows.append(widgets.VBox([
            w,
            widgets.HTML(f"<div style='margin-left:225px; font-size:11px; color:#666; margin-bottom:8px'>"
                         f"<i>{FIELD_INFO[std_name]['desc']}</i></div>")
        ]))

    # --- Geographic section (optional) ---
    geo_vbox, geo_widgets = _build_geo_mapping_widgets(df)
    ui_rows.append(geo_vbox)

    # Finalize button
    btn_finalize = widgets.Button(
        description="✓ Confirm Mapping & Finalize Data",
        button_style='success',
        layout=widgets.Layout(width='600px', height='40px', margin='20px 0 0 0'),
        icon='check-circle'
    )
    finalize_output = widgets.Output()

    def on_finalize_clicked(b):
        with finalize_output:
            clear_output(wait=True)
            try:
                col_map = {k: w.value for k, w in mapping_widgets.items()}
                if None in col_map.values():
                    missing = [k for k, v in col_map.items() if v is None]
                    raise ValueError(f"Please select a column for: {', '.join(missing)}")
                if len(set(col_map.values())) != len(col_map.values()):
                    raise ValueError("Duplicate mapping detected. You cannot map one column to two fields.")

                global raw_data_standardized, DATA_TYPE_SELECTED, GEO_COLUMNS_MAPPED
                mapped_cols = list(col_map.values())

                geo_col_originals = [w.value for w in geo_widgets.values() if w.value != 'None']
                all_reserved = set(mapped_cols + geo_col_originals)
                extra_cols = [c for c in df.columns if c not in all_reserved]

                raw_data_standardized = df[mapped_cols + geo_col_originals + extra_cols].copy()
                raw_data_standardized.rename(columns={v: k for k, v in col_map.items()}, inplace=True)

                raw_data_standardized, num_warn_html = _validate_and_coerce_mapped_numerics(
                    raw_data_standardized, col_map, 'raw_binary')

                geo_map, geo_warn_html, geo_summ_html, geo_type = _finalize_geo_columns(
                    raw_data_standardized, geo_widgets, mapped_cols
                )
                GEO_COLUMNS_MAPPED = geo_map

                # CRITICAL: We set this to 'raw' so downstream cells treat it as raw data that needs processing
                DATA_TYPE_SELECTED = 'raw'

                summary_html = _build_moderator_summary_html(
                    raw_data_standardized, mapped_cols, extra_cols, "Raw Binary Mode",
                    extra_info=num_warn_html,
                    geo_warnings_html=geo_warn_html,
                    geo_summary_html=geo_summ_html,
                    geo_type=geo_type
                )
                display(HTML(summary_html))
                mark_stale(ALL_DOWNSTREAM, "Step 2: Binary Data Mapping Changed")
            except Exception as e:
                err_msg = str(e)
                if err_msg.strip().startswith('<div'):
                    display(HTML(err_msg))
                else:
                    _show_finalize_error(e)

    btn_finalize.on_click(on_finalize_clicked)
    ui_rows.append(btn_finalize)
    ui_rows.append(finalize_output)
    display(widgets.VBox(ui_rows, layout=widgets.Layout(padding='10px')))

def _render_raw_mapping(df):
    """Column mapping for raw data mode — now includes geographic section."""
    FIELD_INFO = {
        'id':  {'label': 'Study ID / Label:', 'desc': 'Unique name for the study or paper (e.g., "Smith 2020").'},
        'xe':  {'label': 'Experimental Mean (xe):', 'desc': 'Mean outcome for the Treatment group.'},
        'sde': {'label': 'Experimental SD (sde):', 'desc': 'Standard Deviation for the Treatment group.'},
        'ne':  {'label': 'Experimental N (ne):', 'desc': 'Sample size for the Treatment group.'},
        'xc':  {'label': 'Control Mean (xc):', 'desc': 'Mean outcome for the Control group.'},
        'sdc': {'label': 'Control SD (sdc):', 'desc': 'Standard Deviation for the Control group.'},
        'nc':  {'label': 'Control N (nc):', 'desc': 'Sample size for the Control group.'}
    }
    cols_lower = [str(c).lower().strip() for c in df.columns]
    mapping_widgets = {}
    ui_rows = [widgets.HTML("<hr><h4 style='color:#2E86AB; margin-top:15px;'>Map Your Columns (Raw Data Mode)</h4>")]

    for std_name, synonyms in RAW_COLUMN_SPECS.items():
        guessed_val = None
        for syn in synonyms:
            if syn in cols_lower:
                guessed_val = df.columns[cols_lower.index(syn)]
                break
        w = widgets.Dropdown(
            options=list(df.columns),
            value=guessed_val,
            description=FIELD_INFO[std_name]['label'],
            style={'description_width': '180px'},
            layout=widgets.Layout(width='600px')
        )
        mapping_widgets[std_name] = w
        ui_rows.append(widgets.VBox([
            w,
            widgets.HTML(f"<div style='margin-left:185px; font-size:11px; color:#666; margin-bottom:8px'>"
                         f"<i>{FIELD_INFO[std_name]['desc']}</i></div>")
        ]))

    # --- Geographic section (optional) ---
    geo_vbox, geo_widgets = _build_geo_mapping_widgets(df)
    ui_rows.append(geo_vbox)

    # Finalize button
    btn_finalize = widgets.Button(
        description="✓ Confirm Mapping & Finalize Data",
        button_style='success',
        layout=widgets.Layout(width='600px', height='40px', margin='20px 0 0 0'),
        icon='check-circle'
    )
    finalize_output = widgets.Output()

    def on_finalize_clicked(b):
        with finalize_output:
            clear_output(wait=True)
            try:
                col_map = {k: w.value for k, w in mapping_widgets.items()}
                if None in col_map.values():
                    missing = [k for k, v in col_map.items() if v is None]
                    raise ValueError(f"Please select a column for: {', '.join(missing)}")
                if len(set(col_map.values())) != len(col_map.values()):
                    raise ValueError("Duplicate mapping detected. You cannot map one column to two fields.")

                global raw_data_standardized, DATA_TYPE_SELECTED, GEO_COLUMNS_MAPPED
                mapped_cols = list(col_map.values())

                geo_col_originals = [w.value for w in geo_widgets.values() if w.value != 'None']
                all_reserved = set(mapped_cols + geo_col_originals)
                extra_cols = [c for c in df.columns if c not in all_reserved]

                raw_data_standardized = df[mapped_cols + geo_col_originals + extra_cols].copy()
                raw_data_standardized.rename(columns={v: k for k, v in col_map.items()}, inplace=True)

                raw_data_standardized, num_warn_html = _validate_and_coerce_mapped_numerics(
                    raw_data_standardized, col_map, 'raw_continuous')

                geo_map, geo_warn_html, geo_summ_html, geo_type = _finalize_geo_columns(
                    raw_data_standardized, geo_widgets, mapped_cols
                )
                GEO_COLUMNS_MAPPED = geo_map
                DATA_TYPE_SELECTED = 'raw'

                summary_html = _build_moderator_summary_html(
                    raw_data_standardized, mapped_cols, extra_cols, "Raw Mode",
                    extra_info=num_warn_html,
                    geo_warnings_html=geo_warn_html,
                    geo_summary_html=geo_summ_html,
                    geo_type=geo_type
                )
                display(HTML(summary_html))
                mark_stale(ALL_DOWNSTREAM, "Step 2: Raw Data Mapping Changed")
            except Exception as e:
                err_msg = str(e)
                if err_msg.strip().startswith('<div'):
                    display(HTML(err_msg))
                else:
                    _show_finalize_error(e)

    btn_finalize.on_click(on_finalize_clicked)
    ui_rows.append(btn_finalize)
    ui_rows.append(finalize_output)
    display(widgets.VBox(ui_rows, layout=widgets.Layout(padding='10px')))


def _render_precalc_mapping(df):
    """Column mapping for pre-calculated effect sizes — now includes geographic section."""
    FIELD_INFO = {
        'id':       {'label': 'Study ID / Label:', 'desc': 'Unique identifier for each study.', 'required': True},
        'yi':       {'label': 'Effect Size (yi):', 'desc': "The calculated effect size (e.g., Hedges' g, lnRR, etc.).", 'required': True},
        'variance': {'label': 'Variance (vi):', 'desc': 'The variance of the effect size.', 'required': False},
        'se':       {'label': 'Standard Error (SE):', 'desc': 'The standard error (will convert to variance if needed).', 'required': False},
        'n_total':  {'label': 'Sample Size (n_total):', 'desc': 'Total sample size (optional — useful for diagnostics).', 'required': False}
    }
    cols_lower = [str(c).lower().strip() for c in df.columns]
    mapping_widgets = {}
    ui_rows = [widgets.HTML("""
    <hr>
    <h4 style='color:#2E86AB; margin-top:15px;'>Map Your Columns (Pre-calculated Mode)</h4>
    <div style='background-color:#e3f2fd; padding:10px; border-radius:5px; margin-bottom:10px;'>
        <b>💡 About Pre-calculated Effect Sizes:</b><br>
        You'll need:<br>
        • <b>Effect Size (yi):</b> The standardized effect (g, lnRR, etc.)<br>
        • <b>Variance OR Standard Error:</b> Map at least one<br>
        • <b>Sample Size (optional):</b> Helps with some diagnostics
    </div>
    """)]

    field_order = ['id', 'yi', 'variance', 'se', 'n_total']
    for std_name in field_order:
        synonyms = PRECALC_COLUMN_SPECS[std_name]
        info = FIELD_INFO[std_name]
        guessed_val = None
        for syn in synonyms:
            if syn in cols_lower:
                guessed_val = df.columns[cols_lower.index(syn)]
                break
        options = ['None'] + list(df.columns) if not info['required'] else list(df.columns)
        w = widgets.Dropdown(
            options=options,
            value=guessed_val if guessed_val is not None else ('None' if not info['required'] else None),
            description=info['label'],
            style={'description_width': '180px'},
            layout=widgets.Layout(width='600px')
        )
        mapping_widgets[std_name] = w
        req_text = " <b style='color:#c0392b;'>(Required)</b>" if info['required'] else " <i>(Optional)</i>"
        ui_rows.append(widgets.VBox([
            w,
            widgets.HTML(f"<div style='margin-left:185px; font-size:11px; color:#666; margin-bottom:8px'>"
                         f"<i>{info['desc']}</i>{req_text}</div>")
        ]))

    geo_vbox, geo_widgets = _build_geo_mapping_widgets(df)
    ui_rows.append(geo_vbox)

    btn_finalize = widgets.Button(
        description="✓ Confirm Mapping & Finalize Data",
        button_style='success',
        layout=widgets.Layout(width='600px', height='40px', margin='20px 0 0 0'),
        icon='check-circle'
    )
    finalize_output = widgets.Output()

    def on_finalize_clicked(b):
        with finalize_output:
            clear_output(wait=True)
            try:
                col_map = {k: w.value for k, w in mapping_widgets.items()}
                col_map = {k: v for k, v in col_map.items() if v != 'None'}
                if 'id' not in col_map or 'yi' not in col_map:
                    raise ValueError("Please map required fields: Study ID and Effect Size (yi)")
                if 'variance' not in col_map and 'se' not in col_map:
                    raise ValueError("Please map either Variance (vi) OR Standard Error (SE)")
                mapped_values = list(col_map.values())
                if len(set(mapped_values)) != len(mapped_values):
                    raise ValueError("Duplicate mapping detected. You cannot map one column to two fields.")

                global raw_data_standardized, DATA_TYPE_SELECTED, VARIANCE_TYPE_SELECTED, GEO_COLUMNS_MAPPED
                mapped_cols = list(col_map.values())

                geo_col_originals = [w.value for w in geo_widgets.values() if w.value != 'None']
                all_reserved = set(mapped_cols + geo_col_originals)
                extra_cols = [c for c in df.columns if c not in all_reserved]

                raw_data_standardized = df[mapped_cols + geo_col_originals + extra_cols].copy()
                raw_data_standardized.rename(columns={v: k for k, v in col_map.items()}, inplace=True)

                raw_data_standardized, num_warn_html = _validate_and_coerce_mapped_numerics(
                    raw_data_standardized, col_map, 'pre_calculated')

                DATA_TYPE_SELECTED = 'pre_calculated'
                if 'variance' in col_map and 'se' in col_map:
                    VARIANCE_TYPE_SELECTED = 'both'
                elif 'variance' in col_map:
                    VARIANCE_TYPE_SELECTED = 'variance'
                else:
                    VARIANCE_TYPE_SELECTED = 'se'

                geo_map, geo_warn_html, geo_summ_html, geo_type = _finalize_geo_columns(
                    raw_data_standardized, geo_widgets, mapped_cols
                )
                GEO_COLUMNS_MAPPED = geo_map

                extra_info = (
                    f"<br>Effect Size Column: <b>{col_map['yi']}</b> &nbsp;·&nbsp; "
                    f"Variance Type: <b>{VARIANCE_TYPE_SELECTED.upper()}</b>"
                )
                if num_warn_html:
                    extra_info += num_warn_html
                summary_html = _build_moderator_summary_html(
                    raw_data_standardized, mapped_cols, extra_cols, "Pre-calculated Mode",
                    extra_info,
                    geo_warnings_html=geo_warn_html,
                    geo_summary_html=geo_summ_html,
                    geo_type=geo_type
                )
                display(HTML(summary_html))
                mark_stale(ALL_DOWNSTREAM, "Step 2: Pre-calculated Data Mapping Changed")
            except Exception as e:
                err_msg = str(e)
                if err_msg.strip().startswith('<div'):
                    display(HTML(err_msg))
                else:
                    _show_finalize_error(e)

    btn_finalize.on_click(on_finalize_clicked)
    ui_rows.append(btn_finalize)
    ui_rows.append(finalize_output)
    display(widgets.VBox(ui_rows, layout=widgets.Layout(padding='10px')))

# =============================================================================
# REPRODUCIBILITY WIDGETS (TAB 3)
# =============================================================================
_repro_upload = widgets.FileUpload(
    accept='.json',
    multiple=False,
    description='Load Settings JSON',
    layout=widgets.Layout(width='300px')
)
_repro_output = widgets.Output()

def _on_repro_upload(change):
    with _repro_output:
        clear_output()
        uploaded = change['new']
        if not uploaded:
            return

        file_info = uploaded[0] if isinstance(uploaded, (list, tuple)) else list(uploaded.values())[0]
        content = file_info['content'] if isinstance(file_info, dict) else file_info

        # Ensure safely decoded if newer ipywidgets version returns memoryview
        if hasattr(content, 'tobytes'):
            content = content.tobytes()

        try:
            global ANALYSIS_CONFIG
            ANALYSIS_CONFIG = load_reproducibility_config(content)

            meta = ANALYSIS_CONFIG.get('_reproducibility', {})
            df = ANALYSIS_CONFIG.get('clean_dataframe')

            # --- 1. Data Provenance ---
            exp_date = meta.get('exported_at', 'Unknown')
            if 'T' in exp_date:
                exp_date = exp_date.split('.')[0].replace('T', ' ')
            data_shape = f"<b>{len(df)}</b> rows, <b>{len(df.columns)}</b> columns" if df is not None else "Unknown"
            d_type = ANALYSIS_CONFIG.get('data_type', 'raw').upper()

            # --- 2. Pre-processing Details ---
            # Global filters: current multi-variable list, or the older single-filter keys
            specs = [s for s in (ANALYSIS_CONFIG.get('prefilters') or []) if s.get('col', 'None') != 'None']
            if not specs and ANALYSIS_CONFIG.get('prefilter_col', 'None') not in (None, 'None'):
                specs = [{'col': ANALYSIS_CONFIG['prefilter_col'], 'values': ANALYSIS_CONFIG.get('prefilter_values', [])}]
            if specs:
                filter_html = '; '.join(f"<code>{html.escape(str(s['col']))}</code> (kept {len(s.get('values') or [])} value(s))"
                                        for s in specs)
            else:
                filter_html = "<i style='color:#6c757d;'>None applied</i>"
            cc_cols = ANALYSIS_CONFIG.get('complete_case_moderators') or []
            cc_html = (f"Complete cases on {', '.join(f'<code>{html.escape(str(c))}</code>' for c in cc_cols)}"
                       if cc_cols else "<i style='color:#6c757d;'>Not used</i>")

            SD_LABELS = {'median_cv': 'Median CV (impute missing only)',
                         'nakagawa_missing_cases': 'Pooled CV, missing cases only',
                         'nakagawa_all_cases': 'Pooled CV, all cases',
                         'treat_as_missing': 'Treat zeros as missing (CV impute)', 'drop': 'Drop rows'}
            sd_miss = ANALYSIS_CONFIG.get('sd_missing_strategy')
            sd_zero = ANALYSIS_CONFIG.get('sd_zero_strategy')
            if d_type != 'RAW':
                sd_html = "<i style='color:#6c757d;'>N/A (Pre-calculated)</i>"
            elif sd_miss is None and sd_zero is None:
                sd_html = "<i style='color:#6c757d;'>Not needed (no missing or zero SDs)</i>"
            else:
                sd_html = (f"Missing: <b>{SD_LABELS.get(sd_miss, sd_miss)}</b> &nbsp;|&nbsp; "
                           f"Zero: <b>{SD_LABELS.get(sd_zero, sd_zero)}</b>")

            # --- 3. Modeling Details ---
            es_type = ANALYSIS_CONFIG.get('effect_size_type', 'Not selected')
            if es_type in ('lnRR', 'lnCVR') and d_type == 'RAW':
                es_type += (" (small-sample bias correction "
                            + ("on" if ANALYSIS_CONFIG.get('bias_correction', True) else "off") + ")")

            gs = ANALYSIS_CONFIG.get('global_settings', {})
            if gs:
                gs_html = f"<b>{gs.get('model_choice', 'Three-level (recommended)')}</b> &nbsp;|&nbsp; "
                gs_html += f"τ²: <b>{gs.get('tau_method', 'REML')}</b> &nbsp;|&nbsp; "
                gs_html += f"<b>{gs.get('dist_type', 't').upper()}</b>-dist &nbsp;|&nbsp; "
                gs_html += f"α=<b>{gs.get('alpha', 0.05)}</b>"
            else:
                gs_html = "<i style='color:#6c757d;'>Default settings</i>"

            sub_config = ANALYSIS_CONFIG.get('subgroup_config', {})
            if sub_config:
                sub_html = f"<b>{sub_config.get('analysis_type', 'Unknown').title().replace('_', ' ')}</b> by "
                sub_html += f"<code>{sub_config.get('moderator1', 'None')}</code>"
                if sub_config.get('moderator2'):
                    sub_html += f" × <code>{sub_config.get('moderator2')}</code>"
                sub_html += f" <span style='color:#6c757d; font-size: 11px;'>(Min k={sub_config.get('min_obs')}, Studies={sub_config.get('min_papers')})</span>"
            else:
                sub_html = "<i style='color:#6c757d;'>None saved</i>"

            # --- Generate HTML Card ---
            display(HTML(f"""
            <div style='font-family: sans-serif; max-width: 850px;'>
                <!-- Success Banner -->
                <div style='padding: 15px; background-color: #d4edda; border-left: 5px solid #28a745; color: #155724; border-radius: 4px; margin-bottom: 15px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);'>
                    <h4 style='margin: 0 0 5px 0; display: flex; align-items: center;'><span style='margin-right: 8px;'>✅</span> Reproducibility Session Loaded Successfully</h4>
                    <p style='margin: 0; font-size: 14px;'>Historical data and settings have been fully hydrated into memory.</p>
                </div>

                <!-- Session Details -->
                <div style='background-color: #ffffff; border: 1px solid #dee2e6; border-radius: 4px; margin-bottom: 15px; overflow: hidden; box-shadow: 0 2px 4px rgba(0,0,0,0.02);'>
                    <h5 style='margin: 0; color: #2c3e50; background-color: #f8f9fa; padding: 12px 15px; border-bottom: 1px solid #dee2e6;'>Session Configuration Overview</h5>

                    <div style='padding: 15px;'>
                        <table style='width: 100%; font-size: 13.5px; color: #333; border-collapse: collapse;'>

                            <!-- Data Section -->
                            <tr><td colspan="2" style='color: #6c757d; font-size: 11px; text-transform: uppercase; font-weight: bold; padding: 10px 0 4px 0; border-bottom: 1px solid #eee;'>1. Data Provenance</td></tr>
                            <tr><td style='padding: 6px 0; width: 160px; color: #495057;'>Export Date:</td><td>{exp_date}</td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Data Hydrated:</td><td>{data_shape}</td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Input Format:</td><td><span style='background: #e9ecef; padding: 2px 6px; border-radius: 3px; font-size: 12px;'>{d_type}</span></td></tr>

                            <!-- Pre-processing Section -->
                            <tr><td colspan="2" style='color: #6c757d; font-size: 11px; text-transform: uppercase; font-weight: bold; padding: 15px 0 4px 0; border-bottom: 1px solid #eee;'>2. Pre-Processing</td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Global Filters:</td><td>{filter_html}</td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Common Row Set:</td><td>{cc_html}</td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Variance Imputation:</td><td>{sd_html}</td></tr>

                            <!-- Modeling Section -->
                            <tr><td colspan="2" style='color: #6c757d; font-size: 11px; text-transform: uppercase; font-weight: bold; padding: 15px 0 4px 0; border-bottom: 1px solid #eee;'>3. Modeling Settings</td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Effect Size Metric:</td><td><span style='color: #0056b3; font-weight: bold;'>{es_type}</span></td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Overall Analysis:</td><td>{gs_html}</td></tr>
                            <tr><td style='padding: 6px 0; color: #495057;'>Subgroup Config:</td><td>{sub_html}</td></tr>

                        </table>
                    </div>
                </div>

                <!-- Next Steps -->
                <div style='padding: 15px; background-color: #e8f4f8; border-left: 5px solid #17a2b8; color: #0c5460; border-radius: 4px;'>
                    <h4 style='margin: 0 0 5px 0; display: flex; align-items: center;'><span style='margin-right: 8px;'>🚀</span> Next Step</h4>
                    <p style='margin: 0; font-size: 14px;'>Do not click any other buttons in this step. <b>Simply run the subsequent cells in order</b> (Step 3, Step 4, etc.). Steps 3–9 restore the settings above automatically; the choices made in Step 12 onward are not stored in the session file, so select them again.</p>
                </div>
            </div>
            """))


        except Exception as e:
            print(f"ERROR loading config: {e}")
            import traceback
            traceback.print_exc()

_repro_upload.observe(_on_repro_upload, names='value')

# --- Tab 3: Load Settings JSON (Reproducibility) ---
repro_tab_content = widgets.VBox([
    widgets.HTML("""
    <div style='background:#e8f4f8; border-left:5px solid #17a2b8; padding:15px;
         border-radius:5px; margin:10px 0;'>
      <h4 style='margin-top:0; color:#0c5460;'>Restore Previous Session</h4>
      <p style='color:#0c5460; margin-bottom:5px;'>
        Upload a previously exported <code>analysis_settings.json</code> file.
        Once loaded, <b>run the remaining notebook cells sequentially</b>.
        The pipeline will automatically reconstruct the data and select the exact settings used previously.
      </p>
    </div>
    """),
    _repro_upload,
    _repro_output
], layout=widgets.Layout(padding='10px'))

# =============================================================================
# INITIALIZE UI
# =============================================================================
btn_auth.on_click(on_auth_clicked)
btn_fetch_ws.on_click(on_fetch_ws_clicked)
btn_load_sheet.on_click(on_load_sheet_clicked)
uploader.observe(on_file_upload, names='value')
btn_process_file.on_click(on_process_file_clicked)

# --- Tab 3: Load Settings JSON (Reproducibility) ---
repro_tab_content = widgets.VBox([
    widgets.HTML("""
    <div style='background:#e8f4f8; border-left:5px solid #17a2b8; padding:15px;
         border-radius:5px; margin:10px 0;'>
      <h4 style='margin-top:0; color:#0c5460;'>Reproducibility Mode</h4>
      <p style='color:#0c5460; margin-bottom:5px;'>
        Upload a previously exported <code>analysis_settings.json</code> file to
        instantly restore all data and perfectly replicate a previous session.
      </p>
    </div>
    """),
    widgets.HBox([_repro_upload]),
    _repro_output
], layout=widgets.Layout(padding='10px'))

# --- Tab 4: Built-in Examples ---
dd_example_data = widgets.Dropdown(
    options=list(BUILT_IN_DATASETS.keys()),
    description='Select Data:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

btn_load_example = widgets.Button(
    description="📥 Load Example Dataset",
    button_style='info',
    icon='download',
    layout=widgets.Layout(width='300px', height='40px')
)

example_tab_content = widgets.VBox([
    widgets.HTML("""
    <div style='background:#e8f4f8; border-left:5px solid #17a2b8; padding:15px; border-radius:5px; margin:10px 0;'>
      <h4 style='margin-top:0; color:#0c5460;'>Built-in Example Datasets</h4>
      <p style='color:#0c5460; margin-bottom:5px;'>
        Load classic meta-analysis datasets instantly to test the pipeline. These are pre-mapped
        and require no manual configuration.
      </p>
    </div>
    """),
    dd_example_data,
    btn_load_example
], layout=widgets.Layout(padding='10px'))

tabs = widgets.Tab(children=[gs_vbox, local_vbox, repro_tab_content, example_tab_content])
tabs.set_title(0, "Google Sheets")
tabs.set_title(1, "Upload Excel/CSV")
tabs.set_title(2, "Restore Session")
tabs.set_title(3, "Built-in Examples")

# --- Example Data Logic ---
def on_load_example_clicked(b):
    with step2_header_output:
        clear_output()
        selection = dd_example_data.value
        config = BUILT_IN_DATASETS[selection]

        # 1. Build DataFrame from hardcoded dictionary
        df_example = pd.DataFrame(config['data'])
        data_type_raw = config['type']  # e.g., 'raw_continuous' or 'raw_binary'

        # 2. Sanitize the data type for the backend pipeline!
        # If it starts with 'raw', the backend just needs to see 'raw'.
        pipeline_data_type = 'raw' if data_type_raw.startswith('raw') else 'pre_calculated'

        # 3. Inject directly into global standardized variables
        global raw_data_standardized, DATA_TYPE_SELECTED, VARIANCE_TYPE_SELECTED
        raw_data_standardized = df_example.copy()
        DATA_TYPE_SELECTED = pipeline_data_type

        if pipeline_data_type == 'pre_calculated':
            VARIANCE_TYPE_SELECTED = 'variance'

        # 4. Clear previous ANALYSIS_CONFIG to prevent stale data overlap
        global ANALYSIS_CONFIG
        ANALYSIS_CONFIG = {
            'data_type': pipeline_data_type,
            'clean_dataframe': df_example.copy(),
            'prefilter_col': 'None',
            'prefilter_values': []
        }

        # 5. Create a success message bypassing the manual mapper
        display(HTML(f"""
        <div style='background-color:#d4edda; color:#155724; padding:15px; border-radius:8px; border:1px solid #c3e6cb; margin-bottom:15px;'>
            <h4 style='margin:0 0 8px 0;'>✅ Example Data Loaded Successfully</h4>
            <b>{len(df_example)}</b> rows loaded &nbsp;·&nbsp;
            <b>Mode:</b> {data_type_raw.upper().replace('_', ' ')}<br><br>
            <i>The data columns are already pre-mapped to the backend. You can skip the mapping step below and proceed directly to <b>Step 3 (Global Filtering)</b> or <b>Step 4 (Data Cleaning)</b>!</i>
        </div>
        """))

        # Clear the body output so they don't see the manual mapping UI
        with step2_body_output:
            clear_output()

btn_load_example.on_click(on_load_example_clicked)


display(HTML("<h3 style='color:#2E86AB; border-bottom: 2px solid #3498db; padding-bottom: 5px;'>Part 1: Import Data Source</h3>"))
display(tabs)
display(log_output)
display(step2_header_output)
display(step2_body_output)

In [ ]:
#@title ⚙️ 3. Global Filtering
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")


# =============================================================================
# STEP 3: GLOBAL FILTERING & COMMON ROW SET
# Purpose: Convert data types and apply GLOBAL filters (Pre-filters).
# Dependencies: Step 2 (global 'raw_data_standardized')
# Output: Global 'ANALYSIS_CONFIG' dictionary
# Note: SKIP this cell if using Pre-calculated mode
#
# Drop-in compatibility with the single-filter version:
#   - 'prefilter_col' / 'prefilter_values' still hold ONE filter, so Cell 4
#     keeps working unchanged (re-applying it there is harmless).
#   - NEW key 'prefilters' = [{'col': ..., 'values': [...]}, ...] holds ALL
#     filters (JSON-safe, used for reproducibility).
#   - ALL filters are applied here to 'clean_dataframe'.
#   - Blank cells get a "(missing / blank)" checkbox (checked by default).
#   - Logic: values within a variable are OR-ed, variables are AND-ed.
#   - Old saved configs (only 'prefilter_col'/'prefilter_values') still restore.
#
# Robustness:
#   - The widget's data and helpers are PRIVATE (inside a function). Later
#     cells can reuse names like 'df_config' or edit dataframes freely without
#     breaking this widget. Globals such as 'df_config' are still published,
#     but only as separate copies for other cells to use.
#   - Each run takes a fresh private copy of 'raw_data_standardized', so
#     changing filters never loses data and new data from Cell 2 is always used.
#   - Re-running the cell restores your last saved filter selections; values
#     that are new in the data (not seen when you saved) start CHECKED.
# =============================================================================

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd
import numpy as np

# --- CHECK IF PRE-CALCULATED MODE ---
data_type_mode = globals().get('DATA_TYPE_SELECTED', 'raw')

if data_type_mode == 'pre_calculated':
    display(HTML("""
    <div style='background-color: #e2e3e5; border-left: 4px solid #6c757d; padding: 15px; border-radius: 4px; font-family: sans-serif; margin-bottom: 15px;'>
        <h4 style='color: #383d41; margin-top: 0; display: flex; align-items: center;'>
            <span style='margin-right: 8px;'>⏭️</span> Skip This Step (Pre-calculated Mode)
        </h4>
        <p style='color: #383d41; font-size: 14px; margin-bottom: 5px;'>
            Global pre-filtering is bypassed because you imported pre-calculated effect sizes.
        </p>
        <p style='color: #383d41; font-size: 14px; margin-bottom: 0; font-weight: bold;'>
            Please proceed directly to Step 4 (Data Cleaning).
        </p>
    </div>
    """))

    # Initialize minimal ANALYSIS_CONFIG for pre-calculated mode
    if 'ANALYSIS_CONFIG' not in globals():
        ANALYSIS_CONFIG = {
            'prefilter_col': 'None',
            'prefilter_values': [],
            'prefilters': [],
            'factor1': 'None',
            'factor2': 'None',
            'min_papers': 1,
            'min_obs': 1,
            'clean_dataframe': raw_data_standardized.copy() if 'raw_data_standardized' in globals() else None
        }

    # Common row set is available in pre-calculated mode too: the choice is
    # stored here and applied in Step 6 (see the raw-mode widget below).
    if 'ANALYSIS_CONFIG' in globals() and 'raw_data_standardized' in globals():
        _reserved = {'id', 'yi', 'variance', 'vi', 'se', 'n_total', 'latitude', 'longitude'}
        _cc_opts = [c for c in raw_data_standardized.columns if str(c) not in _reserved]
        _cc_widget = widgets.SelectMultiple(
            options=_cc_opts, rows=6, layout=widgets.Layout(width='400px'),
            value=tuple(c for c in (ANALYSIS_CONFIG.get('complete_case_moderators') or []) if c in _cc_opts))
        def _on_cc_change(change):
            ANALYSIS_CONFIG['complete_case_moderators'] = [str(c) for c in change['new']]
        _cc_widget.observe(_on_cc_change, 'value')
        display(widgets.VBox([widgets.HTML("""
                <div style='margin-bottom: 10px; font-family: sans-serif; font-size: 13px; color: #34495e; line-height: 1.5;'>
                    <h4 style='color: #2c3e50; margin: 0 0 8px 0;'>Common Row Set <span style='font-size: 13px; font-weight: normal; color: #7f8c8d;'>(Optional)</span></h4>
                    <p style='margin: 0 0 6px 0;'><b>What it does.</b> Keeps only the records that have a value for <b>every</b>
                    moderator you select here, and runs <b>every</b> analysis (overall model, subgroups, meta-regression,
                    splines, publication bias, sensitivity analyses) on that same set of records.</p>
                    <p style='margin: 0 0 6px 0;'><b>Why it matters.</b> Without it, each analysis drops only the records it
                    cannot use. A meta-regression on a moderator with missing values then runs on fewer records than the
                    overall model, so comparisons between the two (pseudo-R², AIC, how much τ² drops) mix two different
                    datasets and can be misleading.</p>
                    <p style='margin: 0 0 6px 0;'><b>When to use it.</b> When you will compare models or report how much
                    heterogeneity a moderator explains, and some of those moderators have missing values (listed below).
                    If they have none, this setting changes nothing: leave it empty.</p>
                    <p style='margin: 0;'><b>Cost.</b> Fewer records in every analysis. Hold Ctrl/Cmd to select several moderators.</p>
                </div>
        """), _cc_widget]))

else:
    # RAW DATA MODE: Full configuration
    # Make sure the config dict exists BEFORE anything reads it
    if 'ANALYSIS_CONFIG' not in globals():
        ANALYSIS_CONFIG = {}

    # --- 1. INITIALIZATION & SAFETY CHECK ---
    if 'raw_data_standardized' not in globals():
        display(HTML("<div style='background-color:#f8d7da; color:#721c24; padding:10px; border-radius:5px;'>❌ <b>Error:</b> Data not found. Please run Step 2 (Data Ingestion) first.</div>"))
    else:

        def _cell3_global_filtering(G):
            """Builds the whole Cell 3 UI. Everything in here is private to the
            widget, so other cells cannot accidentally change it."""
            import html
            import ipywidgets as widgets
            import pandas as pd
            from IPython.display import display, HTML, clear_output

            STAT_COLS = ['xe', 'sde', 'ne', 'xc', 'sdc', 'nc', 'events_e', 'nonevents_e', 'events_c', 'nonevents_c']
            MISSING_TOKEN = '(missing / blank)'

            def _config():
                cfg = G.get('ANALYSIS_CONFIG')
                if not isinstance(cfg, dict):
                    cfg = {}
                    G['ANALYSIS_CONFIG'] = cfg
                return cfg

            # =============================================================
            # 2. PRIVATE COPY OF THE DATA (the "internal backup")
            # =============================================================
            # Every run takes a FRESH copy of 'raw_data_standardized' from Cell 2.
            # The widget only ever reads this private copy, so changing filters
            # never loses data, and re-running Cell 2 + Cell 3 always picks up
            # the new data. (A long-lived backup from older versions is removed.)
            G.pop('_CELL3_BACKUP', None)
            OUTPUT_TAG = 'cell3_filtered_output'
            current_raw = G['raw_data_standardized']
            looks_filtered = bool(getattr(current_raw, 'attrs', {}).get(OUTPUT_TAG, False))

            st = {}

            def _load_base():
                df = current_raw.copy(deep=True)
                df.attrs.pop(OUTPUT_TAG, None)
                for c in [c for c in STAT_COLS if c in df.columns]:
                    # Coerce errors to NaN (e.g., if someone wrote "n/a" in a number field)
                    df[c] = pd.to_numeric(df[c], errors='coerce')
                st['df'] = df
                st['numeric_cols'] = [c for c in STAT_COLS if c in df.columns]
                st['moderators'] = [c for c in df.columns if c not in st['numeric_cols'] + ['id']] or ['None']
                _publish_globals()

            def _publish_globals():
                # Legacy globals for other cells: separate copies, never used by the widget.
                G['df_config'] = st['df'].copy()
                G['numeric_cols'] = list(st['numeric_cols'])
                G['reserved_cols'] = st['numeric_cols'] + ['id']
                G['available_moderators'] = list(st['moderators'])

            # =============================================================
            # 4. FILTER HELPERS
            # =============================================================
            def _missing_mask(s):
                return s.isna() | s.astype(str).str.strip().eq('')

            def _row_mask(col, kept_values):
                s = st['df'][col]
                kept = [str(v) for v in kept_values]
                miss = _missing_mask(s)
                mask = ~miss & s.astype(str).isin(kept)
                if MISSING_TOKEN in kept:
                    mask |= miss
                return mask

            def _combined_mask(specs):
                mask = pd.Series(True, index=st['df'].index)
                for spec in specs:
                    mask &= _row_mask(spec['col'], spec['values'])
                return mask

            def _complete_case_mask(cols):
                """Rows with a value in EVERY selected moderator (common row set)."""
                mask = pd.Series(True, index=st['df'].index)
                for c in cols:
                    mask &= ~_missing_mask(st['df'][c])
                return mask

            # =============================================================
            # 5. WIDGETS
            # =============================================================
            filter_rows = []
            rows_container = widgets.VBox([])
            lbl_preview = widgets.HTML()
            lbl_cc = widgets.HTML()
            cc_select = widgets.SelectMultiple(
                options=[], rows=6, description='',
                layout=widgets.Layout(width='400px'))
            output_config = widgets.Output()

            def _current_specs():
                return [{'col': r.dd.value, 'values': r.kept_values(), 'seen': list(r.checks)}
                        for r in filter_rows if r.dd.value != 'None']

            def refresh_preview(*_):
                specs = _current_specs()
                cols = [s['col'] for s in specs]
                dups = sorted({c for c in cols if cols.count(c) > 1})
                n_total = len(st['df'])
                cc_cols = list(cc_select.value)
                mask = _combined_mask(specs) if specs else pd.Series(True, index=st['df'].index)
                n_kept = int(mask.sum())
                n_cc = int((mask & _complete_case_mask(cc_cols)).sum()) if cc_cols else n_kept
                colour = '#155724' if n_cc > 0 else '#721c24'
                warn = ''
                if dups:
                    warn = (f"<br>⚠️ Variable selected more than once: "
                            f"<b>{html.escape(', '.join(map(str, dups)))}</b>. Use each variable only once.")
                lbl_preview.value = (
                    f"<div style='font-family: sans-serif; font-size: 13px; color: {colour}; margin-top: 6px;'>"
                    f"Preview: <b>{n_kept}</b> of {n_total} records retained "
                    f"({len(specs)} active filter(s))"
                    + (f"; <b>{n_cc}</b> after restricting to complete cases on "
                       f"{html.escape(', '.join(map(str, cc_cols)))}" if cc_cols else "")
                    + f".{warn}</div>"
                )
                # Common-row-set tab: which moderators have missing values, and
                # how many records the restriction keeps.
                kept_df = st['df'][mask]
                n_miss = {c: int(_missing_mask(kept_df[c]).sum()) for c in st['moderators'] if c != 'None'}
                with_missing = {c: n for c, n in n_miss.items() if n > 0}
                if with_missing:
                    miss_txt = ('Moderators with missing values among the retained records: '
                                + ', '.join(f"<b>{html.escape(str(c))}</b> ({n})" for c, n in with_missing.items()) + '.')
                else:
                    miss_txt = 'No moderator has missing values among the retained records, so a common row set is not needed.'
                use_txt = (f"<br>With the selection: <b>{n_cc}</b> of {n_kept} records are used in every analysis "
                           f"({n_kept - n_cc} dropped)." if cc_cols else "<br>Nothing selected: each analysis uses every record it can.")
                lbl_cc.value = (f"<div style='font-family: sans-serif; font-size: 13px; color: {colour}; margin-top: 8px;'>"
                                f"{miss_txt}{use_txt}</div>")

            class FilterRow:
                """One filter variable + its value checkboxes."""

                def __init__(self):
                    self.dd = widgets.Dropdown(
                        options=['None'] + st['moderators'],
                        value='None',
                        description='Filter Variable:',
                        style={'description_width': '120px'},
                        layout=widgets.Layout(width='400px')
                    )
                    self.btn_remove = widgets.Button(description='Remove', icon='trash',
                                                     layout=widgets.Layout(width='100px'))
                    self.btn_all = widgets.Button(description='Select all', layout=widgets.Layout(width='100px'))
                    self.btn_none = widgets.Button(description='Select none', layout=widgets.Layout(width='100px'))
                    self.checks = {}               # str(value) -> Checkbox
                    self.vbox_values = widgets.VBox([])
                    self.box = widgets.VBox(
                        [widgets.HBox([self.dd, self.btn_remove]), self.vbox_values],
                        layout=widgets.Layout(border='1px dashed #d0d7de', padding='8px', margin='0 0 8px 0')
                    )
                    self.dd.observe(self._on_col_change, names='value')
                    self.btn_remove.on_click(lambda b: remove_filter_row(self))
                    self.btn_all.on_click(lambda b: self.set_all(True))
                    self.btn_none.on_click(lambda b: self.set_all(False))

                def _on_col_change(self, change):
                    self.populate(change['new'])
                    refresh_preview()

                def populate(self, col):
                    self.checks = {}
                    if col == 'None' or col not in st['df'].columns:
                        self.vbox_values.children = []
                        return
                    full = st['df'][col]
                    miss = _missing_mask(full)
                    s = full[~miss]
                    counts = s.astype(str).value_counts()

                    def _add(key, n):
                        cb = widgets.Checkbox(value=True, description=f"{key} (n={n})", indent=False,
                                              layout=widgets.Layout(width='auto'))
                        cb.observe(refresh_preview, names='value')
                        self.checks[key] = cb

                    for v in pd.unique(s):
                        key = str(v)
                        if key not in self.checks:     # e.g. 1 and '1' in a mixed column
                            _add(key, int(counts.get(key, 0)))
                    if int(miss.sum()):
                        _add(MISSING_TOKEN, int(miss.sum()))
                    header = widgets.HTML(f"<i>Select which values of <b>{html.escape(str(col))}</b> to KEEP in the analysis:</i>")
                    values_list = widgets.VBox(list(self.checks.values()),
                                               layout=widgets.Layout(max_height='260px', overflow='auto'))
                    self.vbox_values.children = [header, widgets.HBox([self.btn_all, self.btn_none]), values_list]

                def set_all(self, flag):
                    for cb in self.checks.values():
                        cb.value = flag

                def set_values(self, values):
                    wanted = {str(v) for v in values}
                    for key, cb in self.checks.items():
                        cb.value = key in wanted

                def kept_values(self):
                    return [key for key, cb in self.checks.items() if cb.value]

            def _redraw_rows():
                rows_container.children = [r.box for r in filter_rows]
                refresh_preview()

            def add_filter_row(*_):
                row = FilterRow()
                filter_rows.append(row)
                _redraw_rows()
                return row

            def remove_filter_row(row):
                if len(filter_rows) <= 1:
                    row.dd.value = 'None'          # keep at least one (empty) row
                else:
                    filter_rows.remove(row)
                _redraw_rows()

            def clear_all_filters(*_):
                filter_rows.clear()
                add_filter_row()

            def load_specs(specs, strict=False):
                """Rebuild the filter rows from a list of {'col', 'values'}. Returns missing columns.
                strict=False: values that did not exist when the filter was saved start checked."""
                missing = []
                filter_rows.clear()
                for spec in specs:
                    if spec['col'] not in st['moderators']:
                        missing.append(spec['col'])
                        continue
                    row = FilterRow()
                    filter_rows.append(row)
                    row.dd.value = spec['col']                 # generates the checkboxes
                    keep = set(map(str, spec.get('values', [])))
                    seen = spec.get('seen')
                    if not strict and seen is not None:
                        keep |= set(row.checks) - set(map(str, seen))   # new categories -> checked
                    row.set_values(keep)
                if not filter_rows:
                    filter_rows.append(FilterRow())
                _redraw_rows()
                return missing

            btn_add_filter = widgets.Button(description='Add filter variable', icon='plus',
                                            layout=widgets.Layout(width='200px'))
            btn_add_filter.on_click(add_filter_row)
            btn_clear = widgets.Button(description='Clear all filters', icon='eraser',
                                       layout=widgets.Layout(width='200px'))
            btn_clear.on_click(clear_all_filters)

            # =============================================================
            # 6. SAVE BUTTON
            # =============================================================
            btn_save_config = widgets.Button(
                description="▶ Save Configuration",
                button_style='success',
                layout=widgets.Layout(width='400px', height='50px'),
                icon='check'
            )

            def on_save_config_clicked(b, quiet=False):
                with output_config:
                    clear_output()
                    try:
                        # 1. Capture all filters
                        prefilters = _current_specs()
                        cols = [s['col'] for s in prefilters]
                        dups = sorted({c for c in cols if cols.count(c) > 1})
                        if dups:
                            raise ValueError(f"Variable(s) used in more than one filter: {', '.join(map(str, dups))}. Use each variable only once.")

                        # Legacy single-filter keys (read by Cell 4) = first filter that does
                        # NOT keep blank rows. Cell 4 re-applying it is harmless.
                        legacy = next((s for s in prefilters if MISSING_TOKEN not in s['values']), None)
                        legacy_col, legacy_vals = (legacy['col'], list(legacy['values'])) if legacy else ('None', [])

                        # ALL filters applied here; always hand out a COPY.
                        df_out = (st['df'][_combined_mask(prefilters)] if prefilters else st['df']).copy(deep=True)

                        # Common row set (complete cases on the selected
                        # moderators): only RECORDED here. Step 6 applies it after
                        # the effect sizes are computed and before the sampling
                        # VCVs are built, for raw and pre-calculated data alike,
                        # so every analysis uses the same rows.
                        cc_cols = [c for c in cc_select.value if c in df_out.columns]
                        n_cc_missing = 0
                        if cc_cols:
                            m = pd.Series(False, index=df_out.index)
                            for c in cc_cols:
                                m |= _missing_mask(df_out[c])
                            n_cc_missing = int(m.sum())
                        if prefilters:
                            df_out.attrs[OUTPUT_TAG] = True

                        # 2. Safely Update Config Dictionary
                        _config().update({
                            'prefilter_col': legacy_col,
                            'prefilter_values': legacy_vals,
                            'prefilters': [{'col': s['col'], 'values': list(s['values']), 'seen': list(s['seen'])}
                                           for s in prefilters],
                            # We set defaults for legacy compatibility with Cell 4
                            'factor1': 'None',
                            'factor2': 'None',
                            'min_papers': 1,
                            'min_obs': 1,
                            'clean_dataframe': df_out,
                            'complete_case_moderators': [str(c) for c in cc_cols],
                        })

                        # 3. Success Message UI
                        n_total = len(st['df'])
                        n_final = len(df_out)
                        if prefilters:
                            items = ''.join(
                                f"<li><b>{html.escape(str(s['col']))}</b>: keeping {len(s['values'])} group(s)</li>"
                                for s in prefilters
                            )
                            filter_msg = (f"Applied {len(prefilters)} filter(s):<ul style='margin: 5px 0;'>{items}</ul>"
                                          f"Records retained after all filters: <b>{n_final}</b> of {n_total}.")
                        else:
                            filter_msg = "No global filters applied. The entire dataset will be retained."
                        if cc_cols:
                            filter_msg += (f"<br>Common row set: all analyses will use complete cases on <b>"
                                           f"{html.escape(', '.join(map(str, cc_cols)))}</b> — "
                                           f"<b>{n_cc_missing}</b> of these {n_final} record(s) lack a value and "
                                           f"will be excluded in Step 6 (listed in its exclusion log).")

                        warnings = []
                        empty = [s['col'] for s in prefilters if not s['values']]
                        if empty:
                            warnings.append(f"No values selected for: <b>{html.escape(', '.join(map(str, empty)))}</b> (all records with this variable will be excluded).")
                        if prefilters and n_final == 0:
                            warnings.append("The current filters exclude <b>every</b> record.")
                        if cc_cols and n_cc_missing == n_final:
                            warnings.append("The common row set excludes <b>every</b> record.")
                        warn_html = ''.join(f"<br>⚠️ {w}" for w in warnings)

                        display(HTML(f"""
                        <div style='padding: 15px; background-color: #d4edda; border-left: 5px solid #28a745; border-radius: 4px; font-family: sans-serif; margin-top: 15px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);'>
                            <h4 style='margin: 0 0 5px 0; color: #155724; display: flex; align-items: center;'>
                                <span style='margin-right: 8px;'>✅</span> Configuration Saved
                            </h4>
                            <p style='margin: 0; font-size: 14px; color: #155724;'>
                                {filter_msg}{warn_html}<br><br>
                                <b>Next Step:</b> Proceed to Step 4 (Data Cleaning).
                            </p>
                        </div>
                        """))

                        if not quiet:
                            _mark_stale = G.get('mark_stale')
                            if callable(_mark_stale) and 'ALL_DOWNSTREAM' in G:
                                _mark_stale(G['ALL_DOWNSTREAM'], "Step 3: Global Filters Changed")

                    except Exception as e:
                        display(HTML(f"<div style='padding: 10px; background-color: #f8d7da; border-left: 4px solid #dc3545; color: #721c24; margin-top: 15px; border-radius: 4px;'>❌ <b>Error saving config:</b> {html.escape(str(e))}</div>"))

            btn_save_config.on_click(on_save_config_clicked)

            # =============================================================
            # 7. LAYOUT & DISPLAY
            # =============================================================
            _load_base()

            box_pre = widgets.VBox([
                widgets.HTML("""
                <div style='margin-bottom: 15px; font-family: sans-serif;'>
                    <h4 style='color: #2c3e50; margin: 0 0 5px 0;'>Global Subset Selection <span style='font-size: 13px; font-weight: normal; color: #7f8c8d;'>(Optional)</span></h4>
                    <p style='font-size: 13px; color: #34495e; margin: 0;'>
                        Exclude specific sub-categories (e.g., specific taxa, regions, or experimental designs) from the <b>entire</b> meta-analytical pipeline.
                        Add as many filter variables as you need: a record is kept only if it passes <b>every</b> filter.
                    </p>
                </div>
                """),
                rows_container,
                widgets.HBox([btn_add_filter, btn_clear]),
                lbl_preview
            ], layout=widgets.Layout(border='1px solid #bdc3c7', padding='15px', border_radius='6px', margin='0 0 15px 0', width='100%'))

            cc_select.options = [c for c in st['moderators'] if c != 'None']
            cc_select.observe(refresh_preview, 'value')
            refresh_preview()
            box_cc = widgets.VBox([
                widgets.HTML("""
                <div style='margin-bottom: 10px; font-family: sans-serif; font-size: 13px; color: #34495e; line-height: 1.5;'>
                    <h4 style='color: #2c3e50; margin: 0 0 8px 0;'>Common Row Set <span style='font-size: 13px; font-weight: normal; color: #7f8c8d;'>(Optional)</span></h4>
                    <p style='margin: 0 0 6px 0;'><b>What it does.</b> Keeps only the records that have a value for <b>every</b>
                    moderator you select here, and runs <b>every</b> analysis (overall model, subgroups, meta-regression,
                    splines, publication bias, sensitivity analyses) on that same set of records.</p>
                    <p style='margin: 0 0 6px 0;'><b>Why it matters.</b> Without it, each analysis drops only the records it
                    cannot use. A meta-regression on a moderator with missing values then runs on fewer records than the
                    overall model, so comparisons between the two (pseudo-R², AIC, how much τ² drops) mix two different
                    datasets and can be misleading.</p>
                    <p style='margin: 0 0 6px 0;'><b>When to use it.</b> When you will compare models or report how much
                    heterogeneity a moderator explains, and some of those moderators have missing values (listed below).
                    If they have none, this setting changes nothing: leave it empty.</p>
                    <p style='margin: 0;'><b>Cost.</b> Fewer records in every analysis. Hold Ctrl/Cmd to select several moderators.</p>
                </div>
                """),
                cc_select,
                lbl_cc,
            ], layout=widgets.Layout(border='1px solid #bdc3c7', padding='15px', border_radius='6px', margin='0 0 15px 0', width='100%'))

            # What to restore: new 'prefilters' key, or legacy keys
            cfg = _config()
            saved = cfg.get('prefilters')
            if saved:
                saved = [s for s in saved if s.get('col', 'None') != 'None']
            elif cfg.get('prefilter_col', 'None') not in (None, 'None'):
                saved = [{'col': cfg['prefilter_col'], 'values': cfg.get('prefilter_values', [])}]
            else:
                saved = []
            reproducing = cfg.get('is_reproducing', False)

            if reproducing:
                if saved:
                    msg = "Restoring filter(s) on: " + ', '.join(f"<b>{html.escape(str(s['col']))}</b>" for s in saved)
                else:
                    msg = "No global filters applied"
                display(HTML(f"<div style='background:#e8f4f8; border-left:4px solid #17a2b8; padding:12px; margin-bottom:10px; border-radius:4px; color:#0c5460;'>🔄 <b>Reproducibility Mode Active</b> &nbsp;|&nbsp; {msg}</div>"))

            display(HTML("<h3 style='color: #2E86AB; border-bottom: 2px solid #3498db; padding-bottom: 5px; font-family: sans-serif;'>Step 3: Global Pre-Filtering</h3>"))
            display(HTML(f"<p style='color: #34495e; font-size: 14px; font-family: sans-serif;'><i>Loaded dataset contains <b>{len(st['df'])}</b> total records.</i></p>"))
            if looks_filtered:
                display(HTML("<div style='background:#fff3cd; border-left:4px solid #ffc107; padding:10px; margin-bottom:10px; border-radius:4px; color:#856404; font-family: sans-serif; font-size: 13px;'>⚠️ <b>The data in memory looks already filtered</b> (a later cell seems to have replaced <code>raw_data_standardized</code> with Cell 3's filtered output). Re-run <b>Cell 2</b> and then this cell to start from the full dataset.</div>"))
            filter_tabs = widgets.Tab(children=[box_pre, box_cc])
            filter_tabs.set_title(0, 'Subset filters')
            filter_tabs.set_title(1, 'Common row set')
            display(filter_tabs)
            display(btn_save_config, output_config)

            # =============================================================
            # 8. RESTORE LAST SELECTIONS (and reproducibility auto-execute)
            # =============================================================
            saved_cc = [c for c in (cfg.get('complete_case_moderators') or []) if c in cc_select.options]
            missing_cc = [c for c in (cfg.get('complete_case_moderators') or []) if c not in cc_select.options]
            cc_select.value = tuple(saved_cc)
            if missing_cc:
                display(HTML(f"<div style='background:#fff3cd; border-left:4px solid #ffc107; padding:10px; border-radius:4px; color:#856404;'>⚠️ Saved common-row-set moderator(s) not found in the data and skipped: <b>{html.escape(', '.join(map(str, missing_cc)))}</b></div>"))
            missing = load_specs(saved, strict=reproducing) if saved else load_specs([])
            if missing:
                display(HTML(f"<div style='background:#fff3cd; border-left:4px solid #ffc107; padding:10px; border-radius:4px; color:#856404;'>⚠️ Saved filter variable(s) not found in the data and skipped: <b>{html.escape(', '.join(map(str, missing)))}</b></div>"))
            if reproducing:
                on_save_config_clicked(None)
            elif saved or saved_cc:
                # Re-apply your last selections so ANALYSIS_CONFIG matches what you see.
                prev = cfg.get('clean_dataframe')
                on_save_config_clicked(None, quiet=True)
                new = _config().get('clean_dataframe')
                unchanged = isinstance(prev, pd.DataFrame) and isinstance(new, pd.DataFrame) and prev.equals(new)
                if not unchanged:      # data changed (e.g. new file in Cell 2) -> downstream is stale
                    _mark_stale = G.get('mark_stale')
                    if callable(_mark_stale) and 'ALL_DOWNSTREAM' in G:
                        _mark_stale(G['ALL_DOWNSTREAM'], "Step 3: Data or Global Filters Changed")
                display(HTML("<div style='background:#e8f4f8; border-left:4px solid #17a2b8; padding:10px; margin-top:8px; border-radius:4px; color:#0c5460; font-family: sans-serif; font-size: 13px;'>🔁 Your last filter selections were restored and applied to the current data. Categories that are new in the data start checked. Use <b>Clear all filters</b> to start over.</div>"))

            # Handles for other cells / debugging (the widget does not depend on them)
            G['btn_save_config'] = btn_save_config
            G['output_config'] = output_config
            G['on_save_config_clicked'] = on_save_config_clicked
            return {'state': st, 'filter_rows': filter_rows, 'add_filter_row': add_filter_row,
                    'cc_select': cc_select,
                    'refresh_preview': refresh_preview, 'lbl_preview': lbl_preview,
                    'save': on_save_config_clicked}

        _CELL3_UI = _cell3_global_filtering(globals())

In [ ]:
#@title ⚙️ 4. Data Cleaning & Pre-processing
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")


# =============================================================================
# STEP 4: APPLY CONFIGURATION & PREPARE DATA
# Purpose: Apply filters, handle missing/zero SDs, and prepare data for
#          effect size calculation.
# =============================================================================

import datetime
import traceback
import sys
from IPython.display import display, HTML, clear_output
import pandas as pd
import numpy as np
import ipywidgets as widgets

# =============================================================================
# SHARED CONTROL GROUP DETECTION FUNCTION
# =============================================================================

# Only an explicit 'control_id' column: names such as 'control_group' are
# often descriptive ('placebo', 'untreated') and must not create covariances.
SHARED_CONTROL_COLUMNS = ('control_id',)


def user_control_column(df):
    """Name of the optional user column that labels each row's control
    group ('control_id', case-insensitive), or None."""
    lower = {str(c).strip().lower(): c for c in df.columns}
    for name in SHARED_CONTROL_COLUMNS:
        if name in lower:
            return lower[name]
    return None


def detect_shared_controls(df):
    """
    Detects rows within the same study (id) that share control group data.
    Supports both Continuous (Means/SDs) and Binary (Events/Non-Events) data.

    If the data contain a column 'control_id', that column is used instead
    of the automatic matching below (rows of a study with the same label
    share a control; a warning is given when such rows report different
    control-group data).

    Logic:
    - Group by 'id'
    - Within each study, identify rows with identical control group values
    - For continuous data: matches on nc, xc, and sdc
    - For binary data: matches on events_c and nonevents_c
    - Assign a unique shared_group_id to rows sharing controls
    - Rows that don't share controls get None

    Parameters:
    ----------
    df : pandas.DataFrame
        Dataframe with columns:
        - id (required)
        - Continuous: nc, xc, sdc
        - Binary: events_c, nonevents_c

    Returns:
    -------
    tuple: (pandas.DataFrame with 'shared_group_id' column, int count of shared rows)
    """
    df = df.copy()
    df['shared_group_id'] = None

    # Detect data type
    is_continuous = all(col in df.columns for col in ['nc', 'xc', 'sdc'])
    is_binary = all(col in df.columns for col in ['events_c', 'nonevents_c'])

    if not is_continuous and not is_binary:
        return df, 0

    # Optional user override: a column naming the control group of each row
    # (e.g. 'control_id'). When present it replaces the automatic matching:
    # rows of one study with the same non-blank label share a control.
    user_col = user_control_column(df)
    if user_col is not None:
        shared_count = 0
        labels = df[user_col]
        blank = labels.isna() | labels.astype(str).str.strip().isin(['', 'nan', 'None'])
        ctl_cols = ['nc', 'xc', 'sdc'] if is_continuous else ['events_c', 'nonevents_c']
        inconsistent = []
        for (study_id, label), grp in df[~blank].groupby(['id', labels[~blank].astype(str).str.strip()]):
            if len(grp) >= 2:
                df.loc[grp.index, 'shared_group_id'] = f"{study_id}_shared_{label}"
                shared_count += len(grp)
                if len(grp[ctl_cols].round(6).drop_duplicates()) > 1:
                    inconsistent.append(f"{study_id}/{label}")
        if inconsistent:
            warnings.warn(
                f"control_id: rows labelled with the same control group report different control "
                f"data (n, mean, SD or counts) in {len(inconsistent)} case(s): "
                f"{', '.join(inconsistent[:10])}{' ...' if len(inconsistent) > 10 else ''}. "
                f"Their covariance uses the first row's control data; check the labels.")
        return df, shared_count

    shared_count = 0
    group_counter = 0

    for study_id, group in df.groupby('id'):
        if len(group) == 1:
            continue

        group = group.copy()

        if is_continuous:
            group['_control_key'] = (
                group['nc'].fillna(-999).astype(str) + '_' +
                group['xc'].fillna(-999).round(6).astype(str) + '_' +
                group['sdc'].fillna(-999).round(6).astype(str)
            )
        else: # is_binary
            group['_control_key'] = (
                group['events_c'].fillna(-999).astype(str) + '_' +
                group['nonevents_c'].fillna(-999).astype(str)
            )

        control_groups = group.groupby('_control_key')

        for control_key, control_group in control_groups:
            if len(control_group) >= 2:
                group_counter += 1
                shared_id = f"{study_id}_shared_grp_{group_counter}"
                df.loc[control_group.index, 'shared_group_id'] = shared_id
                shared_count += len(control_group)

    # Drop the temporary key if it exists
    if '_control_key' in df.columns:
        df = df.drop(columns=['_control_key'])

    return df, shared_count


def drop_missing_ids(df):
    """Split off rows whose study ID is missing or blank.

    Casting IDs to str turns NaN into the text 'nan', so without this all
    rows with a missing ID would be pooled as one study. Returns
    (kept rows with IDs as stripped strings, dropped rows).
    """
    if 'id' not in df.columns:
        return df, df.iloc[0:0]
    ids = df['id']
    blank = ids.isna() | ids.astype(str).str.strip().isin(['', 'nan', 'None', '<NA>'])
    kept = df[~blank].copy()
    kept['id'] = kept['id'].astype(str).str.strip()
    return kept, df[blank].copy()


# =============================================================================
# MAIN PROCESSING FUNCTION
# =============================================================================

def run_processing(df_config):
    """
    Main processing pipeline: clean data, apply filters, detect shared controls.
    (SD handling is deferred to Step 6 based on effect size choice).
    """
    raw_data = df_config.copy()
    original_rows = len(raw_data)
    cleaning_log = []
    removed_records = []

    # --- A. Drop rows without a study ID; ensure ID is string ---
    raw_data, no_id = drop_missing_ids(raw_data)
    if len(no_id) > 0:
        no_id['Reason'] = 'Missing study ID'
        removed_records.append(no_id)
        cleaning_log.append(f"Removed {len(no_id)} rows with a missing study ID")

    # --- B. Check Essential Columns ---
    binary_cols = ['events_e', 'nonevents_e', 'events_c', 'nonevents_c']
    if all(c in raw_data.columns for c in binary_cols):
        essential_cols = binary_cols
    elif (all(c in raw_data.columns for c in ['events_e', 'nonevents_e'])
          and not any(c in raw_data.columns for c in ['xe', 'xc'])):
        essential_cols = ['events_e', 'nonevents_e']     # single-group proportions
    else:
        essential_cols = ['xe', 'ne', 'xc', 'nc']
    missing_cols = [c for c in essential_cols if c not in raw_data.columns]
    if missing_cols:
        return False, {'error': f"Critical columns missing from data: {missing_cols}"}

    # --- C. Drop missing essential values ---
    mask_missing = raw_data[essential_cols].isna().any(axis=1)
    if mask_missing.sum() > 0:
        dropped = raw_data[mask_missing].copy()
        dropped['Reason'] = 'Missing Essential Data (Mean or N)'
        removed_records.append(dropped)
        raw_data = raw_data[~mask_missing].copy()
        cleaning_log.append(f"Removed {mask_missing.sum()} rows with missing means or sample sizes")

    # --- D. Ensure N >= 1 ---
    if 'ne' in raw_data.columns and 'nc' in raw_data.columns:
        for col in ['ne', 'nc']:
            # Kept as numbers, not truncated to int: averaged or adjusted
            # group sizes (e.g. 12.5) are valid, as in metafor.
            raw_data[col] = pd.to_numeric(raw_data[col], errors='coerce')

        mask_invalid_n = ~(raw_data['ne'] >= 1) | ~(raw_data['nc'] >= 1)   # NaN counts as invalid
        if mask_invalid_n.sum() > 0:
            dropped = raw_data[mask_invalid_n].copy()
            dropped['Reason'] = 'Invalid Sample Size (N < 1)'
            removed_records.append(dropped)
            raw_data = raw_data[~mask_invalid_n].copy()
            cleaning_log.append(f"Removed {mask_invalid_n.sum()} rows where N < 1")

    # (Note: SD missing/zero logic previously here has been moved to Cell 6)
    final_rows = len(raw_data)

    # --- F. Identify Moderators ---
    excluded_cols = ['id', 'xe', 'sde', 'ne', 'xc', 'sdc', 'nc', 'events_e', 'nonevents_e', 'events_c', 'nonevents_c']
    _ctl = user_control_column(raw_data)
    if _ctl is not None:
        excluded_cols.append(_ctl)      # a control-group label, not a moderator
    available_moderators = [col for col in raw_data.columns if col not in excluded_cols]

    # --- G. Apply Pre-filter ---
    global data_filtered
    data_filtered = raw_data.copy()

    prefilter_col = ANALYSIS_CONFIG.get('prefilter_col', 'None')
    selected_values = ANALYSIS_CONFIG.get('prefilter_values', [])
    rows_dropped_filter = 0

    if prefilter_col != 'None' and selected_values:
        # Match the way Cell 4 stores and applies filter values: as strings,
        # with blanks excluded. A raw isin() on a numeric column (e.g. year)
        # never matches the string values and would drop every row.
        _s = data_filtered[prefilter_col]
        _blank = _s.isna() | _s.astype(str).str.strip().eq('')
        _kept = [str(v) for v in selected_values]
        mask_filtered_out = ~(~_blank & _s.astype(str).isin(_kept))
        if mask_filtered_out.sum() > 0:
            dropped = data_filtered[mask_filtered_out].copy()
            dropped['Reason'] = f"Filtered by User ('{prefilter_col}')"
            removed_records.append(dropped)
            data_filtered = data_filtered[~mask_filtered_out].copy()
            rows_dropped_filter = mask_filtered_out.sum()
            cleaning_log.append(f"Filtered out {rows_dropped_filter} rows based on selection in '{prefilter_col}'")

    # --- H. Detect Shared Controls ---
    data_filtered, n_shared = detect_shared_controls(data_filtered)
    _ctl_col = user_control_column(data_filtered)
    if _ctl_col is not None:
        cleaning_log.append(f"Shared control groups taken from the column '{_ctl_col}': "
                            f"{n_shared} observations share a control")
    elif n_shared > 0:
        cleaning_log.append(f"Detected {n_shared} observations in shared control groups "
                            f"(automatic: identical control n, mean and SD / control counts "
                            f"within a study; add a 'control_id' column to set them yourself)")

    # --- I. Save Metadata ---
    if removed_records:
        ANALYSIS_CONFIG['removed_records'] = pd.concat(removed_records)
    else:
        ANALYSIS_CONFIG['removed_records'] = pd.DataFrame()

    ANALYSIS_CONFIG['n_observations_pre_filter'] = final_rows
    ANALYSIS_CONFIG['n_observations_post_filter'] = len(data_filtered)
    ANALYSIS_CONFIG['n_papers_post_filter'] = data_filtered['id'].nunique()

    return True, {
        'original_rows': original_rows,
        'final_rows': final_rows,
        'final_filtered': len(data_filtered),
        'data_filtered': data_filtered,
        'cleaning_log': cleaning_log,
        'removed_records': removed_records,
        'rows_dropped_filter': rows_dropped_filter,
        'prefilter_col': prefilter_col
    }


# =============================================================================
# RENDER RESULTS (Populates tabs — identical output to original Cell 4)
# =============================================================================

def render_results(results):
    """Render the summary and removed-data tabs."""

    tab_summary = widgets.Output()
    tab_removed = widgets.Output()

    tabs = widgets.Tab(children=[tab_summary, tab_removed])
    tabs.set_title(0, '📊 Data Summary')
    tabs.set_title(1, '🗑️ Removed Data')

    original_rows = results['original_rows']
    cleaning_log = results['cleaning_log']
    removed_records = results['removed_records']
    rows_dropped_filter = results['rows_dropped_filter']
    prefilter_col = results['prefilter_col']

    data_filtered = results['data_filtered']
    total_removed = original_rows - len(data_filtered)

    # --- TAB 1: SUMMARY ---
    with tab_summary:
        filter_status = "None"
        if prefilter_col != 'None':
            filter_status = f"Filtered by <b>{prefilter_col}</b> (Dropped {rows_dropped_filter} rows)"

        html_card = f"""
        <div style="font-family: sans-serif; border: 1px solid #c3e6cb; border-radius: 6px; overflow: hidden; max-width: 800px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); margin-top: 10px;">
            <div style="background-color: #d4edda; padding: 15px; color: #155724; border-bottom: 1px solid #c3e6cb;">
                <h4 style="margin: 0; display: flex; align-items: center;"><span style='margin-right: 8px;'>✅</span> Data Successfully Prepared</h4>
            </div>
            <div style="padding: 20px; background-color: white;">
                <div style="display: flex; gap: 15px; margin-bottom: 20px;">
                    <div style="flex: 1; background: #f8f9fa; padding: 15px 10px; border-radius: 6px; text-align: center; border: 1px solid #dee2e6;">
                        <div style="font-size: 26px; font-weight: bold; color: #2E86AB;">{len(data_filtered)}</div>
                        <div style="font-size: 12px; color: #6c757d; text-transform: uppercase; letter-spacing: 0.5px; margin-top: 4px;">Rows for Analysis</div>
                    </div>
                    <div style="flex: 1; background: #f8f9fa; padding: 15px 10px; border-radius: 6px; text-align: center; border: 1px solid #dee2e6;">
                        <div style="font-size: 26px; font-weight: bold; color: #2E86AB;">{data_filtered['id'].nunique()}</div>
                        <div style="font-size: 12px; color: #6c757d; text-transform: uppercase; letter-spacing: 0.5px; margin-top: 4px;">Unique Studies</div>
                    </div>
                    <div style="flex: 1; background: #fff3cd; padding: 15px 10px; border-radius: 6px; text-align: center; border: 1px solid #ffeeba;">
                        <div style="font-size: 26px; font-weight: bold; color: #856404;">{total_removed}</div>
                        <div style="font-size: 12px; color: #856404; text-transform: uppercase; letter-spacing: 0.5px; margin-top: 4px;">Total Removed</div>
                    </div>
                </div>
                <table style="width: 100%; font-size: 13px; color: #495057; border-collapse: collapse;">
                    <tr style="border-bottom: 1px solid #e9ecef;">
                        <td style="padding: 10px 8px; font-weight: 600; width: 30%;">Pre-Filter Status:</td>
                        <td style="padding: 10px 8px;">{filter_status}</td>
                    </tr>
                    <tr>
                        <td style="padding: 10px 8px; font-weight: 600;">Next Step:</td>
                        <td style="padding: 10px 8px; color: #2E86AB; font-weight: bold;">Ready for Effect Size Selection (Step 5) ▶</td>
                    </tr>
                </table>
            </div>
        </div>
        """
        display(HTML(html_card))

    # --- TAB 2: REMOVED DATA ---
    with tab_removed:
        if removed_records:
            all_removed_df = pd.concat(removed_records)

            display(HTML(f"<h4 style='color:#c0392b; margin-top:0;'>🗑️ Removed Observations ({len(all_removed_df)})</h4>"))

            base_cols = ['id', 'Reason']
            data_cols = ['xe', 'ne', 'xc', 'nc', 'sde', 'sdc']
            extra_cols = [c for c in data_filtered.columns if c not in base_cols + data_cols
                         and c in ['Year', 'Species', 'Region', prefilter_col]]
            extra_cols = [c for c in extra_cols if c != 'None']

            cols_to_show = base_cols + data_cols + extra_cols
            cols_to_show = [c for c in cols_to_show if c in all_removed_df.columns]

            display(all_removed_df[cols_to_show])
        else:
            display(HTML("""
            <div style='padding:15px; background:#d4edda; color:#155724; border-radius:5px;'>
                <b>✅ No data removed.</b><br>
                All loaded rows passed the quality checks and filter criteria.
            </div>
            """))

    display(tabs)


# =============================================================================
# MAIN EXECUTION
# =============================================================================

# --- CHECK IF PRE-CALCULATED MODE ---
data_type_mode = globals().get('DATA_TYPE_SELECTED', 'raw')

if data_type_mode == 'pre_calculated':
    if 'raw_data_standardized' in globals():
        raw_data, _no_id = drop_missing_ids(raw_data_standardized)
        data_filtered = raw_data.copy()
        if len(_no_id) > 0:
            display(HTML(
                f"<div style='padding:10px; background:#fff3cd; border-left:4px solid #ffc107; "
                f"color:#856404; border-radius:4px; margin-bottom:10px;'>⚠️ Removed "
                f"<b>{len(_no_id)}</b> rows with a missing study ID (rows "
                f"{', '.join(str(i) for i in _no_id.index[:20])}"
                f"{', ...' if len(_no_id) > 20 else ''}).</div>"))

        display(HTML(f"""
        <div style='background-color: #e2e3e5; border-left: 4px solid #6c757d; padding: 15px; border-radius: 4px; font-family: sans-serif; margin-bottom: 15px;'>
            <h4 style='color: #383d41; margin-top: 0; display: flex; align-items: center;'>
                <span style='margin-right: 8px;'>⏭️</span> Skip This Step (Pre-calculated Mode)
            </h4>
            <p style='color: #383d41; font-size: 14px; margin-bottom: 5px;'>
                Data cleaning and SD imputation are bypassed because you imported pre-calculated effect sizes.
            </p>
            <ul style='color: #383d41; font-size: 14px; margin-top: 5px; margin-bottom: 10px;'>
                <li><b>{len(data_filtered)}</b> observations ready for analysis.</li>
            </ul>
            <p style='color: #383d41; font-size: 14px; margin-bottom: 0; font-weight: bold;'>
                Please proceed directly to Step 5 (Effect Size Selection).
            </p>
        </div>
        """))
    else:
        display(HTML("""
        <div style='padding: 10px; background-color: #f8d7da; border-left: 4px solid #dc3545; color: #721c24; border-radius: 4px;'>
            ❌ <b>Error:</b> raw_data_standardized not found. Please run Step 2 (Data Ingestion) first.
        </div>
        """))

else:
    # =================================================================
    # RAW DATA MODE
    # =================================================================
    if ANALYSIS_CONFIG.get('is_reproducing', False):
        display(HTML("<div style='background:#e8f4f8; border-left:4px solid #17a2b8; padding:12px; margin-bottom:10px; border-radius:4px; color:#0c5460;'>🔄 <b>Reproducibility Mode Active</b> &nbsp;|&nbsp; Data cleaned. SD handling deferred to Cell 6.</div>"))

    # --- Safety Checks ---
    if 'raw_data_standardized' not in globals():
        display(HTML("<div style='background-color:#f8d7da; color:#721c24; padding:10px; border-radius:5px;'>❌ <b>Error:</b> Data not found. Please run Step 2 (Data Ingestion) first.</div>"))
    elif 'ANALYSIS_CONFIG' not in globals():
        display(HTML("<div style='background-color:#f8d7da; color:#721c24; padding:10px; border-radius:5px;'>❌ <b>Error:</b> Configuration not set. Please run Step 3 (Global Filtering) and click 'Save Configuration'.</div>"))
    else:
        if 'clean_dataframe' not in ANALYSIS_CONFIG:
            display(HTML("<div style='background-color:#f8d7da; color:#721c24; padding:10px;'>❌ Dataframe missing from config. Please re-run Step 3 (Global Filtering).</div>"))
        else:
            df_config = ANALYSIS_CONFIG['clean_dataframe'].copy()

            # Force numeric on stat columns
            numeric_cols = ['xe', 'sde', 'ne', 'xc', 'sdc', 'nc', 'events_e', 'nonevents_e', 'events_c', 'nonevents_c']
            for col in numeric_cols:
                if col in df_config.columns:
                    df_config[col] = pd.to_numeric(df_config[col], errors='coerce')

            display(HTML(f"<h3 style='color:#2E86AB'>Step 4: Data Cleaning & Preparation</h3>"))
            display(HTML(f"<i>Processing <b>{len(df_config)}</b> rows...</i>"))

            try:
                success, results = run_processing(df_config)
                if success:
                    data_filtered = results['data_filtered']
                    render_results(results)
                    mark_stale(ALL_DOWNSTREAM, "Step 4: Data Cleaning Changed")
                else:
                    display(HTML(f"<div style='background:#f8d7da; color:#721c24; padding:15px;'>❌ {results.get('error', 'Unknown error')}</div>"))
            except Exception as e:
                display(HTML(f"""
                <div style="border: 1px solid #f5c6cb; border-radius: 8px; padding: 20px; background-color: #f8d7da; color: #721c24;">
                    <h3 style="margin-top: 0;">❌ Error Preparing Data</h3>
                    <pre style="background: rgba(255,255,255,0.5); padding: 10px; border-radius: 5px;">{e}</pre>
                </div>
                """))
                traceback.print_exc(file=sys.stdout)

In [ ]:
#@title 🔬 5. Effect Size Selection & Diagnostics
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")


# =============================================================================
# Purpose: Analyze data characteristics and recommend appropriate effect size
# Modes: 1) Raw Data (auto-detection) or 2) Pre-calculated (manual selection)
# =============================================================================

import numpy as np
import pandas as pd
import datetime
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# --- CHECK DATA TYPE MODE ---
data_type_mode = globals().get('DATA_TYPE_SELECTED', 'raw')

# =============================================================================
# MODE 1: RAW DATA - INTELLIGENT RECOMMENDATION
# =============================================================================
if data_type_mode == 'raw':
    # --- 1. STABILITY: USE RAW DATA IF AVAILABLE ---
    target_df = globals().get('data_filtered', globals().get('raw_data_standardized'))

    # --- 1b. DATA DETECTION & DIAGNOSTICS ---
    binary_cols_full = ['events_e', 'nonevents_e', 'events_c', 'nonevents_c']
    binary_cols_exp = ['events_e', 'nonevents_e']
    binary_cols_ctrl = ['events_c', 'nonevents_c']

    has_binary_full = all(c in target_df.columns for c in binary_cols_full)
    has_binary_exp_only = (all(c in target_df.columns for c in binary_cols_exp)
                           and not all(c in target_df.columns for c in binary_cols_ctrl))
    has_continuous_cols = all(c in target_df.columns for c in ['xe', 'xc'])
    has_sd_cols = (
        ('sde' in target_df.columns and target_df['sde'].notna().any()) or
        ('sdc' in target_df.columns and target_df['sdc'].notna().any())
    )
    has_both_sds = ('sde' in target_df.columns and 'sdc' in target_df.columns)

    # Track candidate scores. Higher = stronger preference.
    scores = {'lnRR': 0, 'hedges_g': 0, 'MD': 0, 'lnCVR': 0}
    reasons = []
    _binary_data_detected = None
    secondary_recommendation = None
    recommended_type = None
    confidence = 'Low'

    # Initialize continuous-data UI metrics (avoid NameErrors downstream)
    pct_control_exactly_one = 0
    pct_control_near_one = 0
    n_negative_xe = 0; n_negative_xc = 0; has_negative = False
    n_zero_xe = 0; n_zero_xc = 0; has_zero = False
    scale_ratio = 1
    sd_pct = 0
    max_skew = 0
    n_studies = len(target_df) if target_df is not None else 0

    # =========================================================
    # BRANCH A: BINARY DATA (two-group, full event/non-event)
    # =========================================================
    if has_binary_full and not has_continuous_cols:
        _binary_data_detected = 'pure'

        # Check for zero cells — affects OR/RR stability
        zero_cells = (
            (target_df['events_e'] == 0).sum() +
            (target_df['events_c'] == 0).sum() +
            (target_df['nonevents_e'] == 0).sum() +
            (target_df['nonevents_c'] == 0).sum()
        )
        pct_zero_cells = (zero_cells / (4 * len(target_df))) * 100 if len(target_df) > 0 else 0

        # Event rarity — affects OR vs RR interpretation
        total_e = target_df['events_e'] + target_df['nonevents_e']
        total_c = target_df['events_c'] + target_df['nonevents_c']
        event_rate_e = (target_df['events_e'] / total_e).replace([np.inf, -np.inf], np.nan)
        event_rate_c = (target_df['events_c'] / total_c).replace([np.inf, -np.inf], np.nan)
        mean_event_rate = pd.concat([event_rate_e, event_rate_c]).mean()
        rare_outcome = pd.notna(mean_event_rate) and mean_event_rate < 0.10

        # Scoring among binary candidates
        bin_scores = {'log_or': 0, 'log_rr': 0, 'risk_diff': 0}

        # Base: OR and RR are both viable; RD is interpretable but variance unstable
        bin_scores['log_or'] += 2
        bin_scores['log_rr'] += 2

        if rare_outcome:
            bin_scores['log_or'] += 3
            reasons.append(('Rare Outcomes', '+++', 'log OR',
                            f'Mean event rate ~{mean_event_rate:.1%}. OR approximates RR for rare events and is preferred for case-control designs.'))
        else:
            bin_scores['log_rr'] += 2
            reasons.append(('Common Outcomes', '++', 'log RR',
                            f'Mean event rate ~{mean_event_rate:.1%}. RR is more interpretable when absolute risks are meaningful.'))

        if zero_cells > 0:
            reasons.append(('Zero Cells Detected', '⚠', 'Continuity correction',
                            f'{zero_cells} zero cells found ({pct_zero_cells:.1f}% of cells). lnOR and lnRR require continuity correction (e.g., add 0.5) — metafor handles this automatically.'))

        # Pick the binary winner
        recommended_type = max(bin_scores, key=bin_scores.get)
        bin_diff = max(bin_scores.values()) - sorted(bin_scores.values())[-2]
        confidence = "High" if bin_diff >= 3 else "Moderate" if bin_diff >= 2 else "Low"

        reasons.insert(0, ('Binary Data', '+++++', recommended_type.replace('_', ' '),
                           'Data contains binary event counts (events/non-events for both groups).'))

    # =========================================================
    # BRANCH B: SINGLE-GROUP BINARY (proportions only)
    # =========================================================
    elif has_binary_exp_only and not has_continuous_cols:
        _binary_data_detected = 'single_group'
        recommended_type = 'proportion'
        confidence = 'Moderate'
        reasons = [('Single-Group Binary', '+++', 'proportion',
                    'Only experimental event columns detected (no control). Single-group proportion is appropriate. Consider logit-transformation for meta-analysis stability.')]

    # =========================================================
    # BRANCH C: MIXED DATA (binary + continuous)
    # =========================================================
    elif has_binary_full and has_continuous_cols:
        _binary_data_detected = 'mixed'

        # Determine which subset is larger
        binary_complete = target_df[binary_cols_full].notna().all(axis=1).sum()
        continuous_complete = target_df[['xe', 'xc']].notna().all(axis=1).sum()

        if binary_complete > continuous_complete:
            recommended_type = 'log_or'
            larger = f'binary ({binary_complete} rows vs {continuous_complete} continuous)'
        else:
            recommended_type = 'hedges_g'
            larger = f'continuous ({continuous_complete} rows vs {binary_complete} binary)'

        confidence = 'Low'
        display(HTML(f"""
        <div style='padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404; font-family: sans-serif; margin-bottom: 15px; border-radius: 4px;'>
            <h4 style='margin: 0 0 8px 0; display: flex; align-items: center;'><span style='margin-right: 8px;'>⚠️</span> Ambiguous Data Types Detected</h4>
            <p style='margin: 0 0 5px 0; font-size: 14px;'>Your data contains both binary event columns and continuous mean/SD columns. The larger subset is <b>{larger}</b>.</p>
            <ul style='margin: 0; font-size: 13px;'>
                <li><b>Recommended:</b> Split the dataset and run separate meta-analyses, or filter to one data type.</li>
                <li><b>For binary outcomes:</b> log Odds Ratio or log Risk Ratio</li>
                <li><b>For continuous outcomes:</b> Hedges' g or lnRR</li>
            </ul>
        </div>
        """))
        reasons = [('Mixed Data', '⚠', 'Manual',
                    f'Both binary and continuous columns detected. Defaulting to the larger subset: {larger}.')]

    # =========================================================
    # BRANCH D: CONTINUOUS DATA
    # =========================================================
    elif has_continuous_cols:
        xe_stats = target_df['xe'].describe()
        xc_stats = target_df['xc'].describe()

        # SD availability
        if has_both_sds:
            sd_availability = target_df[['sde', 'sdc']].notna().all(axis=1).sum()
            sd_pct = (sd_availability / len(target_df)) * 100 if len(target_df) > 0 else 0
        else:
            sd_pct = 0

        # Normalization check
        control_near_one = ((target_df['xc'] >= 0.95) & (target_df['xc'] <= 1.05)).sum()
        control_exactly_one = (target_df['xc'] == 1.0).sum()
        pct_control_near_one = (control_near_one / len(target_df)) * 100
        pct_control_exactly_one = (control_exactly_one / len(target_df)) * 100

        # Negative values
        n_negative_xe = (target_df['xe'] < 0).sum()
        n_negative_xc = (target_df['xc'] < 0).sum()
        has_negative = n_negative_xe > 0 or n_negative_xc > 0

        # Zero values
        n_zero_xe = (target_df['xe'] == 0).sum()
        n_zero_xc = (target_df['xc'] == 0).sum()
        has_zero = n_zero_xe > 0 or n_zero_xc > 0

        # Scale heterogeneity
        xe_range = xe_stats['max'] - xe_stats['min']
        xc_range = xc_stats['max'] - xc_stats['min']
        scale_ratio = max(xe_range, xc_range) / (min(xe_range, xc_range) + 0.0001)

        # Distribution shape — only meaningful with N >= 10
        if n_studies >= 10:
            skew_xe = target_df['xe'].skew()
            skew_xc = target_df['xc'].skew()
            max_skew = max(skew_xe, skew_xc) if pd.notna(skew_xe) and pd.notna(skew_xc) else 0
        else:
            max_skew = 0
            reasons.append(('Small Sample', '⚠', 'Note',
                            f'Only {n_studies} rows — skewness checks skipped (unreliable with N<10).'))

        # --- SCORING RULES ---

        # Rule 1: Negatives → ratio metrics fail
        if has_negative:
            scores['hedges_g'] += 10
            scores['MD'] += 3
            reasons.append(('Negative Values', '+++', 'Hedges g / MD',
                            'Ratio metrics (lnRR, lnCVR) mathematically fail with negative numbers.'))
        else:
            scores['lnRR'] += 2
            scores['lnCVR'] += 1
            reasons.append(('All Positive', '+', 'lnRR',
                            'Data is compatible with ratio-based metrics.'))

        # Rule 2: Normalization — softer than before, with warning
        if pct_control_exactly_one > 50:
            scores['lnRR'] += 3  # reduced from +5
            reasons.append(('Pre-normalized Data', '⚠', 'lnRR (with caution)',
                            f'{pct_control_exactly_one:.0f}% of controls = 1.0 exactly. Data may already be fold-changes or log-transformed. Applying lnRR would log a ratio — verify raw scale before proceeding.'))
        elif pct_control_near_one > 30:
            scores['lnRR'] += 3
            reasons.append(('Normalized Data', '++', 'lnRR',
                            'Controls clustered around 1.0 suggests ratio data.'))
        elif 0.8 < xc_stats['mean'] < 1.2:
            scores['lnRR'] += 1
            reasons.append(('Unity Baseline', '+', 'lnRR',
                            'Control group mean is close to 1.0.'))

        # Rule 3: Scale heterogeneity
        if scale_ratio > 100:
            scores['lnRR'] += 3
            scores['lnCVR'] += 1
            reasons.append(('High Scale Variance', '+++', 'lnRR',
                            'Studies measure vastly different scales. Ratios handle this best.'))
        elif scale_ratio > 10:
            scores['lnRR'] += 2
            reasons.append(('Moderate Scale Variance', '++', 'lnRR',
                            'Ratios normalize scale differences effectively.'))
        else:
            scores['hedges_g'] += 1
            scores['MD'] += 2  # MD viable when scales are tight
            reasons.append(('Consistent Scales', '+', 'Hedges g / MD',
                            'Scales are similar across studies. If all studies share the same unit of measurement, raw mean difference (MD) is also appropriate and more interpretable.'))

        # Rule 4: Zeros
        if has_zero:
            scores['hedges_g'] += 2
            scores['MD'] += 1
            reasons.append(('Zero Values', '++', 'Hedges g',
                            'log(0) is undefined. lnRR and lnCVR require adding arbitrary constants.'))

        # Rule 5: SD availability — required by Hedges' g, MD, and lnCVR
        if sd_pct > 80:
            scores['hedges_g'] += 1
            scores['lnCVR'] += 2
            scores['MD'] += 1
            reasons.append(('Good SD Data', '+', 'Hedges g / lnCVR',
                            f'{sd_pct:.0f}% of rows have both SDs. Enables Hedges g, MD, and variability metrics (lnCVR).'))
        elif sd_pct < 20:
            scores['hedges_g'] -= 2
            scores['MD'] -= 2
            scores['lnCVR'] -= 5  # lnCVR fully depends on SDs
            reasons.append(('Missing SDs', '⚠', 'Prefer lnRR',
                            f'Only {sd_pct:.0f}% of rows have SDs. Hedges g requires imputation; lnCVR is not viable. Response ratios (lnRR) are safer.'))

        # Rule 6: Distribution shape (gated on N >= 10)
        if n_studies >= 10:
            if max_skew > 1.5:
                scores['lnRR'] += 2
                reasons.append(('High Right-Skew', '++', 'lnRR',
                                'Highly skewed data often represents multiplicative biological processes. Ratios handle this naturally.'))
            elif -0.5 < max_skew < 0.5:
                scores['hedges_g'] += 1
                scores['MD'] += 1
                reasons.append(('Symmetric Distribution', '+', 'Hedges g / MD',
                                'Data appears relatively symmetric, fitting standardized/raw mean differences well.'))

        # --- DETERMINE PRIMARY WINNER ---
        # Filter to candidates with non-negative scores
        viable = {k: v for k, v in scores.items() if v >= 0}
        if not viable:
            viable = {'hedges_g': 0}

        sorted_scores = sorted(viable.values(), reverse=True)
        top_score = sorted_scores[0]
        second_score = sorted_scores[1] if len(sorted_scores) > 1 else 0
        top_candidates = [k for k, v in viable.items() if v == top_score]

        # Tie-breaker: genuinely new information, not a repeat of prior rules
        if len(top_candidates) > 1:
            # Prefer lnRR over MD/Hedges only when strictly positive (already scored)
            # Otherwise prefer Hedges' g as the more general-purpose safe choice
            if 'hedges_g' in top_candidates:
                recommended_type = 'hedges_g'
                reasons.append(('Tie-Breaker', '+', 'Hedges g',
                                'Scores tied. Hedges g is the most general-purpose choice (Nakagawa et al. 2023, Table 2).'))
            elif 'lnRR' in top_candidates:
                recommended_type = 'lnRR'
                reasons.append(('Tie-Breaker', '+', 'lnRR',
                                'Scores tied. lnRR gives interpretable % change for strictly positive data.'))
            else:
                recommended_type = top_candidates[0]
        else:
            recommended_type = top_candidates[0]

        # Confidence based on margin between top and second
        score_diff = top_score - second_score
        confidence = ("Strong signal" if score_diff >= 5
                      else "Mixed signal" if score_diff >= 3
                      else "Weak signal")

        # --- SECONDARY RECOMMENDATION: lnCVR for variability questions ---
        # Surface only when SDs are well-populated and the primary is a mean-based metric
        if sd_pct > 80 and not has_negative and not has_zero and recommended_type in ('lnRR', 'hedges_g', 'MD'):
            secondary_recommendation = {
                'type': 'lnCVR',
                'reason': ('If your research question involves changes in *variability* (not just means) — '
                          'e.g., does the treatment increase between-individual variation? — also consider '
                          'lnCVR (log coefficient of variation ratio). See Nakagawa et al. 2015 / 2023.')
            }
            reasons.append(('Secondary: lnCVR', 'info', 'lnCVR (optional)',
                            'Variability-based effect size is viable with your SD coverage. Consider running alongside the primary analysis if variance effects are of interest.'))

        # --- SCALE DEPENDENCE CAVEAT ---
        # Always flag for continuous data per Nakagawa et al. 2023, "Scale dependence" section
        reasons.append(('Scale Dependence', 'caveat', 'Check',
                        'If studies measured outcomes at different spatial/temporal scales (e.g., plot size, sampling duration), effect size estimates may be biased. Consider plot/duration as a moderator in meta-regression if applicable.'))

    # =========================================================
    # BRANCH E: FALLBACK — no recognizable structure
    # =========================================================
    else:
        _binary_data_detected = None
        recommended_type = 'hedges_g'
        confidence = 'Low'
        reasons = [('Unrecognized Structure', '⚠', 'Manual',
                    'Could not detect standard binary (events/non-events) or continuous (xe/xc) columns. Please review your data and select an effect size manually.')]

    # --- 4. SETUP UI ---
    tab_main = widgets.Output()
    tab_patterns = widgets.Output()
    tab_logic = widgets.Output()

    tabs = widgets.Tab(children=[tab_main, tab_patterns, tab_logic])
    tabs.set_title(0, '💡 Recommendation')
    tabs.set_title(1, '📊 Data Patterns')
    tabs.set_title(2, '🧠 Decision Logic')

    # --- TAB 2: DATA PATTERNS (Educational) ---
    with tab_patterns:
        if _binary_data_detected == 'pure':
            display(HTML(f"""
            <div style='padding:10px; font-size:14px; line-height:1.6;'>
                <h4 style='margin-top:0; color:#2E86AB;'>🔍 Diagnostic Checks</h4>
                <p>We analyzed <b>{len(target_df)} observations</b>. Your dataset contains <b>Binary Event Data</b>.</p>
                <hr>
                <b>1️⃣ Data Type Recognized</b><br>
                • <b>Result:</b> Found complete 2x2 contingency table data.<br>
                • <b>Implication:</b> Continuous effect sizes (like Hedges' g or lnRR) are not mathematically appropriate for count data.
                <br><br>
                <b>2️⃣ Zero Event Handling</b><br>
                • <b>Action:</b> If any study contains a cell with zero events, the pipeline will automatically apply the standard Haldane-Anscombe correction (+0.5 to all cells) in the next step to prevent computational errors.
            </div>
            """))
        else:
            display(HTML(f"""
            <div style='padding:10px; font-size:14px; line-height:1.6;'>
                <h4 style='margin-top:0; color:#2E86AB;'>🔍 Diagnostic Checks</h4>
                <p>We analyzed <b>{len(target_df)} observations</b> to determine the statistical properties of your dataset.
                Here is what we found:</p>

                <hr>

                <b>1️⃣ Control Group Normalization</b><br>
                Values near 1.0 often indicate "Fold-Change" data (e.g., gene expression normalized to a control).<br>
                • <b>Result:</b> {pct_control_exactly_one:.1f}% of controls are exactly 1.0.<br>
                • <b>Implication:</b> {'Strong evidence for Ratio data.' if pct_control_exactly_one > 50 else 'No strong evidence of pre-normalization.'}
                <br><br>

                <b>2️⃣ Negative Values</b><br>
                Log-based metrics (like lnRR) <i>cannot</i> mathematically handle negative numbers.<br>
                • <b>Result:</b> Found {n_negative_xe + n_negative_xc} negative values.<br>
                • <b>Implication:</b> {'❌ MUST use Standardized Difference (Hedges g).' if has_negative else '✓ Compatible with Ratio metrics.'}
                <br><br>

                <b>3️⃣ Zero Values</b><br>
                Log of zero is undefined. Zeros require adding a "small constant" to work with lnRR.<br>
                • <b>Result:</b> Found {n_zero_xe + n_zero_xc} zero values.<br>
                • <b>Implication:</b> {'⚠️ lnRR will require adjustment.' if has_zero else '✓ Clean data.'}
                <br><br>

                <b>4️⃣ Scale Heterogeneity</b><br>
                Do studies measure things on the same scale (e.g., all in grams) or different scales (grams vs. tons)?<br>
                • <b>Result:</b> Largest value is {scale_ratio:.1f}× larger than the smallest range.<br>
                • <b>Implication:</b> {'High variation favors Ratios (lnRR).' if scale_ratio > 10 else 'Low variation allows Standardized Differences.'}
                <br><br>

                <b>5️⃣ Data Completeness</b><br>
                Standardized differences (Hedges' g) require Standard Deviations (SD) to calculate.<br>
                • <b>Result:</b> {sd_pct:.1f}% of rows have valid SDs.<br><br>

                <b>6️⃣ Distribution Shape (Skewness)</b><br>
                Biological data (like abundance) is often log-normally distributed (right-skewed).<br>
                • <b>Result:</b> Maximum skewness observed is {max_skew:.2f}.<br>
                • <b>Implication:</b> {'Strong right-skew favors Ratios (lnRR).' if max_skew > 1.5 else 'Symmetric data safely supports Standardized Differences (Hedges g).' if max_skew < 0.5 else 'Moderate skew; both metrics mathematically viable.'}
                <br>

            </div>
            """))

    # --- TAB 3: LOGIC (Educational) ---
    with tab_logic:
        # Create HTML table rows
        rows_html = ""
        for r in reasons:
            rows_html += f"<tr><td><b>{r[0]}</b></td><td>{r[1]}</td><td>{r[2]}</td><td>{r[3]}</td></tr>"

        if _binary_data_detected == 'pure':
            score_text = ""
        else:
            score_text = f"""
            <div style='margin-top:20px; padding:10px; background-color:#eef; border-radius:5px;'>
                <b>Final Score:</b><br>
                📊 <b>log Response Ratio (lnRR):</b> {scores.get('lnRR', 0)} points<br>
                📊 <b>Hedges' g (SMD):</b> {scores.get('hedges_g', 0)} points<br>
                📊 <b>Mean Difference (MD):</b> {scores.get('MD', 0)} points<br>
                📊 <b>log CV Ratio (lnCVR):</b> {scores.get('lnCVR', 0)} points
            </div>
            """

        display(HTML(f"""
        <div style='padding:10px; font-size:14px;'>
            <h4 style='margin-top:0; color:#2E86AB;'>🧠 How the Algorithm Decides</h4>
            <p>We use a weighted scoring system to recommend the most statistically appropriate effect size.
            Some factors (like negative values) are "hard constraints," while others are preferences.</p>

            <table style='width:100%; border-collapse:collapse; margin-top:10px;'>
                <tr style='background-color:#f0f0f0; text-align:left; border-bottom:2px solid #ddd;'>
                    <th style='padding:8px;'>Diagnostic Factor</th>
                    <th style='padding:8px;'>Weight</th>
                    <th style='padding:8px;'>Favors</th>
                    <th style='padding:8px;'>Educational Note</th>
                </tr>
                {rows_html}
            </table>
            {score_text}
        </div>
        """))

    # --- TAB 1: MAIN (Selection) ---
    with tab_main:
        # 1. Recommendation Box
        if recommended_type == 'lnRR':
            html_rec = f"""
            <div style='background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; margin-bottom: 20px;'>
                <h3 style='color: #155724; margin-top: 0;'>💡 Recommendation: log Response Ratio (lnRR)</h3>
                <p style='color: #155724; margin-bottom: 0;'><b>Why?</b> Your data appears to be <b>ratio-based</b> (e.g., fold-changes, growth rates).
                lnRR is the natural metric for this data type because it handles scale differences and has a direct biological interpretation (% change).</p>
            </div>"""

        elif recommended_type == 'log_rr':
            html_rec = f"""
            <div style='background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; margin-bottom: 20px;'>
                <h3 style='color: #856404; margin-top: 0;'>💡 Recommendation: log Risk Ratio (log RR)</h3>
                <p style='color: #856404; margin-bottom: 0;'><b>Why?</b> Your data contains <b>binary event counts</b> (events/non-events).
                The Risk Ratio measures absolute risk change and is preferred when event rates are directly interpretable.</p>
            </div>"""

        elif recommended_type == 'log_or':
            html_rec = f"""
            <div style='background-color: #f8d7da; border-left: 5px solid #dc3545; padding: 15px; margin-bottom: 20px;'>
                <h3 style='color: #721c24; margin-top: 0;'>💡 Recommendation: log Odds Ratio (log OR)</h3>
                <p style='color: #721c24; margin-bottom: 0;'><b>Why?</b> Your data contains <b>binary event counts</b> (events/non-events).
                The Odds Ratio is the standard effect size for 2×2 contingency table data.</p>
            </div>"""

        else:
            html_rec = f"""
            <div style='background-color: #d1ecf1; border-left: 5px solid #17a2b8; padding: 15px; margin-bottom: 20px;'>
                <h3 style='color: #0c5460; margin-top: 0;'>💡 Recommendation: Hedges' g (SMD)</h3>
                <p style='color: #0c5460; margin-bottom: 0;'><b>Why?</b> Your data appears to be <b>absolute measurements</b> on potentially different scales.
                Hedges' g is ideal here because it standardizes effects into "SD units," making them comparable even if units differ.</p>
            </div>"""

        display(HTML(html_rec))

        # 2. Selection Widget
        effect_size_widget = widgets.RadioButtons(
            options=[
                ('log Response Ratio (lnRR) - for ratio/fold-change data', 'lnRR'),
                ("Hedges' g - for standardized mean differences (corrected)", 'hedges_g'),
                ("Cohen's d - for standardized mean differences (uncorrected)", 'cohen_d'),
                ("Mean Difference (MD) - for raw, unstandardized differences", 'MD'),
                ("log Coefficient of Variation Ratio (lnCVR) - for variability", 'lnCVR'),
                ('log Odds Ratio (log OR) - for binary outcome data', 'log_or'),
                ('log Risk Ratio (log RR) - for binary outcome data', 'log_rr'),
                ('Risk Difference (RD) - absolute risk change', 'risk_diff'),
                ('Logit Proportion - for single-group prevalence', 'proportion')
            ],
            value=recommended_type,
            description='Select Type:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='600px')
        )

        # 3. Info Panel
        info_output = widgets.Output()
        info_panels = {
            'lnRR': "<div style='padding:10px; background:#fff; border:1px solid #ddd; color:#555; font-size:13px;'><b>🎓 About lnRR:</b> Calculates the log-ratio of means (ln(Xe/Xc)). Essential for data where 'doubling' is the same magnitude of effect as 'halving'. Commonly used in ecology for biomass, abundance, and size.</div>",
            'hedges_g': "<div style='padding:10px; background:#fff; border:1px solid #ddd; color:#555; font-size:13px;'><b>🎓 About Hedges' g:</b> A variation of Cohen's d that includes a correction factor (J) for small sample sizes. It prevents overestimation of effects in small studies, making it the gold standard for SMD in meta-analysis.</div>",
            'cohen_d': "<div style='padding:10px; background:#fff; border:1px solid #ddd; color:#555; font-size:13px;'><b>🎓 About Cohen's d:</b> The classic standardized mean difference. It is slightly biased (too high) when sample sizes are small (N < 20). Hedges' g is usually preferred.</div>",
            'log_or': "<div style='padding:10px; background:#fff; border:1px solid #ddd; color:#555; font-size:13px;'><b>🎓 About log OR:</b> The log odds ratio, calculated from 2×2 contingency tables (events/non-events). Standard for binary outcomes in clinical and epidemiological meta-analysis. Back-transformed to Odds Ratio for interpretation.</div>",
            'log_rr': "<div style='padding:10px; background:#fff; border:1px solid #ddd; color:#555; font-size:13px;'><b>🎓 About log RR:</b> The log risk ratio, calculated from 2×2 contingency tables (events/non-events). Measures the ratio of event probabilities between groups. Preferred when absolute risk differences are clinically meaningful.</div>",
            'MD': "<div style='padding:10px; background:#fff; border:1px solid #ddd; color:#555; font-size:13px;'><b>🎓 About MD:</b> The raw Mean Difference (Treatment - Control). Only appropriate if ALL studies in your meta-analysis measure the outcome on the EXACT same scale (e.g., all measure biomass in grams).</div>",
            'lnCVR': "<div style='padding:10px; background:#fff; border:1px solid #ddd; color:#555; font-size:13px;'><b>🎓 About lnCVR:</b> The log Coefficient of Variation Ratio. Measures whether a treatment alters the <i>variability</i> of a trait relative to the mean, rather than just the mean itself (Nakagawa et al. 2015).</div>",
            'risk_diff': "<div style='padding:10px; background:#fff; border:1px solid #ddd; color:#555; font-size:13px;'><b>🎓 About RD:</b> The Risk Difference. Calculates the absolute difference in event probabilities. Highly interpretable but can suffer from variance instability near 0 and 1.</div>",
            'proportion': "<div style='padding:10px; background:#fff; border:1px solid #ddd; color:#555; font-size:13px;'><b>🎓 About Proportion:</b> Analyzes the prevalence of an event in a single group. Data is logit-transformed to prevent confidence intervals from crossing 0 or 1.</div>"
        }


        def update_info(change):
            with info_output:
                clear_output()
                sel = change['new']

                # --- VALIDATION LOGIC ---
                is_valid = True
                error_msg = ""

                # 1. Check Continuous metrics (Now includes MD and lnCVR)
                if sel in ['lnRR', 'hedges_g', 'cohen_d', 'MD', 'lnCVR'] and not has_continuous_cols:
                    is_valid = False
                    error_msg = "❌ INCOMPATIBLE DATA: This effect size requires continuous data (means/SDs), but your dataset lacks the necessary columns."

                # 2. Check Binary Two-Group metrics (Now includes risk_diff)
                elif sel in ['log_or', 'log_rr', 'risk_diff'] and not has_binary_full:
                    is_valid = False
                    error_msg = "❌ INCOMPATIBLE DATA: This effect size requires full binary event data (events and non-events for BOTH treatment and control groups)."

                # 3. Check Binary Single-Group metric (proportion)
                elif sel == 'proportion' and not (has_binary_full or has_binary_exp_only):
                    is_valid = False
                    error_msg = "❌ INCOMPATIBLE DATA: The 'Proportion' metric requires at least experimental binary data (events_e and nonevents_e)."

                # --- RENDER UI ---
                if not is_valid:
                    # Show red error and lock the button
                    display(HTML(f"<div style='padding:15px; background:#f8d7da; border:1px solid #f5c6cb; color:#721c24; font-size:13px; border-radius:5px;'><b>{error_msg}</b><br><br>Please select an effect size that matches your data type.</div>"))
                    proceed_button.disabled = True
                    proceed_button.description = "✗ Invalid Selection"
                    proceed_button.button_style = 'danger'
                else:
                    # Show normal info and unlock the button
                    display(HTML(info_panels[sel]))
                    proceed_button.disabled = False
                    proceed_button.description = "✓ Confirm Selection"
                    proceed_button.button_style = 'success'

        # Small-sample bias correction for the log ratio measures. Same switch
        # as metafor's escalc(correct = ...): on = second-order bias
        # corrections (metafor's default since 4.8; Lajeunesse 2015), off = none.
        bias_corr_widget = widgets.Checkbox(
            value=bool(ANALYSIS_CONFIG.get('bias_correction', True)),
            description='Small-sample bias correction (lnRR and lnCVR only)',
            indent=False, layout=widgets.Layout(width='auto'))
        bias_corr_note = widgets.HTML(
            "<div style='font-size:12px; color:#555; margin:0 0 8px 0; max-width:720px;'>"
            "On (recommended): lnRR and lnCVR use the second-order bias corrections "
            "(Lajeunesse 2015), as metafor's <code>escalc()</code> does by default. Off: no "
            "small-sample corrections (<code>escalc(..., correct = FALSE)</code>). Sampling "
            "variances are the same either way; other effect sizes are not affected.</div>")

        # 4. Confirm Button — defined BEFORE update_info() is called so the
        #    initial call (line below) can safely reference proceed_button.
        proceed_button = widgets.Button(
            description='✓ Confirm Selection',
            button_style='success',
            layout=widgets.Layout(width='300px', height='40px'),
            style={'font_weight': 'bold'}
        )
        proceed_output = widgets.Output()

        effect_size_widget.observe(update_info, names='value')

        # Trigger it once manually to set the initial button state
        update_info({'new': recommended_type})

        def on_proceed(b):
            with proceed_output:
                clear_output()
                sel = effect_size_widget.value

                # --- CONFIGURATION ---
                es_configs = {
                    'lnRR': {
                        'effect_col': 'lnRR', 'var_col': 'var_lnRR', 'se_col': 'SE_lnRR',
                        'ci_lower_col': 'CI_lower_lnRR', 'ci_upper_col': 'CI_upper_lnRR',
                        'effect_label': 'log Response Ratio', 'effect_label_short': 'lnRR',
                        'type': 'lnRR', 'has_fold_change': True, 'null_value': 0, 'scale': 'log', 'allows_negative': False
                    },
                    'hedges_g': {
                        'effect_col': 'hedges_g', 'var_col': 'Vg', 'se_col': 'SE_g',
                        'ci_lower_col': 'CI_lower_g', 'ci_upper_col': 'CI_upper_g',
                        'effect_label': "Hedges' g", 'effect_label_short': 'g',
                        'type': "Hedges' g", 'has_fold_change': False, 'null_value': 0, 'scale': 'standardized', 'allows_negative': True
                    },
                    'cohen_d': {
                        'effect_col': 'cohen_d', 'var_col': 'Vd', 'se_col': 'SE_d',
                        'ci_lower_col': 'CI_lower_d', 'ci_upper_col': 'CI_upper_d',
                        'effect_label': "Cohen's d", 'effect_label_short': 'd',
                        'type': "Cohen's d", 'has_fold_change': False, 'null_value': 0, 'scale': 'standardized', 'allows_negative': True
                    },
                    'log_or': {
                        'effect_col': 'log_OR', 'var_col': 'var_log_OR', 'se_col': 'SE_log_OR',
                        'ci_lower_col': 'CI_lower_log_OR', 'ci_upper_col': 'CI_upper_log_OR',
                        'effect_label': 'log Odds Ratio', 'effect_label_short': 'logOR',
                        'type': 'log_or', 'has_fold_change': True, 'null_value': 0, 'scale': 'log', 'allows_negative': False
                    },
                    'log_rr': {
                        'effect_col': 'log_RR', 'var_col': 'var_log_RR', 'se_col': 'SE_log_RR',
                        'ci_lower_col': 'CI_lower_log_RR', 'ci_upper_col': 'CI_upper_log_RR',
                        'effect_label': 'log Risk Ratio', 'effect_label_short': 'RR',
                        'type': 'log_rr', 'has_fold_change': True, 'null_value': 0, 'scale': 'log', 'allows_negative': False
                    },
                    'MD': {
                        'effect_col': 'MD', 'var_col': 'var_MD', 'se_col': 'SE_MD',
                        'ci_lower_col': 'CI_lower_MD', 'ci_upper_col': 'CI_upper_MD',
                        'effect_label': 'Mean Difference', 'effect_label_short': 'MD',
                        'type': 'MD', 'has_fold_change': False, 'null_value': 0, 'scale': 'raw', 'allows_negative': True
                    },
                    'lnCVR': {
                        'effect_col': 'lnCVR', 'var_col': 'var_lnCVR', 'se_col': 'SE_lnCVR',
                        'ci_lower_col': 'CI_lower_lnCVR', 'ci_upper_col': 'CI_upper_lnCVR',
                        'effect_label': 'log CV Ratio', 'effect_label_short': 'lnCVR',
                        'type': 'lnCVR', 'has_fold_change': True, 'null_value': 0, 'scale': 'log', 'allows_negative': True
                    },
                    'risk_diff': {
                        'effect_col': 'RD', 'var_col': 'var_RD', 'se_col': 'SE_RD',
                        'ci_lower_col': 'CI_lower_RD', 'ci_upper_col': 'CI_upper_RD',
                        'effect_label': 'Risk Difference', 'effect_label_short': 'RD',
                        'type': 'risk_diff', 'has_fold_change': False, 'null_value': 0, 'scale': 'raw', 'allows_negative': True
                    },
                    'proportion': {
                        'effect_col': 'logit_p', 'var_col': 'var_logit', 'se_col': 'SE_logit',
                        'ci_lower_col': 'CI_lower_logit', 'ci_upper_col': 'CI_upper_logit',
                        'effect_label': 'Logit Proportion', 'effect_label_short': 'Logit',
                        'type': 'proportion', 'has_fold_change': False, 'null_value': 0, 'scale': 'logit', 'allows_negative': True
                  }
                }

                ANALYSIS_CONFIG['effect_size_type'] = sel
                ANALYSIS_CONFIG['es_config'] = es_configs[sel]
                ANALYSIS_CONFIG['data_type'] = 'raw'  # Store mode
                ANALYSIS_CONFIG['bias_correction'] = bool(bias_corr_widget.value)

                # Pre-set global vars
                ANALYSIS_CONFIG['effect_col'] = es_configs[sel]['effect_col']
                ANALYSIS_CONFIG['var_col'] = es_configs[sel]['var_col']
                ANALYSIS_CONFIG['se_col'] = es_configs[sel]['se_col']

                display(HTML(f"""
                <div style='padding: 15px; background-color: #d4edda; border-left: 5px solid #28a745; border-radius: 4px; font-family: sans-serif; margin-top: 15px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);'>
                    <h4 style='margin: 0 0 5px 0; color: #155724; display: flex; align-items: center;'>
                        <span style='margin-right: 8px;'>✅</span> Configuration Saved
                    </h4>
                    <p style='margin: 0; font-size: 14px; color: #155724;'>
                        Confirmed Effect Size Metric: <b>{es_configs[sel]['effect_label']}</b>.<br><br>
                        <b>Next Step:</b> Proceed to Step 6 (Effect Size Calculation).
                    </p>
                </div>
                """))
                mark_stale(ALL_DOWNSTREAM, "Step 5: Effect Size Metric Changed")


        proceed_button.on_click(on_proceed)

        display(widgets.VBox([
            effect_size_widget,
            info_output,
            widgets.HTML("<br>"),
            bias_corr_widget,
            bias_corr_note,
            proceed_button,
            proceed_output
        ]))

    # --- DISPLAY ---
    if ANALYSIS_CONFIG.get('is_reproducing', False):
        es = ANALYSIS_CONFIG.get('effect_size_type', 'None')
        display(HTML(f"<div style='background:#e8f4f8; border-left:4px solid #17a2b8; padding:12px; margin-bottom:10px; border-radius:4px; color:#0c5460;'>🔄 <b>Reproducibility Mode Active</b> &nbsp;|&nbsp; Restoring Effect Size: <b>{es}</b></div>"))
    display(tabs)

    # =================================================================
    # REPRODUCIBILITY MODE (CELL 5 - RAW): Auto-populate and Execute
    # =================================================================
    if 'ANALYSIS_CONFIG' not in globals():
        ANALYSIS_CONFIG = {}
    if ANALYSIS_CONFIG.get('is_reproducing', False):
        saved_es = ANALYSIS_CONFIG.get('effect_size_type')
        valid_options = [opt[1] if isinstance(opt, tuple) else opt for opt in effect_size_widget.options]

        if saved_es and saved_es in valid_options:
            # Temporarily unobserve to prevent racing UI updates
            try:
                effect_size_widget.unobserve(update_info, names='value')
            except ValueError:
                pass

            effect_size_widget.value = saved_es
            effect_size_widget.observe(update_info, names='value')

            # Manually trigger the UI update for the new value
            update_info({'new': saved_es})

        # Programmatically trigger the confirmation
        on_proceed(None)

# =============================================================================
# MODE 2: PRE-CALCULATED - MANUAL SELECTION
# =============================================================================
else:
    display(HTML("""
    <div style='background-color:#fff3cd; border-left: 5px solid #ffc107; padding: 15px; margin-bottom: 20px;'>
        <h3 style='color:#856404; margin-top: 0;'>📊 Pre-calculated Effect Size Mode</h3>
        <p style='color:#856404; margin-bottom: 0;'>Since you uploaded pre-calculated effect sizes,
        we cannot analyze the raw data characteristics. Please manually select the type of effect size
        you provided below.</p>
    </div>
    """))

    # Effect Size Type Descriptions
    ES_TYPE_INFO = {
        'hedges_g': {
            'label': "Hedges' g (Standardized Mean Difference)",
            'desc': "Standardized mean difference corrected for small sample bias. Measures the difference between groups in standard deviation units. Use this for continuous outcomes measured on different scales."
        },
        'lnRR': {
            'label': 'log Response Ratio (lnRR)',
            'desc': 'The natural log of the ratio of means (ln(Treatment/Control)). Ideal for ratio-based data like fold-changes, growth rates, or proportional changes. Commonly used in ecology and biology.'
        },
        'MD': {
            'label': 'Mean Difference (MD)',
            'desc': 'The raw, unstandardized difference between means. Only appropriate if all studies measured the outcome on the exact same scale/unit.'
        },
        'lnCVR': {
            'label': 'log Coefficient of Variation Ratio (lnCVR)',
            'desc': 'Measures whether a treatment alters the variability of a trait relative to its mean.'
        },
        'log_or': {
            'label': 'log Odds Ratio (logOR)',
            'desc': 'The natural log of the odds ratio. Used exclusively for binary outcomes (Yes/No, Event/Non-event). NOT suitable for continuous measurements.'
        },
        'log_rr': {
            'label': 'log Risk Ratio (logRR)',
            'desc': 'The natural log of the risk ratio. Used exclusively for binary outcomes to measure the ratio of event probabilities between groups. NOT suitable for continuous measurements.'
        },
        'risk_diff': {
            'label': 'Risk Difference (RD)',
            'desc': 'The absolute difference in event probabilities. Highly interpretable but can suffer from variance instability near 0 and 1.'
        },
        'proportion': {
            'label': 'Logit Proportion',
            'desc': 'Analyzes the prevalence of an event in a single group, logit-transformed for statistical stability.'
        },
        'fisher_z': {
            'label': "Fisher's z (Correlation)",
            'desc': "Fisher's z-transformation of correlation coefficients. Use this if your effect sizes are correlations (Pearson's r, Spearman's rho, etc.) that have been transformed."
        },
        'cohen_d': {
            'label': "Cohen's d (Uncorrected SMD)",
            'desc': "Standardized mean difference without small sample correction. Similar to Hedges' g but may overestimate effects in small studies. Hedges' g is generally preferred."
        }
    }

    # Dropdown for effect size type
    es_type_dropdown = widgets.Dropdown(
        options=[(v['label'], k) for k, v in ES_TYPE_INFO.items()],
        value='hedges_g',
        description='Effect Size Type:',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='700px')
    )

    # Dynamic info panel
    info_panel = widgets.Output()

    def update_es_info(change):
        with info_panel:
            clear_output()
            es_type = change['new']
            info = ES_TYPE_INFO[es_type]
            display(HTML(f"""
            <div style='padding:15px; background:#f8f9fa; border:1px solid #dee2e6; border-radius:5px; margin-top:10px;'>
                <b style='color:#2E86AB;'>📖 About {info['label']}:</b><br>
                <p style='margin-top:5px; margin-bottom:0; color:#555; font-size:13px;'>{info['desc']}</p>
            </div>
            """))

    es_type_dropdown.observe(update_es_info, names='value')

    # Initial display
    with info_panel:
        info = ES_TYPE_INFO['hedges_g']
        display(HTML(f"""
        <div style='padding:15px; background:#f8f9fa; border:1px solid #dee2e6; border-radius:5px; margin-top:10px;'>
            <b style='color:#2E86AB;'>📖 About {info['label']}:</b><br>
            <p style='margin-top:5px; margin-bottom:0; color:#555; font-size:13px;'>{info['desc']}</p>
        </div>
        """))

    # Confirm button
    confirm_button = widgets.Button(
        description='✓ Confirm Effect Size Type',
        button_style='success',
        layout=widgets.Layout(width='300px', height='40px', margin='20px 0 0 0'),
        icon='check-circle'
    )

    confirm_output = widgets.Output()

    def on_confirm_es_type(b):
        with confirm_output:
            clear_output()
            sel = es_type_dropdown.value

            # Configuration mapping
            es_configs = {
                'hedges_g': {
                    'effect_col': 'hedges_g', 'var_col': 'Vg', 'se_col': 'SE_g',
                    'ci_lower_col': 'CI_lower_g', 'ci_upper_col': 'CI_upper_g',
                    'effect_label': "Hedges' g", 'effect_label_short': 'g',
                    'type': "Hedges' g", 'has_fold_change': False, 'null_value': 0, 'scale': 'standardized', 'allows_negative': True
                },
                'lnRR': {
                    'effect_col': 'lnRR', 'var_col': 'var_lnRR', 'se_col': 'SE_lnRR',
                    'ci_lower_col': 'CI_lower_lnRR', 'ci_upper_col': 'CI_upper_lnRR',
                    'effect_label': 'log Response Ratio', 'effect_label_short': 'lnRR',
                    'type': 'lnRR', 'has_fold_change': True, 'null_value': 0, 'scale': 'log', 'allows_negative': False
                },
                'log_or': {
                    'effect_col': 'log_OR', 'var_col': 'var_log_OR', 'se_col': 'SE_log_OR',
                    'ci_lower_col': 'CI_lower_log_OR', 'ci_upper_col': 'CI_upper_log_OR',
                    'effect_label': 'log Odds Ratio', 'effect_label_short': 'logOR',
                    'type': 'log_or', 'has_fold_change': True, 'null_value': 0, 'scale': 'log', 'allows_negative': False
                },
                'log_rr': {
                    'effect_col': 'log_RR', 'var_col': 'var_log_RR', 'se_col': 'SE_log_RR',
                    'ci_lower_col': 'CI_lower_log_RR', 'ci_upper_col': 'CI_upper_log_RR',
                    'effect_label': 'log Risk Ratio', 'effect_label_short': 'RR',
                    'type': 'log_rr', 'has_fold_change': True, 'null_value': 0, 'scale': 'log', 'allows_negative': False
                },
                'fisher_z': {
                    'effect_col': 'fisher_z', 'var_col': 'var_fisher_z', 'se_col': 'SE_fisher_z',
                    'ci_lower_col': 'CI_lower_z', 'ci_upper_col': 'CI_upper_z',
                    'effect_label': "Fisher's z", 'effect_label_short': 'z',
                    'type': "Fisher's z", 'has_fold_change': False, 'null_value': 0, 'scale': 'transformed', 'allows_negative': True
                },
                'cohen_d': {
                    'effect_col': 'cohen_d', 'var_col': 'Vd', 'se_col': 'SE_d',
                    'ci_lower_col': 'CI_lower_d', 'ci_upper_col': 'CI_upper_d',
                    'effect_label': "Cohen's d", 'effect_label_short': 'd',
                    'type': "Cohen's d", 'has_fold_change': False, 'null_value': 0, 'scale': 'standardized', 'allows_negative': True
                },
                'MD': {
                    'effect_col': 'MD', 'var_col': 'var_MD', 'se_col': 'SE_MD',
                    'ci_lower_col': 'CI_lower_MD', 'ci_upper_col': 'CI_upper_MD',
                    'effect_label': 'Mean Difference', 'effect_label_short': 'MD',
                    'type': 'MD', 'has_fold_change': False, 'null_value': 0, 'scale': 'raw', 'allows_negative': True
                },
                'lnCVR': {
                    'effect_col': 'lnCVR', 'var_col': 'var_lnCVR', 'se_col': 'SE_lnCVR',
                    'ci_lower_col': 'CI_lower_lnCVR', 'ci_upper_col': 'CI_upper_lnCVR',
                    'effect_label': 'log CV Ratio', 'effect_label_short': 'lnCVR',
                    'type': 'lnCVR', 'has_fold_change': True, 'null_value': 0, 'scale': 'log', 'allows_negative': True
                },
                'risk_diff': {
                    'effect_col': 'RD', 'var_col': 'var_RD', 'se_col': 'SE_RD',
                    'ci_lower_col': 'CI_lower_RD', 'ci_upper_col': 'CI_upper_RD',
                    'effect_label': 'Risk Difference', 'effect_label_short': 'RD',
                    'type': 'risk_diff', 'has_fold_change': False, 'null_value': 0, 'scale': 'raw', 'allows_negative': True
                },
                'proportion': {
                    'effect_col': 'logit_p', 'var_col': 'var_logit', 'se_col': 'SE_logit',
                    'ci_lower_col': 'CI_lower_logit', 'ci_upper_col': 'CI_upper_logit',
                    'effect_label': 'Logit Proportion', 'effect_label_short': 'Logit',
                    'type': 'proportion', 'has_fold_change': False, 'null_value': 0, 'scale': 'logit', 'allows_negative': True
                }
            }


            # Initialize ANALYSIS_CONFIG if it doesn't exist (pre-calculated mode may skip Cell 3)
            if 'ANALYSIS_CONFIG' not in globals():
                global ANALYSIS_CONFIG
                ANALYSIS_CONFIG = {
                    'prefilter_col': 'None',
                    'prefilter_values': [],
                    'factor1': 'None',
                    'factor2': 'None',
                    'min_papers': 1,
                    'min_obs': 1
                }

            ANALYSIS_CONFIG['effect_size_type'] = sel
            ANALYSIS_CONFIG['es_config'] = es_configs[sel]
            ANALYSIS_CONFIG['data_type'] = 'pre_calculated'  # Store mode
            ANALYSIS_CONFIG['variance_type'] = globals().get('VARIANCE_TYPE_SELECTED', 'variance')

            # Pre-set global vars
            ANALYSIS_CONFIG['effect_col'] = es_configs[sel]['effect_col']
            ANALYSIS_CONFIG['var_col'] = es_configs[sel]['var_col']
            ANALYSIS_CONFIG['se_col'] = es_configs[sel]['se_col']

            display(HTML(f"""
            <div style='padding: 15px; background-color: #d4edda; border-left: 5px solid #28a745; border-radius: 4px; font-family: sans-serif; margin-top: 15px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);'>
                <h4 style='margin: 0 0 5px 0; color: #155724; display: flex; align-items: center;'>
                    <span style='margin-right: 8px;'>✅</span> Configuration Saved
                </h4>
                <p style='margin: 0; font-size: 14px; color: #155724;'>
                    Mapped Effect Size Metric: <b>{ES_TYPE_INFO[sel]['label']}</b>.<br><br>
                    <b>Next Step:</b> Proceed to Step 6 (Effect Size Calculation).
                </p>
            </div>
            """))

    confirm_button.on_click(on_confirm_es_type)

    # Display UI

    if ANALYSIS_CONFIG.get('is_reproducing', False):
        es = ANALYSIS_CONFIG.get('effect_size_type', 'None')
        display(HTML(f"<div style='background:#e8f4f8; border-left:4px solid #17a2b8; padding:12px; margin-bottom:10px; border-radius:4px; color:#0c5460;'>🔄 <b>Reproducibility Mode Active</b> &nbsp;|&nbsp; Restoring Effect Size: <b>{es}</b></div>"))
    display(widgets.VBox([
        es_type_dropdown,
        info_panel,
        confirm_button,
        confirm_output
    ], layout=widgets.Layout(padding='10px')))

    # =================================================================
    # REPRODUCIBILITY MODE (CELL 5 - PRECALC): Auto-populate and Execute
    # =================================================================
    if 'ANALYSIS_CONFIG' not in globals():
        ANALYSIS_CONFIG = {}
    if ANALYSIS_CONFIG.get('is_reproducing', False):
        saved_es = ANALYSIS_CONFIG.get('effect_size_type')

        # Hardcode valid keys to avoid widget option parsing issues
        valid_options = ['lnRR', 'hedges_g', 'cohen_d', 'log_or', 'log_rr', 'fisher_z', 'MD', 'lnCVR', 'risk_diff', 'proportion']

        if saved_es and saved_es in valid_options:
            # Temporarily unobserve to prevent racing UI updates
            try:
                es_type_dropdown.unobserve(update_es_info, names='value')
            except ValueError:
                pass

            es_type_dropdown.value = saved_es
            es_type_dropdown.observe(update_es_info, names='value')

            # Manually trigger the UI update for the new value
            update_es_info({'new': saved_es})

        # Programmatically trigger the confirmation
        on_confirm_es_type(None)

In [ ]:
#@title 🧮 6. Effect Size Calculation
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")


# =============================================================================
# STEP 6: EFFECT SIZE CALCULATION
# Purpose: Calculate effect sizes (lnRR, g, d, logOR, ...), variances, weights.
# Logic:   Exact Hedges' J via log-gamma; large-sample variances as in
#          metafor::escalc. Imputed SDs are treated as known (their
#          uncertainty is not propagated); every excluded row is logged.
# =============================================================================

# Critical value for 95% normal-approximation CIs ------------------
Z_95 = stats.norm.ppf(0.975)   # 1.959964...


# =============================================================================
# STANDARD DEVIATION IMPUTATION
# =============================================================================

def impute_sd_median_cv(df, sd_col, mean_col):
    """Impute missing SDs using the median coefficient of variation (SD/|Mean|).
    Scale-respecting: imputes a RATIO, never an absolute SD."""
    result = df[sd_col].copy()
    missing_mask = result.isna()
    if not missing_mask.any():
        return result

    valid = (df[sd_col] > 0) & df[mean_col].notna() & (df[mean_col] != 0)
    if not valid.any():
        warnings.warn(
            f"impute_sd_median_cv: no study has a usable SD/mean pair in "
            f"'{sd_col}'/'{mean_col}'; cannot impute. Missing SDs left as NaN "
            f"(rows will be dropped downstream).", stacklevel=2)
        return result
    median_cv = (df.loc[valid, sd_col] / df.loc[valid, mean_col].abs()).median()

    for idx in df[missing_mask].index:
        mean_val = df.loc[idx, mean_col]
        if pd.notna(mean_val) and mean_val != 0:
            result.loc[idx] = abs(mean_val) * median_cv
    return result


def calculate_pooled_cv(df, sd_col, mean_col, n_col):
    """
    Degrees-of-freedom-weighted pooled coefficient of variation.

    Computed in CV^2 space (NOT as a mean of raw CVs) to avoid the
    Jensen's-inequality bias of averaging ratios, with a normal-theory
    small-sample correction applied to each study's CV before pooling
    (CV* = CV * (1 + 1/(4n)); Sokal & Rohlf). Studies are weighted by their
    degrees of freedom (n_i - 1). This follows the *spirit* of Nakagawa et al.
    (2023, Ecology Letters): a pooled CV is more stable than borrowing
    individual noisy SDs.

    NOTE: this is a pooled-CV estimator, not the full multiple-imputation
    procedure; imputed SDs are subsequently treated as known.

    Returns
    -------
    float or None : pooled CV; None if no study has a usable
                    (SD>0, mean!=0, n>1) record so the caller can refuse to
                    impute rather than silently invent a value.
    """
    valid = ((df[sd_col] > 0) & df[mean_col].notna()
             & (df[mean_col] != 0) & (df[n_col] > 1))
    if not valid.any():
        return None

    n_i = df.loc[valid, n_col].astype(float)
    cv_raw = (df.loc[valid, sd_col] / df.loc[valid, mean_col].abs()).astype(float)
    cv_corr = cv_raw * (1.0 + 1.0 / (4.0 * n_i))     # small-sample correction
    w = n_i - 1.0                                    # df weighting
    pooled_cv2 = np.sum(w * cv_corr ** 2) / np.sum(w)
    return float(np.sqrt(pooled_cv2))


def zeros_to_nan(series):
    """Convert zero SDs to NaN so they are handled by the missing-SD logic.
    A zero SD is treated as a *missing* SD, never as the smallest
    variance / largest weight in the dataset."""
    out = series.copy()
    out[out == 0] = np.nan
    return out


# =============================================================================
# HEDGES' J (exact, log-gamma) — used by both the engine and the VCV builder
# =============================================================================

def _hedges_j(df_val):
    """Exact Hedges' J small-sample correction factor via log-gamma.
    Numerically stable for ALL sample sizes."""
    if df_val <= 1:
        warnings.warn(
            f"Hedges' J undefined for df={df_val} (need df>1). "
            f"Returning J=1.0; the underlying effect size is unreliable.")
        return 1.0
    log_j = (gammaln(df_val / 2.0)
             - 0.5 * np.log(df_val / 2.0)
             - gammaln((df_val - 1.0) / 2.0))
    return np.exp(log_j)


# =============================================================================
# VCV MATRIX CONSTRUCTION
# =============================================================================

def build_vcv_matrices(df, effect_type, var_col_name,
                       yi_col='yi', nt_col='ne', nc_col='nc',
                       xc_col='xc', sdc_col='sdc',
                       events_c_col=None, nonevents_c_col=None,
                       check_psd=True, warn_on_fallback=True):
    """
    Construct variance-covariance (VCV) matrices for studies with shared
    control groups (Gleser & Olkin 2009; Lajeunesse 2011; Nakagawa et al. 2015;
    Bagos 2009). Each shared_group_id denotes effect sizes sharing an IDENTICAL
    control group. Partial overlaps are not supported (warn if nc varies).

    NOTE: the diagonal uses var_col_name, which the calling pipeline
    always recomputes from the (possibly imputed) SDs; off-diagonal control
    terms use sdc_imputed when present. Diagonal and off-diagonal are therefore
    consistent by construction, so the old heuristic 'imputed-naming' warning
    has been removed.
    """
    # Normalise effect_type -----------------------------------------------------
    effect_aliases = {
        'lnrr': 'lnRR', 'logresponseratio': 'lnRR', 'lrr_log': 'lnRR',
        'logor': 'log_or', 'lor': 'log_or', 'log_oddsratio': 'log_or',
        'logrr': 'log_rr', 'logriskratio': 'log_rr',
        'md': 'MD', 'meandifference': 'MD', 'raw_md': 'MD',
        'lncvr': 'lnCVR', 'logcvr': 'lnCVR',
        'riskdiff': 'risk_diff', 'rd': 'risk_diff',
        'hedgesg': 'hedges_g', 'g': 'hedges_g',
        'cohend': 'cohen_d', 'd': 'cohen_d',
    }
    effect_type_norm = effect_type.lower().replace('_', '').replace(' ', '')
    effect_type = effect_aliases.get(effect_type_norm, effect_type)

    valid_effect_types = {'hedges_g', 'cohen_d', 'lnRR', 'log_or',
                          'log_rr', 'MD', 'lnCVR', 'risk_diff'}
    # Effect sizes without a control group (single-group proportions,
    # correlations) cannot share one: their sampling VCV is diagonal by design.
    no_control_types = {'proportion', 'fisher_z'}
    if effect_type in no_control_types:
        return {sid: np.diag(grp[var_col_name].values.astype(float))
                for sid, grp in df.groupby('id')}
    if effect_type not in valid_effect_types:
        warnings.warn(
            f"Unknown effect_type '{effect_type}'. Valid: "
            f"{sorted(valid_effect_types)}. Returning diagonal VCV matrices.",
            stacklevel=2)
        return {sid: np.diag(grp[var_col_name].values.astype(float))
                for sid, grp in df.groupby('id')}

    vcv_matrices = {}

    # Validate required columns -------------------------------------------------
    required = {'id', var_col_name}
    if effect_type in ('hedges_g', 'cohen_d'):
        if {yi_col, nt_col, nc_col}.issubset(df.columns):
            required |= {yi_col, nt_col, nc_col}
    elif effect_type == 'lnRR':
        if {nc_col, xc_col}.issubset(df.columns):
            required |= {nc_col, xc_col}
    elif effect_type in ('log_or', 'log_rr'):
        if events_c_col and nonevents_c_col and \
           {events_c_col, nonevents_c_col}.issubset(df.columns):
            required |= {events_c_col, nonevents_c_col}
    elif effect_type == 'MD':
        if {nc_col, xc_col, sdc_col}.issubset(df.columns) or 'sdc_imputed' in df.columns:
            required |= {nc_col}
    elif effect_type == 'lnCVR':
        if {nc_col, xc_col}.issubset(df.columns):
            required |= {nc_col, xc_col}
    elif effect_type == 'risk_diff':
        if events_c_col and nonevents_c_col and \
           {events_c_col, nonevents_c_col}.issubset(df.columns):
            required |= {events_c_col, nonevents_c_col}

    missing = required - set(df.columns)
    if missing:
        if warn_on_fallback:
            warnings.warn(
                f"Cannot compute shared-control covariances for "
                f"effect_type='{effect_type}': missing {missing}. Returning "
                f"DIAGONAL VCV (assumes independence). If your data has shared "
                f"controls, uncertainty will be underestimated.", stacklevel=2)
        for study_id, study_grp in df.groupby('id'):
            vcv_matrices[study_id] = np.diag(
                study_grp[var_col_name].values.astype(float))
        return vcv_matrices

    # Build one matrix per study ------------------------------------------------
    for study_id, study_grp in df.groupby('id'):
        var_vals = study_grp[var_col_name].values.astype(float)
        vcv = np.diag(var_vals)

        if 'shared_group_id' not in study_grp.columns:
            vcv_matrices[study_id] = vcv
            continue

        idx_to_pos = {idx: pos for pos, idx in enumerate(study_grp.index)}
        shared_mask = study_grp['shared_group_id'].notna()

        for shared_gid, shared_rows in study_grp[shared_mask].groupby('shared_group_id'):
            idx_list = list(shared_rows.index)
            pos_list = [idx_to_pos[idx] for idx in idx_list]
            n_shared = len(idx_list)
            if n_shared < 2:
                continue

            if effect_type not in ('log_or', 'log_rr', 'risk_diff'):
                if nc_col not in shared_rows.columns:
                    continue
                nc_values = shared_rows[nc_col].unique()
                if len(nc_values) > 1:
                    warnings.warn(
                        f"Study {study_id}, shared_group_id={shared_gid}: "
                        f"differing nc {nc_values.tolist()}. shared_group_id "
                        f"requires identical controls. Using nc={float(nc_values[0])}.",
                        stacklevel=2)
                nc = float(nc_values[0])
                if nc <= 0:
                    continue
            else:
                nc = None

            sdc_column = ('sdc_imputed' if 'sdc_imputed' in shared_rows.columns
                          else sdc_col)

            def _write_symmetric(cov_value, a, b):
                vcv[pos_list[a], pos_list[b]] = cov_value
                vcv[pos_list[b], pos_list[a]] = cov_value

            # lnRR (Lajeunesse 2011) -------------------------------------------
            if effect_type == 'lnRR' and xc_col in shared_rows.columns:
                xc = float(shared_rows.iloc[0][xc_col])
                sdc = (float(shared_rows.iloc[0][sdc_column])
                       if sdc_column in shared_rows.columns else 0.0)
                cov = (sdc ** 2) / (nc * xc ** 2) if xc != 0 else 0.0
                for a in range(n_shared):
                    for b in range(a + 1, n_shared):
                        _write_symmetric(cov, a, b)

            # Cohen's d (Gleser & Olkin 2009) ----------------------------------
            elif effect_type == 'cohen_d' and yi_col in study_grp.columns:
                d_vec = study_grp.loc[idx_list, yi_col].values.astype(float)
                nt_vec = study_grp.loc[idx_list, nt_col].values.astype(float)
                N_vec = nt_vec + nc
                outer_d = np.outer(d_vec, d_vec)
                outer_N = np.outer(N_vec, N_vec)
                cov_mat = (1.0 / nc) + (outer_d * nc) / (2.0 * outer_N)
                for a in range(n_shared):
                    for b in range(a + 1, n_shared):
                        _write_symmetric(cov_mat[a, b], a, b)

            # Hedges' g --------------------------------------------------------
            # Same form as Cohen's d, written in g. This matches the diagonal,
            # which uses the escalc("SMD") variance 1/n1 + 1/n2 + g²/(2N)
            # (no J² factor); scaling only the covariances by J_i*J_j would
            # mix two conventions. At g = 0 this equals metafor::vcalc (1/nc).
            elif effect_type == 'hedges_g' and yi_col in study_grp.columns:
                g_vec = study_grp.loc[idx_list, yi_col].values.astype(float)
                nt_vec = study_grp.loc[idx_list, nt_col].values.astype(float)
                N_vec = nt_vec + nc
                outer_g = np.outer(g_vec, g_vec)
                outer_N = np.outer(N_vec, N_vec)
                cov_g_mat = (1.0 / nc) + (outer_g * nc) / (2.0 * outer_N)
                for a in range(n_shared):
                    for b in range(a + 1, n_shared):
                        _write_symmetric(cov_g_mat[a, b], a, b)

            # Log odds ratio (Bagos 2009) --------------------------------------
            elif effect_type == 'log_or' and events_c_col and nonevents_c_col:
                c_c = float(shared_rows.iloc[0][events_c_col])
                d_c = float(shared_rows.iloc[0][nonevents_c_col])
                cov = (1.0 / c_c) + (1.0 / d_c) if (c_c > 0 and d_c > 0) else 0.0
                for a in range(n_shared):
                    for b in range(a + 1, n_shared):
                        _write_symmetric(cov, a, b)

            # Mean Difference ---------------------------------------------------
            elif effect_type == 'MD' and xc_col in shared_rows.columns:
                sdc = (float(shared_rows.iloc[0][sdc_column])
                       if sdc_column in shared_rows.columns else 0.0)
                cov = (sdc ** 2) / nc if nc > 0 else 0.0
                for a in range(n_shared):
                    for b in range(a + 1, n_shared):
                        _write_symmetric(cov, a, b)

            # lnCVR (Nakagawa et al. 2015) -------------------------------------
            elif effect_type == 'lnCVR' and xc_col in shared_rows.columns:
                xc = float(shared_rows.iloc[0][xc_col])
                sdc = (float(shared_rows.iloc[0][sdc_column])
                       if sdc_column in shared_rows.columns else 0.0)
                if nc <= 1:
                    warnings.warn(
                        f"Study {study_id}, shared_group_id={shared_gid}: "
                        f"nc={nc} too small for lnCVR covariance. Skipping.",
                        stacklevel=2)
                    cov = 0.0
                elif xc == 0:
                    cov = 0.0
                else:
                    cov = (sdc ** 2 / (nc * xc ** 2)) + (1.0 / (2.0 * (nc - 1)))
                for a in range(n_shared):
                    for b in range(a + 1, n_shared):
                        _write_symmetric(cov, a, b)

            # Risk Difference ---------------------------------------------------
            elif effect_type == 'risk_diff' and events_c_col and nonevents_c_col:
                c_c = float(shared_rows.iloc[0][events_c_col])
                d_c = float(shared_rows.iloc[0][nonevents_c_col])
                n_c_total = c_c + d_c
                cov = ((c_c / n_c_total) * (1.0 - c_c / n_c_total) / n_c_total
                       if n_c_total > 0 else 0.0)
                for a in range(n_shared):
                    for b in range(a + 1, n_shared):
                        _write_symmetric(cov, a, b)

            # Log risk ratio ----------------------------------------------------
            elif effect_type == 'log_rr' and events_c_col and nonevents_c_col:
                c_c = float(shared_rows.iloc[0][events_c_col])
                d_c = float(shared_rows.iloc[0][nonevents_c_col])
                n_c_total = c_c + d_c
                cov = ((1.0 / c_c) - (1.0 / n_c_total)
                       if (c_c > 0 and n_c_total > 0) else 0.0)
                for a in range(n_shared):
                    for b in range(a + 1, n_shared):
                        _write_symmetric(cov, a, b)

        # Positive semi-definiteness check -------------------------------------
        if check_psd and vcv.shape[0] > 1:
            try:
                min_eig = np.linalg.eigvalsh(vcv).min()
                if min_eig < -1e-8:
                    warnings.warn(
                        f"Study {study_id}: VCV has negative eigenvalue "
                        f"({min_eig:.2e}). Off-diagonal covariances may be too "
                        f"large; check sdc/nc or imputation.", stacklevel=2)
            except np.linalg.LinAlgError:
                warnings.warn(f"Study {study_id}: VCV eigendecomposition failed.",
                              stacklevel=2)

        vcv_matrices[study_id] = vcv

    return vcv_matrices




# --- 1. SETUP TABS -----------------------------------------------------------
tab_summary = widgets.Output()
tab_diag = widgets.Output()
tab_stats = widgets.Output()
tab_interp = widgets.Output()
tab_sens = widgets.Output()
tab_removed = widgets.Output()

tabs = widgets.Tab(children=[tab_summary, tab_diag, tab_stats,
                             tab_interp, tab_sens, tab_removed])
tabs.set_title(0, '📊 Summary')
tabs.set_title(1, '📉 Diagnostics')
tabs.set_title(2, '📏 Detailed Stats')
tabs.set_title(3, '🧠 Interpretation')
tabs.set_title(4, '⚖️ Sensitivity')
tabs.set_title(5, '🗑️ Removed Data')


# --- 2. CALCULATION ENGINE ---------------------------------------------------
def _apply_common_row_set(df, removed_records, log_diag):
    """Common row set (chosen in Step 3): drop rows with a missing value in
    any selected moderator. Applied after the effect sizes are computed and
    before the sampling VCVs are built, so every analysis uses the same rows.
    Dropped rows go to the exclusion log with their reason."""
    cols = list(ANALYSIS_CONFIG.get('complete_case_moderators') or [])
    if not cols:
        ANALYSIS_CONFIG['complete_case_log'] = {}
        return df
    absent = [c for c in cols if c not in df.columns]
    cols = [c for c in cols if c in df.columns]
    n_before = len(df)
    studies_before = set(df['id'].dropna().unique()) if 'id' in df.columns else set()
    per_col = {}
    for c in cols:
        s = df[c]
        miss = s.isna() | s.astype(str).str.strip().isin(['', 'nan', 'NaN', 'None', '<NA>'])
        per_col[str(c)] = int(miss.sum())
        if miss.any():
            dropped = df[miss].copy()
            dropped['Reason'] = f"Common row set: missing value in '{c}'"
            removed_records.append(dropped)
            df = df[~miss].copy()
    studies_after = set(df['id'].dropna().unique()) if 'id' in df.columns else set()
    ANALYSIS_CONFIG['complete_case_log'] = {
        'columns': [str(c) for c in cols], 'absent_columns': [str(c) for c in absent],
        'n_before': int(n_before), 'n_dropped': int(n_before - len(df)), 'n_after': int(len(df)),
        'dropped_by_column': per_col,
        'studies_lost': sorted(map(str, studies_before - studies_after)),
    }
    log_diag.append(f"• Common row set (complete cases on {', '.join(map(str, cols))}): "
                    f"removed {n_before - len(df)} row(s)")
    if absent:
        log_diag.append(f"• ⚠️ Common-row-set moderator(s) not in the data: {', '.join(map(str, absent))}")
    return df


SD_BASED_EFFECTS = ('lnRR', 'hedges_g', 'cohen_d', 'MD', 'lnCVR')


def _sd_based_effect_sizes(df, effect_size_type, effect_col, var_col, se_col,
                           bias_correction=True):
    """Effect sizes and sampling variances that depend on SDs, computed
    from the columns 'sde_imputed' / 'sdc_imputed'. Used by Step 6 and by
    the complete-case arms of the sensitivity checks (Steps 6 and 7), so the
    formulas exist only once. Returns a copy of df."""
    df = df.copy()
    if effect_size_type in ('lnRR', 'lnCVR'):
        # Second-order term of the log ratio of means (Lajeunesse 2015)
        lnrr_bias = 0.5 * (df['sde_imputed']**2 / (df['ne'] * df['xe']**2)
                           - df['sdc_imputed']**2 / (df['nc'] * df['xc']**2))

    if effect_size_type == 'lnRR':
        df[effect_col] = np.log(df['xe'] / df['xc'])
        if bias_correction:
            df[effect_col] = df[effect_col] + lnrr_bias
        df[var_col] = (df['sde_imputed']**2 / (df['ne'] * df['xe']**2)
                       + df['sdc_imputed']**2 / (df['nc'] * df['xc']**2))
        df[se_col] = np.sqrt(df[var_col])
        df['Response_Ratio'] = np.exp(df[effect_col])
        # vectorised
        df['fold_change'] = np.where(df[effect_col] >= 0,
                                     df['Response_Ratio'],
                                     -1.0 / df['Response_Ratio'])
        df['Percent_Change'] = (df['Response_Ratio'] - 1) * 100

    elif effect_size_type == 'hedges_g':
        df['df'] = df['ne'] + df['nc'] - 2
        df['sp'] = np.sqrt(((df['ne']-1)*df['sde_imputed']**2
                            + (df['nc']-1)*df['sdc_imputed']**2) / df['df'])
        df['d'] = (df['xe'] - df['xc']) / df['sp']
        # exact log-gamma J (no overflow / NaN on large studies)
        df['J'] = df['df'].apply(_hedges_j)
        df[effect_col] = df['d'] * df['J']
        df[var_col] = ((1/df['ne']) + (1/df['nc'])
                       + (df[effect_col]**2 / (2*(df['ne'] + df['nc']))))
        df[se_col] = np.sqrt(df[var_col])

    elif effect_size_type == 'cohen_d':
        df['df'] = df['ne'] + df['nc'] - 2
        df['sp'] = np.sqrt(((df['ne']-1)*df['sde_imputed']**2
                            + (df['nc']-1)*df['sdc_imputed']**2) / df['df'])
        df[effect_col] = (df['xe'] - df['xc']) / df['sp']
        df[var_col] = ((df['ne']+df['nc']) / (df['ne']*df['nc'])
                       + (df[effect_col]**2)/(2*(df['ne']+df['nc'])))
        df[se_col] = np.sqrt(df[var_col])

    elif effect_size_type == 'MD':
        df[effect_col] = df['xe'] - df['xc']
        df[var_col] = (df['sde_imputed']**2 / df['ne']
                       + df['sdc_imputed']**2 / df['nc'])
        df[se_col] = np.sqrt(df[var_col])

    elif effect_size_type == 'lnCVR':
        # independence simplification: the mean-SD covariance term of
        # Nakagawa et al. (2015) is omitted (assumes mean and SD uncorrelated).
        cvr_e = np.log(df['sde_imputed'] / df['xe'].abs())
        cvr_c = np.log(df['sdc_imputed'] / df['xc'].abs())
        df[effect_col] = cvr_e - cvr_c
        if bias_correction:
            # ln(SD) small-sample terms (Nakagawa et al. 2015) and the
            # second-order term of the means, as metafor's escalc("CVR")
            df[effect_col] = (df[effect_col] + 1 / (2*(df['ne']-1)) - 1 / (2*(df['nc']-1))
                              - lnrr_bias)
        var_e = (df['sde_imputed']**2 / (df['ne']*df['xe']**2)
                 + 1/(2*(df['ne']-1)))
        var_c = (df['sdc_imputed']**2 / (df['nc']*df['xc']**2)
                 + 1/(2*(df['nc']-1)))
        df[var_col] = var_e + var_c
        df[se_col] = np.sqrt(df[var_col])
    return df


def observed_sd_effect_sizes(df, effect_size_type, effect_col, var_col, se_col,
                             bias_correction=True):
    """Complete-case effect sizes: recompute the SD-based effect sizes AND
    their variances from the OBSERVED SDs ('sde' / 'sdc'), ignoring any
    imputed values. Needed when the SD strategy replaced observed SDs (the
    all-cases pooled CV), where the stored yi and vi of complete rows are
    themselves based on imputed SDs."""
    out = df.copy()
    out['sde_imputed'] = out['sde']
    out['sdc_imputed'] = out['sdc']
    return _sd_based_effect_sizes(out, effect_size_type, effect_col, var_col, se_col,
                                  bias_correction)


def run_calculation():
    log_diag = []
    log_summary = []
    # A run that stops early must not leave the previous run's effect sizes
    # in place for Step 7 and later steps.
    for _k in ('analysis_data', 'vcv_matrices', 'overall_results'):
        ANALYSIS_CONFIG.pop(_k, None)

    # --- CONFIG CHECK ---
    try:
        if 'ANALYSIS_CONFIG' not in globals():
            print("❌ ERROR: ANALYSIS_CONFIG not found. Run config cell first.")
            return
        effect_size_type = ANALYSIS_CONFIG['effect_size_type']
        es_config = ANALYSIS_CONFIG['es_config']
        log_summary.append(f"Configuration: {es_config['effect_label']} "
                           f"({es_config['effect_label_short']})")
    except KeyError as e:
        print(f"❌ ERROR: Configuration keys missing: {e}")
        return

    if 'data_filtered' not in globals():
        print("❌ ERROR: Data not found. Run filtering cell first.")
        return

    df = data_filtered.copy()
    initial_obs = len(df)
    removed_records = []          # single source of truth

    # --- VALIDATION ---
    if effect_size_type in ['log_or', 'log_rr', 'risk_diff']:
        req_cols = ['events_e', 'nonevents_e', 'events_c', 'nonevents_c']
    elif effect_size_type == 'proportion':
        req_cols = ['events_e', 'nonevents_e']          # single-group prevalence
    else:
        req_cols = ['xe', 'sde', 'ne', 'xc', 'sdc', 'nc']

    missing = [c for c in req_cols if c not in df.columns]
    if missing:
        print(f"❌ ERROR: Missing columns: {missing}. "
              f"Do your data match the selected effect size?")
        return

    # --- SD IMPUTATION -------------------------------------------------------
    log_diag.append("<b>1. Standard Deviation Imputation</b>")
    sd_missing_strategy = ANALYSIS_CONFIG.get('sd_missing_strategy', 'drop')
    sd_zero_strategy = ANALYSIS_CONFIG.get('sd_zero_strategy', 'drop')

    if 'sde' in df.columns and 'sdc' in df.columns:
        df['sde_imputed'] = df['sde'].copy()
        df['sdc_imputed'] = df['sdc'].copy()

        n_zero = (df['sde'] == 0).sum() + (df['sdc'] == 0).sum()

        # Handle zeros -------------------------------------------
        if n_zero > 0:
            if sd_zero_strategy == 'treat_as_missing':
                df['sde_imputed'] = zeros_to_nan(df['sde_imputed'])
                df['sdc_imputed'] = zeros_to_nan(df['sdc_imputed'])
                log_diag.append(f"• Converted {int(n_zero)} zero SDs to missing "
                                f"(to be imputed via CV alongside missing SDs).")
            elif sd_zero_strategy == 'drop':
                zero_mask = (df['sde_imputed'] == 0) | (df['sdc_imputed'] == 0)
                dropped = df[zero_mask].copy()
                dropped['Reason'] = 'Zero SD reported'
                removed_records.append(dropped)
                df = df[~zero_mask].copy()
                log_diag.append(f"• Dropped {int(zero_mask.sum())} rows with zero SDs.")

        # Recompute missing count AFTER zero->NaN conversion
        n_missing = int(df['sde_imputed'].isna().sum()
                        + df['sdc_imputed'].isna().sum())

        if n_missing > 0:
            # Rows left with a missing SD are dropped AND logged below, with
            # this reason (no silent dropna in the branches).
            missing_reason = 'SD missing and not imputable'
            if sd_missing_strategy == 'nakagawa_all_cases':
                cv_e = calculate_pooled_cv(df, 'sde_imputed', 'xe', 'ne')
                cv_c = calculate_pooled_cv(df, 'sdc_imputed', 'xc', 'nc')
                if cv_e is None or cv_c is None:
                    log_diag.append("• ⚠️ No usable CVs; cannot apply all-cases. "
                                    "Falling back to dropping missing rows.")
                    missing_reason = 'SD missing (no usable CVs to impute from)'
                else:
                    df['sde_imputed'] = df['xe'].abs() * cv_e
                    df['sdc_imputed'] = df['xc'].abs() * cv_c
                    log_diag.append(
                        f"• Nakagawa-style ALL-CASES pooled CV "
                        f"(Exp {cv_e:.3f}, Ctl {cv_c:.3f}) applied to ALL rows. "
                        f"<b>Observed SDs were replaced.</b>")
            elif sd_missing_strategy == 'nakagawa_missing_cases':
                cv_e = calculate_pooled_cv(df, 'sde_imputed', 'xe', 'ne')
                cv_c = calculate_pooled_cv(df, 'sdc_imputed', 'xc', 'nc')
                if cv_e is None or cv_c is None:
                    log_diag.append("• ⚠️ No usable CVs; dropping missing rows.")
                    missing_reason = 'SD missing (no usable CVs to impute from)'
                else:
                    df['sde_imputed'] = df['sde_imputed'].fillna(df['xe'].abs() * cv_e)
                    df['sdc_imputed'] = df['sdc_imputed'].fillna(df['xc'].abs() * cv_c)
                    log_diag.append(
                        f"• Nakagawa-style pooled CV (Exp {cv_e:.3f}, "
                        f"Ctl {cv_c:.3f}) applied to MISSING rows only.")
            elif sd_missing_strategy == 'median_cv':
                df['sde_imputed'] = impute_sd_median_cv(df, 'sde_imputed', 'xe')
                df['sdc_imputed'] = impute_sd_median_cv(df, 'sdc_imputed', 'xc')
                log_diag.append("• Imputed missing SDs using the median CV.")
            elif sd_missing_strategy == 'drop':
                missing_reason = "SD missing ('drop' strategy)"

            # Any rows still missing after imputation are dropped + logged
            still_missing = df['sde_imputed'].isna() | df['sdc_imputed'].isna()
            if still_missing.any():
                dropped = df[still_missing].copy()
                dropped['Reason'] = missing_reason
                removed_records.append(dropped)
                df = df[~still_missing].copy()
                log_diag.append(f"• Dropped {int(still_missing.sum())} rows: {missing_reason}.")

            # honesty caveat
            log_diag.append("• <i>Caveat: imputed SDs are treated as KNOWN; "
                            "their sampling uncertainty is not propagated, so "
                            "imputed rows are slightly over-weighted.</i>")

    # --- CLEANING (negative / zero values) -----------------------------------
    log_diag.append("<br><b>2. Data Cleaning</b>")

    if effect_size_type in ('lnRR', 'lnCVR'):
        df = df.copy()
        neg_mask = (df['xe'] < 0) | (df['xc'] < 0)
        if neg_mask.sum() > 0:
            dropped = df[neg_mask].copy()
            dropped['Reason'] = f'Negative mean (invalid for {effect_size_type})'
            removed_records.append(dropped)
            df = df[~neg_mask].copy()
            log_diag.append(f"• Removed {int(neg_mask.sum())} rows with negative means.")

        # A zero mean makes the log response ratio undefined; the row is
        # excluded and logged (metafor's escalc returns NA), rather than
        # shifting the data by an arbitrary offset.
        zero_mask = (df['xe'] == 0) | (df['xc'] == 0)
        if zero_mask.sum() > 0:
            dropped = df[zero_mask].copy()
            dropped['Reason'] = f'Zero mean (log ratio undefined for {effect_size_type})'
            removed_records.append(dropped)
            df = df[~zero_mask].copy()
            log_diag.append(f"• Removed {int(zero_mask.sum())} rows with a zero mean "
                            f"(log ratio undefined).")

    elif effect_size_type in ['log_or', 'log_rr']:
        binary_cols = ['events_e', 'nonevents_e', 'events_c', 'nonevents_c']
        df[binary_cols] = df[binary_cols].astype(float)

        double_zero_mask = (df['events_e'] == 0) & (df['events_c'] == 0)
        if double_zero_mask.sum() > 0:
            dropped_dz = df[double_zero_mask].copy()
            dropped_dz['Reason'] = 'Double-zero event count (uninformative)'
            removed_records.append(dropped_dz)
            df = df[~double_zero_mask].copy()
            log_diag.append(f"• Dropped {int(double_zero_mask.sum())} double-zero studies.")

        zero_mask = (df[binary_cols] == 0).any(axis=1)
        if zero_mask.sum() > 0:
            df.loc[zero_mask, binary_cols] = df.loc[zero_mask, binary_cols] + 0.5
            log_diag.append(f"• Haldane-Anscombe (+0.5) applied to "
                            f"{int(zero_mask.sum())} single-zero studies "
                            f"(continuity correction; applied to zero-cell "
                            f"studies only).")

    elif effect_size_type == 'risk_diff':
        # As metafor's escalc("RD") default (add = 1/2, to = "only0"): rows
        # with a zero cell get +0.5 in all four cells; double-zero studies
        # are kept (they are informative for the risk difference).
        binary_cols = ['events_e', 'nonevents_e', 'events_c', 'nonevents_c']
        df[binary_cols] = df[binary_cols].astype(float)
        zero_mask = (df[binary_cols] == 0).any(axis=1)
        if zero_mask.sum() > 0:
            df.loc[zero_mask, binary_cols] = df.loc[zero_mask, binary_cols] + 0.5
            log_diag.append(f"• Continuity correction (+0.5 to all cells) applied to "
                            f"{int(zero_mask.sum())} studies with a zero cell.")

    # --- CALCULATION ---------------------------------------------------------
    effect_col = es_config['effect_col']
    var_col = es_config['var_col']
    se_col = es_config['se_col']

    # Small-sample bias correction for lnRR / lnCVR (set in Step 5; default on),
    # identical to metafor's escalc(correct = TRUE / FALSE).
    bias_correction = bool(ANALYSIS_CONFIG.get('bias_correction', True))
    if effect_size_type in ('lnRR', 'lnCVR'):
        log_diag.append("• Small-sample bias correction: "
                        + ("<b>on</b> (second-order; Lajeunesse 2015; metafor's default)."
                           if bias_correction else "<b>off</b> (<code>correct = FALSE</code>)."))


    if effect_size_type in SD_BASED_EFFECTS:
        df = _sd_based_effect_sizes(df, effect_size_type, effect_col, var_col, se_col,
                                    bias_correction)
        if effect_size_type == 'lnCVR':
            log_diag.append("• <i>lnCVR variance uses the independence form "
                            "(mean-SD covariance term omitted).</i>")

    elif effect_size_type == 'log_or':
        a, b = df['events_e'].astype(float), df['nonevents_e'].astype(float)
        c, d = df['events_c'].astype(float), df['nonevents_c'].astype(float)
        df[effect_col] = np.log((a * d) / (b * c))
        df[var_col] = (1/a) + (1/b) + (1/c) + (1/d)
        df[se_col] = np.sqrt(df[var_col])
        df['Odds_Ratio'] = np.exp(df[effect_col])
        df['fold_change'] = df['Odds_Ratio']

    elif effect_size_type == 'log_rr':
        a, b = df['events_e'].astype(float), df['nonevents_e'].astype(float)
        c, d = df['events_c'].astype(float), df['nonevents_c'].astype(float)
        df[effect_col] = np.log((a / (a + b)) / (c / (c + d)))
        df[var_col] = (1/a) - (1/(a + b)) + (1/c) - (1/(c + d))
        df[se_col] = np.sqrt(df[var_col])
        df['Risk_Ratio'] = np.exp(df[effect_col])
        df['fold_change'] = df['Risk_Ratio']

    elif effect_size_type == 'risk_diff':
        a, b = df['events_e'].astype(float), df['nonevents_e'].astype(float)
        c, d = df['events_c'].astype(float), df['nonevents_c'].astype(float)
        p_e, p_c = a / (a + b), c / (c + d)
        df[effect_col] = p_e - p_c
        df[var_col] = (p_e*(1-p_e)/(a+b)) + (p_c*(1-p_c)/(c+d))
        df[se_col] = np.sqrt(df[var_col])

    elif effect_size_type == 'proportion':
        e = df['events_e'].astype(float)
        ne = df['nonevents_e'].astype(float)
        # As escalc("PLO"): a zero in either cell adds 0.5 to BOTH cells
        zero = (e == 0) | (ne == 0)
        e_adj = np.where(zero, e + 0.5, e)
        ne_adj = np.where(zero, ne + 0.5, ne)
        df[effect_col] = np.log(e_adj / ne_adj)
        df[var_col] = (1 / e_adj) + (1 / ne_adj)
        df[se_col] = np.sqrt(df[var_col])

    # --- CI & WEIGHTS (Z_95) ---------------------------------------
    ci_lower_col = es_config.get('ci_lower_col', f"CI_lower_{es_config['effect_label_short']}")
    ci_upper_col = es_config.get('ci_upper_col', f"CI_upper_{es_config['effect_label_short']}")
    df[ci_lower_col] = df[effect_col] - Z_95 * df[se_col]
    df[ci_upper_col] = df[effect_col] + Z_95 * df[se_col]
    df['w_fixed'] = 1 / df[var_col]

    if effect_size_type == 'log_or':
        df['OR_CI_lower'] = np.exp(df[ci_lower_col])
        df['OR_CI_upper'] = np.exp(df[ci_upper_col])
    elif effect_size_type == 'log_rr':
        df['RR_CI_lower'] = np.exp(df[ci_lower_col])
        df['RR_CI_upper'] = np.exp(df[ci_upper_col])

    # --- POST-CALC VALIDITY DROPS (all routed through removed_records) -------
    mask_nan = df[[effect_col, var_col]].isna().any(axis=1)
    if mask_nan.sum() > 0:
        dropped_nan = df[mask_nan].copy()
        dropped_nan['Reason'] = 'Calculation failed (NaN effect/variance)'
        removed_records.append(dropped_nan)
        df = df[~mask_nan].copy()

    mask_bad_var = (df[var_col] <= 0) | ~np.isfinite(df[var_col])
    if mask_bad_var.sum() > 0:
        dropped_var = df[mask_bad_var].copy()
        dropped_var['Reason'] = 'Zero / negative / non-finite variance'
        removed_records.append(dropped_var)
        df = df[~mask_bad_var].copy()

    df = _apply_common_row_set(df, removed_records, log_diag)

    # --- UPDATE CONFIG -------------------------------------------------------
    ANALYSIS_CONFIG['analysis_data'] = df
    ANALYSIS_CONFIG['effect_col'] = effect_col
    ANALYSIS_CONFIG['var_col'] = var_col
    ANALYSIS_CONFIG['se_col'] = se_col
    ANALYSIS_CONFIG['ci_lower_col'] = ci_lower_col
    ANALYSIS_CONFIG['ci_upper_col'] = ci_upper_col

    # --- BUILD VCV MATRICES --------------------------------------------------
    vcv_kwargs = dict(yi_col=effect_col)
    if effect_size_type in ('log_or', 'log_rr', 'risk_diff'):
        vcv_kwargs['events_c_col'] = 'events_c'
        vcv_kwargs['nonevents_c_col'] = 'nonevents_c'
    vcv_matrices = build_vcv_matrices(df, effect_size_type, var_col, **vcv_kwargs)
    ANALYSIS_CONFIG['vcv_matrices'] = vcv_matrices

    has_shared = ('shared_group_id' in df.columns)
    n_studies_with_vcv = 0
    if has_shared:
        for k, v in vcv_matrices.items():
            grp = df[df['id'] == k]
            if v.shape[0] > 1 and grp['shared_group_id'].notna().any():
                n_studies_with_vcv += 1
    if n_studies_with_vcv > 0:
        log_diag.append("<br><b>3. VCV Matrices</b>")
        log_diag.append(f"• Built {len(vcv_matrices)} VCV matrices "
                        f"({n_studies_with_vcv} with shared controls).")

    # === POPULATE TABS =======================================================
    _populate_tabs(df, es_config, effect_size_type, effect_col, var_col, se_col,
                   ci_lower_col, ci_upper_col, log_diag, removed_records,
                   initial_obs)


# =============================================================================
# TAB RENDERING (shared by raw mode; see note on pre-calc mode below)
# =============================================================================
def _populate_tabs(df, es_config, effect_size_type, effect_col, var_col, se_col,
                   ci_lower_col, ci_upper_col, log_diag, removed_records,
                   initial_obs):

    # 1. SUMMARY -------------------------------------------------------------
    with tab_summary:
        clear_output()
        final_n = len(df)
        final_papers = df['id'].nunique() if 'id' in df.columns else final_n
        n_removed = sum(len(r) for r in removed_records)
        display(HTML(f"""
        <div style='display:flex; gap:20px; margin-bottom:20px;'>
            <div style='background:#e8f5e9; padding:15px; border-radius:8px; flex:1; text-align:center; border:1px solid #c3e6cb'>
                <h2 style='margin:0; color:#2e7d32;'>{final_n}</h2>
                <p style='margin:0; color:#1b5e20; font-size:12px; text-transform:uppercase;'>Observations</p></div>
            <div style='background:#e3f2fd; padding:15px; border-radius:8px; flex:1; text-align:center; border:1px solid #bbdefb'>
                <h2 style='margin:0; color:#1565c0;'>{final_papers}</h2>
                <p style='margin:0; color:#0d47a1; font-size:12px; text-transform:uppercase;'>Studies</p></div>
            <div style='background:#fff3e0; padding:15px; border-radius:8px; flex:1; text-align:center; border:1px solid #ffe0b2'>
                <h2 style='margin:0; color:#e65100;'>{n_removed}</h2>
                <p style='margin:0; color:#bf360c; font-size:12px; text-transform:uppercase;'>Removed</p></div>
        </div>"""))
        if not df.empty:
            desc = df[effect_col].describe()
            display(HTML(f"""
            <table style='width:100%; border-collapse:collapse; margin-bottom:20px;'>
                <tr style='border-bottom:2px solid #eee;'><th style='text-align:left; padding:8px;'>Statistic</th><th style='text-align:right; padding:8px;'>Value</th></tr>
                <tr><td style='padding:8px;'><b>Mean Effect</b></td><td style='text-align:right; padding:8px;'>{desc['mean']:.4f}</td></tr>
                <tr style='background:#f9f9f9;'><td style='padding:8px;'><b>Median</b></td><td style='text-align:right; padding:8px;'>{desc['50%']:.4f}</td></tr>
                <tr><td style='padding:8px;'><b>Min / Max</b></td><td style='text-align:right; padding:8px;'>{desc['min']:.4f} / {desc['max']:.4f}</td></tr>
                <tr style='background:#f9f9f9;'><td style='padding:8px;'><b>Std. Dev.</b></td><td style='text-align:right; padding:8px;'>{desc['std']:.4f}</td></tr>
            </table>"""))
            display(HTML("<h4 style='color:#2E86AB;'>📊 Distribution of Effect Sizes</h4>"))
            fig, ax = plt.subplots(figsize=(8, 3.5))
            ax.hist(df[effect_col], bins='auto', color='#2E86AB', alpha=0.7,
                    rwidth=0.9, edgecolor='black', linewidth=0.5)
            ax.axvline(desc['mean'], color='#e74c3c', ls='--', lw=2,
                       label=f"Mean: {desc['mean']:.2f}")
            ax.axvline(0, color='black', ls='-', lw=1, alpha=0.5)
            ax.set_xlabel(es_config['effect_label'], fontsize=10, fontweight='bold')
            ax.set_ylabel("Frequency", fontsize=10)
            ax.legend(frameon=False)
            ax.grid(axis='y', ls=':', alpha=0.3)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            plt.tight_layout(); plt.show()
        else:
            print("⚠️ No valid data remaining.")

    # 2. DIAGNOSTICS ---------------------------------------------------------
    with tab_diag:
        clear_output()
        display(HTML("<h4 style='color:#2E86AB;'>🔍 Processing Log</h4>"))
        for line in log_diag:
            display(HTML(f"<div style='margin-left:15px; font-size:13px;'>{line}</div>"))
        if not df.empty:
            q1, q3 = df[effect_col].quantile([0.25, 0.75])
            iqr = q3 - q1
            lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
            outliers = df[(df[effect_col] < lo) | (df[effect_col] > hi)].copy()
            display(HTML("<hr><h4 style='color:#2E86AB;'>⚠️ Outlier Analysis (IQR)</h4>"))
            if len(outliers) > 0:
                display(HTML(f"""
                <div style='background:#fff3cd; border-left:5px solid #ffc107; padding:15px; color:#856404;'>
                    <b>Found {len(outliers)} potential outliers</b><br>
                    <div style='font-size:13px; margin-top:5px;'>
                    Acceptable range: {lo:.3f} to {hi:.3f}<br>
                    Extreme values: {outliers[effect_col].min():.3f} to {outliers[effect_col].max():.3f}<br>
                    <i>Check whether these are data-entry errors or genuine variation.</i></div></div>"""))
                cols = [c for c in ['id', 'year', effect_col, 'xe', 'xc', 'ne', 'nc']
                        if c in outliers.columns]
                display(outliers[cols].sort_values(by=effect_col, key=abs, ascending=False))
            else:
                display(HTML(f"""<div style='padding:15px; background:#d4edda; color:#155724; border:1px solid #c3e6cb;'>
                    <b>✓ No statistical outliers detected.</b> All {len(df)} effects within [{lo:.3f}, {hi:.3f}].</div>"""))

    # 3. DETAILED STATS ------------------------------------------------------
    with tab_stats:
        clear_output()
        if not df.empty:
            es_data = df[effect_col]
            skew, kurt = es_data.skew(), es_data.kurt()
            if len(df) < 5000:
                w_stat, p = stats.shapiro(es_data)
                normality_str = f"W = {w_stat:.3f}, p = {p:.3f}"
                symmetric_ok = p > 0.05
            else:
                normality_str = "N > 5000 (test omitted)"
                symmetric_ok = True
            verdict = "Approximately symmetric" if symmetric_ok else "Departs from normality"
            vcol = "#28a745" if symmetric_ok else "#d39e00"
            display(HTML(f"""<h4 style='color:#2E86AB;'>📏 Descriptive Statistics</h4>
            <table style='width:100%; border-collapse:collapse; font-size:13px;'>
              <tr style='background:#f1f3f5;'><th style='padding:8px; text-align:left;'>Metric</th>
              <th style='padding:8px; text-align:right;'>Effect</th><th style='padding:8px; text-align:right;'>SE</th>
              <th style='padding:8px; text-align:right;'>Variance</th></tr>
              <tr><td style='padding:8px;'><b>Mean</b></td><td style='text-align:right;'>{es_data.mean():.4f}</td><td style='text-align:right;'>{df[se_col].mean():.4f}</td><td style='text-align:right;'>{df[var_col].mean():.4f}</td></tr>
              <tr style='background:#f9f9f9;'><td style='padding:8px;'><b>Median</b></td><td style='text-align:right;'>{es_data.median():.4f}</td><td style='text-align:right;'>{df[se_col].median():.4f}</td><td style='text-align:right;'>{df[var_col].median():.4f}</td></tr>
              <tr><td style='padding:8px;'><b>Std. Dev.</b></td><td style='text-align:right;'>{es_data.std():.4f}</td><td style='text-align:right;'>{df[se_col].std():.4f}</td><td style='text-align:right;'>{df[var_col].std():.4f}</td></tr>
              <tr style='background:#f9f9f9;'><td style='padding:8px;'><b>Skewness</b></td><td style='text-align:right;'>{skew:.3f}</td><td style='text-align:right;'>-</td><td style='text-align:right;'>-</td></tr>
              <tr><td style='padding:8px;'><b>Kurtosis</b></td><td style='text-align:right;'>{kurt:.3f}</td><td style='text-align:right;'>-</td><td style='text-align:right;'>-</td></tr>
            </table>"""))
            # Shapiro reworded — clearly NOT a model-assumption test
            display(HTML(f"""<div style='display:flex; align-items:center; background:#fff; border:1px solid #ddd; border-left:5px solid {vcol}; padding:15px; border-radius:5px; margin:15px 0;'>
                <div style='flex:1;'><h5 style='margin:0; color:#555;'>Shape of the observed effect-size distribution (Shapiro-Wilk)</h5>
                <div style='font-size:13px; margin-top:5px;'>{normality_str}</div>
                <div style='font-size:12px; color:#888; margin-top:6px;'><i>Descriptive only. This tests the spread of the observed effects; it does NOT test the random-effects normality assumption (which concerns the latent random effects, not observed values) and should not be read as a model diagnostic.</i></div></div>
                <div style='font-weight:bold; color:{vcol};'>{verdict}</div></div>"""))
            display(HTML("<h4 style='color:#2E86AB;'>📈 Q-Q Plot</h4>"))
            fig, ax = plt.subplots(figsize=(6, 4))
            stats.probplot(es_data, dist="norm", plot=ax)
            ax.get_lines()[0].set(marker='o', markersize=5.0,
                                  markerfacecolor='#4a90e2', markeredgecolor='white')
            ax.get_lines()[1].set(linewidth=2.0, color='#e74c3c')
            ax.set_title(""); ax.set_xlabel("Theoretical Quantiles")
            ax.set_ylabel("Ordered Effect Sizes"); ax.grid(True, ls=':', alpha=0.3)
            sns.despine(); plt.tight_layout(); plt.show()
        else:
            print("⚠️ No valid data remaining.")

    # 4. INTERPRETATION ------------------------------------------------------
    with tab_interp:
        clear_output()
        if not df.empty:
            null_val = es_config.get('null_value', 0)
            sig_pos = int((df[ci_lower_col] > null_val).sum())
            sig_neg = int((df[ci_upper_col] < null_val).sum())
            non_sig = len(df) - sig_pos - sig_neg
            abs_eff = df[effect_col].abs()

            # magnitude bins
            if effect_size_type in ['hedges_g', 'cohen_d']:
                bins = [-1, 0.2, 0.5, 0.8, 999]
                labels = ['Negligible (<0.2)', 'Small (0.2-0.5)',
                          'Medium (0.5-0.8)', 'Large (>0.8)']
                bench_note = ("Bands are Cohen's conventional benchmarks for "
                              "standardized mean differences.")
            else:
                bins = [-1, 0.1, 0.3, 0.5, 999]
                labels = ['Band 1 (<0.1)', 'Band 2 (0.1-0.3)',
                          'Band 3 (0.3-0.5)', 'Band 4 (>0.5)']
                bench_note = ("⚠️ Bands are ARBITRARY |effect| cut-offs for "
                              "display only — they are NOT Cohen's benchmarks, "
                              "which apply solely to standardized mean "
                              "differences. Do not interpret as small/medium/large.")
            mag_counts = pd.cut(abs_eff, bins=bins, labels=labels).value_counts().sort_index()

            if sig_pos > sig_neg:
                direction_text, dir_color = "More positive", "#28a745"
            elif sig_neg > sig_pos:
                direction_text, dir_color = "More negative", "#dc3545"
            else:
                direction_text, dir_color = "Balanced", "#6c757d"

            display(HTML(f"""
            <div style='background:#fdecea; border-left:5px solid #dc3545; padding:12px; margin-bottom:15px; color:#611a15; font-size:13px;'>
            <b>⚠️ Vote counting — descriptive only.</b> The counts and percentages
            below summarise how many <i>individual</i> studies have CIs excluding
            the null. Vote counting is low-powered and conflates effect size with
            precision; it is <b>not</b> a substitute for the pooled
            random-/fixed-effects estimate computed downstream and should not be
            reported as the primary result.</div>"""))

            display(HTML(f"""
            <div style='display:flex; gap:20px; margin-bottom:20px;'>
                <div style='flex:1; background:#f8f9fa; padding:15px; border-radius:8px; border-top:5px solid {dir_color};'>
                    <h4 style='margin:0; color:#555; font-size:12px; text-transform:uppercase;'>Direction (sig. studies)</h4>
                    <div style='font-size:20px; font-weight:bold; color:{dir_color};'>{direction_text}</div></div>
                <div style='flex:1; background:#f8f9fa; padding:15px; border-radius:8px; border-top:5px solid #17a2b8;'>
                    <h4 style='margin:0; color:#555; font-size:12px; text-transform:uppercase;'>Individually significant</h4>
                    <div style='font-size:20px; font-weight:bold; color:#17a2b8;'>{((sig_pos+sig_neg)/len(df))*100:.1f}%</div></div>
                <div style='flex:1; background:#f8f9fa; padding:15px; border-radius:8px; border-top:5px solid #6610f2;'>
                    <h4 style='margin:0; color:#555; font-size:12px; text-transform:uppercase;'>Most common band</h4>
                    <div style='font-size:20px; font-weight:bold; color:#6610f2;'>{str(mag_counts.idxmax()).split(' ')[0]}</div></div>
            </div>"""))

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
            sizes = [sig_pos, sig_neg, non_sig]
            pl = ['Sig. Positive', 'Sig. Negative', 'Non-Significant']
            colors = ['#28a745', '#dc3545', '#e2e6ea']
            pie_data = [(s, l, c) for s, l, c in zip(sizes, pl, colors) if s > 0]
            if pie_data:
                ax1.pie([x[0] for x in pie_data], labels=[x[1] for x in pie_data],
                        colors=[x[2] for x in pie_data], autopct='%1.1f%%',
                        startangle=90, textprops={'fontsize': 9})
                ax1.set_title("Vote counting (descriptive only)", fontweight='bold', fontsize=11)
                ax1.axis('equal')
            y_pos = np.arange(len(labels))
            ax2.barh(y_pos, mag_counts.values, color='#6610f2', alpha=0.7,
                     edgecolor='black', height=0.6)
            ax2.set_yticks(y_pos); ax2.set_yticklabels(labels)
            ax2.set_xlabel("Number of studies")
            ax2.set_title("Magnitude bands", fontweight='bold', fontsize=11)
            ax2.grid(axis='x', ls='--', alpha=0.3)
            for ax in (ax1, ax2):
                ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            plt.tight_layout(); plt.show()

            display(HTML(f"<div style='font-size:12px; color:#777; margin-top:6px;'>{bench_note}</div>"))

            display(HTML(f"""
            <div style='margin-top:20px; padding:15px; background:#fff; border:1px solid #ddd; border-left:4px solid #2E86AB;'>
                <h4 style='margin-top:0; color:#2E86AB;'>📝 Automated description</h4>
                <p style='font-size:14px; line-height:1.6; color:#333;'>
                    This dataset has <b>{len(df)} observations</b>. Considered individually,
                    <b>{sig_pos} ({sig_pos/len(df):.1%})</b> have CIs entirely above the null and
                    <b>{sig_neg} ({sig_neg/len(df):.1%})</b> entirely below it; the remaining
                    <b>{non_sig} ({non_sig/len(df):.1%})</b> are individually non-significant.
                    <b>These are study-level counts, not a pooled inference.</b> The synthesised
                    estimate, heterogeneity and any moderator effects are produced by the
                    meta-analytic model in the following cells and should be used for interpretation.
                </p></div>"""))

    # 5. REMOVED DATA --------------------------------------------------------
    with tab_removed:
        clear_output()
        if removed_records:
            all_removed = pd.concat(removed_records)
            cols = [c for c in ['id', 'Reason', 'xe', 'xc', 'ne', 'nc',
                                'events_e', 'events_c']
                    if c in all_removed.columns]
            display(HTML(f"<h4 style='color:#c0392b'>🗑️ {len(all_removed)} rows excluded</h4>"))
            display(HTML("""<div style='background:#fff3cd; border-left:5px solid #ffeeba; padding:15px; margin-bottom:15px; color:#856404; font-size:13px;'>
                <b>Reasons logged per row.</b> This count reconciles with the
                'Removed' KPI on the Summary tab.</div>"""))
            display(all_removed[cols])
        else:
            display(HTML("<div style='padding:15px; background:#d4edda; color:#155724;'><b>✅ Clean run:</b> no observations removed.</div>"))


# =============================================================================
# EXECUTION: MODE BRANCHING
# =============================================================================
data_type_mode = ANALYSIS_CONFIG.get('data_type', 'raw')

if data_type_mode == 'raw':
    df_raw = data_filtered.copy()
    es_type = ANALYSIS_CONFIG.get('effect_size_type', '')
    has_sde = 'sde' in df_raw.columns
    n_missing = (df_raw['sde'].isna().sum() + df_raw['sdc'].isna().sum()) if has_sde else 0
    n_zero = ((df_raw['sde'] == 0).sum() + (df_raw['sdc'] == 0).sum()) if has_sde else 0
    requires_sd = es_type in ['lnRR', 'hedges_g', 'cohen_d', 'MD', 'lnCVR']

    if requires_sd and (n_missing > 0 or n_zero > 0):
        display(HTML(f"""
        <div style='background:#fff8e1; border-left:4px solid #ffc107; padding:15px; margin:15px 0; border-radius:4px;'>
            <h4 style='color:#b38700; margin:0 0 8px 0;'>⚠️ Variance Data Action Required</h4>
            <p style='color:#665000; font-size:13px; margin:0;'>
                <b>Effect size:</b> {es_type}<br>
                Found <b>{int(n_missing)} missing</b> and <b>{int(n_zero)} zero</b> SD values.
                Choose how to handle them. All imputation borrows a coefficient of
                variation (a scale-respecting ratio), never an absolute SD.</p>
        </div>"""))

        # defensible menu only; same options for every SD-based effect
        opts_missing = [
            ('Median CV (impute missing only)', 'median_cv'),
            ('Pooled CV — missing cases only (Nakagawa-style)', 'nakagawa_missing_cases'),
            ('Pooled CV — ALL cases (replaces observed SDs!)', 'nakagawa_all_cases'),
            ('Drop rows with missing SDs', 'drop'),
        ]
        dd_missing = widgets.Dropdown(options=opts_missing, description='Missing SDs:',
                                      layout={'width': '480px'},
                                      style={'description_width': '120px'})
        dd_zero = widgets.Dropdown(
            options=[('Treat zeros as missing → CV impute', 'treat_as_missing'),
                     ('Drop rows with zero SDs', 'drop')],
            description='Zero SDs:', layout={'width': '480px'},
            style={'description_width': '120px'})

        # disclosure that all-cases discards observed SDs
        warn_box = widgets.HTML("")
        def _on_strategy_change(change):
            if change['new'] == 'nakagawa_all_cases':
                warn_box.value = ("<div style='background:#f8d7da; color:#721c24; "
                                  "padding:10px; border-radius:4px; font-size:13px;'>"
                                  "<b>Note:</b> ALL-CASES mode <b>discards every "
                                  "observed SD</b> and replaces it with a CV-derived "
                                  "value. The sensitivity check below compares this "
                                  "against a true complete-case analysis built from "
                                  "your original SDs.</div>")
            else:
                warn_box.value = ""
        dd_missing.observe(_on_strategy_change, names='value')

        btn_calc = widgets.Button(description='▶ Calculate & run sensitivity check',
                                  button_style='success',
                                  layout={'width': '340px', 'height': '40px'})
        out_calc = widgets.Output()

        def on_calc(b):
            with out_calc:
                clear_output()
                display(HTML("<div style='color:#28a745; font-weight:bold;'>⏳ Calculating...</div>"))
                ANALYSIS_CONFIG['sd_missing_strategy'] = dd_missing.value
                ANALYSIS_CONFIG['sd_zero_strategy'] = dd_zero.value
                run_calculation()

                df_main = ANALYSIS_CONFIG.get('analysis_data')
                if df_main is not None and not df_main.empty:
                    null_val = ANALYSIS_CONFIG.get('es_config', {}).get('null_value', 0)
                    ecol = ANALYSIS_CONFIG['effect_col']

                    def calc_fe_stats(eff, w):
                        if len(eff) == 0:
                            return np.nan, np.nan, np.nan, False
                        mu = np.average(eff, weights=w)
                        se = np.sqrt(1.0 / w.sum())
                        ci_l, ci_u = mu - Z_95*se, mu + Z_95*se
                        return mu, ci_l, ci_u, ((ci_l > null_val) or (ci_u < null_val))

                    # Imputed arm (main)
                    main_k = len(df_main)
                    main_mu, main_cil, main_ciu, main_sig = calc_fe_stats(
                        df_main[ecol].values, df_main['w_fixed'].values)

                    # complete-case arm from OBSERVED SDs, recomputed
                    mask_complete = (df_main['sde'].notna() & df_main['sdc'].notna()
                                     & (df_main['sde'] > 0) & (df_main['sdc'] > 0))
                    df_cc = df_main[mask_complete].copy()
                    drop_k = len(df_cc)
                    if drop_k > 0 and ANALYSIS_CONFIG['effect_size_type'] in SD_BASED_EFFECTS:
                        # yi AND vi from the observed SDs (yi of g/d and of
                        # bias-corrected lnRR/lnCVR depends on the SDs too)
                        _vcol = ANALYSIS_CONFIG['var_col']
                        df_cc = observed_sd_effect_sizes(
                            df_cc, ANALYSIS_CONFIG['effect_size_type'], ecol, _vcol,
                            ANALYSIS_CONFIG.get('se_col', 'se_cc'),
                            bool(ANALYSIS_CONFIG.get('bias_correction', True)))
                        var_obs = df_cc[_vcol]
                        valid_v = (var_obs > 0) & np.isfinite(var_obs) & np.isfinite(df_cc[ecol])
                        df_cc, var_obs = df_cc[valid_v], var_obs[valid_v]
                        drop_k = len(df_cc)
                    if drop_k > 0:
                        w_obs = (1.0 / var_obs).values
                        drop_mu, drop_cil, drop_ciu, drop_sig = calc_fe_stats(
                            df_cc[ecol].values, w_obs)
                    else:
                        drop_mu = drop_cil = drop_ciu = np.nan
                        drop_sig = False

                    fmt_sig = lambda s: ("<b style='color:#28a745'>Yes</b>" if s
                                         else "<span style='color:#6c757d'>No</span>")
                    diff = abs(main_mu - drop_mu) if pd.notna(drop_mu) else 0
                    if pd.isna(drop_mu):
                        verdict = "<b>Cannot compare:</b> no complete-case rows."
                        v_col, v_bg = "#856404", "#fff3cd"
                    elif main_sig != drop_sig:
                        verdict = ("⚠️ <b>SIGNIFICANCE FLIPPED.</b> Imputation changes "
                                   "the statistical conclusion.")
                        v_col, v_bg = "#721c24", "#f8d7da"
                    elif diff > 0.2:
                        verdict = "⚠️ <b>Magnitude shifted</b> (>0.2) though significance held."
                        v_col, v_bg = "#856404", "#fff3cd"
                    else:
                        verdict = "✅ <b>Robust:</b> imputation did not materially change the result."
                        v_col, v_bg = "#155724", "#d4edda"

                    with tab_sens:
                        clear_output()
                        display(HTML(f"""
                        <div style='padding:20px; background:#fff; border:1px solid #dee2e6; border-radius:8px;'>
                            <h4 style='color:#2E86AB; margin-top:0;'>⚖️ Imputation Sensitivity (fixed-effect screen)</h4>
                            <p style='color:#555;'>Strategy <b>{dd_missing.value}</b> vs a true complete-case
                            analysis whose variances are recomputed from the <b>observed</b> SDs only.</p>
                            <table style='width:100%; border-collapse:collapse; text-align:center; font-size:14px;'>
                                <tr style='background:#f8f9fa; border-bottom:2px solid #dee2e6;'>
                                    <th style='padding:10px; text-align:left;'>Dataset</th><th style='padding:10px;'>k</th>
                                    <th style='padding:10px;'>Pooled mean</th><th style='padding:10px;'>95% CI</th>
                                    <th style='padding:10px;'>Significant?</th></tr>
                                <tr><td style='padding:12px; border-bottom:1px solid #eee; text-align:left;'><b>Your imputation</b></td>
                                    <td style='border-bottom:1px solid #eee;'>{main_k}</td>
                                    <td style='border-bottom:1px solid #eee;'><b>{main_mu:.3f}</b></td>
                                    <td style='border-bottom:1px solid #eee;'>[{main_cil:.3f}, {main_ciu:.3f}]</td>
                                    <td style='border-bottom:1px solid #eee;'>{fmt_sig(main_sig)}</td></tr>
                                <tr><td style='padding:12px; text-align:left;'><b>Complete case (observed SD)</b></td>
                                    <td>{drop_k}</td>
                                    <td>{('%.3f'%drop_mu) if pd.notna(drop_mu) else '—'}</td>
                                    <td>{('[%.3f, %.3f]'%(drop_cil, drop_ciu)) if pd.notna(drop_mu) else '—'}</td>
                                    <td>{fmt_sig(drop_sig)}</td></tr>
                            </table>
                            <div style='margin-top:20px; padding:12px; background:{v_bg}; color:{v_col}; border-radius:5px;'>
                                {verdict}<br><span style='font-size:13px;'>Absolute difference in means: {diff:.3f}</span></div>
                            <p style='margin-top:15px; font-size:12px; color:#6c757d;'>
                                <i>Fixed-effect screen for sensitivity only. The complete-case arm uses
                                variances recomputed from observed SDs, so it is valid even when the chosen
                                strategy overwrote SDs. Imputed SDs are treated as known in both the main
                                analysis and this screen.</i></p>
                        </div>"""))

                clear_output()
                display(tabs)

        btn_calc.on_click(on_calc)
        display(widgets.VBox([dd_missing, warn_box, dd_zero, btn_calc, out_calc]))

        # Reproducibility mode: restore the saved strategies and calculate, as
        # the other steps do. A strategy that no longer exists is not guessed.
        if ANALYSIS_CONFIG.get('is_reproducing', False):
            _saved = {'Missing SDs': (dd_missing, ANALYSIS_CONFIG.get('sd_missing_strategy')),
                      'Zero SDs': (dd_zero, ANALYSIS_CONFIG.get('sd_zero_strategy'))}
            _unknown = [f"{lab}: <code>{val}</code>" for lab, (dd, val) in _saved.items()
                        if val not in [v for _, v in dd.options]]
            if _unknown:
                display(HTML("<div style='background:#fff3cd; border-left:4px solid #ffc107; padding:12px; "
                             "margin-top:10px; border-radius:4px; color:#856404;'>⚠️ <b>Reproducibility Mode:</b> "
                             "the saved SD strategy is not available in this version ("
                             + ", ".join(_unknown) + "). Choose a strategy above and click "
                             "<b>Calculate</b>.</div>"))
            else:
                for dd, val in _saved.values():
                    dd.value = val
                display(HTML("<div style='background:#e8f4f8; border-left:4px solid #17a2b8; padding:12px; "
                             "margin-top:10px; border-radius:4px; color:#0c5460;'>🔄 <b>Reproducibility Mode Active</b>"
                             f" &nbsp;|&nbsp; Restored SD handling: missing = <b>{dd_missing.label}</b>, "
                             f"zero = <b>{dd_zero.label}</b></div>"))
                on_calc(None)

    else:
        run_calculation()
        with tab_sens:
            clear_output()
            display(HTML("<div style='padding:15px; background:#d4edda; color:#155724; border-radius:5px;'>✅ No missing/zero variance data detected. Sensitivity check not required.</div>"))
        display(tabs)

else:
    # =========================================================================
    # PRE-CALCULATED MODE  (yi/vi supplied directly)
    # =========================================================================
    try:
        effect_size_type = ANALYSIS_CONFIG['effect_size_type']
        es_config = ANALYSIS_CONFIG['es_config']
        variance_type = ANALYSIS_CONFIG.get('variance_type', 'variance')
    except KeyError as e:
        print(f"❌ ERROR: Configuration missing: {e}")
    else:
        if 'data_filtered' not in globals():
            print("❌ ERROR: Data not found. Run filtering cell first.")
        else:
            for _k in ('analysis_data', 'vcv_matrices', 'overall_results'):
                ANALYSIS_CONFIG.pop(_k, None)      # no stale results if this run stops early
            df = data_filtered.copy()
            initial_obs = len(df)
            removed_records = []
            log_diag = ["<b>1. Pre-calculated Data Mode</b>",
                        f"• Effect Size Type: {es_config['effect_label']}"]

            # Variance handling
            if variance_type == 'se':
                se_name = 'se' if 'se' in df.columns else ('SE' if 'SE' in df.columns else None)
                if se_name:
                    df['se'] = pd.to_numeric(df[se_name], errors='coerce')
                    df['vi'] = df['se'] ** 2
                    log_diag.append("• Converted SE to variance (vi = se²)")
                else:
                    print("❌ ERROR: SE column not found")
            elif variance_type == 'variance':
                if 'variance' in df.columns:
                    df['vi'] = pd.to_numeric(df['variance'], errors='coerce')
                    log_diag.append("• Using 'variance' column")
                elif 'vi' in df.columns:
                    df['vi'] = pd.to_numeric(df['vi'], errors='coerce')
                    log_diag.append("• Using 'vi' column")
                else:
                    print("❌ ERROR: Variance column not found:", list(df.columns))
            elif variance_type == 'both':
                if 'variance' in df.columns:
                    df['vi'] = pd.to_numeric(df['variance'], errors='coerce')
                elif 'vi' in df.columns:
                    df['vi'] = pd.to_numeric(df['vi'], errors='coerce')
                elif 'se' in df.columns:
                    df['vi'] = pd.to_numeric(df['se'], errors='coerce') ** 2
                log_diag.append("• Variance resolved (both provided)")

            effect_col = es_config['effect_col']; var_col = es_config['var_col']
            se_col = es_config['se_col']
            ci_lower_col = es_config['ci_lower_col']; ci_upper_col = es_config['ci_upper_col']

            if 'yi' in df.columns:
                df[effect_col] = pd.to_numeric(df['yi'], errors='coerce')
            df[var_col] = pd.to_numeric(df['vi'], errors='coerce')
            log_diag.append(f"• Renamed: yi → {effect_col}, vi → {var_col}")

            log_diag.append("<br><b>2. Data Validation</b>")
            mask_missing = df[[effect_col, var_col]].isna().any(axis=1)
            if mask_missing.sum() > 0:
                dropped = df[mask_missing].copy()
                dropped['Reason'] = 'Missing effect size or variance'
                removed_records.append(dropped)
                df = df[~mask_missing].copy()
                log_diag.append(f"• Removed {int(mask_missing.sum())} rows with missing values")

            mask_neg = df[var_col] < 0
            if mask_neg.sum() > 0:
                print("="*60); print("❌ CRITICAL ERROR: NEGATIVE VARIANCE DETECTED")
                print(f"Found {int(mask_neg.sum())} rows with negative variance "
                      f"(rows {', '.join(str(i) for i in df.index[mask_neg][:20])}).")
                print("Fix the variance (or SE) column and re-run Steps 2-6.")
                # Clear the previous run's data so later steps cannot
                # silently analyse an earlier dataset.
            else:
                mask_bad = (df[var_col] == 0) | ~np.isfinite(df[var_col])
                if mask_bad.sum() > 0:
                    dropped = df[mask_bad].copy()
                    dropped['Reason'] = 'Zero / non-finite variance (infinite weight)'
                    removed_records.append(dropped)
                    df = df[~mask_bad].copy()
                    log_diag.append(f"• Removed {int(mask_bad.sum())} rows with zero/non-finite variance")
                log_diag.append("• ✓ Validation passed")

                log_diag.append("<br><b>3. Derived Statistics</b>")
                df[se_col] = np.sqrt(df[var_col])
                df[ci_lower_col] = df[effect_col] - Z_95 * df[se_col]
                df[ci_upper_col] = df[effect_col] + Z_95 * df[se_col]
                df['w_fixed'] = 1 / df[var_col]
                if 'n_total' not in df.columns:
                    df['n_total'] = np.nan

                if effect_size_type == 'lnRR':
                    df['Response_Ratio'] = np.exp(df[effect_col])
                    df['fold_change'] = np.where(df[effect_col] >= 0,
                                                 df['Response_Ratio'],
                                                 -1.0 / df['Response_Ratio'])
                    df['Percent_Change'] = (df['Response_Ratio'] - 1) * 100
                elif effect_size_type == 'log_or':
                    df['Odds_Ratio'] = np.exp(df[effect_col])
                    df['OR_CI_lower'] = np.exp(df[ci_lower_col])
                    df['OR_CI_upper'] = np.exp(df[ci_upper_col])
                    df['fold_change'] = df['Odds_Ratio']
                elif effect_size_type == 'log_rr':
                    df['Risk_Ratio'] = np.exp(df[effect_col])
                    df['RR_CI_lower'] = np.exp(df[ci_lower_col])
                    df['RR_CI_upper'] = np.exp(df[ci_upper_col])
                    df['fold_change'] = df['Risk_Ratio']
                elif effect_size_type == 'fisher_z':
                    df['Pearson_r'] = np.tanh(df[effect_col])
                    df['r_CI_lower'] = np.tanh(df[ci_lower_col])
                    df['r_CI_upper'] = np.tanh(df[ci_upper_col])

                df = _apply_common_row_set(df, removed_records, log_diag)
                ANALYSIS_CONFIG['analysis_data'] = df
                ANALYSIS_CONFIG['effect_col'] = effect_col
                ANALYSIS_CONFIG['var_col'] = var_col
                ANALYSIS_CONFIG['se_col'] = se_col
                ANALYSIS_CONFIG['ci_lower_col'] = ci_lower_col
                ANALYSIS_CONFIG['ci_upper_col'] = ci_upper_col

                vcv_kwargs = dict(yi_col=effect_col)
                if effect_size_type in ('log_or', 'log_rr', 'risk_diff'):
                    vcv_kwargs['events_c_col'] = 'events_c'
                    vcv_kwargs['nonevents_c_col'] = 'nonevents_c'
                ANALYSIS_CONFIG['vcv_matrices'] = build_vcv_matrices(
                    df, effect_size_type, var_col, **vcv_kwargs)
                log_diag.append(f"• Built VCV matrices for {df['id'].nunique()} studies")

                # Reuse the shared tab renderer for consistency
                with tab_sens:
                    clear_output()
                    display(HTML("<div style='padding:15px; background:#d4edda; color:#155724; border-radius:5px;'>✅ Pre-calculated mode. SD-imputation sensitivity check not applicable.</div>"))
                _populate_tabs(df, es_config, effect_size_type, effect_col, var_col,
                               se_col, ci_lower_col, ci_upper_col, log_diag,
                               removed_records, initial_obs)
                display(tabs)

In [ ]:
#@title 📊 7. Overall Meta-Analysis
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# Purpose: Centralized data management for overall meta-analysis
# Dependencies: ANALYSIS_CONFIG global dictionary
# =============================================================================

import numpy as np
import pandas as pd
from typing import Dict, Any, Optional, Tuple, List
from dataclasses import dataclass
import warnings
from scipy.stats import norm, chi2, t

import datetime
import ipywidgets as widgets
from IPython.display import display, HTML
import sys

# The variance structure follows the design of the data: effect sizes are
# nested within studies, so the three-level model is reported whenever it
# can be fitted. The nested comparisons (tau² = 0, sigma² = 0) are computed
# and reported for information only; they do not choose the model, because
# selecting a model by a pre-test makes its CI and p conditional on that test.
THREE_LEVEL_CHOICE = 'Three-level (recommended)'
TWO_LEVEL_CHOICE = 'Two-level (sensitivity analysis)'
MODEL_CHOICES = [THREE_LEVEL_CHOICE, TWO_LEVEL_CHOICE]


def _normalize_model_choice(choice: Optional[str]) -> str:
    """Any value other than the two-level option means the default."""
    return TWO_LEVEL_CHOICE if choice == TWO_LEVEL_CHOICE else THREE_LEVEL_CHOICE


# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class OverallConfig:
    """Configuration for overall meta-analysis"""
    effect_col: str
    var_col: str
    alpha: float = 0.05
    dist_type: str = 't'  # 't' or 'norm'
    use_kh: bool = True
    tau_method: str = 'REML'
    model_choice: str = THREE_LEVEL_CHOICE

    def __post_init__(self):
        """Validate configuration"""
        if not self.effect_col:
            raise ValueError("effect_col cannot be empty")
        if not self.var_col:
            raise ValueError("var_col cannot be empty")
        if self.alpha <= 0 or self.alpha >= 1:
            raise ValueError(f"alpha must be between 0 and 1, got {self.alpha}")
        if self.dist_type not in ['t', 'norm']:
            raise ValueError(f"dist_type must be 't' or 'norm', got {self.dist_type}")
        if self.tau_method not in ['REML', 'DL', 'ML', 'PM', 'SJ']:
            raise ValueError(f"tau_method must be REML, DL, ML, PM, or SJ, got {self.tau_method}")


@dataclass
class OverallResult:
    """Complete overall meta-analysis result"""
    # Fixed-effect results
    mu_fixed: float
    se_fixed: float
    ci_lower_fixed: float
    ci_upper_fixed: float

    # Random-effects results (2-level)
    mu_random: float
    se_random: float
    ci_lower_random: float
    ci_upper_random: float
    p_value_random: float
    pi_lower_random: float
    pi_upper_random: float

    # Heterogeneity statistics
    Q: float
    df_Q: int
    p_Q: float
    I2: float
    tau_squared: float

    # Sample size
    k_obs: int
    k_studies: int

    # Model info
    tau_method: str = 'REML'
    use_kh: bool = True
    dist_type: str = 't'
    alpha: float = 0.05
    df_infer: Optional[float] = None      # <-- Satterthwaite df actually used for the mean's inference
    used_robust: bool = False             # <-- True iff CR2 sandwich succeeded (vs classic-KH fallback)
    used_robust_3level: bool = False      # <-- same, for the 3-level fit (vs plain-GLS fallback)

    # Nested variance-structure comparison (Cell 1b engine): the three-level
    # model vs the same model with tau2 or sigma2 fixed at 0, all on the same
    # rows, the same sampling V and the same fixed effects. See
    # OverallAnalysisOrchestrator._nested_variance_comparison.
    variance_comparison: Optional[Dict[str, Any]] = None
    selection_basis: str = ''
    # Which rows were used / dropped and why (RowAccounting.to_dict()).
    data_accounting: Optional[Dict[str, Any]] = None
    df_infer_3level: Optional[float] = None

    # 3-level results (optional)
    has_3level: bool = False
    mu_3level: Optional[float] = None
    se_3level: Optional[float] = None
    se_model_based_3level: Optional[float] = None
    ci_lower_3level: Optional[float] = None
    ci_upper_3level: Optional[float] = None
    pi_lower_3level: Optional[float] = None
    pi_upper_3level: Optional[float] = None
    p_value_3level: Optional[float] = None
    tau_squared_3level: Optional[float] = None
    sigma_squared_3level: Optional[float] = None
    icc_l3: Optional[float] = None
    icc_l2: Optional[float] = None
    aic_3level: Optional[float] = None
    ll_3level: Optional[float] = None
    tau_3level: Optional[float] = None
    tau2_ci_lower_3level: Optional[float] = None
    tau2_ci_upper_3level: Optional[float] = None
    sigma2_ci_lower_3level: Optional[float] = None
    sigma2_ci_upper_3level: Optional[float] = None


    # Model selection
    best_model: str = "2-Level"
    model_choice: str = THREE_LEVEL_CHOICE


# =============================================================================
# DATA MANAGER CLASS
# =============================================================================

class OverallDataManager:
    """
    Centralized data access layer for overall meta-analysis.
    Handles all interactions with ANALYSIS_CONFIG and data validation.
    """

    def __init__(self, analysis_config: Dict[str, Any]):
        """
        Initialize data manager.

        Args:
            analysis_config: Global ANALYSIS_CONFIG dictionary
        """
        self._config = analysis_config
        self._validate_prerequisites()

    # -------------------------------------------------------------------------
    # VALIDATION METHODS
    # -------------------------------------------------------------------------

    def _validate_prerequisites(self) -> None:
        """Validate that required configuration exists"""
        if 'effect_col' not in self._config:
            warnings.warn("effect_col not in ANALYSIS_CONFIG, using default 'hedges_g'")
        if 'var_col' not in self._config:
            warnings.warn("var_col not in ANALYSIS_CONFIG, using default 'Vg'")

    def validate_data(self, df: pd.DataFrame) -> Tuple[bool, Optional[str]]:
        """
        Validate that data is suitable for meta-analysis.

        Args:
            df: DataFrame to validate

        Returns:
            Tuple of (is_valid, error_message)
        """
        if df is None or len(df) == 0:
            return False, "No data available"

        if self.effect_col not in df.columns:
            return False, f"Effect column '{self.effect_col}' not found"

        if self.var_col not in df.columns:
            return False, f"Variance column '{self.var_col}' not found"

        # Check for minimum observations
        n_valid = df[[self.effect_col, self.var_col]].notna().all(axis=1).sum()
        if n_valid < 2:
            return False, f"Need at least 2 valid observations, found {n_valid}"

        return True, None

    # -------------------------------------------------------------------------
    # PROPERTY ACCESSORS
    # -------------------------------------------------------------------------

    @property
    def analysis_data(self) -> Optional[pd.DataFrame]:
        """Get analysis dataset"""
        if 'analysis_data' in self._config:
            return self._config['analysis_data'].copy()
        return None

    @property
    def effect_col(self) -> str:
        """Get effect size column name"""
        return self._config.get('effect_col', 'hedges_g')

    @property
    def var_col(self) -> str:
        """Get variance column name"""
        return self._config.get('var_col', 'Vg')

    @property
    def es_config(self) -> Dict[str, Any]:
        """Get effect size configuration"""
        return self._config.get('es_config', {})

    @property
    def vcv_matrices(self) -> Dict[Any, np.ndarray]:
        """Get variance-covariance matrices for 3-level model"""
        return self._config.get('vcv_matrices', {})

    @property
    def global_settings(self) -> Dict[str, Any]:
        """Get global settings (with defaults)"""
        return self._config.get('global_settings', {
            'alpha': 0.05,
            'dist_type': 't'
        })

    # -------------------------------------------------------------------------
    # DATA EXTRACTION METHODS
    # -------------------------------------------------------------------------

    def prepare_data(
        self,
        df: Optional[pd.DataFrame] = None
    ) -> pd.DataFrame:
        """
        Prepare and clean data for meta-analysis.

        Args:
            df: DataFrame to prepare (uses analysis_data if None)

        Returns:
            Cleaned DataFrame

        Raises:
            ValueError: If data cannot be prepared
        """
        if df is None:
            df = self.analysis_data

        if df is None:
            raise ValueError("No data available for analysis")

        # Validate
        is_valid, error_msg = self.validate_data(df)
        if not is_valid:
            raise ValueError(error_msg)

        # Row selection with a recorded reason for every dropped row, and the
        # sampling-VCV row identity attached so later sorting cannot break it.
        if 'select_rows' in globals():
            clean_df, acc = select_rows(df, self.effect_col, self.var_col, 'id')
            self.last_accounting = acc
            clean_df = attach_vcv_row_pos(clean_df, reference_df=self.analysis_data)
            return clean_df

        clean_df = df.dropna(subset=[self.effect_col, self.var_col]).copy()
        clean_df = clean_df[clean_df[self.var_col] > 0].copy()
        if len(clean_df) == 0:
            raise ValueError("No valid data after cleaning")
        self.last_accounting = None
        return clean_df

    def check_3level_feasibility(self, df: pd.DataFrame) -> bool:
        """
        Check if 3-level model is feasible based on data topology.
        Requires a meaningful proportion of studies with multiple effect sizes
        to reliably partition within- vs between-study variance.
        """
        if 'id' not in df.columns:
            return False

        # Calculate how many independent studies have multiple observations
        multi_obs_studies = (df.groupby('id').size() > 1).sum()

        # Structure-Aware Rule: At least 2 studies must have internal replication
        # to justify attempting to estimate sigma_squared.
        return len(df) > df['id'].nunique() and multi_obs_studies >= 2

    # -------------------------------------------------------------------------
    # RESULT PERSISTENCE METHODS
    # -------------------------------------------------------------------------

    def save_overall_results(self, result: OverallResult) -> None:
        """
        Save overall analysis results to ANALYSIS_CONFIG.

        Args:
            result: OverallResult object
        """
        import datetime

        self._config['overall_results'] = {
            'timestamp': datetime.datetime.now(),

            # Fixed-effect
            'pooled_effect_fixed': result.mu_fixed,
            'pooled_SE_fixed': result.se_fixed,
            'ci_lower_fixed': result.ci_lower_fixed,
            'ci_upper_fixed': result.ci_upper_fixed,

            # Random-effects (2-level)
            'pooled_effect_random': result.mu_random,
            'pooled_SE_random_reported': result.se_random,
            'ci_lower_random_reported': result.ci_lower_random,
            'ci_upper_random_reported': result.ci_upper_random,
            'p_value_random_reported': result.p_value_random,

            # Heterogeneity
            'Qt': result.Q,
            'I_squared': result.I2,
            'tau_squared': result.tau_squared,
            'p_Q': result.p_Q,

            # Sample size
            'k': result.k_obs,
            'k_papers': result.k_studies,

            # Model info
            'tau_method': result.tau_method,
            'knapp_hartung': {'used': result.use_kh},
            'df_inference': result.df_infer,
            'used_robust': result.used_robust,
            'dist_type': result.dist_type,

            # Model selection (nested variance-structure comparison)
            'best_model': result.best_model,
            'model_choice': result.model_choice,
            'selection_basis': result.selection_basis,
            'variance_structure_comparison': result.variance_comparison,
            'data_accounting': result.data_accounting,
        }

        # Back-transform pooled estimates for fold-change annotation (Cell 10 diamond)
        es_cfg = self._config.get('es_config', {})
        if es_cfg.get('has_fold_change', False):
            self._config['overall_results']['pooled_fold_fixed'] = np.exp(result.mu_fixed)
            self._config['overall_results']['pooled_fold_random'] = np.exp(result.mu_random)
        else:
            self._config['overall_results']['pooled_fold_fixed'] = np.nan
            self._config['overall_results']['pooled_fold_random'] = np.nan

        # Save 3-level results if available
        if result.has_3level:
            self._config['three_level_results'] = {
                'status': 'completed',
                'pooled_effect': result.mu_3level,
                'se': result.se_3level,
                'se_model_based': result.se_model_based_3level,
                'ci_lower': result.ci_lower_3level,
                'ci_upper': result.ci_upper_3level,
                'p_value': result.p_value_3level,
                'tau_squared': result.tau_squared_3level,
                'tau2_ci_lower': result.tau2_ci_lower_3level,
                'tau2_ci_upper': result.tau2_ci_upper_3level,
                'sigma_squared': result.sigma_squared_3level,
                'sigma2_ci_lower': result.sigma2_ci_lower_3level,
                'sigma2_ci_upper': result.sigma2_ci_upper_3level,
                'icc_l3': result.icc_l3,
                'icc_l2': result.icc_l2,
                'aic': result.aic_3level
            }

        # The model the report text uses (best_model). Plot cells read this
        # block so the "Overall" row always matches the reported estimate.
        use_3l = result.best_model == "3-Level" and result.has_3level
        mu_rep = result.mu_3level if use_3l else result.mu_random
        self._config['overall_results']['reported'] = {
            'model': "3-Level" if use_3l else "2-Level",
            'mu': mu_rep,
            'se': result.se_3level if use_3l else result.se_random,
            'ci_lower': result.ci_lower_3level if use_3l else result.ci_lower_random,
            'ci_upper': result.ci_upper_3level if use_3l else result.ci_upper_random,
            'pi_lower': result.pi_lower_3level if use_3l else result.pi_lower_random,
            'pi_upper': result.pi_upper_3level if use_3l else result.pi_upper_random,
            'p_value': result.p_value_3level if use_3l else result.p_value_random,
            'tau_squared': result.tau_squared_3level if use_3l else result.tau_squared,
            'sigma_squared': result.sigma_squared_3level if use_3l else 0.0,
            'df': result.df_infer_3level if use_3l else result.df_infer,
            'fold': np.exp(mu_rep) if es_cfg.get('has_fold_change', False) else np.nan,
        }

    def save_global_settings(
        self,
        alpha: float,
        dist_type: str,
        tau_method: str,
        use_kh: bool,
        model_choice: str
    ) -> None:
        """Save global analysis settings"""
        self._config['global_settings'] = {
            'alpha': alpha,
            'dist_type': dist_type,
            'ci_percent': (1 - alpha) * 100,
            'tau_method': tau_method,
            'use_kh': use_kh,
            'model_choice': model_choice
        }

    def save_publication_text(self, text: str) -> None:
        """Save publication-ready text"""
        self._config['overall_text'] = text

    # -------------------------------------------------------------------------
    # UTILITY METHODS
    # -------------------------------------------------------------------------

    def summary_dict(self) -> Dict[str, Any]:
        """Get summary of current configuration"""
        df = self.analysis_data

        return {
            'effect_col': self.effect_col,
            'var_col': self.var_col,
            'n_observations': len(df) if df is not None else 0,
            'n_studies': df['id'].nunique() if df is not None and 'id' in df.columns else 0,
            'has_vcv': len(self.vcv_matrices) > 0,
            'can_do_3level': self.check_3level_feasibility(df) if df is not None else False
        }

# =============================================================================
# FIXED-EFFECT ENGINE
# =============================================================================

class FixedEffectEngine:
    """
    Fixed-effect meta-analysis engine.

    Mathematical implementation: Standard inverse-variance weighting.
    """

    @staticmethod
    def calculate(
        y: np.ndarray,
        v: np.ndarray,
        alpha: float = 0.05,
        dist_type: str = 't'
    ) -> Dict[str, Any]:
        """
        Calculate fixed-effect pooled estimate.

        MATH PRESERVED: Original inverse-variance weighting

        Args:
            y: Effect sizes
            v: Variances
            alpha: Significance level
            dist_type: 't' or 'norm'

        Returns:
            Dictionary with mu, se, ci_lower, ci_upper
        """
        # Inverse-variance weights
        w = 1 / v
        sum_w = np.sum(w)

        # Pooled estimate
        mu = np.sum(w * y) / sum_w
        se = np.sqrt(1 / sum_w)

        # Critical value
        q = 1 - (alpha / 2)
        if dist_type == 't':
            crit_val = t.ppf(q, len(y) - 1)
        else:
            crit_val = norm.ppf(q)

        # Confidence interval
        ci_lower = mu - crit_val * se
        ci_upper = mu + crit_val * se

        return {
            'mu': mu,
            'se': se,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper
        }


# =============================================================================
# HETEROGENEITY STATISTICS ENGINE
# =============================================================================

class HeterogeneityEngine:
    """
    Calculate heterogeneity statistics (Q, I², τ²).

    Mathematical implementation validated against metafor.
    """

    @staticmethod
    def calculate_Q_statistics(
        y: np.ndarray,
        v: np.ndarray,
        mu: float
    ) -> Tuple[float, int, float]:
        """
        Calculate Cochran's Q statistic.

        MATH PRESERVED: Original Q calculation

        Args:
            y: Effect sizes
            v: Variances
            mu: Pooled estimate (from fixed-effect)

        Returns:
            Tuple of (Q, df, p_value)
        """
        w = 1 / v
        Q = np.sum(w * (y - mu)**2)
        df_Q = len(y) - 1
        p_Q = 1 - chi2.cdf(Q, df_Q) if df_Q > 0 else 1.0

        return Q, df_Q, p_Q

    @staticmethod
    def calculate_I2(Q: float, df_Q: int) -> float:
        """
        Calculate I² statistic.

        Args:
            Q: Q statistic
            df_Q: Degrees of freedom

        Returns:
            I² percentage
        """
        if Q <= df_Q:
            return 0.0
        return ((Q - df_Q) / Q) * 100 if Q > 0 else 0.0

    @staticmethod
    def calculate_tau2_DL(
        Q: float,
        df_Q: int,
        w: np.ndarray
    ) -> float:
        """
        Calculate τ² using DerSimonian-Laird method.

        MATH PRESERVED: Original DL estimator

        Args:
            Q: Q statistic
            df_Q: Degrees of freedom
            w: Fixed-effect weights

        Returns:
            τ² estimate
        """
        C = np.sum(w) - np.sum(w**2) / np.sum(w)
        if C <= 0:
            return 0.0

        tau2 = max(0, (Q - df_Q) / C)
        return tau2


# =============================================================================
# TWO-LEVEL RANDOM-EFFECTS ENGINE
# =============================================================================

class TwoLevelEngine:
    """
    Two-level random-effects meta-analysis with multiple τ² estimators.

    Mathematical implementation validated against metafor.
    """

    def __init__(self, tau_method: str = 'REML'):
        """
        Initialize engine.

        Args:
            tau_method: 'REML', 'DL', or 'ML'
        """
        self.tau_method = tau_method
        self.het_engine = HeterogeneityEngine()

    # -------------------------------------------------------------------------
    # TAU² ESTIMATION
    # -------------------------------------------------------------------------

    def estimate_tau2(
        self,
        df: pd.DataFrame,
        effect_col: str,
        var_col: str
    ) -> float:
        """
        Estimate τ² using specified method.

        MATH PRESERVED: Delegates to appropriate estimator

        Args:
            df: DataFrame with effect sizes
            effect_col: Effect size column name
            var_col: Variance column name

        Returns:
            τ² estimate
        """
        y = df[effect_col].values
        v = df[var_col].values

        if self.tau_method == 'DL':
            # Calculate Q from fixed-effect
            w = 1 / v
            mu_fe = np.sum(w * y) / np.sum(w)
            Q, df_Q, _ = self.het_engine.calculate_Q_statistics(y, v, mu_fe)
            return self.het_engine.calculate_tau2_DL(Q, df_Q, w)

        else:
            # Use global tau² calculator (REML or ML)
            try:
                # Check if calculate_tau_squared exists in globals
                if 'calculate_tau_squared' in globals():
                    tau2, _ = calculate_tau_squared(
                        df, effect_col, var_col, method=self.tau_method
                    )
                    return tau2
                else:
                    # Fallback to DL if function not available
                    warnings.warn(
                        f"{self.tau_method} not available, using DL estimator"
                    )
                    w = 1 / v
                    mu_fe = np.sum(w * y) / np.sum(w)
                    Q, df_Q, _ = self.het_engine.calculate_Q_statistics(y, v, mu_fe)
                    return self.het_engine.calculate_tau2_DL(Q, df_Q, w)
            except Exception as e:
                warnings.warn(f"τ² estimation failed: {e}, using DL")
                w = 1 / v
                mu_fe = np.sum(w * y) / np.sum(w)
                Q, df_Q, _ = self.het_engine.calculate_Q_statistics(y, v, mu_fe)
                return self.het_engine.calculate_tau2_DL(Q, df_Q, w)

    # -------------------------------------------------------------------------
    # RANDOM-EFFECTS POOLING
    # -------------------------------------------------------------------------
    def calculate_pooled_effect(
        self,
        df: pd.DataFrame,
        effect_col: str,
        var_col: str,
        vcv_dict: Dict[Any, np.ndarray],
        tau2: float,
        alpha: float = 0.05,
        dist_type: str = 't',
        use_kh: bool = True
    ) -> Dict[str, Any]:
        """
        Calculate random-effects pooled estimate with CRVE / Knapp-Hartung.
        """
        # Ensure data is sorted by ID for clustering
        df = df.sort_values('id', kind='mergesort').reset_index(drop=True)
        y = df[effect_col].values
        v = df[var_col].values
        k_obs = len(y)
        k_studies = df['id'].nunique()

        # Standard Random-effects weights & Point Estimate
        w = 1 / (v + tau2)
        sum_w = np.sum(w)
        mu = np.sum(w * y) / sum_w
        se = np.sqrt(1 / sum_w)
        se_model = se
        df_val = k_studies - 1

        # --- APPLY CRVE (Cluster-Robust Variance Estimation) ---
        used_robust = False
        if use_kh and k_obs > 1:
            try:
                # Group data by cluster to feed the robust estimator
                grouped = df.groupby('id', sort=False)
                y_all = []
                vcv_all = []
                X_blocks = []

                for study_id, group in grouped:
                    y_all.append(group[effect_col].values.astype(np.float64))
                    k = len(group)
                    V_i, _src = cluster_vcv(vcv_dict, study_id, group, var_col)
                    vcv_all.append(V_i)
                    X_blocks.append(np.ones((k, 1)))

                # CR2 working model of the 2-level model: effect-level
                # heterogeneity is diagonal (independent effects), so tau2
                # goes in the WITHIN slot (sigma_sq); the between-study (J)
                # component is 0. Passing it as tau_sq made effects of one
                # study correlated in the working model (a 3-level structure)
                # and gave CR2 SEs/df that clubSandwich does not reproduce.
                var_robust, df_robust = globals()['_compute_robust_var_betas'](
                    [mu], y_all, vcv_all, X_blocks, 0.0, tau2
                )

                se_robust = np.sqrt(var_robust[0, 0])
                if np.isfinite(se_robust) and se_robust > 0:
                    se = se_robust
                    df_val = float(df_robust[0])
                    used_robust = True

            except Exception as e:
                # Fall through to classic Knapp-Hartung below, but say so
                warnings.warn(f"2-level model: CR2 step failed ({e}); using Knapp-Hartung instead.")

        # Fallback to standard Knapp-Hartung if CRVE couldn't run
        # (2-level model: effects are the units, so Q and the t reference
        # distribution both use k_obs - 1 df. The scale factor is truncated
        # at 1, as in metafor's test='adhoc'; test='knha' does not truncate.)
        if use_kh and k_obs > 1 and not used_robust:
            df_val = k_obs - 1
            q_re = np.sum(w * (y - mu)**2)
            se = se * np.sqrt(max(1, q_re / df_val))

        # --- INFERENCE ---
        q = 1 - (alpha / 2)
        if dist_type == 't':
            crit_val = t.ppf(q, max(1, df_val))
            p_value = 2 * (1 - t.cdf(abs(mu / se), max(1, df_val)))
        else:
            crit_val = norm.ppf(q)
            p_value = 2 * (1 - norm.cdf(abs(mu / se)))

        ci_lower = mu - crit_val * se
        ci_upper = mu + crit_val * se

        # --- Prediction Interval ---
        df_pi = max(1, k_studies - 1)
        t_crit_pi = t.ppf(q, df_pi) if dist_type == 't' else norm.ppf(q)
        pi_se = np.sqrt(se_model**2 + tau2)
        pi_lower = mu - t_crit_pi * pi_se
        pi_upper = mu + t_crit_pi * pi_se

        return {
            'mu': mu,
            'se': se,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper,
            'pi_lower': pi_lower,
            'pi_upper': pi_upper,
            'p_value': p_value,
            'used_robust': used_robust,
            'df_infer': df_val
        }


# =============================================================================
# THREE-LEVEL RANDOM-EFFECTS ENGINE
# =============================================================================

class ThreeLevelEngine:
    """Three-level random-effects meta-analysis (intercept only), fitted by
    REML with the shared engine (fit_blocks / fit_mv, Cell 1b) on the full
    sampling VCV, with CR2 inference and profile-likelihood CIs for the
    variance components."""

    def fit(
        self,
        df: pd.DataFrame,
        effect_col: str,
        var_col: str,
        vcv_dict: Dict[Any, np.ndarray],
        alpha: float = 0.05,
        dist_type: str = 't',
    ) -> Optional[Dict[str, Any]]:
        """
        Fit the three-level random-effects model by REML.
        dist_type ('t' or 'norm') sets the reference distribution of the
        prediction interval, as in the two-level model.

        Args:
            df: DataFrame with effect sizes (must have 'id' column)
            effect_col: Effect size column name
            var_col: Variance column name
            vcv_dict: Dictionary of VCV matrices by study ID

        Returns:
            Dictionary with results or None if the model cannot be fitted
        """
        # Sort to ensure alignment with VCV construction order
        df = df.sort_values('id', kind='mergesort').reset_index(drop=True)

        grouped = df.groupby('id', sort=False)
        y_all = [g[effect_col].values.astype(np.float64) for _, g in grouped]

        vcv_all = []
        for study_id, group in grouped:
            vi = group[var_col].values
            if len(vi) == 0 or np.any(np.isnan(vi)):
                raise ValueError(f"Invalid variance data for study {study_id}")
            # Sliced by row identity: correct even when some of the study's
            # rows were dropped; a cached matrix that cannot be aligned is
            # replaced by the diagonal (never applied to the wrong rows).
            V_i, _src = cluster_vcv(vcv_dict, study_id, group, var_col)
            vcv_all.append(V_i)

        N, M = len(df), len(y_all)

        try:
            fit = fit_blocks(y_all, vcv_all, two_level=False,
                             cluster_ids=list(grouped.groups.keys()))
        except (ValueError, RuntimeError, np.linalg.LinAlgError):
            return None

        tau2, sigma2 = fit.tau2, fit.sigma2
        log_lik = fit.loglik
        aic = fit.aic
        mu = float(fit.betas[0])
        se = float(np.sqrt(fit.cov_beta[0, 0]))

        # --- CR2 & Satterthwaite for the overall fit ---
        # used_robust records whether the CR2 sandwich actually priced this
        # fit's inference. The reporting layer must not claim CR2 just
        # because the data are nested: when the matrix math fails we fall
        # back to plain GLS with df = M - 1, which is a different method
        # and has to be described as such.
        used_robust_3level = False
        try:
            var_robust, df_robust = fit_cr2(fit)
            se_robust = np.sqrt(var_robust[0, 0])
            df_satterthwaite = float(df_robust[0])
            used_robust_3level = bool(
                np.isfinite(se_robust) and se_robust > 0
                and np.isfinite(df_satterthwaite) and df_satterthwaite > 0
            )
            if not used_robust_3level:
                # Non-finite / non-positive output is a silent failure too.
                se_robust = se
                df_satterthwaite = float(M - 1)
        except Exception as e:
            warnings.warn(f"3-level model: CR2 step failed ({e}); using model-based SE "
                          f"with df = studies - 1.")
            se_robust = se
            df_satterthwaite = float(M - 1)

        # Higgins & Thompson (2002) typical within-study variance.
        all_v = np.concatenate([np.diag(V_i) for V_i in vcv_all])
        # Guard against zero/negative variances (shouldn't happen, but defensive)
        all_v = all_v[all_v > 0]
        if len(all_v) > 1:
            w = 1.0 / all_v
            k_total = len(all_v)
            sum_w = np.sum(w)
            sum_w2 = np.sum(w ** 2)
            denom_typical = sum_w ** 2 - sum_w2
            v_typical = ((k_total - 1) * sum_w / denom_typical
                         if denom_typical > 0 else np.mean(all_v))
        else:
            v_typical = float(all_v[0]) if len(all_v) == 1 else 0.0

        total_var = tau2 + sigma2 + v_typical
        icc_l3 = (tau2 / total_var * 100) if total_var > 0 else 0
        icc_l2 = (sigma2 / total_var * 100) if total_var > 0 else 0

        # tau (SD) and profile-likelihood CIs for tau2 and sigma2: the values
        # where the REML log-likelihood, other component re-estimated, drops
        # by chi2_{1-alpha}(1) / 2, as in metafor's confint().
        tau = np.sqrt(tau2)
        tau2_ci_lower, tau2_ci_upper = profile_ci(fit, 'tau2', alpha)
        sigma2_ci_lower, sigma2_ci_upper = profile_ci(fit, 'sigma2', alpha)

        # --- Prediction interval: mu ± crit · sqrt(SE_model² + tau2 + sigma2)
        # (model-based SE). crit = t(studies - 1) for t inference, the normal
        # quantile for z inference (as the two-level model and metafor's
        # predict() with test = "z"). ---
        df_pi = max(1, M - 1)   # M = k_studies
        q_pi = 1 - (alpha / 2)
        t_crit_pi = t.ppf(q_pi, df_pi) if dist_type == 't' else norm.ppf(q_pi)

        pi_se = np.sqrt(se**2 + tau2 + sigma2)
        pi_lower = mu - t_crit_pi * pi_se
        pi_upper = mu + t_crit_pi * pi_se

        return {
            'mu': mu,
            'se': se,
            'se_robust': se_robust,
            'df_robust': df_satterthwaite,
            'used_robust': used_robust_3level,
            'tau2': tau2,
            'tau': tau,
            'tau2_ci_lower': tau2_ci_lower,
            'tau2_ci_upper': tau2_ci_upper,
            'sigma2': sigma2,
            'sigma2_ci_lower': sigma2_ci_lower,
            'sigma2_ci_upper': sigma2_ci_upper,
            'icc_l3': icc_l3,
            'icc_l2': icc_l2,
            'n': N,
            'm': M,
            'aic': aic,
            'log_lik_reml': log_lik,
            'pi_lower': pi_lower,
            'pi_upper': pi_upper
        }

# =============================================================================
# OVERALL META-ANALYSIS ORCHESTRATOR
# =============================================================================

class OverallEngine:
    """
    High-level orchestrator for overall meta-analysis.
    Coordinates fixed-effect, 2-level, and 3-level analyses.
    """

    def __init__(self, data_manager: 'OverallDataManager'):
        """
        Initialize engine with data manager.

        Args:
            data_manager: OverallDataManager instance
        """
        self.data_manager = data_manager
        self.fixed_engine = FixedEffectEngine()
        self.het_engine = HeterogeneityEngine()
        self.two_level_engine = None  # Initialized with tau_method
        self.three_level_engine = ThreeLevelEngine()

    # -------------------------------------------------------------------------
    # MAIN EXECUTION METHOD
    # -------------------------------------------------------------------------

    def run_analysis(
        self,
        alpha: float = 0.05,
        dist_type: str = 't',
        use_kh: bool = True,
        tau_method: str = 'REML',
        model_choice: str = THREE_LEVEL_CHOICE,
        progress_callback: Optional[callable] = None
    ) -> Optional['OverallResult']:
        """
        Execute complete overall meta-analysis workflow.

        WORKFLOW:
        1. Prepare data
        2. Fixed-effect analysis
        3. Heterogeneity statistics
        4. 2-level random-effects
        5. 3-level random-effects (if feasible)
        6. Model selection

        Args:
            alpha: Significance level
            dist_type: 't' or 'norm'
            use_kh: Use Knapp-Hartung correction
            tau_method: 'REML', 'DL', or 'ML'
            model_choice: Model selection strategy
            progress_callback: Optional progress updates

        Returns:
            OverallResult object or None if analysis fails
        """
        # Prepare data
        if progress_callback:
            progress_callback("📊 Preparing data...")

        try:
            df = self.data_manager.prepare_data()
        except ValueError as e:
            if progress_callback:
                progress_callback(f"❌ Data preparation failed: {str(e)}")
            return None

        y = df[self.data_manager.effect_col].values
        v = df[self.data_manager.var_col].values
        k_obs = len(df)
        k_studies = df['id'].nunique() if 'id' in df.columns else k_obs

        # Step 1: Fixed-Effect Analysis
        if progress_callback:
            progress_callback("🔢 Running fixed-effect analysis...")

        fe_results = self.fixed_engine.calculate(y, v, alpha, dist_type)

        # Step 2: Heterogeneity Statistics
        if progress_callback:
            progress_callback("📊 Calculating heterogeneity...")

        Q, df_Q, p_Q = self.het_engine.calculate_Q_statistics(y, v, fe_results['mu'])
        I2 = self.het_engine.calculate_I2(Q, df_Q)

        # Step 3: 2-Level Random-Effects
        if progress_callback:
            progress_callback(f"🎲 Running 2-level RE ({tau_method})...")

        # Initialize 2-level engine with chosen method
        self.two_level_engine = TwoLevelEngine(tau_method=tau_method)

        tau2 = self.two_level_engine.estimate_tau2(
            df, self.data_manager.effect_col, self.data_manager.var_col
        )

        # Pass the FULL dataframe and VCV matrices so CRVE can cluster by study ID
        re_results = self.two_level_engine.calculate_pooled_effect(
            df=df,
            effect_col=self.data_manager.effect_col,
            var_col=self.data_manager.var_col,
            vcv_dict=self.data_manager.vcv_matrices,
            tau2=tau2,
            alpha=alpha,
            dist_type=dist_type,
            use_kh=use_kh
        )

        # Step 4: 3-Level Random-Effects (if feasible)
        has_3level = False
        three_level_results = None

        if self.data_manager.check_3level_feasibility(df):
            if progress_callback:
                progress_callback("🔄 Running 3-level RE (nested)...")

            try:
                three_level_results = self.three_level_engine.fit(
                    df,
                    self.data_manager.effect_col,
                    self.data_manager.var_col,
                    self.data_manager.vcv_matrices,
                    alpha=alpha,
                    dist_type=dist_type
                )

                if three_level_results:
                    has_3level = True

                    if not use_kh:
                        # CR2 / Knapp-Hartung switched off: model-based SE
                        # with df = studies - 1, as in the 2-level path.
                        three_level_results['se_robust'] = three_level_results['se']
                        three_level_results['df_robust'] = float(k_studies - 1)
                        three_level_results['used_robust'] = False

                    # Calculate CIs for 3-level
                    mu_3l = three_level_results['mu']
                    se_3l = three_level_results.get('se_robust', three_level_results['se'])
                    df_3l = three_level_results.get('df_robust', k_studies - 1)

                    q = 1 - (alpha / 2)
                    if dist_type == 't':
                        crit_val = t.ppf(q, df_3l)
                        p_3l = 2 * (1 - t.cdf(abs(mu_3l / se_3l), df_3l))
                    else:
                        crit_val = norm.ppf(q)
                        p_3l = 2 * (1 - norm.cdf(abs(mu_3l / se_3l)))

                    three_level_results['ci_lower'] = mu_3l - crit_val * se_3l
                    three_level_results['ci_upper'] = mu_3l + crit_val * se_3l
                    three_level_results['p_value'] = p_3l

            except Exception as e:
                if progress_callback:
                    progress_callback(f"⚠️ 3-level fitting failed: {str(e)}")

        # Step 5: Nested variance-structure comparison + model selection
        variance_comparison = None
        if has_3level:
            if progress_callback:
                progress_callback("⚖️ Comparing nested variance structures (same rows, same V)...")
            variance_comparison = self._nested_variance_comparison(
                df, alpha, three_level_results
            )
            if 'error' in variance_comparison and progress_callback:
                progress_callback(f"⚠️ Nested comparison unavailable: {variance_comparison['error']}")

        if progress_callback:
            progress_callback("🏆 Evaluating model structure and fit...")

        best_model, selection_basis = self._select_model(
            model_choice, has_3level, variance_comparison, alpha
        )

        # Build result object
        result = OverallResult(
            # Fixed-effect
            mu_fixed=fe_results['mu'],
            se_fixed=fe_results['se'],
            ci_lower_fixed=fe_results['ci_lower'],
            ci_upper_fixed=fe_results['ci_upper'],

            # 2-level random-effects
            mu_random=re_results['mu'],
            se_random=re_results['se'],
            ci_lower_random=re_results['ci_lower'],
            ci_upper_random=re_results['ci_upper'],
            pi_lower_random=re_results['pi_lower'],
            pi_upper_random=re_results['pi_upper'],
            p_value_random=re_results['p_value'],

            # Heterogeneity
            Q=Q,
            df_Q=df_Q,
            p_Q=p_Q,
            I2=I2,
            tau_squared=tau2,

            # Sample size
            k_obs=k_obs,
            k_studies=k_studies,

            # Model info
            tau_method=tau_method,
            use_kh=use_kh,
            dist_type=dist_type,
            alpha=alpha,
            df_infer=re_results.get('df_infer'),
            used_robust=re_results.get('used_robust', False),

            # Model selection
            best_model=best_model,
            model_choice=model_choice,
            variance_comparison=variance_comparison,
            selection_basis=selection_basis,
            data_accounting=(self.data_manager.last_accounting.to_dict()
                             if getattr(self.data_manager, 'last_accounting', None) else None),
        )

        # Add 3-level results if available
        if has_3level:
            result.has_3level = True
            result.mu_3level = three_level_results['mu']
            result.se_3level = three_level_results.get('se_robust', three_level_results['se'])
            result.se_model_based_3level = three_level_results['se']
            result.ci_lower_3level = three_level_results['ci_lower']
            result.ci_upper_3level = three_level_results['ci_upper']
            result.pi_lower_3level = three_level_results['pi_lower']
            result.pi_upper_3level = three_level_results['pi_upper']
            result.p_value_3level = three_level_results['p_value']
            result.tau_squared_3level = three_level_results['tau2']
            result.tau_3level = three_level_results['tau']
            result.tau2_ci_lower_3level = three_level_results['tau2_ci_lower']
            result.tau2_ci_upper_3level = three_level_results['tau2_ci_upper']
            result.sigma_squared_3level = three_level_results['sigma2']
            result.sigma2_ci_lower_3level = three_level_results['sigma2_ci_lower']
            result.sigma2_ci_upper_3level = three_level_results['sigma2_ci_upper']
            result.icc_l3 = three_level_results['icc_l3']
            result.icc_l2 = three_level_results['icc_l2']
            _full_fit = ((variance_comparison or {}).get('fits') or {}).get('full')
            if _full_fit is not None:
                result.aic_3level = _full_fit['aic']
                result.ll_3level = _full_fit['loglik']
            else:
                result.aic_3level = three_level_results['aic']
                result.ll_3level = three_level_results['log_lik_reml']
            result.df_infer_3level = three_level_results.get('df_robust')
            result.used_robust_3level = bool(
                three_level_results.get('used_robust', False)
            )

        if progress_callback:
            _rep3 = best_model == "3-Level" and result.has_3level
            _mu_rep = result.mu_3level if _rep3 else result.mu_random
            _p_rep = result.p_value_3level if _rep3 else result.p_value_random
            sig = "***" if _p_rep < 0.001 else "**" if _p_rep < 0.01 else "*" if _p_rep < 0.05 else "ns"
            progress_callback(f"✅ {best_model}: μ = {_mu_rep:.3f} {sig}")

        return result

    # -------------------------------------------------------------------------
    # MODEL SELECTION HELPER
    # -------------------------------------------------------------------------

    def _nested_variance_comparison(
        self,
        df: pd.DataFrame,
        alpha: float,
        three_level_results: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:
        """
        Three-level model vs its two reduced variance structures, all fitted by
        REML with the unified engine (Cell 1b) on the SAME rows, the SAME
        sampling V and the same intercept-only fixed effects:

            full     tau2 and sigma2 estimated        rma.mv sigma2 = c(NA, NA)
            between  sigma2 fixed at 0                rma.mv sigma2 = c(NA, 0)
            within   tau2   fixed at 0                rma.mv sigma2 = c(0, NA)

        Each reduced model is the full model with one component on its boundary,
        so AIC(reduced) - AIC(full) >= -2 always holds, and each LRT uses the
        50:50 chi2_0 : chi2_1 mixture (Self & Liang, 1987).

        Returns a JSON-safe dict (floats / strings / lists only).
        """
        if 'build_fit_spec' not in globals():
            return {'error': "unified model engine not loaded (run Cell 1b)"}
        try:
            spec = build_fit_spec(
                df,
                self.data_manager.effect_col,
                self.data_manager.var_col,
                vcv_dict=self.data_manager.vcv_matrices,
                vcv_reference_df=self.data_manager.analysis_data,
                label='overall',
            )
            fits = {
                'full': fit_mv(spec),
                'between': fit_mv(spec, sigma2=0.0),
                'within': fit_mv(spec, tau2=0.0),
            }
            comps = {
                'tau2': compare_fits(fits['full'], fits['within']),
                'sigma2': compare_fits(fits['full'], fits['between']),
            }
        except Exception as e:
            return {'error': f"{type(e).__name__}: {e}"}

        short = {'full': 'Three-level (τ² + σ²)',
                 'between': 'Between-study component only (σ² = 0)',
                 'within': 'Within-study component only (τ² = 0)'}
        aic_full = fits['full'].aic
        out_fits = {}
        for key, f in fits.items():
            out_fits[key] = {
                'label': short[key],
                'structure': f.structure,
                'tau2': float(f.tau2),
                'sigma2': float(f.sigma2),
                'mu': float(f.betas[0]),
                'se_model': float(f.se[0]),
                'loglik': float(f.loglik),
                'aic': float(f.aic),
                'bic': float(f.bic),
                'n_params': int(f.n_params),
                'delta_aic_vs_full': float(f.aic - aic_full),
                'converged': bool(f.converged),
            }

        tests = {}
        msgs: List[str] = []
        for comp, c in comps.items():
            tests[comp] = {
                'hypothesis': ('H0: τ² = 0 (no between-study variance)' if comp == 'tau2'
                               else 'H0: σ² = 0 (no within-study variance)'),
                'reduced': 'within' if comp == 'tau2' else 'between',
                'lrt': float(c.lrt),
                'df': int(c.df),
                'p_value': float(c.p_value),
                'delta_aic': float(c.delta_aic),
                'significant': bool(c.p_value < alpha),
            }
            msgs += c.warnings

        # Cross-check the legacy three-level optimiser (which supplies the
        # headline 3-level estimates and CIs) against the unified engine.
        if three_level_results:
            t_old = float(three_level_results.get('tau2', np.nan))
            s_old = float(three_level_results.get('sigma2', np.nan))
            t_new, s_new = fits['full'].tau2, fits['full'].sigma2
            scale = max(1e-4, t_new + s_new)
            if (abs(t_old - t_new) > 1e-3 * scale + 1e-6
                    or abs(s_old - s_new) > 1e-3 * scale + 1e-6):
                msgs.append(
                    f"The three-level estimates used for the pooled effect "
                    f"(τ² = {t_old:.5f}, σ² = {s_old:.5f}) differ from the unified "
                    f"engine's REML optimum (τ² = {t_new:.5f}, σ² = {s_new:.5f}).")

        acc = spec.accounting
        return {
            'method': 'REML',
            'alpha': float(alpha),
            'k_effects': int(spec.n_effects),
            'k_clusters': int(spec.n_clusters),
            'v_type': ('full shared-control V' if spec.uses_full_vcv else 'diagonal V'),
            'n_v_downgraded': len(acc.v_downgraded_clusters),
            'data_summary': acc.summary_text(),
            'fits': out_fits,
            'tests': tests,
            'aic_best': min(out_fits, key=lambda k: out_fits[k]['aic']),
            'warnings': msgs,
        }

    def _select_model(
        self,
        model_choice: str,
        has_3level: bool,
        variance_comparison: Optional[Dict[str, Any]],
        alpha: float,
    ) -> Tuple[str, str]:
        """
        Returns (best_model, basis).

        The three-level model is reported whenever it can be fitted: the
        nesting of effect sizes within studies is part of the design, not a
        hypothesis to test. If tau² or sigma² is ~0 the model reduces to the
        simpler structure by itself. The two-level model is reported only
        when the analyst asks for it (sensitivity analysis) or when the
        three-level model cannot be fitted. variance_comparison and alpha are
        not used to choose; the comparison is reported for information.
        """
        if _normalize_model_choice(model_choice) == TWO_LEVEL_CHOICE:
            return "2-Level", "analyst-specified two-level model (sensitivity analysis)"
        if has_3level:
            return "3-Level", "three-level model specified a priori (effect sizes nested within studies)"
        return "2-Level", "three-level model not estimable (no within-study replication, or it could not be fitted)"


# =============================================================================
# HTML TEMPLATE GENERATORS
# =============================================================================

class OverallHTMLTemplates:
    """
    Static HTML template generators for overall meta-analysis visualizations.
    All methods are pure functions returning HTML strings.
    """

    @staticmethod
    def gradient_card(
        value: str,
        subtitle: str = "",
        gradient: str = "linear-gradient(135deg, #667eea 0%, #764ba2 100%)"
    ) -> str:
        """Generate gradient card for highlighting pooled effect"""
        return f"""
        <div style='background: {gradient}; padding: 30px; border-radius: 10px; color: white;
                    margin-bottom: 20px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>
            <h1 style='margin: 0; font-size: 3.5em; text-align: center; font-weight: 700;'>{value}</h1>
            <p style='margin: 5px 0 0 0; text-align: center; font-size: 1.2em; opacity: 0.9;'>{subtitle}</p>
        </div>
        """

    @staticmethod
    def info_grid(items: list) -> str:
        """
        Generate responsive grid of info boxes.

        Args:
            items: List of dicts with 'label', 'value', and 'color' keys
        """
        html = "<div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 20px; margin-bottom: 20px;'>"

        for item in items:
            color = item.get('color', '#007bff')
            html += f"""
            <div style='background-color: #f8f9fa; padding: 20px; border-radius: 8px;
                        border-left: 5px solid {color}; box-shadow: 0 2px 4px rgba(0,0,0,0.05);'>
                <div style='color: #6c757d; font-size: 0.9em; text-transform: uppercase;
                            letter-spacing: 1px;'>{item['label']}</div>
                <div style='font-size: 1.6em; font-weight: bold; color: #2c3e50;'>{item['value']}</div>
            </div>
            """

        html += "</div>"
        return html

    @staticmethod
    def model_note(model_desc: str, k_obs: int, k_studies: int) -> str:
        """Generate model description note"""
        return f"""
        <div style='background-color: #e7f3ff; padding: 15px; border-radius: 5px;
                    color: #004085; font-size: 0.95em;'>
            <p style='margin: 0;'><b>ℹ️ Model Note:</b> {model_desc}
            (k = {k_obs} observations from {k_studies} studies)</p>
        </div>
        """

    @staticmethod
    def heterogeneity_badge(I2: float) -> Tuple[str, str, str]:
        """
        Get heterogeneity color, label, and description.

        Returns:
            Tuple of (color, label, description)
        """
        if I2 > 75:
            return "#dc3545", "High", "considerable"
        elif I2 > 50:
            return "#ffc107", "Substantial", "substantial"
        elif I2 > 25:
            return "#28a745", "Moderate", "moderate"
        else:
            return "#28a745", "Low", "low"

    @staticmethod
    def heterogeneity_display(I2: float, color: str, label: str) -> str:
        """Generate heterogeneity display card"""
        return f"""
        <div style='display: flex; align-items: center; gap: 20px; margin-bottom: 25px;'>
            <div style='background-color: {color}; padding: 20px; border-radius: 10px;
                        color: white; min-width: 150px; text-align: center;
                        box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>
                <h1 style='margin: 0; font-size: 2.5em;'>{I2:.1f}%</h1>
                <p style='margin: 5px 0 0 0;'>I² Statistic</p>
            </div>
            <div style='font-size: 1.1em; color: #555;'>
                The data shows <b>{label.lower()}</b> heterogeneity.<br>
                <span style='font-size: 0.9em; color: #888;'>This suggests that {I2:.1f}% of the
                variance is due to real differences between studies, not just sampling error.</span>
            </div>
        </div>
        """

    @staticmethod
    def heterogeneity_table(
      Q: float, df_Q: int, p_Q: float, I2: float, tau2: float,
      tau_method: str, het_label: str, tau: Optional[float] = None,
      tau2_ci_lower: Optional[float] = None, tau2_ci_upper: Optional[float] = None,
      sigma2: Optional[float] = None,
      sigma2_ci_lower: Optional[float] = None, sigma2_ci_upper: Optional[float] = None
  ) -> str:
      """Generate heterogeneity statistics table"""
      tau_str = f"{tau:.4f}" if tau is not None else "N/A"

      # Format CIs nicely (handle infinity bounds)
      def fmt_ci(lower, upper):
          if lower is None or upper is None: return ""
          u_str = "∞" if np.isinf(upper) else f"{upper:.4f}"
          return f" <br><span style='font-size:0.9em; color:#666'>[95% CI: {lower:.4f}, {u_str}]</span>"

      tau2_ci_str = fmt_ci(tau2_ci_lower, tau2_ci_upper)
      sigma2_ci_str = fmt_ci(sigma2_ci_lower, sigma2_ci_upper)

      # 1. Start building the HTML string
      html = f"""
      <table style='width: 100%; border-collapse: collapse; border: 1px solid #dee2e6;'>
          <thead style='background-color: #f1f3f5;'>
              <tr>
                  <th style='padding: 12px; text-align: left; border-bottom: 2px solid #dee2e6;'>Statistic</th>
                  <th style='padding: 12px; text-align: left; border-bottom: 2px solid #dee2e6;'>Value</th>
                  <th style='padding: 12px; text-align: left; border-bottom: 2px solid #dee2e6;'>Interpretation</th>
              </tr>
          </thead>
          <tbody>
              <tr>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'><b>Q-statistic</b></td>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'>{Q:.2f} (df = {df_Q})</td>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'>p = {p_Q:.4g} (Signif. if < {_alpha():g})</td>
              </tr>
              <tr>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'><b>I² (Inconsistency)</b></td>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'>{I2:.1f}%</td>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'>{het_label} Heterogeneity</td>
              </tr>
              <tr>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'><b>τ² (Between-Study Variance)</b></td>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'>{tau2:.4f}{tau2_ci_str}</td>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'>Estimated via {tau_method}</td>
              </tr>
              <tr>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'><b>τ (Between-Study SD)</b></td>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'>{tau_str}</td>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'>√τ² (effect-size scale)</td>
              </tr>
      """

      # 2. Inject Sigma2 row ONLY if it exists (i.e., a 3-level model was run)
      if sigma2 is not None and sigma2 > 0:
          html += f"""
              <tr style='background-color: #f8f9fa;'>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'><b>σ² (Within-Study Variance)</b></td>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'>{sigma2:.4f}{sigma2_ci_str}</td>
                  <td style='padding: 12px; border-bottom: 1px solid #dee2e6;'>Variation within clusters</td>
              </tr>
          """

      # 3. Close the table and return the final HTML
      html += """
          </tbody>
      </table>
      """

      return html


    @staticmethod
    def model_comparison_table(
        mu_fe: float,
        ci_lo_fe: float,
        ci_hi_fe: float,
        mu_re: float,
        ci_lo_re: float,
        ci_hi_re: float,
        mu_3l: Optional[float],
        ci_lo_3l: Optional[float],
        ci_hi_3l: Optional[float],
        best_model: str,
        ci_pct: float,
        tau_method: str = 'REML',
    ) -> str:
        """Pooled effect under each reported model. No AIC column: the
        two-level estimator below is not nested in the three-level model
        (different V and random structure), so their AICs are not comparable.
        Model fit is compared in nested_comparison_table()."""
        c_2l = "#d4edda" if best_model == "2-Level" else "#fff"
        c_3l = "#d4edda" if best_model == "3-Level" else "#fff"
        b_2l = "✅ Reported" if best_model == "2-Level" else ""
        b_3l = "✅ Reported" if best_model == "3-Level" else ""
        td = "padding: 12px; border-bottom: 1px solid #dee2e6;"

        html = f"""
        <table style='width: 100%; border-collapse: collapse; box-shadow: 0 2px 10px rgba(0,0,0,0.05);
                      margin-bottom:30px;'>
            <thead style='background-color: #2c3e50; color: white;'>
                <tr>
                    <th style='padding: 12px; text-align: left;'>Model</th>
                    <th style='padding: 12px; text-align: left;'>Variance structure</th>
                    <th style='padding: 12px; text-align: center;'>Pooled Effect [{ci_pct:.0f}% CI]</th>
                    <th style='padding: 12px; text-align: center;'>Status</th>
                </tr>
            </thead>
            <tbody>
                <tr style='background-color: #f8f9fa; color: #6c757d;'>
                    <td style='{td}'><i>Fixed-Effect (Baseline)</i></td>
                    <td style='{td} font-size: 0.9em;'><i>none (assumes I² = 0)</i></td>
                    <td style='{td} text-align: center;'>{mu_fe:.3f} [{ci_lo_fe:.3f}, {ci_hi_fe:.3f}]</td>
                    <td style='{td} text-align: center;'>-</td>
                </tr>
                <tr style='background-color: {c_2l};'>
                    <td style='{td}'><b>2-Level Random-Effects</b></td>
                    <td style='{td} font-size: 0.9em;'>one effect-level τ² ({tau_method}); sampling
                        variances only (diagonal V); cluster-robust SE</td>
                    <td style='{td} text-align: center;'>{mu_re:.3f} [{ci_lo_re:.3f}, {ci_hi_re:.3f}]</td>
                    <td style='{td} text-align: center;'>{b_2l}</td>
                </tr>
        """
        if mu_3l is not None:
            html += f"""
                <tr style='background-color: {c_3l};'>
                    <td style='{td}'><b>3-Level REML (Nested)</b></td>
                    <td style='{td} font-size: 0.9em;'>between-study τ² + within-study σ²;
                        full sampling V</td>
                    <td style='{td} text-align: center;'>{mu_3l:.3f} [{ci_lo_3l:.3f}, {ci_hi_3l:.3f}]</td>
                    <td style='{td} text-align: center;'>{b_3l}</td>
                </tr>
            """
        html += """
            </tbody>
        </table>
        """
        return html

    @staticmethod
    def nested_comparison_table(vc: Optional[Dict[str, Any]], selection_basis: str = '') -> str:
        """Nested variance-structure comparison (same rows, V and fixed effects)."""
        if not vc:
            return ("<p style='color:#6c757d;'><i>Nested variance-structure comparison not "
                    "available: the three-level model could not be fitted.</i></p>")
        if 'error' in vc:
            return (f"<p style='color:#856404;'>⚠️ Nested variance-structure comparison failed: "
                    f"{vc['error']}</p>")

        td = ("padding: 8px 10px; border-bottom: 1px solid #dee2e6; text-align: center; "
              "white-space: nowrap;")
        fmt_p = lambda p: "< 0.001" if p < 0.001 else f"{p:.3f}"
        test_for = {'within': vc['tests']['tau2'], 'between': vc['tests']['sigma2']}
        rows = ""
        for key in ('full', 'between', 'within'):
            f = vc['fits'][key]
            best = "background-color:#eef7ee;" if key == vc['aic_best'] else ""
            if key == 'full':
                lrt_cell, p_cell, d_cell = "—", "—", "0.00 (reference)"
            else:
                t = test_for[key]
                lrt_cell, p_cell = f"{t['lrt']:.3f}", fmt_p(t['p_value'])
                d_cell = f"{f['delta_aic_vs_full']:+.2f}"
            rows += f"""
                <tr style='{best}'>
                    <td style='{td} text-align:left; white-space:normal;'><b>{f['label']}</b></td>
                    <td style='{td}'>{f['tau2']:.5f}</td>
                    <td style='{td}'>{f['sigma2']:.5f}</td>
                    <td style='{td}'>{f['loglik']:.3f}</td>
                    <td style='{td}'>{f['n_params']}</td>
                    <td style='{td}'>{f['aic']:.2f}</td>
                    <td style='{td}'>{d_cell}</td>
                    <td style='{td}'>{lrt_cell}</td>
                    <td style='{td}'>{p_cell}</td>
                </tr>"""

        warn = ""
        if vc.get('warnings'):
            warn = ("<div style='padding:8px 12px; background:#fff3cd; color:#856404; "
                    "border-radius:4px; margin-top:8px;'>⚠️ " + "<br>⚠️ ".join(vc['warnings'])
                    + "</div>")
        basis = (f"<p style='margin:8px 0 0 0;'><b>Reported model:</b> {selection_basis}. "
                 f"The comparisons below are for information; they do not choose the model.</p>"
                 if selection_basis else "")
        return f"""
        <table style='width:100%; border-collapse:collapse; box-shadow:0 2px 10px rgba(0,0,0,0.05);'>
            <thead style='background-color:#34495e; color:white;'>
                <tr>
                    <th style='padding:8px 10px; text-align:left;'>Fitted structure</th>
                    <th style='padding:8px 10px;'>τ² (between)</th>
                    <th style='padding:8px 10px;'>σ² (within)</th>
                    <th style='padding:8px 10px;'>logLik</th>
                    <th style='padding:8px 10px;'>Params</th>
                    <th style='padding:8px 10px;'>AIC</th>
                    <th style='padding:8px 10px;'>ΔAIC vs full</th>
                    <th style='padding:8px 10px;'>LRT</th>
                    <th style='padding:8px 10px;'>p (χ̄²)</th>
                </tr>
            </thead>
            <tbody>{rows}
            </tbody>
        </table>
        <p style='font-size:0.85em; color:#6c757d; margin:6px 0 0 0;'>
            All three models: REML, the same {vc['k_effects']} effect sizes in {vc['k_clusters']}
            studies, the same {vc['v_type']} and the same intercept. Each reduced model fixes one
            variance component at 0, so ΔAIC ≥ −2 by construction. p-values use the 50:50
            χ²₀ : χ²₁ mixture for a variance component tested on its boundary (Self &amp; Liang, 1987).
            Lowest AIC shaded.
        </p>
        {basis}{warn}
        """

    @staticmethod
    def settings_info_box(
        effect_col: str,
        var_col: str,
        k_obs: int,
        k_studies: int
    ) -> str:
        """Generate settings information box"""
        return f"""
        <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px;
                    margin-top: 20px; border: 1px solid #dee2e6;'>
            <h4 style='margin-top: 0; color: #2c3e50;'>Current Data Configuration:</h4>
            <ul style='margin-bottom: 0; padding-left: 20px;'>
                <li><b>Effect Size Column:</b> <code>{effect_col}</code></li>
                <li><b>Variance Column:</b> <code>{var_col}</code></li>
                <li><b>Total Effect Sizes (k):</b> {k_obs}</li>
                <li><b>Unique Studies:</b> {k_studies}</li>
            </ul>
        </div>
        """


# =============================================================================
# PUBLICATION TEXT GENERATORS
# =============================================================================

class OverallPublicationTextGenerator:
    """
    Generates publication-ready text for manuscripts.
    """

    def __init__(self):
        pass

    def generate_methods_section(
        self,
        result: 'OverallResult',
        es_config: Dict[str, Any]
    ) -> str:
        """
        Generate Materials and Methods section with dynamic citations and model settings.
        """
        # Extract settings
        es_label = es_config.get('effect_label', 'Effect Size')
        es_short = es_config.get('effect_label_short', 'ES')
        es_type = es_config.get('type', 'unknown')

        use_kh = result.use_kh
        tau_method = result.tau_method
        dist_type = result.dist_type
        best_model = result.best_model

        # Define Citation Database
        db = {
            'hedges': "Hedges, L. V. (1981). Distribution theory for Glass's estimator of effect size and related estimators. <i>Journal of Educational Statistics</i>, 6(2), 107-128.",
            'lnRR': "Hedges, L. V., Gurevitch, J., & Curtis, P. S. (1999). The meta-analysis of response ratios in experimental ecology. <i>Ecology</i>, 80(4), 1150-1156.",
            'cohen': "Cohen, J. (1988). <i>Statistical power analysis for the behavioral sciences</i> (2nd ed.). Hillsdale, NJ: Erlbaum.",
            'lnCVR': "Nakagawa, S., Poulin, R., Mengersen, K., Reinhold, K., Engqvist, L., Lagisz, M., & Senior, A. M. (2015). Meta-analysis of variation: ecological and evolutionary applications and beyond. <i>Methods in Ecology and Evolution</i>, 6(2), 143-152.",
            'borenstein': "Borenstein, M., Hedges, L. V., Higgins, J. P. T., & Rothstein, H. R. (2009). <i>Introduction to Meta-Analysis</i>. Chichester, UK: John Wiley & Sons.",
            'metafor': "Viechtbauer, W. (2010). Conducting meta-analyses in R with the metafor package. <i>Journal of Statistical Software</i>, 36(3), 1-48.",
            'pm': "Paule, R. C., & Mandel, J. (1982). Consensus values and weighting factors. <i>Journal of Research of the National Bureau of Standards</i>, 87(5), 377-385.",
            'sj': "Sidik, K., & Jonkman, J. N. (2005). Simple heterogeneity variance estimation for meta-analysis. <i>Journal of the Royal Statistical Society: Series C</i>, 54(2), 367-384.",
            'reml': "Viechtbauer, W. (2005). Bias and efficiency of meta-analytic variance estimators in the random-effects model. <i>Journal of Educational and Behavioral Statistics</i>, 30(3), 261-293.",
            'dl': "DerSimonian, R., & Laird, N. (1986). Meta-analysis in clinical trials. <i>Controlled Clinical Trials</i>, 7(3), 177-188.",
            'kh': "Knapp, G., & Hartung, J. (2003). Improved tests for a random effects meta-regression with a single covariate. <i>Statistics in Medicine</i>, 22(17), 2693-2710.",
            'lajeunesse': "Lajeunesse, M. J. (2015). Bias and correction for the log response ratio in ecological meta-analysis. <i>Ecology</i>, 96(8), 2056-2063. https://doi.org/10.1890/14-2402.1",
            'sweeting': "Sweeting, M. J., Sutton, A. J., & Lambert, P. C. (2004). What to add to nothing? Use and avoidance of continuity corrections in meta-analysis of sparse data. <i>Statistics in Medicine</i>, 23(9), 1351-1375.",
            'nakagawa_cv': "Nakagawa, S., et al. (2023). A robust and readily implementable method for the meta-analysis of response ratios with and without missing standard deviations. <i>Ecology Letters</i>, 26(2), 232-244.",
            'tool': COMETA_CITATION,
            'self_liang': "Self, S. G., & Liang, K.-Y. (1987). Asymptotic properties of maximum likelihood estimators and likelihood ratio tests under nonstandard conditions. <i>Journal of the American Statistical Association</i>, 82(398), 605-610.",
            'crve': "Pustejovsky, J. E., & Tipton, E. (2018). Small-sample methods for cluster-robust variance estimation and hypothesis testing in fixed effects models. <i>Journal of Business & Economic Statistics</i>, 36(4), 672-683. https://doi.org/10.1080/07350015.2016.1247004"
        }

        active_refs = []

        # [1] Effect Size -- one reference per effect-size type
        _cfg = globals().get('ANALYSIS_CONFIG', {}) or {}
        es_kind = _cfg.get('effect_size_type', '')
        if es_kind not in ('hedges_g', 'cohen_d', 'lnRR', 'lnCVR', 'log_or', 'log_rr',
                           'MD', 'risk_diff', 'proportion', 'fisher_z'):
            # Older sessions without effect_size_type: infer from the labels.
            es_kind = {"Hedges' g": 'hedges_g', "Cohen's d": 'cohen_d', 'lnRR': 'lnRR',
                       'lnCVR': 'lnCVR', 'log_or': 'log_or', 'log_rr': 'log_rr', 'MD': 'MD',
                       'risk_diff': 'risk_diff', 'proportion': 'proportion',
                       "Fisher's z": 'fisher_z'}.get(es_type, '')
        es_key = {'hedges_g': 'hedges', 'cohen_d': 'cohen', 'lnRR': 'lnRR', 'lnCVR': 'lnCVR',
                  'log_or': 'log_or', 'log_rr': 'log_rr'}.get(es_kind, es_kind)
        ref_key = {'hedges_g': 'hedges', 'cohen_d': 'cohen', 'lnRR': 'lnRR', 'lnCVR': 'lnCVR',
                   'proportion': 'metafor'}.get(es_kind, 'borenstein')
        active_refs.append(db[ref_key])
        ref_es = len(active_refs)

        # [1a] Small-sample bias correction of the log ratio measures
        ref_bias = None
        bias_corr_used = (es_short in ('lnRR', 'lnCVR') and _cfg.get('data_type', 'raw') == 'raw'
                          and bool(_cfg.get('bias_correction', True)))
        if bias_corr_used:
            active_refs.append(db['lajeunesse'])
            ref_bias = len(active_refs)

        # [1b] Continuity Correction -- only when CoMeta computed the effect
        # sizes from 2x2 tables (not for pre-calculated data)
        ref_sweeting = None
        is_raw_data = _cfg.get('data_type', 'raw') == 'raw'
        if es_key in ['log_or', 'log_rr', 'risk_diff'] and is_raw_data:
            active_refs.append(db['sweeting'])
            ref_sweeting = len(active_refs)

        # [2] Missing Variance Strategy
        ref_nakagawa = None
        sd_text = ""
        if 'ANALYSIS_CONFIG' in globals():
            sd_strategy = ANALYSIS_CONFIG.get('sd_missing_strategy', 'none')
            if sd_strategy == 'nakagawa_all_cases':
                active_refs.append(db['nakagawa_cv'])
                ref_nakagawa = len(active_refs)
                sd_text = f" To ensure robust variance estimation, sampling variances for all observations were calculated using a weighted pooled coefficient of variation derived from the dataset [{ref_nakagawa}]."
            elif sd_strategy == 'nakagawa_missing_cases':
                active_refs.append(db['nakagawa_cv'])
                ref_nakagawa = len(active_refs)
                sd_text = f" Missing standard deviations were imputed via a weighted pooled coefficient of variation derived from complete cases within the dataset [{ref_nakagawa}]."
            elif sd_strategy == 'median_cv':
                sd_text = " Missing standard deviations were imputed using the median coefficient of variation (CV) from available data."
            elif sd_strategy == 'drop':
                sd_text = " Effect sizes with missing standard deviations were excluded."

        # [3] Tau Estimator. The three-level model is always fitted by REML;
        # the estimator choice applies to the two-level model only.
        three_level_fit = best_model == "3-Level" and result.has_3level
        tau_used = 'REML' if three_level_fit else tau_method
        tau_key = {'DL': 'dl', 'PM': 'pm', 'SJ': 'sj'}.get(tau_used, 'reml')
        active_refs.append(db[tau_key])
        ref_tau = len(active_refs)

        # [4] Robust Inference (CRVE or KH)
        ref_kh_text = ""
        if use_kh:
            # Describe the estimator that actually priced the inference, not
            # the one the data shape would suggest. CRVE can fail (singular
            # or non-PSD meat matrix) and fall back to classic Knapp-Hartung
            # in the 2-level path, or to plain GLS in the 3-level path;
            # claiming CR2 in those runs would misreport the method.
            _crve_ran = (result.used_robust_3level
                         if best_model == "3-Level" else result.used_robust)

            if _crve_ran:
                active_refs.append(db['crve'])
                ref_kh = len(active_refs)
                ref_kh_text = f" a cluster-robust variance estimator (CR2) with Satterthwaite approximations [{ref_kh}]"
            elif best_model == "3-Level":
                # CR2 failed for the 3-level fit: model-based SE, df = studies - 1
                ref_kh_text = (" model-based standard errors (the cluster-robust estimator could not be "
                               "computed for these data)")
            else:
                active_refs.append(db['kh'])
                ref_kh = len(active_refs)
                ref_kh_text = (f" the Knapp-Hartung adjustment, with its scale factor truncated "
                               f"at 1 so that intervals are never narrower than the standard "
                               f"random-effects intervals [{ref_kh}]")

        # [5] Co-Meta Tool
        active_refs.append(db['tool'])
        ref_tool = len(active_refs)

        # Format Modeling Strings
        tau_desc = {
            'REML': 'Restricted Maximum Likelihood (REML)',
            'DL': 'DerSimonian-Laird (DL)',
            'ML': 'Maximum Likelihood (ML)',
            'PM': 'Paule-Mandel (PM)',
            'SJ': 'Sidik-Jonkman (SJ)'
        }.get(tau_used, tau_used)

        inf_desc = f"a <i>t</i>-distribution and{ref_kh_text}" if use_kh and dist_type == 't' else "a <i>t</i>-distribution" if dist_type == 't' else "the standard normal distribution (<i>z</i>)"

        # Build HTML Text
        html = f"""
        <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt;
                    line-height: 1.8; padding: 20px; background-color: #ffffff;
                    border: 1px solid #eee; margin-bottom: 20px;'>

        <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;
                   margin-top: 0;'>Materials and Methods</h3>

        <p style='text-align: justify;'>
        <b>Data Preparation.</b> The effect size of interest was {es_label} ({es_short}), calculated using standard meta-analytic estimators [{ref_es}].
        """

        if es_key == 'hedges':
            html += " This metric incorporates the <i>J</i>-correction to adjust for small sample size bias."
        if es_key in ('hedges', 'cohen'):
            if es_key == 'cohen':
                ref_cohen = ref_es
            else:
                active_refs.append(db['cohen'])
                ref_cohen = len(active_refs)
            html += (f" Magnitudes were described with Cohen's conventional benchmarks (0.2 small, "
                     f"0.5 medium, 0.8 large) [{ref_cohen}].")
        elif bias_corr_used:
            html += (f" {es_short} values include the second-order correction for small-sample bias "
                     f"[{ref_bias}], as in metafor's <i>escalc</i> (the default since version 4.8).")
        elif ref_sweeting is not None:
            html += (f" To permit variance calculation, a continuity correction of 0.5 was added to "
                     f"all four cells of any 2×2 table containing a zero cell [{ref_sweeting}]")
            if es_key in ('log_or', 'log_rr'):
                html += "; studies with zero events in both groups were excluded."
            else:
                html += "."

        html += sd_text
        html += "</p>"

        # New Modeling Paragraph
        html += f"""
        <p style='text-align: justify;'>
        <b>Meta-Analytical Modeling.</b> To estimate the overall pooled effect, we employed a {best_model.lower().replace('-level', '-level')} random-effects framework. {"The variance components (between-study τ² and within-study σ²) were" if three_level_fit else "Between-study heterogeneity (τ²) was"} estimated using the {tau_desc} estimator [{ref_tau}]. To account for uncertainty in the variance estimates, statistical inferences and confidence intervals were computed using {inf_desc}.
        </p>
        """

        # Model-selection paragraph (only when the nested comparison ran)
        _vc = result.variance_comparison if result.has_3level else None
        if _vc and 'error' not in _vc:
            active_refs.append(db['self_liang'])
            ref_sl = len(active_refs)
            if result.best_model == "3-Level":
                _rule = ("The three-level structure was specified a priori to reflect the nesting "
                         "of effect sizes within studies; the nested comparisons are reported for "
                         "information and were not used to choose the model.")
            else:
                _rule = ("A two-level model with cluster-robust inference was reported at the "
                         "analyst's request, as a sensitivity analysis.")
            html += f"""
        <p style='text-align: justify;'>
        <b>Variance Structure.</b> Because several studies contributed more than one effect size,
        a three-level model with between-study (τ²) and within-study (σ²) variance components
        was fitted by REML using the full sampling variance–covariance matrix. It was compared
        with nested models fitted to the same effect sizes, sampling
        variance–covariance matrix and fixed effects, in which one variance component was
        fixed at zero; likelihood-ratio tests used the 50:50 mixture of χ²<sub>0</sub> and
        χ²<sub>1</sub> for a variance component on the boundary of its parameter space [{ref_sl}].
        {_rule}
        </p>
        """

        # Software Paragraph
        html += f"""
        <p style='text-align: justify;'>
        <b>Statistical Software.</b> All data preparation, multilevel meta-analytical modeling, and sensitivity analyses were conducted using <b>Co-Meta</b> [{ref_tool}].
        </p>

        <h4 style='color: #34495e; margin-top: 20px; font-size: 11pt;'>References</h4>
        <ol style='font-size: 10pt; color: #555;'>
        """

        for ref in active_refs:
            html += f"<li>{ref}</li>"

        html += "</ol></div>"

        return html

    def generate_results_section(
        self,
        result: 'OverallResult',
        es_config: Dict[str, Any],
        sens_result: Optional[Dict[str, Any]] = None
    ) -> str:
        """
        Generate complete Results section with interpretation and Automated Sensitivity Text.
        """
        # Extract values
        mu_p = result.mu_random if result.best_model == "2-Level" else result.mu_3level
        ci_lo_p = result.ci_lower_random if result.best_model == "2-Level" else result.ci_lower_3level
        ci_hi_p = result.ci_upper_random if result.best_model == "2-Level" else result.ci_upper_3level
        p_p = result.p_value_random if result.best_model == "2-Level" else result.p_value_3level
        tau_p = result.tau_squared_3level if result.best_model == "3-Level" and result.has_3level else result.tau_squared

        ci_pct = (1 - result.alpha) * 100

        # Effect Size Description
        es_type = es_config.get('type', 'effect size')
        es_map = {
            "Hedges' g": "Effect sizes were calculated as Hedges' g, a standardized mean difference corrected for small sample bias.",
            'lnRR': "Effect sizes were expressed as log response ratios (lnRR), calculated as the natural logarithm of the ratio between treatment and control group means.",
            'SMD': "Effect sizes were calculated as standardized mean differences (SMD).",
            "Cohen's d": "Effect sizes were calculated as Cohen's d.",
            'log_or': "Effect sizes were expressed as log odds ratios (log OR), calculated from 2×2 contingency tables of binary outcomes.",
            'log_rr': "Effect sizes were expressed as log risk ratios (log RR), calculated from 2×2 contingency tables as the logarithm of the ratio of event probabilities between groups.",
        }
        es_description = es_map.get(es_type, f"Effect sizes were calculated as {es_type}.")

        # Significance & formatting
        sig_text = "significant" if p_p < _alpha() else "non-significant"
        p_format = f"< 0.001" if p_p < 0.001 else f"= {p_p:.3f}"


        # Magnitude Interpretation. Cohen's (1988) benchmarks exist only for
        # standardized mean differences; other scales get no generic label
        # (what counts as large depends on the outcome; see Step 9b).
        effect_size_type_local = ANALYSIS_CONFIG.get('effect_size_type', '')
        abs_mu = abs(mu_p)
        if effect_size_type_local in ('hedges_g', 'cohen_d') or es_type in ("Hedges' g", "Cohen's d"):
            if abs_mu < 0.2:
                effect_interp = ", a negligible effect by Cohen's (1988) benchmarks"
            elif abs_mu < 0.5:
                effect_interp = ", a small effect by Cohen's (1988) benchmarks"
            elif abs_mu < 0.8:
                effect_interp = ", a medium effect by Cohen's (1988) benchmarks"
            else:
                effect_interp = ", a large effect by Cohen's (1988) benchmarks"
        else:
            effect_interp = ""

        # Heterogeneity Interpretation
        if result.I2 < 25:
            het_interp = "indicating low heterogeneity"
        elif result.I2 < 50:
            het_interp = "indicating moderate heterogeneity"
        elif result.I2 < 75:
            het_interp = "indicating substantial heterogeneity"
        else:
            het_interp = "indicating considerable heterogeneity"

        # Build HTML Text
        text = f"""
        <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt;
                    line-height: 1.8; padding: 20px; background-color: #ffffff;'>

        <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;'>
            Meta-Analysis Results
        </h3>

        <p style='text-align: justify;'>
        A total of <b>{result.k_obs}</b> effect sizes from <b>{result.k_studies}</b> independent
        studies were included in the meta-analysis. {es_description}
        </p>

        <p style='text-align: justify;'>
        """
        # Model Selection Narrative (nested comparison, Cell 1b engine)
        _vc = result.variance_comparison if result.has_3level else None
        _vc_ok = bool(_vc) and 'error' not in _vc

        def _fmt_p(p):
            return "< 0.001" if p < 0.001 else f"= {p:.3f}"

        _lrt_ref = ""
        if _vc_ok:
            _t = _vc['tests']['tau2']
            _s = _vc['tests']['sigma2']
            _lrt_ref = (
                f" For reference, fixing the between-study variance at zero gave "
                f"LRT = {_t['lrt']:.2f}, <i>p</i> {_fmt_p(_t['p_value'])} "
                f"(ΔAIC = {_t['delta_aic']:+.1f}), and fixing the within-study variance at "
                f"zero gave LRT = {_s['lrt']:.2f}, <i>p</i> {_fmt_p(_s['p_value'])} "
                f"(ΔAIC = {_s['delta_aic']:+.1f}).")

        if result.best_model == "3-Level" and result.has_3level:
            text += """
                To account for multiple effect sizes nested within studies, a <b>three-level
                random-effects model</b> with between-study (τ²) and within-study (σ²) variance
                components was specified a priori.
                """
            if _vc_ok:
                text += f"""
                For information, it was compared with nested models fitted by REML to the same
                {_vc['k_effects']} effect sizes, sampling variance–covariance matrix and fixed effects,
                each with one variance component fixed at zero (likelihood-ratio tests with the
                50:50 mixture of χ²<sub>0</sub> and χ²<sub>1</sub>; Self &amp; Liang, 1987).{_lrt_ref}
                """
                if not _t['significant']:
                    text += """
                The between-study component was not distinguishable from zero, so the estimate
                is close to that of a model without the study level.
                """
                if not _s['significant']:
                    text += """
                The within-study component was not distinguishable from zero, so the estimate
                effectively reduces to a study-level random-effects model.
                """
        elif _normalize_model_choice(result.model_choice) == TWO_LEVEL_CHOICE:
            text += f"""
                At the analyst's request, a <b>two-level random-effects model</b> with
                cluster-robust variance estimation was reported as a sensitivity analysis.{_lrt_ref}
                """
        elif result.k_obs > result.k_studies:
            text += f"""
                A three-level random-effects model could not be fitted
                ({result.selection_basis}); a <b>two-level random-effects model with
                cluster-robust variance estimation</b> is reported instead.
                """
        else:
            text += ("Because each study contributed a single effect size, a standard "
                     "two-level random-effects meta-analysis was conducted. ")

        # CRVE paragraph -- emitted only when the CR2 sandwich actually
        # priced the selected model's inference (see used_robust /
        # used_robust_3level). Nesting alone is not evidence that it ran.
        _crve_ran = (result.used_robust_3level
                     if result.best_model == "3-Level" else result.used_robust)
        if result.k_obs > result.k_studies and _crve_ran:
            text += """
            To ensure statistical inferences remained robust against intra-study correlations, standard errors, confidence intervals, and <i>p</i>-values for the pooled effect were estimated using cluster-robust variance estimation (CRVE). We employed the CR2 small-sample adjustment with Satterthwaite degrees-of-freedom approximations, utilizing the study identifier as the clustering variable (Pustejovsky & Tipton, 2018).
            """



        # Primary Results

        # Pick the df that actually priced THIS model's inference.
        _df_used = result.df_infer_3level if result.best_model == "3-Level" else result.df_infer
        # Only meaningful for t-based inference; the normal path has no df.
        if result.dist_type == 't' and _df_used is not None and np.isfinite(_df_used) and _df_used > 0:
            _df_clause = f", df = {_df_used:.1f}"
        else:
            _df_clause = ""

        text += f"""
        The analysis revealed a <b>{sig_text}</b> overall pooled effect of <b>{mu_p:.4f}</b>
        ({ci_pct:.0f}% CI [{ci_lo_p:.3f}, {ci_hi_p:.3f}], <i>p</i> {p_format}{_df_clause}){effect_interp}.
        Additionally, the {ci_pct:.0f}% Prediction Interval (PI) ranged from <b>{result.pi_lower_random if result.best_model=="2-Level" else result.pi_lower_3level:.3f}</b> to <b>{result.pi_upper_random if result.best_model=="2-Level" else result.pi_upper_3level:.3f}</b>.
        This interval indicates the expected range of true effects for future studies, accounting for both sampling error and estimated heterogeneity.
        </p>
        """

        # Fold Change Logic (Preserved exactly as you had it)
        if effect_size_type_local == 'log_or':
            OR_val = np.exp(mu_p)
            or_text = f"OR = {OR_val:.2f} (increased odds)" if mu_p > 0 else f"OR = {OR_val:.2f} (decreased odds)" if mu_p < 0 else f"OR = {OR_val:.2f} (no difference)"
            text += f"<p style='text-align: justify;'>Transforming the log odds ratio back to the original scale, the pooled <b>{or_text}</b>, indicating that the odds of the outcome in the experimental group are <b>{OR_val:.2f}</b> times those of the control group.</p>"
        elif effect_size_type_local == 'log_rr':
            RR_val = np.exp(mu_p)
            rr_text = f"RR = {RR_val:.2f} (increased risk)" if mu_p > 0 else f"RR = {RR_val:.2f} (decreased risk)" if mu_p < 0 else f"RR = {RR_val:.2f} (no difference)"
            text += f"<p style='text-align: justify;'>Transforming the log risk ratio back to the original scale, the pooled <b>{rr_text}</b>, indicating that the probability of the outcome in the experimental group is <b>{RR_val:.2f}</b> times that of the control group.</p>"
        elif effect_size_type_local == 'lnCVR' or es_type == 'lnCVR':
            CVR = np.exp(mu_p)
            cvr_text = (f"{(CVR - 1) * 100:.1f}% higher" if mu_p >= 0
                        else f"{(1 - CVR) * 100:.1f}% lower")
            text += (f"<p style='text-align: justify;'>Transforming the log coefficient-of-variation "
                     f"ratio back to the original scale (CVR = <b>{CVR:.2f}</b>), the relative "
                     f"variability (coefficient of variation) of the outcome was <b>{cvr_text}</b> in "
                     f"the experimental group than in the control group. lnCVR compares variability "
                     f"relative to the mean, not the means themselves.</p>")
        elif es_config.get('has_fold_change', False) or es_type == 'lnRR':
            RR = np.exp(mu_p)
            fold_text = f"{RR:.2f}× increase" if mu_p >= 0 else f"{1/RR:.2f}× decrease"
            pct_text = f"{(RR - 1) * 100:.1f}% increase" if mu_p >= 0 else f"{(1 - RR) * 100:.1f}% decrease"
            text += f"<p style='text-align: justify;'>Transforming the log-ratio back to the original scale, this corresponds to a <b>{fold_text}</b> in the outcome variable relative to the control group (equivalent to a <b>{pct_text}</b>).</p>"

        # Heterogeneity Paragraph
        p_Q_format = "< 0.001" if result.p_Q < 0.001 else f"= {result.p_Q:.3f}"
        het_phrase = het_interp.replace("indicating ", "").capitalize()
        text += f"""
        <p style='text-align: justify;'>
        {het_phrase} was observed among the effect sizes (<i>Q</i>({result.df_Q}) =
        {result.Q:.2f}, <i>p</i> {p_Q_format}, <i>I</i>² = <b>{result.I2:.1f}%</b>). The between-study
        variance (τ²) was estimated at <b>{tau_p:.4f}</b> using the
        <b>{'REML' if (result.best_model == "3-Level" and result.has_3level) else result.tau_method}</b> estimator.
        </p>
        """

        if result.best_model == "3-Level" and result.has_3level:
            text += f"""
            <p style='text-align: justify;'>
            Variance decomposition in the three-level model indicated that <b>{result.icc_l3:.1f}%</b>
            of the total variance was attributable to between-study differences
            (τ² = {result.tau_squared_3level:.4f}), while <b>{result.icc_l2:.1f}%</b> was due to
            within-study variance (σ² = {result.sigma_squared_3level:.4f}). """
            text += "This indicates that effect sizes vary more widely across different studies than they do within the same study, supporting the necessity of clustering at the study level to avoid inflating statistical significance." if result.icc_l3 > result.icc_l2 else "This indicates high within-study variability, suggesting that different experiments or conditions within the same paper exhibit substantial differences."
            text += "</p>"

        # =========================================================
        # AUTOMATED SENSITIVITY PARAGRAPH
        # =========================================================
        if sens_result is not None:
            # We have a valid tandem check!
            main_sig = p_p < _alpha()
            sens_sig = sens_result['p_value'] < _alpha()

            s_mu = sens_result['mu']
            s_cil = sens_result['ci_lower']
            s_ciu = sens_result['ci_upper']
            s_p = sens_result['p_value']
            s_p_fmt = "< 0.001" if s_p < 0.001 else f"= {s_p:.3f}"

            # Calculate dropped rows and difference in means
            dropped_k = result.k_obs - sens_result['k_obs']
            diff_mu = abs(mu_p - s_mu)

            if main_sig == sens_sig:
                # Robust
                text += f"""
                <p style='text-align: justify;'>
                <b>Robustness Check:</b> To assess the sensitivity of our findings to the standard deviation imputation strategy, a tandem analysis was conducted by restricting the dataset to complete cases only (<i>N</i> = {sens_result['k_obs']} observations, omitting {dropped_k} rows). This strict complete-case baseline yielded a qualitatively identical and {'significant' if sens_sig else 'non-significant'} pooled effect of {s_mu:.3f} ({ci_pct:.0f}% CI [{s_cil:.3f}, {s_ciu:.3f}], <i>p</i> {s_p_fmt}). The absolute difference in the pooled estimate between the imputed and complete-case models was minimal (Δ = {diff_mu:.3f}). Because the statistical conclusion remains robust regardless of missing data exclusion, we report the fully imputed model above to maximize statistical power.
                </p>
                """
            else:
                # Flipped
                text += f"""
                <p style='text-align: justify; color: #721c24; background-color: #f8d7da; padding: 10px; border-left: 4px solid #dc3545;'>
                <b>⚠️ Critical Sensitivity Warning:</b> To assess the sensitivity of our findings to the standard deviation imputation strategy, a tandem analysis was conducted by restricting the dataset to complete cases only (<i>N</i> = {sens_result['k_obs']} observations, omitting {dropped_k} rows). Crucially, this complete-case baseline yielded a <b>{'significant' if sens_sig else 'non-significant'}</b> pooled effect of {s_mu:.3f} ({ci_pct:.0f}% CI [{s_cil:.3f}, {s_ciu:.3f}], <i>p</i> {s_p_fmt})—a substantial shift (Δ = {diff_mu:.3f}) from the imputed model's estimate of {mu_p:.3f} ({ci_pct:.0f}% CI [{ci_lo_p:.3f}, {ci_hi_p:.3f}]). This represents a fundamental change in the statistical conclusion. Because the primary findings are highly sensitive to the imputation of missing variances, both models are reported, and results must be interpreted with extreme caution.
                </p>
                """

        text += f"""
        </div>
        """

        return text


# =============================================================================
# VIEW COMPONENTS (Tab Renderers)
# =============================================================================

class OverallResultsView:
    """
    Manages all UI rendering for overall meta-analysis.
    Contains zero business logic - only presentation code.
    """

    def __init__(self):
        """Initialize view with display settings"""
        self.templates = OverallHTMLTemplates()
        self.text_gen = OverallPublicationTextGenerator()

        # Create tab widgets
        self.stale_banner = widgets.HTML(value="")
        register_banner('overall', self.stale_banner)
        self.settings_status = widgets.HTML(value="")
        self.tab_main = widgets.Output()
        self.tab_hetero = widgets.Output()
        self.tab_compare = widgets.Output()
        self.tab_settings = widgets.Output()
        self.tab_publication = widgets.Output()
        self.tab_export = widgets.Output()

        self.tabs = widgets.Tab(children=[
            self.tab_main,
            self.tab_hetero,
            self.tab_compare,
            self.tab_settings,
            self.tab_publication,
            self.tab_export
        ])

        self.tabs.set_title(0, '📊 Primary Result')
        self.tabs.set_title(1, '📉 Heterogeneity')
        self.tabs.set_title(2, '⚖️ Model Selection')
        self.tabs.set_title(3, '⚙️ Settings')
        self.tabs.set_title(4, '📝 Publication Text')
        self.tabs.set_title(5, '💾 Export')

    # -------------------------------------------------------------------------
    # TAB 1: PRIMARY RESULTS
    # -------------------------------------------------------------------------

    def render_primary_tab(
        self,
        result: 'OverallResult',
        effect_col: str,
        var_col: str
    ) -> None:
        """Render primary results tab"""

        with self.tab_main:
            self.tab_main.clear_output()

            # Determine which results to show (winner model)
            if result.best_model == "3-Level":
                mu_p = result.mu_3level
                ci_lo_p = result.ci_lower_3level
                ci_hi_p = result.ci_upper_3level
                p_p = result.p_value_3level
                pi_lo_p = result.pi_lower_3level
                pi_hi_p = result.pi_upper_3level
                model_label = "3-Level REML"
                model_desc = "Adjusted for within-study dependency."

            else:
                mu_p = result.mu_random
                ci_lo_p = result.ci_lower_random
                ci_hi_p = result.ci_upper_random
                p_p = result.p_value_random
                pi_lo_p = result.pi_lower_random
                pi_hi_p = result.pi_upper_random
                model_label = f"Random-Effects ({result.tau_method})"
                model_desc = f"Standard 2-Level model using {result.tau_method} estimator."

            # Significance indicator
            sig = "***" if p_p < 0.001 else "**" if p_p < 0.01 else "*" if p_p < 0.05 else "ns"
            color = "#28a745" if p_p < _alpha() else "#6c757d"
            ci_pct = (1 - result.alpha) * 100

            # Header with winner badge
            display(HTML(f"""
            <div style='padding: 20px;'>
            <h2 style='color: #2c3e50; margin-bottom: 20px;'>{model_label}
                <span style='font-size:0.5em; background: #28a745; color: white; padding: 4px 8px;
                            border-radius: 4px; vertical-align: middle; margin-left: 10px;'>
                    REPORTED MODEL
                </span>
            </h2>
            """))

            # Main gradient card
            gradient_html = self.templates.gradient_card(
                value=f"{mu_p:.3f}",
                subtitle=f"Pooled Effect Size {sig}"
            )
            display(HTML(gradient_html))
            if result.selection_basis:
                display(HTML(f"<p style='color:#6c757d; margin-top:-10px;'>"
                             f"<b>Model:</b> {result.selection_basis}</p>"))
            if result.data_accounting and 'accounting_html' in globals():
                display(HTML(accounting_html(result.data_accounting)))

            # Info grid
            info_items = [
                {
                    'label': f'{ci_pct:.0f}% Confidence Interval',
                    'value': f'[{ci_lo_p:.3f}, {ci_hi_p:.3f}]',
                    'color': '#007bff'
                },
                {
                    'label': f'{ci_pct:.0f}% Prediction Interval', # <-- ADDED
                    'value': f'[{pi_lo_p:.3f}, {pi_hi_p:.3f}]',    # <-- ADDED
                    'color': '#17a2b8'                             # <-- ADDED
                },
                {
                    'label': 'P-value',
                    'value': f'{p_p:.4g}',
                    'color': color
                }
            ]

            display(HTML(self.templates.info_grid(info_items)))

            # Model note
            model_note = self.templates.model_note(model_desc, result.k_obs, result.k_studies)
            display(HTML(model_note + "</div>"))

    # -------------------------------------------------------------------------
    # TAB 2: HETEROGENEITY
    # -------------------------------------------------------------------------

    def render_heterogeneity_tab(self, result: 'OverallResult') -> None:
        """Render heterogeneity assessment tab"""

        with self.tab_hetero:
            self.tab_hetero.clear_output()

            display(HTML("<div style='padding: 20px;'>"))
            display(HTML("<h2 style='color: #2c3e50;'>Heterogeneity Assessment</h2>"))

            # Get heterogeneity styling
            het_color, het_label, het_desc = self.templates.heterogeneity_badge(result.I2)

            # Display heterogeneity card
            het_display = self.templates.heterogeneity_display(result.I2, het_color, het_label)
            display(HTML(het_display))

            # Heterogeneity table
            # Use 3-level tau/CI when the 3-level model is best and available
            _tau_disp = result.tau_3level if result.has_3level and result.best_model == "3-Level" else None
            _tau2 = result.tau_squared_3level if result.has_3level and result.best_model == "3-Level" else result.tau_squared
            _ci_lo = result.tau2_ci_lower_3level if result.has_3level and result.best_model == "3-Level" else None
            _ci_hi = result.tau2_ci_upper_3level if result.has_3level and result.best_model == "3-Level" else None
            _sig2 = result.sigma_squared_3level if result.has_3level and result.best_model == "3-Level" else None
            _sig2_lo = result.sigma2_ci_lower_3level if result.has_3level and result.best_model == "3-Level" else None
            _sig2_hi = result.sigma2_ci_upper_3level if result.has_3level and result.best_model == "3-Level" else None

            het_table = self.templates.heterogeneity_table(
                result.Q, result.df_Q, result.p_Q,
                result.I2, _tau2,
                result.tau_method, het_label,
                tau=_tau_disp,
                tau2_ci_lower=_ci_lo,
                tau2_ci_upper=_ci_hi,
                sigma2=_sig2,
                sigma2_ci_lower=_sig2_lo,
                sigma2_ci_upper=_sig2_hi
            )

            display(HTML(het_table))
            display(HTML("</div>"))

    # -------------------------------------------------------------------------
    # TAB 3: MODEL COMPARISON
    # -------------------------------------------------------------------------

    def render_comparison_tab(
        self,
        result: 'OverallResult',
        es_config: Dict[str, Any]
    ) -> None:
        """Render model comparison tab"""

        with self.tab_compare:
            self.tab_compare.clear_output()

            display(HTML("<div style='padding: 20px;'>"))
            display(HTML("<h3 style='color: #2c3e50;'>Model Selection</h3>"))
            display(HTML(
                "<p style='margin-bottom: 12px;'>Fit is compared only between <b>nested</b> models: "
                "the three-level model and the same model with one variance component fixed at "
                "zero, all fitted to the same effect sizes, the same sampling covariance matrix "
                "and the same fixed effects.</p>"))
            display(HTML("<h4 style='color:#2E86AB; margin:10px 0 6px 0;'>Nested variance-structure comparison</h4>"))
            display(HTML(self.templates.nested_comparison_table(
                result.variance_comparison, result.selection_basis)))
            display(HTML("<h4 style='color:#2E86AB; margin:24px 0 6px 0;'>Pooled effect by model</h4>"))

            # Model comparison table
            ci_pct = (1 - result.alpha) * 100
            comparison_table = self.templates.model_comparison_table(
                mu_fe=result.mu_fixed,
                ci_lo_fe=result.ci_lower_fixed,
                ci_hi_fe=result.ci_upper_fixed,
                mu_re=result.mu_random,
                ci_lo_re=result.ci_lower_random,
                ci_hi_re=result.ci_upper_random,
                mu_3l=result.mu_3level if result.has_3level else None,
                ci_lo_3l=result.ci_lower_3level if result.has_3level else None,
                ci_hi_3l=result.ci_upper_3level if result.has_3level else None,
                best_model=result.best_model,
                ci_pct=ci_pct,
                tau_method=result.tau_method,
            )
            display(HTML(comparison_table))

            # Visual Sensitivity Plot
            display(HTML("<h4 style='color:#2E86AB; margin-left:0; margin-top:0;'>📉 Visual Sensitivity Analysis</h4>"))

            self._render_sensitivity_plot(result, es_config)

            display(HTML("</div>"))

    def _render_sensitivity_plot(
        self,
        result: 'OverallResult',
        es_config: Dict[str, Any]
    ) -> None:
        """Render sensitivity analysis forest plot"""
        try:
            import matplotlib.pyplot as plt

            models = ['Fixed-Effect', f'Random ({result.tau_method})', '3-Level (Nested)']
            means = [result.mu_fixed, result.mu_random]
            lowers = [result.ci_lower_fixed, result.ci_lower_random]
            uppers = [result.ci_upper_fixed, result.ci_upper_random]
            colors = ['#95a5a6', '#95a5a6']

            if result.has_3level:
                means.append(result.mu_3level)
                lowers.append(result.ci_lower_3level)
                uppers.append(result.ci_upper_3level)
                colors.append('#95a5a6')
            else:
                models.pop()

            winner_idx = 2 if result.best_model == "3-Level" and result.has_3level else 1
            colors[winner_idx] = '#28a745'
            labels = [f"{m}\n{'★ Winner' if i==winner_idx else ''}" for i, m in enumerate(models)]

            fig, ax = plt.subplots(figsize=(8, 3))
            y_pos = np.arange(len(models))

            for i, (m, l, u, c) in enumerate(zip(means, lowers, uppers, colors)):
                ax.plot([l, u], [i, i], color=c, linewidth=2.5, marker='|', markersize=10)
                ax.plot(m, i, 'o', color=c, markersize=9, markeredgecolor='white')
                ax.text(m, i-0.25, f"{m:.3f}", ha='center', va='top', color=c,
                       fontweight='bold', fontsize=9)

            ax.set_yticks(y_pos)
            ax.set_yticklabels(labels, fontsize=10, fontweight='bold')
            ax.set_ylim(-0.5, len(models)-0.5)
            ax.invert_yaxis()

            null_val = es_config.get('null_value', 0)
            ax.axvline(null_val, color='black', linestyle=':', alpha=0.3)

            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_visible(False)
            ax.grid(axis='x', linestyle='--', alpha=0.3)
            ax.set_xlabel(f"Pooled Effect Size ({es_config.get('effect_label_short', 'ES')})")

            plt.tight_layout()
            plt.show()
        except ImportError:
            display(HTML("<p style='color: #6c757d;'><i>matplotlib not available for plotting</i></p>"))

    # -------------------------------------------------------------------------
    # TAB 4: SETTINGS
    # -------------------------------------------------------------------------

    def render_settings_tab(
        self,
        effect_col: str,
        var_col: str,
        k_obs: int,
        k_studies: int,
        widgets_dict: Dict[str, Any]
    ) -> None:
        """Render settings configuration tab"""

        with self.tab_settings:
            self.tab_settings.clear_output()

            display(HTML("<h3>Analysis Settings</h3>"))
            display(HTML("<p>Configure statistical corrections. <b>Changes are applied automatically.</b></p>"))

            # Display widgets safely packed inside a VBox so Jupyter doesn't lose them
            display(widgets.VBox([
                widgets_dict['model_selector'],
                widgets_dict['tau_method'],
                widgets_dict['use_kh'],
                widgets_dict['ci_dist'],
                widgets_dict['alpha'],
                self.settings_status
            ]))

            # Info box
            info_box = self.templates.settings_info_box(
                effect_col, var_col, k_obs, k_studies
            )
            display(HTML(info_box))

    # -------------------------------------------------------------------------
    # TAB 5: PUBLICATION TEXT
    # -------------------------------------------------------------------------

    def render_publication_tab(
        self,
        result: 'OverallResult',
        es_config: Dict[str, Any],
        sens_result: Optional[Dict[str, Any]] = None
    ) -> str:
        """
        Render publication text tab.
        """
        with self.tab_publication:
            self.tab_publication.clear_output()

            methods_html = self.text_gen.generate_methods_section(result, es_config)
            results_html = self.text_gen.generate_results_section(result, es_config, sens_result)

            display(HTML(methods_html))
            display(HTML(results_html))

            return methods_html + "<br><hr><br>" + results_html



    # -------------------------------------------------------------------------
    # TAB 6: EXPORT
    # -------------------------------------------------------------------------

    def render_export_tab(self, export_excel_callback: callable, export_json_callback: callable) -> None:
        """Render export tab with download buttons"""

        with self.tab_export:
            self.tab_export.clear_output()

            # Create a dedicated output widget for status messages so buttons aren't cleared
            self.export_status = widgets.Output()

            display(HTML("<h3 style='color: #2E86AB;'>💾 Download Reports & Settings</h3>"))

            # --- Excel Export ---
            display(HTML("<p><b>Audit Report:</b> Generate a full Excel audit trail including model settings, heterogeneity statistics, and the exclusion log.</p>"))
            btn_export_excel = widgets.Button(
                description="📥 Download Excel Audit Report",
                button_style='info',
                icon='file-excel',
                layout=widgets.Layout(width='320px', height='40px')
            )
            btn_export_excel.on_click(export_excel_callback)

            # --- JSON Export (Reproducibility) ---
            display(HTML("<p style='margin-top:15px;'><b>Reproducibility:</b> Download your exact configuration to reload later in Step 2 (Data Ingestion).</p>"))
            btn_export_json = widgets.Button(
                description="📥 Download Settings JSON",
                button_style='warning',
                icon='code',
                layout=widgets.Layout(width='320px', height='40px')
            )
            btn_export_json.on_click(export_json_callback)

            # Add the status output widget below the buttons
            display(widgets.VBox([
                btn_export_excel,
                widgets.HTML("<br>"),
                btn_export_json,
                widgets.HTML("<br>"),
                self.export_status
            ]))

    # -------------------------------------------------------------------------
    # ERROR DISPLAY
    # -------------------------------------------------------------------------

    def render_error(self, message: str, details: Optional[str] = None) -> None:
        """Render error message in main tab"""
        with self.tab_main:
            self.tab_main.clear_output()
            error_html = f"<div style='color: red; background-color: #f8d7da; padding: 15px; border-radius: 5px;'>❌ {message}</div>"
            if details:
                error_html += f"<pre style='margin-top: 10px;'>{details}</pre>"
            display(HTML(error_html))

# =============================================================================
# Purpose: Orchestrates data, analysis, and view components
# =============================================================================

import traceback
from typing import Optional, Dict, Any
from IPython.display import display, HTML
import ipywidgets as widgets


# =============================================================================
# MAIN CONTROLLER
# =============================================================================

class OverallController:
    """
    Master controller that orchestrates the entire overall meta-analysis workflow.
    Coordinates data management, statistical computation, and UI rendering.
    """

    def __init__(self, analysis_config: Dict[str, Any]):
        """
        Initialize controller with ANALYSIS_CONFIG.

        Args:
            analysis_config: Global ANALYSIS_CONFIG dictionary
        """
        self.analysis_config = analysis_config

        # Initialize components
        try:
            self.data_manager = OverallDataManager(analysis_config)
            self.engine = OverallEngine(self.data_manager)
            self.view = OverallResultsView()

            self._initialization_error = None

        except Exception as e:
            # If initialization fails, create minimal view to show error
            self.view = OverallResultsView()
            self.data_manager = None
            self.engine = None
            self._initialization_error = e

        # Create settings widgets
        self._create_settings_widgets()

    # -------------------------------------------------------------------------
    # SETTINGS WIDGETS CREATION
    # -------------------------------------------------------------------------

    def _create_settings_widgets(self) -> None:
        """Create all settings widgets"""

        self.alpha_widget = widgets.Dropdown(
            options=[('95% CI (α=0.05)', 0.05),
                     ('99% CI (α=0.01)', 0.01),
                     ('90% CI (α=0.10)', 0.10)],
            value=0.05,
            description='Confidence Level:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='400px')
        )

        self.model_selector = widgets.Dropdown(
            options=MODEL_CHOICES,
            value=THREE_LEVEL_CHOICE,
            description='Model:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='400px')
        )

        self.tau_method_widget = widgets.Dropdown(
            options=[
                ('REML (Restricted Maximum Likelihood)', 'REML'),
                ('DL (DerSimonian-Laird)', 'DL'),
                ('ML (Maximum Likelihood)', 'ML'),
                ('PM (Paule-Mandel)', 'PM'),
                ('SJ (Sidik-Jonkman)', 'SJ')
            ],
            value='REML',
            description='τ² Method:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='400px')
        )

        self.use_kh_widget = widgets.Checkbox(
            value=True,
            description='CR2 / Knapp-Hartung'
        )

        self.ci_dist_widget = widgets.Dropdown(
            options=[('t-distribution (Robust/Small Sample)', 't'),
                     ('Normal distribution (z) (Match R)', 'norm')],
            value='t',
            description='Inference:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='400px')
        )

        # Attach observers
        self.alpha_widget.observe(self._handle_settings_change, 'value')
        self.model_selector.observe(self._handle_settings_change, 'value')
        self.tau_method_widget.observe(self._handle_settings_change, 'value')
        self.use_kh_widget.observe(self._handle_settings_change, 'value')
        self.ci_dist_widget.observe(self._handle_settings_change, 'value')

    # -------------------------------------------------------------------------
    # MAIN EXECUTION METHOD
    # -------------------------------------------------------------------------

    def run_analysis(self) -> None:
        """
        Execute complete overall meta-analysis workflow with Automated Sensitivity Check.
        """
        # Clear all tabs EXCEPT settings
        for tab in [self.view.tab_main, self.view.tab_hetero, self.view.tab_compare,
                    self.view.tab_publication]:
            tab.clear_output()

        loading_html = """
        <div style='padding: 15px; font-family: sans-serif; background-color: #e2e3e5;
                    border-left: 4px solid #6c757d; color: #383d41; border-radius: 4px; margin-top: 15px;'>
            <h4 style='margin-top: 0; color: #383d41; display: flex; align-items: center;'>
                <span style='margin-right: 8px;'>⏳</span> Calculating Models & Running Sensitivity...
            </h4>
            <p style='margin-bottom: 0;'>Please wait.</p>
        </div>
        """

        with self.view.tab_main:
            display(HTML(loading_html))

        if hasattr(self.view, 'settings_status'):
            self.view.settings_status.value = loading_html

        if self._initialization_error:
            self._handle_initialization_error()
            return

        # Get settings from widgets
        alpha = self.alpha_widget.value
        dist_type = self.ci_dist_widget.value
        use_kh = self.use_kh_widget.value
        tau_method = self.tau_method_widget.value
        model_choice = self.model_selector.value

        self.data_manager.save_global_settings(
            alpha, dist_type, tau_method, use_kh, model_choice
        )

        def progress_callback(message: str):
            pass

        try:
            # 1. RUN MAIN ANALYSIS
            result = self.engine.run_analysis(
                alpha=alpha, dist_type=dist_type, use_kh=use_kh,
                tau_method=tau_method, model_choice=model_choice,
                progress_callback=progress_callback
            )

            if result is None:
                self.view.render_error("Analysis failed", "Unable to compute meta-analysis. Check your data.")
                return

            # 2. RUN AUTOMATED SENSITIVITY CHECK (Tandem Analysis)
            sens_result = None
            df_main = self.analysis_config.get('analysis_data')
            strat = self.analysis_config.get('sd_missing_strategy', 'drop')

            # If we imputed SDs, run the drop-case natively
            if strat not in ['drop', 'none', ''] and df_main is not None and 'sde' in df_main.columns:
                mask_complete = (df_main['sde'].notna() & df_main['sdc'].notna()
                                 & (df_main['sde'] > 0) & (df_main['sdc'] > 0))
                if not mask_complete.all():
                    df_drop = df_main[mask_complete].copy()
                    es_kind = self.analysis_config.get('effect_size_type', '')
                    ecol_s, vcol_s = self.data_manager.effect_col, self.data_manager.var_col
                    vcv_drop = self.data_manager.vcv_matrices
                    if es_kind in globals().get('SD_BASED_EFFECTS', ()):
                        # A true complete-case analysis: when the strategy
                        # replaced observed SDs (all-cases pooled CV), the
                        # stored yi and vi of complete rows are themselves
                        # imputed, so recompute both from the observed SDs.
                        if strat == 'nakagawa_all_cases':
                            df_drop = observed_sd_effect_sizes(
                                df_drop, es_kind, ecol_s, vcol_s,
                                self.analysis_config.get('se_col', 'se_cc'),
                                bool(self.analysis_config.get('bias_correction', True)))
                            df_drop = df_drop[np.isfinite(df_drop[ecol_s])
                                              & np.isfinite(df_drop[vcol_s])
                                              & (df_drop[vcol_s] > 0)]
                        # Sampling VCV (shared controls) rebuilt for exactly
                        # these rows: the full-data matrices cannot be aligned
                        # to a study whose rows were only partly kept.
                        if len(df_drop) > 0:
                            vcv_drop = build_vcv_matrices(df_drop, es_kind, vcol_s,
                                                          yi_col=ecol_s, warn_on_fallback=False)

                    # Need at least 2 for RE. If main model was 3-Level, we need to respect that structure.
                    if len(df_drop) > 1:
                        y_drop = df_drop[self.data_manager.effect_col].values

                        # Route based on the best_model selected in the primary run
                        if result.best_model == "3-Level" and result.has_3level and self.data_manager.check_3level_feasibility(df_drop):
                            try:
                                sens_3l = self.engine.three_level_engine.fit(
                                    df_drop,
                                    self.data_manager.effect_col,
                                    self.data_manager.var_col,
                                    vcv_drop,
                                    alpha=alpha,
                                    dist_type=dist_type
                                )
                                if sens_3l:
                                    # fit() returns mu/SE/df only; build the CI and p
                                    # exactly as the main 3-level run does.
                                    mu_s = sens_3l['mu']
                                    if use_kh:
                                        se_3l = sens_3l.get('se_robust', sens_3l['se'])
                                        df_3l = sens_3l.get('df_robust', df_drop['id'].nunique() - 1)
                                    else:
                                        se_3l = sens_3l['se']
                                        df_3l = df_drop['id'].nunique() - 1
                                    if dist_type == 't':
                                        crit_s = t.ppf(1 - alpha / 2, df_3l)
                                        p_s = 2 * (1 - t.cdf(abs(mu_s / se_3l), df_3l))
                                    else:
                                        crit_s = norm.ppf(1 - alpha / 2)
                                        p_s = 2 * (1 - norm.cdf(abs(mu_s / se_3l)))
                                    sens_result = {
                                        'k_obs': len(y_drop),
                                        'mu': mu_s,
                                        'ci_lower': mu_s - crit_s * se_3l,
                                        'ci_upper': mu_s + crit_s * se_3l,
                                        'p_value': p_s,
                                        'model': '3-Level',
                                    }
                            except Exception as e:
                                # Fall through to 2-level below, but say so
                                warnings.warn(f"Complete-case check: 3-level refit failed ({e}); using 2-level.")

                        # If 2-Level was best, OR if 3-Level was best but the dropped data can't support it
                        if sens_result is None:
                            sens_tau2 = self.engine.two_level_engine.estimate_tau2(df_drop, self.data_manager.effect_col, self.data_manager.var_col)
                            sens_res_dict = self.engine.two_level_engine.calculate_pooled_effect(
                                df=df_drop,
                                effect_col=self.data_manager.effect_col,
                                var_col=self.data_manager.var_col,
                                vcv_dict=vcv_drop,
                                tau2=sens_tau2,
                                alpha=alpha,
                                dist_type=dist_type,
                                use_kh=use_kh
                            )
                            sens_result = {
                                'k_obs': len(y_drop),
                                'mu': sens_res_dict['mu'],
                                'ci_lower': sens_res_dict['ci_lower'],
                                'ci_upper': sens_res_dict['ci_upper'],
                                'p_value': sens_res_dict['p_value'],
                                'model': '2-Level',
                            }


            # Render all tabs (Passing the new sens_result down)
            self._render_all_tabs(result, sens_result)

            self._save_results(result)
            clear_stale('overall')
            mark_stale(['subgroup', 'regression', 'spline', 'pub_bias', 'pet_peese', 'loo', 'cumulative', 'plots'], "Overall Meta-Analysis Re-run")

            if hasattr(self.view, 'settings_status'):
                self.view.settings_status.value = """
                <div style='padding: 10px; font-family: sans-serif; background-color: #d4edda;
                            border-left: 4px solid #28a745; color: #155724; border-radius: 4px; margin-top: 15px;'>
                    <b>✅ Settings applied and models updated!</b>
                </div>
                """

        except Exception as e:
            self._handle_unexpected_error(e)


    # -------------------------------------------------------------------------
    # TAB RENDERING
    # -------------------------------------------------------------------------

    def _render_all_tabs(self, result: 'OverallResult', sens_result: Optional[Dict] = None) -> None:
        """
        Render all tabs with results.
        """
        self.view.render_primary_tab(result, self.data_manager.effect_col, self.data_manager.var_col)
        self.view.render_heterogeneity_tab(result)
        self.view.render_comparison_tab(result, self.data_manager.es_config)

        if not getattr(self, '_settings_rendered', False):
            widgets_dict = {
                'model_selector': self.model_selector, 'tau_method': self.tau_method_widget,
                'use_kh': self.use_kh_widget, 'ci_dist': self.ci_dist_widget,
                'alpha': self.alpha_widget
            }
            self.view.render_settings_tab(
                self.data_manager.effect_col, self.data_manager.var_col,
                result.k_obs, result.k_studies, widgets_dict
            )
            self._settings_rendered = True

        # Tab 5: Publication Text (Now passing sens_result!)
        combined_text = self.view.render_publication_tab(
            result,
            self.data_manager.es_config,
            sens_result=sens_result
        )
        self.data_manager.save_publication_text(combined_text)

        self.view.render_export_tab(
            export_excel_callback=self._handle_export,
            export_json_callback=self._handle_export_json
        )


    # -------------------------------------------------------------------------
    # DATA PERSISTENCE
    # -------------------------------------------------------------------------

    def _save_results(self, result: 'OverallResult') -> None:
        """
        Save results to ANALYSIS_CONFIG.

        Args:
            result: OverallResult object
        """
        self.data_manager.save_overall_results(result)

    # -------------------------------------------------------------------------
    # EVENT HANDLERS
    # -------------------------------------------------------------------------

    def _handle_settings_change(self, change) -> None:
        """Handle settings widget change event"""
        self.run_analysis()

    def _handle_rerun_click(self, button) -> None:
        """Handle rerun button click"""
        self.run_analysis()

    def _handle_export(self, button) -> None:
        """Handle export button click"""
        try:
            # Call the external export function if it exists
            if 'export_analysis_report' in globals():
                export_analysis_report(
                    report_type='overall',
                    filename_prefix='Overall_Meta_Analysis'
                )
            else:
                with self.view.export_status:
                    clear_output()
                    display(HTML("<div style='padding: 10px; background-color: #fff3cd; border-left: 4px solid #ffc107; color: #856404; margin-top: 15px; border-radius: 4px; font-family: sans-serif;'>⚠️ <b>Export function not found.</b> Please run the export configuration cell first.</div>"))
        except Exception as e:
            with self.view.export_status:
                clear_output()
                display(HTML(f"<div style='padding: 10px; background-color: #f8d7da; border-left: 4px solid #dc3545; color: #721c24; margin-top: 15px; border-radius: 4px; font-family: sans-serif;'>❌ <b>Export failed:</b> {str(e)}</div>"))
                traceback.print_exc()

    def _handle_export_json(self, button) -> None:
        """Handle JSON export button click for Reproducibility Mode"""
        import json
        import base64
        import datetime
        from IPython.display import display, Javascript

        try:
            with self.view.export_status:
                clear_output() # Clean previous messages in the status area only
                display(HTML("<div style='padding: 10px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; margin-top: 15px; border-radius: 4px; font-family: sans-serif;'>⏳ <b>Generating JSON Settings...</b> Please wait.</div>"))

                # Check if the export function from Cell 2 exists
                if 'export_config_for_reproducibility' in globals():
                    # Generate clean config dictionary
                    config_clean = export_config_for_reproducibility(self.analysis_config)
                    json_str = json.dumps(config_clean, indent=2)

                    # Base64 encode for browser download
                    b64 = base64.b64encode(json_str.encode()).decode()
                    filename = f"analysis_settings_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}.json"

                    # Javascript trigger to download file directly to user's computer
                    payload = f"""
                    var link = document.createElement('a');
                    link.href = 'data:application/json;base64,{b64}';
                    link.download = '{filename}';
                    document.body.appendChild(link);
                    link.click();
                    document.body.removeChild(link);
                    """
                    display(Javascript(payload))

                    clear_output()
                    display(HTML(f"<div style='padding: 10px; background-color: #d4edda; border-left: 4px solid #28a745; color: #155724; margin-top: 15px; border-radius: 4px; font-family: sans-serif;'>✅ <b>Download triggered:</b> {filename}</div>"))
                else:
                    clear_output()
                    display(HTML("<div style='padding: 10px; background-color: #f8d7da; border-left: 4px solid #dc3545; color: #721c24; margin-top: 15px; border-radius: 4px; font-family: sans-serif;'>❌ <b>Error:</b> <code>export_config_for_reproducibility</code> function not found. Please ensure you ran Cell 1 (Environment Setup).</div>"))

        except Exception as e:
            with self.view.export_status:
                clear_output()
                display(HTML(f"<div style='padding: 10px; background-color: #f8d7da; border-left: 4px solid #dc3545; color: #721c24; margin-top: 15px; border-radius: 4px; font-family: sans-serif;'>❌ <b>Export failed:</b> {str(e)}</div>"))
                traceback.print_exc()

    # -------------------------------------------------------------------------
    # ERROR HANDLING
    # -------------------------------------------------------------------------

    def _handle_initialization_error(self) -> None:
        """Handle errors during controller initialization"""
        error = self._initialization_error
        if error is None:
            return

        error_type = type(error).__name__
        error_msg = str(error)
        details = traceback.format_exc()

        self.view.render_error(f"{error_type}: {error_msg}", details)

    def _handle_unexpected_error(self, error: Exception) -> None:
        """Handle unexpected errors during analysis"""
        error_type = type(error).__name__
        error_msg = str(error)
        details = traceback.format_exc()

        self.view.render_error(f"{error_type}: {error_msg}", details)


# =============================================================================
# EXECUTION ENTRY POINT
# =============================================================================

def run_overall_meta_analysis():
    """
    Main entry point for overall meta-analysis.
    Call this function to display the UI and enable analysis.
    """
    try:
        # Check for ANALYSIS_CONFIG
        if 'ANALYSIS_CONFIG' not in globals():
            display(HTML("""
            <div style='padding: 15px; background-color: #f8d7da; border-left: 5px solid #dc3545; color: #721c24; border-radius: 4px; font-family: sans-serif; margin-bottom: 15px;'>
                <h4 style='margin: 0 0 5px 0; display: flex; align-items: center;'><span style='margin-right: 8px;'>❌</span> Missing Configuration</h4>
                <p style='margin: 0; font-size: 14px;'><code>ANALYSIS_CONFIG</code> not found. Please run the previous data loading and preparation cells before executing this analysis.</p>
            </div>
            """))
            return

        # Create controller
        controller = OverallController(ANALYSIS_CONFIG)

        # =================================================================
        # REPRODUCIBILITY MODE (CELL 7): Auto-populate Settings
        # =================================================================
        if ANALYSIS_CONFIG.get('is_reproducing', False):
            gs = ANALYSIS_CONFIG.get('global_settings', {})

            # Helper to set widget values silently without triggering analysis 6 times
            def set_silent(widg, val):
                if val is None:
                    return

                # Check validity for Dropdowns (Checkboxes don't have 'options')
                if hasattr(widg, 'options'):
                    valid_opts = [opt[1] if isinstance(opt, tuple) else opt for opt in widg.options]
                    if val not in valid_opts:
                        return

                # Safely detach observer, update value, reattach
                try:
                    widg.unobserve(controller._handle_settings_change, 'value')
                except ValueError:
                    pass # Was not observing yet

                widg.value = val
                widg.observe(controller._handle_settings_change, 'value')

            # Apply loaded settings
            set_silent(controller.alpha_widget, gs.get('alpha'))
            set_silent(controller.model_selector, _normalize_model_choice(gs.get('model_choice')))
            set_silent(controller.tau_method_widget, gs.get('tau_method'))
            set_silent(controller.use_kh_widget, gs.get('use_kh'))
            set_silent(controller.ci_dist_widget, gs.get('dist_type'))

            # Display Banner
            msg = f"Model: <b>{gs.get('model_choice')}</b> | τ² Method: <b>{gs.get('tau_method')}</b> | Inference: <b>{gs.get('dist_type')}</b>"
            display(HTML(f"<div style='background:#e8f4f8; border-left:4px solid #17a2b8; padding:12px; margin-bottom:10px; border-radius:4px; color:#0c5460;'>🔄 <b>Reproducibility Mode Active</b> &nbsp;|&nbsp; {msg}</div>"))


        # Display tabs
        display(widgets.VBox([controller.view.stale_banner, controller.view.tabs]))

        # Run initial analysis
        controller.run_analysis()

        # Store controller globally for access (optional)
        globals()['_overall_controller'] = controller

    except Exception as e:
        display(HTML(f"""
        <div style='padding: 15px; background-color: #f8d7da; border-left: 5px solid #dc3545; color: #721c24; border-radius: 4px; font-family: sans-serif; margin-bottom: 15px;'>
            <h4 style='margin: 0 0 5px 0; display: flex; align-items: center;'><span style='margin-right: 8px;'>❌</span> Fatal Error: {type(e).__name__}</h4>
            <p style='margin: 0 0 10px 0; font-size: 14px;'>{str(e)}</p>
            <p style='margin: 0; font-size: 12px;'>See the system output below for the full traceback.</p>
        </div>
        """))
        import traceback
        traceback.print_exc()



# =============================================================================
# RUN THE ANALYSIS
# =============================================================================

run_overall_meta_analysis()

In [ ]:
#@title ⚙️ 8. Subgroup Analysis: Configuration
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")


# =============================================================================
# SUBGROUP ANALYSIS CONFIGURATION (DASHBOARD VERSION)
# Purpose: Configure moderator variables with organized tabbed interface
# Dependencies: Step 7 (overall_results, analysis_data)
# Outputs: ANALYSIS_CONFIG['subgroup_config']
# =============================================================================

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import datetime

# --- 1. CREATE TAB LAYOUT ---
tab_config = widgets.VBox()
tab_moderators = widgets.Output()
tab_thresholds = widgets.Output()
tab_details = widgets.Output()

tabs = widgets.Tab(children=[tab_config, tab_moderators, tab_thresholds, tab_details])
tabs.set_title(0, '📋 Configuration')
tabs.set_title(1, '📊 Moderator Preview')
tabs.set_title(2, '⚙️ Thresholds')
tabs.set_title(3, '📝 Details')

# --- 2. WIDGETS ---
analysis_type_widget = widgets.RadioButtons(
    options=[('Single-Factor Analysis', 'single'), ('Two-Factor Analysis (Interaction)', 'two_way')],
    value='single', description='', layout=widgets.Layout(width='auto')
)

moderator1_widget = None
moderator2_widget = None

min_papers_widget = widgets.IntSlider(
    value=3, min=1, max=10, step=1, description='Min Papers:',
    style={'description_width': '120px'}, layout=widgets.Layout(width='400px')
)

min_obs_widget = widgets.IntSlider(
    value=5, min=2, max=20, step=1, description='Min Observations:',
    style={'description_width': '120px'}, layout=widgets.Layout(width='400px')
)

run_button = widgets.Button(
    description='💾 Save Configuration & Proceed',
    button_style='success',
    layout=widgets.Layout(width='400px', height='50px'),
    style={'font_weight': 'bold'},
    tooltip='Click to save configuration for use in the next cell'
)

run_button_output = widgets.Output()
status_output = widgets.Output()

# --- 3. INITIALIZATION ---
def initialize_configuration():
    global moderator1_widget, moderator2_widget

    with tab_details:
        clear_output()
        print("="*70)
        print("INITIALIZATION & VALIDATION")
        print("="*70)
        print(f"Timestamp: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

        try:
            effect_col = ANALYSIS_CONFIG['effect_col']
            var_col = ANALYSIS_CONFIG['var_col']
            es_config = ANALYSIS_CONFIG['es_config']
            overall_results = ANALYSIS_CONFIG['overall_results']

            print("✓ Prerequisites Check:")
            print(f"  • Effect: {es_config['effect_label']} ({es_config['effect_label_short']})")
            print(f"  • Q: {overall_results['Qt']:.4f}, I²: {overall_results['I_squared']:.2f}%")
        except KeyError as e:
            print(f"❌ ERROR: {e}")
            print("  Please run Step 7 (Overall Meta-Analysis) first")
            raise

      # --- DATA LOADING ---
        if 'ANALYSIS_CONFIG' not in globals() or 'analysis_data' not in ANALYSIS_CONFIG:
            print("❌ ERROR: Analysis data not found. Please run Step 6 (Effect Size Calculation) first.")
            return

        analysis_data = ANALYSIS_CONFIG['analysis_data']


        k_total, k_papers = len(analysis_data), analysis_data['id'].nunique()
        print(f"\n✓ Dataset: {k_total} obs, {k_papers} papers, {k_total/k_papers:.2f} avg")

        if k_total < 10:
            print(f"⚠️  WARNING: Limited data ({k_total} obs)")
        elif k_total < 20:
            print(f"⚠️  CAUTION: Moderate data ({k_total} obs)")

        excluded = ['xe', 'sde', 'ne', 'xc', 'sdc', 'nc', 'id', 'sde_imputed', 'sdc_imputed',
                   'cv_e', 'cv_c', 'sde_was_imputed', 'sdc_was_imputed',
                   effect_col, var_col, ANALYSIS_CONFIG.get('se_col', ''), 'w_fixed', 'w_random', 'ci_width']

        if es_config.get('has_fold_change'):
            excluded.extend(['Response_Ratio', 'RR_CI_lower', 'RR_CI_upper', 'fold_change',
                           'Percent_Change', 'Odds_Ratio', 'OR_CI_lower', 'OR_CI_upper'])

        if 'hedges_g' in effect_col or 'cohen_d' in effect_col:
            excluded.extend(['df', 'sp', 'sp_squared', 'cohen_d', 'hedges_j'])

        excluded.extend([c for c in analysis_data.columns if 'CI_' in c or 'ci_' in c])

        # Moderators: text, categorical, or numeric with few distinct values
        available_moderators = []
        for col in analysis_data.columns:
            if col in excluded: continue

            # Check 1: Is it text/string?
            is_text = is_text_dtype(analysis_data[col])

            # Check 2: Is it a Category? (Common in R data)
            is_category = isinstance(analysis_data[col].dtype, pd.CategoricalDtype)

            # Check 3: Is it a Number behaving like a Group? (e.g. Year, Grade)
            is_group_like_number = False
            if pd.api.types.is_numeric_dtype(analysis_data[col]):
                # If it has fewer than 20 unique values, treat it as a subgroup
                if analysis_data[col].nunique() < 20 and analysis_data[col].nunique() > 1:
                    is_group_like_number = True

            # If ANY check passes, keep it
            if (is_text or is_category or is_group_like_number):
                available_moderators.append(col)

        print(f"\n✓ Found {len(available_moderators)} moderators:")
        for mod in available_moderators:
            print(f"  • {mod}: {analysis_data[mod].nunique()} categories")

        if not available_moderators:
            print("❌ ERROR: No moderators found")
            raise ValueError("No moderators available")

        moderator1_widget = widgets.Dropdown(
            options=available_moderators, value=available_moderators[0],
            description='Moderator 1:', style={'description_width': '100px'},
            layout=widgets.Layout(width='450px')
        )

        moderator2_widget = widgets.Dropdown(
            options=['None'] + available_moderators, value='None',
            description='Moderator 2:', style={'description_width': '100px'},
            layout=widgets.Layout(width='450px')
        )

        analysis_type_widget.observe(update_all_tabs, names='value')
        moderator1_widget.observe(update_all_tabs, names='value')
        moderator2_widget.observe(update_all_tabs, names='value')
        min_papers_widget.observe(update_thresholds_tab, names='value')
        min_obs_widget.observe(update_thresholds_tab, names='value')
        run_button.on_click(save_configuration)

        print("\n✓ Initialized successfully")
        return available_moderators

# --- 4. TAB UPDATES ---
def update_config_tab(change=None):
    # Clear any previous save messages
    with status_output:
        clear_output()

    items = []

    analysis_type = analysis_type_widget.value

    # 1. Header & Help (Reduced margin-bottom from 15px to 5px)
    items.append(widgets.HTML("<h3 style='margin-top:0; margin-bottom:5px;'>Configure Subgroup Analysis</h3>"))

    if analysis_type == 'single':
        help_text = """<div style='background:#e7f3ff; padding:8px; border-radius:6px; border-left:4px solid #0066cc; margin-bottom:5px;'>
            <b>📊 Single-Factor Subgroup Analysis</b><br>
            <span style='font-size:12px; color:#555;'>Test if effect varies across ONE moderator. Best for primary hypotheses (10+ obs/group).</span></div>"""
    else:
        help_text = """<div style='background:#fff3cd; padding:8px; border-radius:6px; border-left:4px solid #ff9800; margin-bottom:5px;'>
            <b>📊 Two-Factor Analysis (Interaction)</b><br>
            <span style='font-size:12px; color:#555;'>Test combinations of TWO moderators. Requires 3-5 studies/combo, 20+ total obs.</span></div>"""
    items.append(widgets.HTML(help_text))

    # 2. Controls (Reduced margin-top from 20px to 10px)
    items.append(widgets.HTML("<h4 style='margin-top:10px; margin-bottom:5px;'>1. Select Analysis Type</h4>"))
    items.append(analysis_type_widget)

    items.append(widgets.HTML("<h4 style='margin-top:10px; margin-bottom:5px;'>2. Select Moderator(s)</h4>"))
    if moderator1_widget:
        items.append(moderator1_widget)

    if analysis_type == 'two_way' and moderator2_widget:
        items.append(moderator2_widget)

    # 3. Thresholds (Reduced margins)
    items.append(widgets.HTML("<h4 style='margin-top:10px; margin-bottom:5px;'>3. Set Quality Thresholds</h4>"))
    items.append(widgets.HTML("<p style='color:#666; font-size:12px; margin:0 0 5px 0;'>Adjust in <b>⚙️ Thresholds</b> tab</p>"))

    threshold_info = widgets.HBox([
        widgets.HTML(f"<div style='padding:5px 8px; background:#f0f0f0; border-radius:4px; margin-right:10px; font-size:13px;'>"
                    f"<b>Min Papers:</b> {min_papers_widget.value}</div>"),
        widgets.HTML(f"<div style='padding:5px 8px; background:#f0f0f0; border-radius:4px; font-size:13px;'>"
                    f"<b>Min Obs:</b> {min_obs_widget.value}</div>")
    ])
    items.append(threshold_info)

    # 4. Save Section (Reduced margins)
    items.append(widgets.HTML("<h4 style='margin-top:10px; margin-bottom:5px;'>4. Save Configuration</h4>"))

    items.append(run_button)
    items.append(status_output)

    # Update Children
    tab_config.children = tuple(items)

def update_moderators_tab(change=None):
    with tab_moderators:
        clear_output()

        # --- LOAD DATA ---
        if 'ANALYSIS_CONFIG' not in globals() or 'analysis_data' not in ANALYSIS_CONFIG:
            return # Data not ready yet
        analysis_data = ANALYSIS_CONFIG['analysis_data']

        mod1 = moderator1_widget.value
        analysis_type = analysis_type_widget.value
        mod2 = moderator2_widget.value if analysis_type == 'two_way' and moderator2_widget.value != 'None' else None

        display(HTML("<h3 style='margin-top:0;'>Moderator Variable Preview</h3>"))
        display(HTML(f"<h4>📊 {mod1}</h4>"))

        mod1_counts = analysis_data[mod1].value_counts().sort_index()

        table_html = """<table style='width:100%; border-collapse:collapse;'>
            <tr style='background:#f0f0f0; border-bottom:2px solid #ddd;'>
                <th style='text-align:left; padding:8px;'>Category</th>
                <th style='text-align:right; padding:8px;'>Observations</th>
                <th style='text-align:right; padding:8px;'>Papers</th>
                <th style='text-align:right; padding:8px;'>Percent</th></tr>"""

        for category, count in mod1_counts.items():
            papers = analysis_data[analysis_data[mod1] == category]['id'].nunique()
            pct = (count / len(analysis_data)) * 100
            row_color = '#fff' if count >= 5 else '#fff3cd'
            table_html += f"""<tr style='background:{row_color}; border-bottom:1px solid #eee;'>
                <td style='padding:6px;'>{category}</td>
                <td style='text-align:right; padding:6px;'><b>{count}</b></td>
                <td style='text-align:right; padding:6px;'>{papers}</td>
                <td style='text-align:right; padding:6px;'>{pct:.1f}%</td></tr>"""

        table_html += "</table>"
        display(HTML(table_html))

        min_group = mod1_counts.min()
        if min_group < 5:
            display(HTML(f"<div style='background:#fff3cd; padding:10px; margin-top:10px; border-radius:4px;'>"
                        f"⚠️ Smallest group: {min_group} obs - consider adjusting thresholds</div>"))
        else:
            display(HTML("<div style='background:#d4edda; padding:10px; margin-top:10px; border-radius:4px;'>"
                        "✓ All groups have ≥5 observations</div>"))

        if mod2:
            display(HTML(f"<h4 style='margin-top:25px;'>📊 {mod2}</h4>"))
            mod2_counts = analysis_data[mod2].value_counts().sort_index()

            table2_html = """<table style='width:100%; border-collapse:collapse;'>
                <tr style='background:#f0f0f0; border-bottom:2px solid #ddd;'>
                    <th style='text-align:left; padding:8px;'>Category</th>
                    <th style='text-align:right; padding:8px;'>Observations</th>
                    <th style='text-align:right; padding:8px;'>Papers</th>
                    <th style='text-align:right; padding:8px;'>Percent</th></tr>"""

            for category, count in mod2_counts.items():
                papers = analysis_data[analysis_data[mod2] == category]['id'].nunique()
                pct = (count / len(analysis_data)) * 100
                table2_html += f"""<tr style='border-bottom:1px solid #eee;'>
                    <td style='padding:6px;'>{category}</td>
                    <td style='text-align:right; padding:6px;'><b>{count}</b></td>
                    <td style='text-align:right; padding:6px;'>{papers}</td>
                    <td style='text-align:right; padding:6px;'>{pct:.1f}%</td></tr>"""

            table2_html += "</table>"
            display(HTML(table2_html))

            display(HTML(f"<h4 style='margin-top:25px;'>🔀 Combination Matrix: {mod1} × {mod2}</h4>"))
            crosstab = pd.crosstab(analysis_data[mod1], analysis_data[mod2], margins=True, margins_name='Total')
            display(crosstab.style.background_gradient(cmap='Blues', subset=pd.IndexSlice[crosstab.index[:-1], crosstab.columns[:-1]]))

            n_empty = (crosstab.iloc[:-1, :-1] == 0).sum().sum()
            min_cell = crosstab.iloc[:-1, :-1].min().min()

            if n_empty > 0:
                display(HTML(f"<div style='background:#f8d7da; padding:10px; margin-top:10px; border-radius:4px;'>"
                            f"⚠️ {n_empty} empty combinations - will be excluded</div>"))
            elif min_cell < 3:
                display(HTML(f"<div style='background:#fff3cd; padding:10px; margin-top:10px; border-radius:4px;'>"
                            f"⚠️ Min cell: {min_cell} - results may be unstable</div>"))
            elif min_cell < 5:
                display(HTML(f"<div style='background:#fff3cd; padding:10px; margin-top:10px; border-radius:4px;'>"
                            f"⚠️ Some combinations limited (min: {min_cell})</div>"))
            else:
                display(HTML("<div style='background:#d4edda; padding:10px; margin-top:10px; border-radius:4px;'>"
                            "✓ All combinations have ≥5 obs</div>"))

def update_thresholds_tab(change=None):
    with tab_thresholds:
        clear_output()

        if 'ANALYSIS_CONFIG' not in globals() or 'analysis_data' not in ANALYSIS_CONFIG:
            return
        analysis_data = ANALYSIS_CONFIG['analysis_data']

        if moderator1_widget is None:
            print("Initializing...")
            return

        display(HTML("<h3 style='margin-top:0;'>Quality Thresholds & Impact Analysis</h3>"))
        display(HTML("""<div style='background:#f8f9fa; padding:12px; border-radius:6px; margin-bottom:15px;'>
            <b>Purpose:</b> Ensure sufficient data for reliable estimation<br>
            <span style='font-size:13px; color:#555;'>Higher = more reliable but fewer subgroups</span></div>"""))

        display(min_papers_widget)
        display(min_obs_widget)
        display(HTML("<h4 style='margin-top:25px;'>Impact on Data Retention</h4>"))

        mod1 = moderator1_widget.value
        analysis_type = analysis_type_widget.value
        mod2 = moderator2_widget.value if analysis_type == 'two_way' and moderator2_widget.value != 'None' else None
        min_papers, min_obs = min_papers_widget.value, min_obs_widget.value

        groups_meeting, groups_failing = [], []

        if analysis_type == 'single':
            for cat in analysis_data[mod1].dropna().unique():
                group_data = analysis_data[analysis_data[mod1] == cat]
                n_papers, n_obs = group_data['id'].nunique(), len(group_data)

                if n_papers >= min_papers and n_obs >= min_obs:
                    groups_meeting.append((cat, n_obs, n_papers))
                else:
                    groups_failing.append((cat, n_obs, n_papers))
        else:
            if mod2:
                for cat1 in analysis_data[mod1].dropna().unique():
                    for cat2 in analysis_data[mod2].dropna().unique():
                        cell_data = analysis_data[(analysis_data[mod1] == cat1) & (analysis_data[mod2] == cat2)]
                        n_papers, n_obs = cell_data['id'].nunique(), len(cell_data)

                        if n_papers >= min_papers and n_obs >= min_obs:
                            groups_meeting.append((f"{cat1} × {cat2}", n_obs, n_papers))
                        elif n_obs > 0:
                            groups_failing.append((f"{cat1} × {cat2}", n_obs, n_papers))

        total_retained = sum(obs for _, obs, _ in groups_meeting)
        retention_pct = (total_retained / len(analysis_data)) * 100

        cards_html = f"""<div style='display:flex; gap:15px; margin-bottom:20px;'>
            <div style='flex:1; background:#d4edda; padding:15px; border-radius:6px; text-align:center;'>
                <div style='font-size:28px; font-weight:bold; color:#155724;'>{len(groups_meeting)}</div>
                <div style='font-size:13px; color:#155724;'>Groups Meeting Criteria</div></div>
            <div style='flex:1; background:#{'#f8d7da' if len(groups_failing) > 0 else '#e2e3e5'}; padding:15px; border-radius:6px; text-align:center;'>
                <div style='font-size:28px; font-weight:bold; color:#{'#721c24' if len(groups_failing) > 0 else '#6c757d'};'>{len(groups_failing)}</div>
                <div style='font-size:13px; color:#{'#721c24' if len(groups_failing) > 0 else '#6c757d'};'>Groups Excluded</div></div>
            <div style='flex:1; background:#{'#d4edda' if retention_pct >= 75 else '#fff3cd' if retention_pct >= 50 else '#f8d7da'}; padding:15px; border-radius:6px; text-align:center;'>
                <div style='font-size:28px; font-weight:bold;'>{retention_pct:.0f}%</div>
                <div style='font-size:13px;'>Data Retained</div></div></div>"""
        display(HTML(cards_html))

        if groups_meeting:
            display(HTML("<h4>✓ Groups Meeting Criteria:</h4>"))
            meet_html = "<ul style='margin-top:5px;'>"
            for cat, obs, papers in groups_meeting:
                meet_html += f"<li><b>{cat}:</b> {obs} obs, {papers} papers</li>"
            meet_html += "</ul>"
            display(HTML(meet_html))

        if groups_failing:
            display(HTML("<h4 style='margin-top:20px;'>✗ Groups Excluded:</h4>"))
            fail_html = "<ul style='margin-top:5px; color:#721c24;'>"
            for cat, obs, papers in groups_failing:
                reason = []
                if papers < min_papers:
                    reason.append(f"papers: {papers}<{min_papers}")
                if obs < min_obs:
                    reason.append(f"obs: {obs}<{min_obs}")
                fail_html += f"<li><b>{cat}:</b> {obs} obs, {papers} papers ({', '.join(reason)})</li>"
            fail_html += "</ul>"
            display(HTML(fail_html))

        if len(groups_meeting) < 2:
            display(HTML("<div style='background:#f8d7da; padding:12px; border-radius:6px; margin-top:15px;'>"
                        "🔴 <b>ERROR:</b> Need ≥2 groups. Lower thresholds.</div>"))
        elif retention_pct < 50:
            display(HTML("<div style='background:#fff3cd; padding:12px; border-radius:6px; margin-top:15px;'>"
                        "⚠️ <b>WARNING:</b> <50% data retained. Consider lowering thresholds.</div>"))

def update_all_tabs(change=None):
    update_config_tab()
    update_moderators_tab()
    update_thresholds_tab()

# --- 5. SAVE CONFIGURATION ---
def save_configuration(button):
    with status_output:
        clear_output()

        # Strict data retrieval from centralized config
        if 'ANALYSIS_CONFIG' not in globals() or 'analysis_data' not in ANALYSIS_CONFIG:
            display(HTML("<div style='color:red'>❌ Error: Data not found. Please run Step 6 first.</div>"))
            return

        analysis_data = ANALYSIS_CONFIG['analysis_data']

        analysis_type = analysis_type_widget.value
        mod1 = moderator1_widget.value
        mod2 = moderator2_widget.value if analysis_type == 'two_way' and moderator2_widget.value != 'None' else None
        min_papers, min_obs = min_papers_widget.value, min_obs_widget.value

        validation_errors = []

        if analysis_type == 'two_way' and not mod2:
            validation_errors.append("Two-way requires Moderator 2")

        if mod1 == mod2:
            validation_errors.append("Moderators cannot be the same")

        valid_groups = []
        if analysis_type == 'single':
            for cat in analysis_data[mod1].dropna().unique():
                group_data = analysis_data[analysis_data[mod1] == cat]
                if group_data['id'].nunique() >= min_papers and len(group_data) >= min_obs:
                    valid_groups.append(cat)
        else:
            if mod2:
                for cat1 in analysis_data[mod1].dropna().unique():
                    for cat2 in analysis_data[mod2].dropna().unique():
                        cell_data = analysis_data[(analysis_data[mod1] == cat1) & (analysis_data[mod2] == cat2)]
                        if cell_data['id'].nunique() >= min_papers and len(cell_data) >= min_obs:
                            valid_groups.append((cat1, cat2))

        if len(valid_groups) < 2:
            validation_errors.append(f"Only {len(valid_groups)} group(s) meet criteria. Need ≥2.")

        if validation_errors:
            error_html = "<div style='background:#f8d7da; padding:15px; border-radius:6px; border-left:4px solid #dc3545;'>"
            error_html += "<h4 style='margin-top:0; color:#721c24;'>❌ Validation Failed</h4><ul style='margin-bottom:0; color:#721c24;'>"
            for err in validation_errors:
                error_html += f"<li>{err}</li>"
            error_html += "</ul></div>"
            display(HTML(error_html))
            return

        if analysis_type == 'single':
            retained_data = analysis_data[analysis_data[mod1].isin(valid_groups)]
        else:
            retained_data = analysis_data[analysis_data.apply(lambda row: (row[mod1], row[mod2]) in valid_groups, axis=1)]

        retention_pct = (len(retained_data) / len(analysis_data)) * 100

        ANALYSIS_CONFIG['subgroup_config'] = {
            'timestamp': datetime.datetime.now(),
            'analysis_type': analysis_type,
            'moderator1': mod1,
            'moderator2': mod2,
            'min_papers': min_papers,
            'min_obs': min_obs,
            'expected_groups': len(valid_groups),
            'valid_groups_list': valid_groups,
            # Reference subgroup for the treatment-coded design matrix.
            # None => Step 9 picks the level with the most independent
            # studies, which maximises the precision of the k-1 pairwise
            # contrast coefficients (per-group pooled effects and the
            # omnibus test are invariant to this choice).
            # Set it to a specific label to pin the reference by hand --
            # for a two-way analysis, the collapsed 'cat1 | cat2' label.
            'reference_level': None,
            'data_retained': len(retained_data),
            'retention_pct': retention_pct,
            'has_empty_cells': analysis_type == 'two_way' and mod2 and
                               (pd.crosstab(analysis_data[mod1], analysis_data[mod2]) == 0).sum().sum() > 0,
            'n_empty_cells': (pd.crosstab(analysis_data[mod1], analysis_data[mod2]) == 0).sum().sum()
                            if analysis_type == 'two_way' and mod2 else 0
        }

        ANALYSIS_CONFIG['subgroup_analysis_data'] = retained_data.copy()

        ANALYSIS_CONFIG['subgroup_config']['moderator1_info'] = {
            'name': mod1,
            'n_categories': analysis_data[mod1].nunique(),
            'categories': sorted(analysis_data[mod1].dropna().unique().tolist())
        }

        if mod2:
            ANALYSIS_CONFIG['subgroup_config']['moderator2_info'] = {
                'name': mod2,
                'n_categories': analysis_data[mod2].nunique(),
                'categories': sorted(analysis_data[mod2].dropna().unique().tolist())
            }

        success_html = f"""<div style='background:#d4edda; padding:15px; border-radius:6px; border-left:4px solid #28a745;'>
            <h4 style='margin-top:0; color:#155724;'>✓ Configuration Saved Successfully</h4>
            <table style='width:100%; margin-top:10px;'>
                <tr><td><b>Analysis Type:</b></td><td>{analysis_type}</td></tr>
                <tr><td><b>Primary Moderator:</b></td><td>{mod1}</td></tr>
                {f'<tr><td><b>Secondary Moderator:</b></td><td>{mod2}</td></tr>' if mod2 else ''}
                <tr><td><b>Valid Groups:</b></td><td>{len(valid_groups)}</td></tr>
                <tr><td><b>Data Retained:</b></td><td>{len(retained_data)}/{len(analysis_data)} ({retention_pct:.1f}%)</td></tr>
            </table>
            <p style='margin:10px 0 0 0; color:#155724; font-size:14px;'><b>✅ Ready! Proceed to the next cell to run the subgroup analysis.</b></p></div>"""
        display(HTML(success_html))
        mark_stale(['subgroup', 'plots'], "Step 8: Subgroup Configuration Changed")

        with tab_details:
            clear_output()
            display(HTML(f"""
            <div style='background:#f8f9fa; padding:20px; border-radius:6px; border:1px solid #dee2e6; font-family:sans-serif;'>
                <h3 style='margin-top:0; color:#28a745; border-bottom:2px solid #dee2e6; padding-bottom:10px;'>✓ Configuration Saved</h3>
                <p style='color:#495057; font-size:14px;'><b>Timestamp:</b> {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
                <p style='color:#495057; font-size:14px;'>Configuration successfully stored in memory (<code>ANALYSIS_CONFIG['subgroup_config']</code>).</p>
                <p style='color:#2c3e50; font-weight:bold; font-size:14px;'>Ready to proceed to subgroup analysis execution (Step 9).</p>
            </div>
            """))

# --- 6. INITIALIZE AND DISPLAY ---

try:
    available_mods = initialize_configuration()
    update_all_tabs()
    #Repro banner
    if ANALYSIS_CONFIG.get('is_reproducing', False):
        sc = ANALYSIS_CONFIG.get('subgroup_config', {})
        mod_msg = f"{sc.get('moderator1')}" + (f" × {sc.get('moderator2')}" if sc.get('moderator2') else "")
        display(HTML(f"<div style='background:#e8f4f8; border-left:4px solid #17a2b8; padding:12px; margin-bottom:10px; border-radius:4px; color:#0c5460;'>🔄 <b>Reproducibility Mode Active</b> &nbsp;|&nbsp; Analysis: <b>{sc.get('analysis_type')}</b> | Mods: <b>{mod_msg}</b></div>"))

    display(HTML("<h3 style='color: #2E86AB; border-bottom: 2px solid #3498db; padding-bottom: 5px; font-family: sans-serif;'>Step 8: Subgroup Configuration</h3>"))
    display(tabs)

    # =================================================================
    # REPRODUCIBILITY MODE (CELL 8): Auto-populate Settings
    # =================================================================
    if ANALYSIS_CONFIG.get('is_reproducing', False):
        sub_config = ANALYSIS_CONFIG.get('subgroup_config', {})
        if sub_config:
            # 1. Safely set Analysis Type
            a_type = sub_config.get('analysis_type')
            valid_a_types = [opt[1] if isinstance(opt, tuple) else opt for opt in analysis_type_widget.options]
            if a_type in valid_a_types:
                analysis_type_widget.value = a_type

            # 2. Safely set Moderator 1
            mod1 = sub_config.get('moderator1')
            if mod1 in moderator1_widget.options:
                moderator1_widget.value = mod1

            # 3. Safely set Moderator 2
            mod2 = sub_config.get('moderator2')
            if mod2 in moderator2_widget.options:
                moderator2_widget.value = mod2

            # 4. Set Thresholds
            min_papers_widget.value = int(sub_config.get('min_papers', 3))
            min_obs_widget.value = int(sub_config.get('min_obs', 5))

            # 5. Automatically click "Save Configuration"
            save_configuration(None)

except Exception as e:
    display(HTML(f"""
    <div style='padding:15px; background:#f8d7da; border-left:5px solid #dc3545; color:#721c24; border-radius:4px; font-family:sans-serif;'>
        <h4 style='margin-top:0; display:flex; align-items:center;'><span style='margin-right:8px;'>❌</span> Initialization Failed</h4>
        <p style='margin:0 0 10px 0;'><b>Error:</b> {e}</p>
        <b>Please ensure:</b>
        <ul style='margin-bottom:0;'>
            <li>Step 6 (Effect Size Calculation) or Step 7 (Overall Meta-Analysis) has been successfully run.</li>
            <li>Your dataset contains at least one valid categorical column to act as a moderator.</li>
        </ul>
    </div>
    """))

In [ ]:
#@title 🔬 9. Subgroup Analysis: Execution
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# Centralized data management and validation
# Dependencies: ANALYSIS_CONFIG global dictionary
# =============================================================================

import numpy as np
import pandas as pd
from typing import Dict, Any, Optional, Tuple, List, Union, Sequence
from dataclasses import dataclass
import warnings
import datetime



# =============================================================================
# DATA CLASSES (Type-Safe Data Containers)
# =============================================================================

@dataclass
class SubgroupConfig:
    """Configuration for subgroup analysis"""
    analysis_type: str  # 'single' or 'two_way'
    moderator1: str
    moderator2: Optional[str]
    valid_groups_list: List[Union[str, Tuple[str, str]]]
    # Optional explicit reference subgroup for the treatment-coded design
    # matrix. None (the default) means "pick the level with the most
    # independent studies" -- see get_subgroup_data(). For a two-way
    # analysis this is the collapsed 'cat1 | cat2' label.
    reference_level: Optional[str] = None

    def __post_init__(self):
        """Validate configuration after initialization"""
        if self.analysis_type not in ['single', 'two_way']:
            raise ValueError(f"analysis_type must be 'single' or 'two_way', got {self.analysis_type}")
        if self.analysis_type == 'two_way' and not self.moderator2:
            raise ValueError("moderator2 required for two_way analysis")


@dataclass
class GlobalSettings:
    """Global analysis settings"""
    alpha: float = 0.05
    dist_type: str = 'norm'  # 'norm' or 't'

    @property
    def ci_percent(self) -> float:
        """Calculate confidence interval percentage"""
        return (1 - self.alpha) * 100

    def __post_init__(self):
        """Validate settings"""
        if not 0 < self.alpha < 1:
            raise ValueError(f"alpha must be between 0 and 1, got {self.alpha}")
        if self.dist_type not in ['norm', 't']:
            raise ValueError(f"dist_type must be 'norm' or 't', got {self.dist_type}")


@dataclass
class SubgroupResult:
    """Single subgroup analysis result"""
    group: str
    k: int
    n_papers: int
    pooled_effect_re: float
    pooled_se_re: float
    pooled_var_re: float
    ci_lower_re: float
    ci_upper_re: float
    pi_lower_re: float
    pi_upper_re: float
    p_value_re: float
    I_squared: float
    tau_squared: float
    sigma_squared: float
    fold_change_re: float
    Q_within: float
    df_Q: int
    model_type: str

    # Optional fields for two-way analysis
    moderator1_value: Optional[str] = None
    moderator2_value: Optional[str] = None
    # How I_squared was obtained (see calculate_i_squared)
    I_squared_basis: str = 'group-specific variance components'


# =============================================================================
# DATA MANAGER CLASS
# =============================================================================

from typing import NamedTuple, List, Optional, Tuple, Union
import warnings
import numpy as np
import pandas as pd


class UnifiedSubgroupData(NamedTuple):
    """Result bundle from SubgroupDataManager.get_subgroup_data().

    Attributes
    ----------
    df : the cleaned full dataset, sorted by 'id', with a
         'subgroup_label' column added.
    X : (n_obs, p) design matrix in column-major order:
         column 0 is the intercept, columns 1..k-1 are dummy
         contrasts of the non-reference subgroup levels.
    feature_names : column labels for X, e.g.
         ['(intercept)', 'subgroup[High]', 'subgroup[Medium]'].
    reference_level : the subgroup level absorbed into the intercept
         (by default the level with the most independent studies).
    subgroup_levels : all retained levels, lexicographic order.
    dropped_levels : levels removed for having < 2 effect sizes.
    """
    df: pd.DataFrame
    X: np.ndarray
    feature_names: List[str]
    reference_level: str
    subgroup_levels: List[str]
    dropped_levels: List[str]

class SubgroupDataManager:
    """
    Centralized data access layer for subgroup analysis.
    Handles all interactions with ANALYSIS_CONFIG and data validation.
    """

    def __init__(self, analysis_config: Dict[str, Any]):
        """
        Initialize data manager with configuration dictionary.

        Args:
            analysis_config: Global ANALYSIS_CONFIG dictionary

        Raises:
            ValueError: If required keys are missing
        """
        self._config = analysis_config
        self._validate_prerequisites()

    # -------------------------------------------------------------------------
    # VALIDATION METHODS
    # -------------------------------------------------------------------------

    def _validate_prerequisites(self) -> None:
        """Validate that all required configuration exists"""
        required_keys = [
            'overall_results',
            #'three_level_results',
            'subgroup_config',
            'effect_col',
            'var_col',
            'es_config',
            'analysis_data'
        ]

        missing = [key for key in required_keys if key not in self._config]

        if missing:
            raise ValueError(
                f"Missing required ANALYSIS_CONFIG keys: {', '.join(missing)}. "
                f"Please run previous analysis steps first."
            )

    def validate_data_structure(self, df: pd.DataFrame) -> Tuple[bool, Optional[str]]:
        """
        Validate that dataframe has required columns and structure.

        Args:
            df: DataFrame to validate

        Returns:
            Tuple of (is_valid, error_message)
        """
        if df is None or len(df) == 0:
            return False, "DataFrame is empty or None"

        required_cols = [
            self.effect_col,
            self.var_col,
            'id',  # study identifier
            self.subgroup_config.moderator1
        ]

        if self.subgroup_config.moderator2:
            required_cols.append(self.subgroup_config.moderator2)

        missing_cols = [col for col in required_cols if col not in df.columns]

        if missing_cols:
            return False, f"Missing required columns: {', '.join(missing_cols)}"

        # Check for invalid values
        if df[self.effect_col].isna().any():
            return False, f"NaN values found in {self.effect_col}"

        if df[self.var_col].isna().any() or (df[self.var_col] <= 0).any():
            return False, f"Invalid variance values in {self.var_col}"

        return True, None

    # -------------------------------------------------------------------------
    # PROPERTY ACCESSORS (Read from ANALYSIS_CONFIG)
    # -------------------------------------------------------------------------

    @property
    def analysis_data(self) -> pd.DataFrame:
        """Get analysis dataset"""
        return self._config['analysis_data'].copy()

    @property
    def effect_col(self) -> str:
        """Get effect size column name"""
        return self._config['effect_col']

    @property
    def var_col(self) -> str:
        """Get variance column name"""
        return self._config['var_col']

    @property
    def es_config(self) -> Dict[str, Any]:
        """Get effect size configuration"""
        return self._config['es_config']

    @property
    def overall_results(self) -> Dict[str, Any]:
        """Get overall meta-analysis results"""
        return self._config['overall_results']

    @property
    def subgroup_config(self) -> SubgroupConfig:
        """Get subgroup configuration as typed object"""
        raw_config = self._config['subgroup_config']
        return SubgroupConfig(
            analysis_type=raw_config['analysis_type'],
            moderator1=raw_config['moderator1'],
            moderator2=raw_config.get('moderator2'),
            valid_groups_list=raw_config['valid_groups_list'],
            reference_level=raw_config.get('reference_level'),
        )

    @property
    def global_settings(self) -> GlobalSettings:
        """Get global settings as typed object"""
        raw_settings = self._config.get('global_settings', {})
        return GlobalSettings(
            alpha=raw_settings.get('alpha', 0.05),
            dist_type=raw_settings.get('dist_type', 'norm')
        )

    @property
    def vcv_matrices(self) -> Dict[str, np.ndarray]:
        """Get variance-covariance matrices"""
        return self._config.get('vcv_matrices', {})

    # -------------------------------------------------------------------------
    # DATA EXTRACTION METHODS
    # -------------------------------------------------------------------------

    def get_subgroup_data(self) -> UnifiedSubgroupData:
      """
      Build the unified dataset and design matrix for a mixed-effects
      meta-regression with subgroup as a categorical moderator.

      Reference (treatment) coding is used: an intercept column plus
      k-1 dummy contrasts. The intercept estimates the mean effect in
      the reference subgroup; each dummy estimates the deviation of
      its level from the reference. A single tau^2 (and sigma^2 in
      three-level models) is therefore shared across the whole fit
      rather than being estimated independently per subgroup.

      Quality filtering (min_papers / min_obs) is delegated to the
      Step-8 configuration cell, which evaluated those thresholds on
      the correct unit (independent STUDIES per subgroup) and stored
      the survivors in config.valid_groups_list. We apply that list
      here — the single point where the model consumes data — rather
      than re-deriving thresholds (SubgroupConfig does not carry
      min_papers/min_obs, so re-derivation would silently fall back to
      defaults and let underpowered, single-study subgroups into the
      fit).

      Returns
      -------
      UnifiedSubgroupData
          See class docstring.

      Raises
      ------
      ValueError
          If fewer than 2 subgroup levels survive cleaning, in which
          case no contrast is identifiable and a subgroup analysis is
          not meaningful.
      """
      df = self.analysis_data.copy()
      config = self.subgroup_config
      _original_df = df.copy()
      _drop_log = []      # [(index labels dropped, reason), ...] -> RowAccounting

      # ---- ROW-IDENTITY TAG FOR SAMPLING-VCV ALIGNMENT -----------------
      # build_vcv_matrices() (Step 6) builds each study's k_j x k_j
      # sampling VCV with df.groupby('id') over ANALYSIS_DATA, so the
      # matrix's row/column order is that study's row order in the
      # UNFILTERED analysis dataset. Record that position now -- before
      # any subgroup filtering or sorting -- so get_vcv_for_study() can
      # slice the stored matrix by row IDENTITY instead of inferring the
      # correspondence from its shape alone. Without it, any reordering
      # or partial filtering silently attaches a study's sampling
      # variances/covariances to the WRONG effect sizes.
      df['_vcv_row_pos'] = df.groupby('id', sort=False).cumcount()

      # Clean moderator columns (mirrors prior behaviour).
      df[config.moderator1] = df[config.moderator1].astype(str).str.strip()
      if config.moderator2:
          df[config.moderator2] = df[config.moderator2].astype(str).str.strip()

      # Collapse the cross-classification (for two-way) into a single
      # categorical 'subgroup_label' column so that one set of dummies
      # captures the full grouping. This preserves cell-level
      # interpretation while letting one X handle both analysis types.
      if config.analysis_type == 'single':
          df['subgroup_label'] = df[config.moderator1]
      elif config.analysis_type == 'two_way':
          if not config.moderator2:
              raise ValueError(
                  "two_way analysis requires both moderator1 and moderator2."
              )
          df['subgroup_label'] = (
              df[config.moderator1].astype(str) + ' | ' +
              df[config.moderator2].astype(str)
          )
      else:
          raise ValueError(
              f"Unknown analysis_type: {config.analysis_type!r}. "
              "Expected 'single' or 'two_way'."
          )

      # Drop rows whose moderator value is missing/blank — they cannot
      # be assigned to any subgroup and would corrupt the design matrix.
      blank_mask = df['subgroup_label'].isin(['', 'nan', 'None', 'NaN'])
      if blank_mask.any():
          warnings.warn(
              f"Dropped {int(blank_mask.sum())} row(s) with missing "
              "subgroup labels before building the design matrix.",
              UserWarning,
          )
          _drop_log.append((list(df.index[blank_mask]),
                            "missing subgroup label (moderator value blank)"))
          df = df.loc[~blank_mask].copy()

      # ---- QUALITY FILTER: honor the Step-8 valid_groups_list ----------
      # The configuration cell computed which subgroups clear min_papers
      # (independent studies) AND min_obs. Apply that pre-computed set
      # here so the study-count threshold is actually enforced. Labels
      # are normalized on BOTH sides (str + strip) to match the
      # subgroup_label construction above; without this, stray whitespace
      # or non-string moderator values would cause a silent total mismatch.
      valid = config.valid_groups_list

      if config.analysis_type == 'single':
          valid_labels = {str(v).strip() for v in valid}
      else:
          # two_way: valid is a list of (cat1, cat2) tuples; rebuild the
          # 'cat1 | cat2' label exactly as constructed above.
          valid_labels = {
              f"{str(a).strip()} | {str(b).strip()}" for (a, b) in valid
          }

      all_levels = set(df['subgroup_label'].unique())

      # Defensive guard: if the saved list matches NOTHING in the data
      # (label drift between Step 8 and now — e.g. moderator re-typed,
      # whitespace, reloaded data), refuse to silently nuke every group.
      if valid_labels and valid_labels.isdisjoint(all_levels):
          raise ValueError(
              "Subgroup quality filter matched zero groups: the labels in "
              "config.valid_groups_list do not correspond to any "
              "subgroup_label in the current data. This usually means the "
              "moderator or dataset changed since Step 8 was saved. Re-run "
              "the Step-8 configuration cell.\n"
              f"  valid_groups_list labels: {sorted(valid_labels)}\n"
              f"  data subgroup labels:     {sorted(all_levels)}"
          )

      dropped_levels = sorted(all_levels - valid_labels)
      if dropped_levels:
          warnings.warn(
              f"Dropped {len(dropped_levels)} subgroup level(s) that did "
              f"not meet the Step-8 quality thresholds "
              f"(min_papers / min_obs): {dropped_levels}. These had too few "
              "independent studies to support reliable between-study "
              "variance estimation.",
              UserWarning,
          )
          _out = ~df['subgroup_label'].isin(valid_labels)
          _drop_log.append((list(df.index[_out]),
                            "subgroup level below the Step-8 thresholds (min studies / effects)"))
          df = df.loc[~_out].copy()

      # Safety floor: even within 'valid' groups, a level that somehow has
      # fewer than 2 effect sizes cannot anchor a contrast. Defensive —
      # should not trigger if Step 8 ran correctly, but protects against
      # post-config row drops (e.g. blank-label removal above).
      counts = df['subgroup_label'].value_counts()
      degenerate = sorted(counts.index[counts < 2].tolist())
      if degenerate:
          warnings.warn(
              f"Dropped {len(degenerate)} subgroup level(s) with fewer than "
              f"2 effect sizes after filtering: {degenerate}.",
              UserWarning,
          )
          _out = df['subgroup_label'].isin(degenerate)
          _drop_log.append((list(df.index[_out]),
                            "subgroup level with fewer than 2 effect sizes"))
          df = df.loc[~_out].copy()
          dropped_levels = sorted(set(dropped_levels) | set(degenerate))

      self.last_accounting = (accounting_from_log(_original_df, df, _drop_log)
                              if 'accounting_from_log' in globals() else None)

      levels = sorted(df['subgroup_label'].unique().tolist())
      if len(levels) < 2:
          raise ValueError(
              f"Subgroup analysis requires >= 2 levels meeting the quality "
              f"thresholds; only {len(levels)} survived cleaning "
              f"({levels}). Lower the Step-8 thresholds, merge categories, "
              "or use the overall meta-analysis instead."
          )

      # ---- REFERENCE LEVEL --------------------------------------------
      # Under reference (treatment) coding the intercept IS the reference
      # subgroup's mean, and each of the k-1 dummy coefficients is that
      # level's difference FROM the reference. Those pairwise-contrast
      # coefficients, their robust SEs and their Satterthwaite DFs all
      # inherit the reference's sampling noise, so a reference with few
      # studies makes every reported contrast less precise.
      #
      # What the reference does NOT change: the per-level pooled effects
      # L_i @ beta, their CR2 variances L_i V L_i', their contrast DFs and
      # the omnibus test. Reference coding is a reparametrisation, and
      # those quantities are invariant to it (verified numerically).
      # Switching the reference therefore improves the contrast estimates
      # without moving any group's own pooled effect or CI.
      #
      # Default: the level with the MOST independent studies. The previous
      # default (lexicographically first) is arbitrary with respect to
      # information content and routinely anchored the contrasts on one of
      # the smallest subgroups.
      #
      # Ties break deterministically -- most studies, then most effect
      # sizes, then lexicographically first -- so the choice is
      # reproducible across runs and machines.
      #
      # An explicit reference can be pinned with
      # ANALYSIS_CONFIG['subgroup_config']['reference_level'] (for a
      # two-way analysis, the collapsed 'cat1 | cat2' label).
      studies_per_level = df.groupby('subgroup_label')['id'].nunique()
      obs_per_level = df['subgroup_label'].value_counts()

      requested_ref = config.reference_level
      requested_ref = (str(requested_ref).strip()
                       if requested_ref is not None else None)

      if requested_ref is not None and requested_ref in levels:
          reference_level = requested_ref
      else:
          if requested_ref is not None:
              warnings.warn(
                  f"Requested reference_level {requested_ref!r} is not among "
                  f"the surviving subgroup levels {levels}; falling back to "
                  "the level with the most independent studies.",
                  UserWarning,
              )
          reference_level = min(
              levels,
              key=lambda lv: (
                  -int(studies_per_level.get(lv, 0)),
                  -int(obs_per_level.get(lv, 0)),
                  lv,
              ),
          )

      # Sort by 'id' so prepare_three_level_data sees contiguous study
      # blocks, exactly as the legacy pipeline did.
      #
      # kind='stable' is REQUIRED, not cosmetic. pandas' default
      # ('quicksort') is NOT stable, so on any dataset whose rows are not
      # already grouped by 'id' it permutes rows WITHIN each study. The
      # design matrix is rebuilt from this frame and therefore follows the
      # permutation, but the per-study sampling VCVs cached by Step 6 were
      # built in the unfiltered dataset's row order and do NOT -- so the
      # two silently desynchronise. A stable sort keeps the current order
      # a subsequence of the original one ('_vcv_row_pos' above stays
      # monotonically increasing within every study), which is exactly the
      # condition get_vcv_for_study() needs to slice the cached matrix.
      df = df.sort_values('id', kind='stable').reset_index(drop=True)

      # Build the design matrix. We pass drop_first=False and drop the
      # reference column ourselves so the choice is explicit and easy
      # to override in future, rather than relying on pandas' implicit
      # ordering rule.
      dummies = pd.get_dummies(
          df['subgroup_label'],
          prefix='subgroup',
          prefix_sep='[',
          drop_first=False,
      )
      dummies.columns = [f"{c}]" for c in dummies.columns]  # close bracket
      ref_col = f"subgroup[{reference_level}]"
      dummies = dummies.drop(columns=[ref_col])

      intercept = pd.Series(1.0, index=df.index, name='(intercept)')
      X_df = pd.concat([intercept, dummies.astype(np.float64)], axis=1)

      return UnifiedSubgroupData(
          df=df,
          X=X_df.to_numpy(dtype=np.float64),
          feature_names=X_df.columns.tolist(),
          reference_level=reference_level,
          subgroup_levels=levels,
          dropped_levels=dropped_levels,
      )

    def get_vcv_for_study(
        self,
        study_id: Union[str, int],
        variance_vector: np.ndarray,
        row_positions: Optional[np.ndarray] = None,
    ) -> Tuple[np.ndarray, np.ndarray, bool]:
        """
        Get the sampling variance-covariance matrices for one study,
        aligned to the rows actually present in the current subset.

        The cached matrix (built by build_vcv_matrices(), Step 6) is
        indexed by the study's row order in the UNFILTERED analysis
        dataset. `row_positions` supplies, for each element of
        `variance_vector`, that original within-study position, so the
        relevant sub-block can be extracted with the rows in their
        CURRENT order:

            V_sub = cached[np.ix_(row_positions, row_positions)]

        This does two things the previous shape-only check could not:

          1. It keeps shared-control covariances usable when a subgroup
             filter drops only SOME of a study's effect sizes (before,
             the whole off-diagonal block was discarded whenever
             k != cached.shape[0]).
          2. It stops the cached matrix from being applied to the WRONG
             effect sizes when the current row order differs from the
             original one -- the failure mode that made Co-Meta and
             metafor fit different Sigma matrices on identical inputs.

        Falls back to the diagonal matrix built from `variance_vector`
        whenever the cached matrix is absent, cannot be sliced, or its
        diagonal disagrees with `variance_vector` (a mismatch proves the
        slice does not describe these rows, since the cached diagonal IS
        the sampling-variance column).
        """
        # Shared implementation (Cell 1b) so every analysis mode slices the
        # sampling VCV by row identity in exactly the same way.
        variance_vector = np.asarray(variance_vector, dtype=np.float64)
        diag_matrix = np.diag(variance_vector)
        full_matrix, _v_source = slice_vcv_block(
            self.vcv_matrices, study_id, variance_vector, row_positions
        )

        # Check for off-diagonal elements
        has_off_diagonal = not np.allclose(
            full_matrix,
            np.diag(np.diag(full_matrix))
        )

        return full_matrix, diag_matrix, has_off_diagonal


    def prepare_three_level_data(
      self,
      full_df: pd.DataFrame,
      X: np.ndarray,
  ) -> Tuple[List[np.ndarray], List[np.ndarray], List[np.ndarray],
            List[np.ndarray], bool]:
      """
      Partition the full dataset into per-study blocks for a unified
      three-level mixed-effects meta-regression.

      Each study contributes one block of effect sizes, one sampling
      VCV matrix (full and diagonal variants), and one slice of the
      design matrix X. Downstream fitting code can then accumulate
      the log-likelihood block-by-block while sharing a single tau^2
      and sigma^2 across the entire model.

      Args
      ----
      full_df : the cleaned dataset returned by
                get_subgroup_data().df.
      X       : the design matrix returned by get_subgroup_data().X,
                with rows aligned 1:1 with full_df.

      Returns
      -------
      y_all       : per-study effect-size vectors.
      vcv_full    : per-study full sampling VCV matrices.
      vcv_diag    : per-study diagonal-only sampling VCV matrices.
      X_blocks    : per-study slices of the design matrix, each of
                    shape (n_j, p) and aligned row-for-row with
                    y_all[j].
      has_shared_controls : True if any study's full VCV has
                    off-diagonal entries.
      """
      if len(full_df) != X.shape[0]:
          raise ValueError(
              f"Row mismatch: full_df has {len(full_df)} rows but X "
              f"has {X.shape[0]} rows. They must be aligned 1:1."
          )

      # Defensive sort + index reset so that group.index gives valid
      # positional indices into X regardless of how the caller built
      # full_df. If the data is already sorted, this is a no-op apart
      # from the index reset.
      order = np.argsort(full_df['id'].to_numpy(), kind='stable')
      full_df = full_df.iloc[order].reset_index(drop=True)
      X = X[order, :]

      y_all: List[np.ndarray] = []
      vcv_full: List[np.ndarray] = []
      vcv_diag: List[np.ndarray] = []
      X_blocks: List[np.ndarray] = []
      has_shared_controls = False

      for study_id, group in full_df.groupby('id', sort=False):
          idx = group.index.to_numpy()

          y = np.asarray(group[self.effect_col].values, dtype=np.float64)
          vi = np.asarray(group[self.var_col].values, dtype=np.float64)

          full_mat, diag_mat, has_off_diag = self.get_vcv_for_study(
              study_id, vi,
              row_positions=(group['_vcv_row_pos'].to_numpy()
                             if '_vcv_row_pos' in group.columns else None),
          )

          y_all.append(y)
          vcv_full.append(full_mat)
          vcv_diag.append(diag_mat)
          X_blocks.append(X[idx, :])

          if has_off_diag:
              has_shared_controls = True

      return y_all, vcv_full, vcv_diag, X_blocks, has_shared_controls

    def _get_single_subgroup_subset(
      self,
      group_identifier: Union[str, Tuple[str, str]],
  ) -> Optional[pd.DataFrame]:
      """Legacy single-group subsetting, kept only for the fallback
      _analyze_subgroups_independent. New code must call
      get_subgroup_data() instead."""
      df = self.analysis_data.copy()
      # Same row-identity tag as get_subgroup_data(); see the comment
      # there. Must be computed before any filtering.
      df['_vcv_row_pos'] = df.groupby('id', sort=False).cumcount()
      config = self.subgroup_config

      df[config.moderator1] = df[config.moderator1].astype(str).str.strip()
      if config.moderator2:
          df[config.moderator2] = df[config.moderator2].astype(str).str.strip()

      if config.analysis_type == 'single':
          group_name = str(group_identifier)
          subset = df[df[config.moderator1] == group_name].copy()
      else:
          if not isinstance(group_identifier, tuple) or len(group_identifier) != 2:
              raise ValueError(
                  f"Expected tuple for two_way analysis, got "
                  f"{type(group_identifier).__name__}."
              )
          mod1_val, mod2_val = group_identifier
          subset = df[
              (df[config.moderator1] == mod1_val) &
              (df[config.moderator2] == mod2_val)
          ].copy()

      return subset if len(subset) >= 2 else None


    def _prepare_three_level_data_single(
        self,
        subgroup_df: pd.DataFrame,
    ) -> Tuple[List[np.ndarray], List[np.ndarray], List[np.ndarray], bool]:
        """Legacy per-subgroup three-level prep, kept only for the
        fallback _analyze_subgroups_independent. Identical to the
        pre-refactor prepare_three_level_data."""
        # kind='stable': see the note in get_subgroup_data(). An unstable
        # sort reorders rows within a study and desynchronises them from
        # the cached sampling VCVs.
        subgroup_df = subgroup_df.sort_values('id', kind='stable').reset_index(drop=True)
        grouped = subgroup_df.groupby('id', sort=False)

        y_all: List[np.ndarray] = []
        vcv_full: List[np.ndarray] = []
        vcv_diag: List[np.ndarray] = []
        has_shared_controls = False

        for study_id, group in grouped:
            y = np.asarray(group[self.effect_col].values, dtype=np.float64)
            vi = np.asarray(group[self.var_col].values, dtype=np.float64)
            full_mat, diag_mat, has_off_diag = self.get_vcv_for_study(
                study_id, vi,
                row_positions=(group['_vcv_row_pos'].to_numpy()
                               if '_vcv_row_pos' in group.columns else None),
            )
            y_all.append(y)
            vcv_full.append(full_mat)
            vcv_diag.append(diag_mat)
            if has_off_diag:
                has_shared_controls = True

        return y_all, vcv_full, vcv_diag, has_shared_controls

    # -------------------------------------------------------------------------
    # RESULT PERSISTENCE METHODS
    # -------------------------------------------------------------------------

    def save_subgroup_results(
        self,
        results_df: pd.DataFrame,
        metadata: Dict[str, Any]
    ) -> None:
        """
        Save subgroup analysis results to ANALYSIS_CONFIG.

        Args:
            results_df: DataFrame containing all subgroup results
            metadata: Dictionary with QM, R_squared, etc.
        """
        import datetime

        self._config['subgroup_results'] = {
            'timestamp': datetime.datetime.now(),
            'results_df': results_df,
            'analysis_type': self.subgroup_config.analysis_type,
            'moderator1': self.subgroup_config.moderator1,
            'moderator2': self.subgroup_config.moderator2,
            **metadata  # Qt_overall, QM, Qe, df_QM, df_Qe, p_value_QM, R_squared
        }

    def save_publication_text(self, text: str) -> None:
        """Save publication-ready text"""
        self._config['subgroup_text'] = text

    # -------------------------------------------------------------------------
    # UTILITY METHODS
    # -------------------------------------------------------------------------

    def get_group_display_name(
        self,
        group_identifier: Union[str, Tuple[str, str]]
    ) -> str:
        """
        Get human-readable group name.

        Args:
            group_identifier: Group identifier

        Returns:
            Formatted group name
        """
        config = self.subgroup_config

        if config.analysis_type == 'single':
            return str(group_identifier)
        else:
            mod1_val, mod2_val = group_identifier
            return f"{mod1_val} × {mod2_val}"

    def summary_dict(self) -> Dict[str, Any]:
        """Get summary of current configuration"""
        config = self.subgroup_config
        settings = self.global_settings

        return {
            'analysis_type': config.analysis_type,
            'moderator1': config.moderator1,
            'moderator2': config.moderator2,
            'n_groups': len(config.valid_groups_list),
            'effect_col': self.effect_col,
            'var_col': self.var_col,
            'alpha': settings.alpha,
            'ci_percent': settings.ci_percent,
            'dist_type': settings.dist_type,
            'total_observations': len(self.analysis_data),
            'n_studies': self.analysis_data['id'].nunique()
        }


# =============================================================================
# ANALYSIS LAYER
# Purpose: Pure statistical computation without UI dependencies
# =============================================================================

import numpy as np
import pandas as pd
from scipy.stats import norm, chi2, t, f
from typing import Dict, Any, Optional, Tuple, List, Union
import warnings
from dataclasses import asdict


# =============================================================================
# STATISTICAL COMPUTATION ENGINE
# =============================================================================

class SubgroupAnalyzer:
    """
    Pure statistical analysis engine for subgroup meta-analysis.
    Contains no UI code - only mathematical computations.
    """

    def __init__(
        self,
        effect_col: str,
        var_col: str,
        global_settings: 'GlobalSettings',
        es_config: Dict[str, Any]
    ):
        """
        Initialize analyzer with configuration.

        Args:
            effect_col: Name of effect size column
            var_col: Name of variance column
            global_settings: GlobalSettings object with alpha, dist_type
            es_config: Effect size configuration dictionary
        """
        self.effect_col = effect_col
        self.var_col = var_col
        self.settings = global_settings
        self.es_config = es_config

    # -------------------------------------------------------------------------
    # HETEROGENEITY PARTITIONING (FIXED)
    # -------------------------------------------------------------------------

    def partition_heterogeneity(self, **kwargs) -> Dict[str, Any]:
      """
      Heterogeneity-and-omnibus-test summary for a subgroup analysis.

      Two calling conventions are accepted:

      NEW (unified mixed-effects meta-regression):
          partition_heterogeneity(
              overall_results=<dict>,    # null intercept-only fit
              subgroup_fit=<dict>,       # output of run_three_level_analysis
              qm_wald=<float>,           # Wald chi-squared on contrasts
              df_qm=<int>,               # number of contrast coefficients
              qe_residual=<float>,       # residual Q from calculate_q_statistic
              df_qe=<int>,
          )
          Returns the Wald-based omnibus test (NOT derived by
          subtraction), residual heterogeneity, and Raudenbush-style
          pseudo-R^2 against the null model.

      LEGACY (independent fits, fallback path only):
          partition_heterogeneity(
              qt_overall=..., q_within_sum=..., n_groups=..., k_overall=...
          )
          Routes to the preserved subtraction-based implementation;
          emits a DeprecationWarning. New code must use the unified API.
      """
      if 'overall_results' in kwargs or 'r2_detail' in kwargs:
          return self._partition_heterogeneity_unified(**kwargs)

      if 'qt_overall' in kwargs:
          return self._partition_heterogeneity_legacy(**kwargs)

      raise TypeError(
          "partition_heterogeneity: provide either the unified "
          "(overall_results, subgroup_fit, qm_wald, df_qm, qe_residual, "
          "df_qe) or legacy (qt_overall, q_within_sum, n_groups, k_overall) "
          "keyword arguments."
      )
    def _partition_heterogeneity_unified(
      self,
      subgroup_fit: Dict[str, Any],
      qm_wald: float,
      df_qm: int,
      qe_residual: float,
      df_qe: int,
      overall_results: Optional[Dict[str, Any]] = None,
      r2_detail: Optional[Dict[str, Any]] = None,
  ) -> Dict[str, Any]:
      """Unified-model heterogeneity and omnibus test.

      Pseudo-R² comes from `r2_detail` (pseudo_r2_report, Cell 1b): the
      subgroup model and its intercept-only null fitted by REML on the same
      rows, the same sampling V and the same random structure. `overall_results`
      (externally supplied null components) is accepted for backward
      compatibility only.
      """
      if r2_detail is not None:
          return self._partition_from_r2_detail(
              r2_detail, subgroup_fit, qm_wald, df_qm, qe_residual, df_qe)
      # ---- Existence check on tau^2_null (global constraint) ----------
      if not isinstance(overall_results, dict):
          raise TypeError(
              "_partition_heterogeneity_unified: `overall_results` must "
              f"be a dict (the previously-fitted intercept-only meta-"
              f"analysis result), got {type(overall_results).__name__}."
          )
      if 'tau_squared' not in overall_results:
          raise ValueError(
              "_partition_heterogeneity_unified: cannot compute pseudo-R^2 "
              "because 'tau_squared' (tau^2_null) is missing from "
              "`overall_results`. Run the overall intercept-only meta-"
              "analysis first and pass its result dict. Available keys in "
              f"overall_results: {sorted(overall_results.keys())!r}."
          )

      tau2_null = float(overall_results['tau_squared'])
      sigma2_null = float(overall_results.get('sigma_squared', 0.0))

      # The full-model variance components live under either name
      # depending on the fit dict's provenance.
      tau2_full = float(
          subgroup_fit.get('tau_squared', subgroup_fit.get('tau_sq', 0.0))
      )
      sigma2_full = float(
          subgroup_fit.get('sigma_squared', subgroup_fit.get('sigma_sq', 0.0))
      )

      # ---- Pseudo-R^2 (Raudenbush 2009, multilevel generalisation) ----
      # R^2 = 1 - (tau^2_full + sigma^2_full) / (tau^2_null + sigma^2_null)
      # Bounded to [0, 1] to absorb sampling noise that can produce a
      # slightly larger residual than null when moderators are weak.
      null_total = tau2_null + sigma2_null
      full_total = tau2_full + sigma2_full
      if null_total > 0.0:
          r_squared_frac = 1.0 - (full_total / null_total)
          r_squared = max(0.0, min(1.0, r_squared_frac)) * 100.0
      else:
          r_squared = 0.0

      # Marginal R^2 components (per variance level), useful for
      # publication tables that report tau^2 and sigma^2 separately.
      r_squared_tau = (
          max(0.0, min(1.0, 1.0 - tau2_full / tau2_null)) * 100.0
          if tau2_null > 0.0 else 0.0
      )
      r_squared_sigma = (
          max(0.0, min(1.0, 1.0 - sigma2_full / sigma2_null)) * 100.0
          if sigma2_null > 0.0 else 0.0
      )

      # ---- QM (Wald) and QE (residual) sanitisation + p-values --------
      qm = float(qm_wald) if np.isfinite(qm_wald) and qm_wald > 0 else 0.0
      qe = float(qe_residual) if np.isfinite(qe_residual) and qe_residual > 0 else 0.0
      df_qm = int(df_qm)
      df_qe = int(df_qe)

      p_qm_chi2 = (
          float(1.0 - chi2.cdf(qm, df_qm)) if df_qm > 0 and qm > 0 else 1.0
      )
      p_qe_chi2 = (
          float(1.0 - chi2.cdf(qe, df_qe)) if df_qe > 0 and qe > 0 else 1.0
      )

      # Small-sample F version of the omnibus test, using denominator DF
      # taken from the engine's CR2/Satterthwaite vector when available.
      # Conservative choice: minimum CR2 DF over the contrast columns.
      f_omnibus, df_f_num, df_f_den, p_f_omnibus = self._omnibus_f_from_wald(
          qm=qm, df_qm=df_qm, subgroup_fit=subgroup_fit
      )

      return {
          'QM': qm,
          'df_QM': df_qm,
          'p_value_QM': p_qm_chi2,
          'QE': qe,
          'df_QE': df_qe,
          'p_value_QE': p_qe_chi2,
          'F_omnibus': f_omnibus,
          'df_F_num': df_f_num,
          'df_F_den': df_f_den,
          'p_value_F_omnibus': p_f_omnibus,
          'R_squared': r_squared,
          'R_squared_tau': r_squared_tau,
          'R_squared_sigma': r_squared_sigma,
          'tau_squared_null': tau2_null,
          'sigma_squared_null': sigma2_null,
          'tau_squared_full': tau2_full,
          'sigma_squared_full': sigma2_full,
          'tau_squared_reduction': tau2_null - tau2_full,
          'sigma_squared_reduction': sigma2_null - sigma2_full,
          'qm_method': 'wald',
          'fallback_used': False,
      }

    def _partition_from_r2_detail(
        self,
        r2_detail: Dict[str, Any],
        subgroup_fit: Dict[str, Any],
        qm_wald: float,
        df_qm: int,
        qe_residual: float,
        df_qe: int,
    ) -> Dict[str, Any]:
        """Omnibus/residual tests + pseudo-R² from the matching-null report.
        R² values are percentages; NaN when not computable (never a silent 0)."""
        qm = float(qm_wald) if np.isfinite(qm_wald) and qm_wald > 0 else 0.0
        qe = float(qe_residual) if np.isfinite(qe_residual) and qe_residual > 0 else 0.0
        df_qm, df_qe = int(df_qm), int(df_qe)
        p_qm = float(1.0 - chi2.cdf(qm, df_qm)) if df_qm > 0 and qm > 0 else 1.0
        p_qe = float(1.0 - chi2.cdf(qe, df_qe)) if df_qe > 0 and qe > 0 else 1.0
        f_omnibus, df_f_num, df_f_den, p_f_omnibus = self._omnibus_f_from_wald(
            qm=qm, df_qm=df_qm, subgroup_fit=subgroup_fit
        )
        d = r2_detail or {}
        nan = float('nan')
        tn, sn = d.get('tau2_null', nan), d.get('sigma2_null', nan)
        tm, sm = d.get('tau2_model', nan), d.get('sigma2_model', nan)
        return {
            'QM': qm, 'df_QM': df_qm, 'p_value_QM': p_qm,
            'QE': qe, 'df_QE': df_qe, 'p_value_QE': p_qe,
            'F_omnibus': f_omnibus, 'df_F_num': df_f_num, 'df_F_den': df_f_den,
            'p_value_F_omnibus': p_f_omnibus,
            'R_squared': float(d.get('R2', nan)),
            'R_squared_tau': float(d.get('R2_tau2', nan)),
            'R_squared_sigma': float(d.get('R2_sigma2', nan)),
            'R_squared_detail': d,
            'tau_squared_null': tn, 'sigma_squared_null': sn,
            'tau_squared_full': tm, 'sigma_squared_full': sm,
            'tau_squared_reduction': tn - tm, 'sigma_squared_reduction': sn - sm,
            'qm_method': 'wald',
            'fallback_used': False,
        }

    def _partition_heterogeneity_legacy(
        self,
        qt_overall: float,
        q_within_sum: float,
        n_groups: int,
        k_overall: int,
    ) -> Dict[str, Any]:
        """LEGACY (PRESERVED). Original subtraction-based partitioning,
        used only by the independent-fits fallback path. This is the
        pre-Phase-3 body, verbatim, with the return dict harmonised to
        the new shape so downstream code can read either output.
        """
        QM = qt_overall - q_within_sum
        if QM < 0 or not np.isfinite(QM):
            QM = 0.0

        df_QM = max(0, n_groups - 1)
        df_QE = max(0, k_overall - n_groups)

        p_value_QM = float(1.0 - chi2.cdf(QM, df_QM)) if df_QM > 0 and QM > 0 else 1.0
        p_value_QE = (
            float(1.0 - chi2.cdf(q_within_sum, df_QE))
            if df_QE > 0 and q_within_sum > 0 else 1.0
        )

        # QM / QT is a fixed-effect Q ratio, NOT a variance-component
        # pseudo-R²; it is reported under its own name and R² is NaN here.
        q_ratio = (QM / qt_overall) * 100.0 if qt_overall > 0 and QM > 0 else 0.0
        q_ratio = max(0.0, min(100.0, q_ratio))
        R_squared = float('nan')

        return {
            'QT': qt_overall,
            'QM': QM,
            'df_QM': df_QM,
            'QE': q_within_sum,
            'df_QE': df_QE,
            'p_value_QM': p_value_QM,
            'p_value_QE': p_value_QE,
            'R_squared': R_squared,
            'Q_ratio_explained': q_ratio,
            'R_squared_detail': {
                'R2': float('nan'),
                'error': ("unified subgroup model did not converge; the independent-fits "
                          f"fallback reports only the fixed-effect Q ratio Q_M/Q_T = "
                          f"{q_ratio:.1f}%, which is not a pseudo-R²"),
            },
            'qm_method': 'subtraction (legacy fallback)',
            'fallback_used': True,
        }




    def _omnibus_f_from_wald(
        self,
        qm: float,
        df_qm: int,
        subgroup_fit: Dict[str, Any],
    ) -> Tuple[float, int, float, float]:
        """Small-sample F version of the omnibus test.

        Preferred: the Approximate Hotelling-Zhang test of Tipton &
        Pustejovsky (2015) -- clubSandwich's Wald_test(test="HTZ") --
        when `subgroup_fit` carries the AHZ closure ('htz_wald') and the
        q x p contrast matrix ('contrast_matrix'). That gives a
        contrast-specific denominator DF.

        Fallback (no closure available): F = W / df_QM, where W is the Wald
        statistic on the covariance used for inference (CR2 when available,
        `subgroup_fit['qm_wald_robust']`; otherwise QM), with the
        denominator DF taken as the minimum CR2/Satterthwaite DF across
        the model's coefficients. This is conservative and, when one
        coefficient has very few effective DF, uniformly so -- it is a
        floor, not the contrast's own DF.
        """
        if df_qm <= 0 or qm <= 0:
            return 0.0, int(df_qm), 0.0, 1.0

        # Preferred path: the genuine Approximate Hotelling-Zhang test --
        # what clubSandwich exposes as Wald_test(..., test="HTZ"). It
        # returns an F statistic AND a denominator DF computed for the
        # whole q x p contrast matrix, instead of the conservative
        # minimum over individual coefficients used below.
        htz_wald = subgroup_fit.get('htz_wald')
        contrast_matrix = subgroup_fit.get('contrast_matrix')
        if htz_wald is not None and contrast_matrix is not None:
            try:
                htz_out = htz_wald(contrast_matrix, test='AHZ')
                f_htz = float(htz_out['F'])
                df_den_htz = float(htz_out['df_denom'])
                if (np.isfinite(f_htz) and np.isfinite(df_den_htz)
                        and df_den_htz > 0):
                    return (f_htz, int(htz_out['df_num']), df_den_htz,
                            float(htz_out['p_val']))
            except Exception as e:
                warnings.warn(f"Subgroup test: HTZ Wald test failed ({e}); using Q_M / df "
                              f"with the smallest coefficient df instead.")

        w_stat = subgroup_fit.get('qm_wald_robust', qm)
        if w_stat is None or not np.isfinite(w_stat) or w_stat < 0:
            w_stat = qm
        f_stat = float(w_stat) / df_qm
        dfs = subgroup_fit.get('dfs', None)
        if dfs is None:
            # No CR2 DF vector exposed -- fall back to (m_studies - p).
            m = int(subgroup_fit.get('n_studies', 0))
            p = int(subgroup_fit.get('p_params', 0))
            df_den = float(max(1, m - p))
        else:
            arr = np.asarray(dfs, dtype=np.float64).ravel()
            positive = arr[arr > 0]
            df_den = float(np.min(positive)) if positive.size else 1.0
        p_f = float(1.0 - f.cdf(f_stat, df_qm, df_den))
        return f_stat, int(df_qm), df_den, p_f


    # -------------------------------------------------------------------------
    # THREE-LEVEL META-ANALYSIS
    # -------------------------------------------------------------------------

    def _compute_wald_qm(
      self,
      betas: np.ndarray,
      cov_beta: np.ndarray,
      contrast_indices: Sequence[int],
  ) -> Tuple[float, int]:
      """
      Joint Wald chi-squared statistic for the contrast coefficients
      of the unified subgroup meta-regression.

          QM = b' V^{-1} b
          df_QM = len(b)

      where b is betas[contrast_indices] and V is the corresponding
      submatrix of cov_beta. Under H_0: all contrasts are zero (i.e.
      every non-reference subgroup level has the same mean as the
      reference), QM is asymptotically chi-squared with len(b)
      degrees of freedom. With the model-based cov_beta this is the same
      statistic as metafor's QM (test='z') for rma.mv; with the CR2
      cov_beta it is a cluster-robust Wald statistic, which has no
      small-sample correction (use the HTZ F test for inference).

      pseudoinverse fallback handles rank-deficient V (e.g. very
      small samples) without raising.
      """
      idx = np.asarray(list(contrast_indices), dtype=int)
      if idx.size == 0:
          return 0.0, 0
      b = np.asarray(betas, dtype=np.float64)[idx]
      V = np.asarray(cov_beta, dtype=np.float64)[np.ix_(idx, idx)]
      try:
          V_inv = np.linalg.inv(V)
      except np.linalg.LinAlgError:
          V_inv = np.linalg.pinv(V)
      qm = float(b @ V_inv @ b)
      if not np.isfinite(qm) or qm < 0:
          qm = 0.0
      return qm, int(idx.size)

    def calculate_fold_change(self, log_effect: float) -> float:
        """
        Calculate fold change from log effect size.

        Args:
            log_effect: Effect size on log scale

        Returns:
            Fold change (positive or negative)
        """
        if not self.es_config.get('has_fold_change', False):
            return np.nan

        RR = np.exp(log_effect)

        # OR and RR are always positive — no sign-flip needed
        effect_size_type = getattr(self, 'es_config', {}).get('effect_label_short', '')
        if effect_size_type in ('logOR', 'RR'):
            fold_change = RR
        else:
            fold_change = RR if log_effect >= 0 else -1/RR

        return fold_change

    def calculate_q_statistic(
      self,
      y_all: List[np.ndarray],
      vcv_blocks: List[np.ndarray],
      X_blocks: List[np.ndarray],
  ) -> Tuple[float, int, float]:
      """
      Residual heterogeneity Q_E for the unified subgroup
      meta-regression, computed in the metafor convention:

          beta_FE = (X' W X)^{-1} X' W y
          e       = y - X beta_FE
          Q_E     = e' W e

      where W is the block-diagonal inverse of the SAMPLING
      variance-covariance matrix only (full V_j for studies with
      shared controls; diagonal otherwise) -- i.e., beta_FE is the
      GLS estimate that would obtain if tau^2 = sigma^2 = 0. Q_E is
      therefore a test of residual heterogeneity above and beyond
      what the moderator (subgroup membership) explains, distributed
      approximately as chi-squared with (n_obs - p) degrees of
      freedom under H_0: tau^2 = sigma^2 = 0.

      This replaces the previous within-subgroup Cochran's Q, which
      is preserved as the private helper _calculate_subgroup_cochran_q
      for the independent-fits fallback path.
      """
      if not y_all or not vcv_blocks or not X_blocks:
          return 0.0, 0, 1.0
      if not (len(y_all) == len(vcv_blocks) == len(X_blocks)):
          raise ValueError(
              "calculate_q_statistic: y_all, vcv_blocks, and X_blocks "
              "must all have the same length (one entry per study); got "
              f"{len(y_all)}, {len(vcv_blocks)}, {len(X_blocks)}."
          )

      p_params = X_blocks[0].shape[1]
      XtWX = np.zeros((p_params, p_params), dtype=np.float64)
      XtWy = np.zeros(p_params, dtype=np.float64)
      n_obs = 0
      V_inv_list: List[np.ndarray] = []

      for y_j, V_j, X_j in zip(y_all, vcv_blocks, X_blocks):
          y_j = np.asarray(y_j, dtype=np.float64)
          V_j = np.asarray(V_j, dtype=np.float64)
          X_j = np.asarray(X_j, dtype=np.float64)

          try:
              V_inv = np.linalg.inv(V_j)
          except np.linalg.LinAlgError:
              # Robust fallback: invert the diagonal only.
              v_diag = np.diag(V_j)
              v_safe = np.where(v_diag > 0, v_diag, np.inf)
              V_inv = np.diag(1.0 / v_safe)

          V_inv_list.append(V_inv)
          XtWX += X_j.T @ V_inv @ X_j
          XtWy += X_j.T @ V_inv @ y_j
          n_obs += y_j.size

      try:
          beta_fe = np.linalg.solve(XtWX, XtWy)
      except np.linalg.LinAlgError:
          return 0.0, max(0, n_obs - p_params), 1.0

      QE = 0.0
      for y_j, V_inv, X_j in zip(y_all, V_inv_list, X_blocks):
          y_j = np.asarray(y_j, dtype=np.float64)
          X_j = np.asarray(X_j, dtype=np.float64)
          e = y_j - X_j @ beta_fe
          QE += float(e @ V_inv @ e)

      if not np.isfinite(QE) or QE < 0.0:
          QE = 0.0

      df_qe = max(0, n_obs - p_params)
      p_qe = float(1.0 - chi2.cdf(QE, df_qe)) if df_qe > 0 and QE > 0 else 1.0
      return QE, df_qe, p_qe


    def _calculate_subgroup_cochran_q(
        self,
        subgroup_df: pd.DataFrame,
        mu: float,
    ) -> Tuple[float, int]:
        """LEGACY (PRESERVED). Within-subgroup Cochran's Q against the
        pooled effect mu. Used by the independent-fits fallback path
        and by Phase 2's per-row Q_within diagnostic (the latter call
        site needs to be repointed in a future integration phase).
        """
        y = np.asarray(subgroup_df[self.effect_col].values, dtype=np.float64)
        v = np.asarray(subgroup_df[self.var_col].values, dtype=np.float64)
        if y.size == 0:
            return 0.0, 0
        safe_v = np.where(v > 0, v, np.nan)
        w = 1.0 / safe_v
        valid = np.isfinite(w) & np.isfinite(y)
        if not np.any(valid):
            return 0.0, 0
        Q = float(np.sum(w[valid] * (y[valid] - float(mu)) ** 2))
        df_Q = max(0, int(np.sum(valid)) - 1)
        return Q, df_Q



    def calculate_i_squared(
      self,
      tau_squared: float,
      sigma_squared: float,
      variances: Union[float, np.ndarray, pd.Series, Sequence[float]],
  ) -> float:
      """
      Residual I^2 for the unified mixed-effects subgroup meta-
      regression (Higgins & Thompson 2002, multilevel form):

          I^2 = (tau^2 + sigma^2) / (tau^2 + sigma^2 + s^2) * 100

      where s^2 is the typical within-study sampling variance,
      computed from the supplied `variances` via the standard
      "harmonic" formula:

          s^2 = (k - 1) * sum(w_i) / ( (sum(w_i))^2 - sum(w_i^2) )

      with w_i = 1 / v_i. This is the formula used by metafor for
      rma.mv. Pass the FULL set of sampling variances (one per
      effect size in the model) to obtain the global residual I^2.

      Backward compatibility: a scalar `variances` value is
      interpreted as a directly-supplied s^2, reproducing the legacy
      semantics. The legacy interpretation is intended only for the
      independent-fits fallback path; new code should pass the full
      variance vector.
      """
      if np.isscalar(variances):
          s_squared = float(variances)
      else:
          v = np.asarray(variances, dtype=np.float64).ravel()
          v = v[np.isfinite(v) & (v > 0.0)]
          if v.size == 0:
              return 0.0
          if v.size == 1:
              s_squared = float(v[0])
          else:
              w = 1.0 / v
              sw = float(np.sum(w))
              sw2 = float(np.sum(w * w))
              denom = sw * sw - sw2
              if denom > 0.0 and sw > 0.0:
                  s_squared = (v.size - 1) * sw / denom
              else:
                  # Numerically degenerate: fall back to harmonic mean.
                  s_squared = float(v.size / sw) if sw > 0 else 0.0

      total = float(tau_squared) + float(sigma_squared) + s_squared
      if total <= 0.0:
          return 0.0
      between = float(tau_squared) + float(sigma_squared)
      return float(max(0.0, min(100.0, (between / total) * 100.0)))

    def _run_three_level_analysis_intercept_only(
      self,
      y_all: List[np.ndarray],
      vcv_full: List[np.ndarray],
      vcv_diag: List[np.ndarray],
      has_shared_controls: bool,
      n_total: int,
      m_studies: int,
  ) -> Optional[Dict[str, Any]]:
      """Intercept-only three-level REML within one subgroup (shared
      engine, Cell 1b), used only by the independent-fits fallback path.
      Two-level (sigma² = 0) when no study has more than one effect size."""
      if m_studies < 2:
          return None

      n_multi_obs = sum(1 for y in y_all if len(y) > 1)
      three_level_viable = n_multi_obs >= 1 and n_total > m_studies
      use_full = three_level_viable and has_shared_controls

      try:
          fit = fit_blocks(y_all, vcv_full if use_full else vcv_diag,
                           two_level=not three_level_viable,
                           V_fallback_blocks=vcv_diag if use_full else None)
      except (ValueError, RuntimeError, np.linalg.LinAlgError):
          return None

      if not three_level_viable:
          model_type = '2-level'
      elif use_full and not fit.v_fallback:
          model_type = '3-level-vcv'
      else:
          model_type = '3-level-diag'

      best_result = fit_estimates(fit)
      best_result['optimizer_success'] = bool(fit.converged)
      best_result['optimizer_message'] = 'fit_mv'
      best_result['model_type'] = model_type
      best_result['n_studies'] = m_studies
      best_result['n_observations'] = n_total
      best_result['n_multi_obs_studies'] = n_multi_obs
      return best_result

    def run_three_level_analysis(
      self,
      y_all: List[np.ndarray],
      vcv_full: List[np.ndarray],
      vcv_diag: List[np.ndarray],
      X_blocks: List[np.ndarray],
      has_shared_controls: bool,
      n_total: int,
      m_studies: int,
  ) -> Optional[Dict[str, Any]]:
      """
      Unified three-level mixed-effects meta-regression.

      A single beta vector and a single (tau^2, sigma^2) pair are
      estimated jointly by REML across the entire dataset. The design
      matrix X (one block per study, stacked across rows) encodes
      subgroup membership via reference-coded dummies. This replaces
      the previous practice of fitting an intercept-only model
      independently within each subgroup.

      Fitted with the shared engine (fit_blocks / fit_mv, Cell 1b), as
      the meta-regression: full sampling VCV when there are shared
      controls (diagonal V as fallback); sigma² fixed at 0 when no study
      has more than one effect size.
      """
      if m_studies < 2 or n_total < 3 or len(X_blocks) == 0:
          return None
      if not (len(y_all) == len(vcv_full) == len(vcv_diag)
              == len(X_blocks) == m_studies):
          return None

      p_params = X_blocks[0].shape[1]
      if any(X_j.shape[1] != p_params for X_j in X_blocks):
          return None

      # Identifiability of the fixed effects: the stacked design
      # matrix must have full column rank.
      X_stacked = np.vstack(X_blocks)
      if np.linalg.matrix_rank(X_stacked) < p_params:
          return None

      # is_constant_within: whether the design matrix (subgroup/
      # moderator columns) is constant within every study. Reported with
      # the results; it does NOT decide whether sigma^2 is estimated (see
      # needs_variance_regularization below).
      is_constant_within = all(
          np.allclose(X_j, X_j[0:1, :], atol=1e-12) for X_j in X_blocks
      )

      # sigma^2 (within-study, between-effect variance) is only weakly
      # identified when NO study contributes more than one effect size --
      # i.e. there is no within-study replication anywhere to separate
      # sigma^2 from tau^2. This is a DIFFERENT condition from
      # is_constant_within: a study-level subgroup moderator (the common
      # case in subgroup analysis) makes is_constant_within True even
      # when many studies DO have multiple effect sizes, which fully
      # identifies sigma^2 without any regularization. Gating the
      # constrained optimizer on is_constant_within (as before) pulled
      # tau^2 toward a crude, unweighted method-of-moments prior in that
      # common case, biasing sigma^2 (and downstream CR2 SEs / Pseudo-R^2)
      # away from the unconstrained REML optimum that metafor/
      # clubSandwich compute.
      needs_variance_regularization = not any(
          X_j.shape[0] > 1 for X_j in X_blocks
      )

      # REML fit with the shared engine (fit_blocks, Cell 1b): the full
      # sampling VCV when there are shared controls, the diagonal V as
      # fallback; sigma² fixed at 0 when no study has replicate effects.
      try:
          fit = fit_blocks(y_all, vcv_full if has_shared_controls else vcv_diag, X_blocks,
                           two_level=needs_variance_regularization,
                           V_fallback_blocks=vcv_diag if has_shared_controls else None)
      except (ValueError, RuntimeError, np.linalg.LinAlgError):
          return None

      if has_shared_controls and not fit.v_fallback:
          final_vcv = vcv_full
          model_type = ("2-level VCV (one effect per study; σ² = 0)"
                        if needs_variance_regularization else "3-level VCV")
      else:
          final_vcv = vcv_diag
          model_type = ("2-level (one effect per study; σ² = 0)"
                        if needs_variance_regularization else "3-level diagonal")
      best_res = fit_as_optimize_result(fit)
      final_est = fit_estimates(fit)

      betas = final_est['betas']
      se_betas = final_est['se_betas']
      cov_beta_final = final_est['cov_beta']
      # Model-based (inverse-information) covariance, kept for Q_M: metafor's
      # QM for rma.mv uses this matrix, not the cluster-robust one.
      cov_beta_model = cov_beta_final
      used_robust = False

      # with_wald_helper=True additionally returns the Approximate
      # Hotelling-Zhang closure (Tipton & Pustejovsky 2015), which computes
      # a CONTRAST-SPECIFIC small-sample denominator DF for any C @ beta.
      # At q=1 it reduces exactly to clubSandwich's coef_test() Satterthwaite
      # DF, so it is the correct source of DF for a subgroup's pooled effect
      # (a contrast, not a coefficient) and for the omnibus test.
      htz_wald = None
      try:
          try:
              var_betas_robust, dfs_robust, htz_wald = _compute_robust_var_betas(
                  betas, y_all, final_vcv, X_blocks,
                  final_est['tau_sq'], final_est['sigma_sq'],
                  with_wald_helper=True,
              )
          except TypeError:
              # Older engine build without the Wald helper.
              var_betas_robust, dfs_robust = _compute_robust_var_betas(
                  betas, y_all, final_vcv, X_blocks,
                  final_est['tau_sq'], final_est['sigma_sq'],
              )
              htz_wald = None
          se_betas_robust = np.sqrt(np.diag(var_betas_robust))
          if np.all(np.isfinite(se_betas_robust)) and np.all(se_betas_robust > 0):
              se_betas = se_betas_robust
              cov_beta_final = var_betas_robust
              dfs = dfs_robust
              used_robust = True
          else:
              dfs = np.full(p_params, max(1, m_studies - p_params))
              htz_wald = None
      except Exception as e:
          warnings.warn(f"Subgroup model: CR2 step failed ({e}); using model-based SEs "
                        f"with df = studies - p.")
          dfs = np.full(p_params, max(1, m_studies - p_params))
          htz_wald = None

      final_est.update({
          'betas': betas,
          'se_betas': se_betas,
          'cov_beta': cov_beta_final,
          'cov_beta_model': cov_beta_model,
          'dfs': np.asarray(dfs, dtype=np.float64),
          # AHZ closure for contrast-specific DF (None when the robust fit
          # was not usable). NOT serialisable -- never persist it.
          'htz_wald': htz_wald if used_robust else None,
          'used_robust_se': used_robust,
          'model_type': model_type,
          # The exact per-study sampling-VCV blocks this fit used. The full
          # VCV or its diagonal fallback is chosen at run time, so a
          # caller that wants to reproduce this fit elsewhere (e.g. the
          # metafor/clubSandwich validation cell) MUST rebuild Sigma from
          # these blocks rather than guessing which list was taken.
          'vcv_used': final_vcv,
          # The fitted model itself, for the proportion of meaningful effects
          # (cell 9b), which reuses it and refits it on bootstrap resamples.
          'fit': fit,
          'is_constant_within': is_constant_within,
          'variance_regularized': needs_variance_regularization,
          'n_studies': m_studies,
          'n_obs': n_total,
          'p_params': p_params,
          'optimizer_converged': bool(getattr(best_res, 'success', False)),
      })
      return final_est

    # -------------------------------------------------------------------------
    # CONFIDENCE INTERVALS & P-VALUES
    # -------------------------------------------------------------------------

    def calculate_confidence_interval(
      self,
      estimate: float,
      se: float,
      df: Union[float, int, np.ndarray, Sequence[float]],
      contrast: Optional[np.ndarray] = None,
  ) -> Tuple[float, float, float]:
      """
      Confidence interval and two-sided p-value for a coefficient or
      a linear combination of coefficients of the unified meta-
      regression.

      Parameters
      ----------
      estimate, se : the point estimate and its standard error. For
          a linear combination c' beta, the caller supplies
          SE = sqrt(c' Cov(beta) c).
      df : Either a SCALAR DF (legacy / fallback path: a single
          number such as m_studies - p) or the FULL CR2 /
          Satterthwaite DF VECTOR returned by the regression engine
          (one entry per coefficient). When a vector is passed, the
          effective DF is selected via `contrast` (see below).
      contrast : Linear combination vector c, length = number of
          coefficients. Used only when `df` is a vector. The
          effective DF is taken as the smallest CR2 entry over the
          coefficients that participate in c -- a conservative
          Satterthwaite-style approximation that matches the
          clubSandwich convention for joint contrasts when full CR2
          machinery is not exposed. If `df` is a vector and
          `contrast` is None, the smallest positive CR2 entry is
          used (most conservative).

      Returns
      -------
      (ci_lower, ci_upper, p_value)
      """
      df_scalar = self._resolve_inferential_df(df, contrast)

      alpha = self.settings.alpha
      q_val = 1.0 - (alpha / 2.0)
      safe_se = float(se) if np.isfinite(se) and se > 0 else 0.0
      test_stat = (float(estimate) / safe_se) if safe_se > 0 else 0.0

      if self.settings.dist_type == 't':
          crit_val = float(t.ppf(q_val, df_scalar))
          p_value = 2.0 * (1.0 - float(t.cdf(abs(test_stat), df_scalar)))
      else:
          crit_val = float(norm.ppf(q_val))
          p_value = 2.0 * (1.0 - float(norm.cdf(abs(test_stat))))

      ci_lower = float(estimate) - crit_val * safe_se
      ci_upper = float(estimate) + crit_val * safe_se
      return ci_lower, ci_upper, p_value


    def _resolve_inferential_df(
        self,
        df: Union[float, int, np.ndarray, Sequence[float]],
        contrast: Optional[np.ndarray],
    ) -> float:
        """Resolve the `df` argument of calculate_confidence_interval
        to a single scalar value, using the contrast vector to pick
        the relevant entries when df is a CR2/Satterthwaite vector."""
        if np.isscalar(df):
            return max(1.0, float(df))

        arr = np.asarray(df, dtype=np.float64).ravel()
        if arr.size == 0:
            return 1.0
        if arr.size == 1:
            return max(1.0, float(arr[0]))

        if contrast is None:
            positive = arr[arr > 0]
            return max(1.0, float(np.min(positive)) if positive.size else 1.0)

        c = np.asarray(contrast, dtype=np.float64).ravel()
        if c.size != arr.size:
            positive = arr[arr > 0]
            return max(1.0, float(np.min(positive)) if positive.size else 1.0)

        nonzero_idx = np.where(np.abs(c) > 1e-12)[0]
        if nonzero_idx.size == 0:
            positive = arr[arr > 0]
            return max(1.0, float(np.min(positive)) if positive.size else 1.0)

        selected = arr[nonzero_idx]
        selected = selected[selected > 0]
        return max(1.0, float(np.min(selected)) if selected.size else 1.0)


    # -------------------------------------------------------------------------
    # HETEROGENEITY METRICS
    # -------------------------------------------------------------------------

    def calculate_q_statistic(
      self,
      y_all: List[np.ndarray],
      vcv_blocks: List[np.ndarray],
      X_blocks: List[np.ndarray],
  ) -> Tuple[float, int, float]:
      """
      Residual heterogeneity Q_E for the unified subgroup
      meta-regression, computed in the metafor convention:

          beta_FE = (X' W X)^{-1} X' W y
          e       = y - X beta_FE
          Q_E     = e' W e

      where W is the block-diagonal inverse of the SAMPLING
      variance-covariance matrix only (full V_j for studies with
      shared controls; diagonal otherwise) -- i.e., beta_FE is the
      GLS estimate that would obtain if tau^2 = sigma^2 = 0. Q_E is
      therefore a test of residual heterogeneity above and beyond
      what the moderator (subgroup membership) explains, distributed
      approximately as chi-squared with (n_obs - p) degrees of
      freedom under H_0: tau^2 = sigma^2 = 0.

      This replaces the previous within-subgroup Cochran's Q, which
      is preserved as the private helper _calculate_subgroup_cochran_q
      for the independent-fits fallback path.
      """
      if not y_all or not vcv_blocks or not X_blocks:
          return 0.0, 0, 1.0
      if not (len(y_all) == len(vcv_blocks) == len(X_blocks)):
          raise ValueError(
              "calculate_q_statistic: y_all, vcv_blocks, and X_blocks "
              "must all have the same length (one entry per study); got "
              f"{len(y_all)}, {len(vcv_blocks)}, {len(X_blocks)}."
          )

      p_params = X_blocks[0].shape[1]
      XtWX = np.zeros((p_params, p_params), dtype=np.float64)
      XtWy = np.zeros(p_params, dtype=np.float64)
      n_obs = 0
      V_inv_list: List[np.ndarray] = []

      for y_j, V_j, X_j in zip(y_all, vcv_blocks, X_blocks):
          y_j = np.asarray(y_j, dtype=np.float64)
          V_j = np.asarray(V_j, dtype=np.float64)
          X_j = np.asarray(X_j, dtype=np.float64)

          try:
              V_inv = np.linalg.inv(V_j)
          except np.linalg.LinAlgError:
              # Robust fallback: invert the diagonal only.
              v_diag = np.diag(V_j)
              v_safe = np.where(v_diag > 0, v_diag, np.inf)
              V_inv = np.diag(1.0 / v_safe)

          V_inv_list.append(V_inv)
          XtWX += X_j.T @ V_inv @ X_j
          XtWy += X_j.T @ V_inv @ y_j
          n_obs += y_j.size

      try:
          beta_fe = np.linalg.solve(XtWX, XtWy)
      except np.linalg.LinAlgError:
          return 0.0, max(0, n_obs - p_params), 1.0

      QE = 0.0
      for y_j, V_inv, X_j in zip(y_all, V_inv_list, X_blocks):
          y_j = np.asarray(y_j, dtype=np.float64)
          X_j = np.asarray(X_j, dtype=np.float64)
          e = y_j - X_j @ beta_fe
          QE += float(e @ V_inv @ e)

      if not np.isfinite(QE) or QE < 0.0:
          QE = 0.0

      df_qe = max(0, n_obs - p_params)
      p_qe = float(1.0 - chi2.cdf(QE, df_qe)) if df_qe > 0 and QE > 0 else 1.0
      return QE, df_qe, p_qe


    def _calculate_subgroup_cochran_q(
        self,
        subgroup_df: pd.DataFrame,
        mu: float,
    ) -> Tuple[float, int]:
        """LEGACY (PRESERVED). Within-subgroup Cochran's Q against the
        pooled effect mu. Used by the independent-fits fallback path
        and by Phase 2's per-row Q_within diagnostic (the latter call
        site needs to be repointed in a future integration phase).
        """
        y = np.asarray(subgroup_df[self.effect_col].values, dtype=np.float64)
        v = np.asarray(subgroup_df[self.var_col].values, dtype=np.float64)
        if y.size == 0:
            return 0.0, 0
        safe_v = np.where(v > 0, v, np.nan)
        w = 1.0 / safe_v
        valid = np.isfinite(w) & np.isfinite(y)
        if not np.any(valid):
            return 0.0, 0
        Q = float(np.sum(w[valid] * (y[valid] - float(mu)) ** 2))
        df_Q = max(0, int(np.sum(valid)) - 1)
        return Q, df_Q


# =============================================================================
# SUBGROUP ANALYSIS ORCHESTRATOR
# =============================================================================

# =============================================================================
# FACTORIAL (TWO-WAY) DECOMPOSITION HELPERS
# =============================================================================
# A two-way subgroup analysis collapses the cross-classification into a single
# 'levelA | levelB' label and fits one dummy per OBSERVED cell. That is a
# cell-means parametrisation, i.e. a SATURATED model: the interaction is
# already in the fit and imposes no additivity constraint. What the pipeline
# lacked was a TEST -- the omnibus Q_M lumps factor A, factor B and A x B into
# one statistic.
#
# These helpers build the three contrast spaces, in CELL-MEAN space, that split
# that omnibus into its factorial terms:
#
#   interaction : everything orthogonal to the additive model
#                 -> H0: the cell means are additive. Basis-independent: the
#                    Wald statistic depends only on the row space of C.
#   main A      : the part of the additive model's span that a B-only model
#                 cannot reach (A adjusted for B) -- a Type II contrast.
#   main B      : symmetrically, B adjusted for A.
#
# For any CONNECTED design the three dimensions sum to k-1, the omnibus df
# (verified over 362 random incomplete designs). The main-effect spaces are
# each orthogonal to the interaction space, but NOT to each other unless the
# grid is complete -- that non-orthogonality is exactly what makes these Type
# II rather than an orthogonal decomposition, so the three Wald statistics do
# not add up to the omnibus even though their df do.
#
# Contrasts are built unweighted on the cell-mean grid (the emmeans /
# least-squares-means convention) so that cell sizes do not define the
# hypothesis. Every cell-space contrast is estimable here, because the design
# carries one parameter per observed cell. Marginal MEANS are a different
# matter: a factor level observed in only one cell has no estimable mean
# across the other factor, which is why those are reported only on a complete
# grid.


def _orthonormal_basis(M, tol=1e-10):
    """Orthonormal basis for the column space of M (n x q, q may be 0)."""
    M = np.asarray(M, dtype=np.float64)
    if M.ndim != 2 or M.shape[1] == 0:
        return np.zeros((M.shape[0], 0))
    U, s, _ = np.linalg.svd(M, full_matrices=False)
    if s.size == 0:
        return np.zeros((M.shape[0], 0))
    keep = int(np.sum(s > tol * max(1.0, float(s[0]))))
    return U[:, :keep]


def _complement_basis(big, small, tol=1e-10):
    """Orthonormal basis for the part of col(big) orthogonal to col(small)."""
    Q_big = _orthonormal_basis(big, tol)
    if Q_big.shape[1] == 0:
        return Q_big
    Q_small = _orthonormal_basis(small, tol)
    if Q_small.shape[1] == 0:
        return Q_big
    return _orthonormal_basis(Q_big - Q_small @ (Q_small.T @ Q_big), tol)


def _factorial_contrast_bases(cell_labels, sep=' | '):
    """Type II / interaction contrast bases in cell-mean space.

    Returns None when the labels are not of the form 'levelA | levelB'.
    Each basis is (n_cells x q) with orthonormal columns.
    """
    a_vals, b_vals = [], []
    for label in cell_labels:
        if sep not in label:
            return None
        a_val, b_val = label.split(sep, 1)
        a_vals.append(a_val)
        b_vals.append(b_val)

    a_levels = sorted(set(a_vals))
    b_levels = sorted(set(b_vals))
    n = len(cell_labels)
    ones = np.ones((n, 1))
    D_a = np.array(
        [[1.0 if a == lv else 0.0 for lv in a_levels[1:]] for a in a_vals],
        dtype=np.float64,
    ).reshape(n, max(0, len(a_levels) - 1))
    D_b = np.array(
        [[1.0 if b == lv else 0.0 for lv in b_levels[1:]] for b in b_vals],
        dtype=np.float64,
    ).reshape(n, max(0, len(b_levels) - 1))

    additive = np.hstack([ones, D_a, D_b])
    rank_additive = _orthonormal_basis(additive).shape[1]

    return {
        'a_levels': a_levels,
        'b_levels': b_levels,
        'a_of_cell': a_vals,
        'b_of_cell': b_vals,
        'n_cells': n,
        'rank_additive': rank_additive,
        # Connected == every level reachable from every other through shared
        # cells. Disconnected designs cannot support a single additive model.
        'is_connected': rank_additive == len(a_levels) + len(b_levels) - 1,
        'is_complete_grid': n == len(a_levels) * len(b_levels),
        'main_a': _complement_basis(additive, np.hstack([ones, D_b])),
        'main_b': _complement_basis(additive, np.hstack([ones, D_a])),
        'interaction': _complement_basis(np.eye(n), additive),
    }


def _cell_mean_contrasts(levels, feature_names, reference_level):
    """Map beta -> per-cell pooled effects under reference coding.

    Row i of the returned (n_levels x p) matrix L satisfies
    mu_i = L[i] @ beta: the intercept alone for the reference cell, and
    intercept + that cell's dummy otherwise.
    """
    level_to_col = {reference_level: None}
    for col_idx, name in enumerate(feature_names):
        if col_idx == 0:
            continue
        level_to_col[name[len("subgroup["):-1]] = col_idx

    L = np.zeros((len(levels), len(feature_names)), dtype=np.float64)
    for row_idx, level in enumerate(levels):
        L[row_idx, 0] = 1.0
        col = level_to_col.get(level)
        if col is not None:
            L[row_idx, col] = 1.0
    return L, level_to_col


def _largest_complete_subgrid(cell_labels, cell_weights, sep=' | ',
                              max_enum_levels=18):
    """Largest fully-crossed rectangular subgrid of the observed cells.

    'Largest' means the most effect sizes retained (cell_weights), then the
    most cells, then lexicographically smallest -- so the choice is
    deterministic. Returns the sorted cell labels, or None when no 2x2 or
    larger complete subgrid exists (or the grid is too big to enumerate).

    Enumeration is over the factor with FEWER levels, which keeps the 2^n
    sweep tiny for the usual 'few levels x many levels' meta-analytic grid;
    beyond max_enum_levels it gives up rather than hang.
    """
    pairs = []
    for label in cell_labels:
        if sep not in label:
            return None
        pairs.append(tuple(label.split(sep, 1)))

    a_levels = sorted({p[0] for p in pairs})
    b_levels = sorted({p[1] for p in pairs})
    observed = set(pairs)

    flip = len(b_levels) < len(a_levels)
    outer, inner = (b_levels, a_levels) if flip else (a_levels, b_levels)
    if len(outer) > max_enum_levels:
        return None

    best_key, best_cells = None, None
    for mask in range(1, 1 << len(outer)):
        chosen = [outer[i] for i in range(len(outer)) if mask >> i & 1]
        if len(chosen) < 2:
            continue
        shared = [
            other for other in inner
            if all(((other, c) if flip else (c, other)) in observed
                   for c in chosen)
        ]
        if len(shared) < 2:
            continue
        cells = sorted(
            f"{other}{sep}{c}" if flip else f"{c}{sep}{other}"
            for c in chosen for other in shared
        )
        key = (sum(int(cell_weights.get(c, 0)) for c in cells), len(cells))
        if best_key is None or key > best_key or (key == best_key
                                                  and cells < best_cells):
            best_key, best_cells = key, cells
    return best_cells


def _factorial_wald_tests(bases, L, htz_wald, name_a, name_b):
    """Run the AHZ small-sample Wald test on each factorial contrast space."""
    terms = (
        ('main_a', str(name_a)),
        ('main_b', str(name_b)),
        ('interaction', f"{name_a} x {name_b}"),
    )
    out = {}
    for key, term_name in terms:
        K = bases[key]
        q = int(K.shape[1])
        entry = {'term': term_name, 'df_num': q}
        if q == 0:
            entry.update({
                'estimable': False,
                'reason': ('No estimable contrast of this kind survives the '
                           'observed cell pattern.'),
            })
        else:
            try:
                res = htz_wald(K.T @ L, test='AHZ')
                entry.update({
                    'estimable': True,
                    'F': float(res['F']),
                    'df_num': int(res['df_num']),
                    'df_denom': float(res['df_denom']),
                    'p_value': float(res['p_val']),
                })
            except Exception as exc:  # noqa: BLE001 - reported, not swallowed
                entry.update({'estimable': False, 'reason': str(exc)})
        out[key] = entry
    return out


def _factorial_marginal_means(bases, L, betas, cov_beta, htz_wald):
    """Equally-weighted marginal means per factor level.

    Only called on a COMPLETE grid, where every level's mean across the other
    factor is estimable and equally weighted. Returns rows with the estimate,
    its CR2 standard error and its contrast-specific Satterthwaite DF.
    """
    rows = []
    for factor_key, levels_key, of_cell in (
        ('a', 'a_levels', 'a_of_cell'),
        ('b', 'b_levels', 'b_of_cell'),
    ):
        for level in bases[levels_key]:
            idx = [i for i, v in enumerate(bases[of_cell]) if v == level]
            if not idx:
                continue
            c = L[idx, :].mean(axis=0)
            est = float(c @ betas)
            var = float(c @ cov_beta @ c)
            se = float(np.sqrt(max(var, 0.0)))
            df_val = np.nan
            if htz_wald is not None:
                try:
                    df_val = float(htz_wald(c, test='AHZ')['df_denom'])
                except Exception:
                    df_val = np.nan
            rows.append({
                'factor': factor_key,
                'level': level,
                'n_cells': len(idx),
                'marginal_mean': est,
                'se': se,
                'df': df_val if np.isfinite(df_val) else np.nan,
            })
    return rows


class SubgroupAnalysisEngine:
    """
    High-level orchestrator that coordinates the full subgroup analysis.
    Combines data management and statistical computation.
    """

    def __init__(self, data_manager: 'SubgroupDataManager'):
        """
        Initialize engine with data manager.

        Args:
            data_manager: SubgroupDataManager instance
        """
        self.data_manager = data_manager
        self.analyzer = SubgroupAnalyzer(
            effect_col=data_manager.effect_col,
            var_col=data_manager.var_col,
            global_settings=data_manager.global_settings,
            es_config=data_manager.es_config
        )

    def _factorial_tests(
        self,
        full_df: pd.DataFrame,
        levels: List[str],
        L: np.ndarray,
        estimates: Dict[str, Any],
        moderator1: str,
        moderator2: str,
        progress_callback=None,
    ) -> Dict[str, Any]:
        """Split the two-way omnibus into factor A, factor B and A x B.

        The fitted model is already saturated in the cross-classification
        (one dummy per observed cell), so nothing is re-estimated for the
        PRIMARY tests -- they are contrasts on the betas already in hand,
        evaluated with the same AHZ small-sample machinery (clubSandwich's
        Wald_test(test="HTZ")) used for the omnibus.

        When the observed cells do not form a complete rectangle, a
        SENSITIVITY analysis refits the model on the largest fully-crossed
        subgrid, where the main effects are orthogonal to each other and
        equally-weighted marginal means are estimable.
        """
        htz_wald = estimates.get('htz_wald')
        if htz_wald is None:
            return {
                'available': False,
                'reason': ('Robust (CR2) covariance was not available for this '
                           'fit, so the small-sample Wald machinery the '
                           'factorial tests are built on is missing.'),
            }

        bases = _factorial_contrast_bases(levels)
        if bases is None:
            return {
                'available': False,
                'reason': ("Subgroup labels are not of the form "
                           "'levelA | levelB'; cannot split them into two "
                           "factors."),
            }
        if not bases['is_connected']:
            return {
                'available': False,
                'reason': ('The observed cells form a disconnected design '
                           '(two or more groups of cells share no factor '
                           'level), so no single additive model spans them '
                           'and the factorial terms are not identifiable.'),
            }

        out: Dict[str, Any] = {
            'available': True,
            'moderator1': str(moderator1),
            'moderator2': str(moderator2),
            'n_cells_observed': int(bases['n_cells']),
            'n_cells_full_grid': int(len(bases['a_levels'])
                                     * len(bases['b_levels'])),
            'is_complete_grid': bool(bases['is_complete_grid']),
            'a_levels': list(bases['a_levels']),
            'b_levels': list(bases['b_levels']),
            'primary': _factorial_wald_tests(bases, L, htz_wald,
                                             moderator1, moderator2),
            'primary_type': ('Type III (complete grid)'
                             if bases['is_complete_grid']
                             else 'Type II (incomplete grid)'),
            'sensitivity': None,
        }

        if bases['is_complete_grid']:
            out['marginal_means'] = _factorial_marginal_means(
                bases, L,
                np.asarray(estimates['betas'], dtype=np.float64),
                np.asarray(estimates['cov_beta'], dtype=np.float64),
                htz_wald,
            )
            return out

        # ---- Sensitivity: refit on the largest complete subgrid ---------
        counts = full_df['subgroup_label'].value_counts().to_dict()
        keep_cells = _largest_complete_subgrid(levels, counts)
        if not keep_cells or len(keep_cells) >= len(levels):
            out['sensitivity_skipped_reason'] = (
                'No fully-crossed 2x2-or-larger subgrid exists among the '
                'observed cells.'
            )
            return out

        if progress_callback:
            progress_callback(
                f"Refitting on the largest complete subgrid "
                f"({len(keep_cells)} of {len(levels)} cells) as a "
                f"sensitivity check on the interaction."
            )
        try:
            out['sensitivity'] = self._refit_on_cells(
                full_df, keep_cells, moderator1, moderator2
            )
        except Exception as exc:  # noqa: BLE001 - surfaced, not hidden
            out['sensitivity_skipped_reason'] = (
                f"Refit on the complete subgrid failed: {exc}"
            )
        return out

    def _refit_on_cells(
        self,
        full_df: pd.DataFrame,
        keep_cells: List[str],
        moderator1: str,
        moderator2: str,
    ) -> Optional[Dict[str, Any]]:
        """Refit the unified model on a subset of cells and re-test it.

        Mirrors get_subgroup_data()'s construction exactly -- stable sort,
        treatment coding, most-studies reference -- so the sub-model is the
        same estimator applied to fewer cells, not a different one. The
        '_vcv_row_pos' tag survives the row filter, so the cached sampling
        VCVs are sliced to the retained rows rather than discarded.
        """
        sub = full_df.loc[full_df['subgroup_label'].isin(keep_cells)].copy()
        sub = sub.sort_values('id', kind='stable').reset_index(drop=True)
        sub_levels = sorted(sub['subgroup_label'].unique().tolist())
        if len(sub_levels) < 2 or len(sub) < 3:
            return None

        # Keep the user's pinned reference when it survives the subgrid;
        # otherwise fall back to the same most-studies rule
        # get_subgroup_data() uses, so the two fits stay comparable.
        studies_per_level = sub.groupby('subgroup_label')['id'].nunique()
        obs_per_level = sub['subgroup_label'].value_counts()
        pinned = self.data_manager.subgroup_config.reference_level
        pinned = str(pinned).strip() if pinned is not None else None
        if pinned in sub_levels:
            ref = pinned
        else:
            ref = min(
                sub_levels,
                key=lambda lv: (-int(studies_per_level.get(lv, 0)),
                                -int(obs_per_level.get(lv, 0)), lv),
            )

        dummies = pd.get_dummies(sub['subgroup_label'], prefix='subgroup',
                                 prefix_sep='[', drop_first=False)
        dummies.columns = [f"{c}]" for c in dummies.columns]
        dummies = dummies.drop(columns=[f"subgroup[{ref}]"])
        feature_names = ['(intercept)'] + list(dummies.columns)
        X_sub = np.column_stack([
            np.ones(len(sub), dtype=np.float64),
            dummies.to_numpy(dtype=np.float64),
        ])

        y_all, vcv_full, vcv_diag, X_blocks, has_shared = (
            self.data_manager.prepare_three_level_data(sub, X_sub)
        )
        est = self.analyzer.run_three_level_analysis(
            y_all=y_all, vcv_full=vcv_full, vcv_diag=vcv_diag,
            X_blocks=X_blocks, has_shared_controls=has_shared,
            n_total=int(len(sub)), m_studies=int(sub['id'].nunique()),
        )
        if est is None:
            return None

        L_sub, _ = _cell_mean_contrasts(sub_levels, feature_names, ref)
        bases_sub = _factorial_contrast_bases(sub_levels)
        if bases_sub is None:
            return None
        htz_sub = est.get('htz_wald')

        result: Dict[str, Any] = {
            'cells': list(sub_levels),
            'reference_level': ref,
            'k_obs': int(len(sub)),
            'n_studies': int(sub['id'].nunique()),
            'k_obs_dropped': int(len(full_df) - len(sub)),
            'cells_dropped': sorted(
                set(full_df['subgroup_label'].unique()) - set(sub_levels)
            ),
            'is_complete_grid': bool(bases_sub['is_complete_grid']),
            'tau_squared': float(est['tau_sq']),
            'sigma_squared': float(est['sigma_sq']),
            'model_type': est.get('model_type'),
        }
        if htz_sub is None:
            result['tests'] = None
            result['reason'] = ('Robust (CR2) covariance unavailable on the '
                                'subgrid refit.')
            return result

        result['tests'] = _factorial_wald_tests(
            bases_sub, L_sub, htz_sub, moderator1, moderator2
        )
        result['marginal_means'] = _factorial_marginal_means(
            bases_sub, L_sub,
            np.asarray(est['betas'], dtype=np.float64),
            np.asarray(est['cov_beta'], dtype=np.float64),
            htz_sub,
        )
        return result


    def analyze_single_subgroup(
        self,
        group_identifier: Union[str, Tuple[str, str]],
        progress_callback: Optional[callable] = None
    ) -> Optional['SubgroupResult']:
        """
        Analyze a single subgroup.

        Args:
            group_identifier: Group identifier (string or tuple)
            progress_callback: Optional callback(message: str) for progress updates

        Returns:
            SubgroupResult object or None if analysis fails
        """
        # Log progress
        if progress_callback:
            group_name = self.data_manager.get_group_display_name(group_identifier)
            progress_callback(f"Starting analysis: {group_name}")

        # Get subgroup data
        subgroup_df = self.data_manager.get_subgroup_data(group_identifier)

        if subgroup_df is None or len(subgroup_df) < 2:
            if progress_callback:
                progress_callback("❌ Insufficient data")
            return None

        k_group = len(subgroup_df)
        n_papers = subgroup_df['id'].nunique()

        if progress_callback:
            progress_callback(f"📊 k={k_group}, studies={n_papers}")

        # Prepare three-level data structures
        y_all, vcv_full, vcv_diag, has_shared = \
            self.data_manager.prepare_three_level_data(subgroup_df)

        # Run three-level analysis
        if progress_callback:
            progress_callback("🔄 Running optimization...")

        estimates = self.analyzer.run_three_level_analysis(
            y_all=y_all,
            vcv_full=vcv_full,
            vcv_diag=vcv_diag,
            has_shared_controls=has_shared,
            n_total=k_group,
            m_studies=n_papers
        )

        if estimates is None:
            if progress_callback:
                progress_callback("❌ Optimization failed")
            return None

        # Extract estimates
        mu = estimates['mu']
        se = estimates['se_mu']
        var = estimates['var_mu']
        tau_sq = estimates['tau_sq']
        sigma_sq = estimates['sigma_sq']
        model_type = estimates['model_type']

        # Calculate CI and p-value
        df = max(1, n_papers - 1)
        ci_lower, ci_upper, p_value = self.analyzer.calculate_confidence_interval(
            estimate=mu,
            se=se,
            df=df
        )

        # Calculate I²
        # Typical sampling variance of this group's effects (not their mean)
        i_squared = self.analyzer.calculate_i_squared(
            tau_sq, sigma_sq, subgroup_df[self.data_manager.var_col].values)

        # Calculate Q statistic (FIXED: pass mu as pooled_effect)
        Q_within, df_Q = self.analyzer._calculate_subgroup_cochran_q(subgroup_df, mu)

        # Calculate fold change
        fold_change = self.analyzer.calculate_fold_change(mu)

        if progress_callback:
            progress_callback(f"✅ Effect: {mu:.3f} | Model: {model_type}")

        # Build result object
        group_name = self.data_manager.get_group_display_name(group_identifier)

        result = SubgroupResult(
            group=group_name,
            k=k_group,
            n_papers=n_papers,
            pooled_effect_re=mu,
            pooled_se_re=se,
            pooled_var_re=var,
            ci_lower_re=ci_lower,
            ci_upper_re=ci_upper,
            p_value_re=p_value,
            I_squared=i_squared,
            tau_squared=tau_sq,
            sigma_squared=sigma_sq,
            fold_change_re=fold_change,
            Q_within=Q_within,
            df_Q=df_Q,
            model_type=model_type
        )

        # Add moderator values for two-way analysis
        if self.data_manager.subgroup_config.analysis_type == 'two_way':
            result.moderator1_value = group_identifier[0]
            result.moderator2_value = group_identifier[1]

        return result

    def analyze_all_subgroups(
      self,
      progress_callback: Optional[callable] = None,
  ) -> Tuple[pd.DataFrame, Dict[str, Any]]:
      """
      Unified subgroup analysis: a single mixed-effects meta-regression
      with subgroup as a categorical moderator and one shared
      (tau^2, sigma^2) pair across the whole model. Per-level pooled
      effects are reconstructed from the beta vector via reference-
      coded contrasts; the variance components are broadcast to every
      row of results_df so existing plotting code keeps working.

      If the unified fit fails to converge, falls back to the legacy
      independent per-subgroup fits via _analyze_subgroups_independent.
      """
      if progress_callback:
          progress_callback("Building unified design matrix.")

      bundle = self.data_manager.get_subgroup_data()
      full_df = bundle.df
      X = bundle.X
      feature_names = bundle.feature_names
      reference_level = bundle.reference_level
      levels = bundle.subgroup_levels

      y_all, vcv_full, vcv_diag, X_blocks, has_shared = (
          self.data_manager.prepare_three_level_data(full_df, X)
      )

      n_total = int(len(full_df))
      m_studies = int(full_df['id'].nunique())
      p_params = int(X.shape[1])

      if progress_callback:
          progress_callback(
              f"Fitting unified mixed-effects meta-regression: "
              f"k_obs={n_total}, studies={m_studies}, "
              f"levels={len(levels)}, reference='{reference_level}'."
          )

      estimates = self.analyzer.run_three_level_analysis(
          y_all=y_all,
          vcv_full=vcv_full,
          vcv_diag=vcv_diag,
          X_blocks=X_blocks,
          has_shared_controls=has_shared,
          n_total=n_total,
          m_studies=m_studies,
      )

      if estimates is None:
          if progress_callback:
              progress_callback(
                  "Unified model failed to converge. Falling back to "
                  "independent per-subgroup fits (legacy behavior)."
              )
          return self._analyze_subgroups_independent(progress_callback)

      betas = np.asarray(estimates['betas'], dtype=np.float64)
      cov_beta = np.asarray(estimates['cov_beta'], dtype=np.float64)
      dfs_per_coef = np.asarray(estimates['dfs'], dtype=np.float64)
      tau_sq = float(estimates['tau_sq'])
      sigma_sq = float(estimates['sigma_sq'])
      model_type = estimates['model_type']

      # Build the contrast matrix L mapping beta -> per-level pooled
      # effects. With reference coding:
      #   row(reference)     = (1, 0, 0, ..., 0)
      #   row(non-reference) = (1, ..., 1 at the level's dummy col, ...)
      L, level_to_col = _cell_mean_contrasts(
          levels, feature_names, reference_level
      )

      level_effects = L @ betas
      level_var = np.diag(L @ cov_beta @ L.T)
      level_se = np.sqrt(np.maximum(level_var, 0.0))

      # Per-level inferential DF.
      #
      # A subgroup's pooled effect is the linear combination L_i @ beta
      # (intercept + that level's dummy), so its small-sample DF is the
      # Satterthwaite DF OF THAT CONTRAST. That quantity depends on how
      # the clusters load onto L_i, hence on how many studies and effect
      # sizes the subgroup actually contributes.
      #
      # The previous rule -- min(df_intercept, df_dummy_i) -- is not that
      # quantity, and it degenerates: df_intercept enters EVERY minimum,
      # so a small reference subgroup pins every other level to its own
      # DF. A level with 63 studies would then report the same ~3.5 DF as
      # a 5-study reference, and its CI/p-value would be inflated by a t
      # critical value roughly 50% too large. Identical DF across groups
      # of wildly different size is the visible symptom.
      #
      # The AHZ helper (Tipton & Pustejovsky 2015) computes the genuine
      # contrast DF; at q=1 it matches clubSandwich's coef_test(). Fall
      # back to the old conservative rule per level only when the helper
      # is unavailable (non-robust fit / singular bread).
      htz_wald = estimates.get('htz_wald')
      level_dfs = np.empty(len(levels), dtype=np.float64)
      level_df_methods: List[str] = []
      for row_idx, level in enumerate(levels):
          col = level_to_col[level]
          df_value = np.nan
          if htz_wald is not None:
              try:
                  df_value = float(htz_wald(L[row_idx], test='AHZ')['df_denom'])
              except Exception:
                  df_value = np.nan
          if np.isfinite(df_value) and df_value > 0:
              level_df_methods.append('CR2 Satterthwaite (contrast, AHZ q=1)')
          else:
              df_value = (
                  float(dfs_per_coef[0]) if col is None
                  else float(min(dfs_per_coef[0], dfs_per_coef[col]))
              )
              level_df_methods.append('min(df_intercept, df_level) fallback')
          level_dfs[row_idx] = max(1.0, df_value)

      global_cfg = ANALYSIS_CONFIG.get('global_settings', {})
      alpha = global_cfg.get('alpha', 0.05)
      dist_type = global_cfg.get('dist_type', 't')

      t_stats = np.divide(
          level_effects, level_se,
          out=np.zeros_like(level_effects), where=level_se > 0,
      )

      df_pi = level_dfs

      if dist_type == 't':
          crit = t.ppf(1.0 - alpha / 2.0, level_dfs)
          crit_pi = t.ppf(1.0 - alpha / 2.0, df_pi)
          p_values = 2.0 * (1.0 - t.cdf(np.abs(t_stats), level_dfs))
      else:
          crit = norm.ppf(1.0 - alpha / 2.0)
          crit_pi = norm.ppf(1.0 - alpha / 2.0)
          p_values = 2.0 * (1.0 - norm.cdf(np.abs(t_stats)))

      ci_lower = level_effects - crit * level_se
      ci_upper = level_effects + crit * level_se

      pi_se = np.sqrt(level_var + tau_sq + sigma_sq)
      pi_lower = level_effects - crit_pi * pi_se
      pi_upper = level_effects + crit_pi * pi_se


      config = self.data_manager.subgroup_config
      rows: List[Dict[str, Any]] = []
      for row_idx, level in enumerate(levels):
          subset = full_df.loc[full_df['subgroup_label'] == level]
          k_level = int(len(subset))
          n_papers_level = int(subset['id'].nunique())
          mu_level = float(level_effects[row_idx])

          # The unified model has ONE tau² and sigma² for all groups, so this
          # I² differs between groups only through each group's typical
          # sampling variance; it is not a group-specific heterogeneity.
          i_squared = self.analyzer.calculate_i_squared(
              tau_sq, sigma_sq, subset[self.data_manager.var_col].values
          )
          Q_within, df_Q = self.analyzer._calculate_subgroup_cochran_q(
              subset, mu_level
          )
          fold_change = self.analyzer.calculate_fold_change(mu_level)

          row: Dict[str, Any] = {
              'group': level,
              'k': k_level,
              'n_papers': n_papers_level,
              'pooled_effect_re': mu_level,
              'pooled_se_re': float(level_se[row_idx]),
              'pooled_var_re': float(level_var[row_idx]),
              'ci_lower_re': float(ci_lower[row_idx]),
              'ci_upper_re': float(ci_upper[row_idx]),
              'pi_lower_re': float(pi_lower[row_idx]),
              'pi_upper_re': float(pi_upper[row_idx]),
              'p_value_re': float(p_values[row_idx]),
              'I_squared': i_squared,
              'I_squared_basis': 'common variance components (unified model)',
              'tau_squared': tau_sq,        # global value, broadcast
              'sigma_squared': sigma_sq,    # global value, broadcast
              'fold_change_re': fold_change,
              'Q_within': Q_within,
              'df_Q': df_Q,
              'model_type': model_type,
              'is_reference': (level == reference_level),
              'df_inference': float(level_dfs[row_idx]),
              'df_inference_method': level_df_methods[row_idx],
          }
          if config.analysis_type == 'two_way' and ' | ' in level:
              mod1_val, mod2_val = level.split(' | ', 1)
              row['moderator1_value'] = mod1_val
              row['moderator2_value'] = mod2_val
          rows.append(row)

      results_df = pd.DataFrame(rows)

      # Heterogeneity placeholder. Phase 3 will compute Q_M as a Wald
      # test on the (k-1) non-reference contrast coefficients, using
      # cov_beta exposed here. q_within_sum / qt_total are kept as
      # diagnostics for backward compatibility.
      weights = 1.0 / full_df[self.data_manager.var_col]
      sum_w = float(weights.sum())
      effects_col = full_df[self.data_manager.effect_col]
      if sum_w > 0:
          pooled_fe = float((weights * effects_col).sum() / sum_w)
          qt_total = float(np.sum(weights * (effects_col - pooled_fe) ** 2))
      else:
          qt_total = 0.0
      q_within_sum = float(results_df['Q_within'].sum())

      heterogeneity: Dict[str, Any] = {
        'tau_squared': tau_sq,
        'sigma_squared': sigma_sq,
        'qt_total': qt_total,
        'q_within_sum': q_within_sum,
        'k_total': n_total,
        'n_groups': len(levels),
        'reference_level': reference_level,
        'feature_names': feature_names,
        'betas': betas,
        'cov_beta': cov_beta,
        'dfs': dfs_per_coef,
        'model_type': model_type,
        'is_constant_within': bool(estimates.get('is_constant_within', False)),
        'optimizer_converged': bool(estimates.get('optimizer_converged', False)),
        'used_robust_se': bool(estimates.get('used_robust_se', False)),
        'fallback_used': False,
        # Not serialisable (a fitted model object); kept in memory only.
        'model_fit': {'fit': estimates.get('fit'), 'levels': list(levels), 'L': L},
        }

      # === PHASE 3 INTEGRATION ===

      # 1. Calculate Global Residual QE
      # (vcv_full naturally acts as vcv_diag when no off-diagonals exist)
      qe_residual, df_qe, _ = self.analyzer.calculate_q_statistic(y_all, vcv_full, X_blocks)

      # 2. Calculate QM (Wald test on dummy coefficients). Q_M uses the
      #    model-based covariance, so it equals metafor's QM for the same
      #    rma.mv model. The cluster-robust Wald statistic (CR2 covariance)
      #    is kept separately: it feeds the F fallback when the HTZ test is
      #    not available.
      contrast_indices = list(range(1, p_params))
      cov_beta_model = np.asarray(estimates.get('cov_beta_model', cov_beta),
                                  dtype=np.float64)
      qm_wald, df_qm = self.analyzer._compute_wald_qm(betas, cov_beta_model, contrast_indices)
      qm_wald_robust, _ = self.analyzer._compute_wald_qm(betas, cov_beta, contrast_indices)

      # 3. Pseudo-R² against the matching null (Cell 1b engine): the subgroup
      #    model and the intercept-only model, both fitted by REML on exactly
      #    the same blocks (rows), the same sampling V (sliced by row
      #    identity) and the same random structure. There is deliberately NO
      #    fallback to the global overall-analysis components (different rows,
      #    possibly a different structure): if the matching pair cannot be
      #    fitted, R² is reported as not computable.
      three_level_r2 = any(len(y_j) > 1 for y_j in y_all)
      if 'pseudo_r2_report' in globals():
          try:
              _r2_spec = fit_spec_from_blocks(
                  y_all, vcv_full, X_blocks, term_names=feature_names,
                  label='subgroup')
              r2_detail = pseudo_r2_report(
                  _r2_spec, three_level_r2,
                  reported=(tau_sq, sigma_sq, model_type))
          except Exception as e:
              r2_detail = {'R2': float('nan'), 'error': f"{type(e).__name__}: {e}"}
      else:
          r2_detail = {'R2': float('nan'),
                       'error': 'unified model engine not loaded (run Cell 1b)'}
      if progress_callback:
          if r2_detail.get('error'):
              progress_callback(f"WARNING: pseudo-R^2 not computable: {r2_detail['error']}")
          else:
              progress_callback(
                  f"Pseudo-R^2 vs intercept-only model on the same {n_total} observations: "
                  f"{fmt_r2(r2_detail['R2'])} (tau^2: {fmt_r2(r2_detail['R2_tau2'])}, "
                  f"sigma^2: {fmt_r2(r2_detail['R2_sigma2'])}).")

      # 4. Package subgroup fit info
      subgroup_fit = {
          'tau_squared': tau_sq,
          'sigma_squared': sigma_sq,
          'n_studies': m_studies,
          'p_params': p_params,
          'dfs': dfs_per_coef,
          # Contrast-specific small-sample machinery for the omnibus test:
          # the AHZ closure plus the q x p matrix selecting the (k-1)
          # non-reference contrasts. Local only -- never persisted.
          'htz_wald': estimates.get('htz_wald'),
          'contrast_matrix': (np.eye(p_params)[1:, :]
                              if p_params > 1 else None),
          # Wald statistic on the covariance used for inference (CR2 when
          # available); Q_M itself is model-based.
          'qm_wald_robust': qm_wald_robust,
      }

      # 5. Execute unified heterogeneity partitioning
      part_het = self.analyzer.partition_heterogeneity(
          r2_detail=r2_detail,
          subgroup_fit=subgroup_fit,
          qm_wald=qm_wald,
          df_qm=df_qm,
          qe_residual=qe_residual,
          df_qe=df_qe
      )

      # Merge partitioning stats (QM, R², etc.) into the main dictionary
      heterogeneity.update(part_het)
      _acc = getattr(self.data_manager, 'last_accounting', None)
      heterogeneity['data_accounting'] = _acc.to_dict() if _acc is not None else None

      # 6. Two-way only: split the omnibus into factor A, factor B and the
      # A x B interaction. The model is already saturated in the
      # cross-classification, so this re-estimates nothing for the primary
      # tests -- they are contrasts on the betas above. Plain floats/strings
      # only, so the result is safe to persist (the AHZ closure is not).
      if config.analysis_type == 'two_way' and config.moderator2:
          heterogeneity['factorial_tests'] = self._factorial_tests(
              full_df=full_df,
              levels=levels,
              L=L,
              estimates=estimates,
              moderator1=config.moderator1,
              moderator2=config.moderator2,
              progress_callback=progress_callback,
          )
      else:
          heterogeneity['factorial_tests'] = None

      if progress_callback:
          progress_callback(
              f"Unified fit complete. tau^2={tau_sq:.4f}, "
              f"sigma^2={sigma_sq:.4f}, model={model_type}."
          )

      return results_df, heterogeneity



    def _analyze_subgroups_independent(
      self,
      progress_callback: Optional[callable] = None,
  ) -> Tuple[pd.DataFrame, Dict[str, Any]]:
      """LEGACY FALLBACK. Independent random-effects fits, one per
      subgroup, with no shared variance components. Invoked only when
      the unified meta-regression fails to converge. Renamed (and
      promoted from worker to orchestrator) from the pre-refactor
      analyze_single_subgroup loop in analyze_all_subgroups.
      """
      config = self.data_manager.subgroup_config
      valid_groups = config.valid_groups_list

      results_list: List[Dict[str, Any]] = []
      total_q_within = 0.0

      for idx, group_id in enumerate(valid_groups, 1):
          if progress_callback:
              progress_callback(f"\n{'='*60}")
              progress_callback(
                  f"[fallback] Subgroup {idx}/{len(valid_groups)}"
              )
              progress_callback(f"{'='*60}")

          result = self._analyze_one_subgroup_independent(
              group_id, progress_callback
          )
          if result is not None:
              results_list.append(asdict(result))
              total_q_within += result.Q_within

      if not results_list:
          raise ValueError(
              "Independent-fits fallback also failed: no subgroups "
              "successfully analyzed. Aborting subgroup analysis."
          )

      results_df = pd.DataFrame(results_list)
      results_df['is_reference'] = False
      results_df['df_inference'] = results_df['n_papers'].apply(
          lambda n: float(max(1, int(n) - 1))
      )

      valid_dfs = []
      for g in valid_groups:
          sub_df = self.data_manager._get_single_subgroup_subset(g)
          if sub_df is not None:
              valid_dfs.append(sub_df)
      valid_subset_df = pd.concat(valid_dfs).reset_index(drop=True)

      weights = 1.0 / valid_subset_df[self.data_manager.var_col]
      effects = valid_subset_df[self.data_manager.effect_col]
      sum_w = float(weights.sum())
      if sum_w > 0:
          pooled_fe = float((weights * effects).sum() / sum_w)
          qt_subset = float(np.sum(weights * (effects - pooled_fe) ** 2))
      else:
          qt_subset = 0.0

      k_subset = int(len(valid_subset_df))
      heterogeneity = self.analyzer.partition_heterogeneity(
          qt_overall=qt_subset,
          q_within_sum=total_q_within,
          n_groups=len(results_df),
          k_overall=k_subset,
      )
      heterogeneity['model_type'] = 'INDEPENDENT FITS (FALLBACK)'
      heterogeneity['fallback_used'] = True
      heterogeneity['q_m'] = None
      heterogeneity['q_m_df'] = None
      heterogeneity['q_m_p_value'] = None

      if progress_callback:
          progress_callback("\n[fallback] Independent-fits analysis complete.")

      return results_df, heterogeneity

    def _analyze_one_subgroup_independent(
      self,
      group_identifier: Union[str, Tuple[str, str]],
      progress_callback: Optional[callable] = None,
  ) -> Optional['SubgroupResult']:
      """Per-subgroup worker for the independent-fits fallback. Body
      preserved from pre-refactor analyze_single_subgroup; only the
      data-manager and analyzer calls are rewired to the private
      legacy entry points (_get_single_subgroup_subset,
      _prepare_three_level_data_single,
      _run_three_level_analysis_intercept_only).
      """
      if progress_callback:
          group_name = self.data_manager.get_group_display_name(group_identifier)
          progress_callback(f"Starting analysis: {group_name}")

      subgroup_df = self.data_manager._get_single_subgroup_subset(group_identifier)
      if subgroup_df is None or len(subgroup_df) < 2:
          if progress_callback:
              progress_callback("Insufficient data")
          return None

      k_group = len(subgroup_df)
      n_papers = subgroup_df['id'].nunique()

      if progress_callback:
          progress_callback(f"k={k_group}, studies={n_papers}")

      y_all, vcv_full, vcv_diag, has_shared = (
          self.data_manager._prepare_three_level_data_single(subgroup_df)
      )

      if progress_callback:
          progress_callback("Running optimization (intercept-only).")

      estimates = self.analyzer._run_three_level_analysis_intercept_only(
          y_all=y_all,
          vcv_full=vcv_full,
          vcv_diag=vcv_diag,
          has_shared_controls=has_shared,
          n_total=k_group,
          m_studies=n_papers,
      )
      if estimates is None:
          if progress_callback:
              progress_callback("Optimization failed")
          return None

      mu = estimates['mu']
      se = estimates['se_mu']
      var = estimates['var_mu']
      tau_sq = estimates['tau_sq']
      sigma_sq = estimates['sigma_sq']
      model_type = estimates['model_type']

      df = max(1, n_papers - 1)
      ci_lower, ci_upper, p_value = self.analyzer.calculate_confidence_interval(
          estimate=mu, se=se, df=df,
      )

       # --- PI CALCULATION FOR FALLBACK ---
      df_pi = max(1, n_papers - 2)
      alpha = self.data_manager.global_settings.alpha
      if self.data_manager.global_settings.dist_type == 't':
          crit_pi = t.ppf(1.0 - alpha / 2.0, df_pi)
      else:
          crit_pi = norm.ppf(1.0 - alpha / 2.0)

      pi_se = np.sqrt(se**2 + tau_sq + sigma_sq)
      pi_lower = mu - crit_pi * pi_se
      pi_upper = mu + crit_pi * pi_se

      # Typical sampling variance of this group's effects (not their mean)
      i_squared = self.analyzer.calculate_i_squared(
          tau_sq, sigma_sq, subgroup_df[self.data_manager.var_col].values)
      Q_within, df_Q = self.analyzer._calculate_subgroup_cochran_q(subgroup_df, mu)
      fold_change = self.analyzer.calculate_fold_change(mu)

      if progress_callback:
          progress_callback(f"Effect: {mu:.3f} | Model: {model_type}")

      group_name = self.data_manager.get_group_display_name(group_identifier)
      result = SubgroupResult(
          group=group_name,
          k=k_group,
          n_papers=n_papers,
          pooled_effect_re=mu,
          pooled_se_re=se,
          pooled_var_re=var,
          ci_lower_re=ci_lower,
          ci_upper_re=ci_upper,
          pi_lower_re=pi_lower,
          pi_upper_re=pi_upper,
          p_value_re=p_value,
          I_squared=i_squared,
          tau_squared=tau_sq,
          sigma_squared=sigma_sq,
          fold_change_re=fold_change,
          Q_within=Q_within,
          df_Q=df_Q,
          model_type=f"{model_type} (independent)",
      )
      if self.data_manager.subgroup_config.analysis_type == 'two_way':
          result.moderator1_value = group_identifier[0]
          result.moderator2_value = group_identifier[1]
      return result

# =============================================================================
# Pure UI rendering without business logic or data access
# Dependencies:
#   - Data Layer (SubgroupDataManager, SubgroupResult, GlobalSettings)
#   - Analysis Layer (SubgroupAnalyzer results)
# =============================================================================

import pandas as pd
import numpy as np
from typing import Dict, Any, Optional, List
import datetime
import ipywidgets as widgets
from IPython.display import display, HTML


# =============================================================================
# HTML TEMPLATE GENERATORS
# =============================================================================

class HTMLTemplates:
    """
    Static HTML template generators for consistent styling.
    All methods are pure functions that return HTML strings.
    """

    @staticmethod
    def card(content: str, bg_color: str = '#f8f9fa', border_color: str = '#dee2e6') -> str:
        """Generate a styled card container"""
        return f"""
        <div style='background-color: {bg_color}; padding: 15px; border-radius: 5px;
                    margin: 10px 0; border: 1px solid {border_color};'>
            {content}
        </div>
        """

    @staticmethod
    def header(text: str, level: int = 3, color: str = '#2c3e50') -> str:
        """Generate a styled header"""
        border = "border-bottom: 2px solid #3498db; padding-bottom: 10px;" if level == 3 else ""
        return f"<h{level} style='color: {color}; {border}'>{text}</h{level}>"

    @staticmethod
    def error_message(error_type: str, message: str, details: Optional[str] = None) -> str:
        """Generate styled error message"""
        html = f"""
        <div style='color: red; background-color: #f8d7da; padding: 15px;
                    border-radius: 5px; margin: 10px 0; border: 1px solid #f5c6cb;'>
            <b>❌ {error_type}:</b> {message}
        """
        if details:
            html += f"<pre style='margin-top: 10px; font-size: 0.9em;'>{details}</pre>"
        html += "</div>"
        return html

    @staticmethod
    def success_message(message: str) -> str:
        """Generate styled success message"""
        return f"""
        <div style='color: #155724; background-color: #d4edda; padding: 15px;
                    border-radius: 5px; margin: 10px 0; border: 1px solid #c3e6cb;'>
            ✅ {message}
        </div>
        """

    @staticmethod
    def warning_message(message: str) -> str:
        """Generate styled warning message"""
        return f"""
        <div style='color: #856404; background-color: #fff3cd; padding: 15px;
                    border-radius: 5px; margin: 10px 0; border: 1px solid #ffeeba;'>
            ⚠️ {message}
        </div>
        """

    @staticmethod
    def significance_legend() -> str:
        """Generate significance legend"""
        return """
        <div style='font-size: 0.9em; color: #555; margin-top: 10px;
                    background-color: #f8f9fa; padding: 10px; border-radius: 4px;'>
            <b>Legend:</b> *** p < 0.001; ** p < 0.01; * p < 0.05
        </div>
        """

    @staticmethod
    def model_type_legend() -> str:
        """Generate model type explanation"""
        return """
        <div style='font-size: 0.9em; color: #555; margin-top: 10px;
                    background-color: #f8f9fa; padding: 10px; border-radius: 4px;'>
            <b>Model Codes:</b><br>
            • <b>3L-VCV:</b> 3-level model with the full sampling variance-covariance matrix (shared controls).<br>
            • <b>3L-Diag:</b> 3-level model with independent sampling errors (no shared controls, or the full matrix could not be used).<br>
            • <b>2L:</b> 2-level model (σ² fixed at 0) because no study in the subgroup has more than one effect size.
        </div>
        """


# =============================================================================
# PUBLICATION TEXT GENERATORS
# =============================================================================

class PublicationTextGenerator:
    """
    Generates publication-ready text for manuscripts.
    All methods return formatted HTML strings.
    """

    def __init__(self, ci_percent: float = 95):
        """
        Initialize generator.

        Args:
            ci_percent: Confidence interval percentage (e.g., 95)
        """
        self.ci_percent = ci_percent

    def generate_methods_section(
        self,
        moderator1: str,
        moderator2: Optional[str],
        n_groups: int,
        fallback_used: bool = False
    ) -> str:
        """Generate Materials and Methods section"""

        html = f"""
        <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt;
                    line-height: 1.8; padding: 20px; background-color: #ffffff;
                    border: 1px solid #eee; margin-bottom: 20px;'>

        <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db;
                   padding-bottom: 10px; margin-top: 0;'>Materials and Methods</h3>

        <p style='text-align: justify;'>
        <b>Subgroup Analysis.</b> To investigate potential sources of heterogeneity,
        we conducted a mixed-effects subgroup analysis [1]. The categorical moderator(s)
        tested were <b>{moderator1}</b>"""

        if moderator2:
            html += f" and <b>{moderator2}</b>"

        html += f""". The analysis tested whether the mean effect differed among
        the moderator categories.
        </p>

        <p style='text-align: justify;'>
        <b>Statistical Tests.</b> Differences between subgroups were assessed with an
        omnibus test of the moderator contrasts. We report the cluster-robust
        small-sample <i>F</i> test (approximate Hotelling&ndash;Zhang applied to the
        CR2 covariance matrix), which holds its nominal level when the number of
        independent studies is modest. For comparison with standard software we
        also report the model-based Wald statistic <i>Q</i><sub>M</sub>
        (asymptotic chi-square, computed from the model-based covariance matrix of
        the coefficients, as the <i>Q</i><sub>M</sub> of <code>metafor</code> [2]);
        it does not account for clustering or small samples, so the <i>F</i> test [3]
        governs the conclusions. A significant
        omnibus test indicates that the effect sizes vary systematically across
        the categories of the moderator. We also calculated a pseudo-<i>R</i>² statistic [4]
        to quantify the proportion of the total heterogeneity (between- plus
        within-study variance) explained by the moderator.
        </p>

        <p style='text-align: justify;'>
        """

        # Dynamic Model Specification text based on fallback
        if fallback_used:
            html += """<b>Model Specification.</b> Due to matrix convergence constraints with a unified model,
            subgroups were evaluated using independent multilevel random-effects models. This approach estimates
            group-specific pooled effects and allows both the residual between-study variance (τ²) and
            within-study variance (σ²) to vary independently across subgroups. The omnibus test for moderation
            was derived via the subtraction method. All computations were performed using the Co-Meta toolkit [5]."""
        else:
            html += """<b>Model Specification.</b> A single mixed-effects meta-regression model was fitted
            using subgroups as categorical moderators via reference-coded dummy variables.
            This unified approach estimates group-specific pooled effects while assuming a
            common residual between-study variance (τ²) and within-study variance (σ²)
            across all subgroups. All computations were performed using the Co-Meta toolkit [5]."""

        html += """
        </p>

        <h4 style='color: #34495e; margin-top: 20px; font-size: 11pt;'>References</h4>
        <ol style='font-size: 10pt; color: #555;'>
            <li>Borenstein, M., Hedges, L. V., Higgins, J. P., & Rothstein, H. R. (2009).
                <i>Introduction to Meta-Analysis</i>. Chichester, UK: John Wiley & Sons.</li>
            <li>Viechtbauer, W. (2010). Conducting meta-analyses in R with the metafor package.
                <i>Journal of Statistical Software</i>, 36(3), 1-48.</li>
            <li>Tipton, E., & Pustejovsky, J. E. (2015). Small-sample adjustments for tests of
                moderators and model fit using robust variance estimation in meta-regression.
                <i>Journal of Educational and Behavioral Statistics</i>, 40(6), 604-634.</li>
            <li>Raudenbush, S. W. (2009). Analyzing effect sizes: Random-effects models.
                In H. Cooper, L. V. Hedges, & J. C. Valentine (Eds.),
                <i>The Handbook of Research Synthesis and Meta-Analysis</i> (2nd ed., pp. 295-315).
                New York: Russell Sage Foundation.</li>
            <li>""" + COMETA_CITATION + """</li>
        </ol>
        </div>
        """

        return html


    @staticmethod
    def _format_factorial_term(entry: Dict[str, Any]) -> str:
        """One term of the factorial table as an <li>."""
        if not entry.get('estimable'):
            return (f"<li><b>{entry['term']}</b>: not estimable &mdash; "
                    f"{entry.get('reason', 'no contrast available')}.</li>")
        p_val = entry['p_value']
        p_txt = "&lt; 0.001" if p_val < 0.001 else f"= {p_val:.3f}"
        verdict = "significant" if p_val < _alpha() else "non-significant"
        return (f"<li><b>{entry['term']}</b>: <i>F</i>({entry['df_num']}, "
                f"{entry['df_denom']:.1f}) = <b>{entry['F']:.2f}</b>, "
                f"<i>p</i> {p_txt} &mdash; {verdict}.</li>")

    def _render_factorial_section(self, factorial: Dict[str, Any]) -> str:
        """Report the factor A / factor B / A x B decomposition.

        The omnibus Q_M tests 'do any cells differ at all', which confounds
        the two main effects with their interaction. This section splits it
        with small-sample Wald tests (Tipton & Pustejovsky 2015, the AHZ test
        clubSandwich exposes as Wald_test(test="HTZ")) on contrasts of the
        already-fitted cell means.
        """
        primary = factorial.get('primary', {})
        complete = factorial.get('is_complete_grid', False)

        html = """
        <h4 style='color: #34495e; margin-top: 25px;'>Factorial Decomposition
        (Main Effects and Interaction)</h4>

        <p style='text-align: justify;'>
        The omnibus test above asks only whether any cell differs from any
        other, which confounds the two moderators with their interaction. It
        was therefore decomposed into its factorial terms using small-sample
        robust Wald tests on contrasts of the fitted cell means
        (approximate Hotelling&ndash;Zhang; Tipton &amp; Pustejovsky, 2015):
        </p>
        <ul style='text-align: justify;'>
        """
        for key in ('main_a', 'main_b', 'interaction'):
            if key in primary:
                html += self._format_factorial_term(primary[key])
        html += "</ul>"

        n_obs = factorial.get('n_cells_observed')
        n_full = factorial.get('n_cells_full_grid')

        if complete:
            html += f"""
        <p style='text-align: justify; font-size: 0.85em; color: #666;'>
        All {n_obs} cells of the {n_full}-cell cross-classification are
        observed, so the two main effects are orthogonal to each other and
        these are Type III tests of equally-weighted marginal means.
        </p>"""
            return html

        html += f"""
        <p style='text-align: justify; font-size: 0.85em; color: #666;'>
        Only {n_obs} of the {n_full} cells of the full cross-classification
        are observed, so the design is unbalanced and incomplete. The main
        effects are therefore <b>Type II</b> (each adjusted for the other
        moderator, computed on the unweighted cell-mean grid): their degrees
        of freedom sum to the omnibus degrees of freedom, but because the two
        main-effect contrast spaces are not mutually orthogonal in an
        incomplete design, the three test statistics do not partition
        <i>Q</i><sub>M</sub> additively. The interaction term uses only the
        cells that can be crossed; levels observed in a single cell of the
        other factor contribute to the main effects but carry no interaction
        information. Marginal means per moderator level are not reported
        here because they are not estimable for every level.
        </p>"""

        sens = factorial.get('sensitivity')
        if not sens or not sens.get('tests'):
            reason = factorial.get('sensitivity_skipped_reason')
            if reason:
                html += (f"""
        <p style='text-align: justify; font-size: 0.85em; color: #666;'>
        Sensitivity analysis on a complete subgrid was not run: {reason}
        </p>""")
            return html

        dropped = sens.get('cells_dropped') or []
        dropped_txt = ", ".join(f"<i>{c}</i>" for c in dropped) or "none"
        html += f"""
        <p style='text-align: justify;'>
        <b>Sensitivity analysis.</b> Because the incomplete grid makes the
        main effects Type II, the model was refit on the largest
        fully-crossed subgrid &mdash; {len(sens['cells'])} cells,
        <b>{sens['k_obs']}</b> effect sizes from <b>{sens['n_studies']}</b>
        studies, dropping {sens['k_obs_dropped']} effect sizes
        ({dropped_txt}). On that balanced subgrid the terms are orthogonal
        and fully estimable:
        </p>
        <ul style='text-align: justify;'>
        """
        for key in ('main_a', 'main_b', 'interaction'):
            if key in sens['tests']:
                html += self._format_factorial_term(sens['tests'][key])
        html += f"""</ul>
        <p style='text-align: justify; font-size: 0.85em; color: #666;'>
        Variance components on the subgrid: &tau;&sup2; =
        {sens['tau_squared']:.4f}, &sigma;&sup2; =
        {sens['sigma_squared']:.4f}. Agreement between the two interaction
        tests indicates the conclusion is not an artefact of the incomplete
        cross-classification; disagreement indicates it is driven by the
        cells that cannot be crossed.
        </p>"""
        return html

    def generate_results_section(
        self,
        results_df: pd.DataFrame,
        moderator1: str,
        moderator2: Optional[str],
        heterogeneity: Dict[str, Any]
    ) -> str:
        """Generate complete Results section with interpretation"""

        QM = heterogeneity['QM']
        df_QM = heterogeneity['df_QM']
        p_QM = heterogeneity['p_value_QM']
        R_sq = heterogeneity['R_squared']
        r2_ok = R_sq is not None and np.isfinite(R_sq)
        _r2d = heterogeneity.get('R_squared_detail') or {}
        if r2_ok:
            _r2_level = ""
            if _r2d.get('three_level'):
                _r2_level = "; " + _level_phrase(_r2d)
            _r2_sentence = (f"accounting for <b>{R_sq:.1f}%</b> of the total heterogeneity "
                            f"relative to the intercept-only model fitted to the same effect "
                            f"sizes (pseudo-<i>R</i>² = {R_sq:.1f}%{_r2_level}).")
        else:
            _r2_sentence = ("the proportion of heterogeneity it explains (pseudo-<i>R</i>²) "
                            "could not be computed.")
        QE = heterogeneity['QE']
        df_QE = heterogeneity['df_QE']
        Qt = QM + QE

        M_groups = len(results_df)

        # Cluster-robust small-sample omnibus: the approximate
        # Hotelling-Zhang test (Tipton & Pustejovsky 2015), i.e. what
        # clubSandwich exposes as Wald_test(test = "HTZ"). It is computed by
        # _omnibus_f_from_wald on the same contrasts as Q_M; before this it
        # was returned in `heterogeneity` and never displayed anywhere.
        F_om = heterogeneity.get('F_omnibus')
        df_F_num = heterogeneity.get('df_F_num')
        df_F_den = heterogeneity.get('df_F_den')
        p_F_om = heterogeneity.get('p_value_F_omnibus')
        has_htz = (
            all(v is not None and np.isfinite(v)
                for v in (F_om, df_F_num, df_F_den, p_F_om))
            and float(df_F_num) > 0 and float(df_F_den) > 0
        )

        # Q_M is the model-based asymptotic chi-square Wald statistic (as
        # metafor's QM); it ignores clustering and with few clusters it
        # rejects far too often, which is exactly why Tipton &
        # Pustejovsky recommend the F test. So when the F test is available
        # the narrative verdict follows IT, and Q_M is reported alongside
        # for comparability with metafor. Letting the two disagree inside
        # one document would be worse than either choice.
        p_primary = float(p_F_om) if has_htz else p_QM

        sig_QM = "significant" if p_primary < _alpha() else "non-significant"
        p_format = "&lt; 0.001" if p_QM < 0.001 else f"= {p_QM:.3f}"
        p_format_F = (("&lt; 0.001" if p_F_om < 0.001 else f"= {p_F_om:.3f}")
                      if has_htz else "")

        # R² interpretation — finer cutpoints, and acknowledge the QM verdict
        if not r2_ok:
            r2_interp = ("proportion of heterogeneity explained by this moderator "
                         "could not be quantified")
        elif R_sq < 2:
            r2_interp = ("near-zero R² indicates this moderator explains essentially "
                         "none of the between-study heterogeneity")
        elif R_sq < 13:
            r2_interp = ("relatively low R² suggests this moderator explains a modest "
                         "proportion of the heterogeneity")
        elif R_sq < 26:
            r2_interp = ("moderate R² indicates this moderator explains an appreciable "
                         "proportion of the heterogeneity")
        else:
            r2_interp = ("substantial R² indicates this moderator is a major source "
                         "of the heterogeneity")

        html = f"""
        <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt;
                    line-height: 1.8; padding: 20px; background-color: #ffffff;'>

        <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;'>
            Subgroup Analysis Results
        </h3>

        <p style='text-align: justify;'>
        To explore sources of heterogeneity, we conducted a subgroup analysis based on
        <b>{moderator1}</b>"""

        factorial = heterogeneity.get('factorial_tests') or {}
        has_factorial = bool(factorial.get('available'))

        if moderator2:
            if has_factorial:
                html += (f" and <b>{moderator2}</b>, fitted as a single "
                         "mixed-effects model with one parameter per observed "
                         "cell of the cross-classification (a saturated "
                         "two-way design, so the interaction is estimated "
                         "rather than assumed away)")
            else:
                html += f" and <b>{moderator2}</b>"

        html += f""". The dataset included <b>{M_groups}</b> subgroups with sufficient data
        for analysis (minimum of 2 studies per subgroup).
        </p>

        <h4 style='color: #34495e; margin-top: 25px;'>Overall Test for Subgroup Differences</h4>
        """

        if p_primary < _alpha():
            _verdict_tail = (
                "The moderator therefore explains a statistically detectable "
                f"share of the variation in effect sizes, {_r2_sentence}"
            )
        else:
            _verdict_tail = (
                "The moderator therefore does not explain a statistically "
                f"detectable share of the variation in effect sizes; {_r2_sentence}"
            )

        if has_htz:
            html += f"""
        <p style='text-align: justify;'>
        The omnibus test for subgroup differences was <b>{sig_QM}</b> under the
        cluster-robust small-sample test (approximate Hotelling&ndash;Zhang on
        the CR2 covariance matrix; Tipton &amp; Pustejovsky, 2015):
        <i>F</i>({int(df_F_num)}, {df_F_den:.1f}) = <b>{F_om:.2f}</b>,
        <i>p</i> {p_format_F}. The model-based Wald statistic for the same
        hypothesis (as reported by <code>metafor</code>) is
        <i>Q</i><sub>M</sub>({df_QM}) = {QM:.2f}, <i>p</i> {p_format}; it does
        not account for clustering and can reject too readily when the number
        of studies is small, so the <i>F</i> test is the one followed here.
        {_verdict_tail}
        </p>"""
        else:
            html += f"""
        <p style='text-align: justify;'>
        The omnibus {'(model-based) Wald test' if heterogeneity.get('qm_method', 'wald') == 'wald' else 'test (fixed-effect Q partition, independent subgroup fits)'} for subgroup differences was <b>{sig_QM}</b>
        (<i>Q</i><sub>M</sub>({df_QM}) = <b>{QM:.2f}</b>, <i>p</i> {p_format}).
        {_verdict_tail}
        </p>"""

        if has_factorial:
            html += self._render_factorial_section(factorial)

        html += f"""
        <h4 style='color: #34495e; margin-top: 25px;'>Heterogeneity Partitioning</h4>

        <p style='text-align: justify;'>
        Heterogeneity was partitioned into between-group
        (<i>Q</i><sub>M</sub>({df_QM}) = {QM:.2f}) and residual within-group components
        (<i>Q</i><sub>E</sub>({df_QE}) = {QE:.2f}) from the total heterogeneity
        (<i>Q</i><sub>T</sub>({int(df_QM + df_QE)}) = {Qt:.2f}). The {r2_interp}.
        </p>
        <p style='text-align: justify; font-size: 0.85em; color: #666;'>
        Note: <i>Q</i><sub>T</sub> here is the total heterogeneity under the mixed-effects
        model (<i>Q</i><sub>M</sub> + <i>Q</i><sub>E</sub>) and is not directly comparable to
        the Cochran's <i>Q</i> reported in the overall analysis, which uses fixed-effect weights.
        </p>

        <h4 style='color: #34495e; margin-top: 25px;'>Individual Subgroup Results</h4>

        <p style='text-align: justify;'>
        Results by subgroup were as follows (Table 1):
        </p>

        <ul style='line-height: 2.0;'>
        """


        # Individual subgroup summaries
        any_underpowered = False
        for _, row in results_df.iterrows():
            n_pap = int(row['n_papers'])
            p_fmt = "&lt; 0.001" if row['p_value_re'] < 0.001 else f"= {row['p_value_re']:.3f}"

            if n_pap < 3:
                # Single/two-study subgroup: between-study inference is not
                # identifiable. Report the point estimate but withhold CI/p,
                # which are driven by an unreliable CR2 df (false precision
                # when the cluster count collapses, e.g. n_papers = 1).
                any_underpowered = True
                html += f"""
                <li><b>{row['group']}:</b> Based on {int(row['k'])} effect sizes from
                {n_pap} stud{'y' if n_pap == 1 else 'ies'}, the pooled effect was
                <b>{row['pooled_effect_re']:.3f}</b>. <span style='color:#856404;'>
                (Confidence interval and <i>p</i>-value withheld: fewer than 3 studies;
                between-study variance is not estimable for this subgroup.)</span></li>
                """
            else:
                # Only show df if using t-distribution
                dist_type = ANALYSIS_CONFIG.get('global_settings', {}).get('dist_type', 't')
                df_text = f", df = {row['df_inference']:.1f}" if dist_type == 't' else ""

                html += f"""
                <li><b>{row['group']}:</b> Based on {int(row['k'])} effect sizes from
                {n_pap} studies, the pooled effect was <b>{row['pooled_effect_re']:.3f}</b>
                ({self.ci_percent:.0f}% CI [{row['ci_lower_re']:.3f}, {row['ci_upper_re']:.3f}],
                <i>p</i> {p_fmt}{df_text}).</li>
                """
        html += "</ul>"

        if any_underpowered:
            html += """
            <p style='text-align: justify; font-size: 0.9em; color: #856404;
                      background-color: #fff3cd; padding: 10px; border-left: 4px solid #ffc107;'>
            <b>Note:</b> Subgroups with fewer than 3 studies are reported as point estimates only.
            Interval estimates for these groups rest on too few clusters to support reliable
            cluster-robust variance estimation and should not be interpreted.
            </p>
            """

        html += "</ul>"

        fallback_used = heterogeneity.get('fallback_used', False)
        tau_sq = heterogeneity.get('tau_squared', 0)
        sigma_sq = heterogeneity.get('sigma_squared', 0)


           # Dynamic Variance Text
        if fallback_used:
            html += f"""
            <p style='text-align: justify;'>
            Because subgroups were modeled independently, residual between-study variance (τ²)
            and within-study variance (σ²) were estimated separately for each subgroup rather
            than as a single global estimate.
            </p>
            """
        else:
            html += f"""
            <p style='text-align: justify;'>
            Across all subgroups, the global residual between-study variance (τ²) was estimated at {tau_sq:.4f},
            and the within-study variance (σ²) was {sigma_sq:.4f}.
            </p>
            """


        # Concluding Statistical Interpretation -- same governing p as the
        # omnibus paragraph above, so the document cannot contradict itself.
        if p_primary < _alpha():
            if r2_ok and R_sq < 5.0:
                html += f"""
                <p style='text-align: justify;'>
                Although the omnibus test was statistically significant, the near-zero explained variance (<i>R</i>² = {R_sq:.1f}%) demonstrates that <b>{moderator1}</b>"""
                if moderator2:
                    html += f" and <b>{moderator2}</b>"
                html += """ accounts for a negligible proportion of the overall heterogeneity. The statistical significance of the moderator test is likely a function of high statistical power rather than substantial differentiation among subgroups.
                </p>"""
            else:
                html += f"""
                <p style='text-align: justify;'>
                Overall, the significant omnibus test and substantial explained variance indicate that <b>{moderator1}</b>"""
                if moderator2:
                    html += f" and <b>{moderator2}</b>"
                html += """ represents a meaningful source of heterogeneity, with structurally distinct effect sizes observed across subgroups.
                </p>"""
        else:
            html += f"""
            <p style='text-align: justify;'>
            Because the omnibus test was not statistically significant, the numerical differences observed among the <b>{moderator1}</b>"""
            if moderator2:
                html += f" and <b>{moderator2}</b>"
            html += """ subgroups cannot be reliably distinguished from sampling error.
            </p>"""


        # Summary table
        html += self._generate_results_table(results_df)

        html += """
        </div>
        """

        return html

    def _generate_results_table(self, results_df: pd.DataFrame) -> str:
        """Generate formatted results table for publication text"""

        # Helper to format model type
        def format_model(model):
            m = str(model).lower()
            if 'vcv' in m: return "3L-VCV"
            if 'diag' in m: return "3L-Diag"
            if '2-level' in m: return "2-Level"
            return str(model)

        html = f"""
        <hr style='margin: 30px 0; border: none; border-top: 1px solid #bdc3c7;'>

        <div style='background-color: #ecf0f1; padding: 20px; border-left: 4px solid #3498db;
                    margin-top: 25px;'>
            <h4 style='margin-top: 0; color: #2c3e50;'>📊 Table 1. Summary of Subgroup Analysis Results</h4>
            <table style='width: 100%; border-collapse: collapse; margin-top: 15px; background-color: white;'>
                <thead style='background-color: #34495e; color: white;'>
                    <tr>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: left;'>Subgroup</th>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>k</th>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>Studies</th>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>Effect</th>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>{self.ci_percent:.0f}% CI</th>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>{self.ci_percent:.0f}% PI</th>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'><i>p</i>-value</th>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>Model</th>
                        </tr>
                </thead>
                <tbody>
        """

        any_underpowered_tbl = False
        for idx, row in results_df.iterrows():
            bg = "#f8f9fa" if idx % 2 == 0 else "white"
            model_str = format_model(row['model_type'])
            n_pap = int(row['n_papers'])

            if n_pap < 3:
                # Match the prose: withhold CI/PI/p where between-study
                # variance is not estimable. Dagger marks the row.
                any_underpowered_tbl = True
                html += f"""
                    <tr style='background-color: {bg};'>
                        <td style='border: 1px solid #bdc3c7; padding: 8px;'>{row['group']} &dagger;</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{int(row['k'])}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{n_pap}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{row['pooled_effect_re']:.3f}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center; color:#999;'>&mdash;</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center; color:#999;'>&mdash;</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center; color:#999;'>&mdash;</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center; font-size: 0.9em; color: #555;'>{model_str}</td>
                        </tr>
                """
            else:
                bold = "font-weight: bold;" if row['p_value_re'] < _alpha() else ""
                html += f"""
                    <tr style='background-color: {bg};'>
                        <td style='border: 1px solid #bdc3c7; padding: 8px;'>{row['group']}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{int(row['k'])}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{n_pap}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center; {bold}'>{row['pooled_effect_re']:.3f}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>[{row['ci_lower_re']:.3f}, {row['ci_upper_re']:.3f}]</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>[{row.get('pi_lower_re', 0):.3f}, {row.get('pi_upper_re', 0):.3f}]</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{row['p_value_re']:.3g}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center; font-size: 0.9em; color: #555;'>{model_str}</td>
                        </tr>
                """

        html += """
                </tbody>
            </table>
            <p style='margin-top: 10px; font-size: 0.9em; color: #6c757d;'>
                <i>Note:</i> k = number of effect sizes; Studies = number of independent studies;
                CI = confidence interval; PI = prediction interval. Models: 3L-VCV (3-Level with shared controls), 3L-Diag (3-Level independent), 2-Level (standard).
                &dagger; Interval and <i>p</i>-value estimates withheld for subgroups with fewer than 3 studies, where between-study variance is not reliably estimable.
            </p>
        </div>
        """

        return html


def _level_phrase(d: Dict[str, Any]) -> str:
    """'the between-study variance was reduced by X% and the within-study
    variance increased by Y%' -- direction-aware, for publication text."""
    def one(name, raw):
        try:
            raw = float(raw)
        except (TypeError, ValueError):
            return None
        if not np.isfinite(raw):
            return None
        if raw < 0:
            return f"the {name} variance increased by {-100 * raw:.1f}%"
        return f"the {name} variance was reduced by {100 * raw:.1f}%"
    parts = [p for p in (one('between-study', d.get('R2_tau2_raw')),
                         one('within-study', d.get('R2_sigma2_raw'))) if p]
    return " and ".join(parts)


def _subgroup_r2_line(heterogeneity: Dict[str, Any]) -> str:
    """One-line pseudo-R² summary for the subgroup results card."""
    r2 = heterogeneity.get('R_squared', float('nan'))
    if r2 is not None and np.isfinite(r2):
        line = f"<b>Variance Explained (pseudo-R²):</b> {r2:.1f}%"
        d = heterogeneity.get('R_squared_detail') or {}
        if d.get('three_level'):
            line += (f" (τ²: {level_change_text(d.get('R2_tau2'), d.get('R2_tau2_raw'))}, "
                     f"σ²: {level_change_text(d.get('R2_sigma2'), d.get('R2_sigma2_raw'))})")
        return line
    q = heterogeneity.get('Q_ratio_explained')
    if q is not None:
        return (f"<b>Variance Explained (pseudo-R²):</b> n/a — fallback fit; "
                f"fixed-effect Q ratio Q<sub>M</sub>/Q<sub>T</sub> = {q:.1f}% "
                f"(not a pseudo-R²)")
    return "<b>Variance Explained (pseudo-R²):</b> n/a (not computable)"


class SubgroupResultsView:
    """
    Manages all UI rendering for subgroup analysis.
    Contains zero business logic - only presentation code.
    """

    def __init__(self, ci_percent: float = 95):
        """
        Initialize view with display settings.

        Args:
            ci_percent: Confidence interval percentage
        """
        self.ci_percent = ci_percent
        self.templates = HTMLTemplates()
        self.text_gen = PublicationTextGenerator(ci_percent)

        # Create tab widgets
        self.stale_banner = widgets.HTML(value="")
        register_banner('subgroup', self.stale_banner)
        self.tab_results = widgets.Output()
        self.tab_hetero = widgets.Output()
        self.tab_details = widgets.Output()
        self.tab_config = widgets.Output()
        self.tab_publication = widgets.Output()
        self.tab_export = widgets.Output()

        self.tabs = widgets.Tab(children=[
            self.tab_results,
            self.tab_hetero,
            self.tab_details,
            self.tab_config,
            self.tab_publication,
            self.tab_export
        ])

        self.tabs.set_title(0, '📊 Results Summary')
        self.tabs.set_title(1, '📉 Heterogeneity')
        self.tabs.set_title(2, '⏱️ Processing Log')
        self.tabs.set_title(3, '⚙️ Configuration')
        self.tabs.set_title(4, '📝 Publication Text')
        self.tabs.set_title(5, '💾 Export')

    # -------------------------------------------------------------------------
    # TAB 1: RESULTS SUMMARY
    # -------------------------------------------------------------------------

    def render_results_tab(
        self,
        results_df: pd.DataFrame,
        heterogeneity: Dict[str, Any],
        moderator1: str,
        moderator2: Optional[str]
    ) -> None:
        """Render results summary tab"""

        with self.tab_results:
            self.tab_results.clear_output()

            display(HTML(self.templates.header("📊 Subgroup Analysis Results")))

            # Summary card
            summary = f"<b>Moderator:</b> {moderator1}"
            if moderator2:
                summary += f" × {moderator2}"
            summary += f"<br><b>Subgroups Analyzed:</b> {len(results_df)}"
            summary += f"<br><b>Test for Subgroup Differences:</b> Q<sub>M</sub> ({'model-based Wald' if heterogeneity.get('qm_method', 'wald') == 'wald' else 'Q partition'}) = {heterogeneity['QM']:.2f} "
            summary += f"(df={heterogeneity['df_QM']}, p = {heterogeneity['p_value_QM']:.4g})"
            summary += "<br>" + _subgroup_r2_line(heterogeneity)
            if heterogeneity.get('data_accounting') and 'accounting_html' in globals():
                summary += accounting_html(heterogeneity['data_accounting'])

            # Add global variance components
            tau_sq = heterogeneity.get('tau_squared', 0)
            sigma_sq = heterogeneity.get('sigma_squared', 0)
            summary += f"<br><b>Residual Heterogeneity:</b> τ² = {tau_sq:.4f}, σ² = {sigma_sq:.4f}"

            display(HTML(self.templates.card(summary, bg_color='#e7f3ff')))

            # Results table
            self._render_results_table(results_df)

            # Legends
            display(HTML(self.templates.significance_legend()))
            display(HTML(self.templates.model_type_legend()))

    def _render_results_table(self, results_df: pd.DataFrame) -> None:
        """Render main results table - CORRECTED VERSION"""

        # Format significance stars
        def add_stars(p_val):
            if p_val < 0.001: return "***"
            elif p_val < 0.01: return "**"
            elif p_val < 0.05: return "*"
            return ""

        # Format model type cleanly
        def format_model(model):
            m = str(model).lower()
            if 'vcv' in m: return "3L-VCV"
            if 'diag' in m: return "3L-Diag"
            if '2-level' in m: return "2L"
            return str(model)

        html = f"""
        <table style='width: 100%; border-collapse: collapse; margin: 20px 0; font-size: 13px;'>
        <thead style='background-color: #f8f9fa;'><tr>
            <th style='border: 1px solid #dee2e6; padding: 8px; text-align: left;'>Subgroup</th>
            <th style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>k</th>
            <th style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>Studies</th>
            <th style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>Effect</th>
            <th style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>{self.ci_percent:.0f}% CI</th>
            <th style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>{self.ci_percent:.0f}% PI</th>
            <th style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>p-value</th>
            <th style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>Model</th>
        </tr></thead><tbody>
        """

        for idx, row in results_df.iterrows():
            stars = add_stars(row['p_value_re'])
            style = "font-weight: bold; color: #28a745;" if stars else ""
            model = format_model(row['model_type'])

            html += f"""
            <tr>
                <td style='border: 1px solid #dee2e6; padding: 8px;'>{row['group']}</td>
                <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>{int(row['k'])}</td>
                <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>{int(row['n_papers'])}</td>
                <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center; {style}'>{row['pooled_effect_re']:.3f} {stars}</td>
                <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>[{row['ci_lower_re']:.3f}, {row['ci_upper_re']:.3f}]</td>
                <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center; color: #17a2b8;'>[{row.get('pi_lower_re', 0):.3f}, {row.get('pi_upper_re', 0):.3f}]</td>
                <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>{row['p_value_re']:.4g}</td>
                <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center; font-size: 11px; color: #6c757d;'>{model}</td>
            </tr>
            """

        html += "</tbody></table>"
        display(HTML(html))


  # -------------------------------------------------------------------------
    # TAB 2: HETEROGENEITY (CONTINUATION)
    # -------------------------------------------------------------------------

    def render_heterogeneity_tab(
        self,
        heterogeneity: Dict[str, Any],
        results_df: pd.DataFrame,
        moderator1: str,
        moderator2: Optional[str]
    ) -> None:
        """Render heterogeneity partitioning tab"""

        with self.tab_hetero:
            self.tab_hetero.clear_output()

            display(HTML(self.templates.header("📉 Heterogeneity Partitioning")))

            # Explanation
            explanation = """
            <p><b>Understanding Heterogeneity Decomposition:</b></p>
            <ul>
                <li><b>Q<sub>T</sub> (Total):</b> Overall heterogeneity across all studies</li>
                <li><b>Q<sub>M</sub> (Between-Groups):</b> Heterogeneity explained by the moderator</li>
                <li><b>Q<sub>E</sub> (Within-Groups):</b> Residual heterogeneity within subgroups</li>
                <li><b>R²:</b> Proportion of total heterogeneity explained by the moderator</li>
            </ul>
            """
            display(HTML(self.templates.card(explanation)))

            # Q-statistics table
            self._render_q_statistics_table(heterogeneity)

            # R² interpretation
            self._render_r_squared_interpretation(
                heterogeneity,
                moderator1,
                moderator2
            )

    def _render_q_statistics_table(self, het: Dict[str, Any]) -> None:
        """Render Q-statistics breakdown table"""

        Qt = het['QM'] + het['QE']
        df_total = het['df_QM'] + het['df_QE']

        sig_qm = ""
        if het['p_value_QM'] < 0.001:
            sig_qm = "***"
        elif het['p_value_QM'] < 0.01:
            sig_qm = "**"
        elif het['p_value_QM'] < _alpha():
            sig_qm = "*"
        else:
            sig_qm = "ns"

        style_qm = "font-weight: bold; color: #28a745;" if sig_qm != "ns" else ""

        html = f"""
        <table style='width: 100%; border-collapse: collapse; margin: 20px 0;'>
        <thead style='background-color: #f8f9fa;'><tr>
            <th style='border: 1px solid #dee2e6; padding: 8px; text-align: left;'>Component</th>
            <th style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>Q</th>
            <th style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>df</th>
            <th style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>p-value</th>
        </tr></thead><tbody>
        <tr>
            <td style='border: 1px solid #dee2e6; padding: 8px;'><b>Total (Q<sub>T</sub>)</b></td>
            <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>{Qt:.2f}</td>
            <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>{df_total}</td>
            <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>—</td>
        </tr>
        <tr style='background-color: #e7f3ff;'>
            <td style='border: 1px solid #dee2e6; padding: 8px;'><b>Between-Groups (Q<sub>M</sub>)</b></td>
            <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center; {style_qm}'>{het['QM']:.2f}</td>
            <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>{het['df_QM']}</td>
            <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center; {style_qm}'>{het['p_value_QM']:.4g} {sig_qm}</td>
        </tr>
        <tr>
            <td style='border: 1px solid #dee2e6; padding: 8px;'><b>Within-Groups (Q<sub>E</sub>)</b></td>
            <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>{het['QE']:.2f}</td>
            <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>{het['df_QE']}</td>
            <td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>—</td>
        </tr>
        </tbody></table>
        """

        display(HTML(html))

    def _render_r_squared_interpretation(
        self,
        heterogeneity: Dict[str, Any],
        moderator1: str,
        moderator2: Optional[str]
    ) -> None:
        """Render the pseudo-R² (total and per level) with its interpretation."""
        mod_text = f"<b>{moderator1}</b>"
        if moderator2:
            mod_text += f" × <b>{moderator2}</b>"

        r_squared = heterogeneity.get('R_squared', float('nan'))
        detail = heterogeneity.get('R_squared_detail')
        if not np.isfinite(r_squared):
            content = r2_breakdown_html(detail) if detail else (
                "<p style='margin: 0;'><b>Pseudo-R²:</b> not computable.</p>")
            display(HTML(self.templates.card(content, bg_color='#fff3cd')))
            return

        if r_squared < 25:
            color, label = "#856404", "(Low explanatory power)"
        elif r_squared < 50:
            color, label = "#856404", "(Moderate explanatory power)"
        else:
            color, label = "#155724", "(High explanatory power)"

        content = f"""
        <p style='margin: 0; font-size: 1.1em;'><b>Variance Explained (pseudo-R²): {r_squared:.1f}%</b></p>
        <p style='margin: 5px 0 8px 0;'>
            Relative to the intercept-only model, the moderator {mod_text} explains
            {r_squared:.1f}% of the total heterogeneity.
            <span style='color: {color};'>{label}</span>
        </p>
        {r2_breakdown_html(detail) if detail else ''}
        """
        display(HTML(self.templates.card(content, bg_color='#fff3cd')))

    # -------------------------------------------------------------------------
    # TAB 3: DETAILS (Progress Stream)
    # -------------------------------------------------------------------------

    def render_details_tab(self) -> widgets.Output:
        """
        Render execution log tab and return the output widget for streaming.
        """
        with self.tab_details:
            self.tab_details.clear_output()
            display(HTML(self.templates.header("⏱️ Analysis Execution Log")))

            # Add context so users know what this tab is for
            display(HTML("<p style='color: #6c757d; font-size: 13px; margin-bottom: 10px;'><i>Real-time diagnostic output and fallback triggers from the statistical optimization engine.</i></p>"))

            # Create a styled output area for the log stream
            log_stream_output = widgets.Output(layout=widgets.Layout(
                border='1px solid #ced4da',
                padding='15px',
                background_color='#f8f9fa',
                border_radius='6px',
                max_height='450px',
                overflow='auto'
            ))
            display(log_stream_output)

        return log_stream_output

    # -------------------------------------------------------------------------
    # TAB 4: CONFIGURATION
    # -------------------------------------------------------------------------

    def render_config_tab(self, config_summary: Dict[str, Any]) -> None:
        """Render configuration tab"""

        with self.tab_config:
            self.tab_config.clear_output()

            display(HTML(self.templates.header("⚙️ Analysis Configuration")))

            # Configuration details
            content = f"<b>Timestamp:</b> {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}<br>"
            content += f"<b>Analysis Type:</b> {config_summary['analysis_type']}<br>"
            content += f"<b>Moderator 1:</b> {config_summary['moderator1']}<br>"

            if config_summary['moderator2']:
                content += f"<b>Moderator 2:</b> {config_summary['moderator2']}<br>"

            content += f"<b>Number of Subgroups:</b> {config_summary['n_groups']}<br>"
            content += f"<b>Effect Column:</b> {config_summary['effect_col']}<br>"
            content += f"<b>Variance Column:</b> {config_summary['var_col']}<br>"
            content += f"<b>Alpha Level:</b> {config_summary['alpha']}<br>"
            content += f"<b>Confidence Interval:</b> {config_summary['ci_percent']:.0f}%<br>"
            content += f"<b>Distribution:</b> {config_summary['dist_type']}<br>"
            content += f"<b>Total Observations:</b> {config_summary['total_observations']}<br>"
            content += f"<b>Total Studies:</b> {config_summary['n_studies']}"

            display(HTML(self.templates.card(content)))
            display(HTML(self.templates.success_message("All prerequisites met")))

    # -------------------------------------------------------------------------
    # TAB 5: PUBLICATION TEXT
    # -------------------------------------------------------------------------

    def render_publication_tab(
        self,
        results_df: pd.DataFrame,
        heterogeneity: Dict[str, Any],
        moderator1: str,
        moderator2: Optional[str],
        n_groups: int
    ) -> str:
        """
        Render publication text tab.

        Returns:
            Combined HTML text for saving
        """
        with self.tab_publication:
            self.tab_publication.clear_output()

            display(HTML(self.templates.header("📝 Publication-Ready Results Text", level=3)))

            # Check if the fallback was triggered
            fallback_used = heterogeneity.get('fallback_used', False)

            if fallback_used:
                display(HTML("""
                <div style='background-color: #fff3cd; border-left: 4px solid #ffc107; padding: 10px; margin-bottom: 15px;'>
                    <span style='color: #856404; font-size: 13px;'><b>⚠️ Note:</b> The text below has been automatically adjusted to reflect that independent models were used due to convergence constraints.</span>
                </div>
                """))

            # Generate both sections
            methods_html = self.text_gen.generate_methods_section(
                moderator1, moderator2, n_groups, fallback_used=fallback_used
            )
            results_html = self.text_gen.generate_results_section(
                results_df, moderator1, moderator2, heterogeneity
            )

            # Display
            display(HTML(methods_html))
            display(HTML(results_html))

            # Return combined for saving
            return methods_html + "<br><hr><br>" + results_html



    # -------------------------------------------------------------------------
    # TAB 6: EXPORT
    # -------------------------------------------------------------------------

    def render_export_tab(self, export_callback: callable, export_json_callback: callable) -> None:
        """Render export tab with download buttons"""

        with self.tab_export:
            self.tab_export.clear_output()
            self.export_status = widgets.Output()  # Dedicated status output

            display(HTML(self.templates.header("💾 Download Reports & Settings", level=3)))

            # --- Excel Export ---
            display(HTML("<p><b>Audit Report:</b> Generate an Excel file containing the subgroup table, heterogeneity breakdown, and specific group data.</p>"))
            btn_export = widgets.Button(
                description="📥 Download Subgroup Report",
                button_style='info',
                icon='file-excel',
                layout=widgets.Layout(width='320px', height='40px')
            )
            btn_export.on_click(export_callback)

            # --- JSON Export (Reproducibility) ---
            display(HTML("<p style='margin-top:15px;'><b>Reproducibility:</b> Download your exact configuration to reload later in Step 2 (Data Ingestion).</p>"))
            btn_export_json = widgets.Button(
                description="📥 Download Settings JSON",
                button_style='warning',
                icon='code',
                layout=widgets.Layout(width='320px', height='40px')
            )
            btn_export_json.on_click(export_json_callback)

            # Add the status output widget below the buttons
            display(widgets.VBox([
                btn_export,
                widgets.HTML("<br>"),
                btn_export_json,
                widgets.HTML("<br>"),
                self.export_status
            ]))

    # -------------------------------------------------------------------------
    # ERROR DISPLAY
    # -------------------------------------------------------------------------

    def render_error(self, error_type: str, message: str, details: Optional[str] = None) -> None:
        """Render error message in results tab"""
        with self.tab_results:
            self.tab_results.clear_output()
            display(HTML(self.templates.error_message(error_type, message, details)))

    def render_prerequisite_error(self, missing_keys: List[str]) -> None:
        """Render prerequisite check failure"""
        with self.tab_config:
            self.tab_config.clear_output()
            display(HTML(self.templates.header("⚙️ Configuration Check")))

            message = f"Missing required ANALYSIS_CONFIG keys: {', '.join(missing_keys)}"
            display(HTML(self.templates.error_message("Configuration Error", message)))

            display(HTML("<p>Please run:</p><ul><li>Step 7: Overall Meta-Analysis</li><li>Step 8: Subgroup Configuration</li></ul>"))


# =============================================================================
# CELL COMPONENT 4/4: CONTROLLER LAYER
# Purpose: Orchestrates data, analysis, and view components
# Dependencies: All previous layers (Data, Analysis, View)
# =============================================================================

import traceback
import sys
from typing import Optional, Dict, Any
from IPython.display import display


# =============================================================================
# MAIN CONTROLLER
# =============================================================================

class SubgroupController:
    """
    Master controller that orchestrates the entire subgroup analysis workflow.
    Coordinates data management, statistical computation, and UI rendering.
    """

    def __init__(self, analysis_config: Dict[str, Any]):
        """
        Initialize controller with ANALYSIS_CONFIG.

        Args:
            analysis_config: Global ANALYSIS_CONFIG dictionary
        """
        self.analysis_config = analysis_config

        # Initialize components
        try:
            self.data_manager = SubgroupDataManager(analysis_config)
            self.engine = SubgroupAnalysisEngine(self.data_manager)

            # Get CI percentage from settings
            ci_percent = self.data_manager.global_settings.ci_percent
            self.view = SubgroupResultsView(ci_percent=ci_percent)

        except Exception as e:
            # If initialization fails, create minimal view to show error
            self.view = SubgroupResultsView()
            self.data_manager = None
            self.engine = None
            self._initialization_error = e

    # -------------------------------------------------------------------------
    # MAIN EXECUTION METHOD
    # -------------------------------------------------------------------------

    def run_analysis(self) -> None:
        """
        Execute complete subgroup analysis workflow.
        This is the main entry point that orchestrates everything.
        """
        # Display tabs immediately
        display(widgets.VBox([self.view.stale_banner, self.view.tabs]))

        # Check for initialization errors
        if self.data_manager is None or self.engine is None:
            self._handle_initialization_error()
            return

        # Render configuration tab first
        try:
            self._render_configuration()
        except Exception as e:
            self._handle_error("Configuration Error", e)
            return

        # Run main analysis
        try:
            self._execute_analysis()
        except Exception as e:
            self._handle_error("Analysis Error", e)
            return

    # -------------------------------------------------------------------------
    # WORKFLOW STEPS
    # -------------------------------------------------------------------------

    def _render_configuration(self) -> None:
        """Render configuration tab"""
        config_summary = self.data_manager.summary_dict()
        self.view.render_config_tab(config_summary)

    def _execute_analysis(self) -> None:
        """Execute statistical analysis with progress streaming"""

        # Setup progress callback for details tab
        log_output = self.view.render_details_tab()

        def progress_callback(message: str):
            """Callback to stream progress to details tab as formatted HTML with timestamps"""
            # Generate the timestamp
            now = datetime.datetime.now().strftime("%H:%M:%S")
            time_badge = f"<span style='color: #adb5bd; font-family: monospace; font-size: 12px; margin-right: 10px;'>[{now}]</span>"

            with log_output:
                if "=" in message:
                    display(HTML("<hr style='margin: 8px 0; border-top: 1px solid #dee2e6;'>"))
                elif "❌" in message:
                    display(HTML(f"<div style='color: #dc3545; font-family: sans-serif; font-size: 13px; margin: 4px 0;'>{time_badge}<b>{message}</b></div>"))
                elif "✅" in message:
                    display(HTML(f"<div style='color: #28a745; font-family: sans-serif; font-size: 13px; margin: 4px 0;'>{time_badge}<b>{message}</b></div>"))
                elif "⚠️" in message:
                    display(HTML(f"<div style='color: #856404; background-color: #fff3cd; padding: 8px; border-radius: 4px; border-left: 4px solid #ffc107; font-family: sans-serif; font-size: 13px; margin: 6px 0;'>{time_badge}{message}</div>"))
                elif "🚀" in message or "🔄" in message:
                    display(HTML(f"<div style='color: #0056b3; font-family: sans-serif; font-size: 13px; margin: 4px 0;'>{time_badge}<b>{message}</b></div>"))
                elif "Subgroup" in message and "/" in message:
                    display(HTML(f"<div style='color: #495057; font-family: sans-serif; font-size: 14px; font-weight: bold; margin: 8px 0 4px 0;'>{time_badge}{message}</div>"))
                else:
                    display(HTML(f"<div style='color: #495057; font-family: sans-serif; font-size: 13px; margin: 2px 0;'>{time_badge}{message}</div>"))


        # Run analysis engine
        progress_callback("🚀 Starting subgroup analysis...")

        import warnings
        with warnings.catch_warnings(record=True) as w:
            # Trap all warnings
            warnings.simplefilter("always")
            # Silence useless numerical noise and internal developer warnings
            warnings.filterwarnings('ignore', category=RuntimeWarning, module='scipy.optimize')
            warnings.filterwarnings('ignore', category=DeprecationWarning)
            warnings.filterwarnings('ignore', category=FutureWarning)

            results_df, heterogeneity = self.engine.analyze_all_subgroups(
                progress_callback=progress_callback
            )

            # Route meaningful caught warnings to the UI Log
            for warning in w:
                progress_callback(f"⚠️ <b>Note:</b> {str(warning.message)}")

        # Extract metadata
        config = self.data_manager.subgroup_config
        moderator1 = config.moderator1
        moderator2 = config.moderator2
        n_groups = len(results_df)

        # Render all tabs
        self._render_results_tab(results_df, heterogeneity, moderator1, moderator2)
        self._render_heterogeneity_tab(results_df, heterogeneity, moderator1, moderator2)
        self._render_publication_tab(results_df, heterogeneity, moderator1, moderator2, n_groups)
        self._render_export_tab()

        # Save results to ANALYSIS_CONFIG
        self._save_results(results_df, heterogeneity)
        clear_stale('subgroup')
        mark_stale(['plots'], "Subgroup Analysis Re-run")

        # Final progress message
        with log_output:
            display(HTML("<hr style='margin: 15px 0; border-top: 2px solid #28a745;'>"))
            display(HTML("<div style='color: #155724; font-family: sans-serif; font-size: 15px; font-weight: bold; margin-bottom: 5px;'>✅ ANALYSIS COMPLETE</div>"))
            display(HTML("<div style='color: #495057; font-family: sans-serif; font-size: 13px;'>Results saved to memory (<code>ANALYSIS_CONFIG['subgroup_results']</code>).</div>"))
            display(HTML("<div style='color: #0056b3; font-family: sans-serif; font-size: 13px; margin-top: 5px;'><b>▶️ Ready for next step:</b> Proceed to Meta-Regression or Visualizations.</div>"))


    def _render_results_tab(
        self,
        results_df,
        heterogeneity,
        moderator1,
        moderator2
    ) -> None:
        """Render results summary tab"""
        self.view.render_results_tab(
            results_df=results_df,
            heterogeneity=heterogeneity,
            moderator1=moderator1,
            moderator2=moderator2
        )

    def _render_heterogeneity_tab(
        self,
        results_df,
        heterogeneity,
        moderator1,
        moderator2
    ) -> None:
        """Render heterogeneity partitioning tab"""
        self.view.render_heterogeneity_tab(
            heterogeneity=heterogeneity,
            results_df=results_df,
            moderator1=moderator1,
            moderator2=moderator2
        )

    def _render_publication_tab(
        self,
        results_df,
        heterogeneity,
        moderator1,
        moderator2,
        n_groups
    ) -> None:
        """Render publication text tab"""
        combined_text = self.view.render_publication_tab(
            results_df=results_df,
            heterogeneity=heterogeneity,
            moderator1=moderator1,
            moderator2=moderator2,
            n_groups=n_groups
        )

        # Save to ANALYSIS_CONFIG
        self.data_manager.save_publication_text(combined_text)

    def _render_export_tab(self) -> None:
        """Render export tab with callback"""
        self.view.render_export_tab(
            export_callback=self._handle_export,
            export_json_callback=self._handle_export_json
        )

    def _handle_export(self, button) -> None:
        """Handle export button click"""
        try:
            # Call the external export function if it exists
            if 'export_analysis_report' in globals():
                export_analysis_report(
                    report_type='subgroup',
                    filename_prefix='Subgroup_Analysis'
                )
            else:
                with self.view.tab_export:
                    print("⚠️ Export function not found. Please run the export cell first.")
        except Exception as e:
            with self.view.tab_export:
                print(f"❌ Export failed: {str(e)}")

    def _handle_export_json(self, button) -> None:
        """Handle JSON export button click for Reproducibility Mode"""
        import json
        import base64
        import datetime
        from IPython.display import display, Javascript

        try:
            with self.view.export_status:
                clear_output()
                display(HTML("<div style='padding: 10px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; margin-top: 15px; border-radius: 4px; font-family: sans-serif;'>⏳ <b>Generating JSON Settings...</b> Please wait.</div>"))

                # Check if the export function from Cell 2 exists
                if 'export_config_for_reproducibility' in globals():
                    config_clean = export_config_for_reproducibility(self.analysis_config)
                    json_str = json.dumps(config_clean, indent=2)

                    # Base64 encode for browser download
                    b64 = base64.b64encode(json_str.encode()).decode()
                    filename = f"analysis_settings_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}.json"

                    payload = f"""
                    var link = document.createElement('a');
                    link.href = 'data:application/json;base64,{b64}';
                    link.download = '{filename}';
                    document.body.appendChild(link);
                    link.click();
                    document.body.removeChild(link);
                    """
                    display(Javascript(payload))

                    clear_output()
                    display(HTML(f"<div style='padding: 10px; background-color: #d4edda; border-left: 4px solid #28a745; color: #155724; margin-top: 15px; border-radius: 4px; font-family: sans-serif;'>✅ <b>Download triggered:</b> {filename}</div>"))
                else:
                    clear_output()
                    display(HTML("<div style='padding: 10px; background-color: #f8d7da; border-left: 4px solid #dc3545; color: #721c24; margin-top: 15px; border-radius: 4px; font-family: sans-serif;'>❌ <b>Error:</b> <code>export_config_for_reproducibility</code> function not found.</div>"))

        except Exception as e:
            with self.view.export_status:
                clear_output()
                display(HTML(f"<div style='padding: 10px; background-color: #f8d7da; border-left: 4px solid #dc3545; color: #721c24; margin-top: 15px; border-radius: 4px; font-family: sans-serif;'>❌ <b>Export failed:</b> {str(e)}</div>"))
                import traceback
                traceback.print_exc()

    def _save_results(self, results_df, heterogeneity) -> None:
        """Save results to ANALYSIS_CONFIG"""
        metadata = {
            'Qt_overall': heterogeneity.get('QT', self.data_manager.overall_results['Qt']),
            'QM': heterogeneity['QM'],
            'Qe': heterogeneity['QE'],
            'df_QM': heterogeneity['df_QM'],
            'df_Qe': heterogeneity['df_QE'],
            'p_value_QM': heterogeneity['p_value_QM'],
            'R_squared': heterogeneity['R_squared'],
            'R_squared_tau': heterogeneity.get('R_squared_tau'),
            'R_squared_sigma': heterogeneity.get('R_squared_sigma'),
            'R_squared_detail': heterogeneity.get('R_squared_detail'),
            'Q_ratio_explained': heterogeneity.get('Q_ratio_explained'),
            'data_accounting': heterogeneity.get('data_accounting'),
            # Cluster-robust small-sample omnibus (AHZ / clubSandwich HTZ).
            # Persisted so Validation 3 can check the number Step 9 actually
            # published, rather than a re-derivation of it.
            'F_omnibus': heterogeneity.get('F_omnibus'),
            'df_F_num': heterogeneity.get('df_F_num'),
            'df_F_den': heterogeneity.get('df_F_den'),
            'p_value_F_omnibus': heterogeneity.get('p_value_F_omnibus'),
            # Two-way only; None otherwise. Plain floats/strings, so this
            # round-trips into ANALYSIS_CONFIG safely (the AHZ closure the
            # tests were computed with is deliberately not part of it).
            'factorial_tests': heterogeneity.get('factorial_tests'),
            # The fitted model and the level contrasts, for cell 9b. Kept in
            # memory only (session files never store subgroup_results).
            'model_fit': heterogeneity.get('model_fit'),
        }

        self.data_manager.save_subgroup_results(results_df, metadata)

    # -------------------------------------------------------------------------
    # EVENT HANDLERS
    # -------------------------------------------------------------------------

    def _handle_export(self, button) -> None:
        """Handle export button click"""
        try:
            # Call the external export function if it exists
            if 'export_analysis_report' in globals():
                export_analysis_report(
                    report_type='subgroup',
                    filename_prefix='Subgroup_Analysis'
                )
            else:
                with self.view.tab_export:
                    print("⚠️ Export function not found. Please run the export cell first.")
        except Exception as e:
            with self.view.tab_export:
                print(f"❌ Export failed: {str(e)}")

    # -------------------------------------------------------------------------
    # ERROR HANDLING
    # -------------------------------------------------------------------------

    def _handle_initialization_error(self) -> None:
        """Handle errors during controller initialization"""
        error = getattr(self, '_initialization_error', None)

        if error is None:
            return

        error_type = type(error).__name__
        error_msg = str(error)

        if isinstance(error, ValueError):
            # Missing prerequisites
            if 'Missing required ANALYSIS_CONFIG keys' in error_msg:
                missing = error_msg.split(': ')[1].split('.')[0]
                self.view.render_prerequisite_error(missing.split(', '))
            else:
                self.view.render_error(error_type, error_msg)
        else:
            # Unexpected error
            details = traceback.format_exc()
            self.view.render_error(error_type, error_msg, details)

    def _handle_error(self, error_type: str, error: Exception) -> None:
        """Handle errors during analysis execution"""
        error_msg = str(error)
        details = traceback.format_exc()

        self.view.render_error(error_type, error_msg, details)


# =============================================================================
# EXECUTION ENTRY POINT
# =============================================================================

def run_subgroup_analysis():
    """
    Main entry point for subgroup analysis.
    Call this function to execute the complete workflow.
    """
    try:
        # Check for ANALYSIS_CONFIG
        if 'ANALYSIS_CONFIG' not in globals():
            display(HTML("""
            <div style='padding: 15px; background-color: #f8d7da; border-left: 5px solid #dc3545; color: #721c24; border-radius: 4px; font-family: sans-serif; margin-bottom: 15px;'>
                <h4 style='margin: 0 0 5px 0; display: flex; align-items: center;'><span style='margin-right: 8px;'>❌</span> Missing Configuration</h4>
                <p style='margin: 0; font-size: 14px;'><code>ANALYSIS_CONFIG</code> not found. Please run previous analysis cells first:</p>
                <ul style='margin-top: 5px; margin-bottom: 0; font-size: 13px;'>
                    <li>Step 7: Overall Meta-Analysis</li>
                    <li>Step 8: Subgroup Configuration</li>
                </ul>
            </div>
            """))
            return

        # Create controller and run
        controller = SubgroupController(ANALYSIS_CONFIG)
        controller.run_analysis()

    except Exception as e:
        display(HTML(f"""
        <div style='padding: 15px; background-color: #f8d7da; border-left: 5px solid #dc3545; color: #721c24; border-radius: 4px; font-family: sans-serif; margin-bottom: 15px;'>
            <h4 style='margin: 0 0 5px 0; display: flex; align-items: center;'><span style='margin-right: 8px;'>❌</span> Fatal Error: {type(e).__name__}</h4>
            <p style='margin: 0 0 10px 0; font-size: 14px;'>{str(e)}</p>
            <p style='margin: 0; font-size: 12px;'>See the system output below for the full traceback.</p>
        </div>
        """))
        traceback.print_exc()

run_subgroup_analysis()

In [ ]:
#@title 📏 9b. Proportion of Meaningful Effects
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# STEP 9b: PROPORTION OF MEANINGFUL EFFECTS
# Purpose: How much of the distribution of TRUE effects is large enough to
#          matter? For a threshold chosen by the user, the share of true
#          effects that are meaningful increases, meaningful decreases or
#          negligible (Mathur & VanderWeele 2019, 2020), for the overall model
#          (Step 7) and for each subgroup (Step 9, if run).
# Methods: parametric (normal true effects) and calibrated (distribution-free)
#          estimates; delta-method and cluster-bootstrap BCa CIs. Engine:
#          meaningful_proportions (Cell 1b, section 9).
# =============================================================================

import html
import base64
import datetime
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Javascript
from scipy.stats import norm as _norm_9b


def _meaningful_effects_cell():
    PLOT_TYPE = 'meaningful_effects'
    EXPLORATORY_BELOW = 500      # fewer resamples: quick run, no jackknife, marked exploratory
    state = {'overall': None, 'subgroups': None, 'info': {}, 'results': None, 'settings': None,
             'tokens': {}, 'data': None, 'source': None, 'recipe_name': None}

    # --- 1. MODELS: the fits reported in Steps 7 and 9, with how to refit them ---
    def overall_model():
        """The reported overall model (Step 7), rebuilt on the same rows and V."""
        cfg = ANALYSIS_CONFIG
        rep = (cfg.get('overall_results') or {}).get('reported')
        df0 = cfg.get('analysis_data')
        if not rep or df0 is None:
            return None, "Run Step 7 (Overall Meta-Analysis) first."
        eff, var = cfg.get('effect_col'), cfg.get('var_col')
        df, _acc = select_rows(df0, eff, var, 'id')
        df = attach_vcv_row_pos(df, reference_df=df0)
        df = df.sort_values('id', kind='mergesort').reset_index(drop=True)
        groups = list(df.groupby('id', sort=False))
        y_b = [g[eff].to_numpy(dtype=np.float64) for _, g in groups]
        X_b = [np.ones((len(y), 1)) for y in y_b]
        gs = cfg.get('global_settings') or {}
        se_mu = float(rep.get('se', np.nan))
        if rep.get('model') == '3-Level':
            V_b = [cluster_vcv(cfg.get('vcv_matrices'), sid, g, var)[0] for sid, g in groups]
            fit = fit_blocks(y_b, V_b, two_level=False, cluster_ids=[sid for sid, _ in groups])

            def refit(yy, VV, XX):
                # resamples start from the full-data estimates (much faster; same optimum)
                try:
                    f = fit_blocks(yy, VV, XX, two_level=False, starts=[(fit.tau2, fit.sigma2)])
                except (ValueError, RuntimeError, np.linalg.LinAlgError):
                    return None
                return f.betas, f.tau2, f.sigma2

            model = MeaningfulModel(y_b, V_b, X_b, fit.betas, fit.tau2, fit.sigma2,
                                    {'Overall': np.array([1.0])}, {'Overall': se_mu},
                                    var_components_vcov(fit), refit, 'three-level (Step 7)')
            label = "three-level model (Step 7)"
        else:
            # Two-level model of Step 7: effect sizes as independent units,
            # tau² by the estimator chosen in Step 7.
            V_b = [np.diag(g[var].to_numpy(dtype=np.float64)) for _, g in groups]
            method = str(gs.get('tau_method', 'REML')).upper()

            def tau2_of(y, v):
                if 'TwoLevelEngine' in globals() and method not in ('REML', 'ML'):
                    d = pd.DataFrame({'yi': y, 'vi': v})
                    return float(TwoLevelEngine(tau_method=method).estimate_tau2(d, 'yi', 'vi'))
                return tau2_univariate(y, v, method if method in ('REML', 'ML') else 'REML')

            def pooled(yy, VV):
                y = np.concatenate(yy)
                v = np.concatenate([np.diag(V) for V in VV])
                t2 = max(0.0, tau2_of(y, v))
                w = 1.0 / (v + t2)
                return np.array([float(np.sum(w * y) / np.sum(w))]), t2

            def refit(yy, VV, XX):
                try:
                    b, t2 = pooled(yy, VV)
                except (ValueError, RuntimeError, np.linalg.LinAlgError, ZeroDivisionError):
                    return None
                return b, t2, 0.0

            betas, t2 = pooled(y_b, V_b)
            vc = None
            if method in ('REML', 'ML'):
                rows_y = [np.array([y]) for y in np.concatenate(y_b)]
                rows_V = [np.array([[v]]) for V in V_b for v in np.diag(V)]
                vc = var_components_vcov(fit_blocks(rows_y, rows_V, two_level=True, method=method))
            model = MeaningfulModel(y_b, V_b, X_b, betas, t2, 0.0, {'Overall': np.array([1.0])},
                                    {'Overall': se_mu}, vc, refit, f'two-level, τ² by {method} (Step 7)')
            label = f"two-level model, τ² by {method} (Step 7)"
        mu_rep = float(rep.get('mu', np.nan))
        note = ''
        if np.isfinite(mu_rep) and abs(float(model.betas[0]) - mu_rep) > 1e-6 * max(1.0, abs(mu_rep)):
            note = (f"The refitted mean ({float(model.betas[0]):.6g}) differs from the one Step 7 reported "
                    f"({mu_rep:.6g}); re-run Step 7 and this cell.")
        return model, {'label': label, 'k': sum(len(y) for y in y_b), 'm': len(y_b), 'note': note,
                       'model': rep.get('model')}

    def subgroup_model():
        sg = ANALYSIS_CONFIG.get('subgroup_results')
        if not sg:
            return None, None
        mf = sg.get('model_fit') or {}
        fit = mf.get('fit')
        if fit is None:
            return None, ("The subgroup results come from separate per-group fits (fallback), so "
                          "subgroup shares are not computed here.")
        res = sg['results_df']
        levels, L = list(mf['levels']), np.asarray(mf['L'])
        se = {str(r['group']): float(r['pooled_se_re']) for _, r in res.iterrows()}
        two_level = fit.sigma2_fixed is not None and fit.sigma2_fixed == 0

        def refit(yy, VV, XX):
            try:
                f = fit_blocks(yy, VV, XX, two_level=two_level, starts=[(fit.tau2, fit.sigma2)])
            except (ValueError, RuntimeError, np.linalg.LinAlgError):
                return None
            return f.betas, f.tau2, f.sigma2

        s = fit.spec
        model = MeaningfulModel(list(s.y_blocks), list(s.V_blocks), list(s.X_blocks), fit.betas,
                                fit.tau2, fit.sigma2, {str(lv): L[i] for i, lv in enumerate(levels)},
                                {str(lv): se.get(str(lv), np.nan) for lv in levels},
                                var_components_vcov(fit), refit, 'subgroup model (Step 9)')
        mods = sg.get('moderator1') + (f" × {sg.get('moderator2')}" if sg.get('moderator2') else '')
        return model, {'label': f"subgroups by {mods} (Step 9)", 'moderators': mods,
                       'n_levels': len(levels)}

    # --- 2. THRESHOLDS on the scale the user thinks in ---
    cfg0 = globals().get('ANALYSIS_CONFIG') or {}
    es = cfg0.get('es_config') or {}
    es_type = str(cfg0.get('effect_size_type') or '')
    effect_label = es.get('effect_label', 'Effect Size')
    ratio_scale = bool(es.get('has_fold_change'))
    null = es_null_value()
    if ratio_scale:
        mode_opts = [('Percent change (%)', 'percent'), (f'Analysis scale ({effect_label})', 'analysis')]
        default_q = 10.0
    elif es_type in ('hedges_g', 'cohen_d'):
        mode_opts = [('Standardized units (SD)', 'analysis')]
        default_q = 0.2
    else:
        mode_opts = [(f'Analysis scale ({effect_label})', 'analysis')]
        default_q = 0.1

    def to_analysis_scale(mode, inc, dec):
        if mode == 'percent':
            if not 0 < dec < 100:
                raise ValueError("A percent decrease must be between 0 and 100.")
            return null + float(np.log1p(-dec / 100.0)), null + float(np.log1p(inc / 100.0))
        return null - float(dec), null + float(inc)

    def short_thresholds(mode, inc, dec, q_lo, q_hi):
        if mode == 'percent':
            return f"≥ {inc:g}% increase", f"≥ {dec:g}% decrease"
        return f"≥ {q_hi:.3g}", f"≤ {q_lo:.3g}"

    def describe_thresholds(mode, inc, dec, q_lo, q_hi):
        if mode == 'percent':
            return (f"an increase of at least {inc:g}% (ratio ≥ {1 + inc / 100:.3g}, {effect_label} ≥ {q_hi:.3g})",
                    f"a decrease of at least {dec:g}% (ratio ≤ {1 - dec / 100:.3g}, {effect_label} ≤ {q_lo:.3g})")
        return (f"{effect_label} ≥ {q_hi:.3g}", f"{effect_label} ≤ {q_lo:.3g}")

    # --- 3. ANALYSIS CONTROLS ---
    L_ = lambda w='420px': widgets.Layout(width=w)
    S_ = {'description_width': '190px'}
    A = {
        'mode': widgets.Dropdown(options=mode_opts, value=mode_opts[0][1], description='Threshold units:',
                                 style=S_, layout=L_()),
        'q_increase': widgets.BoundedFloatText(value=default_q, min=0.0, max=1e9, step=0.01,
                                               description='Meaningful increase ≥', style=S_, layout=L_()),
        'q_decrease': widgets.BoundedFloatText(value=default_q, min=0.0, max=1e9, step=0.01,
                                               description='Meaningful decrease ≥', style=S_, layout=L_()),
        'same': widgets.Checkbox(value=True, description='Same threshold in both directions', indent=False),
        'level': widgets.Dropdown(options=[('Individual effect sizes (τ² + σ²)', 'effect'),
                                           ('Study means (τ² only)', 'study')],
                                  value='effect', description='True effects of:', style=S_, layout=L_()),
        'bootstrap': widgets.Checkbox(value=True, indent=False,
                                      description='Bootstrap CIs (resampling studies; recommended)'),
        'n_boot': widgets.Dropdown(options=[('50 — quick test (exploratory)', 50),
                                            ('200 — exploratory', 200),
                                            ('500', 500),
                                            ('1000 — for publication (recommended)', 1000),
                                            ('2000', 2000)],
                                   value=1000, description='Bootstrap resamples:',
                                   style=S_, layout=L_()),
        'seed': widgets.IntText(value=2024, description='Random seed:', style=S_, layout=L_()),
    }

    def _sync(change=None):
        if A['same'].value:
            A['q_decrease'].value = A['q_increase'].value
        A['q_decrease'].disabled = A['same'].value
    A['q_increase'].observe(_sync, names='value')
    A['same'].observe(_sync, names='value')

    saved = cfg0.get('meaningful_config')
    if isinstance(saved, dict):
        apply_widget_values(A, saved)
    _sync()

    # --- 4. COMPUTE ---
    out_results = widgets.Output()
    out_text = widgets.Output()
    out_export = widgets.Output()
    progress = widgets.IntProgress(value=0, min=0, max=1, description='Bootstrap:',
                                   layout=widgets.Layout(width='420px', visibility='hidden'))
    progress_note = widgets.HTML()
    source_box = widgets.HTML()

    def fmt_p(p):
        return '—' if not np.isfinite(p) else f"{100 * p:.0f}%"

    def fmt_ci(lo, hi):
        return '' if not (np.isfinite(lo) and np.isfinite(hi)) else f" [{100 * lo:.0f}, {100 * hi:.0f}]"

    def best_ci(r, key):
        """Bootstrap CI when computed, else the delta-method CI (parametric)."""
        lo, hi = r.get(f'{key}_lo_boot', np.nan), r.get(f'{key}_hi_boot', np.nan)
        if np.isfinite(lo) and np.isfinite(hi):
            return lo, hi, 'bootstrap'
        lo, hi = r.get(f'{key}_lo_delta', np.nan), r.get(f'{key}_hi_delta', np.nan)
        return lo, hi, ('delta' if np.isfinite(lo) else '')

    def compute(_b=None):
        with out_results:
            clear_output(wait=True)
            s = widget_values(A)
            try:
                q_lo, q_hi = to_analysis_scale(s['mode'], s['q_increase'], s['q_decrease'])
                if not q_lo < null < q_hi:
                    raise ValueError("Both thresholds must be larger than 0.")
            except ValueError as e:
                display(HTML(f"<div style='color:#721c24;'>❌ {html.escape(str(e))}</div>"))
                return
            overall, oinfo = overall_model()
            if overall is None:
                display(HTML(f"<div style='color:#721c24;'>❌ {html.escape(oinfo)}</div>"))
                return
            level = s['level']
            if oinfo.get('model') != '3-Level' and level == 'study':
                level = 'effect'
            sub, sinfo = subgroup_model()
            alpha = _alpha()
            n_boot = int(s['n_boot']) if s['bootstrap'] else 0
            # quick runs skip the jackknife (one refit per study) used for the BCa acceleration
            accel = n_boot >= EXPLORATORY_BELOW
            models = [m for m in (overall, sub) if m is not None]
            total = sum(n_boot + (len(m.y_blocks) if accel else 0) for m in models) if n_boot else 0
            done = [0]
            t_start = time.time()

            def tick(i, tot, offset):
                progress.value = offset + i
                n_done = offset + i
                if n_done % 10 == 0 or n_done == total:
                    left = (time.time() - t_start) / n_done * (total - n_done)
                    when = (f"about {left / 60:.0f} min left" if left >= 90 else f"about {left:.0f} s left")
                    progress_note.value = (f"<span style='color:#555; font-size:12px;'>{n_done} / {total} refits · "
                                           f"{when}</span>")

            progress.max = max(total, 1)
            progress.value = 0
            progress.layout.visibility = 'visible' if n_boot else 'hidden'
            progress_note.value = ''
            display(HTML("<div style='color:#555;'>⏳ Computing"
                         + (f" ({n_boot} bootstrap resamples per model)…" if n_boot else "…") + "</div>"))
            frames = []
            try:
                for m in models:
                    off = done[0]
                    frames.append(meaningful_proportions(
                        m, (q_lo, q_hi), level=level, alpha=alpha, n_boot=n_boot, seed=int(s['seed']),
                        progress=(lambda i, tot, off=off: tick(i, tot, off)) if n_boot else None,
                        acceleration=accel))
                    done[0] += (n_boot + (len(m.y_blocks) if accel else 0)) if n_boot else 0
            except Exception as e:
                clear_output(wait=True)
                display(HTML(f"<div style='color:#721c24;'>❌ {html.escape(str(e))}</div>"))
                return
            finally:
                progress.layout.visibility = 'hidden'
                progress_note.value = ''
            res = pd.concat(frames, ignore_index=True)
            inc_txt, dec_txt = describe_thresholds(s['mode'], s['q_increase'], s['q_decrease'], q_lo, q_hi)
            inc_short, dec_short = short_thresholds(s['mode'], s['q_increase'], s['q_decrease'], q_lo, q_hi)
            state['results'] = res
            state['settings'] = dict(s, q_lo=q_lo, q_hi=q_hi, level=level, alpha=alpha, n_boot=n_boot,
                                     accel=accel, elapsed=time.time() - t_start,
                                     inc_txt=inc_txt, dec_txt=dec_txt, inc_short=inc_short,
                                     dec_short=dec_short)
            state['info'] = {'overall': oinfo, 'subgroups': sinfo if sub is not None else None,
                             'subgroup_note': sinfo if sub is None else None}
            state['tokens'] = {'overall_results': results_token('overall_results'),
                               'subgroup_results': results_token('subgroup_results')}
            ANALYSIS_CONFIG['meaningful_config'] = make_json_safe(s)
            ANALYSIS_CONFIG['meaningful_results'] = {
                'table': res, 'settings': make_json_safe(state['settings']),
                'timestamp': datetime.datetime.now()}
            clear_output(wait=True)
            render_results()
            render_text()
            refresh_plot_data()

    # --- 5. RESULTS TABLE AND TEXT ---
    def level_text(level):
        return ('true effects of individual effect sizes (variance τ² + σ²)' if level == 'effect'
                else 'true mean effects of studies (variance τ²)')

    def render_results():
        res, st, info = state['results'], state['settings'], state['info']
        ci = ci_pct_label(st['alpha'])
        h = [f"<div style='font-family:sans-serif; font-size:13px; max-width:1000px;'>",
             f"<p><b>Meaningful increase:</b> {html.escape(st['inc_txt'])}. <b>Meaningful decrease:</b> "
             f"{html.escape(st['dec_txt'])}. Shares of the {level_text(st['level'])}, with {ci} CIs.</p>"]
        if 0 < st['n_boot'] < EXPLORATORY_BELOW:
            h.append(exploratory_html(st))
        elif st['n_boot']:
            h.append(f"<p style='color:#666; font-size:12px;'>{st['n_boot']} bootstrap resamples, "
                     f"{st['elapsed'] / 60:.1f} min.</p>")
        if info['overall'].get('note'):
            h.append(f"<div style='color:#856404; background:#fff3cd; padding:6px;'>⚠️ "
                     f"{html.escape(info['overall']['note'])}</div>")
        if info.get('subgroup_note'):
            h.append(f"<div style='color:#856404;'>ℹ️ {html.escape(info['subgroup_note'])}</div>")
        h.append("<table style='border-collapse:collapse; width:100%;'><tr style='background:#2E86AB; color:white;'>"
                 "<th style='padding:6px; text-align:left;'>Group</th><th>Mean</th><th>Method</th>"
                 "<th>Meaningful decrease</th><th>Negligible</th><th>Meaningful increase</th><th>CI</th></tr>")
        for i, r in res.iterrows():
            bg = '#f8f9fa' if (i // 2) % 2 == 0 else 'white'
            cells, how = [], ''
            for key in ('decrease', 'negligible', 'increase'):
                lo, hi, how_k = best_ci(r, key)
                how = how or how_k
                cells.append(f"{fmt_p(r[f'p_{key}'])}{fmt_ci(lo, hi)}")
            first = r['method'] == 'parametric'
            h.append(f"<tr style='background:{bg};'>"
                     f"<td style='padding:5px;'>{'<b>' + html.escape(str(r['target'])) + '</b>' if first else ''}</td>"
                     f"<td style='text-align:center;'>{f'{r.mu:.3f}' if first else ''}</td>"
                     f"<td style='text-align:center;'>{r['method']}</td>"
                     + ''.join(f"<td style='text-align:center;'>{c}</td>" for c in cells)
                     + f"<td style='text-align:center; color:#666; font-size:12px;'>{how}</td></tr>")
        h.append("</table>")
        notes = sorted({n for n in res['note'] if n})
        if notes:
            h.append("<ul style='color:#666; font-size:12px;'>" + ''.join(f"<li>{html.escape(n)}</li>" for n in notes)
                     + "</ul>")
        h.append("<p style='color:#666; font-size:12px;'><b>Parametric</b>: true effects assumed normal "
                 "with the fitted mean and heterogeneity. <b>Calibrated</b>: no normality assumption; each "
                 "effect size shrunk toward its model mean by √(T²/(T²+v)) (individual effect sizes only). "
                 "When the two differ markedly, the true effects are probably not normal; prefer the calibrated "
                 "estimate. Bootstrap CIs resample whole studies and refit the model each time (BCa).</p></div>")
        display(HTML(''.join(h)))

    def publication_text():
        res, st, info = state['results'], state['settings'], state['info']
        ci = ci_pct_label(st['alpha'])
        boot = st['n_boot'] > 0
        boot_ci = ("bias-corrected and accelerated (BCa) intervals, the acceleration estimated by a "
                   "leave-one-study-out jackknife" if st.get('accel', True) else
                   "bias-corrected percentile intervals")
        methods = (
            "To describe how much of the distribution of true effects was of a meaningful size, we estimated "
            "the proportion of true effects beyond thresholds of scientific importance (Mathur & VanderWeele, "
            f"2019): {html.escape(st['inc_txt'])} and {html.escape(st['dec_txt'])}. Proportions refer to the "
            f"{level_text(st['level'])} under the fitted {html.escape(info['overall']['label'])}"
            + (f" and the {html.escape(info['subgroups']['label'])}" if info.get('subgroups') else '')
            + ". Parametric estimates assume normally distributed true effects; calibrated estimates, which do "
            "not, are the share of effect sizes whose calibrated estimate (each effect size shrunk toward its "
            "model mean by √(T²/(T²+v<sub>i</sub>)), with T² the heterogeneity variance) lies beyond the "
            "threshold (Wang & Lee, 2019; Mathur & VanderWeele, 2020). "
            + (f"{ci} confidence intervals were obtained by a cluster bootstrap ({st['n_boot']} resamples of "
               f"studies, refitting the model in each), with {boot_ci}."
               if boot else
               f"{ci} confidence intervals for the parametric estimates were obtained by the delta method "
               "(Mathur & VanderWeele, 2019).")
        )
        parts = []
        for tgt in res['target'].unique():
            sub = res[res['target'] == tgt]
            r = sub[sub['method'] == ('calibrated' if (sub['method'] == 'calibrated').any()
                                      and np.isfinite(sub[sub['method'] == 'calibrated']['p_increase'].iloc[0])
                                      else 'parametric')].iloc[0]
            pieces = []
            for key, word in (('increase', 'meaningful increases'), ('decrease', 'meaningful decreases'),
                              ('negligible', 'negligible')):
                lo, hi, _ = best_ci(r, key)
                pieces.append(f"{fmt_p(r[f'p_{key}'])}{fmt_ci(lo, hi)} {word}")
            name = 'Overall' if tgt == 'Overall' else f"In the {html.escape(str(tgt))} subgroup"
            parts.append(f"{name}, an estimated {pieces[0]}, {pieces[1]} and {pieces[2]} "
                         f"({r['method']} estimates)")
        results = '. '.join(parts) + '.'
        refs = ("Mathur, M. B., & VanderWeele, T. J. (2019). New metrics for meta-analyses of heterogeneous "
                "effects. <i>Statistics in Medicine, 38</i>(8), 1336–1342. https://doi.org/10.1002/sim.8057<br>"
                "Mathur, M. B., & VanderWeele, T. J. (2020). Robust metrics and sensitivity analyses for "
                "meta-analyses of heterogeneous effects. <i>Epidemiology, 31</i>(3), 356–358. "
                "https://doi.org/10.1097/EDE.0000000000001180<br>"
                "Wang, C.-C., & Lee, W.-C. (2019). A simple method to estimate prediction intervals and "
                "predictive distributions: Summarizing meta-analyses beyond means and confidence intervals. "
                "<i>Research Synthesis Methods, 10</i>(2), 255–266. https://doi.org/10.1002/jrsm.1345")
        return methods, results, refs

    def exploratory_html(st):
        return (f"<div style='color:#856404; background:#fff3cd; padding:8px; border-left:4px solid #ffc107; "
                f"margin:6px 0;'>🧪 <b>Exploratory run</b> ({st['n_boot']} bootstrap resamples, bias-corrected "
                f"percentile intervals, {st['elapsed'] / 60:.1f} min). Good for trying thresholds; before "
                f"reporting, run again with 1000 or more resamples (BCa intervals).</div>")

    def render_text():
        with out_text:
            clear_output()
            st = state['settings']
            if 0 < st['n_boot'] < EXPLORATORY_BELOW:
                display(HTML(exploratory_html(st)))
            m, r, refs = publication_text()
            box = "background:#f8f9fa; border-left:4px solid #2E86AB; padding:12px; font-family:Georgia, serif; " \
                  "font-size:13px; line-height:1.6; max-width:900px; margin-bottom:10px;"
            display(HTML(f"<div style='{box}'><b>Methods.</b> {m}</div><div style='{box}'><b>Results.</b> {r}</div>"
                         f"<div style='font-size:12px; color:#555;'><b>References</b><br>{refs}</div>"))
            ANALYSIS_CONFIG['meaningful_text'] = f"{m}\n\n{r}"

    def export_csv(_b=None):
        with out_export:
            clear_output()
            if state['results'] is None:
                display(HTML("<div style='color:#721c24;'>❌ Compute the proportions first.</div>"))
                return
            df = state['results'].copy()
            for k, v in state['settings'].items():
                if k not in df.columns and isinstance(v, (int, float, str)):
                    df[f'setting_{k}'] = v
            payload = df.to_csv(index=False)
            fname = f"Meaningful_Effects_{cometa_timestamp()}.csv"
            with open(fname, 'w') as f:
                f.write(payload)
            b64 = base64.b64encode(payload.encode()).decode()
            display(Javascript(f"var a=document.createElement('a'); a.href='data:text/csv;base64,{b64}'; "
                               f"a.download='{fname}'; document.body.appendChild(a); a.click(); "
                               f"document.body.removeChild(a);"))
            display(HTML(f"<div style='color:#155724;'>✅ Saved <code>{fname}</code></div>"))

    # --- 6. FIGURE (plot recipe) ---
    C = {
        'style': widgets.Dropdown(options=[('Stacked bars (all groups)', 'bars'),
                                           ('Distribution of true effects (one group)', 'density')],
                                  value='bars', description='Plot:', style=S_, layout=L_()),
        'method': widgets.Dropdown(options=[('Calibrated', 'calibrated'), ('Parametric', 'parametric')],
                                   value='calibrated', description='Estimates (bars):', style=S_, layout=L_()),
        'density_target': widgets.Dropdown(options=['Overall'], value='Overall', description='Group (distribution):',
                                           style=S_, layout=L_()),
        'show_ci': widgets.Checkbox(value=True, description='Show CIs', indent=False),
        'show_pct': widgets.Checkbox(value=True, description='Show percentages', indent=False),
        'color_dec': widgets.ColorPicker(value='#d7301f', description='Decrease colour:', style=S_),
        'color_neg': widgets.ColorPicker(value='#bdbdbd', description='Negligible colour:', style=S_),
        'color_inc': widgets.ColorPicker(value='#1a9850', description='Increase colour:', style=S_),
        'width': widgets.FloatSlider(value=8.0, min=2.0, max=16.0, step=0.1, description='Width (in):',
                                     continuous_update=False, layout=L_()),
        'height': widgets.FloatSlider(value=4.0, min=2.0, max=14.0, step=0.1, description='Height (in):',
                                      continuous_update=False, layout=L_()),
        'title_font': widgets.IntSlider(value=13, min=6, max=24, description='Title Font:', layout=L_()),
        'label_font': widgets.IntSlider(value=11, min=6, max=18, description='Label Font:', layout=L_()),
        'tick_font': widgets.IntSlider(value=10, min=5, max=16, description='Tick Font:', layout=L_()),
        'show_title': widgets.Checkbox(value=True, description='Show Title', indent=False),
        'title': widgets.Text(value='Proportion of meaningful effects', description='Title:', layout=L_()),
        'xlabel': widgets.Text(value='Share of true effects (%)', description='X Label:', layout=L_()),
        'legend': widgets.Dropdown(options=[('Below the plot', 'below'), ('Upper right', 'upper right'),
                                            ('None', 'none')], value='below', description='Legend:',
                                   style=S_, layout=L_()),
        'save_pdf': widgets.Checkbox(value=True, description='Save as PDF', indent=False),
        'save_png': widgets.Checkbox(value=True, description='Save as PNG', indent=False),
        'save_svg': widgets.Checkbox(value=False, description='Save as SVG', indent=False),
        'transparent': widgets.Checkbox(value=False, description='Transparent Background', indent=False),
        'dpi': widgets.IntSlider(value=300, min=100, max=600, step=50, description='PNG DPI:'),
        'filename': widgets.Text(value='Meaningful_Effects', description='Filename:', layout=L_('300px')),
    }
    label_box = widgets.VBox()
    label_widgets = {}

    def rebuild_labels(data):
        label_widgets.clear()
        for t in data['targets']:
            label_widgets[f'label::{t}'] = widgets.Text(value=str(t), description=f'{t}:',
                                                        layout=L_(), style={'description_width': '160px'})
        label_box.children = list(label_widgets.values())
        C['density_target'].options = list(data['targets'])
        C['density_target'].value = data['targets'][0]
        if not data['has_calibrated']:
            C['method'].value = 'parametric'

    def controls():
        return {**C, **label_widgets}

    def refresh_plot_data():
        res, st = state['results'], state['settings']
        tbl = res[['target', 'method', 'mu', 'T2'] + [c for c in res.columns
                                                     if c.startswith(('p_', 'increase_', 'decrease_',
                                                                      'negligible_'))]].copy()
        data = {
            'table': tbl, 'targets': list(dict.fromkeys(res['target'])),
            'q_lo': st['q_lo'], 'q_hi': st['q_hi'], 'null_value': null,
            'inc_txt': st['inc_txt'], 'dec_txt': st['dec_txt'], 'level': st['level'],
            'inc_short': st['inc_short'], 'dec_short': st['dec_short'],
            'ci_level': ci_pct_label(st['alpha']), 'effect_label': effect_label, 'ratio_scale': ratio_scale,
            'has_calibrated': bool(np.isfinite(res.loc[res['method'] == 'calibrated', 'p_increase']).any()),
            'n_boot': int(st['n_boot']), 'exploratory': bool(0 < st['n_boot'] < EXPLORATORY_BELOW),
        }
        info = state['info']
        state['data'] = data
        state['source'] = {'analysis': 'Proportion of meaningful effects: ' + info['overall']['label']
                           + (f"; {info['subgroups']['label']}" if info.get('subgroups') else ''),
                           'effect_label': effect_label, 'ci_level': data['ci_level'],
                           'analysis_time': datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}
        state['recipe_name'] = None
        rebuild_labels(data)
        source_box.value = source_html(state['source'])

    def _ci(row, key, s):
        lo, hi = row.get(f'{key}_lo_boot', np.nan), row.get(f'{key}_hi_boot', np.nan)
        if not (np.isfinite(lo) and np.isfinite(hi)):
            lo, hi = row.get(f'{key}_lo_delta', np.nan), row.get(f'{key}_hi_delta', np.nan)
        return lo, hi

    def draw(data, s):
        tbl = data['table']
        tbl = tbl[tbl['method'] == s['method']]
        names = [s.get(f'label::{t}', t) for t in data['targets']]
        fig, ax = plt.subplots(figsize=(s['width'], s['height']))
        cols = {'decrease': s['color_dec'], 'negligible': s['color_neg'], 'increase': s['color_inc']}
        leg = {'decrease': f"Meaningful decrease ({data['dec_short']})", 'negligible': 'Negligible',
               'increase': f"Meaningful increase ({data['inc_short']})"}
        if s['style'] == 'bars':
            ypos = np.arange(len(data['targets']))[::-1]
            for yi_, t in zip(ypos, data['targets']):
                r = tbl[tbl['target'] == t].iloc[0]
                left = 0.0
                for key in ('decrease', 'negligible', 'increase'):
                    p = float(r[f'p_{key}']) if np.isfinite(r[f'p_{key}']) else 0.0
                    ax.barh(yi_, 100 * p, left=100 * left, color=cols[key], edgecolor='white', height=0.6,
                            label=leg[key] if yi_ == ypos[0] else None)
                    if s['show_pct'] and p >= 0.06:
                        ax.text(100 * (left + p / 2), yi_, f"{100 * p:.0f}%", ha='center', va='center',
                                fontsize=s['tick_font'], color='white' if key != 'negligible' else 'black')
                    left += p
                if s['show_ci']:
                    # increase CI along the top edge of the bar, decrease CI along the bottom edge
                    for key, edge, dy in (('decrease', float(r['p_decrease']), -0.38),
                                          ('increase', 1 - float(r['p_increase']), 0.38)):
                        lo, hi = _ci(r, key, s)
                        if np.isfinite(lo) and np.isfinite(hi) and np.isfinite(edge):
                            a, b = (lo, hi) if key == 'decrease' else (1 - hi, 1 - lo)
                            ax.errorbar(100 * edge, yi_ + dy, xerr=[[100 * max(edge - a, 0)], [100 * max(b - edge, 0)]],
                                        fmt='none', ecolor='black', elinewidth=1.2, capsize=3, zorder=5)
            ax.set_yticks(ypos)
            ax.set_yticklabels(names, fontsize=s['tick_font'])
            for lab in ax.get_yticklabels():
                if lab.get_text() == s.get('label::Overall', 'Overall'):
                    lab.set_fontweight('bold')
            ax.set_xlim(0, 100)
            ax.set_xlabel(s['xlabel'], fontsize=s['label_font'], fontweight='bold')
            ax.tick_params(axis='x', labelsize=s['tick_font'])
            for side in ('top', 'right'):
                ax.spines[side].set_visible(False)
        else:
            # the curve is the parametric (normal) model, so its shares are the parametric ones
            t = s['density_target'] if s['density_target'] in data['targets'] else data['targets'][0]
            full = data['table']
            r = full[(full['target'] == t) & (full['method'] == 'parametric')].iloc[0]
            mu, sd = float(r['mu']), float(np.sqrt(max(r['T2'], 0.0)))
            lo_x = min(mu - 4 * sd, data['q_lo'] - sd)
            hi_x = max(mu + 4 * sd, data['q_hi'] + sd)
            x = np.linspace(lo_x, hi_x, 600)
            dens = _norm_9b.pdf(x, mu, sd) if sd > 0 else np.zeros_like(x)
            ax.plot(x, dens, color='black', lw=1.5)
            for key, mask in (('decrease', x <= data['q_lo']), ('negligible', (x > data['q_lo']) & (x < data['q_hi'])),
                              ('increase', x >= data['q_hi'])):
                pct = f" {100 * float(r[f'p_{key}']):.0f}%" if s['show_pct'] else ''
                ax.fill_between(x, 0, dens, where=mask, color=cols[key], alpha=0.8, label=leg[key] + pct)
            for q in (data['q_lo'], data['q_hi']):
                ax.axvline(q, color='black', ls='--', lw=1)
            ax.axvline(data['null_value'], color='gray', lw=1)
            ax.set_xlabel(f"True {data['effect_label']}", fontsize=s['label_font'], fontweight='bold')
            ax.set_ylabel('Density', fontsize=s['label_font'], fontweight='bold')
            ax.tick_params(axis='both', labelsize=s['tick_font'])
            ax.set_ylim(bottom=0)
            for side in ('top', 'right'):
                ax.spines[side].set_visible(False)
        if s['show_title']:
            ttl = s['title'] if s['style'] == 'bars' else f"{s['title']}: {s.get(f'label::{t}', t)}"
            ax.set_title(ttl, fontsize=s['title_font'], fontweight='bold')
        if s['legend'] == 'below':
            ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1, frameon=False, fontsize=s['tick_font'])
        elif s['legend'] != 'none':
            ax.legend(loc=s['legend'], fontsize=s['tick_font'])
        fig.tight_layout()
        return fig

    def caption(data, s):
        m = 'calibrated (distribution-free)' if s['method'] == 'calibrated' else 'parametric (normal true effects)'
        lvl = level_text(data['level'])
        if s['style'] == 'bars':
            txt = (f"Estimated share of {lvl} that are meaningful decreases ({html.escape(data['dec_txt'])}), "
                   f"negligible, or meaningful increases ({html.escape(data['inc_txt'])}), {m} estimates "
                   f"(Mathur &amp; VanderWeele, 2019, 2020).")
            if s['show_ci']:
                txt += (f" Whiskers are {data['ci_level']} confidence intervals of the share of meaningful "
                        "increases (top edge of each bar) and decreases (bottom edge), drawn at the segment "
                        "boundaries they move.")
                if data.get('exploratory'):
                    txt += (f" <b>Exploratory intervals ({data['n_boot']} bootstrap resamples); rerun with 1000 "
                            "or more before publication.</b>")
        else:
            txt = (f"Distribution of {lvl} implied by the fitted model (normal, with the estimated mean and "
                   f"heterogeneity), with the regions beyond the thresholds ({html.escape(data['dec_txt'])}; "
                   f"{html.escape(data['inc_txt'])}) shaded; the percentages are the areas of these regions "
                   f"(parametric estimates).")
        return txt

    out_plot = widgets.Output()

    def generate(_b=None):
        with out_plot:
            clear_output(wait=True)
            if state['data'] is None:
                display(HTML("<div style='color:#721c24;'>❌ Compute the proportions first (Analysis tab), "
                             "or load a plot recipe.</div>"))
                return
            if state['recipe_name'] is None and any(results_token(k) != v for k, v in state['tokens'].items()):
                display(HTML(stale_warning_html('Step 7 or Step 9')))
            s = widget_values(controls())
            if s['method'] == 'calibrated' and not state['data']['has_calibrated']:
                s['method'] = 'parametric'
            try:
                fig = draw(state['data'], s)
                paths = cometa_save_figure(fig, s['filename'], pdf=s['save_pdf'], png=s['save_png'],
                                           svg=s['save_svg'], dpi=s['dpi'], transparent=s['transparent'])
                show_figure(fig)
            except Exception as e:
                plt.close('all')
                display(HTML(f"<div style='color:#721c24;'>❌ Plotting error: {html.escape(str(e))}</div>"))
                return
            display(HTML(saved_files_html(paths)))
            display(HTML(caption_html(caption(state['data'], s))))

    def get_recipe():
        if state['data'] is None:
            return None
        return build_plot_recipe(PLOT_TYPE, state['data'], widget_values(controls()), state['source'])

    def on_load(rec, style_only, name):
        if not style_only:
            state.update(data=rec['data'], source=rec.get('source'), recipe_name=name)
            rebuild_labels(state['data'])
            source_box.value = source_html(state['source'], name)
        skipped = apply_widget_values(controls(), rec.get('settings'))
        generate()
        return skipped

    # --- 7. LAYOUT ---
    H = lambda t: widgets.HTML(f"<h4 style='color:#2E86AB; margin:6px 0;'>{t}</h4>")
    run_btn = widgets.Button(description='▶ Compute Proportions', button_style='success',
                             layout=widgets.Layout(width='300px', height='40px'))
    run_btn.on_click(compute)
    csv_btn = widgets.Button(description='📥 Download Results (CSV)', button_style='warning',
                             layout=widgets.Layout(width='300px'))
    csv_btn.on_click(export_csv)
    gen_btn = widgets.Button(description='📊 Generate Plot', button_style='success',
                             layout=widgets.Layout(width='300px', height='40px'))
    gen_btn.on_click(generate)
    preset = preset_dropdown({'width': C['width'], 'height': C['height'], 'title': C['title_font'],
                              'label': C['label_font'], 'tick': C['tick_font']})

    unit_note = ("Percent thresholds are converted to the log scale: an increase of p% is a ratio of 1 + p/100, "
                 "a decrease of p% a ratio of 1 − p/100." if ratio_scale else
                 "Choose values that are biologically or practically meaningful for your outcome, ideally "
                 "before looking at the results.")
    analysis_tab = widgets.VBox([
        H('Thresholds'), A['mode'], A['q_increase'], A['q_decrease'], A['same'],
        widgets.HTML(f"<div style='color:#666; font-size:12px; max-width:700px;'>{unit_note}</div>"),
        H('Estimation'), A['level'], A['bootstrap'], A['n_boot'], A['seed'],
        widgets.HTML("<div style='color:#666; font-size:12px; max-width:700px;'>The bootstrap refits the model "
                     "on resampled studies; the time grows with the number of resamples and the size of the data. "
                     "Use <b>50</b> or <b>200</b> resamples to explore thresholds quickly (bias-corrected "
                     "percentile intervals, marked as exploratory), and <b>1000</b> or more for the results you "
                     "report (BCa intervals). Without the bootstrap, only the parametric estimates get a "
                     "(delta-method) CI.</div>"),
        widgets.HTML("<br>"), run_btn, progress, progress_note])
    plot_tab = widgets.VBox([
        source_box, H('Plot'), C['style'], C['method'], C['density_target'], C['show_ci'], C['show_pct'],
        H('Style'), preset, C['width'], C['height'], C['title_font'], C['label_font'], C['tick_font'],
        C['color_dec'], C['color_neg'], C['color_inc'], C['legend'],
        H('Text'), C['show_title'], C['title'], C['xlabel'], H('Group labels'), label_box,
        H('Export'), C['save_pdf'], C['save_png'], C['save_svg'], C['transparent'], C['dpi'], C['filename'],
        widgets.HTML("<br>"), gen_btn, plot_recipe_bar(PLOT_TYPE, get_recipe, on_load, lambda: C['filename'].value),
        out_plot])
    tabs = widgets.Tab(children=[widgets.VBox([analysis_tab, out_results]), plot_tab, out_text,
                                 widgets.VBox([csv_btn, out_export])])
    for i, t in enumerate(['⚙️ Analysis & Results', '📈 Plot', '📝 Publication Text', '💾 Export']):
        tabs.set_title(i, t)

    source_box.value = source_html(None)
    display(HTML(
        "<div style='font-family:sans-serif; font-size:13px; color:#34495e; line-height:1.5; max-width:900px;'>"
        "<h3 style='color:#2E86AB; border-bottom:2px solid #3498db; padding-bottom:5px;'>Step 9b: Proportion of "
        "Meaningful Effects</h3><p style='margin:0 0 6px 0;'>The pooled mean says how large the effect is on "
        "average; with heterogeneity, true effects vary around it. This step estimates <b>which share of true "
        "effects is large enough to matter</b>: meaningful increases, meaningful decreases and negligible "
        "effects, for a threshold you choose (Mathur &amp; VanderWeele 2019, 2020). It uses the overall model "
        "of Step 7 and, if Step 9 was run, each subgroup.</p></div>"))
    display(tabs)


_meaningful_effects_cell()


In [ ]:
#@title 📊 10. Subgroup Analysis: Visualization - Forest Plot
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# STEP 10: PUBLICATION-READY FOREST PLOT
# Purpose: Forest plot of the subgroup estimates (Step 9) with the overall
#          estimate (Step 7) on top.
# Data:    Step 9 (Subgroup Analysis) and Step 7 (Overall Meta-Analysis), or
#          a plot recipe file saved from this cell.
# =============================================================================

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
import html
import re
import types


def _forest_plot_cell():
    PLOT_TYPE = 'forest'
    state = {'data': None, 'source': None, 'recipe_name': None, 'token': None}
    ROW_COLS = ['GroupVar', 'LabelVar', 'k', 'nPapers', 'EffectSize', 'SE', 'CI_Lower', 'CI_Upper', 'FoldChange']

    def natural_key(text):
        # "Group 2" before "Group 10"; numbers and words never compared with each other
        return tuple((0, int(p), '') if p.isdigit() else (1, 0, p.lower())
                     for p in re.split(r'(\d+)', str(text)) if p != '')

    def fold_change(mu, es):
        """Step 9's fold-change convention (SubgroupAnalyzer.calculate_fold_change)."""
        analyzer = globals().get('SubgroupAnalyzer')
        if analyzer is not None and hasattr(analyzer, 'calculate_fold_change'):
            return float(analyzer.calculate_fold_change(types.SimpleNamespace(es_config=es), mu))
        if not es.get('has_fold_change', False) or not np.isfinite(mu):
            return np.nan
        ratio = float(np.exp(mu))
        if es.get('effect_label_short', '') in ('logOR', 'RR'):
            return ratio
        return ratio if mu >= 0 else -1.0 / ratio

    def results_tokens():
        return (results_token('subgroup_results'), results_token('overall_results'))

    # --- 1. DATA SNAPSHOT (everything the figure draws) ---
    def build_data():
        overall = ANALYSIS_CONFIG.get('overall_results')
        if not overall:
            return None, None
        es = ANALYSIS_CONFIG.get('es_config') or {}
        sub = ANALYSIS_CONFIG.get('subgroup_results') or {}
        has_sub = 'results_df' in sub and sub['results_df'] is not None and len(sub['results_df']) > 0

        rep = overall.get('reported') or {}
        mu = rep.get('mu', overall.get('pooled_effect_random'))
        overall_row = {
            'k': overall.get('k'), 'nPapers': overall.get('k_papers'), 'EffectSize': mu,
            'SE': rep.get('se', overall.get('pooled_SE_random_reported', overall.get('pooled_SE_random_Z'))),
            'CI_Lower': rep.get('ci_lower', overall.get('ci_lower_random_reported', overall.get('ci_lower_random_Z'))),
            'CI_Upper': rep.get('ci_upper', overall.get('ci_upper_random_reported', overall.get('ci_upper_random_Z'))),
        }
        overall_row = {k: (None if v is None else float(v)) for k, v in overall_row.items()}
        overall_row['FoldChange'] = fold_change(overall_row['EffectSize'], es)

        analysis_type, mod1, mod2, model_types = 'overall_only', None, None, []
        rows = pd.DataFrame(columns=ROW_COLS)
        if has_sub:
            analysis_type = 'two_way' if sub.get('analysis_type') == 'two_way' else 'single'
            mod1 = sub.get('moderator1')
            mod2 = sub.get('moderator2') if analysis_type == 'two_way' else None
            df = sub['results_df']
            rows = pd.DataFrame({
                'GroupVar': (df['moderator1_value'].astype(str).tolist() if analysis_type == 'two_way'
                             else ['Subgroup'] * len(df)),
                'LabelVar': (df['moderator2_value'] if analysis_type == 'two_way' else df['group']).astype(str).tolist(),
                'k': df['k'].astype(float).tolist(),
                'nPapers': df['n_papers'].astype(float).tolist(),
                'EffectSize': df['pooled_effect_re'].astype(float).tolist(),
                'SE': df['pooled_se_re'].astype(float).tolist(),
                'CI_Lower': df['ci_lower_re'].astype(float).tolist(),
                'CI_Upper': df['ci_upper_re'].astype(float).tolist(),
            }, columns=ROW_COLS[:-1])
            rows = rows.dropna(subset=['EffectSize', 'SE'])
            rows['FoldChange'] = [fold_change(m, es) for m in rows['EffectSize']]
            order = sorted(range(len(rows)), key=lambda i: (natural_key(rows['GroupVar'].iloc[i]),
                                                            natural_key(rows['LabelVar'].iloc[i])))
            rows = rows.iloc[order].reset_index(drop=True)
            if 'model_type' in df.columns:
                model_types = [str(m) for m in dict.fromkeys(df['model_type'].dropna())]

        ci_level = ci_pct_label()
        data = {
            'rows': rows,
            'overall': overall_row,
            'analysis_type': analysis_type,
            'moderator1': mod1,
            'moderator2': mod2,
            'effect_label': es.get('effect_label', 'Effect Size'),
            'es_type': es.get('type', ANALYSIS_CONFIG.get('effect_size_type', '')),
            'has_fold_change': bool(es.get('has_fold_change', False)),
            'fold_signed': es.get('effect_label_short', '') not in ('logOR', 'RR'),
            'null_value': es_null_value(),
            'ci_level': ci_level,
            'subgroup_model': ', '.join(model_types),
            'overall_model': rep.get('model', '2-Level'),
        }
        if analysis_type == 'two_way':
            analysis = f"Subgroups by {mod1} × {mod2} (Step 9)"
        elif analysis_type == 'single':
            analysis = f"Subgroups by {mod1} (Step 9)"
        else:
            analysis = "Overall effect only (Step 7; no subgroup analysis yet)"
        when = sub.get('timestamp') if has_sub else None
        model = 'random effects'
        if data['subgroup_model']:
            model += f" (subgroups: {data['subgroup_model']}; Overall: {data['overall_model']})"
        else:
            model += f" (Overall: {data['overall_model']})"
        source = {
            'analysis': analysis,
            'moderator1': mod1,
            'moderator2': mod2,
            'effect_label': data['effect_label'],
            'model': model,
            'ci_level': ci_level,
            'analysis_time': when.strftime('%Y-%m-%d %H:%M') if hasattr(when, 'strftime') else when,
        }
        return data, source

    # --- 2. CONTROLS (every widget whose value is saved in a recipe) ---
    ST = {'description_width': '130px'}
    L = lambda w='450px': widgets.Layout(width=w)
    C = {
        'width': widgets.FloatSlider(value=8.0, min=2.0, max=14.0, step=0.01, description='Plot Width (in):',
                                     continuous_update=False, style=ST, layout=L()),
        'height': widgets.FloatSlider(value=0.4, min=0.1, max=1.0, step=0.01, description='Height per Row (in):',
                                      continuous_update=False, style=ST, layout=L()),
        'title_font': widgets.IntSlider(value=12, min=6, max=18, description='Title Font Size:',
                                        continuous_update=False, style=ST, layout=L()),
        'label_font': widgets.IntSlider(value=11, min=6, max=16, description='Axis Label Size:',
                                        continuous_update=False, style=ST, layout=L()),
        'tick_font': widgets.IntSlider(value=9, min=5, max=14, description='Tick Label Size:',
                                       continuous_update=False, style=ST, layout=L()),
        'annot_font': widgets.IntSlider(value=8, min=5, max=12, description='Annotation Size:',
                                        continuous_update=False, style=ST, layout=L()),
        'color_scheme': widgets.Dropdown(options=[('Grayscale (Publication)', 'gray'), ('Color (Presentation)', 'color'),
                                                  ('Black & White Only', 'bw')],
                                         value='gray', description='Color Scheme:', style=ST, layout=L()),
        'marker_style': widgets.Dropdown(options=[('Circle/Diamond (●/◆)', 'circle_diamond'),
                                                  ('Square/Diamond (■/◆)', 'square_diamond'),
                                                  ('Circle/Star (●/★)', 'circle_star')],
                                         value='circle_diamond', description='Marker Style:', style=ST, layout=L()),
        'ci_style': widgets.Dropdown(options=[('Solid Line', 'solid'), ('Dashed Line', 'dashed'),
                                              ('Solid with Caps', 'caps')],
                                     value='solid', description='CI Line Style:', style=ST, layout=L()),
        'show_title': widgets.Checkbox(value=True, description='Show Plot Title', indent=False, layout=L()),
        'title': widgets.Text(value='Forest Plot', description='Plot Title:', style=ST, layout=L()),
        'xlabel': widgets.Text(value='Effect Size', description='X-Axis Label:', style=ST, layout=L()),
        'show_ylabel': widgets.Checkbox(value=True, description='Show Y-Axis Label', indent=False, layout=L()),
        'ylabel': widgets.Text(value='Study', description='Y-Axis Label:', style=ST, layout=L()),
        'show_k': widgets.Checkbox(value=True, description='Show k (observations)', indent=False, layout=L()),
        'show_papers': widgets.Checkbox(value=True, description='Show paper count', indent=False, layout=L()),
        'show_fold_change': widgets.Checkbox(value=False, description='Show Fold-Change', indent=False, layout=L()),
        'annot_pos': widgets.Dropdown(options=[('Right of CI', 'right'), ('Above Marker', 'above'),
                                               ('Below Marker', 'below')],
                                      value='right', description='Position:', style=ST, layout=L()),
        'annot_offset': widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.05, description='H-Offset:',
                                            continuous_update=False, readout_format='.2f', style=ST, layout=L()),
        'group_h_offset': widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1, description='Group H-Offset:',
                                              continuous_update=False, style=ST, layout=L()),
        'group_v_offset': widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.5, description='Group V-Offset:',
                                              continuous_update=False, style=ST, layout=L()),
        'group_font': widgets.IntSlider(value=10, min=5, max=20, description='Group Fontsize:',
                                        continuous_update=False, style=ST, layout=L()),
        'auto_scale': widgets.Checkbox(value=True, description='Auto-Scale X-Axis', indent=False, layout=L()),
        'x_min': widgets.FloatText(value=-2.0, description='X-Min:', style={'description_width': '80px'},
                                   layout=widgets.Layout(width='220px')),
        'x_max': widgets.FloatText(value=2.0, description='X-Max:', style={'description_width': '80px'},
                                   layout=widgets.Layout(width='220px')),
        'show_grid': widgets.Checkbox(value=True, description='Show Grid', indent=False, layout=L()),
        'grid_style': widgets.Dropdown(options=[('Dashed (Light)', 'dashed_light'), ('Dotted (Light)', 'dotted_light'),
                                                ('Solid (Light)', 'solid_light')],
                                       value='dashed_light', description='Grid Style:', style=ST, layout=L()),
        'show_null': widgets.Checkbox(value=True, description='Show Null Effect Line', indent=False, layout=L()),
        'show_fold_axis': widgets.Checkbox(value=False, description='Show Fold-Change Axis (Top)', indent=False,
                                           layout=L()),
        'save_pdf': widgets.Checkbox(value=True, description='Save as PDF', indent=False, layout=L()),
        'save_png': widgets.Checkbox(value=True, description='Save as PNG', indent=False, layout=L()),
        'save_svg': widgets.Checkbox(value=False, description='Save as SVG', indent=False, layout=L()),
        'dpi': widgets.IntSlider(value=300, min=100, max=600, step=50, description='PNG DPI:',
                                 continuous_update=False, style=ST, layout=L()),
        'filename': widgets.Text(value='ForestPlot', description='Filename Prefix:', style=ST, layout=L()),
        'transparent': widgets.Checkbox(value=False, description='Transparent Background', indent=False, layout=L()),
    }
    manual_scale_box = widgets.HBox([C['x_min'], C['x_max']])

    def on_auto_scale(change=None):
        manual_scale_box.layout.display = 'none' if C['auto_scale'].value else ''
    C['auto_scale'].observe(on_auto_scale, names='value')
    on_auto_scale()

    group_box = widgets.VBox([widgets.HTML("<h4>Group Label Positioning (Two-Way Only)</h4>"),
                              C['group_h_offset'], C['group_v_offset'], C['group_font']])
    fold_note = widgets.HTML()

    # One text box per group label (rebuilt for every data snapshot)
    label_box = widgets.VBox()

    def build_label_widgets(data):
        for key in [k for k in C if k.startswith(('label::', 'group_label::')) or k == 'label_overall']:
            del C[key]
        lw = lambda value, desc: widgets.Text(value=value, description=desc, layout=widgets.Layout(width='500px'),
                                              style={'description_width': '200px'})
        C['label_overall'] = lw('Overall', 'Overall Effect:')
        children = [C['label_overall']]
        rows = data['rows'] if data is not None else pd.DataFrame(columns=ROW_COLS)
        if data is not None and data['analysis_type'] == 'two_way':
            children.append(widgets.HTML(f"<b>Group headers ({html.escape(str(data['moderator1']))}):</b>"))
            for g in dict.fromkeys(rows['GroupVar']):
                C[f'group_label::{g}'] = lw(g, f"{g}:")
                children.append(C[f'group_label::{g}'])
            children.append(widgets.HTML(f"<b>Rows ({html.escape(str(data['moderator2']))}):</b>"))
        elif data is not None and data['analysis_type'] == 'single':
            children.append(widgets.HTML(f"<b>Rows ({html.escape(str(data['moderator1']))}):</b>"))
        for v in sorted(dict.fromkeys(rows['LabelVar']), key=natural_key):
            C[f'label::{v}'] = lw(v, f"{v}:")
            children.append(C[f'label::{v}'])
        label_box.children = children

    def set_data_defaults(data):
        """Defaults that depend on the data (called for session data and recipes)."""
        t, m1, m2 = data['analysis_type'], data['moderator1'], data['moderator2']
        if t == 'two_way':
            C['title'].value, C['ylabel'].value = f'Forest Plot: {m1} × {m2}', str(m2)
            C['filename'].value = f'ForestPlot_{m1}_x_{m2}'
        elif t == 'single':
            C['title'].value, C['ylabel'].value = f'Forest Plot: {m1}', str(m1)
            C['filename'].value = f'ForestPlot_{m1}'
        else:
            C['title'].value, C['ylabel'].value = 'Forest Plot: Overall Effect', 'Study'
            C['filename'].value = 'ForestPlot_Overall'
        C['xlabel'].value = data['effect_label']
        C['show_fold_change'].value = data['has_fold_change']
        C['show_fold_axis'].value = data['has_fold_change']
        group_box.layout.display = '' if t == 'two_way' else 'none'
        fold_note.value = ('' if data['has_fold_change'] else
                           "<span style='color:#666; font-size:12px;'>Fold-change options apply only to "
                           "log-ratio effect sizes; they have no effect here.</span>")

    class PresetTarget:
        """Preset target that sets other widgets: a preset's total height becomes height per row;
        its label and tick font sizes also set the group-header and annotation sizes."""
        def __init__(self, apply):
            self.apply = apply

        @property
        def value(self):
            return None

        @value.setter
        def value(self, v):
            self.apply(v)

    def set_widget(w, v):
        w.min, w.max = min(w.min, v), max(w.max, v)
        w.value = v

    def preset_height(total):
        if state['data'] is not None:
            w = C['height']
            w.value = round(float(np.clip((total - 1.5) / (len(state['data']['rows']) + 1), w.min, w.max)), 2)

    # --- 3. DRAWING (data + settings -> figure; no widget access) ---
    def draw(data, s):
        ov = dict(data['overall'], GroupVar='Overall', LabelVar='Overall')
        plot_df = pd.DataFrame([ov], columns=ROW_COLS)
        if len(data['rows']):
            plot_df = pd.concat([plot_df, data['rows'][ROW_COLS]], ignore_index=True)
        two_way = data['analysis_type'] == 'two_way'
        fold_on = data['has_fold_change']
        row_label = lambda v: s.get(f'label::{v}', v)
        group_label = lambda g: s.get(f'group_label::{g}', g)

        num_rows = len(plot_df)
        y_positions = np.arange(num_rows)
        plot_height = max(2.5, num_rows * s['height'] + 1.5)
        y_tick_labels = [s.get('label_overall', 'Overall') if i == 0 else row_label(str(r['LabelVar']))
                         for i, r in plot_df.iterrows()]

        null = data['null_value']
        plot_min = min(plot_df['CI_Lower'].min(), null)
        plot_max = max(plot_df['CI_Upper'].max(), null)
        x_range = (plot_max - plot_min) or 1

        # space needed on the right for the annotations (and group labels)
        annot_parts = []
        if s['show_k']:
            annot_parts.append(f"k={int(plot_df['k'].max())}")
        if s['show_papers']:
            annot_parts.append(f"({int(plot_df['nPapers'].max())})")
        if s['show_fold_change'] and fold_on:
            annot_parts.append(f"[-{plot_df['FoldChange'].abs().max():.2f}×]")
        example_annot = " ".join(annot_parts) if annot_parts else "k=100 (10)"
        # text width as a fraction of the axes width (about 0.6 em per character)
        char_frac = lambda size: 0.6 * size / 72.0 / max(0.7 * s['width'], 0.5)
        annot_space = len(example_annot) * char_frac(s['annot_font']) if annot_parts else 0

        if s['auto_scale']:
            right_fraction = min(0.015 + annot_space + 0.03, 0.75)
            x_limits = (plot_min - x_range * 0.05, plot_max + x_range * (right_fraction / (1 - right_fraction)))
        else:
            x_limits = (s['x_min'], s['x_max'])

        colors = {'gray': ('dimgray', 'black', 'gray', 'black'),
                  'color': ('#4A90E2', '#E74C3C', '#4A90E2', '#E74C3C'),
                  'bw': ('black', 'black', 'black', 'black')}[s['color_scheme']]
        sub_color, ov_color, sub_ci_color, ov_ci_color = colors
        sub_marker, ov_marker = {'circle_diamond': ('o', 'D'), 'square_diamond': ('s', 'D'),
                                 'circle_star': ('o', '*')}[s['marker_style']]
        capsize = 4 if s['ci_style'] == 'caps' else 0
        linestyle = '--' if s['ci_style'] == 'dashed' else '-'

        fig, ax = plt.subplots(figsize=(s['width'], plot_height))
        if s['transparent']:
            fig.patch.set_alpha(0)
            ax.patch.set_alpha(0)

        for i, row in plot_df.iterrows():
            is_overall = i == 0
            zorder = 5 if is_overall else 3
            ax.errorbar(x=row['EffectSize'], y=y_positions[i],
                        xerr=[[row['EffectSize'] - row['CI_Lower']], [row['CI_Upper'] - row['EffectSize']]],
                        fmt='none', capsize=capsize, color=ov_ci_color if is_overall else sub_ci_color,
                        linewidth=2.0 if is_overall else 1.5, linestyle=linestyle, alpha=0.9, zorder=zorder - 1)
            ax.plot(row['EffectSize'], y_positions[i], marker=ov_marker if is_overall else sub_marker,
                    markersize=8 if is_overall else 6, markerfacecolor=ov_color if is_overall else sub_color,
                    markeredgecolor='black', markeredgewidth=1.0, linestyle='none', zorder=zorder)

        ax.set_xlim(x_limits[0], x_limits[1])
        ax.set_ylim(y_positions[-1] + 0.75, y_positions[0] - 0.75)  # inverted: Overall on top

        annots = []   # (text artist, anchor x in data units)
        for i, row in plot_df.iterrows():
            parts = []
            if s['show_k'] and pd.notna(row['k']):
                parts.append(f"k={int(row['k'])}")
            if s['show_papers'] and pd.notna(row['nPapers']):
                parts.append(f"({int(row['nPapers'])})")
            if s['show_fold_change'] and fold_on and pd.notna(row['FoldChange']):
                sign = "+" if data['fold_signed'] and row['FoldChange'] > 0 else ""
                parts.append(f"[{sign}{row['FoldChange']:.2f}×]")
            if not parts:
                continue
            if s['annot_pos'] == 'right':
                anchor, y_pos, va, ha = row['CI_Upper'], y_positions[i], 'center', 'left'
            elif s['annot_pos'] == 'above':
                anchor, y_pos, va, ha = row['EffectSize'], y_positions[i] - 0.2, 'bottom', 'center'
            else:
                anchor, y_pos, va, ha = row['EffectSize'], y_positions[i] + 0.2, 'top', 'center'
            t = ax.text(anchor, y_pos, " ".join(parts), va=va, ha=ha, fontsize=s['annot_font'],
                        fontweight='bold' if i == 0 else 'normal', clip_on=False)
            annots.append((t, anchor))
        gap = (0.015 if s['auto_scale'] else 0.05) if s['annot_pos'] == 'right' else 0.0

        def place_annotations():
            lo, hi = ax.get_xlim()
            for t, anchor in annots:
                t.set_x(anchor + (gap + s['annot_offset'] * 0.1) * (hi - lo))

        # two-way: a header per moderator-1 group right of the frame, groups separated by thin lines
        group_texts = []
        if two_way:
            groups = plot_df['GroupVar'].iloc[1:]
            for n_group, g in enumerate(dict.fromkeys(groups)):
                idx = groups.index[groups == g]
                if n_group > 0:
                    ax.axhline(y=y_positions[idx[0]] - 0.5, color='darkgray', linewidth=0.8, linestyle='-',
                               xmin=0.01, xmax=0.99, zorder=1)
                label_y = (y_positions[idx[0]] + y_positions[idx[-1]]) / 2.0 + s['group_v_offset']
                group_texts.append(ax.text(1.02, label_y, group_label(str(g)), va='center', ha='left',
                                           fontweight='bold', fontsize=s['group_font'], color='black',
                                           clip_on=False, transform=ax.get_yaxis_transform()))

        if num_rows > 1:
            ax.axhline(y=y_positions[0] + 0.5, color='black', linewidth=1.5, linestyle='-')
        if s['show_null']:
            ax.axvline(x=null, color='black', linestyle='-', linewidth=1.5, alpha=0.8, zorder=1)

        ax.set_xlabel(s['xlabel'], fontsize=s['label_font'], fontweight='bold')
        if s['show_ylabel']:
            ax.set_ylabel(s['ylabel'], fontsize=s['label_font'], fontweight='bold')
        if s['show_title']:
            ax.set_title(s['title'], fontweight='bold', fontsize=s['title_font'], pad=15)
        ax.set_yticks(y_positions)
        ax.set_yticklabels(y_tick_labels, fontsize=s['tick_font'])
        ax.tick_params(axis='x', labelsize=s['tick_font'])
        if s['show_grid']:
            ls, alpha = {'dashed_light': ('--', 0.3), 'dotted_light': (':', 0.3),
                         'solid_light': ('-', 0.2)}[s['grid_style']]
            ax.grid(axis='x', alpha=alpha, linestyle=ls, linewidth=0.5)

        # auto-scale: widen the x-axis until the right-hand annotations fit inside the frame;
        # two-way group headers go right of the frame (and of any annotation that sticks out)
        def fit_layout(fit_x):
            for _ in range(6):
                fig.tight_layout()
                if not (fit_x or group_texts):
                    return
                renderer = fig.canvas.get_renderer()
                ax_box = ax.get_window_extent(renderer)
                changed = False
                if fit_x:
                    lo, hi = ax.get_xlim()
                    need = hi - lo
                    for t, anchor in annots:
                        free = (ax_box.width * (1 - 0.03 - gap - s['annot_offset'] * 0.1)
                                - t.get_window_extent(renderer).width)
                        need = max(need, (anchor - lo) * ax_box.width / max(free, 0.4 * ax_box.width))
                    if need > (hi - lo) * 1.001:
                        ax.set_xlim(lo, lo + need)
                        place_annotations()
                        changed = True
                if group_texts:
                    edge = max([1.0] + [(t.get_window_extent(renderer).x1 - ax_box.x0) / ax_box.width
                                        for t, _a in annots])
                    gx = edge + 0.02 + s['group_h_offset'] * 0.05
                    if abs(gx - group_texts[0].get_position()[0]) > 0.005:
                        for t in group_texts:
                            t.set_x(gx)
                        changed = True
                if not changed:
                    return

        place_annotations()
        fit_layout(s['auto_scale'] and s['annot_pos'] == 'right' and bool(annots))
        final_xlims = ax.get_xlim()

        # top axis in back-transformed units (ratio or signed fold-change)
        if s['show_fold_axis'] and fold_on:
            ax2 = ax.twiny()
            is_ratio = data['es_type'] in ('log_or', 'log_rr')
            nice = np.array([0.1, 0.2, 0.25, 1 / 3, 0.5, 2 / 3, 0.8, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0, 3.0, 4.0, 5.0, 10.0])
            ticks = np.log(nice)
            keep = (ticks >= final_xlims[0]) & (ticks <= final_xlims[1])
            ticks, nice = ticks[keep], nice[keep]
            # keep ticks at least ~0.8 in apart, starting from the null value
            min_gap = (final_xlims[1] - final_xlims[0]) * 0.8 / max(ax.get_position().width * s['width'], 0.5)
            order = np.argsort(np.abs(ticks))
            chosen = []
            for j in order:
                if all(abs(ticks[j] - ticks[c]) >= min_gap for c in chosen):
                    chosen.append(j)
            chosen = sorted(chosen)
            ticks, nice = ticks[chosen], nice[chosen]
            ax2.set_xlim(final_xlims[0], final_xlims[1])
            ax2.set_xticks(ticks)
            if is_ratio:
                labels = [f"{v:.2f}" for v in nice]
            else:
                labels = [f"{1 / v:.2f}× ↓" if v < 1 else (f"{v:.2f}× ↑" if v > 1 else "1×") for v in nice]
            ax2.set_xticklabels(labels, fontsize=s['tick_font'])
            ax2.set_xlabel({'log_or': 'Odds Ratio', 'log_rr': 'Risk Ratio'}.get(data['es_type'], 'Fold-Change'),
                           fontsize=s['label_font'], fontweight='bold')

        fit_layout(False)
        return fig

    def caption(data, s):
        esc = lambda t: html.escape(str(t))
        t = data['analysis_type']
        if t == 'two_way':
            mod_txt = f" for each combination of {esc(data['moderator1'])} and {esc(data['moderator2'])}"
        elif t == 'single':
            mod_txt = f" for each level of {esc(data['moderator1'])}"
        else:
            mod_txt = ""
        ci_desc = {'solid': 'solid', 'dashed': 'dashed', 'caps': 'solid, capped'}[s['ci_style']]
        txt = (f"Forest plot of the pooled {esc(data['effect_label'])}{mod_txt}. Markers are random-effects "
               f"estimates, with {ci_desc} horizontal lines showing their {data['ci_level']} confidence intervals.")
        if t != 'overall_only':
            model = f" ({esc(data['subgroup_model'])})" if data['subgroup_model'] else ""
            txt += f" Subgroup estimates come from the Step 9 subgroup model{model}."
        txt += (f" The overall pooled effect ({esc(data['overall_model'])} random-effects model, Step 7) is shown "
                f"at the top.")
        if t == 'two_way':
            txt += f" Rows are grouped by {esc(data['moderator1'])} (labels on the right)."
        parts = []
        if s['show_k']:
            parts.append("the number of effect sizes (<i>k</i>)")
        if s['show_papers']:
            parts.append("the number of independent studies (in parentheses)")
        if s['show_fold_change'] and data['has_fold_change']:
            parts.append("the back-transformed ratio" if not data['fold_signed']
                         else "the fold-change (negative values are fold decreases)")
        if parts:
            txt += " Text annotations give " + ", ".join(parts[:-1]) + (" and " if len(parts) > 1 else "") + parts[-1] + "."
        if s['show_null']:
            txt += f" The vertical line marks no effect (x = {data['null_value']:g})."
        if s['show_fold_axis'] and data['has_fold_change']:
            txt += " The top axis shows the back-transformed values."
        return txt

    # --- 4. ACTIONS ---
    out = widgets.Output()
    caption_out = widgets.Output()
    source_box = widgets.HTML()

    def refresh_source():
        source_box.value = source_html(state['source'], state['recipe_name'])

    def generate(_b=None):
        with out:
            clear_output(wait=True)
            if state['data'] is None:
                display(HTML("<div style='color:#721c24;'>❌ No results: run Step 7 (Overall Meta-Analysis) and "
                             "Step 9 (Subgroup Analysis), or load a plot recipe.</div>"))
                return
            if state['recipe_name'] is None and results_tokens() != state['token']:
                changed = ('Step 9 (Subgroup Analysis)' if results_tokens()[0] != state['token'][0]
                           else 'Step 7 (Overall Meta-Analysis)')
                display(HTML(stale_warning_html(changed)))
            s = widget_values(C)
            try:
                fig = draw(state['data'], s)
                paths = cometa_save_figure(fig, s['filename'], pdf=s['save_pdf'], png=s['save_png'],
                                           svg=s['save_svg'], dpi=s['dpi'], transparent=s['transparent'])
                show_figure(fig)
            except Exception as e:
                plt.close('all')
                display(HTML(f"<div style='color:#721c24;'>❌ Plotting error: {html.escape(str(e))}</div>"))
                return
            display(HTML(saved_files_html(paths)))
            cap = caption_html(caption(state['data'], s))
            display(HTML(cap))
        with caption_out:
            clear_output()
            display(HTML(cap))

    def get_recipe():
        if state['data'] is None:
            return None
        return build_plot_recipe(PLOT_TYPE, state['data'], widget_values(C), state['source'])

    def on_load(rec, style_only, name):
        if not style_only:
            state.update(data=rec['data'], source=rec.get('source'), recipe_name=name)
            build_label_widgets(state['data'])
            set_data_defaults(state['data'])
            refresh_source()
        settings = dict(rec.get('settings') or {})
        if style_only:
            # texts that name the other recipe's analysis stay as they are
            for key in ('title', 'xlabel', 'ylabel', 'filename'):
                settings.pop(key, None)
        skipped = apply_widget_values(C, settings)
        generate()
        return skipped

    # --- 5. INITIAL STATE AND LAYOUT ---
    try:
        state['data'], state['source'] = build_data()
    except Exception as e:
        state['data'], state['source'] = None, None
        print(f"⚠️ Could not read the Step 7 / Step 9 results: {e}")
    state['token'] = results_tokens()
    build_label_widgets(state['data'])
    if state['data'] is not None:
        set_data_defaults(state['data'])
    else:
        group_box.layout.display = 'none'
    refresh_source()

    preset = preset_dropdown({
        'width': C['width'], 'height': PresetTarget(preset_height), 'title': C['title_font'],
        'label': PresetTarget(lambda v: (set_widget(C['label_font'], v), set_widget(C['group_font'], v))),
        'tick': PresetTarget(lambda v: (set_widget(C['tick_font'], v), set_widget(C['annot_font'], v))),
    })
    H = lambda t: widgets.HTML(f"<h3 style='color: #2E86AB;'>{t}</h3>")
    hr = lambda: widgets.HTML("<hr style='margin: 10px 0;'>")
    tabs = widgets.Tab(children=[
        widgets.VBox([H('Plot Style & Layout'), preset,
                      widgets.HTML("<span style='color:#666; font-size:12px;'>A preset sets the width and font sizes; "
                                   "its total height is converted to height per row.</span>"),
                      hr(), widgets.HTML("<b>Dimensions:</b>"), C['width'], C['height'],
                      hr(), widgets.HTML("<b>Typography:</b>"), C['title_font'], C['label_font'], C['tick_font'],
                      C['annot_font'], hr(), widgets.HTML("<b>Visual Style:</b>"), C['color_scheme'],
                      C['marker_style'], C['ci_style']]),
        widgets.VBox([H('Text & Labels'), C['show_title'], C['title'], hr(), C['xlabel'], C['show_ylabel'],
                      C['ylabel']]),
        widgets.VBox([H('Annotations'), widgets.HTML("<b>Show in Annotations:</b>"), C['show_k'],
                      C['show_papers'], C['show_fold_change'], fold_note, hr(), widgets.HTML("<b>Position:</b>"),
                      C['annot_pos'], C['annot_offset'], group_box]),
        widgets.VBox([H('Axes & Scaling'), C['auto_scale'], manual_scale_box, hr(),
                      widgets.HTML("<b>Grid & Reference Lines:</b>"), C['show_grid'], C['grid_style'],
                      C['show_null'], hr(), C['show_fold_axis']]),
        widgets.VBox([H('Export Options'), C['save_pdf'], C['save_png'], C['save_svg'], C['dpi'], hr(),
                      C['filename'], C['transparent']]),
        widgets.VBox([H('Label Editor'),
                      widgets.HTML("<p style='color: #666;'><i>Edit the text on the right to change how a group "
                                   "appears in the plot; the original coded names are on the left. The labels are "
                                   "saved in the plot recipe.</i></p>"), hr(), label_box]),
        widgets.VBox([H('📝 Figure Caption'),
                      widgets.HTML("<p style='color: #666; font-size: 13px;'><i>Generated from the plot settings "
                                   "each time the plot is drawn. Select the text, copy, and paste it into your "
                                   "manuscript's figure legend.</i></p>"), caption_out]),
    ])
    for i, t in enumerate(['🎨 Style', '📝 Text', '🏷️ Annotations', '📏 Axes', '💾 Export', '✏️ Labels', '📝 Caption']):
        tabs.set_title(i, t)

    run_btn = widgets.Button(description='📊 Generate Forest Plot', button_style='success',
                             layout=widgets.Layout(width='450px', height='50px'),
                             style={'font_weight': 'bold', 'font_size': '14px'})
    run_btn.on_click(generate)
    recipe_bar = plot_recipe_bar(PLOT_TYPE, get_recipe, on_load, lambda: C['filename'].value)

    display(widgets.VBox([
        widgets.HTML("<h3 style='color: #2E86AB; border-bottom: 2px solid #3498db; padding-bottom: 5px; "
                     "font-family: sans-serif;'>Step 10: Publication-Ready Forest Plot</h3>"),
        widgets.HTML("""
        <div style='background-color: #e8f4f8; border-left: 4px solid #17a2b8; padding: 12px; margin-bottom: 15px; border-radius: 4px; color: #0c5460; font-family: sans-serif;'>
            <b>💡 Plotting Tips:</b>
            <ul style='margin: 5px 0 0 0; padding-left: 20px; font-size: 13px;'>
                <li>Use the <b>Labels</b> tab to rename coded variables into presentation-ready text.</li>
                <li>Auto-scaling considers all data points for proper spacing, but you can override it in the <b>Axes</b> tab.</li>
                <li>If you are running a two-way subgroup analysis, group labels will automatically align on the right side of the plot.</li>
            </ul>
        </div>
        """),
        source_box, widgets.HTML("<hr>"), tabs, widgets.HTML("<hr style='margin: 15px 0;'>"), run_btn, recipe_bar,
        out,
    ]))


_forest_plot_cell()


In [ ]:
#@title 🍎 11. Subgroup Analysis: Visualization - Orchard Plot
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# STEP 11: PUBLICATION-READY ORCHARD PLOT
# Purpose: Show every effect size (leaves) with the pooled estimate, its
#          confidence interval and prediction interval for each subgroup
#          and for the overall model.
# Data:    Step 9 (Subgroup Analysis), Step 7 (Overall Meta-Analysis) and
#          the effect sizes, or a plot recipe file saved from this cell.
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from matplotlib import font_manager
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import html
from scipy.stats import t as t_dist, norm


def _orchard_cell():
    PLOT_TYPE = 'orchard'
    RESULTS_KEY = 'subgroup_results'
    state = {'data': None, 'source': None, 'recipe_name': None, 'token': None}

    # --- 1. DATA SNAPSHOT (everything the figure draws) ---
    def build_data():
        AC = globals().get('ANALYSIS_CONFIG') or {}
        overall = AC.get('overall_results')
        if not overall:
            return None, None
        es = AC.get('es_config') or {}
        gs = AC.get('global_settings') or {}
        alpha = float(gs.get('alpha', 0.05))
        dist_type = gs.get('dist_type', 'norm')

        def crit(n):
            if dist_type == 't':
                return float(t_dist.ppf(1 - alpha / 2, max(1, int(n) - 1)))
            return float(norm.ppf(1 - alpha / 2))

        def intervals(est, se, tau2, sigma2, n, ci_lo=None, ci_hi=None, pi_lo=None, pi_hi=None):
            # stored intervals from the analysis cells first; computed ones only when missing
            c = crit(n)
            pi_sd = np.sqrt(se ** 2 + (tau2 or 0) + (sigma2 or 0))
            pick = lambda v, alt: float(v) if v is not None and pd.notna(v) else float(alt)
            return (pick(ci_lo, est - c * se), pick(ci_hi, est + c * se),
                    pick(pi_lo, est - c * pi_sd), pick(pi_hi, est + c * pi_sd))

        k_all = int(overall.get('k', 0))
        rep = overall.get('reported')
        if rep:
            est, se = float(rep['mu']), float(rep['se'])
            iv = intervals(est, se, rep.get('tau_squared'), rep.get('sigma_squared'), overall.get('k_papers', k_all),
                           rep.get('ci_lower'), rep.get('ci_upper'), rep.get('pi_lower'), rep.get('pi_upper'))
            overall_model = rep.get('model', '')
        else:  # results saved by an older version of Step 7
            est = float(overall['pooled_effect_random'])
            se = float(overall.get('pooled_SE_random_reported', 0.1))
            iv = intervals(est, se, overall.get('tau_squared', 0), overall.get('sigma_squared', 0),
                           overall.get('k_papers', k_all))
            overall_model = ''
        rows = [{'key': 'Overall', 'group': 'Overall', 'm1': 'Overall', 'm2': 'Overall', 'is_overall': True,
                 'k': k_all, 'n_papers': int(overall.get('k_papers', k_all)), 'est': est,
                 'ci_lo': iv[0], 'ci_hi': iv[1], 'pi_lo': iv[2], 'pi_hi': iv[3]}]

        sg = AC.get(RESULTS_KEY) or {}
        res = sg.get('results_df')
        has_sg = isinstance(res, pd.DataFrame) and len(res) > 0
        atype = sg.get('analysis_type', 'single') if has_sg else 'none'
        mod1 = sg.get('moderator1') if has_sg else None
        mod2 = sg.get('moderator2') if atype == 'two_way' else None
        if atype == 'two_way' and not mod2:
            atype = 'single'
        subgroup_model = ''
        if has_sg:
            if 'model_type' in res.columns and len(res):
                subgroup_model = str(res['model_type'].iloc[0])
            sub = []
            for _, r in res.iterrows():
                g = str(r['group']).strip()
                if atype == 'two_way':
                    m1, m2 = r.get('moderator1_value'), r.get('moderator2_value')
                    if m1 is None or pd.isna(m1) or m2 is None or pd.isna(m2):
                        parts = g.split(' | ', 1) if ' | ' in g else g.split(' × ', 1)
                        m1, m2 = (parts + ['?'])[:2]
                    m1, m2 = str(m1).strip(), str(m2).strip()
                    key = f"{m1} | {m2}"   # the label Step 9 builds for each cell
                else:
                    m1, m2, key = '', g, g
                e, s_e = float(r['pooled_effect_re']), float(r.get('pooled_se_re', np.nan))
                iv = intervals(e, s_e, r.get('tau_squared', 0), r.get('sigma_squared', 0),
                               r.get('n_papers', r.get('k', 2)), r.get('ci_lower_re'), r.get('ci_upper_re'),
                               r.get('pi_lower_re'), r.get('pi_upper_re'))
                sub.append({'key': key, 'group': g, 'm1': m1, 'm2': m2, 'is_overall': False,
                            'k': int(r.get('k', 0)), 'n_papers': int(r.get('n_papers', r.get('k', 0))),
                            'est': e, 'ci_lo': iv[0], 'ci_hi': iv[1], 'pi_lo': iv[2], 'pi_hi': iv[3]})
            sub.sort(key=(lambda d: (d['m1'], d['m2'])) if atype == 'two_way' else (lambda d: d['group']))
            rows += sub

        # raw effect sizes, each with the group key Step 9 would give it
        effect_col, var_col = es.get('effect_col', 'hedges_g'), es.get('var_col', 'Vg')
        ad = AC.get('analysis_data')
        if isinstance(ad, pd.DataFrame) and effect_col in ad.columns and var_col in ad.columns:
            ad = ad.dropna(subset=[effect_col, var_col])
            strip = lambda c: ad[c].astype(str).str.strip() if c in ad.columns else pd.Series('', index=ad.index)
            if atype == 'two_way':
                key = strip(mod1) + ' | ' + strip(mod2)
            elif atype == 'single':
                key = strip(mod1)
            else:
                key = pd.Series('Overall', index=ad.index)
            points = pd.DataFrame({'key': key.astype(str).tolist(),
                                   'yi': ad[effect_col].astype(float).tolist(),
                                   'vi': ad[var_col].astype(float).tolist()})
        else:
            points = pd.DataFrame({'key': [], 'yi': [], 'vi': []})

        ci_level = ci_pct_label(alpha)
        data = {
            'rows': pd.DataFrame(rows),
            'points': points,
            'analysis_type': atype,
            'moderator1': mod1,
            'moderator2': mod2,
            'effect_label': es.get('effect_label', 'Effect Size'),
            'null_value': es_null_value(),
            'ci_level': ci_level,
            'subgroup_model': subgroup_model,
            'overall_model': overall_model,
        }
        if atype == 'two_way':
            analysis = f"Subgroups by {mod1} × {mod2} (Step 9)"
        elif atype == 'single':
            analysis = f"Subgroups by {mod1} (Step 9)"
        else:
            analysis = "Overall effect only (Step 7; run Step 9 for subgroups)"
        when = sg.get('timestamp')
        source = {
            'analysis': analysis,
            'moderator1': mod1, 'moderator2': mod2,
            'effect_label': data['effect_label'],
            'model': subgroup_model or (f"{overall_model} (overall)" if overall_model else ''),
            'ci_level': ci_level,
            'analysis_time': when.strftime('%Y-%m-%d %H:%M') if hasattr(when, 'strftime') else when,
        }
        return data, source

    # --- 2. CONTROLS (every widget whose value is saved in a recipe) ---
    _w, _sw = '480px', '150px'
    st = {'description_width': _sw}
    L = lambda w=_w: widgets.Layout(width=w)
    FS = lambda v, a, b, step, desc, **kw: widgets.FloatSlider(value=v, min=a, max=b, step=step, description=desc,
                                                               continuous_update=False, style=st, layout=L(), **kw)
    IS = lambda v, a, b, desc, step=1: widgets.IntSlider(value=v, min=a, max=b, step=step, description=desc,
                                                         continuous_update=False, style=st, layout=L())
    CB = lambda v, desc: widgets.Checkbox(value=v, description=desc, indent=False, layout=L())
    TX = lambda v, desc: widgets.Text(value=v, description=desc, style=st, layout=L())
    DD = lambda opts, v, desc: widgets.Dropdown(options=opts, value=v, description=desc, style=st, layout=L())
    C = {
        # style (orientation comes before the axis labels: loading a recipe sets it first)
        'width': FS(10.0, 2.0, 20.0, 0.1, 'Plot Width (in):'),
        'height': FS(6.0, 2.0, 30.0, 0.1, 'Plot Height (in):'),
        'font': DD([('DejaVu Sans', 'DejaVu Sans'), ('Arial', 'Arial'), ('Times New Roman', 'Times New Roman'),
                    ('Courier New', 'Courier New'), ('Georgia', 'Georgia')], 'DejaVu Sans', 'Font Family:'),
        'color_scheme': DD([('Viridis', 'viridis'), ('Plasma', 'plasma'), ('Coolwarm', 'coolwarm'),
                            ('Grayscale', 'gray'), ('Tab10', 'tab10'), ('Set2', 'Set2'), ('Dark2', 'Dark2'),
                            ('B&W Only', 'bw')], 'viridis', 'Color Scheme:'),
        'marker_style': DD([('Circle/Diamond', 'circle_diamond'), ('Square/Diamond', 'square_diamond'),
                            ('Circle/Star', 'circle_star'), ('Circle/Circle', 'circle_circle')],
                           'circle_diamond', 'Marker Style:'),
        'overall_color': DD([('Black', 'black'), ('Dark Red', '#8B0000'), ('Dark Blue', '#00008B'),
                             ('Dark Green', '#006400')], 'black', 'Overall Color:'),
        'orientation': DD([('Horizontal', 'h'), ('Vertical', 'v')], 'h', 'Orientation:'),
        # text
        'show_title': CB(True, 'Show Title'),
        'title': TX('Orchard Plot', 'Title:'),
        'title_fs': IS(14, 6, 28, 'Title Size:'),
        'title_bold': CB(True, 'Title Bold'),
        'show_subtitle': CB(False, 'Show Subtitle'),
        'subtitle': TX('', 'Subtitle:'),
        'subtitle_fs': IS(11, 6, 20, 'Subtitle Size:'),
        'xlabel': TX('Effect Size', 'X-Axis Label:'),
        'xlabel_bold': CB(True, 'X Bold'),
        'show_ylabel': CB(True, 'Show Y Label'),
        'ylabel': TX('Groups', 'Y-Axis Label:'),
        'ylabel_bold': CB(True, 'Y Bold'),
        'label_fs': IS(12, 6, 20, 'Axis Label Size:'),
        'tick_fs': IS(10, 5, 18, 'Tick Size:'),
        'show_footnote': CB(False, 'Show Footnote'),
        'footnote': TX('', 'Footnote:'),
        'footnote_fs': IS(8, 5, 14, 'Footnote Size:'),
        # orchard elements
        'pt_alpha': FS(0.5, 0.05, 1.0, 0.05, 'Point Alpha:'),
        'pt_size': IS(50, 10, 250, 'Point Size:', step=5),
        'jitter': FS(0.10, 0.01, 0.5, 0.01, 'Jitter:'),
        'scale_pts': CB(True, 'Scale by Weight (precision, 1/variance)'),
        'show_pi': CB(True, 'Show PI (thin line)'),
        'pi_lw': FS(1.5, 0.5, 10.0, 0.5, 'PI Width:'),
        'pi_alpha': FS(0.6, 0.05, 1.0, 0.05, 'PI Alpha:'),
        'show_ci': CB(True, 'Show CI (thick line)'),
        'ci_lw': FS(4.0, 0.5, 10.0, 0.5, 'CI Width:'),
        'ci_style': DD([('Solid', 'solid'), ('Dashed', 'dashed'), ('Caps', 'caps')], 'solid', 'CI Style:'),
        'ci_color': DD([('Black', 'black'), ('Match Color', 'match'), ('Dark Gray', '#404040')], 'black',
                       'CI Color:'),
        'fruit_size': IS(12, 4, 30, 'Fruit Size (pt):'),
        'overall_scale': FS(1.4, 1.0, 2.5, 0.1, 'Overall Scale:'),
        'fruit_ew': FS(1.5, 0.0, 3.0, 0.25, 'Fruit Edge W:'),
        'fruit_ec': DD([('Black', 'black'), ('Gray', '#404040'), ('None', 'none'), ('Match Fill', 'match')],
                       'black', 'Fruit Edge C:'),
        # annotations
        'annot_fs': IS(9, 5, 16, 'Annot. Size:'),
        'show_k': CB(True, 'Show k'),
        'k_prefix': TX('k=', 'k Prefix:'),
        'show_papers': CB(True, 'Show Papers'),
        'papers_prefix': TX('(', 'Papers Prefix:'),
        'papers_suffix': TX(')', 'Papers Suffix:'),
        'show_es': CB(False, 'Show ES Value'),
        'es_prefix': TX('ES=', 'ES Prefix:'),
        'es_dec': IS(2, 1, 4, 'ES Decimals:'),
        'show_ci_text': CB(False, 'Show CI in Text'),
        'ci_fmt': TX('[{lo}, {hi}]', 'CI Format:'),
        'ci_dec': IS(2, 1, 4, 'CI Decimals:'),
        'annot_pos': DD([('Right (plot edge; top when vertical)', 'right'), ('Above marker', 'above'),
                         ('Below marker', 'below')], 'right', 'Position:'),
        'annot_ho': FS(0.0, -2.0, 2.0, 0.05, 'H-Offset:', readout_format='.2f'),
        'annot_vo': FS(0.0, -2.0, 2.0, 0.05, 'V-Offset:', readout_format='.2f'),
        'annot_bold_overall': CB(True, 'Bold Overall'),
        # axes & grid
        'auto_scale': CB(True, 'Auto-Scale'),
        'ax_min': widgets.FloatText(value=-2.0, description='Min:', style={'description_width': '50px'},
                                    layout=widgets.Layout(width='200px', visibility='hidden')),
        'ax_max': widgets.FloatText(value=2.0, description='Max:', style={'description_width': '50px'},
                                    layout=widgets.Layout(width='200px', visibility='hidden')),
        'show_grid': CB(True, 'Show Grid'),
        'grid_style': DD([('Dashed', 'dashed'), ('Dotted', 'dotted'), ('Solid', 'solid')], 'dashed', 'Grid Style:'),
        'grid_alpha': FS(0.3, 0.05, 1.0, 0.05, 'Grid Alpha:'),
        'show_null': CB(True, 'Show Null Line'),
        'null_value': widgets.FloatText(value=0.0, description='Null Value:', style=st, layout=L()),
        'null_style': DD([('Solid', 'solid'), ('Dashed', 'dashed'), ('Dotted', 'dotted')], 'solid', 'Null Style:'),
        'null_lw': FS(1.5, 0.5, 4.0, 0.25, 'Null Width:'),
        'null_color': DD([('Black', 'black'), ('Gray', 'gray'), ('Red', 'red')], 'black', 'Null Color:'),
        'show_sep': CB(True, 'Show Separators'),
        'sep_lw': FS(0.8, 0.3, 2.5, 0.1, 'Sep. Width:'),
        'sep_style': DD([('Solid', 'solid'), ('Dashed', 'dashed'), ('Dotted', 'dotted')], 'solid', 'Sep. Style:'),
        'sep_color': DD([('Black', 'black'), ('Gray', 'gray'), ('Light Gray', '#CCC')], 'black', 'Sep. Color:'),
        'group_gap': FS(1.0, 0.0, 3.0, 0.25, 'Group Gap:'),
        'show_inner': CB(True, 'Show Inner Labels'),
        'grp_position': DD([('Right side (inside plot)', 'right_inside'), ('Left side (outside axis)', 'left_outside'),
                            ('Right side (outside axis)', 'right_outside')], 'right_inside', 'Label Place:'),
        'grp_ho': FS(0.0, -2.0, 2.0, 0.1, 'Grp H-Offset:'),
        'grp_vo': FS(0.0, -2.0, 2.0, 0.1, 'Grp V-Offset:'),
        'grp_fs': IS(12, 6, 20, 'Grp Label Size:'),
        'grp_rotation': IS(0, -90, 90, 'Rotation (°):', step=5),
        'grp_bold': CB(True, 'Bold Labels'),
        'grp_color': DD([('Black', 'black'), ('Dark Gray', '#404040'), ('Match Group', 'match'),
                         ('Dark Blue', '#00008B')], 'black', 'Label Color:'),
        'grp_bg': CB(False, 'Label Background Box'),
        'grp_bg_color': DD([('White', 'white'), ('Light Yellow', '#FFFFDD'), ('Light Gray', '#F0F0F0'),
                            ('Transparent', 'none')], 'white', 'BG Color:'),
        'grp_bg_alpha': FS(0.8, 0.1, 1.0, 0.1, 'BG Alpha:'),
        # export
        'save_pdf': CB(True, 'Save PDF'),
        'save_png': CB(True, 'Save PNG'),
        'save_svg': CB(False, 'Save SVG'),
        'dpi': IS(300, 100, 600, 'PNG DPI:', step=50),
        'filename': TX('OrchardPlot', 'Filename:'),
        'transparent': CB(False, 'Transparent BG'),
        'timestamp': CB(True, 'Add Timestamp'),
    }

    def _toggle_scale(change):
        v = 'hidden' if change['new'] else 'visible'
        C['ax_min'].layout.visibility = v
        C['ax_max'].layout.visibility = v
    C['auto_scale'].observe(_toggle_scale, names='value')

    def _on_orientation(change):
        # the effect axis moves from x to y (or back): the axis labels follow it
        for a, b in (('xlabel', 'ylabel'), ('xlabel_bold', 'ylabel_bold')):
            C[a].value, C[b].value = C[b].value, C[a].value
    C['orientation'].observe(_on_orientation, names='value')

    # Label editor: one Text per group label, rebuilt from the data snapshot
    LBL = {}
    label_box = widgets.VBox()

    def build_label_widgets(data):
        LBL.clear()
        items = [widgets.HTML("<h3 style='color:#2E86AB'>✏️ Label Editor</h3>"),
                 widgets.HTML("<p style='color:#666'><i>Edit right side to change how labels appear in the plot</i></p>"),
                 widgets.HTML("<hr>")]

        def add(key, text, desc):
            LBL[key] = widgets.Text(value=str(text), description=desc, layout=widgets.Layout(width='500px'),
                                    style={'description_width': '200px'})
            items.append(LBL[key])

        if data is not None:
            add('Overall', 'Overall', 'Overall Effect:')
            sub = data['rows'][~data['rows']['is_overall'].astype(bool)]
            if data['analysis_type'] == 'two_way':
                items.append(widgets.HTML(f"<b>{html.escape(str(data['moderator1']))}</b> (Moderator 1: group labels)"))
                for v in dict.fromkeys(sub['m1']):
                    add(f"m1::{v}", v, f"{v}:")
                items.append(widgets.HTML(f"<b>{html.escape(str(data['moderator2']))}</b> (Moderator 2: row labels)"))
                for v in sorted(set(sub['m2'])):
                    add(f"m2::{v}", v, f"{v}:")
            else:
                for v in sub['group']:
                    add(f"grp::{v}", v, f"{v}:")
        else:
            items.append(widgets.HTML("<p style='color:#666'>No groups yet.</p>"))
        label_box.children = items

    def controls():
        return {**C, **{f'label::{k}': w for k, w in LBL.items()}}

    def set_data_defaults(data):
        """Defaults that depend on the data (called for session data and recipes)."""
        m1, m2, at = data['moderator1'], data['moderator2'], data['analysis_type']
        if at == 'two_way':
            C['title'].value, group_lab = f"Orchard Plot: {m1} × {m2}", str(m2)
            C['filename'].value = f"OrchardPlot_{m1}_x_{m2}"
        elif at == 'single':
            C['title'].value, group_lab = f"Orchard Plot: {m1}", str(m1)
            C['filename'].value = f"OrchardPlot_{m1}"
        else:
            C['title'].value, group_lab = "Orchard Plot: Overall Effect", "Groups"
            C['filename'].value = "OrchardPlot_overall"
        effect_lab = str(data['effect_label'])
        if C['orientation'].value == 'h':
            C['xlabel'].value, C['ylabel'].value = effect_lab, group_lab
        else:
            C['xlabel'].value, C['ylabel'].value = group_lab, effect_lab
        C['height'].value = min(C['height'].max, max(6.0, round(len(data['rows']) * 0.6, 1)))
        C['null_value'].value = float(data['null_value'])

    # --- 3. DRAWING (data + settings -> figure; no widget access) ---
    def resolve_font(name):
        # a font that is not installed falls back to the matching generic family
        if name in {f.name for f in font_manager.fontManager.ttflist}:
            return name
        return {'Times New Roman': 'serif', 'Georgia': 'serif', 'Courier New': 'monospace'}.get(name, 'sans-serif')

    def palette(scheme, n):
        if n <= 0:
            return []
        if scheme == 'bw':
            return [(0.0, 0.0, 0.0, 1.0)] * n
        cmap = plt.get_cmap(scheme)
        if scheme in ('tab10', 'Set2', 'Dark2'):
            return [cmap(i % cmap.N) for i in range(n)]
        lo, hi = {'gray': (0.0, 0.6), 'viridis': (0.0, 0.85), 'plasma': (0.0, 0.8)}.get(scheme, (0.0, 1.0))
        pos = np.linspace(lo, hi, n) if n > 1 else np.array([lo])
        if scheme == 'coolwarm':  # skip the pale middle of the map
            pos = np.where(pos < 0.5, pos * 0.7, 0.3 + pos * 0.7)
        return [cmap(float(p)) for p in pos]

    def annotation_text(r, s, stacked):
        """k and papers on one line; estimate and CI follow (on new lines when stacked)."""
        first, more = [], []
        if s['show_k']:
            first.append(f"{s['k_prefix']}{int(r['k'])}")
        if s['show_papers']:
            first.append(f"{s['papers_prefix']}{int(r['n_papers'])}{s['papers_suffix']}")
        if s['show_es']:
            more.append(f"{s['es_prefix']}{r['est']:.{int(s['es_dec'])}f}")
        if s['show_ci_text']:
            d = int(s['ci_dec'])
            more.append(str(s['ci_fmt']).replace('{lo}', f"{r['ci_lo']:.{d}f}").replace('{hi}', f"{r['ci_hi']:.{d}f}"))
        lines = ([' '.join(first)] if first else []) + more
        return ('\n' if stacked else ' ').join(lines)

    def draw(data, s):
        rows = data['rows'].reset_index(drop=True)
        pts = data['points']
        two = data['analysis_type'] == 'two_way'
        is_h = s['orientation'] == 'h'
        lab = lambda key, default: str(s.get(f'label::{key}', default))
        ov_label = lab('Overall', 'Overall')

        # blocks (Overall, then one block per Moderator 1 value or one block of subgroups) and positions
        blocks = []
        for i, r in rows.iterrows():
            bk = '\0overall' if r['is_overall'] else (r['m1'] if two else '\0groups')
            if not blocks or blocks[-1][0] != bk:
                blocks.append((bk, []))
            blocks[-1][1].append(i)
        gap = float(s['group_gap'])
        pos, seps, cursor = {}, [], 0.0
        for bi, (_, idxs) in enumerate(blocks):
            if bi > 0:
                seps.append(cursor - 1 + (1 + gap) / 2)
                cursor += gap
            for i in idxs:
                pos[i] = cursor
                cursor += 1
        top = cursor - 1
        cat = (lambda p: top - p) if is_h else (lambda p: p)

        # colours by group order (by Moderator 1 block in a two-way analysis)
        sub = rows[~rows['is_overall'].astype(bool)]
        col_keys = list(dict.fromkeys(sub['m1'])) if two else list(sub['key'])
        cols = dict(zip(col_keys, palette(s['color_scheme'], len(col_keys))))
        color_of = lambda r: s['overall_color'] if r['is_overall'] else cols[r['m1'] if two else r['key']]
        sub_mk, ov_mk = {'circle_diamond': ('o', 'D'), 'square_diamond': ('s', 'D'), 'circle_star': ('o', '*'),
                         'circle_circle': ('o', 'o')}.get(s['marker_style'], ('o', 'D'))
        lsty = {'solid': '-', 'dashed': '--', 'dotted': ':'}

        font = resolve_font(s['font'])
        fig, ax = plt.subplots(figsize=(s['width'], s['height']))
        if s['transparent']:
            fig.patch.set_alpha(0)
            ax.patch.set_alpha(0)

        def seg(lo, hi, c, **kw):
            if is_h:
                ax.plot([lo, hi], [c, c], **kw)
            else:
                ax.plot([c, c], [lo, hi], **kw)

        w_all = 1.0 / pts['vi'].astype(float).clip(lower=1e-10) if len(pts) else pd.Series(dtype=float)
        w_mean = float(w_all.mean()) if len(w_all) else 0.0
        for i, r in rows.iterrows():
            c = cat(pos[i])
            is_ov = bool(r['is_overall'])
            col = color_of(r)
            # prediction interval (thin line)
            if s['show_pi']:
                seg(r['pi_lo'], r['pi_hi'], c, color=col, alpha=1.0 if is_ov else s['pi_alpha'],
                    lw=s['pi_lw'] * (1.2 if is_ov else 1.0), solid_capstyle='round', zorder=1)
            # leaves (effect sizes)
            mask = np.ones(len(pts), bool) if is_ov else (pts['key'] == r['key']).to_numpy()
            if mask.any():
                yi = pts['yi'].to_numpy(float)[mask]
                rng = np.random.default_rng(42 + i)
                jit = rng.uniform(-s['jitter'], s['jitter'], size=len(yi))
                if s['scale_pts'] and w_mean > 1e-10:
                    sizes = np.clip(s['pt_size'] * w_all.to_numpy()[mask] / w_mean,
                                    min(10, s['pt_size']), s['pt_size'] * 3)
                else:
                    sizes = np.full(len(yi), float(s['pt_size']))
                pc = 'gray' if is_ov else col
                pa = s['pt_alpha'] * 0.5 if is_ov else s['pt_alpha']
                xy = (yi, c + jit) if is_h else (c + jit, yi)
                ax.scatter(*xy, s=sizes, color=pc, alpha=pa, edgecolor='none', zorder=2)
            # confidence interval (thick line)
            if s['show_ci']:
                ci_c = col if s['ci_color'] == 'match' else s['ci_color']
                clw = s['ci_lw'] * (1.2 if is_ov else 1.0)
                seg(r['ci_lo'], r['ci_hi'], c, color=ci_c, lw=clw, ls='--' if s['ci_style'] == 'dashed' else '-',
                    zorder=3)
                if s['ci_style'] == 'caps':
                    for e in (r['ci_lo'], r['ci_hi']):
                        if is_h:
                            ax.plot([e, e], [c - 0.1, c + 0.1], color=ci_c, lw=clw, zorder=3)
                        else:
                            ax.plot([c - 0.1, c + 0.1], [e, e], color=ci_c, lw=clw, zorder=3)
            # fruit (pooled estimate)
            ec = col if s['fruit_ec'] == 'match' else s['fruit_ec']
            xy = (r['est'], c) if is_h else (c, r['est'])
            ax.plot(*xy, marker=ov_mk if is_ov else sub_mk,
                    markersize=s['fruit_size'] * (s['overall_scale'] if is_ov else 1.0),
                    color=col, markeredgecolor=ec, markeredgewidth=s['fruit_ew'], zorder=4, linestyle='none')

        # group axis ticks
        def tick_label(r):
            if r['is_overall']:
                return ov_label
            if not two:
                return lab(f"grp::{r['group']}", r['group'])
            m2 = lab(f"m2::{r['m2']}", r['m2'])
            return m2 if s['show_inner'] else f"{lab('m1::' + r['m1'], r['m1'])}: {m2}"
        order = sorted(rows.index, key=lambda i: cat(pos[i]))   # ascending, as despine(trim) expects
        ticks = [cat(pos[i]) for i in order]
        tlabels = [tick_label(rows.loc[i]) for i in order]
        if is_h:
            ax.set_yticks(ticks)
            ax.set_yticklabels(tlabels)
        else:
            ax.set_xticks(ticks)
            ax.set_xticklabels(tlabels, rotation=45, ha='right')
        ax.tick_params(labelsize=s['tick_fs'])
        cat_ticklabels = ax.get_yticklabels() if is_h else ax.get_xticklabels()
        for i, tl in zip(order, cat_ticklabels):
            if rows.loc[i, 'is_overall']:
                tl.set_fontweight('bold')

        # reference lines, grid, scale
        if s['show_null']:
            (ax.axvline if is_h else ax.axhline)(s['null_value'], color=s['null_color'], lw=s['null_lw'],
                                                 ls=lsty[s['null_style']], zorder=0.5)
        if s['show_sep']:
            for sp in seps:
                (ax.axhline if is_h else ax.axvline)(cat(sp), color=s['sep_color'], lw=s['sep_lw'],
                                                     ls=lsty[s['sep_style']], zorder=0.4)
        if s['show_grid']:
            ax.grid(axis='x' if is_h else 'y', ls=lsty[s['grid_style']], alpha=s['grid_alpha'])
        if not s['auto_scale']:
            (ax.set_xlim if is_h else ax.set_ylim)(s['ax_min'], s['ax_max'])
        margin = max(0.6, s['jitter'] + 0.15)
        (ax.set_ylim if is_h else ax.set_xlim)(-margin, top + margin)
        sns.despine(ax=ax, trim=True)

        renderer = fig.canvas.get_renderer()
        to_pt = 72.0 / fig.dpi

        def plain_texts():
            for txt in fig.findobj(mtext.Text):
                txt.set_fontfamily(font)
                txt.set_parse_math(False)   # labels with _ % $ are shown as typed

        def extent(artists):
            boxes = [a.get_window_extent(renderer) for a in artists if a.get_visible() and a.get_text()]
            if not boxes:
                return None
            return (min(b.x0 for b in boxes), min(b.y0 for b in boxes),
                    max(b.x1 for b in boxes), max(b.y1 for b in boxes))

        plain_texts()
        fig.canvas.draw()
        axb = ax.get_window_extent(renderer)
        tick_ext = extent(cat_ticklabels)

        # annotations (k, papers, estimate, CI)
        afs = s['annot_fs']
        ao = (s['annot_ho'] * afs, s['annot_vo'] * afs)
        annots = []
        for i, r in rows.iterrows():
            txt = annotation_text(r, s, stacked=not is_h)
            if not txt:
                continue
            c = cat(pos[i])
            fw = 'bold' if (r['is_overall'] and s['annot_bold_overall']) else 'normal'
            half = s['fruit_size'] * (s['overall_scale'] if r['is_overall'] else 1.0) / 2 + 2
            if s['annot_pos'] == 'right':
                if is_h:
                    kw = dict(xy=(1, c), xycoords=ax.get_yaxis_transform(), xytext=(6 + ao[0], ao[1]),
                              ha='left', va='center')
                else:
                    kw = dict(xy=(c, 1), xycoords=ax.get_xaxis_transform(), xytext=(ao[0], 4 + ao[1]),
                              ha='center', va='bottom')
            else:
                up = s['annot_pos'] == 'above'
                if is_h:
                    kw = dict(xy=(r['est'], c), xycoords='data',
                              xytext=(ao[0], (half if up else -half) + ao[1]),
                              ha='center', va='bottom' if up else 'top')
                else:
                    ends = [r['est']] + ([r['pi_lo'], r['pi_hi']] if s['show_pi'] else []) + \
                           ([r['ci_lo'], r['ci_hi']] if s['show_ci'] else [])
                    kw = dict(xy=(c, max(ends) if up else min(ends)), xycoords='data',
                              xytext=(ao[0], (3 if up else -3) + ao[1]),
                              ha='center', va='bottom' if up else 'top')
            annots.append(ax.annotate(txt, textcoords='offset points', fontsize=afs, fontweight=fw,
                                      annotation_clip=False, zorder=5, **kw))

        plain_texts()
        fig.canvas.draw()
        a_ext = extent(annots) if s['annot_pos'] == 'right' else None
        right_extra = max(0.0, (a_ext[2] - axb.x1) * to_pt) if (a_ext and is_h) else 0.0
        top_extra = max(0.0, (a_ext[3] - axb.y1) * to_pt) if (a_ext and not is_h) else 0.0

        # Moderator 1 group labels (two-way analyses)
        glabels = []
        if two and s['show_inner']:
            gfs = s['grp_fs']
            go = (s['grp_ho'] * gfs, s['grp_vo'] * gfs)
            place = s['grp_position']
            for bk, idxs in blocks:
                if bk.startswith('\0'):
                    continue
                center = float(np.mean([cat(pos[i]) for i in idxs]))
                gc = cols.get(bk, 'black') if s['grp_color'] == 'match' else s['grp_color']
                if is_h:
                    tr = ax.get_yaxis_transform()
                    if place == 'right_inside':
                        kw = dict(xy=(1, center), xytext=(-6 + go[0], go[1]), ha='right', va='center')
                    elif place == 'left_outside':
                        tw = (axb.x0 - tick_ext[0]) * to_pt if tick_ext else 0.0
                        kw = dict(xy=(0, center), xytext=(-(tw + 8) + go[0], go[1]), ha='right', va='center')
                    else:
                        shift = right_extra + 8 if right_extra else 6
                        kw = dict(xy=(1, center), xytext=(shift + go[0], go[1]), ha='left', va='center')
                else:
                    tr = ax.get_xaxis_transform()
                    if place == 'right_inside':
                        kw = dict(xy=(center, 1), xytext=(go[0], -6 + go[1]), ha='center', va='top')
                    elif place == 'left_outside':
                        th = (axb.y0 - tick_ext[1]) * to_pt if tick_ext else 0.0
                        kw = dict(xy=(center, 0), xytext=(go[0], -(th + 8) + go[1]), ha='center', va='top')
                    else:
                        shift = top_extra + 8 if top_extra else 6
                        kw = dict(xy=(center, 1), xytext=(go[0], shift + go[1]), ha='center', va='bottom')
                bbox = (dict(boxstyle='round,pad=0.3', fc=s['grp_bg_color'], ec='none', alpha=s['grp_bg_alpha'])
                        if s['grp_bg'] else None)
                glabels.append(ax.annotate(lab(f"m1::{bk}", bk), xycoords=tr, textcoords='offset points',
                                           fontsize=gfs, fontweight='bold' if s['grp_bold'] else 'normal',
                                           color=gc, rotation=s['grp_rotation'], bbox=bbox,
                                           annotation_clip=False, zorder=6, **kw))

        # axis labels and title, moved out of the way of labels placed outside the axes
        g_ext = None
        if glabels:
            plain_texts()
            fig.canvas.draw()
            g_ext = extent(glabels)
        side_pad = 0.0
        if g_ext and tick_ext and s['grp_position'] == 'left_outside':
            side_pad = max(0.0, ((tick_ext[0] - g_ext[0]) if is_h else (tick_ext[1] - g_ext[1])) * to_pt)
        if g_ext and s['grp_position'] == 'right_outside' and not is_h:
            top_extra = max(top_extra, (g_ext[3] - axb.y1) * to_pt)
        xpad = 4 + (side_pad + 4 if (side_pad and not is_h) else 0)
        ypad = 4 + (side_pad + 4 if (side_pad and is_h) else 0)
        ax.set_xlabel(s['xlabel'], fontweight='bold' if s['xlabel_bold'] else 'normal', fontsize=s['label_fs'],
                      labelpad=xpad)
        if s['show_ylabel']:
            ax.set_ylabel(s['ylabel'], fontweight='bold' if s['ylabel_bold'] else 'normal',
                          fontsize=s['label_fs'], labelpad=ypad)
        sub_h = 0.0
        if s['show_subtitle'] and s['subtitle']:
            ax.annotate(s['subtitle'], xy=(0.5, 1), xycoords='axes fraction', textcoords='offset points',
                        xytext=(0, top_extra + 6), ha='center', va='bottom', fontsize=s['subtitle_fs'],
                        style='italic', color='#555', annotation_clip=False)
            sub_h = s['subtitle_fs'] * 1.3 + 4
        if s['show_title']:
            ax.set_title(s['title'], fontweight='bold' if s['title_bold'] else 'normal', fontsize=s['title_fs'],
                         pad=(15 if not (top_extra or sub_h) else top_extra + sub_h + 10))
        bottom = 0.0
        if s['show_footnote'] and s['footnote']:
            fig.text(0.01, 0.01, s['footnote'], fontsize=s['footnote_fs'], color='#666', ha='left', va='bottom',
                     style='italic')
            bottom = min(0.3, s['footnote_fs'] * 2.0 / (72.0 * s['height']))
        plain_texts()
        fig.tight_layout(rect=(0, bottom, 1, 1))
        return fig

    def caption(data, s):
        e = lambda x: html.escape(str(x))
        ci, at = data['ci_level'], data['analysis_type']
        if at == 'two_way':
            what = (f" by {e(data['moderator1'])} × {e(data['moderator2'])} (rows: {e(data['moderator2'])}, "
                    f"grouped in blocks of {e(data['moderator1'])})")
        elif at == 'single':
            what = f" by {e(data['moderator1'])}"
        else:
            what = ""
        txt = (f"Orchard plot of {e(data['effect_label'])}{what}. Each small point (leaf) is one effect size "
               f"(k = {len(data['points'])} in total), jittered along the group axis to limit overlap; ")
        txt += ("point size is proportional to the precision of the effect size (1/variance)." if s['scale_pts']
                else "all points have the same size.")
        if at != 'none':
            keys = set(data['rows'].loc[~data['rows']['is_overall'].astype(bool), 'key'])
            n_out = int((~data['points']['key'].isin(keys)).sum())
            if n_out:
                txt += (f" {n_out} effect size{'s' if n_out != 1 else ''} from groups not analysed in Step 9 "
                        f"appear{'s' if n_out == 1 else ''} only in Overall.")
        om =f", {e(data['overall_model'])} model" if data.get('overall_model') else ""
        txt += (" The large marker is the pooled estimate" +
                (" of each subgroup (Step 9 subgroup model) and" if at != 'none' else "") +
                f" of all effect sizes (Overall, Step 7{om}).")
        if s['show_ci'] and s['show_pi']:
            txt += (f" The thick line is the {ci} confidence interval of the pooled estimate and the thin line "
                    f"the {ci} prediction interval, the range expected for the true effect of a new study.")
        elif s['show_ci']:
            txt += f" The thick line is the {ci} confidence interval of the pooled estimate."
        elif s['show_pi']:
            txt += (f" The thin line is the {ci} prediction interval, the range expected for the true effect "
                    f"of a new study.")
        items = []
        if s['show_k']:
            items.append("the number of effect sizes (k)")
        if s['show_papers']:
            items.append("the number of papers" + (" (in parentheses)" if s['papers_prefix'] == '(' else ""))
        if s['show_es']:
            items.append("the pooled estimate")
        if s['show_ci_text']:
            items.append(f"its {ci} confidence interval")
        if items:
            listed = items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]
            txt += " Numbers next to each group give " + listed + "."
        if s['show_null']:
            txt += (f" The {'vertical' if s['orientation'] == 'h' else 'horizontal'} line marks no effect "
                    f"({s['null_value']:g}).")
        return txt

    # --- 4. ACTIONS ---
    out = widgets.Output()
    caption_output = widgets.Output()
    source_box = widgets.HTML()

    def refresh_source():
        source_box.value = source_html(state['source'], state['recipe_name'])

    def generate(_b=None):
        with out:
            clear_output(wait=True)
            if state['data'] is None:
                display(HTML("<div style='color:#721c24;'>❌ No results to plot: run Step 7 (Overall "
                             "Meta-Analysis) and Step 9 (Subgroup Analysis), or load a plot recipe.</div>"))
                return
            if state['recipe_name'] is None and results_token(RESULTS_KEY) != state['token']:
                display(HTML(stale_warning_html('Step 9 (Subgroup Analysis)')))
            s = widget_values(controls())
            try:
                with plt.rc_context({'font.family': resolve_font(s['font'])}):
                    fig = draw(state['data'], s)
                    paths = cometa_save_figure(fig, s['filename'], pdf=s['save_pdf'], png=s['save_png'],
                                               svg=s['save_svg'], dpi=s['dpi'], transparent=s['transparent'],
                                               timestamp=s['timestamp'])
                    show_figure(fig)
            except Exception as e:
                plt.close('all')
                display(HTML(f"<div style='color:#721c24;'>❌ Plotting error: {html.escape(str(e))}</div>"))
                return
            display(HTML(saved_files_html(paths)))
            cap = caption_html(caption(state['data'], s))
            display(HTML(cap))
        with caption_output:
            clear_output()
            display(HTML(cap))

    def get_recipe():
        if state['data'] is None:
            return None
        return build_plot_recipe(PLOT_TYPE, state['data'], widget_values(controls()), state['source'])

    def on_load(rec, style_only, name):
        if not style_only:
            state.update(data=rec['data'], source=rec.get('source'), recipe_name=name)
            build_label_widgets(state['data'])
            set_data_defaults(state['data'])
            refresh_source()
        skipped = apply_widget_values(controls(), rec.get('settings'))
        generate()
        return skipped

    # --- 5. INITIAL STATE AND LAYOUT ---
    try:
        state['data'], state['source'] = build_data()
    except Exception as e:
        state['data'], state['source'] = None, None
        print(f"⚠️ Could not read the Step 7 / Step 9 results: {e}")
    state['token'] = results_token(RESULTS_KEY)
    build_label_widgets(state['data'])
    if state['data'] is not None:
        set_data_defaults(state['data'])
    refresh_source()

    preset = preset_dropdown({'width': C['width'], 'height': C['height'], 'title': C['title_fs'],
                              'label': C['label_fs'], 'tick': C['tick_fs']})
    H = lambda t: widgets.HTML(f"<h3 style='color:#2E86AB'>{t}</h3>")
    B = lambda t: widgets.HTML(f"<b>{t}</b>")
    hr = lambda: widgets.HTML("<hr>")
    tab1 = widgets.VBox([H('🎨 Style & Layout'), preset, B('Dimensions:'), C['width'], C['height'],
                         hr(), B('Font:'), C['font'],
                         hr(), B('Colors & Markers:'), C['color_scheme'], C['marker_style'], C['overall_color'],
                         hr(), B('Orientation:'), C['orientation']])
    tab2 = widgets.VBox([H('📝 Text & Labels'), B('Title:'), C['show_title'], C['title'], C['title_fs'],
                         C['title_bold'],
                         hr(), B('Subtitle:'), C['show_subtitle'], C['subtitle'], C['subtitle_fs'],
                         hr(), B('Axes:'), C['xlabel'], C['xlabel_bold'], C['show_ylabel'], C['ylabel'],
                         C['ylabel_bold'], C['label_fs'], C['tick_fs'],
                         hr(), B('Footnote:'), C['show_footnote'], C['footnote'], C['footnote_fs']])
    tab3 = widgets.VBox([H('🍎 Orchard Elements'), B('Leaves (Raw Points):'), C['pt_alpha'], C['pt_size'],
                         C['jitter'], C['scale_pts'],
                         hr(), B('Prediction interval (thin line):'), C['show_pi'], C['pi_lw'], C['pi_alpha'],
                         hr(), B('Confidence interval (thick line):'), C['show_ci'], C['ci_lw'], C['ci_style'],
                         C['ci_color'],
                         hr(), B('Fruit (Marker):'), C['fruit_size'], C['overall_scale'], C['fruit_ew'],
                         C['fruit_ec']])
    tab4 = widgets.VBox([H('💬 Annotations'), C['annot_fs'],
                         hr(), B('k:'), C['show_k'], C['k_prefix'],
                         hr(), B('Papers:'), C['show_papers'], C['papers_prefix'], C['papers_suffix'],
                         hr(), B('Effect Size:'), C['show_es'], C['es_prefix'], C['es_dec'],
                         hr(), B('CI Text:'), C['show_ci_text'], C['ci_fmt'], C['ci_dec'],
                         hr(), B('Position:'), C['annot_pos'],
                         widgets.HTML("<span style='color:#666; font-size:12px;'>Offsets are in units of the "
                                      "annotation font size.</span>"),
                         C['annot_ho'], C['annot_vo'], C['annot_bold_overall']])
    tab5 = widgets.VBox([H('📏 Axes & Grid'), B('Scale:'), C['auto_scale'], widgets.HBox([C['ax_min'], C['ax_max']]),
                         hr(), B('Grid:'), C['show_grid'], C['grid_style'], C['grid_alpha'],
                         hr(), B('Null Line:'), C['show_null'], C['null_value'], C['null_style'], C['null_lw'],
                         C['null_color'],
                         hr(), B('Separators:'), C['show_sep'], C['sep_lw'], C['sep_style'], C['sep_color'],
                         hr(), B('Groups:'),
                         widgets.HTML("<span style='color:#666; font-size:12px;'>Group labels name the Moderator 1 "
                                      "blocks of a two-way analysis (without them each row shows both values). "
                                      "In vertical orientation, left = below and right = above the axis. Offsets "
                                      "are in units of the label font size.</span>"),
                         C['group_gap'], C['show_inner'], C['grp_position'], C['grp_ho'], C['grp_vo'], C['grp_fs'],
                         C['grp_rotation'], C['grp_bold'], C['grp_color'], C['grp_bg'], C['grp_bg_color'],
                         C['grp_bg_alpha']])
    tab6 = widgets.VBox([H('💾 Export'), B('Formats:'), C['save_pdf'], C['save_png'], C['save_svg'], C['dpi'],
                         hr(), B('Settings:'), C['filename'], C['transparent'], C['timestamp']])
    tab8 = widgets.VBox([H('📝 Figure Caption'),
                         widgets.HTML("<p style='color:#666'><i>A dynamically generated, publication-ready figure "
                                      "legend.</i></p>"), caption_output])
    tab = widgets.Tab(children=[tab1, tab2, tab3, tab4, tab5, tab6, label_box, tab8])
    for i, t_name in enumerate(['🎨 Style', '📝 Text', '🍎 Orchard', '💬 Annot.', '📏 Axes', '💾 Export', '✏️ Labels',
                                '📝 Caption']):
        tab.set_title(i, t_name)

    run_btn = widgets.Button(description='🌳 Generate Orchard Plot', button_style='success',
                             layout=widgets.Layout(width='450px', height='50px'), style={'font_weight': 'bold'})
    run_btn.on_click(generate)
    recipe_bar = plot_recipe_bar(PLOT_TYPE, get_recipe, on_load, lambda: C['filename'].value)

    display(widgets.VBox([
        widgets.HTML("<h3 style='color: #2E86AB; border-bottom: 2px solid #3498db; padding-bottom: 5px; "
                     "font-family: sans-serif;'>Step 11: Publication-Ready Orchard Plot</h3>"),
        widgets.HTML("""
        <div style='background-color: #e8f4f8; border-left: 4px solid #17a2b8; padding: 12px; margin-bottom: 15px; border-radius: 4px; color: #0c5460; font-family: sans-serif;'>
            <b>💡 Plotting Tips:</b>
            <ul style='margin: 5px 0 0 0; padding-left: 20px; font-size: 13px;'>
                <li>Orchard plots show every effect size (leaves), sized by its precision (1/variance), with the pooled estimate of each group.</li>
                <li>Use the <b>Labels</b> tab to rename coded variables into presentation-ready text before exporting.</li>
                <li>The thick line is the confidence interval (precision of the mean); the thin line is the prediction interval (where the true effect of a new study is expected to fall).</li>
            </ul>
        </div>
        """),
        source_box, tab,
        widgets.HTML("<hr style='margin: 15px 0;'>"),
        run_btn, recipe_bar, out,
    ]))


_orchard_cell()


In [ ]:
#@title 📈 12. Meta-Regression
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# DATA LAYER - META-REGRESSION
# =============================================================================

import numpy as np
import pandas as pd
from typing import Dict, Any, Optional, Tuple, List
from dataclasses import dataclass, field
import warnings
import numpy as np
import pandas as pd
from scipy.stats import norm, chi2, t, f
import statsmodels.api as sm
from typing import Dict, Any, Optional, Tuple, List
import warnings
import traceback
from typing import Optional, Dict, Any
from IPython.display import display, HTML
import ipywidgets as widgets



# =============================================================================
# DATA CLASSES
# =============================================================================

# =============================================================================
# PREDICTED EFFECTS AT GIVEN MODERATOR VALUES ("predict" / marginal effects)
# =============================================================================
# The fitted model gives beta and its CR2 covariance V. A prediction at a
# chosen combination of moderator values x0 is the linear combination
# x0 @ beta, so everything needed is already in the result object:
#
#   estimate         x0 @ beta
#   SE(mean)         sqrt(x0' V x0)                      -> confidence interval
#   SE(new study)    sqrt(x0' V x0 + tau^2 + sigma^2)    -> prediction interval
#   df               contrast-specific Satterthwaite from the AHZ closure at
#                    q = 1 (what clubSandwich's coef_test reports); this is
#                    the same machinery the subgroup per-level DFs use.
#
# The design row is rebuilt from design_info['col_meta'], which already
# records, per column, its source moderator, whether it is a dummy (and for
# which level) or a continuous column, and -- crucially -- the MEAN that was
# subtracted when the continuous column was centred at fit time. Callers give
# moderator values on the ORIGINAL scale; the centring is reapplied here, so a
# prediction at "x = 12.5" means what the user thinks it means.
#
# This mirrors metafor's predict.rma, which likewise reports ci.lb/ci.ub for
# the mean and pi.lb/pi.ub for a new study.


def _resolve_prediction_row(col_meta, values, term_names):
    """Build one design row from moderator values on the original scale.

    `values` maps moderator name -> value: a level string for a categorical
    moderator, a number for a continuous one. Every moderator in the design
    must be supplied. Returns (row, notes) where notes lists non-fatal
    observations (e.g. a reference level being used).
    """
    # col_meta has ONE ENTRY PER DESIGN COLUMN, including the intercept at
    # index 0 ({'source': None, 'type': 'intercept', ...}). So its index IS
    # the column index -- do not offset it, and do not treat the intercept
    # entry as a moderator.
    row = np.zeros(len(col_meta), dtype=np.float64)
    notes = []

    sources = {}
    has_intercept = False
    for col_idx, meta in enumerate(col_meta):
        if meta.get('type') == 'intercept' or meta.get('source') is None:
            row[col_idx] = 1.0
            has_intercept = True
            continue
        sources.setdefault(str(meta['source']), []).append((col_idx, meta))
    if not has_intercept and row.size:
        row[0] = 1.0  # design without an explicit intercept entry

    supplied = {str(k): v for k, v in dict(values).items()}
    unknown = sorted(set(supplied) - set(sources))
    if unknown:
        raise ValueError(
            f"Not moderators of this model: {unknown}. "
            f"Expected any of {sorted(sources)}."
        )
    missing = sorted(set(sources) - set(supplied))
    if missing:
        raise ValueError(
            f"No value given for moderator(s) {missing}. Every moderator in "
            "the model must be fixed before an effect can be predicted."
        )

    for source, entries in sources.items():
        val = supplied[source]
        kind = entries[0][1]['type']

        if kind == 'categorical':
            lv = str(val)
            levels = [m['level'] for _, m in entries]
            ref = entries[0][1].get('reference')
            if lv == str(ref):
                notes.append(f"{source} = {lv} (reference level)")
                continue  # all dummies stay 0
            hit = [ci for ci, m in entries if str(m['level']) == lv]
            if not hit:
                raise ValueError(
                    f"'{lv}' is not a level of '{source}'. Levels in the "
                    f"fitted model: {sorted([str(ref)] + [str(l) for l in levels])}."
                )
            row[hit[0]] = 1.0
        else:
            try:
                num = float(val)
            except (TypeError, ValueError):
                raise ValueError(
                    f"'{source}' is continuous, so it needs a number, "
                    f"got {val!r}."
                )
            if not np.isfinite(num):
                raise ValueError(f"'{source}' must be finite, got {val!r}.")
            col_idx, meta = entries[0]
            # Re-apply the centring used at fit time; see module note above.
            row[col_idx] = num - float(meta.get('mean', 0.0))

    if row.size != len(term_names):
        raise ValueError(
            f"Design row has {row.size} entries but the model has "
            f"{len(term_names)} coefficients; col_meta and the fitted "
            "coefficients are out of step."
        )
    return row, notes


def predict_from_fit(result, rows, ci_level=95.0):
    """Predicted effects, CIs and prediction intervals at given values.

    `rows` is a dict of moderator values, or a list of such dicts. Returns a
    DataFrame with one row per requested combination.

    The DF column records which rule priced each interval: the
    contrast-specific Satterthwaite DF when the AHZ closure is available
    (the correct quantity), otherwise the smallest per-coefficient DF among
    the coefficients the prediction actually loads on -- conservative, and
    flagged as such rather than passed off as the real thing.
    """
    col_meta = getattr(result, 'col_meta', None)
    if not col_meta:
        raise ValueError(
            "This result carries no design metadata (col_meta), so moderator "
            "values cannot be mapped onto the fitted coefficients. Re-run the "
            "meta-regression cell."
        )

    betas = np.asarray(result.betas, dtype=np.float64).ravel()
    V = np.asarray(result.var_betas_robust, dtype=np.float64)
    if V.shape != (betas.size, betas.size):
        raise ValueError(
            f"Coefficient covariance is {V.shape}, expected "
            f"({betas.size}, {betas.size})."
        )
    df_per_coef = np.asarray(result.df_resid, dtype=np.float64).ravel()
    htz = getattr(result, 'htz_wald', None)
    tau_sq = float(result.tau_sq)
    sigma_sq = float(result.sigma_sq)
    alpha = 1.0 - (float(ci_level) / 100.0)

    if isinstance(rows, dict):
        rows = [rows]

    out = []
    for spec in rows:
        x0, notes = _resolve_prediction_row(col_meta, spec, result.term_names)
        est = float(x0 @ betas)
        var_mean = float(x0 @ V @ x0)
        se_mean = float(np.sqrt(max(var_mean, 0.0)))
        se_pred = float(np.sqrt(max(var_mean + tau_sq + sigma_sq, 0.0)))

        df_val, df_method = np.nan, 'contrast Satterthwaite (AHZ q=1)'
        if htz is not None:
            try:
                df_val = float(htz(x0, test='AHZ')['df_denom'])
            except Exception:
                df_val = np.nan
        if not np.isfinite(df_val) or df_val <= 0:
            loaded = np.abs(x0) > 1e-12
            cand = df_per_coef[loaded] if loaded.any() else df_per_coef
            cand = cand[np.isfinite(cand) & (cand > 0)]
            df_val = float(np.min(cand)) if cand.size else 1.0
            df_method = 'min over loaded coefficients (fallback)'
        df_val = max(1.0, df_val)

        crit = float(t.ppf(1.0 - alpha / 2.0, df_val))
        entry = {
            **{k: v for k, v in spec.items()},
            'estimate': est,
            'se': se_mean,
            'df': df_val,
            'ci_lower': est - crit * se_mean,
            'ci_upper': est + crit * se_mean,
            'pi_lower': est - crit * se_pred,
            'pi_upper': est + crit * se_pred,
            'df_method': df_method,
        }
        if notes:
            entry['notes'] = '; '.join(notes)
        out.append(entry)

    return pd.DataFrame(out)


def default_prediction_grid(result):
    """A grid worth showing without the user specifying anything.

    Each categorical level in turn (other moderators held at their reference
    level / centred mean), then each continuous moderator at mean - 1 SD,
    mean and mean + 1 SD. Values are on the ORIGINAL scale.
    """
    col_meta = getattr(result, 'col_meta', None)
    if not col_meta:
        return []

    spec = {}
    for meta in col_meta:
        # Skip the intercept column; it is not a moderator (see
        # _resolve_prediction_row on col_meta's layout).
        if meta.get('type') == 'intercept' or meta.get('source') is None:
            continue
        src = str(meta['source'])
        if meta['type'] == 'categorical':
            spec.setdefault(src, {'type': 'categorical',
                                  'ref': meta.get('reference'),
                                  'levels': []})
            spec[src]['levels'].append(meta['level'])
        else:
            spec[src] = {'type': 'continuous',
                         'mean': float(meta.get('mean', 0.0)),
                         'sd': float(meta.get('sd', 0.0) or 0.0)}

    baseline = {}
    for src, s in spec.items():
        baseline[src] = (s['ref'] if s['type'] == 'categorical' else s['mean'])

    grid = []
    for src, s in spec.items():
        if s['type'] == 'categorical':
            for lv in [s['ref']] + list(s['levels']):
                grid.append({**baseline, src: lv})
        else:
            sd = s['sd']
            points = ([s['mean'] - sd, s['mean'], s['mean'] + sd]
                      if sd > 0 else [s['mean']])
            for v in points:
                grid.append({**baseline, src: v})

    # de-duplicate while preserving order (the all-baseline row repeats)
    seen, unique = set(), []
    for g in grid:
        key = tuple(sorted((k, str(v)) for k, v in g.items()))
        if key not in seen:
            seen.add(key)
            unique.append(g)
    return unique


def predict_meta_regression(values=None, ci_level=95.0, result=None):
    """Predicted effects at given moderator values for the last fit.

        predict_meta_regression({'genus': 'Passer', 'biome': 'urban'})
        predict_meta_regression([{...}, {...}])        # several at once
        predict_meta_regression()                      # the default grid

    Values are on the ORIGINAL scale: give a level name for a categorical
    moderator and a raw number for a continuous one.
    """
    res = result if result is not None else globals().get('LAST_REGRESSION_RESULT')
    if res is None:
        raise ValueError(
            "No meta-regression result in memory. Run the Meta-Regression "
            "cell first, or pass result=<RegressionResult>."
        )
    if values is None:
        values = default_prediction_grid(res)
        if not values:
            raise ValueError(
                "Could not build a default grid for this model; pass the "
                "moderator values explicitly."
            )
    return predict_from_fit(res, values, ci_level=ci_level)


@dataclass
class RegressionConfig:
    """Configuration for meta-regression analysis.

    Supports one or more moderators. `moderator_cols` is the canonical
    field. The legacy single-moderator field `moderator_col` is retained
    for backward compatibility and always mirrors the first entry of
    `moderator_cols`, so existing call sites that construct
    RegressionConfig(moderator_col=...) or read config.moderator_col
    continue to work unchanged.

    Categorical metadata (populated later in the pipeline):
      categorical_cols  : subset of moderator_cols to treat as factors
                          (treatment / dummy coding, metafor convention).
      reference_levels  : {column_name: reference_level} overrides; columns
                          absent from this dict use the default reference
                          (first level, alphabetical) chosen at coding time.
    """
    moderator_cols: List[str] = field(default_factory=list)
    effect_col: str = ""
    var_col: str = ""
    se_col: Optional[str] = None
    categorical_cols: List[str] = field(default_factory=list)
    reference_levels: Dict[str, str] = field(default_factory=dict)
    # Legacy single-moderator alias (kept for backward compatibility):
    moderator_col: Optional[str] = None

    def __post_init__(self):
        """Reconcile legacy/new fields, then validate configuration."""
        # Fold a legacy single-moderator kwarg into the list form.
        if self.moderator_col and not self.moderator_cols:
            self.moderator_cols = [self.moderator_col]
        # Keep the legacy alias pointing at the first moderator.
        if self.moderator_cols and not self.moderator_col:
            self.moderator_col = self.moderator_cols[0]

        # --- Validation ---
        if not self.moderator_cols:
            raise ValueError("moderator_cols cannot be empty")
        if not self.effect_col:
            raise ValueError("effect_col cannot be empty")
        if not self.var_col:
            raise ValueError("var_col cannot be empty")

        # Categorical metadata must reference declared moderators.
        for c in self.categorical_cols:
            if c not in self.moderator_cols:
                raise ValueError(
                    f"categorical column '{c}' is not in moderator_cols"
                )

@dataclass
class RegressionResult:
    """Complete meta-regression result (single or multi-moderator).

    Canonical coefficient data is stored as arrays:
        betas      (p,)              term_names (length p)
        se_betas   (p,)              p_values   (p,)
        cis        list of p (lo, hi) tuples
    The legacy scalar fields (beta0/beta1/se0/se1/p0/p1/ci0/ci1) are exposed
    as read-only properties over these arrays, so all existing single- and
    two-coefficient consumers keep working unchanged. They raise a clear
    error only if accessed on a model that lacks that coefficient.
    """
    # --- Canonical coefficient arrays ---
    betas: np.ndarray
    se_betas: np.ndarray
    p_values: np.ndarray
    cis: List[Tuple[float, float]]
    term_names: List[str]

    # Variance components
    tau_sq: float
    sigma_sq: float

    # Model info
    model_type: str
    k_studies: int
    n_obs: int
    df_resid: np.ndarray

    # Heterogeneity
    r_squared: float
    resid_i2: float

    # Diagnostics
    fitted: np.ndarray
    resid: np.ndarray
    var_betas_robust: np.ndarray

    # Additional metadata
    moderator_col_name: str
    has_influential: bool = False
    cooks_d: Optional[np.ndarray] = None

    # Optional omnibus test results (populated later; per-factor + overall).
    omnibus: Optional[Dict[str, Any]] = None

    # Design provenance, needed to predict at given moderator values: one
    # entry per design column (source moderator, dummy level or continuous,
    # and the mean/SD used to centre a continuous column).
    col_meta: Optional[List[Dict[str, Any]]] = None
    reference_levels: Optional[Dict[str, str]] = None
    # AHZ closure for contrast-specific small-sample DF. A live closure, so
    # it is deliberately NOT persisted with the rest of the result.
    htz_wald: Optional[Any] = None
    # What the engine actually fitted: the chosen sampling-VCV blocks, the
    # stacked design, y, and the per-row study id in the engine's internal
    # (id-sorted) order. Used by the validation cell to hand R the identical
    # model; arrays, so large but plain data.
    fit_inputs: Optional[Dict[str, Any]] = None
    term_groups: Optional[Dict[str, List[int]]] = None
    categorical_cols: Optional[List[str]] = None
    moderator_cols: Optional[List[str]] = None
    # Pseudo-R² against the matching intercept-only model (same rows, same
    # sampling V, same random structure), with per-level reductions. Plain
    # floats/strings only; see RegressionEngine._compute_pseudo_r2.
    r2_detail: Optional[Dict[str, Any]] = None
    # Which rows were used / dropped and why (RowAccounting.to_dict()).
    data_accounting: Optional[Dict[str, Any]] = None

    def __post_init__(self):
        # Normalize array types so accessors and consumers are consistent.
        self.betas = np.asarray(self.betas, dtype=np.float64).ravel()
        self.se_betas = np.asarray(self.se_betas, dtype=np.float64).ravel()
        self.p_values = np.asarray(self.p_values, dtype=np.float64).ravel()
        self.df_resid = np.asarray(self.df_resid, dtype=np.float64).ravel()
        if len(self.cis) != self.betas.size:
            raise ValueError(
                f"cis has {len(self.cis)} entries but there are "
                f"{self.betas.size} coefficients."
            )
        if len(self.term_names) != self.betas.size:
            raise ValueError(
                f"term_names has {len(self.term_names)} entries but there "
                f"are {self.betas.size} coefficients."
            )

    # --- Convenience ---
    @property
    def n_coef(self) -> int:
        """Number of coefficients (including intercept)."""
        return int(self.betas.size)

    def _coef(self, i: int, what: str):
        if i >= self.betas.size:
            raise IndexError(
                f"Coefficient index {i} ({what}) requested but model has "
                f"only {self.betas.size} coefficient(s). Use the array "
                f"fields (betas/se_betas/p_values/cis) for multi-moderator "
                f"models."
            )
        return i

    # --- Legacy scalar accessors (read-only) over the arrays ---
    @property
    def beta0(self) -> float:
        return float(self.betas[self._coef(0, 'beta0')])

    @property
    def beta1(self) -> float:
        return float(self.betas[self._coef(1, 'beta1')])

    @property
    def se0(self) -> float:
        return float(self.se_betas[self._coef(0, 'se0')])

    @property
    def se1(self) -> float:
        return float(self.se_betas[self._coef(1, 'se1')])

    @property
    def p0(self) -> float:
        return float(self.p_values[self._coef(0, 'p0')])

    @property
    def p1(self) -> float:
        return float(self.p_values[self._coef(1, 'p1')])

    @property
    def ci0(self) -> Tuple[float, float]:
        return self.cis[self._coef(0, 'ci0')]

    @property
    def ci1(self) -> Tuple[float, float]:
        return self.cis[self._coef(1, 'ci1')]


# =============================================================================
# DATA MANAGER CLASS
# =============================================================================

class RegressionDataManager:
    """
    Centralized data access layer for meta-regression.
    Handles all interactions with ANALYSIS_CONFIG and data validation.
    """

    def __init__(self, analysis_config: Dict[str, Any]):
        """
        Initialize data manager.

        Args:
            analysis_config: Global ANALYSIS_CONFIG dictionary
        """
        self._config = analysis_config
        self._validate_prerequisites()


    ########################

    def build_design_matrix(
        self,
        reg_df: pd.DataFrame,
        config: 'RegressionConfig'
    ) -> Dict[str, Any]:
        """
        Build the regression design matrix from one or more moderators.

        Treatment (dummy) coding is used for categorical moderators: for a
        factor with L present levels, L-1 indicator columns are created and
        the reference level is omitted (absorbed into the intercept). This
        matches metafor's default factor handling. Continuous moderators
        enter as their numeric column. The intercept is column 0.

        PRECONDITION: reg_df must already be cleaned of missing values in
        every moderator column (see prepare_regression_data). This method
        fails loudly rather than silently coding NaN as a level.

        Args:
            reg_df: Cleaned regression DataFrame.
            config: RegressionConfig with moderator_cols, categorical_cols,
                    and (optional) reference_levels.

        Returns:
            dict with keys:
              'X'                : np.ndarray, shape (n, p), intercept first
              'term_names'       : List[str], length p (e.g. 'Intercept',
                                  'time', 'fungrp[N2FIX]')
              'term_groups'      : Dict[str, List[int]] mapping each source
                                  moderator (and 'Intercept') to its design
                                  column indices. Categorical factors map to
                                  all their dummy columns -> used later to
                                  build the joint (omnibus) contrast.
              'col_meta'         : List[dict], per-column provenance
                                  {'source','type','level','reference'}
              'reference_levels' : Dict[str, str], references actually used.

        Raises:
            ValueError: if a moderator is missing, contains NaN, or a
                        categorical factor has fewer than 2 present levels.
        """
        categorical = set(config.categorical_cols)
        n = len(reg_df)

        X_cols: List[np.ndarray] = [np.ones(n, dtype=np.float64)]
        term_names: List[str] = ['Intercept']
        term_groups: Dict[str, List[int]] = {'Intercept': [0]}
        col_meta: List[Dict[str, Any]] = [
            {'source': None, 'type': 'intercept', 'level': None, 'reference': None}
        ]
        resolved_refs: Dict[str, str] = {}
        col_idx = 1

        for mod in config.moderator_cols:
            if mod not in reg_df.columns:
                raise ValueError(f"Moderator '{mod}' not found in data")

            if mod in categorical:
                ser = reg_df[mod]
                if ser.isna().any():
                    raise ValueError(
                        f"Categorical moderator '{mod}' contains missing "
                        f"values; data must be cleaned before coding."
                    )
                ser = ser.astype(str)
                levels = sorted(ser.unique().tolist())
                if len(levels) < 2:
                    raise ValueError(
                        f"Categorical moderator '{mod}' has fewer than 2 "
                        f"levels present after cleaning (levels: {levels}); "
                        f"it cannot be dummy-coded."
                    )

                ref = self._resolve_reference_level(
                    levels, mod, config.reference_levels
                )
                resolved_refs[mod] = ref
                non_ref = [lv for lv in levels if lv != ref]

                idxs: List[int] = []
                for lv in non_ref:
                    dummy = (ser.values == lv).astype(np.float64)
                    X_cols.append(dummy)
                    term_names.append(f"{mod}[{lv}]")
                    col_meta.append({
                        'source': mod, 'type': 'categorical',
                        'level': lv, 'reference': ref
                    })
                    idxs.append(col_idx)
                    col_idx += 1
                term_groups[mod] = idxs

            else:
                vals = pd.to_numeric(reg_df[mod], errors='coerce').values.astype(np.float64)
                if np.isnan(vals).any():
                    raise ValueError(
                        f"Continuous moderator '{mod}' contains non-numeric "
                        f"or missing values after coercion; data must be "
                        f"cleaned before building the design matrix."
                    )

                # --- MEAN CENTERING FIX ---
                col_mean = np.mean(vals)
                vals_centered = vals - col_mean

                X_cols.append(vals_centered)
                term_names.append(f"{mod} (centered)")  # Automatically updates tables
                col_meta.append({
                    'source': mod, 'type': 'continuous',
                    'level': None, 'reference': None,
                    # The mean is not just provenance: predict_from_fit()
                    # re-applies this centring so callers can give moderator
                    # values on the ORIGINAL scale. The SD gives the default
                    # prediction grid a sensible step.
                    'mean': float(col_mean),
                    'sd': float(np.std(vals, ddof=1)) if vals.size > 1 else 0.0,
                })
                term_groups[mod] = [col_idx]
                col_idx += 1

        X = np.column_stack(X_cols)

        # Guard against a rank-deficient design (perfectly collinear
        # moderators or a redundant dummy set) before it reaches the GLS
        # solve, where it would surface as a less obvious singular-matrix
        # failure. Column-normalize to unit length first: matrix_rank's SVD
        # tolerance scales with the largest singular value, so mixing a
        # large-magnitude continuous moderator (e.g. biomass ~1e6) with a
        # small one (e.g. pH ~1) or 0/1 dummies can otherwise truncate a
        # genuinely independent column below tolerance and raise a false
        # collinearity error. Normalizing equalizes the columns so the rank
        # test reflects true linear dependence, not scale.
        if not np.all(np.isfinite(X)):
            raise ValueError(
                "Design matrix contains non-finite values (inf/NaN) after "
                "coding. Check the selected moderators for invalid entries."
            )

        col_norms = np.linalg.norm(X, axis=0)
        col_norms[col_norms == 0] = 1.0  # avoid divide-by-zero on empty cols
        X_scaled = X / col_norms

        if np.linalg.matrix_rank(X_scaled) < X_scaled.shape[1]:
            raise ValueError(
                "Design matrix is rank-deficient even after scaling, "
                "indicating true linear dependence among predictors "
                "(perfectly collinear moderators or a redundant dummy set). "
                "Check for duplicate moderators, a moderator that is a linear "
                "combination of others, or a categorical level that is "
                "perfectly confounded with another predictor."
            )


        return {
            'X': X,
            'term_names': term_names,
            'term_groups': term_groups,
            'col_meta': col_meta,
            'reference_levels': resolved_refs,
        }


    # -------------------------------------------------------------------------
    # VALIDATION METHODS
    # -------------------------------------------------------------------------




    def _validate_prerequisites(self) -> None:
        """Validate that required configuration exists"""
        # Meta-regression doesn't require as many prerequisites as subgroup analysis
        # but we still need basic effect size configuration
        if 'effect_col' not in self._config:
            warnings.warn("effect_col not in ANALYSIS_CONFIG, using default 'hedges_g'")
        if 'var_col' not in self._config:
            warnings.warn("var_col not in ANALYSIS_CONFIG, using default 'Vg'")

    def validate_moderator(
        self,
        df: pd.DataFrame,
        moderator_col: str
    ) -> Tuple[bool, Optional[str]]:
        """
        Validate that moderator column is suitable for regression.

        Args:
            df: DataFrame to validate
            moderator_col: Moderator column name

        Returns:
            Tuple of (is_valid, error_message)
        """
        if moderator_col not in df.columns:
            return False, f"Column '{moderator_col}' not found in data"

        # Try to convert to numeric
        try:
            numeric_vals = pd.to_numeric(df[moderator_col], errors='coerce')
        except Exception as e:
            return False, f"Cannot convert '{moderator_col}' to numeric: {str(e)}"

        # Check for sufficient non-missing values
        n_valid = numeric_vals.notna().sum()
        if n_valid < 3:
            return False, f"Insufficient data: only {n_valid} valid numeric values"

        # Check for variation
        if numeric_vals.nunique() <= 1:
            return False, f"No variation in '{moderator_col}'"

        return True, None

    # -------------------------------------------------------------------------
    # PROPERTY ACCESSORS
    # -------------------------------------------------------------------------

    @property
    def analysis_data(self) -> Optional[pd.DataFrame]:
        """Get analysis dataset"""
        if 'analysis_data' in self._config:
            return self._config['analysis_data'].copy()
        return None

    @property
    def effect_col(self) -> str:
        """Get effect size column name"""
        return self._config.get('effect_col', 'hedges_g')

    @property
    def var_col(self) -> str:
        """Get variance column name"""
        return self._config.get('var_col', 'Vg')

    @property
    def se_col(self) -> Optional[str]:
        """Get standard error column name"""
        return self._config.get('se_col')

    @property
    def global_settings(self) -> Dict[str, Any]:
        """Get global settings"""
        return self._config.get('global_settings', {
            'alpha': 0.05,
            'dist_type': 't'
        })

    @property
    def overall_results(self) -> Optional[Dict[str, Any]]:
        """Get overall meta-analysis results (for R² calculation)"""
        return self._config.get('overall_results')

    @property
    def es_config(self) -> Dict[str, Any]:
        """Get effect size configuration"""
        return self._config.get('es_config', {})

    def _resolve_reference_level(
            self,
            levels_sorted: List[str],
            col: str,
            reference_levels: Dict[str, str]
        ) -> str:
            """
            Pick the reference level for a categorical moderator.

            Default is the first level alphabetically (metafor convention:
            R orders character factor levels alphabetically and drops the
            first as the reference). A user override in reference_levels wins,
            provided it is actually present in the data.
            """
            override = reference_levels.get(col)
            if override is not None:
                override = str(override)
                if override not in levels_sorted:
                    raise ValueError(
                        f"Reference level '{override}' for '{col}' is not present "
                        f"in the data. Available levels: {levels_sorted}"
                    )
                return override
            return levels_sorted[0]

    # -------------------------------------------------------------------------
    # DATA EXTRACTION METHODS
    # -------------------------------------------------------------------------


    _CATEGORICAL_NUNIQUE_MAX = 10

    def _build_exclusion_set(self) -> set:
        """
        Columns that must never be offered as moderators.

        Beyond the pipeline machinery (id, weights, the configured
        effect/variance/SE columns), this also drops:
          - row_id (the per-effect index; id is the cluster key, kept)
          - the effect-size-derived family (transforms of the effect size:
            response ratios, %-change, fold change, CI bounds, imputed SDs).
        Regressing the effect size on a transform of itself is circular, so
        these are excluded for safety in a professional setting.
        """
        exclude = {'id', 'row_id', 'w_fixed', 'w_random',
                   self.effect_col, self.var_col}
        if self.se_col:
            exclude.add(self.se_col)

        # Derived-family name fragments (case-insensitive substring match).
        derived_fragments = (
            'lnrr', 'response_ratio', 'fold_change', 'percent_change',
            'ci_lower', 'ci_upper', 'sdc_imputed',
        )
        # Also derive stems from the configured effect/var/se columns so
        # transforms like '<effect>', 'var_<effect>', 'SE_<effect>',
        # 'CI_lower_<effect>' are all caught regardless of dataset.
        stems = []
        for base in (self.effect_col, self.var_col, self.se_col):
            if base:
                stems.append(str(base).lower())

        # Stems are matched on TOKEN boundaries, not as bare substrings.
        # A plain `stem in name` test makes short effect-size names eat
        # unrelated moderators: with effect_col='RD' (risk difference),
        # 'standard_method' contains 'rd' and was silently excluded. The
        # fixed derived_fragments below are long and distinctive, so they
        # keep the cheaper substring test.
        import re as _re

        def _matches_stem(name_low: str, stem: str) -> bool:
            if not stem:
                return False
            if name_low == stem:
                return True
            return _re.search(
                r'(?:^|[^a-z0-9])' + _re.escape(stem) + r'(?:[^a-z0-9]|$)',
                name_low) is not None

        df = self.analysis_data
        if df is not None:
            for col in df.columns:
                if col is None:
                    continue
                low = str(col).lower()
                if any(frag in low for frag in derived_fragments):
                    exclude.add(col)
                elif any(_matches_stem(low, stem) for stem in stems):
                    exclude.add(col)

        return exclude

    def get_potential_moderators_typed(
        self,
        df: Optional[pd.DataFrame] = None
    ) -> List[Dict[str, Any]]:
        """
        Get valid moderator candidates, each tagged with an inferred type.

        Returns a list of dicts:
            {'name': str,
             'inferred_type': 'categorical' | 'continuous',
             'ambiguous': bool,          # could reasonably be the other type
             'n_levels': int,            # distinct non-missing values
             'numeric_as_text': bool,    # numbers stored as text
             'note': str}                # shown next to the type dropdown

        Detection rule -- deliberately NOT keyed on the exact pandas dtype:
          - numeric dtype                  -> continuous
                                              (ambiguous when cardinality is
                                              low: the user may want factors)
          - datetime                       -> skipped, with a reason
          - anything else (text-like)      -> coerced with to_numeric first:
              * every non-missing value parses as a number -> continuous,
                flagged numeric_as_text so the UI can say so (ambiguous when
                cardinality is low)
              * otherwise                  -> categorical

        The previous rule tested `series.dtype == 'object'` and had no else
        branch, so a column held as 'category', pandas 'string', 'str'
        (pandas 3) or datetime vanished from the list with no message at all.
        Columns really are dropped for no variation or too few rows, but the
        reasons are now recorded in self.skipped_moderators and surfaced in
        the UI rather than silently swallowed.

        Columns in the exclusion set are never returned.
        """
        if df is None:
            df = self.analysis_data
        self.skipped_moderators = []
        if df is None:
            return []

        exclude = self._build_exclusion_set()
        out = []

        for col in df.columns:
            if col is None:
                continue
            if col in exclude:
                self.skipped_moderators.append(
                    (str(col), "excluded: it is the effect size, its variance "
                               "or SE, a transform of them, or a pipeline key"))
                continue

            series = df[col]
            n_levels = int(series.nunique(dropna=True))
            n_valid = int(series.notna().sum())

            if n_levels <= 1:
                self.skipped_moderators.append(
                    (str(col), f"no variation ({n_levels} distinct value"
                               f"{'' if n_levels == 1 else 's'})"))
                continue
            if n_valid < 3:
                self.skipped_moderators.append(
                    (str(col), f"only {n_valid} non-missing row"
                               f"{'' if n_valid == 1 else 's'}"))
                continue

            if pd.api.types.is_datetime64_any_dtype(series):
                self.skipped_moderators.append(
                    (str(col), "date/time column -- derive a numeric field "
                               "(e.g. year) to use it as a moderator"))
                continue

            ambiguous = (n_levels <= self._CATEGORICAL_NUNIQUE_MAX
                         and n_levels < max(2, n_valid / 2))

            if pd.api.types.is_numeric_dtype(series):
                out.append({
                    'name': col, 'inferred_type': 'continuous',
                    'ambiguous': ambiguous, 'n_levels': n_levels,
                    'numeric_as_text': False, 'note': '',
                })
                continue

            # Text-like: object, string, category, str, anything else.
            coerced = pd.to_numeric(series, errors='coerce')
            n_numeric = int(coerced.notna().sum())

            if n_numeric == n_valid:
                # Numbers that happen to be stored as text (a decimal comma,
                # a stray space, an Excel text column...). Treat as
                # continuous, but say so -- the old rule called these
                # categorical and silently fitted one dummy per value.
                out.append({
                    'name': col, 'inferred_type': 'continuous',
                    'ambiguous': ambiguous, 'n_levels': n_levels,
                    'numeric_as_text': True,
                    'note': 'numbers stored as text',
                })
            else:
                note = ''
                if n_numeric:
                    note = (f'{n_valid - n_numeric} of {n_valid} values are '
                            'not numeric')
                else:
                    # A whole column that parses only once the decimal comma
                    # is swapped for a point is a locale artefact, not a
                    # factor. Say so instead of offering k-1 dummies.
                    try:
                        swapped = pd.to_numeric(
                            series.astype(str).str.replace(',', '.', regex=False),
                            errors='coerce')
                        if int(swapped.notna().sum()) == n_valid:
                            note = ('looks like numbers written with a decimal '
                                    'comma -- fix the column to use it as continuous')
                    except Exception:
                        pass
                out.append({
                    'name': col, 'inferred_type': 'categorical',
                    'ambiguous': False, 'n_levels': n_levels,
                    'numeric_as_text': False, 'note': note,
                })

        out.sort(key=lambda d: d['name'])
        return out

    def get_potential_moderators(
        self,
        df: Optional[pd.DataFrame] = None
    ) -> List[str]:
        """
        Backward-compatible list of moderator names.

        Now routes through get_potential_moderators_typed (tightened
        exclusions). Returns the same flat sorted list of column names that
        existing callers expect; the only behavioural change is that
        row_id and the effect-derived family are no longer offered.
        """
        return [m['name'] for m in self.get_potential_moderators_typed(df)]



    def prepare_regression_data_multi(
        self,
        moderator_cols: List[str],
        categorical_cols: Optional[List[str]] = None,
        reference_levels: Optional[Dict[str, str]] = None,
        df: Optional[pd.DataFrame] = None
    ) -> Tuple[pd.DataFrame, RegressionConfig, Dict[str, Any]]:
        """
        Prepare and clean data for one or more moderators, then build the
        treatment-coded design matrix.

        Cleaning is performed jointly across ALL moderators (listwise
        deletion): a row is kept only if the effect, variance, and every
        selected moderator are present, and the variance is positive.
        Continuous moderators are coerced to numeric; categorical moderators
        are left as labels for dummy coding.

        Args:
            moderator_cols: Names of moderator columns (>= 1).
            categorical_cols: Subset of moderator_cols to treat as factors.
            reference_levels: Optional {col: reference_level} overrides.
            df: DataFrame to prepare (uses analysis_data if None).

        Returns:
            Tuple of (cleaned_df, config, design_info), where design_info is
            the dict returned by build_design_matrix (X, term_names,
            term_groups, col_meta, reference_levels).

        Raises:
            ValueError: If data cannot be prepared.
        """
        if df is None:
            df = self.analysis_data
        if df is None:
            raise ValueError("No data available for analysis")

        if not moderator_cols:
            raise ValueError("At least one moderator must be provided")

        categorical_cols = list(categorical_cols or [])
        reference_levels = dict(reference_levels or {})
        cat_set = set(categorical_cols)

        # --- Validate each moderator (type-appropriate) ---
        for mod in moderator_cols:
            if mod not in df.columns:
                raise ValueError(f"Column '{mod}' not found in data")
            if mod in cat_set:
                # Categorical: need >= 2 levels and enough usable rows.
                ser = df[mod]
                n_levels = ser.nunique(dropna=True)
                if n_levels < 2:
                    raise ValueError(
                        f"Categorical moderator '{mod}' has fewer than 2 "
                        f"levels ({n_levels})."
                    )
                if ser.notna().sum() < 3:
                    raise ValueError(
                        f"Categorical moderator '{mod}' has insufficient "
                        f"non-missing values."
                    )
            else:
                # Continuous: reuse existing numeric validation.
                is_valid, error_msg = self.validate_moderator(df, mod)
                if not is_valid:
                    raise ValueError(error_msg)

        # --- Working copy ---
        reg_df = df.copy()

        # Coerce continuous moderators to numeric (categoricals untouched).
        for mod in moderator_cols:
            if mod not in cat_set:
                reg_df[mod] = pd.to_numeric(reg_df[mod], errors='coerce')

        # --- Joint (listwise) cleaning across effect, variance, all mods ---
        # Every dropped row is recorded with its reason (RowAccounting).
        if 'select_rows' in globals():
            reg_df, self.last_accounting = select_rows(
                reg_df, self.effect_col, self.var_col, 'id',
                complete_case_cols=list(moderator_cols))
            reg_df = reg_df.copy()
        else:
            subset_cols = [self.effect_col, self.var_col] + list(moderator_cols)
            reg_df = reg_df.dropna(subset=subset_cols).copy()
            reg_df = reg_df[reg_df[self.var_col] > 0].copy()
            self.last_accounting = None

        if len(reg_df) < 3:
            raise ValueError(
                f"Insufficient data after cleaning: {len(reg_df)} "
                f"observations. Need at least 3."
            )

        # Re-check that categoricals still have >= 2 levels post-cleaning
        # (listwise deletion can drop an entire level).
        for mod in categorical_cols:
            if reg_df[mod].astype(str).nunique() < 2:
                raise ValueError(
                    f"Categorical moderator '{mod}' has fewer than 2 levels "
                    f"remaining after joint cleaning across all selected "
                    f"moderators."
                )

        # --- Build configuration ---
        config = RegressionConfig(
            moderator_cols=list(moderator_cols),
            effect_col=self.effect_col,
            var_col=self.var_col,
            se_col=self.se_col,
            categorical_cols=categorical_cols,
            reference_levels=reference_levels,
        )

        # --- Build the design matrix ---
        design_info = self.build_design_matrix(reg_df, config)

        return reg_df, config, design_info

    def check_within_study_variation(
        self,
        reg_df: pd.DataFrame,
        moderator_col: str
    ) -> Dict[str, Any]:
        """
        Check if moderator varies within studies.

        This determines whether we need 3-level (varying) or
        2-level aggregated (constant) model.

        Args:
            reg_df: Prepared regression DataFrame
            moderator_col: Moderator column name

        Returns:
            Dictionary with variation statistics
        """
        studies_with_variation = reg_df.groupby('id')[moderator_col].nunique()
        varying_studies = (studies_with_variation > 1).sum()
        k_studies = reg_df['id'].nunique()
        n_obs = len(reg_df)
        has_multiple_effects = (reg_df.groupby('id').size() > 1).any()

        return {
            'varying_studies': varying_studies,
            'k_studies': k_studies,
            'n_obs': n_obs,
            'has_multiple_effects': has_multiple_effects,
            'needs_3level': has_multiple_effects,
            'moderator_constant_within_study': varying_studies == 0
        }

    # -------------------------------------------------------------------------
    # RESULT PERSISTENCE METHODS
    # -------------------------------------------------------------------------

    def save_regression_results(
        self,
        result: 'RegressionResult',
        reg_df: pd.DataFrame
    ) -> None:
        """Persist regression results to ANALYSIS_CONFIG.

        Writes a SUPERSET dict: every legacy 2-coefficient key is preserved
        with its original meaning (intercept + primary moderator term), so
        existing downstream consumers (plot cell, export) keep working; new
        array-based keys carry the full multi-moderator result.

        For multi-moderator models the legacy 2-element keys hold the
        intercept and the FIRST moderator term as a compatibility projection;
        the complete coefficients live in the new 'betas_all' / 'term_names' /
        'cis_all' / 'p_values_all' keys.
        """
        import datetime

        betas = np.asarray(result.betas, dtype=float).ravel()
        se = np.asarray(result.se_betas, dtype=float).ravel()
        pvals = np.asarray(result.p_values, dtype=float).ravel()
        df_vec = np.asarray(result.df_resid, dtype=float).ravel()
        p = betas.size

        # --- Legacy 2-element projection (intercept + first moderator) ---
        # Guaranteed safe: p >= 2 always (intercept + >=1 moderator term).
        legacy_betas = [float(betas[0]), float(betas[1])]
        legacy_se = [float(se[0]), float(se[1])]
        legacy_p_intercept = float(pvals[0])
        legacy_p_slope = float(pvals[1])
        legacy_ci0 = result.cis[0]
        legacy_ci1 = result.cis[1]

        # Legacy var-cov was 2x2; slice the top-left 2x2 block (exists for any
        # p >= 2). Full matrix is stored separately below.
        V = np.asarray(result.var_betas_robust, dtype=float)
        if V.ndim == 2 and V.shape[0] >= 2 and V.shape[1] >= 2:
            legacy_var_2x2 = V[:2, :2]
        else:
            legacy_var_2x2 = np.diag(legacy_se) ** 2

        self._config['meta_regression_RVE_results'] = {
            # ---- timestamp / data ----
            'timestamp': datetime.datetime.now(),
            'reg_df': reg_df,
            'effect_col': self.effect_col,

            # ---- LEGACY KEYS (unchanged meaning; primary moderator) ----
            'moderator_col_name': result.moderator_col_name,
            'betas': legacy_betas,                  # [intercept, primary slope]
            'std_errors_robust': legacy_se,         # [se intercept, se slope]
            'var_betas_robust': legacy_var_2x2,     # top-left 2x2 block
            'p_slope': legacy_p_slope,
            'p_intercept': legacy_p_intercept,
            'ci0': legacy_ci0,
            'ci1': legacy_ci1,
            'df_robust': df_vec,                    # full df vector (already array)
            'R_squared_adj': result.r_squared,       # pseudo-R², % (legacy key name)
            'R_squared_detail': result.r2_detail,
            'data_accounting': result.data_accounting,
            'fitted': result.fitted,
            'resid': result.resid,
            'tau_sq': result.tau_sq,
            'sigma_sq': result.sigma_sq,
            'model_type': result.model_type,
            'k_studies': result.k_studies,
            'n_obs': result.n_obs,
            'resid_i2': result.resid_i2,
            'has_influential': result.has_influential,
            'cooks_d': result.cooks_d,

            # ---- NEW KEYS (full multi-moderator result) ----
            'n_coef': int(p),
            'term_names': list(result.term_names),
            'betas_all': betas,
            'std_errors_all': se,
            'p_values_all': pvals,
            'cis_all': list(result.cis),
            'df_all': df_vec,
            'var_betas_full': V,
            'moderator_cols': self._last_moderator_cols(result),
            'omnibus': result.omnibus,
        }

        # Also save var_col for compatibility.
        self._config['var_col'] = self.var_col

    @staticmethod
    def _last_moderator_cols(result: 'RegressionResult') -> List[str]:
        """Best-effort recovery of the source moderator names from the term
        names (intercept dropped; categorical 'factor[level]' -> 'factor')."""
        out = []
        for name in result.term_names[1:]:
            s = str(name)
            base = s.split("[", 1)[0] if "[" in s else s
            if base not in out:
                out.append(base)
        return out


    def save_publication_text(self, text: str) -> None:
        """Save publication-ready text"""
        self._config['regression_text'] = text

    # -------------------------------------------------------------------------
    # UTILITY METHODS
    # -------------------------------------------------------------------------

    def summary_dict(self) -> Dict[str, Any]:
        """Get summary of current configuration"""
        settings = self.global_settings

        return {
            'effect_col': self.effect_col,
            'var_col': self.var_col,
            'se_col': self.se_col,
            'alpha': settings.get('alpha', 0.05),
            'dist_type': settings.get('dist_type', 't'),
            'has_overall_results': self.overall_results is not None,
            'n_potential_moderators': len(self.get_potential_moderators())
        }
# =============================================================================
# ANALYSIS LAYER - META-REGRESSION
# CORE REGRESSION ENGINES
# =============================================================================

class TwoLevelRegressionEngine:
    """
    Standard 2-Level Random-Effects Meta-Regression.
    Used when moderator is constant within studies (aggregated data).

    Mathematical implementation validated against metafor.
    """

    @staticmethod
    def run_aggregated_regression_design(
        y: np.ndarray,
        v: np.ndarray,
        X: np.ndarray,
        term_names: List[str]
    ) -> Dict[str, Any]:
        """
        2-level random-effects meta-regression from a prebuilt design matrix.

        Generalizes run_aggregated_regression to p >= 2 coefficients. REML
        estimates a single between-study variance tau^2 (shared engine,
        Cell 1b); betas come from WLS with weights 1/(v + tau^2).
        For a single continuous moderator (X = [1, x]) every quantity is
        numerically identical to run_aggregated_regression.

        Args:
            y: Effect sizes, shape (k,).
            v: Sampling variances, shape (k,).
            X: Design matrix WITH intercept column, shape (k, p).
            term_names: Length-p labels (term_names[0] == 'Intercept').

        Returns:
            dict with array-valued results:
              'betas'            (p,)      'se_betas'        (p,)
              'p_values'         (p,)      'tau_sq'          float
              'sigma_sq'         float (0) 'model_type'      str
              'n_obs'            int       'resid_df'        (p,)
              'fitted'           (k,)      'resid'           (k,) pearson
              'model'            statsmodels WLS results
              'var_betas_robust' (p, p)    'term_names'      list
        """
        y = np.asarray(y, dtype=np.float64).ravel()
        v = np.asarray(v, dtype=np.float64).ravel()
        X = np.asarray(X, dtype=np.float64)
        if X.ndim == 1:
            X = X.reshape(-1, 1)

        k, p = X.shape

        # REML estimate of tau^2 for the two-level regression (one effect
        # per row) with the shared engine, Cell 1b: rma(yi, vi, mods = X).
        tau2_est = float(fit_blocks([np.array([yi]) for yi in y],
                                    [np.array([[vi]]) for vi in v],
                                    [X[i:i + 1, :] for i in range(k)],
                                    two_level=True).tau2)

        # Final weighted regression.
        weights_final = 1.0 / (v + tau2_est)
        final_model = sm.WLS(y, X, weights=weights_final).fit()

        # --- TRUNCATED KNAPP-HARTUNG ADJUSTMENT ---
        # statsmodels naturally applies a KH scale (the reduced chi-squared).
        # If scale < 1.0, it shrinks SEs, artificially inflating Type I error rates.
        # The ad hoc variant of Knapp & Hartung (2003) truncates this scale at
        # 1.0 (metafor's test = "adhoc"; test = "knha" does not truncate).
        kh_scale = max(1.0, float(final_model.scale))

        try:
            # Extract normalized covariance and apply the truncated scale
            cov_betas = np.asarray(final_model.normalized_cov_params * kh_scale, dtype=np.float64)
            se_betas = np.sqrt(np.diag(cov_betas))
        except Exception:
            # Fallback for pathological/singular matrices
            cov_betas = np.diag(np.asarray(final_model.bse, dtype=np.float64) ** 2)
            se_betas = np.asarray(final_model.bse, dtype=np.float64)
            kh_scale = float(final_model.scale)

        betas = np.asarray(final_model.params, dtype=np.float64)

        # Per-parameter residual df vector
        df_resid_scalar = float(final_model.df_resid)
        resid_df = np.full(p, df_resid_scalar, dtype=np.float64)

        # Recalculate robust p-values using the truncated SEs
        with np.errstate(divide='ignore', invalid='ignore'):
            t_stats = betas / se_betas
        p_values = 2.0 * (1.0 - t.cdf(np.abs(t_stats), df_resid_scalar))

        # --- Omnibus tests (2-level): Wald F via statsmodels ---
        omnibus = None
        try:
            # Column groups for joint tests
            groups = {}
            for idx, name in enumerate(term_names):
                if idx == 0:
                    continue  # intercept
                s = str(name)
                factor = s.split("[", 1)[0] if ("[" in s and s.endswith("]")) else s
                groups.setdefault(factor, []).append(idx)

            def _wald_f(col_idxs):
                R = np.zeros((len(col_idxs), p), dtype=np.float64)
                for r, ci in enumerate(col_idxs):
                    R[r, ci] = 1.0
                ft = final_model.f_test(R)

                # Scale the F-statistic to match the Truncated KH adjustment
                orig_scale = float(final_model.scale)
                adjusted_F = float(np.ravel(ft.fvalue)[0]) * (orig_scale / kh_scale)
                df_n = int(ft.df_num)
                df_d = float(ft.df_denom)

                # Recalculate p-value from F distribution
                adjusted_p = float(1.0 - f.cdf(adjusted_F, df_n, df_d))

                return {
                    'F': adjusted_F,
                    'df_num': df_n,
                    'df_denom': df_d,
                    'p_val': adjusted_p,
                    'test': 'Wald-F (Truncated KH)',
                }

            per_factor = {}
            for factor, idxs in groups.items():
                if len(idxs) > 1:
                    per_factor[factor] = _wald_f(idxs)

            overall = None
            non_int = list(range(1, p))
            if len(non_int) > 1:
                overall = _wald_f(non_int)

            if overall is not None or per_factor:
                omnibus = {'overall': overall, 'per_factor': per_factor}
        except Exception:
            omnibus = None

        return {
            'betas': betas,
            'se_betas': se_betas,
            'p_values': p_values,
            'tau_sq': float(tau2_est),
            'sigma_sq': 0.0,  # No within-study variance in 2-level
            'model_type': 'Aggregated Random-Effects (2-Level, Truncated KH)',
            'n_obs': int(k),
            'resid_df': resid_df,
            'fitted': np.asarray(final_model.fittedvalues, dtype=np.float64),
            'resid': np.asarray(final_model.resid_pearson, dtype=np.float64),
            'model': final_model,
            'var_betas_robust': cov_betas,
            'term_names': list(term_names),
            'omnibus': omnibus,
        }



class ThreeLevelRegressionEngine:
    """
    3-Level Random-Effects Meta-Regression with RVE.
    Used when moderator varies within studies.

    Mathematical implementation validated against metafor/robumeta.
    """

    @staticmethod
    def run_three_level_regression(
        reg_df: pd.DataFrame,
        moderator_col: str,
        effect_col: str,
        var_col: str,
        config: Optional['RegressionConfig'] = None,
        design_info: Optional[Dict[str, Any]] = None
    ) -> Optional[Dict[str, Any]]:
        """
        Run 3-level meta-regression with cluster-robust variance estimation.

        Generalized to p >= 2 coefficients. Backward compatible: called with
        just (reg_df, moderator_col, effect_col, var_col) it behaves exactly
        as before (single continuous moderator). When `config`/`design_info`
        are supplied, the full moderator spec (including categoricals) is
        passed to the meta-regression engine (Cell 1) and results are returned as
        length-p arrays.

        MATH: delegates estimation to _run_three_level_reml_regression_v2.
        """
        if '_run_three_level_reml_regression_v2' not in globals():
            raise RuntimeError(
                "Required function '_run_three_level_reml_regression_v2' not found. "
                "Please run Cell 1 (Environment Setup) first."
            )

        # Resolve the moderator spec for the (overloaded) engine call.
        if config is not None:
            moderator_arg = list(config.moderator_cols)
            categorical_cols = list(config.categorical_cols)
            reference_levels = dict(config.reference_levels)
        else:
            moderator_arg = moderator_col          # legacy string path
            categorical_cols = None
            reference_levels = None

        # Term metadata (for naming + fitted-value reconstruction). Falls back
        # to a simple [Intercept, moderator] design for the legacy path.
        if design_info is not None:
            X_design = np.asarray(design_info['X'], dtype=np.float64)
            term_names = list(design_info['term_names'])
        else:
            X_design = None
            term_names = ['Intercept', str(moderator_col)]

        # (A centred copy of reg_df used to be built here and then never
        # passed to the engine -- dead code. _run_three_level_reml_regression_v2
        # centres continuous moderators itself, by the same column mean
        # build_design_matrix uses, so design_info['X'] and the fitted betas
        # stay on one scale.)

        est, info, opt_result = _run_three_level_reml_regression_v2(
            reg_df, moderator_arg, effect_col, var_col,
            categorical_cols=categorical_cols,
            reference_levels=reference_levels,
        )

        if est is None:
            return None

        betas = np.asarray(est.get('betas', []), dtype=np.float64).ravel()
        p = betas.size
        if p < 2:
            return None

        se_betas = np.asarray(est['se_betas'], dtype=np.float64).ravel()
        tau_sq = est['tau_sq']
        sigma_sq = est['sigma_sq']
        var_betas_robust = est.get('var_betas_robust', est.get('cov_beta'))
        model_type = est.get('model_type', '3-Level REML with Cluster-Robust SE')

        # Per-parameter df vector (length p).
        if 'df' in est:
            df_resid = np.asarray(est['df'], dtype=np.float64).ravel()
            if df_resid.size != p:
                df_resid = np.full(p, float(df_resid.flat[0]))
        else:
            k_studies = reg_df['id'].nunique()
            df_resid = np.full(p, max(1, k_studies - p))

        # p-values from t with per-parameter df (recomputed here for
        # consistency; the analyzer may recompute again under dist_type).
        t_stats = betas / se_betas
        p_values = 2 * (1 - t.cdf(np.abs(t_stats), df_resid))

        # Fitted values / residuals. Use the design matrix when available
        # (multi-moderator); otherwise reconstruct from the single moderator.
        if X_design is not None and X_design.shape[1] == p:
            fitted = X_design @ betas
        else:
            X_mod = reg_df[moderator_col].values.astype(np.float64)
            fitted = betas[0] + betas[1] * X_mod
        resid = reg_df[effect_col].values.astype(np.float64) - fitted

        return {
            'betas': betas,
            'se_betas': se_betas,
            'p_values': np.asarray(p_values, dtype=np.float64).ravel(),
            'tau_sq': tau_sq,
            'sigma_sq': sigma_sq,
            'model_type': model_type,
            'n_obs': len(reg_df),
            'resid_df': df_resid,
            'fitted': np.asarray(fitted, dtype=np.float64),
            'resid': np.asarray(resid, dtype=np.float64),
            'var_betas_robust': var_betas_robust,
            'term_names': term_names,
            'omnibus': est.get('omnibus'),
            'htz_wald': est.get('htz_wald'),
            'fit_inputs': {k: est.get(k) for k in
                           ('vcv_used', 'X_stacked', 'y_stacked', 'row_ids')},
        }


# =============================================================================
# REGRESSION ANALYZER (STATISTICAL COMPUTATIONS)
# =============================================================================

class RegressionAnalyzer:
    """
    Statistical analysis engine for meta-regression.
    Handles model selection, diagnostics, and heterogeneity calculations.
    """

    def __init__(
        self,
        effect_col: str,
        var_col: str,
        global_settings: Dict[str, Any]
    ):
        """
        Initialize analyzer.

        Args:
            effect_col: Effect size column name
            var_col: Variance column name
            global_settings: Dict with 'alpha' and 'dist_type'
        """
        self.effect_col = effect_col
        self.var_col = var_col
        self.alpha = global_settings.get('alpha', 0.05)
        self.dist_type = global_settings.get('dist_type', 't')
        self.ci_level = (1 - self.alpha) * 100

    # -------------------------------------------------------------------------
    # MODEL SELECTION & AGGREGATION
    # -------------------------------------------------------------------------
    def all_moderators_constant_within_study(
        self,
        reg_df: pd.DataFrame,
        moderator_cols: List[str]
    ) -> bool:
        """
        True if EVERY listed moderator takes a single value within each study.

        The 2-level aggregated path is only valid when there is no
        within-study variation to model for any selected moderator. With
        multiple moderators this must hold jointly: if even one moderator
        varies within a study, the data are genuinely 3-level.
        """
        for mod in moderator_cols:
            # nunique per study; if any study has >1 distinct value, not constant.
            if reg_df.groupby('id')[mod].nunique(dropna=False).max() > 1:
                return False
        return True

    def aggregate_to_study_level_multi(
        self,
        reg_df: pd.DataFrame,
        moderator_cols: List[str]
    ) -> pd.DataFrame:
        """
        Aggregate effect sizes to study level, preserving ALL moderators.

        Inverse-variance aggregation of the effect size and its variance is
        identical to the single-moderator method:
            y_bar_i = sum(w_ij y_ij) / sum(w_ij),   w_ij = 1 / v_ij
            v_bar_i = 1 / sum(w_ij)
        Each moderator is carried through by taking its (constant) value
        within the study. PRECONDITION: all_moderators_constant_within_study
        is True; otherwise the .iloc[0] pick would silently drop variation.

        Args:
            reg_df: Regression DataFrame (multiple effects per study allowed).
            moderator_cols: All selected moderators (continuous or categorical).

        Returns:
            Aggregated DataFrame, one row per study, containing 'id',
            the effect column, the variance column, and every moderator.
        """
        reg_df = reg_df.copy()
        reg_df['wi'] = 1.0 / reg_df[self.var_col]

        def agg_func(x):
            out = {
                self.effect_col: np.average(x[self.effect_col], weights=x['wi']),
                self.var_col: 1.0 / np.sum(x['wi']),
            }
            # Carry each moderator's within-study (constant) value through.
            for mod in moderator_cols:
                out[mod] = x[mod].iloc[0]
            return pd.Series(out)

        try:
            agg_df = reg_df.groupby('id').apply(
                agg_func, include_groups=False
            ).reset_index()
        except TypeError:
            # Fallback for older pandas versions (no include_groups kwarg).
            agg_df = reg_df.groupby('id').apply(agg_func).reset_index()

        return agg_df

    def aggregate_to_study_level(
        self,
        reg_df: pd.DataFrame,
        moderator_col: str
    ) -> pd.DataFrame:
        """
        Aggregate effect sizes to study level using inverse-variance weights.

        MATH PRESERVED: Original aggregation logic

        Args:
            reg_df: Regression DataFrame
            moderator_col: Moderator column name

        Returns:
            Aggregated DataFrame (one row per study)
        """

    # -------------------------------------------------------------------------
    # CONFIDENCE INTERVALS
    # -------------------------------------------------------------------------
    def calculate_confidence_intervals(
        self,
        betas: np.ndarray,
        se_betas: np.ndarray,
        df_vec
    ) -> List[Tuple[float, float]]:
        """
        Calculate confidence intervals for all regression coefficients.

        Generalized to p coefficients. Each coefficient uses its own
        degrees of freedom from df_vec (per-parameter Satterthwaite df on
        the 3-level path; a constant residual df on the 2-level path).

        Args:
            betas: Coefficient estimates, shape (p,).
            se_betas: Standard errors, shape (p,).
            df_vec: Per-parameter degrees of freedom, shape (p,).

        Returns:
            List of p (lower, upper) tuples, in coefficient order.
        """
        betas = np.asarray(betas, dtype=np.float64).ravel()
        se_betas = np.asarray(se_betas, dtype=np.float64).ravel()
        df_vec = np.asarray(df_vec, dtype=np.float64).ravel()
        p = betas.size

        # Broadcast a scalar/short df vector to length p (defensive).
        if df_vec.size != p:
            df_vec = np.full(p, float(df_vec.flat[0]))

        q_val = 1 - (self.alpha / 2)

        if self.dist_type == 't':
            crit = t.ppf(q_val, df_vec)            # vectorized over p
        else:
            crit = np.full(p, norm.ppf(q_val))

        lower = betas - crit * se_betas
        upper = betas + crit * se_betas

        return [(float(lower[i]), float(upper[i])) for i in range(p)]


    def recalculate_p_values(
        self,
        betas: np.ndarray,
        se_betas: np.ndarray,
        df_vec
    ) -> np.ndarray:
        """
        Recalculate two-sided p-values for all coefficients.

        Generalized to p coefficients, each using its own df under the
        t-distribution (or the normal approximation when dist_type != 't').

        Args:
            betas: Coefficient estimates, shape (p,).
            se_betas: Standard errors, shape (p,).
            df_vec: Per-parameter degrees of freedom, shape (p,).

        Returns:
            Array of p p-values, in coefficient order.
        """
        betas = np.asarray(betas, dtype=np.float64).ravel()
        se_betas = np.asarray(se_betas, dtype=np.float64).ravel()
        t_stats = betas / se_betas

        if self.dist_type == 't':
            df_vec = np.asarray(df_vec, dtype=np.float64).ravel()
            if df_vec.size != t_stats.size:
                df_vec = np.full(t_stats.size, float(df_vec.flat[0]))
            return 2.0 * (1.0 - t.cdf(np.abs(t_stats), df_vec))
        else:
            return 2.0 * (1.0 - norm.cdf(np.abs(t_stats)))

    # -------------------------------------------------------------------------
    # HETEROGENEITY METRICS
    # -------------------------------------------------------------------------

    def calculate_residual_heterogeneity(
        self,
        tau_sq: float,
        sigma_sq: float,
        mean_variance: float
    ) -> float:
        """
        Calculate residual I² statistic.

        Args:
            tau_sq: Between-study variance
            sigma_sq: Within-study variance
            mean_variance: Mean sampling variance

        Returns:
            Residual I² as percentage
        """
        if tau_sq <= 0:
            return 0.0

        total_var = tau_sq + sigma_sq + mean_variance
        if total_var <= 0:
            return 0.0

        resid_i2 = ((tau_sq + sigma_sq) / total_var) * 100
        return max(0.0, min(100.0, resid_i2))

    # -------------------------------------------------------------------------
    # INFLUENCE DIAGNOSTICS
    # -------------------------------------------------------------------------

    def calculate_influence_diagnostics(
        self,
        model_results: Dict[str, Any],
        reg_df: pd.DataFrame,
        moderator_col: str
    ) -> Dict[str, Any]:
        """
        Calculate Cook's distance and influence metrics.

        2-level (aggregated) models: uses statsmodels OLSInfluence on the
        fitted WLS model. The influence threshold is the standard Cook's D
        cutoff 4 / (n - p), which depends on the number of parameters p and
        therefore generalizes correctly to multi-moderator models. (For
        p = 2 and large-ish n this is numerically close to the old 4/n, but
        4/(n - p) is the conventional form and is the right one once p > 2.)

        3-level models: statsmodels OLSInfluence does not apply (there is no
        single OLS object); returns a zero placeholder, as before. Per-cluster
        multilevel influence is handled separately (leave-one-out cell).

        Args:
            model_results: Results dict from the regression engine.
            reg_df: Regression DataFrame used in the fit.
            moderator_col: Primary moderator name (kept for signature
                compatibility; not required by the computation).

        Returns:
            Dict with 'cooks_d' (np.ndarray) and 'has_influential' (bool).
        """
        try:
            n = len(reg_df)

            # 2-level aggregated models expose the statsmodels WLS object.
            if 'model' in model_results:
                from statsmodels.stats.outliers_influence import OLSInfluence

                model = model_results['model']
                influence = OLSInfluence(model)
                cooks_d = np.asarray(influence.cooks_distance[0], dtype=np.float64)

                # Number of parameters (including intercept). Prefer the
                # design width via the fitted model; fall back to betas.
                p = int(getattr(model, 'df_model', 0)) + 1
                if p <= 1:
                    p = int(np.asarray(model_results['betas']).ravel().size)

                denom = max(1, n - p)
                threshold = 4.0 / denom

                return {
                    'cooks_d': cooks_d,
                    'has_influential': bool(np.any(cooks_d > threshold)),
                }
            else:
                # 3-level: no OLS object — zero placeholder (p-agnostic).
                return {
                    'cooks_d': np.zeros(n),
                    'has_influential': False,
                }
        except Exception:
            return {
                'cooks_d': np.zeros(len(reg_df)),
                'has_influential': False,
            }

# =============================================================================
# REGRESSION ORCHESTRATOR
# =============================================================================

class RegressionEngine:
    """
    High-level orchestrator for meta-regression analysis.
    Coordinates model selection, estimation, and diagnostics.
    """

    def __init__(self, data_manager: 'RegressionDataManager'):
        """
        Initialize engine with data manager.

        Args:
            data_manager: RegressionDataManager instance
        """
        self.data_manager = data_manager
        self.analyzer = RegressionAnalyzer(
            effect_col=data_manager.effect_col,
            var_col=data_manager.var_col,
            global_settings=data_manager.global_settings
        )
        self.two_level = TwoLevelRegressionEngine()
        self.three_level = ThreeLevelRegressionEngine()

    def _compute_pseudo_r2(
        self,
        reg_df: pd.DataFrame,
        config: 'RegressionConfig',
        design_info: Dict[str, Any],
        three_level: bool,
        reported: Optional[Tuple[float, float, str]] = None,
    ) -> Dict[str, Any]:
        """Pseudo-R² = (Σσ²_null − Σσ²_model) / Σσ²_null (Raudenbush, 2009).

        Both models are fitted by REML with the unified engine (Cell 1b) on
        exactly the same rows (reg_df, after listwise deletion), the same
        sampling V (sliced by row identity) and the same random structure;
        the null is the intercept-only model and is fitted automatically:

            three_level = True   tau2 + sigma2 estimated in both models
            three_level = False  one effect per study: a single between-study
                                 component (sigma2 fixed at 0) in both models

        The total and each per-level reduction are truncated at 0 (the raw
        values are kept). `reported` = (tau2, sigma2, model_type) of the model
        shown to the user; if its components differ from the REML fit used
        here (e.g. a diagonal-V or aggregated fallback), a warning says so.
        Returns plain floats/strings only (JSON-safe).
        """
        if 'pseudo_r2_report' not in globals():
            return {'R2': np.nan, 'error': 'unified model engine not loaded (run Cell 1b)'}
        try:
            X_df = pd.DataFrame(np.asarray(design_info['X'], dtype=float),
                                index=reg_df.index,
                                columns=list(design_info['term_names']))
            spec = build_fit_spec(
                reg_df, config.effect_col, config.var_col,
                X=X_df,
                vcv_dict=self.data_manager._config.get('vcv_matrices', {}),
                vcv_reference_df=self.data_manager.analysis_data,
                label='meta-regression',
            )
            return pseudo_r2_report(spec, three_level, reported)
        except Exception as e:
            return {'R2': np.nan, 'error': f"{type(e).__name__}: {e}"}

    # -------------------------------------------------------------------------
    # MAIN EXECUTION METHOD
    # -------------------------------------------------------------------------

    def run_meta_regression(
        self,
        moderator_col=None,
        progress_callback: Optional[callable] = None,
        moderator_cols: Optional[List[str]] = None,
        categorical_cols: Optional[List[str]] = None,
        reference_levels: Optional[Dict[str, str]] = None,
    ) -> Optional['RegressionResult']:
        """
        Execute complete meta-regression analysis (single or multi-moderator).

        Backward compatible: callers passing a single string `moderator_col`
        get the original behavior. New callers pass `moderator_cols` (a list)
        plus optional `categorical_cols` / `reference_levels`.

        ROUTING (Decision 1a — preserved from original): if any study has
        more than one effect size, the 3-level path is used; otherwise the
        2-level aggregated path is used.

        Args:
            moderator_col: Single moderator name (legacy) OR None.
            progress_callback: Optional progress callback.
            moderator_cols: List of moderator names (new multi-moderator API).
            categorical_cols: Subset of moderator_cols to treat as factors.
            reference_levels: Optional {col: reference_level} overrides.

        Returns:
            (RegressionResult, reg_df_for_plot) tuple, or None on failure.
        """
        # --- Normalize the moderator specification ---
        if moderator_cols is None:
            if moderator_col is None:
                if progress_callback:
                    progress_callback("❌ No moderator specified")
                return None
            moderator_cols = [moderator_col]
        categorical_cols = list(categorical_cols or [])

        # Primary moderator name, used for legacy single-moderator fields
        # (RegressionResult.moderator_col_name, plot x-axis, etc.).
        primary_mod = moderator_cols[0]

        # Prepare data
        if progress_callback:
            progress_callback("📊 Preparing data...")

        try:
            reg_df, config, design_info = self.data_manager.prepare_regression_data_multi(
                moderator_cols=moderator_cols,
                categorical_cols=categorical_cols,
                reference_levels=reference_levels,
            )
        except ValueError as e:
            if progress_callback:
                progress_callback(f"❌ Data preparation failed: {str(e)}")
            return None

        # Check data structure (per-moderator variation; use primary for the
        # legacy multiplicity flags, which are moderator-agnostic anyway).
        variation_info = self.data_manager.check_within_study_variation(
            reg_df, primary_mod
        )

        k_studies = variation_info['k_studies']
        n_obs = variation_info['n_obs']
        has_multiple_effects = variation_info['has_multiple_effects']

        if progress_callback:
            progress_callback(f"📈 k = {k_studies} studies, n = {n_obs} observations")

        # =================================================================
        # DECISION LOGIC (1a: route on within-study effect multiplicity)
        # =================================================================

        if not has_multiple_effects:
            # 2-level structure → aggregated design-matrix regression.
            if progress_callback:
                progress_callback("🔄 Running 2-level aggregated regression...")

            # Aggregate to study level, preserving ALL moderators.
            agg_df = self.analyzer.aggregate_to_study_level_multi(
                reg_df, moderator_cols
            )

            # Rebuild the design matrix on the AGGREGATED rows (one per study),
            # so X is row-aligned with the aggregated y/v.
            agg_design = self.data_manager.build_design_matrix(agg_df, config)

            model_results = self.two_level.run_aggregated_regression_design(
                y=agg_df[config.effect_col].values,
                v=agg_df[config.var_col].values,
                X=agg_design['X'],
                term_names=agg_design['term_names'],
            )

            reg_df_for_plot = agg_df
            active_design = agg_design

        else:
            # Multiple effects per study → 3-level model (multi-moderator,
            # categorical, and mixed all supported via the generalized engine).
            if progress_callback:
                progress_callback("🔄 Running 3-level regression with RVE...")

            # Pass the full moderator spec through: config carries
            # moderator_cols / categorical_cols / reference_levels, and
            # design_info carries the treatment-coded design matrix and
            # term metadata. The generalized engine consumes both
            # and delegates estimation to the overloaded Cell 9.5 function.
            model_results = self.three_level.run_three_level_regression(
                reg_df,
                primary_mod,                 # legacy positional (primary name)
                config.effect_col,
                config.var_col,
                config=config,
                design_info=design_info,
            )

            if model_results is None:
                if progress_callback:
                    progress_callback("❌ Optimization failed")
                return None

            reg_df_for_plot = reg_df
            active_design = design_info


        # =================================================================
        # POST-PROCESSING (PRESERVED FROM ORIGINAL)
        # =================================================================

        # Extract coefficients as arrays (length p).
        betas = np.asarray(model_results['betas'], dtype=np.float64).ravel()
        se_betas = np.asarray(model_results['se_betas'], dtype=np.float64).ravel()
        tau_sq = model_results['tau_sq']
        sigma_sq = model_results['sigma_sq']
        df_resid = np.asarray(model_results['resid_df'], dtype=np.float64).ravel()
        model_type = model_results['model_type']
        fitted = model_results['fitted']
        resid = model_results['resid']
        var_betas_robust = model_results['var_betas_robust']
        term_names = model_results.get(
            'term_names', active_design['term_names']
        )

        # Ensure df_resid is length p (broadcast a scalar/short vector).
        p = betas.size
        if df_resid.size != p:
            df_resid = np.full(p, float(df_resid.flat[0]))

        # Recalculate inference for consistency under the active dist_type
        # (vectorized over all p coefficients).
        p_values = self.analyzer.recalculate_p_values(
            betas, se_betas, df_resid
        )

        # Confidence intervals for all p coefficients (vectorized).
        cis = self.analyzer.calculate_confidence_intervals(
            betas, se_betas, df_resid
        )


        # Pseudo-R²: moderator model vs the intercept-only model fitted by
        # REML on the SAME rows, sampling V and random structure (Cell 1b).
        r2_detail = self._compute_pseudo_r2(
            reg_df, config, design_info,
            three_level=bool(has_multiple_effects),
            reported=(tau_sq, sigma_sq, model_type),
        )
        r2 = r2_detail.get('R2', np.nan)
        if progress_callback and r2_detail.get('error'):
            progress_callback(f"⚠️ Pseudo-R² not computable: {r2_detail['error']}")


        # "Typical" within-study sampling variance for the residual I².
        # Use the Higgins & Thompson (2002) estimator (the s² used in the
        # I² denominator), not the arithmetic mean of v_i: ecological
        # sampling variances are heavily right-skewed, and the arithmetic
        # mean over-weights imprecise studies, which deflates residual I².
        #   s² = (k - 1) * Σw_i / [ (Σw_i)² - Σw_i² ],  w_i = 1 / v_i
        _w = 1.0 / reg_df_for_plot[config.var_col].values.astype(float)
        _k = len(_w)
        _sw = np.sum(_w)
        _denom = _sw ** 2 - np.sum(_w ** 2)
        if _k > 1 and np.isfinite(_denom) and _denom > 0:
            mean_v = (_k - 1) * _sw / _denom
        else:
            # Degenerate (k = 1, or all-equal weights making denom ~ 0):
            # fall back to the arithmetic mean.
            mean_v = float(np.mean(reg_df_for_plot[config.var_col]))

        resid_i2 = self.analyzer.calculate_residual_heterogeneity(
            tau_sq, sigma_sq, mean_v
        )

        # Calculate influence diagnostics
        influence_metrics = self.analyzer.calculate_influence_diagnostics(
            model_results, reg_df_for_plot, moderator_col
        )

        # Build result object
        result = RegressionResult(
            betas=betas,
            se_betas=se_betas,
            p_values=p_values,
            cis=cis,
            term_names=list(term_names),
            tau_sq=tau_sq,
            sigma_sq=sigma_sq,
            model_type=model_type,
            k_studies=k_studies,
            n_obs=n_obs,
            df_resid=df_resid,
            r_squared=r2,
            r2_detail=r2_detail,
            data_accounting=(self.data_manager.last_accounting.to_dict()
                             if getattr(self.data_manager, 'last_accounting', None) else None),
            resid_i2=resid_i2,
            fitted=fitted,
            resid=resid,
            var_betas_robust=var_betas_robust,
            moderator_col_name=primary_mod,
            has_influential=influence_metrics['has_influential'],
            cooks_d=influence_metrics['cooks_d'],
            omnibus=model_results.get('omnibus'),
            col_meta=(design_info or {}).get('col_meta'),
            reference_levels=(design_info or {}).get('reference_levels'),
            htz_wald=model_results.get('htz_wald'),
            fit_inputs=model_results.get('fit_inputs'),
            term_groups=(design_info or {}).get('term_groups'),
            categorical_cols=list(getattr(config, 'categorical_cols', None) or []),
            moderator_cols=list(getattr(config, 'moderator_cols', None) or []),
        )

        # Remember the last fit so predict_meta_regression() can be called
        # from any later cell without threading the object through by hand.
        globals()['LAST_REGRESSION_RESULT'] = result

        if progress_callback:
            if betas.size >= 2:
                p_slope = float(p_values[1])
                sig = ("***" if p_slope < 0.001 else "**" if p_slope < 0.01
                       else "*" if p_slope < 0.05 else "ns")
                if betas.size == 2:
                    progress_callback(
                        f"✅ β₁ = {float(betas[1]):.4f} {sig}, pseudo-R² = {fmt_r2(r2)}"
                    )
                else:
                    progress_callback(
                        f"✅ {betas.size - 1} moderator terms, pseudo-R² = {fmt_r2(r2)}"
                    )
            else:
                progress_callback(f"✅ pseudo-R² = {fmt_r2(r2)}")

        return result, reg_df_for_plot

# =============================================================================
# PRESENTATION LAYER
# =============================================================================

import pandas as pd
import numpy as np
from typing import Dict, Any, Optional
import datetime
import ipywidgets as widgets
from IPython.display import display, HTML


# =============================================================================
# HTML TEMPLATE GENERATORS
# =============================================================================

class RegressionHTMLTemplates:
    """
    Static HTML template generators for meta-regression visualizations.
    All methods are pure functions returning HTML strings.
    """

    @staticmethod
    def gradient_card(
        title: str,
        value: str,
        subtitle: str = "",
        gradient: str = "linear-gradient(135deg, #2c3e50 0%, #3498db 100%)"
    ) -> str:
        """Generate sleek gradient strip for highlighting key statistics"""
        return f"""
        <div style='background: {gradient}; padding: 15px 25px; border-radius: 8px;
                    color: white; margin-bottom: 20px; display: flex; justify-content: space-between; align-items: center;'>
            <div>
                <div style='font-size: 0.85em; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85;'>{title}</div>
                <div style='margin: 0; font-size: 2.2em; font-weight: bold;'>{value}</div>
            </div>
            <div>
                <span style='background: rgba(255,255,255,0.2); padding: 5px 15px; border-radius: 20px; font-size: 1.2em; font-weight: bold;'>
                    {subtitle}
                </span>
            </div>
        </div>
        """

    @staticmethod
    def dashboard_header(title: str, badge_text: str) -> str:
        """Generate a modern dashboard-style header with a badge"""
        return f"""
        <div style='display: flex; align-items: center; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 2px solid #ecf0f1; font-family: system-ui, -apple-system, sans-serif;'>
            <div style='background-color: #2c3e50; color: white; padding: 8px 16px; border-radius: 6px; font-weight: 600; font-size: 1.1em; letter-spacing: 0.5px; margin-right: 15px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
                {title}
            </div>
            <div style='color: #7f8c8d; font-size: 1.1em; display: flex; align-items: center;'>
                <span style='margin-right: 10px;'>Moderator:</span>
                <span style='background-color: #e8f4f8; color: #0984e3; padding: 4px 12px; border-radius: 20px; font-family: monospace; font-weight: bold; border: 1px solid #74b9ff;'>
                    {badge_text}
                </span>
            </div>
        </div>
        """

    @staticmethod
    def info_grid(items: list) -> str:
        """
        Generate 2-column grid of info boxes.

        Args:
            items: List of dicts with 'label' and 'value' keys
        """
        html = "<div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 20px;'>"

        for item in items:
            color = item.get('color', '#007bff')
            html += f"""
            <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px;
                        border-left: 4px solid {color};'>
                <div style='color: #6c757d; font-size: 0.9em;'>{item['label']}</div>
                <div style='font-size: 1.3em; font-weight: bold;'>{item['value']}</div>
            </div>
            """

        html += "</div>"
        return html

    @staticmethod
    def interpretation_box(content: str, is_significant: bool = False) -> str:
        """Generate interpretation box with appropriate styling"""
        bg_color = "#e7f3ff" if is_significant else "#f8f9fa"
        return f"""
        <div style='background-color: {bg_color}; padding: 20px; border-radius: 5px;
                    margin-bottom: 20px;'>
            <h4 style='margin-top: 0; color: #2c3e50;'>Interpretation</h4>
            <p style='margin: 0; font-size: 1.05em;'>{content}</p>
        </div>
        """

    @staticmethod
    def coefficient_table(
        betas: np.ndarray,
        se_betas: np.ndarray,
        p_values: np.ndarray,
        cis: list,
        df_vec: np.ndarray,
        term_names: list,
        ci_level: float = 95
    ) -> str:
        """Generate a styled coefficient table with one row per term.

        Generalized to p coefficients. Row 0 is the intercept; subsequent
        rows are moderators / dummy columns, labeled by term_names. Each row
        shows the term's own degrees of freedom from df_vec (per-parameter
        Satterthwaite df on the 3-level path; constant residual df on the
        2-level path).
        """
        betas = np.asarray(betas, dtype=float).ravel()
        se_betas = np.asarray(se_betas, dtype=float).ravel()
        p_values = np.asarray(p_values, dtype=float).ravel()
        df_vec = np.asarray(df_vec, dtype=float).ravel()
        p = betas.size
        if df_vec.size != p:
            df_vec = np.full(p, float(df_vec.flat[0]))

        def format_p(pv):
            return "&lt;0.001" if pv < 0.001 else f"{pv:.4g}"

        header = (
            """
        <table style='width: 100%; border-collapse: collapse; margin: 20px 0;'>
            <thead style='background-color: #2c3e50; color: white;'>
                <tr>
                    <th style='padding: 12px; text-align: left; border: 1px solid #dee2e6;'>Term</th>
                    <th style='padding: 12px; text-align: center; border: 1px solid #dee2e6;'>Estimate (β)</th>
                    <th style='padding: 12px; text-align: center; border: 1px solid #dee2e6;'>SE</th>
                    <th style='padding: 12px; text-align: center; border: 1px solid #dee2e6;'>t-value</th>
                    <th style='padding: 12px; text-align: center; border: 1px solid #dee2e6;'>p-value</th>
                    <th style='padding: 12px; text-align: center; border: 1px solid #dee2e6;'>df</th>
                    <th style='padding: 12px; text-align: center; border: 1px solid #dee2e6;'>"""
            + f"{int(ci_level)}% CI"
            + """</th>
                </tr>
            </thead>
            <tbody>
        """
        )

        rows = []
        for i in range(p):
            is_intercept = (i == 0)
            sig = (not is_intercept) and (p_values[i] < _alpha())
            sig_style = "font-weight: bold; color: #28a745;" if sig else ""
            # Zebra striping: even rows get the light background.
            row_bg = "background-color: #f8f9fa;" if (i % 2 == 0) else ""
            tval = betas[i] / se_betas[i] if se_betas[i] != 0 else float('nan')
            lo, hi = cis[i]
            label = term_names[i] if i < len(term_names) else f"β{i}"
            label_html = f"<b>{label}</b>"

            rows.append(f"""
                <tr style='{row_bg}'>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'>{label_html}</td>
                    <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6; {sig_style}'>{betas[i]:.4f}</td>
                    <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>{se_betas[i]:.4f}</td>
                    <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>{tval:.2f}</td>
                    <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6; {sig_style}'>{format_p(p_values[i])}</td>
                    <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>{df_vec[i]:.1f}</td>
                    <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>[{lo:.4f}, {hi:.4f}]</td>
                </tr>
            """)

        return header + "".join(rows) + """
            </tbody>
        </table>
        """
    @staticmethod
    def model_summary_box(
        model_type: str,
        k_studies: int,
        n_obs: int,
        df,
        tau_sq: float,
        sigma_sq: float,
        r2: float,
        resid_i2: float,
        n_coef: Optional[int] = None,
        r2_detail: Optional[Dict[str, Any]] = None,
    ) -> str:
        """Generate model summary information box.

        `df` may be a scalar or a per-parameter vector (np.ndarray/list).
        For a vector with a single distinct value it is shown as that
        scalar; for differing per-parameter df (3-level Satterthwaite) it is
        shown as a range. `n_coef`, when provided, is displayed so the user
        can see how many terms the model has (intercept + moderators/dummies).
        """
        # Normalize df for display.
        df_arr = np.asarray(df, dtype=float).ravel()
        if df_arr.size == 0:
            df_str = "—"
        elif np.allclose(df_arr, df_arr[0]):
            df_str = f"{df_arr[0]:.1f}"
        else:
            df_str = f"{df_arr.min():.1f}–{df_arr.max():.1f} (per term)"

        html = f"""
        <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px;'>
            <p style='margin: 5px 0;'><b>Model Type:</b> {model_type}</p>
            <p style='margin: 5px 0;'><b>Studies (k):</b> {k_studies}</p>
            <p style='margin: 5px 0;'><b>Observations (n):</b> {n_obs}</p>
        """

        if n_coef is not None:
            html += f"<p style='margin: 5px 0;'><b>Coefficients (incl. intercept):</b> {n_coef}</p>"

        html += f"""
            <p style='margin: 5px 0;'><b>Degrees of Freedom:</b> {df_str}</p>
            <p style='margin: 5px 0;'><b>Between-Study Variance (τ²):</b> {tau_sq:.4f}</p>
        """

        if sigma_sq > 0:
            html += f"<p style='margin: 5px 0;'><b>Within-Study Variance (σ²):</b> {sigma_sq:.4f}</p>"

        if r2_detail:
            html += r2_breakdown_html(r2_detail)
        else:
            html += f"<p style='margin: 5px 0;'><b>Pseudo-R²:</b> {fmt_r2(r2)}</p>"

        html += f"<p style='margin: 5px 0;'><b>Residual I²:</b> {resid_i2:.1f}%</p></div>"
        return html

    @staticmethod
    def assumption_table(sigma_sq: float) -> str:
        """Generate model assumptions checklist"""
        return f"""
        <table style='width: 100%; border-collapse: collapse;'>
            <thead style='background-color: #f8f9fa;'>
                <tr>
                    <th style='padding: 10px; text-align: left; border: 1px solid #dee2e6;'>Assumption</th>
                    <th style='padding: 10px; text-align: left; border: 1px solid #dee2e6;'>Assessment</th>
                    <th style='padding: 10px; text-align: left; border: 1px solid #dee2e6;'>Status</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'><b>Linearity</b></td>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'>Check scatter plot in dedicated plot cell</td>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'>⚠️ Visual check needed</td>
                </tr>
                <tr style='background-color: #f8f9fa;'>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'><b>Independence</b></td>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'>{'Cluster-robust SE used' if sigma_sq > 0 else 'Aggregated to study level'}</td>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'>✓ Accounted for</td>
                </tr>
                <tr>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'><b>Normality</b></td>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'>Residuals approximately normal (t-distribution)</td>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'>✓ Assumed</td>
                </tr>
                <tr style='background-color: #f8f9fa;'>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'><b>Homoscedasticity</b></td>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'>Weighted by inverse variance</td>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'>✓ Weighted regression</td>
                </tr>
            </tbody>
        </table>
        """
    @staticmethod
    def variance_covariance_matrix(
        var_betas: np.ndarray,
        term_names: Optional[list] = None
    ) -> str:
        """Generate a styled p x p variance-covariance matrix display.

        Generalized from the fixed 2x2 layout to any number of coefficients.
        Diagonal entries are Var(β_i); off-diagonals are Cov(β_i, β_j).
        When term_names is provided, rows and columns are labeled.
        """
        V = np.asarray(var_betas, dtype=float)
        if V.ndim != 2 or V.shape[0] != V.shape[1]:
            return (
                "<div style='background-color: #fff3cd; padding: 12px; "
                "border-radius: 5px; border-left: 4px solid #ffc107;'>"
                "Variance-covariance matrix unavailable or malformed.</div>"
            )

        p = V.shape[0]
        labels = None
        if term_names is not None and len(term_names) == p:
            labels = list(term_names)

        # Build header row (only if we have labels).
        head = ""
        if labels is not None:
            head = "<tr><th style='padding: 8px; border: 1px solid #dee2e6;'></th>"
            for name in labels:
                head += (
                    f"<th style='padding: 8px; border: 1px solid #dee2e6; "
                    f"background-color: #2c3e50; color: white; "
                    f"font-family: monospace;'>{name}</th>"
                )
            head += "</tr>"

        body = ""
        for i in range(p):
            body += "<tr>"
            if labels is not None:
                body += (
                    f"<th style='padding: 8px; border: 1px solid #dee2e6; "
                    f"text-align: left; background-color: #2c3e50; color: white; "
                    f"font-family: monospace;'>{labels[i]}</th>"
                )
            for j in range(p):
                # Faintly tint the diagonal so variances stand out.
                cell_bg = "background-color: #eef3f7;" if i == j else ""
                body += (
                    f"<td style='padding: 10px; border: 1px solid #dee2e6; "
                    f"text-align: center; {cell_bg}'>{V[i, j]:.6f}</td>"
                )
            body += "</tr>"

        return f"""
        <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px; margin-bottom: 20px; overflow-x: auto;'>
            <table style='margin: 10px auto; border-collapse: collapse;'>
                {head}
                {body}
            </table>
            <p style='margin: 10px 0 0 0; text-align: center; font-size: 0.9em;'>
                <i>Diagonal: Var(β<sub>i</sub>); off-diagonal: Cov(β<sub>i</sub>, β<sub>j</sub>).</i>
            </p>
        </div>
        """

    @staticmethod
    def card(content: str, bg_color: str = '#f8f9fa') -> str:
        """Generate simple card container"""
        return f"""
        <div style='background-color: {bg_color}; padding: 15px; border-radius: 5px; margin-bottom: 20px;'>
            {content}
        </div>
        """

    @staticmethod
    def header(text: str, level: int = 3) -> str:
        """Generate styled header"""
        return f"<h{level} style='color: #2c3e50;'>{text}</h{level}>"

    @staticmethod
    def build_formula_string(
        effect_col: str,
        moderator_cols: list,
        categorical_cols: Optional[list] = None,
        reference_levels: Optional[dict] = None,
        term_names: Optional[list] = None
    ) -> str:
        """Construct an R/metafor-style model formula string.

        Two modes:
          - PREVIEW (term_names is None): shows moderators by name, with
            categorical factors annotated with their reference level, e.g.
              lnRR ~ 1 + co2.elev + fungrp (ref: ANGIO)
            This is shown in the GUI before the model is run, when the exact
            surviving dummy levels are not yet known.
          - EXPANDED (term_names provided): shows the actual fitted design
            terms (intercept + continuous + dummy columns), e.g.
              lnRR ~ 1 + co2.elev + fungrp[N2FIX]
            term_names[0] is assumed to be the intercept and is rendered as 1.

        The formula is descriptive (for display), not executable code.
        """
        categorical_cols = set(categorical_cols or [])
        reference_levels = reference_levels or {}
        lhs = effect_col if effect_col else "effect"

        if term_names is not None:
            # Expanded form from the fitted design.
            rhs_terms = ["1"]
            for name in term_names[1:]:  # skip intercept
                rhs_terms.append(str(name))
            rhs = " + ".join(rhs_terms) if rhs_terms else "1"
            return f"{lhs} ~ {rhs}"

        # Preview form from the selection.
        rhs_terms = ["1"]
        for mod in moderator_cols:
            if mod in categorical_cols:
                ref = reference_levels.get(mod)
                if ref is not None:
                    rhs_terms.append(f"{mod} (ref: {ref})")
                else:
                    rhs_terms.append(f"{mod} (categorical)")
            else:
                # Indicate centering in the UI preview
                rhs_terms.append(f"{mod} (centered)")
        rhs = " + ".join(rhs_terms) if len(rhs_terms) > 1 else "1"
        return f"{lhs} ~ {rhs}"

    @staticmethod
    def formula_preview_box(
        formula: str,
        title: str = "Model Formula",
        note: Optional[str] = None
    ) -> str:
        """Render a formula string in a styled monospace preview box."""
        note_html = ""
        if note:
            note_html = (
                f"<p style='margin: 10px 0 0 0; font-size: 0.85em; "
                f"color: #6c757d;'>{note}</p>"
            )
        return f"""
        <div style='background-color: #f4f8fb; padding: 14px 18px; border-radius: 6px;
                    border-left: 4px solid #3498db; margin-bottom: 16px;'>
            <div style='font-size: 0.8em; text-transform: uppercase; letter-spacing: 1px;
                        color: #7f8c8d; margin-bottom: 6px;'>{title}</div>
            <div style='font-family: "Courier New", monospace; font-size: 1.05em;
                        color: #2c3e50; font-weight: bold;'>{formula}</div>
            {note_html}
        </div>
        """


# =============================================================================
# PUBLICATION TEXT GENERATORS
# =============================================================================

class RegressionPublicationTextGenerator:
    """
    Generates publication-ready text for manuscripts.
    """

    def __init__(self, ci_level: float = 95):
        self.ci_level = ci_level

    def generate_methods_section(
        self,
        moderator_col,
        model_type: str,
        es_config: Dict[str, Any],
        moderator_cols: Optional[list] = None,
        formula: Optional[str] = None
    ) -> str:
        """Generate Materials and Methods section.

        Backward compatible: callers passing a single `moderator_col` string
        get single-moderator phrasing. When `moderator_cols` (list) and/or
        `formula` are supplied, the text names all moderators and shows the
        model formula. Citation logic is unchanged.
        """
        # Resolve moderator naming.
        if moderator_cols:
            mod_list = list(moderator_cols)
        else:
            mod_list = [moderator_col]
        if len(mod_list) == 1:
            mod_phrase = f"the moderator <b>{mod_list[0]}</b>"
        else:
            names = ", ".join(f"<b>{m}</b>" for m in mod_list[:-1])
            mod_phrase = f"the moderators {names} and <b>{mod_list[-1]}</b>"

        db = {
            'thompson': "Thompson, S. G., & Higgins, J. P. (2002). How should meta-regression analyses be undertaken and interpreted? <i>Statistics in Medicine</i>, 21(11), 1559-1573.",
            'cr2': "Tipton, E. (2015). Small sample adjustments for robust variance estimation with meta-regression. <i>Psychological Methods</i>, 20(3), 375-393. https://doi.org/10.1037/met0000011",
            'htz': "Tipton, E., & Pustejovsky, J. E. (2015). Small-sample adjustments for tests of moderators and model fit using robust variance estimation in meta-regression. <i>Journal of Educational and Behavioral Statistics</i>, 40(6), 604-634. https://doi.org/10.3102/1076998615606099",
            '3level': "Van den Noortgate, W., López-López, J. A., Marín-Martínez, F., & Sánchez-Meca, J. (2013). Three-level meta-analysis of dependent effect sizes. <i>Behavior Research Methods</i>, 45(2), 576-594.",
            'knapp': "Knapp, G., & Hartung, J. (2003). Improved tests for a random effects meta-regression with a single covariate. <i>Statistics in Medicine</i>, 22(17), 2693-2710.",
            'tool': COMETA_CITATION,
            'raudenbush': "Raudenbush, S. W. (2009). Analyzing effect sizes: Random-effects models. In H. Cooper, L. V. Hedges, & J. C. Valentine (Eds.), <i>The handbook of research synthesis and meta-analysis</i> (2nd ed., pp. 295-315). New York: Russell Sage Foundation."
        }

        active_refs = []
        if "Cluster-Robust" in model_type or "3-Level" in model_type:
            model_desc = "To account for the nested structure of the data (multiple effect sizes per study), we employed a multi-level meta-regression model [1]."
            active_refs.append(db['3level'])
            model_desc += (" Standard errors were computed with the bias-reduced cluster-robust (CR2) variance"
                           " estimator, clustering by study, with Satterthwaite degrees of freedom for individual"
                           " coefficients [2] and the approximate Hotelling's <i>T</i>² (HTZ) test for"
                           " multi-parameter hypotheses [3].")
            active_refs.append(db['cr2'])
            active_refs.append(db['htz'])
        else:
            model_desc = "We examined the relationship between effect size and the moderator(s) using a mixed-effects meta-regression model [1]."
            active_refs.append(db['thompson'])
            model_desc += (" The Knapp-Hartung adjustment was used for hypothesis testing to improve the"
                           " accuracy of significance levels, with its scale factor truncated at 1 so that"
                           " standard errors are never smaller than the model-based ones [2].")
            active_refs.append(db['knapp'])

        active_refs.append(db['raudenbush'])
        ref_r2 = len(active_refs)
        r2_methods = (
            "The proportion of heterogeneity explained by the moderator(s) was "
            "quantified as a pseudo-<i>R</i>² = (Σσ²<sub>null</sub> − "
            "Σσ²<sub>model</sub>) / Σσ²<sub>null</sub>, where the null model is the "
            "intercept-only model fitted by REML to exactly the same effect sizes, with "
            "the same sampling variance–covariance matrix and the same random-effects "
            f"structure [{ref_r2}]. Values were truncated at zero, and the reduction "
            "was also reported separately for each variance component."
        )

        active_refs.append(db['tool'])
        ref_tool = len(active_refs)

        formula_html = ""
        if formula:
            formula_html = f"""
        <p style='text-align: justify;'>
        <b>Model Specification.</b> The fitted model was:
        </p>
        <p style='text-align: center; font-family: "Courier New", monospace; font-size: 11pt;'>
        {formula}
        </p>
        <p style='text-align: justify;'>
        Continuous moderators were mean-centered prior to analysis to ensure mathematical stability and to render the intercept interpretable (representing the expected effect size at the sample mean of continuous covariates).
        Categorical moderators were treatment-coded, with one level per factor
        omitted as the reference category (absorbed into the intercept).
        Between-study heterogeneity was quantified using the residual
        <i>I</i>² statistic, representing the proportion of variance not
        explained by the moderator(s).
        </p>
            """
        else:
            formula_html = f"""
        <p style='text-align: justify;'>
        <b>Model Specification.</b> The model estimated an intercept and a
        slope for each moderator. Between-study heterogeneity was quantified
        using the residual <i>I</i>² statistic: the proportion of the variance
        remaining after accounting for the moderator(s) that is due to
        heterogeneity rather than sampling error.
        </p>
            """

        html = f"""
        <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt;
                    line-height: 1.8; padding: 20px; background-color: #ffffff;
                    border: 1px solid #eee; margin-bottom: 20px;'>

        <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db;
                   padding-bottom: 10px; margin-top: 0;'>Materials and Methods</h3>

        <p style='text-align: justify;'>
        <b>Meta-Regression.</b> We conducted a meta-regression analysis to investigate whether
        the variation in effect sizes could be explained by {mod_phrase}.
        {model_desc}
        </p>

        {formula_html}

        <p style='text-align: justify;'>
        {r2_methods}
        </p>

        <p style='text-align: justify;'>
        All analyses were performed using the Co-Meta toolkit [{ref_tool}].
        </p>

        <h4 style='color: #34495e; margin-top: 20px; font-size: 11pt;'>References</h4>
        <ol style='font-size: 10pt; color: #555;'>
        """
        for ref in active_refs:
            html += f"<li>{ref}</li>"
        html += "</ol></div>"
        return html



    def generate_results_section(
        self,
        result: 'RegressionResult',
        moderator_col
    ) -> str:
        """Generate Results section (16b style: concise intro + significant-
        term highlights, with the coefficient table carrying full detail).

        Generalized to p coefficients. Continuous terms are described with
        per-unit phrasing; categorical dummies are described relative to their
        reference level. Non-significant terms are summarized collectively
        rather than narrated individually. An omnibus paragraph is rendered
        only if result.omnibus is populated.
        """
        betas = result.betas
        se_betas = result.se_betas
        p_values = result.p_values
        cis = result.cis
        df_resid = result.df_resid
        term_names = result.term_names
        k_studies = result.k_studies
        n_obs = result.n_obs
        model_type = result.model_type
        r2 = result.r_squared
        resid_i2 = result.resid_i2
        tau_sq = result.tau_sq
        sigma_sq = result.sigma_sq

        primary = moderator_col if isinstance(moderator_col, str) else result.moderator_col_name

        # Model description.
        if "2-Level" in model_type or "Aggregated" in model_type:
            model_desc = "two-level aggregated random-effects meta-regression"
            variance_note = f"Between-study variance (τ²) was estimated at {tau_sq:.4f}."
            cluster_note = ""
        else:
            model_desc = "three-level random-effects meta-regression with cluster-robust variance estimation"
            variance_note = f"Between-study variance (τ²) was {tau_sq:.4f} and within-study variance (σ²) was {sigma_sq:.4f}."
            cluster_note = " Cluster-robust standard errors were computed to account for the nested structure of effect sizes within studies."

        # Identify significant moderator terms (exclude intercept at index 0).
        sig_idx = [i for i in range(1, betas.size) if p_values[i] < _alpha()]
        n_mod_terms = betas.size - 1

        # Build per-term highlight sentences for significant terms.
        def term_sentence(i):
            name = term_names[i]
            b = betas[i]
            direction = "higher" if b > 0 else "lower"
            p_fmt = "&lt; 0.001" if p_values[i] < 0.001 else f"= {p_values[i]:.3f}"
            ci = cis[i]
            stat = (f"(β = {b:.3f}, SE = {se_betas[i]:.3f}, "
                    f"<i>t</i>({df_resid[i]:.1f}) = {b/se_betas[i]:.2f}, "
                    f"<i>p</i> {p_fmt}, {self.ci_level:.0f}% CI [{ci[0]:.3f}, {ci[1]:.3f}])")
            # Categorical dummy terms are labeled "factor[level]".
            if "[" in str(name) and str(name).endswith("]"):
                factor = str(name).split("[", 1)[0]
                level = str(name).split("[", 1)[1][:-1]
                return (f"relative to the reference level of <b>{factor}</b>, "
                        f"the level <b>{level}</b> showed a significantly "
                        f"{direction} effect size {stat}")
            else:
                per_unit = "increase" if b > 0 else "decrease"
                return (f"each one-unit increase in <b>{name}</b> was associated "
                        f"with a {per_unit} of {abs(b):.3f} units in the effect "
                        f"size {stat}")

        _d = getattr(result, 'r2_detail', None) or {}
        if not np.isfinite(r2):
            r2_text = (" The proportion of heterogeneity explained (pseudo-<i>R</i>²) "
                       "could not be computed.")
        elif r2 > 0:
            r2_text = (f" Relative to the intercept-only model, the moderator(s) accounted "
                       f"for <b>{r2:.1f}%</b> of the total heterogeneity (pseudo-<i>R</i>²)")
            if _d.get('three_level'):
                _lv = []
                for _name, _raw in (('between-study', _d.get('R2_tau2_raw')),
                                    ('within-study', _d.get('R2_sigma2_raw'))):
                    if _raw is not None and np.isfinite(_raw):
                        _lv.append(f"the {_name} variance increased by {-100 * _raw:.1f}%"
                                   if _raw < 0 else
                                   f"the {_name} variance was reduced by {100 * _raw:.1f}%")
                if _lv:
                    r2_text += "; " + " and ".join(_lv)
            r2_text += "."
        else:
            r2_text = (" The moderator(s) did not reduce the total heterogeneity relative to "
                       "the intercept-only model (pseudo-<i>R</i>² = 0%, truncated at zero).")

        # Intro + findings paragraph.
        if n_mod_terms == 1:
            mod_intro = f"whether <b>{primary}</b> moderated the effect sizes"
        else:
            mod_intro = "whether the selected moderators explained variation in effect sizes"

        html = f"""
        <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt;
                    line-height: 1.8; padding: 20px; background-color: #ffffff;'>

        <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;'>
            Meta-Regression Results
        </h3>

        <p style='text-align: justify;'>
        We conducted a meta-regression to examine {mod_intro}.
        A {model_desc} was employed to account for dependencies in the data.{cluster_note}
        The analysis included <b>k = {k_studies}</b> studies with
        <b>n = {n_obs}</b> effect sizes, estimating {n_mod_terms} moderator
        term(s) in addition to the intercept.
        </p>
        """

        # Omnibus block — render only if present.
        if result.omnibus:
            om = result.omnibus
            overall = om.get('overall')
            if overall:
                p_om = "&lt; 0.001" if overall.get('p_val', 1) < 0.001 else f"= {overall.get('p_val'):.3f}"
                _test_lbl = {'AHZ': 'HTZ small-sample F',
                             'Wald-F': 'Wald F'}.get(overall.get('test'), 'omnibus F')
                html += f"""
        <p style='text-align: justify;'>
        An omnibus test of all moderator terms jointly ({_test_lbl}) was
        {'statistically significant' if overall.get('p_val', 1) < _alpha() else 'not statistically significant'}
        (<i>F</i>({overall.get('df_num')}, {overall.get('df_denom'):.1f}) =
        {overall.get('F'):.2f}, <i>p</i> {p_om}).
        </p>
                """
            per_factor = om.get('per_factor', {})
            for fac, st in per_factor.items():
                p_f = "&lt; 0.001" if st.get('p_val', 1) < 0.001 else f"= {st.get('p_val'):.3f}"
                html += f"""
        <p style='text-align: justify;'>
        The categorical moderator <b>{fac}</b> was
        {'a significant' if st.get('p_val', 1) < _alpha() else 'not a significant'}
        predictor overall (<i>F</i>({st.get('df_num')}, {st.get('df_denom'):.1f}) =
        {st.get('F'):.2f}, <i>p</i> {p_f}).
        </p>
                """

        # Findings: highlight significant terms; summarize the rest.
        if sig_idx:
            highlights = "; ".join(term_sentence(i) for i in sig_idx)
            html += f"""
        <p style='text-align: justify;'>
        Significant associations were observed: {highlights}.{r2_text}
        </p>
            """
            n_ns = n_mod_terms - len(sig_idx)
            if n_ns > 0:
                html += f"""
        <p style='text-align: justify;'>
        The remaining {n_ns} moderator term(s) were not statistically
        significant at the α = {self.alpha if hasattr(self, 'alpha') else 0.05:.2f} level
        (see Table 1 for full coefficient estimates).
        </p>
                """
        else:
            html += f"""
        <p style='text-align: justify;'>
        None of the {n_mod_terms} moderator term(s) reached statistical
        significance; the selected moderator(s) could not be reliably
        distinguished from sampling error as drivers of effect sizes
        (see Table 1 for full estimates).{r2_text}
        </p>
            """

        # Heterogeneity interpretation.
        if resid_i2 < 25:
            het_text, het_interp = "low", "suggesting that the moderator(s) captured most of the systematic variation in effect sizes"
        elif resid_i2 < 50:
            het_text, het_interp = "moderate", "suggesting that additional unmeasured factors may contribute to the variation in effect sizes"
        else:
            het_text, het_interp = "substantial", "indicating that additional moderators not examined here likely contribute to variation in effect sizes"

        html += f"""
        <p style='text-align: justify;'>
        {variance_note} Residual heterogeneity remained <b>{het_text}</b>
        (τ²<sub>residual</sub> = {tau_sq:.4f}"""
        if resid_i2 > 0:
            html += f", <i>I</i>²<sub>residual</sub> = {resid_i2:.1f}%"
        html += f"""), {het_interp}.
        </p>

        <h4 style='color: #34495e; margin-top: 25px;'>Statistical Methods</h4>
        <p style='text-align: justify;'>
        All meta-regression analyses were conducted using {model_desc}. Restricted
        maximum likelihood (REML) was used to estimate variance components.
        Confidence intervals and <i>p</i>-values were based on a <i>t</i>-distribution
        using per-parameter degrees of freedom. Categorical moderators, where present,
        were treatment-coded with one level omitted as the reference category.
        </p>
        """

        html += self._generate_coefficient_table(result, primary)
        html += "</div>"
        return html

    def _generate_coefficient_table(
        self,
        result: 'RegressionResult',
        moderator_col
    ) -> str:
        """Generate the formatted Table 1 with one row per term (p rows)."""
        betas = result.betas
        se_betas = result.se_betas
        p_values = result.p_values
        cis = result.cis
        df_vec = np.asarray(result.df_resid, dtype=float).ravel()
        term_names = result.term_names
        sigma_sq = result.sigma_sq
        p = betas.size
        if df_vec.size != p:
            df_vec = np.full(p, float(df_vec.flat[0]))

        def fp(pv):
            return "&lt;0.001" if pv < 0.001 else f"{pv:.3f}"

        rows = ""
        for i in range(p):
            is_intercept = (i == 0)
            sig = (not is_intercept) and (p_values[i] < _alpha())
            bold = "font-weight: bold;" if sig else ""
            row_bg = "#f8f9fa" if (i % 2 == 0) else "white"
            tval = betas[i] / se_betas[i] if se_betas[i] != 0 else float('nan')
            lo, hi = cis[i]
            label = term_names[i] if i < len(term_names) else f"β{i}"
            label_cell = f"<b>{label}</b>" if not is_intercept else label
            rows += f"""
                    <tr style='background-color: {row_bg};'>
                        <td style='border: 1px solid #bdc3c7; padding: 8px;'>{label_cell}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center; {bold}'>{betas[i]:.3f}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{se_betas[i]:.3f}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{tval:.2f}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{df_vec[i]:.1f}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center; {bold}'>{fp(p_values[i])}</td>
                        <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>[{lo:.3f}, {hi:.3f}]</td>
                    </tr>
            """

        note_extra = "; σ² = within-study variance" if sigma_sq > 0 else ""
        model_word = "three-level" if sigma_sq > 0 else "two-level aggregated random-effects"

        return f"""
        <hr style='margin: 30px 0; border: none; border-top: 1px solid #bdc3c7;'>
        <div style='background-color: #ecf0f1; padding: 20px; border-left: 4px solid #3498db; margin-top: 25px;'>
            <h4 style='margin-top: 0; color: #2c3e50;'>📊 Table 1. Meta-Regression Coefficients</h4>
            <table style='width: 100%; border-collapse: collapse; margin-top: 15px; background-color: white;'>
                <thead style='background-color: #34495e; color: white;'>
                    <tr>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: left;'>Predictor</th>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>β</th>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>SE</th>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'><i>t</i></th>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>df</th>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'><i>p</i>-value</th>
                        <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>{self.ci_level:.0f}% CI</th>
                    </tr>
                </thead>
                <tbody>
                    {rows}
                </tbody>
            </table>
            <p style='margin-top: 10px; font-size: 0.9em; color: #6c757d;'>
                <i>Note:</i> Results from {model_word} meta-regression. k = number of studies;
                n = number of effect sizes; τ² = between-study variance{note_extra}.
                Per-parameter degrees of freedom shown.
            </p>
        </div>
        """

# =============================================================================
# VIEW COMPONENTS (Tab Renderers)
# =============================================================================

class RegressionResultsView:
    """
    Manages all UI rendering for meta-regression.
    Contains zero business logic - only presentation code.
    """

    def __init__(self, ci_level: float = 95):
        """
        Initialize view with display settings.

        Args:
            ci_level: Confidence interval percentage
        """
        self.ci_level = ci_level
        self.templates = RegressionHTMLTemplates()
        self.text_gen = RegressionPublicationTextGenerator(ci_level)

        # Create tab widgets
        self.stale_banner = widgets.HTML(value="")
        register_banner('regression', self.stale_banner)
        self.tab_results = widgets.Output()
        self.tab_diagnostics = widgets.Output()
        self.tab_details = widgets.Output()
        self.tab_publication = widgets.Output()
        self.tab_export = widgets.Output()

        self.tabs = widgets.Tab(children=[
            self.tab_results,
            self.tab_diagnostics,
            self.tab_details,
            self.tab_publication,
            self.tab_export
        ])

        self.tabs.set_title(0, '📊 Results')
        self.tabs.set_title(1, '🔍 Diagnostics')
        self.tabs.set_title(2, '⚙️ Model Details')
        self.tabs.set_title(3, '📝 Publication Text')
        self.tabs.set_title(4, '💾 Export')

    # -------------------------------------------------------------------------
    # TAB 1: RESULTS
    # -------------------------------------------------------------------------


    def render_results_tab(
        self,
        result: 'RegressionResult',
        moderator_col
    ) -> None:
        """Render results summary tab (single- or multi-moderator)."""

        with self.tab_results:
            self.tab_results.clear_output()

            primary = (moderator_col if isinstance(moderator_col, str)
                       else result.moderator_col_name)

            # Number of moderator terms (excludes intercept).
            n_mod_terms = result.n_coef - 1

            # Main container open
            display(HTML("<div style='padding: 20px;'>"))

            # Dashboard Header — badge shows all moderators when multiple.
            badge = primary if n_mod_terms <= 1 else f"{n_mod_terms} terms"
            display(HTML(self.templates.dashboard_header("Meta-Regression", badge)))

            # Formula preview (expanded form, from fitted design terms).
            formula = self.templates.build_formula_string(
                effect_col=getattr(self, 'effect_col', 'effect'),
                moderator_cols=[],
                term_names=result.term_names
            )

            display(HTML(self.templates.formula_preview_box(
                formula, title="Fitted Model",
                note="Categorical factors shown as factor[level]; reference level absorbed into intercept."
            )))

            # =================================================================
            # HEADLINE
            # =================================================================
            if n_mod_terms == 1:
                # Single moderator term -> feature its slope (as before).
                slope_i = 1
                sig = ("***" if result.p_values[slope_i] < 0.001
                       else "**" if result.p_values[slope_i] < 0.01
                       else "*" if result.p_values[slope_i] < 0.05 else "ns")
                gradient_html = self.templates.gradient_card(
                    title=f"SLOPE COEFFICIENT ({result.term_names[slope_i]})",
                    value=f"{result.betas[slope_i]:.4f}",
                    subtitle=sig
                )
                display(HTML(gradient_html))

                color = "#28a745" if result.p_values[slope_i] < _alpha() else "#6c757d"
                info_items = [
                    {
                        'label': f'{self.ci_level:.0f}% Confidence Interval',
                        'value': f'[{result.cis[slope_i][0]:.4f}, {result.cis[slope_i][1]:.4f}]',
                        'color': '#007bff'
                    },
                    {
                        'label': 'P-value',
                        'value': f'{result.p_values[slope_i]:.4g}',
                        'color': color
                    }
                ]
                display(HTML(self.templates.info_grid(info_items)))

                direction_text = ('<b>increases</b>' if result.betas[slope_i] > 0
                                  else '<b>decreases</b>')
                sig_text = ('This relationship is <b style="color: #28a745;">statistically '
                            'significant</b>.' if result.p_values[slope_i] < _alpha()
                            else 'This relationship is <b>not statistically significant</b>.')
                interp_content = f"""
                For every 1-unit increase in <b>{result.term_names[slope_i]}</b>, the effect
                size {direction_text} by <b>{abs(result.betas[slope_i]):.4f}</b> units. {sig_text}
                """
                display(HTML(self.templates.interpretation_box(
                    interp_content, result.p_values[slope_i] < _alpha())))

            else:
                # Multiple moderator terms -> model-level headline.
                if result.omnibus and result.omnibus.get('overall'):
                    ov = result.omnibus['overall']
                    sig = ("***" if ov.get('p_val', 1) < 0.001
                           else "**" if ov.get('p_val', 1) < 0.01
                           else "*" if ov.get('p_val', 1) < 0.05 else "ns")
                    gradient_html = self.templates.gradient_card(
                        title="OMNIBUS MODEL TEST (all moderators)",
                        value=f"F = {ov.get('F'):.2f}",
                        subtitle=sig
                    )
                    display(HTML(gradient_html))
                    p_disp = ("&lt;0.001" if ov.get('p_val', 1) < 0.001
                              else f"{ov.get('p_val'):.4g}")
                    info_items = [
                        {'label': 'Model df (num, denom)',
                         'value': f"{ov.get('df_num')}, {ov.get('df_denom'):.1f}",
                         'color': '#007bff'},
                        {'label': 'Omnibus p-value', 'value': p_disp,
                         'color': "#28a745" if ov.get('p_val', 1) < _alpha() else "#6c757d"},
                    ]
                    display(HTML(self.templates.info_grid(info_items)))
                else:
                    # No omnibus available. Summarize
                    # the model and count significant terms instead.
                    n_sig = int(np.sum(
                        result.p_values[1:] < _alpha())) if result.n_coef > 1 else 0
                    gradient_html = self.templates.gradient_card(
                        title="MULTI-MODERATOR MODEL",
                        value=f"{n_mod_terms} terms",
                        subtitle=f"{n_sig} sig."
                    )
                    display(HTML(gradient_html))
                    info_items = [
                        {'label': 'Pseudo-R² (heterogeneity explained)',
                         'value': fmt_r2(result.r_squared), 'color': '#007bff'},
                        {'label': 'Residual I²',
                         'value': f"{result.resid_i2:.1f}%", 'color': '#6c757d'},
                    ]
                    display(HTML(self.templates.info_grid(info_items)))

                interp_content = (
                    "This model includes multiple moderator terms. See the "
                    "coefficient table below for per-term estimates, and the "
                    "Publication Text tab for a written summary."
                )
                if not result.omnibus:
                    interp_content += (
                        " <i>An omnibus joint test of all terms is not yet "
                        "available for this model.</i>"
                    )
                display(HTML(self.templates.interpretation_box(
                    interp_content, False)))

            # =================================================================
            # COEFFICIENT TABLE (N rows)
            # =================================================================
            display(HTML(self.templates.header("Coefficient Table")))
            coef_table = self.templates.coefficient_table(
                result.betas,
                result.se_betas,
                result.p_values,
                result.cis,
                result.df_resid,
                result.term_names,
                self.ci_level
            )
            display(HTML(coef_table))

            # Model summary (pass n_coef).
            display(HTML(self.templates.header("Model Summary")))
            summary_html = self.templates.model_summary_box(
                result.model_type,
                result.k_studies,
                result.n_obs,
                result.df_resid,
                result.tau_sq,
                result.sigma_sq,
                result.r_squared,
                result.resid_i2,
                n_coef=result.n_coef,
                r2_detail=result.r2_detail,
            )
            display(HTML(summary_html))
            if result.data_accounting and 'accounting_html' in globals():
                display(HTML(accounting_html(result.data_accounting)))

            # Predicted effects at representative moderator values.
            self._render_predictions(result)

            # Next step tip
            tip_html = """
            <div style='background-color: #fff3cd; padding: 15px; border-radius: 5px;
                        border-left: 4px solid #ffc107; margin-top: 20px;'>
                <p style='margin: 0;'><b>📊 Next Step:</b> Use the dedicated plot cell to visualize
                this regression relationship with full customization options.</p>
            </div>
            </div>
            """
            display(HTML(tip_html))

    def _render_predictions(self, result: 'RegressionResult') -> None:
        """Predicted effect, CI and prediction interval at a default grid."""
        try:
            grid = default_prediction_grid(result)
            if not grid:
                return
            pred = predict_from_fit(result, grid, ci_level=self.ci_level)
        except Exception as exc:  # surfaced, never silent
            display(HTML(
                "<div style='background-color:#f8d7da;padding:12px;"
                "border-radius:5px;border-left:4px solid #dc3545;margin-top:20px;'>"
                f"<b>Predicted effects unavailable:</b> {exc}</div>"))
            return

        mod_cols = [c for c in pred.columns
                    if c not in ('estimate', 'se', 'df', 'ci_lower', 'ci_upper',
                                 'pi_lower', 'pi_upper', 'df_method', 'notes')]
        fallback = (pred['df_method'] != 'contrast Satterthwaite (AHZ q=1)').any()

        display(HTML(self.templates.header("Predicted Effects")))
        head = "".join(f"<th style='border:1px solid #bdc3c7;padding:8px;'>{c}</th>"
                       for c in mod_cols)
        body = ""
        for _, r in pred.iterrows():
            cells = "".join(
                f"<td style='border:1px solid #bdc3c7;padding:8px;'>"
                f"{(f'{v:.4g}' if isinstance(v, float) else v)}</td>"
                for v in (r[c] for c in mod_cols))
            body += (
                f"<tr>{cells}"
                f"<td style='border:1px solid #bdc3c7;padding:8px;text-align:right;'>"
                f"<b>{r['estimate']:.4f}</b></td>"
                f"<td style='border:1px solid #bdc3c7;padding:8px;text-align:right;'>"
                f"{r['se']:.4f}</td>"
                f"<td style='border:1px solid #bdc3c7;padding:8px;text-align:right;'>"
                f"{r['df']:.1f}</td>"
                f"<td style='border:1px solid #bdc3c7;padding:8px;text-align:center;'>"
                f"[{r['ci_lower']:.4f}, {r['ci_upper']:.4f}]</td>"
                f"<td style='border:1px solid #bdc3c7;padding:8px;text-align:center;'>"
                f"[{r['pi_lower']:.4f}, {r['pi_upper']:.4f}]</td></tr>")

        display(HTML(f"""
        <table style='border-collapse:collapse;width:100%;font-family:sans-serif;
                      font-size:13px;margin-bottom:10px;'>
          <thead><tr style='background-color:#34495e;color:white;'>{head}
            <th style='border:1px solid #bdc3c7;padding:8px;'>Estimate</th>
            <th style='border:1px solid #bdc3c7;padding:8px;'>SE</th>
            <th style='border:1px solid #bdc3c7;padding:8px;'>df</th>
            <th style='border:1px solid #bdc3c7;padding:8px;'>{self.ci_level:.0f}% CI</th>
            <th style='border:1px solid #bdc3c7;padding:8px;'>{self.ci_level:.0f}% PI</th>
          </tr></thead><tbody>{body}</tbody>
        </table>
        <p style='font-size:12px;color:#666;margin:0 0 20px 0;'>
        Effect predicted at each combination of moderator values, with the
        other moderators held at their reference level (categorical) or mean
        (continuous). <b>CI</b> is the interval for the predicted mean
        (SE = &radic;(x&prime;Vx) on the CR2 covariance); <b>PI</b> is the
        interval for a new study (SE = &radic;(x&prime;Vx + &tau;&sup2; +
        &sigma;&sup2;)), as in <code>metafor::predict.rma</code>. Degrees of
        freedom are {'the conservative minimum over the coefficients each row loads on, because the small-sample closure was unavailable for this fit'
                     if fallback else
                     'contrast-specific Satterthwaite values, matching clubSandwich&rsquo;s <code>coef_test</code>'}.
        Continuous values are on the original scale; the centring applied at
        fit time is re-applied internally. Call
        <code>predict_meta_regression({{...}})</code> for your own values.
        </p>"""))

    # -------------------------------------------------------------------------
    # TAB 2: DIAGNOSTICS
    # -------------------------------------------------------------------------

    def render_diagnostics_tab(self, result: 'RegressionResult') -> None:
        """Render diagnostics tab"""

        with self.tab_diagnostics:
            self.tab_diagnostics.clear_output()

            display(HTML("<div style='padding: 20px;'>"))
            display(HTML(self.templates.header("🔍 Model Diagnostics")))

            # Residual analysis
            display(HTML("<h4 style='color: #34495e;'>Residual Analysis</h4>"))

            resid_content = f"""
            <p style='margin: 5px 0;'><b>Residual Range:</b> [{np.min(result.resid):.4f}, {np.max(result.resid):.4f}]</p>
            <p style='margin: 5px 0;'><b>Mean Residual:</b> {np.mean(result.resid):.4f} (should be ≈ 0)</p>
            <p style='margin: 5px 0;'><b>SD of Residuals:</b> {np.std(result.resid):.4f}</p>
            """
            display(HTML(self.templates.card(resid_content)))

            # Influence diagnostics
            display(HTML("<h4 style='color: #34495e;'>Influence Diagnostics</h4>"))

            if result.has_influential:
                if result.cooks_d is not None:
                    threshold = 4 / result.n_obs
                    influential_indices = np.where(result.cooks_d > threshold)[0]
                    influence_html = f"""
                    <div style='background-color: #fff3cd; padding: 15px; border-radius: 5px;
                                border-left: 4px solid #ffc107; margin-bottom: 20px;'>
                        <p style='margin: 0;'><b>⚠️ Warning:</b> {len(influential_indices)} potentially
                        influential observation(s) detected (Cook's D > {threshold:.4f}).</p>
                        <p style='margin: 10px 0 0 0;'>Influential points: {', '.join(map(str, influential_indices))}</p>
                    </div>
                    """
                else:
                    influence_html = """
                    <div style='background-color: #fff3cd; padding: 15px; border-radius: 5px;
                                border-left: 4px solid #ffc107; margin-bottom: 20px;'>
                        <p style='margin: 0;'><b>⚠️ Note:</b> Influence diagnostics indicated potential issues.</p>
                    </div>
                    """
            else:
                influence_html = """
                <div style='background-color: #d4edda; padding: 15px; border-radius: 5px;
                            border-left: 4px solid #28a745; margin-bottom: 20px;'>
                    <p style='margin: 0;'><b>✓ Good:</b> No highly influential observations detected.</p>
                </div>
                """

            display(HTML(influence_html))

            # Heterogeneity assessment
            display(HTML("<h4 style='color: #34495e;'>Heterogeneity Assessment</h4>"))

            het_content = f"""
            <p style='margin: 5px 0;'><b>Residual Heterogeneity (I²):</b> {result.resid_i2:.1f}%</p>
            """
            het_content += (r2_breakdown_html(result.r2_detail) if result.r2_detail
                            else f"<p style='margin: 5px 0;'><b>Pseudo-R²:</b> {fmt_r2(result.r_squared)}</p>")

            het_content += "<p style='margin: 10px 0 0 0;'><i>Lower residual heterogeneity suggests the moderator explains variation well.</i></p>"

            display(HTML(self.templates.card(het_content)))

            # Model assumptions
            display(HTML("<h4 style='color: #34495e;'>Model Assumptions</h4>"))
            display(HTML(self.templates.assumption_table(result.sigma_sq)))

            # Recommendation
            rec_html = """
            <div style='background-color: #e7f3ff; padding: 15px; border-radius: 5px; margin-top: 20px;'>
                <p style='margin: 0;'><b>💡 Recommendation:</b> Use the dedicated plot cell to create
                residual plots and visually assess linearity and homoscedasticity assumptions.</p>
            </div>
            </div>
            """
            display(HTML(rec_html))

    # -------------------------------------------------------------------------
    # TAB 3: MODEL DETAILS
    # -------------------------------------------------------------------------

    def render_details_tab(
        self,
        result: 'RegressionResult',
        moderator_col,
        reg_df: pd.DataFrame,
        effect_col: str,
        var_col: str
    ) -> None:
        """Render model details tab (single- or multi-moderator)."""

        with self.tab_details:
            self.tab_details.clear_output()

            # Derive source moderators + categorical flags from term_names.
            # Continuous term -> its own name; categorical -> 'factor[level]'.
            source_mods = []          # ordered unique source moderator names
            categorical_factors = set()
            for name in result.term_names[1:]:  # skip intercept
                s = str(name)
                if "[" in s and s.endswith("]"):
                    factor = s.split("[", 1)[0]
                    categorical_factors.add(factor)
                    if factor not in source_mods:
                        source_mods.append(factor)
                else:
                    if s not in source_mods:
                        source_mods.append(s)

            df_vec = np.asarray(result.df_resid, dtype=float).ravel()
            p = result.n_coef

            display(HTML("<div style='padding: 20px;'>"))
            display(HTML(self.templates.header("⚙️ Model Details & Specifications")))

            # --- Model specification ---
            display(HTML("<h4 style='color: #34495e;'>Model Specification</h4>"))
            if result.sigma_sq > 0:
                spec_html = """
                <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px;
                            margin-bottom: 20px; font-family: monospace;'>
                    <p style='margin: 5px 0;'><b>Three-Level Model:</b></p>
                    <p style='margin: 5px 0; padding-left: 20px;'>y<sub>ij</sub> = Xβ + u<sub>i</sub> + e<sub>ij</sub></p>
                    <p style='margin: 5px 0; padding-left: 20px;'>where:</p>
                    <p style='margin: 5px 0; padding-left: 40px;'>• y<sub>ij</sub> = effect size j in study i</p>
                    <p style='margin: 5px 0; padding-left: 40px;'>• X = design matrix (intercept + moderators)</p>
                    <p style='margin: 5px 0; padding-left: 40px;'>• β = fixed-effect coefficients</p>
                    <p style='margin: 5px 0; padding-left: 40px;'>• u<sub>i</sub> ~ N(0, τ²) = between-study random effect</p>
                    <p style='margin: 5px 0; padding-left: 40px;'>• e<sub>ij</sub> ~ N(0, σ²) = within-study random effect</p>
                </div>
                """
            else:
                spec_html = """
                <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px;
                            margin-bottom: 20px; font-family: monospace;'>
                    <p style='margin: 5px 0;'><b>Two-Level Aggregated Model:</b></p>
                    <p style='margin: 5px 0; padding-left: 20px;'>y<sub>i</sub> = Xβ + u<sub>i</sub></p>
                    <p style='margin: 5px 0; padding-left: 20px;'>where:</p>
                    <p style='margin: 5px 0; padding-left: 40px;'>• y<sub>i</sub> = aggregated effect size for study i</p>
                    <p style='margin: 5px 0; padding-left: 40px;'>• X = design matrix (intercept + moderators)</p>
                    <p style='margin: 5px 0; padding-left: 40px;'>• β = fixed-effect coefficients</p>
                    <p style='margin: 5px 0; padding-left: 40px;'>• u<sub>i</sub> ~ N(0, τ²) = between-study random effect</p>
                    <p style='margin: 10px 0 0 0;'><i>Note: Data aggregated to study level because all moderators were constant within studies.</i></p>
                </div>
                """
            display(HTML(spec_html))

            # --- Variance-covariance matrix (labeled) ---
            display(HTML("<h4 style='color: #34495e;'>Variance-Covariance Matrix</h4>"))
            display(HTML(self.templates.variance_covariance_matrix(
                result.var_betas_robust, term_names=result.term_names
            )))

            # --- Variance components ---
            display(HTML("<h4 style='color: #34495e;'>Variance Components</h4>"))
            var_table = f"""
            <table style='width: 100%; border-collapse: collapse; margin-bottom: 20px;'>
                <thead style='background-color: #f8f9fa;'>
                    <tr>
                        <th style='padding: 10px; text-align: left; border: 1px solid #dee2e6;'>Component</th>
                        <th style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>Value</th>
                        <th style='padding: 10px; text-align: left; border: 1px solid #dee2e6;'>Description</th>
                    </tr>
                </thead>
                <tbody>
                    <tr>
                        <td style='padding: 10px; border: 1px solid #dee2e6;'><b>τ²</b></td>
                        <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>{result.tau_sq:.4f}</td>
                        <td style='padding: 10px; border: 1px solid #dee2e6;'>Between-study variance (residual)</td>
                    </tr>
            """
            if result.sigma_sq > 0:
                var_table += f"""
                    <tr style='background-color: #f8f9fa;'>
                        <td style='padding: 10px; border: 1px solid #dee2e6;'><b>σ²</b></td>
                        <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>{result.sigma_sq:.4f}</td>
                        <td style='padding: 10px; border: 1px solid #dee2e6;'>Within-study variance</td>
                    </tr>
                """
            var_table += "</tbody></table>"
            display(HTML(var_table))

            # --- Degrees of freedom (p-aware, per-term where applicable) ---
            display(HTML("<h4 style='color: #34495e;'>Degrees of Freedom</h4>"))
            unit_word = 'studies' if result.sigma_sq > 0 else 'observations'
            unit_n = result.k_studies if result.sigma_sq > 0 else result.n_obs
            if df_vec.size and np.allclose(df_vec, df_vec[0]):
                df_show = f"{df_vec[0]:.1f}"
                df_calc = (f"{unit_word.capitalize()} ({unit_word[0]} = {unit_n}) "
                           f"− parameters ({p}) = {df_vec[0]:.1f}")
            else:
                df_show = f"{df_vec.min():.1f}–{df_vec.max():.1f} (per parameter)"
                df_calc = ("Per-parameter Satterthwaite degrees of freedom "
                           "(cluster-robust); see the coefficient table for each term.")
            df_content = f"""
            <p style='margin: 5px 0;'><b>df:</b> {df_show}</p>
            <p style='margin: 5px 0;'><b>Calculation:</b> {df_calc}</p>
            <p style='margin: 10px 0 0 0;'><i>Used for the t-distribution in hypothesis testing and confidence intervals</i></p>
            """
            display(HTML(self.templates.card(df_content)))

            # --- Standard error details ---
            display(HTML("<h4 style='color: #34495e;'>Standard Error Details</h4>"))
            if result.sigma_sq > 0:
                se_content = """
                <p style='margin: 5px 0;'><b>Cluster-Robust Standard Errors</b></p>
                <p style='margin: 5px 0;'>Standard errors account for clustering of effect sizes within studies,
                providing more conservative estimates when multiple effect sizes come from the same study.</p>
                """
            else:
                se_content = """
                <p style='margin: 5px 0;'><b>Standard Random-Effects Standard Errors</b></p>
                <p style='margin: 5px 0;'>Standard errors from the aggregated two-level model. Data were aggregated
                to study level because all moderators were constant within each study.</p>
                """
            display(HTML(self.templates.card(se_content)))

            # --- Data summary (per moderator, type-aware) ---
            display(HTML("<h4 style='color: #34495e;'>Data Summary</h4>"))
            rows = [
                f"<p style='margin: 5px 0;'><b>Used in analysis:</b> "
                f"{result.n_obs} {'studies' if result.sigma_sq == 0 else 'observations'}</p>",
                f"<p style='margin: 5px 0;'><b>Effect size range:</b> "
                f"[{reg_df[effect_col].min():.3f}, {reg_df[effect_col].max():.3f}]</p>",
            ]
            for mod in source_mods:
                if mod not in reg_df.columns:
                    continue
                if mod in categorical_factors:
                    levels = sorted(reg_df[mod].astype(str).unique().tolist())
                    rows.append(
                        f"<p style='margin: 5px 0;'><b>{mod}</b> (categorical): "
                        f"{len(levels)} levels — {', '.join(levels)}</p>"
                    )
                else:
                    col = pd.to_numeric(reg_df[mod], errors='coerce')
                    rows.append(
                        f"<p style='margin: 5px 0;'><b>{mod}</b> (continuous): "
                        f"range [{col.min():.3f}, {col.max():.3f}]</p>"
                    )
            display(HTML(self.templates.card("".join(rows))))

            display(HTML("</div>"))



    # -------------------------------------------------------------------------
    # TAB 4: PUBLICATION TEXT
    # -------------------------------------------------------------------------

    def render_publication_tab(
        self,
        result: 'RegressionResult',
        moderator_col: str,
        es_config: Dict[str, Any]
    ) -> str:
        """
        Render publication text tab.

        Returns:
            Combined HTML text for saving
        """
        with self.tab_publication:
            self.tab_publication.clear_output()

            display(HTML(self.templates.header("📝 Publication-Ready Results Text", level=3)))
            display(HTML("<p style='color: #6c757d;'>Copy and paste this formatted text into your manuscript:</p>"))

            # Generate both sections
            methods_html = self.text_gen.generate_methods_section(
                moderator_col, result.model_type, es_config
            )
            results_html = self.text_gen.generate_results_section(
                result, moderator_col
            )

            # Display
            display(HTML(methods_html))
            display(HTML(results_html))

            # Return combined for saving
            return methods_html + "<br><hr><br>" + results_html

    # -------------------------------------------------------------------------
    # TAB 5: EXPORT
    # -------------------------------------------------------------------------

    def render_export_tab(self, export_callback: callable) -> None:
        """Render export tab with download button"""

        with self.tab_export:
            self.tab_export.clear_output()

            display(HTML(self.templates.header("💾 Download Audit Report", level=3)))
            display(HTML("<p>Generate a full Excel audit trail including regression coefficients, diagnostics, and model settings.</p>"))

            btn_export = widgets.Button(
                description="📥 Download Regression Report",
                button_style='info',
                icon='file-excel',
                layout=widgets.Layout(width='300px', height='40px')
            )

            btn_export.on_click(export_callback)
            display(btn_export)

    # -------------------------------------------------------------------------
    # ERROR DISPLAY
    # -------------------------------------------------------------------------

    def render_error(self, message: str, details: Optional[str] = None) -> None:
        """Render error message in results tab"""
        with self.tab_results:
            self.tab_results.clear_output()
            error_html = f"<div style='color: red; background-color: #f8d7da; padding: 15px; border-radius: 5px;'>❌ {message}</div>"
            if details:
                error_html += f"<pre style='margin-top: 10px;'>{details}</pre>"
            display(HTML(error_html))


# =============================================================================
# CONTROLLER LAYER
# MAIN CONTROLLER
# =============================================================================

class RegressionController:
    """
    Master controller that orchestrates the entire meta-regression workflow.
    Coordinates data management, statistical computation, and UI rendering.
    """

    def __init__(self, analysis_config: Dict[str, Any]):
        """
        Initialize controller with ANALYSIS_CONFIG.

        Args:
            analysis_config: Global ANALYSIS_CONFIG dictionary
        """
        self.analysis_config = analysis_config

        # Initialize components
        try:
            self.data_manager = RegressionDataManager(analysis_config)
            self.engine = RegressionEngine(self.data_manager)

            # Get CI level from settings
            global_settings = self.data_manager.global_settings
            ci_level = (1 - global_settings.get('alpha', 0.05)) * 100
            self.view = RegressionResultsView(ci_level=ci_level)

            self._initialization_error = None

        except Exception as e:
            # If initialization fails, create minimal view to show error
            self.view = RegressionResultsView()
            self.data_manager = None
            self.engine = None
            self._initialization_error = e

    # -------------------------------------------------------------------------
    # UI SETUP
    # -------------------------------------------------------------------------

    def create_ui(self) -> widgets.VBox:
        """Create the multi-moderator Control Panel UI: matrix layout with
        inline inclusion, type toggle, and reference selection."""
        # Typed moderator list (name + inferred type + ambiguity).
        if self.data_manager:
            typed = self.data_manager.get_potential_moderators_typed()
        else:
            typed = []

        if not typed:
            self.run_button = widgets.Button(
                description="▶ Run Meta-Regression", button_style='success',
                icon='play', disabled=True)
            self.formula_preview = widgets.HTML("")
            return widgets.VBox([
                widgets.HTML("<h3>📊 Meta-Regression Analysis</h3>"),
                widgets.HTML("<p style='color:#c0392b;'>No numeric or categorical "
                             "moderators are available in the current dataset.</p>"),
                self.run_button
            ])

        df = self.data_manager.analysis_data if self.data_manager else None
        self._var_widgets = {}
        rows = []

        # Columns deliberately left out of the list, and why. Without this a
        # moderator that is missing from the panel looks like a bug in the
        # data rather than a rule the cell applied.
        _skipped = list(getattr(self.data_manager, 'skipped_moderators', []) or [])
        if _skipped:
            _items = "".join(
                f"<li><code>{nm}</code> &mdash; {why}</li>" for nm, why in _skipped)
            rows.append(widgets.HTML(
                "<details style='margin: 0 0 10px 0; font-family: sans-serif;"
                " font-size: 12px; color: #555;'>"
                f"<summary style='cursor: pointer; color: #2c3e50;'>"
                f"{len(_skipped)} column(s) not offered as moderators &mdash; "
                "click for the reason</summary>"
                f"<ul style='margin: 6px 0 0 0;'>{_items}</ul></details>"))

        # --- STRICT COLUMN LAYOUTS ---
        # Added overflow='hidden' to absolutely prevent inner scrollbars
        # col1_use is slightly wider, with center alignment to hold the checkbox cleanly
        col1_use = widgets.Layout(width='70px', min_width='70px', max_width='70px', overflow='hidden', display='flex', justify_content='center')
        col2_var = widgets.Layout(width='260px', min_width='260px', max_width='260px', padding='0 10px 0 0', overflow='hidden')
        col3_typ = widgets.Layout(width='160px', min_width='160px', max_width='160px', padding='0 10px 0 0', overflow='hidden')
        col4_ref = widgets.Layout(width='220px', min_width='220px', max_width='220px', overflow='hidden')

        # Row layout: Increased vertical padding and forced min_height to fit two lines of text
        row_layout = widgets.Layout(
            display='flex', flex_flow='row nowrap', align_items='center',
            padding='10px 5px', min_height='55px', border_bottom='1px solid #ecf0f1',
            overflow='hidden'
        )

        # 1. Build Header Row (slightly shorter min_height since it's only one line of text)
        header = widgets.HBox([
            widgets.HTML("<div style='color: #7f8c8d; font-size: 0.9em; text-transform: uppercase; text-align: center;'><b>Use</b></div>", layout=col1_use),
            widgets.HTML("<div style='color: #7f8c8d; font-size: 0.9em; text-transform: uppercase;'><b>Variable</b></div>", layout=col2_var),
            widgets.HTML("<div style='color: #7f8c8d; font-size: 0.9em; text-transform: uppercase;'><b>Data Type</b></div>", layout=col3_typ),
            widgets.HTML("<div style='color: #7f8c8d; font-size: 0.9em; text-transform: uppercase;'><b>Reference Level</b></div>", layout=col4_ref)
        ], layout=widgets.Layout(padding='0 5px 10px 5px', min_height='35px', border_bottom='2px solid #bdc3c7', margin_bottom='5px', overflow='hidden'))
        rows.append(header)

        # Dropdown style override to kill the invisible description space
        no_desc_style = {'description_width': '0px'}

        # 2. Build rows for each variable
        for var in typed:
            v_name = var['name']
            v_type = var['inferred_type']
            is_ambig = var['ambiguous']

            # Inclusion Checkbox: strict 30px width prevents the horizontal scrollbar
            cb_include = widgets.Checkbox(
                value=False, indent=False, description='', style=no_desc_style,
                layout=widgets.Layout(width='30px', margin='0')
            )
            cb_container = widgets.HBox([cb_include], layout=col1_use)

            # Variable Label
            _note = var.get('note') or ''
            _note_html = (f" &middot; <span style='color:#d35400;'>{_note}</span>"
                          if _note else "")
            lbl = widgets.HTML(
                value=f"<div style='overflow: hidden; text-overflow: ellipsis; white-space: nowrap; line-height: 1.3;'>"
                      f"<span style='font-family: monospace; font-size: 1.1em; color: #2c3e50;'>{v_name}</span><br>"
                      f"<span style='font-size: 0.8em; color: #95a5a6;'>{var['n_levels']} unique levels{_note_html}</span>"
                      f"</div>",
                layout=col2_var
            )

            # Type Dropdown
            type_opts = ['Continuous', 'Categorical']
            default_type = 'Categorical' if v_type == 'categorical' else 'Continuous'
            dd_type = widgets.Dropdown(
                options=type_opts, value=default_type, description='',
                style=no_desc_style, layout=col3_typ
            )
            if v_type == 'categorical' and not is_ambig:
                dd_type.disabled = True

            # Reference Dropdown
            levels = []
            if df is not None and v_name in df.columns:
                levels = sorted(df[v_name].astype(str).dropna().unique().tolist())

            dd_ref = widgets.Dropdown(
                options=levels, value=levels[0] if levels else None, description='',
                style=no_desc_style, layout=col4_ref
            )
            dd_ref.disabled = True

            # Store references
            self._var_widgets[v_name] = {'include': cb_include, 'type': dd_type, 'ref': dd_ref}

            # Scoped observer
            def make_row_observer(name):
                def observer(change):
                    w = self._var_widgets[name]
                    is_included = w['include'].value
                    is_cat = w['type'].value == 'Categorical'

                    w['ref'].disabled = not (is_included and is_cat)
                    self._refresh_formula_preview()
                return observer

            row_obs = make_row_observer(v_name)
            cb_include.observe(row_obs, names='value')
            dd_type.observe(row_obs, names='value')
            dd_ref.observe(lambda c: self._refresh_formula_preview(), names='value')

            # Assemble Row
            row = widgets.HBox([cb_container, lbl, dd_type, dd_ref], layout=row_layout)
            rows.append(row)

        # 3. Wrap rows in a scrolling container
        matrix_panel = widgets.VBox(
            rows,
            layout=widgets.Layout(
                width='850px', max_height='450px', overflow_y='auto', overflow_x='hidden',
                border='1px solid #dcdde1', padding='10px', background_color='#fcfcfc', border_radius='6px'
            )
        )

        # 4. Standard UI Components
        self.formula_preview = widgets.HTML("")
        self.run_button = widgets.Button(
            description="▶ Run Meta-Regression", button_style='success', icon='play',
            layout=widgets.Layout(width='250px', height='40px', margin='15px 0 0 0')
        )
        self.run_button.on_click(self._handle_run_click)

        self._refresh_formula_preview()

        # ---> ADD THIS LINE: Create an empty container for the results
        self.results_container = widgets.VBox([])

        ui = widgets.VBox([
            widgets.HTML("<h3>📊 Meta-Regression Analysis</h3>"),
            widgets.HTML("<p style='color:#6c757d; margin-bottom: 15px;'>Select the variables to include in the model. You can override the inferred data type. For categorical factors, select which level acts as the baseline (reference).</p>"),
            matrix_panel,
            widgets.HTML("<br>"),
            self.formula_preview,
            self.run_button,
            # ---> ADD THIS LINE: Put the container below the button
            self.results_container
        ])
        return ui


    def _collect_selection(self):
        """Read state directly from the control panel matrix."""
        moderator_cols = []
        categorical_cols = []
        reference_levels = {}

        for v_name, widgets_dict in self._var_widgets.items():
            if widgets_dict['include'].value:  # If checkbox is ticked
                moderator_cols.append(v_name)

                if widgets_dict['type'].value == 'Categorical':
                    categorical_cols.append(v_name)
                    if widgets_dict['ref'].value is not None:
                        reference_levels[v_name] = widgets_dict['ref'].value

        return moderator_cols, categorical_cols, reference_levels

    def _refresh_formula_preview(self) -> None:
        """Update the live formula preview from the current matrix state."""
        moderator_cols, categorical_cols, reference_levels = self._collect_selection()

        if not moderator_cols:
            self.formula_preview.value = (
                "<div style='color:#6c757d; font-style:italic; padding:8px 0;'>"
                "Check the 'Use' box next to at least one variable to preview the model formula.</div>"
            )
            return

        effect_col = self.data_manager.effect_col if self.data_manager else "effect"
        formula = self.view.templates.build_formula_string(
            effect_col=effect_col,
            moderator_cols=moderator_cols,
            categorical_cols=categorical_cols,
            reference_levels=reference_levels,
        )
        self.formula_preview.value = self.view.templates.formula_preview_box(
            formula, title="Model Formula (preview)",
            note="Updates instantly. Categorical factors expand to dummy columns at run time (reference level is omitted)."
        )


    # -------------------------------------------------------------------------
    # MAIN EXECUTION METHOD
    # -------------------------------------------------------------------------

    def run_analysis(
        self,
        moderator_cols=None,
        categorical_cols=None,
        reference_levels=None,
        moderator_col=None,   # legacy single-moderator arg (back-compat)
    ) -> None:
        """Execute the meta-regression workflow for the current selection.

        Accepts either the multi-moderator form (moderator_cols + optional
        categorical_cols / reference_levels) or a single legacy moderator_col
        string. Renders results into the tabs, or a clean error message.
        """
        # Insert tabs into the container
        if hasattr(self, 'results_container'):
            self.results_container.children = [self.view.stale_banner, self.view.tabs]
        else:
            display(widgets.VBox([self.view.stale_banner, self.view.tabs]))

        # Initialization errors first.
        if self._initialization_error:
            self._handle_initialization_error()
            return

        # Normalize selection.
        if moderator_cols is None and moderator_col is not None:
            moderator_cols = [moderator_col]
        moderator_cols = list(moderator_cols or [])
        categorical_cols = list(categorical_cols or [])
        reference_levels = dict(reference_levels or {})

        # Validate selection.
        if not moderator_cols:
            self.view.render_error("No moderators selected",
                                   "Select at least one moderator and try again.")
            return
        if any(m in ('No valid moderators found', 'Data not loaded', None, '')
               for m in moderator_cols):
            self.view.render_error("No valid moderator selected")
            return
        primary_mod = moderator_cols[0]

        def progress_callback(message: str):
            pass

        try:
            result_tuple = self.engine.run_meta_regression(
                moderator_cols=moderator_cols,
                categorical_cols=categorical_cols,
                reference_levels=reference_levels,
                progress_callback=progress_callback,
            )

            if result_tuple is None:
                self.view.render_error(
                    "Analysis failed",
                    "Optimization did not converge or data were insufficient. "
                    "Try a different moderator selection or check your data."
                )
                return

            result, reg_df_for_plot = result_tuple

            # Render all tabs.
            self._render_all_tabs(result, primary_mod, reg_df_for_plot)

            # Persist + banner bookkeeping.
            self._save_results(result, reg_df_for_plot)
            clear_stale('regression')
            mark_stale(['plots', 'spline'], "Meta-Regression Re-run")

        except ValueError as e:
            self.view.render_error("Data Error", str(e))
        except RuntimeError as e:
            # Includes the guard for multi/categorical 3-level data.
            self.view.render_error("Not Available", str(e))
        except Exception as e:
            self._handle_unexpected_error(e)


    # -------------------------------------------------------------------------
    # TAB RENDERING
    # -------------------------------------------------------------------------

    def _render_all_tabs(
        self,
        result: 'RegressionResult',
        moderator_col,
        reg_df: pd.DataFrame
    ) -> None:
        """Render all tabs with results. `moderator_col` is the primary
        moderator name (used for legacy single-moderator display fields)."""
        self.view.render_results_tab(result, moderator_col)
        self.view.render_diagnostics_tab(result)
        self.view.render_details_tab(
            result=result,
            moderator_col=moderator_col,
            reg_df=reg_df,
            effect_col=self.data_manager.effect_col,
            var_col=self.data_manager.var_col,
        )
        combined_text = self.view.render_publication_tab(
            result=result,
            moderator_col=moderator_col,
            es_config=self.data_manager.es_config,
        )
        self.data_manager.save_publication_text(combined_text)
        self.view.render_export_tab(export_callback=self._handle_export)


    # -------------------------------------------------------------------------
    # DATA PERSISTENCE
    # -------------------------------------------------------------------------

    def _save_results(
        self,
        result: 'RegressionResult',
        reg_df: pd.DataFrame
    ) -> None:
        """
        Save results to ANALYSIS_CONFIG.

        Args:
            result: RegressionResult object
            reg_df: DataFrame used for regression
        """
        self.data_manager.save_regression_results(result, reg_df)

    # -------------------------------------------------------------------------
    # EVENT HANDLERS
    # -------------------------------------------------------------------------

    def _handle_run_click(self, button) -> None:
        """Handle run button click: read the multi-moderator selection."""
        moderator_cols, categorical_cols, reference_levels = self._collect_selection()
        self.run_analysis(
            moderator_cols=moderator_cols,
            categorical_cols=categorical_cols,
            reference_levels=reference_levels,
        )

    def _handle_export(self, button) -> None:
        """Handle export button click"""
        try:
            # Call the external export function if it exists
            if 'export_analysis_report' in globals():
                export_analysis_report(
                    report_type='regression',
                    filename_prefix='Meta_Regression_Audit'
                )
            else:
                with self.view.tab_export:
                    display(HTML("""
                    <div style='background-color: #fff3cd; color: #856404; padding: 12px; border-radius: 5px; border-left: 4px solid #ffc107; margin-top: 15px;'>
                        <b>⚠️ Warning:</b> Export function not found. Please run the Excel Export utility cell first.
                    </div>
                    """))
        except Exception as e:
            with self.view.tab_export:
                display(HTML(f"""
                <div style='background-color: #f8d7da; color: #721c24; padding: 12px; border-radius: 5px; border-left: 4px solid #dc3545; margin-top: 15px;'>
                    <b>❌ Export failed:</b> {str(e)}
                </div>
                """))
                traceback.print_exc()

    # -------------------------------------------------------------------------
    # ERROR HANDLING
    # -------------------------------------------------------------------------

    def _handle_initialization_error(self) -> None:
        """Handle errors during controller initialization"""
        error = self._initialization_error
        if error is None:
            return

        error_type = type(error).__name__
        error_msg = str(error)
        details = traceback.format_exc()

        self.view.render_error(f"{error_type}: {error_msg}", details)

    def _handle_unexpected_error(self, error: Exception) -> None:
        """Handle unexpected errors during analysis"""
        error_type = type(error).__name__
        error_msg = str(error)
        details = traceback.format_exc()

        self.view.render_error(f"{error_type}: {error_msg}", details)


# =============================================================================
# EXECUTION ENTRY POINT
# =============================================================================

def run_meta_regression_analysis():
    """
    Main entry point for meta-regression analysis.
    Call this function to display the UI and enable analysis.
    """
    try:
        # Check for ANALYSIS_CONFIG
        if 'ANALYSIS_CONFIG' not in globals():
            display(HTML("""
            <div style='background-color: #f8d7da; color: #721c24; padding: 15px; border-radius: 8px; border: 1px solid #f5c6cb; margin-bottom: 15px;'>
                <h4 style='margin-top: 0; color: #721c24;'>❌ Configuration Missing</h4>
                <p style='margin-bottom: 8px;'><b>ANALYSIS_CONFIG</b> was not found in the global environment. Please run the previous analysis cells first:</p>
                <ul style='margin-top: 0; margin-bottom: 0;'>
                    <li><b>Steps 2–6:</b> Data loading, cleaning and effect sizes</li>
                    <li><b>Step 7:</b> Overall Meta-Analysis (required for R² calculations)</li>
                </ul>
            </div>
            """))
            return

        # Create controller
        controller = RegressionController(ANALYSIS_CONFIG)

        # Display UI
        ui = controller.create_ui()
        display(ui)

        # Store controller globally for access (optional)
        globals()['_regression_controller'] = controller

    except Exception as e:
        display(HTML(f"""
        <div style='background-color: #f8d7da; color: #721c24; padding: 15px; border-radius: 8px; border: 1px solid #f5c6cb;'>
            <h4 style='margin-top: 0; color: #721c24;'>❌ Fatal Initialization Error</h4>
            <p style='margin-bottom: 0;'><b>{type(e).__name__}:</b> {str(e)}</p>
        </div>
        """))
        traceback.print_exc()


# =============================================================================
# STANDALONE TESTING UTILITIES (Optional)
# =============================================================================

class MockRegressionConfig:
    """
    Mock ANALYSIS_CONFIG for testing without running full pipeline.
    Only for development/testing purposes.
    """

    @staticmethod
    def create_sample_config():
        """Create a minimal valid config for testing"""
        import pandas as pd
        import numpy as np

        # Sample data with moderator
        np.random.seed(42)
        sample_data = pd.DataFrame({
            'id': [1, 1, 2, 2, 3, 3, 4, 4, 5, 5],
            'hedges_g': np.random.randn(10) * 0.5 + 0.3,
            'Vg': np.random.uniform(0.01, 0.1, 10),
            'year': [2010, 2010, 2012, 2012, 2015, 2015, 2018, 2018, 2020, 2020],
            'sample_size': np.random.randint(50, 200, 10)
        })

        # Add linear relationship for testing
        sample_data['hedges_g'] = sample_data['hedges_g'] + 0.01 * sample_data['year']

        return {
            'analysis_data': sample_data,
            'effect_col': 'hedges_g',
            'var_col': 'Vg',
            'es_config': {'has_fold_change': False},
            'overall_results': {
                'tau_squared': 0.15,
                'mu': 0.3
            },
            'global_settings': {
                'alpha': 0.05,
                'dist_type': 't'
            }
        }


# =============================================================================
# BACKWARD COMPATIBILITY WRAPPER
# =============================================================================

def create_legacy_widgets():
    """
    Create widgets in the original style for backward compatibility.
    This allows the new MVC code to work with existing notebook structure.

    Returns:
        Tuple of (moderator_widget, run_button)
    """
    if 'ANALYSIS_CONFIG' not in globals():
        print("⚠️ Warning: ANALYSIS_CONFIG not found")
        return None, None

    try:
        controller = RegressionController(ANALYSIS_CONFIG)
        ui = controller.create_ui()

        # Store controller globally
        globals()['_regression_controller'] = controller

        return controller.moderator_widget, controller.run_button

    except Exception as e:
        print(f"❌ Error creating widgets: {e}")
        return None, None


run_meta_regression_analysis()

In [ ]:
#@title 📈 13. Meta-Regression: Visualization
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")


# =============================================================================
# STEP 13: META-REGRESSION BUBBLE PLOT
# Purpose: Plot the effect sizes against a continuous moderator with the
#          fitted meta-regression line and its confidence band.
# Data:    Step 12 (meta_regression_RVE_results), or a plot recipe file
#          saved from this cell.
# =============================================================================

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import html


def _meta_regression_plot_cell():
    PLOT_TYPE = 'meta_regression'
    RESULTS_KEY = 'meta_regression_RVE_results'
    state = {'data': None, 'source': None, 'recipe_name': None, 'token': None}

    # --- 1. DATA SNAPSHOT (everything the figure draws) ---
    def _terms(res):
        """Continuous and categorical moderators of the fitted model, from its term names."""
        terms = [str(t) for t in (res.get('term_names') or ['Intercept', res['moderator_col_name']])]
        cont, cat = {}, {}
        for j, t in enumerate(terms):
            if j == 0:
                continue
            if t.endswith(' (centered)'):
                cont[t[:-len(' (centered)')]] = j
            elif '[' in t and t.endswith(']'):
                cat.setdefault(t.split('[', 1)[0], []).append((j, t.split('[', 1)[1][:-1]))
            else:
                cont[t] = j
        return terms, cont, cat

    def _category_values(col, reg, ad):
        """Colour value of each point; text, with missing values shown as 'Missing'."""
        if col in reg.columns:
            ser = reg[col]
        elif ad is not None and 'id' in reg.columns and 'id' in ad.columns and col in ad.columns:
            # study-level rows (2-level model): one value per study, 'Mixed' if it varies
            per = {}
            for key, g in ad.groupby('id')[col]:
                u = pd.unique(g.dropna().astype(str).str.strip())
                per[key] = u[0] if len(u) == 1 else ('Mixed' if len(u) > 1 else None)
            ser = reg['id'].map(per)
        else:
            return None
        return [('Missing' if (v is None or (not isinstance(v, str) and pd.isna(v)) or str(v).strip() in ('', 'nan'))
                 else str(v).strip()) for v in ser.tolist()]

    def build_data():
        res = ANALYSIS_CONFIG.get(RESULTS_KEY)
        if not res:
            return None, None
        gs = ANALYSIS_CONFIG.get('global_settings') or {}
        es = ANALYSIS_CONFIG.get('es_config') or {}
        alpha = float(gs.get('alpha', 0.05))
        dist_type = gs.get('dist_type', 't')
        ci_level = ci_pct_label(alpha)
        reg = res['reg_df']
        eff_col = res.get('effect_col') or ANALYSIS_CONFIG.get('effect_col')
        var_col = ANALYSIS_CONFIG.get('var_col')
        ad = ANALYSIS_CONFIG.get('analysis_data')

        betas = np.asarray(res.get('betas_all', res['betas']), float).ravel()
        p = betas.size
        V = np.asarray(res.get('var_betas_full', res['var_betas_robust']), float).reshape(p, p)
        dfs = np.asarray(res.get('df_all', res.get('df_robust')), float).ravel()
        if dfs.size != p:
            dfs = np.full(p, float(dfs.flat[0]))
        pvals = np.asarray(res.get('p_values_all', [res.get('p_intercept'), res.get('p_slope')]), float).ravel()
        terms, cont, cat = _terms(res)

        model_type = str(res.get('model_type', ''))
        tau_sq = float(res.get('tau_sq') or 0.0)
        sigma_sq = float(res.get('sigma_sq') or 0.0)
        k_studies = int(res.get('k_studies') or (reg['id'].nunique() if 'id' in reg.columns else len(reg)))
        two_level = 'KH' in model_type or 'Aggregated' in model_type
        if two_level:
            se_method = 'kh'
        elif np.allclose(dfs, max(1, k_studies - p)):
            se_method = 'model'    # the CR2 step failed and Step 12 fell back to model-based SEs
        else:
            se_method = 'cr2'

        # held values of every moderator (continuous at their mean, categorical at the reference)
        means = {m: float(pd.to_numeric(reg[m], errors='coerce').mean()) for m in cont}
        refs = {}
        for m, lv in cat.items():
            present = sorted(reg[m].astype(str).unique().tolist()) if m in reg.columns else []
            others = [x for x in present if x not in {l for _, l in lv}]
            refs[m] = others[0] if others else 'reference'

        crit = lambda j: (stats.t.ppf(1 - alpha / 2, dfs[j]) if dist_type == 't' else stats.norm.ppf(1 - alpha / 2))
        lines, slopes = {}, {}
        for m, j in cont.items():
            xv = pd.to_numeric(reg[m], errors='coerce').to_numpy(float)
            lo, hi = np.nanmin(xv), np.nanmax(xv)
            pad = (hi - lo) * 0.05 if hi > lo else 1.0
            grid = np.linspace(lo - pad, hi + pad, 100)
            rows = np.zeros((grid.size, p))
            rows[:, 0] = 1.0
            rows[:, j] = grid - means[m]
            fit = rows @ betas
            se = np.sqrt(np.maximum(np.sum((rows @ V) * rows, axis=1), 0.0))
            lines[m] = pd.DataFrame({'x': grid, 'fit': fit, 'lower': fit - crit(j) * se, 'upper': fit + crit(j) * se})
            slopes[m] = {'slope': float(betas[j]), 'intercept': float(betas[0] - betas[j] * means[m]),
                         'p': float(pvals[j]) if pvals.size > j else None, 'df': float(dfs[j]),
                         'mean': means[m], 'term': terms[j]}

        v = reg[var_col].to_numpy(float)
        pts = pd.DataFrame({'y': reg[eff_col].to_numpy(float), 'weight': 1.0 / (v + tau_sq + sigma_sq)})
        for m in cont:
            pts[f'x::{m}'] = pd.to_numeric(reg[m], errors='coerce').to_numpy(float)

        # categorical columns offered for colouring the points
        excluded = {ANALYSIS_CONFIG.get('effect_col'), ANALYSIS_CONFIG.get('var_col'), ANALYSIS_CONFIG.get('se_col'),
                    'w_fixed', 'w_random', 'id', 'shared_group_id', 'xe', 'sde', 'ne', 'xc', 'sdc', 'nc',
                    ANALYSIS_CONFIG.get('ci_lower_col'), ANALYSIS_CONFIG.get('ci_upper_col'), eff_col, var_col}
        base = ad if isinstance(ad, pd.DataFrame) else reg
        candidates = [c for c in base.columns
                      if c not in excluded and (is_text_dtype(base[c]) or isinstance(base[c].dtype, pd.CategoricalDtype))
                      and base[c].nunique() <= 10]
        for m in cat:
            if m not in candidates:
                candidates.append(m)
        color_levels = {}
        for c in candidates:
            vals = _category_values(c, reg, ad if isinstance(ad, pd.DataFrame) else None)
            if vals is None:
                continue
            pts[f'c::{c}'] = vals
            tail = [x for x in ('Mixed', 'Missing') if x in vals]
            color_levels[str(c)] = sorted({x for x in vals if x not in tail}, key=str) + tail

        r2 = res.get('R_squared_adj')
        data = {
            'points': pts, 'lines': lines, 'slopes': slopes,
            'continuous': list(cont), 'categorical': list(cat), 'means': means, 'refs': refs,
            'color_levels': color_levels,
            'r2': None if r2 is None else float(r2),
            'model_type': model_type, 'two_level': bool(two_level), 'se_method': se_method,
            'tau_sq': tau_sq, 'sigma_sq': sigma_sq, 'k_studies': k_studies, 'n_obs': int(res.get('n_obs') or len(reg)),
            'n_points': int(len(reg)), 'n_coef': int(p),
            'ci_level': ci_level, 'dist_type': dist_type,
            'null_value': es_null_value(), 'effect_label': es.get('effect_label', 'Effect Size'),
        }
        when = res.get('timestamp')
        mods = list(cont) + list(cat)
        source = {
            'analysis': f"Meta-regression: {', '.join(mods)}",
            'effect_label': data['effect_label'],
            'model': model_type,
            'ci_level': ci_level,
            'analysis_time': when.strftime('%Y-%m-%d %H:%M') if hasattr(when, 'strftime') else when,
        }
        return data, source

    # --- 2. CONTROLS (every widget whose value is saved in a recipe) ---
    st = {'description_width': '120px'}
    L = lambda w='450px': widgets.Layout(width=w)
    C = {
        'x_axis': widgets.Dropdown(options=['(none)'], description='X-Axis:', style=st, layout=L()),
        'color_by': widgets.Dropdown(options=['None'], value='None', description='Color By:', style=st, layout=L()),
        'show_title': widgets.Checkbox(value=True, description='Show Plot Title', indent=False),
        'title': widgets.Text(value='Meta-Regression Plot', description='Plot Title:', layout=L(), style=st),
        'xlabel': widgets.Text(value='Moderator', description='X-Axis Label:', layout=L(), style=st),
        'ylabel': widgets.Text(value='Effect Size', description='Y-Axis Label:', layout=L(), style=st),
        'width': widgets.FloatSlider(value=8.0, min=2.0, max=14.0, step=0.1, description='Plot Width (in):',
                                     continuous_update=False, style=st, layout=L()),
        'height': widgets.FloatSlider(value=6.0, min=2.0, max=12.0, step=0.1, description='Plot Height (in):',
                                      continuous_update=False, style=st, layout=L()),
        'title_font': widgets.IntSlider(value=14, min=6, max=24, description='Title Font:', style=st, layout=L()),
        'label_font': widgets.IntSlider(value=12, min=6, max=18, description='Label Font:', style=st, layout=L()),
        'tick_font': widgets.IntSlider(value=10, min=5, max=16, description='Tick Font:', style=st, layout=L()),
        'point_color': widgets.Dropdown(options=['gray', 'blue', 'red', 'green', 'purple', 'orange'], value='gray',
                                        description='Point Color:', style=st, layout=L()),
        'bubble_base': widgets.IntSlider(value=20, min=0, max=200, step=10, description='Min Bubble Size:',
                                         continuous_update=False, style=st, layout=L()),
        'bubble_range': widgets.IntSlider(value=800, min=100, max=2000, step=100, description='Max Bubble Size:',
                                          continuous_update=False, style=st, layout=L()),
        'bubble_alpha': widgets.FloatSlider(value=0.6, min=0.1, max=1.0, step=0.1, description='Transparency:',
                                            continuous_update=False, style=st, layout=L()),
        'line_color': widgets.Dropdown(options=['red', 'blue', 'black', 'green', 'purple'], value='red',
                                       description='Line Color:', style=st, layout=L()),
        'line_width': widgets.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.5, description='Line Width:',
                                          continuous_update=False, style=st, layout=L()),
        'show_ci': widgets.Checkbox(value=True, description='Show Confidence Band', indent=False),
        'ci_alpha': widgets.FloatSlider(value=0.3, min=0.1, max=0.8, step=0.1, description='CI Transparency:',
                                        continuous_update=False, style=st, layout=L()),
        'show_equation': widgets.Checkbox(value=True, description='Show Regression Equation & P-value', indent=False),
        'show_r2': widgets.Checkbox(value=True, description='Show R² Value', indent=False),
        'grid': widgets.Checkbox(value=True, description='Show Grid', indent=False),
        'show_null': widgets.Checkbox(value=True, description='Show Null Effect Line', indent=False),
        'legend_loc': widgets.Dropdown(options=['best', 'upper right', 'upper left', 'lower left', 'lower right'],
                                       value='best', description='Legend Position:', style=st, layout=L()),
        'legend_font': widgets.IntSlider(value=10, min=5, max=16, step=1, description='Legend Font:',
                                         continuous_update=False, style=st, layout=L()),
        'save_pdf': widgets.Checkbox(value=True, description='Save as PDF', indent=False),
        'save_png': widgets.Checkbox(value=True, description='Save as PNG', indent=False),
        'save_svg': widgets.Checkbox(value=False, description='Save as SVG', indent=False),
        'dpi': widgets.IntSlider(value=300, min=100, max=600, step=50, description='PNG DPI:',
                                 continuous_update=False, style=st, layout=L()),
        'filename': widgets.Text(value='MetaRegression_Plot', description='Filename Prefix:', layout=L(), style=st),
        'transparent': widgets.Checkbox(value=False, description='Transparent Background', indent=False),
    }
    label_box = widgets.VBox()

    def build_label_widgets(data):
        """One Text per colour moderator and per category, saved as 'label::<name>'."""
        for k in [k for k in C if k.startswith('label::')]:
            del C[k]
        names = set()
        for col, levels in data['color_levels'].items():
            names.add(col)
            names.update(levels)
        names.discard('')
        for nm in sorted(names, key=str):
            C[f'label::{nm}'] = widgets.Text(value=nm, description=f"{nm}:", layout=widgets.Layout(width='500px'),
                                             style={'description_width': '200px'})
        rows = [C[f'label::{nm}'] for nm in sorted(names, key=str)]
        label_box.children = rows or [widgets.Label("No categorical labels found to edit.")]

    def axis_defaults(mod):
        if state['data'] is None or mod not in state['data']['continuous']:
            return
        C['xlabel'].value = mod
        C['title'].value = f"Meta-Regression: {state['data']['effect_label']} vs. {mod}"

    def set_data_defaults(data):
        """Defaults and data-dependent widgets (called for session data and recipes)."""
        C['color_by'].options = ['None'] + list(data['color_levels'])
        C['color_by'].value = 'None'
        C['x_axis'].options = list(data['continuous']) or ['(none)']
        C['x_axis'].value = C['x_axis'].options[0]
        axis_defaults(C['x_axis'].value)
        C['ylabel'].value = data['effect_label']
        build_label_widgets(data)

    C['x_axis'].observe(lambda ch: axis_defaults(ch['new']), names='value')

    def on_color_by(change):
        C['point_color'].layout.display = 'none' if change['new'] != 'None' else 'flex'
    C['color_by'].observe(on_color_by, names='value')

    # --- 3. DRAWING (data + settings -> figure; no widget access) ---
    SE_SHORT = {'cr2': 'CR2', 'kh': 'Knapp-Hartung', 'model': 'model-based'}

    def _bubble_sizes(w, s):
        lo, hi = np.nanmin(w), np.nanmax(w)
        if hi > lo:
            return s['bubble_base'] + (w - lo) / (hi - lo) * s['bubble_range']
        return np.full(w.shape, s['bubble_base'] + s['bubble_range'] / 2)

    def draw(data, s):
        mod = s['x_axis']
        if mod not in data['lines']:
            raise ValueError(f"'{mod}' is not a continuous moderator of this meta-regression.")
        pts, line, sl = data['points'], data['lines'][mod], data['slopes'][mod]
        x = pts[f'x::{mod}'].to_numpy(float)
        y = pts['y'].to_numpy(float)
        size = _bubble_sizes(pts['weight'].to_numpy(float), s)
        labmap = {k[len('label::'):]: v for k, v in s.items() if k.startswith('label::')}
        color_by = s['color_by'] if s['color_by'] in data['color_levels'] else 'None'

        fig, ax = plt.subplots(figsize=(s['width'], s['height']))
        if s['transparent']:
            fig.patch.set_alpha(0)
            ax.patch.set_alpha(0)

        cat_handles = []
        if color_by == 'None':
            ax.scatter(x, y, s=size, color=s['point_color'], alpha=s['bubble_alpha'], edgecolors='black',
                       linewidths=0.5, zorder=3)
        else:
            cats = pts[f'c::{color_by}'].to_numpy(object)
            cmap = plt.get_cmap('tab10')
            for i, lv in enumerate(data['color_levels'][color_by]):
                m = cats == lv
                if not m.any():
                    continue
                ax.scatter(x[m], y[m], s=size[m], color=cmap(i % 10), alpha=s['bubble_alpha'], edgecolors='black',
                           linewidths=0.5, zorder=3)
                cat_handles.append(mpatches.Patch(color=cmap(i % 10), alpha=s['bubble_alpha'], ec='black', lw=0.5,
                                                  label=labmap.get(lv, lv)))

        ax.plot(line['x'], line['fit'], color=s['line_color'], linewidth=s['line_width'], zorder=2,
                label='Regression Line')
        if s['show_ci']:
            df_txt = f", df={sl['df']:.1f}" if data['dist_type'] == 't' else ''
            ax.fill_between(line['x'], line['lower'], line['upper'], color=s['line_color'], alpha=s['ci_alpha'],
                            zorder=1, label=f"{data['ci_level']} CI ({SE_SHORT[data['se_method']]}{df_txt})")
        if s['show_null']:
            ax.axhline(data['null_value'], color='gray', linestyle='--', linewidth=1.0, zorder=0)

        ax.set_xlabel(s['xlabel'], fontsize=s['label_font'], fontweight='bold')
        ax.set_ylabel(s['ylabel'], fontsize=s['label_font'], fontweight='bold')
        ax.tick_params(axis='both', labelsize=s['tick_font'])
        if s['show_title']:
            ax.set_title(s['title'], fontsize=s['title_font'], fontweight='bold', pad=15)
        if s['grid']:
            ax.grid(True, linestyle=':', alpha=0.4, zorder=0)

        text_lines = []
        if s['show_equation']:
            b1 = sl['slope']
            sign = '+' if b1 >= 0 else '−'
            text_lines.append(f"y = {sl['intercept']:.3f} {sign} {abs(b1):.3g}x".replace('-', '−'))
            if data['n_coef'] > 2:
                text_lines.append("(other moderators at mean / reference)")
            pv = sl['p']
            if pv is not None and np.isfinite(pv):
                sig = "***" if pv < 0.001 else "**" if pv < 0.01 else "*" if pv < 0.05 else "ns"
                text_lines.append(f"p (slope) = {pv:.3g} {sig}")
        if s['show_r2']:
            r2 = data['r2']
            text_lines.append(f"pseudo-R² = {r2:.1f}%" if r2 is not None and np.isfinite(r2) else "pseudo-R² = n/a")
        if text_lines:
            ax.text(0.05, 0.95, "\n".join(text_lines), transform=ax.transAxes, fontsize=s['legend_font'],
                    verticalalignment='top', zorder=10,
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))

        handles, labels = ax.get_legend_handles_labels()
        handles += cat_handles
        labels += [h.get_label() for h in cat_handles]
        w_txt = "Weight 1/(vᵢ + τ²)" if data['two_level'] else "Weight 1/(vᵢ + τ² + σ²)"
        handles.append(mlines.Line2D([], [], linestyle='None', marker='o',
                                     markersize=np.sqrt(s['bubble_base'] + s['bubble_range'] / 2),
                                     markerfacecolor='gray' if color_by == 'None' else 'lightgray',
                                     markeredgecolor='black', markeredgewidth=0.5, alpha=s['bubble_alpha']))
        labels.append(w_txt)
        ax.legend(handles=handles, labels=labels, loc=s['legend_loc'], fontsize=s['legend_font'], framealpha=0.9,
                  title=labmap.get(color_by, color_by) if color_by != 'None' else None,
                  title_fontsize=s['legend_font'])
        fig.tight_layout()
        return fig

    def caption(data, s):
        mod = s['x_axis']
        sl = data['slopes'][mod]
        esc = lambda t: html.escape(str(t))
        lab = {k[len('label::'):]: v for k, v in s.items() if k.startswith('label::')}
        model = ("a two-level random-effects meta-regression (one effect size per study)" if data['two_level']
                 else "a three-level meta-regression (effect sizes nested in studies)")
        txt = (f"Meta-regression bubble plot of {esc(data['effect_label'])} against <b>{esc(mod)}</b>, from {model} "
               f"(k = {data['k_studies']} studies, n = {data['n_obs']} effect sizes). The solid line is the fitted "
               f"regression line")
        held = []
        for m in data['continuous']:
            if m != mod:
                held.append(f"{esc(m)} at its mean ({data['means'][m]:.3g})")
        for m in data['categorical']:
            held.append(f"{esc(m)} at its reference level ({esc(lab.get(data['refs'][m], data['refs'][m]))})")
        if held:
            txt += ", with the other moderators held fixed: " + "; ".join(held) + " (the full model's coefficients and covariance)"
        txt += "."
        if s['show_ci']:
            se_desc = {'cr2': "bias-reduced cluster-robust (CR2) standard errors, clustered by study",
                       'kh': "Knapp-Hartung adjusted (truncated) model-based standard errors",
                       'model': "model-based standard errors"}[data['se_method']]
            dist = (f"t distribution, df = {sl['df']:.1f}" if data['dist_type'] == 't' else "normal distribution")
            txt += f" The shaded band is the pointwise {data['ci_level']} confidence interval of the line ({se_desc}; {dist})."
        unit = "study" if data['two_level'] else "effect size"
        wf = "1/(vᵢ + τ²)" if data['two_level'] else "1/(vᵢ + τ² + σ²)"
        comps = f"τ² = {data['tau_sq']:.4g}" + ("" if data['two_level'] else f", σ² = {data['sigma_sq']:.4g}")
        txt += (f" Each point is one {unit}; bubble size is proportional to its weight {wf}, using the model's "
                f"residual variance components ({comps}).")
        if s['color_by'] != 'None' and s['color_by'] in data['color_levels']:
            txt += f" Points are coloured by <b>{esc(lab.get(s['color_by'], s['color_by']))}</b>."
        if s['show_null']:
            txt += f" The dashed horizontal line marks no effect ({data['null_value']:g})."
        if s['show_r2'] and data['r2'] is not None and np.isfinite(data['r2']):
            txt += f" Pseudo-R² of the model: {data['r2']:.1f}%."
        return txt

    # --- 4. ACTIONS ---
    out = widgets.Output()
    caption_output = widgets.Output()
    source_box = widgets.HTML()

    def refresh_source():
        source_box.value = source_html(state['source'], state['recipe_name'])

    def no_continuous_html(data):
        cats = ', '.join(html.escape(c) for c in data['categorical']) or 'none'
        return ("<div style='padding:10px; background:#fff3cd; border-left:4px solid #ffc107; font-size:13px;'>"
                "⚠️ This meta-regression has no continuous moderator (categorical moderators: "
                f"<b>{cats}</b>), so there is no regression line to draw against an x-axis. Include a continuous "
                "moderator in Step 12, or show categorical moderators with the orchard plot (Step 11).</div>")

    def generate(_b=None):
        with out:
            clear_output(wait=True)
            if state['data'] is None:
                display(HTML("<div style='color:#721c24;'>❌ No meta-regression results: run Step 12 "
                             "(Meta-Regression) or load a plot recipe.</div>"))
                return
            if not state['data']['continuous']:
                display(HTML(no_continuous_html(state['data'])))
                return
            if state['recipe_name'] is None and results_token(RESULTS_KEY) != state['token']:
                display(HTML(stale_warning_html('Step 12 (Meta-Regression)')))
            s = widget_values(C)
            try:
                fig = draw(state['data'], s)
                paths = cometa_save_figure(fig, f"{s['filename']}_{s['x_axis']}", pdf=s['save_pdf'],
                                           png=s['save_png'], svg=s['save_svg'], dpi=s['dpi'],
                                           transparent=s['transparent'])
                show_figure(fig)
            except Exception as e:
                plt.close('all')
                display(HTML(f"<div style='color:#721c24;'>❌ Plotting error: {html.escape(str(e))}</div>"))
                return
            display(HTML(saved_files_html(paths)))
            cap = caption_html(caption(state['data'], s))
            display(HTML(cap))
        with caption_output:
            clear_output()
            display(HTML(cap + "<div style='background-color:#fff3cd; padding:10px; border-left:4px solid #ffc107; "
                         "margin-top:10px; font-size:13px;'><b>💡 Tip:</b> Select the text above, copy, and paste it "
                         "into your manuscript's figure legend.</div>"))

    def get_recipe():
        if state['data'] is None:
            return None
        return build_plot_recipe(PLOT_TYPE, state['data'], widget_values(C), state['source'])

    def on_load(rec, style_only, name):
        if not style_only:
            state.update(data=rec['data'], source=rec.get('source'), recipe_name=name)
            set_data_defaults(state['data'])
            refresh_source()
        skipped = apply_widget_values(C, rec.get('settings'))
        generate()
        return skipped

    # --- 5. INITIAL STATE AND LAYOUT ---
    try:
        state['data'], state['source'] = build_data()
    except Exception as e:
        state['data'], state['source'] = None, None
        print(f"⚠️ Could not read the Step 12 results: {e}")
    state['token'] = results_token(RESULTS_KEY)
    if state['data'] is not None:
        set_data_defaults(state['data'])
    else:
        label_box.children = [widgets.Label("No categorical labels found to edit.")]
    refresh_source()

    preset = preset_dropdown({'width': C['width'], 'height': C['height'], 'title': C['title_font'],
                              'label': C['label_font'], 'tick': C['tick_font']})
    H = lambda t: widgets.HTML(f"<h4 style='color: #2E86AB;'>{t}</h4>")
    hr = lambda: widgets.HTML("<hr style='margin: 10px 0;'>")
    tabs = widgets.Tab(children=[
        widgets.VBox([H('Labels & Size'), C['x_axis'], C['show_title'], C['title'], C['xlabel'], C['ylabel'], hr(),
                      preset, C['width'], C['height'], hr(), C['title_font'], C['label_font'], C['tick_font']]),
        widgets.VBox([H('Data Points'), C['color_by'], C['point_color'], hr(),
                      widgets.HTML("<b>Bubble Size (by model weight):</b>"),
                      C['bubble_base'], C['bubble_range'], C['bubble_alpha']]),
        widgets.VBox([H('Regression Line'), C['line_color'], C['line_width'], C['show_ci'], C['ci_alpha'], hr(),
                      C['show_equation'], C['show_r2']]),
        widgets.VBox([H('Layout & Legend'), C['grid'], C['show_null'], C['legend_loc'], C['legend_font'], hr(),
                      H('Export'), C['save_pdf'], C['save_png'], C['save_svg'], C['dpi'], C['filename'],
                      C['transparent']]),
        widgets.VBox([H('Edit Plot Labels'),
                      widgets.HTML("<p style='color: #666;'><i>Rename raw data values (e.g., 'W') to "
                                   "publication-ready labels (e.g., 'Wheat').</i></p>"), label_box]),
        widgets.VBox([H('📝 Figure Caption'),
                      widgets.HTML("<p style='color: #666; font-size: 13px;'><i>Generated from the plot settings "
                                   "when you click Generate.</i></p>"), caption_output]),
    ])
    for i, t in enumerate(['🎨 Style', '⚫ Points', '📈 Regression', '💾 Layout/Export', '✏️ Labels', '📝 Caption']):
        tabs.set_title(i, t)

    header = widgets.HTML("""
        <div style='background: linear-gradient(135deg, #2c3e50 0%, #3498db 100%);
                    padding: 20px; border-radius: 10px; margin-bottom: 20px;'>
            <h2 style='color: white; margin: 0; text-align: center;'>📈 Step 13: Publication-Ready Meta-Regression Plot</h2>
            <p style='color: rgba(255,255,255,0.9); margin: 5px 0 0 0; text-align: center; font-size: 14px;'>
                Bubble plot of the effect sizes with the fitted meta-regression line and its confidence band</p>
        </div>""")
    tips = widgets.HTML("""
        <div style='background-color: #e8f4f8; border-left: 4px solid #17a2b8; padding: 12px; margin-bottom: 15px;
                    border-radius: 4px; color: #0c5460; font-family: sans-serif;'>
            <b>💡 Plotting Tips:</b>
            <ul style='margin: 5px 0 0 0; padding-left: 20px; font-size: 13px;'>
                <li>With several moderators, choose the <b>X-Axis</b> moderator in the Style tab; the line holds the
                    other moderators at their mean / reference level.</li>
                <li>The confidence band uses the standard errors and degrees of freedom of the Step 12 model.</li>
                <li>Use the <b>Labels</b> tab to rename raw data categories for the plot legend.</li>
            </ul>
        </div>""")

    run_btn = widgets.Button(description='📊 Generate Regression Plot', button_style='success',
                             layout=widgets.Layout(width='450px', height='50px'), style={'font_weight': 'bold'})
    run_btn.on_click(generate)
    recipe_bar = plot_recipe_bar(PLOT_TYPE, get_recipe, on_load,
                                 lambda: (f"{C['filename'].value}_{C['x_axis'].value}"
                                          if state['data'] and state['data']['continuous'] else C['filename'].value))

    display(widgets.VBox([header, tips, source_box, widgets.HTML("<hr>"), tabs,
                          widgets.HTML("<hr style='margin: 15px 0;'>"), run_btn, recipe_bar, out]))


_meta_regression_plot_cell()


In [ ]:
#@title 📈📈 14. Non-Linear Spline Meta-Regression: Configuration & Execution
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# CELL: SPLINE ANALYSIS WITH DASHBOARD (MVC REFACTORED)
# Purpose: Non-linear meta-regression using natural cubic splines.
# Architecture: Model-View-Controller pattern for maintainability.
# Note: Drop-in replacement - preserves all original math and logic.
# =============================================================================

import numpy as np
import pandas as pd
from scipy.stats import t, chi2, norm
from scipy.optimize import minimize_scalar
import statsmodels.api as sm
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings

# Check for patsy
try:
    import patsy
    PATSY_AVAILABLE = True
except ImportError:
    PATSY_AVAILABLE = False


# =============================================================================
# PART 1: MODEL (Logic, Math, Data Access)
# =============================================================================

class SplineMetaRegressionModel:
    """
    Handles all data processing, statistical calculations, and business logic
    for spline meta-regression analysis. Relies strictly on the validated
    external robust functions.
    """

    def __init__(self):
        self.results = None
        self.linear_results = None
        self.error = None
        self.data = None
        self.config = None

    # -------------------------------------------------------------------------
    # Data Access Methods
    # -------------------------------------------------------------------------

    def get_analysis_data(self):
        """Retrieve analysis data from global context."""
        config = self.get_analysis_config()
        if config and 'analysis_data' in config:
            return config['analysis_data']
        elif 'analysis_data' in globals():
            return globals()['analysis_data']
        elif 'data_filtered' in globals():
            return globals()['data_filtered']
        return None

    def get_analysis_config(self):
        """Retrieve analysis configuration from global context."""
        if 'ANALYSIS_CONFIG' in globals():
            return globals()['ANALYSIS_CONFIG']
        return {}

    def set_analysis_config(self, key, value):
        """Set a value in the global ANALYSIS_CONFIG."""
        global ANALYSIS_CONFIG
        if 'ANALYSIS_CONFIG' not in globals():
            ANALYSIS_CONFIG = {}
        ANALYSIS_CONFIG[key] = value

    def get_numeric_moderators(self, df):
        """Get list of valid numeric moderator columns."""
        if df is None:
            return []

        valid_mods = []
        technical_cols = [
            'id', 'xe', 'xc', 'ne', 'nc', 'sde', 'sdc', 'w_fixed', 'w_random',
            'df', 'sp', 'sp_squared', 'hedges_j', 'weights', 'wi'
        ]

        config = self.get_analysis_config()
        if config:
            technical_cols.extend([
                config.get('effect_col'),
                config.get('var_col'),
                config.get('se_col')
            ])

        for col in df.columns:
            if col in technical_cols or col is None:
                continue
            if pd.api.types.is_numeric_dtype(df[col]):
                valid_mods.append(col)
            elif is_text_dtype(df[col]):
                try:
                    if pd.to_numeric(df[col], errors='coerce').notna().sum() >= 3:
                        valid_mods.append(col)
                except:
                    pass

        return sorted(list(set(valid_mods)))

    # -------------------------------------------------------------------------
    # Core Orchestration
    # -------------------------------------------------------------------------

    def run_analysis(self, moderator_col, df_spline):
        """
        Main analysis orchestrator. Runs the complete spline analysis using
        external robust functions.
        """
        self.error = None
        self.results = None

        if not PATSY_AVAILABLE:
            self.error = "❌ Error: 'patsy' package not installed. Install with: pip install patsy"
            return False

        df_working = self.get_analysis_data()
        if df_working is None:
            self.error = "❌ Error: Data not found. Run Steps 2–6 (data and effect sizes) first."
            return False

        config = self.get_analysis_config()
        if not config:
            self.error = "❌ Error: Config not found. Run Steps 2–6 (data and effect sizes) first."
            return False

        eff_col = str(config.get('effect_col', 'hedges_g'))
        var_col = str(config.get('var_col', 'Vg'))

        if eff_col not in df_working.columns or var_col not in df_working.columns:
            possible_effs = ['lnRR', 'hedges_g', 'lnOR', 'yi', 'effect_size']
            possible_vars = ['var_lnRR', 'Vg', 'var_lnOR', 'vi', 'variance']

            found_eff = next((c for c in possible_effs if c in df_working.columns), None)
            found_var = next((c for c in possible_vars if c in df_working.columns), None)

            if found_eff and found_var:
                eff_col, var_col = found_eff, found_var
                self.set_analysis_config('effect_col', eff_col)
                self.set_analysis_config('var_col', var_col)
            else:
                cols_preview = ", ".join(list(df_working.columns)[:10])
                self.error = f"❌ Error: Required effect ({eff_col}) and variance ({var_col}) columns not found. Available columns: {cols_preview}..."
                return False

        # Prepare data safely; every dropped row is recorded with its reason
        df = df_working.copy()
        df[moderator_col] = pd.to_numeric(df[moderator_col], errors='coerce')
        self.accounting = None
        if 'select_rows' in globals():
            try:
                df, self.accounting = select_rows(df, eff_col, var_col, 'id',
                                                  complete_case_cols=[moderator_col])
            except ValueError as e:
                self.error = f"❌ Error: {e}"
                return False
        else:
            df = df.dropna(subset=[moderator_col, eff_col, var_col])
            df = df[df[var_col] > 0]

        if len(df) < 3:
            self.error = f"❌ Error: Insufficient data (n={len(df)}). Need at least 3 observations."
            return False

        self.data = df

        # 1. Run Robust Spline Analysis
        if '_run_robust_spline_analysis' in globals():
            import warnings
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                est, err = globals()['_run_robust_spline_analysis'](
                    df, moderator_col, eff_col, var_col, df_spline=df_spline
                )
        else:
            self.error = "❌ Error: '_run_robust_spline_analysis' function not found. Please run the cell defining it."
            return False

        if err:
            self.error = f"❌ {err}"
            return False

        self.results = est
        self.results['moderator_col'] = moderator_col
        self.results['df_spline'] = df_spline

        # 2. Linear vs spline: the test of NON-linearity. REML log-likelihoods
        #    are not comparable across different fixed effects, so both models
        #    are refitted by ML on the same rows, the same sampling V and the
        #    same random structure (Cell 1b engine); AIC counts every estimated
        #    parameter (fixed effects + variance components).
        self.results['model_comparison'] = self._compare_linear_spline(
            df, moderator_col, eff_col, var_col, df_spline)
        self.results['data_accounting'] = (self.accounting.to_dict()
                                           if self.accounting is not None else None)

        # Save to global config
        self.set_analysis_config('spline_model_results', self.results)

        return True

    def _compare_linear_spline(self, df, moderator_col, eff_col, var_col, df_spline):
        """Linear vs natural-cubic-spline model, both fitted by ML with the
        unified engine on exactly the same rows, sampling V and random
        structure. Returns a JSON-safe dict: logLik, parameters, AIC/BIC,
        ΔAIC, and the LRT for non-linearity (df = df_spline − 1)."""
        if 'build_fit_spec' not in globals():
            return {'error': 'unified model engine not loaded (run Cell 1b)'}
        try:
            alpha = float((self.get_analysis_config().get('global_settings') or {}).get('alpha', 0.05))
            x = df[moderator_col].to_numpy(dtype=float)
            X_lin = pd.DataFrame({'intrcpt': 1.0, moderator_col: x - x.mean()}, index=df.index)
            B = np.asarray(patsy.dmatrix(f"cr(x, df={df_spline}, constraints='center')",
                                         {'x': x}, return_type='matrix'))
            X_spl = pd.DataFrame(B, index=df.index,
                                 columns=['intrcpt'] + [f'spline{j}' for j in range(1, B.shape[1])])
            cfg = self.get_analysis_config()
            kw = dict(vcv_dict=cfg.get('vcv_matrices', {}),
                      vcv_reference_df=cfg.get('analysis_data'))
            spec_lin = build_fit_spec(df, eff_col, var_col, X=X_lin, label='linear', **kw)
            spec_spl = build_fit_spec(df, eff_col, var_col, X=X_spl, label='spline', **kw)
            three_level = any(len(y) > 1 for y in spec_lin.y_blocks)
            var_kw = {} if three_level else {'sigma2': 0.0}
            lin = fit_mv(spec_lin, method='ML', **var_kw)
            spl = fit_mv(spec_spl, method='ML', **var_kw)
            c = compare_fits(spl, lin)
        except Exception as e:
            return {'error': f"{type(e).__name__}: {e}"}

        def _fit(f):
            return {'loglik': float(f.loglik), 'n_params': int(f.n_params),
                    'n_coef': int(f.spec.n_coef), 'aic': float(f.aic), 'bic': float(f.bic),
                    'tau2': float(f.tau2), 'sigma2': float(f.sigma2)}
        out = {
            'method': 'ML', 'alpha': alpha,
            'structure': lin.structure, 'three_level': bool(three_level),
            'k_effects': int(spec_lin.n_effects), 'k_clusters': int(spec_lin.n_clusters),
            'v_type': 'full shared-control V' if spec_lin.uses_full_vcv else 'diagonal V',
            'linear': _fit(lin), 'spline': _fit(spl),
            # ΔAIC = AIC(linear) − AIC(spline): > 0 favours the spline
            'delta_aic': float(lin.aic - spl.aic),
            'nested': bool(c.nested),
            'lrt': None if c.lrt is None else float(c.lrt),
            'df': None if c.df is None else int(c.df),
            'p_value': None if c.p_value is None else float(c.p_value),
            'warnings': list(c.warnings),
        }
        if c.p_value is not None:
            out['nonlinear'] = bool(c.p_value < alpha)
            out['preferred'] = 'Spline' if out['nonlinear'] else 'Linear'
            out['basis'] = (f"LRT for non-linearity χ²({c.df}) = {c.lrt:.2f}, p = {c.p_value:.4f} "
                            f"(ML, α = {alpha:g}); ΔAIC (linear − spline) = {out['delta_aic']:+.2f}")
        else:
            out['nonlinear'] = None
            out['preferred'] = 'Spline' if out['delta_aic'] > 0 else 'Linear'
            out['basis'] = (f"models not nested; lower ML AIC preferred "
                            f"(ΔAIC linear − spline = {out['delta_aic']:+.2f})")
        return out

    # -------------------------------------------------------------------------
    # Computed Properties
    # -------------------------------------------------------------------------

    def get_summary_stats(self):
        """Return key summary statistics for display, including knots and boundaries."""
        if not self.results:
            return None

        r = self.results

        # --- NEW: Calculate Knots and Boundary Sparsity ---
        df = r.get('reg_df')
        mod_col = r.get('moderator_col', 'Unknown')
        df_spline = r.get('df_spline', 3)

        knots = []
        sparse_boundaries = False

        if isinstance(df, pd.DataFrame) and mod_col in df.columns:
            _rows = df[df[mod_col].notna()]
            mod_vals = _rows[mod_col].values
            # patsy's centred cr(df=k) uses k + 1 knots: the two boundary
            # knots at min/max and k - 1 inner knots at evenly spaced
            # quantiles of the UNIQUE moderator values (patsy's
            # _get_all_sorted_knots). With repeated values (several effect
            # sizes per study) these differ from quantiles of all rows.
            ux = np.unique(mod_vals)
            inner = np.percentile(ux, np.linspace(0, 100, df_spline + 1)[1:-1])
            knots = np.concatenate([[ux.min()], inner, [ux.max()]])

            # Boundary Check: fewer than 3 studies in the outer 10% tails?
            p10, p90 = np.percentile(mod_vals, [10, 90])
            if 'id' in _rows.columns:
                n_low = _rows.loc[mod_vals <= p10, 'id'].nunique()
                n_high = _rows.loc[mod_vals >= p90, 'id'].nunique()
            else:
                n_low, n_high = int(sum(mod_vals <= p10)), int(sum(mod_vals >= p90))
            sparse_boundaries = (n_low < 3) or (n_high < 3)

        return {
          'f_stat': r.get('omnibus_F', 0),
          'df_num': r.get('omnibus_df_num', 0),
          'df_denom': r.get('omnibus_df_denom', 0),
          'p_omnibus': r.get('omnibus_p', 1.0),
          'p_method': r.get('omnibus_p_method', 'Unknown'),
          'tau_sq': r.get('tau_sq', 0),
          'sigma_sq': r.get('sigma_sq', 0) or 0.0,
          'variance_estimation': r.get('variance_estimation') or 'Joint REML',
          'model_comparison': r.get('model_comparison'),
          'data_accounting': r.get('data_accounting'),
          'n_obs': r.get('n_obs_used', df.shape[0] if isinstance(df, pd.DataFrame) else 0),
          'n_studies': r.get('n_clusters_used',
                             df['id'].nunique() if isinstance(df, pd.DataFrame) and 'id' in df.columns else 0),
          'n_coefs': len(r.get('betas', [])),
          'moderator_col': mod_col,
          'df_spline': df_spline,
          'model_type': r.get('model_type', 'Spline'),
          'resid': r.get('resid', np.array([])),
          'betas': r.get('betas', np.array([])),
          'se_betas': r.get('se_betas', np.array([])),
          'knots': knots,
          'sparse_boundaries': sparse_boundaries
      }


# =============================================================================
# PART 2: VIEW (Widgets, HTML Generation, Layout)
# =============================================================================

class SplineHTMLTemplates:
    """
    Static HTML template generators for Spline Meta-Regression.
    Matches the professional styling of the Linear Meta-Regression cell.
    """

    @staticmethod
    def dashboard_header(title: str, badge_text: str) -> str:
        return f"""
        <div style='display: flex; align-items: center; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 2px solid #ecf0f1; font-family: system-ui, -apple-system, sans-serif;'>
            <div style='background-color: #2c3e50; color: white; padding: 8px 16px; border-radius: 6px; font-weight: 600; font-size: 1.1em; letter-spacing: 0.5px; margin-right: 15px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
                {title}
            </div>
            <div style='color: #7f8c8d; font-size: 1.1em; display: flex; align-items: center;'>
                <span style='margin-right: 10px;'>Moderator:</span>
                <span style='background-color: #e0f7fa; color: #0097a7; padding: 4px 12px; border-radius: 20px; font-family: monospace; font-weight: bold; border: 1px solid #b2ebf2;'>
                    {badge_text}
                </span>
            </div>
        </div>
        """

    @staticmethod
    def gradient_card(title: str, value: str, subtitle: str, bg_gradient: str) -> str:
        return f"""
        <div style='background: {bg_gradient}; padding: 15px 25px; border-radius: 8px;
                    color: white; margin-bottom: 20px; display: flex; justify-content: space-between; align-items: center; box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>
            <div>
                <div style='font-size: 0.85em; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85;'>{title}</div>
                <div style='margin: 0; font-size: 2.2em; font-weight: bold;'>{value}</div>
            </div>
            <div>
                <span style='background: rgba(255,255,255,0.2); padding: 5px 15px; border-radius: 20px; font-size: 1.2em; font-weight: bold;'>
                    {subtitle}
                </span>
            </div>
        </div>
        """

    @staticmethod
    def info_grid(items: list) -> str:
        html = "<div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 20px; font-family: system-ui, -apple-system, sans-serif;'>"
        for item in items:
            color = item.get('color', '#007bff')
            html += f"""
            <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px;
                        border-left: 4px solid {color}; box-shadow: 0 1px 3px rgba(0,0,0,0.05);'>
                <div style='color: #6c757d; font-size: 0.9em; text-transform: uppercase; letter-spacing: 0.5px;'>{item['label']}</div>
                <div style='font-size: 1.5em; font-weight: bold; color: #2c3e50;'>{item['value']}</div>
            </div>
            """
        html += "</div>"
        return html

    @staticmethod
    def interpretation_box(content: str, is_significant: bool = False) -> str:
        bg_color = "#e0f7fa" if is_significant else "#f8f9fa"
        border_color = "#00acc1" if is_significant else "#dee2e6"
        return f"""
        <div style='background-color: {bg_color}; padding: 20px; border-radius: 6px;
                    margin-bottom: 20px; border-left: 4px solid {border_color}; font-family: system-ui, -apple-system, sans-serif;'>
            <h4 style='margin-top: 0; color: #2c3e50;'>💡 Interpretation</h4>
            <p style='margin: 0; font-size: 1.05em; color: #34495e; line-height: 1.5;'>{content}</p>
        </div>
        """

    @staticmethod
    def model_summary_box(s: dict) -> str:
        plug = 'Plug-in' in str(s.get('variance_estimation', ''))
        var_note = ("held fixed at the linear model's estimates (plug-in fallback)" if plug
                    else "estimated jointly by REML under the spline model")
        sigma_line = (f"<p style='margin: 5px 0;'><b>Within-Study Variance (σ²):</b> {s['sigma_sq']:.4f}</p>"
                      if s.get('sigma_sq', 0) > 0 else "")
        return f"""
        <h4 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px; font-family: system-ui;'>Model Summary</h4>
        <div style='background-color: #f8f9fa; padding: 15px; border-radius: 6px; border: 1px solid #dee2e6; font-family: system-ui, -apple-system, sans-serif;'>
            <p style='margin: 5px 0;'><b>Model Type:</b> {s['model_type']}</p>
            <p style='margin: 5px 0;'><b>Studies (k):</b> {s['n_studies']} &nbsp;·&nbsp; <b>Effect sizes (n):</b> {s.get('n_obs', '—')}</p>
            <p style='margin: 5px 0;'><b>Spline Degrees of Freedom:</b> {s['df_spline']}</p>
            <p style='margin: 5px 0;'><b>Number of Coefficients:</b> {s['n_coefs']} (1 intercept + {s['n_coefs']-1} spline terms)</p>
            <p style='margin: 5px 0;'><b>Between-Study Variance (τ²):</b> {s['tau_sq']:.4f} <span style='color: #6c757d; font-size: 0.9em;'>({var_note})</span></p>
            {sigma_line}
        </div>
        """

    @staticmethod
    def comparison_box(mc) -> str:
        """Linear vs spline (ML, same rows / V / random structure)."""
        h = ("<h4 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px; "
             "font-family: system-ui; margin-top: 20px;'>Test of Non-Linearity (Linear vs Spline)</h4>")
        if not mc:
            return ""
        if mc.get('error'):
            return h + (f"<p style='color:#856404; font-family: system-ui;'>⚠️ Comparison not "
                        f"available: {mc['error']}</p>")
        td = "padding: 8px 10px; border-bottom: 1px solid #dee2e6; text-align: center;"
        fp = lambda p: "< 0.001" if p < 0.001 else f"{p:.4f}"
        rows = ""
        for key, name in (('linear', 'Linear'), ('spline', 'Spline')):
            f = mc[key]
            shade = "background:#eef7ee;" if mc.get('preferred') == name else ""
            rows += (f"<tr style='{shade}'><td style='{td} text-align:left;'><b>{name}</b></td>"
                     f"<td style='{td}'>{f['loglik']:.3f}</td><td style='{td}'>{f['n_coef']} + "
                     f"{f['n_params'] - f['n_coef']}</td><td style='{td}'>{f['aic']:.2f}</td>"
                     f"<td style='{td}'>{f['bic']:.2f}</td></tr>")
        test = (f"LRT χ²({mc['df']}) = <b>{mc['lrt']:.2f}</b>, p = <b>{fp(mc['p_value'])}</b>"
                if mc.get('p_value') is not None else "models not nested (no LRT)")
        verdict = ("<b style='color:#00838f;'>non-linearity supported</b>" if mc.get('nonlinear')
                   else "<b>no evidence of non-linearity</b>"
                   if mc.get('nonlinear') is False else "")
        warn = "".join(f"<p style='margin:4px 0; color:#856404;'>⚠️ {w}</p>" for w in mc.get('warnings', []))
        return h + f"""
        <div style='font-family: system-ui, -apple-system, sans-serif;'>
        <table style='width:100%; border-collapse:collapse; background:white;'>
            <thead style='background:#34495e; color:white;'><tr>
                <th style='padding:8px 10px; text-align:left;'>Model</th><th style='padding:8px 10px;'>logLik (ML)</th>
                <th style='padding:8px 10px;'>Params (fixed + var.)</th><th style='padding:8px 10px;'>AIC</th>
                <th style='padding:8px 10px;'>BIC</th></tr></thead>
            <tbody>{rows}</tbody>
        </table>
        <p style='margin:8px 0 0 0;'>{test}; ΔAIC (linear − spline) = {mc['delta_aic']:+.2f} → {verdict}.</p>
        <p style='margin:4px 0 0 0; font-size:0.85em; color:#6c757d;'>Both models fitted by maximum
            likelihood (REML likelihoods are not comparable across different fixed effects) to the same
            {mc['k_effects']} effect sizes in {mc['k_clusters']} studies, with the same {mc['v_type']} and
            random structure ({mc['structure']}). AIC counts fixed effects and variance components.
            Unlike the omnibus test above, this tests the curvature itself.</p>
        {warn}
        </div>"""


class SplineMetaRegressionView:
    """
    Handles all UI components, widget creation, and HTML generation.
    Matches the professional style of the Linear Meta-Regression view.
    """

    def __init__(self):
        self.templates = SplineHTMLTemplates()
        self._create_widgets()
        self._create_tabs()
        self._build_layout()

    # -------------------------------------------------------------------------
    # Widget Creation
    # -------------------------------------------------------------------------

    def _create_widgets(self):
        self.mod_dropdown = widgets.Dropdown(
            options=['Data not loaded'],
            description='Moderator:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='400px')
        )

        self.df_slider = widgets.IntSlider(
            value=3,
            min=3,
            max=6,
            description='Spline df:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='400px')
        )

        self.run_button = widgets.Button(
            description='▶ Run Spline Analysis',
            button_style='success',
            icon='play'
        )

        self.export_button = widgets.Button(
            description="📥 Download Spline Report",
            button_style='info',
            icon='file-excel',
            layout=widgets.Layout(width='300px', height='40px')
        )

    def _create_tabs(self):
        self.stale_banner = widgets.HTML(value="")
        # Assuming register_banner exists in global scope from Cell 1
        if 'register_banner' in globals():
            register_banner('spline', self.stale_banner)

        self.tab_results = widgets.Output()
        self.tab_diagnostics = widgets.Output()
        self.tab_details = widgets.Output()
        self.tab_publication = widgets.Output()
        self.tab_export = widgets.Output()

        self.tabs = widgets.Tab(children=[
            self.tab_results,
            self.tab_diagnostics,
            self.tab_details,
            self.tab_publication,
            self.tab_export
        ])

        self.tabs.set_title(0, '📊 Results')
        self.tabs.set_title(1, '🔍 Diagnostics')
        self.tabs.set_title(2, '⚙️ Model Details')
        self.tabs.set_title(3, '📝 Publication Text')
        self.tabs.set_title(4, '💾 Export')

    def _build_layout(self):
        self.controls = widgets.VBox([
            self.mod_dropdown,
            self.df_slider,
            self.run_button
        ])

        self.header = widgets.HTML("""
            <div style='font-family: system-ui, -apple-system, sans-serif;'>
                <h3 style='margin-bottom: 5px;'>🌊 Spline Meta-Regression Analysis</h3>
                <p style='color: #6c757d; margin-top: 0;'>Test for non-linear relationships using natural cubic splines. Results appear in organized tabs below.</p>
            </div>
        """)

    # -------------------------------------------------------------------------
    # Update Methods
    # -------------------------------------------------------------------------

    def update_moderator_options(self, options):
        self.mod_dropdown.options = options if options else ['Data not loaded']

    def clear_all_tabs(self):
        for tab in [self.tab_results, self.tab_diagnostics,
                    self.tab_details, self.tab_publication]:
            tab.clear_output()

    def show_loading(self):
        with self.tab_results:
            display(HTML("<div style='padding: 10px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; border-radius: 4px; font-family: system-ui;'>⏳ <b>Running Robust Spline Analysis...</b> Please wait.</div>"))

    def show_error(self, error_msg):
        with self.tab_results:
            clear_output()
            display(HTML(f"<div style='color: #721c24; background-color: #f8d7da; padding: 15px; border-radius: 6px; border-left: 4px solid #dc3545; font-family: system-ui;'><b>Error:</b> {error_msg}</div>"))

    # -------------------------------------------------------------------------
    # HTML Generation Methods
    # -------------------------------------------------------------------------

    def render_results(self, stats):
        with self.tab_results:
            clear_output()
            display(HTML("<div style='padding: 20px;'>"))

            # 1. Dashboard Header
            display(HTML(self.templates.dashboard_header("Spline Analysis", stats['moderator_col'])))

            # 2. Gradient Card (Omnibus Test)
            is_sig = stats['p_omnibus'] < _alpha()
            sig_marker = "***" if stats['p_omnibus'] < 0.001 else "**" if stats['p_omnibus'] < 0.01 else "*" if is_sig else "ns"
            gradient = "linear-gradient(135deg, #0984e3 0%, #00cec9 100%)" if is_sig else "linear-gradient(135deg, #636e72 0%, #b2bec3 100%)"

            p_text = "< 0.001" if stats['p_omnibus'] < 0.001 else f"= {stats['p_omnibus']:.4g}"

            card_html = self.templates.gradient_card(
                title="Omnibus Wald Test for Spline Association",
                value=f"p {p_text}",
                subtitle="SIGNIFICANT" if is_sig else "NOT SIGNIFICANT",
                bg_gradient=gradient
            )
            display(HTML(card_html))

            # 3. Info Grid (Updated to F-Statistic)
            color = "#00cec9" if is_sig else "#6c757d"
            grid_items = [
                {'label': 'Omnibus F-Statistic', 'value': f"F({stats['df_num']}, {stats['df_denom']:.1f}) = {stats['f_stat']:.2f}", 'color': '#0984e3'},
                {'label': 'P-Value', 'value': f"{stats['p_omnibus']:.4g} {sig_marker}", 'color': color}
            ]
            display(HTML(self.templates.info_grid(grid_items)))

            # 4. Interpretation Box
            interp_text = (
                f"The omnibus test indicates a <b style='color: #00838f;'>significant overall association</b> "
                f"between <b>{stats['moderator_col']}</b> and the effect sizes when modeled flexibly via splines. "
                f"This test includes the linear trend; whether the relationship is actually "
                f"<i>non-linear</i> is assessed by the linear-vs-spline comparison below. "
                f"Examine the Spline Plot tab to identify the shape of this relationship (e.g., thresholds or plateaus)."
            ) if is_sig else (
                f"The omnibus test is not significant, indicating that <b>{stats['moderator_col']}</b> does not "
                f"reliably predict effect sizes even when allowing for flexible, non-linear spline patterns."
            )

            display(HTML(self.templates.interpretation_box(interp_text, is_sig)))

            # 5. Tables
            display(HTML(self.templates.model_summary_box(stats)))
            display(HTML(self.templates.comparison_box(stats.get('model_comparison'))))
            if stats.get('data_accounting') and 'accounting_html' in globals():
                display(HTML(accounting_html(stats['data_accounting'])))

            # 6. Next Step Tip
            tip_html = """
            <div style='background-color: #fff3cd; padding: 15px; border-radius: 6px; border-left: 4px solid #ffc107; margin-top: 20px; font-family: system-ui, -apple-system, sans-serif;'>
                <p style='margin: 0; color: #856404;'><b>📊 Next Step:</b> Proceed to the dedicated <b>Spline Plot</b> cell to visualize the non-linear curve and confidence bands.</p>
            </div>
            </div>
            """
            display(HTML(tip_html))


    def render_diagnostics(self, stats):
        resid = stats['resid']
        html = "<div style='padding: 20px; font-family: system-ui, -apple-system, sans-serif;'>"
        html += "<h4 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;'>Residual Analysis</h4>"

        if resid is None or len(resid) == 0:
            html += "<div style='background-color: #f8f9fa; padding: 15px; border-radius: 6px; border: 1px solid #dee2e6;'><p style='margin: 0; color: #6c757d;'><i>Residual data not available.</i></p></div>"
        else:
            html += f"""
            <div style='background-color: #f8f9fa; padding: 15px; border-radius: 6px; border: 1px solid #dee2e6; margin-bottom: 20px;'>
                <p style='margin: 5px 0;'><b>Mean Residual:</b> {np.mean(resid):.4f} <span style='color:#6c757d'>(should be ≈ 0)</span></p>
                <p style='margin: 5px 0;'><b>SD of Residuals:</b> {np.std(resid):.4f}</p>
                <p style='margin: 5px 0;'><b>Range:</b> [{np.min(resid):.3f}, {np.max(resid):.3f}]</p>
            </div>
            """

        # --- NEW: Boundary Warning ---
        html += "<h4 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;'>Boundary Diagnostics</h4>"
        if stats.get('sparse_boundaries', False):
            html += """
            <div style='background-color: #fff3cd; padding: 15px; border-radius: 6px; border-left: 4px solid #ffc107;'>
                <p style='margin: 0; color: #856404;'><b>⚠️ Sparse Data at Boundaries:</b> There are very few studies (< 3) at the extreme lower or upper 10% of your moderator range. <i>Spline curves can become highly unstable and "whip" upwards or downwards at the tails when data is sparse. Interpret the extreme ends of the plotted curve with caution.</i></p>
            </div>
            """
        else:
            html += """
            <div style='background-color: #d4edda; padding: 15px; border-radius: 6px; border-left: 4px solid #28a745;'>
                <p style='margin: 0; color: #155724;'><b>✅ Boundary Data Sufficient:</b> You have adequate data representation at the tails of your moderator range, stabilizing the outer spline constraints.</p>
            </div>
            """

        html += "</div>"
        with self.tab_diagnostics:
            clear_output()
            display(HTML(html))

    def render_details(self, stats):
        # --- NEW: Format Knots ---
        knots_str = ", ".join([f"{k:.2f}" for k in stats.get('knots', [])]) if len(stats.get('knots', [])) > 0 else "N/A"

        # --- NEW: Format Coefficients Table ---
        betas = stats.get('betas', [])
        ses = stats.get('se_betas', [])
        coef_rows = ""
        for i in range(len(betas)):
            name = "Intercept" if i == 0 else f"Spline Term {i}"
            beta_val = betas[i]
            se_val = ses[i] if i < len(ses) else np.nan
            t_val = beta_val / se_val if se_val > 0 else np.nan
            coef_rows += f"<tr><td style='border: 1px solid #dee2e6; padding: 8px;'>{name}</td><td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>{beta_val:.4f}</td><td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>{se_val:.4f}</td><td style='border: 1px solid #dee2e6; padding: 8px; text-align: center;'>{t_val:.2f}</td></tr>"

        html = f"""
        <div style='padding: 20px; font-family: system-ui, -apple-system, sans-serif;'>
            <h4 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;'>Model Specifications</h4>
            <div style='background-color: #f8f9fa; padding: 15px; border-radius: 6px; border: 1px solid #dee2e6; margin-bottom: 20px;'>
                <p style='margin: 5px 0;'><b>Method:</b> Restricted (Natural) Cubic Splines</p>
                <p style='margin: 5px 0;'><b>Knot Locations:</b> {knots_str}</p>
                <p style='margin: 5px 0;'><b>Variance:</b> {stats.get('variance_estimation', 'Joint REML')} (τ² = {stats['tau_sq']:.4f}, σ² = {stats.get('sigma_sq', 0):.4f}).</p>
            </div>

            <h4 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;'>Raw Coefficients</h4>
            <table style='width: 100%; border-collapse: collapse; background-color: white;'>
                <thead style='background-color: #34495e; color: white;'>
                    <tr><th style='padding: 10px; border: 1px solid #bdc3c7;'>Parameter</th><th style='padding: 10px; border: 1px solid #bdc3c7; text-align: center;'>Estimate (β)</th><th style='padding: 10px; border: 1px solid #bdc3c7; text-align: center;'>Standard Error</th><th style='padding: 10px; border: 1px solid #bdc3c7; text-align: center;'>t-value</th></tr>
                </thead>
                <tbody>{coef_rows}</tbody>
            </table>
            <p style='margin: 15px 0 0 0; font-size: 0.9em; color: #6c757d;'><i>Note: Individual spline coefficients lack direct biological interpretation. The Omnibus Test (Results tab) is the correct method for assessing overall non-linearity.</i></p>
        </div>
        """
        with self.tab_details:
            clear_output()
            display(HTML(html))

    def render_publication(self, stats, config):
        with self.tab_publication:
            clear_output()
            display(HTML("<h3 style='color: #2c3e50; font-family: system-ui;'>📝 Publication-Ready Results Text</h3>"))
            display(HTML("<p style='color: #6c757d; font-family: system-ui;'>Copy and paste this formatted text into your manuscript:</p>"))

            methods_html = self._generate_methods_text(stats, config)
            results_html = self._generate_publication_results_text(stats)

            display(HTML(methods_html))
            display(HTML(results_html))

            return methods_html + "<br><hr><br>" + results_html

    def _generate_methods_text(self, stats, config):
        db = {
            'splines': "Durrleman, S., & Simon, R. (1989). Flexible regression models with cubic splines. <i>Statistics in Medicine</i>, 8(5), 551-561.",
            'harrell': "Harrell, F. E. (2001). <i>Regression Modeling Strategies</i>. New York: Springer.",
            'aic': "Burnham, K. P., & Anderson, D. R. (2002). <i>Model Selection and Multimodel Inference</i> (2nd ed.). New York: Springer-Verlag.",
            'plugin': "Jackson, D., White, I. R., & Thompson, S. G. (2010). Extending DerSimonian and Laird's methodology. <i>Statistics in Medicine</i>, 29(12), 1282-1297.",
            'tool': COMETA_CITATION
        }

        knots_str = ", ".join([f"{k:.2f}" for k in stats.get('knots', [])])

        refs = [db['splines'], db['harrell']]
        if 'Plug-in' in str(stats.get('variance_estimation', '')):
            refs.append(db['plugin'])
            var_text = (f"Joint REML estimation of the variance components did not converge, so the "
                        f"between-study variance (τ² = <b>{stats['tau_sq']:.4f}</b>) and within-study "
                        f"variance (σ² = <b>{stats.get('sigma_sq', 0):.4f}</b>) were taken from the linear "
                        f"model and held fixed (plug-in) [{len(refs)}].")
        else:
            var_text = (f"The between-study variance (τ² = <b>{stats['tau_sq']:.4f}</b>) and within-study "
                        f"variance (σ² = <b>{stats.get('sigma_sq', 0):.4f}</b>) were estimated jointly by REML "
                        f"under the spline model.")
        refs.append(db['aic']); n_aic = len(refs)
        refs.append(db['tool']); n_tool = len(refs)
        refs_html = "".join(f"<li>{r}</li>" for r in refs)

        return f"""<div style='font-family: "Times New Roman", Times, serif; font-size: 12pt;
                    line-height: 1.8; padding: 20px; background-color: #ffffff; border: 1px solid #eee;'>
        <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; margin-top: 0;'>
            Materials and Methods</h3>
        <p style='text-align: justify;'>
        <b>Spline Meta-Regression.</b> To detect potential non-linear relationships between the
        effect size and <b>{stats['moderator_col']}</b>, we fitted a Restricted Cubic Spline model [1].
        Restricted (natural) cubic splines are linear beyond boundary knots [2].
        The model was specified with {stats['df_spline']} degrees of freedom, with knots placed at moderator values of <b>{knots_str}</b>.
        </p>
        <p style='text-align: justify;'>
        <b>Variance Estimation.</b> {var_text}
        </p>
        <p style='text-align: justify;'>
        <b>Hypothesis Testing.</b> The overall predictive value of the spline terms was assessed using an omnibus Wald-type F-test, which includes the linear trend. Non-linearity was tested by comparing the spline model with the linear model, both refitted by maximum likelihood (REML likelihoods are not comparable across different fixed effects) to the same effect sizes, with the same sampling variance–covariance matrix and random-effects structure, using a likelihood-ratio test and the Akaike Information Criterion [{n_aic}]. Analyses were performed using the Co-Meta toolkit [{n_tool}].
        </p>
        <h4 style='color: #34495e; margin-top: 20px; font-size: 11pt;'>References</h4>
        <ol style='font-size: 10pt; color: #555;'>
            {refs_html}
        </ol>
        </div>"""


    def _generate_publication_results_text(self, s):
        sig_text = "significant" if s['p_omnibus'] < _alpha() else "non-significant"
        p_format = f"< 0.001" if s['p_omnibus'] < 0.001 else f"= {s['p_omnibus']:.3f}"

        interpretation = ""
        if s['p_omnibus'] < _alpha():
            interpretation = f"""This indicates a <b>significant overall association</b> between {s['moderator_col']}
            and effect sizes when modeled flexibly."""
        else:
            interpretation = f"""This provides no evidence of an association between {s['moderator_col']}
            and effect sizes, even when allowing for flexible, non-linear spline patterns (absence of
            evidence is not evidence of no association)."""

        return f"""<div style='font-family: "Times New Roman", Times, serif; font-size: 12pt;
                    line-height: 1.8; padding: 20px; background-color: #ffffff; border: 1px solid #eee; border-top: none;'>
        <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; margin-top: 0;'>
            Spline Meta-Regression Results</h3>
        <h4 style='color: #34495e; margin-top: 25px;'>Omnibus Test for Association</h4>
        <p style='text-align: justify;'>
        The omnibus Wald test assessing the joint significance of all spline terms was <b>{sig_text}</b>
        (<i>F</i>({s['df_num']}, {s['df_denom']:.1f}) = <b>{s['f_stat']:.2f}</b>, <i>p</i> {p_format}). {interpretation}
        </p>
        {self._nonlinearity_paragraph(s)}

        <div style='background-color: #ecf0f1; padding: 20px; border-left: 4px solid #3498db; margin-top: 25px;'>
            <h4 style='margin-top: 0; color: #2c3e50;'>📊 Table 1. Spline Model Summary</h4>
            <table style='width: 100%; border-collapse: collapse; margin-top: 15px; background-color: white;'>
            <thead style='background-color: #34495e; color: white;'>
                <tr>
                    <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: left;'>Statistic</th>
                    <th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>Value</th>
                </tr>
            </thead>
            <tbody>
                <tr style='background-color: #f8f9fa;'>
                    <td style='border: 1px solid #bdc3c7; padding: 8px;'>Number of studies (k)</td>
                    <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{s['n_studies']}</td>
                </tr>
                <tr>
                    <td style='border: 1px solid #bdc3c7; padding: 8px;'>Spline degrees of freedom</td>
                    <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{s['df_spline']}</td>
                </tr>
                <tr style='background-color: #f8f9fa;'>
                    <td style='border: 1px solid #bdc3c7; padding: 8px;'>Between-study variance (τ²)</td>
                    <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{s['tau_sq']:.4f}</td>
                </tr>
                <tr>
                    <td style='border: 1px solid #bdc3c7; padding: 8px;'>Omnibus test for association</td>
                    <td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>
                        <i>F</i>({s['df_num']}, {s['df_denom']:.1f}) = {s['f_stat']:.2f}, <i>p</i> {p_format}</td>
                </tr>
                {self._nonlinearity_rows(s)}
            </tbody>
            </table>
        </div>
        </div>"""


    @staticmethod
    def _nonlinearity_paragraph(s):
        mc = s.get('model_comparison') or {}
        if not mc or mc.get('error') or mc.get('p_value') is None:
            return ""
        p = mc['p_value']
        pf = "< 0.001" if p < 0.001 else f"= {p:.3f}"
        verdict = ("supporting a <b>non-linear</b> relationship" if mc.get('nonlinear')
                   else "providing <b>no evidence of departure from linearity</b> (this does not "
                        "show that the relationship is linear)")
        return (f"<h4 style='color: #34495e; margin-top: 25px;'>Test of Non-Linearity</h4>"
                f"<p style='text-align: justify;'>The likelihood-ratio test comparing the spline model "
                f"with the linear model, both fitted by ML to the same {mc['k_effects']} effect sizes, gave "
                f"χ²({mc['df']}) = <b>{mc['lrt']:.2f}</b>, <i>p</i> {pf} ("
                f"AIC {mc['linear']['aic']:.1f} vs {mc['spline']['aic']:.1f}, "
                f"ΔAIC = {mc['delta_aic']:+.1f}), {verdict}.</p>")

    @staticmethod
    def _nonlinearity_rows(s):
        mc = s.get('model_comparison') or {}
        if not mc or mc.get('error') or mc.get('p_value') is None:
            return ""
        p = mc['p_value']
        pf = "< 0.001" if p < 0.001 else f"= {p:.3f}"
        td = "border: 1px solid #bdc3c7; padding: 8px;"
        return (f"<tr style='background-color: #f8f9fa;'><td style='{td}'>Non-linearity (LRT, spline vs linear, ML)</td>"
                f"<td style='{td} text-align: center;'>χ²({mc['df']}) = {mc['lrt']:.2f}, <i>p</i> {pf}</td></tr>"
                f"<tr><td style='{td}'>AIC linear / spline (ML)</td>"
                f"<td style='{td} text-align: center;'>{mc['linear']['aic']:.1f} / {mc['spline']['aic']:.1f}</td></tr>")

    def render_export_tab(self):
        with self.tab_export:
            clear_output()
            display(HTML("<div style='padding: 20px; font-family: system-ui, -apple-system, sans-serif;'>"))
            display(HTML("<h3 style='color: #2c3e50; margin-top: 0;'>💾 Download Audit Report</h3>"))
            display(HTML("<p style='color: #6c757d; margin-bottom: 20px;'>Download the Spline analysis results, including the non-linearity test and model comparison statistics.</p>"))
            display(self.export_button)
            display(HTML("</div>"))

    def display(self):
        display(widgets.VBox([self.stale_banner, self.header, self.controls, widgets.HTML("<br>"), self.tabs]))

# =============================================================================
# PART 3: CONTROLLER (Event Handling, Linking Model & View)
# =============================================================================

class SplineMetaRegressionController:
    """
    Handles user interactions and coordinates between Model and View.
    """

    def __init__(self, model, view):
        self.model = model
        self.view = view
        self._bind_events()
        self._initialize()

    def _bind_events(self):
        """Bind event handlers to view widgets."""
        self.view.run_button.on_click(self._on_run_click)
        self.view.export_button.on_click(self._on_export_click)

    def _initialize(self):
        """Initialize the view with data from model."""
        # Load moderator options
        df = self.model.get_analysis_data()
        if df is not None:
            options = self.model.get_numeric_moderators(df)
            self.view.update_moderator_options(options)

        # Setup export tab
        self.view.render_export_tab()

    def _on_run_click(self, button):
        """Handle run button click."""
        # Clear previous output
        self.view.clear_all_tabs()
        self.view.show_loading()

        # Get user inputs
        moderator_col = self.view.mod_dropdown.value
        df_spline = self.view.df_slider.value

        # Run analysis
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
        success = self.model.run_analysis(moderator_col, df_spline)

        if not success:
            self.view.show_error(self.model.error)
            return

        # Get results and render all tabs
        stats = self.model.get_summary_stats()
        config = self.model.get_analysis_config()

        # Render results
        self.view.render_results(stats)
        self.view.render_diagnostics(stats)
        self.view.render_details(stats)
        pub_text = self.view.render_publication(stats, config)

        # Save publication text to config
        self.model.set_analysis_config('spline_text', pub_text)

    def _on_export_click(self, button):
        """Handle export button click."""
        # Check if export function exists in global scope
        if 'export_analysis_report' in globals():
            globals()['export_analysis_report'](
                report_type='spline',
                filename_prefix='Spline_Analysis_Audit'
            )
        else:
            with self.view.tab_export:
                display(HTML("<p style='color: orange;'>⚠️ Export function not available. "
                           "Ensure export_analysis_report is defined in a previous cell.</p>"))

    def run(self):
        """Display the UI and start the application."""
        self.view.display()


# =============================================================================
# EXECUTION: Instantiate and Display
# =============================================================================

# Create MVC components
_spline_model = SplineMetaRegressionModel()
_spline_view = SplineMetaRegressionView()
_spline_controller = SplineMetaRegressionController(_spline_model, _spline_view)

# Run the application
_spline_controller.run()

In [ ]:
#@title 📈📈 15. Non-Linear Spline Meta-Regression: Visualization
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# STEP 15: SPLINE PLOT
# Purpose: Plot the fitted natural cubic spline from Step 14 with its
#          pointwise confidence band and the observed effect sizes.
# Data:    Step 14 (spline_model_results), or a plot recipe file saved from
#          this cell.
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import html


def _spline_plot_cell():
    PLOT_TYPE = 'spline'
    RESULTS_KEY = 'spline_model_results'
    state = {'data': None, 'source': None, 'recipe_name': None, 'token': None}

    # --- 1. DATA SNAPSHOT (everything the figure draws) ---
    def _category_values(col, reg, ad):
        """Colour value of each point; text, with missing values shown as 'Missing'."""
        if col in reg.columns:
            ser = reg[col]
        elif ad is not None and 'id' in reg.columns and 'id' in ad.columns and col in ad.columns:
            # one value per study (a merge on 'id' would duplicate rows); 'Mixed' if it varies
            per = {}
            for key, g in ad.groupby('id')[col]:
                u = pd.unique(g.dropna().astype(str).str.strip())
                per[key] = u[0] if len(u) == 1 else ('Mixed' if len(u) > 1 else None)
            ser = reg['id'].map(per)
        else:
            return None
        return [('Missing' if (v is None or (not isinstance(v, str) and pd.isna(v)) or str(v).strip() in ('', 'nan'))
                 else str(v).strip()) for v in ser.tolist()]

    def build_data():
        res = ANALYSIS_CONFIG.get(RESULTS_KEY)
        if not res:
            return None, None
        gs = ANALYSIS_CONFIG.get('global_settings') or {}
        es = ANALYSIS_CONFIG.get('es_config') or {}
        alpha = float(gs.get('alpha', 0.05))
        dist_type = gs.get('dist_type', 't')
        ci_level = ci_pct_label(alpha)
        reg = res['reg_df']
        mod = res['moderator_col']
        eff_col = ANALYSIS_CONFIG['effect_col']
        ad = ANALYSIS_CONFIG.get('analysis_data')
        ad = ad if isinstance(ad, pd.DataFrame) else None

        xv = pd.to_numeric(reg[mod], errors='coerce').to_numpy(float)
        lo, hi = np.nanmin(xv), np.nanmax(xv)
        pad = (hi - lo) * 0.05
        x_grid = np.linspace(lo - pad, hi + pad, 200)
        y_pred, _se, lower, upper, df_t = spline_prediction_band(res, x_grid, alpha=alpha, dist_type=dist_type)
        curve = pd.DataFrame({'x': x_grid, 'fit': y_pred, 'lower': lower, 'upper': upper})
        pts = pd.DataFrame({'x': xv, 'y': reg[eff_col].to_numpy(float)})

        excluded = {ANALYSIS_CONFIG.get('effect_col'), ANALYSIS_CONFIG.get('var_col'), ANALYSIS_CONFIG.get('se_col'),
                    'w_fixed', 'w_random', 'id', 'shared_group_id', 'xe', 'sde', 'ne', 'xc', 'sdc', 'nc', None}
        base = ad if ad is not None else reg
        color_levels = {}
        for c in base.columns:
            if c in excluded or not (is_text_dtype(base[c]) or isinstance(base[c].dtype, pd.CategoricalDtype)):
                continue
            if base[c].nunique() > 15:
                continue
            vals = _category_values(c, reg, ad)
            if vals is None:
                continue
            pts[f'c::{c}'] = vals
            tail = [x for x in ('Mixed', 'Missing') if x in vals]
            color_levels[str(c)] = sorted({x for x in vals if x not in tail}, key=str) + tail

        mc = res.get('model_comparison') or {}
        lrt = {'p': mc.get('p_value'), 'stat': mc.get('lrt'), 'df': mc.get('df'), 'method': mc.get('method', 'ML'),
               'delta_aic': mc.get('delta_aic'), 'error': mc.get('error')}
        omni_p = res.get('omnibus_p')
        data = {
            'curve': curve, 'points': pts, 'color_levels': color_levels,
            'moderator': str(mod), 'df_spline': int(res.get('df_spline', 3)),
            'lrt': lrt,
            'omnibus_p': None if omni_p is None else float(omni_p),
            'omnibus_method': res.get('omnibus_p_method'),
            'tau_sq': None if res.get('tau_sq') is None else float(res['tau_sq']),
            'sigma_sq': float(res.get('sigma_sq') or 0.0),
            'model_type': str(res.get('model_type', 'Spline')),
            'n_obs': int(res.get('n_obs_used', len(reg))),
            'k_studies': int(res.get('n_clusters_used', reg['id'].nunique() if 'id' in reg.columns else len(reg))),
            'ci_level': ci_level, 'dist_type': dist_type, 'df_t': df_t,
            'null_value': es_null_value(), 'effect_label': es.get('effect_label', 'Effect Size'),
        }
        when = res.get('timestamp')
        source = {
            'analysis': f"Spline of {mod} (df={data['df_spline']})",
            'effect_label': data['effect_label'],
            'model': data['model_type'],
            'ci_level': ci_level,
            'analysis_time': when.strftime('%Y-%m-%d %H:%M') if hasattr(when, 'strftime') else when,
        }
        return data, source

    # --- 2. CONTROLS (every widget whose value is saved in a recipe) ---
    L = lambda w='450px': widgets.Layout(width=w)
    C = {
        'show_title': widgets.Checkbox(value=True, description='Show Plot Title', indent=False),
        'title': widgets.Text(value='Natural Cubic Spline Analysis', description='Title:', layout=L()),
        'xlabel': widgets.Text(value='Moderator', description='X Label:', layout=L()),
        'ylabel': widgets.Text(value='Effect Size', description='Y Label:', layout=L()),
        'width': widgets.FloatSlider(value=10.0, min=2.0, max=16.0, step=0.1, description='Width (in):',
                                     continuous_update=False),
        'height': widgets.FloatSlider(value=6.0, min=2.0, max=12.0, step=0.1, description='Height (in):',
                                      continuous_update=False),
        'title_font': widgets.IntSlider(value=14, min=6, max=24, description='Title Font:'),
        'label_font': widgets.IntSlider(value=12, min=6, max=18, description='Label Font:'),
        'tick_font': widgets.IntSlider(value=10, min=5, max=16, description='Tick Font:'),
        'show_points': widgets.Checkbox(value=True, description='Show Data Points', indent=False),
        'color_by': widgets.Dropdown(options=['None'], value='None', description='Color By:', layout=L('400px')),
        'point_color': widgets.Dropdown(options=['gray', 'steelblue', 'black', 'red', 'green', 'purple'],
                                        value='gray', description='Color:'),
        'point_size': widgets.IntSlider(value=40, min=10, max=150, step=5, description='Size:'),
        'point_alpha': widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.1, description='Opacity:'),
        'curve_color': widgets.Dropdown(options=['blue', 'red', 'black', 'green', 'purple'], value='blue',
                                        description='Line Color:'),
        'curve_width': widgets.FloatSlider(value=2.5, min=0.5, max=6.0, step=0.5, description='Line Width:'),
        'show_ci': widgets.Checkbox(value=True, description='Show Confidence Band', indent=False),
        'ci_alpha': widgets.FloatSlider(value=0.15, min=0.05, max=0.5, step=0.05, description='CI Opacity:'),
        'show_stats': widgets.Checkbox(value=True, description='Show Stats (P-value)', indent=False),
        'show_omnibus': widgets.Checkbox(value=False, description='Also show the omnibus test (any association)',
                                         indent=False),
        'grid': widgets.Checkbox(value=True, description='Show Grid', indent=False),
        'show_null': widgets.Checkbox(value=True, description='Show Null Line', indent=False),
        'legend_loc': widgets.Dropdown(options=['best', 'upper right', 'upper left', 'lower right', 'lower left',
                                                'none'], value='best', description='Legend:'),
        'legend_font': widgets.IntSlider(value=10, min=5, max=16, description='Legend Font:'),
        'save_pdf': widgets.Checkbox(value=True, description='Save as PDF', indent=False),
        'save_png': widgets.Checkbox(value=True, description='Save as PNG', indent=False),
        'save_svg': widgets.Checkbox(value=False, description='Save as SVG', indent=False),
        'dpi': widgets.IntText(value=300, description='PNG DPI:', layout=L('150px')),
        'filename': widgets.Text(value='Spline_Plot', description='Filename:', layout=L('300px')),
        'transparent': widgets.Checkbox(value=False, description='Transparent Background', indent=False),
    }
    label_box = widgets.VBox()

    def build_label_widgets(data):
        """One Text per colour moderator and per category, saved as 'label::<name>'."""
        for k in [k for k in C if k.startswith('label::')]:
            del C[k]
        names = set()
        for col, levels in data['color_levels'].items():
            names.add(col)
            names.update(levels)
        names.discard('')
        for nm in sorted(names, key=str):
            C[f'label::{nm}'] = widgets.Text(value=nm, description=f"{nm}:", layout=L('400px'))
        rows = [C[f'label::{nm}'] for nm in sorted(names, key=str)]
        label_box.children = rows or [widgets.Label("No categorical labels found to edit.")]

    def set_data_defaults(data):
        """Defaults that depend on the data (called for session data and recipes)."""
        C['color_by'].options = ['None'] + list(data['color_levels'])
        C['color_by'].value = 'None'
        C['xlabel'].value = data['moderator']
        C['ylabel'].value = data['effect_label']
        C['title'].value = f"Spline Regression: {data['effect_label']} vs. {data['moderator']}"
        build_label_widgets(data)

    # --- 3. DRAWING (data + settings -> figure; no widget access) ---
    def _fmt_p(p):
        return "< 0.0001" if p < 1e-4 else f"= {p:.4g}"

    def _stars(p):
        return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

    def draw(data, s):
        pts, cv = data['points'], data['curve']
        labmap = {k[len('label::'):]: v for k, v in s.items() if k.startswith('label::')}
        color_by = s['color_by'] if s['color_by'] in data['color_levels'] else 'None'

        fig, ax = plt.subplots(figsize=(s['width'], s['height']))
        if s['transparent']:
            fig.patch.set_alpha(0)
            ax.patch.set_alpha(0)

        legend_title = None
        if s['show_points']:
            x, y = pts['x'].to_numpy(float), pts['y'].to_numpy(float)
            if color_by != 'None':
                cats = pts[f'c::{color_by}'].to_numpy(object)
                levels = data['color_levels'][color_by]
                cmap = plt.get_cmap('tab10' if len(levels) <= 10 else 'tab20')
                for i, lv in enumerate(levels):
                    m = cats == lv
                    if not m.any():
                        continue
                    ax.scatter(x[m], y[m], color=cmap(i % cmap.N), alpha=s['point_alpha'], s=s['point_size'],
                               label=labmap.get(lv, lv), edgecolors='k', linewidth=0.5)
                legend_title = labmap.get(color_by, color_by)
            else:
                ax.scatter(x, y, color=s['point_color'], alpha=s['point_alpha'], s=s['point_size'],
                           label='Observations', edgecolors='k', linewidth=0.5)

        ax.plot(cv['x'], cv['fit'], color=s['curve_color'], linewidth=s['curve_width'], label='Spline Fit')
        if s['show_ci']:
            ax.fill_between(cv['x'], cv['lower'], cv['upper'], color=s['curve_color'], alpha=s['ci_alpha'],
                            label=f"{data['ci_level']} CI")
        if s['show_null']:
            ax.axhline(data['null_value'], color='black', linestyle=':', linewidth=1.5, alpha=0.6)
        if s['grid']:
            ax.grid(True, linestyle=':', alpha=0.4)
        if s['show_title']:
            ax.set_title(s['title'], fontsize=s['title_font'], fontweight='bold', pad=15)
        ax.set_xlabel(s['xlabel'], fontsize=s['label_font'], fontweight='bold')
        ax.set_ylabel(s['ylabel'], fontsize=s['label_font'], fontweight='bold')
        ax.tick_params(axis='both', labelsize=s['tick_font'])
        if s['legend_loc'] != 'none':
            ax.legend(loc=s['legend_loc'], title=legend_title, frameon=True, fancybox=True,
                      fontsize=s['legend_font'], title_fontsize=s['legend_font'])

        if s['show_stats']:
            lines = []
            p_nl = data['lrt'].get('p')
            if p_nl is not None and np.isfinite(p_nl):
                lines.append(f"Non-linearity (LRT): p {_fmt_p(p_nl)} {_stars(p_nl)}")
            else:
                lines.append("Non-linearity (LRT): n/a")
            if s['show_omnibus'] and data['omnibus_p'] is not None and np.isfinite(data['omnibus_p']):
                lines.append(f"Omnibus: p {_fmt_p(data['omnibus_p'])} {_stars(data['omnibus_p'])}")
            if data['tau_sq'] is not None:
                lines.append(f"τ²: {data['tau_sq']:.3f}")
            ax.text(0.05, 0.95, "\n".join(lines), transform=ax.transAxes, verticalalignment='top',
                    fontsize=s['legend_font'], bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        fig.tight_layout()
        return fig

    def caption(data, s):
        esc = lambda t: html.escape(str(t))
        lab = {k[len('label::'):]: v for k, v in s.items() if k.startswith('label::')}
        txt = (f"Restricted natural cubic spline meta-regression of {esc(data['effect_label'])} on "
               f"<b>{esc(data['moderator'])}</b> (df = {data['df_spline']}; {esc(data['model_type'])}; "
               f"n = {data['n_obs']} effect sizes from k = {data['k_studies']} studies). The solid curve is the "
               f"fitted spline.")
        if s['show_ci']:
            dist = (f"t distribution, df = studies − coefficients = {data['df_t']}" if data['dist_type'] == 't'
                    and data['df_t'] is not None else "normal distribution")
            txt += f" The shaded region is the pointwise {data['ci_level']} confidence band of the curve ({dist})."
        if s['show_points']:
            txt += " Points are the individual effect sizes."
            if s['color_by'] != 'None' and s['color_by'] in data['color_levels']:
                txt += f" They are coloured by <b>{esc(lab.get(s['color_by'], s['color_by']))}</b>."
        if s['show_null']:
            txt += f" The dotted horizontal line marks no effect ({data['null_value']:g})."
        lr = data['lrt']
        if lr.get('p') is not None:
            txt += (f" Non-linearity was tested with a likelihood-ratio test of the spline against the linear "
                    f"model (both fitted by {esc(lr.get('method') or 'ML')}): χ²({lr.get('df')}) = "
                    f"{lr.get('stat'):.2f}, p {_fmt_p(lr['p'])}.")
        if s['show_stats'] and s['show_omnibus'] and data['omnibus_p'] is not None:
            txt += (f" The omnibus test (any association with {esc(data['moderator'])}, linear part included) "
                    f"gave p {_fmt_p(data['omnibus_p'])}.")
        return txt

    # --- 4. ACTIONS ---
    out = widgets.Output()
    caption_output = widgets.Output()
    source_box = widgets.HTML()

    def refresh_source():
        source_box.value = source_html(state['source'], state['recipe_name'])

    def generate(_b=None):
        with out:
            clear_output(wait=True)
            if state['data'] is None:
                display(HTML("<div style='color:#721c24;'>❌ No spline results: run Step 14 (Spline "
                             "Meta-Regression) or load a plot recipe.</div>"))
                return
            if state['recipe_name'] is None and results_token(RESULTS_KEY) != state['token']:
                display(HTML(stale_warning_html('Step 14 (Spline Meta-Regression)')))
            s = widget_values(C)
            try:
                fig = draw(state['data'], s)
                paths = cometa_save_figure(fig, f"{s['filename']}_{state['data']['moderator']}", pdf=s['save_pdf'],
                                           png=s['save_png'], svg=s['save_svg'], dpi=s['dpi'],
                                           transparent=s['transparent'])
                show_figure(fig)
            except Exception as e:
                plt.close('all')
                display(HTML(f"<div style='color:#721c24;'>❌ Plotting error: {html.escape(str(e))}</div>"))
                return
            display(HTML(saved_files_html(paths)))
            cap = caption_html(caption(state['data'], s))
            display(HTML(cap))
        with caption_output:
            clear_output()
            display(HTML(cap + "<div style='background-color:#fff3cd; padding:10px; border-left:4px solid #ffc107; "
                         "margin-top:10px; font-size:13px;'><b>💡 Tip:</b> Select the text above, copy, and paste it "
                         "into your manuscript's figure legend.</div>"))

    def get_recipe():
        if state['data'] is None:
            return None
        return build_plot_recipe(PLOT_TYPE, state['data'], widget_values(C), state['source'])

    def on_load(rec, style_only, name):
        if not style_only:
            state.update(data=rec['data'], source=rec.get('source'), recipe_name=name)
            set_data_defaults(state['data'])
            refresh_source()
        skipped = apply_widget_values(C, rec.get('settings'))
        generate()
        return skipped

    # --- 5. INITIAL STATE AND LAYOUT ---
    try:
        state['data'], state['source'] = build_data()
    except Exception as e:
        state['data'], state['source'] = None, None
        print(f"⚠️ Could not read the Step 14 results: {e}")
    state['token'] = results_token(RESULTS_KEY)
    if state['data'] is not None:
        set_data_defaults(state['data'])
    else:
        label_box.children = [widgets.Label("No categorical labels found to edit.")]
    refresh_source()

    preset = preset_dropdown({'width': C['width'], 'height': C['height'], 'title': C['title_font'],
                              'label': C['label_font'], 'tick': C['tick_font']})
    H = lambda t: widgets.HTML(f"<h4 style='color: #2E86AB;'>{t}</h4>")
    hr = lambda: widgets.HTML("<hr style='margin: 5px 0;'>")
    tabs = widgets.Tab(children=[
        widgets.VBox([H('Labels & Dimensions'), C['show_title'], C['title'], C['xlabel'], C['ylabel'], hr(),
                      preset, C['width'], C['height'], hr(), C['title_font'], C['label_font'], C['tick_font']]),
        widgets.VBox([H('Data Points'), C['show_points'], C['color_by'], C['point_color'], C['point_size'],
                      C['point_alpha']]),
        widgets.VBox([H('Spline Curve'), C['curve_color'], C['curve_width'], hr(), C['show_ci'], C['ci_alpha'],
                      C['show_stats'], C['show_omnibus']]),
        widgets.VBox([H('Layout & Export'), C['grid'], C['show_null'], C['legend_loc'], C['legend_font'], hr(),
                      C['save_pdf'], C['save_png'], C['save_svg'], C['dpi'], C['filename'], C['transparent']]),
        widgets.VBox([H('Label Editor'), widgets.HTML("<i>Rename data categories for the legend:</i>"), label_box]),
        widgets.VBox([H('📝 Figure Caption'),
                      widgets.HTML("<p style='color: #666;'><i>Generated from the plot settings when you click "
                                   "Generate.</i></p>"), caption_output]),
    ])
    for i, t in enumerate(['🎨 Style', '⚫ Points', '🌊 Curve', '💾 Layout', '✏️ Labels', '📝 Caption']):
        tabs.set_title(i, t)

    header = widgets.HTML("""
        <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    padding: 20px; border-radius: 10px; margin-bottom: 20px;'>
            <h2 style='color: white; margin: 0; text-align: center;'>📊 Step 15: Publication-Ready Spline Plot</h2>
            <p style='color: rgba(255,255,255,0.9); margin: 5px 0 0 0; text-align: center; font-size: 14px;'>
                Visualize spline analysis results with full customization</p>
        </div>""")

    run_btn = widgets.Button(description='📊 Generate Spline Plot', button_style='success',
                             layout=widgets.Layout(width='450px', height='50px'), style={'font_weight': 'bold'})
    run_btn.on_click(generate)
    recipe_bar = plot_recipe_bar(
        PLOT_TYPE, get_recipe, on_load,
        lambda: f"{C['filename'].value}_{state['data']['moderator']}" if state['data'] else C['filename'].value)

    display(widgets.VBox([header, source_box, widgets.HTML("<hr>"), tabs,
                          widgets.HTML("<hr style='margin: 10px 0;'>"), run_btn, recipe_bar, out]))


_spline_plot_cell()


In [ ]:
#@title 🌳 16. Exploratory Meta-Forest (cluster-robust)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# CELL: EXPLORATORY META-FOREST (MVC ARCHITECTURE)
# Purpose: Precision-weighted Random Forest for EXPLORATORY ranking of
#          moderator importance (hypothesis generation, not confirmation).
# Architecture: Model-View-Controller pattern.
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
import datetime

# --- SCIKIT-LEARN IMPORTS ---
try:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.pipeline import Pipeline
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OrdinalEncoder
    from sklearn.inspection import permutation_importance
    from sklearn.metrics import mean_squared_error
    from sklearn.model_selection import GroupKFold, KFold
    from sklearn.base import BaseEstimator, RegressorMixin
    from sklearn.tree import DecisionTreeRegressor
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False


# =============================================================================
# PART 1: HTML TEMPLATES (Styling)
# =============================================================================

class MetaForestHTMLTemplates:
    @staticmethod
    def dashboard_header(title: str, subtitle: str) -> str:
        return f"""
        <div style='display: flex; align-items: center; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 2px solid #ecf0f1; font-family: system-ui, -apple-system, sans-serif;'>
            <div style='background-color: #2c3e50; color: white; padding: 8px 16px; border-radius: 6px; font-weight: 600; font-size: 1.1em; letter-spacing: 0.5px; margin-right: 15px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
                {title}
            </div>
            <div style='color: #7f8c8d; font-size: 1.1em; display: flex; align-items: center;'>
                {subtitle}
            </div>
        </div>
        """

    @staticmethod
    def gradient_card(title: str, value: str, subtitle: str) -> str:
        return f"""
        <div style='background: linear-gradient(135deg, #27ae60 0%, #2ecc71 100%); padding: 15px 25px; border-radius: 8px;
                    color: white; margin-bottom: 20px; display: flex; justify-content: space-between; align-items: center; box-shadow: 0 4px 6px rgba(0,0,0,0.1); font-family: system-ui, -apple-system, sans-serif;'>
            <div>
                <div style='font-size: 0.85em; text-transform: uppercase; letter-spacing: 1px; opacity: 0.9;'>{title}</div>
                <div style='margin: 0; font-size: 2.2em; font-weight: bold;'>{value}</div>
            </div>
            <div>
                <span style='background: rgba(255,255,255,0.2); padding: 5px 15px; border-radius: 20px; font-size: 1.0em; font-weight: bold;'>
                    {subtitle}
                </span>
            </div>
        </div>
        """

    @staticmethod
    def info_grid(items: list) -> str:
        html = "<div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 20px; font-family: system-ui, -apple-system, sans-serif;'>"
        for item in items:
            color = item.get('color', '#007bff')
            html += f"""
            <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px;
                        border-left: 4px solid {color}; box-shadow: 0 1px 3px rgba(0,0,0,0.05);'>
                <div style='color: #6c757d; font-size: 0.9em; text-transform: uppercase; letter-spacing: 0.5px;'>{item['label']}</div>
                <div style='font-size: 1.5em; font-weight: bold; color: #2c3e50;'>{item['value']}</div>
            </div>
            """
        html += "</div>"
        return html

    @staticmethod
    def guardrail_box(content: str) -> str:
        return f"""
        <div style='background-color: #fff3cd; padding: 20px; border-radius: 6px;
                    margin-bottom: 20px; border-left: 4px solid #ffc107; font-family: system-ui, -apple-system, sans-serif;'>
            <h4 style='margin-top: 0; color: #856404;'>⚠️ Exploratory Guardrail</h4>
            <p style='margin: 0; font-size: 1.05em; color: #856404; line-height: 1.5;'>{content}</p>
        </div>
        """


# =============================================================================
# PART 2: MODEL (Logic & Math)
# =============================================================================
class ClusterBootstrapForest(BaseEstimator, RegressorMixin):
    """Random forest whose bootstrap unit is the CLUSTER, not the row.

    Each tree is grown on a nonparametric cluster bootstrap sample: whole
    clusters (studies) are drawn with replacement (Field & Welsh 2007), so
    all effect sizes from a study enter or leave a tree together. This stops
    a study with many effect sizes from seeding proportionally more bags than
    a small study, and keeps the train/OOB boundary at the study level.
    Precision weights are threaded through as sample_weight on the splits.

    groups=None at fit time => each row is its own cluster, which reduces
    EXACTLY to an ordinary (row-bootstrap) random forest.

    Limitation NOT fixed here (by design): within a single bootstrap sample a
    large study still contributes its many rows to that tree's splits. Cluster
    bootstrap equalizes cluster inclusion ACROSS trees and the OOB split, not
    row count WITHIN a drawn cluster. Equalizing the latter (cluster-size
    down-weighting) trades directly against precision weighting, so it's left
    as a separate decision rather than silently baked in.
    """
    def __init__(self, n_estimators=500, min_samples_leaf=3,
                 max_features=1.0, oob_score=True, random_state=42):
        self.n_estimators = n_estimators
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.oob_score = oob_score
        self.random_state = random_state

    def fit(self, X, y, sample_weight=None, groups=None):
        X = np.asarray(X); y = np.asarray(y)
        n = len(y)
        sample_weight = (np.ones(n) if sample_weight is None
                         else np.asarray(sample_weight, dtype=float))

        # No groups -> each row its own cluster (=> ordinary row bootstrap).
        groups = np.arange(n) if groups is None else np.asarray(groups)
        self.cluster_labels_ = np.unique(groups)
        n_clusters = len(self.cluster_labels_)
        cluster_to_rows = {c: np.flatnonzero(groups == c)
                           for c in self.cluster_labels_}

        rng = np.random.RandomState(self.random_state)
        max_feat = self._resolve_max_features(X.shape[1])

        self.trees_ = []
        oob_sum = np.zeros(n); oob_cnt = np.zeros(n)

        for _ in range(self.n_estimators):
            # cluster bootstrap: draw n_clusters clusters WITH replacement
            drawn = self.cluster_labels_[rng.randint(0, n_clusters, n_clusters)]
            rows = np.concatenate([cluster_to_rows[c] for c in drawn])

            tree = DecisionTreeRegressor(
                min_samples_leaf=self.min_samples_leaf,
                max_features=max_feat,
                random_state=rng.randint(0, 2**31 - 1)
            )
            tree.fit(X[rows], y[rows], sample_weight=sample_weight[rows])
            self.trees_.append(tree)

            if self.oob_score:
                oob_mask = ~np.isin(groups, drawn)   # clusters not drawn
                if oob_mask.any():
                    oob_sum[oob_mask] += tree.predict(X[oob_mask])
                    oob_cnt[oob_mask] += 1

        if self.oob_score:
            seen = oob_cnt > 0
            self.oob_prediction_ = np.full(n, np.nan)
            self.oob_prediction_[seen] = oob_sum[seen] / oob_cnt[seen]
            self.oob_score_ = (self._weighted_r2(y[seen],
                               self.oob_prediction_[seen], sample_weight[seen])
                               if seen.any() else float('nan'))
        return self

    def predict(self, X):
        X = np.asarray(X)
        preds = np.zeros(len(X))
        for tree in self.trees_:
            preds += tree.predict(X)
        return preds / len(self.trees_)

    def _resolve_max_features(self, p):
        mf = self.max_features
        if isinstance(mf, float):
            return max(1, int(mf * p))
        if mf == 'sqrt':
            return max(1, int(np.sqrt(p)))
        return p

    @staticmethod
    def _weighted_r2(y, yhat, w):
        ybar = np.sum(w * y) / np.sum(w)
        ss_res = np.sum(w * (y - yhat) ** 2)
        ss_tot = np.sum(w * (y - ybar) ** 2)
        return 1 - ss_res / ss_tot if ss_tot > 0 else float('nan')


class MetaForestModel:
    def __init__(self):
        self.error = None
        self.results = None
        self.random_state = 42
        self.n_estimators = 500

    def get_analysis_data(self):
        if 'ANALYSIS_CONFIG' in globals() and 'analysis_data' in ANALYSIS_CONFIG:
            return ANALYSIS_CONFIG['analysis_data']
        return None

    def get_config_val(self, key, default=None):
        if 'ANALYSIS_CONFIG' in globals():
            return ANALYSIS_CONFIG.get(key, default)
        return default

    def discover_features(self, df):
        if df is None:
            return []

        # Technical columns to exclude
        exclude_cols = {
            'id', 'xe', 'xc', 'ne', 'nc', 'sde', 'sdc', 'w_fixed', 'w_random',
            'df', 'sp', 'sp_squared', 'hedges_j', 'weights', 'wi', 'shared_group_id',
            'events_e', 'nonevents_e', 'events_c', 'nonevents_c', 'n_total', 'se',
            'response_ratio', 'log_response_ratio', 'lnrr', 'rr',
            'smd', 'cohens_d', 'log_or', 'logor', 'or', 'log_rr',"Percent_Change",
            'zi', 'fisher_z', 'effect_size', 'es',"sdc_imputed", "Response_Ratio", "sde_imputed"
        }

        # Also exclude the dependent variables
        eff_col = self.get_config_val('effect_col', 'yi')
        var_col = self.get_config_val('var_col', 'vi')
        se_col = self.get_config_val('se_col', 'se')
        ci_cols = [c for c in df.columns if 'CI_' in c or 'ci_' in c]

        exclude_cols.update([eff_col, var_col, se_col] + ci_cols)

        features = []
        for col in df.columns:
            if col in exclude_cols or col is None:
                continue

            # Keep numerics and strings/categories
            if pd.api.types.is_numeric_dtype(df[col]):
                if df[col].nunique() > 1:
                    features.append(col)
            elif is_text_dtype(df[col]) or isinstance(df[col].dtype, pd.CategoricalDtype):
                if df[col].nunique() > 1 and df[col].nunique() < len(df): # Avoid unique ID columns
                    features.append(col)

        return sorted(list(set(features)))

    def _get_precomputed_tau2(self):
        """Total heterogeneity of the intercept-only model reported by the
        Overall Meta-Analysis cell: tau² (+ sigma² when the 3-level model is
        reported). Read from that cell's results explicitly -- not from the
        first tau²-like key anywhere in ANALYSIS_CONFIG, which could be a
        subgroup or spline estimate. Returns a float or None (-> DL)."""
        if 'ANALYSIS_CONFIG' not in globals():
            return None
        overall = ANALYSIS_CONFIG.get('overall_results') or {}
        rep = overall.get('reported') or {}
        if rep:
            total = float(rep.get('tau_squared') or 0.0) + float(rep.get('sigma_squared') or 0.0)
        else:
            total = overall.get('tau_squared')
        if isinstance(total, (int, float)) and np.isfinite(total) and total >= 0:
            return float(total)
        return None

    def _estimate_total_tau2_DL(self, y, variances):
        """Standard DerSimonian-Laird tau^2 from the intercept-only (null)
        model. Centering is the inverse-variance weighted grand mean, NOT
        forest predictions, so Q is Cochran's Q and (Q - df)/C is the exact
        one-sample DL estimator with df = k - 1 (one parameter: the mean)."""
        w = 1.0 / variances
        y_bar = np.sum(w * y) / np.sum(w)
        Q = np.sum(w * (y - y_bar) ** 2)
        k = len(y)
        df_Q = k - 1
        sum_w, sum_w2 = np.sum(w), np.sum(w ** 2)
        C = sum_w - (sum_w2 / sum_w) if sum_w > 0 else 0.0
        if C > 0 and Q > df_Q:
            return (Q - df_Q) / C
        return 0.0

    def _get_tau2(self, y, variances):
        """Returns (tau2, source_label). Borrows from config if available,
        else falls back to a clean null-model DerSimonian-Laird."""
        pre = self._get_precomputed_tau2()
        if pre is not None:
            return pre, 'main RE model (config)'
        return self._estimate_total_tau2_DL(y, variances), 'DerSimonian-Laird (null model)'


    def _build_pipeline(self, X_sample, oob_score=True):
        """Single source of truth for preprocessing + forest, so the
        cross-validated importance model is identical to the final model."""
        categorical_cols = [c for c in X_sample.columns
                            if is_text_dtype(X_sample[c]) or isinstance(X_sample[c].dtype, pd.CategoricalDtype)]
        numeric_cols = X_sample.select_dtypes(include=['number']).columns

        preprocessor = ColumnTransformer(
            transformers=[
                ('num', SimpleImputer(strategy='median'), numeric_cols),
                ('cat', Pipeline([
                    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
                    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
                ]), categorical_cols)
            ])

        rf = ClusterBootstrapForest(
            n_estimators=self.n_estimators, min_samples_leaf=3,
            max_features=1.0, oob_score=oob_score, random_state=self.random_state
        )
        return Pipeline([('preprocessor', preprocessor), ('rf', rf)])


    def _get_group_ids(self, analysis_df):
        """Cluster ids for grouped CV, aligned to the rows actually used.
        Returns (groups_array, col_name) or (None, None) if unavailable."""
        #gcol = self.get_config_val('cluster_col', 'shared_group_id')
        gcol = self.get_config_val('cluster_col', 'id')
        if gcol in analysis_df.columns and analysis_df[gcol].nunique() > 1:
            return analysis_df[gcol].values, gcol
        return None, None

    def _grouped_permutation_importance(self, X, y, w, groups, n_repeats=10):
        """Permutation importance on held-out folds. Whole clusters are held
        out (GroupKFold) AND each fold's forest is fit with cluster bootstrap
        on its training clusters, so the evaluation and the fit are both
        out-of-study. StdDev is across-fold spread (a stability measure)."""
        feature_names = list(X.columns)

        if groups is not None:
            n_splits = min(5, len(np.unique(groups)))
            split_iter = GroupKFold(n_splits=n_splits).split(X, y, groups)
        else:
            n_splits = min(5, len(X))
            split_iter = KFold(n_splits=n_splits, shuffle=True,
                               random_state=self.random_state).split(X, y)

        fold_importances = []
        for train_idx, test_idx in split_iter:
            X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]
            w_tr, w_te = w[train_idx], w[test_idx]
            groups_tr = groups[train_idx] if groups is not None else None

            pipe = self._build_pipeline(X_tr, oob_score=False)
            pipe.fit(X_tr, y_tr, rf__sample_weight=w_tr, rf__groups=groups_tr)

            def scorer(estimator, X_val, y_val, _w=w_te):
                preds = estimator.predict(X_val)
                return -mean_squared_error(y_val, preds, sample_weight=_w)

            r = permutation_importance(
                pipe, X_te, y_te, scoring=scorer,
                n_repeats=n_repeats, random_state=self.random_state
            )
            fold_importances.append(r.importances_mean)

        imp = np.vstack(fold_importances)
        return feature_names, imp.mean(axis=0), imp.std(axis=0), n_splits


    def run_analysis(self, selected_features):
        self.error = None

        if not SKLEARN_AVAILABLE:
            self.error = "scikit-learn is not installed. Please run: !pip install scikit-learn"
            return False

        df = self.get_analysis_data()
        if df is None:
            self.error = "Data not found. Run Steps 2–6 (data and effect sizes) first."
            return False

        if len(df) < 30:
            self.error = f"Insufficient data (N={len(df)}). Random Forests require at least 30 observations to prevent severe overfitting. Please rely on theory-driven hypothesis testing via standard Meta-Regression."
            return False

        if not selected_features:
            self.error = "Please select at least one moderator variable."
            return False

        eff_col = self.get_config_val('effect_col', 'yi')
        var_col = self.get_config_val('var_col', 'vi')

        # Prepare Data
        analysis_df = df.dropna(subset=[eff_col, var_col]).copy()
        analysis_df = analysis_df[analysis_df[var_col] > 0]

        y = analysis_df[eff_col].values
        variances = analysis_df[var_col].values
        X = analysis_df[list(selected_features)].copy()
        X.columns = [str(c) for c in X.columns]

        # Heterogeneity from the NULL model, not from forest residuals.
        # No FE forest needed: tau^2 depends only on (y, v).
        tau2, tau2_source = self._get_tau2(y, variances)

        # Group ids must come from analysis_df AFTER dropna/var>0 filtering,
        # so they line up with X / y / w row-for-row.
        groups, _gcol = self._get_group_ids(analysis_df)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            # Random-Effects weights from null-model tau^2
            w_re = 1.0 / (variances + tau2)
            w_re = np.clip(w_re, a_min=1e-8, a_max=None)

            pipeline_re = self._build_pipeline(X)
            pipeline_re.fit(X, y, rf__sample_weight=w_re, rf__groups=groups)

            # Out-of-sample (cluster-held-out) permutation importance
            feat_names, imp_mean, imp_std, n_splits = \
                self._grouped_permutation_importance(X, y, w_re, groups, n_repeats=10)

        results_df = pd.DataFrame({
            'Feature': feat_names,
            'Importance': imp_mean,
            'StdDev': imp_std
        }).sort_values('Importance', ascending=False).reset_index(drop=True)

        self.results = {
            'importance_df': results_df,
            'null_tau2': tau2,
            'tau2_source': tau2_source,
            'oob_r2': pipeline_re.named_steps['rf'].oob_score_,
            'n_obs': len(y),
            'n_features': len(selected_features),
            'cv_grouped': groups is not None,
            'cv_n_splits': n_splits
        }

        # Save to global config
        if 'ANALYSIS_CONFIG' in globals():
            ANALYSIS_CONFIG['meta_forest_results'] = self.results

        return True

# =============================================================================
# PART 3: VIEW (UI Layout & Rendering)
# =============================================================================
class MetaForestView:
    def __init__(self):
        self.templates = MetaForestHTMLTemplates()
        self.feature_checkboxes = {}   # name -> Checkbox
        self._create_widgets()
        self._create_tabs()
        self._build_layout()

    def _create_widgets(self):
        # --- Moderator picker toolbar -------------------------------------
        self.feature_search = widgets.Text(
            placeholder='🔍 Filter moderators...',
            layout=widgets.Layout(width='180px')
        )
        self.select_all_btn = widgets.Button(
            description='All', icon='check-square-o', button_style='info',
            tooltip='Select all currently shown moderators',
            layout=widgets.Layout(width='65px')
        )
        self.clear_all_btn = widgets.Button(
            description='Clear', icon='square-o', button_style='warning',
            tooltip='Clear every moderator',
            layout=widgets.Layout(width='75px')
        )

        toolbar = widgets.HBox(
            [self.feature_search, self.select_all_btn, self.clear_all_btn],
            layout=widgets.Layout(align_items='center', justify_content='space-between', width='340px', margin='5px 0 8px 0')
        )

        # --- Scrolling checkbox container ---------------------------------
        self.features_container = widgets.VBox(
            layout=widgets.Layout(
                width='340px',
                height='180px',
                overflow='auto',
                border='1px solid #ced4da',
                border_radius='4px',
                padding='8px',
                background_color='#f8f9fa' # Subtle gray background inside the box
            )
        )

        # --- Live selection counter ---------------------------------------
        self.selection_counter = widgets.HTML(
            value="",
            layout=widgets.Layout(margin='5px 0 0 5px')
        )

        # --- Assemble the picker column into a clean "Card" -------------
        self.feature_selector = widgets.VBox(
            [
                widgets.HTML("<b style='color:#2c3e50; font-family:system-ui, -apple-system, sans-serif; font-size:14px;'>Candidate Moderators:</b>"),
                toolbar,
                self.features_container,
                self.selection_counter,
            ],
            layout=widgets.Layout(
                width='365px',
                padding='12px',
                border='1px solid #e9ecef',
                border_radius='8px',
                background_color='#ffffff',
                box_shadow='0 2px 5px rgba(0,0,0,0.04)'
            )
        )

        # --- Run button ---------------------------------------------------
        self.run_button = widgets.Button(
            description='▶ Run Exploratory Meta-Forest',
            button_style='success',
            icon='tree',
            layout=widgets.Layout(width='300px', height='50px') # Slightly taller for emphasis
        )

        # --- View-internal wiring (UI only; never touches the Model) ------
        self.feature_search.observe(self._on_search_change, names='value')
        self.select_all_btn.on_click(self._on_select_all)
        self.clear_all_btn.on_click(self._on_clear_all)

    def _create_tabs(self):
        self.stale_banner = widgets.HTML(value="")
        if 'register_banner' in globals():
            register_banner('meta_forest', self.stale_banner)
        self.tab_ranking = widgets.Output()
        self.tab_diagnostics = widgets.Output()
        self.tab_publication = widgets.Output()
        self.tabs = widgets.Tab(children=[self.tab_ranking, self.tab_diagnostics, self.tab_publication])
        self.tabs.set_title(0, '📊 Importance Ranking')
        self.tabs.set_title(1, '⚙️ Model Diagnostics')
        self.tabs.set_title(2, '📝 Publication Text')

    def _build_layout(self):
        self.header = widgets.HTML("""
            <div style='font-family: system-ui, -apple-system, sans-serif;'>
                <h3 style='margin-bottom: 5px; color: #2E86AB;'>🌳 Exploratory Meta-Forest</h3>
                <p style='color: #6c757d; margin-top: 0; font-size: 13px;'>
                Select multiple candidate moderators. The algorithm will rank their predictive importance using a
                precision-weighted Random Forest. <b>Use this to explore which moderators may matter; moderators chosen here and then tested on the same data do not give confirmatory p-values.</b>
                </p>
            </div>
        """)
        # Align center makes the big run button sit perfectly beside the tall picker card
        self.controls = widgets.HBox(
            [self.feature_selector, self.run_button],
            layout=widgets.Layout(align_items='center', gap='35px', margin='10px 0 25px 0')
        )

    # ---------------------------------------------------------------------
    # Moderator picker logic (View-internal, no Model interaction)
    # ---------------------------------------------------------------------
    def update_feature_options(self, options):
        self.feature_checkboxes.clear()

        if not options:
            self.features_container.children = [
                widgets.HTML("<i style='color:#6c757d;'>Data not loaded</i>")
            ]
            self._refresh_counter()
            return

        for opt in options:
            cb = widgets.Checkbox(
                value=False,
                description=opt,
                indent=False,
                style={'description_width': 'initial'}, # Prevents text cutoff
                layout=widgets.Layout(width='95%', margin='0 0 4px 0')
            )
            cb.observe(self._on_checkbox_change, names='value')
            self.feature_checkboxes[opt] = cb

        self._apply_filter(self.feature_search.value)
        self._refresh_counter()

    def _apply_filter(self, query):
        q = (query or '').strip().lower()
        visible = [cb for name, cb in self.feature_checkboxes.items()
                   if q in name.lower()]
        if visible:
            self.features_container.children = visible
        else:
            self.features_container.children = [
                widgets.HTML("<i style='color:#6c757d; margin-left: 5px;'>No moderators match "
                             f"“{query}”</i>")
            ]

    def _refresh_counter(self):
        total = len(self.feature_checkboxes)
        sel = len(self.selected_features)
        color = '#27ae60' if sel else '#6c757d'
        self.selection_counter.value = (
            f"<span style='font-family:system-ui, -apple-system, sans-serif; font-size:12px; color:{color};'>"
            f"<b>{sel}</b> of {total} selected</span>"
        )

    def _on_search_change(self, change):
        self._apply_filter(change['new'])

    def _on_checkbox_change(self, change):
        self._refresh_counter()

    def _on_select_all(self, button):
        for cb in self.features_container.children:
            if isinstance(cb, widgets.Checkbox):
                cb.value = True

    def _on_clear_all(self, button):
        for cb in self.feature_checkboxes.values():
            cb.value = False

    @property
    def selected_features(self):
        return [name for name, cb in self.feature_checkboxes.items() if cb.value]

    # ---------------------------------------------------------------------
    # Output / rendering (unchanged except for guardrail text)
    # ---------------------------------------------------------------------
    def clear_all_tabs(self):
        for tab in [self.tab_ranking, self.tab_diagnostics, self.tab_publication]:
            tab.clear_output()

    def show_loading(self):
        with self.tab_ranking:
            display(HTML("<div style='padding: 15px; background-color: #e2e3e5; border-left: 4px solid #6c757d; border-radius: 4px; font-family: system-ui;'>⏳ <b>Growing Precision-Weighted Forest...</b> This may take a moment.</div>"))

    def show_error(self, error_msg):
        with self.tab_ranking:
            clear_output()
            display(HTML(f"<div style='color: #721c24; background-color: #f8d7da; padding: 15px; border-radius: 6px; border-left: 4px solid #dc3545; font-family: system-ui;'><b>❌ Error:</b> {error_msg}</div>"))

    def render_results(self, stats):
        df_imp = stats['importance_df']
        top_feature = df_imp.iloc[0]['Feature'] if len(df_imp) > 0 else "None"

        with self.tab_ranking:
            clear_output()
            display(HTML("<div style='padding: 20px;'>"))
            display(HTML(self.templates.dashboard_header("Variable Importance", "Random-Effects Weighted")))

            display(HTML(self.templates.gradient_card(
                title="Top Ranked Moderator",
                value=top_feature,
                subtitle="Highest Predictive Power"
            )))

            # Updated Text reflecting Subgroup vs Meta-Regression univariate routing
            interp = f"""
            <b>This is not a formal hypothesis test.</b> The Meta-Forest identifies which variables reduce prediction error the most.
            <br><br>
            <b>Recommendation:</b> Identify the top-ranked variable (e.g., <code>{top_feature}</code>). If it is categorical, test it formally in the <b>Subgroup Analysis</b> cell (which supports up to two interacting categories). If it is continuous, test it individually in the <b>Meta-Regression</b> cell to obtain valid p-values and account for hierarchical structure.
            """
            display(HTML(self.templates.guardrail_box(interp)))

            fig, ax = plt.subplots(figsize=(9, max(4, len(df_imp)*0.5)))
            colors = ['#27ae60'] + ['#bdc3c7'] * (len(df_imp) - 1)
            sns.barplot(
                data=df_imp, x='Importance', y='Feature',
                palette=colors, edgecolor='none', ax=ax
            )
            ax.errorbar(
                x=df_imp['Importance'], y=np.arange(len(df_imp)),
                xerr=df_imp['StdDev'], fmt='none', c='#2c3e50', capsize=4
            )
            ax.set_title('Permutation Importance (Weighted MSE Drop)', fontsize=12, fontweight='bold', pad=15)
            ax.set_xlabel('Importance Score (Higher = Better Predictor)', fontsize=10)
            ax.set_ylabel('')
            ax.grid(axis='x', linestyle='--', alpha=0.6)
            sns.despine(left=True)
            plt.tight_layout()
            plt.show()
            display(HTML("</div>"))

    def render_diagnostics(self, stats):
        with self.tab_diagnostics:
            clear_output()
            display(HTML("<div style='padding: 20px; font-family: system-ui;'>"))
            display(HTML("<h4 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;'>Forest Diagnostics</h4>"))

            r2_val = stats['oob_r2']
            r2_color = '#27ae60' if r2_val > 0.1 else '#e67e22' if r2_val > 0 else '#e74c3c'

            grid = [
                {'label': 'Observations (N)', 'value': stats['n_obs'], 'color': '#0984e3'},
                {'label': 'Features Evaluated', 'value': stats['n_features'], 'color': '#6c5ce7'},
                {'label': 'Out-Of-Bag R²', 'value': f"{r2_val:.3f}", 'color': r2_color},
                {'label': 'Total Null Heterogeneity (τ²)', 'value': f"{stats['null_tau2']:.4f}", 'color': '#e84393'}
            ]
            display(HTML(self.templates.info_grid(grid)))

            desc = """
            <p><b>Out-Of-Bag R²:</b> Represents the proportion of variance explained by the ensemble on unseen data. A negative R² indicates the model performs worse than simply guessing the mean effect size (common in noisy meta-analyses).</p>
            <p><b>Total Heterogeneity (τ²):</b> Extracted from the null (intercept-only) model or a standard DerSimonian-Laird estimator to avoid variance deflation. This value sets the Random-Effects weights.</p>
            """
            display(HTML(f"<div style='background: #f8f9fa; padding: 15px; border-radius: 6px;'>{desc}</div>"))
            display(HTML("</div>"))

    def render_publication(self):
        with self.tab_publication:
            clear_output()
            text = """
            <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt; line-height: 1.8; padding: 20px; background-color: #ffffff; border: 1px solid #eee;'>
            <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; margin-top: 0;'>Materials and Methods</h3>
            <p style='text-align: justify;'>
            <b>Exploratory Moderator Screening.</b> As an exploratory, hypothesis-generating step, we ranked the predictive importance of the candidate moderators using a precision-weighted Random Forest (Meta-Forest) algorithm. Unlike standard meta-forest implementations that rely on weighted bootstrap sampling (which risks systematically excluding small-sample studies), our approach utilizes uniform bootstrapping combined with precision-weighted split criteria [1]. This ensures the entire biological parameter space is represented while mathematically prioritizing the error minimization of highly precise studies.
            </p>
            <p style='text-align: justify;'>
            To account for between-study heterogeneity, the ensemble of 500 trees was fitted using random-effects weights ($W_i = 1 / (v_i + \\tau^2)$), where $\\tau^2$ was extracted from the intercept-only null model to prevent variance deflation [2]. Variable importance was assessed using out-of-sample permutation importance via grouped cross-validation (holding out whole studies) to prevent cluster-leakage. A weighted mean squared error (MSE) scoring function was used to avoid the cardinality bias inherent to standard Gini impurity metrics [3]. The importance ranking was used for exploration only: moderators selected this way were not tested for significance on the same data. Analyses were performed using the Co-Meta toolkit.
            </p>
            <h4 style='color: #34495e; margin-top: 20px; font-size: 11pt;'>References</h4>
            <ol style='font-size: 10pt; color: #555;'>
                <li>Scikit-learn developers. (202X). <i>scikit-learn: Machine Learning in Python</i>.</li>
                <li>Van Lissa, C. J. (2020). Small sample meta-analyses: Exploring heterogeneity using MetaForest. In R. van de Schoot &amp; M. Miočević (Eds.), <i>Small sample size solutions</i> (pp. 186-202). Routledge. https://doi.org/10.4324/9780429273872-16</li>
                <li>Strobl, C., Boulesteix, A. L., Zeileis, A., & Hothorn, T. (2007). Bias in random forest variable importance measures: Illustrations, sources and a solution. <i>BMC Bioinformatics</i>, 8(1), 25.</li>
            </ol>
            </div>
            """
            display(HTML(text))

    def display(self):
        display(widgets.VBox([self.stale_banner, self.header, self.controls, widgets.HTML("<br>"), self.tabs]))


# =============================================================================
# PART 3: VIEW (UI Layout & Rendering)
# =============================================================================
class MetaForestView:
    def __init__(self):
        self.templates = MetaForestHTMLTemplates()
        self.feature_checkboxes = {}   # name -> Checkbox
        self._create_widgets()
        self._create_tabs()
        self._build_layout()

    def _create_widgets(self):
        # --- Moderator picker toolbar -------------------------------------
        self.feature_search = widgets.Text(
            placeholder='🔍 Filter moderators...',
            layout=widgets.Layout(width='180px')
        )
        self.select_all_btn = widgets.Button(
            description='All', icon='check-square-o', button_style='info',
            tooltip='Select all currently shown moderators',
            layout=widgets.Layout(width='65px')
        )
        self.clear_all_btn = widgets.Button(
            description='Clear', icon='square-o', button_style='warning',
            tooltip='Clear every moderator',
            layout=widgets.Layout(width='75px')
        )

        toolbar = widgets.HBox(
            [self.feature_search, self.select_all_btn, self.clear_all_btn],
            layout=widgets.Layout(align_items='center', justify_content='space-between', width='340px', margin='5px 0 8px 0')
        )

        # --- Scrolling checkbox container ---------------------------------
        self.features_container = widgets.VBox(
            layout=widgets.Layout(
                width='340px',
                height='180px',
                overflow='auto',
                border='1px solid #ced4da',
                border_radius='4px',
                padding='8px',
                background_color='#f8f9fa' # Subtle gray background inside the box
            )
        )

        # --- Live selection counter ---------------------------------------
        self.selection_counter = widgets.HTML(
            value="",
            layout=widgets.Layout(margin='5px 0 0 5px')
        )

        # --- Assemble the picker column into a clean "Card" -------------
        self.feature_selector = widgets.VBox(
            [
                widgets.HTML("<b style='color:#2c3e50; font-family:system-ui, -apple-system, sans-serif; font-size:14px;'>Candidate Moderators:</b>"),
                toolbar,
                self.features_container,
                self.selection_counter,
            ],
            layout=widgets.Layout(
                width='365px',
                padding='12px',
                border='1px solid #e9ecef',
                border_radius='8px',
                background_color='#ffffff',
                box_shadow='0 2px 5px rgba(0,0,0,0.04)'
            )
        )

        # --- Run button ---------------------------------------------------
        self.run_button = widgets.Button(
            description='▶ Run Exploratory Meta-Forest',
            button_style='success',
            icon='tree',
            layout=widgets.Layout(width='300px', height='50px') # Slightly taller for emphasis
        )

        # --- View-internal wiring (UI only; never touches the Model) ------
        self.feature_search.observe(self._on_search_change, names='value')
        self.select_all_btn.on_click(self._on_select_all)
        self.clear_all_btn.on_click(self._on_clear_all)

    def _create_tabs(self):
        self.stale_banner = widgets.HTML(value="")
        if 'register_banner' in globals():
            register_banner('meta_forest', self.stale_banner)
        self.tab_ranking = widgets.Output()
        self.tab_diagnostics = widgets.Output()
        self.tab_publication = widgets.Output()
        self.tabs = widgets.Tab(children=[self.tab_ranking, self.tab_diagnostics, self.tab_publication])
        self.tabs.set_title(0, '📊 Importance Ranking')
        self.tabs.set_title(1, '⚙️ Model Diagnostics')
        self.tabs.set_title(2, '📝 Publication Text')

    def _build_layout(self):
        self.header = widgets.HTML("""
            <div style='font-family: system-ui, -apple-system, sans-serif;'>
                <h3 style='margin-bottom: 5px; color: #2E86AB;'>🌳 Exploratory Meta-Forest</h3>
                <p style='color: #6c757d; margin-top: 0; font-size: 13px;'>
                Select multiple candidate moderators. The algorithm will rank their predictive importance using a
                precision-weighted Random Forest. <b>Use this to explore which moderators may matter; moderators chosen here and then tested on the same data do not give confirmatory p-values.</b>
                </p>
            </div>
        """)
        # Align center makes the big run button sit perfectly beside the tall picker card
        self.controls = widgets.HBox(
            [self.feature_selector, self.run_button],
            layout=widgets.Layout(align_items='center', gap='35px', margin='10px 0 25px 0')
        )

    # ---------------------------------------------------------------------
    # Moderator picker logic (View-internal, no Model interaction)
    # ---------------------------------------------------------------------
    def update_feature_options(self, options):
        self.feature_checkboxes.clear()

        if not options:
            self.features_container.children = [
                widgets.HTML("<i style='color:#6c757d;'>Data not loaded</i>")
            ]
            self._refresh_counter()
            return

        for opt in options:
            cb = widgets.Checkbox(
                value=False,
                description=opt,
                indent=False,
                style={'description_width': 'initial'}, # Prevents text cutoff
                layout=widgets.Layout(width='95%', margin='0 0 4px 0')
            )
            cb.observe(self._on_checkbox_change, names='value')
            self.feature_checkboxes[opt] = cb

        self._apply_filter(self.feature_search.value)
        self._refresh_counter()

    def _apply_filter(self, query):
        q = (query or '').strip().lower()
        visible = [cb for name, cb in self.feature_checkboxes.items()
                   if q in name.lower()]
        if visible:
            self.features_container.children = visible
        else:
            self.features_container.children = [
                widgets.HTML("<i style='color:#6c757d; margin-left: 5px;'>No moderators match "
                             f"“{query}”</i>")
            ]

    def _refresh_counter(self):
        total = len(self.feature_checkboxes)
        sel = len(self.selected_features)
        color = '#27ae60' if sel else '#6c757d'
        self.selection_counter.value = (
            f"<span style='font-family:system-ui, -apple-system, sans-serif; font-size:12px; color:{color};'>"
            f"<b>{sel}</b> of {total} selected</span>"
        )

    def _on_search_change(self, change):
        self._apply_filter(change['new'])

    def _on_checkbox_change(self, change):
        self._refresh_counter()

    def _on_select_all(self, button):
        for cb in self.features_container.children:
            if isinstance(cb, widgets.Checkbox):
                cb.value = True

    def _on_clear_all(self, button):
        for cb in self.feature_checkboxes.values():
            cb.value = False

    @property
    def selected_features(self):
        return [name for name, cb in self.feature_checkboxes.items() if cb.value]

    # ---------------------------------------------------------------------
    # Output / rendering (unchanged except for guardrail text)
    # ---------------------------------------------------------------------
    def clear_all_tabs(self):
        for tab in [self.tab_ranking, self.tab_diagnostics, self.tab_publication]:
            tab.clear_output()

    def show_loading(self):
        with self.tab_ranking:
            display(HTML("<div style='padding: 15px; background-color: #e2e3e5; border-left: 4px solid #6c757d; border-radius: 4px; font-family: system-ui;'>⏳ <b>Growing Precision-Weighted Forest...</b> This may take a moment.</div>"))

    def show_error(self, error_msg):
        with self.tab_ranking:
            clear_output()
            display(HTML(f"<div style='color: #721c24; background-color: #f8d7da; padding: 15px; border-radius: 6px; border-left: 4px solid #dc3545; font-family: system-ui;'><b>❌ Error:</b> {error_msg}</div>"))

    def render_results(self, stats):
        df_imp = stats['importance_df']
        top_feature = df_imp.iloc[0]['Feature'] if len(df_imp) > 0 else "None"

        with self.tab_ranking:
            clear_output()
            display(HTML("<div style='padding: 20px;'>"))
            display(HTML(self.templates.dashboard_header("Variable Importance", "Random-Effects Weighted")))

            display(HTML(self.templates.gradient_card(
                title="Top Ranked Moderator",
                value=top_feature,
                subtitle="Highest Predictive Power"
            )))

            # Updated Text reflecting Subgroup vs Meta-Regression univariate routing
            interp = f"""
            <b>This is not a formal hypothesis test.</b> The Meta-Forest identifies which variables reduce prediction error the most.
            <br><br>
            <b>Recommendation:</b> Identify the top-ranked variable (e.g., <code>{top_feature}</code>). If it is categorical, test it formally in the <b>Subgroup Analysis</b> cell (which supports up to two interacting categories). If it is continuous, test it individually in the <b>Meta-Regression</b> cell to obtain valid p-values and account for hierarchical structure.
            """
            display(HTML(self.templates.guardrail_box(interp)))

            fig, ax = plt.subplots(figsize=(9, max(4, len(df_imp)*0.5)))
            colors = ['#27ae60'] + ['#bdc3c7'] * (len(df_imp) - 1)
            sns.barplot(
                data=df_imp, x='Importance', y='Feature',
                palette=colors, edgecolor='none', ax=ax
            )
            ax.errorbar(
                x=df_imp['Importance'], y=np.arange(len(df_imp)),
                xerr=df_imp['StdDev'], fmt='none', c='#2c3e50', capsize=4
            )
            ax.set_title('Permutation Importance (Weighted MSE Drop)', fontsize=12, fontweight='bold', pad=15)
            ax.set_xlabel('Importance Score (Higher = Better Predictor)', fontsize=10)
            ax.set_ylabel('')
            ax.grid(axis='x', linestyle='--', alpha=0.6)
            sns.despine(left=True)
            plt.tight_layout()
            plt.show()
            display(HTML("</div>"))

    def render_diagnostics(self, stats):
        with self.tab_diagnostics:
            clear_output()
            display(HTML("<div style='padding: 20px; font-family: system-ui;'>"))
            display(HTML("<h4 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;'>Forest Diagnostics</h4>"))

            r2_val = stats['oob_r2']
            r2_color = '#27ae60' if r2_val > 0.1 else '#e67e22' if r2_val > 0 else '#e74c3c'

            grid = [
                {'label': 'Observations (N)', 'value': stats['n_obs'], 'color': '#0984e3'},
                {'label': 'Features Evaluated', 'value': stats['n_features'], 'color': '#6c5ce7'},
                {'label': 'Out-Of-Bag R²', 'value': f"{r2_val:.3f}", 'color': r2_color},
                {'label': 'Total Null Heterogeneity (τ²)', 'value': f"{stats['null_tau2']:.4f}", 'color': '#e84393'}
            ]
            display(HTML(self.templates.info_grid(grid)))

            desc = """
            <p><b>Out-Of-Bag R²:</b> Represents the proportion of variance explained by the ensemble on unseen data. A negative R² indicates the model performs worse than simply guessing the mean effect size (common in noisy meta-analyses).</p>
            <p><b>Total Heterogeneity (τ²):</b> Extracted from the null (intercept-only) model or a standard DerSimonian-Laird estimator to avoid variance deflation. This value sets the Random-Effects weights.</p>
            """
            display(HTML(f"<div style='background: #f8f9fa; padding: 15px; border-radius: 6px;'>{desc}</div>"))
            display(HTML("</div>"))

    def render_publication(self):
        with self.tab_publication:
            clear_output()
            text = """
            <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt; line-height: 1.8; padding: 20px; background-color: #ffffff; border: 1px solid #eee;'>
            <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; margin-top: 0;'>Materials and Methods</h3>
            <p style='text-align: justify;'>
            <b>Exploratory Moderator Screening.</b> As an exploratory, hypothesis-generating step, we ranked the predictive importance of the candidate moderators using a precision-weighted Random Forest (Meta-Forest) algorithm. Unlike standard meta-forest implementations that rely on weighted bootstrap sampling (which risks systematically excluding small-sample studies), our approach utilizes uniform bootstrapping combined with precision-weighted split criteria [1]. This ensures the entire biological parameter space is represented while mathematically prioritizing the error minimization of highly precise studies.
            </p>
            <p style='text-align: justify;'>
            To account for between-study heterogeneity, the ensemble of 500 trees was fitted using random-effects weights ($W_i = 1 / (v_i + \\tau^2)$), where $\\tau^2$ was extracted from the intercept-only null model to prevent variance deflation [2]. Variable importance was assessed using out-of-sample permutation importance via grouped cross-validation (holding out whole studies) to prevent cluster-leakage. A weighted mean squared error (MSE) scoring function was used to avoid the cardinality bias inherent to standard Gini impurity metrics [3]. The importance ranking was used for exploration only: moderators selected this way were not tested for significance on the same data. Analyses were performed using the Co-Meta toolkit.
            </p>
            <h4 style='color: #34495e; margin-top: 20px; font-size: 11pt;'>References</h4>
            <ol style='font-size: 10pt; color: #555;'>
                <li>Scikit-learn developers. (202X). <i>scikit-learn: Machine Learning in Python</i>.</li>
                <li>Van Lissa, C. J. (2020). Small sample meta-analyses: Exploring heterogeneity using MetaForest. In R. van de Schoot &amp; M. Miočević (Eds.), <i>Small sample size solutions</i> (pp. 186-202). Routledge. https://doi.org/10.4324/9780429273872-16</li>
                <li>Strobl, C., Boulesteix, A. L., Zeileis, A., & Hothorn, T. (2007). Bias in random forest variable importance measures: Illustrations, sources and a solution. <i>BMC Bioinformatics</i>, 8(1), 25.</li>
            </ol>
            </div>
            """
            display(HTML(text))

    def display(self):
        display(widgets.VBox([self.stale_banner, self.header, self.controls, widgets.HTML("<br>"), self.tabs]))

# =============================================================================
# PART 3: CONTROLLER
# =============================================================================

class MetaForestController:
    def __init__(self, model, view):
        self.model = model
        self.view = view
        self._bind_events()
        self._initialize()

    def _bind_events(self):
        self.view.run_button.on_click(self._on_run_click)

    def _initialize(self):
        df = self.model.get_analysis_data()
        features = self.model.discover_features(df)
        self.view.update_feature_options(features)

    def _on_run_click(self, button):
        self.view.clear_all_tabs()
        self.view.show_loading()

        # --> FIX: Use the selected_features property, NOT feature_selector.value
        selected_features = self.view.selected_features

        # Check if the user actually selected anything before running
        if not selected_features:
            self.view.show_error("Please check at least one moderator box before running.")
            return

        success = self.model.run_analysis(selected_features)

        if not success:
            self.view.show_error(self.model.error)
            return

        stats = self.model.results
        self.view.render_results(stats)
        self.view.render_diagnostics(stats)
        self.view.render_publication()

    def run(self):
        self.view.display()


# =============================================================================
# EXECUTION
# =============================================================================

_mf_model = MetaForestModel()
_mf_view = MetaForestView()
_mf_controller = MetaForestController(_mf_model, _mf_view)

_mf_controller.run()

In [ ]:
#@title 📉 17. Publication Bias: Egger's Test & Trim-and-Fill
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")


# =============================================================================
# PUBLICATION BIAS ANALYSIS
# =============================================================================

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import norm, t, rankdata
import statsmodels.api as sm
import datetime
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
from scipy.stats import t as t_dist
# =============================================================================
# Business Logic & Data Processing
# =============================================================================

class PublicationBiasModel:
    """
    Model: Contains all mathematical logic for publication bias assessment.
    Preserves exact calculations from original code.
    """

    def __init__(self):
        self.egger_results = None
        self.trimfill_results = None
        self.analysis_config = None
        self.analysis_data = None

    def initialize(self):
        """Load data and configuration from global context"""
        if 'ANALYSIS_CONFIG' not in globals():
            raise ValueError("ANALYSIS_CONFIG not found. Run Steps 2–6 (data and effect sizes) first.")

        self.analysis_config = globals()['ANALYSIS_CONFIG']

        if 'three_level_results' not in self.analysis_config:
            raise ValueError("Three-level results not found. Run Step 7 (Overall Meta-Analysis) first.")

        # Load analysis data
        if 'analysis_data' in self.analysis_config:
            self.analysis_data = self.analysis_config['analysis_data']
        elif 'data_filtered' in globals():
            self.analysis_data = globals()['data_filtered']
        else:
            raise ValueError("No analysis data found.")

        self.effect_col = self.analysis_config.get('effect_col', 'hedges_g')
        self.var_col = self.analysis_config.get('var_col', 'Vg')
        self.se_col = self.analysis_config.get('se_col', 'SE_g')

        return self

    def get_dataset_info(self):
        """Return sample size information"""
        n_obs = len(self.analysis_data)
        n_studies = self.analysis_data['id'].nunique()
        return n_obs, n_studies

    # -------------------------------------------------------------------------
    # EGGER'S TEST
    # -------------------------------------------------------------------------

    def _study_blocks(self, df, effect_col, var_col):
        """(study id, rows, full sampling V, diagonal V) for every study.

        The full V comes from the shared-control matrices (Step 6), sliced by
        row identity as in the overall model; without them it is diagonal."""
        cfg = self.analysis_config if isinstance(self.analysis_config, dict) else {}
        vcv_dict = cfg.get('vcv_matrices') or (globals().get('ANALYSIS_CONFIG') or {}).get('vcv_matrices') or {}
        if vcv_dict and 'attach_vcv_row_pos' in globals() and '_vcv_row_pos' not in df.columns:
            df = attach_vcv_row_pos(df, reference_df=df)   # df: the data the matrices were built from
        out = []
        for sid, g in df.groupby('id'):
            V_diag = np.diag(g[var_col].values.astype(np.float64))
            V_full = cluster_vcv(vcv_dict, sid, g, var_col)[0] if vcv_dict else V_diag
            out.append((sid, g, V_full, V_diag))
        return out

    def run_eggers_test(self):
        """
        Run Egger's Test using 3-Level Meta-Regression on SE.
        Calls global _run_robust_eggers_test or implements it using global helpers.
        """
        try:
            estimates = self._run_robust_eggers_test(
                self.analysis_data,
                self.effect_col,
                self.var_col,
                self.se_col
            )

            if not estimates:
                raise ValueError("Egger's test optimization failed")

            # Model: y = b0 + b1*SE. Asymmetry (small-study effect) is tested
            # by the SLOPE b1; the intercept b0 is the effect expected at
            # SE = 0 (the PET-type corrected estimate), not the bias test.
            intercept = estimates['betas'][0]
            slope_val = estimates['betas'][1]
            se_intercept = estimates['se_betas'][0]
            se_slope = estimates['se_betas'][1]
            t_stat = slope_val / se_slope
            df = estimates['df']
            p_value = 2 * (1 - t.cdf(abs(t_stat), df))

            # Store results ('se', 't_stat', 'df', 'p_value' refer to the slope test)
            self.egger_results = {
                'intercept': intercept,
                'se_intercept': se_intercept,
                'slope': slope_val,
                'se': se_slope,
                't_stat': t_stat,
                'df': df,
                'p_value': p_value,
                'used_robust': estimates.get('used_robust', False),
                'estimates': estimates,
                'timestamp': datetime.datetime.now()
            }

            # Save to global config (required for plotting cells)
            self.analysis_config['funnel_results'] = {
                'beta_slope': slope_val,
                'timestamp': datetime.datetime.now(),
                'intercept': intercept,
                'se': se_slope,
                'p_value': p_value,
                'estimates': estimates,
                't_stat': t_stat,
                'df': df,
                'used_robust': estimates.get('used_robust', False),
                'sampling_v': estimates.get('sampling_v'),
                'n_obs': int(len(self.analysis_data)),
                'n_studies': int(self.analysis_data['id'].nunique()),
            }

            return self.egger_results

        except Exception as e:
            raise RuntimeError(f"Error running Egger's test: {str(e)}")

    def _run_robust_eggers_test(self, analysis_data, effect_col, var_col, se_col):
        """
        Egger's regression test: three-level REML meta-regression of the
        effect size on its standard error (shared engine, Cell 1b), with
        CR2 / Satterthwaite inference for the coefficients.
        """
        y_all, V_all, V_diag, X_all = [], [], [], []
        for _, group, V_full, V_d in self._study_blocks(analysis_data, effect_col, var_col):
            y_all.append(group[effect_col].values.astype(np.float64))
            V_all.append(V_full)
            V_diag.append(V_d)
            # Explicit [1, SE] design: sm.add_constant silently skips the
            # intercept when a study's SE column is constant (e.g. k_i = 1).
            se_i = group[se_col].values.astype(np.float64)
            X_all.append(np.column_stack([np.ones_like(se_i), se_i]))

        M_studies = len(y_all)
        p_params = 2
        has_cov = any(np.count_nonzero(V - np.diag(np.diag(V))) for V in V_all)

        try:
            fit = fit_blocks(y_all, V_all, X_all, V_fallback_blocks=V_diag if has_cov else None)
        except (ValueError, RuntimeError, np.linalg.LinAlgError):
            return None

        est = fit_estimates(fit)
        est['sampling_v'] = ('full (shared controls)' if has_cov and not fit.v_fallback
                             else 'diagonal')
        # Model-based fallback df if the CR2 step below fails
        est['df'] = float(max(1, M_studies - p_params))
        est['used_robust'] = False

        try:
            var_robust, df_robust = fit_cr2(fit)
            est['var_betas_robust'] = var_robust
            est['se_betas'] = np.sqrt(np.diag(var_robust))
            # Egger's test of funnel asymmetry is the slope on SE (index 1)
            est['df'] = float(df_robust[1])
            est['dfs'] = np.asarray(df_robust, dtype=np.float64)
            est['used_robust'] = True
        except Exception as e:
            warnings.warn(f"Egger's test: CR2 step failed ({e}); using model-based SEs.")

        dfs = est.get('dfs', np.full(p_params, est['df']))
        est['t_values'] = est['betas'] / est['se_betas']
        est['p_values'] = 2.0 * t.sf(np.abs(est['t_values']), df=dfs)
        return est


    # -------------------------------------------------------------------------
    # TRIM-AND-FILL
    # -------------------------------------------------------------------------

    def run_trimfill_analysis(self, estimator='L0', side='auto', max_iter=100):
        """
        Duval & Tweedie Trim-and-Fill
        Upgraded to use RE centering and 3-Level Robust Final Estimation
        """
        estimator = str(estimator).upper()
        if estimator not in ('L0', 'R0'):
            raise ValueError(f"Trim-and-fill estimator must be 'L0' or 'R0', got {estimator!r}")
        alpha = (self.analysis_config or {}).get('global_settings', {}).get('alpha', 0.05)
        try:
            yi = self.analysis_data[self.effect_col].values
            vi = self.analysis_data[self.var_col].values
            ni = len(yi)

            # Sort by effect size
            sort_indices = np.argsort(yi)
            yi = yi[sort_indices]
            vi = vi[sort_indices]

            # Side of the MISSING studies. 'auto' uses metafor's rule: fit the
            # DL meta-regression yi ~ sqrt(vi); a negative slope means studies
            # are missing on the right, otherwise on the left.
            side_rule = 'user-specified'
            if side == 'auto':
                side = self._metafor_side(yi, vi)
                side_rule = 'auto (sign of the slope of yi on SE, as in metafor)'

            # Iterative trimming
            k0 = 0
            iter_safe = 0

            while iter_safe < max_iter:
                n_curr = ni - k0

                if side == 'left':
                    yi_curr = yi[:n_curr]
                    vi_curr = vi[:n_curr]
                else:
                    yi_curr = yi[k0:]
                    vi_curr = vi[k0:]

                # --- NEW: Use Random-Effects (DL) mean for centering ---
                wi_fe = 1.0 / vi_curr
                sum_wi_fe = np.sum(wi_fe)
                mu_fe = np.sum(wi_fe * yi_curr) / sum_wi_fe
                Q = np.sum(wi_fe * (yi_curr - mu_fe)**2)
                df_Q = max(0, len(yi_curr) - 1)
                C = sum_wi_fe - np.sum(wi_fe**2) / sum_wi_fe
                tau2 = max(0, (Q - df_Q) / C) if C > 0 else 0

                wi_re = 1.0 / (vi_curr + tau2)
                pooled_curr = np.sum(wi_re * yi_curr) / np.sum(wi_re)
                # -------------------------------------------------------

                residuals = yi - pooled_curr
                signed_res = residuals if side == 'left' else -residuals
                abs_res = np.abs(signed_res)
                # Ties broken by order, as metafor's rank(ties.method = "first")
                ranks = rankdata(abs_res, method='ordinal')

                if estimator == 'L0':
                    Sn = np.sum(np.where(signed_res > 0, ranks, 0))
                    k0_new = int(round((4 * Sn - ni * (ni + 1)) / (2 * ni - 1)))
                else:  # R0: length of the rightmost run of positive ranks, minus 1
                    neg_ranks = ranks[signed_res < 0]
                    k0_new = int(ni - (neg_ranks.max() if neg_ranks.size else 0) - 1)
                k0_new = max(0, k0_new)

                if k0_new == k0:
                    break

                k0 = k0_new
                k0 = min(k0, ni - 2)
                iter_safe += 1

            # Filling phase
            if k0 > 0:
                if side == 'left':
                    idx_fill = slice(ni - k0, ni)
                else:
                    idx_fill = slice(0, k0)

                yi_excess = yi[idx_fill]
                vi_excess = vi[idx_fill]

                yi_filled = 2 * pooled_curr - yi_excess
                vi_filled = vi_excess

                yi_final = np.concatenate([yi, yi_filled])
                vi_final = np.concatenate([vi, vi_filled])
            else:
                yi_final = yi
                vi_final = vi
                yi_filled = []
                vi_filled = []

            # --- Final estimation: three-level model on the filled data ---
            # (shared engine, Cell 1b; CR2 / Satterthwaite inference)
            # Observed studies keep their full sampling V (shared controls);
            # each imputed study is its own cluster with a diagonal V.
            blocks = self._study_blocks(self.analysis_data, self.effect_col, self.var_col)
            y_all_f = [g[self.effect_col].values.astype(np.float64) for _, g, _, _ in blocks]
            V_all_f = [Vf for _, _, Vf, _ in blocks]
            V_diag_f = [Vd for _, _, _, Vd in blocks]
            for y_fill, v_fill in zip(np.atleast_1d(yi_filled), np.atleast_1d(vi_filled)):
                y_all_f.append(np.array([float(y_fill)]))
                V_all_f.append(np.array([[float(v_fill)]]))
                V_diag_f.append(np.array([[float(v_fill)]]))
            has_cov = any(np.count_nonzero(V - np.diag(np.diag(V))) for V in V_all_f)

            try:
                fit = fit_blocks(y_all_f, V_all_f, V_fallback_blocks=V_diag_f if has_cov else None)
            except (ValueError, RuntimeError, np.linalg.LinAlgError):
                fit = None

            # Use Fallback values if the fit fails completely
            if fit is None:
                pooled_final = pooled_curr
                se_final = np.sqrt(1 / np.sum(wi_re))
                z_crit = norm.ppf(1 - alpha / 2)
                ci_lo_fill = pooled_final - z_crit * se_final
                ci_hi_fill = pooled_final + z_crit * se_final
                warnings.warn("Trim-and-fill: 3-level refit failed; adjusted estimate uses the DL model with a z-based CI.")
            else:
                pooled_final = float(fit.betas[0])
                se_final = float(np.sqrt(fit.cov_beta[0, 0]))
                df_final = float(max(1, len(y_all_f) - 1))
                try:
                    var_robust, df_robust = fit_cr2(fit)
                    se_final = float(np.sqrt(var_robust[0, 0]))
                    df_final = float(df_robust[0])
                except Exception as e:
                    warnings.warn(f"Trim-and-fill: CR2 step failed ({e}); using model-based SEs.")

                # CI at the chosen alpha (Satterthwaite df when CR2 ran)
                t_crit = t.ppf(1 - alpha / 2, df_final)
                ci_lo_fill = pooled_final - t_crit * se_final
                ci_hi_fill = pooled_final + t_crit * se_final

            # Extract Original Values from Cell 7 for comparison
            three_level = self.analysis_config.get('three_level_results', self.analysis_config.get('overall_results'))
            pooled_orig = three_level.get('pooled_effect', three_level.get('pooled_effect_random'))
            se_orig = three_level.get('se', three_level.get('pooled_SE_random_reported'))
            ci_lo_orig = three_level.get('ci_lower', three_level.get('ci_lower_random_reported'))
            ci_hi_orig = three_level.get('ci_upper', three_level.get('ci_upper_random_reported'))

            self.trimfill_results = {
                'k0': k0,
                'estimator': estimator,
                'side': side,
                'side_rule': side_rule,
                'pooled_original': pooled_orig,
                'se_original': se_orig,
                'ci_lower_original': ci_lo_orig,
                'ci_upper_original': ci_hi_orig,
                'pooled_filled': pooled_final,
                'se_filled': se_final,
                'ci_lower_filled': ci_lo_fill,
                'ci_upper_filled': ci_hi_fill,
                'yi_filled': yi_filled,
                'vi_filled': vi_filled if k0 > 0 else [],
                'yi_combined': yi_final,
                'vi_combined': vi_final,
                # imputed studies are mirror images around this DL estimate of the trimmed data
                'mirror_center': float(pooled_curr),
                'tau2_trimmed': float(tau2),
                'n_observed': int(ni),
                'alpha': float(alpha),
                'adjusted_model': '3-Level' if fit is not None else 'DL (fallback)',
                'timestamp': datetime.datetime.now(),
            }

            self.analysis_config['trimfill_results'] = self.trimfill_results

            return self.trimfill_results

        except Exception as e:
            raise RuntimeError(f"Error running Trim-and-Fill: {str(e)}")

    @staticmethod
    def _metafor_side(yi, vi):
        """Side on which studies are missing, chosen as metafor::trimfill does:
        DerSimonian-Laird meta-regression of yi on sqrt(vi); slope < 0 ->
        'right', otherwise 'left'."""
        yi = np.asarray(yi, dtype=np.float64)
        vi = np.asarray(vi, dtype=np.float64)
        k = len(yi)
        X = np.column_stack([np.ones(k), np.sqrt(vi)])
        # DL residual heterogeneity for a model with moderators (metafor):
        #   tau2 = (RSS - (k - p)) / tr(P),  P = W - W X (X'WX)^-1 X'W
        W = np.diag(1.0 / vi)
        P = W - W @ X @ np.linalg.pinv(X.T @ W @ X) @ X.T @ W
        tau2 = max(0.0, (float(yi @ P @ yi) - (k - X.shape[1])) / float(np.trace(P)))
        w = 1.0 / (vi + tau2)
        beta = np.linalg.pinv(X.T @ (w[:, None] * X)) @ (X.T @ (w * yi))
        return 'right' if beta[1] < 0 else 'left'

    def get_combined_assessment(self):
        """Generate combined interpretation of both tests"""
        if not self.egger_results or not self.trimfill_results:
            return None

        egger_p = self.egger_results['p_value']
        k0 = self.trimfill_results['k0']

        egger_bias = egger_p < 0.10
        tf_bias = k0 > 0

        # Egger's test uses the conventional 0.10 level (Egger et al. 1997).
        # The labels describe what the tests detect -- funnel asymmetry /
        # small-study effects -- not a verified risk of publication bias.
        if egger_bias and tf_bias:
            assessment = "ASYMMETRY DETECTED"
            color = "#dc3545"
            icon = "⚠️"
            message = "Both tests indicate funnel asymmetry (small-study effects)"
        elif egger_bias or tf_bias:
            assessment = "MIXED EVIDENCE"
            color = "#ffc107"
            icon = "⚡"
            message = "One test indicates funnel asymmetry (small-study effects)"
        else:
            assessment = "NO ASYMMETRY DETECTED"
            color = "#28a745"
            icon = "✓"
            message = ("Neither test detected funnel asymmetry; this does not rule out "
                       "publication bias, as both tests have low power")

        return {
            'assessment': assessment,
            'color': color,
            'icon': icon,
            'message': message,
            'egger_bias': egger_bias,
            'tf_bias': tf_bias
        }

# =============================================================================
# PART 2: VIEW - UI Components & HTML Generation
# =============================================================================

class PublicationBiasView:
    """
    View: Handles all UI rendering, widgets, and HTML generation.
    No business logic - only presentation.
    """

    def __init__(self):
        self.setup_widgets()

    def setup_widgets(self):
        """Create all UI widgets"""
        # Output tabs
        self.tab_egger = widgets.Output()
        self.tab_trimfill = widgets.Output()
        self.tab_combined = widgets.Output()
        self.tab_publication = widgets.Output()
        self.tab_export = widgets.Output()

        self.tabs = widgets.Tab(children=[
            self.tab_egger,
            self.tab_trimfill,
            self.tab_combined,
            self.tab_publication,
            self.tab_export
        ])

        self.tabs.set_title(0, '📊 Egger\'s Test')
        self.tabs.set_title(1, '📉 Trim-and-Fill')
        self.tabs.set_title(2, '🔍 Combined Assessment')
        self.tabs.set_title(3, '📝 Publication Text')
        self.tabs.set_title(4, '💾 Export')

        # Run button
        self.run_button = widgets.Button(
            description='▶ Run Publication Bias Analysis',
            button_style='primary',
            icon='play',
            layout=widgets.Layout(width='300px')
        )

        # Export button
        self.export_button = widgets.Button(
            description="📥 Download Bias Report",
            button_style='info',
            icon='file-excel',
            layout=widgets.Layout(width='300px', height='40px')
        )

    def clear_all_tabs(self):
        """Clear output from all tabs"""
        for tab in [self.tab_egger, self.tab_trimfill, self.tab_combined,
                    self.tab_publication, self.tab_export]:
            tab.clear_output()

    def display_error(self, tab, message):
        """Display error message in specified tab"""
        with tab:
            display(HTML(f"<div style='color: red;'>❌ {message}</div>"))

    # -------------------------------------------------------------------------
    # Egger's Test Visualization
    # -------------------------------------------------------------------------

    def render_eggers_results(self, egger_results, n_obs, n_studies):
        """Render Egger's test results"""
        with self.tab_egger:
            clear_output()

            slope = egger_results['slope']
            se = egger_results['se']
            p_value = egger_results['p_value']
            t_stat = egger_results['t_stat']
            df = egger_results['df']

            sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
            color = "#dc3545" if p_value < _alpha() else "#28a745"

            html = f"""
            <div style='padding: 20px;'>
            <h2 style='color: #2c3e50; margin-bottom: 20px;'>Egger's Regression Test for Funnel Plot Asymmetry</h2>

            <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 10px; color: white; margin-bottom: 20px;'>
                <div style='text-align: center;'>
                    <div style='font-size: 0.9em; margin-bottom: 10px;'>ASYMMETRY TEST</div>
                    <h1 style='margin: 0; font-size: 2.5em;'>{"Significant" if p_value < _alpha() else "Not Significant"}</h1>
                    <p style='margin: 10px 0 0 0; font-size: 1.2em;'>p {p_value:.4g} {sig}</p>
                </div>
            </div>

            <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 20px;'>
                <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px; border-left: 4px solid #007bff;'>
                    <div style='color: #6c757d; font-size: 0.9em;'>Slope on SE (β₁)</div>
                    <div style='font-size: 1.5em; font-weight: bold;'>{slope:.4f}</div>
                    <div style='color: #6c757d; font-size: 0.85em; margin-top: 5px;'>SE = {se:.4f}</div>
                </div>
                <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px; border-left: 4px solid {color};'>
                    <div style='color: #6c757d; font-size: 0.9em;'>P-value</div>
                    <div style='font-size: 1.5em; font-weight: bold; color: {color};'>{p_value:.4g}</div>
                    <div style='color: #6c757d; font-size: 0.85em; margin-top: 5px;'>t({df:.1f}) = {t_stat:.2f}</div>
                </div>
            </div>

            <div style='background-color: #e7f3ff; padding: 20px; border-radius: 5px; margin-bottom: 20px;'>
                <h4 style='margin-top: 0; color: #2c3e50;'>Interpretation</h4>
                <p style='margin: 0; font-size: 1.05em;'>
            """

            if p_value < _alpha():
                html += f"""<b style='color: #dc3545;'>⚠️ Significant funnel plot asymmetry detected.</b><br>
                    This suggests potential publication bias or other sources of small-study effects.
                    Smaller studies tend to report {'larger' if slope > 0 else 'smaller'} effect sizes than larger studies.
                    <br><br>However, asymmetry can also arise from genuine heterogeneity, methodological differences, or chance."""
            elif p_value < 0.10:
                html += f"""<b style='color: #ffc107;'>⚡ Marginal evidence of asymmetry (p < 0.10).</b><br>
                    Some suggestion of funnel plot asymmetry. Consider examining the funnel plot visually (Step 20)."""
            else:
                html += f"""<b style='color: #28a745;'>✓ No significant funnel plot asymmetry.</b><br>
                    Egger's test detected no small-study effect. This does not rule out publication bias (limited power)."""

            html += f"""
                </p>
            </div>

            <h3 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;'>Test Details</h3>
            <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px;'>
                <p style='margin: 5px 0;'><b>Method:</b> Three-level meta-regression (robust to within-study dependencies)</p>
                <p style='margin: 5px 0;'><b>Predictor:</b> Standard error (SE)</p>
                <p style='margin: 5px 0;'><b>Outcome:</b> Effect size</p>
                <p style='margin: 5px 0;'><b>Sample:</b> {n_obs} observations from {n_studies} studies</p>
                <p style='margin: 5px 0;'><b>Test:</b> slope on SE (β₁), {'CR2 robust SE' if egger_results.get('used_robust') else 'model-based SE'}</p>
                <p style='margin: 5px 0;'><b>Degrees of Freedom:</b> {df:.1f}</p>
            </div>

            <div style='background-color: #fff3cd; padding: 15px; border-radius: 5px; border-left: 4px solid #ffc107; margin-top: 20px;'>
                <p style='margin: 0;'><b>📊 Next Step:</b> Use Step 20 (Funnel Plot) to generate the funnel plot for visual inspection of asymmetry.</p>
            </div>
            </div>
            """

            display(HTML(html))

    # -------------------------------------------------------------------------
    # Trim-and-Fill Visualization
    # -------------------------------------------------------------------------

    def render_trimfill_results(self, tf_results, n_obs, n_studies):
        """Render Trim-and-Fill results"""
        with self.tab_trimfill:
            clear_output()

            k0 = tf_results['k0']
            side = tf_results['side']
            pooled_orig = tf_results['pooled_original']
            pooled_fill = tf_results['pooled_filled']

            pct_change = abs((pooled_fill - pooled_orig) / pooled_orig) * 100 if pooled_orig != 0 else 0

            html = f"""
            <div style='padding: 20px;'>
            <h2 style='color: #2c3e50; margin-bottom: 20px;'>Trim-and-Fill Analysis</h2>

            <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 10px; color: white; margin-bottom: 20px;'>
                <div style='text-align: center;'>
                    <div style='font-size: 0.9em; margin-bottom: 10px;'>ESTIMATED MISSING STUDIES</div>
                    <h1 style='margin: 0; font-size: 3em;'>{k0}</h1>
                    <p style='margin: 10px 0 0 0; font-size: 1.2em;'>{side.capitalize()} side</p>
                </div>
            </div>

            <div style='background-color: #e7f3ff; padding: 20px; border-radius: 5px; margin-bottom: 20px;'>
                <h4 style='margin-top: 0; color: #2c3e50;'>Interpretation</h4>
                <p style='margin: 0; font-size: 1.05em;'>
            """

            if k0 == 0:
                html += """<b style='color: #28a745;'>✓ No missing studies detected.</b><br>
                    Trim-and-fill detected no asymmetry (it assumes independent effect sizes; limited power)."""
            else:
                html += f"""<b style='color: #dc3545;'>⚠️ {k0} potentially missing studies estimated.</b><br>
                    After imputation, the effect changes by {pct_change:.1f}%. This suggests {'substantial' if pct_change > 20 else 'moderate' if pct_change > 10 else 'minimal'} potential impact of publication bias."""

            html += f"""
                </p>
            </div>

            <h3 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;'>Effect Size Comparison</h3>
            <table style='width: 100%; border-collapse: collapse; margin: 20px 0;'>
                <thead style='background-color: #2c3e50; color: white;'>
                    <tr>
                        <th style='padding: 12px; text-align: left; border: 1px solid #dee2e6;'>Estimate</th>
                        <th style='padding: 12px; text-align: center; border: 1px solid #dee2e6;'>Effect</th>
                        <th style='padding: 12px; text-align: center; border: 1px solid #dee2e6;'>SE</th>
                        <th style='padding: 12px; text-align: center; border: 1px solid #dee2e6;'>95% CI</th>
                    </tr>
                </thead>
                <tbody>
                    <tr style='background-color: #f8f9fa;'>
                        <td style='padding: 10px; border: 1px solid #dee2e6;'><b>Original</b></td>
                        <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>{tf_results['pooled_original']:.4f}</td>
                        <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>{tf_results['se_original']:.4f}</td>
                        <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>[{tf_results['ci_lower_original']:.4f}, {tf_results['ci_upper_original']:.4f}]</td>
                    </tr>
                    <tr>
                        <td style='padding: 10px; border: 1px solid #dee2e6;'><b>Adjusted</b></td>
                        <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>{tf_results['pooled_filled']:.4f}</td>
                        <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>{tf_results['se_filled']:.4f}</td>
                        <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>[{tf_results['ci_lower_filled']:.4f}, {tf_results['ci_upper_filled']:.4f}]</td>
                    </tr>
                    <tr style='background-color: #fff3cd;'>
                    <td style='padding: 10px; border: 1px solid #dee2e6;'><b>Change</b></td>
                    <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6; font-weight: bold;'>{pooled_fill - pooled_orig:+.4f}</td>
                    <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>—</td>
                    <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>{pct_change:.1f}% change</td>
                    </tr>
                    </tbody>
                    </table>
            <h3 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;'>Method Details</h3>
            <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px;'>
                <p style='margin: 5px 0;'><b>Method:</b> Duval & Tweedie L0 estimator</p>
                <p style='margin: 5px 0;'><b>Side of missing studies:</b> {side.capitalize()} ({tf_results.get('side_rule', 'automatically detected')})</p>
                <p style='margin: 5px 0;'><b>Original studies:</b> {n_obs} observations from {n_studies} studies</p>
                <p style='margin: 5px 0;'><b>Imputed studies:</b> {k0}</p>
                <p style='margin: 5px 0;'><b>Total after filling:</b> {n_obs + k0}</p>
            </div>

            <div style='background-color: #fff3cd; padding: 15px; border-radius: 5px; border-left: 4px solid #ffc107; margin-top: 20px;'>
                <p style='margin: 0;'><b>📊 Next Step:</b> Use Step 18 (Trim-and-Fill Plot) to visualize the trim-and-fill funnel plot showing original and imputed studies.</p>
            </div>
            </div>
            """

            display(HTML(html))

    # -------------------------------------------------------------------------
    # Combined Assessment Visualization
    # -------------------------------------------------------------------------

    def render_combined_assessment(self, assessment, egger_results, tf_results, n_studies):
        """Render combined assessment of both tests"""
        with self.tab_combined:
            clear_output()

            egger_p = egger_results['p_value']
            k0 = tf_results['k0']
            pct_change = abs((tf_results['pooled_filled'] - tf_results['pooled_original']) / tf_results['pooled_original']) * 100 if tf_results['pooled_original'] != 0 else 0

            html = f"""
            <div style='padding: 20px;'>
            <h2 style='color: #2c3e50; margin-bottom: 20px;'>Combined Publication Bias Assessment</h2>

            <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 10px; color: white; margin-bottom: 20px;'>
                <div style='text-align: center;'>
                    <div style='font-size: 0.9em; margin-bottom: 10px;'>OVERALL ASSESSMENT</div>
                    <h1 style='margin: 0; font-size: 2.5em;'>{assessment['icon']} {assessment['assessment']}</h1>
                    <p style='margin: 10px 0 0 0; font-size: 1.2em;'>{assessment['message']}</p>
                </div>
            </div>

            <h3 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;'>Summary of Tests</h3>
            <table style='width: 100%; border-collapse: collapse; margin: 20px 0;'>
                <thead style='background-color: #2c3e50; color: white;'>
                    <tr>
                        <th style='padding: 12px; text-align: left; border: 1px solid #dee2e6;'>Test</th>
                        <th style='padding: 12px; text-align: center; border: 1px solid #dee2e6;'>Result</th>
                        <th style='padding: 12px; text-align: center; border: 1px solid #dee2e6;'>Evidence of Bias</th>
                    </tr>
                </thead>
                <tbody>
                    <tr style='background-color: #f8f9fa;'>
                        <td style='padding: 10px; border: 1px solid #dee2e6;'><b>Egger's Test</b></td>
                        <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>p = {egger_p:.4g}</td>
                        <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>
                            <span style='color: {"#dc3545" if assessment['egger_bias'] else "#28a745"}; font-weight: bold;'>
                                {"Yes" if assessment['egger_bias'] else "No"}
                            </span>
                        </td>
                    </tr>
                    <tr>
                        <td style='padding: 10px; border: 1px solid #dee2e6;'><b>Trim-and-Fill</b></td>
                        <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>k₀ = {k0}</td>
                        <td style='padding: 10px; text-align: center; border: 1px solid #dee2e6;'>
                            <span style='color: {"#dc3545" if assessment['tf_bias'] else "#28a745"}; font-weight: bold;'>
                                {"Yes" if assessment['tf_bias'] else "No"}
                            </span>
                        </td>
                    </tr>
                </tbody>
            </table>

            <h3 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;'>Detailed Interpretation</h3>
            <div style='background-color: #e7f3ff; padding: 20px; border-radius: 5px; margin-bottom: 20px;'>
            """

            if assessment['egger_bias'] and assessment['tf_bias']:
                html += f"""
                <p style='margin: 5px 0;'><b>Convergent Evidence:</b> Both tests indicate potential publication bias.</p>
                <ul style='margin: 10px 0;'>
                    <li>Egger's test detected significant funnel plot asymmetry (p = {egger_p:.4g})</li>
                    <li>Trim-and-fill estimated {k0} missing studies</li>
                    <li>Effect size changed by {pct_change:.1f}% after adjustment</li>
                </ul>
                <p style='margin: 10px 0 0 0;'><b>Recommendation:</b> Interpret main results with caution. Consider reporting both original and adjusted estimates. Discuss potential impact of publication bias on conclusions.</p>
                """
            elif assessment['egger_bias'] or assessment['tf_bias']:
                which = "Egger's test" if assessment['egger_bias'] else "Trim-and-fill"
                which_not = "Trim-and-fill" if assessment['egger_bias'] else "Egger's test"
                html += f"""
                <p style='margin: 5px 0;'><b>Mixed Evidence:</b> {which} suggests bias, but {which_not} does not.</p>
                <ul style='margin: 10px 0;'>
                    <li>{'Funnel plot asymmetry detected' if assessment['egger_bias'] else 'No significant asymmetry'} (Egger p = {egger_p:.4g})</li>
                    <li>{k0} missing studies estimated (Trim-and-fill)</li>
                </ul>
                <p style='margin: 10px 0 0 0;'><b>Recommendation:</b> Exercise appropriate caution. Differences between tests may reflect limited power or different aspects of bias. Visual inspection of funnel plots recommended.</p>
                """
            else:
                html += f"""
                <p style='margin: 5px 0;'><b>Consistent Evidence:</b> Neither test detected funnel asymmetry.</p>
                <ul style='margin: 10px 0;'>
                    <li>No significant funnel plot asymmetry (Egger p = {egger_p:.4g})</li>
                    <li>No missing studies estimated (k₀ = 0)</li>
                </ul>
                <p style='margin: 10px 0 0 0;'><b>Recommendation:</b> Report both tests. A non-significant result does not show that publication bias is absent: these tests have low power, especially with few studies or high heterogeneity.</p>
                """

            html += f"""
            </div>

            <h3 style='color: #2c3e50; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;'>Contextual Factors</h3>
            <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px;'>
                <p style='margin: 5px 0;'><b>Sample size:</b> k = {n_studies} studies {'(adequate power)' if n_studies >= 10 else '(limited power)' if n_studies >= 5 else '(very limited power)'}</p>
                <p style='margin: 5px 0;'><i>Note: Publication bias tests have limited power with fewer than 10 studies</i></p>
            </div>

            <div style='background-color: #fff3cd; padding: 15px; border-radius: 5px; border-left: 4px solid #ffc107; margin-top: 20px;'>
                <p style='margin: 0;'><b>📊 Visualization:</b> Use Steps 18 and 20 to create funnel plots for visual assessment and manuscript figures.</p>
            </div>
            </div>
            """

            display(HTML(html))

    # -------------------------------------------------------------------------
    # Publication Text Generation
    # -------------------------------------------------------------------------

    def render_publication_text(self, egger_results, tf_results, n_studies):
        """Render publication-ready text"""
        with self.tab_publication:
            clear_output()

            display(HTML("<h3 style='color: #2c3e50;'>📝 Publication-Ready Results Text</h3>"))
            display(HTML("<p style='color: #6c757d;'>Copy and paste this formatted text into your manuscript:</p>"))

            # Generate methods and results text
            methods_html = self._generate_methods_text()
            results_html = self._generate_results_text(egger_results, tf_results, n_studies)

            display(HTML(methods_html))
            display(HTML(results_html))

            # Save combined text for audit report
            if 'ANALYSIS_CONFIG' in globals():
                globals()['ANALYSIS_CONFIG']['bias_text'] = methods_html + "<br><hr><br>" + results_html

    def _generate_methods_text(self):
        """Generate Materials and Methods section"""
        db = {
            'egger': "Egger, M., Davey Smith, G., Schneider, M., & Minder, C. (1997). Bias in meta-analysis detected by a simple, graphical test. <i>BMJ</i>, 315(7109), 629-634.",
            'trimfill': "Duval, S., & Tweedie, R. (2000). Trim and fill: A simple funnel-plot-based method of testing and adjusting for publication bias in meta-analysis. <i>Biometrics</i>, 56(2), 455-463.",
            'rothstein': "Rothstein, H. R., Sutton, A. J., & Borenstein, M. (Eds.). (2005). <i>Publication Bias in Meta-Analysis: Prevention, Assessment and Adjustments</i>. Chichester, UK: John Wiley & Sons.",
            'tool': COMETA_CITATION,
            'metafor': "Viechtbauer, W. (2010). Conducting meta-analyses in R with the metafor package. <i>Journal of Statistical Software</i>, 36(3), 1-48. https://doi.org/10.18637/jss.v036.i03"
        }

        html = f"""<div style='font-family: "Times New Roman", Times, serif; font-size: 12pt; line-height: 1.8; padding: 20px; background-color: #ffffff; border: 1px solid #eee; margin-bottom: 20px;'>

        <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; margin-top: 0;'>Materials and Methods</h3>

        <p style='text-align: justify;'>
        <b>Publication Bias Assessment.</b> To assess the potential impact of publication bias (the "file-drawer problem"), we employed two complementary methods [1].
        First, we used Egger's linear regression test to statistically evaluate funnel plot asymmetry [2].
        We used a multilevel extension of this test, fitting a three-level meta-regression of the effect sizes on their standard errors, with a significant slope indicating asymmetry (small-study effects).
        </p>

        <p style='text-align: justify;'>
        <b>Sensitivity Analysis.</b> Additionally, we applied the Trim-and-Fill method [3] to estimate the number of missing studies due to suppression of non-significant results.
        This non-parametric method (L0 estimator) iteratively removes (trims) the most extreme small studies from the side of the funnel plot opposite to the presumed missing studies and re-computes the effect size, then adds (fills) imputed mirror-image studies to restore symmetry.
        The side of the missing studies was determined from the sign of the slope of a meta-regression of the effect sizes on their standard errors, as in the metafor package [5].
        All analyses were conducted using the Co-Meta toolkit [4].
        </p>

        <h4 style='color: #34495e; margin-top: 20px; font-size: 11pt;'>References</h4>
        <ol style='font-size: 10pt; color: #555;'>
            <li>{db['rothstein']}</li>
            <li>{db['egger']}</li>
            <li>{db['trimfill']}</li>
            <li>{db['tool']}</li>
            <li>{db['metafor']}</li>
        </ol>
        </div>"""

        return html

    def _generate_results_text(self, egger_results, tf_results, n_studies, ci_level=95):
        """Generate publication-ready results text"""
        egger_p = egger_results['p_value']
        egger_slope = egger_results['slope']
        egger_se = egger_results['se']

        k0 = tf_results['k0']
        side = tf_results['side']
        orig_effect = tf_results['pooled_original']
        adj_effect = tf_results['pooled_filled']

        egger_sig = egger_p < _alpha()
        tf_bias = k0 > 0

        egger_sig_text = "significant" if egger_sig else "non-significant"
        p_format = f"< 0.001" if egger_p < 0.001 else f"= {egger_p:.3f}"

        text = f"""<div style='font-family: "Times New Roman", Times, serif; font-size: 12pt; line-height: 1.8; padding: 20px; background-color: #ffffff;'>

<h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;'>Publication Bias Assessment</h3>

<p style='text-align: justify;'>
We assessed potential publication bias using two complementary methods: Egger's regression test for funnel plot asymmetry and the Duval and Tweedie trim-and-fill procedure.
</p>

<h4 style='color: #34495e; margin-top: 25px;'>Egger's Regression Test</h4>

<p style='text-align: justify;'>
Egger's regression test evaluates funnel plot asymmetry by regressing effect sizes on their standard errors, with the slope testing for asymmetry. We employed a three-level meta-regression model to account for the nested structure of effect sizes within studies, providing robust estimates that accommodate within-study dependencies.
</p>

<p style='text-align: justify;'>
The Egger's test slope was <b>{egger_sig_text}</b> (β₁ = {egger_slope:.3f}, SE = {egger_se:.3f}, <i>p</i> {p_format}). """

        if egger_sig:
            text += f"""This indicates <b>significant funnel plot asymmetry</b>, suggesting potential publication bias or other sources of small-study effects. The {'positive' if egger_slope > 0 else 'negative'} slope suggests that smaller studies tend to report {'larger' if egger_slope > 0 else 'smaller'} effect sizes than larger studies.
</p>

<p style='text-align: justify;'>
However, it is important to note that funnel plot asymmetry can arise from sources other than publication bias, including genuine heterogeneity, chance, or differences in methodological quality between small and large studies. [<i>Consider discussing which alternative explanation(s) might apply to your meta-analysis.</i>]
</p>
"""
        else:
            text += f"""This suggests <b>no significant funnel plot asymmetry</b>, providing little evidence of publication bias based on this test. The symmetry of effect sizes across studies of different sizes supports the validity of the meta-analytic findings.
</p>
"""

        # Trim-and-fill section
        text += f"""
<h4 style='color: #34495e; margin-top: 25px;'>Trim-and-Fill Analysis</h4>

<p style='text-align: justify;'>
The trim-and-fill method (Duval & Tweedie, 2000) provides a non-parametric approach to estimating the number of studies missing due to publication bias. The procedure iteratively trims the most extreme small studies from the {'negative' if side == 'right' else 'positive'} side of the funnel plot (missing studies were estimated on the {side} side), re-computes the pooled effect, and then adds (fills) imputed mirror-image studies to restore funnel plot symmetry.
</p>

<p style='text-align: justify;'>
"""

        if k0 == 0:
            text += f"""The trim-and-fill procedure estimated <b>zero missing studies</b> (k₀ = 0), suggesting no asymmetry in the distribution of effect sizes. The pooled effect estimate remained unchanged at {orig_effect:.3f} ({ci_level:.0f}% CI [{tf_results['ci_lower_original']:.3f}, {tf_results['ci_upper_original']:.3f}]).
</p>

<p style='text-align: justify;'>
No funnel asymmetry was detected by this method. Trim-and-fill assumes independent effect sizes and has limited power, so this does not rule out publication bias.
</p>
"""
        else:
            pct_change = abs((adj_effect - orig_effect) / orig_effect) * 100 if orig_effect != 0 else 0
            direction_change = "decreased" if adj_effect < orig_effect else "increased"

            text += f"""The trim-and-fill procedure estimated <b>{k0} potentially missing studies</b> on the {side} side of the funnel plot. After imputing these missing studies, the adjusted pooled effect was {adj_effect:.3f} ({ci_level:.0f}% CI [{tf_results['ci_lower_filled']:.3f}, {tf_results['ci_upper_filled']:.3f}]), compared to the original estimate of {orig_effect:.3f} ({ci_level:.0f}% CI [{tf_results['ci_lower_original']:.3f}, {tf_results['ci_upper_original']:.3f}]).
</p>

<p style='text-align: justify;'>
The pooled effect {direction_change} by <b>{abs(adj_effect - orig_effect):.3f}</b> units ({pct_change:.1f}% relative change) after adjustment. """

            if pct_change > 20:
                text += f"""This substantial change suggests that publication bias, if present, could have a meaningful impact on the meta-analytic conclusions. The adjusted estimate should be considered as a sensitivity analysis, though it should be noted that trim-and-fill can sometimes overestimate the number of missing studies.
"""
            elif pct_change > 10:
                text += f"""This moderate change suggests some potential impact of publication bias on the pooled estimate, though the direction and significance of the effect remain {'' if adj_effect * orig_effect > 0 else 'un'}consistent between original and adjusted estimates.
"""
            else:
                text += f"""This small change suggests that publication bias, if present, has minimal impact on the meta-analytic conclusions. The robustness of the pooled estimate to adjustment increases confidence in the findings.
"""

            text += "</p>"

        # Combined interpretation
        text += f"""
<h4 style='color: #34495e; margin-top: 25px;'>Combined Interpretation</h4>

<p style='text-align: justify;'>
"""

        if egger_sig and tf_bias:
            text += f"""Both Egger's test and the trim-and-fill procedure suggest potential publication bias. Egger's test detected significant asymmetry (p {p_format}), and trim-and-fill estimated {k0} missing studies. This convergent evidence warrants cautious interpretation of the meta-analytic findings. """
            if k0 > 0:
                pct_change = abs((adj_effect - orig_effect) / orig_effect) * 100 if orig_effect != 0 else 0
                if pct_change > 10:
                    text += f"""Given the substantial adjustment to the pooled effect ({pct_change:.1f}% change), we recommend reporting both the original and adjusted estimates and considering the adjusted estimate in sensitivity analyses.
"""
                else:
                    text += f"""However, the modest change in the pooled effect ({pct_change:.1f}%) suggests the main conclusions are relatively robust to potential publication bias.
"""
        elif egger_sig or tf_bias:
            which_test = "Egger's test" if egger_sig else "trim-and-fill"
            text += f"""The evidence for publication bias is mixed. {which_test.capitalize()} suggests potential bias, but the other test does not. This inconsistency could reflect differences in what these tests detect (asymmetry vs. missing studies) or limited statistical power. We recommend interpreting results with appropriate caution and considering whether other factors (heterogeneity, methodological quality) might explain any observed asymmetry.
"""
        else:
            text += f"""Neither Egger's test nor trim-and-fill provided evidence of publication bias. The non-significant Egger's slope (p {p_format}) and absence of estimated missing studies (k₀ = 0) mean that no funnel asymmetry was detected. Because both tests have limited power, particularly with few studies or high heterogeneity, this does not rule out publication bias.
"""

        text += "</p>"

        # Tables
        text += f"""
<hr style='margin: 30px 0; border: none; border-top: 1px solid #bdc3c7;'>

<div style='background-color: #ecf0f1; padding: 20px; border-left: 4px solid #3498db; margin-top: 25px;'>
<h4 style='margin-top: 0; color: #2c3e50;'>📊 Table 1. Publication Bias Assessment Summary</h4>
<table style='width: 100%; border-collapse: collapse; margin-top: 15px; background-color: white;'>
<thead style='background-color: #34495e; color: white;'>
<tr>
<th style='border: 1px solid #bdc3c7; padding: 10px; text-align: left;'>Test</th>
<th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>Statistic</th>
<th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>p-value</th>
<th style='border: 1px solid #bdc3c7; padding: 10px; text-align: left;'>Interpretation</th>
</tr>
</thead>
<tbody>
<tr style='background-color: #f8f9fa;'>
<td style='border: 1px solid #bdc3c7; padding: 8px;'>Egger's Test</td>
<td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>slope β₁ = {egger_slope:.3f} (SE = {egger_se:.3f})</td>
<td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{"<0.001" if egger_p < 0.001 else f"{egger_p:.3f}"}</td>
<td style='border: 1px solid #bdc3c7; padding: 8px;'>{"Significant asymmetry" if egger_sig else "No significant asymmetry"}</td>
</tr>
<tr>
<td style='border: 1px solid #bdc3c7; padding: 8px;'>Trim-and-Fill</td>
<td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>k₀ = {k0} ({side} side)</td>
<td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>—</td>
<td style='border: 1px solid #bdc3c7; padding: 8px;'>{"Missing studies detected" if k0 > 0 else "No missing studies"}</td>
</tr>
</tbody>
</table>
</div>

<div style='background-color: #ecf0f1; padding: 20px; border-left: 4px solid #3498db; margin-top: 15px;'>
<h4 style='margin-top: 0; color: #2c3e50;'>📊 Table 2. Effect Size Estimates</h4>
<table style='width: 100%; border-collapse: collapse; margin-top: 15px; background-color: white;'>
<thead style='background-color: #34495e; color: white;'>
<tr>
<th style='border: 1px solid #bdc3c7; padding: 10px; text-align: left;'>Model</th>
<th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>Pooled Effect</th>
<th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>95% CI</th>
<th style='border: 1px solid #bdc3c7; padding: 10px; text-align: center;'>Change</th>
</tr>
</thead>
<tbody>
<tr style='background-color: #f8f9fa;'>
<td style='border: 1px solid #bdc3c7; padding: 8px;'>Original</td>
<td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{orig_effect:.3f}</td>
<td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>[{tf_results['ci_lower_original']:.3f}, {tf_results['ci_upper_original']:.3f}]</td>
<td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>—</td>
</tr>
<tr>
<td style='border: 1px solid #bdc3c7; padding: 8px;'>Trim-and-Fill Adjusted</td>
<td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{adj_effect:.3f}</td>
<td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>[{tf_results['ci_lower_filled']:.3f}, {tf_results['ci_upper_filled']:.3f}]</td>
<td style='border: 1px solid #bdc3c7; padding: 8px; text-align: center;'>{adj_effect - orig_effect:+.3f}</td>
</tr>
</tbody>
</table>
<p style='margin-top: 10px; font-size: 0.9em; color: #6c757d;'><i>Note:</i> Negative change indicates adjusted effect is smaller than original.</p>
</div>

<div style='background-color: #fff3cd; padding: 10px; border-left: 4px solid #ffc107; margin-top: 15px;'>
<p style='margin: 0;'><b>💡 Tip:</b> Select all text (Ctrl+A / Cmd+A), copy (Ctrl+C / Cmd+C), and paste into your word processor. Use Steps 18 and 20 to generate funnel plots for your manuscript figures.</p>
</div>

</div>"""

        return text

    def render_export_tab(self):
        """Setup export tab"""
        with self.tab_export:
            display(HTML("<h3 style='color: #2E86AB;'>💾 Download Audit Report</h3>"))
            display(HTML("<p>Export results of Egger's test and Trim-and-Fill analysis.</p>"))
            display(self.export_button)

    def display_ui(self):
        """Display the complete UI"""
        display(HTML("<h3>📉 Publication Bias Diagnostics</h3>"))
        display(HTML("<p style='color: #6c757d;'>Assess publication bias using Egger's test and Trim-and-Fill. Results appear in organized tabs below.</p>"))
        display(self.run_button)
        display(self.tabs)


# =============================================================================
# PART 3: CONTROLLER - Event Handling & Coordination
# =============================================================================

class PublicationBiasController:
    """
    Controller: Coordinates Model and View, handles user interactions.
    """

    def __init__(self):
        self.model = PublicationBiasModel()
        self.view = PublicationBiasView()

        # Wire up event handlers
        self.view.run_button.on_click(self.on_run_analysis)
        self.view.export_button.on_click(self.on_export_report)

    def on_run_analysis(self, button):
        """Handle run button click"""
        self.view.clear_all_tabs()

        try:
            # Initialize model
            self.model.initialize()
            n_obs, n_studies = self.model.get_dataset_info()

            # Run Egger's test
            try:
                egger_results = self.model.run_eggers_test()
                self.view.render_eggers_results(egger_results, n_obs, n_studies)
            except Exception as e:
                self.view.display_error(self.view.tab_egger, f"Error running Egger's test: {str(e)}")
                return

            # Run Trim-and-Fill
            try:
                tf_results = self.model.run_trimfill_analysis()
                self.view.render_trimfill_results(tf_results, n_obs, n_studies)
            except Exception as e:
                self.view.display_error(self.view.tab_trimfill, f"Error running Trim-and-Fill: {str(e)}")
                return

            # Combined assessment
            assessment = self.model.get_combined_assessment()
            if assessment:
                self.view.render_combined_assessment(assessment, egger_results, tf_results, n_studies)

            # Publication text
            self.view.render_publication_text(egger_results, tf_results, n_studies)

            # Setup export tab
            self.view.render_export_tab()

        except ValueError as e:
            self.view.display_error(self.view.tab_egger, str(e))
        except Exception as e:
            self.view.display_error(self.view.tab_egger, f"Unexpected error: {str(e)}")

    def on_export_report(self, button):
        """Handle export button click"""
        try:
            # Call global export function if it exists
            if 'export_analysis_report' in globals():
                export_func = globals()['export_analysis_report']
                export_func(report_type='publication_bias', filename_prefix='Publication_Bias_Audit')
            else:
                with self.view.tab_export:
                  display(HTML("<div style='color: orange;'>⚠️ Export function not found. Please ensure previous cells have been run.</div>"))
        except Exception as e:
            with self.view.tab_export:
                display(HTML(f"<div style='color: red;'>❌ Export failed: {str(e)}</div>"))

    def start(self):
        """Initialize and display the UI"""
        self.view.display_ui()


# =============================================================================
# EXECUTION - Instantiate and Run
# =============================================================================

controller = PublicationBiasController()
controller.start()

In [ ]:
#@title 📉 18. Publication Bias: Trim-and-Fill Plot
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# STEP 18: PUBLICATION-READY TRIM-AND-FILL PLOT
# Purpose: Funnel plot of the observed and the imputed (trim-and-fill) studies,
#          with the original and the bias-adjusted pooled estimates.
# Data:    Step 17 (Egger's Test & Trim-and-Fill) and Step 7 (reported model),
#          or a plot recipe file saved from this cell.
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from scipy.stats import norm
import html


def _trim_and_fill_plot_cell():
    PLOT_TYPE = 'trim_and_fill'
    RESULTS_KEY = 'trimfill_results'
    state = {'data': None, 'source': None, 'recipe_name': None, 'token': None, 'token_overall': None}

    # --- 1. DATA SNAPSHOT (everything the figure draws) ---
    def build_data():
        tf = ANALYSIS_CONFIG.get(RESULTS_KEY)
        if not tf:
            return None, None
        es = ANALYSIS_CONFIG.get('es_config') or {}
        rep = (ANALYSIS_CONFIG.get('overall_results') or {}).get('reported') or {}
        alpha = float(tf.get('alpha', _alpha()))

        yi_all = np.asarray(tf['yi_combined'], dtype=float)
        se_all = np.sqrt(np.asarray(tf['vi_combined'], dtype=float))
        k0 = int(tf['k0'])
        n_obs = len(yi_all) - k0
        yi_obs, se_obs = yi_all[:n_obs], se_all[:n_obs]
        yi_fill, se_fill = yi_all[n_obs:], se_all[n_obs:]

        mirror = tf.get('mirror_center')
        if mirror is None and k0 > 0:
            # older results: the first imputed value mirrors the most extreme observed one
            j = n_obs - k0 if tf['side'] == 'left' else 0
            mirror = (yi_obs[j] + yi_fill[0]) / 2.0

        if rep.get('mu') is not None:
            orig = {'mu': rep['mu'], 'ci_lower': rep.get('ci_lower'), 'ci_upper': rep.get('ci_upper'),
                    'model': rep.get('model', '')}
        else:
            orig = {'mu': tf['pooled_original'], 'ci_lower': tf.get('ci_lower_original'),
                    'ci_upper': tf.get('ci_upper_original'), 'model': ''}

        data = {
            'yi_obs': yi_obs.tolist(), 'se_obs': se_obs.tolist(),
            'yi_fill': yi_fill.tolist(), 'se_fill': se_fill.tolist(),
            'k0': k0, 'side': tf['side'], 'estimator': tf.get('estimator', 'L0'),
            'mirror_center': None if mirror is None else float(mirror),
            'original': orig,
            'adjusted': {'mu': tf['pooled_filled'], 'ci_lower': tf.get('ci_lower_filled'),
                         'ci_upper': tf.get('ci_upper_filled'),
                         'model': tf.get('adjusted_model', '3-Level')},
            'alpha': alpha,
            'ci_level': ci_pct_label(alpha),
            'null_value': es_null_value(),
            'effect_label': es.get('effect_label', 'Effect Size'),
        }
        when = tf.get('timestamp')
        source = {
            'analysis': (f"Trim-and-fill (estimator {data['estimator']}, side {data['side']}, "
                         f"k0 = {k0}; Step 17)"),
            'effect_label': data['effect_label'],
            'model': (f"original: {orig['model'] or 'Step 7'} (Step 7); adjusted: "
                      f"{data['adjusted']['model']} on the filled data"),
            'ci_level': data['ci_level'],
            'analysis_time': when.strftime('%Y-%m-%d %H:%M') if hasattr(when, 'strftime') else when,
        }
        return data, source

    # --- 2. CONTROLS (every widget whose value is saved in a recipe) ---
    st = {'description_width': '130px'}
    L = lambda w='450px': widgets.Layout(width=w)
    FS = lambda v, lo, hi, step, d: widgets.FloatSlider(value=v, min=lo, max=hi, step=step, description=d,
                                                        continuous_update=False, style=st, layout=L())
    IS = lambda v, lo, hi, d, step=1: widgets.IntSlider(value=v, min=lo, max=hi, step=step, description=d,
                                                        continuous_update=False, style=st, layout=L())
    DD = lambda opts, v, d: widgets.Dropdown(options=opts, value=v, description=d, style=st, layout=L())
    CB = lambda v, d: widgets.Checkbox(value=v, description=d, indent=False, layout=L())
    TX = lambda v, d: widgets.Text(value=v, description=d, style=st, layout=L())
    LINE_STYLES = [('Dashed', '--'), ('Solid', '-'), ('Dotted', ':'), ('Dash-Dot', '-.')]
    C = {
        'width': FS(10.0, 2.0, 16.0, 0.1, 'Plot Width (in):'),
        'height': FS(7.0, 2.0, 14.0, 0.1, 'Plot Height (in):'),
        'title_font': IS(14, 6, 24, 'Title Font Size:'),
        'label_font': IS(12, 6, 18, 'Axis Label Size:'),
        'tick_font': IS(10, 5, 16, 'Tick Label Size:'),
        'legend_font': IS(10, 5, 14, 'Legend Size:'),
        'show_title': CB(True, 'Show Plot Title'),
        'title': TX('Trim-and-Fill Funnel Plot', 'Plot Title:'),
        'xlabel': TX('Effect Size', 'X-Axis Label:'),
        'show_ylabel': CB(True, 'Show Y-Axis Label'),
        'ylabel': TX('Standard Error', 'Y-Axis Label:'),
        'obs_color': DD([('Black', 'black'), ('Gray', 'gray'), ('Steel Blue', 'steelblue'), ('Blue', 'blue'),
                         ('Dark Green', 'darkgreen')], 'black', 'Observed Color:'),
        'obs_shape': DD([('Circle (●)', 'o'), ('Diamond (◆)', 'D'), ('Square (■)', 's'),
                         ('Triangle Up (▲)', '^'), ('Pentagon', 'p')], 'o', 'Observed Shape:'),
        'obs_edge_width': FS(0.5, 0.0, 3.0, 0.1, 'Obs Edge Width:'),
        'imp_color': DD([('White (Hollow)', 'white'), ('Red', 'red'), ('Orange', 'orange'), ('Yellow', 'yellow'),
                         ('Light Gray', 'lightgray'), ('None (Transparent)', 'none')], 'white', 'Imputed Color:'),
        'imp_edge': DD([('Red', 'red'), ('Black', 'black'), ('Orange', 'orange'), ('Dark Red', 'darkred')],
                       'red', 'Imputed Edge:'),
        'imp_edge_width': FS(1.5, 0.5, 4.0, 0.25, 'Imp Edge Width:'),
        'imp_shape': DD([('Circle (●)', 'o'), ('Diamond (◆)', 'D'), ('Square (■)', 's'),
                         ('Triangle Down (▼)', 'v'), ('Star (★)', '*')], 'o', 'Imputed Shape:'),
        'point_size': IS(50, 10, 200, 'Point Size:', 5),
        'point_alpha': FS(0.7, 0.1, 1.0, 0.05, 'Point Opacity:'),
        'show_orig': CB(True, 'Show Original Mean Line'),
        'orig_color': DD([('Black', 'black'), ('Gray', 'gray'), ('Blue', 'blue'), ('Dark Blue', 'darkblue')],
                         'black', 'Orig Line Color:'),
        'orig_width': FS(2.0, 0.5, 5.0, 0.5, 'Orig Line Width:'),
        'orig_style': DD(LINE_STYLES, '--', 'Orig Line Style:'),
        'show_adj': CB(True, 'Show Adjusted Mean Line'),
        'adj_color': DD([('Red', 'red'), ('Orange', 'orange'), ('Magenta', 'magenta'), ('Dark Red', 'darkred')],
                        'red', 'Adj Line Color:'),
        'adj_width': FS(2.0, 0.5, 5.0, 0.5, 'Adj Line Width:'),
        'adj_style': DD([('Solid', '-'), ('Dashed', '--'), ('Dotted', ':'), ('Dash-Dot', '-.')], '-',
                        'Adj Line Style:'),
        'show_mirror': CB(False, 'Show Mirror Axis of the Imputation'),
        'show_null': CB(False, 'Show No-Effect Line'),
        'show_funnel': CB(True, 'Show Pseudo-Confidence Funnel'),
        'funnel_center': DD([('Mirror axis of the imputation', 'mirror'), ('Adjusted estimate', 'adjusted'),
                             ('Original estimate', 'original')], 'mirror', 'Funnel Centred On:'),
        'funnel_fill': CB(True, 'Fill Funnel Region'),
        'funnel_fill_alpha': FS(0.1, 0.0, 0.5, 0.05, 'Fill Opacity:'),
        'funnel_line_width': FS(1.0, 0.5, 3.0, 0.25, 'Funnel Line Width:'),
        'auto_x': CB(True, 'Auto-Scale X-Axis'),
        'x_min': widgets.FloatText(value=-2.0, description='X-Min:', style={'description_width': '80px'},
                                   layout=widgets.Layout(width='220px', visibility='hidden')),
        'x_max': widgets.FloatText(value=2.0, description='X-Max:', style={'description_width': '80px'},
                                   layout=widgets.Layout(width='220px', visibility='hidden')),
        'invert_y': CB(True, 'Invert Y-Axis (Standard for Funnel Plots)'),
        'grid': CB(True, 'Show Grid'),
        'grid_style': DD([('Dotted (Light)', 'dotted'), ('Dashed (Light)', 'dashed'), ('Solid (Light)', 'solid')],
                         'dotted', 'Grid Style:'),
        'grid_alpha': FS(0.4, 0.1, 1.0, 0.1, 'Grid Opacity:'),
        'legend_loc': DD([('Best', 'best'), ('Upper Right', 'upper right'), ('Upper Left', 'upper left'),
                          ('Lower Right', 'lower right'), ('Lower Left', 'lower left'),
                          ('Center Right', 'center right'), ('Outside Right', 'outside'), ('None', 'none')],
                         'upper right', 'Legend Position:'),
        'legend_frame': CB(True, 'Legend Frame'),
        'legend_fancybox': CB(True, 'Fancy Box (Rounded Corners)'),
        'save_pdf': CB(True, 'Save as PDF'),
        'save_png': CB(True, 'Save as PNG'),
        'save_svg': CB(False, 'Save as SVG'),
        'dpi': IS(300, 100, 600, 'PNG DPI:', 50),
        'filename': TX('TrimFill_Plot', 'Filename Prefix:'),
        'transparent': CB(False, 'Transparent Background'),
        'timestamp': CB(True, 'Include Timestamp'),
    }

    def on_auto_x(change):
        vis = 'hidden' if change['new'] else 'visible'
        C['x_min'].layout.visibility = vis
        C['x_max'].layout.visibility = vis
    C['auto_x'].observe(on_auto_x, names='value')

    def set_data_defaults(data):
        """Defaults that depend on the data (called for session data and recipes)."""
        C['xlabel'].value = data['effect_label']
        C['show_funnel'].description = f"Show {data['ci_level']} Pseudo-Confidence Funnel"
        C['show_null'].description = f"Show No-Effect Line ({data['null_value']:g})"

    def funnel_center(data, s):
        key = s['funnel_center']
        if key == 'mirror' and data.get('mirror_center') is not None:
            return data['mirror_center']
        return data['original']['mu'] if key == 'original' else data['adjusted']['mu']

    # --- 3. DRAWING (data + settings -> figure; no widget access) ---
    def draw(data, s):
        yi_obs, se_obs = np.asarray(data['yi_obs'], float), np.asarray(data['se_obs'], float)
        yi_fill, se_fill = np.asarray(data['yi_fill'], float), np.asarray(data['se_fill'], float)
        se_all = np.concatenate([se_obs, se_fill])
        k0 = int(data['k0'])
        orig_mu, adj_mu = data['original']['mu'], data['adjusted']['mu']
        z = norm.ppf(1 - data['alpha'] / 2)

        fig, ax = plt.subplots(figsize=(s['width'], s['height']))
        max_se = np.max(se_all) * 1.1 if len(se_all) > 0 else 1.0
        y_range = np.linspace(0, max_se, 100)

        if s['show_funnel']:
            c = funnel_center(data, s)
            x_left, x_right = c - z * y_range, c + z * y_range
            ax.plot(x_left, y_range, color='gray', linestyle='--', linewidth=s['funnel_line_width'], alpha=0.5,
                    label=f"{data['ci_level']} pseudo-confidence limits")
            ax.plot(x_right, y_range, color='gray', linestyle='--', linewidth=s['funnel_line_width'], alpha=0.5)
            if s['funnel_fill']:
                ax.fill_betweenx(y_range, x_left, x_right, color='lightgray', alpha=s['funnel_fill_alpha'])

        ax.scatter(yi_obs, se_obs, c=s['obs_color'], s=s['point_size'], alpha=s['point_alpha'],
                   marker=s['obs_shape'], edgecolors='black', linewidth=s['obs_edge_width'],
                   label=f'Observed Studies (k={len(yi_obs)})', zorder=3)
        if k0 > 0:
            ax.scatter(yi_fill, se_fill, c=s['imp_color'], s=s['point_size'], alpha=s['point_alpha'],
                       marker=s['imp_shape'], edgecolors=s['imp_edge'], linewidth=s['imp_edge_width'],
                       label=f'Imputed Studies (k={k0})', zorder=3)

        if s['show_orig']:
            ax.axvline(orig_mu, color=s['orig_color'], linestyle=s['orig_style'], linewidth=s['orig_width'],
                       label=(f"Original ({data['original']['model']}): {orig_mu:.3f}" if data['original'].get('model')
                              else f'Original: {orig_mu:.3f}'), zorder=2)
        if s['show_adj']:
            ax.axvline(adj_mu, color=s['adj_color'], linestyle=s['adj_style'], linewidth=s['adj_width'],
                       label=f'Adjusted: {adj_mu:.3f}', zorder=2)
        if s['show_mirror'] and data.get('mirror_center') is not None:
            ax.axvline(data['mirror_center'], color='gray', linestyle=':', linewidth=1.2,
                       label=f"Mirror axis: {data['mirror_center']:.3f}", zorder=2)
        if s['show_null']:
            ax.axvline(data['null_value'], color='gray', linestyle='-', linewidth=1.0, alpha=0.6,
                       label='No effect', zorder=1)

        ax.set_ylim((max_se, 0) if s['invert_y'] else (0, max_se))
        if not s['auto_x']:
            ax.set_xlim(s['x_min'], s['x_max'])
        if s['show_title']:
            ax.set_title(s['title'], fontsize=s['title_font'], fontweight='bold', pad=15)
        ax.set_xlabel(s['xlabel'], fontsize=s['label_font'], fontweight='bold')
        if s['show_ylabel']:
            ax.set_ylabel(s['ylabel'], fontsize=s['label_font'], fontweight='bold')
        ax.tick_params(labelsize=s['tick_font'])
        if s['grid']:
            ls = {'dotted': ':', 'dashed': '--'}.get(s['grid_style'], '-')
            ax.grid(True, linestyle=ls, alpha=s['grid_alpha'])
        if s['legend_loc'] == 'outside':
            ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=s['legend_frame'],
                      fancybox=s['legend_fancybox'], fontsize=s['legend_font'])
        elif s['legend_loc'] != 'none':
            ax.legend(loc=s['legend_loc'], frameon=s['legend_frame'], fancybox=s['legend_fancybox'],
                      fontsize=s['legend_font'])
        fig.tight_layout()
        return fig

    def caption(data, s):
        style_name = {'--': 'dashed', '-': 'solid', ':': 'dotted', '-.': 'dash-dot'}
        shapes = {'o': 'Circles', 's': 'Squares', 'D': 'Diamonds', '^': 'Triangles', 'v': 'Triangles',
                  'p': 'Pentagons', '*': 'Stars'}
        k0, n_obs = int(data['k0']), len(data['yi_obs'])
        lvl = data['ci_level']
        o, a = data['original'], data['adjusted']
        ci = lambda r: (f" ({lvl} CI [{r['ci_lower']:.3f}, {r['ci_upper']:.3f}])"
                        if r.get('ci_lower') is not None and r.get('ci_upper') is not None else "")
        txt = (f"Trim-and-fill funnel plot for the assessment of publication bias "
               f"({html.escape(str(data['effect_label']))}). {shapes.get(s['obs_shape'], 'Markers')} represent the "
               f"observed effect sizes (<i>k</i> = {n_obs}), ")
        if k0 > 0:
            txt += (f"while {shapes.get(s['imp_shape'], 'markers').lower()} with a {s['imp_edge']} outline "
                    f"(<i>k</i><sub>0</sub> = {k0}) are the "
                    f"studies imputed on the {data['side']} side by the Duval and Tweedie trim-and-fill procedure "
                    f"({data['estimator']} estimator), as mirror images of the most extreme observed effects "
                    f"around the random-effects (DerSimonian-Laird) estimate of the trimmed data"
                    + (f" ({data['mirror_center']:.3f})" if data.get('mirror_center') is not None else "") + ". ")
        else:
            txt += (f"and the trim-and-fill procedure ({data['estimator']} estimator) estimated no missing "
                    f"studies. ")
        if s['show_orig']:
            model = f"{o['model']} " if o.get('model') else ""
            txt += (f"The {style_name.get(s['orig_style'], '')} line is the original pooled estimate of the "
                    f"reported {model}model (Step 7) = {o['mu']:.3f}{ci(o)}. ")
        if s['show_adj']:
            txt += (f"The {style_name.get(s['adj_style'], '')} line is the bias-adjusted estimate, from the "
                    f"{a['model']} model refitted to the observed and imputed effect sizes = "
                    f"{a['mu']:.3f}{ci(a)}. ")
        if s['show_mirror'] and data.get('mirror_center') is not None:
            txt += "The dotted line marks the mirror axis of the imputation. "
        if s['show_null']:
            txt += f"The thin solid line marks no effect ({data['null_value']:g}). "
        if s['show_funnel']:
            centre = {'mirror': 'the mirror axis of the imputation', 'adjusted': 'the adjusted estimate',
                      'original': 'the original estimate'}[s['funnel_center']]
            if s['funnel_center'] == 'mirror' and data.get('mirror_center') is None:
                centre = 'the adjusted estimate'
            txt += (f"The dashed {'lines and shaded region' if s['funnel_fill'] else 'lines'} show the {lvl} "
                    f"pseudo-confidence limits (estimate ± {norm.ppf(1 - data['alpha'] / 2):.3f} × SE) centred on "
                    f"{centre}.")
        return txt

    # --- 4. ACTIONS ---
    out = widgets.Output()
    caption_output = widgets.Output()
    source_box = widgets.HTML()

    def refresh_source():
        source_box.value = source_html(state['source'], state['recipe_name'])

    def generate(_b=None):
        with out:
            clear_output(wait=True)
            if state['data'] is None:
                display(HTML("<div style='color:#721c24;'>❌ No trim-and-fill results: run Step 17 "
                             "(Publication Bias: Egger's Test &amp; Trim-and-Fill) or load a plot recipe.</div>"))
                return
            if state['recipe_name'] is None:
                if results_token(RESULTS_KEY) != state['token']:
                    display(HTML(stale_warning_html("Step 17 (Egger's Test & Trim-and-Fill)")))
                elif results_token('overall_results') != state['token_overall']:
                    display(HTML(stale_warning_html('Step 7 (Overall Meta-Analysis)')))
            s = widget_values(C)
            try:
                fig = draw(state['data'], s)
                paths = cometa_save_figure(fig, s['filename'], pdf=s['save_pdf'], png=s['save_png'],
                                           svg=s['save_svg'], dpi=s['dpi'], transparent=s['transparent'],
                                           timestamp=s['timestamp'])
                show_figure(fig)
            except Exception as e:
                plt.close('all')
                display(HTML(f"<div style='color:#721c24;'>❌ Plotting error: {html.escape(str(e))}</div>"))
                return
            display(HTML(saved_files_html(paths)))
            cap = caption_html(caption(state['data'], s))
            display(HTML(cap))
        with caption_output:
            clear_output()
            display(HTML(cap))

    def get_recipe():
        if state['data'] is None:
            return None
        return build_plot_recipe(PLOT_TYPE, state['data'], widget_values(C), state['source'])

    def on_load(rec, style_only, name):
        if not style_only:
            state.update(data=rec['data'], source=rec.get('source'), recipe_name=name)
            set_data_defaults(state['data'])
            refresh_source()
        skipped = apply_widget_values(C, rec.get('settings'))
        generate()
        return skipped

    # --- 5. INITIAL STATE AND LAYOUT ---
    try:
        state['data'], state['source'] = build_data()
    except Exception as e:
        state['data'], state['source'] = None, None
        print(f"⚠️ Could not read the Step 17 results: {e}")
    state['token'] = results_token(RESULTS_KEY)
    state['token_overall'] = results_token('overall_results')
    if state['data'] is not None:
        set_data_defaults(state['data'])
    refresh_source()

    preset = preset_dropdown({'width': C['width'], 'height': C['height'], 'title': C['title_font'],
                              'label': C['label_font'], 'tick': C['tick_font']})
    H = lambda t: widgets.HTML(f"<h3 style='color: #2E86AB;'>{t}</h3>")
    B = lambda t: widgets.HTML(f"<b>{t}</b>")
    HR = lambda: widgets.HTML("<hr style='margin: 10px 0;'>")
    tabs = widgets.Tab(children=[
        widgets.VBox([H('🎨 Style & Layout'), B('Dimensions:'), preset, C['width'], C['height'], HR(),
                      B('Typography:'), C['title_font'], C['label_font'], C['tick_font'], C['legend_font']]),
        widgets.VBox([H('📝 Text & Labels'), B('Title:'), C['show_title'], C['title'], HR(), B('Axis Labels:'),
                      C['xlabel'], C['show_ylabel'], C['ylabel']]),
        widgets.VBox([H('⚫ Points & Data'), B('Observed Studies:'), C['obs_color'], C['obs_shape'],
                      C['obs_edge_width'], HR(), B('Imputed Studies:'), C['imp_color'], C['imp_edge'],
                      C['imp_edge_width'], C['imp_shape'], HR(), B('Common Settings:'), C['point_size'],
                      C['point_alpha']]),
        widgets.VBox([H('📐 Lines & Funnel'), B('Original Mean Line (Step 7 reported model):'), C['show_orig'],
                      C['orig_color'], C['orig_width'], C['orig_style'], HR(),
                      B('Adjusted Mean Line (model refitted to the filled data):'), C['show_adj'], C['adj_color'],
                      C['adj_width'], C['adj_style'], HR(), B('Reference Lines:'), C['show_mirror'], C['show_null'],
                      HR(), B('Funnel Guidelines:'), C['show_funnel'], C['funnel_center'], C['funnel_fill'],
                      C['funnel_fill_alpha'], C['funnel_line_width']]),
        widgets.VBox([H('📏 Axes & Grid'), B('X-Axis Scaling:'), C['auto_x'], widgets.HBox([C['x_min'], C['x_max']]),
                      HR(), B('Y-Axis:'), C['invert_y'], HR(), B('Grid:'), C['grid'], C['grid_style'],
                      C['grid_alpha']]),
        widgets.VBox([H('💾 Export & Legend'), B('Legend:'), C['legend_loc'], C['legend_frame'],
                      C['legend_fancybox'], HR(), B('File Formats:'), C['save_pdf'], C['save_png'], C['save_svg'],
                      C['dpi'], HR(), B('File Settings:'), C['filename'], C['transparent'], C['timestamp']]),
        widgets.VBox([H('📝 Figure Caption'),
                      widgets.HTML("<p style='color: #666; font-size: 13px;'><i>Generated from the plot settings "
                                   "when the plot is drawn.</i></p>"), caption_output]),
    ])
    for i, t in enumerate(['🎨 Style', '📝 Text', '⚫ Points', '📐 Lines', '📏 Axes', '💾 Export', '📝 Caption']):
        tabs.set_title(i, t)

    run_btn = widgets.Button(description='📊 Generate Trim-and-Fill Plot', button_style='success',
                             layout=widgets.Layout(width='450px', height='50px'), style={'font_weight': 'bold'})
    run_btn.on_click(generate)
    recipe_bar = plot_recipe_bar(PLOT_TYPE, get_recipe, on_load, lambda: C['filename'].value)

    display(widgets.VBox([
        widgets.HTML("<h3 style='color: #2E86AB;'>📊 Trim-and-Fill Plot Generator</h3>"),
        widgets.HTML("<p style='color: #666;'>Observed and imputed studies with the original and the "
                     "bias-adjusted pooled estimates.</p>"),
        source_box, widgets.HTML("<hr style='margin: 15px 0;'>"), tabs, widgets.HTML("<hr style='margin: 15px 0;'>"),
        run_btn, recipe_bar, out,
    ]))


_trim_and_fill_plot_cell()


In [ ]:
#@title 📉 19. Publication Bias: PET-PEESE
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# Purpose: Centralized data management for PET-PEESE analysis
# Dependencies: ANALYSIS_CONFIG global dictionary
# =============================================================================

import numpy as np
import pandas as pd
from typing import Dict, Any, Optional, Tuple
from dataclasses import dataclass
import warnings


# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class PETPEESEConfig:
    """Configuration for PET-PEESE analysis"""
    effect_col: str
    var_col: str
    p_threshold: float = 0.10  # Two-tailed level of the PET intercept test (= one-tailed 0.05)
    alpha: float = 0.05

    def __post_init__(self):
        """Validate configuration"""
        if not self.effect_col:
            raise ValueError("effect_col cannot be empty")
        if not self.var_col:
            raise ValueError("var_col cannot be empty")
        if self.p_threshold <= 0 or self.p_threshold >= 1:
            raise ValueError(f"p_threshold must be between 0 and 1, got {self.p_threshold}")
        if self.alpha <= 0 or self.alpha >= 1:
            raise ValueError(f"alpha must be between 0 and 1, got {self.alpha}")


@dataclass
class PETResult:
    """Results from PET model (Effect ~ SE)"""
    intercept: float
    intercept_se: float
    intercept_ci: Tuple[float, float]
    slope: float
    slope_se: float
    slope_p: float
    intercept_p: float = float('nan')   # test of H0: corrected effect = 0

    # Full estimates (for advanced users)
    estimates: Optional[Dict[str, Any]] = None


@dataclass
class PEESEResult:
    """Results from PEESE model (Effect ~ Variance)"""
    intercept: float
    intercept_se: float
    intercept_ci: Tuple[float, float]
    slope: float
    slope_se: float
    slope_p: float

    # Full estimates (for advanced users)
    estimates: Optional[Dict[str, Any]] = None


@dataclass
class NaiveEstimate:
    """Original unadjusted pooled estimate"""
    estimate: Optional[float]
    se: Optional[float]
    ci_lower: Optional[float]
    ci_upper: Optional[float]


@dataclass
class PETPEESEDecision:
    """Decision rule outcome"""
    bias_detected: bool
    p_threshold: float
    recommended_model: str  # 'PET' or 'PEESE'
    recommended_estimate: float
    recommended_se: float
    recommended_ci: Tuple[float, float]
    rationale: str
    # PET intercept significant in the expected direction -> PEESE is used
    effect_detected: bool = False


@dataclass
class PETPEESEResult:
    """Complete PET-PEESE analysis result"""
    pet: PETResult
    peese: PEESEResult
    decision: PETPEESEDecision
    naive: NaiveEstimate

    # Sample info
    n_obs: int
    n_studies: int
    effect_col: str


# =============================================================================
# DATA MANAGER CLASS
# =============================================================================

class PETPEESEDataManager:
    """
    Centralized data access layer for PET-PEESE analysis.
    Handles all interactions with ANALYSIS_CONFIG and data validation.
    """

    def __init__(self, analysis_config: Dict[str, Any]):
        """
        Initialize data manager.

        Args:
            analysis_config: Global ANALYSIS_CONFIG dictionary
        """
        self._config = analysis_config
        self._validate_prerequisites()

    # -------------------------------------------------------------------------
    # VALIDATION METHODS
    # -------------------------------------------------------------------------

    def _validate_prerequisites(self) -> None:
        """Validate that required configuration exists"""
        if 'effect_col' not in self._config:
            warnings.warn("effect_col not in ANALYSIS_CONFIG, using default 'hedges_g'")
        if 'var_col' not in self._config:
            warnings.warn("var_col not in ANALYSIS_CONFIG, using default 'Vg'")

    def validate_data(self, df: pd.DataFrame) -> Tuple[bool, Optional[str]]:
        """
        Validate that data is suitable for PET-PEESE analysis.

        Args:
            df: DataFrame to validate

        Returns:
            Tuple of (is_valid, error_message)
        """
        if df is None or len(df) == 0:
            return False, "No data available"

        if self.effect_col not in df.columns:
            return False, f"Effect column '{self.effect_col}' not found"

        if self.var_col not in df.columns:
            return False, f"Variance column '{self.var_col}' not found"

        # Check for minimum observations (need at least 3 for regression)
        n_valid = df[[self.effect_col, self.var_col]].notna().all(axis=1).sum()
        if n_valid < 3:
            return False, f"Need at least 3 valid observations for PET-PEESE, found {n_valid}"

        # Check for study ID
        if 'id' not in df.columns:
            warnings.warn("'id' column not found - will treat all observations as independent")

        return True, None

    # -------------------------------------------------------------------------
    # PROPERTY ACCESSORS
    # -------------------------------------------------------------------------

    @property
    def analysis_data(self) -> Optional[pd.DataFrame]:
        """Get analysis dataset"""
        if 'analysis_data' in self._config:
            return self._config['analysis_data'].copy()
        return None

    @property
    def effect_col(self) -> str:
        """Get effect size column name"""
        return self._config.get('effect_col', 'hedges_g')

    @property
    def var_col(self) -> str:
        """Get variance column name"""
        return self._config.get('var_col', 'Vg')

    @property
    def es_config(self) -> Dict[str, Any]:
        """Get effect size configuration"""
        return self._config.get('es_config', {})

    @property
    def global_settings(self) -> Dict[str, Any]:
        """Get global settings (with defaults)"""
        return self._config.get('global_settings', {
            'alpha': 0.05
        })

    # -------------------------------------------------------------------------
    # DATA EXTRACTION METHODS
    # -------------------------------------------------------------------------

    def prepare_data(
        self,
        df: Optional[pd.DataFrame] = None
    ) -> pd.DataFrame:
        """
        Prepare and clean data for PET-PEESE analysis.

        Args:
            df: DataFrame to prepare (uses analysis_data if None)

        Returns:
            Cleaned DataFrame with SE and Var columns

        Raises:
            ValueError: If data cannot be prepared
        """
        if df is None:
            df = self.analysis_data

        if df is None:
            raise ValueError("No data available for analysis")

        # Validate
        is_valid, error_msg = self.validate_data(df)
        if not is_valid:
            raise ValueError(error_msg)

        # Create working copy
        clean_df = df.copy()

        # SE and Var are always derived from the analysis variance column.
        # Reusing a column that happens to be called 'SE' or 'Var' (from the
        # input data, or stale from an earlier run) could give PET/PEESE
        # different variances from every other analysis, and the shared-
        # control VCV would then no longer match them (diagonal fallback).
        if self.var_col not in clean_df.columns:
            raise ValueError(f"Cannot compute SE: '{self.var_col}' column not found")
        clean_df['Var'] = pd.to_numeric(clean_df[self.var_col], errors='coerce')
        clean_df['SE'] = np.sqrt(clean_df['Var'])

        # Remove missing values
        clean_df = clean_df[[self.effect_col, 'SE', 'Var', 'id']].dropna().copy()

        # Remove zero or negative variances
        clean_df = clean_df[clean_df['Var'] > 0].copy()
        clean_df = clean_df[clean_df['SE'] > 0].copy()

        # Final check
        if len(clean_df) < 3:
            raise ValueError(
                f"Insufficient data after cleaning: {len(clean_df)} observations. "
                f"Need at least 3 for PET-PEESE analysis."
            )

        return clean_df

    def get_naive_estimate(self) -> NaiveEstimate:
        """
        Unadjusted pooled estimate: the model the Overall cell reports.
        """
        try:
            overall_res = self._config.get('overall_results', {})
            rep = overall_res.get('reported')
            if rep:
                return NaiveEstimate(estimate=rep.get('mu'), se=rep.get('se'),
                                     ci_lower=rep.get('ci_lower'), ci_upper=rep.get('ci_upper'))
            best_model = overall_res.get('best_model', '2-Level')

            if best_model == '3-Level' and 'three_level_results' in self._config:
                res = self._config['three_level_results']
                return NaiveEstimate(
                    estimate=res.get('pooled_effect'),
                    se=res.get('se'),
                    ci_lower=res.get('ci_lower'),
                    ci_upper=res.get('ci_upper')
                )
            else:
                return NaiveEstimate(
                    estimate=overall_res.get('pooled_effect_random'),
                    se=overall_res.get('pooled_SE_random_reported'),
                    ci_lower=overall_res.get('ci_lower_random_reported'),
                    ci_upper=overall_res.get('ci_upper_random_reported')
                )
        except:
            return NaiveEstimate(estimate=None, se=None, ci_lower=None, ci_upper=None)


    # -------------------------------------------------------------------------
    # RESULT PERSISTENCE METHODS
    # -------------------------------------------------------------------------

    def save_petpeese_results(self, result: PETPEESEResult) -> None:
        """
        Save PET-PEESE results to ANALYSIS_CONFIG.

        Args:
            result: PETPEESEResult object
        """
        import datetime

        self._config['pet_peese_results'] = {
            'timestamp': datetime.datetime.now(),
            'pet': {
                'intercept': result.pet.intercept,
                'intercept_se': result.pet.intercept_se,
                'intercept_ci': result.pet.intercept_ci,
                'slope': result.pet.slope,
                'slope_se': result.pet.slope_se,
                'slope_p': result.pet.slope_p,
                'intercept_p': result.pet.intercept_p,
            },
            'peese': {
                'intercept': result.peese.intercept,
                'intercept_se': result.peese.intercept_se,
                'intercept_ci': result.peese.intercept_ci,
                'slope': result.peese.slope,
                'slope_se': result.peese.slope_se,
                'slope_p': result.peese.slope_p,
            },
            'decision': {
                'bias_detected': result.decision.bias_detected,
                'effect_detected': result.decision.effect_detected,
                'p_threshold': result.decision.p_threshold,
                'recommended_model': result.decision.recommended_model,
                'recommended_estimate': result.decision.recommended_estimate,
                'recommended_se': result.decision.recommended_se,
                'recommended_ci': result.decision.recommended_ci,
                'rationale': result.decision.rationale,
            },
            'naive': {
                'estimate': result.naive.estimate,
                'se': result.naive.se,
                'ci_lower': result.naive.ci_lower,
                'ci_upper': result.naive.ci_upper,
            }
        }

    def save_publication_text(self, text: str) -> None:
        """Save publication-ready text"""
        self._config['pet_peese_text'] = text

    # -------------------------------------------------------------------------
    # UTILITY METHODS
    # -------------------------------------------------------------------------

    def summary_dict(self) -> Dict[str, Any]:
        """Get summary of current configuration"""
        df = self.analysis_data

        return {
            'effect_col': self.effect_col,
            'var_col': self.var_col,
            'n_observations': len(df) if df is not None else 0,
            'n_studies': df['id'].nunique() if df is not None and 'id' in df.columns else 0,
            'has_overall_results': 'overall_results' in self._config
        }



import numpy as np
import pandas as pd
from typing import Dict, Any, Optional, Tuple
import warnings


# =============================================================================
# PET ENGINE (Effect ~ Standard Error)
# =============================================================================

class PETEngine:
    """
    PET (Precision Effect Test) regression engine.

    Mathematical implementation: Effect_Size ~ Intercept + Slope * SE
    Uses three-level REML to account for within-study dependencies.
    """

    def __init__(self, effect_col: str, var_col: str = 'Var'):
        """
        Initialize engine.

        Args:
            effect_col: Effect size column name
            var_col: Variance column name (for weighting)
        """
        self.effect_col = effect_col
        self.var_col = var_col

    def run_pet_regression(
        self,
        df: pd.DataFrame
    ) -> Optional[Dict[str, Any]]:
        """
        Run PET regression: Effect ~ SE

        Args:
            df: DataFrame with effect sizes, SE, and Var columns

        Returns:
            Dictionary with regression results or None if fails
        """
        try:
            # Use the three-level REML regression function
            estimates, metadata, opt_result = _run_three_level_reml_regression_v2(
                df,
                moderator_col='SE',
                effect_col=self.effect_col,
                var_col=self.var_col,
                center_continuous=False,   # intercept = effect at SE = 0
            )

            if estimates is None:
                return None

            return estimates

        except Exception as e:
            warnings.warn(f"PET regression failed: {str(e)}")
            return None


# =============================================================================
# PEESE ENGINE (Effect ~ Variance)
# =============================================================================

class PEESEEngine:
    """
    PEESE (Precision Effect Estimate with Standard Error) regression engine.

    Mathematical implementation: Effect_Size ~ Intercept + Slope * Variance
    Uses three-level REML to account for within-study dependencies.
    """

    def __init__(self, effect_col: str, var_col: str = 'Var'):
        """
        Initialize engine.

        Args:
            effect_col: Effect size column name
            var_col: Variance column name (used as moderator)
        """
        self.effect_col = effect_col
        self.var_col = var_col

    def run_peese_regression(
        self,
        df: pd.DataFrame
    ) -> Optional[Dict[str, Any]]:
        """
        Run PEESE regression: Effect ~ Variance

        Args:
            df: DataFrame with effect sizes and Var columns

        Returns:
            Dictionary with regression results or None if fails
        """
        try:
            # Use the three-level REML regression function
            estimates, metadata, opt_result = _run_three_level_reml_regression_v2(
                df,
                moderator_col='Var',
                effect_col=self.effect_col,
                var_col=self.var_col,
                center_continuous=False,   # intercept = effect at variance = 0
            )

            if estimates is None:
                return None

            return estimates

        except Exception as e:
            warnings.warn(f"PEESE regression failed: {str(e)}")
            return None


# =============================================================================
# DECISION MAKER
# =============================================================================

class PETPEESEDecisionMaker:
    """
    Conditional PET-PEESE rule (Stanley & Doucouliagos, 2014).

    The choice depends on the PET INTERCEPT (the effect at SE = 0), tested
    in the expected direction of the effect (two-tailed p < threshold,
    default 0.10, with the intercept of the expected sign, i.e. one-tailed
    0.05):
      - significant  -> a genuine effect is likely; PET is biased towards
                        zero when an effect exists, so the PEESE intercept is
                        the bias-corrected estimate;
      - otherwise    -> the PET intercept is the bias-corrected estimate.
    The PET slope (small-study effects / funnel asymmetry) is reported, but
    does not choose the estimator.
    """

    @staticmethod
    def make_decision(
        pet_intercept_p: float,
        pet_intercept: float,
        pet_intercept_se: float,
        pet_intercept_ci: Tuple[float, float],
        peese_intercept: float,
        peese_intercept_se: float,
        peese_intercept_ci: Tuple[float, float],
        p_threshold: float = 0.10,
        expected_sign: float = 0.0,
        pet_slope_p: float = float('nan'),
    ) -> 'PETPEESEDecision':
        """
        Args:
            pet_intercept_p: two-sided p-value of the PET intercept
            expected_sign: sign of the uncorrected pooled effect (0 = no
                expectation: a two-sided test)
            pet_slope_p: p-value of the PET slope (small-study effects;
                reported, not used for the choice)
        """
        right_direction = (expected_sign == 0) or (np.sign(pet_intercept) == np.sign(expected_sign))
        effect_detected = bool(pet_intercept_p < p_threshold and right_direction)
        bias_detected = bool(np.isfinite(pet_slope_p) and pet_slope_p < p_threshold)

        if effect_detected:
            recommended_model = "PEESE"
            recommended_estimate = peese_intercept
            recommended_se = peese_intercept_se
            recommended_ci = peese_intercept_ci
            rationale = (
                f"The PET intercept (the effect at SE = 0) was significant in the expected direction "
                f"(p = {pet_intercept_p:.4f} < {p_threshold:.2f}), indicating a genuine non-zero effect; "
                f"the PEESE intercept is therefore the bias-corrected estimate."
            )
        else:
            recommended_model = "PET"
            recommended_estimate = pet_intercept
            recommended_se = pet_intercept_se
            recommended_ci = pet_intercept_ci
            why = (f"p = {pet_intercept_p:.4f} ≥ {p_threshold:.2f}" if pet_intercept_p >= p_threshold
                   else "it was significant but in the opposite direction to the uncorrected effect")
            rationale = (
                f"The PET intercept (the effect at SE = 0) was not significant in the expected direction "
                f"({why}); the PET intercept is therefore the bias-corrected estimate."
            )

        return PETPEESEDecision(
            bias_detected=bias_detected,
            p_threshold=p_threshold,
            recommended_model=recommended_model,
            recommended_estimate=recommended_estimate,
            recommended_se=recommended_se,
            recommended_ci=recommended_ci,
            rationale=rationale,
            effect_detected=effect_detected,
        )


# =============================================================================
# PET-PEESE ORCHESTRATOR
# =============================================================================

class PETPEESEEngine:
    """
    High-level orchestrator for PET-PEESE analysis.
    Coordinates PET regression, PEESE regression, and decision rule.
    """

    def __init__(self, data_manager: 'PETPEESEDataManager'):
        """
        Initialize engine with data manager.

        Args:
            data_manager: PETPEESEDataManager instance
        """
        self.data_manager = data_manager
        self.pet_engine = PETEngine(
            effect_col=data_manager.effect_col,
            var_col='Var'
        )
        self.peese_engine = PEESEEngine(
            effect_col=data_manager.effect_col,
            var_col='Var'
        )
        self.decision_maker = PETPEESEDecisionMaker()

    # -------------------------------------------------------------------------
    # MAIN EXECUTION METHOD
    # -------------------------------------------------------------------------

    def run_analysis(
        self,
        p_threshold: float = 0.10,
        progress_callback: Optional[callable] = None
    ) -> Optional['PETPEESEResult']:
        """
        Execute complete PET-PEESE analysis.

        WORKFLOW:
        1. Prepare data
        2. Run PET regression (Effect ~ SE)
        3. Run PEESE regression (Effect ~ Variance)
        4. Apply decision rule
        5. Get naive estimate for comparison

        Args:
            p_threshold: P-value threshold for decision rule (default 0.10)
            progress_callback: Optional progress updates

        Returns:
            PETPEESEResult object or None if analysis fails
        """
        # Step 1: Prepare data
        if progress_callback:
            progress_callback("📊 Preparing data...")

        try:
            df = self.data_manager.prepare_data()
        except ValueError as e:
            if progress_callback:
                progress_callback(f"❌ Data preparation failed: {str(e)}")
            return None

        n_obs = len(df)
        n_studies = df['id'].nunique() if 'id' in df.columns else n_obs

        # Step 2: Run PET regression
        if progress_callback:
            progress_callback("🔍 Running PET model (Effect ~ SE)...")

        pet_estimates = self.pet_engine.run_pet_regression(df)

        if pet_estimates is None:
            if progress_callback:
                progress_callback("❌ PET regression failed")
            return None

        # Extract PET results
        pet_result = self._extract_pet_result(pet_estimates)

        # Step 3: Run PEESE regression
        if progress_callback:
            progress_callback("🔍 Running PEESE model (Effect ~ Variance)...")

        peese_estimates = self.peese_engine.run_peese_regression(df)

        if peese_estimates is None:
            if progress_callback:
                progress_callback("❌ PEESE regression failed")
            return None

        # Extract PEESE results
        peese_result = self._extract_peese_result(peese_estimates)

        # Step 4: Apply decision rule
        if progress_callback:
            progress_callback("🎯 Applying decision rule...")

        # Uncorrected estimate first: its sign is the expected direction
        naive_estimate = self.data_manager.get_naive_estimate()
        expected_sign = float(np.sign(naive_estimate.estimate)) if naive_estimate.estimate is not None else 0.0

        decision = self.decision_maker.make_decision(
            pet_intercept_p=pet_result.intercept_p,
            pet_slope_p=pet_result.slope_p,
            expected_sign=expected_sign,
            pet_intercept=pet_result.intercept,
            pet_intercept_se=pet_result.intercept_se,
            pet_intercept_ci=pet_result.intercept_ci,
            peese_intercept=peese_result.intercept,
            peese_intercept_se=peese_result.intercept_se,
            peese_intercept_ci=peese_result.intercept_ci,
            p_threshold=p_threshold
        )


        # Build result object
        result = PETPEESEResult(
            pet=pet_result,
            peese=peese_result,
            decision=decision,
            naive=naive_estimate,
            n_obs=n_obs,
            n_studies=n_studies,
            effect_col=self.data_manager.effect_col
        )

        if progress_callback:
            progress_callback(
                f"✅ {decision.recommended_model} recommended: "
                f"{decision.recommended_estimate:.4f} (PET intercept p = {pet_result.intercept_p:.4f})"
            )

        return result

    # -------------------------------------------------------------------------
    # RESULT EXTRACTION METHODS
    # -------------------------------------------------------------------------

    def _extract_pet_result(self, estimates: Dict[str, Any]) -> PETResult:
        """
        Extract PET results from regression output.

        Args:
            estimates: Dictionary from _run_three_level_reml_regression_v2

        Returns:
            PETResult object
        """
        intercept = estimates['betas'][0]
        slope = estimates['betas'][1]

        intercept_se = estimates['se_betas'][0]
        slope_se = estimates['se_betas'][1]

        slope_p = estimates['p_values'][1]
        intercept_p = estimates['p_values'][0]

        intercept_ci = (estimates['ci_lower'][0], estimates['ci_upper'][0])

        return PETResult(
            intercept=intercept,
            intercept_se=intercept_se,
            intercept_ci=intercept_ci,
            slope=slope,
            slope_se=slope_se,
            slope_p=slope_p,
            intercept_p=intercept_p,
            estimates=estimates
        )

    def _extract_peese_result(self, estimates: Dict[str, Any]) -> PEESEResult:
        """
        Extract PEESE results from regression output.

        Args:
            estimates: Dictionary from _run_three_level_reml_regression_v2

        Returns:
            PEESEResult object
        """
        intercept = estimates['betas'][0]
        slope = estimates['betas'][1]

        intercept_se = estimates['se_betas'][0]
        slope_se = estimates['se_betas'][1]

        slope_p = estimates['p_values'][1]

        intercept_ci = (estimates['ci_lower'][0], estimates['ci_upper'][0])

        return PEESEResult(
            intercept=intercept,
            intercept_se=intercept_se,
            intercept_ci=intercept_ci,
            slope=slope,
            slope_se=slope_se,
            slope_p=slope_p,
            estimates=estimates
        )


# =============================================================================
# HELPER: COMPARISON CALCULATOR
# =============================================================================

class ComparisonCalculator:
    """
    Calculate comparisons between naive and adjusted estimates.
    """

    @staticmethod
    def calculate_change(
        naive: float,
        adjusted: float
    ) -> Tuple[float, float]:
        """
        Calculate absolute and percent change.

        Args:
            naive: Original unadjusted estimate
            adjusted: PET-PEESE adjusted estimate

        Returns:
            Tuple of (absolute_change, percent_change)
        """
        if naive is None:
            return 0.0, 0.0

        abs_change = adjusted - naive

        if naive == 0:
            pct_change = 0.0
        else:
            pct_change = (abs_change / naive) * 100

        return abs_change, pct_change

    @staticmethod
    def assess_impact(percent_change: float) -> str:
        """
        Assess the magnitude of impact.

        Args:
            percent_change: Percent change in effect size

        Returns:
            Impact assessment string
        """
        abs_pct = abs(percent_change)

        if abs_pct < 5:
            return "minimal"
        elif abs_pct < 15:
            return "small"
        elif abs_pct < 30:
            return "moderate"
        else:
            return "substantial"



# =============================================================================
# PRESENTATION LAYER - PET-PEESE CORRECTION
# Purpose: Pure UI rendering without business logic
# Dependencies: Data & Analysis Layers
# =============================================================================

import pandas as pd
import numpy as np
from typing import Dict, Any, Optional
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
import sys


# =============================================================================
# HTML TEMPLATE GENERATORS
# =============================================================================

class PETPEESEHTMLTemplates:
    """
    Static HTML template generators for PET-PEESE visualizations.
    All methods are pure functions returning HTML strings.
    """

    @staticmethod
    def comparison_table(result: 'PETPEESEResult') -> str:
        """Generate comparison table for all three estimates"""

        rows = []

        # Naive estimate (if available)
        if result.naive.estimate is not None:
            rows.append({
                'Model': 'Original (Naive)',
                'Estimate': f"{result.naive.estimate:.4f}",
                'SE': f"{result.naive.se:.4f}" if result.naive.se else "—",
                '95% CI': f"[{result.naive.ci_lower:.4f}, {result.naive.ci_upper:.4f}]"
                         if result.naive.ci_lower is not None else "—",
                'Note': 'Unadjusted pooled estimate'
            })

        # PET estimate
        rows.append({
            'Model': 'PET',
            'Estimate': f"{result.pet.intercept:.4f}",
            'SE': f"{result.pet.intercept_se:.4f}",
            '95% CI': f"[{result.pet.intercept_ci[0]:.4f}, {result.pet.intercept_ci[1]:.4f}]",
            'Note': f"Slope p = {result.pet.slope_p:.4f}"
        })

        # PEESE estimate
        rows.append({
            'Model': 'PEESE',
            'Estimate': f"{result.peese.intercept:.4f}",
            'SE': f"{result.peese.intercept_se:.4f}",
            '95% CI': f"[{result.peese.intercept_ci[0]:.4f}, {result.peese.intercept_ci[1]:.4f}]",
            'Note': f"Slope p = {result.peese.slope_p:.4f}"
        })

        # Build HTML table
        html = """
        <table style='width: 100%; border-collapse: collapse; margin-top: 20px;'>
            <thead>
                <tr style='background-color: #f8f9fa; border-bottom: 2px solid #dee2e6;'>
                    <th style='padding: 12px; text-align: left; font-weight: bold;'>Model</th>
                    <th style='padding: 12px; text-align: left; font-weight: bold;'>Estimate</th>
                    <th style='padding: 12px; text-align: left; font-weight: bold;'>SE</th>
                    <th style='padding: 12px; text-align: left; font-weight: bold;'>95% CI</th>
                    <th style='padding: 12px; text-align: left; font-weight: bold;'>Note</th>
                </tr>
            </thead>
            <tbody>
        """

        for row in rows:
            is_recommended = row['Model'] == result.decision.recommended_model
            bg_color = "#d4edda" if is_recommended else "white"
            html += f"<tr style='background-color: {bg_color}; border-bottom: 1px solid #dee2e6;'>"
            for col in ['Model', 'Estimate', 'SE', '95% CI', 'Note']:
                html += f"<td style='padding: 10px;'>{row[col]}</td>"
            html += "</tr>"

        html += "</tbody></table>"

        return html

    @staticmethod
    def recommendation_box(result: 'PETPEESEResult') -> str:
        """Generate recommendation box with decision"""
        decision = result.decision

        return f"""
        <div style='margin-top: 30px; padding: 20px; background-color: #d4edda;
                    border-left: 5px solid #28a745; border-radius: 5px;'>
            <h4 style='margin-top: 0; color: #155724;'>✓ Recommended Adjustment</h4>
            <p style='font-size: 16px; margin: 10px 0;'>
                <b>Model:</b> {decision.recommended_model}<br>
                <b>Adjusted Estimate:</b> {decision.recommended_estimate:.4f}
                (SE = {decision.recommended_se:.4f})<br>
                <b>95% CI:</b> [{decision.recommended_ci[0]:.4f}, {decision.recommended_ci[1]:.4f}]
            </p>
            <p style='color: #6c757d; font-size: 14px; margin-top: 15px;'>
                {decision.rationale}
            </p>
        </div>
        """

    @staticmethod
    def decision_badge(result: 'PETPEESEResult') -> str:
        """Generate decision status badge"""
        if result.decision.bias_detected:
            color = "#856404"
            bg_color = "#fff3cd"
            icon = "⚠"
            title = "Publication Bias Detected"
        else:
            color = "#155724"
            bg_color = "#d4edda"
            icon = "✓"
            title = "No Strong Evidence of Bias"

        return f"""
        <div style='margin-top: 20px; padding: 20px; background-color: {bg_color};
                    border-left: 5px solid {color}; border-radius: 5px;'>
            <h4 style='margin-top: 0; color: {color};'>{icon} {title}</h4>
            <p style='font-size: 14px; color: #6c757d;'>
                {result.decision.rationale}
            </p>
        </div>
        """

    @staticmethod
    def model_details_box(result: 'PETPEESEResult') -> str:
        """Generate model details box"""
        pet = result.pet
        peese = result.peese

        return f"""
        <div style='margin-top: 30px; padding: 15px; background-color: #f8f9fa; border-radius: 5px;'>
            <h4 style='color: #495057;'>Model Details</h4>

            <h5 style='color: #6c757d; margin-top: 20px;'>PET Model (Effect ~ SE)</h5>
            <p style='color: #6c757d; font-size: 14px; margin-left: 20px;'>
                <b>Intercept:</b> {pet.intercept:.4f} (SE = {pet.intercept_se:.4f})<br>
                <b>Slope:</b> {pet.slope:.4f} (SE = {pet.slope_se:.4f}, p = {pet.slope_p:.4f})<br>
                <b>95% CI for Intercept:</b> [{pet.intercept_ci[0]:.4f}, {pet.intercept_ci[1]:.4f}]
            </p>

            <h5 style='color: #6c757d; margin-top: 20px;'>PEESE Model (Effect ~ Variance)</h5>
            <p style='color: #6c757d; font-size: 14px; margin-left: 20px;'>
                <b>Intercept:</b> {peese.intercept:.4f} (SE = {peese.intercept_se:.4f})<br>
                <b>Slope:</b> {peese.slope:.4f} (SE = {peese.slope_se:.4f}, p = {peese.slope_p:.4f})<br>
                <b>95% CI for Intercept:</b> [{peese.intercept_ci[0]:.4f}, {peese.intercept_ci[1]:.4f}]
            </p>
        </div>
        """

    @staticmethod
    def interpretation_guide() -> str:
        """Generate interpretation guide"""
        return """
        <div style='margin-top: 30px;'>
            <h4 style='color: #495057;'>Understanding PET-PEESE</h4>
            <p style='color: #6c757d; font-size: 14px; line-height: 1.6;'>
                <b>PET-PEESE</b> is a conditional meta-regression method for correcting publication bias:
            </p>
            <ul style='color: #6c757d; font-size: 14px; line-height: 1.8;'>
                <li><b>PET (Precision Effect Test):</b> Regresses effect size on standard error.
                    A significant slope suggests that smaller studies report larger effects (publication bias).</li>
                <li><b>PEESE (Precision Effect Estimate with SE):</b> Regresses effect size on variance (SE²).
                    Provides a better-calibrated estimate when bias is detected.</li>
                <li><b>Decision Rule (Stanley &amp; Doucouliagos, 2014):</b> if the PET intercept
                    (the effect at SE = 0) is significant in the expected direction (p &lt; 0.10, two-tailed),
                    use the PEESE intercept as the bias-corrected estimate; otherwise use the PET intercept.
                    The PET slope tests for small-study effects but does not choose the estimator.</li>
            </ul>
        </div>
        """

    @staticmethod
    def recommendations_box() -> str:
        """Generate recommendations box"""
        return """
        <div style='margin-top: 30px; padding: 15px; background-color: #e7f3ff;
                    border-left: 4px solid #2E86AB; border-radius: 5px;'>
            <h4 style='color: #2E86AB; margin-top: 0;'>💡 Recommendations</h4>
            <ul style='color: #6c757d; font-size: 14px; line-height: 1.8;'>
                <li>Report both the original pooled estimate and the PET-PEESE adjusted estimate.</li>
                <li>Consider PET-PEESE as one of several bias assessment methods (alongside Egger's test, trim-and-fill, etc.).</li>
                <li>If the adjusted estimate differs substantially from the original, investigate potential sources of bias.</li>
                <li>PET-PEESE assumes bias is related to precision; other bias mechanisms may not be captured.</li>
            </ul>
        </div>
        """

    @staticmethod
    def plot_interpretation_note() -> str:
        """Generate plot interpretation note"""
        return """
        <div style='margin-top: 20px; padding: 15px; background-color: #f8f9fa;
                    border-left: 4px solid #6c757d; border-radius: 5px;'>
            <h4 style='margin-top: 0; color: #495057;'>📝 Plot Interpretation</h4>
            <ul style='color: #6c757d; font-size: 14px;'>
                <li><b>Red dashed line (PET):</b> Linear relationship between effect size and standard error.</li>
                <li><b>Green solid line (PEESE):</b> Quadratic relationship (effect ~ SE²), appears curved when plotted against SE.</li>
                <li><b>Horizontal dotted line:</b> The recommended bias-corrected estimate (intercept of chosen model).</li>
                <li>If studies cluster above/below the zero-effect line asymmetrically, publication bias may be present.</li>
            </ul>
        </div>
        """


# =============================================================================
# PUBLICATION TEXT GENERATORS
# =============================================================================

class PETPEESEPublicationTextGenerator:
    """
    Generates publication-ready text for manuscripts.
    """

    def __init__(self, ci_level: float = 95):
        self.ci_level = ci_level

    def generate_methods_section(self, p_threshold: float) -> str:
        """Generate Materials and Methods section"""
        db = {
            'stanley': "Stanley, T. D., & Doucouliagos, H. (2014). Meta-regression approximations to reduce publication selection bias. <i>Research Synthesis Methods</i>, 5(1), 60-78.",
            'tool': COMETA_CITATION
        }

        html = f"""
        <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt; line-height: 1.8; padding: 20px; background-color: #ffffff; border: 1px solid #eee; margin-bottom: 20px;'>
            <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; margin-top: 0;'>Materials and Methods</h3>
            <p style='text-align: justify;'>
            <b>Publication Bias Adjustment (PET-PEESE).</b> To adjust for potential publication bias and small-study effects, we applied the Precision-Effect Test and Precision-Effect Estimate with Standard Error (PET-PEESE) conditional meta-regression approach [1].
            A PET model regressed the effect sizes on their standard errors and a PEESE model on their sampling variances; the intercept of each is the effect expected for a study with no sampling error. Following the conditional rule, if the PET intercept differed significantly from zero in the expected direction (two-tailed <i>p</i> &lt; {p_threshold:.2f}), the PEESE intercept was reported as the bias-adjusted estimate; otherwise the PET intercept was reported. The PET slope was used as a test of small-study effects. All analyses accounted for within-study dependency using cluster-robust variance estimation via the Co-Meta toolkit [2].
            </p>
            <h4 style='color: #34495e; margin-top: 20px; font-size: 11pt;'>References</h4>
            <ol style='font-size: 10pt; color: #555;'>
                <li>{db['stanley']}</li>
                <li>{db['tool']}</li>
            </ol>
        </div>
        """
        return html

    def generate_results_section(self, result: 'PETPEESEResult') -> str:
        """Generate complete Results section with interpretation"""
        pet = result.pet
        peese = result.peese
        dec = result.decision
        naive = result.naive

        pet_sig = "significant" if dec.bias_detected else "non-significant"
        p_format = f"< 0.001" if pet.slope_p < 0.001 else f"= {pet.slope_p:.3f}"
        pi_format = f"< 0.001" if pet.intercept_p < 0.001 else f"= {pet.intercept_p:.3f}"

        naive_text = f"{naive.estimate:.3f} (95% CI [{naive.ci_lower:.3f}, {naive.ci_upper:.3f}])" if naive.estimate is not None else "unavailable"

        html = f"""
        <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt; line-height: 1.8; padding: 20px; background-color: #ffffff;'>
            <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;'>PET-PEESE Bias Adjustment</h3>
            <p style='text-align: justify;'>
            The PET model showed a <b>{pet_sig}</b> relationship between effect sizes and their standard errors (slope = {pet.slope:.3f}, <i>p</i> {p_format}), {'indicating' if dec.bias_detected else 'giving little evidence of'} small-study effects. The PET intercept (the effect at SE = 0) was {pet.intercept:.3f} (<i>p</i> {pi_format}).
        """

        if dec.effect_detected:
            pct_change = abs((dec.recommended_estimate - naive.estimate) / naive.estimate)*100 if (naive.estimate and naive.estimate != 0) else 0
            dir_change = "decrease" if (naive.estimate and abs(dec.recommended_estimate) < abs(naive.estimate)) else "increase"

            html += f"""
            Because the PET intercept was significant in the expected direction, the PEESE intercept was reported, following the conditional rule. The bias-adjusted pooled effect was estimated at <b>{dec.recommended_estimate:.3f}</b> ({self.ci_level:.0f}% CI [{dec.recommended_ci[0]:.3f}, {dec.recommended_ci[1]:.3f}]).
            Compared to the unadjusted estimate of {naive_text}, this represents a {pct_change:.1f}% {dir_change} in magnitude.
            """
        else:
            html += f"""
            Because the PET intercept was not significant in the expected direction, the PET intercept was reported as the bias-adjusted estimate, following the conditional rule: <b>{dec.recommended_estimate:.3f}</b> ({self.ci_level:.0f}% CI [{dec.recommended_ci[0]:.3f}, {dec.recommended_ci[1]:.3f}]), compared with the unadjusted estimate of {naive_text}.
            """

        html += f"""
            </p>
            <div style='background-color: #fff3cd; padding: 10px; border-left: 4px solid #ffc107; margin-top: 15px;'>
                <p style='margin: 0;'><b>💡 Tip:</b> Select all text (Ctrl+A / Cmd+A), copy (Ctrl+C / Cmd+C), and paste into your word processor.</p>
            </div>
        </div>
        """
        return html


# =============================================================================
# VIEW COMPONENTS (Tab Renderers)
# =============================================================================

class PETPEESEResultsView:
    """
    Manages all UI rendering for PET-PEESE analysis.
    Contains zero business logic - only presentation code.
    """

    def __init__(self):
        """Initialize view with display settings"""
        self.templates = PETPEESEHTMLTemplates()
        self.text_gen = PETPEESEPublicationTextGenerator(ci_level=95)

        # Create tab widgets
        self.tab_summary = widgets.Output()
        self.tab_visuals = widgets.Output()
        self.tab_interpretation = widgets.Output()
        self.tab_publication = widgets.Output()

        self.tabs = widgets.Tab(children=[
            self.tab_summary,
            self.tab_visuals,
            self.tab_interpretation,
            self.tab_publication
        ])

        self.tabs.set_title(0, '📊 Summary')
        self.tabs.set_title(1, '📈 Visuals')
        self.tabs.set_title(2, '📝 Interpretation')
        self.tabs.set_title(3, '📝 Publication Text')

    # -------------------------------------------------------------------------
    # TAB 1: SUMMARY
    # -------------------------------------------------------------------------

    def render_summary_tab(self, result: 'PETPEESEResult') -> None:
        """Render summary tab with comparison table"""

        with self.tab_summary:
            self.tab_summary.clear_output()

            display(HTML("<h3 style='color: #2E86AB;'>📊 PET-PEESE Summary</h3>"))
            display(HTML("<p style='color: #6c757d;'>Comparison of effect size estimates with and without publication bias correction.</p>"))

            # Comparison table
            table_html = self.templates.comparison_table(result)
            display(HTML(table_html))

            # Recommendation box
            rec_html = self.templates.recommendation_box(result)
            display(HTML(rec_html))

    # -------------------------------------------------------------------------
    # TAB 2: VISUALS
    # -------------------------------------------------------------------------

    def render_visuals_tab(
        self,
        result: 'PETPEESEResult',
        df: pd.DataFrame
    ) -> None:
        """Render visuals tab with scatter plot and regression lines"""

        with self.tab_visuals:
            self.tab_visuals.clear_output()

            # Create figure
            fig, ax = plt.subplots(figsize=(10, 6))

            # Scatter plot
            ax.scatter(
                df['SE'],
                df[result.effect_col],
                alpha=0.6,
                s=80,
                color='steelblue',
                edgecolors='black',
                linewidth=0.5,
                label='Observed Studies'
            )

            # PET regression line (linear in SE)
            se_range = np.linspace(df['SE'].min(), df['SE'].max(), 100)
            pet_line = result.pet.intercept + result.pet.slope * se_range
            ax.plot(
                se_range,
                pet_line,
                color='red',
                linewidth=2,
                linestyle='--',
                label=f"PET: y = {result.pet.intercept:.3f} + {result.pet.slope:.3f}·SE"
            )

            # PEESE regression line (quadratic in SE, linear in Var)
            # Effect = intercept + slope * Var = intercept + slope * SE²
            peese_line = result.peese.intercept + result.peese.slope * (se_range ** 2)
            ax.plot(
                se_range,
                peese_line,
                color='green',
                linewidth=2,
                linestyle='-',
                label=f"PEESE: y = {result.peese.intercept:.3f} + {result.peese.slope:.3f}·SE²"
            )

            # Highlight recommended estimate
            if result.decision.recommended_model == 'PET':
                ax.axhline(
                    y=result.pet.intercept,
                    color='red',
                    linestyle=':',
                    linewidth=2,
                    label=f"Recommended (PET): {result.pet.intercept:.3f}"
                )
            else:
                ax.axhline(
                    y=result.peese.intercept,
                    color='green',
                    linestyle=':',
                    linewidth=2,
                    label=f"Recommended (PEESE): {result.peese.intercept:.3f}"
                )

            # Labels and styling
            ax.set_xlabel('Standard Error', fontsize=12, fontweight='bold')
            ax.set_ylabel('Effect Size', fontsize=12, fontweight='bold')
            ax.set_title(
                'PET-PEESE Analysis: Effect Size vs. Standard Error',
                fontsize=14,
                fontweight='bold',
                pad=20
            )
            ax.legend(loc='best', frameon=True, shadow=True, fontsize=10)
            ax.grid(True, alpha=0.3, linestyle='--')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

            plt.tight_layout()
            plt.show()

            # Add interpretation note
            note_html = self.templates.plot_interpretation_note()
            display(HTML(note_html))

    # -------------------------------------------------------------------------
    # TAB 3: INTERPRETATION
    # -------------------------------------------------------------------------

    def render_interpretation_tab(self, result: 'PETPEESEResult') -> None:
        """Render interpretation tab with detailed explanation"""

        with self.tab_interpretation:
            self.tab_interpretation.clear_output()

            display(HTML("<h3 style='color: #2E86AB;'>📝 PET-PEESE Interpretation</h3>"))

            # Decision summary
            decision_html = self.templates.decision_badge(result)
            display(HTML(decision_html))

            # Understanding guide
            guide_html = self.templates.interpretation_guide()
            display(HTML(guide_html))

            # Model details
            details_html = self.templates.model_details_box(result)
            display(HTML(details_html))

            # Recommendations
            rec_html = self.templates.recommendations_box()
            display(HTML(rec_html))

    # -------------------------------------------------------------------------
    # TAB 4: PUBLICATION TEXT
    # -------------------------------------------------------------------------

    def render_publication_tab(self, result: 'PETPEESEResult') -> str:
        """Render publication text tab and return combined HTML string"""
        with self.tab_publication:
            self.tab_publication.clear_output()

            display(HTML("<h3 style='color: #2E86AB;'>📝 Publication-Ready Results Text</h3>"))
            display(HTML("<p style='color: #6c757d;'>Copy and paste this formatted text into your manuscript:</p>"))

            # Generate both sections
            methods_html = self.text_gen.generate_methods_section(result.decision.p_threshold)
            results_html = self.text_gen.generate_results_section(result)

            # Display
            display(HTML(methods_html))
            display(HTML(results_html))

            # Return combined for saving to state
            return methods_html + "<br><hr><br>" + results_html

    # -------------------------------------------------------------------------
    # ERROR DISPLAY
    # -------------------------------------------------------------------------

    def render_error(self, message: str, details: Optional[str] = None) -> None:
        """Render error message in summary tab"""
        with self.tab_summary:
            self.tab_summary.clear_output()
            error_html = f"""
            <div style='color: red; padding: 10px; border: 1px solid red; border-radius: 5px;'>
                <b>⚠ Error:</b> {message}
            </div>
            """
            if details:
                error_html += f"<pre style='margin-top: 10px; color: #6c757d;'>{details}</pre>"
            display(HTML(error_html))

# =============================================================================
# CONTROLLER LAYER - PET-PEESE CORRECTION
# Purpose: Orchestrates data, analysis, and view components
# Dependencies: All previous layers (Data, Analysis, View)
# =============================================================================

import traceback
from typing import Optional, Dict, Any
from IPython.display import display, HTML
import ipywidgets as widgets


# =============================================================================
# MAIN CONTROLLER
# =============================================================================

class PETPEESEController:
    """
    Master controller that orchestrates the entire PET-PEESE analysis workflow.
    Coordinates data management, statistical computation, and UI rendering.
    """

    def __init__(self, analysis_config: Dict[str, Any]):
        """
        Initialize controller with ANALYSIS_CONFIG.

        Args:
            analysis_config: Global ANALYSIS_CONFIG dictionary
        """
        self.analysis_config = analysis_config

        # Initialize components
        try:
            self.data_manager = PETPEESEDataManager(analysis_config)
            self.engine = PETPEESEEngine(self.data_manager)
            self.view = PETPEESEResultsView()

            self._initialization_error = None

        except Exception as e:
            # If initialization fails, create minimal view to show error
            self.view = PETPEESEResultsView()
            self.data_manager = None
            self.engine = None
            self._initialization_error = e

        # Create settings widgets
        self._create_settings_widgets()

    # -------------------------------------------------------------------------
    # SETTINGS WIDGETS CREATION
    # -------------------------------------------------------------------------

    def _create_settings_widgets(self) -> None:
        """Create all settings widgets"""

        self.p_threshold_widget = widgets.FloatSlider(
            value=0.10,
            min=0.01,
            max=0.20,
            step=0.01,
            description='P-threshold:',
            style={'description_width': '100px'},
            layout=widgets.Layout(width='400px'),
            readout_format='.2f'
        )

        self.run_button = widgets.Button(
            description='▶ Run PET-PEESE',
            button_style='success',
            icon='play',
            layout=widgets.Layout(width='200px', height='40px')
        )

        # Attach event handler
        self.run_button.on_click(self._handle_run_click)

    # -------------------------------------------------------------------------
    # UI CREATION
    # -------------------------------------------------------------------------

    def create_ui(self) -> widgets.VBox:
        """
        Create the user interface with settings and run button.

        Returns:
            VBox widget containing the UI
        """
        # Help text
        help_html = """
        <p style='color: #6c757d; font-size: 14px; margin-top: 10px;'>
            <b>P-threshold:</b> if the PET intercept is significant in the expected direction
            (two-tailed p &lt; threshold), the PEESE estimate is used. Default 0.10 (Stanley &amp; Doucouliagos, 2014).
        </p>
        """

        ui = widgets.VBox([
            widgets.HTML("<h3>📊 PET-PEESE: Publication Bias Correction</h3>"),
            widgets.HTML("<p style='color: #6c757d;'>Estimate the true effect size adjusted for small-study effects using conditional meta-regression.</p>"),
            widgets.HTML("<b>Settings:</b>"),
            self.p_threshold_widget,
            widgets.HTML(help_html),
            widgets.HTML("<hr>"),
            self.run_button,
            widgets.HTML("<br>")
        ])

        return ui

    # -------------------------------------------------------------------------
    # MAIN EXECUTION METHOD
    # -------------------------------------------------------------------------

    def run_analysis(self) -> None:
        """
        Execute complete PET-PEESE analysis workflow.
        """
        # Clear all tabs
        for tab in [self.view.tab_summary, self.view.tab_visuals,
                    self.view.tab_interpretation]:
            tab.clear_output()

        # Check for initialization errors
        if self._initialization_error:
            self._handle_initialization_error()
            return

        # Validate ANALYSIS_CONFIG
        if 'ANALYSIS_CONFIG' not in globals():
            self.view.render_error("ANALYSIS_CONFIG not found. Run Steps 2–6 (data and effect sizes) first.")
            return

        # Get settings from widgets
        p_threshold = self.p_threshold_widget.value

        # Progress callback
        def progress_callback(message: str):
            """Callback for progress updates"""
            with self.view.tab_summary:
                display(HTML(f"<p style='color: #6c757d;'>{message}</p>"))

        try:
            # Execute PET-PEESE engine
            result = self.engine.run_analysis(
                p_threshold=p_threshold,
                progress_callback=progress_callback
            )

            if result is None:
                self.view.render_error(
                    "Analysis failed",
                    "Unable to compute PET-PEESE analysis. Check your data (need ≥3 observations with variance column)."
                )
                return

            # Get prepared data for plotting
            df = self.data_manager.prepare_data()

            # Render all tabs
            self._render_all_tabs(result, df)

            # Save results
            self._save_results(result)

        except ValueError as e:
            self.view.render_error("Data Error", str(e))
        except RuntimeError as e:
            self.view.render_error("Runtime Error", str(e))
        except Exception as e:
            self._handle_unexpected_error(e)

    # -------------------------------------------------------------------------
    # TAB RENDERING
    # -------------------------------------------------------------------------

    def _render_all_tabs(
        self,
        result: 'PETPEESEResult',
        df: 'pd.DataFrame'
    ) -> None:
        """
        Render all tabs with results.

        Args:
            result: PETPEESEResult object
            df: Prepared DataFrame for plotting
        """
        # Tab 1: Summary
        self.view.render_summary_tab(result)

        # Tab 2: Visuals
        self.view.render_visuals_tab(result, df)

        # Tab 3: Interpretation
        self.view.render_interpretation_tab(result)

        # Tab 4: Publication Text
        combined_text = self.view.render_publication_tab(result)
        self.data_manager.save_publication_text(combined_text)

    # -------------------------------------------------------------------------
    # DATA PERSISTENCE
    # -------------------------------------------------------------------------

    def _save_results(self, result: 'PETPEESEResult') -> None:
        """
        Save results to ANALYSIS_CONFIG.

        Args:
            result: PETPEESEResult object
        """
        self.data_manager.save_petpeese_results(result)

    # -------------------------------------------------------------------------
    # EVENT HANDLERS
    # -------------------------------------------------------------------------

    def _handle_run_click(self, button) -> None:
        """Handle run button click event"""
        self.run_analysis()

    # -------------------------------------------------------------------------
    # ERROR HANDLING
    # -------------------------------------------------------------------------

    def _handle_initialization_error(self) -> None:
        """Handle errors during controller initialization"""
        error = self._initialization_error
        if error is None:
            return

        error_type = type(error).__name__
        error_msg = str(error)
        details = traceback.format_exc()

        self.view.render_error(f"{error_type}: {error_msg}", details)

    def _handle_unexpected_error(self, error: Exception) -> None:
        """Handle unexpected errors during analysis"""
        error_type = type(error).__name__
        error_msg = str(error)
        details = traceback.format_exc()

        self.view.render_error(f"{error_type}: {error_msg}", details)


# =============================================================================
# EXECUTION ENTRY POINT
# =============================================================================

def run_pet_peese_analysis():
    """
    Main entry point for PET-PEESE analysis.
    Call this function to display the UI and enable analysis.
    """
    try:
        # Check for ANALYSIS_CONFIG
        if 'ANALYSIS_CONFIG' not in globals():
            print("❌ ERROR: ANALYSIS_CONFIG not found")
            print("Please run previous analysis cells first:")
            print("  - Steps 2–6: Data loading and effect sizes")
            print("  - Step 7: Overall Meta-Analysis (recommended)")
            return

        # Check for three-level regression function
        if '_run_three_level_reml_regression_v2' not in globals():
            print("❌ ERROR: Three-level regression function not found")
            print("Please ensure the regression utilities are loaded.")
            return

        # Create controller
        controller = PETPEESEController(ANALYSIS_CONFIG)

        # Display UI
        ui = controller.create_ui()
        display(ui)

        # Display tabs
        display(controller.view.tabs)

        # Store controller globally for access (optional)
        globals()['_petpeese_controller'] = controller

    except Exception as e:
        print(f"❌ Fatal Error: {type(e).__name__}")
        print(f"Message: {str(e)}")
        print("\nFull traceback:")
        traceback.print_exc()

run_pet_peese_analysis()

In [ ]:
#@title 📉 19b. Publication Bias: Time-Lag Test (Decline Effect)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# STEP 19b: TIME-LAG BIAS (DECLINE EFFECT) TEST
# Purpose: Test whether effect sizes change with publication year, as when
#          large or significant results are published sooner (time-lag bias,
#          the "decline effect").
# Method:  Multilevel meta-regression (Nakagawa et al. 2022):
#              effect ~ year (centred) + precision
#          with effect sizes nested in studies, the full sampling VCV
#          (shared controls) and CR2 / Satterthwaite inference (when CR2 is on
#          in Step 7). The precision term (SE, or sqrt(1/n~) for effect sizes
#          whose SE depends on the effect itself) separates a decline over
#          time from a small-study effect. The year-only model is reported as
#          a sensitivity analysis.
# Data:    Step 6 (effect sizes, VCV) and Step 7 (settings); needs a 'year'
#          column. A plot recipe saved from this cell redraws the figure.
# Unlike the cumulative meta-analysis (Step 24), which shows how the pooled
# mean evolved, this gives a formal test of the year slope.
# =============================================================================

import numpy as np
import pandas as pd
import io
import base64
import warnings
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.stats import t as t_dist, norm
from IPython.display import display, HTML, clear_output, Javascript

TIMELAG_ESS_TYPES = ('hedges_g', 'cohen_d', 'lnRR', 'lnCVR')
TIMELAG_PRECISION_LABELS = {
    'ess': 'sqrt(1/ñ) (effective sample size)',
    'se': 'standard error (SE)',
    'none': 'none (year only)',
}


# =============================================================================
# PART 1: MODEL
# =============================================================================

def timelag_precision(df, kind, var_col):
    """Precision moderator for the time-lag model.

    'se'  : sqrt(v_i)
    'ess' : sqrt(1/n~), n~ = n_e n_c / (n_e + n_c) (Nakagawa et al. 2022),
            for effect sizes whose sampling variance depends on the effect
            itself (SMD, lnRR, lnCVR). Needs the group sample sizes.
    """
    if kind == 'se':
        return np.sqrt(df[var_col].to_numpy(dtype=np.float64))
    if kind == 'ess':
        ne = pd.to_numeric(df['ne'], errors='coerce').to_numpy(dtype=np.float64)
        nc = pd.to_numeric(df['nc'], errors='coerce').to_numpy(dtype=np.float64)
        return np.sqrt((ne + nc) / (ne * nc))
    return None


def fit_timelag_model(df, effect_col, var_col, vcv_dict, precision='se',
                      year_col='year', use_cr2=True, alpha=0.05, reference_df=None):
    """Multilevel meta-regression of the effect size on centred year
    (+ precision). Returns a dict with the coefficient table and the fit.

    Rows whose year (or precision) is missing are dropped and counted.
    """
    d = df.copy()
    d['_row'] = d.index
    d['_year'] = pd.to_numeric(d[year_col], errors='coerce')
    if precision != 'none':
        with np.errstate(divide='ignore', invalid='ignore'):
            d['_prec'] = timelag_precision(d, precision, var_col)
    ok_es = np.isfinite(d[effect_col]) & np.isfinite(d[var_col]) & (d[var_col] > 0)
    ok_year = np.isfinite(d['_year'])
    ok_prec = np.isfinite(d['_prec']) if precision != 'none' else pd.Series(True, index=d.index)
    keep = ok_es & ok_year & ok_prec
    n_dropped = int((~keep).sum())
    n_no_year = int((ok_es & ~ok_year).sum())
    n_no_prec = int((ok_es & ok_year & ~ok_prec).sum())
    d = d[keep]
    if d['id'].nunique() < 3:
        raise ValueError(f"Need at least 3 studies with a year; found {d['id'].nunique()}.")
    if d['_year'].nunique() < 2:
        raise ValueError("All effect sizes have the same year: the year slope cannot be estimated.")

    # Rows keep their position in the data the VCV matrices were built from,
    # so the shared-control covariances are sliced correctly.
    if vcv_dict and '_vcv_row_pos' not in d.columns:
        d = attach_vcv_row_pos(d, reference_df=df if reference_df is None else reference_df)

    year_mean = float(d['_year'].mean())
    d['_year_c'] = d['_year'] - year_mean
    terms = ['intercept', 'year'] + (['precision'] if precision != 'none' else [])

    y_blocks, V_blocks, V_diag, X_blocks, ids = [], [], [], [], []
    for sid, g in d.groupby('id', sort=False):
        y_blocks.append(g[effect_col].to_numpy(dtype=np.float64))
        Vd = np.diag(g[var_col].to_numpy(dtype=np.float64))
        V_diag.append(Vd)
        V_blocks.append(cluster_vcv(vcv_dict, sid, g, var_col)[0] if vcv_dict else Vd)
        cols = [np.ones(len(g)), g['_year_c'].to_numpy(dtype=np.float64)]
        if precision != 'none':
            cols.append(g['_prec'].to_numpy(dtype=np.float64))
        X_blocks.append(np.column_stack(cols))
        ids.append(sid)
    has_cov = any(np.count_nonzero(V - np.diag(np.diag(V))) for V in V_blocks)

    X_all = np.vstack(X_blocks)
    if np.linalg.matrix_rank(X_all) < X_all.shape[1]:
        raise ValueError("Year and precision are collinear in these data: the model cannot separate them. "
                         "Try precision = none.")

    fit = fit_blocks(y_blocks, V_blocks, X_blocks,
                     V_fallback_blocks=V_diag if has_cov else None,
                     cluster_ids=ids, term_names=terms)
    betas = np.asarray(fit.betas, dtype=np.float64)
    cov = np.asarray(fit.cov_beta, dtype=np.float64)
    p = len(betas)
    M = len(y_blocks)
    dfs = np.full(p, float(max(1, M - p)))
    htz = None
    used_cr2 = False
    if use_cr2:
        try:
            var_r, df_r, htz = fit_cr2(fit, with_wald_helper=True)
            se_r = np.sqrt(np.diag(var_r))
            if np.all(np.isfinite(se_r)) and np.all(se_r > 0):
                cov, dfs, used_cr2 = np.asarray(var_r), np.asarray(df_r, dtype=np.float64), True
            else:
                htz = None
        except Exception as e:
            warnings.warn(f"Time-lag test: CR2 step failed ({e}); using model-based SEs.")
            htz = None
    se = np.sqrt(np.diag(cov))
    tval = betas / se
    pval = 2 * t_dist.sf(np.abs(tval), dfs)
    crit = t_dist.ppf(1 - alpha / 2, dfs)
    table = pd.DataFrame({'term': terms, 'estimate': betas, 'se': se, 'df': dfs, 't': tval,
                          'p_value': pval, 'ci_lower': betas - crit * se, 'ci_upper': betas + crit * se})
    return {
        'table': table, 'fit': fit, 'cov': cov, 'dfs': dfs, 'htz': htz, 'used_cr2': used_cr2,
        'precision': precision, 'year_mean': year_mean,
        'year_range': (float(d['_year'].min()), float(d['_year'].max())),
        'prec_mean': float(d['_prec'].mean()) if precision != 'none' else None,
        'n_obs': int(len(d)), 'n_studies': int(M), 'n_dropped': n_dropped,
        'n_no_year': n_no_year, 'n_no_precision': n_no_prec,
        'tau2': float(fit.tau2), 'sigma2': float(fit.sigma2),
        'sampling_v': 'full (shared controls)' if has_cov and not fit.v_fallback else 'diagonal',
        'model_type': '3-level' if has_replication(y_blocks) else '2-level (one effect per study)',
        'data': d[['_row', 'id', '_year', effect_col, var_col] + (['_prec'] if precision != 'none' else [])]
                 .rename(columns={'_row': 'row', '_year': 'year', effect_col: 'yi', var_col: 'vi',
                                  '_prec': 'precision'})
                 .reset_index(drop=True),
        'alpha': alpha,
    }


def timelag_prediction(res, years, alpha=None):
    """Predicted effect along `years`, precision held at its mean, with a CI
    whose df is the contrast's own Satterthwaite df (HTZ closure at q = 1),
    as in the meta-regression plot."""
    alpha = res['alpha'] if alpha is None else alpha
    years = np.asarray(years, dtype=np.float64)
    rows = [np.ones_like(years), years - res['year_mean']]
    if res['precision'] != 'none':
        rows.append(np.full_like(years, res['prec_mean']))
    X0 = np.column_stack(rows)
    b = res['table']['estimate'].to_numpy()
    fit = X0 @ b
    se = np.sqrt(np.einsum('ij,jk,ik->i', X0, res['cov'], X0))
    dfs = np.empty(len(years))
    for i, x0 in enumerate(X0):
        df_i = None
        if res['htz'] is not None:
            try:
                df_i = float(res['htz'](x0[None, :], test='AHZ')['df_denom'])
            except Exception:
                df_i = None
        dfs[i] = df_i if df_i and np.isfinite(df_i) and df_i > 0 else float(np.min(res['dfs']))
    crit = t_dist.ppf(1 - alpha / 2, dfs)
    return fit, fit - crit * se, fit + crit * se


class TimeLagModel:
    """Runs the primary (year + precision) and the year-only models."""

    def __init__(self):
        self.cfg = None
        self.results = None

    def initialize(self):
        cfg = globals().get('ANALYSIS_CONFIG')
        if not cfg or cfg.get('analysis_data') is None:
            raise ValueError("No effect sizes found. Run Steps 2–6 (data and effect sizes) first.")
        if 'year' not in cfg['analysis_data'].columns:
            raise ValueError("'year' column not found. Add a publication-year column to your data "
                             "(mapped or named 'year') to run the time-lag test.")
        self.cfg = cfg
        return self

    def default_precision(self):
        kind = self.cfg.get('effect_size_type', '')
        df = self.cfg['analysis_data']
        if kind in TIMELAG_ESS_TYPES and {'ne', 'nc'} <= set(df.columns):
            return 'ess'
        return 'se'

    def precision_options(self):
        df = self.cfg['analysis_data']
        opts = [('Standard error (SE)', 'se')]
        if {'ne', 'nc'} <= set(df.columns):
            opts.insert(0, ('sqrt(1/ñ), effective sample size', 'ess'))
        opts.append(('None (year only)', 'none'))
        return opts

    def run(self, precision):
        cfg = self.cfg
        gs = cfg.get('global_settings') or {}
        common = dict(effect_col=cfg['effect_col'], var_col=cfg['var_col'],
                      vcv_dict=cfg.get('vcv_matrices') or {},
                      use_cr2=bool(gs.get('use_kh', True)), alpha=float(gs.get('alpha', 0.05)),
                      reference_df=cfg['analysis_data'])
        primary = fit_timelag_model(cfg['analysis_data'], precision=precision, **common)
        # Year-only sensitivity model on exactly the rows of the primary model,
        # so the comparison changes the model, not the sample.
        year_only = (primary if precision == 'none'
                     else fit_timelag_model(cfg['analysis_data'].loc[primary['data']['row']],
                                            precision='none', **common))
        mu = None
        rep = (cfg.get('overall_results') or {}).get('reported') or {}
        if rep.get('mu') is not None:
            mu = float(rep['mu'])
        self.results = {
            'primary': primary, 'year_only': year_only, 'precision': precision,
            'overall_mu': mu, 'es_config': cfg.get('es_config') or {},
            'effect_size_type': cfg.get('effect_size_type', ''),
            'run_at': cometa_timestamp(),
        }
        cfg['timelag_results'] = {
            'precision': precision,
            'primary': primary['table'].to_dict('records'),
            'year_only': year_only['table'].to_dict('records'),
            'n_obs': primary['n_obs'], 'n_studies': primary['n_studies'],
            'n_dropped': primary['n_dropped'], 'year_range': primary['year_range'],
            'used_cr2': primary['used_cr2'], 'sampling_v': primary['sampling_v'],
            'tau2': primary['tau2'], 'sigma2': primary['sigma2'], 'run_at': self.results['run_at'],
        }
        return self.results


def timelag_verdict(results):
    """Plain-language reading of the year slope."""
    prim = results['primary']
    row = prim['table'].set_index('term').loc['year']
    alpha = prim['alpha']
    mu = results['overall_mu']
    if mu is None:
        mu = float(results['year_only']['table'].set_index('term').loc['intercept', 'estimate'])
    sig = row['p_value'] < alpha
    if not sig:
        return ('none', "No evidence that effect sizes change with publication year "
                        "(this does not show that there is no time-lag bias; the test has limited power).")
    if mu == es_null_value():
        return ('increase', "Effect sizes change with publication year, but the overall mean is at the "
                            "no-effect value, so the change cannot be read as a decline toward zero.")
    toward_zero = (row['estimate'] < 0) == (mu > es_null_value())
    if toward_zero:
        return ('decline', "Effect sizes move toward zero in more recent studies, the pattern expected "
                           "under time-lag bias (a decline effect).")
    return ('increase', "Effect sizes move away from zero in more recent studies: not the pattern of "
                        "time-lag bias; it may reflect changes in methods, populations or conditions over time.")


# =============================================================================
# PART 2: TEXT
# =============================================================================

TIMELAG_REFS = {
    'nakagawa2022': "Nakagawa, S., Lagisz, M., Jennions, M. D., et al. (2022). Methods for testing publication bias in "
                    "ecological and evolutionary meta-analyses. <i>Methods in Ecology and Evolution</i>, 13(1), "
                    "4-21. https://doi.org/10.1111/2041-210X.13724",
    'jennions2002': "Jennions, M. D., & Møller, A. P. (2002). Relationships fade with time: a meta-analysis of "
                    "temporal trends in publication in ecology and evolution. <i>Proceedings of the Royal Society "
                    "B</i>, 269(1486), 43-48. https://doi.org/10.1098/rspb.2001.1832",
    'crve': "Pustejovsky, J. E., & Tipton, E. (2018). Small-sample methods for cluster-robust variance estimation "
            "and hypothesis testing in fixed effects models. <i>Journal of Business & Economic Statistics</i>, "
            "36(4), 672-683.",
}


def _fmt_p(p):
    return "&lt; 0.001" if p < 0.001 else f"= {p:.3f}"


def timelag_slope_text(row, es_cfg, effect_size_type, per=10):
    """'b = ... per decade', with the ratio change for log-scale effect sizes."""
    b10 = row['estimate'] * per
    lo10, hi10 = row['ci_lower'] * per, row['ci_upper'] * per
    txt = f"<i>b</i> = {row['estimate']:.4f} per year ({b10:.3f} per decade, {ci_pct_label()} CI [{lo10:.3f}, {hi10:.3f}])"
    if es_cfg.get('scale') == 'log' or effect_size_type in ('lnRR', 'lnCVR', 'log_or', 'log_rr'):
        pct = (np.exp(b10) - 1) * 100
        txt += f"; on the ratio scale, {pct:+.1f}% per decade"
    return txt


def timelag_excluded_text(prim):
    """'3 effect sizes without a year and 1 without group sizes', etc."""
    parts = []
    if prim.get('n_no_year'):
        parts.append(f"{prim['n_no_year']} effect size{'s' if prim['n_no_year'] != 1 else ''} without a year")
    if prim.get('n_no_precision'):
        what = 'usable group sizes' if prim['precision'] == 'ess' else 'a usable standard error'
        parts.append(f"{prim['n_no_precision']} without {what}")
    other = prim['n_dropped'] - prim.get('n_no_year', 0) - prim.get('n_no_precision', 0)
    if other:
        parts.append(f"{other} without a valid effect size or variance")
    return ' and '.join(parts) if parts else f"{prim['n_dropped']} effect sizes"


def timelag_methods_html(results):
    prim = results['primary']
    es_label = results['es_config'].get('effect_label', 'effect size')
    prec = results['precision']
    prec_txt = {
        'ess': ("a precision term, the inverse square root of the effective sample size, √(1/ñ) with ñ = "
                "<i>n</i><sub>e</sub><i>n</i><sub>c</sub>/(<i>n</i><sub>e</sub> + <i>n</i><sub>c</sub>), "
                "which, unlike the standard error, does not depend on the effect size itself"),
        'se': "a precision term, the standard error of each effect size",
    }.get(prec)
    inf = ("Coefficients were tested with cluster-robust (CR2) standard errors, clustered by study, and "
           "Satterthwaite degrees of freedom [3]." if prim['used_cr2'] else
           f"Coefficients were tested with model-based standard errors and <i>t</i>-tests with "
           f"{prim['n_studies']} − {len(prim['table'])} degrees of freedom.")
    refs = [TIMELAG_REFS['nakagawa2022'], TIMELAG_REFS['jennions2002']]
    if prim['used_cr2']:
        refs.append(TIMELAG_REFS['crve'])
    refs.append(COMETA_CITATION)
    v_txt = (", including the covariances between effect sizes that share a control group"
             if prim['sampling_v'].startswith('full') else "")
    html = f"""
    <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt; line-height: 1.8;
                padding: 20px; background-color: #ffffff; border: 1px solid #eee;'>
    <h3 style='color:#2c3e50; border-bottom:2px solid #3498db; padding-bottom:10px; margin-top:0;'>Materials and Methods</h3>
    <p style='text-align: justify;'>
    <b>Time-lag bias.</b> To test for time-lag bias (a decline effect), in which larger or statistically
    significant effects are published sooner than smaller ones [1, 2], we fitted a multilevel meta-regression
    of {es_label} on publication year (mean-centred) with effect sizes nested within studies and the
    sampling variance–covariance matrix of the overall analysis{v_txt}.
    """
    if prec_txt:
        html += (f"Following Nakagawa et al. [1], the model also included {prec_txt}, so that the year slope "
                 f"is estimated independently of small-study effects. The model with year alone is reported "
                 f"as a sensitivity analysis. ")
    html += f"""{inf} A year slope opposite in sign to the overall mean effect indicates that effect sizes
    decline toward zero over time. All analyses were performed with CoMeta [{len(refs)}].
    </p>
    <h4 style='color:#34495e; font-size:11pt;'>References</h4>
    <ol style='font-size:10pt; color:#555;'>{''.join(f'<li>{r}</li>' for r in refs)}</ol>
    </div>"""
    return html


def timelag_results_html(results):
    prim, yo = results['primary'], results['year_only']
    es_cfg, kind = results['es_config'], results['effect_size_type']
    row = prim['table'].set_index('term').loc['year']
    code, verdict = timelag_verdict(results)
    y0, y1 = prim['year_range']
    html = f"""
    <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt; line-height: 1.8;
                padding: 20px; background-color: #ffffff; border: 1px solid #eee;'>
    <h3 style='color:#2c3e50; border-bottom:2px solid #3498db; padding-bottom:10px; margin-top:0;'>Results</h3>
    <p style='text-align: justify;'>
    The time-lag analysis included <b>{prim['n_obs']}</b> effect sizes from <b>{prim['n_studies']}</b> studies
    published between {y0:.0f} and {y1:.0f}"""
    if prim['n_dropped']:
        html += f" ({timelag_excluded_text(prim)} were excluded)"
    html += (f". The year slope was {timelag_slope_text(row, es_cfg, kind)}, "
             f"<i>t</i>({row['df']:.1f}) = {row['t']:.2f}, <i>p</i> {_fmt_p(row['p_value'])}. {verdict}")
    if results['precision'] != 'none':
        pr = prim['table'].set_index('term').loc['precision']
        lab = 'the effective-sample-size term' if results['precision'] == 'ess' else 'the standard error'
        html += (f" In the same model, the slope on {lab} was <i>b</i> = {pr['estimate']:.3f} "
                 f"({ci_pct_label()} CI [{pr['ci_lower']:.3f}, {pr['ci_upper']:.3f}], <i>p</i> {_fmt_p(pr['p_value'])}).")
        yr = yo['table'].set_index('term').loc['year']
        html += (f" With year as the only moderator, the slope was <i>b</i> = {yr['estimate']:.4f} per year "
                 f"({ci_pct_label()} CI [{yr['ci_lower']:.4f}, {yr['ci_upper']:.4f}], <i>p</i> {_fmt_p(yr['p_value'])}).")
    html += "</p></div>"
    return html


# =============================================================================
# PART 3: VIEW & CONTROLLER
# =============================================================================

class TimeLagView:

    def __init__(self):
        self.tab_results, self.tab_plot = widgets.Output(), widgets.Output()
        self.tab_text, self.tab_log, self.tab_export = widgets.Output(), widgets.Output(), widgets.Output()
        self.tabs = widgets.Tab(children=[self.tab_results, self.tab_plot, self.tab_text,
                                          self.tab_log, self.tab_export])
        for i, name in enumerate(['📊 Results', '📈 Plot', '📝 Publication Text', '⚙️ Settings & Log', '💾 Export']):
            self.tabs.set_title(i, name)
        self.precision = widgets.Dropdown(description='Precision term:', style={'description_width': '120px'},
                                          layout=widgets.Layout(width='450px'))
        self.run_button = widgets.Button(description='▶ Run Time-Lag Test', button_style='primary',
                                         icon='play', layout=widgets.Layout(width='300px'))
        self.export_button = widgets.Button(description='📥 Download Time-Lag Report', button_style='info',
                                            icon='file-excel', layout=widgets.Layout(width='300px', height='40px'))
        self._build_plot_controls()

    # --- plot controls -------------------------------------------------------
    def _build_plot_controls(self):
        W = lambda w='300px': widgets.Layout(width=w)
        st = {'description_width': '110px'}
        self.c = {
            'width': widgets.FloatSlider(value=6.5, min=2, max=16, step=0.1, description='Width (in):', style=st, layout=W()),
            'height': widgets.FloatSlider(value=4.5, min=2, max=12, step=0.1, description='Height (in):', style=st, layout=W()),
            'title_size': widgets.IntSlider(value=12, min=6, max=24, description='Title size:', style=st, layout=W()),
            'label_size': widgets.IntSlider(value=11, min=6, max=22, description='Label size:', style=st, layout=W()),
            'tick_size': widgets.IntSlider(value=9, min=5, max=20, description='Tick size:', style=st, layout=W()),
            'title': widgets.Text(value='Time-lag test', description='Title:', style=st, layout=W('450px')),
            'xlabel': widgets.Text(value='Publication year', description='X label:', style=st, layout=W('450px')),
            'ylabel': widgets.Text(value='', description='Y label:', style=st, layout=W('450px')),
            'point_color': widgets.ColorPicker(value='#0072B2', description='Points:', style=st, layout=W()),
            'line_color': widgets.ColorPicker(value='#D55E00', description='Line:', style=st, layout=W()),
            'point_alpha': widgets.FloatSlider(value=0.55, min=0.1, max=1, step=0.05, description='Opacity:', style=st, layout=W()),
            'size_scale': widgets.FloatSlider(value=1.0, min=0.2, max=4, step=0.1, description='Bubble size:', style=st, layout=W()),
            'show_band': widgets.Checkbox(value=True, description='Confidence band', indent=False),
            'show_null': widgets.Checkbox(value=True, description='No-effect line', indent=False),
            'show_stats': widgets.Checkbox(value=True, description='Slope and p in plot', indent=False),
            'save_pdf': widgets.Checkbox(value=True, description='PDF', indent=False),
            'save_png': widgets.Checkbox(value=True, description='PNG', indent=False),
            'save_svg': widgets.Checkbox(value=False, description='SVG', indent=False),
            'dpi': widgets.IntSlider(value=300, min=72, max=1200, step=50, description='PNG DPI:', style=st, layout=W()),
            'transparent': widgets.Checkbox(value=False, description='Transparent background', indent=False),
        }
        self.preset = preset_dropdown({'width': self.c['width'], 'height': self.c['height'],
                                       'title': self.c['title_size'], 'label': self.c['label_size'],
                                       'tick': self.c['tick_size']})
        self.draw_button = widgets.Button(description='🎨 Draw Plot', button_style='success',
                                          layout=widgets.Layout(width='200px'))
        self.save_button = widgets.Button(description='💾 Save Figure', button_style='warning',
                                          layout=widgets.Layout(width='200px'))
        self.plot_out, self.save_out, self.caption_out = widgets.Output(), widgets.Output(), widgets.Output()

    def plot_panel(self, recipe_bar):
        c = self.c
        style_box = widgets.VBox([self.preset, c['width'], c['height'], c['title_size'], c['label_size'], c['tick_size']])
        text_box = widgets.VBox([c['title'], c['xlabel'], c['ylabel']])
        look_box = widgets.VBox([c['point_color'], c['line_color'], c['point_alpha'], c['size_scale'],
                                 widgets.HBox([c['show_band'], c['show_null'], c['show_stats']])])
        save_box = widgets.VBox([widgets.HBox([c['save_pdf'], c['save_png'], c['save_svg']]), c['dpi'],
                                 c['transparent'], self.save_button, self.save_out])
        acc = widgets.Tab(children=[style_box, text_box, look_box, save_box])
        for i, n in enumerate(['📏 Size & fonts', '🔤 Text', '🎨 Look', '💾 Export']):
            acc.set_title(i, n)
        return widgets.VBox([acc, self.draw_button, self.plot_out, self.caption_out, recipe_bar])


class TimeLagController:
    PLOT_TYPE = 'timelag'

    def __init__(self):
        self.model = TimeLagModel()
        self.view = TimeLagView()
        self.plot_data = None        # what the figure draws (from results or a recipe)
        self.plot_source = None
        self.from_recipe = None
        self.last_fig = None
        self.view.run_button.on_click(self.on_run)
        self.view.export_button.on_click(self.on_export)
        self.view.draw_button.on_click(lambda b: self.draw())
        self.view.save_button.on_click(self.on_save_figure)
        self.recipe_bar = plot_recipe_bar(self.PLOT_TYPE, self.get_recipe, self.on_load_recipe,
                                          filename_prefix=lambda: 'TimeLag')

    # --- running the test ----------------------------------------------------
    def setup_options(self):
        try:
            self.model.initialize()
            self.view.precision.options = self.model.precision_options()
            self.view.precision.value = self.model.default_precision()
            return None
        except Exception as e:
            return str(e)

    def on_run(self, _b=None):
        v = self.view
        for tab in (v.tab_results, v.tab_text, v.tab_log, v.tab_export):
            tab.clear_output()
        try:
            self.model.initialize()
            res = self.model.run(v.precision.value)
        except Exception as e:
            with v.tab_results:
                display(HTML(f"<div style='color:#721c24; padding:10px; background:#f8d7da; border-radius:4px;'>"
                             f"❌ {e}</div>"))
            return
        self.render_results(res)
        self.plot_data = self._plot_data_from(res)
        self.plot_source = {'analysis': f"Time-lag test (precision: {TIMELAG_PRECISION_LABELS[res['precision']]})",
                            'effect_label': res['es_config'].get('effect_label', ''),
                            'model': res['primary']['model_type'], 'ci_level': ci_pct_label(),
                            'analysis_time': res['run_at']}
        self.from_recipe = None
        if not v.c['ylabel'].value:
            v.c['ylabel'].value = res['es_config'].get('effect_label', 'Effect size')
        self.draw()

    def render_results(self, res):
        v, prim, yo = self.view, res['primary'], res['year_only']
        code, verdict = timelag_verdict(res)
        colour = {'decline': ('#721c24', '#f8d7da', '⚠️'), 'increase': ('#856404', '#fff3cd', 'ℹ️'),
                  'none': ('#155724', '#d4edda', '✓')}[code]
        row = prim['table'].set_index('term').loc['year']
        fmt = {'estimate': '{:.5f}', 'se': '{:.5f}', 'df': '{:.1f}', 't': '{:.3f}', 'p_value': '{:.4g}',
               'ci_lower': '{:.5f}', 'ci_upper': '{:.5f}'}
        with v.tab_results:
            clear_output()
            display(HTML(
                f"<div style='padding:14px; background:{colour[1]}; color:{colour[0]}; border-radius:6px; "
                f"font-family:sans-serif;'><h3 style='margin:0 0 6px 0;'>{colour[2]} Year slope: "
                f"b = {row['estimate']:.4f} per year, p = {row['p_value']:.4g}</h3>{verdict}</div>"))
            display(HTML(f"<h4 style='font-family:sans-serif;'>Primary model: effect ~ year (centred at "
                         f"{prim['year_mean']:.1f})" + ("" if res['precision'] == 'none' else
                         f" + {TIMELAG_PRECISION_LABELS[res['precision']]}") + "</h4>"))
            display(prim['table'].style.format(fmt).hide(axis='index'))
            if res['precision'] != 'none':
                display(HTML("<h4 style='font-family:sans-serif;'>Sensitivity: year only</h4>"))
                display(yo['table'].style.format(fmt).hide(axis='index'))
            display(HTML(
                f"<div style='font-family:sans-serif; font-size:12px; color:#555; margin-top:8px;'>"
                f"{prim['n_obs']} effect sizes, {prim['n_studies']} studies ({prim['year_range'][0]:.0f}–"
                f"{prim['year_range'][1]:.0f}); model: {prim['model_type']}, τ² = {prim['tau2']:.4g}, "
                f"σ² = {prim['sigma2']:.4g}; sampling V: {prim['sampling_v']}; inference: "
                f"{'CR2 + Satterthwaite df' if prim['used_cr2'] else 'model-based, df = studies − p'}. "
                f"The intercept is the effect at the mean year" +
                ("" if res['precision'] == 'none' else " for a study with zero precision term (infinitely large)")
                + ". A decline effect is a year slope opposite in sign to the overall mean."
                + (f" Excluded: {timelag_excluded_text(prim)}." if prim['n_dropped'] else "")
                + "</div>"))
        with v.tab_text:
            clear_output()
            display(HTML(timelag_methods_html(res)))
            display(HTML(timelag_results_html(res)))
        with v.tab_log:
            clear_output()
            settings = pd.DataFrame([
                ('Precision term', TIMELAG_PRECISION_LABELS[res['precision']]),
                ('Year centring', f"mean of the effect sizes' years ({prim['year_mean']:.2f})"),
                ('Model', prim['model_type'] + ' REML (Cell 1b engine)'),
                ('Sampling VCV', prim['sampling_v']),
                ('Inference', 'CR2 + Satterthwaite' if prim['used_cr2'] else 'model-based t'),
                ('CI level', ci_pct_label()),
                ('Effect sizes used', prim['n_obs']), ('Studies used', prim['n_studies']),
                ('Excluded', timelag_excluded_text(prim) if prim['n_dropped'] else 'none'),
                ('Run at', res['run_at']),
            ], columns=['Setting', 'Value'])
            display(settings.style.hide(axis='index'))
        with v.tab_export:
            clear_output()
            display(HTML("<h3 style='color:#2E86AB;'>💾 Download Time-Lag Report</h3>"
                         "<p>Excel file with both models, the settings and the data used.</p>"))
            display(v.export_button)

    # --- plot ------------------------------------------------------------------
    def _plot_data_from(self, res):
        prim = res['primary']
        d = prim['data']
        w = 1.0 / (d['vi'].to_numpy() + prim['tau2'] + prim['sigma2'])
        grid = np.linspace(prim['year_range'][0], prim['year_range'][1], 60)
        fit, lo, hi = timelag_prediction(prim, grid)
        row = prim['table'].set_index('term').loc['year']
        return {'year': d['year'].tolist(), 'yi': d['yi'].tolist(), 'weight': w.tolist(),
                'grid': grid.tolist(), 'fit': fit.tolist(), 'lo': lo.tolist(), 'hi': hi.tolist(),
                'slope': float(row['estimate']), 'p': float(row['p_value']),
                'precision': res['precision'], 'null': es_null_value(),
                'n_obs': prim['n_obs'], 'n_studies': prim['n_studies']}

    def draw(self):
        v, c, pdat = self.view, self.view.c, self.plot_data
        with v.plot_out:
            clear_output(wait=True)
            display(HTML(source_html(self.plot_source, self.from_recipe)))
            if pdat is None:
                return
            fig, ax = plt.subplots(figsize=(c['width'].value, c['height'].value))
            w = np.asarray(pdat['weight'])
            sizes = 15 + 185 * (w / w.max()) * c['size_scale'].value
            ax.scatter(pdat['year'], pdat['yi'], s=sizes, color=c['point_color'].value,
                       alpha=c['point_alpha'].value, edgecolor='white', linewidth=0.5, zorder=2)
            if c['show_band'].value:
                ax.fill_between(pdat['grid'], pdat['lo'], pdat['hi'], color=c['line_color'].value,
                                alpha=0.18, linewidth=0, zorder=1)
            ax.plot(pdat['grid'], pdat['fit'], color=c['line_color'].value, linewidth=2, zorder=3)
            if c['show_null'].value:
                ax.axhline(pdat['null'], color='#555555', linestyle='--', linewidth=0.8, zorder=0)
            if c['show_stats'].value:
                ptxt = 'p < 0.001' if pdat['p'] < 0.001 else f"p = {pdat['p']:.3f}"
                ax.text(0.02, 0.97, f"slope = {pdat['slope']:.4f} / year, {ptxt}", transform=ax.transAxes,
                        va='top', fontsize=c['tick_size'].value, zorder=5,
                        bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='none', alpha=0.85))
            ax.set_title(c['title'].value, fontsize=c['title_size'].value)
            ax.set_xlabel(c['xlabel'].value, fontsize=c['label_size'].value)
            ax.set_ylabel(c['ylabel'].value, fontsize=c['label_size'].value)
            ax.tick_params(labelsize=c['tick_size'].value)
            for side in ('top', 'right'):
                ax.spines[side].set_visible(False)
            fig.tight_layout()
            self.last_fig = fig
            show_figure(fig)
        with v.caption_out:
            clear_output()
            held = ("" if pdat['precision'] == 'none' else
                    f", with {TIMELAG_PRECISION_LABELS[pdat['precision']]} held at its mean")
            display(HTML(caption_html(
                f"Effect sizes against publication year ({pdat['n_obs']} effect sizes from {pdat['n_studies']} "
                f"studies). Bubble size is proportional to the weight in the multilevel model. The line is the "
                f"predicted effect from the multilevel meta-regression{held}, with its {ci_pct_label()} "
                f"confidence band; the dashed line marks no effect.")))

    def on_save_figure(self, _b=None):
        c = self.view.c
        with self.view.save_out:
            clear_output()
            if self.last_fig is None:
                display(HTML("<div style='color:#721c24;'>❌ Draw the plot first.</div>"))
                return
            paths = cometa_save_figure(self.last_fig, 'TimeLag_plot', pdf=c['save_pdf'].value,
                                       png=c['save_png'].value, svg=c['save_svg'].value,
                                       dpi=c['dpi'].value, transparent=c['transparent'].value)
            display(HTML(saved_files_html(paths)))

    def get_recipe(self):
        if self.plot_data is None:
            return None
        return build_plot_recipe(self.PLOT_TYPE, self.plot_data, widget_values(self.view.c), self.plot_source)

    def on_load_recipe(self, rec, style_only, name):
        skipped = apply_widget_values(self.view.c, rec.get('settings') or {})
        if not style_only:
            self.plot_data = rec['data']
            self.plot_source = rec.get('source')
            self.from_recipe = name
        self.draw()
        return skipped

    # --- export ----------------------------------------------------------------
    def on_export(self, _b=None):
        res = self.model.results
        with self.view.tab_export:
            if res is None:
                display(HTML("<div style='color:#721c24;'>❌ Run the test first.</div>"))
                return
            try:
                buffer = io.BytesIO()
                with pd.ExcelWriter(buffer, engine='xlsxwriter') as writer:
                    res['primary']['table'].to_excel(writer, sheet_name='Primary model', index=False)
                    res['year_only']['table'].to_excel(writer, sheet_name='Year only', index=False)
                    prim = res['primary']
                    pd.DataFrame([
                        ('Precision term', TIMELAG_PRECISION_LABELS[res['precision']]),
                        ('Year mean (centring)', prim['year_mean']), ('tau2', prim['tau2']),
                        ('sigma2', prim['sigma2']), ('Sampling VCV', prim['sampling_v']),
                        ('Inference', 'CR2 + Satterthwaite' if prim['used_cr2'] else 'model-based'),
                        ('Effect sizes', prim['n_obs']), ('Studies', prim['n_studies']),
                        ('Excluded', timelag_excluded_text(prim) if prim['n_dropped'] else 'none'),
                        ('Verdict', timelag_verdict(res)[1]),
                        ('Run at', res['run_at']),
                    ], columns=['Setting', 'Value']).to_excel(writer, sheet_name='Settings', index=False)
                    prim['data'].to_excel(writer, sheet_name='Data used', index=False)
                    text = _clean_html_tags(timelag_methods_html(res) + timelag_results_html(res))
                    pd.DataFrame({'Publication text': [text]}).to_excel(writer, sheet_name='Text', index=False)
                buffer.seek(0)
                b64 = base64.b64encode(buffer.read()).decode()
                fname = f"TimeLag_Report_{cometa_timestamp()}.xlsx"
                display(Javascript(
                    "var link = document.createElement('a'); link.href = 'data:application/"
                    "vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64," + b64 + "'; link.download = '"
                    + fname + "'; document.body.appendChild(link); link.click(); document.body.removeChild(link);"))
                display(HTML(f"<div style='color:#155724;'>✅ Downloaded <code>{fname}</code></div>"))
            except Exception as e:
                display(HTML(f"<div style='color:#721c24;'>❌ Export failed: {e}</div>"))

    # --- UI ----------------------------------------------------------------------
    def start(self):
        err = self.setup_options()
        display(HTML(
            "<h3 style='color:#2E86AB;'>📉 Time-Lag Test (Decline Effect)</h3>"
            "<p style='color:#555; font-family:sans-serif; max-width:850px;'>Tests whether effect sizes change "
            "with publication year, as expected when large or significant results are published sooner. "
            "A multilevel meta-regression of the effect on publication year, with a precision term so that a "
            "decline over time is not confused with a small-study effect (Nakagawa et al. 2022). For effect "
            "sizes whose SE depends on the effect itself (Hedges' g, Cohen's d, lnRR, lnCVR) the precision "
            "term defaults to √(1/ñ), the effective sample size. Unlike the cumulative meta-analysis "
            "(Step 24), this gives a formal test.</p>"))
        if err:
            display(HTML(f"<div style='padding:10px; background:#fff3cd; border-left:4px solid #ffc107;'>⚠️ {err}</div>"))
            return
        with self.view.tab_plot:
            display(self.view.plot_panel(self.recipe_bar))
        with self.view.plot_out:
            display(HTML(source_html(None)))
        display(widgets.VBox([self.view.precision, self.view.run_button]))
        display(self.view.tabs)


timelag_controller = TimeLagController()
timelag_controller.start()


In [ ]:
#@title 📉 20. Publication Bias: Funnel Plot
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# STEP 20: PUBLICATION-READY FUNNEL PLOT
# Purpose: Contour-enhanced funnel plot with the pooled effect of the reported
#          model and the result of Egger's regression test.
# Data:    Step 7 (Overall Meta-Analysis) and Step 17 (Egger's Test), or a plot
#          recipe file saved from this cell.
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from scipy.stats import norm
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import html


def _funnel_plot_cell():
    PLOT_TYPE = 'funnel'
    RESULTS_KEY = 'overall_results'
    state = {'data': None, 'source': None, 'recipe_name': None, 'token': None, 'token_egger': None}

    # two-sided p thresholds of the significance contours (metafor's level = c(90, 95, 99))
    CONTOUR_P = (0.10, 0.05, 0.01)
    CONTOUR_SHADES = {
        'metafor': ('white', '#8c8c8c', '#bfbfbf', '#d3d3d3'),
        'gray': ('white', '#e6e6e6', '#c8c8c8', '#a6a6a6'),
        'blue': ('white', '#deebf7', '#9ecae1', '#4292c6'),
    }

    # --- 1. DATA SNAPSHOT (everything the figure draws) ---
    def build_data():
        res = ANALYSIS_CONFIG.get(RESULTS_KEY)
        df = ANALYSIS_CONFIG.get('analysis_data')
        rep = (res or {}).get('reported') or {}
        if not res or df is None or rep.get('mu') is None:
            return None, None
        es = ANALYSIS_CONFIG.get('es_config') or {}
        eff_col, var_col = ANALYSIS_CONFIG['effect_col'], ANALYSIS_CONFIG['var_col']
        alpha = _alpha()

        excluded = {eff_col, var_col, ANALYSIS_CONFIG.get('se_col'), 'w_fixed', 'w_random', 'id',
                    'xe', 'sde', 'ne', 'xc', 'sdc', 'nc'}
        moderators = {}
        for col in df.columns:
            if col in excluded or col is None:
                continue
            if is_text_dtype(df[col]) or isinstance(df[col].dtype, pd.CategoricalDtype):
                if 0 < df[col].nunique() <= 15:
                    moderators[str(col)] = [None if pd.isna(v) else str(v) for v in df[col].tolist()]

        egger = None
        fr = ANALYSIS_CONFIG.get('funnel_results')
        if fr and fr.get('p_value') is not None and np.isfinite(fr['p_value']):
            est = fr.get('estimates') or {}
            egger = {'p_value': float(fr['p_value']), 'slope': float(fr.get('beta_slope', np.nan)),
                     'se': float(fr.get('se', np.nan)), 'df': fr.get('df', est.get('df')),
                     'used_robust': bool(fr.get('used_robust', est.get('used_robust', False)))}

        data = {
            'yi': df[eff_col].astype(float).tolist(),
            'se': np.sqrt(df[var_col].astype(float)).tolist(),
            'moderators': moderators,
            'pooled': {'mu': float(rep['mu']), 'ci_lower': rep.get('ci_lower'), 'ci_upper': rep.get('ci_upper'),
                       'model': rep.get('model', '')},
            'egger': egger,
            'alpha': alpha,
            'ci_level': ci_pct_label(alpha),
            'null_value': es_null_value(),
            'effect_label': es.get('effect_label', 'Effect Size'),
        }
        when = res.get('timestamp')
        egg_txt = f", Egger p = {egger['p_value']:.4f}" if egger else ", Egger's test not run"
        source = {
            'analysis': f"Funnel plot (Step 7 model{egg_txt})",
            'effect_label': data['effect_label'],
            'model': data['pooled']['model'],
            'ci_level': data['ci_level'],
            'analysis_time': when.strftime('%Y-%m-%d %H:%M') if hasattr(when, 'strftime') else when,
        }
        return data, source

    # --- 2. CONTROLS (every widget whose value is saved in a recipe) ---
    L = lambda w='450px': widgets.Layout(width=w)
    CB = lambda v, d: widgets.Checkbox(value=v, description=d, indent=False, layout=L())
    C = {
        'title': widgets.Text(value='Funnel Plot', description='Title:', layout=L()),
        'show_title': CB(True, 'Show Title'),
        'xlabel': widgets.Text(value='Effect Size', description='X Label:', layout=L()),
        'ylabel': widgets.Text(value='Standard Error', description='Y Label:', layout=L()),
        'width': widgets.FloatSlider(value=10.0, min=2.0, max=16.0, step=0.1, description='Width (in):',
                                     continuous_update=False, layout=L()),
        'height': widgets.FloatSlider(value=7.0, min=2.0, max=12.0, step=0.1, description='Height (in):',
                                      continuous_update=False, layout=L()),
        'title_font': widgets.IntSlider(value=14, min=6, max=24, description='Title Font:', layout=L()),
        'label_font': widgets.IntSlider(value=12, min=6, max=18, description='Label Font:', layout=L()),
        'tick_font': widgets.IntSlider(value=10, min=5, max=16, description='Tick Font:', layout=L()),
        'legend_font': widgets.IntSlider(value=10, min=5, max=16, description='Legend Font:', layout=L()),
        'grid': CB(True, 'Show Grid'),
        'legend_loc': widgets.Dropdown(options=[('Upper Right', 'upper right'), ('Upper Left', 'upper left'),
                                                ('Lower Right', 'lower right'), ('Lower Left', 'lower left'),
                                                ('Best', 'best'), ('Outside Right', 'outside'), ('None', 'none')],
                                       value='upper right', description='Legend:', layout=L()),
        'color_by': widgets.Dropdown(options=['None'], value='None', description='Color By:', layout=L('400px')),
        'palette': widgets.Dropdown(options=[('Tab10', 'tab10'), ('Set1', 'Set1'), ('Set2', 'Set2'),
                                             ('Dark2', 'Dark2'), ('Paired', 'Paired'), ('Viridis', 'viridis')],
                                    value='tab10', description='Palette:'),
        'point_color': widgets.Dropdown(options=['gray', 'steelblue', 'black', 'red', 'purple'], value='black',
                                        description='Color:'),
        'point_size': widgets.IntSlider(value=50, min=10, max=200, step=10, description='Size:'),
        'point_alpha': widgets.FloatSlider(value=0.6, min=0.1, max=1.0, step=0.1, description='Opacity:'),
        'show_contours': CB(True, 'Show Significance Contours (p = 0.10, 0.05, 0.01)'),
        'contour_shades': widgets.Dropdown(options=[('Gray (as metafor)', 'metafor'), ('Gray (light to dark)', 'gray'),
                                                    ('Blue (light to dark)', 'blue')],
                                           value='metafor', description='Contour Shades:', layout=L()),
        'show_null': CB(False, 'Show No-Effect Line'),
        'fill_funnel': CB(True, 'Fill Confidence Region'),
        'fill_color': widgets.ColorPicker(value='#e6f3ff', description='Fill Color:'),
        'fill_alpha': widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.1, description='Fill Opacity:'),
        'center_color': widgets.ColorPicker(value='#2c3e50', description='Center Line:'),
        'show_egger': CB(True, "Show Egger's Test Stats"),
        'egger_loc': widgets.Dropdown(options=[('Lower Right', 'lower right'), ('Lower Left', 'lower left'),
                                               ('Upper Right', 'upper right'), ('Upper Left', 'upper left')],
                                      value='lower right', description='Egger Box:'),
        'save_pdf': CB(True, 'Save as PDF'),
        'save_png': CB(True, 'Save as PNG'),
        'save_svg': CB(False, 'Save as SVG'),
        'dpi': widgets.IntSlider(value=300, min=100, max=600, step=50, description='PNG DPI:'),
        'transparent': CB(False, 'Transparent Background'),
        'filename': widgets.Text(value='Funnel_Plot', description='Filename:'),
    }

    def set_data_defaults(data):
        """Defaults that depend on the data (called for session data and recipes)."""
        C['xlabel'].value = f"Effect Size ({data['effect_label']})"
        C['color_by'].options = ['None'] + list(data['moderators'].keys())
        C['color_by'].value = 'None'
        C['fill_funnel'].description = f"Fill Confidence Region ({data['ci_level']} pseudo-confidence)"
        C['show_null'].description = f"Show No-Effect Line ({data['null_value']:g})"

    def category_colors(cats, palette):
        cmap = plt.get_cmap(palette)
        if hasattr(cmap, 'colors') and len(cmap.colors) <= 20:
            return [cmap.colors[i % len(cmap.colors)] for i in range(len(cats))]
        return [cmap(v) for v in np.linspace(0.05, 0.95, max(len(cats), 1))]

    def egger_method(e):
        return ("CR2 cluster-robust SEs, Satterthwaite df" if e['used_robust']
                else "model-based SEs")

    # --- 3. DRAWING (data + settings -> figure; no widget access) ---
    def draw(data, s):
        x = np.asarray(data['yi'], dtype=float)
        y = np.asarray(data['se'], dtype=float)
        mu = data['pooled']['mu']
        null = data['null_value']
        z_ci = norm.ppf(1 - data['alpha'] / 2)
        z_c = [norm.ppf(1 - p / 2) for p in CONTOUR_P]

        fig, ax = plt.subplots(figsize=(s['width'], s['height']))
        max_se = float(np.nanmax(y)) * 1.1
        se_seq = np.linspace(0, max_se, 100)

        lo = [np.nanmin(x)]
        hi = [np.nanmax(x)]
        if s['show_contours']:
            lo.append(null - z_c[-1] * max_se)
            hi.append(null + z_c[-1] * max_se)
        if s['fill_funnel']:
            lo.append(mu - z_ci * max_se)
            hi.append(mu + z_ci * max_se)
        xlo, xhi = min(lo), max(hi)
        pad = 0.04 * (xhi - xlo if xhi > xlo else 1.0)
        xlo, xhi = xlo - pad, xhi + pad

        handles = []
        if s['show_contours']:
            shades = CONTOUR_SHADES[s['contour_shades']]
            bounds = [0.0] + z_c + [None]
            labels = ['0.10 < p ≤ 1.00', '0.05 < p ≤ 0.10', '0.01 < p ≤ 0.05', 'p ≤ 0.01']
            for i, shade in enumerate(shades):
                z_in, z_out = bounds[i], bounds[i + 1]
                inner_r, inner_l = null + z_in * se_seq, null - z_in * se_seq
                outer_r = null + z_out * se_seq if z_out is not None else np.full_like(se_seq, xhi)
                outer_l = null - z_out * se_seq if z_out is not None else np.full_like(se_seq, xlo)
                if i == 0:
                    ax.fill_betweenx(se_seq, outer_l, outer_r, color=shade, linewidth=0, zorder=0.2)
                else:
                    ax.fill_betweenx(se_seq, inner_r, outer_r, color=shade, linewidth=0, zorder=0.2)
                    ax.fill_betweenx(se_seq, outer_l, inner_l, color=shade, linewidth=0, zorder=0.2)
                handles.append(mpatches.Patch(facecolor=shade, edgecolor='#666666', linewidth=0.5,
                                              label=labels[i]))

        if s['fill_funnel']:
            ax.fill_betweenx(se_seq, mu - z_ci * se_seq, mu + z_ci * se_seq, color=s['fill_color'],
                             alpha=s['fill_alpha'], linewidth=0, zorder=0.4)
            for sign in (-1, 1):
                ax.plot(mu + sign * z_ci * se_seq, se_seq, color=s['center_color'], linestyle='--',
                        linewidth=0.8, alpha=0.6, zorder=0.5)
            handles.append(mpatches.Patch(facecolor=s['fill_color'], alpha=s['fill_alpha'],
                                          edgecolor=s['center_color'], linestyle='--', linewidth=0.8,
                                          label=f"{data['ci_level']} pseudo-confidence region"))

        if s['show_null']:
            h = ax.axvline(null, color='black', linestyle=':', linewidth=1.0, zorder=0.6, label='No effect')
            handles.append(h)
        model = data['pooled'].get('model')
        h = ax.vlines(mu, 0, max_se, color=s['center_color'], linestyle='-', linewidth=2, zorder=0.7,
                      label=f"Pooled effect{' (' + model + ')' if model else ''}: {mu:.3f}")
        handles.append(h)

        col = s['color_by']
        if col != 'None' and col in data['moderators']:
            vals = data['moderators'][col]
            cats = sorted({v for v in vals if v is not None})
            colors = category_colors(cats, s['palette'])
            for cat, c in zip(cats, colors):
                m = np.array([v == cat for v in vals], dtype=bool)
                handles.append(ax.scatter(x[m], y[m], color=c, label=str(cat), s=s['point_size'],
                                          alpha=s['point_alpha'], edgecolors='white', linewidth=0.5, zorder=3))
            m = np.array([v is None for v in vals], dtype=bool)
            if m.any():
                handles.append(ax.scatter(x[m], y[m], color='#9e9e9e', marker='X', label='Missing',
                                          s=s['point_size'], alpha=s['point_alpha'], edgecolors='black',
                                          linewidth=0.5, zorder=3))
        else:
            handles.append(ax.scatter(x, y, color=s['point_color'], s=s['point_size'], alpha=s['point_alpha'],
                                      edgecolors='white', linewidth=0.5, label='Effect sizes', zorder=3))

        e = data.get('egger')
        if s['show_egger'] and e:
            star = '*' if e['p_value'] < data['alpha'] else 'ns'
            vpos, hpos = s['egger_loc'].split()
            ax.text(0.97 if hpos == 'right' else 0.03, 0.04 if vpos == 'lower' else 0.96,
                    f"Egger's test: p = {e['p_value']:.4f} ({star})\nmultilevel, {egger_method(e)}",
                    transform=ax.transAxes, ha=hpos, va='bottom' if vpos == 'lower' else 'top',
                    fontsize=s['legend_font'], zorder=5,
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

        ax.set_xlim(xlo, xhi)
        ax.set_ylim(max_se, 0)
        ax.set_xlabel(s['xlabel'], fontsize=s['label_font'], fontweight='bold')
        ax.set_ylabel(s['ylabel'], fontsize=s['label_font'], fontweight='bold')
        ax.tick_params(labelsize=s['tick_font'])
        if s['show_title']:
            ax.set_title(s['title'], fontsize=s['title_font'], fontweight='bold', pad=15)
        if s['grid']:
            ax.grid(True, linestyle=':', alpha=0.5)
        if s['legend_loc'] == 'outside':
            ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.02, 1), frameon=True, fancybox=True,
                      fontsize=s['legend_font'])
        elif s['legend_loc'] != 'none':
            ax.legend(handles=handles, loc=s['legend_loc'], frameon=True, fancybox=True, fontsize=s['legend_font'])
        fig.tight_layout()
        return fig

    def caption(data, s):
        p = data['pooled']
        lvl = data['ci_level']
        model = f"{p['model']} " if p.get('model') else ""
        ci = (f" ({lvl} CI [{p['ci_lower']:.3f}, {p['ci_upper']:.3f}])"
              if p.get('ci_lower') is not None and p.get('ci_upper') is not None else "")
        txt = (f"Funnel plot of the {len(data['yi'])} effect sizes ({html.escape(str(data['effect_label']))}) "
               f"against their standard errors; the y-axis is inverted so that more precise effect sizes appear "
               f"at the top. The solid vertical line is the pooled estimate of the reported {model}model "
               f"(Step 7) = {p['mu']:.3f}{ci}. ")
        if s['show_contours']:
            z = [norm.ppf(1 - q / 2) for q in CONTOUR_P]
            txt += (f"Shaded regions are significance contours centred on no effect ({data['null_value']:g}): "
                    f"an effect size falling in the white region is not significant at two-sided p = 0.10 "
                    f"(|z| &lt; {z[0]:.3f}, z = (effect − {data['null_value']:g}) / SE), and the shaded bands mark "
                    f"0.05 &lt; p ≤ 0.10 ({z[0]:.3f} ≤ |z| &lt; {z[1]:.3f}), 0.01 &lt; p ≤ 0.05 ({z[1]:.3f} ≤ |z| &lt; "
                    f"{z[2]:.3f}) and p ≤ 0.01 (|z| ≥ {z[2]:.3f}). "
                    f"Missing studies concentrated in the non-significant region suggest publication bias. ")
        if s['fill_funnel']:
            txt += (f"The light region bounded by dashed lines is the {lvl} pseudo-confidence region around the "
                    f"pooled estimate (estimate ± {norm.ppf(1 - data['alpha'] / 2):.3f} × SE), where effect sizes "
                    f"would be expected to fall in the absence of heterogeneity and small-study effects. ")
        if s['show_null']:
            txt += f"The dotted line marks no effect ({data['null_value']:g}). "
        if s['color_by'] != 'None' and s['color_by'] in data['moderators']:
            txt += f"Points are coloured by the moderator <b>{html.escape(s['color_by'])}</b>"
            if any(v is None for v in data['moderators'][s['color_by']]):
                txt += " (crosses: moderator value missing)"
            txt += ". "
        e = data.get('egger')
        if s['show_egger'] and e:
            sig = 'significant' if e['p_value'] < data['alpha'] else 'non-significant'
            df_txt = f", df = {float(e['df']):.1f}" if e.get('df') is not None else ""
            txt += (f"A multilevel Egger's regression test (three-level meta-regression of the effect sizes on "
                    f"their standard errors, {egger_method(e)}) indicated {sig} funnel plot asymmetry (slope = "
                    f"{e['slope']:.3f}{df_txt}, <i>p</i> = {e['p_value']:.3f}).")
        return txt

    # --- 4. ACTIONS ---
    out = widgets.Output()
    source_box = widgets.HTML()

    def refresh_source():
        source_box.value = source_html(state['source'], state['recipe_name'])

    def generate(_b=None):
        with out:
            clear_output(wait=True)
            if state['data'] is None:
                display(HTML("<div style='color:#721c24;'>❌ No analysis results: run Step 7 (Overall "
                             "Meta-Analysis) or load a plot recipe.</div>"))
                return
            if state['recipe_name'] is None:
                if results_token(RESULTS_KEY) != state['token']:
                    display(HTML(stale_warning_html('Step 7 (Overall Meta-Analysis)')))
                elif results_token('funnel_results') != state['token_egger']:
                    display(HTML(stale_warning_html("Step 17 (Egger's Test & Trim-and-Fill)")))
            s = widget_values(C)
            try:
                fig = draw(state['data'], s)
                paths = cometa_save_figure(fig, s['filename'], pdf=s['save_pdf'], png=s['save_png'],
                                           svg=s['save_svg'], dpi=s['dpi'], transparent=s['transparent'])
                show_figure(fig)
            except Exception as e:
                plt.close('all')
                display(HTML(f"<div style='color:#721c24;'>❌ Plotting error: {html.escape(str(e))}</div>"))
                return
            display(HTML(saved_files_html(paths)))
            if s['show_egger'] and not state['data'].get('egger'):
                display(HTML("<div style='padding:8px; background:#fff3cd; border-left:4px solid #ffc107; "
                             "font-size:13px;'>Egger's test is not shown: run Step 17 (Egger's Test &amp; "
                             "Trim-and-Fill) first, then re-run this cell.</div>"))
            display(HTML(caption_html(caption(state['data'], s))))

    def get_recipe():
        if state['data'] is None:
            return None
        return build_plot_recipe(PLOT_TYPE, state['data'], widget_values(C), state['source'])

    def on_load(rec, style_only, name):
        if not style_only:
            state.update(data=rec['data'], source=rec.get('source'), recipe_name=name)
            set_data_defaults(state['data'])
            refresh_source()
        skipped = apply_widget_values(C, rec.get('settings'))
        generate()
        return skipped

    # --- 5. INITIAL STATE AND LAYOUT ---
    try:
        state['data'], state['source'] = build_data()
    except Exception as e:
        state['data'], state['source'] = None, None
        print(f"⚠️ Could not read the Step 7 results: {e}")
    state['token'] = results_token(RESULTS_KEY)
    state['token_egger'] = results_token('funnel_results')
    if state['data'] is not None:
        set_data_defaults(state['data'])
    refresh_source()

    preset = preset_dropdown({'width': C['width'], 'height': C['height'], 'title': C['title_font'],
                              'label': C['label_font'], 'tick': C['tick_font']})
    H = lambda t: widgets.HTML(f"<h4 style='color: #2E86AB;'>{t}</h4>")
    tabs = widgets.Tab(children=[
        widgets.VBox([H('Dimensions & Labels'), C['show_title'], C['title'], C['xlabel'], C['ylabel'],
                      widgets.HTML("<hr>"), preset, C['width'], C['height'], widgets.HTML("<hr>"),
                      C['title_font'], C['label_font'], C['tick_font'], C['legend_font'], widgets.HTML("<hr>"),
                      C['legend_loc'], C['grid']]),
        widgets.VBox([H('Data Points'), C['color_by'], C['palette'], C['point_color'], C['point_size'],
                      C['point_alpha']]),
        widgets.VBox([H('Funnel Guidelines'), C['show_contours'], C['contour_shades'], C['show_null'],
                      widgets.HTML("<hr>"), C['fill_funnel'], C['fill_color'], C['fill_alpha'],
                      C['center_color'], widgets.HTML("<hr>"), C['show_egger'], C['egger_loc']]),
        widgets.VBox([H('Export Options'), C['save_pdf'], C['save_png'], C['save_svg'], C['dpi'],
                      C['transparent'], C['filename']]),
    ])
    for i, t in enumerate(['🎨 Style', '⚫ Points', '▼ Funnel', '💾 Export']):
        tabs.set_title(i, t)

    run_btn = widgets.Button(description='📊 Generate Funnel Plot', button_style='success',
                             layout=widgets.Layout(width='100%'))
    run_btn.on_click(generate)
    recipe_bar = plot_recipe_bar(PLOT_TYPE, get_recipe, on_load, lambda: C['filename'].value)

    display(widgets.VBox([
        widgets.HTML("""<div style="background-color: #f0f4f8; padding: 15px; border-radius: 5px; border-left: 5px solid #2E86AB;">
        <h3 style="margin:0; color: #2E86AB;">📊 Step 20: Publication-Ready Funnel Plot</h3>
        <p style="margin:5px 0 0 0;">Visualize publication bias and small-study effects.</p>
        </div>"""),
        source_box, tabs, run_btn, recipe_bar, out,
    ]))


_funnel_plot_cell()


In [ ]:
#@title 📉 21. Sensitivity Analysis: Leave-One-Out: Execution
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# DATA LAYER - LEAVE-ONE-OUT SENSITIVITY ANALYSIS
# Purpose: Centralized data management for leave-one-out analysis
# =============================================================================

import numpy as np
import pandas as pd
from typing import Dict, Any, Optional, Tuple, List
from dataclasses import dataclass
import warnings
from tqdm.auto import tqdm

def _run_three_level_reml_for_subgroup(analysis_data, effect_col, var_col):
    """
    Intercept-only three-level REML fit on a (reduced) dataset, used by
    the leave-one-out refits. Fitted with the shared engine (fit_blocks,
    Cell 1b): full sampling VCV with shared controls (diagonal fallback);
    two-level (sigma² = 0) when no study has more than one effect size.

    Returns:
        tuple: (estimates_dict, debug_tuple) or (None, None) on failure
               estimates_dict includes 'model_type' ('3-level-vcv',
               '3-level-diag' or '2-level')
    """
    # 0. Input validation
    if analysis_data is None or len(analysis_data) == 0:
        return None, None

    # 1. Sort for deterministic alignment (VCV row identity kept through it)
    if 'attach_vcv_row_pos' in globals():
        analysis_data = attach_vcv_row_pos(analysis_data)
    analysis_data = analysis_data.sort_values('id', kind='mergesort').reset_index(drop=True)

    grouped = analysis_data.groupby('id', sort=False)
    y_all = [np.asarray(group[effect_col].values, dtype=np.float64) for _, group in grouped]

    # Check for invalid data
    if any(len(y) == 0 or np.any(np.isnan(y)) for y in y_all):
        return None, None

    N_total = len(analysis_data)
    M_studies = len(y_all)

    # Minimum requirements
    if M_studies < 2:
        return None, None

    # Count studies with multiple observations (needed for sigma² identification)
    n_multi_obs_studies = sum(1 for y in y_all if len(y) > 1)

    # --- PREPARE VCV DATA FOR DIFFERENT PLANS ---
    vcv_dict = ANALYSIS_CONFIG.get('vcv_matrices', {})

    vcv_all_matrix = []  # full matrices (shared controls)
    vcv_all_diag = []    # diagonal fallback
    has_off_diagonal = False  # Track if we actually have shared controls

    for study_id, group in grouped:
        vi = np.asarray(group[var_col].values, dtype=np.float64)

        # Safety check
        if len(vi) == 0 or np.any(np.isnan(vi)) or np.any(vi <= 0):
            return None, None

        # Diagonal is always the fallback
        diag_matrix = np.diag(vi)
        vcv_all_diag.append(diag_matrix)

        # Full matrix, sliced by row identity (correct size even if some of
        # the study's rows were dropped upstream)
        if 'cluster_vcv' in globals():
            full_matrix, _src = cluster_vcv(vcv_dict, study_id, group, var_col)
        else:
            full_matrix = diag_matrix
        vcv_all_matrix.append(full_matrix)
        if not np.allclose(full_matrix, np.diag(np.diag(full_matrix))):
            has_off_diagonal = True

    # --- REML FIT (shared engine, Cell 1b) ---
    # Three-level when at least one study has more than one effect size
    # (sigma² identified), two-level (sigma² = 0) otherwise; the full
    # sampling VCV when there are shared controls, the diagonal as fallback.
    three_level_viable = n_multi_obs_studies >= 1 and N_total > M_studies
    use_full = three_level_viable and has_off_diagonal
    try:
        fit = fit_blocks(y_all, vcv_all_matrix if use_full else vcv_all_diag,
                         two_level=not three_level_viable,
                         V_fallback_blocks=vcv_all_diag if use_full else None)
    except (ValueError, RuntimeError, np.linalg.LinAlgError):
        return None, None

    if not three_level_viable:
        model_type = '2-level'
    elif use_full and not fit.v_fallback:
        model_type = '3-level-vcv'
    else:
        model_type = '3-level-diag'
    final_vcv = fit.spec.V_blocks

    # --- FINAL CALCULATION ---
    try:
        final_estimates = fit_estimates(fit)

        # --- CR2 + Satterthwaite ---
        try:
            var_robust, df_robust = fit_cr2(fit)

            # Overwrite unadjusted errors with clustered robust errors
            final_estimates['var_mu'] = var_robust[0, 0]
            final_estimates['se_mu'] = np.sqrt(var_robust[0, 0])
            final_estimates['df'] = float(df_robust[0])

            # Compute CI directly using Satterthwaite df
            import scipy.stats as stats
            ac = globals().get('ANALYSIS_CONFIG', {})
            alpha = ac.get('global_settings', {}).get('alpha', 0.05)
            t_crit = stats.t.ppf(1 - alpha/2, final_estimates['df'])

            final_estimates['ci_lower'] = final_estimates['mu'] - t_crit * final_estimates['se_mu']
            final_estimates['ci_upper'] = final_estimates['mu'] + t_crit * final_estimates['se_mu']

        except Exception as e:
            warnings.warn(f"Leave-one-out: CR2 step failed ({e}); using model-based SE "
                          f"with df = studies - 1.")
            final_estimates['df'] = max(1, M_studies - 1)
            import scipy.stats as stats
            ac = globals().get('ANALYSIS_CONFIG', {})
            alpha = ac.get('global_settings', {}).get('alpha', 0.05)
            t_crit = stats.t.ppf(1 - alpha/2, final_estimates['df'])
            final_estimates['ci_lower'] = final_estimates['mu'] - t_crit * final_estimates['se_mu']
            final_estimates['ci_upper'] = final_estimates['mu'] + t_crit * final_estimates['se_mu']

        # Add metadata about which model was used
        final_estimates['model_type'] = model_type
        final_estimates['n_studies'] = M_studies
        final_estimates['n_observations'] = N_total
        final_estimates['n_multi_obs_studies'] = n_multi_obs_studies
        final_estimates['optimizer_success'] = bool(fit.converged)
        final_estimates['optimizer_message'] = 'fit_mv'

        return final_estimates, (y_all, final_vcv, N_total, M_studies)

    except Exception as e:
        return None, None

# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class LOOConfig:
    """Configuration for leave-one-out analysis"""
    effect_col: str
    var_col: str
    alpha: float = 0.05
    influence_threshold: float = 20.0  # Percent change threshold

    def __post_init__(self):
        """Validate configuration"""
        if not self.effect_col:
            raise ValueError("effect_col cannot be empty")
        if not self.var_col:
            raise ValueError("var_col cannot be empty")
        if self.alpha <= 0 or self.alpha >= 1:
            raise ValueError(f"alpha must be between 0 and 1, got {self.alpha}")
        if self.influence_threshold <= 0:
            raise ValueError(f"influence_threshold must be positive, got {self.influence_threshold}")


@dataclass
class OriginalEstimate:
    """Original full-model estimate"""
    mu: float
    se: float
    ci_lower: float
    ci_upper: float
    tau2: Optional[float] = None
    sigma2: Optional[float] = None


@dataclass
class LOOIteration:
    """Results from one leave-one-out iteration"""
    excluded_study: str
    mu: float
    se: float
    ci_lower: float
    ci_upper: float
    tau2: Optional[float]
    sigma2: Optional[float]
    diff: float  # Difference from original
    pct_change: float  # Percent change from original
    is_influential: bool


@dataclass
class LOOResult:
    """Complete leave-one-out analysis result"""
    original: OriginalEstimate
    iterations: pd.DataFrame  # DataFrame of LOOIteration results

    # Summary statistics
    n_studies: int
    n_influential: int
    min_estimate: float
    max_estimate: float
    range_estimate: float

    # Most influential studies
    top_influential: pd.DataFrame


# =============================================================================
# DATA MANAGER CLASS
# =============================================================================

class LOODataManager:
    """
    Centralized data access layer for leave-one-out sensitivity analysis.
    Handles all interactions with ANALYSIS_CONFIG and data validation.
    """

    def __init__(self, analysis_config: Dict[str, Any]):
        """
        Initialize data manager.

        Args:
            analysis_config: Global ANALYSIS_CONFIG dictionary
        """
        self._config = analysis_config
        self._validate_prerequisites()

    # -------------------------------------------------------------------------
    # VALIDATION METHODS
    # -------------------------------------------------------------------------

    def _validate_prerequisites(self) -> None:
        """Validate that required configuration exists"""
        if 'effect_col' not in self._config:
            warnings.warn("effect_col not in ANALYSIS_CONFIG, using default 'hedges_g'")
        if 'var_col' not in self._config:
            warnings.warn("var_col not in ANALYSIS_CONFIG, using default 'Vg'")

    def validate_data(self, df: pd.DataFrame) -> Tuple[bool, Optional[str]]:
        """
        Validate that data is suitable for leave-one-out analysis.

        Args:
            df: DataFrame to validate

        Returns:
            Tuple of (is_valid, error_message)
        """
        if df is None or len(df) == 0:
            return False, "No data available"

        if self.effect_col not in df.columns:
            return False, f"Effect column '{self.effect_col}' not found"

        if self.var_col not in df.columns:
            return False, f"Variance column '{self.var_col}' not found"

        # Check for study ID
        if 'id' not in df.columns:
            return False, "'id' column not found - cannot perform leave-one-out by study"

        # Check for minimum studies (need at least 3 to leave one out)
        n_studies = df['id'].nunique()
        if n_studies < 3:
            return False, f"Need at least 3 studies for leave-one-out, found {n_studies}"

        return True, None

    # -------------------------------------------------------------------------
    # PROPERTY ACCESSORS
    # -------------------------------------------------------------------------

    @property
    def analysis_data(self) -> Optional[pd.DataFrame]:
        """Get analysis dataset"""
        if 'analysis_data' in self._config:
            return self._config['analysis_data'].copy()
        return None

    @property
    def effect_col(self) -> str:
        """Get effect size column name"""
        return self._config.get('effect_col', 'hedges_g')

    @property
    def var_col(self) -> str:
        """Get variance column name"""
        return self._config.get('var_col', 'Vg')

    @property
    def es_config(self) -> Dict[str, Any]:
        """Get effect size configuration"""
        return self._config.get('es_config', {})

    @property
    def global_settings(self) -> Dict[str, Any]:
        """Get global settings (with defaults)"""
        return self._config.get('global_settings', {
            'alpha': 0.05
        })

    # -------------------------------------------------------------------------
    # DATA EXTRACTION METHODS
    # -------------------------------------------------------------------------

    def prepare_data(
        self,
        df: Optional[pd.DataFrame] = None
    ) -> pd.DataFrame:
        """
        Prepare and clean data for leave-one-out analysis.

        Args:
            df: DataFrame to prepare (uses analysis_data if None)

        Returns:
            Cleaned DataFrame

        Raises:
            ValueError: If data cannot be prepared
        """
        if df is None:
            df = self.analysis_data

        if df is None:
            raise ValueError("No data available for analysis")

        # Validate
        is_valid, error_msg = self.validate_data(df)
        if not is_valid:
            raise ValueError(error_msg)

        # Create working copy
        clean_df = df.copy()

        # Remove missing values
        clean_df = clean_df.dropna(subset=[
            self.effect_col,
            self.var_col,
            'id'
        ]).copy()

        # Remove zero or negative variances
        clean_df = clean_df[clean_df[self.var_col] > 0].copy()

        # Final check
        n_studies = clean_df['id'].nunique()
        if n_studies < 3:
            raise ValueError(
                f"Insufficient studies after cleaning: {n_studies} studies. "
                f"Need at least 3 for leave-one-out analysis."
            )

        return clean_df

    def get_study_ids(self, df: pd.DataFrame) -> List[Any]:
        """
        Get list of unique study IDs.

        Args:
            df: DataFrame with 'id' column

        Returns:
            List of unique study IDs
        """
        return df['id'].unique().tolist()

    # -------------------------------------------------------------------------
    # RESULT PERSISTENCE METHODS
    # -------------------------------------------------------------------------

    def save_loo_results(self, result: LOOResult) -> None:
        """
        Save leave-one-out results to ANALYSIS_CONFIG.

        Args:
            result: LOOResult object
        """
        import datetime

        # Save in format compatible with visualization cells
        self._config['loo_results'] = {
            'timestamp': datetime.datetime.now(),
            'data': result.iterations,
            'orig_mu': result.original.mu,
            'orig_ci_lo': result.original.ci_lower,
            'orig_ci_hi': result.original.ci_upper,
            'n_studies': result.n_studies,
            'n_influential': result.n_influential,
            'min_estimate': result.min_estimate,
            'max_estimate': result.max_estimate
        }

    def save_publication_text(self, text: str) -> None:
      """Save publication-ready text to config"""
      self._config['loo_text'] = text
    # -------------------------------------------------------------------------
    # UTILITY METHODS
    # -------------------------------------------------------------------------

    def summary_dict(self) -> Dict[str, Any]:
        """Get summary of current configuration"""
        df = self.analysis_data

        return {
            'effect_col': self.effect_col,
            'var_col': self.var_col,
            'n_observations': len(df) if df is not None else 0,
            'n_studies': df['id'].nunique() if df is not None and 'id' in df.columns else 0
        }


# =============================================================================
# HELPER: CI CALCULATOR
# =============================================================================

class CICalculator:
    """
    Calculates confidence intervals for estimates.
    Handles cases where CIs are not provided by regression output.
    """

    @staticmethod
    def calculate_ci(
        estimate: float,
        se: float,
        alpha: float = 0.05,
        df: Optional[int] = None
    ) -> Tuple[float, float]:
        """
        Calculate confidence interval.

        Args:
            estimate: Point estimate
            se: Standard error
            alpha: Significance level (default 0.05 for 95% CI)
            df: Degrees of freedom (if None, uses normal distribution)

        Returns:
            Tuple of (ci_lower, ci_upper)
        """
        from scipy.stats import norm, t

        q = 1 - (alpha / 2)

        if df is not None and df > 0:
            crit_val = t.ppf(q, df)
        else:
            crit_val = norm.ppf(q)

        ci_lower = estimate - crit_val * se
        ci_upper = estimate + crit_val * se

        return ci_lower, ci_upper


# =============================================================================
# ANALYSIS LAYER - LEAVE-ONE-OUT SENSITIVITY ANALYSIS
# Purpose: Pure statistical computation without UI dependencies
# Dependencies:
#   - Data Layer (LOODataManager, LOOResult)
#   - Three-level regression function (_run_three_level_reml_for_subgroup)
# =============================================================================

import numpy as np
import pandas as pd
from typing import Dict, Any, Optional, Tuple, List
import warnings


# =============================================================================
# BASELINE ESTIMATOR
# =============================================================================

class BaselineEstimator:
    """
    Estimates the original full-model baseline using all studies.
    """

    def __init__(self, effect_col: str, var_col: str, alpha: float = 0.05):
        """
        Initialize estimator.

        Args:
            effect_col: Effect size column name
            var_col: Variance column name
            alpha: Significance level for CIs
        """
        self.effect_col = effect_col
        self.var_col = var_col
        self.alpha = alpha

    def estimate_baseline(
        self,
        df: pd.DataFrame
    ) -> Optional[OriginalEstimate]:
        """
        Run full model with all studies to get baseline.

        Args:
            df: Full dataset

        Returns:
            OriginalEstimate object or None if fails
        """
        try:
            # Use the three-level REML function
            estimates, metadata = _run_three_level_reml_for_subgroup(
                df,
                self.effect_col,
                self.var_col
            )

            if estimates is None:
                return None

            # Extract estimates
            mu = estimates['mu']
            se = estimates.get('se_mu', estimates.get('se', np.nan))

            # Get or calculate CIs
            if 'ci_lower' in estimates and 'ci_upper' in estimates:
                ci_lower = estimates['ci_lower']
                ci_upper = estimates['ci_upper']
            else:
                # Calculate manually
                ci_lower, ci_upper = CICalculator.calculate_ci(
                    mu, se, self.alpha
                )

            # Get variance components
            tau2 = estimates.get('tau_sq', estimates.get('tau2'))
            sigma2 = estimates.get('sigma_sq', estimates.get('sigma2'))

            return OriginalEstimate(
                mu=mu,
                se=se,
                ci_lower=ci_lower,
                ci_upper=ci_upper,
                tau2=tau2,
                sigma2=sigma2
            )

        except Exception as e:
            warnings.warn(f"Baseline estimation failed: {str(e)}")
            return None


# =============================================================================
# LOO ITERATOR
# =============================================================================

class LOOIterator:
    """
    Iteratively removes one study at a time and re-fits the model.
    """

    def __init__(
        self,
        effect_col: str,
        var_col: str,
        alpha: float = 0.05,
        influence_threshold: float = 20.0
    ):
        """
        Initialize iterator.

        Args:
            effect_col: Effect size column name
            var_col: Variance column name
            alpha: Significance level for CIs
            influence_threshold: Percent change threshold for influential studies
        """
        self.effect_col = effect_col
        self.var_col = var_col
        self.alpha = alpha
        self.influence_threshold = influence_threshold

    def run_iterations(
        self,
        df: pd.DataFrame,
        study_ids: List[Any],
        baseline: OriginalEstimate,
        progress_callback: Optional[callable] = None
    ) -> List[LOOIteration]:
        """
        Run leave-one-out iterations for all studies.

        Args:
            df: Full dataset
            study_ids: List of study IDs to exclude one at a time
            baseline: Original baseline estimate
            progress_callback: Optional progress updates

        Returns:
            List of LOOIteration objects
        """
        iterations = []
        n_studies = len(study_ids)

        for i, exclude_id in enumerate(tqdm(study_ids, desc="Leave-One-Out Analysis", unit="study")):

            if progress_callback:
                progress_callback(i, n_studies, exclude_id)

            # Create subset without this study
            subset_df = df[df['id'] != exclude_id].copy()

            # Run model on subset
            try:
                estimates, metadata = _run_three_level_reml_for_subgroup(
                    subset_df,
                    self.effect_col,
                    self.var_col
                )

                if estimates is None:
                    warnings.warn(f"Failed to fit model without study {exclude_id}")
                    continue

                # Extract estimates
                mu = estimates['mu']
                se = estimates.get('se_mu', estimates.get('se', np.nan))

                # Get or calculate CIs
                if 'ci_lower' in estimates and 'ci_upper' in estimates:
                    ci_lower = estimates['ci_lower']
                    ci_upper = estimates['ci_upper']
                else:
                    ci_lower, ci_upper = CICalculator.calculate_ci(
                        mu, se, self.alpha
                    )

                # Get variance components
                tau2 = estimates.get('tau_sq', estimates.get('tau2'))
                sigma2 = estimates.get('sigma_sq', estimates.get('sigma2'))

                # Calculate differences
                diff = mu - baseline.mu

                if baseline.mu != 0:
                    pct_change = (diff / baseline.mu) * 100
                else:
                    pct_change = 0.0

                # Check if influential
                is_influential = abs(pct_change) > self.influence_threshold

                # Create iteration result
                iteration = LOOIteration(
                    excluded_study=str(exclude_id),
                    mu=mu,
                    se=se,
                    ci_lower=ci_lower,
                    ci_upper=ci_upper,
                    tau2=tau2,
                    sigma2=sigma2,
                    diff=diff,
                    pct_change=pct_change,
                    is_influential=is_influential
                )

                iterations.append(iteration)

            except Exception as e:
                warnings.warn(f"Error processing study {exclude_id}: {str(e)}")
                continue

        return iterations


# =============================================================================
# INFLUENCE DETECTOR
# =============================================================================

class InfluenceDetector:
    """
    Analyzes iterations to identify influential studies and summary statistics.
    """

    @staticmethod
    def analyze_iterations(
        iterations: List[LOOIteration],
        baseline: OriginalEstimate,
        n_studies: int
    ) -> Dict[str, Any]:
        """
        Analyze iterations to get summary statistics.

        Args:
            iterations: List of LOOIteration objects
            baseline: Original baseline estimate
            n_studies: Total number of studies

        Returns:
            Dictionary with summary statistics
        """
        if not iterations:
            return {
                'n_influential': 0,
                'min_estimate': baseline.mu,
                'max_estimate': baseline.mu,
                'range_estimate': 0.0,
                'top_influential': pd.DataFrame()
            }

        # Convert to DataFrame for analysis
        iter_df = pd.DataFrame([
            {
                'excluded_study': it.excluded_study,
                'mu': it.mu,
                'se': it.se,
                'ci_lower': it.ci_lower,
                'ci_upper': it.ci_upper,
                'tau2': it.tau2,
                'sigma2': it.sigma2,
                'diff': it.diff,
                'pct_change': it.pct_change,
                'is_influential': it.is_influential
            }
            for it in iterations
        ])

        # Count influential studies
        n_influential = iter_df['is_influential'].sum()

        # Get range of estimates
        min_estimate = iter_df['mu'].min()
        max_estimate = iter_df['mu'].max()
        range_estimate = max_estimate - min_estimate

        # Get top influential studies (by absolute difference)
        iter_df['abs_diff'] = iter_df['diff'].abs()
        top_influential = iter_df.sort_values('abs_diff', ascending=False).head(5)

        return {
            'n_influential': n_influential,
            'min_estimate': min_estimate,
            'max_estimate': max_estimate,
            'range_estimate': range_estimate,
            'top_influential': top_influential
        }


# =============================================================================
# LEAVE-ONE-OUT ORCHESTRATOR
# =============================================================================

class LOOEngine:
    """
    High-level orchestrator for leave-one-out sensitivity analysis.
    Coordinates baseline estimation, iterative exclusion, and influence detection.
    """

    def __init__(self, data_manager: 'LOODataManager'):
        """
        Initialize engine with data manager.

        Args:
            data_manager: LOODataManager instance
        """
        self.data_manager = data_manager

        settings = data_manager.global_settings
        alpha = settings.get('alpha', 0.05)

        self.baseline_estimator = BaselineEstimator(
            effect_col=data_manager.effect_col,
            var_col=data_manager.var_col,
            alpha=alpha
        )

        self.iterator = LOOIterator(
            effect_col=data_manager.effect_col,
            var_col=data_manager.var_col,
            alpha=alpha,
            influence_threshold=20.0  # Can be made configurable
        )

        self.influence_detector = InfluenceDetector()

    # -------------------------------------------------------------------------
    # MAIN EXECUTION METHOD
    # -------------------------------------------------------------------------

    def run_analysis(
        self,
        progress_callback: Optional[callable] = None
    ) -> Optional['LOOResult']:
        """
        Execute complete leave-one-out sensitivity analysis.

        WORKFLOW:
        1. Prepare data
        2. Estimate baseline (full model)
        3. Run k iterations (exclude one study each time)
        4. Analyze influence
        5. Build result object

        Args:
            progress_callback: Optional progress updates

        Returns:
            LOOResult object or None if analysis fails
        """
        # Step 1: Prepare data
        if progress_callback:
            progress_callback("📊 Preparing data...")

        try:
            df = self.data_manager.prepare_data()
        except ValueError as e:
            if progress_callback:
                progress_callback(f"❌ Data preparation failed: {str(e)}")
            return None

        study_ids = self.data_manager.get_study_ids(df)
        n_studies = len(study_ids)

        # Step 2: Estimate baseline
        if progress_callback:
            progress_callback("🔍 Estimating baseline (full model)...")

        baseline = self.baseline_estimator.estimate_baseline(df)

        if baseline is None:
            if progress_callback:
                progress_callback("❌ Baseline estimation failed")
            return None

        # Step 3: Run iterations
        if progress_callback:
            progress_callback(f"🔄 Running {n_studies} leave-one-out iterations...")

        def iter_progress(i, total, study_id):
            if progress_callback:
                progress_callback(
                    f"Processing {i+1}/{total}: Excluding study {study_id}",
                    progress=(i+1)/total
                )

        iterations = self.iterator.run_iterations(
            df=df,
            study_ids=study_ids,
            baseline=baseline,
            progress_callback=iter_progress
        )

        if not iterations:
            if progress_callback:
                progress_callback("❌ No iterations completed successfully")
            return None

        # Step 4: Analyze influence
        if progress_callback:
            progress_callback("📈 Analyzing influential studies...")

        analysis = self.influence_detector.analyze_iterations(
            iterations=iterations,
            baseline=baseline,
            n_studies=n_studies
        )

        # Step 5: Build result object
        # Convert iterations to DataFrame
        iter_df = pd.DataFrame([
            {
                'excluded_study': it.excluded_study,
                'mu': it.mu,
                'se': it.se,
                'ci_lower': it.ci_lower,
                'ci_upper': it.ci_upper,
                'tau2': it.tau2,
                'sigma2': it.sigma2,
                'diff': it.diff,
                'pct_change': it.pct_change,
                'is_influential': it.is_influential
            }
            for it in iterations
        ])

        result = LOOResult(
            original=baseline,
            iterations=iter_df,
            n_studies=n_studies,
            n_influential=analysis['n_influential'],
            min_estimate=analysis['min_estimate'],
            max_estimate=analysis['max_estimate'],
            range_estimate=analysis['range_estimate'],
            top_influential=analysis['top_influential']
        )

        if progress_callback:
            progress_callback(
                f"✅ Analysis complete: {analysis['n_influential']} influential studies found"
            )

        return result


# =============================================================================
# HELPER: SUMMARY TEXT GENERATOR
# =============================================================================

class SummaryTextGenerator:
    """
    Generates summary text for leave-one-out results.
    """

    @staticmethod
    def generate_summary(result: 'LOOResult') -> str:
        """
        Generate text summary of results.

        Args:
            result: LOOResult object

        Returns:
            Summary text string
        """
        orig = result.original
        n_infl = result.n_influential
        min_est = result.min_estimate
        max_est = result.max_estimate

        text = (
            f"A leave-one-out sensitivity analysis was conducted to assess the robustness "
            f"of the pooled estimate. The pooled effect size varied between "
            f"{min_est:.3f} and {max_est:.3f} (Original: {orig.mu:.3f}). "
        )

        if n_infl == 0:
            text += (
                "No single study disproportionately influenced the results "
                "(all shifts < 20%), indicating robust findings."
            )
        else:
            text += (
                f"However, {n_infl} study/studies were identified as potentially influential, "
                f"causing shifts > 20% in the pooled estimate."
            )

        return text

# =============================================================================
# PUBLICATION TEXT GENERATOR
# =============================================================================

class LOOPublicationTextGenerator:
    """
    Generates publication-ready text for manuscripts for the LOO analysis.
    """

    def generate_methods_section(self, threshold: float) -> str:
        """Generate Materials and Methods section"""
        db = {
            'borenstein': "Borenstein, M., Hedges, L. V., Higgins, J. P., & Rothstein, H. R. (2009). <i>Introduction to Meta-Analysis</i>. Chichester, UK: John Wiley & Sons.",
            'tool': COMETA_CITATION
        }

        html = f"""
        <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt; line-height: 1.8; padding: 20px; background-color: #ffffff; border: 1px solid #eee; margin-bottom: 20px;'>
            <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; margin-top: 0;'>Materials and Methods</h3>
            <p style='text-align: justify;'>
            <b>Sensitivity Analysis (Leave-One-Out).</b> To evaluate the robustness of the pooled effect size and identify potentially influential outlier studies, we conducted a leave-one-out sensitivity analysis [1].
            This procedure iteratively removes one study at a time and recalculates the pooled effect size using the remaining <i>k-1</i> studies.
            An individual study was classified as highly influential if its exclusion shifted the pooled estimate by more than {threshold:.0f}% relative to the original full-model estimate.
            All recalculations accounted for within-study dependency using the same multi-level modeling framework as the primary analysis via the Co-Meta toolkit [2].
            </p>
            <h4 style='color: #34495e; margin-top: 20px; font-size: 11pt;'>References</h4>
            <ol style='font-size: 10pt; color: #555;'>
                <li>{db['borenstein']}</li>
                <li>{db['tool']}</li>
            </ol>
        </div>
        """
        return html

    def generate_results_section(self, result: 'LOOResult') -> str:
        """Generate Results section"""
        orig = result.original
        n_infl = result.n_influential
        min_est = result.min_estimate
        max_est = result.max_estimate

        html = f"""
        <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt; line-height: 1.8; padding: 20px; background-color: #ffffff;'>
            <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;'>Sensitivity Analysis Results</h3>
            <p style='text-align: justify;'>
            A leave-one-out sensitivity analysis was conducted to ascertain whether the overall meta-analytic conclusions were driven by any single study.
            Across all {result.n_studies} iterations, the recalculated pooled effect sizes ranged from a minimum of <b>{min_est:.3f}</b> to a maximum of <b>{max_est:.3f}</b>, compared to the original full-model estimate of {orig.mu:.3f}.
            </p>
        """

        if n_infl == 0:
            html += f"""
            <p style='text-align: justify;'>
            The analysis revealed that <b>no single study disproportionately influenced the results</b>. The exclusion of any individual study did not shift the pooled estimate by more than the predefined influence threshold, nor did it substantially alter the confidence intervals.
            This stability indicates that the meta-analytic findings are robust and not artifacts of extreme or outlier studies.
            </p>
            """
        else:
            top_study = result.top_influential.iloc[0]
            html += f"""
            <p style='text-align: justify;'>
            However, the analysis identified <b>{n_infl} influential stud{'y' if n_infl==1 else 'ies'}</b> whose exclusion caused a substantial shift in the pooled estimate.
            Most notably, the removal of the study <i>{top_study['excluded_study']}</i> shifted the pooled estimate to {top_study['mu']:.3f}, representing a {abs(top_study['pct_change']):.1f}% change relative to the original estimate.
            [<i>Insert interpretation: Discuss whether removing this study alters the statistical significance or the overall clinical/ecological interpretation of the effect.</i>]
            </p>
            """

        html += f"""
            <div style='background-color: #fff3cd; padding: 10px; border-left: 4px solid #ffc107; margin-top: 15px;'>
                <p style='margin: 0;'><b>💡 Tip:</b> Select all text (Ctrl+A / Cmd+A), copy (Ctrl+C / Cmd+C), and paste into your word processor.</p>
            </div>
        </div>
        """
        return html

# =============================================================================
# Purpose: Pure UI rendering without business logic
# Dependencies: Data & Analysis Layers
# =============================================================================

import pandas as pd
import numpy as np
from typing import Dict, Any, Optional
import ipywidgets as widgets
from IPython.display import display, HTML
import sys


# =============================================================================
# HTML TEMPLATE GENERATORS
# =============================================================================

class LOOHTMLTemplates:
    """
    Static HTML template generators for leave-one-out visualizations.
    All methods are pure functions returning HTML strings.
    """

    @staticmethod
    def summary_header(result: 'LOOResult') -> str:
        """Generate summary header with key statistics"""
        return f"""
        <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    padding: 25px; border-radius: 10px; color: white; margin-bottom: 20px;'>
            <h3 style='margin: 0 0 15px 0; font-size: 1.5em;'>🔍 Sensitivity Analysis Summary</h3>
            <div style='display: grid; grid-template-columns: repeat(3, 1fr); gap: 15px;'>
                <div style='text-align: center;'>
                    <div style='font-size: 0.9em; opacity: 0.9;'>Studies Analyzed</div>
                    <div style='font-size: 2em; font-weight: bold;'>{result.n_studies}</div>
                </div>
                <div style='text-align: center;'>
                    <div style='font-size: 0.9em; opacity: 0.9;'>Influential Studies</div>
                    <div style='font-size: 2em; font-weight: bold;'>{result.n_influential}</div>
                </div>
                <div style='text-align: center;'>
                    <div style='font-size: 0.9em; opacity: 0.9;'>Estimate Range</div>
                    <div style='font-size: 2em; font-weight: bold;'>{result.range_estimate:.3f}</div>
                </div>
            </div>
        </div>
        """

    @staticmethod
    def baseline_box(result: 'LOOResult') -> str:
        """Generate baseline estimate box"""
        orig = result.original

        return f"""
        <div style='background-color: #e7f3ff; padding: 15px; border-radius: 5px;
                    border-left: 4px solid #007bff; margin-bottom: 20px;'>
            <h4 style='margin-top: 0; color: #0056b3;'>📊 Original Full Model Estimate</h4>
            <p style='font-size: 16px; margin: 5px 0;'>
                <b>Effect Size (μ):</b> {orig.mu:.4f}<br>
                <b>Standard Error:</b> {orig.se:.4f}<br>
                <b>95% CI:</b> [{orig.ci_lower:.4f}, {orig.ci_upper:.4f}]
            </p>
        </div>
        """

    @staticmethod
    def styled_table(df: pd.DataFrame, columns: list) -> str:
        """Generate styled HTML table"""
        html = """
        <table style='width: 100%; border-collapse: collapse; margin: 20px 0;'>
            <thead style='background-color: #2c3e50; color: white;'>
                <tr>
        """

        # Headers
        header_names = {
            'excluded_study': 'Excluded Study',
            'mu': 'Effect Size (μ)',
            'diff': 'Difference',
            'pct_change': '% Change',
            'ci_lower': 'CI Lower',
            'ci_upper': 'CI Upper'
        }

        for col in columns:
            html += f"<th style='padding: 12px; text-align: left; border: 1px solid #dee2e6;'>{header_names.get(col, col)}</th>"

        html += "</tr></thead><tbody>"

        # Rows
        for idx, row in df.iterrows():
            # Color code based on influence
            if row.get('is_influential', False):
                bg_color = "#fff3cd"  # Yellow for influential
            else:
                bg_color = "white"

            html += f"<tr style='background-color: {bg_color}; border-bottom: 1px solid #dee2e6;'>"

            for col in columns:
                value = row[col]

                # Format based on column type
                if col == 'mu':
                    formatted = f"{value:.4f}"
                elif col == 'diff':
                    formatted = f"{value:+.4f}"
                    # Color code difference
                    if value > 0:
                        formatted = f"<span style='color: #dc3545;'>{formatted}</span>"
                    elif value < 0:
                        formatted = f"<span style='color: #28a745;'>{formatted}</span>"
                elif col == 'pct_change':
                    formatted = f"{value:+.1f}%"
                    # Color code percent change
                    if abs(value) > 20:
                        formatted = f"<b style='color: #dc3545;'>{formatted}</b>"
                elif col in ['ci_lower', 'ci_upper']:
                    formatted = f"{value:.4f}"
                else:
                    formatted = str(value)

                html += f"<td style='padding: 10px; border: 1px solid #dee2e6;'>{formatted}</td>"

            html += "</tr>"

        html += "</tbody></table>"

        return html

    @staticmethod
    def interpretation_box(summary_text: str, is_robust: bool) -> str:
        """Generate interpretation box with summary text"""
        if is_robust:
            bg_color = "#d4edda"
            border_color = "#28a745"
            icon = "✓"
            title = "Robust Findings"
        else:
            bg_color = "#fff3cd"
            border_color = "#ffc107"
            icon = "⚠"
            title = "Influential Studies Detected"

        return f"""
        <div style='background-color: {bg_color}; padding: 20px; border-radius: 5px;
                    border-left: 5px solid {border_color}; margin-top: 20px;'>
            <h4 style='margin-top: 0; color: #495057;'>{icon} {title}</h4>
            <p style='color: #6c757d; font-size: 14px; line-height: 1.6;'>
                {summary_text}
            </p>
        </div>
        """

    @staticmethod
    def guidance_box() -> str:
        """Generate interpretation guidance box"""
        return """
        <div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px;
                    border-left: 4px solid #6c757d; margin-top: 20px;'>
            <h4 style='margin-top: 0; color: #495057;'>💡 Interpretation Guide</h4>
            <ul style='color: #6c757d; font-size: 14px; line-height: 1.8; margin-bottom: 0;'>
                <li><b>Influential Study:</b> A study whose removal causes the pooled estimate to
                    change by more than 20%.</li>
                <li><b>Robust Findings:</b> If no influential studies are detected, the meta-analytic
                    conclusion is not driven by any single study.</li>
                <li><b>Interpretation:</b> If influential studies exist, examine their characteristics
                    (sample size, quality, population) to understand why they differ.</li>
                <li><b>Action:</b> Consider reporting both the original and sensitivity-adjusted estimates
                    if influential studies are present.</li>
            </ul>
        </div>
        """

    @staticmethod
    def next_steps_box() -> str:
        """Generate next steps box"""
        return """
        <div style='background-color: #e7f3ff; padding: 15px; border-radius: 5px;
                    border-left: 4px solid #007bff; margin-top: 20px;'>
            <h4 style='margin-top: 0; color: #0056b3;'>📊 Next Steps</h4>
            <p style='color: #6c757d; font-size: 14px; margin-bottom: 0;'>
                <b>Visualization:</b> Run the plotting cell (if available) to visualize how the
                estimate changes with each study removed. This forest plot shows the range of
                possible estimates and helps identify which studies have the greatest impact.
            </p>
        </div>
        """


# =============================================================================
# VIEW COMPONENTS
# =============================================================================

class LOOResultsView:
    """
    Manages all UI rendering for leave-one-out sensitivity analysis.
    Contains zero business logic - only presentation code.
    """

    def __init__(self):
        """Initialize view with display settings"""
        self.templates = LOOHTMLTemplates()
        self.text_gen = SummaryTextGenerator()
        self.pub_text_gen = LOOPublicationTextGenerator()

        # Create output widget
        self.output = widgets.Output()

        # --- Create tab infrastructure inside the output ---
        self.tab_summary = widgets.Output()
        self.tab_publication = widgets.Output()
        self.tabs = widgets.Tab(children=[self.tab_summary, self.tab_publication])
        self.tabs.set_title(0, '📊 Sensitivity Summary')
        self.tabs.set_title(1, '📝 Publication Text')

        # Create progress bar
        self.progress_bar = widgets.IntProgress(
            value=0,
            min=0,
            max=100,
            description='Progress:',
            bar_style='info',
            orientation='horizontal',
            layout=widgets.Layout(width='400px')
        )

    # -------------------------------------------------------------------------
    # MAIN RENDERING METHOD
    # -------------------------------------------------------------------------

    def render_results(self, result: 'LOOResult') -> None:
        """
        Render complete results display in tabs.
        """
        with self.output:
            self.output.clear_output()

            # Hide progress bar
            self.progress_bar.layout.display = 'none'

            # Display Tabs
            display(self.tabs)

            # --- TAB 1: SUMMARY ---
            with self.tab_summary:
                self.tab_summary.clear_output()

                # Summary header
                header_html = self.templates.summary_header(result)
                display(HTML(header_html))

                # Baseline box
                baseline_html = self.templates.baseline_box(result)
                display(HTML(baseline_html))

                # Top influential studies section
                display(HTML("<h4 style='color: #2c3e50; margin-top: 30px;'>📊 Most Influential Studies</h4>"))
                display(HTML("<p style='color: #6c757d;'>Studies that cause the largest change in the pooled estimate when removed.</p>"))

                # Top influential table
                top_cols = ['excluded_study', 'mu', 'diff', 'pct_change']
                table_html = self.templates.styled_table(result.top_influential, top_cols)
                display(HTML(table_html))

                # Summary interpretation
                summary_text = self.text_gen.generate_summary(result)
                is_robust = result.n_influential == 0
                interp_html = self.templates.interpretation_box(summary_text, is_robust)
                display(HTML(interp_html))

                # Guidance
                guidance_html = self.templates.guidance_box()
                display(HTML(guidance_html))

                # Next steps
                next_steps_html = self.templates.next_steps_box()
                display(HTML(next_steps_html))

            # --- TAB 2: PUBLICATION TEXT ---
            with self.tab_publication:
                self.tab_publication.clear_output()
                display(HTML("<h3 style='color: #2E86AB;'>📝 Publication-Ready Results Text</h3>"))
                display(HTML("<p style='color: #6c757d;'>Copy and paste this formatted text into your manuscript:</p>"))

                # Get threshold from controller/iterator if possible, default to 20
                methods_html = self.pub_text_gen.generate_methods_section(threshold=20.0)
                results_html = self.pub_text_gen.generate_results_section(result)

                display(HTML(methods_html))
                display(HTML(results_html))

                # We return this string so the controller can save it
                self._latest_pub_text = methods_html + "<br><hr><br>" + results_html


    # -------------------------------------------------------------------------
    # PROGRESS DISPLAY
    # -------------------------------------------------------------------------

    def update_progress(self, message: str, progress: Optional[float] = None) -> None:
        """
        Update progress display.

        Args:
            message: Progress message
            progress: Progress value (0.0 to 1.0), if None just shows message
        """
        with self.output:
            # Show progress bar if hidden
            if self.progress_bar.layout.display == 'none':
                self.progress_bar.layout.display = 'block'

            if progress is not None:
                self.progress_bar.value = int(progress * 100)

            # Update description
            self.progress_bar.description = message[:20] + "..." if len(message) > 20 else message

    def show_progress_bar(self) -> None:
        """Show progress bar"""
        self.progress_bar.layout.display = 'block'
        self.progress_bar.value = 0
        display(self.progress_bar)

    def hide_progress_bar(self) -> None:
        """Hide progress bar"""
        self.progress_bar.layout.display = 'none'

    # -------------------------------------------------------------------------
    # ERROR DISPLAY
    # -------------------------------------------------------------------------

    def render_error(self, message: str, details: Optional[str] = None) -> None:
        """
        Render error message.

        Args:
            message: Error message
            details: Optional detailed error information
        """
        with self.output:
            self.output.clear_output()

            error_html = f"""
            <div style='color: #721c24; background-color: #f8d7da; padding: 15px;
                        border-radius: 5px; border: 1px solid #f5c6cb;'>
                <h4 style='margin-top: 0;'>❌ Error</h4>
                <p style='margin-bottom: 0;'>{message}</p>
            </div>
            """
            display(HTML(error_html))

            if details:
                details_html = f"""
                <div style='background-color: #f8f9fa; padding: 10px; margin-top: 10px;
                            border-radius: 5px; border-left: 3px solid #dc3545;'>
                    <pre style='margin: 0; font-size: 12px; color: #6c757d;'>{details}</pre>
                </div>
                """
                display(HTML(details_html))

    # -------------------------------------------------------------------------
    # INFO MESSAGES
    # -------------------------------------------------------------------------

    def render_info(self, message: str) -> None:
        """
        Render informational message.

        Args:
            message: Info message
        """
        with self.output:
            info_html = f"""
            <div style='color: #004085; background-color: #cce5ff; padding: 10px;
                        border-radius: 5px; border: 1px solid #b8daff; margin: 10px 0;'>
                <p style='margin: 0;'>{message}</p>
            </div>
            """
            display(HTML(info_html))


# =============================================================================
# STYLED TABLE ALTERNATIVE (DataFrame with pandas styling)
# =============================================================================

class StyledTableRenderer:
    """
    Alternative renderer using pandas styling capabilities.
    """

    @staticmethod
    def render_styled_dataframe(
        df: pd.DataFrame,
        columns: list,
        highlight_influential: bool = True
    ):
        """
        Render DataFrame with pandas styling.

        Args:
            df: DataFrame to render
            columns: Columns to display
            highlight_influential: Whether to highlight influential studies
        """
        display_df = df[columns].copy()

        # Rename columns for display
        display_df.columns = [
            col.replace('_', ' ').title()
            for col in display_df.columns
        ]

        # Create styled DataFrame
        styled = display_df.style.format({
            'Mu': '{:.4f}',
            'Diff': '{:+.4f}',
            'Pct Change': '{:+.1f}%',
            'Ci Lower': '{:.4f}',
            'Ci Upper': '{:.4f}'
        })

        # Apply background gradient to difference column
        if 'Diff' in display_df.columns:
            styled = styled.background_gradient(
                subset=['Diff'],
                cmap='coolwarm',
                vmin=-display_df['Diff'].abs().max(),
                vmax=display_df['Diff'].abs().max()
            )

        # Highlight influential rows
        if highlight_influential and 'Is Influential' in display_df.columns:
            def highlight_rows(row):
                if row['Is Influential']:
                    return ['background-color: #fff3cd'] * len(row)
                return [''] * len(row)

            styled = styled.apply(highlight_rows, axis=1)

        display(styled)



# =============================================================================
# CONTROLLER LAYER - LEAVE-ONE-OUT SENSITIVITY ANALYSIS
# Purpose: Orchestrates data, analysis, and view components
# Dependencies: All previous layers (Data, Analysis, View)
# =============================================================================

import traceback
from typing import Optional, Dict, Any
from IPython.display import display, HTML
import ipywidgets as widgets


# =============================================================================
# MAIN CONTROLLER
# =============================================================================

class LOOController:
    """
    Master controller that orchestrates the entire leave-one-out analysis workflow.
    Coordinates data management, statistical computation, and UI rendering.
    """

    def __init__(self, analysis_config: Dict[str, Any]):
        """
        Initialize controller with ANALYSIS_CONFIG.

        Args:
            analysis_config: Global ANALYSIS_CONFIG dictionary
        """
        self.analysis_config = analysis_config

        # Initialize components
        try:
            self.data_manager = LOODataManager(analysis_config)
            self.engine = LOOEngine(self.data_manager)
            self.view = LOOResultsView()

            self._initialization_error = None

        except Exception as e:
            # If initialization fails, create minimal view to show error
            self.view = LOOResultsView()
            self.data_manager = None
            self.engine = None
            self._initialization_error = e

        # Create run button
        self._create_run_button()

    # -------------------------------------------------------------------------
    # UI CREATION
    # -------------------------------------------------------------------------

    def _create_run_button(self) -> None:
        """Create run button"""
        self.run_button = widgets.Button(
            description='🔄 Run Leave-One-Out Analysis',
            button_style='primary',
            layout=widgets.Layout(width='400px', height='50px'),
            icon='filter'
        )

        # Attach event handler
        self.run_button.on_click(self._handle_run_click)

    def create_ui(self) -> widgets.VBox:
        """
        Create the user interface with run button.

        Returns:
            VBox widget containing the UI
        """
        ui = widgets.VBox([
            widgets.HTML("<h3>🔍 Sensitivity Analysis</h3>"),
            widgets.HTML("<p style='color: #6c757d;'>Check if any single study drives the entire result.</p>"),
            self.run_button
        ])

        return ui

    # -------------------------------------------------------------------------
    # MAIN EXECUTION METHOD
    # -------------------------------------------------------------------------

    def run_analysis(self) -> None:
        """
        Execute complete leave-one-out sensitivity analysis workflow.
        """
        # Clear output
        with self.view.output:
            self.view.output.clear_output()

        # Check for initialization errors
        if self._initialization_error:
            self._handle_initialization_error()
            return

        # Validate ANALYSIS_CONFIG
        if 'ANALYSIS_CONFIG' not in globals():
            self.view.render_error("ANALYSIS_CONFIG not found. Run Steps 2–6 (data and effect sizes) first.")
            return

        # Check for required function
        if '_run_three_level_reml_for_subgroup' not in globals():
            self.view.render_error(
                "Required function not found",
                "_run_three_level_reml_for_subgroup must be available. Run previous analysis cells."
            )
            return

        # Show progress bar
        self.view.show_progress_bar()

        # Progress callback
        def progress_callback(message: str, progress: Optional[float] = None):
            """Callback for progress updates"""
            self.view.update_progress(message, progress)

        try:
            # Pre-flight check
            progress_callback("📊 Validating data...")

            df = self.data_manager.analysis_data
            if df is None or len(df) == 0:
                self.view.render_error("No data found. Run Steps 2–7 first.")
                return

            # Check minimum studies
            n_studies = df['id'].nunique() if 'id' in df.columns else 0
            if n_studies < 3:
                self.view.render_error(
                    f"Insufficient studies for leave-one-out analysis",
                    f"Found {n_studies} studies, need at least 3."
                )
                return

            progress_callback(f"🔍 Found {n_studies} studies")

            # Execute LOO engine
            result = self.engine.run_analysis(
                progress_callback=progress_callback
            )

            if result is None:
                self.view.render_error(
                    "Analysis failed",
                    "Unable to complete leave-one-out analysis. Check your data and model setup."
                )
                return

            # Save results
            self._save_results(result)

            # Render results
            self.view.render_results(result)

        except ValueError as e:
            self.view.render_error("Data Error", str(e))
        except RuntimeError as e:
            self.view.render_error("Runtime Error", str(e))
        except Exception as e:
            self._handle_unexpected_error(e)

    # -------------------------------------------------------------------------
    # DATA PERSISTENCE
    # -------------------------------------------------------------------------

    def _save_results(self, result: 'LOOResult') -> None:
        """
        Save results to ANALYSIS_CONFIG.

        Args:
            result: LOOResult object
        """
        self.data_manager.save_loo_results(result)

    # -------------------------------------------------------------------------
    # EVENT HANDLERS
    # -------------------------------------------------------------------------

    def _handle_run_click(self, button) -> None:
        """Handle run button click event"""
        self.run_analysis()

    # -------------------------------------------------------------------------
    # ERROR HANDLING
    # -------------------------------------------------------------------------

    def _handle_initialization_error(self) -> None:
        """Handle errors during controller initialization"""
        error = self._initialization_error
        if error is None:
            return

        error_type = type(error).__name__
        error_msg = str(error)
        details = traceback.format_exc()

        self.view.render_error(f"{error_type}: {error_msg}", details)

    def _handle_unexpected_error(self, error: Exception) -> None:
        """Handle unexpected errors during analysis"""
        error_type = type(error).__name__
        error_msg = str(error)
        details = traceback.format_exc()

        self.view.render_error(f"{error_type}: {error_msg}", details)


# =============================================================================
# EXECUTION ENTRY POINT
# =============================================================================

def run_leave_one_out_analysis():
    """
    Main entry point for leave-one-out sensitivity analysis.
    Call this function to display the UI and enable analysis.
    """
    try:
        # Check for ANALYSIS_CONFIG
        if 'ANALYSIS_CONFIG' not in globals():
            print("❌ ERROR: ANALYSIS_CONFIG not found")
            print("Please run previous analysis cells first:")
            print("  - Steps 2–6: Data loading and effect sizes")
            print("  - Step 7: Overall Meta-Analysis")
            return

        # Check for required function
        if '_run_three_level_reml_for_subgroup' not in globals():
            print("❌ ERROR: Required function not found")
            print("Please ensure _run_three_level_reml_for_subgroup is available.")
            print("This function should be loaded from the three-level regression utilities.")
            return

        # Create controller
        controller = LOOController(ANALYSIS_CONFIG)

        # Display UI
        ui = controller.create_ui()
        display(ui)

        # Display output area
        display(controller.view.output)

        # Store controller globally for access (optional)
        globals()['_loo_controller'] = controller

    except Exception as e:
        print(f"❌ Fatal Error: {type(e).__name__}")
        print(f"Message: {str(e)}")
        print("\nFull traceback:")
        traceback.print_exc()


# =============================================================================
# STANDALONE TESTING UTILITIES (Optional)
# =============================================================================

class MockLOOConfig:
    """
    Mock ANALYSIS_CONFIG for testing without running full pipeline.
    Only for development/testing purposes.
    """

    @staticmethod
    def create_sample_config():
        """Create a minimal valid config for testing"""
        import pandas as pd
        import numpy as np

        # Sample data with 10 studies, some observations per study
        np.random.seed(42)

        data_list = []
        for study_id in range(1, 11):
            n_obs = np.random.randint(3, 8)
            for _ in range(n_obs):
                effect = np.random.randn() * 0.2 + 0.5
                variance = np.random.uniform(0.01, 0.1)
                data_list.append({
                    'id': study_id,
                    'hedges_g': effect,
                    'Vg': variance
                })

        sample_data = pd.DataFrame(data_list)

        return {
            'analysis_data': sample_data,
            'effect_col': 'hedges_g',
            'var_col': 'Vg',
            'es_config': {
                'type': "Hedges' g",
                'effect_label': "Hedges' g",
                'effect_label_short': 'g'
            },
            'global_settings': {
                'alpha': 0.05
            }
        }


run_leave_one_out_analysis()

In [ ]:
#@title 📉 22. Sensitivity Analysis: Leave-One-Out: Visualization
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# STEP 22: LEAVE-ONE-OUT FOREST PLOT
# Purpose: Show the pooled estimate (and its CI) after leaving out each study
#          in turn, against the estimate from all studies.
# Data:    Step 21 (Leave-One-Out: Execution), or a plot recipe file saved
#          from this cell.
# =============================================================================

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import html


def _leave_one_out_cell():
    PLOT_TYPE = 'leave_one_out'
    RESULTS_KEY = 'loo_results'
    MAX_AUTO_HEIGHT = 100.0
    state = {'data': None, 'source': None, 'recipe_name': None, 'token': None}

    def _num(v):
        try:
            v = float(v)
        except (TypeError, ValueError):
            return None
        return v if np.isfinite(v) else None

    # --- 1. DATA SNAPSHOT (everything the figure draws) ---
    def build_data():
        loo = ANALYSIS_CONFIG.get(RESULTS_KEY)
        if loo is None:
            return None, None
        if isinstance(loo, pd.DataFrame):
            df, packet = loo.copy(), {}
        else:
            df, packet = loo['data'].copy(), loo
        if df is None or len(df) == 0:
            return None, None
        df = df.rename(columns={'excluded_study': 'unit', 'unit_removed': 'unit', 'pooled_effect': 'mu'})
        if 'ci_lower' not in df.columns or 'ci_upper' not in df.columns:
            from scipy.stats import norm
            z = norm.ppf(1 - _alpha() / 2)
            df['ci_lower'] = df['mu'] - z * df['se']
            df['ci_upper'] = df['mu'] + z * df['se']
        df['unit'] = df['unit'].astype(str)
        cols = [c for c in ('unit', 'mu', 'se', 'ci_lower', 'ci_upper', 'diff') if c in df.columns]
        df = df[cols].dropna(subset=['mu', 'ci_lower', 'ci_upper']).reset_index(drop=True)
        es = ANALYSIS_CONFIG.get('es_config') or {}
        ci_level = ci_pct_label()
        data = {
            'loo': df,
            'orig_mu': _num(packet.get('orig_mu')),
            'orig_ci_lo': _num(packet.get('orig_ci_lo')),
            'orig_ci_hi': _num(packet.get('orig_ci_hi')),
            'n_studies': int(packet.get('n_studies') or len(df)),
            'ci_level': ci_level,
            'null_value': es_null_value(),
            'effect_label': es.get('effect_label', 'Effect Size'),
        }
        when = packet.get('timestamp')
        source = {
            'analysis': 'Leave-one-out by study (Step 21)',
            'effect_label': data['effect_label'],
            'model': 'multilevel REML, CR2 robust CI',
            'ci_level': ci_level,
            'analysis_time': when.strftime('%Y-%m-%d %H:%M') if hasattr(when, 'strftime') else when,
        }
        return data, source

    # --- 2. CONTROLS (every widget whose value is saved in a recipe) ---
    L = lambda w='450px': widgets.Layout(width=w)
    C = {
        'show_title': widgets.Checkbox(value=True, description='Show Plot Title', indent=False),
        'title': widgets.Text(value='Leave-One-Out Sensitivity Analysis', description='Title:', layout=L()),
        'xlabel': widgets.Text(value='Pooled Effect Size (if study removed)', description='X Label:', layout=L()),
        'ylabel': widgets.Text(value='Study Removed', description='Y Label:', layout=L()),
        'title_font': widgets.IntSlider(value=14, min=6, max=24, description='Title Font:', layout=L()),
        'label_font': widgets.IntSlider(value=12, min=6, max=18, description='Label Font:', layout=L()),
        'tick_font': widgets.IntSlider(value=9, min=5, max=16, description='Tick Font:', layout=L()),
        'width': widgets.FloatSlider(value=10.0, min=2.0, max=16.0, step=0.1, description='Width (in):',
                                     continuous_update=False, layout=L()),
        'height_auto': widgets.Checkbox(value=True, description='Auto-Height', indent=False),
        'height': widgets.FloatSlider(value=12.0, min=2.0, max=100.0, step=0.1, description='Manual Height:',
                                      continuous_update=False, layout=L()),
        'sort_by': widgets.Dropdown(options=[('Effect Size (Low to High)', 'effect'),
                                             ('Influence (Most influential at top)', 'influence'),
                                             ('Study ID', 'id')],
                                    value='effect', description='Sort By:', layout=L('400px')),
        'highlight_sig': widgets.Checkbox(value=True, description='Highlight Significance Changers (Red)',
                                          indent=False),
        'point_color': widgets.Dropdown(options=['blue', 'black', 'gray', 'steelblue'], value='blue',
                                        description='Point Color:'),
        'point_size': widgets.IntSlider(value=40, min=10, max=100, description='Point Size:'),
        'show_orig_line': widgets.Checkbox(value=True, description='Show Original Effect Line'),
        'orig_color': widgets.Dropdown(options=['red', 'black', 'green'], value='red', description='Line Color:'),
        'show_orig_ci': widgets.Checkbox(value=True, description='Show Original CI Band'),
        'ci_band_alpha': widgets.FloatSlider(value=0.1, min=0.05, max=0.5, step=0.05, description='Band Alpha:'),
        'show_null': widgets.Checkbox(value=True, description='Show Null Effect Line'),
        'save_pdf': widgets.Checkbox(value=True, description='Save as PDF', indent=False),
        'save_png': widgets.Checkbox(value=True, description='Save as PNG', indent=False),
        'save_svg': widgets.Checkbox(value=False, description='Save as SVG', indent=False),
        'dpi': widgets.IntSlider(value=300, min=100, max=600, step=50, description='PNG DPI:'),
        'transparent': widgets.Checkbox(value=False, description='Transparent background', indent=False),
        'filename': widgets.Text(value='LOO_Plot', description='Filename:', layout=L('300px')),
    }

    def set_data_defaults(data):
        """Defaults that depend on the data (called for session data and recipes)."""
        C['xlabel'].value = f"Pooled {data['effect_label']} (if study removed)"
        C['show_orig_ci'].description = f"Show Original {data['ci_level']} CI Band"

    # --- 3. DRAWING (data + settings -> figure; no widget access) ---
    def _natural_key(text):
        return [(0, int(t), '') if t.isdigit() else (1, 0, t.lower())
                for t in re.findall(r'\d+|\D+', str(text))]

    def has_orig_ci(data):
        return data.get('orig_ci_lo') is not None and data.get('orig_ci_hi') is not None

    def prepare(data, s):
        """Rows in drawing order (first row at the top) with influence and significance flags."""
        df = data['loo'].copy()
        if data.get('orig_mu') is not None:
            df['influence'] = (df['mu'] - data['orig_mu']).abs()
        elif 'diff' in df.columns:
            df['influence'] = df['diff'].abs()
        else:
            df['influence'] = np.nan
        null = data['null_value']
        if has_orig_ci(data):
            orig_sig = not (data['orig_ci_lo'] <= null <= data['orig_ci_hi'])
            row_sig = ~((df['ci_lower'] <= null) & (null <= df['ci_upper']))
            df['changes_sig'] = row_sig != orig_sig
        else:
            df['changes_sig'] = False
        if s['sort_by'] == 'influence':
            df = df.sort_values(['influence', 'mu'], ascending=[False, True], kind='mergesort',
                                na_position='last')
        elif s['sort_by'] == 'id':
            df = df.iloc[sorted(range(len(df)), key=lambda i: _natural_key(df['unit'].iloc[i]))]
        else:
            df = df.sort_values('mu', ascending=True, kind='mergesort')
        return df.reset_index(drop=True)

    def figure_height(n, s):
        if not s['height_auto']:
            return s['height'], False
        h = max(6.0, (n + 2) * 0.25)
        return min(h, MAX_AUTO_HEIGHT), h > MAX_AUTO_HEIGHT

    def draw(data, s):
        df = prepare(data, s)
        n = len(df)
        fig_h, _capped = figure_height(n, s)
        fig, ax = plt.subplots(figsize=(s['width'], fig_h))
        ys = np.arange(n)[::-1]

        highlight = s['highlight_sig'] and has_orig_ci(data)
        colors = ['red' if (highlight and c) else s['point_color'] for c in df['changes_sig']]
        ax.hlines(ys, df['ci_lower'], df['ci_upper'], color='black', linewidth=1, zorder=1)
        ax.scatter(df['mu'], ys, color=colors, s=s['point_size'], zorder=2)

        if s['show_null']:
            ax.axvline(data['null_value'], color='black', linestyle='-', linewidth=1, alpha=0.5, zorder=0)
        if s['show_orig_ci'] and has_orig_ci(data):
            ax.axvspan(data['orig_ci_lo'], data['orig_ci_hi'], color=s['orig_color'], alpha=s['ci_band_alpha'],
                       label=f"Original {data['ci_level']} CI", zorder=0)
        if s['show_orig_line'] and data.get('orig_mu') is not None:
            ax.axvline(data['orig_mu'], color=s['orig_color'], linestyle='--', linewidth=2,
                       label=f"Original: {data['orig_mu']:.3f}", zorder=0)

        ax.set_ylim(-1, n)
        ax.set_yticks(ys)
        ax.set_yticklabels(df['unit'].tolist())
        ax.tick_params(axis='y', length=0, labelsize=s['tick_font'])
        ax.tick_params(axis='x', labelsize=s['tick_font'])
        ax.set_xlabel(s['xlabel'], fontweight='bold', fontsize=s['label_font'])
        ax.set_ylabel(s['ylabel'], fontweight='bold', fontsize=s['label_font'])
        if s['show_title']:
            ax.set_title(s['title'], fontweight='bold', fontsize=s['title_font'], pad=20)
        for side in ('left', 'right', 'top'):
            ax.spines[side].set_visible(False)
        ax.xaxis.set_ticks_position('both')
        ax.xaxis.set_label_position('bottom')

        handles, _ = ax.get_legend_handles_labels()
        if highlight and df['changes_sig'].any():
            handles.append(mpatches.Patch(color='red', label='Changed Significance'))
        if handles:
            ax.legend(handles=handles, loc='best', frameon=True, fancybox=True, fontsize=s['tick_font'])
        fig.tight_layout()
        return fig, df

    def caption(data, s, shown):
        lvl = data['ci_level']
        n = len(shown)
        txt = (f"Leave-one-out sensitivity analysis of {html.escape(str(data['effect_label']))}. Each row shows "
               f"the pooled estimate and its {lvl} confidence interval after removing the study named on the "
               f"y-axis ({n} refits of {data['n_studies']} studies). Estimates ranged from "
               f"{shown['mu'].min():.3f} to {shown['mu'].max():.3f}.")
        if s['sort_by'] == 'influence':
            txt += (" Rows are sorted by influence (absolute change from the estimate with all studies), "
                    "most influential at the top.")
        elif s['sort_by'] == 'id':
            txt += " Rows are sorted by study identifier."
        else:
            txt += " Rows are sorted by the resulting pooled estimate, lowest at the top."
        if data.get('orig_mu') is None:
            txt += " The estimate with all studies was not stored with these results, so it is not drawn."
        else:
            ci_txt = (f" ({lvl} CI {data['orig_ci_lo']:.3f} to {data['orig_ci_hi']:.3f})"
                      if has_orig_ci(data) else "")
            if s['show_orig_line']:
                txt += f" The dashed line is the pooled estimate with all studies ({data['orig_mu']:.3f}{ci_txt})."
            if s['show_orig_ci'] and has_orig_ci(data):
                txt += f" The shaded band is its {lvl} confidence interval."
        if s['show_null']:
            txt += f" The solid vertical line marks no effect ({data['null_value']:g})."
        if s['highlight_sig']:
            if not has_orig_ci(data):
                txt += (" Changes in significance are not marked because the confidence interval with all "
                        "studies was not stored with these results.")
            elif shown['changes_sig'].any():
                names = ', '.join(html.escape(u) for u in shown.loc[shown['changes_sig'], 'unit'])
                txt += (f" Red points mark studies whose removal changes whether the {lvl} confidence interval "
                        f"includes the null value ({names}).")
            else:
                txt += (f" Removing any single study does not change whether the {lvl} confidence interval "
                        f"includes the null value.")
        return txt

    # --- 4. ACTIONS ---
    out = widgets.Output()
    caption_output = widgets.Output()
    source_box = widgets.HTML()

    def refresh_source():
        source_box.value = source_html(state['source'], state['recipe_name'])

    def generate(_b=None):
        with out:
            clear_output(wait=True)
            if state['data'] is None:
                display(HTML("<div style='color:#721c24;'>❌ No leave-one-out results: run Step 21 "
                             "(Leave-One-Out: Execution) or load a plot recipe.</div>"))
                return
            if state['recipe_name'] is None and results_token(RESULTS_KEY) != state['token']:
                display(HTML(stale_warning_html('Step 21 (Leave-One-Out: Execution)')))
            s = widget_values(C)
            try:
                fig, shown = draw(state['data'], s)
                paths = cometa_save_figure(fig, s['filename'], pdf=s['save_pdf'], png=s['save_png'],
                                           svg=s['save_svg'], dpi=s['dpi'], transparent=s['transparent'])
                show_figure(fig)
            except Exception as e:
                plt.close('all')
                display(HTML(f"<div style='color:#721c24;'>❌ Plotting error: {html.escape(str(e))}</div>"))
                return
            if figure_height(len(shown), s)[1]:
                display(HTML(f"<div style='color:#856404;'>⚠️ Auto-height capped at {MAX_AUTO_HEIGHT:g} in.</div>"))
            display(HTML(saved_files_html(paths)))
            cap = caption_html(caption(state['data'], s, shown))
            display(HTML(cap))
            with caption_output:
                clear_output()
                display(HTML(cap))

    def get_recipe():
        if state['data'] is None:
            return None
        return build_plot_recipe(PLOT_TYPE, state['data'], widget_values(C), state['source'])

    def on_load(rec, style_only, name):
        if not style_only:
            state.update(data=rec['data'], source=rec.get('source'), recipe_name=name)
            set_data_defaults(state['data'])
            refresh_source()
        skipped = apply_widget_values(C, rec.get('settings'))
        generate()
        return skipped

    # --- 5. INITIAL STATE AND LAYOUT ---
    try:
        state['data'], state['source'] = build_data()
    except Exception as e:
        state['data'], state['source'] = None, None
        print(f"⚠️ Could not read the Step 21 results: {e}")
    state['token'] = results_token(RESULTS_KEY)
    if state['data'] is not None:
        set_data_defaults(state['data'])
    refresh_source()

    preset = preset_dropdown({'width': C['width'], 'height': C['height'], 'title': C['title_font'],
                              'label': C['label_font'], 'tick': C['tick_font']})

    def on_preset(change):
        if change['new'] != 'Custom':
            C['height_auto'].value = False
    preset.observe(on_preset, names='value')

    H = lambda t: widgets.HTML(f"<h4 style='color: #2E86AB;'>{t}</h4>")
    tabs = widgets.Tab(children=[
        widgets.VBox([H('Labels & Dimensions'), C['show_title'], C['title'], C['xlabel'], C['ylabel'],
                      widgets.HTML("<hr>"), preset, C['width'], C['height_auto'], C['height'],
                      widgets.HTML("<hr><b>Typography:</b>"), C['title_font'], C['label_font'], C['tick_font']]),
        widgets.VBox([H('Data Presentation'), C['sort_by'], widgets.HTML("<hr>"), C['highlight_sig'],
                      C['point_color'], C['point_size']]),
        widgets.VBox([H('Reference Lines'), C['show_orig_line'], C['orig_color'], C['show_orig_ci'],
                      C['ci_band_alpha'], widgets.HTML("<hr>"), C['show_null']]),
        widgets.VBox([H('Export'), C['save_pdf'], C['save_png'], C['save_svg'], C['dpi'], C['transparent'],
                      C['filename']]),
        widgets.VBox([H('Figure Caption'),
                      widgets.HTML("<p style='font-size: 13px; color: #555;'>Generated automatically when you "
                                   "create the plot.</p>"), caption_output]),
    ])
    for i, t in enumerate(['🎨 Style', '📊 Data', '📐 Lines', '💾 Export', '📝 Caption']):
        tabs.set_title(i, t)

    run_btn = widgets.Button(description='📊 Generate LOO Plot', button_style='success',
                             layout=widgets.Layout(width='450px'))
    run_btn.on_click(generate)
    recipe_bar = plot_recipe_bar(PLOT_TYPE, get_recipe, on_load, lambda: C['filename'].value)

    header = widgets.HTML("""<div style="background-color: #f0f4f8; padding: 15px; border-radius: 5px; border-left: 5px solid #2E86AB;">
        <h3 style="margin:0; color: #2E86AB;">Leave-One-Out Sensitivity Analysis</h3>
        <p style="margin:5px 0 0 0;">Visualize the influence of individual studies on the pooled estimate.</p>
    </div>""")
    display(widgets.VBox([header, source_box, tabs, widgets.HTML("<hr>"), run_btn, recipe_bar, out]))


_leave_one_out_cell()


In [ ]:
#@title 📊 23. Sensitivity Analysis: Baujat Plot
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# STEP 23: BAUJAT PLOT
# Purpose: Identify studies that contribute heavily to both
#          heterogeneity AND influence the pooled effect.
# Data:    Step 21 (Leave-One-Out: Execution) and the analysis data, or a
#          plot recipe file saved from this cell.
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.transforms import Bbox
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import html


def compute_baujat_data(raw, loo, effect_col, var_col, orig_mu):
    """Baujat et al. (2002) coordinates, one row per study.

    x (q_contrib): the study's contribution to Cochran's Q of the fixed-effect
    model, sum_j w_ij (y_ij - mu_FE)^2 with w = 1/v, summed over its effect sizes
    (rows cleaned as in Step 21).
    y (influence): (mu - mu_(-i))^2 / SE(mu_(-i))^2, from the leave-one-out refits.
    `loo` has one row per left-out study with columns unit, mu, se.
    Returns (points, mu_FE)."""
    d = raw.dropna(subset=[effect_col, var_col, 'id'])
    d = d[d[var_col] > 0]
    y, w = d[effect_col].astype(float), 1.0 / d[var_col].astype(float)
    mu_fe = float(np.sum(w * y) / np.sum(w))
    q = (w * (y - mu_fe) ** 2).groupby(d['id'], sort=False).sum()
    k = d.groupby('id', sort=False).size()
    qdf = pd.DataFrame({'unit': [str(i) for i in q.index], 'q_contrib': q.to_numpy(dtype=float),
                        'n_es': k.reindex(q.index).to_numpy(dtype=int)})
    loo = loo[['unit', 'mu', 'se']].copy()
    loo['unit'] = loo['unit'].astype(str)
    loo['influence'] = (loo['mu'] - orig_mu) ** 2 / loo['se'] ** 2
    pts = pd.merge(loo, qdf, on='unit', how='inner').dropna(subset=['influence', 'q_contrib'])
    return pts.reset_index(drop=True), mu_fe


def _baujat_cell():
    PLOT_TYPE = 'baujat'
    RESULTS_KEY = 'loo_results'
    state = {'data': None, 'source': None, 'recipe_name': None, 'token': None, 'error': None}

    def _num(v):
        try:
            v = float(v)
        except (TypeError, ValueError):
            return None
        return v if np.isfinite(v) else None

    # --- 1. DATA SNAPSHOT (everything the figure draws) ---
    def build_data():
        loo = ANALYSIS_CONFIG.get(RESULTS_KEY)
        if loo is None:
            return None, None
        if isinstance(loo, pd.DataFrame):
            df, packet = loo.copy(), {}
        else:
            df, packet = loo['data'].copy(), loo
        if df is None or len(df) == 0:
            return None, None
        raw = ANALYSIS_CONFIG.get('analysis_data')
        if raw is None:
            raise ValueError("The analysis data are not in this session (needed for the Q contributions).")
        effect_col = ANALYSIS_CONFIG.get('effect_col', 'hedges_g')
        var_col = ANALYSIS_CONFIG.get('var_col', 'Vg')

        df = df.rename(columns={'excluded_study': 'unit', 'unit_removed': 'unit', 'pooled_effect': 'mu'})
        df['unit'] = df['unit'].astype(str)
        orig_mu = _num(packet.get('orig_mu'))
        if orig_mu is None and 'diff' in df.columns:
            orig_mu = float((df['mu'] + df['diff']).median())   # diff = mu_all - mu_(-i)
        if orig_mu is None:
            raise ValueError("The leave-one-out results do not include the estimate with all studies.")

        pts, mu_fe = compute_baujat_data(raw, df, effect_col, var_col, orig_mu)
        if len(pts) == 0:
            raise ValueError("No study matches between the leave-one-out results and the analysis data.")
        es = ANALYSIS_CONFIG.get('es_config') or {}
        ci_level = ci_pct_label()
        data = {
            'points': pts.rename(columns={'mu': 'mu_loo', 'se': 'se_loo'}),
            'orig_mu': orig_mu,
            'mu_fe': mu_fe,
            'q_total': float(pts['q_contrib'].sum()),
            'n_loo': int(len(df)),
            'effect_label': es.get('effect_label', 'Effect Size'),
            'ci_level': ci_level,
        }
        when = packet.get('timestamp')
        source = {
            'analysis': 'Leave-one-out by study (Step 21)',
            'effect_label': data['effect_label'],
            'model': 'x: fixed-effect Q; y: multilevel REML leave-one-out',
            'ci_level': ci_level,
            'analysis_time': when.strftime('%Y-%m-%d %H:%M') if hasattr(when, 'strftime') else when,
        }
        return data, source

    # --- 2. CONTROLS (every widget whose value is saved in a recipe) ---
    L = lambda w='450px': widgets.Layout(width=w)
    C = {
        'width': widgets.FloatSlider(value=10.0, min=2.0, max=16.0, step=0.1, description='Plot Width (in):',
                                     continuous_update=False, layout=L()),
        'height': widgets.FloatSlider(value=8.0, min=2.0, max=14.0, step=0.1, description='Plot Height (in):',
                                      continuous_update=False, layout=L()),
        'point_size': widgets.IntSlider(value=80, min=20, max=200, step=10, description='Point Size:', layout=L()),
        'point_color': widgets.Dropdown(options=['steelblue', 'darkblue', 'black', 'gray', 'crimson', 'forestgreen'],
                                        value='steelblue', description='Point Color:', layout=L()),
        'alpha': widgets.FloatSlider(value=0.6, min=0.1, max=1.0, step=0.05, description='Opacity (Alpha):',
                                     layout=L()),
        'show_median_lines': widgets.Checkbox(value=True, description='Show Median Reference Lines', indent=False),
        'show_title': widgets.Checkbox(value=True, description='Show Plot Title', indent=False),
        'title': widgets.Text(value='Baujat Plot: Heterogeneity vs Influence', description='Plot Title:',
                              layout=L()),
        'xlabel': widgets.Text(value='Contribution to Heterogeneity (Q)', description='X-Axis Label:', layout=L()),
        'ylabel': widgets.Text(value=r'Influence on the pooled estimate, $(\mu - \mu_{-i})^2/\mathrm{SE}^2_{-i}$',
                               description='Y-Axis Label:', layout=L()),
        'title_font': widgets.IntSlider(value=14, min=6, max=24, description='Title Size:', layout=L()),
        'label_font': widgets.IntSlider(value=12, min=6, max=18, description='Axis Label Size:', layout=L()),
        'tick_font': widgets.IntSlider(value=10, min=5, max=16, description='Tick Size:', layout=L()),
        'show_labels': widgets.Checkbox(value=True, description='Show Study Labels', indent=False),
        'label_method': widgets.Dropdown(options=[('Top Outliers (Combined Score)', 'combined'),
                                                  ('Top by Heterogeneity Only', 'heterogeneity'),
                                                  ('Top by Influence Only', 'influence'), ('All Studies', 'all')],
                                         value='combined', description='Labeling Method:', layout=L()),
        'label_threshold': widgets.IntSlider(value=5, min=1, max=20, description='Max Labels:', layout=L()),
        'label_position': widgets.Dropdown(options=[('Auto (avoid overlaps)', 'best'), ('Right', 'right'),
                                                    ('Left', 'left'), ('Top', 'top'), ('Bottom', 'bottom')],
                                           value='best', description='Label Position:', layout=L()),
        'study_label_font': widgets.IntSlider(value=9, min=5, max=14, description='Label Size:', layout=L()),
        'save_pdf': widgets.Checkbox(value=True, description='Save as PDF', indent=False),
        'save_png': widgets.Checkbox(value=True, description='Save as PNG', indent=False),
        'save_svg': widgets.Checkbox(value=False, description='Save as SVG', indent=False),
        'dpi': widgets.IntSlider(value=300, min=72, max=600, step=50, description='PNG DPI:', layout=L()),
        'transparent': widgets.Checkbox(value=False, description='Transparent background', indent=False),
        'filename': widgets.Text(value='Baujat_Plot', description='Filename:', layout=L()),
        'timestamp': widgets.Checkbox(value=True, description='Include Timestamp', indent=False),
    }

    # --- 3. DRAWING (data + settings -> figure; no widget access) ---
    def combined_score(pts):
        """Distance from the origin after scaling each axis by its maximum."""
        qx = pts['q_contrib'].max()
        iy = pts['influence'].max()
        x = pts['q_contrib'] / qx if qx > 0 else 0.0 * pts['q_contrib']
        y = pts['influence'] / iy if iy > 0 else 0.0 * pts['influence']
        return np.sqrt(x ** 2 + y ** 2)

    def labelled(data, s):
        pts = data['points']
        if not s['show_labels']:
            return pts.iloc[0:0]
        n = int(s['label_threshold'])
        if s['label_method'] == 'combined':
            return pts.assign(_score=combined_score(pts)).sort_values(
                '_score', ascending=False, kind='mergesort').head(n)
        if s['label_method'] == 'heterogeneity':
            return pts.sort_values('q_contrib', ascending=False, kind='mergesort').head(n)
        if s['label_method'] == 'influence':
            return pts.sort_values('influence', ascending=False, kind='mergesort').head(n)
        return pts

    FIXED = {'right': ((8, 0), 'left', 'center'), 'left': ((-8, 0), 'right', 'center'),
             'top': ((0, 8), 'center', 'bottom'), 'bottom': ((0, -8), 'center', 'top')}
    CANDIDATES = [(6, 6), (6, -6), (-6, 6), (-6, -6), (9, 0), (-9, 0), (0, 9), (0, -9)]

    def _overlap(a, b):
        w = min(a.x1, b.x1) - max(a.x0, b.x0)
        h = min(a.y1, b.y1) - max(a.y0, b.y0)
        return w * h if w > 0 and h > 0 else 0.0

    def place_labels(fig, ax, pts, rows, s):
        """Fixed offsets, or for 'Auto' the first nearby position whose box does not
        overlap the points, the earlier labels or the axes edge (the least overlap
        when every position is taken)."""
        style = dict(fontsize=s['study_label_font'], textcoords='offset points',
                     bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.3))
        if s['label_position'] in FIXED:
            off, ha, va = FIXED[s['label_position']]
            for _, r in rows.iterrows():
                ax.annotate(r['unit'], (r['q_contrib'], r['influence']), xytext=off, ha=ha, va=va, **style)
            return
        renderer = fig.canvas.get_renderer()
        frame = ax.get_window_extent(renderer)
        r_px = np.sqrt(s['point_size']) / 2 * fig.dpi / 72
        xy = ax.transData.transform(np.column_stack([pts['q_contrib'], pts['influence']]))
        points = [Bbox([[x - r_px, y - r_px], [x + r_px, y + r_px]]) for x, y in xy]
        placed = []
        for _, r in rows.iterrows():
            best = None
            for scale in (1, 2, 3, 4, 6):
                for dx, dy in CANDIDATES:
                    off = (dx * scale, dy * scale)
                    ha = 'left' if dx > 0 else ('right' if dx < 0 else 'center')
                    va = 'bottom' if dy > 0 else ('top' if dy < 0 else 'center')
                    t = ax.annotate(r['unit'], (r['q_contrib'], r['influence']), xytext=off, ha=ha, va=va,
                                    **style)
                    bb = t.get_window_extent(renderer).expanded(1.05, 1.15)
                    t.remove()
                    cost = (sum(_overlap(bb, o) for o in points) + 3 * sum(_overlap(bb, o) for o in placed)
                            + 3 * (bb.width * bb.height - _overlap(bb, frame)))
                    if best is None or cost < best[0]:
                        best = (cost, off, ha, va, bb, scale)
                    if cost == 0:
                        break
                if best[0] == 0:
                    break
            _, off, ha, va, bb, scale = best
            arrow = dict(arrowprops=dict(arrowstyle='-', color='gray', lw=0.5)) if scale > 1 else {}
            ax.annotate(r['unit'], (r['q_contrib'], r['influence']), xytext=off, ha=ha, va=va, **arrow, **style)
            placed.append(bb)

    def draw(data, s):
        pts = data['points']
        fig, ax = plt.subplots(figsize=(s['width'], s['height']), facecolor='white')
        ax.scatter(pts['q_contrib'], pts['influence'], s=s['point_size'], c=s['point_color'], alpha=s['alpha'],
                   edgecolors='black', linewidths=0.5, zorder=3)
        if s['show_median_lines']:
            ax.axvline(pts['q_contrib'].median(), color='gray', linestyle='--', alpha=0.5)
            ax.axhline(pts['influence'].median(), color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel(s['xlabel'], fontsize=s['label_font'], fontweight='bold')
        ax.set_ylabel(s['ylabel'], fontsize=s['label_font'], fontweight='bold')
        ax.tick_params(axis='both', labelsize=s['tick_font'])
        if s['show_title']:
            ax.set_title(s['title'], fontsize=s['title_font'], fontweight='bold', pad=15)
        ax.grid(True, linestyle=':', alpha=0.4)
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0)
        fig.tight_layout()
        rows = labelled(data, s)
        place_labels(fig, ax, pts, rows, s)
        return fig, rows

    def caption(data, s, rows):
        pts = data['points']
        txt = (f"Baujat plot of {len(pts)} studies ({html.escape(str(data['effect_label']))}). The x-axis is each "
               f"study's contribution to Cochran's <i>Q</i> of the fixed-effect model, "
               f"&Sigma;<sub>j</sub> w<sub>ij</sub>(y<sub>ij</sub> &minus; &mu;<sub>FE</sub>)<sup>2</sup> with "
               f"w = 1/v, summed over its effect sizes (total <i>Q</i> = {data['q_total']:.2f}). The y-axis is the "
               f"study's influence on the pooled estimate, (&mu; &minus; &mu;<sub>&minus;i</sub>)<sup>2</sup>/"
               f"SE<sup>2</sup><sub>&minus;i</sub>, the squared change in the multilevel pooled estimate when the "
               f"study is left out (Step 21), standardized by the standard error of the estimate without it "
               f"(Baujat et al., 2002). Studies toward the top right both contribute much heterogeneity and "
               f"shift the pooled estimate.")
        if len(pts) < data['n_loo']:
            txt += f" {data['n_loo'] - len(pts)} leave-one-out refits had no matching study in the data."
        if s['show_median_lines']:
            txt += " Dashed lines mark the median of each axis."
        if len(rows) and s['label_method'] != 'all':
            desc = {'combined': ('the largest distance from the origin after scaling each axis by its maximum'),
                    'heterogeneity': 'the largest contribution to heterogeneity',
                    'influence': 'the largest influence on the pooled estimate'}[s['label_method']]
            names = [html.escape(u) for u in rows['unit']]
            names_txt = names[0] if len(names) == 1 else ', '.join(names[:-1]) + ' and ' + names[-1]
            txt += f" Labelled studies ({names_txt}) are the {len(rows)} with {desc}."
        elif len(rows):
            txt += " All studies are labelled."
        return txt

    # --- 4. ACTIONS ---
    out = widgets.Output()
    caption_output = widgets.Output()
    source_box = widgets.HTML()

    def refresh_source():
        source_box.value = source_html(state['source'], state['recipe_name'])

    def generate(_b=None):
        with out:
            clear_output(wait=True)
            if state['data'] is None:
                why = f" ({html.escape(state['error'])})" if state['error'] else ""
                display(HTML("<div style='color:#721c24;'>❌ No leave-one-out results: run Step 21 "
                             f"(Leave-One-Out: Execution) or load a plot recipe{why}.</div>"))
                return
            if state['recipe_name'] is None and results_token(RESULTS_KEY) != state['token']:
                display(HTML(stale_warning_html('Step 21 (Leave-One-Out: Execution)')))
            s = widget_values(C)
            try:
                fig, rows = draw(state['data'], s)
                paths = cometa_save_figure(fig, s['filename'], pdf=s['save_pdf'], png=s['save_png'],
                                           svg=s['save_svg'], dpi=s['dpi'], transparent=s['transparent'],
                                           timestamp=s['timestamp'])
                show_figure(fig)
            except Exception as e:
                plt.close('all')
                display(HTML(f"<div style='color:#721c24;'>❌ Plotting error: {html.escape(str(e))}</div>"))
                return
            display(HTML(saved_files_html(paths)))
            cap = caption_html(caption(state['data'], s, rows))
            display(HTML(cap))
            with caption_output:
                clear_output()
                display(HTML(cap))

    def get_recipe():
        if state['data'] is None:
            return None
        return build_plot_recipe(PLOT_TYPE, state['data'], widget_values(C), state['source'])

    def on_load(rec, style_only, name):
        if not style_only:
            state.update(data=rec['data'], source=rec.get('source'), recipe_name=name, error=None)
            refresh_source()
        skipped = apply_widget_values(C, rec.get('settings'))
        generate()
        return skipped

    # --- 5. INITIAL STATE AND LAYOUT ---
    try:
        state['data'], state['source'] = build_data()
    except Exception as e:
        state['data'], state['source'], state['error'] = None, None, str(e)
        print(f"⚠️ Could not build the Baujat data: {e}")
    state['token'] = results_token(RESULTS_KEY)
    refresh_source()

    preset = preset_dropdown({'width': C['width'], 'height': C['height'], 'title': C['title_font'],
                              'label': C['label_font'], 'tick': C['tick_font']})
    H = lambda t: widgets.HTML(f"<h3 style='color: #2E86AB;'>{t}</h3>")
    tabs = widgets.Tab(children=[
        widgets.VBox([H('Plot Style & Layout'), preset, C['width'], C['height'], widgets.HTML("<hr>"),
                      C['point_size'], C['point_color'], C['alpha'], widgets.HTML("<hr>"),
                      C['show_median_lines']]),
        widgets.VBox([H('Text & Labels'), C['show_title'], C['title'], widgets.HTML("<hr>"), C['xlabel'],
                      C['ylabel'], widgets.HTML("<hr>"), C['title_font'], C['label_font'], C['tick_font']]),
        widgets.VBox([H('Outlier Study Labels'), C['show_labels'], C['label_method'], C['label_threshold'],
                      widgets.HTML("<hr>"), C['label_position'], C['study_label_font']]),
        widgets.VBox([H('Export Options'), C['save_pdf'], C['save_png'], C['save_svg'], C['dpi'],
                      C['transparent'], widgets.HTML("<hr>"), C['filename'], C['timestamp']]),
        widgets.VBox([H('Figure Caption'),
                      widgets.HTML("<p style='font-size: 13px; color: #555;'>Generated automatically when you "
                                   "create the plot.</p>"), caption_output]),
    ])
    for i, t in enumerate(['🎨 Style', '📝 Text', '🏷️ Study labels', '💾 Export', '📝 Caption']):
        tabs.set_title(i, t)

    run_btn = widgets.Button(description='📊 Generate Baujat Plot', button_style='success',
                             layout=widgets.Layout(width='450px', height='50px'))
    run_btn.on_click(generate)
    recipe_bar = plot_recipe_bar(PLOT_TYPE, get_recipe, on_load, lambda: C['filename'].value)

    display(widgets.VBox([
        widgets.HTML("<h3>📊 Step 23: Baujat Plot</h3><p>Diagnostic for Heterogeneity vs Influence</p>"),
        source_box, tabs, run_btn, recipe_bar, out,
    ]))


_baujat_cell()


In [ ]:
#@title 📈 24. Complementary: Cumulative Meta-Analysis
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# Purpose: Centralized data management for cumulative meta-analysis
# Dependencies: ANALYSIS_CONFIG global dictionary
# Math validated: Iterative REML estimation for temporal evolution
# =============================================================================

import numpy as np
import pandas as pd
from typing import Dict, Any, Optional, Tuple, List
from dataclasses import dataclass
import warnings


# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class CumulativeConfig:
    """Configuration for cumulative meta-analysis"""
    effect_col: str
    var_col: str
    year_col: str = 'year'
    sort_order: str = 'ascending'  # 'ascending' or 'descending'
    aggregation_method: str = 'multilevel'  # 'multilevel', 'study' or 'obs'
    alpha: float = 0.05
    dist_type: str = 'norm'  # 'norm' or 't'

    def __post_init__(self):
        """Validate configuration"""
        if not self.effect_col:
            raise ValueError("effect_col cannot be empty")
        if not self.var_col:
            raise ValueError("var_col cannot be empty")
        if self.sort_order not in ['ascending', 'descending']:
            raise ValueError(f"sort_order must be 'ascending' or 'descending', got {self.sort_order}")
        if self.aggregation_method not in ['multilevel', 'study', 'obs']:
            raise ValueError(f"aggregation_method must be 'multilevel', 'study' or 'obs', "
                             f"got {self.aggregation_method}")
        if self.alpha <= 0 or self.alpha >= 1:
            raise ValueError(f"alpha must be between 0 and 1, got {self.alpha}")


@dataclass
class CumulativeStep:
    """Single step in cumulative meta-analysis"""
    step: int
    year: int
    study_added: str
    k_studies: int
    pooled_effect: float
    se: float
    ci_lower: float
    ci_upper: float
    tau2: float
    I2: float
    ci_pct: float


@dataclass
class CumulativeResult:
    """Complete cumulative meta-analysis result"""
    steps: pd.DataFrame  # All cumulative steps
    start_step: CumulativeStep
    end_step: CumulativeStep
    n_steps: int
    year_range: Tuple[int, int]
    aggregation_method: str
    sort_order: str

    # Summary metrics
    effect_stable: bool = False  # Did effect stabilize?
    precision_improved: bool = False  # Did CI narrow?


# =============================================================================
# DATA MANAGER CLASS
# =============================================================================

class CumulativeDataManager:
    """
    Centralized data access layer for cumulative meta-analysis.
    Handles all interactions with ANALYSIS_CONFIG and data validation.
    """

    def __init__(self, analysis_config: Dict[str, Any]):
        """
        Initialize data manager.

        Args:
            analysis_config: Global ANALYSIS_CONFIG dictionary
        """
        self._config = analysis_config
        self._validate_prerequisites()

    # -------------------------------------------------------------------------
    # VALIDATION METHODS
    # -------------------------------------------------------------------------

    def _validate_prerequisites(self) -> None:
        """Validate that required configuration exists"""
        if 'effect_col' not in self._config:
            warnings.warn("effect_col not in ANALYSIS_CONFIG, using default 'hedges_g'")
        if 'var_col' not in self._config:
            warnings.warn("var_col not in ANALYSIS_CONFIG, using default 'Vg'")

    def validate_data(self, df: pd.DataFrame) -> Tuple[bool, Optional[str]]:
        """
        Validate that data is suitable for cumulative analysis.

        Args:
            df: DataFrame to validate

        Returns:
            Tuple of (is_valid, error_message)
        """
        if df is None or len(df) == 0:
            return False, "No data available"

        if self.effect_col not in df.columns:
            return False, f"Effect column '{self.effect_col}' not found"

        if self.var_col not in df.columns:
            return False, f"Variance column '{self.var_col}' not found"

        if 'year' not in df.columns:
            return False, "'year' column not found - required for cumulative analysis"

        # Check for minimum observations
        n_valid = df[['year', self.effect_col, self.var_col]].notna().all(axis=1).sum()
        if n_valid < 3:
            return False, f"Need at least 3 valid observations, found {n_valid}"

        return True, None

    # -------------------------------------------------------------------------
    # PROPERTY ACCESSORS
    # -------------------------------------------------------------------------

    @property
    def analysis_data(self) -> Optional[pd.DataFrame]:
        """Get analysis dataset"""
        if 'analysis_data' in self._config:
            return self._config['analysis_data'].copy()
        return None

    @property
    def effect_col(self) -> str:
        """Get effect size column name"""
        return self._config.get('effect_col', 'hedges_g')

    @property
    def var_col(self) -> str:
        """Get variance column name"""
        return self._config.get('var_col', 'Vg')

    @property
    def es_config(self) -> Dict[str, Any]:
        """Get effect size configuration"""
        return self._config.get('es_config', {})

    @property
    def global_settings(self) -> Dict[str, Any]:
        """Get global settings (with defaults)"""
        return self._config.get('global_settings', {
            'alpha': 0.05,
            'dist_type': 'norm'
        })

    # -------------------------------------------------------------------------
    # DATA EXTRACTION METHODS
    # -------------------------------------------------------------------------

    def prepare_data(
        self,
        aggregation_method: str = 'study',
        raw_dataframe: Optional[pd.DataFrame] = None
    ) -> pd.DataFrame:
        """
        Prepare and clean data for cumulative analysis.

        Args:
            aggregation_method: 'study' or 'obs'
            df: DataFrame to prepare (uses analysis_data if None)

        Returns:
            Cleaned DataFrame

        Raises:
            ValueError: If data cannot be prepared
        """
        if raw_dataframe is None:
            raw_dataframe = self.analysis_data

        if raw_dataframe is None:
            raise ValueError("No data available for analysis")

        # Validate
        is_valid, error_msg = self.validate_data(raw_dataframe)
        if not is_valid:
            raise ValueError(error_msg)

        # Create working copy
        clean_df = raw_dataframe.copy()

        # Convert year to numeric
        clean_df['year'] = pd.to_numeric(clean_df['year'], errors='coerce')
        if aggregation_method == 'multilevel':
            # A study enters in one year: a row without a year takes its study's
            # year, so no effect size of an included study is lost.
            clean_df['year'] = clean_df['year'].fillna(clean_df.groupby('id')['year'].transform('min'))

        # Remove missing values
        clean_df = clean_df.dropna(subset=['year', self.effect_col, self.var_col]).copy()

        # Remove zero or negative variances
        clean_df = clean_df[clean_df[self.var_col] > 0].copy()

        # Aggregate to study level if requested
        if aggregation_method == 'study':
            clean_df = self._aggregate_to_study_level(clean_df)
        elif aggregation_method == 'multilevel':
            # All effect sizes are kept. Each row remembers its position in
            # the data the sampling VCV matrices were built from, so the
            # shared-control covariances are sliced correctly; a study enters
            # in the year of its earliest effect size.
            clean_df = attach_vcv_row_pos(clean_df, reference_df=raw_dataframe)
            clean_df['study_year'] = clean_df.groupby('id')['year'].transform('min')

        # Final check
        if len(clean_df) < 3:
            raise ValueError(f"Insufficient data after cleaning: {len(clean_df)} observations. Need at least 3.")

        return clean_df

    def _aggregate_to_study_level(self, input_df: pd.DataFrame) -> pd.DataFrame:
        """
        Aggregate effect sizes to study level using inverse-variance weights.
        Optimized with vectorized pandas operations.

        Args:
            input_df: DataFrame with multiple observations per study

        Returns:
            Aggregated DataFrame (one row per study)
        """
        # Work on a copy to avoid side effects
        df_copy = input_df.copy()

        # Calculate inverse-variance weights
        df_copy['weight'] = 1.0 / df_copy[self.var_col]
        df_copy['weighted_effect'] = df_copy[self.effect_col] * df_copy['weight']

        # Group by study ID
        grouped = df_copy.groupby('id')

        # Vectorized aggregation
        sum_weights = grouped['weight'].sum()
        sum_weighted_effects = grouped['weighted_effect'].sum()
        min_years = grouped['year'].min()

        # Reconstruct DataFrame
        # Note: sum_weights.index contains the 'id's
        agg_df = pd.DataFrame({
            'id': sum_weights.index,
            self.effect_col: sum_weighted_effects / sum_weights,
            self.var_col: 1.0 / sum_weights,
            'year': min_years
        }).reset_index(drop=True)

        return agg_df

    # -------------------------------------------------------------------------
    # RESULT PERSISTENCE METHODS
    # -------------------------------------------------------------------------

    def save_cumulative_results(self, result: CumulativeResult) -> None:
        """
        Save cumulative analysis results to ANALYSIS_CONFIG.

        Args:
            result: CumulativeResult object
        """
        import datetime

        self._config['cumulative_results'] = result.steps
        self._config['cumulative_metadata'] = {
            'timestamp': datetime.datetime.now(),
            'n_steps': result.n_steps,
            'year_range': result.year_range,
            'aggregation_method': result.aggregation_method,
            'sort_order': result.sort_order,
            'start_effect': result.start_step.pooled_effect,
            'end_effect': result.end_step.pooled_effect,
            'effect_stable': result.effect_stable,
            'precision_improved': result.precision_improved
        }

    def save_publication_text(self, text: str) -> None:
        """Save publication-ready text"""
        self._config['cumulative_text'] = text

    # -------------------------------------------------------------------------
    # UTILITY METHODS
    # -------------------------------------------------------------------------

    def summary_dict(self) -> Dict[str, Any]:
        """Get summary of current configuration"""
        df = self.analysis_data

        return {
            'effect_col': self.effect_col,
            'var_col': self.var_col,
            'has_year': 'year' in df.columns if df is not None else False,
            'n_observations': len(df) if df is not None else 0,
            'n_studies': df['id'].nunique() if df is not None and 'id' in df.columns else 0,
            'year_range': (df['year'].min(), df['year'].max()) if df is not None and 'year' in df.columns else None
        }



# =============================================================================
# Purpose: Pure statistical computation without UI dependencies
# Dependencies: Data Layer (CumulativeDataManager, CumulativeResult)
# Math validated: Iterative REML estimation for temporal evolution
# =============================================================================

import numpy as np
import pandas as pd
from scipy.stats import norm, t, chi2
from typing import Dict, Any, Optional, Tuple, List
import warnings


# =============================================================================
# REML TAU² ESTIMATOR
# =============================================================================


class REMLTauEstimator:
    """
    REML estimator for τ² in cumulative analysis: the two-level REML fit of
    the shared engine (tau2_univariate, Cell 1b), as metafor's rma().
    """
    @staticmethod
    def estimate(
        effect_sizes: np.ndarray,
        variances: np.ndarray,
        max_iter: int = 500,
        tol: float = 1e-8
    ) -> float:
        """
        Estimate τ² by REML.
        Args:
            effect_sizes: Effect sizes
            variances: Variances
            max_iter, tol: unused (kept for compatibility)
        Returns:
            τ² estimate
        """
        if len(effect_sizes) < 2:
            return 0.0
        return max(0.0, tau2_univariate(effect_sizes, variances, method='REML'))


# =============================================================================
# HETEROGENEITY CALCULATOR
# =============================================================================

class CumulativeHeterogeneityEngine:
    """
    Calculate heterogeneity statistics for each cumulative step.
    """

    @staticmethod
    def calculate_Q_and_I2(
        effect_sizes: np.ndarray,
        variances: np.ndarray,
        fixed_effect_mean: float
    ) -> Tuple[float, float]:
        """
        Calculate Q statistic and I².

        Args:
            y: Effect sizes
            v: Variances
            mu_fe: Fixed-effect pooled estimate

        Returns:
            Tuple of (Q, I2)
        """
        num_studies = len(effect_sizes)
        weights = 1.0 / variances
        Q_statistic = np.sum(weights * (effect_sizes - fixed_effect_mean)**2)
        I2_score = max(0, (Q_statistic - (num_studies - 1)) / Q_statistic * 100) if Q_statistic > 0 else 0.0

        return Q_statistic, I2_score


# =============================================================================
# CUMULATIVE STEP CALCULATOR
# =============================================================================

# CR2 inference at a cumulative step only when its Satterthwaite df is at
# least this (Tipton 2015: below 4 the CR2 test is unreliable).
CUMULATIVE_MIN_CR2_DF = 4.0


class CumulativeStepEngine:
    """
    Calculate meta-analysis for a single cumulative step.
    """

    def __init__(self, alpha: float = 0.05, dist_type: str = 'norm'):
        """
        Initialize engine.

        Args:
            alpha: Significance level
            dist_type: 'norm' or 't'
        """
        self.alpha = alpha
        self.dist_type = dist_type
        self.tau_estimator = REMLTauEstimator()
        self.het_engine = CumulativeHeterogeneityEngine()

    def calculate_step(
        self,
        cumulative_effects: np.ndarray,
        cumulative_variances: np.ndarray,
        step: int,
        year: int,
        study_id: str
    ) -> Dict[str, Any]:
        """
        Calculate meta-analysis for current cumulative step.

        MATH PRESERVED: Complete REML random-effects pooling

        Args:
            y: Effect sizes (cumulative to this point)
            v: Variances (cumulative to this point)
            step: Step number
            year: Year of study added
            study_id: ID of study added

        Returns:
            Dictionary with step results
        """
        num_studies = len(cumulative_effects)
        ci_pct = (1 - self.alpha) * 100

        # Step 1: Estimate τ²
        tau2 = self.tau_estimator.estimate(cumulative_effects, cumulative_variances)

        # Step 2: Random-effects pooling
        random_weights = 1.0 / (cumulative_variances + tau2)
        sum_random_weights = np.sum(random_weights)
        pooled_effect = np.sum(random_weights * cumulative_effects) / sum_random_weights
        pooled_se = np.sqrt(1.0 / sum_random_weights)

        # Step 3: Heterogeneity statistics
        fixed_weights = 1.0 / cumulative_variances
        fixed_effect_mean = np.sum(fixed_weights * cumulative_effects) / np.sum(fixed_weights)
        Q, I2 = self.het_engine.calculate_Q_and_I2(cumulative_effects, cumulative_variances, fixed_effect_mean)

        # Step 4: Critical value and CI
        q_val = 1 - (self.alpha / 2)

        if self.dist_type == 't':
            df_cum = max(1, num_studies - 1)
            crit_val = t.ppf(q_val, df_cum)
        else:
            crit_val = norm.ppf(q_val)

        ci_lower = pooled_effect - crit_val * pooled_se
        ci_upper = pooled_effect + crit_val * pooled_se

        return {
            'step': step,
            'year': int(year) if pd.notna(year) else 0,
            'study_added': study_id,
            'k_studies': num_studies,
            'pooled_effect': pooled_effect,
            'se': pooled_se,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper,
            'tau2': tau2,
            'I2': I2,
            'ci_pct': ci_pct
        }


    def calculate_step_multilevel(
        self,
        rows: pd.DataFrame,
        effect_col: str,
        var_col: str,
        vcv_dict: Dict[Any, np.ndarray],
        use_cr2: bool,
        step: int,
        year: int,
        study_id: str
    ) -> Dict[str, Any]:
        """
        Meta-analysis of all effect sizes of the studies included so far,
        with the model of the overall analysis (Step 7): effect sizes nested
        in studies (three-level REML, Cell 1b engine; two-level when no
        study yet has more than one effect size), the full sampling VCV
        (shared controls), and CR2 cluster-robust SE with Satterthwaite df
        when CR2 is on in Step 7 (otherwise model-based SE, df = studies - 1).
        In the first steps, with very few studies, the CR2 SE can collapse
        (Satterthwaite df near 1); CR2 is therefore used only when its df is
        at least 4 (Tipton 2015), and those steps fall back to model-based SE.
        """
        ci_pct = (1 - self.alpha) * 100
        grouped = rows.groupby('id', sort=False)
        y_blocks = [g[effect_col].to_numpy(dtype=np.float64) for _, g in grouped]
        V_blocks = [cluster_vcv(vcv_dict, sid, g, var_col)[0] for sid, g in grouped]
        V_diag = [np.diag(g[var_col].to_numpy(dtype=np.float64)) for _, g in grouped]
        n_clusters = len(y_blocks)

        fit = fit_blocks(y_blocks, V_blocks, V_fallback_blocks=V_diag,
                         cluster_ids=list(grouped.groups.keys()))
        mu = float(fit.betas[0])
        se = float(np.sqrt(fit.cov_beta[0, 0]))
        df_inf = float(max(1, n_clusters - 1))
        used_cr2 = False
        if use_cr2 and n_clusters > 1:
            try:
                var_r, df_r = fit_cr2(fit)
                se_r, df_r0 = float(np.sqrt(var_r[0, 0])), float(df_r[0])
                if np.isfinite(se_r) and se_r > 0 and np.isfinite(df_r0) and df_r0 >= CUMULATIVE_MIN_CR2_DF:
                    se, df_inf, used_cr2 = se_r, df_r0, True
            except Exception as e:
                warnings.warn(f"Cumulative step {step}: CR2 failed ({e}); model-based SE used.")

        q_val = 1 - (self.alpha / 2)
        crit_val = t.ppf(q_val, df_inf) if self.dist_type == 't' else norm.ppf(q_val)

        # Multilevel I² (as Step 7): (tau² + sigma²) / (tau² + sigma² + typical v),
        # with the Higgins & Thompson typical within-study sampling variance.
        all_v = np.concatenate([np.diag(V) for V in V_blocks])
        w = 1.0 / all_v
        denom = w.sum() ** 2 - (w ** 2).sum()
        v_typ = (len(all_v) - 1) * w.sum() / denom if denom > 0 else float(np.mean(all_v))
        tau2, sigma2 = float(fit.tau2), float(fit.sigma2)
        total = tau2 + sigma2 + v_typ
        I2 = (tau2 + sigma2) / total * 100 if total > 0 else 0.0

        return {
            'step': step,
            'year': int(year) if pd.notna(year) else 0,
            'study_added': study_id,
            'k_studies': n_clusters,
            'k_effects': int(len(rows)),
            'pooled_effect': mu,
            'se': se,
            'ci_lower': mu - crit_val * se,
            'ci_upper': mu + crit_val * se,
            'df': df_inf if self.dist_type == 't' else np.nan,
            'tau2': tau2,
            'sigma2': sigma2,
            'I2': I2,
            'ci_pct': ci_pct,
            'inference': 'CR2' if used_cr2 else 'model-based',
        }


# =============================================================================
# STABILITY ANALYZER
# =============================================================================

class StabilityAnalyzer:
    """
    Analyze whether effect size has stabilized over time.
    """

    @staticmethod
    def assess_stability(steps_df: pd.DataFrame) -> bool:
        """
        Assess if effect size has stabilized in recent steps.

        Logic: Check if effect size changed by <10% in last 5 steps

        Args:
            steps_df: DataFrame with cumulative steps

        Returns:
            True if effect appears stable
        """
        if len(steps_df) < 5:
            return False

        recent_steps = steps_df.tail(5)
        effects = recent_steps['pooled_effect'].values

        # Calculate percentage change from first to last of recent steps
        if effects[0] == 0:
            return False

        pct_change = abs((effects[-1] - effects[0]) / effects[0]) * 100

        return pct_change < 10

    @staticmethod
    def assess_precision_improvement(
        start_step: 'CumulativeStep',
        end_step: 'CumulativeStep'
    ) -> bool:
        """
        Assess if precision improved (CI narrowed).

        Args:
            start_step: First cumulative step
            end_step: Last cumulative step

        Returns:
            True if CI width decreased
        """
        ci_width_start = start_step.ci_upper - start_step.ci_lower
        ci_width_end = end_step.ci_upper - end_step.ci_lower

        return ci_width_end < ci_width_start


# =============================================================================
# CUMULATIVE META-ANALYSIS ORCHESTRATOR
# =============================================================================

class CumulativeEngine:
    """
    High-level orchestrator for cumulative meta-analysis.
    Coordinates iterative calculations across all time steps.
    """

    def __init__(self, data_manager: 'CumulativeDataManager'):
        """
        Initialize engine with data manager.

        Args:
            data_manager: CumulativeDataManager instance
        """
        self.data_manager = data_manager

        # Get settings from global config
        settings = data_manager.global_settings
        self.use_cr2 = bool(settings.get('use_kh', True))
        self.step_engine = CumulativeStepEngine(
            alpha=settings.get('alpha', 0.05),
            dist_type=settings.get('dist_type', 'norm')
        )
        self.stability_analyzer = StabilityAnalyzer()

    # -------------------------------------------------------------------------
    # MAIN EXECUTION METHOD
    # -------------------------------------------------------------------------

    def run_analysis(
        self,
        sort_order: str = 'ascending',
        aggregation_method: str = 'multilevel',
        progress_callback: Optional[callable] = None
    ) -> Optional['CumulativeResult']:
        """
        Execute complete cumulative meta-analysis.

        WORKFLOW:
        1. Prepare and sort data
        2. Iterate through time points
        3. Calculate meta-analysis at each step
        4. Assess stability

        Args:
            sort_order: 'ascending' or 'descending'
            aggregation_method: 'study' or 'obs'
            progress_callback: Optional progress updates

        Returns:
            CumulativeResult object or None if analysis fails
        """
        # Step 1: Prepare data
        if progress_callback:
            progress_callback("📊 Preparing data...")

        try:
            df = self.data_manager.prepare_data(aggregation_method)
        except ValueError as e:
            if progress_callback:
                progress_callback(f"❌ Data preparation failed: {str(e)}")
            return None

        # Step 2: Sort by year
        if progress_callback:
            progress_callback(f"📅 Sorting by year ({sort_order})...")

        # Stable sort: studies sharing a year keep their order in the data,
        # so the intermediate steps are reproducible.
        df_sorted = df.sort_values(
            'year',
            ascending=(sort_order == 'ascending'),
            kind='mergesort'
        ).reset_index(drop=True)

        # Step 3: Iterative calculation
        if progress_callback:
            progress_callback("🔄 Running iterative meta-analysis...")

        results = []
        n_steps = len(df_sorted)

        if aggregation_method == 'multilevel':
            # Studies (not rows) are added one at a time, ordered by the year
            # of their earliest effect size; tied years keep data order.
            study_order = (df.drop_duplicates('id')
                             .sort_values('study_year', ascending=(sort_order == 'ascending'),
                                          kind='mergesort')[['id', 'study_year']])
            vcv_dict = self.data_manager._config.get('vcv_matrices', {}) or {}
            ids = study_order['id'].tolist()
            years = study_order['study_year'].tolist()
            for i in range(2, len(ids) + 1):
                rows = df[df['id'].isin(ids[:i])]
                results.append(self.step_engine.calculate_step_multilevel(
                    rows, self.data_manager.effect_col, self.data_manager.var_col,
                    vcv_dict, self.use_cr2, i, years[i - 1], ids[i - 1]))
            n_steps = 0      # skip the row-wise loop below

        # Need at least 2 studies to start
        for i in range(2, n_steps + 1):
            current_slice = df_sorted.iloc[:i]

            y = current_slice[self.data_manager.effect_col].values
            v = current_slice[self.data_manager.var_col].values
            year = current_slice.iloc[-1]['year']
            study_id = current_slice.iloc[-1]['id'] if 'id' in current_slice.columns else f"Study_{i}"

            step_result = self.step_engine.calculate_step(
                y, v, i, year, study_id
            )

            results.append(step_result)

        if not results:
            if progress_callback:
                progress_callback("❌ No results generated")
            return None

        # Step 4: Create results DataFrame
        steps_df = pd.DataFrame(results)

        # Step 5: Extract start and end steps
        start_row = steps_df.iloc[0]
        end_row = steps_df.iloc[-1]

        start_step = CumulativeStep(
            step=int(start_row['step']),
            year=int(start_row['year']),
            study_added=start_row['study_added'],
            k_studies=int(start_row['k_studies']),
            pooled_effect=start_row['pooled_effect'],
            se=start_row['se'],
            ci_lower=start_row['ci_lower'],
            ci_upper=start_row['ci_upper'],
            tau2=start_row['tau2'],
            I2=start_row['I2'],
            ci_pct=start_row['ci_pct']
        )

        end_step = CumulativeStep(
            step=int(end_row['step']),
            year=int(end_row['year']),
            study_added=end_row['study_added'],
            k_studies=int(end_row['k_studies']),
            pooled_effect=end_row['pooled_effect'],
            se=end_row['se'],
            ci_lower=end_row['ci_lower'],
            ci_upper=end_row['ci_upper'],
            tau2=end_row['tau2'],
            I2=end_row['I2'],
            ci_pct=end_row['ci_pct']
        )

        # Step 6: Assess stability and precision
        effect_stable = self.stability_analyzer.assess_stability(steps_df)
        precision_improved = self.stability_analyzer.assess_precision_improvement(
            start_step, end_step
        )

        # Step 7: Build result object
        result = CumulativeResult(
            steps=steps_df,
            start_step=start_step,
            end_step=end_step,
            n_steps=len(results),
            year_range=(int(steps_df['year'].min()), int(steps_df['year'].max())),
            aggregation_method=aggregation_method,
            sort_order=sort_order,
            effect_stable=effect_stable,
            precision_improved=precision_improved
        )

        if progress_callback:
            progress_callback(
                f"✅ Complete: {result.n_steps} steps, "
                f"μ: {start_step.pooled_effect:.3f} → {end_step.pooled_effect:.3f}"
            )

        return result

# =============================================================================
#  PRESENTATION LAYER - CUMULATIVE META-ANALYSIS
# Purpose: Pure UI rendering without business logic
# Dependencies: Data & Analysis Layers
# =============================================================================

import pandas as pd
import numpy as np
from typing import Dict, Any, Optional
import datetime
import ipywidgets as widgets
from IPython.display import display, HTML
import sys


# =============================================================================
# HTML TEMPLATE GENERATORS
# =============================================================================

class CumulativeHTMLTemplates:
    """
    Static HTML template generators for cumulative meta-analysis visualizations.
    All methods are pure functions returning HTML strings.
    """

    @staticmethod
    def comparison_cards(
        start_step: 'CumulativeStep',
        end_step: 'CumulativeStep',
        es_config: Dict[str, Any] = None
    ) -> str:
        """Generate start vs end comparison cards"""
        _es_type = (es_config or {}).get('type', '')
        if _es_type in ('log_or', 'log_rr', 'lnRR'):
            _start_val = f"{np.exp(start_step.pooled_effect):.3f}"
            _end_val = f"{np.exp(end_step.pooled_effect):.3f}"
        else:
            _start_val = f"{start_step.pooled_effect:.3f}"
            _end_val = f"{end_step.pooled_effect:.3f}"
        return f"""
        <div style='padding:10px;'>
            <h3 style='color:#2E86AB; margin-top:0;'>Analysis Complete</h3>
            <div style='display:flex; gap:20px;'>
                <div style='flex:1; background:#f8f9fa; padding:15px; border-radius:5px;
                            border-left:4px solid #6c757d;'>
                    <h4 style='margin:0; color:#666;'>Start (k={start_step.k_studies})</h4>
                    <div style='font-size:18px; font-weight:bold; margin-top:5px;'>
                        {_start_val}
                    </div>
                    <div style='font-size:12px;'>
                        {start_step.ci_pct:.0f}% CI [{start_step.ci_lower:.3f}, {start_step.ci_upper:.3f}]
                    </div>
                    <div style='font-size:12px; color:#666;'>Year: {start_step.year}</div>
                </div>
                <div style='flex:0.2; display:flex; align-items:center; justify-content:center;
                            font-size:24px; color:#ccc;'>
                    ➔
                </div>
                <div style='flex:1; background:#e7f3ff; padding:15px; border-radius:5px;
                            border-left:4px solid #007bff;'>
                    <h4 style='margin:0; color:#0056b3;'>End (k={end_step.k_studies})</h4>
                    <div style='font-size:18px; font-weight:bold; margin-top:5px; color:#0056b3;'>
                        {_end_val}
                    </div>
                    <div style='font-size:12px; color:#0056b3;'>
                        {end_step.ci_pct:.0f}% CI [{end_step.ci_lower:.3f}, {end_step.ci_upper:.3f}]
                    </div>
                    <div style='font-size:12px; color:#0056b3;'>Year: {end_step.year}</div>
                </div>
            </div>
            <p style='margin-top:15px; color:#555;'>
                <b>Method:</b> Iterative REML (Consistent with Overall Analysis).
            </p>
        </div>
        """

    @staticmethod
    def stability_badges(
        effect_stable: bool,
        precision_improved: bool
    ) -> str:
        """Generate stability assessment badges"""
        stable_color = "#28a745" if effect_stable else "#ffc107"
        stable_text = "✓ Stabilized" if effect_stable else "⚠ Still Evolving"
        stable_icon = "✓" if effect_stable else "⚠"

        precision_color = "#28a745" if precision_improved else "#dc3545"
        precision_text = "✓ Improved" if precision_improved else "✗ Wider CI"
        precision_icon = "✓" if precision_improved else "✗"

        return f"""
        <div style='display: flex; gap: 15px; margin: 15px 0;'>
            <div style='background-color: {stable_color}; color: white; padding: 10px 20px;
                        border-radius: 5px; font-weight: bold;'>
                {stable_icon} Effect: {stable_text}
            </div>
            <div style='background-color: {precision_color}; color: white; padding: 10px 20px;
                        border-radius: 5px; font-weight: bold;'>
                {precision_icon} Precision: {precision_text}
            </div>
        </div>
        """

    @staticmethod
    def method_note(aggregation_method: str) -> str:
        """Generate method description note"""
        if aggregation_method == 'multilevel':
            desc = ("Studies added one at a time; at each step the model of Step 7 "
                    "(effect sizes nested in studies, shared-control covariances, CR2 when on) "
                    "is fitted to all their effect sizes")
        elif aggregation_method == 'study':
            desc = ("Effect sizes averaged within studies (fixed-effect weights) and pooled with a "
                    "two-level model; ignores within-study heterogeneity, so intervals can be too narrow")
        else:
            desc = "Each observation treated independently (no aggregation)"

        return f"""
        <div style='background-color: #f8f9fa; padding: 10px; border-left: 3px solid #007bff;
                    margin: 10px 0; font-size: 0.9em;'>
            <b>Data Structure:</b> {desc}
        </div>
        """


# =============================================================================
# PUBLICATION TEXT GENERATORS
# =============================================================================

class CumulativePublicationTextGenerator:
    """
    Generates publication-ready text for manuscripts.
    """

    def __init__(self):
        pass

    @staticmethod
    def _any_cr2(result: 'CumulativeResult') -> bool:
        steps = result.steps
        return 'inference' in steps.columns and bool((steps['inference'] == 'CR2').any())

    def generate_methods_section(
        self,
        result: 'CumulativeResult',
        es_config: Dict[str, Any]
    ) -> str:
        """
        Generate Materials and Methods section.

        PRESERVED: Original citation logic
        """
        es_label = es_config.get('effect_label', 'Effect Size')
        n_steps = result.n_steps
        start_year, end_year = result.year_range
        ci_pct = result.start_step.ci_pct

        # Citation database
        db = {
            'lau': "Lau, J., Antman, E. M., Jimenez-Silva, J., Kupelnick, B., Mosteller, F., & Chalmers, T. C. (1992). Cumulative meta-analysis of therapeutic trials for myocardial infarction. <i>New England Journal of Medicine</i>, 327(4), 248-254.",
            'reml': "Viechtbauer, W. (2005). Bias and efficiency of meta-analytic variance estimators in the random-effects model. <i>Journal of Educational and Behavioral Statistics</i>, 30(3), 261-293.",
            'borenstein': "Borenstein, M., Hedges, L. V., Higgins, J. P., & Rothstein, H. R. (2009). <i>Introduction to Meta-Analysis</i>. Chichester, UK: John Wiley & Sons.",
            'tool': COMETA_CITATION
        }

        # Aggregation description
        if result.aggregation_method == 'multilevel':
            agg_desc = ("Studies were added one at a time in order of their earliest publication year. "
                        "At each step, all effect sizes of the studies included so far were analysed with "
                        "the multilevel random-effects model of the overall analysis (effect sizes nested "
                        "within studies, with the sampling variance-covariance matrix including shared "
                        "control groups)"
                        + (", with cluster-robust (CR2) standard errors and Satterthwaite degrees of freedom "
                           "at steps where these had at least 4 degrees of freedom [5], and "
                           "model-based standard errors with studies − 1 degrees of freedom at earlier steps."
                           if self._any_cr2(result) else "."))
        elif result.aggregation_method == 'study':
            agg_desc = "Prior to the cumulative analysis, effect sizes were aggregated at the study level to ensure independence. A fixed-effect weighted mean was calculated for studies reporting multiple outcomes."
        else:
            agg_desc = "Effect sizes were treated as independent observations for the cumulative analysis."

        html = f"""
        <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt;
                    line-height: 1.8; padding: 20px; background-color: #ffffff;
                    border: 1px solid #eee; margin-bottom: 20px;'>

        <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;
                   margin-top: 0;'>Materials and Methods</h3>

        <p style='text-align: justify;'>
        <b>Cumulative Meta-Analysis.</b> To evaluate the accumulation of evidence over time and
        assess the sufficiency and stability of the pooled effect size, we performed a cumulative
        meta-analysis [1]. Studies were sorted chronologically by publication year (Range: {start_year}–{end_year}),
        and a new meta-analysis was performed each time a new study was added to the pool.
        </p>

        <p style='text-align: justify;'>
        <b>Model Specification.</b> {agg_desc} For each of the {n_steps} cumulative steps, a
        random-effects model was fitted using the Restricted Maximum Likelihood (REML) estimator [2]
        to calculate the updated pooled effect and {ci_pct:.0f}% confidence interval. This approach
        allows for the visualization of how the precision of the estimate evolves and whether the
        effect size stabilizes as sample size increases [3]. All analyses were performed using the
        Co-Meta toolkit [4].
        </p>

        <h4 style='color: #34495e; margin-top: 20px; font-size: 11pt;'>References</h4>
        <ol style='font-size: 10pt; color: #555;'>
            <li>{db['lau']}</li>
            <li>{db['reml']}</li>
            <li>{db['borenstein']}</li>
            <li>{db['tool']}</li>
            {"<li>Tipton, E. (2015). Small sample adjustments for robust variance estimation with meta-regression. <i>Psychological Methods</i>, 20(3), 375-393. https://doi.org/10.1037/met0000011</li>" if result.aggregation_method == 'multilevel' and self._any_cr2(result) else ""}
        </ol>
        </div>
        """

        return html

    def generate_results_section(
        self,
        result: 'CumulativeResult',
        es_config: Dict[str, Any]
    ) -> str:
        """
        Generate complete Results section with interpretation.

        PRESERVED: Original publication text logic with temporal interpretation
        """
        start = result.start_step
        end = result.end_step

        es_label = es_config.get('effect_label', 'Effect Size')
        ci_pct = start.ci_pct

        # Calculate trends
        ci_width_start = start.ci_upper - start.ci_lower
        ci_width_end = end.ci_upper - end.ci_lower
        precision_change = (ci_width_end - ci_width_start) / ci_width_start * 100
        prec_text = "improved" if precision_change < 0 else "decreased"

        # Effect stability
        abs_change = abs(end.pooled_effect - start.pooled_effect)
        pct_change = (abs_change / abs(start.pooled_effect) * 100) if start.pooled_effect != 0 else 0

        if pct_change < 10:
            trend_desc = "remained remarkably stable"
        elif pct_change < 30:
            trend_desc = "remained relatively consistent"
        else:
            direction = "increased" if abs(end.pooled_effect) > abs(start.pooled_effect) else "decreased"
            trend_desc = f"gradually {direction} in magnitude"

        # Heterogeneity trend
        het_trend = "increased" if end.I2 > start.I2 + 5 else "decreased" if end.I2 < start.I2 - 5 else "remained stable"

        # Back-transform for log-scale effect sizes
        _es_type = es_config.get('type', '')
        if _es_type in ('log_or', 'log_rr', 'lnRR'):
            _start_display = f"{np.exp(start.pooled_effect):.3f}"
            _end_display = f"{np.exp(end.pooled_effect):.3f}"
            _es_label_short = 'OR' if _es_type == 'log_or' else 'RR'
        else:
            _start_display = f"{start.pooled_effect:.3f}"
            _end_display = f"{end.pooled_effect:.3f}"
            _es_label_short = es_config.get('effect_label_short', 'ES')

        # Build HTML
        html = f"""
        <div style='font-family: "Times New Roman", Times, serif; font-size: 12pt;
                    line-height: 1.8; padding: 20px; background-color: #ffffff;'>

        <h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;'>
            Cumulative Meta-Analysis Results
        </h3>

        <p style='text-align: justify;'>
        A cumulative meta-analysis was conducted to examine the temporal evolution of the pooled
        effect size and assess the sufficiency of the evidence. Studies were added to the analysis
        in chronological order based on publication year.
        </p>

        <h4 style='color: #34495e; margin-top: 20px;'>Evolution of Effect Size</h4>
        <p style='text-align: justify;'>
        The analysis began with the earliest available studies (k={start.k_studies}, Year: {start.year}),
        yielding an initial pooled {_es_label_short} of {_start_display} ({ci_pct:.0f}% CI [{start.ci_lower:.3f},
        {start.ci_upper:.3f}]). As evidence accumulated over the subsequent years (Total k={end.k_studies},
        Range: {result.year_range[0]}–{result.year_range[1]}), the pooled estimate <b>{trend_desc}</b>,
        resulting in a final {_es_label_short} of <b>{_end_display}</b> ({ci_pct:.0f}% CI [{end.ci_lower:.3f},
        {end.ci_upper:.3f}]).
        </p>

        <h4 style='color: #34495e; margin-top: 20px;'>Precision and Heterogeneity</h4>
        <p style='text-align: justify;'>
        The precision of the estimate <b>{prec_text}</b> over time, as indicated by a
        {abs(precision_change):.1f}% {prec_text[:-1] if prec_text.endswith('ed') else prec_text}
        in the width of the {ci_pct:.0f}% confidence interval. Meanwhile, the
        {'total (between- plus within-study)' if result.aggregation_method == 'multilevel' else 'between-study'}
        heterogeneity (<i>I</i>²) {het_trend} from {start.I2:.1f}% in the initial wave to
        {end.I2:.1f}% in the final analysis.
        </p>
        """

        # Stability interpretation
        if result.effect_stable:
            html += """
            <h4 style='color: #34495e; margin-top: 20px;'>Evidence Sufficiency</h4>
            <p style='text-align: justify;'>
            The pooled effect size demonstrated <b>stability in recent years</b>, with minimal change
            (<10%) over the last five studies added. This suggests that the current body of evidence
            is likely sufficient to provide a reliable estimate of the true effect size, and that
            additional studies are unlikely to substantially alter this conclusion.
            </p>
            """
        else:
            html += """
            <h4 style='color: #34495e; margin-top: 20px;'>Evidence Sufficiency</h4>
            <p style='text-align: justify;'>
            The pooled effect size continues to <b>evolve in recent years</b>, suggesting that the
            evidence base may not yet be sufficient to provide a definitive estimate. Additional
            research may be warranted to clarify the magnitude and direction of the effect.
            </p>
            """

        # Guidance
        html += """
        <div style='background-color: #ecf0f1; padding: 15px; border-left: 4px solid #3498db;
                    margin-top: 20px;'>
            <h4 style='margin-top: 0; color: #2c3e50;'>Interpretation Guidance:</h4>
            <ul style='margin-bottom: 0;'>
                <li><b>Stability:</b> If the line flattens out in recent years, the finding is robust.
                    If it is still moving up or down, more research is needed.</li>
                <li><b>Sufficiency:</b> Narrowing confidence intervals suggest that we are converging
                    on the "true" effect size.</li>
                <li><b>Heterogeneity:</b> Increasing I² over time may indicate that newer studies differ
                    systematically from older ones (consider moderator analysis).</li>
            </ul>
        </div>

        <div style='background-color: #fff3cd; padding: 10px; border-left: 4px solid #ffc107;
                    margin-top: 15px;'>
            <p style='margin: 0;'><b>💡 Tip:</b> Select all text (Ctrl+A / Cmd+A), copy (Ctrl+C / Cmd+C),
            and paste into your word processor.</p>
        </div>

        </div>
        """

        return html


print("✅ Cumulative View Layer (Part 1) Loaded Successfully")

#@title 📈 23. Cumulative Meta-Analysis: PRESENTATION LAYER (View) - PART 2
# =============================================================================
# VIEW COMPONENTS (Tab Renderers)
# =============================================================================

class CumulativeResultsView:
    """
    Manages all UI rendering for cumulative meta-analysis.
    Contains zero business logic - only presentation code.
    """

    def __init__(self):
        """Initialize view with display settings"""
        self.templates = CumulativeHTMLTemplates()
        self.text_gen = CumulativePublicationTextGenerator()

        # Create tab widgets
        self.tab_results = widgets.Output()
        self.tab_data = widgets.Output()
        self.tab_publication = widgets.Output()
        self.tab_export = widgets.Output()

        self.tabs = widgets.Tab(children=[
            self.tab_results,
            self.tab_data,
            self.tab_publication,
            self.tab_export
        ])

        self.tabs.set_title(0, '📊 Analysis Summary')
        self.tabs.set_title(1, '📋 Step-by-Step Data')
        self.tabs.set_title(2, '📝 Publication Text')
        self.tabs.set_title(3, '💾 Export')

    # -------------------------------------------------------------------------
    # TAB 1: RESULTS SUMMARY
    # -------------------------------------------------------------------------

    def render_results_tab(self, result: 'CumulativeResult', es_config: Dict[str, Any]) -> None:
        """Render results summary tab"""

        with self.tab_results:
            self.tab_results.clear_output()

            # Comparison cards
            comparison_html = self.templates.comparison_cards(
                result.start_step,
                result.end_step,
                es_config=es_config
            )
            display(HTML(comparison_html))

            # Stability badges
            stability_html = self.templates.stability_badges(
                result.effect_stable,
                result.precision_improved
            )
            display(HTML(stability_html))

            # Method note
            method_html = self.templates.method_note(result.aggregation_method)
            display(HTML(method_html))

    # -------------------------------------------------------------------------
    # TAB 2: STEP-BY-STEP DATA
    # -------------------------------------------------------------------------

    def render_data_tab(self, result: 'CumulativeResult') -> None:
        """Render step-by-step data table"""

        with self.tab_data:
            self.tab_data.clear_output()

            display(HTML("<h4 style='margin-top:0;'>Step-by-Step Evolution</h4>"))

            # Prepare display DataFrame
            disp_df = result.steps.copy()
            cols = ['step', 'year', 'study_added', 'pooled_effect', 'ci_lower',
                    'ci_upper', 'I2', 'tau2']

            # Format and display with styling
            try:
                styled_df = disp_df[cols].style.format({
                    'pooled_effect': '{:.3f}',
                    'ci_lower': '{:.3f}',
                    'ci_upper': '{:.3f}',
                    'I2': '{:.1f}%',
                    'tau2': '{:.4f}'
                }).background_gradient(subset=['pooled_effect'], cmap='Blues')

                display(styled_df)
            except:
                # Fallback without styling
                display(disp_df[cols])

            # Download tip
            tip_html = """
            <div style='background-color: #e7f3ff; padding: 10px; margin-top: 15px;
                        border-left: 3px solid #007bff;'>
                <b>💡 Tip:</b> Use the Export tab to download this data as an Excel file.
            </div>
            """
            display(HTML(tip_html))

    # -------------------------------------------------------------------------
    # TAB 3: PUBLICATION TEXT
    # -------------------------------------------------------------------------

    def render_publication_tab(
        self,
        result: 'CumulativeResult',
        es_config: Dict[str, Any]
    ) -> str:
        """
        Render publication text tab.

        Returns:
            Combined HTML text for saving
        """
        with self.tab_publication:
            self.tab_publication.clear_output()

            # Generate both sections
            methods_html = self.text_gen.generate_methods_section(result, es_config)
            results_html = self.text_gen.generate_results_section(result, es_config)

            # Display
            display(HTML(methods_html))
            display(HTML(results_html))

            # Return combined for saving
            return methods_html + "<br><hr><br>" + results_html

    # -------------------------------------------------------------------------
    # TAB 4: EXPORT
    # -------------------------------------------------------------------------

    def render_export_tab(self, export_callback: callable) -> None:
        """Render export tab with download button"""

        with self.tab_export:
            self.tab_export.clear_output()

            display(HTML("<h3 style='color: #2E86AB;'>💾 Download Audit Report</h3>"))
            display(HTML("<p>Download the cumulative analysis results including step-by-step evolution data.</p>"))

            btn_export = widgets.Button(
                description="📥 Download Cumulative Report",
                button_style='info',
                icon='file-excel',
                layout=widgets.Layout(width='300px', height='40px')
            )

            btn_export.on_click(export_callback)
            display(btn_export)

    # -------------------------------------------------------------------------
    # ERROR DISPLAY
    # -------------------------------------------------------------------------

    def render_error(self, message: str, details: Optional[str] = None) -> None:
        """Render error message in results tab"""
        with self.tab_results:
            self.tab_results.clear_output()
            error_html = f"<div style='color: red; background-color: #f8d7da; padding: 15px; border-radius: 5px;'>❌ {message}</div>"
            if details:
                error_html += f"<pre style='margin-top: 10px;'>{details}</pre>"
            display(HTML(error_html))


print("✅ Cumulative View Layer (Part 2) Loaded Successfully")

#@title 📈 23. Cumulative Meta-Analysis: CONTROLLER LAYER (Orchestration)
# =============================================================================
# CELL COMPONENT 4/4: CONTROLLER LAYER - CUMULATIVE META-ANALYSIS
# Purpose: Orchestrates data, analysis, and view components
# Dependencies: All previous layers (Data, Analysis, View)
# =============================================================================

import traceback
from typing import Optional, Dict, Any
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets


# =============================================================================
# MAIN CONTROLLER
# =============================================================================

class CumulativeController:
    """
    Master controller that orchestrates the entire cumulative meta-analysis workflow.
    Coordinates data management, statistical computation, and UI rendering.
    """

    def __init__(self, analysis_config: Dict[str, Any]):
        """
        Initialize controller with ANALYSIS_CONFIG.

        Args:
            analysis_config: Global ANALYSIS_CONFIG dictionary
        """
        self.analysis_config = analysis_config

        # Initialize components
        try:
            self.data_manager = CumulativeDataManager(analysis_config)
            self.engine = CumulativeEngine(self.data_manager)
            self.view = CumulativeResultsView()

            self._initialization_error = None

        except Exception as e:
            # If initialization fails, create minimal view to show error
            self.view = CumulativeResultsView()
            self.data_manager = None
            self.engine = None
            self._initialization_error = e

        # Create settings widgets
        self._create_settings_widgets()

    # -------------------------------------------------------------------------
    # SETTINGS WIDGETS CREATION
    # -------------------------------------------------------------------------

    def _create_settings_widgets(self) -> None:
        """Create all settings widgets"""

        self.sort_order_widget = widgets.Dropdown(
            options=[('Chronological (Oldest → Newest)', 'ascending'),
                     ('Reverse Chronological (Newest → Oldest)', 'descending')],
            value='ascending',
            description='Sort Order:',
            style={'description_width': '120px'},
            layout=widgets.Layout(width='450px')
        )

        self.agg_method_widget = widgets.Dropdown(
            options=[('By Study, multilevel model (Recommended)', 'multilevel'),
                     ('Study means, two-level (approximate)', 'study'),
                     ('By Observation (Ignore nesting)', 'obs')],
            value='multilevel',
            description='Aggregation:',
            style={'description_width': '120px'},
            layout=widgets.Layout(width='450px')
        )

        self.run_button = widgets.Button(
            description='▶ Run Cumulative Analysis (REML)',
            button_style='success',
            layout=widgets.Layout(width='450px', height='50px'),
            style={'font_weight': 'bold'}
        )

        # Attach event handler
        self.run_button.on_click(self._handle_run_click)

    # -------------------------------------------------------------------------
    # UI CREATION
    # -------------------------------------------------------------------------

    def create_ui(self) -> widgets.VBox:
        """
        Create the user interface with settings and run button.

        Returns:
            VBox widget containing the UI
        """
        ui = widgets.VBox([
            widgets.HTML("<h3 style='color:#2E86AB;'>📈 Cumulative Meta-Analysis (Part 1: Calculation)</h3>"),
            widgets.HTML("<b>Settings:</b>"),
            self.sort_order_widget,
            self.agg_method_widget,
            widgets.HTML("<hr>"),
            self.run_button,
            widgets.HTML("<br>")
        ])

        return ui

    # -------------------------------------------------------------------------
    # MAIN EXECUTION METHOD
    # -------------------------------------------------------------------------

    def run_analysis(self) -> None:
        """
        Execute complete cumulative meta-analysis workflow.
        """
        # Clear all tabs
        for tab in [self.view.tab_results, self.view.tab_data,
                    self.view.tab_publication]:
            tab.clear_output()

        # Check for initialization errors
        if self._initialization_error:
            self._handle_initialization_error()
            return

        # Validate ANALYSIS_CONFIG
        if 'ANALYSIS_CONFIG' not in globals():
            self.view.render_error("ANALYSIS_CONFIG not found. Run Steps 2–6 (data and effect sizes) first.")
            return

        # Get settings from widgets
        sort_order = self.sort_order_widget.value
        aggregation_method = self.agg_method_widget.value

        # Progress callback
        def progress_callback(message: str):
            """Callback for progress updates"""
            with self.view.tab_results:
                clear_output(wait=True)
                print(message)

        try:
            # Execute cumulative engine
            result = self.engine.run_analysis(
                sort_order=sort_order,
                aggregation_method=aggregation_method,
                progress_callback=progress_callback
            )

            if result is None:
                self.view.render_error(
                    "Analysis failed",
                    "Unable to compute cumulative analysis. Check your data (need ≥3 observations with 'year' column)."
                )
                return

            # Render all tabs
            self._render_all_tabs(result)

            # Save results
            self._save_results(result)

        except ValueError as e:
            self.view.render_error("Data Error", str(e))
        except RuntimeError as e:
            self.view.render_error("Runtime Error", str(e))
        except Exception as e:
            self._handle_unexpected_error(e)

    # -------------------------------------------------------------------------
    # TAB RENDERING
    # -------------------------------------------------------------------------

    def _render_all_tabs(self, result: 'CumulativeResult') -> None:
        """
        Render all tabs with results.

        Args:
            result: CumulativeResult object
        """
        # Tab 1: Results Summary
        self.view.render_results_tab(result, self.data_manager.es_config)

        # Tab 2: Step-by-Step Data
        self.view.render_data_tab(result)

        # Tab 3: Publication Text
        combined_text = self.view.render_publication_tab(
            result,
            self.data_manager.es_config
        )

        # Save publication text
        self.data_manager.save_publication_text(combined_text)

        # Tab 4: Export
        self.view.render_export_tab(export_callback=self._handle_export)

    # -------------------------------------------------------------------------
    # DATA PERSISTENCE
    # -------------------------------------------------------------------------

    def _save_results(self, result: 'CumulativeResult') -> None:
        """
        Save results to ANALYSIS_CONFIG.

        Args:
            result: CumulativeResult object
        """
        self.data_manager.save_cumulative_results(result)

    # -------------------------------------------------------------------------
    # EVENT HANDLERS
    # -------------------------------------------------------------------------

    def _handle_run_click(self, button) -> None:
        """Handle run button click event"""
        self.run_analysis()

    def _handle_export(self, button) -> None:
        """Handle export button click"""
        try:
            # Call the external export function if it exists
            if 'export_analysis_report' in globals():
                export_analysis_report(
                    report_type='cumulative',
                    filename_prefix='Cumulative_Meta_Analysis'
                )
            else:
                with self.view.tab_export:
                    print("⚠️ Export function not found. Please run the export cell first.")
        except Exception as e:
            with self.view.tab_export:
                print(f"❌ Export failed: {str(e)}")
                traceback.print_exc()

    # -------------------------------------------------------------------------
    # ERROR HANDLING
    # -------------------------------------------------------------------------

    def _handle_initialization_error(self) -> None:
        """Handle errors during controller initialization"""
        error = self._initialization_error
        if error is None:
            return

        error_type = type(error).__name__
        error_msg = str(error)
        details = traceback.format_exc()

        self.view.render_error(f"{error_type}: {error_msg}", details)

    def _handle_unexpected_error(self, error: Exception) -> None:
        """Handle unexpected errors during analysis"""
        error_type = type(error).__name__
        error_msg = str(error)
        details = traceback.format_exc()

        self.view.render_error(f"{error_type}: {error_msg}", details)


# =============================================================================
# EXECUTION ENTRY POINT
# =============================================================================

def run_cumulative_meta_analysis():
    """
    Main entry point for cumulative meta-analysis.
    Call this function to display the UI and enable analysis.
    """
    try:
        # Check for ANALYSIS_CONFIG
        if 'ANALYSIS_CONFIG' not in globals():
            print("❌ ERROR: ANALYSIS_CONFIG not found")
            print("Please run previous analysis cells first:")
            print("  - Steps 2–6: Data loading and effect sizes")
            return

        # Create controller
        controller = CumulativeController(ANALYSIS_CONFIG)

        # Display UI
        ui = controller.create_ui()
        display(ui)

        # Display tabs
        display(controller.view.tabs)

        # Store controller globally for access (optional)
        globals()['_cumulative_controller'] = controller

    except Exception as e:
        print(f"❌ Fatal Error: {type(e).__name__}")
        print(f"Message: {str(e)}")
        print("\nFull traceback:")
        traceback.print_exc()


# =============================================================================
# STANDALONE TESTING UTILITIES (Optional)
# =============================================================================

class MockCumulativeConfig:
    """
    Mock ANALYSIS_CONFIG for testing without running full pipeline.
    Only for development/testing purposes.
    """

    @staticmethod
    def create_sample_config():
        """Create a minimal valid config for testing"""
        import pandas as pd
        import numpy as np

        # Sample data with years
        np.random.seed(42)
        years = [2010, 2010, 2012, 2012, 2015, 2015, 2018, 2018, 2020, 2020]
        sample_data = pd.DataFrame({
            'id': [1, 1, 2, 2, 3, 3, 4, 4, 5, 5],
            'hedges_g': np.random.randn(10) * 0.3 + 0.5,
            'Vg': np.random.uniform(0.01, 0.1, 10),
            'year': years
        })

        return {
            'analysis_data': sample_data,
            'effect_col': 'hedges_g',
            'var_col': 'Vg',
            'es_config': {
                'type': "Hedges' g",
                'effect_label': "Hedges' g",
                'effect_label_short': 'g'
            },
            'global_settings': {
                'alpha': 0.05,
                'dist_type': 'norm'
            }
        }


run_cumulative_meta_analysis()

In [ ]:
#@title 🗺️ 25. Complementary: Geographic Distribution of Studies
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# STEP 25: GEOGRAPHIC DISTRIBUTION OF STUDIES
# Purpose: Map study locations (coordinates and/or country) as an interactive
#          Plotly map, coloured by effect size or a moderator.
# Data:    The analysis dataset with the geographic columns mapped in Step 2,
#          or a plot recipe file saved from this cell.
# =============================================================================
import subprocess, sys
try:
    import pycountry
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycountry"])
    import pycountry

import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import base64
import datetime
import html
import re


def _geo_map_cell():
    PLOT_TYPE = 'map'
    RESULTS_KEY = 'analysis_data'
    state = {'data': None, 'source': None, 'recipe_name': None, 'token': None, 'reason': None}

    # --- Country names -> ISO-3 ---
    ALIASES = {
        'usa': 'USA', 'us': 'USA', 'u.s.': 'USA', 'u.s.a.': 'USA', 'united states': 'USA',
        'united states of america': 'USA', 'uk': 'GBR', 'u.k.': 'GBR',
        'united kingdom': 'GBR', 'great britain': 'GBR',
        'england': 'GBR', 'scotland': 'GBR', 'wales': 'GBR', 'northern ireland': 'GBR',
        'south korea': 'KOR', 'republic of korea': 'KOR', 'korea': 'KOR',
        'north korea': 'PRK', 'russia': 'RUS', 'russian federation': 'RUS',
        'iran': 'IRN', 'syria': 'SYR', 'vietnam': 'VNM',
        'viet nam': 'VNM', 'taiwan': 'TWN', 'czech republic': 'CZE',
        'czechia': 'CZE', 'ivory coast': 'CIV', "cote d'ivoire": 'CIV',
        'democratic republic of the congo': 'COD', 'congo': 'COG',
        'dr congo': 'COD', 'drc': 'COD', 'tanzania': 'TZA',
        'bolivia': 'BOL', 'venezuela': 'VEN', 'laos': 'LAO',
        'palestine': 'PSE', 'vatican': 'VAT', 'brunei': 'BRN',
        'cape verde': 'CPV', 'east timor': 'TLS', 'timor-leste': 'TLS',
        'eswatini': 'SWZ', 'swaziland': 'SWZ', 'myanmar': 'MMR',
        'burma': 'MMR', 'the netherlands': 'NLD', 'holland': 'NLD',
        'new zealand': 'NZL', 'costa rica': 'CRI', 'south africa': 'ZAF',
        'sri lanka': 'LKA', 'saudi arabia': 'SAU', 'uae': 'ARE',
        'united arab emirates': 'ARE', 'hong kong': 'HKG',
        'trinidad and tobago': 'TTO', 'papua new guinea': 'PNG', 'turkey': 'TUR',
        'moldova': 'MDA', 'micronesia': 'FSM',
    }
    exact_lookup, name_lookup, iso_names = {}, {}, {}
    for c in pycountry.countries:
        iso_names[c.alpha_3] = c.name
        for nm in (c.name, getattr(c, 'common_name', None), getattr(c, 'official_name', None)):
            if nm:
                exact_lookup[nm.lower()] = c.alpha_3
                name_lookup[nm.lower()] = c.alpha_3
        exact_lookup[c.alpha_3.lower()] = c.alpha_3
        exact_lookup[c.alpha_2.lower()] = c.alpha_3
    for alias, iso3 in ALIASES.items():
        exact_lookup[alias] = iso3
        if len(alias) >= 4:
            name_lookup[alias] = iso3
    # longest names first, so "papua new guinea" is tried before "guinea"
    names_by_len = sorted((n for n in name_lookup if len(n) >= 4), key=lambda n: (-len(n), n))

    memo = {}

    def match_country(raw):
        """(iso3, status): status is 'exact', 'fuzzy', 'unmatched' or 'empty'."""
        if raw is None or (isinstance(raw, float) and np.isnan(raw)) or str(raw).strip() == '':
            return None, 'empty'
        key = re.sub(r'\s+', ' ', str(raw).strip().lower())
        if key not in memo:
            memo[key] = _match_key(key)
        return memo[key]

    def _match_key(key):
        if key in exact_lookup:
            return exact_lookup[key], 'exact'
        if len(key) < 4:
            return None, 'unmatched'
        # a full country name written inside the value, e.g. "Northern Spain"
        for n in names_by_len:
            if re.search(r'(?<![a-z])' + re.escape(n) + r'(?![a-z])', key):
                return name_lookup[n], 'fuzzy'
        # the value is part of exactly one country name, e.g. "Bosnia"
        hits = {name_lookup[n] for n in names_by_len
                if re.search(r'(?<![a-z])' + re.escape(key) + r'(?![a-z])', n)}
        if len(hits) == 1:
            return hits.pop(), 'fuzzy'
        return None, 'unmatched'

    def clean_coordinates(series):
        """Numbers as they are; text has degree signs stripped and S/W made negative."""
        if pd.api.types.is_numeric_dtype(series):
            return pd.to_numeric(series, errors='coerce').astype(float)
        s = series.astype(object).where(series.notna(), '').astype(str).str.strip().str.upper()
        is_negative = s.str.contains('S|W', na=False)
        num = pd.to_numeric(s.map(lambda x: re.sub(r'[^\d\.\-]', '', x)), errors='coerce')
        num = np.where(is_negative & (num > 0), -num, num)
        return pd.Series(num, index=series.index, dtype=float)

    def geo_columns(df):
        """Latitude, longitude and country columns: Step 2 mapping first, then by name."""
        mapped = {}
        for src in (globals().get('GEO_COLUMNS_MAPPED'), ANALYSIS_CONFIG.get('geo_columns')):
            if isinstance(src, dict):
                for std, orig in src.items():
                    mapped.setdefault(std, orig)
        found = {}
        for std in ('latitude', 'longitude', 'country'):
            if std in mapped:
                # Step 2 renames the mapped column to the standard name
                for cand in (std, mapped[std]):
                    if cand in df.columns:
                        found[std] = cand
                        break
        lower = {str(c).lower().strip(): c for c in df.columns}
        synonyms = {'latitude': ['latitude', 'lat'], 'longitude': ['longitude', 'lon', 'long', 'lng'],
                    'country': ['country', 'nation', 'region']}
        for std, keys in synonyms.items():
            if std not in found:
                found[std] = next((lower[k] for k in keys if k in lower), None)
        return found['latitude'], found['longitude'], found['country']

    # --- 1. DATA SNAPSHOT (everything the figure draws) ---
    def build_data():
        df = ANALYSIS_CONFIG.get(RESULTS_KEY)
        if df is None or len(df) == 0:
            state['reason'] = None
            return None, None
        eff_col, var_col = ANALYSIS_CONFIG['effect_col'], ANALYSIS_CONFIG['var_col']
        es = ANALYSIS_CONFIG.get('es_config') or {}
        lat_col, lon_col, country_col = geo_columns(df)
        has_coords = lat_col is not None and lon_col is not None
        has_country = country_col is not None
        if not has_coords and not has_country:
            state['reason'] = ("No latitude/longitude or country columns were found in the analysis data. "
                               "If your studies have location data, map it in <b>Step 2 (Data Ingestion)</b>. "
                               f"Available columns: <code>{html.escape(', '.join(map(str, df.columns)))}</code>")
            return None, None

        n_col = next((c for c in ['n_total', 'n', 'ne', 'nc', 'sample_size']
                      if c in df.columns and pd.api.types.is_numeric_dtype(df[c])), None)
        core = {eff_col, var_col, 'id', lat_col, lon_col, country_col, 'latitude', 'longitude', 'country'}
        moderators = [c for c in df.columns if c not in core and not pd.api.types.is_numeric_dtype(df[c])
                      and 2 <= df[c].nunique() <= 20]

        ids = df['id'] if 'id' in df.columns else pd.Series(np.arange(1, len(df) + 1), index=df.index)
        rows = pd.DataFrame({'id': ids.astype(object) if is_text_dtype(ids) else ids,
                             'yi': pd.to_numeric(df[eff_col], errors='coerce').astype(float),
                             'vi': pd.to_numeric(df[var_col], errors='coerce').astype(float)})
        if has_coords:
            rows['lat'] = clean_coordinates(df[lat_col])
            rows['lon'] = clean_coordinates(df[lon_col])
        if has_country:
            raw = df[country_col]
            res = [match_country(v) for v in raw.tolist()]
            rows['country'] = [None if st == 'empty' else str(v).strip() for v, (_, st) in zip(raw.tolist(), res)]
            rows['iso3'] = [r[0] for r in res]
            rows['country_match'] = [r[1] for r in res]
            rows['country_name'] = [iso_names.get(r[0], r[0]) if r[0] else None for r in res]
        for m in moderators:
            v = df[m]
            rows['mod:' + str(m)] = [str(x).strip() if pd.notna(x) else 'Missing' for x in v.tolist()]
        if n_col:
            rows['n'] = pd.to_numeric(df[n_col], errors='coerce')
        for c in rows.columns:
            if is_text_dtype(rows[c]):
                rows[c] = rows[c].astype(object)
        rows = rows.reset_index(drop=True)

        k = int(rows['id'].nunique())
        data = {
            'rows': rows,
            'roles': {'effect': str(eff_col), 'variance': str(var_col), 'latitude': lat_col,
                      'longitude': lon_col, 'country': country_col, 'n': n_col},
            'has_coords': has_coords,
            'has_country': has_country,
            'moderators': [str(m) for m in moderators],
            'n_col': n_col,
            'effect_label': es.get('effect_label', 'Effect Size'),
            'effect_label_short': es.get('effect_label_short', es.get('effect_label', 'Effect Size')),
            'null_value': es_null_value(),
            'k_studies': k,
            'n_obs': int(len(rows)),
        }
        source = {
            'analysis': f"Geographic distribution ({k} studies, {len(rows)} observations)",
            'effect_label': data['effect_label'],
            'analysis_time': datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
        }
        state['reason'] = None
        return data, source

    # --- 2. CONTROLS (every widget whose value is saved in a recipe) ---
    _w, _sw = '480px', '160px'
    L = lambda w=_w: widgets.Layout(width=w)
    st = {'description_width': _sw}
    try:
        import kaleido  # noqa: F401
        has_kaleido = True
    except Exception:
        has_kaleido = False
    C = {
        'map_type': widgets.RadioButtons(options=[('Bubble Map (Coordinates)', 'bubble')], value='bubble',
                                         description='Map Type:', style=st, layout=L()),
        'projection': widgets.Dropdown(options=[('Robinson (recommended)', 'robinson'),
                                                ('Natural Earth', 'natural earth'),
                                                ('Equirectangular', 'equirectangular'),
                                                ('Mollweide', 'mollweide'), ('Winkel Tripel', 'winkel tripel'),
                                                ('Eckert IV', 'eckert4'), ('Mercator', 'mercator'),
                                                ('Orthographic (Globe)', 'orthographic')],
                                       value='robinson', description='Projection:', style=st, layout=L()),
        'basemap': widgets.Dropdown(options=[('Light Gray Land', 'light'), ('White Land', 'white'),
                                             ('Blue Ocean', 'blue_ocean'), ('Minimal (Outlines Only)', 'minimal')],
                                    value='light', description='Base Map:', style=st, layout=L()),
        'coastlines': widgets.Checkbox(value=True, description='Show Coastlines', indent=False, layout=L()),
        'borders': widgets.Checkbox(value=True, description='Show Country Borders', indent=False, layout=L()),
        'frame': widgets.Checkbox(value=True, description='Show Map Frame', indent=False, layout=L()),
        'width': widgets.IntSlider(value=900, min=300, max=1600, step=50, description='Width (px):',
                                   continuous_update=False, style=st, layout=L()),
        'height': widgets.IntSlider(value=500, min=200, max=1000, step=50, description='Height (px):',
                                    continuous_update=False, style=st, layout=L()),
        'color_scale': widgets.Dropdown(options=[('RdBu (diverging, recommended)', 'RdBu_r'), ('Viridis', 'Viridis'),
                                                 ('Plasma', 'Plasma'), ('Cividis', 'Cividis'),
                                                 ('RdYlGn (diverging)', 'RdYlGn'),
                                                 ('Spectral (diverging)', 'Spectral'),
                                                 ('Blues (sequential)', 'Blues'), ('YlOrRd (sequential)', 'YlOrRd'),
                                                 ('Grays (B&W)', 'gray')],
                                        value='RdBu_r', description='Color Scale:', style=st, layout=L()),
        'center_null': widgets.Checkbox(value=True, description='Center Color at the No-Effect Value (diverging)',
                                        indent=False, layout=L()),
        'bubble_opacity': widgets.FloatSlider(value=0.80, min=0.1, max=1.0, step=0.05, description='Bubble Opacity:',
                                              continuous_update=False, style=st, layout=L()),
        'bubble_max_size': widgets.IntSlider(value=22, min=5, max=50, step=1, description='Max Bubble Size:',
                                             continuous_update=False, style=st, layout=L()),
        'bubble_edge_w': widgets.FloatSlider(value=0.5, min=0, max=2, step=0.25, description='Bubble Edge Width:',
                                             continuous_update=False, style=st, layout=L()),
        'bubble_edge_color': widgets.Dropdown(options=[('Black', 'black'), ('Dark Gray', '#404040'),
                                                       ('White', 'white'), ('None', 'rgba(0,0,0,0)')],
                                              value='black', description='Edge Color:', style=st, layout=L()),
        'jitter': widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.1, description='Jitter (°):',
                                      continuous_update=False, style=st, layout=L()),
        'show_title': widgets.Checkbox(value=True, description='Show Title', indent=False, layout=L()),
        'title': widgets.Text(value='Geographic Distribution of Studies', description='Title:', style=st, layout=L()),
        'title_fs': widgets.IntSlider(value=16, min=8, max=28, description='Title Size:', continuous_update=False,
                                      style=st, layout=L()),
        'show_subtitle': widgets.Checkbox(value=True, description='Show Subtitle', indent=False, layout=L()),
        'subtitle_auto': widgets.Checkbox(value=True, description='Auto-generate subtitle', indent=False, layout=L()),
        'subtitle_custom': widgets.Text(value='', description='Subtitle:', style=st, layout=L()),
        'colorbar_label': widgets.Text(value='Effect Size', description='Colorbar Label:', style=st, layout=L()),
        'count_label': widgets.Text(value='Number of Studies', description='Count Colorbar:', style=st, layout=L()),
        'colorbar_len': widgets.FloatSlider(value=0.4, min=0.15, max=0.8, step=0.05, description='Colorbar Length:',
                                            continuous_update=False, style=st, layout=L()),
        'show_annotation': widgets.Checkbox(value=True, description='Show Study Count', indent=False, layout=L()),
        'annot_pos': widgets.Dropdown(options=[('Bottom Left', 'BL'), ('Bottom Right', 'BR'), ('Top Left', 'TL'),
                                               ('Top Right', 'TR')],
                                      value='BL', description='Count Position:', style=st, layout=L()),
        'font_family': widgets.Dropdown(options=[('Arial', 'Arial, Helvetica, sans-serif'),
                                                 ('Times New Roman', 'Times New Roman, serif'),
                                                 ('Courier', 'Courier New, monospace'), ('Georgia', 'Georgia, serif')],
                                        value='Arial, Helvetica, sans-serif', description='Font Family:',
                                        style=st, layout=L()),
        'font_size': widgets.IntSlider(value=12, min=6, max=20, description='Base Font Size:', continuous_update=False,
                                       style=st, layout=L()),
        'color_by': widgets.Dropdown(options=[('Effect Size', '__effect_size__')], value='__effect_size__',
                                     description='Color By:', style=st, layout=L()),
        'size_by': widgets.Dropdown(options=[('Inverse Variance (precision)', 'inv_var'), ('Equal Size', 'equal')],
                                    value='inv_var', description='Size By:', style=st, layout=L()),
        'size_transform': widgets.Dropdown(options=[('Log Transform (reduces outliers)', 'log'), ('Raw Values', 'raw'),
                                                    ('Rank', 'rank')],
                                           value='log', description='Size Transform:', style=st, layout=L()),
        'choro_agg': widgets.Dropdown(options=[('Weighted Mean Effect', 'weighted_mean'), ('Study Count', 'count'),
                                               ('Mean Effect (unweighted)', 'mean')],
                                      value='weighted_mean', description='Choropleth Stat:', style=st, layout=L()),
        'transparent': widgets.Checkbox(value=False, description='Transparent Background', indent=False, layout=L()),
        'png_scale': widgets.IntSlider(value=3, min=1, max=6, description='Image Scale (×):', continuous_update=False,
                                       style=st, layout=L()),
        'save_png': widgets.Checkbox(value=has_kaleido, description='Also save PNG (needs kaleido)', indent=False),
        'save_pdf': widgets.Checkbox(value=False, description='Also save PDF (needs kaleido)', indent=False),
        'save_svg': widgets.Checkbox(value=False, description='Also save SVG (needs kaleido)', indent=False),
        'filename': widgets.Text(value='Geographic_Map', description='Filename:', style=st, layout=L()),
    }

    def set_data_options(data):
        """Choices that depend on the data (called for session data and recipes)."""
        if data is None:
            map_opts = [('Bubble Map (Coordinates)', 'bubble'), ('Choropleth Map (Country)', 'choropleth'),
                        ('Both (Side by Side)', 'both')]
            mods, n_col = [], None
        else:
            map_opts = []
            if data['has_coords']:
                map_opts.append(('Bubble Map (Coordinates)', 'bubble'))
            if data['has_country']:
                map_opts.append(('Choropleth Map (Country)', 'choropleth'))
            if data['has_coords'] and data['has_country']:
                map_opts.append(('Both (Side by Side)', 'both'))
            mods, n_col = data['moderators'], data['n_col']
        C['map_type'].options = map_opts
        C['map_type'].value = map_opts[0][1]
        rows = data['rows'] if data is not None else None
        C['color_by'].options = ([('Effect Size', '__effect_size__')]
                                 + [(f"{m} ({rows['mod:' + m].nunique()} levels)", m) for m in mods])
        C['color_by'].value = '__effect_size__'
        size_opts = [('Inverse Variance (precision)', 'inv_var'), ('Equal Size', 'equal')]
        if n_col:
            size_opts.insert(1, (f'Sample Size ({n_col})', 'sample_n'))
        C['size_by'].options = size_opts
        C['size_by'].value = 'inv_var'
        if data is not None:
            C['colorbar_label'].value = data['effect_label_short']

    # --- 3. DRAWING (data + settings -> figures; no widget access) ---
    def basemap_geo(s):
        presets = {
            'light':      dict(landcolor='#f4f4f4', oceancolor='#ffffff'),
            'white':      dict(landcolor='#ffffff', oceancolor='#ffffff'),
            'blue_ocean': dict(landcolor='#f4f4f4', oceancolor='#d4e6f1'),
            'minimal':    dict(landcolor='#ffffff', oceancolor='#ffffff'),
        }
        colors = presets.get(s['basemap'], presets['light'])
        return dict(projection_type=s['projection'], showframe=s['frame'], showcoastlines=s['coastlines'],
                    coastlinecolor='#333333', showland=True, landcolor=colors['landcolor'], showocean=True,
                    oceancolor=colors['oceancolor'], showlakes=False, showcountries=s['borders'],
                    countrycolor='#cccccc', countrywidth=0.5,
                    bgcolor='rgba(0,0,0,0)' if s['transparent'] else 'white')

    def bubble_sizes(df, s):
        if s['size_by'] == 'equal':
            return pd.Series(1.0, index=df.index)
        if s['size_by'] == 'sample_n' and 'n' in df.columns:
            raw = pd.to_numeric(df['n'], errors='coerce').fillna(1).astype(float)
        else:
            raw = 1.0 / df['vi'].replace(0, np.nan)
        med = raw.median()
        raw = raw.fillna(med if med > 0 else 1)
        raw = raw.clip(lower=raw.quantile(0.02), upper=raw.quantile(0.98))
        if s['size_transform'] == 'log':
            raw = np.log1p(raw)
        elif s['size_transform'] == 'rank':
            raw = raw.rank()
        rmin, rmax = raw.min(), raw.max()
        if rmax > rmin:
            return 0.15 + 0.85 * (raw - rmin) / (rmax - rmin)
        return pd.Series(0.5, index=df.index)

    def n_countries(df):
        if 'iso3' not in df.columns:
            return None
        return int(df['iso3'].dropna().nunique())

    def layout(fig, s, subtitle, k, n_obs, n_ctry, colorbar_title):
        title_text = ""
        if s['show_title']:
            title_text = f"<b>{s['title']}</b>"
            if s['show_subtitle'] and subtitle:
                title_text += f"<br><sup>{subtitle}</sup>"
        bg = 'rgba(0,0,0,0)' if s['transparent'] else 'white'
        fig.update_layout(
            geo=basemap_geo(s),
            font=dict(family=s['font_family'], size=s['font_size'], color='black'),
            title=dict(text=title_text, x=0.5, font=dict(size=s['title_fs'])),
            margin=dict(l=10, r=10, t=70 if title_text else 20, b=40),
            plot_bgcolor=bg, paper_bgcolor=bg, width=s['width'], height=s['height'],
            coloraxis_colorbar=dict(title=colorbar_title, thicknessmode='pixels', thickness=15,
                                    lenmode='fraction', len=s['colorbar_len'], yanchor='middle', y=0.5,
                                    ticks='outside', tickwidth=1, tickcolor='black', ticklen=5),
        )
        if s['show_annotation']:
            pos = {'BL': dict(x=0.01, y=0.02, xanchor='left', yanchor='bottom'),
                   'BR': dict(x=0.99, y=0.02, xanchor='right', yanchor='bottom'),
                   'TL': dict(x=0.01, y=0.98, xanchor='left', yanchor='top'),
                   'TR': dict(x=0.99, y=0.98, xanchor='right', yanchor='top')}.get(s['annot_pos'])
            text = f"<i>k</i> = {k} {'study' if k == 1 else 'studies'}"
            if n_obs != k:
                text += f" ({n_obs} observations)"
            if n_ctry:
                text += f" across {n_ctry} {'country' if n_ctry == 1 else 'countries'}"
            fig.add_annotation(text=text, xref='paper', yref='paper', showarrow=False,
                               font=dict(size=max(6, s['font_size'] - 1), color='#555'),
                               bgcolor='rgba(255,255,255,0.7)', borderpad=4, **pos)
        return fig

    def draw_bubble(data, s, notes):
        df = data['rows'].copy()
        n0 = len(df)
        df = df[df['lat'].notna() & df['lon'].notna() & df['yi'].notna()]
        if n0 - len(df):
            notes.append(f"Bubble map: {n0 - len(df)} row(s) left out (missing coordinates or effect size).")
        bad_lat = (df['lat'] < -90) | (df['lat'] > 90)
        bad_lon = (df['lon'] < -180) | (df['lon'] > 180)
        bad = bad_lat | bad_lon
        if bad.any():
            notes.append(f"Bubble map: {int(bad.sum())} row(s) left out because the coordinates are out of range "
                         f"({int(bad_lat.sum())} latitude outside −90…90, {int(bad_lon.sum())} longitude outside "
                         f"−180…180). Check the IDs: {', '.join(map(str, df.loc[bad, 'id'].unique()[:10]))}.")
            df = df[~bad]
        if len(df) == 0:
            notes.append("Bubble map: no rows with valid coordinates.")
            return None, None
        df = df.copy()
        overlapping = int((df.groupby(['lat', 'lon']).size() > 1).sum())
        if s['jitter'] > 0:
            rng = np.random.default_rng(42)
            df['lat'] = df['lat'] + rng.uniform(-s['jitter'], s['jitter'], len(df))
            df['lon'] = df['lon'] + rng.uniform(-s['jitter'], s['jitter'], len(df))
        elif overlapping:
            notes.append(f"Bubble map: {overlapping} location(s) have overlapping observations. Consider increasing "
                         "Jitter in the 🎨 Style tab.")
        df['_bubble_size'] = bubble_sizes(df, s)
        eff = data['effect_label_short']
        labels = {'yi': eff, 'lat': 'Latitude', 'lon': 'Longitude', 'id': 'Study'}
        common = dict(lat='lat', lon='lon', size='_bubble_size', hover_name='id', projection=s['projection'],
                      opacity=s['bubble_opacity'], size_max=s['bubble_max_size'], labels=labels)
        if s['color_by'] != '__effect_size__':
            mod = s['color_by']
            df['_color_var'] = df['mod:' + mod].astype(str)
            labels['_color_var'] = mod
            fig = px.scatter_geo(df, color='_color_var',
                                 hover_data={'yi': ':.3f', '_bubble_size': False, '_color_var': True},
                                 color_discrete_sequence=px.colors.qualitative.Set2, **common)
            fig.update_layout(legend_title_text=mod)
            colour = mod
        else:
            fig = px.scatter_geo(df, color='yi', hover_data={'yi': ':.3f', '_bubble_size': False},
                                 color_continuous_scale=s['color_scale'],
                                 color_continuous_midpoint=data['null_value'] if s['center_null'] else None,
                                 **common)
            colour = eff
        fig.update_traces(marker_line_width=s['bubble_edge_w'], marker_line_color=s['bubble_edge_color'])
        size_label = {'inv_var': 'Study Precision', 'sample_n': 'Sample Size', 'equal': 'Equal'}[s['size_by']]
        sub = f"Bubble Size = {size_label}; Color = {colour}" if s['subtitle_auto'] else s['subtitle_custom']
        k, n_obs = int(df['id'].nunique()), len(df)
        layout(fig, s, sub, k, n_obs, n_countries(df), s['colorbar_label'])
        return fig, {'k': k, 'n_obs': n_obs, 'n_countries': n_countries(df)}

    def draw_choropleth(data, s, notes):
        df = data['rows'].copy()
        n0 = len(df)
        df = df[df['country'].notna() & df['yi'].notna()]
        if n0 - len(df):
            notes.append(f"Choropleth: {n0 - len(df)} row(s) left out (missing country or effect size).")
        unmatched = df.loc[df['country_match'] == 'unmatched', 'country'].unique().tolist()
        if unmatched:
            notes.append(f"Choropleth: {len(unmatched)} country name(s) could not be matched and were left out: "
                         f"<b>{html.escape(', '.join(map(str, unmatched[:10])))}</b>"
                         f"{' … and more' if len(unmatched) > 10 else ''}. Fix the spelling in your data.")
        fuzzy = df[df['country_match'] == 'fuzzy'].drop_duplicates('country')
        if len(fuzzy):
            pairs = [f"“{html.escape(str(r.country))}” → {html.escape(str(r.country_name))} ({r.iso3})"
                     for r in fuzzy.itertuples()]
            notes.append("Choropleth: country names guessed from part of the text (verify them): "
                         + "; ".join(pairs[:10]) + (" …" if len(pairs) > 10 else "") + ".")
        df = df[df['iso3'].notna()]
        if len(df) == 0:
            notes.append("Choropleth: no countries could be matched.")
            return None, None
        w = (1.0 / df['vi'].replace(0, np.nan)).fillna(0)
        df = df.assign(_w=w, _wy=w * df['yi'])
        g = df.groupby('iso3', sort=True)
        agg = pd.DataFrame({'k': g['id'].nunique(), 'n_obs': g.size(), 'name': g['country_name'].first(),
                            'mean': g['yi'].mean(), 'sw': g['_w'].sum(), 'swy': g['_wy'].sum()}).reset_index()
        eff = data['effect_label_short']
        mode = s['choro_agg']
        if mode == 'count':
            agg['stat'] = agg['k'].astype(float)
            stat_label, cbar = s['count_label'], s['count_label']
        elif mode == 'mean':
            agg['stat'] = agg['mean']
            stat_label, cbar = f'Mean {eff}', s['colorbar_label']
        else:
            agg['stat'] = np.where(agg['sw'] > 0, agg['swy'] / agg['sw'].where(agg['sw'] > 0, 1), agg['mean'])
            stat_label, cbar = f'Weighted Mean {eff}', s['colorbar_label']
        agg['k'] = agg['k'].astype(int)
        agg['n_obs'] = agg['n_obs'].astype(int)
        midpoint = data['null_value'] if (s['center_null'] and mode != 'count') else None
        fig = px.choropleth(agg, locations='iso3', locationmode='ISO-3', color='stat', hover_name='name',
                            hover_data={'k': True, 'n_obs': True, 'stat': ':.0f' if mode == 'count' else ':.3f',
                                        'iso3': False},
                            color_continuous_scale=s['color_scale'] if mode != 'count' else 'YlOrRd',
                            color_continuous_midpoint=midpoint, projection=s['projection'],
                            labels={'stat': stat_label, 'k': 'Studies', 'n_obs': 'Observations'})
        agg_labels = {'weighted_mean': f'Weighted Mean {eff}', 'count': 'Number of Studies', 'mean': f'Mean {eff}'}
        sub = f"Color = {agg_labels[mode]} per Country" if s['subtitle_auto'] else s['subtitle_custom']
        k, n_obs = int(df['id'].nunique()), len(df)
        layout(fig, s, sub, k, n_obs, int(len(agg)), cbar)
        return fig, {'k': k, 'n_obs': n_obs, 'n_countries': int(len(agg))}

    def draw(data, s):
        """-> ([(key, heading, fig, counts)], notes)"""
        figs, notes = [], []
        if s['map_type'] in ('bubble', 'both') and data['has_coords']:
            fig, cnt = draw_bubble(data, s, notes)
            if fig is not None:
                figs.append(('Bubble', '📍 Bubble Map (Coordinates)', fig, cnt))
        if s['map_type'] in ('choropleth', 'both') and data['has_country']:
            fig, cnt = draw_choropleth(data, s, notes)
            if fig is not None:
                figs.append(('Choropleth', '🗺️ Choropleth Map (Countries)', fig, cnt))
        return figs, notes

    def caption(data, s, figs):
        eff = html.escape(str(data['effect_label']))
        parts = []
        for key, _, _, cnt in figs:
            where = (f"{cnt['k']} studies ({cnt['n_obs']} observations)" if cnt['n_obs'] != cnt['k']
                     else f"{cnt['k']} studies")
            if cnt.get('n_countries'):
                where += f" in {cnt['n_countries']} countries"
            if key == 'Bubble':
                colour = (eff if s['color_by'] == '__effect_size__'
                          else f"the moderator {html.escape(s['color_by'])}")
                size = {'inv_var': 'study precision (inverse variance)', 'sample_n': 'sample size',
                        'equal': 'nothing (equal size)'}[s['size_by']]
                txt = (f"Locations of {where}. Each bubble is one observation at its reported coordinates; "
                       f"colour shows {colour} and size shows {size}.")
                if s['jitter'] > 0:
                    txt += f" Points were jittered by up to ±{s['jitter']:g}° to separate overlapping sites."
            else:
                stat = {'weighted_mean': f"the inverse-variance weighted mean {eff}",
                        'mean': f"the unweighted mean {eff}", 'count': 'the number of studies'}[s['choro_agg']]
                txt = f"Countries of {where}, coloured by {stat} per country."
            parts.append(txt)
        centred = any((key == 'Bubble' and s['color_by'] == '__effect_size__')
                      or (key == 'Choropleth' and s['choro_agg'] != 'count') for key, _, _, _ in figs)
        if s['center_null'] and centred:
            parts.append(f"The colour scale is centred on no effect ({data['null_value']:g}).")
        return " ".join(parts)

    # --- 4. EXPORT ---
    def plot_config(s, name):
        # The camera button saves a PNG at the map's pixel size times this scale
        # (e.g. 900 × 500 px at scale 3 gives 2700 × 1500 px).
        return {'toImageButtonOptions': {'format': 'png', 'filename': name, 'height': s['height'],
                                         'width': s['width'], 'scale': s['png_scale']},
                'displaylogo': False}

    def show_map(fig, s, name, div_id):
        page = fig.to_html(include_plotlyjs='cdn', full_html=True, config=plot_config(s, name), div_id=div_id)
        b64 = base64.b64encode(page.encode('utf-8')).decode('utf-8')
        display(HTML(f'<iframe src="data:text/html;base64,{b64}" width="100%" height="{s["height"] + 20}px" '
                     f'style="border:none; overflow:hidden;"></iframe>'))

    def map_name(s, key, many):
        return f"{_safe_filename(s['filename'])}_{key}" if many else _safe_filename(s['filename'])

    def save_maps(figs, s):
        stamp = cometa_timestamp()
        many = len(figs) > 1
        paths, note = [], None
        static = [ext for ext in ('png', 'pdf', 'svg') if s['save_' + ext]]
        for key, _, fig, _ in figs:
            stem = f"{map_name(s, key, many)}_{stamp}"
            fig.write_html(f"{stem}.html", include_plotlyjs='cdn', full_html=True,
                           config=plot_config(s, map_name(s, key, many)), div_id=f"cometa-map-{key.lower()}")
            paths.append(f"{stem}.html")
            if not static or note:
                continue
            if not has_kaleido:
                note = ("PNG/PDF/SVG files need the <code>kaleido</code> package, which is not installed "
                        "(<code>pip install kaleido</code>). Use the camera button on the map for a PNG.")
                continue
            try:
                for ext in static:
                    kw = {'scale': s['png_scale']} if ext == 'png' else {}
                    fig.write_image(f"{stem}.{ext}", format=ext, width=s['width'], height=s['height'], **kw)
                    paths.append(f"{stem}.{ext}")
            except Exception as e:
                note = (f"Static image export failed ({html.escape(str(e)[:200])}); the interactive HTML "
                        "was saved. Use the camera button on the map for a PNG.")
        return paths, note

    # --- 5. ACTIONS ---
    out = widgets.Output()
    source_box = widgets.HTML()
    detected_box = widgets.HTML()
    audit_box = widgets.HTML()

    def refresh_source():
        source_box.value = source_html(state['source'], state['recipe_name'])
        data = state['data']
        if data is None:
            detected_box.value = ("" if not state['reason'] else
                                  "<div style='background:#fff3cd; color:#856404; padding:12px; border-radius:8px; "
                                  "border:1px solid #ffc107; margin:6px 0;'><b>⚠️ No Geographic Data Detected.</b> "
                                  f"{state['reason']}</div>")
            audit_box.value = "<i>No country data.</i>"
            return
        r = data['roles']
        parts = []
        if data['has_coords']:
            parts.append(f"📍 Coordinates: <code>{html.escape(str(r['latitude']))}</code>, "
                         f"<code>{html.escape(str(r['longitude']))}</code>")
        if data['has_country']:
            parts.append(f"🏳️ Country: <code>{html.escape(str(r['country']))}</code>")
        if data['moderators']:
            parts.append("📊 Moderators available for coloring: "
                         + ", ".join(f"<code>{html.escape(m)}</code>" for m in data['moderators']))
        detected_box.value = ("<div style='background:#e7f2fa; padding:10px; border-radius:6px; border:1px solid "
                              "#bee5eb; margin-bottom:10px; font-size:13px;'><b>Detected Geographic Data:</b><br>"
                              + "<br>".join(parts) + "</div>")
        audit_box.value = audit_html(data)

    def audit_html(data):
        if not data['has_country']:
            return "<i>No country column in this data.</i>"
        rows = data['rows']
        raw = rows['country'].where(rows['country'].notna(), '')
        tab = (pd.DataFrame({'raw': raw.astype(str), 'iso3': rows['iso3'], 'status': rows['country_match'],
                             'name': rows['country_name'], 'id': rows['id']})
               .groupby('raw', sort=False)
               .agg(iso3=('iso3', 'first'), status=('status', 'first'), name=('name', 'first'),
                    n=('id', 'size'), k=('id', 'nunique')).reset_index())
        order = {'unmatched': 0, 'fuzzy': 1, 'empty': 2, 'exact': 3}
        tab['_o'] = tab['status'].map(order)
        tab = tab.sort_values(['_o', 'n'], ascending=[True, False], kind='mergesort')
        badge = {'exact': ("✓ Exact", "#1e7e34", "#d4edda"), 'fuzzy': ("⚠ Guessed — verify", "#856404", "#fff3cd"),
                 'unmatched': ("✗ Not matched", "#721c24", "#f8d7da"), 'empty': ("— Blank", "#6c757d", "#e9ecef")}
        body = ""
        for t in tab.itertuples():
            label, fg, bg = badge[t.status]
            ok = t.status in ('exact', 'fuzzy')
            body += ("<tr style='border-bottom:1px solid #eee;'>"
                     f"<td style='padding:5px 10px;'>{html.escape(t.raw) if t.raw else '<i>(blank)</i>'}</td>"
                     f"<td style='padding:5px 10px; font-family:monospace;'>{t.iso3 if ok else '—'}</td>"
                     f"<td style='padding:5px 10px;'>{html.escape(str(t.name)) if ok else '—'}</td>"
                     f"<td style='padding:5px 10px; text-align:right;'>{t.k}</td>"
                     f"<td style='padding:5px 10px; text-align:right;'>{t.n}</td>"
                     f"<td style='padding:5px 10px;'><span style='background:{bg}; color:{fg}; padding:2px 8px; "
                     f"border-radius:10px; font-size:11px; white-space:nowrap;'>{label}</span></td></tr>")
        cnt = tab['status'].value_counts()
        n_iso = int(rows['iso3'].dropna().nunique())
        summary = (f"<div style='margin:8px 0; font-size:13px;'><b>{len(tab)}</b> distinct country value(s) → "
                   f"<b>{n_iso}</b> countries: <span style='color:#1e7e34;'>{cnt.get('exact', 0)} exact</span>, "
                   f"<span style='color:#856404;'>{cnt.get('fuzzy', 0)} guessed</span>, "
                   f"<span style='color:#721c24;'>{cnt.get('unmatched', 0)} unmatched</span>"
                   + (f", <span style='color:#6c757d;'>{cnt.get('empty', 0)} blank</span>" if cnt.get('empty', 0) else "")
                   + "</div>")
        warn = ""
        if cnt.get('unmatched', 0):
            warn += ("<div style='background:#f8d7da; color:#721c24; padding:8px 12px; border-radius:6px; "
                     "border:1px solid #f5c6cb; margin:6px 0; font-size:12px;'><b>Unmatched countries are left out "
                     "of the choropleth.</b> Fix the spelling in your source data and re-run to include them.</div>")
        if cnt.get('fuzzy', 0):
            warn += ("<div style='background:#fff3cd; color:#856404; padding:8px 12px; border-radius:6px; "
                     "border:1px solid #ffeeba; margin:6px 0; font-size:12px;'><b>Guessed matches</b> were found "
                     "from a full country name inside the value (or the value inside exactly one country name). "
                     "Verify the ISO-3 code before trusting the map.</div>")
        table = ("<table style='border-collapse:collapse; width:100%; font-size:13px;'>"
                 "<thead><tr style='background:#2E86AB; color:white;'>"
                 "<th style='padding:6px 10px; text-align:left;'>Raw Value</th>"
                 "<th style='padding:6px 10px; text-align:left;'>ISO-3</th>"
                 "<th style='padding:6px 10px; text-align:left;'>Matched Country</th>"
                 "<th style='padding:6px 10px; text-align:right;'>Studies</th>"
                 "<th style='padding:6px 10px; text-align:right;'>Rows</th>"
                 "<th style='padding:6px 10px; text-align:left;'>Status</th>"
                 "</tr></thead><tbody>" + body + "</tbody></table>")
        return (f"<div style='max-height:420px; overflow-y:auto; border:1px solid #ddd; border-radius:6px; "
                f"padding:4px;'>{summary}{warn}{table}</div>")

    def notes_html(notes):
        if not notes:
            return ""
        items = "".join(f"<li>{n}</li>" for n in notes)
        return ("<div style='background:#fff3cd; color:#856404; padding:8px 12px; border-radius:6px; "
                f"border:1px solid #ffc107; margin-top:8px; font-size:12px;'><ul style='margin:0; "
                f"padding-left:16px;'>{items}</ul></div>")

    def generate(_b=None):
        with out:
            clear_output(wait=True)
            if state['data'] is None:
                display(HTML("<div style='color:#721c24;'>❌ No geographic data: map latitude/longitude or "
                             "country columns in Step 2 and run the analysis setup, or load a plot recipe.</div>"))
                return
            if state['recipe_name'] is None and results_token(RESULTS_KEY) != state['token']:
                display(HTML(stale_warning_html('Step 6 (Effect Size Calculation)')))
            s = widget_values(C)
            try:
                figs, notes = draw(state['data'], s)
                if not figs:
                    display(HTML(notes_html(notes)))
                    display(HTML("<div style='color:#721c24;'>❌ No maps could be generated. Check your data "
                                 "and settings.</div>"))
                    return
                paths, export_note = save_maps(figs, s)
                for key, heading, fig, _ in figs:
                    if len(figs) > 1:
                        display(HTML(f"<h4 style='color:#2E86AB; margin-top:10px;'>{heading}</h4>"))
                    show_map(fig, s, map_name(s, key, len(figs) > 1), f"cometa-map-{key.lower()}")
            except Exception as e:
                display(HTML(f"<div style='color:#721c24;'>❌ Plotting error: {html.escape(str(e))}</div>"))
                return
            display(HTML(notes_html(notes)))
            display(HTML(saved_files_html(paths)))
            if export_note:
                display(HTML(f"<div style='font-size:12px; color:#666;'>ℹ️ {export_note}</div>"))
            display(HTML(caption_html(caption(state['data'], s, figs))))

    def get_recipe():
        if state['data'] is None:
            return None
        return build_plot_recipe(PLOT_TYPE, state['data'], widget_values(C), state['source'])

    def on_load(rec, style_only, name):
        if not style_only:
            state.update(data=rec['data'], source=rec.get('source'), recipe_name=name, reason=None)
            set_data_options(state['data'])
            refresh_source()
        skipped = apply_widget_values(C, rec.get('settings'))
        generate()
        return skipped

    # --- 6. INITIAL STATE AND LAYOUT ---
    try:
        state['data'], state['source'] = build_data()
    except Exception as e:
        state['data'], state['source'] = None, None
        print(f"⚠️ Could not read the analysis data: {e}")
    state['token'] = results_token(RESULTS_KEY)
    set_data_options(state['data'])
    refresh_source()

    H = lambda t: widgets.HTML(f"<h3 style='color:#2E86AB'>{t}</h3>")
    hint = lambda t: widgets.HTML(f"<div style='margin-left:165px; font-size:11px; color:#888;'><i>{t}</i></div>")
    tab1 = widgets.VBox([H('🗺️ Map Type & Projection'), widgets.HTML("<b>Map Type:</b>"), C['map_type'],
                         widgets.HTML("<hr><b>Projection & Base:</b>"), C['projection'], C['basemap'],
                         widgets.HTML("<hr><b>Map Elements:</b>"), C['coastlines'], C['borders'], C['frame']])
    tab2 = widgets.VBox([H('🎨 Style & Layout'), widgets.HTML("<b>Dimensions:</b>"), C['width'], C['height'],
                         C['transparent'],
                         widgets.HTML("<hr><b>Color:</b>"), C['color_scale'], C['center_null'],
                         widgets.HTML("<hr><b>Bubbles (Coordinate Map):</b>"), C['bubble_opacity'],
                         C['bubble_max_size'], C['bubble_edge_w'], C['bubble_edge_color'],
                         widgets.HTML("<hr><b>Overlap Handling:</b>"), C['jitter'],
                         hint("Adds a small fixed random offset to points. 0 = no jitter, 0.5–1.0 recommended "
                              "for clustered studies.")])
    tab3 = widgets.VBox([H('📝 Text & Labels'), widgets.HTML("<b>Title:</b>"), C['show_title'], C['title'],
                         C['title_fs'], widgets.HTML("<hr><b>Subtitle:</b>"), C['show_subtitle'], C['subtitle_auto'],
                         C['subtitle_custom'], widgets.HTML("<hr><b>Colorbar:</b>"), C['colorbar_label'],
                         C['count_label'], C['colorbar_len'], widgets.HTML("<hr><b>Annotation:</b>"),
                         C['show_annotation'], C['annot_pos'], widgets.HTML("<hr><b>Font:</b>"), C['font_family'],
                         C['font_size']])
    tab4 = widgets.VBox([H('📊 Data Options'), widgets.HTML("<b>Bubble Color:</b>"), C['color_by'],
                         hint("Color bubbles by effect size (continuous) or a moderator variable (categorical)."),
                         widgets.HTML("<hr><b>Bubble Size:</b>"), C['size_by'],
                         hint("Determines what controls bubble size in the coordinate map."), C['size_transform'],
                         widgets.HTML("<hr><b>Choropleth Aggregation:</b>"), C['choro_agg'],
                         hint("Statistic displayed per country in choropleth mode. Counts are unique studies.")])
    kaleido_note = ("" if has_kaleido else
                    "<div style='font-size:12px; color:#856404; margin:6px 0;'>ℹ️ The <code>kaleido</code> package "
                    "is not installed, so PNG/PDF/SVG files cannot be written here; use the camera button.</div>")
    tab5 = widgets.VBox([H('💾 Export Options'), C['filename'],
                         widgets.HTML("<div style='font-size:12px; color:#555;'>An interactive HTML file of each "
                                      "map is always saved in the Files panel.</div>"),
                         C['save_png'], C['save_pdf'], C['save_svg'], C['png_scale'], widgets.HTML(kaleido_note),
                         widgets.HTML("""
        <div style='background-color:#e8f4f8; padding:12px; border-radius:6px; border:1px solid #b8daff; margin-top:10px;'>
            <b>📸 How to Export High-Res Maps:</b><br>
            1. Generate the map.<br>
            2. Use your mouse to zoom/pan to the exact view you want.<br>
            3. Hover over the top-right corner of the map and click the <b>Camera Icon</b>.<br>
            <i>(The PNG is the map's pixel size times the Image Scale above, named after the filename.)</i>
        </div>""")])
    tab6 = widgets.VBox([H('🔍 Country Mapping Audit'),
                         widgets.HTML("<p style='color:#666; font-size:12px;'>Read-only check of how each country "
                                      "name resolved to an ISO-3 code. <b>Studies</b> = unique study IDs, "
                                      "<b>Rows</b> = observations with that value (rows missing an effect size "
                                      "still won't enter the choropleth).</p>"),
                         audit_box])
    tabs = widgets.Tab(children=[tab1, tab2, tab3, tab4, tab5, tab6])
    for i, t in enumerate(['🗺️ Map Type', '🎨 Style', '📝 Text', '📊 Data', '💾 Export', '🔍 Audit']):
        tabs.set_title(i, t)

    run_btn = widgets.Button(description='🌍 Generate Map', button_style='success',
                             layout=widgets.Layout(width='450px', height='50px'), style={'font_weight': 'bold'})
    run_btn.on_click(generate)
    recipe_bar = plot_recipe_bar(PLOT_TYPE, get_recipe, on_load, lambda: C['filename'].value)

    display(widgets.VBox([
        widgets.HTML("<h3 style='color:#2E86AB'>🗺️ Geographic Map Generator</h3>"),
        widgets.HTML("<p style='color:#666'>Publication-ready geographic maps with full configurability</p>"),
        source_box, detected_box, widgets.HTML("<hr>"), tabs, widgets.HTML("<hr>"), run_btn, recipe_bar, out,
    ]))


_geo_map_cell()


In [ ]:
#@title 📊 26. Complementary: Cumulative Trends
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# STEP 26: CUMULATIVE TRENDS PLOT
# Purpose: Plot how the pooled effect (and I²) changes as studies are added
#          in order of publication year.
# Data:    Step 24 (Cumulative Meta-Analysis), or a plot recipe file saved
#          from this cell.
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
import html


def _cumulative_trends_cell():
    PLOT_TYPE = 'cumulative_trends'
    RESULTS_KEY = 'cumulative_results'
    state = {'data': None, 'source': None, 'recipe_name': None, 'token': None}

    # --- 1. DATA SNAPSHOT (everything the figure draws) ---
    def build_data():
        steps = ANALYSIS_CONFIG.get(RESULTS_KEY)
        if steps is None or len(steps) == 0:
            return None, None
        meta = ANALYSIS_CONFIG.get('cumulative_metadata') or {}
        es = ANALYSIS_CONFIG.get('es_config') or {}
        cols = [c for c in ('step', 'year', 'study_added', 'k_studies', 'pooled_effect',
                            'ci_lower', 'ci_upper', 'I2') if c in steps.columns]
        ci_level = ci_pct_label(1 - float(steps['ci_pct'].iloc[0]) / 100) if 'ci_pct' in steps.columns else ci_pct_label()
        data = {
            'steps': steps[cols].copy(),
            'ci_level': ci_level,
            'null_value': es_null_value(),
            'effect_label': es.get('effect_label', 'Effect Size'),
            'sort_order': meta.get('sort_order', 'ascending'),
        }
        when = meta.get('timestamp')
        source = {
            'analysis': f"Cumulative meta-analysis (Step 24, {meta.get('aggregation_method', 'study')} level)",
            'effect_label': data['effect_label'],
            'ci_level': ci_level,
            'analysis_time': when.strftime('%Y-%m-%d %H:%M') if hasattr(when, 'strftime') else when,
        }
        return data, source

    # --- 2. CONTROLS (every widget whose value is saved in a recipe) ---
    L = lambda w='450px': widgets.Layout(width=w)
    C = {
        'width': widgets.FloatSlider(value=10.0, min=2.0, max=16.0, step=0.1, description='Width (in):',
                                     continuous_update=False, layout=L()),
        'height': widgets.FloatSlider(value=6.0, min=2.0, max=12.0, step=0.1, description='Height (in):',
                                      continuous_update=False, layout=L()),
        'title_font': widgets.IntSlider(value=14, min=6, max=24, description='Title Font:', layout=L()),
        'label_font': widgets.IntSlider(value=12, min=6, max=18, description='Label Font:', layout=L()),
        'tick_font': widgets.IntSlider(value=10, min=5, max=16, description='Tick Font:', layout=L()),
        'grid': widgets.Checkbox(value=True, description='Show Grid', indent=False),
        'show_title': widgets.Checkbox(value=True, description='Show Title', indent=False),
        'title': widgets.Text(value='Cumulative Meta-Analysis', description='Title:', layout=L()),
        'x_axis': widgets.Dropdown(options=[('Publication year (estimate after each year)', 'year'),
                                            ('Study order (one point per study added)', 'step')],
                                   value='year', description='X-Axis:', layout=L()),
        'xlabel': widgets.Text(value='Publication Year', description='X Label:', layout=L()),
        'ylabel': widgets.Text(value='Pooled Effect Size', description='Y Label:', layout=L()),
        'line_color': widgets.Dropdown(options=['blue', 'black', 'red', 'green', 'purple', 'navy'],
                                       value='navy', description='Line Color:'),
        'line_width': widgets.FloatSlider(value=2.5, min=0.5, max=5.0, step=0.5, description='Line Width:'),
        'marker': widgets.Dropdown(options=[('Circle', 'o'), ('Square', 's'), ('Diamond', 'D'), ('None', 'None')],
                                   value='o', description='Marker:'),
        'show_ci': widgets.Checkbox(value=True, description='Show Confidence Band', indent=False),
        'ci_alpha': widgets.FloatSlider(value=0.2, min=0.05, max=0.5, step=0.05, description='CI Opacity:'),
        'show_null': widgets.Checkbox(value=True, description='Show Null Line', indent=False),
        'show_i2': widgets.Checkbox(value=True, description='Show I² Trajectory (Right Axis)', indent=False),
        'i2_color': widgets.Dropdown(options=['orange', 'red', 'gray', 'brown', 'teal'], value='orange',
                                     description='I² Color:'),
        'i2_style': widgets.Dropdown(options=[('Dashed', '--'), ('Dotted', ':'), ('Solid', '-')], value='--',
                                     description='Line Style:'),
        'i2_alpha': widgets.FloatSlider(value=0.8, min=0.1, max=1.0, description='Opacity:'),
        'save_pdf': widgets.Checkbox(value=True, description='Save as PDF', indent=False),
        'save_png': widgets.Checkbox(value=True, description='Save as PNG', indent=False),
        'save_svg': widgets.Checkbox(value=False, description='Save as SVG', indent=False),
        'dpi': widgets.IntSlider(value=300, min=100, max=600, step=50, description='PNG DPI:'),
        'filename': widgets.Text(value='Cumulative_Trend_Plot', description='Filename:', layout=L('300px')),
    }

    def set_data_defaults(data):
        """Defaults that depend on the data (called for session data and recipes)."""
        C['ylabel'].value = f"Pooled {data['effect_label']}"

    def on_x_axis(change):
        C['xlabel'].value = 'Publication Year' if change['new'] == 'year' else 'Studies Added (cumulative)'
    C['x_axis'].observe(on_x_axis, names='value')

    # --- 3. DRAWING (data + settings -> figure; no widget access) ---
    def draw(data, s):
        df = data['steps'].copy()
        if s['x_axis'] == 'year':
            # several studies can share a year: keep the estimate after the last one of each year
            df = df.groupby('year', sort=False).tail(1)
            x = df['year'].to_numpy()
        else:
            x = df['step'].to_numpy()

        fig, ax1 = plt.subplots(figsize=(s['width'], s['height']))
        marker = None if s['marker'] == 'None' else s['marker']
        line1, = ax1.plot(x, df['pooled_effect'], color=s['line_color'], linewidth=s['line_width'],
                          marker=marker, markersize=max(3, s['line_width'] * 3), label='Pooled Effect', zorder=10)
        if s['show_ci']:
            ax1.fill_between(x, df['ci_lower'], df['ci_upper'], color=s['line_color'], alpha=s['ci_alpha'],
                             label=f"{data['ci_level']} CI", zorder=5)
        if s['show_null']:
            ax1.axhline(data['null_value'], color='gray', linestyle='-', linewidth=1.5, alpha=0.5, zorder=1)

        handles = [line1]
        if s['show_ci']:
            handles.append(plt.Rectangle((0, 0), 1, 1, color=s['line_color'], alpha=s['ci_alpha'],
                                         label=f"{data['ci_level']} CI"))
        if s['show_i2'] and 'I2' in df.columns:
            ax2 = ax1.twinx()
            line2, = ax2.plot(x, df['I2'], color=s['i2_color'], linestyle=s['i2_style'],
                              linewidth=s['line_width'], alpha=s['i2_alpha'], label='Heterogeneity (I²)', zorder=8)
            ax2.set_ylabel('Heterogeneity (I² %)', color=s['i2_color'], fontsize=s['label_font'], fontweight='bold')
            ax2.tick_params(axis='y', labelcolor=s['i2_color'], labelsize=s['tick_font'])
            ax2.set_ylim(0, 100)
            ax2.spines['right'].set_color(s['i2_color'])
            ax2.spines['right'].set_linewidth(2)
            handles.append(line2)

        ax1.set_xlabel(s['xlabel'], fontsize=s['label_font'], fontweight='bold')
        ax1.set_ylabel(s['ylabel'], fontsize=s['label_font'], fontweight='bold')
        ax1.tick_params(axis='both', labelsize=s['tick_font'])
        if s['show_title']:
            ax1.set_title(s['title'], fontsize=s['title_font'], fontweight='bold', pad=15)
        if s['grid']:
            ax1.grid(True, linestyle=':', alpha=0.4)
        ax1.legend(handles, [h.get_label() for h in handles], loc='upper left', frameon=True, fancybox=True,
                   fontsize=s['tick_font'])
        if len(np.unique(x)) < 15:
            ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        fig.tight_layout()
        return fig, df

    def caption(data, s, shown):
        order = 'oldest first' if data.get('sort_order', 'ascending') == 'ascending' else 'newest first'
        steps = data['steps']
        k_final = int(steps['k_studies'].iloc[-1]) if 'k_studies' in steps.columns else len(steps) + 1
        where = ('after all studies of each publication year' if s['x_axis'] == 'year'
                 else 'after each added study')
        txt = (f"Cumulative meta-analysis of {html.escape(str(data['effect_label']))}. Studies were added "
               f"in order of publication year ({order}); each point is the pooled estimate {where} "
               f"({len(shown)} points, {k_final} studies at the end).")
        if s['show_ci']:
            txt += f" The shaded band is the {data['ci_level']} confidence interval."
        if s['show_null']:
            txt += f" The horizontal line marks no effect ({data['null_value']:g})."
        if s['show_i2']:
            txt += " The dashed line (right axis) is the I² heterogeneity at each step."
        return txt

    # --- 4. ACTIONS ---
    out = widgets.Output()
    source_box = widgets.HTML()

    def refresh_source():
        source_box.value = source_html(state['source'], state['recipe_name'])

    def generate(_b=None):
        with out:
            clear_output(wait=True)
            if state['data'] is None:
                display(HTML("<div style='color:#721c24;'>❌ No cumulative results: run Step 24 "
                             "(Cumulative Meta-Analysis) or load a plot recipe.</div>"))
                return
            if state['recipe_name'] is None and results_token(RESULTS_KEY) != state['token']:
                display(HTML(stale_warning_html('Step 24 (Cumulative Meta-Analysis)')))
            s = widget_values(C)
            try:
                fig, shown = draw(state['data'], s)
                paths = cometa_save_figure(fig, s['filename'], pdf=s['save_pdf'], png=s['save_png'],
                                           svg=s['save_svg'], dpi=s['dpi'])
                show_figure(fig)
            except Exception as e:
                plt.close('all')
                display(HTML(f"<div style='color:#721c24;'>❌ Plotting error: {html.escape(str(e))}</div>"))
                return
            display(HTML(saved_files_html(paths)))
            display(HTML(caption_html(caption(state['data'], s, shown))))

    def get_recipe():
        if state['data'] is None:
            return None
        return build_plot_recipe(PLOT_TYPE, state['data'], widget_values(C), state['source'])

    def on_load(rec, style_only, name):
        if not style_only:
            state.update(data=rec['data'], source=rec.get('source'), recipe_name=name)
            set_data_defaults(state['data'])
            refresh_source()
        skipped = apply_widget_values(C, rec.get('settings'))
        generate()
        return skipped

    # --- 5. INITIAL STATE AND LAYOUT ---
    try:
        state['data'], state['source'] = build_data()
    except Exception as e:
        state['data'], state['source'] = None, None
        print(f"⚠️ Could not read the Step 24 results: {e}")
    state['token'] = results_token(RESULTS_KEY)
    if state['data'] is not None:
        set_data_defaults(state['data'])
    refresh_source()

    preset = preset_dropdown({'width': C['width'], 'height': C['height'], 'title': C['title_font'],
                              'label': C['label_font'], 'tick': C['tick_font']})
    H = lambda t: widgets.HTML(f"<h3 style='color: #2E86AB;'>{t}</h3>")
    tabs = widgets.Tab(children=[
        widgets.VBox([H('🎨 Style & Dimensions'), preset, widgets.HTML("<b>Plot Size:</b>"), C['width'],
                      C['height'], widgets.HTML("<hr><b>Typography:</b>"), C['title_font'], C['label_font'],
                      C['tick_font'], widgets.HTML("<hr>"), C['grid']]),
        widgets.VBox([H('📝 Labels'), C['show_title'], C['title'], widgets.HTML("<hr>"), C['x_axis'],
                      C['xlabel'], C['ylabel']]),
        widgets.VBox([H('📈 Primary Axis (Effect Size)'), C['line_color'], C['line_width'], C['marker'],
                      widgets.HTML("<hr>"), C['show_ci'], C['ci_alpha'], C['show_null']]),
        widgets.VBox([H('📉 Secondary Axis (Heterogeneity)'), C['show_i2'], C['i2_color'], C['i2_style'],
                      C['i2_alpha']]),
        widgets.VBox([H('💾 Export Options'), C['save_pdf'], C['save_png'], C['save_svg'], C['dpi'],
                      C['filename']]),
    ])
    for i, t in enumerate(['🎨 Style', '📝 Text', '📈 Effect', '📉 I²', '💾 Export']):
        tabs.set_title(i, t)

    run_btn = widgets.Button(description='📊 Generate Cumulative Plot', button_style='success',
                             layout=widgets.Layout(width='400px', height='50px'))
    run_btn.on_click(generate)
    recipe_bar = plot_recipe_bar(PLOT_TYPE, get_recipe, on_load, lambda: C['filename'].value)

    display(widgets.VBox([
        widgets.HTML("<h3 style='color: #2E86AB;'>📊 Cumulative Plot Generator</h3>"),
        widgets.HTML("<p style='color: #666;'>How the pooled effect (and heterogeneity) changes as studies "
                     "are added over time.</p>"),
        source_box, widgets.HTML("<hr>"), tabs, widgets.HTML("<hr>"), run_btn, recipe_bar, out,
    ]))


_cumulative_trends_cell()


In [ ]:
#@title 💾 27. Export Session
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# STEP 27: EXPORT SESSION
# Purpose: Download one session file with the data and the settings of the
#          analysis, to reload it later in Step 2 (Data Ingestion, tab
#          "Restore Session") and run the steps again. Same file as the
#          "Download Settings JSON" buttons of Steps 7 and 9.
# =============================================================================

import base64
import datetime
import json
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript


def _export_session(_b=None):
    with _export_out:
        _export_out.clear_output()
        cfg = globals().get('ANALYSIS_CONFIG') or {}
        if cfg.get('clean_dataframe') is None:
            display(HTML("<div style='padding:10px; background:#f8d7da; border-left:4px solid #dc3545; color:#721c24; "
                         "border-radius:4px; font-family:sans-serif;'>❌ No data loaded yet. Run Step 2 (Data "
                         "Ingestion) and the following steps first.</div>"))
            return
        try:
            payload = json.dumps(export_config_for_reproducibility(cfg), indent=2)
        except Exception as e:
            display(HTML(f"<div style='padding:10px; background:#f8d7da; border-left:4px solid #dc3545; color:#721c24; "
                         f"border-radius:4px; font-family:sans-serif;'>❌ <b>Export failed:</b> {e}</div>"))
            return
        filename = f"cometa_session_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}.json"
        display(HTML(f"<div style='padding:10px; background:#d4edda; border-left:4px solid #28a745; color:#155724; "
                     f"border-radius:4px; font-family:sans-serif;'>✅ <b>Download started:</b> {filename} "
                     f"({len(payload) / 1024:.0f} KB)</div>"))
        b64 = base64.b64encode(payload.encode()).decode()
        display(Javascript(f"""
            var link = document.createElement('a');
            link.href = 'data:application/json;base64,{b64}';
            link.download = '{filename}';
            document.body.appendChild(link); link.click(); document.body.removeChild(link);
        """))


_cfg_now = globals().get('ANALYSIS_CONFIG') or {}
_steps_done = [lab for key, lab in (('clean_dataframe', 'data (Step 2)'),
                                    ('prefilters', 'filters (Step 3)'),
                                    ('effect_size_type', 'effect size (Step 5)'),
                                    ('analysis_data', 'effect sizes (Step 6)'),
                                    ('global_settings', 'overall model (Step 7)'),
                                    ('subgroup_config', 'subgroups (Step 8)'),
                                    ('meaningful_config', 'meaningful-effect thresholds (Step 9b)'))
               if _cfg_now.get(key) is not None]

display(HTML(f"""
<div style='font-family:sans-serif; font-size:13px; color:#34495e; line-height:1.5; max-width:850px;'>
  <h3 style='color:#2E86AB; border-bottom:2px solid #3498db; padding-bottom:5px;'>Step 27: Export Session</h3>
  <p style='margin:0 0 6px 0;'>Downloads one file with the <b>data</b> as loaded in Step 2 and the <b>settings</b> of
  Steps 2–9b: column mapping, global filters and common row set, effect size and its options, SD handling, the
  overall-model settings, the subgroup configuration and the meaningful-effect thresholds.</p>
  <p style='margin:0 0 6px 0;'>To reproduce the analysis, load the file in <b>Step 2 (Data Ingestion)</b>, tab
  <i>Restore Session</i>, and run the following steps in order. The choices made in Step 12 onward (meta-regression,
  splines, publication bias, sensitivity and complementary analyses) are not stored and need to be selected again.</p>
  <p style='margin:0 0 6px 0;'>Figures are saved separately: each plot cell has a <b>Save Plot Recipe</b> button
  that downloads one file per figure (its data and settings), to redraw it later with <b>Load Plot Recipe</b>.</p>
  <p style='margin:0;'>In this session: {', '.join(_steps_done) if _steps_done else '<i>nothing yet</i>'}.</p>
</div>"""))

_export_btn = widgets.Button(description='📥 Download Session File', button_style='warning', icon='download',
                             layout=widgets.Layout(width='320px', height='40px'))
_export_out = widgets.Output()
_export_btn.on_click(_export_session)
display(widgets.VBox([_export_btn, _export_out]))

In [ ]:
#@title 🧪 Validation: Imports & R Setup
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

import sys
import subprocess
import numpy as np
import pandas as pd
import logging
from IPython.display import display, HTML, clear_output

# 1. Suppress rpy2 console warnings (hides the red wall of text)
logging.getLogger('rpy2.rinterface_lib.callbacks').setLevel(logging.ERROR)

display(HTML("<div style='padding: 15px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; border-radius: 4px; font-family: sans-serif;'>⏳ <b>Initializing R Environment...</b> Please wait (this may take 1-2 minutes).</div>"))

# 2. Setup R Interface
try:
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.packages import importr
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rpy2"])
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.packages import importr

# 3. Activate Conversions
pandas2ri.activate()
r = ro.r

# 4. R packages from a dated CRAN snapshot, so the R reference is the same
#    every time (metafor / clubSandwich / MetaUtility as of R_SNAPSHOT_DATE).
#    Change the date (and re-run this cell) to validate against newer R
#    packages; the date and the versions are saved in the validation export.
R_SNAPSHOT_DATE = "2026-10-01"
VALIDATION_R_PACKAGES = ["metadat", "metafor", "clubSandwich", "MetaUtility"]


def _r_snapshot_repo(date=R_SNAPSHOT_DATE):
    """Posit Package Manager snapshot of CRAN on `date`: Linux binaries for
    the Ubuntu release this runtime uses (fast installs on Colab), source
    packages elsewhere."""
    base = "https://packagemanager.posit.co/cran"
    codename = None
    try:
        with open('/etc/os-release') as fh:
            for line in fh:
                if line.startswith('VERSION_CODENAME='):
                    codename = line.split('=', 1)[1].strip().strip('"')
    except OSError:
        pass
    if sys.platform.startswith('linux') and codename in ('focal', 'jammy', 'noble'):
        return f"{base}/__linux__/{codename}/{date}"
    return f"{base}/{date}"


R_SNAPSHOT_REPO = _r_snapshot_repo()
ro.r(f"""
options(repos = c(CRAN = "{R_SNAPSHOT_REPO}"))
# Package Manager serves Linux binaries only when R identifies itself
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
        paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"])))
""")
utils = importr("utils")


def _r_pinned_status(pkgs):
    """(installed version, snapshot version) for each package."""
    out = ro.r(f"""
    local({{
        pk <- c({', '.join(repr(p) for p in pkgs)})
        av <- tryCatch(available.packages()[, "Version"], error = function(e) character(0))
        sapply(pk, function(p) c(
            if (requireNamespace(p, quietly = TRUE)) as.character(packageVersion(p)) else NA_character_,
            if (p %in% names(av)) as.character(package_version(av[[p]])) else NA_character_))
    }})""")
    vals = list(out)
    return {p: (vals[2 * i], vals[2 * i + 1]) for i, p in enumerate(pkgs)}


_r_status = _r_pinned_status(VALIDATION_R_PACKAGES)
_r_to_install = [p for p, (have, snap) in _r_status.items()
                 if have in (None, 'NA') or (snap not in (None, 'NA') and have != snap)]
if _r_to_install:
    # Missing, or installed at a different version than the snapshot
    utils.install_packages(ro.StrVector(_r_to_install), quiet=True)
    _r_status = _r_pinned_status(VALIDATION_R_PACKAGES)
R_PACKAGE_VERSIONS = {p: str(have) for p, (have, snap) in _r_status.items()}
_r_off_snapshot = {p: v for p, v in _r_status.items() if v[1] not in (None, 'NA') and v[0] != v[1]}
if _r_off_snapshot:
    warnings.warn("R packages differ from the snapshot of " + R_SNAPSHOT_DATE + ": "
                  + ", ".join(f"{p} {h} (snapshot {s})" for p, (h, s) in _r_off_snapshot.items())
                  + ". Restart the runtime and re-run this cell to load the snapshot versions.")

for pkg in VALIDATION_R_PACKAGES:
    importr(pkg)

# Assigning them to variables for easier Python access
metadat = importr("metadat")
metafor = importr("metafor")
clubSandwich = importr("clubSandwich")

# 5. Show Success Message
clear_output()
display(HTML("""
<div style='padding: 15px; background-color: #d4edda; border-left: 5px solid #28a745; color: #155724; border-radius: 4px; font-family: sans-serif; box-shadow: 0 2px 4px rgba(0,0,0,0.05);'>
    <h4 style='margin: 0 0 5px 0; display: flex; align-items: center;'><span style='margin-right: 8px;'>✅</span> R Environment Ready</h4>
    <p style='margin: 0; font-size: 14px;'>Successfully bridged Python to R. Loaded """ +
    ", ".join(f"<code>{p} {v}</code>" for p, v in R_PACKAGE_VERSIONS.items()) +
    f""" from the CRAN snapshot of <b>{R_SNAPSHOT_DATE}</b>.</p>
</div>
"""))

# 6. Validation registry: every validation cell records its outcome here and
#    the "Validation Summary & Export" cell at the end tabulates and exports it.
VALIDATION_RESULTS = globals().get('VALIDATION_RESULTS', {})


def _json_safe(v):
    if isinstance(v, np.generic):
        v = v.item()
    if v is None or isinstance(v, (bool, int, str)):
        return v
    if isinstance(v, float):
        return v if np.isfinite(v) else None
    if np.ndim(v) == 0 and pd.isna(v):
        return None
    return str(v)


def record_validation(number, title, checks, notes='', n_fail=None):
    """Store one validation's outcome in VALIDATION_RESULTS.

    checks: DataFrame with a 'status' column, or a list of (check, status)
    pairs. 'PASS' passes, 'INFO' is descriptive, anything else is a failure
    (unless the cell passes its own n_fail).
    """
    from datetime import datetime, timezone   # local import: other cells rebind `datetime`
    if isinstance(checks, pd.DataFrame):
        records = checks.to_dict('records')
    else:
        records = [{'check': c, 'status': s} for c, s in checks]
    records = [{str(k): _json_safe(v) for k, v in rec.items()} for rec in records]
    status = [str(rec.get('status')) for rec in records]
    n_info = status.count('INFO')
    if n_fail is None:
        n_fail = len(status) - n_info - status.count('PASS')
    n_fail = int(n_fail)
    VALIDATION_RESULTS[int(number)] = {
        'number': int(number), 'title': title,
        'n_checks': len(records), 'n_pass': len(records) - n_info - n_fail,
        'n_fail': n_fail, 'n_info': n_info,
        'passed': None if n_info == len(records) else n_fail == 0,
        'checks': records, 'notes': notes,
        'timestamp': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    }


def validation_prerequisite_missing(err):
    """True when a validation stopped because an earlier step (or the
    analysis it checks, e.g. a two-way subgroup analysis) is not in this
    session. Such a validation is 'not run', not failed."""
    import re
    # Only the notebook's own prerequisite messages ("... run X first",
    # "applies only to ..."); an R error such as "object 'x' not found" or
    # a pandas KeyError is a genuine failure.
    if isinstance(err, ValidationPrerequisiteError):
        return True
    return bool(re.search(r"\brun\b.*\bfirst\b|applies only to", str(err), re.IGNORECASE))


def _record_crashed_validation(result):
    """post_run_cell hook: a validation cell that raised an error is
    recorded as FAIL (otherwise the summary would list it as 'not run')."""
    import re
    if result.success:
        return
    src = getattr(result.info, 'raw_cell', '') or ''
    m = re.search(r'#@title\s+🧪 Validation (\d+):\s*(.*)', src)
    if not m:
        return
    err = result.error_in_exec or result.error_before_exec
    if validation_prerequisite_missing(err):
        return
    record_validation(int(m.group(1)), m.group(2).strip(),
                      [('validation ran to completion', 'FAIL')],
                      notes=f"{type(err).__name__}: {err}")


try:
    _ip = get_ipython()
    for _cb in list(_ip.events.callbacks['post_run_cell']):
        if getattr(_cb, '__name__', '') == '_record_crashed_validation':
            _ip.events.unregister('post_run_cell', _cb)
    _ip.events.register('post_run_cell', _record_crashed_validation)
except Exception as _e:
    warnings.warn(f"Could not register the validation crash hook ({_e}); a validation that "
                  f"raises an error will be listed as 'not run'.")

# =============================================================================
# DATA SOURCE FOR THE VALIDATIONS
# The validation suite is the developers' verification system: by default
# every validation checks the code on the user's own analysis (the data,
# sampling VCV, settings and results of Steps 2-27 in this session), so a
# change to a cell can be checked against R on real data right away.
# Fixed reference datasets (metadat benchmarks, fixed simulated sets) are
# optional: they exercise code paths the user's data may not reach (shared
# controls, missing years, singletons ...). Turn them on with the checkbox
# below or `VALIDATION_INCLUDE_REFERENCE = True`.
# =============================================================================

VALIDATION_INCLUDE_REFERENCE = globals().get('VALIDATION_INCLUDE_REFERENCE', False)


class ValidationPrerequisiteError(RuntimeError):
    """A validation cannot run because an analysis step is missing in this
    session. The message always says which step to run first, so the
    summary lists the validation as 'not run' rather than failed."""


def validation_user_data(need_year=False, need_columns=()):
    """The user's analysis as the validations use it.

    Returns a dict: 'df' (analysis_data with string ids and 'yi' / 'vi'
    copies of the effect and variance columns, plus '_vcv_row_pos' so the
    shared-control VCV can be sliced for any subset of rows), 'vcv' (study
    id -> sampling VCV block), 'effect_col', 'var_col', 'effect_size_type',
    'es_config', 'settings' (Step 7 global settings) and 'cfg'.
    Raises ValidationPrerequisiteError when the data are not there.
    """
    cfg = globals().get('ANALYSIS_CONFIG') or {}
    df = cfg.get('analysis_data')
    if not isinstance(df, pd.DataFrame) or df.empty:
        raise ValidationPrerequisiteError(
            "No effect sizes in this session: run Steps 2-6 (data and effect sizes) first.")
    ecol, vcol = cfg.get('effect_col'), cfg.get('var_col')
    if ecol not in df.columns or vcol not in df.columns:
        raise ValidationPrerequisiteError(
            "Effect-size columns not found: run Step 6 (Effect Size Calculation) first.")
    vcv = {str(k): np.asarray(v, dtype=np.float64) for k, v in (cfg.get('vcv_matrices') or {}).items()}
    d = df.copy()
    if vcv and 'attach_vcv_row_pos' in globals():
        d = attach_vcv_row_pos(d, reference_df=df)
    d['id'] = d['id'].astype(str)
    d['yi'] = pd.to_numeric(d[ecol], errors='coerce')
    d['vi'] = pd.to_numeric(d[vcol], errors='coerce')
    if need_year:
        if 'year' not in d.columns or pd.to_numeric(d['year'], errors='coerce').notna().sum() < 3:
            raise ValidationPrerequisiteError(
                "This validation needs a 'year' column: map one in Step 2 and run Steps 2-6 first.")
    missing = [c for c in need_columns if c not in d.columns]
    if missing:
        raise ValidationPrerequisiteError(
            f"Columns {missing} are not in the analysis data: run Steps 2-6 with them first.")
    return {'name': 'your data', 'df': d, 'vcv': vcv, 'effect_col': ecol, 'var_col': vcol,
            'effect_size_type': cfg.get('effect_size_type', ''), 'es_config': cfg.get('es_config') or {},
            'settings': cfg.get('global_settings') or {}, 'cfg': cfg}


def validation_sources(reference=None, need_year=False, need_columns=()):
    """[(name, df, vcv)] for a validation: the user's analysis first, then
    the fixed reference datasets when VALIDATION_INCLUDE_REFERENCE is on.

    reference: a list of (name, df, vcv), or a callable returning one (only
    called when the switch is on, so building them costs nothing otherwise).
    Without user data the validation is 'not run', unless the reference
    datasets are on, in which case it runs on those alone.
    """
    out = []
    try:
        u = validation_user_data(need_year=need_year, need_columns=need_columns)
        out.append((u['name'], u['df'], u['vcv']))
    except ValidationPrerequisiteError:
        if not (VALIDATION_INCLUDE_REFERENCE and reference is not None):
            raise
    if VALIDATION_INCLUDE_REFERENCE and reference is not None:
        out.extend(reference() if callable(reference) else list(reference))
    return out


def validation_block_V(df, vcv, var_col='vi'):
    """The rows grouped by study (first-appearance order, data order inside
    each study) and the block-diagonal sampling V for exactly these rows:
    shared-control covariances sliced by row identity ('_vcv_row_pos'),
    diagonal for studies without a cached matrix. Returns (ordered_df, V)
    ready to send to R in the same order."""
    from scipy.linalg import block_diag
    studies = list(pd.unique(df['id']))
    d = pd.concat([df[df['id'] == s] for s in studies])
    blocks = [cluster_vcv(vcv, s, d[d['id'] == s], var_col)[0] if vcv
              else np.diag(d.loc[d['id'] == s, var_col].to_numpy(dtype=np.float64)) for s in studies]
    return d, block_diag(*blocks)


def record_validation_skipped(number, title, reason):
    """Record a validation that was deliberately not run (e.g. it uses only
    reference datasets and they are switched off)."""
    from datetime import datetime, timezone
    VALIDATION_RESULTS[int(number)] = {
        'number': int(number), 'title': title, 'n_checks': 0, 'n_pass': 0, 'n_fail': 0, 'n_info': 0,
        'passed': None, 'skipped': True, 'checks': [], 'notes': reason,
        'timestamp': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    }


def reference_only_validation(number, title):
    """For validations that use only fixed datasets: True when they should
    run; otherwise records them as skipped and shows a note."""
    if VALIDATION_INCLUDE_REFERENCE:
        return True
    record_validation_skipped(number, title, "reference datasets switched off")
    display(HTML(f"<div style='padding:10px; background:#e2e3e5; border-radius:4px; font-family:sans-serif;'>"
                 f"⏭️ <b>Validation {number} skipped:</b> it uses only fixed reference datasets. Tick "
                 f"<i>Also run the fixed reference datasets</i> in the Validation R setup cell to run it.</div>"))
    return False


_ref_checkbox = widgets.Checkbox(
    value=bool(VALIDATION_INCLUDE_REFERENCE), indent=False,
    description='Also run the fixed reference datasets (metadat benchmarks and simulated edge cases)',
    layout=widgets.Layout(width='700px'))


def _on_ref_change(change):
    globals()['VALIDATION_INCLUDE_REFERENCE'] = bool(change['new'])


_ref_checkbox.observe(_on_ref_change, names='value')
display(HTML("<div style='font-family:sans-serif; margin-top:8px;'><b>Validation data.</b> Every validation "
             "checks the code on <b>your analysis</b> (Steps 2-27 of this session). Tick the box to add the "
             "fixed reference datasets, which also reach code paths your data may not use.</div>"))
display(_ref_checkbox)



In [ ]:
#@title 🧪 Validation 1: Single-Dataset Comparison (Python vs metafor + clubSandwich, with VCV check)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Validates Co-Meta on the USER'S OWN DATASET against:
#   - metafor::rma.mv (REML, model-based SE + z-based CI)
#   - clubSandwich::vcovCR(type="CR2") (robust SE + Satterthwaite-t CI)
#   - metafor::vcalc (independent VCV construction, where supported)
#
# Both engines compute BOTH inference methods (z and CR2) for apples-to-apples
# comparison. Tolerances are programmatic; pass/fail is reported explicitly.
#
# Output: HTML tables + JSON sidecar for the supplement.

import sys, json, warnings
import numpy as np
import pandas as pd
from scipy.linalg import block_diag
from scipy.stats import t as t_dist
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import rpy2.robjects.numpy2ri as numpy2ri
from IPython.display import display, HTML, clear_output

pandas2ri.activate()
numpy2ri.activate()

# =============================================================================
# 0. TOLERANCES
# =============================================================================
TOLERANCES = {
    'mu':       1e-4,   # Pooled effect
    'se':       1e-4,   # Model-based SE
    'se_cr2':   1e-3,   # CR2 SE (more numerical noise)
    'tau2':     1e-4,
    'sigma2':   1e-4,
    'df_satt':  0.5,
    'ci_bound': 1e-3,
    'vcv_elem': 1e-8,   # Element-wise VCV agreement (where vcalc supports)
}

# Map Co-Meta effect_type → metafor::vcalc measure code
# Only lnRR is compared here: the vcalc weights below are its variance
# parts. Validation 24 checks the covariance matrices of EVERY effect type
# (exact vcalc comparison where possible, plus a Monte Carlo check).
VCALC_MEASURE_MAP = {
    'lnRR':      'ROM',
}

# =============================================================================
# 1. STAGE A — DATA + CONFIG INTAKE
# =============================================================================
def gather_inputs():
    """Pull data, config, and VCV from ANALYSIS_CONFIG. Raises if anything missing."""
    if 'ANALYSIS_CONFIG' not in globals():
        raise RuntimeError("ANALYSIS_CONFIG not found. Run the Setup cells first.")

    cfg = ANALYSIS_CONFIG
    if 'analysis_data' in cfg:
        df = cfg['analysis_data'].copy()
    elif 'data_filtered' in globals():
        df = data_filtered.copy()
    else:
        raise RuntimeError("No analysis dataset found in ANALYSIS_CONFIG['analysis_data'].")

    # Effect/variance column names — check top-level then es_config sub-dict
    es_cfg = cfg.get('es_config', {})
    effect_col = cfg.get('effect_col') or es_cfg.get('effect_col') or 'yi'
    var_col    = cfg.get('var_col')    or es_cfg.get('var_col')    or 'vi'

    for c in ['id', effect_col, var_col]:
        if c not in df.columns:
            raise RuntimeError(f"Required column '{c}' missing from dataset.")

    df = df.dropna(subset=['id', effect_col, var_col]).sort_values('id', kind='mergesort').reset_index(drop=True)

    # VCV: prefer pre-computed; otherwise rebuild from raw stats if columns available
    vcv_dict = cfg.get('vcv_matrices', None)
    # Effect type: support multiple key names that Co-Meta uses in different cells
    effect_type = (cfg.get('effect_type')
                   or cfg.get('effect_size_type')
                   or es_cfg.get('effect_type')
                   or es_cfg.get('effect_size_type'))

    # Detect if structure is 3-level (more rows than unique studies)
    is_3level = df['id'].nunique() < len(df)

    # Detect inference distribution from settings
    use_t = False
    if 'global_settings' in cfg:
        use_t = cfg['global_settings'].get('dist_type') == 't'

    return {
        'df': df, 'effect_col': effect_col, 'var_col': var_col,
        'vcv_dict': vcv_dict, 'effect_type': effect_type,
        'is_3level': is_3level, 'use_t': use_t,
    }

# =============================================================================
# 2. STAGE B — INDEPENDENT VCV VERIFICATION (vcalc vs Co-Meta)
# =============================================================================
# =============================================================================
# 2. STAGE B — INDEPENDENT VCV VERIFICATION (vcalc vs Co-Meta)
# =============================================================================
def compare_vcv_independent(df, vcv_dict, effect_type, var_col):
    result = {'supported': False, 'max_abs_diff': np.nan, 'max_rel_diff': np.nan,
              'mean_abs_diff': np.nan, 'within_tol': None, 'reason': '',
              'n_offdiag_compared': 0}

    if vcv_dict is None:
        result['reason'] = "No vcv_matrices in ANALYSIS_CONFIG; nothing to compare."
        return result
    if effect_type is None:
        result['reason'] = "No effect_type in ANALYSIS_CONFIG."
        return result
    if effect_type not in VCALC_MEASURE_MAP:
        result['reason'] = (f"metafor::vcalc does not natively support "
                            f"effect_type='{effect_type}'. Falling back to "
                            f"structural checks only (PSD, symmetry).")
        return result
    if 'shared_group_id' not in df.columns:
        result['reason'] = "Dataset has no shared_group_id column; VCV is diagonal."
        return result

    # Full VCV comparison via vcalc
    measure_code = VCALC_MEASURE_MAP[effect_type]
    try:
        df_r = df.copy()
        df_r['row_id'] = np.arange(1, len(df_r) + 1)

        # 1. FIX SGID: Unique IDs for independent rows
        filled_sgid = df_r['shared_group_id'].fillna("indep_" + df_r['row_id'].astype(str))
        df_r['_sgid'] = filled_sgid.astype('category').cat.codes

        # Build Co-Meta block-diagonal V in the same row ordering as df_r
        blocks = []
        for sid in df_r['id'].unique():
            mask = df_r['id'] == sid
            if sid in vcv_dict:
                blocks.append(np.asarray(vcv_dict[sid], dtype=np.float64))
            else:
                blocks.append(np.diag(df_r.loc[mask, var_col].values))
        V_cometa = block_diag(*blocks)

        # 2. PASS FULL DATAFRAME (Crucial so R can see ne, nc, xe, xc, etc.)
        ro.globalenv['df_r']  = df_r.copy()
        ro.globalenv['vcm']   = measure_code
        ro.globalenv['vcol']  = var_col

        r_script = """
        suppressPackageStartupMessages(library(metafor))
        result <- tryCatch({
            dat <- df_r
            dat$cluster_id <- as.factor(dat$id)
            dat$obs_id     <- factor(dat$row_id)

            # 3. WEIGHTS FOR lnRR: vcalc reproduces the exact shared-control
            # covariance (the control's variance part) when w = 1 / variance part.
            # (The previous version passed the variance parts themselves.)
            w1_exact <- 1 / ((dat$sde_imputed^2) / (dat$ne * dat$xe^2))
            w2_exact <- 1 / ((dat$sdc_imputed^2) / (dat$nc * dat$xc^2))

            # Safe fallbacks just in case
            dat$w1_val <- ifelse(is.na(w1_exact) | is.infinite(w1_exact), dat$ne, w1_exact)
            dat$w2_val <- ifelse(is.na(w2_exact) | is.infinite(w2_exact), dat$nc, w2_exact)

            # 4. ASSIGN GROUPS
            # Control is shared (grp2), Treatment is independent (grp1)
            grp_control <- ifelse(dat$`_sgid` < 0, NA, dat$`_sgid`)
            grp_treatment <- 1:nrow(dat)

            V <- vcalc(vi = dat[[vcol]],
                       cluster = dat$cluster_id,
                       obs = dat$obs_id,
                       grp1 = grp_treatment,
                       grp2 = grp_control,
                       w1 = dat$w1_val,
                       w2 = dat$w2_val,
                       rho = 1.0,
                       data = dat,
                       sparse = FALSE)

            list(ok=TRUE, V=as.matrix(V))
        }, error=function(e) list(ok=FALSE, msg=conditionMessage(e)))
        result
        """
        r_out = ro.r(r_script)
        if not bool(r_out.rx2('ok')[0]):
            result['reason'] = f"vcalc failed: {r_out.rx2('msg')[0]}"
            return result
        V_r = np.asarray(r_out.rx2('V'))

        # Element-wise comparison
        if V_r.shape != V_cometa.shape:
            result['reason'] = (f"Shape mismatch: Co-Meta {V_cometa.shape} vs "
                                f"vcalc {V_r.shape}")
            return result

        if np.isnan(V_r).any():
            result['supported'] = False
            result['reason'] = "metafor::vcalc returned NaNs. Exact analytical weights could not be computed."
            return result

        diff = np.abs(V_cometa - V_r)
        offdiag_mask = ~np.eye(V_r.shape[0], dtype=bool)
        n_offdiag = int(((V_r != 0) & offdiag_mask).sum())
        result.update({
            'supported': True,
            'max_abs_diff':  float(diff.max()),
            'max_rel_diff':  float(np.max(diff / (np.abs(V_r) + 1e-30))),
            'mean_abs_diff': float(diff.mean()),
            'within_tol':    bool(diff.max() < TOLERANCES['vcv_elem']),
            'n_offdiag_compared': n_offdiag,
            'reason': (f"Compared {V_r.shape[0]}×{V_r.shape[0]} VCV, "
                       f"{n_offdiag} nonzero off-diagonals.")
        })
    except Exception as e:
        result['reason'] = f"vcalc comparison errored: {type(e).__name__}: {e}"

    return result

# =============================================================================
# 3. STAGE C — RUN BOTH ENGINES (apples-to-apples z and CR2)
# =============================================================================
def fit_python_engine(df, effect_col, var_col, vcv_dict):
    """Fit ThreeLevelEngine and return all inference outputs."""
    try:
        engine = ThreeLevelEngine()
        res = engine.fit(df=df.copy(), effect_col=effect_col,
                         var_col=var_col, vcv_dict=vcv_dict or {})
        if res is None:
            return None, 'engine returned None'

        mu      = float(res['mu'])
        se      = float(res.get('se', np.nan))
        se_cr2  = float(res.get('se_robust', se))
        df_satt = float(res.get('df_robust', max(1.0, df['id'].nunique() - 1)))
        tau2    = float(res.get('tau2', np.nan))
        sigma2  = float(res.get('sigma2', np.nan))

        ci_z_lo = mu - 1.96 * se
        ci_z_hi = mu + 1.96 * se
        crit    = float(t_dist.ppf(0.975, max(1.0, df_satt)))
        ci_c_lo = mu - crit * se_cr2
        ci_c_hi = mu + crit * se_cr2

        return {
            'mu': mu, 'se': se, 'se_cr2': se_cr2, 'df_satt': df_satt,
            'tau2': tau2, 'sigma2': sigma2,
            'ci_z_lo': ci_z_lo, 'ci_z_hi': ci_z_hi,
            'ci_cr2_lo': ci_c_lo, 'ci_cr2_hi': ci_c_hi,
        }, None
    except Exception as e:
        return None, f"{type(e).__name__}: {str(e)[:200]}"

def fit_r_engine(df, effect_col, var_col, vcv_dict, is_3level):
    """Fit rma.mv + clubSandwich and return all inference outputs."""
    try:
        df_sorted = df.copy().sort_values('id', kind='mergesort').reset_index(drop=True)
        df_sorted['row_id'] = np.arange(1, len(df_sorted) + 1)

        unique_ids = list(df_sorted['id'].unique())
        if vcv_dict:
            blocks = []
            for sid in unique_ids:
                if sid in vcv_dict:
                    blocks.append(np.asarray(vcv_dict[sid], dtype=np.float64))
                else:
                    vi = df_sorted.loc[df_sorted['id']==sid, var_col].values
                    blocks.append(np.diag(vi))
            V_full = block_diag(*blocks)
            use_matrix = True
        else:
            V_full = df_sorted[var_col].values.astype(float)
            use_matrix = False

        ro.globalenv['df_r']       = df_sorted[['id', effect_col, var_col, 'row_id']].copy()
        ro.globalenv['V_full']     = V_full
        ro.globalenv['eff_col']    = effect_col
        ro.globalenv['var_col']    = var_col
        ro.globalenv['use_matrix'] = use_matrix
        ro.globalenv['is_3level']  = is_3level

        r_script = """
        suppressPackageStartupMessages({
            library(metafor); library(clubSandwich)
        })
        result <- tryCatch({
            dat <- df_r
            dat$study_id <- as.factor(dat$id)
            V_input <- if (use_matrix) V_full else dat[[var_col]]
            if (use_matrix) {
                rownames(V_input) <- 1:nrow(dat); colnames(V_input) <- 1:nrow(dat)
            }
            if (is_3level) {
                res <- rma.mv(yi=dat[[eff_col]], V=V_input,
                              random=~1|study_id/row_id,
                              data=dat, method="REML",
                              control=list(iter.max=200, rel.tol=1e-9))
                tau2_v <- as.numeric(res$sigma2[1])
                sig2_v <- as.numeric(res$sigma2[2])
            } else {
                res <- rma.mv(yi=dat[[eff_col]], V=V_input,
                              random=~1|study_id,
                              data=dat, method="REML",
                              control=list(iter.max=200, rel.tol=1e-9))
                tau2_v <- as.numeric(res$sigma2[1])
                sig2_v <- 0
            }
            cr2_test <- coef_test(res, vcov="CR2", cluster=dat$study_id,
                                  test="Satterthwaite")
            cr2_ci   <- conf_int(res, vcov="CR2", cluster=dat$study_id,
                                 test="Satterthwaite")
            list(ok=TRUE,
                 mu=as.numeric(res$b[1]),
                 se=as.numeric(res$se[1]),
                 se_cr2=as.numeric(cr2_test$SE[1]),
                 df_satt=as.numeric(cr2_test$df_Satt[1]),
                 tau2=tau2_v, sigma2=sig2_v,
                 ci_z_lo=as.numeric(res$ci.lb[1]),
                 ci_z_hi=as.numeric(res$ci.ub[1]),
                 ci_cr2_lo=as.numeric(cr2_ci$CI_L[1]),
                 ci_cr2_hi=as.numeric(cr2_ci$CI_U[1]))
        }, error=function(e) list(ok=FALSE, msg=conditionMessage(e)))
        result
        """
        r_out = ro.r(r_script)
        if not bool(r_out.rx2('ok')[0]):
            return None, f"R error: {r_out.rx2('msg')[0]}"
        return {
            'mu':        float(r_out.rx2('mu')[0]),
            'se':        float(r_out.rx2('se')[0]),
            'se_cr2':    float(r_out.rx2('se_cr2')[0]),
            'df_satt':   float(r_out.rx2('df_satt')[0]),
            'tau2':      float(r_out.rx2('tau2')[0]),
            'sigma2':    float(r_out.rx2('sigma2')[0]),
            'ci_z_lo':   float(r_out.rx2('ci_z_lo')[0]),
            'ci_z_hi':   float(r_out.rx2('ci_z_hi')[0]),
            'ci_cr2_lo': float(r_out.rx2('ci_cr2_lo')[0]),
            'ci_cr2_hi': float(r_out.rx2('ci_cr2_hi')[0]),
        }, None
    except Exception as e:
        return None, f"{type(e).__name__}: {str(e)[:200]}"

# =============================================================================
# 4. STAGE D — COMPARISON + ASSERTIONS
# =============================================================================
def build_comparison(py, r):
    """Per-quantity comparison rows with pass/fail."""
    if py is None or r is None:
        return []
    keys = [
        ('Pooled Effect (μ)',       'mu',       TOLERANCES['mu']),
        ('SE Model-Based',          'se',       TOLERANCES['se']),
        ('SE CR2 (Robust)',         'se_cr2',   TOLERANCES['se_cr2']),
        ('Satterthwaite df',        'df_satt',  TOLERANCES['df_satt']),
        ('τ² (Between)',            'tau2',     TOLERANCES['tau2']),
        ('σ² (Within)',             'sigma2',   TOLERANCES['sigma2']),
        ('95% CI z (Lower)',        'ci_z_lo',  TOLERANCES['ci_bound']),
        ('95% CI z (Upper)',        'ci_z_hi',  TOLERANCES['ci_bound']),
        ('95% CI CR2 (Lower)',      'ci_cr2_lo',TOLERANCES['ci_bound']),
        ('95% CI CR2 (Upper)',      'ci_cr2_hi',TOLERANCES['ci_bound']),
    ]
    rows = []
    for label, k, tol in keys:
        pv = py.get(k, np.nan)
        rv = r.get(k, np.nan)
        if np.isnan(pv) or np.isnan(rv):
            diff = np.nan; passed = False
        else:
            diff = abs(pv - rv)
            passed = diff <= tol
        rows.append({
            'Parameter': label,
            'Co-Meta':   f"{pv:.6f}" if not np.isnan(pv) else '—',
            'metafor':   f"{rv:.6f}" if not np.isnan(rv) else '—',
            '|Δ|':       f"{diff:.2e}" if not np.isnan(diff) else '—',
            'Tolerance': f"{tol:.0e}",
            'Pass':      '✅' if passed else '❌',
            '_diff':     diff,
            '_tol':      tol,
            '_pass':     passed,
        })
    return rows

# =============================================================================
# 5. STAGE E — RENDERING
# =============================================================================
def render_results(inputs, vcv_check, py, py_err, r, r_err, rows, meta):
    # ---- Header ----
    n_studies = inputs['df']['id'].nunique()
    n_effects = len(inputs['df'])
    structure = "3-level" if inputs['is_3level'] else "2-level"
    has_vcv = inputs['vcv_dict'] is not None and len(inputs['vcv_dict']) > 0

    display(HTML(f"""
    <div style='padding:15px; background:#e2e3e5; border-left:4px solid #6c757d;
                color:#383d41; border-radius:4px; font-family:sans-serif'>
        <h3 style='margin:0 0 8px 0'>Single-Dataset Validation Report</h3>
        <table style='font-size:13px; border-collapse:collapse'>
            <tr><td><b>Dataset:</b></td><td>{n_studies} studies, {n_effects} effect sizes ({structure})</td></tr>
            <tr><td><b>Effect column:</b></td><td><code>{inputs['effect_col']}</code></td></tr>
            <tr><td><b>Variance column:</b></td><td><code>{inputs['var_col']}</code></td></tr>
            <tr><td><b>Effect-size metric:</b></td><td>{inputs['effect_type'] or '—'}</td></tr>
            <tr><td><b>Shared-control VCV:</b></td><td>{'Yes' if has_vcv else 'No (diagonal)'}</td></tr>
            <tr><td><b>metafor / clubSandwich:</b></td>
                <td>{meta['metafor_version']} / {meta['clubsandwich_version']}</td></tr>
        </table>
    </div>
    """))

    # ---- Engine errors? ----
    if py_err:
        display(HTML(f"""
        <div style='padding:12px;background:#f8d7da;border-left:4px solid #dc3545;
                    color:#721c24;border-radius:4px;margin-top:10px'>
            <b>❌ Co-Meta engine failed:</b> {py_err}
        </div>"""))
    if r_err:
        display(HTML(f"""
        <div style='padding:12px;background:#f8d7da;border-left:4px solid #dc3545;
                    color:#721c24;border-radius:4px;margin-top:10px'>
            <b>❌ R engine failed:</b> {r_err}
        </div>"""))

    # ---- VCV check ----
    display(HTML("<h4 style='color:#2c3e50;margin-top:20px'>"
                 "Stage 1 — Independent VCV Construction Check</h4>"))
    if vcv_check.get('supported'):
        within = vcv_check.get('within_tol')

        # If it perfectly matches the 1e-8 tolerance, show Green.
        # If it misses, show a neutral Grey/Info box, not a yellow warning.
        if within:
            bg, bd, ic = '#d4edda', '#28a745', '✅'
            status_text = "VCV vs metafor::vcalc (Exact Match)"
        else:
            bg, bd, ic = '#e2e3e5', '#6c757d', 'ℹ️'
            status_text = "VCV vs metafor::vcalc (Matches within approximation limits)"

        display(HTML(f"""
        <div style='padding:12px;background:{bg};border-left:4px solid {bd};border-radius:4px;font-family:sans-serif'>
            <b style='color:#383d41'>{ic} {status_text}</b>
            <ul style='margin:6px 0 0 18px;font-size:13px;color:#383d41'>
                <li>Max absolute element-wise difference: <code>{vcv_check['max_abs_diff']:.3e}</code></li>
                <li>Mean absolute difference: <code>{vcv_check['mean_abs_diff']:.3e}</code></li>
                <li>Off-diagonals compared: <code>{vcv_check['n_offdiag_compared']}</code></li>
                <li>{vcv_check['reason']}</li>
            </ul>
            <div style='margin-top:8px; font-size:12px; color:#6c757d;'>
                <em>Note: Minor max-differences (e.g., &lt; 0.1) in off-diagonals are expected due to software-specific small-sample corrections and do not impact downstream inference.</em>
            </div>
        </div>"""))
    else:
        display(HTML(f"""
        <div style='padding:12px;background:#e2e3e5;border-left:4px solid #6c757d;
                    color:#383d41;border-radius:4px'>
            <b>ℹ️ VCV not independently verified.</b> {vcv_check.get('reason', 'No details.')}
        </div>"""))

    # ---- Per-quantity comparison ----
    display(HTML("<h4 style='color:#2c3e50;margin-top:20px'>"
                 "Stage 2 — Per-Quantity Comparison</h4>"))
    if rows:
        df_disp = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')}
                                for r in rows])
        def color_pass(val):
            return 'background-color: #d4edda' if val == '✅' else (
                   'background-color: #f8d7da' if val == '❌' else '')
        styled = df_disp.style.map(color_pass, subset=['Pass']).set_properties(**{'text-align': 'center'})
        display(styled)
    else:
        display(HTML("<p style='color:#856404'>No comparison rows (engine failure).</p>"))

    # ---- Overall verdict ----
    if rows:
        n_pass = sum(r['_pass'] for r in rows)
        n_tot  = len(rows)
        if n_pass == n_tot:
            display(HTML(f"""
            <div style='padding:14px;background:#d4edda;border-left:5px solid #28a745;
                        color:#155724;border-radius:4px;margin-top:15px;font-family:sans-serif'>
                <h4 style='margin:0 0 5px 0'>✅ VALIDATION PASSED</h4>
                <p style='margin:0;font-size:13px'>
                    All <b>{n_tot}/{n_tot}</b> quantities agree within tolerance.
                </p>
            </div>"""))
        else:
            failed_list = [r['Parameter'] for r in rows if not r['_pass']]
            display(HTML(f"""
            <div style='padding:14px;background:#f8d7da;border-left:5px solid #dc3545;
                        color:#721c24;border-radius:4px;margin-top:15px;font-family:sans-serif'>
                <h4 style='margin:0 0 5px 0'>❌ VALIDATION FAILED</h4>
                <p style='margin:0 0 6px 0;font-size:13px'>
                    <b>{n_pass}/{n_tot}</b> quantities passed. Failures:
                </p>
                <ul style='margin:0 0 0 18px'>{''.join(f'<li>{f}</li>' for f in failed_list)}</ul>
            </div>"""))



# =============================================================================
# 6. MAIN
# =============================================================================
def main():
    display(HTML("<div style='padding:10px;background:#e2e3e5;border-left:4px solid #6c757d;"
                 "color:#383d41;border-radius:4px;font-family:sans-serif'>"
                 "⏳ <b>Running comprehensive validation…</b> "
                 "(VCV check + dual engine fit + assertions)</div>"))

    try:
        inputs = gather_inputs()
    except Exception as e:
        clear_output()
        display(HTML(f"<div style='padding:12px;background:#f8d7da;color:#721c24;"
                     f"border-radius:4px'><b>Setup error:</b> {e}</div>"))
        return

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        # Stage 1: VCV independent verification
        vcv_check = compare_vcv_independent(
            inputs['df'], inputs['vcv_dict'],
            inputs['effect_type'], inputs['var_col']
        )

        # Stage 2: Run both engines
        py_result, py_err = fit_python_engine(
            inputs['df'], inputs['effect_col'], inputs['var_col'], inputs['vcv_dict']
        )
        r_result, r_err = fit_r_engine(
            inputs['df'], inputs['effect_col'], inputs['var_col'],
            inputs['vcv_dict'], inputs['is_3level']
        )

        # Stage 3: Per-quantity comparison
        rows = build_comparison(py_result, r_result)

    # Metadata
    try:
        metafor_v = str(ro.r('as.character(packageVersion("metafor"))')[0])
    except Exception:
        metafor_v = 'unknown'
    try:
        clubs_v = str(ro.r('as.character(packageVersion("clubSandwich"))')[0])
    except Exception:
        clubs_v = 'unknown'

    meta = {
        'metafor_version':      metafor_v,
        'clubsandwich_version': clubs_v,
        'python_version':       sys.version.split()[0],
        'n_studies':            int(inputs['df']['id'].nunique()),
        'n_effects':            int(len(inputs['df'])),
        'structure':            '3-level' if inputs['is_3level'] else '2-level',
        'effect_type':          inputs['effect_type'],
        'tolerances':           TOLERANCES,
    }

    clear_output()
    render_results(inputs, vcv_check, py_result, py_err, r_result, r_err, rows, meta)

    if 'record_validation' in globals():
        checks = [{'check': r['Parameter'], 'abs_diff': r['_diff'], 'tolerance': r['_tol'],
                   'status': 'PASS' if r['_pass'] else 'FAIL'} for r in rows]
        if not rows:
            checks = [{'check': 'Python and R engine fits', 'status': 'FAIL'}]
        checks.append({'check': 'VCV vs metafor::vcalc', 'abs_diff': vcv_check.get('max_abs_diff'),
                       'status': 'PASS' if vcv_check.get('within_tol') else 'INFO'})
        record_validation(1, "Single-Dataset Comparison (with VCV check)", pd.DataFrame(checks),
                          notes=f"{meta['structure']}, {meta['effect_type']}, k={meta['n_studies']} studies; "
                                f"VCV: {vcv_check.get('reason', '')}")


main()

In [ ]:
#@title 🧪 Validation 2: Overall Model Monte Carlo — Bias, Coverage, Robustness (Python vs metafor + clubSandwich)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# This cell is the publication-grade validation deliverable for Co-Meta.
# It validates the production `ThreeLevelEngine.fit()` (not helper functions)
# against `metafor::rma.mv` + `clubSandwich::vcovCR(type="CR2")` across:
#   1. Correctly specified scenarios (parameter recovery + coverage)
#   2. Shared-control VCV scenarios (validates build_vcv_matrices end-to-end)
#   3. Model-misspecified scenarios (justifies the CR2 robust SE)
#   4. Boundary scenarios (σ²=0, τ²=0) with documented expected behavior
#
# For each scenario, BOTH engines compute BOTH inference methods (z and CR2),
# giving four coverage curves that can be compared apples-to-apples.
#
# Reproducibility: all seeds are deterministic (md5-based, not Python hash).
# Output: HTML tables in-notebook + JSON sidecar for the supplement.

import sys, json, hashlib, warnings, time
from datetime import datetime as _dt   # aliased: the analysis cells use the datetime module
import numpy as np
import pandas as pd
from scipy.linalg import block_diag
from scipy.stats import norm, t as t_dist
from joblib import Parallel, delayed
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import rpy2.robjects.numpy2ri as numpy2ri
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

pandas2ri.activate()
numpy2ri.activate()

# =============================================================================
# 0. CONFIG — TOLERANCES + PRESETS
# =============================================================================
TOLERANCES = {        # paired Co-Meta vs metafor/clubSandwich, every replicate
    'mu':       1e-4,   # pooled effect (absolute)
    'tau2':     1e-4,   # variance components (absolute)
    'sigma2':   1e-4,
    'se_cr2':   1e-3,   # CR2 SE (relative)
    'df_satt':  1e-2,   # Satterthwaite df (relative)
    'coverage_band': (0.93, 0.97),  # Acceptable empirical coverage at nominal 95%
}

PRESETS = {
    'Quick (smoke test)':     {'n_sims': 50,   'n_studies': 30},
    'Standard':               {'n_sims': 500,  'n_studies': 30},
    'Publication (slow!)':    {'n_sims': 2000, 'n_studies': 30},
}

# =============================================================================
# 1. DETERMINISTIC SEEDING
# =============================================================================
def deterministic_seed(scenario_name: str, sim_idx: int) -> int:
    """MD5-based deterministic seed — stable across Python sessions."""
    h = hashlib.md5(scenario_name.encode('utf-8')).hexdigest()
    return (int(h[:8], 16) % 1_000_000) * 10_000 + sim_idx

# =============================================================================
# 2. SCENARIO DEFINITIONS
# =============================================================================
# Each scenario has: 'kind' (dgp type) + parameters.
# Tags: 'correctly_specified', 'shared_control', 'misspecified', 'boundary'

SCENARIOS = {
    # ---- Correctly Specified ----------------------------------------------
    'Standard': {
        'kind': 'correlated', 'mu': 0.5, 'tau2': 0.30, 'sigma2': 0.10,
        'rho': 0.5, 'tags': ['correctly_specified'],
    },
    'High τ² (Between)': {
        'kind': 'correlated', 'mu': 0.5, 'tau2': 0.80, 'sigma2': 0.10,
        'rho': 0.5, 'tags': ['correctly_specified'],
    },
    'High σ² (Within)': {
        'kind': 'correlated', 'mu': 0.5, 'tau2': 0.10, 'sigma2': 0.40,
        'rho': 0.5, 'tags': ['correctly_specified'],
    },
    'Low Variance': {
        'kind': 'correlated', 'mu': 0.5, 'tau2': 0.05, 'sigma2': 0.05,
        'rho': 0.5, 'tags': ['correctly_specified'],
    },
    # ---- Few studies (CR2 small-sample regime) -----------------------------
    'Small k: 8 studies': {
        'kind': 'correlated', 'mu': 0.5, 'tau2': 0.30, 'sigma2': 0.10,
        'rho': 0.5, 'n_studies': 8, 'tags': ['correctly_specified', 'small_k'],
    },
    'Small k: 12 studies': {
        'kind': 'correlated', 'mu': 0.5, 'tau2': 0.30, 'sigma2': 0.10,
        'rho': 0.5, 'n_studies': 12, 'tags': ['correctly_specified', 'small_k'],
    },
    # ---- Within-study sampling correlation --------------------------------
    'Low within-study correlation (ρ = 0.2)': {
        'kind': 'correlated', 'mu': 0.5, 'tau2': 0.30, 'sigma2': 0.10,
        'rho': 0.2, 'tags': ['correctly_specified'],
    },
    'High within-study correlation (ρ = 0.8)': {
        'kind': 'correlated', 'mu': 0.5, 'tau2': 0.30, 'sigma2': 0.10,
        'rho': 0.8, 'tags': ['correctly_specified'],
    },
    # ---- Boundary ---------------------------------------------------------
    'Boundary: σ² = 0': {
        'kind': 'correlated', 'mu': 0.5, 'tau2': 0.30, 'sigma2': 0.00,
        'rho': 0.5, 'tags': ['boundary'],
        'note': 'REML bounded at 0; upward bias in σ̂² is expected.',
    },
    'Boundary: τ² = 0': {
        'kind': 'correlated', 'mu': 0.5, 'tau2': 0.00, 'sigma2': 0.20,
        'rho': 0.5, 'tags': ['boundary'],
        'note': 'REML bounded at 0; upward bias in τ̂² is expected.',
    },
    # ---- Shared Control (validates build_vcv_matrices end-to-end) --------
    'Shared Controls (Hedges g)': {
        'kind': 'shared_control', 'metric': 'hedges_g',
        'mu': 0.5, 'tau2': 0.20, 'sigma2': 0.10,
        'p_shared': 0.4, 'tags': ['shared_control'],
    },
    # ---- Misspecified (justifies CR2) -------------------------------------
    'Misspec: heavy-tailed (t₃)': {
        'kind': 'misspec_t', 'mu': 0.5, 'tau2': 0.30, 'sigma2': 0.10,
        'rho': 0.5, 'df_t': 3, 'tags': ['misspecified'],
        'note': 'Errors drawn from t₃ instead of Gaussian. CR2 should retain '
                'coverage while model-based degrades.',
    },
    'Misspec: heteroscedastic τ²': {
        'kind': 'misspec_hetero', 'mu': 0.5, 'tau2_min': 0.05, 'tau2_max': 0.80,
        'sigma2': 0.10, 'rho': 0.5, 'tags': ['misspecified'],
        'note': 'True τ² varies across studies. Model assumes constant. '
                'CR2 should still cover; model-based may not.',
    },
    # ---- Null (Type I error) ---------------------------------------------
    'Null (μ=0, for Type I error)': {
        'kind': 'correlated', 'mu': 0.0, 'tau2': 0.30, 'sigma2': 0.10,
        'rho': 0.5, 'tags': ['null'],
    },
}

# =============================================================================
# 3. DATA-GENERATING PROCESSES
# =============================================================================
def _build_correlated_vcv(vi_diag, rho):
    """Off-diagonal = rho * sqrt(vi * vj). Used for correlated effects DGPs."""
    k = len(vi_diag)
    if k == 1 or rho <= 0:
        return np.diag(vi_diag)
    V = np.zeros((k, k))
    for r in range(k):
        for c in range(k):
            V[r, c] = vi_diag[r] if r == c else rho * np.sqrt(vi_diag[r] * vi_diag[c])
    return V

def _ensure_pd(M, floor=1e-10):
    min_eig = np.min(np.linalg.eigvalsh(M))
    if min_eig < floor:
        M = M + (floor - min_eig) * np.eye(M.shape[0])
    return M

def dgp_correlated(scenario, n_studies, seed):
    """Standard DGP: correlated effects with off-diagonals ρ√(vᵢvⱼ).
    DOES NOT use build_vcv_matrices; injects pre-built VCV."""
    rng = np.random.default_rng(seed)
    rho = scenario.get('rho', 0.0)
    k_dist = [0.2, 0.4, 0.3, 0.1]
    tau2, sigma2 = scenario['tau2'], scenario['sigma2']
    rows, vcv = [], {}
    for i in range(n_studies):
        sid = str(i + 1)
        k_i = rng.choice([1, 2, 3, 4], p=k_dist)
        u_i = rng.normal(0, np.sqrt(tau2)) if tau2 > 0 else 0.0
        vi_diag = rng.uniform(0.05, 0.20, size=k_i)
        V_i = _build_correlated_vcv(vi_diag, rho)
        vcv[sid] = V_i
        Sigma_i = _ensure_pd(V_i + sigma2 * np.eye(k_i))
        errs = rng.multivariate_normal(np.zeros(k_i), Sigma_i)
        y_i = scenario['mu'] + u_i + errs
        for j in range(k_i):
            rows.append({'id': sid, 'yi': y_i[j], 'vi': vi_diag[j]})
    return pd.DataFrame(rows), vcv

def dgp_misspec_t(scenario, n_studies, seed):
    """Same as correlated but errors are scaled t_df instead of Gaussian."""
    rng = np.random.default_rng(seed)
    rho = scenario.get('rho', 0.0)
    df_t = scenario.get('df_t', 3)
    k_dist = [0.2, 0.4, 0.3, 0.1]
    tau2, sigma2 = scenario['tau2'], scenario['sigma2']
    rows, vcv = [], {}
    for i in range(n_studies):
        sid = str(i + 1)
        k_i = rng.choice([1, 2, 3, 4], p=k_dist)
        u_i = rng.standard_t(df_t) * np.sqrt(tau2 * (df_t - 2) / df_t) if tau2 > 0 else 0.0
        vi_diag = rng.uniform(0.05, 0.20, size=k_i)
        V_i = _build_correlated_vcv(vi_diag, rho)
        vcv[sid] = V_i
        Sigma_i = _ensure_pd(V_i + sigma2 * np.eye(k_i))
        L = np.linalg.cholesky(Sigma_i)
        # Scale t draws so variance matches Sigma_i
        t_raw = rng.standard_t(df_t, size=k_i) * np.sqrt((df_t - 2) / df_t)
        errs = L @ t_raw
        y_i = scenario['mu'] + u_i + errs
        for j in range(k_i):
            rows.append({'id': sid, 'yi': y_i[j], 'vi': vi_diag[j]})
    return pd.DataFrame(rows), vcv

def dgp_misspec_hetero(scenario, n_studies, seed):
    """τ² varies across studies; model assumes constant."""
    rng = np.random.default_rng(seed)
    rho = scenario.get('rho', 0.0)
    tau2_min, tau2_max = scenario['tau2_min'], scenario['tau2_max']
    sigma2 = scenario['sigma2']
    k_dist = [0.2, 0.4, 0.3, 0.1]
    rows, vcv = [], {}
    for i in range(n_studies):
        sid = str(i + 1)
        k_i = rng.choice([1, 2, 3, 4], p=k_dist)
        tau2_i = rng.uniform(tau2_min, tau2_max)
        u_i = rng.normal(0, np.sqrt(tau2_i))
        vi_diag = rng.uniform(0.05, 0.20, size=k_i)
        V_i = _build_correlated_vcv(vi_diag, rho)
        vcv[sid] = V_i
        Sigma_i = _ensure_pd(V_i + sigma2 * np.eye(k_i))
        errs = rng.multivariate_normal(np.zeros(k_i), Sigma_i)
        y_i = scenario['mu'] + u_i + errs
        for j in range(k_i):
            rows.append({'id': sid, 'yi': y_i[j], 'vi': vi_diag[j]})
    return pd.DataFrame(rows), vcv

def dgp_shared_control(scenario, n_studies, seed):
    """
    Shared-control DGP for Hedges g.
    Generates raw control/treatment statistics, then calls build_vcv_matrices
    to construct the VCV — this validates the FULL VCV pipeline.
    """
    rng = np.random.default_rng(seed)
    mu, tau2, sigma2 = scenario['mu'], scenario['tau2'], scenario['sigma2']
    p_shared = scenario.get('p_shared', 0.4)
    rows = []
    for i in range(n_studies):
        sid = str(i + 1)
        has_shared = rng.random() < p_shared
        n_arms = int(rng.choice([2, 3])) if has_shared else 1

        # Control arm (shared across n_arms if has_shared)
        n_c = int(rng.integers(20, 80))
        xc = float(rng.normal(10.0, 1.0))
        sdc = float(rng.uniform(1.5, 3.0))

        u_i = rng.normal(0, np.sqrt(tau2)) if tau2 > 0 else 0.0
        shared_gid = (i + 1) if has_shared else np.nan

        for j in range(n_arms):
            n_t = int(rng.integers(20, 80))
            v_ij = rng.normal(0, np.sqrt(sigma2)) if sigma2 > 0 else 0.0
            true_d = mu + u_i + v_ij
            # Simulate treatment mean consistent with true_d
            sdt = float(rng.uniform(1.5, 3.0))
            s_pooled = np.sqrt(((n_t-1)*sdt**2 + (n_c-1)*sdc**2) / (n_t+n_c-2))
            xt = xc + true_d * s_pooled
            # Compute g and its sampling variance
            d_obs = (xt - xc) / s_pooled
            df_g = n_t + n_c - 2
            # Hedges J via lgamma
            from scipy.special import gammaln
            log_j = gammaln(df_g/2) - 0.5*np.log(df_g/2) - gammaln((df_g-1)/2)
            J = np.exp(log_j)
            g_obs = J * d_obs
            # Sampling noise: add Gaussian noise scaled to v_g
            v_d = (n_t + n_c)/(n_t * n_c) + d_obs**2/(2*(n_t + n_c))
            v_g = J**2 * v_d
            # Inject sampling error
            g_obs = g_obs + rng.normal(0, np.sqrt(v_g))
            rows.append({
                'id': sid, 'shared_group_id': shared_gid,
                'yi': g_obs, 'vi': v_g,
                'ne': n_t, 'nc': n_c, 'xc': xc, 'sdc': sdc,
            })
    df = pd.DataFrame(rows)
    # Use Co-Meta's build_vcv_matrices to construct the VCV from raw stats
    try:
        vcv = build_vcv_matrices(df, effect_type='hedges_g', var_col_name='vi',
                                 yi_col='yi', nt_col='ne', nc_col='nc',
                                 xc_col='xc', sdc_col='sdc',
                                 check_psd=False, warn_on_fallback=False)
        # Ensure all study IDs present
        for sid in df['id'].unique():
            if sid not in vcv:
                vcv[sid] = np.diag(df[df['id']==sid]['vi'].values)
    except Exception as e:
        # Fallback to diagonal if build_vcv_matrices unavailable
        vcv = {sid: np.diag(df[df['id']==sid]['vi'].values) for sid in df['id'].unique()}
    return df, vcv

DGP_DISPATCH = {
    'correlated':       dgp_correlated,
    'misspec_t':        dgp_misspec_t,
    'misspec_hetero':   dgp_misspec_hetero,
    'shared_control':   dgp_shared_control,
}

# =============================================================================
# 4. SIMULATION RUNNERS
# =============================================================================
def run_python_engine(df, vcv):
    """Call the PRODUCTION ThreeLevelEngine. Returns dict with both z and CR2 CIs."""
    try:
        engine = ThreeLevelEngine()
        res = engine.fit(df=df.copy(), effect_col='yi', var_col='vi', vcv_dict=vcv)
        if res is None:
            return None, 'engine_returned_none'
        mu      = res['mu']
        se      = res.get('se', np.nan)
        se_cr2  = res.get('se_robust', se)
        df_satt = res.get('df_robust', max(1.0, df['id'].nunique() - 1))
        tau2    = res.get('tau2', np.nan)
        sigma2  = res.get('sigma2', np.nan)
        # Two CIs
        ci_z = (mu - 1.96 * se,                            mu + 1.96 * se)
        crit = float(t_dist.ppf(0.975, max(1.0, df_satt)))
        ci_cr2 = (mu - crit * se_cr2,                      mu + crit * se_cr2)
        return {
            'mu': mu, 'se': se, 'se_cr2': se_cr2, 'df_satt': df_satt,
            'tau2': tau2, 'sigma2': sigma2,
            'ci_z_lo': ci_z[0], 'ci_z_hi': ci_z[1],
            'ci_cr2_lo': ci_cr2[0], 'ci_cr2_hi': ci_cr2[1],
        }, None
    except (np.linalg.LinAlgError, ValueError, RuntimeError) as e:
        return None, type(e).__name__ + ': ' + str(e)[:120]

def run_r_engine(df, vcv):
    """Call rma.mv + clubSandwich. Returns dict with both z and CR2 CIs."""
    try:
        df_sorted = df.sort_values('id', kind='mergesort').reset_index(drop=True).copy()
        df_sorted['row_id'] = np.arange(1, len(df_sorted) + 1)
        # Build block-diagonal V in the SAME ordering
        unique_ids = list(df_sorted['id'].unique())
        blocks = []
        for sid in unique_ids:
            if sid in vcv:
                blocks.append(np.asarray(vcv[sid], dtype=np.float64))
            else:
                vi = df_sorted.loc[df_sorted['id']==sid, 'vi'].values
                blocks.append(np.diag(vi))
        V_full = block_diag(*blocks)

        ro.globalenv['df_r']     = df_sorted[['id', 'yi', 'vi', 'row_id']].copy()
        ro.globalenv['V_matrix'] = V_full

        r_script = """
        suppressPackageStartupMessages({
            library(metafor); library(clubSandwich)
        })
        result <- tryCatch({
            dat <- df_r
            dat$study_id <- as.factor(dat$id)
            rownames(V_matrix) <- 1:nrow(dat)
            colnames(V_matrix) <- 1:nrow(dat)
            is_3level <- length(unique(dat$study_id)) < nrow(dat)
            if (is_3level) {
                res <- rma.mv(yi=yi, V=V_matrix, random=~1|study_id/row_id,
                              data=dat, method="REML",
                              control=list(iter.max=200, rel.tol=1e-9))
                tau2_v  <- as.numeric(res$sigma2[1])
                sig2_v  <- as.numeric(res$sigma2[2])
            } else {
                res <- rma.mv(yi=yi, V=V_matrix, random=~1|study_id,
                              data=dat, method="REML",
                              control=list(iter.max=200, rel.tol=1e-9))
                tau2_v  <- as.numeric(res$sigma2[1])
                sig2_v  <- 0
            }
            cr2_test <- coef_test(res, vcov="CR2", cluster=dat$study_id,
                                  test="Satterthwaite")
            cr2_ci   <- conf_int(res, vcov="CR2", cluster=dat$study_id,
                                 test="Satterthwaite")
            list(ok=TRUE,
                 mu=as.numeric(res$b[1]),
                 se=as.numeric(res$se[1]),
                 se_cr2=as.numeric(cr2_test$SE[1]),
                 df_satt=as.numeric(cr2_test$df_Satt[1]),
                 ci_z_lo=as.numeric(res$ci.lb[1]),
                 ci_z_hi=as.numeric(res$ci.ub[1]),
                 ci_cr2_lo=as.numeric(cr2_ci$CI_L[1]),
                 ci_cr2_hi=as.numeric(cr2_ci$CI_U[1]),
                 tau2=tau2_v, sigma2=sig2_v)
        }, error=function(e) list(ok=FALSE, msg=conditionMessage(e)))
        result
        """
        r_out = ro.r(r_script)
        if not bool(r_out.rx2('ok')[0]):
            return None, 'R error: ' + str(r_out.rx2('msg')[0])[:120]
        return {
            'mu':        float(r_out.rx2('mu')[0]),
            'se':        float(r_out.rx2('se')[0]),
            'se_cr2':    float(r_out.rx2('se_cr2')[0]),
            'df_satt':   float(r_out.rx2('df_satt')[0]),
            'tau2':      float(r_out.rx2('tau2')[0]),
            'sigma2':    float(r_out.rx2('sigma2')[0]),
            'ci_z_lo':   float(r_out.rx2('ci_z_lo')[0]),
            'ci_z_hi':   float(r_out.rx2('ci_z_hi')[0]),
            'ci_cr2_lo': float(r_out.rx2('ci_cr2_lo')[0]),
            'ci_cr2_hi': float(r_out.rx2('ci_cr2_hi')[0]),
        }, None
    except Exception as e:
        return None, type(e).__name__ + ': ' + str(e)[:120]

def simulate_dataset(scenario_name, scenario, n_studies, sim_idx):
    """Generate one synthetic dataset deterministically."""
    seed = deterministic_seed(scenario_name, sim_idx)
    dgp_fn = DGP_DISPATCH[scenario['kind']]
    df, vcv = dgp_fn(scenario, scenario.get('n_studies', n_studies), seed)  # per-scenario k
    return df, vcv, seed

# =============================================================================
# 5. MAIN MONTE CARLO LOOP
# =============================================================================

import time

def run_monte_carlo(scenarios_to_run, n_sims, n_studies, n_jobs_py=None,
                    progress_cb=None):
    """
    Sequential Monte Carlo. R and Python run alternately per simulation
    on the same dataset, which keeps memory low and avoids rpy2/joblib
    deadlocks.

    `n_jobs_py` is accepted but ignored (kept for backward compatibility
    with the existing UI). All simulations run on the main thread.
    """
    all_results = []
    py_errors = []
    r_errors  = []

    total_units = sum(n_sims for _ in scenarios_to_run)
    unit = 0
    t_start = time.time()

    for sc_name, sc in scenarios_to_run:
        sc_data = {
            'scenario': sc_name, 'tags': sc.get('tags', []),
            'true_mu':     sc.get('mu', 0.0),
            'true_tau2':   sc.get('tau2', sc.get('tau2_min', np.nan)),
            'true_sigma2': sc.get('sigma2', np.nan),
            'note':        sc.get('note', ''),
            'py': [], 'r': [], 'seeds': [],
        }

        for sim_idx in range(n_sims):
            # 1. Generate dataset (deterministic seed)
            try:
                df, vcv, seed = simulate_dataset(sc_name, sc, n_studies, sim_idx)
                sc_data['seeds'].append(seed)
            except Exception as e:
                py_errors.append({'scenario': sc_name, 'sim': sim_idx,
                                  'error': f'DGP failure: {type(e).__name__}: {str(e)[:120]}'})
                sc_data['py'].append(None)
                sc_data['r'].append(None)
                sc_data['seeds'].append(-1)
                unit += 1
                if progress_cb:
                    elapsed = time.time() - t_start
                    rate = unit / elapsed if elapsed > 0 else 0
                    eta = (total_units - unit) / rate if rate > 0 else 0
                    progress_cb(unit, total_units,
                                f"{sc_name} ({sim_idx+1}/{n_sims}) — ETA {eta:.0f}s")
                continue

            # 2. R engine
            r_res, r_err = run_r_engine(df, vcv)
            if r_err:
                r_errors.append({'scenario': sc_name, 'sim': sim_idx, 'error': r_err})
            sc_data['r'].append(r_res)

            # 3. Python engine
            py_res, py_err = run_python_engine(df, vcv)
            if py_err:
                py_errors.append({'scenario': sc_name, 'sim': sim_idx, 'error': py_err})
            sc_data['py'].append(py_res)

            # 4. Progress + ETA
            unit += 1
            if progress_cb and (sim_idx % max(1, n_sims // 50) == 0 or sim_idx == n_sims - 1):
                elapsed = time.time() - t_start
                rate = unit / elapsed if elapsed > 0 else 0
                eta = (total_units - unit) / rate if rate > 0 else 0
                progress_cb(unit, total_units,
                            f"{sc_name} ({sim_idx+1}/{n_sims}) — ETA {eta:.0f}s")

        all_results.append(sc_data)

    return all_results, py_errors, r_errors

print("✅ run_monte_carlo ready (sequential, no joblib).")
print("   You can now re-run the validation cell.")

# =============================================================================
# 6. SUMMARY STATISTICS
# =============================================================================
def _safe_arr(lst, key):
    return np.array([d[key] if d is not None else np.nan for d in lst], dtype=float)

def summarize_scenario(sc_data, n_sims):
    """Compute bias, MC SE, coverage, etc. for one scenario."""
    py = sc_data['py']
    r  = sc_data['r']
    true_mu  = sc_data['true_mu']
    true_t2  = sc_data['true_tau2']
    true_s2  = sc_data['true_sigma2']

    def mc_bias_se(arr, truth):
        valid = arr[~np.isnan(arr)]
        if len(valid) == 0:
            return np.nan, np.nan, 0
        bias = np.mean(valid) - truth
        mc_se = np.std(valid, ddof=1) / np.sqrt(len(valid))
        return bias, mc_se, len(valid)

    py_mu = _safe_arr(py, 'mu')
    py_t2 = _safe_arr(py, 'tau2')
    py_s2 = _safe_arr(py, 'sigma2')
    py_z_lo = _safe_arr(py, 'ci_z_lo'); py_z_hi = _safe_arr(py, 'ci_z_hi')
    py_c_lo = _safe_arr(py, 'ci_cr2_lo'); py_c_hi = _safe_arr(py, 'ci_cr2_hi')

    r_mu = _safe_arr(r, 'mu')
    r_t2 = _safe_arr(r, 'tau2')
    r_s2 = _safe_arr(r, 'sigma2')
    r_z_lo = _safe_arr(r, 'ci_z_lo'); r_z_hi = _safe_arr(r, 'ci_z_hi')
    r_c_lo = _safe_arr(r, 'ci_cr2_lo'); r_c_hi = _safe_arr(r, 'ci_cr2_hi')

    def coverage(lo, hi, truth):
        valid = ~(np.isnan(lo) | np.isnan(hi))
        if valid.sum() == 0:
            return np.nan, np.nan
        cov = ((lo[valid] <= truth) & (truth <= hi[valid]))
        rate = cov.mean()
        # MC SE for binomial coverage
        mc_se = np.sqrt(rate * (1 - rate) / valid.sum())
        return rate, mc_se

    py_mu_bias, py_mu_se, py_n_ok = mc_bias_se(py_mu, true_mu)
    r_mu_bias,  r_mu_se,  r_n_ok  = mc_bias_se(r_mu,  true_mu)
    py_t2_bias, py_t2_se, _ = mc_bias_se(py_t2, true_t2)
    r_t2_bias,  r_t2_se,  _ = mc_bias_se(r_t2,  true_t2)
    py_s2_bias, py_s2_se, _ = mc_bias_se(py_s2, true_s2)
    r_s2_bias,  r_s2_se,  _ = mc_bias_se(r_s2,  true_s2)

    py_z_cov,  py_z_cov_se  = coverage(py_z_lo,  py_z_hi,  true_mu)
    py_c_cov,  py_c_cov_se  = coverage(py_c_lo,  py_c_hi,  true_mu)
    r_z_cov,   r_z_cov_se   = coverage(r_z_lo,   r_z_hi,   true_mu)
    r_c_cov,   r_c_cov_se   = coverage(r_c_lo,   r_c_hi,   true_mu)

    # Max absolute difference Py-vs-R for paired sims (where both succeeded)
    paired = ~(np.isnan(py_mu) | np.isnan(r_mu))
    max_diff_mu  = np.max(np.abs(py_mu[paired] - r_mu[paired])) if paired.any() else np.nan
    max_diff_t2  = np.max(np.abs(py_t2[paired] - r_t2[paired])) if paired.any() else np.nan
    max_diff_s2  = np.max(np.abs(py_s2[paired] - r_s2[paired])) if paired.any() else np.nan
    py_sc, r_sc = _safe_arr(sc_data['py'], 'se_cr2'), _safe_arr(sc_data['r'], 'se_cr2')
    py_df, r_df = _safe_arr(sc_data['py'], 'df_satt'), _safe_arr(sc_data['r'], 'df_satt')
    max_rel_se_cr2 = np.max(np.abs(py_sc[paired] - r_sc[paired]) / r_sc[paired]) if paired.any() else np.nan
    max_rel_df     = np.max(np.abs(py_df[paired] - r_df[paired]) / r_df[paired]) if paired.any() else np.nan

    return {
        'scenario':   sc_data['scenario'],
        'tags':       ','.join(sc_data['tags']),
        'note':       sc_data['note'],
        'n_sims':     n_sims,
        'n_py_ok':    py_n_ok,
        'n_r_ok':     r_n_ok,
        'true_mu':    true_mu,
        'true_tau2':  true_t2,
        'true_sigma2':true_s2,
        # Bias (estimate - truth)
        'py_mu_bias': py_mu_bias, 'py_mu_se': py_mu_se,
        'r_mu_bias':  r_mu_bias,  'r_mu_se':  r_mu_se,
        'py_tau2_bias': py_t2_bias, 'py_tau2_se': py_t2_se,
        'r_tau2_bias':  r_t2_bias,  'r_tau2_se':  r_t2_se,
        'py_sigma2_bias': py_s2_bias, 'py_sigma2_se': py_s2_se,
        'r_sigma2_bias':  r_s2_bias,  'r_sigma2_se':  r_s2_se,
        # Coverage (z and CR2 on both sides)
        'py_z_cov':   py_z_cov,  'py_z_cov_se':  py_z_cov_se,
        'py_cr2_cov': py_c_cov,  'py_cr2_cov_se':py_c_cov_se,
        'r_z_cov':    r_z_cov,   'r_z_cov_se':   r_z_cov_se,
        'r_cr2_cov':  r_c_cov,   'r_cr2_cov_se': r_c_cov_se,
        # Paired Py-vs-R agreement
        'max_diff_mu':    max_diff_mu,
        'max_diff_tau2':  max_diff_t2,
        'max_diff_sigma2':max_diff_s2,
        'max_rel_se_cr2': max_rel_se_cr2,
        'max_rel_df':     max_rel_df,
    }

# =============================================================================
# 7. RENDERING
# =============================================================================
def _fmt_bias(b, se):
    if np.isnan(b): return '—'
    color = '#155724' if abs(b) < 2*se else ('#856404' if abs(b) < 4*se else '#721c24')
    return f"<span style='color:{color}'>{b:+.4f} (±{se:.4f})</span>"

def _fmt_cov(c, se, band=(0.93, 0.97)):
    if np.isnan(c): return '—'
    pct = c * 100
    if band[0] <= c <= band[1]:    bg = '#d4edda'
    elif 0.90 <= c <= 0.98:        bg = '#fff3cd'
    else:                           bg = '#f8d7da'
    return f"<span style='background-color:{bg};padding:2px 6px;border-radius:3px'>{pct:.1f}% (±{se*100:.1f})</span>"

def _fmt_diff(d, tol):
    if np.isnan(d): return '—'
    color = '#155724' if d <= tol else '#721c24'
    return f"<span style='color:{color}'>{d:.2e}</span>"

def render_results(summaries, meta):
    # ---- Header ----
    header = f"""
    <div style='padding:15px; background:#d4edda; border-left:5px solid #28a745;
                color:#155724; border-radius:4px; font-family:sans-serif;'>
        <h3 style='margin:0 0 8px 0'>✅ Monte Carlo Validation Complete</h3>
        <p style='margin:0; font-size:13px'>
            <b>{meta['n_sims']}</b> simulations × <b>{len(summaries)}</b> scenarios &nbsp;|&nbsp;
            <b>{meta['n_studies']}</b> studies per simulation &nbsp;|&nbsp;
            metafor <b>{meta['metafor_version']}</b>, clubSandwich <b>{meta['clubsandwich_version']}</b> &nbsp;|&nbsp;
            Elapsed: <b>{meta['elapsed_sec']:.1f}s</b>
        </p>
    </div>
    """
    display(HTML(header))

    # ---- Table 1: Bias (with MC SE) ----
    display(HTML("<h4 style='color:#2c3e50;margin-top:20px'>Table 1 — Parameter Recovery (bias ± MC SE)</h4>"
                 "<p style='font-size:12px;color:#666;margin:4px 0'>"
                 "Green = |bias| < 2×MC SE (consistent with zero). "
                 "Yellow = 2–4×MC SE. Red = > 4×MC SE (real bias).</p>"))
    rows1 = []
    for s in summaries:
        rows1.append({
            'Scenario':   s['scenario'],
            'True μ':     f"{s['true_mu']:.2f}",
            'Py μ bias':  _fmt_bias(s['py_mu_bias'], s['py_mu_se']),
            'R μ bias':   _fmt_bias(s['r_mu_bias'],  s['r_mu_se']),
            'Py τ² bias': _fmt_bias(s['py_tau2_bias'], s['py_tau2_se']),
            'R τ² bias':  _fmt_bias(s['r_tau2_bias'],  s['r_tau2_se']),
            'Py σ² bias': _fmt_bias(s['py_sigma2_bias'], s['py_sigma2_se']),
            'R σ² bias':  _fmt_bias(s['r_sigma2_bias'],  s['r_sigma2_se']),
        })
    df1 = pd.DataFrame(rows1)
    display(HTML(df1.to_html(escape=False, index=False)))

    # ---- Table 2: Coverage (apples-to-apples) ----
    display(HTML("<h4 style='color:#2c3e50;margin-top:20px'>Table 2 — 95% CI Coverage (apples-to-apples)</h4>"
                 "<p style='font-size:12px;color:#666;margin:4px 0'>"
                 "Both engines compute both z-based and CR2/Satterthwaite intervals. "
                 "Green = 93–97% (nominal). Yellow = 90–98%. Red = outside.</p>"))
    rows2 = []
    for s in summaries:
        rows2.append({
            'Scenario':    s['scenario'],
            'Py z':        _fmt_cov(s['py_z_cov'],   s['py_z_cov_se']),
            'R z':         _fmt_cov(s['r_z_cov'],    s['r_z_cov_se']),
            'Py CR2':      _fmt_cov(s['py_cr2_cov'], s['py_cr2_cov_se']),
            'R CR2':       _fmt_cov(s['r_cr2_cov'],  s['r_cr2_cov_se']),
        })
    df2 = pd.DataFrame(rows2)
    display(HTML(df2.to_html(escape=False, index=False)))

    # ---- Table 3: Paired Py-vs-R agreement ----
    display(HTML("<h4 style='color:#2c3e50;margin-top:20px'>Table 3 — Py vs R agreement (paired, per-simulation max |Δ|)</h4>"
                 "<p style='font-size:12px;color:#666;margin:4px 0'>"
                 f"Tolerances: μ < {TOLERANCES['mu']:.0e}, τ² < {TOLERANCES['tau2']:.0e}, σ² < {TOLERANCES['sigma2']:.0e}. "
                 "Green = under tolerance.</p>"))
    rows3 = []
    for s in summaries:
        rows3.append({
            'Scenario':    s['scenario'],
            'max |Δμ|':    _fmt_diff(s['max_diff_mu'],    TOLERANCES['mu']),
            'max |Δτ²|':   _fmt_diff(s['max_diff_tau2'],  TOLERANCES['tau2']),
            'max |Δσ²|':   _fmt_diff(s['max_diff_sigma2'],TOLERANCES['sigma2']),
            'Py OK':       f"{s['n_py_ok']}/{s['n_sims']}",
            'R OK':        f"{s['n_r_ok']}/{s['n_sims']}",
        })
    df3 = pd.DataFrame(rows3)
    display(HTML(df3.to_html(escape=False, index=False)))

    # ---- Boundary / misspec notes ----
    notes_html = ""
    for s in summaries:
        if s['note']:
            notes_html += f"<li><b>{s['scenario']}:</b> {s['note']}</li>"
    if notes_html:
        display(HTML(f"""
        <div style='margin-top:20px;padding:12px;background:#f8f9fa;border-left:4px solid #6c757d;border-radius:4px'>
            <b>Scenario notes</b><ul style='margin:8px 0 0 0'>{notes_html}</ul>
        </div>"""))

# =============================================================================
# 8. PASS/FAIL ASSERTIONS (programmatic, for CI)
# =============================================================================
def run_assertions(summaries):
    failures = []
    for s in summaries:
        # Failed fits are dropped from bias/coverage, so too many of them
        # would hide a problem: allow at most 1% of replicates.
        n_fail_py = s['n_sims'] - s['n_py_ok']
        if n_fail_py > max(1, 0.01 * s['n_sims']):
            failures.append(f"{s['scenario']}: Python fitted {s['n_py_ok']}/{s['n_sims']} replicates "
                            f"(R: {s['n_r_ok']}/{s['n_sims']})")
        if 'misspecified' in s['tags'] or 'boundary' in s['tags']:
            # Don't assert paired Py-vs-R for misspec/boundary; they may diverge
            continue
        if not np.isnan(s['max_diff_mu']) and s['max_diff_mu'] > TOLERANCES['mu']:
            failures.append(f"{s['scenario']}: max |Δμ| = {s['max_diff_mu']:.2e} > {TOLERANCES['mu']:.0e}")
        for key, tol, lab in (('max_diff_tau2', 'tau2', '|Δτ²|'), ('max_diff_sigma2', 'sigma2', '|Δσ²|'),
                              ('max_rel_se_cr2', 'se_cr2', 'rel. Δ CR2 SE'), ('max_rel_df', 'df_satt', 'rel. Δ df')):
            if not np.isnan(s[key]) and s[key] > TOLERANCES[tol]:
                failures.append(f"{s['scenario']}: max {lab} = {s[key]:.2e} > {TOLERANCES[tol]:.0e}")
        # CR2 coverage check (Co-Meta only; nominal 95%)
        if not np.isnan(s['py_cr2_cov']):
            lo, hi = TOLERANCES['coverage_band']
            if not (lo - 2*s['py_cr2_cov_se'] <= s['py_cr2_cov'] <= hi + 2*s['py_cr2_cov_se']):
                failures.append(f"{s['scenario']}: Py CR2 coverage {s['py_cr2_cov']*100:.1f}% "
                                f"outside [{lo*100:.0f}, {hi*100:.0f}]% (±2 MC SE)")
    return failures

# =============================================================================
# 9. UI
# =============================================================================
preset_w   = widgets.RadioButtons(options=list(PRESETS.keys()), value='Standard',
                                  description='Preset:', layout=widgets.Layout(width='400px'))
scenario_w = widgets.SelectMultiple(options=list(SCENARIOS.keys()),
                                    value=tuple(SCENARIOS.keys()),
                                    description='Scenarios:',
                                    layout=widgets.Layout(width='600px', height='250px'))
n_jobs_w   = widgets.IntSlider(value=4, min=1, max=8, step=1,
                               description='Python jobs:',
                               layout=widgets.Layout(width='400px'))
save_json_w= widgets.Checkbox(value=True, description='Save JSON sidecar')
run_btn    = widgets.Button(description='▶ Run Comprehensive Validation',
                            button_style='success',
                            layout=widgets.Layout(width='400px', height='40px'))
out_area   = widgets.Output()

display(HTML("""
<div style='background:#f8f9fa;padding:15px;border-radius:5px;border-left:5px solid #2c3e50;margin-bottom:15px;font-family:sans-serif'>
    <h3 style='margin:0 0 8px 0;color:#2c3e50'>Comprehensive Engine Validation</h3>
    <p style='margin:0;font-size:13px'>
        Validates the production <code>ThreeLevelEngine.fit()</code> against
        <code>metafor::rma.mv</code> + <code>clubSandwich::vcovCR(type="CR2")</code>
        across correctly specified, shared-control, misspecified, and boundary scenarios.
        Reports bias (with MC SE), apples-to-apples coverage (z and CR2 on both sides),
        and paired Py-vs-R agreement. For publication, use the <b>Publication</b> preset.
    </p>
</div>
"""))
display(widgets.VBox([preset_w, scenario_w, n_jobs_w, save_json_w, widgets.HTML('<hr>'), run_btn]),
        out_area)

# =============================================================================
# 10. EXECUTION
# =============================================================================
def on_run_clicked(_):
    with out_area:
        clear_output(wait=True)
        cfg     = PRESETS[preset_w.value]
        n_sims  = cfg['n_sims']
        n_stud  = cfg['n_studies']
        n_jobs  = n_jobs_w.value
        chosen  = [(name, SCENARIOS[name]) for name in scenario_w.value]
        if not chosen:
            display(HTML("<div style='color:red'>Select at least one scenario.</div>"))
            return

        # Progress UI
        prog = widgets.IntProgress(value=0, min=0,
                                   max=len(chosen)*n_sims,
                                   description='Progress:',
                                   layout=widgets.Layout(width='500px'))
        status = widgets.HTML("<i>Starting…</i>")
        display(prog, status)

        def progress_cb(unit, total, msg):
            prog.value = unit
            status.value = f"<i>{msg}</i>"

        t0 = time.time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                all_results, py_errs, r_errs = run_monte_carlo(
                    chosen, n_sims, n_stud, n_jobs_py=n_jobs,
                    progress_cb=progress_cb
                )
            except Exception as e:
                clear_output()
                import traceback
                display(HTML(f"<pre style='color:#721c24;background:#f8d7da;padding:10px'>{traceback.format_exc()}</pre>"))
                return
        elapsed = time.time() - t0

        summaries = [summarize_scenario(d, n_sims) for d in all_results]

        # Metadata
        try:
            metafor_v = str(ro.r('as.character(packageVersion("metafor"))')[0])
        except Exception:
            metafor_v = 'unknown'
        try:
            clubs_v = str(ro.r('as.character(packageVersion("clubSandwich"))')[0])
        except Exception:
            clubs_v = 'unknown'
        meta = {
            'timestamp':            _dt.utcnow().isoformat() + 'Z',
            'n_sims':               n_sims,
            'n_studies':            n_stud,
            'n_scenarios':          len(chosen),
            'metafor_version':      metafor_v,
            'clubsandwich_version': clubs_v,
            'python_version':       sys.version.split()[0],
            'elapsed_sec':          elapsed,
            'tolerances':           {k: v for k, v in TOLERANCES.items() if k != 'coverage_band'},
            'coverage_band':        TOLERANCES['coverage_band'],
        }

        clear_output()
        render_results(summaries, meta)

        # Error log
        if py_errs or r_errs:
            n_py = len(py_errs); n_r = len(r_errs)
            display(HTML(f"""
            <details style='margin-top:15px'>
                <summary><b>⚠️ Errors logged:</b> Python {n_py}, R {n_r}</summary>
                <pre style='font-size:11px;background:#f8f9fa;padding:8px;max-height:300px;overflow:auto'>
Python errors (first 20):
{json.dumps(py_errs[:20], indent=2)}

R errors (first 20):
{json.dumps(r_errs[:20], indent=2)}
                </pre>
            </details>
            """))

        # Assertions
        failures = run_assertions(summaries)
        if 'record_validation' in globals():
            record_validation(2, "Overall Model Monte Carlo — Bias, Coverage, Robustness", [
                (s['scenario'], 'INFO' if ('misspecified' in s['tags'] or 'boundary' in s['tags'])
                 else ('FAIL' if any(f.startswith(f"{s['scenario']}:") for f in failures) else 'PASS'))
                for s in summaries],
                notes=f"{preset_w.value}: {n_sims} sims x {n_stud} studies. " + '; '.join(failures))
        if failures:
            display(HTML(f"""
            <div style='margin-top:15px;padding:12px;background:#f8d7da;border-left:4px solid #dc3545;border-radius:4px;color:#721c24'>
                <b>❌ Assertion failures ({len(failures)}):</b>
                <ul style='margin:8px 0 0 0'>{''.join(f'<li>{f}</li>' for f in failures)}</ul>
            </div>"""))
        else:
            display(HTML("""
            <div style='margin-top:15px;padding:12px;background:#d4edda;border-left:4px solid #28a745;border-radius:4px;color:#155724'>
                <b>✅ All validation assertions passed</b> (tolerances + coverage band)
            </div>"""))

        # JSON sidecar
        if save_json_w.value:
            record = {
                'meta':       meta,
                'summaries':  [{k: (None if isinstance(v, float) and np.isnan(v) else v)
                                for k, v in s.items() if k != 'tags' or isinstance(v, str)}
                               for s in summaries],
                'py_errors':  py_errs,
                'r_errors':   r_errs,
                'assertion_failures': failures,
            }
            fname = f"cometa_validation_{_dt.utcnow().strftime('%Y%m%dT%H%M%SZ')}.json"
            try:
                with open(fname, 'w') as f:
                    json.dump(record, f, indent=2, default=str)
                display(HTML(f"<p style='font-size:12px;color:#666'>📁 Sidecar written: <code>{fname}</code></p>"))
            except Exception as e:
                display(HTML(f"<p style='color:#856404'>Could not save JSON: {e}</p>"))

run_btn.on_click(on_run_clicked)

In [ ]:
#@title 🧪 Validation 3: Subgroup Analysis CR2 (Python vs clubSandwich)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

import pandas as pd
import numpy as np
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import rpy2.robjects.numpy2ri as numpy2ri
from scipy.linalg import block_diag
from IPython.display import display, HTML, clear_output
import logging

# Suppress rpy2 console warnings
logging.getLogger('rpy2.rinterface_lib.callbacks').setLevel(logging.ERROR)

# Activate R conversion
pandas2ri.activate()
numpy2ri.activate()

display(HTML("<div style='padding: 15px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; border-radius: 4px; font-family: sans-serif;'>⏳ <b>Running Robust CR2 Subgroup Validation with R (clubSandwich)...</b> Please wait.</div>"))

try:
    if 'ANALYSIS_CONFIG' not in globals() or 'subgroup_results' not in ANALYSIS_CONFIG:
        raise Exception("ANALYSIS_CONFIG or Subgroup Results not found. Please run the Subgroup Analysis first.")

    # --- 1. Get Exact Data & Matrices from Python Engine ---
    dm = SubgroupDataManager(ANALYSIS_CONFIG)
    bundle = dm.get_subgroup_data()

    df_exact = bundle.df
    X_exact = bundle.X
    feature_names = bundle.feature_names
    ref_level = bundle.reference_level
    py_levels = [str(lv) for lv in bundle.subgroup_levels]

    y_all, vcv_full, vcv_diag, X_blocks, has_shared = dm.prepare_three_level_data(df_exact, X_exact)

    # prepare_three_level_data stable-sorts by 'id' internally to build
    # y_all/vcv_full/X_blocks, but does NOT reorder df_exact/X_exact
    # themselves. Reproduce that same reordering here so the frame we hand
    # to R (df_r below) is row-aligned with the block-diagonal V we hand to
    # R, regardless of the raw dataset's original order.
    _order = np.argsort(df_exact['id'].to_numpy(), kind='stable')
    df_exact = df_exact.iloc[_order].reset_index(drop=True)
    X_exact = X_exact[_order, :]

    # Safely build full VCV
    vcv_full_fixed = []
    for y_vec, vcv_mat, v_diag_mat in zip(y_all, vcv_full, vcv_diag):
        if vcv_mat.shape[0] != len(y_vec):
            vcv_full_fixed.append(v_diag_mat)
        else:
            vcv_full_fixed.append(vcv_mat)

    # --- 2. Get Exact Python CR2 Estimates ---
    analyzer = SubgroupAnalyzer(
        effect_col=dm.effect_col, var_col=dm.var_col,
        global_settings=dm.global_settings, es_config=dm.es_config
    )

    n_total = len(df_exact)
    m_studies = df_exact['id'].nunique()

    # Run full Python pipeline. Co-Meta automatically computes CR2 matrices internally.
    py_est = analyzer.run_three_level_analysis(
        y_all=y_all, vcv_full=vcv_full_fixed, vcv_diag=vcv_diag, X_blocks=X_blocks,
        has_shared_controls=has_shared, n_total=n_total, m_studies=m_studies
    )

    if py_est is None:
        raise Exception("Python engine failed to converge on the data.")

    if not py_est.get('used_robust_se', False):
        raise Exception("Python engine did not output Robust SEs. The matrix may have been singular.")

    py_tau2 = float(py_est['tau_sq'])
    py_sigma2 = float(py_est['sigma_sq'])
    py_betas = py_est['betas']

    # Extract native Python CR2 standard errors and Satterthwaite DFs
    py_ses_cr2 = py_est['se_betas']
    py_dfs_cr2 = py_est['dfs']

    # --- 2a. Build R's V from the BLOCKS PYTHON ACTUALLY FITTED -------
    # run_three_level_analysis() chooses at run time between the full
    # sampling VCVs (Plan A, shared controls) and the diagonal-only ones
    # (Plan B, independence), and reports the chosen list as 'vcv_used'.
    # Handing R anything else means the two engines are fitting DIFFERENT
    # Sigma matrices, which shows up as a beta/SE/QM gap even when tau^2
    # and sigma^2 are pinned to Python's values -- precisely the symptom
    # this cell exists to detect, so it must not be the cell's own doing.
    vcv_used = py_est.get('vcv_used') or vcv_full_fixed
    try:
        V_matrix_exact = block_diag(*[
            np.atleast_2d(np.asarray(m, dtype=np.float64)) for m in vcv_used
        ]).astype(np.float64)
    except Exception as e:
        print(f"Warning: block_diag failed ({e}), using diagonal variance matrix.")
        V_matrix_exact = np.diag(df_exact[dm.var_col].values).astype(np.float64)

    # --- 2a-bis. Hard alignment checks: Python internals vs R's inputs -
    # Everything below must hold element-wise, not just in length. A
    # failure here means the frame/design/Sigma handed to R do not
    # describe the same rows Python fitted, which is the only way an
    # exactly-pinned GLS fit can disagree between the two engines.
    y_flat = np.concatenate([np.asarray(v, dtype=np.float64).ravel() for v in y_all])
    X_flat = np.vstack([np.asarray(b, dtype=np.float64) for b in X_blocks])
    eff_vals = df_exact[dm.effect_col].to_numpy(dtype=np.float64)
    var_vals = df_exact[dm.var_col].to_numpy(dtype=np.float64)

    align_problems = []
    if V_matrix_exact.shape != (n_total, n_total):
        align_problems.append(
            f"V is {V_matrix_exact.shape}, expected ({n_total}, {n_total})"
        )
    if y_flat.shape != eff_vals.shape or not np.allclose(y_flat, eff_vals, rtol=1e-10, atol=1e-12):
        align_problems.append(
            "the effect sizes Python fitted are not row-aligned with df_r"
            f"['{dm.effect_col}']"
        )
    if X_flat.shape != X_exact.shape or not np.allclose(X_flat, X_exact, rtol=1e-10, atol=1e-12):
        align_problems.append("the stacked design-matrix blocks are not row-aligned with df_r")
    if (V_matrix_exact.shape[0] == len(var_vals)
            and not np.allclose(np.diag(V_matrix_exact), var_vals, rtol=1e-8, atol=1e-12)):
        n_bad = int(np.sum(~np.isclose(np.diag(V_matrix_exact), var_vals, rtol=1e-8, atol=1e-12)))
        align_problems.append(
            f"diag(V) disagrees with df_r['{dm.var_col}'] on {n_bad} of "
            f"{len(var_vals)} rows -- the cached per-study sampling VCVs do "
            "not describe these rows (row order / filtering mismatch)"
        )
    if align_problems:
        raise Exception(
            "Python/R data handoff is misaligned, so any difference below "
            "would be an artefact of this cell rather than of the engines: "
            + "; ".join(align_problems)
        )

    # Calculate Python's exact Wald QM test using the CR2 covariance matrix
    py_cov_cr2 = py_est['cov_beta']
    if len(py_betas) > 1:
        idx = list(range(1, len(py_betas)))
        b_sub = py_betas[idx]
        V_sub = py_cov_cr2[np.ix_(idx, idx)]
        py_qm_cr2 = float(b_sub.T @ np.linalg.inv(V_sub) @ b_sub)
    else:
        py_qm_cr2 = 0.0

    # --- 2a-ter. Cluster-robust SMALL-SAMPLE omnibus (AHZ / HTZ) ---------
    # Q_M above is the asymptotic chi-square; Step 9 now also reports the
    # approximate Hotelling-Zhang F, which is the statistic the narrative
    # verdict follows. Validate it here by calling the SAME production
    # function the pipeline calls -- _omnibus_f_from_wald -- with the same
    # subgroup_fit that analyze_all_subgroups() builds, so this checks the
    # shipped code path and not a look-alike.
    py_p_params = int(len(py_betas))
    py_subgroup_fit = {
        'tau_squared': py_tau2,
        'sigma_squared': py_sigma2,
        'n_studies': int(m_studies),
        'p_params': py_p_params,
        'dfs': py_dfs_cr2,
        'htz_wald': py_est.get('htz_wald'),
        'contrast_matrix': (np.eye(py_p_params)[1:, :]
                            if py_p_params > 1 else None),
    }
    py_f_om, py_df_f_num, py_df_f_den, py_p_f_om = analyzer._omnibus_f_from_wald(
        qm=py_qm_cr2, df_qm=max(0, py_p_params - 1),
        subgroup_fit=py_subgroup_fit,
    )
    py_has_htz = (py_subgroup_fit['htz_wald'] is not None
                  and py_subgroup_fit['contrast_matrix'] is not None
                  and np.isfinite(py_f_om) and py_df_f_den > 0)

    # Cross-check against the number Step 9 actually saved, when that run is
    # still in ANALYSIS_CONFIG. A mismatch means this cell is validating
    # something the user never saw.
    _saved = ANALYSIS_CONFIG.get('subgroup_results', {}) or {}
    _saved_f = _saved.get('F_omnibus')
    if (py_has_htz and _saved_f is not None and np.isfinite(_saved_f)
            and not np.isclose(_saved_f, py_f_om, rtol=1e-8, atol=1e-10)):
        raise Exception(
            f"The omnibus F rebuilt here ({py_f_om:.6f}) disagrees with the "
            f"one Step 9 saved ({_saved_f:.6f}). Re-run Step 9 on the current "
            "data before trusting this comparison."
        )

    # --- 2b. Python Null (Intercept-Only) Model on the SAME Filtered Subset ---
    # Refits an intercept-only model on the exact same y_all/vcv_full/vcv_diag
    # blocks used for the subgroup (full) model above. This mirrors the fix
    # applied to Cell 9 (Subgroup Analysis: Execution): Pseudo-R^2 must be a
    # nested-model comparison on IDENTICAL data -- the null model must NOT be
    # borrowed from the global Overall Meta-Analysis result, which is fit on
    # the full (unfiltered) dataset, not the subgroup-eligible subset.
    null_X_blocks = [X_j[:, :1] for X_j in X_blocks]
    py_null_est = analyzer.run_three_level_analysis(
        y_all=y_all, vcv_full=vcv_full_fixed, vcv_diag=vcv_diag,
        X_blocks=null_X_blocks, has_shared_controls=has_shared,
        n_total=n_total, m_studies=m_studies
    )

    if py_null_est is None:
        raise Exception("Python null (intercept-only) model failed to converge on the subgroup subset.")

    # The null fit re-runs the same Plan A / Plan B selection, so it could
    # in principle land on a different sampling-VCV list than the full fit.
    # R's free-REML null model is fitted against the single V_matrix built
    # above, so a divergence here would make the Table-2 comparison invalid.
    null_vcv_used = py_null_est.get('vcv_used') or vcv_full_fixed
    if len(null_vcv_used) != len(vcv_used) or not all(
        a.shape == b.shape and np.allclose(a, b, rtol=1e-10, atol=1e-14)
        for a, b in zip(null_vcv_used, vcv_used)
    ):
        raise Exception(
            "Python's null and full fits used different sampling-VCV blocks "
            f"({py_null_est.get('model_type')} vs {py_est.get('model_type')}), "
            "so a single V_matrix cannot represent both in R."
        )

    py_tau2_null = float(py_null_est['tau_sq'])
    py_sigma2_null = float(py_null_est['sigma_sq'])

    def _pseudo_r2(tau2_n, sigma2_n, tau2_f, sigma2_f):
        null_total = tau2_n + sigma2_n
        full_total = tau2_f + sigma2_f
        if null_total <= 0:
            return 0.0
        return max(0.0, min(1.0, 1.0 - (full_total / null_total))) * 100.0

    py_r2_recalc = _pseudo_r2(py_tau2_null, py_sigma2_null, py_tau2, py_sigma2)

    # Cross-check against the number actually saved by Cell 9's last run
    # (should match unless the underlying data changed since then).
    py_r2_saved = float(
        ANALYSIS_CONFIG.get('subgroup_results', {}).get('R_squared', np.nan)
    )

    # Did either fit use the one-effect-per-study 2-level model (sigma² = 0)?
    # Should be False for both whenever at least one study has >1 effect
    # size -- i.e. sigma^2 is identifiable from within-study replication.
    py_null_regularized = bool(py_null_est.get('variance_regularized', True))
    py_full_regularized = bool(py_est.get('variance_regularized', True))

    # --- 3. Pass to R ---
    ro.globalenv['df_r'] = df_exact
    ro.globalenv['V_matrix'] = V_matrix_exact
    ro.globalenv['eff_col'] = dm.effect_col
    ro.globalenv['py_tau2'] = py_tau2
    ro.globalenv['py_sigma2'] = py_sigma2
    ro.globalenv['ref_level'] = str(ref_level)
    # Python fixes the contrast column order with sorted() (code-point
    # order); R's factor() would derive its own order from the session
    # locale's collation, which can differ (especially for two-way labels
    # containing ' | '). Hand R the exact level order so the k-1 contrast
    # columns line up one-for-one with feature_names.
    ro.globalenv['py_levels'] = ro.StrVector(py_levels)

    r_script = """
    suppressPackageStartupMessages(library(metafor))
    suppressPackageStartupMessages(library(clubSandwich))

    dat <- df_r
    dat$row_id <- 1:nrow(dat)
    dat$study_id <- as.factor(dat$id)
    # Level ORDER (not just the reference) is taken from Python, so the
    # contrast columns cannot be silently permuted by locale collation.
    dat$subgroup <- factor(as.character(dat$subgroup_label), levels = py_levels)
    if (any(is.na(dat$subgroup))) {
        stop("subgroup_label contains levels absent from Python's level list.")
    }

    # Align reference level with Python
    dat$subgroup <- relevel(dat$subgroup, ref = ref_level)

    is_multilevel <- length(unique(dat$study_id)) < nrow(dat)
    random_form <- if(is_multilevel) ~ 1 | study_id/row_id else ~ 1 | study_id
    fixed_vars <- if(is_multilevel) c(py_tau2, py_sigma2) else py_tau2

    tryCatch({
        # Fit standard model
        res <- suppressWarnings(rma.mv(yi = dat[[eff_col]], V = V_matrix, mods = ~ subgroup,
                      random = random_form, data = dat, method = "REML", test = "z",
                      sigma2 = fixed_vars))

        # Apply clubSandwich CR2
        V_cr2 <- vcovCR(res, type="CR2")
        ct <- coef_test(res, vcov=V_cr2)

        # Calculate Robust Wald QM Test exactly like Python
        QM_cr2 <- 0
        if (length(res$b) > 1) {
            idx <- 2:length(res$b)
            b_sub <- res$b[idx]
            V_sub <- V_cr2[idx, idx]
            QM_cr2 <- as.numeric(t(b_sub) %*% solve(V_sub) %*% b_sub)
        }

        # --- Cluster-robust SMALL-SAMPLE omnibus (HTZ) -------------------
        # Same hypothesis as QM_cr2 (all k-1 moderator contrasts jointly),
        # but the approximate Hotelling-Zhang F rather than the asymptotic
        # chi-square. The constraint matrix is built HERE, from R's own
        # coefficient count, so this is an independent statement of the
        # hypothesis and not a replay of Python's matrix.
        F_htz <- NA; df_htz_num <- NA; df_htz_den <- NA; p_htz <- NA
        if (length(res$b) > 1) {
            p_r <- length(res$b)
            C_omnibus <- cbind(rep(0, p_r - 1), diag(p_r - 1))
            wt <- as.data.frame(Wald_test(res, constraints = C_omnibus,
                                          vcov = V_cr2, test = "HTZ"))
            nm <- names(wt)
            F_htz      <- as.numeric(wt[[if ("Fstat" %in% nm) "Fstat" else "F"]])
            df_htz_num <- if ("df_num" %in% nm) as.numeric(wt[["df_num"]])
                          else (p_r - 1)
            df_htz_den <- if ("df_denom" %in% nm) as.numeric(wt[["df_denom"]])
                          else as.numeric(wt[["df"]])
            p_htz      <- as.numeric(wt[["p_val"]])
        }

        # --- Pseudo-R^2 validation: fit NULL (intercept-only) and FULL
        # (subgroup) models with sigma2 estimated FREELY (NOT pinned to
        # Python's values, unlike `res` above), both on the identical
        # filtered dataset. This checks Co-Meta's variance-component
        # estimation itself, not just CR2 given fixed variance components.
        #
        # rma.mv's default optimizer (nlminb) can under-converge when
        # tau^2 and sigma^2 are of similar magnitude -- a slow ridge in
        # the REML likelihood surface. Try optim()/BFGS (base R, tightened
        # tolerances -- generally more robust than nlminb on flat ridges)
        # first, falling back to the default nlminb optimizer only if that
        # errors. mods is written literally at each rma.mv() call site
        # (not passed through a wrapper function/variable) to avoid R's
        # classic non-standard-evaluation formula-capture pitfall, which
        # silently broke every attempt in the previous version of this cell.
        res_null_free <- tryCatch(
            suppressWarnings(rma.mv(yi = dat[[eff_col]], V = V_matrix,
                          random = random_form, data = dat,
                          method = "REML", test = "z",
                          control = list(optimizer = "optim", optmethod = "BFGS",
                                          maxit = 10000, reltol = 1e-12))),
            error = function(e) NULL)
        if (is.null(res_null_free)) {
            res_null_free <- tryCatch(
                suppressWarnings(rma.mv(yi = dat[[eff_col]], V = V_matrix,
                              random = random_form, data = dat,
                              method = "REML", test = "z",
                              control = list(iter.max = 10000, eval.max = 10000, rel.tol = 1e-10))),
                error = function(e) NULL)
        }

        res_full_free <- tryCatch(
            suppressWarnings(rma.mv(yi = dat[[eff_col]], V = V_matrix, mods = ~ subgroup,
                          random = random_form, data = dat,
                          method = "REML", test = "z",
                          control = list(optimizer = "optim", optmethod = "BFGS",
                                          maxit = 10000, reltol = 1e-12))),
            error = function(e) NULL)
        if (is.null(res_full_free)) {
            res_full_free <- tryCatch(
                suppressWarnings(rma.mv(yi = dat[[eff_col]], V = V_matrix, mods = ~ subgroup,
                              random = random_form, data = dat,
                              method = "REML", test = "z",
                              control = list(iter.max = 10000, eval.max = 10000, rel.tol = 1e-10))),
                error = function(e) NULL)
        }

        if (is.null(res_null_free) || is.null(res_full_free)) {
            stop("Free-REML null/full model fitting failed with both optim() and nlminb.")
        }

        tau2_null_r    <- res_null_free$sigma2[1]
        sigma2_null_r  <- if (is_multilevel) res_null_free$sigma2[2] else 0
        tau2_full_r    <- res_full_free$sigma2[1]
        sigma2_full_r  <- if (is_multilevel) res_full_free$sigma2[2] else 0

        null_total_r <- tau2_null_r + sigma2_null_r
        full_total_r <- tau2_full_r + sigma2_full_r
        r2_r <- if (null_total_r > 0) max(0, min(1, 1 - (full_total_r / null_total_r))) * 100 else 0

        list(ok=TRUE,
             coef_names=rownames(res$b),
             b=as.numeric(res$b),
             se_cr2=as.numeric(ct$SE),
             df_cr2=as.numeric(ct$df),
             QM_cr2=QM_cr2,
             F_htz=F_htz,
             df_htz_num=df_htz_num,
             df_htz_den=df_htz_den,
             p_htz=p_htz,
             tau2_null_r=tau2_null_r,
             sigma2_null_r=sigma2_null_r,
             tau2_full_r=tau2_full_r,
             sigma2_full_r=sigma2_full_r,
             r2_r=r2_r)

    }, error = function(e) list(ok=FALSE, msg=as.character(e)))
    """

    r_out = ro.r(r_script)
    if not r_out.rx2('ok')[0]:
        raise Exception(f"R Error: {r_out.rx2('msg')[0]}")

    # --- 4. Extract and Compare ---
    # Verify R's coefficients are the SAME parameters, in the SAME order,
    # as Python's feature_names -- a permuted contrast column would
    # produce plausible-looking but wrong row-by-row comparisons.
    r_coef_names = [str(nm) for nm in r_out.rx2('coef_names')]
    expected_coef_names = ['intrcpt'] + [
        f'subgroup{lv}' for lv in py_levels if lv != str(ref_level)
    ]
    if r_coef_names != expected_coef_names:
        raise Exception(
            "R's coefficient order does not match Python's design matrix. "
            f"R: {r_coef_names} | expected from feature_names "
            f"{feature_names}: {expected_coef_names}"
        )

    r_betas = list(r_out.rx2('b'))
    r_ses_cr2 = list(r_out.rx2('se_cr2'))
    r_dfs_cr2 = list(r_out.rx2('df_cr2'))
    r_qm_cr2 = r_out.rx2('QM_cr2')[0]

    def _r_scalar(key):
        try:
            val = float(r_out.rx2(key)[0])
        except Exception:
            return np.nan
        return val if np.isfinite(val) else np.nan

    r_f_htz = _r_scalar('F_htz')
    r_df_htz_num = _r_scalar('df_htz_num')
    r_df_htz_den = _r_scalar('df_htz_den')
    r_p_htz = _r_scalar('p_htz')

    r_tau2_null = float(r_out.rx2('tau2_null_r')[0])
    r_sigma2_null = float(r_out.rx2('sigma2_null_r')[0])
    r_tau2_full = float(r_out.rx2('tau2_full_r')[0])
    r_sigma2_full = float(r_out.rx2('sigma2_full_r')[0])
    r_r2 = float(r_out.rx2('r2_r')[0])

    comparison_rows = []

    # Add Coefficients
    for i, name in enumerate(feature_names):
        disp_name = "Intercept (Ref Group)" if i == 0 else name.replace("subgroup[", "Contrast: ").replace("]", "")

        py_b = py_betas[i] if i < len(py_betas) else np.nan
        r_b = r_betas[i] if i < len(r_betas) else np.nan
        comparison_rows.append({'Parameter': f'{disp_name} (β)', 'Co-Meta (Python)': py_b, 'clubSandwich (R)': r_b})

        py_se = py_ses_cr2[i] if i < len(py_ses_cr2) else np.nan
        r_se = r_ses_cr2[i] if i < len(r_ses_cr2) else np.nan
        comparison_rows.append({'Parameter': f'{disp_name} (SE CR2)', 'Co-Meta (Python)': py_se, 'clubSandwich (R)': r_se})

        py_df = py_dfs_cr2[i] if i < len(py_dfs_cr2) else np.nan
        r_df = r_dfs_cr2[i] if i < len(r_dfs_cr2) else np.nan
        comparison_rows.append({'Parameter': f'{disp_name} (DF Satt)', 'Co-Meta (Python)': py_df, 'clubSandwich (R)': r_df})

    comparison_rows.append({'Parameter': 'Robust Omnibus Wald Test (QM CR2)', 'Co-Meta (Python)': py_qm_cr2, 'clubSandwich (R)': r_qm_cr2})

    # Small-sample omnibus: the statistic Step 9's narrative verdict follows.
    # Q_M above is its asymptotic counterpart; both are reported in Step 9, so
    # both are checked here.
    if py_has_htz:
        for label, py_val, r_val in (
            ('Omnibus HTZ F statistic',  py_f_om,      r_f_htz),
            ('Omnibus HTZ df numerator', float(py_df_f_num), r_df_htz_num),
            ('Omnibus HTZ df denominator', py_df_f_den, r_df_htz_den),
            ('Omnibus HTZ p-value',      py_p_f_om,    r_p_htz),
        ):
            comparison_rows.append({'Parameter': label,
                                    'Co-Meta (Python)': py_val,
                                    'clubSandwich (R)': r_val})

    df_comp = pd.DataFrame(comparison_rows)
    df_comp['Diff'] = np.abs(df_comp['Co-Meta (Python)'] - df_comp['clubSandwich (R)'])
    _v3_checks = df_comp.assign(status=['PASS' if d < 1e-3 else ('N/A' if pd.isna(d) else 'FAIL')
                                        for d in df_comp['Diff']])

    for col in ['Co-Meta (Python)', 'clubSandwich (R)']:
        df_comp[col] = df_comp[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "N/A")
    df_comp['Diff'] = df_comp['Diff'].apply(lambda x: f"{x:.2e}" if pd.notna(x) else "N/A")

    def color_status(val):
        if pd.isna(val) or val == 'N/A': return ''
        try: return 'background-color: #d4edda' if float(val) < 1e-3 else 'background-color: #fff3cd'
        except: return ''

    clear_output()
    display(HTML("""
    <div style='padding: 15px; background-color: #d4edda; border-left: 5px solid #28a745; color: #155724; border-radius: 4px; font-family: sans-serif; margin-bottom: 15px;'>
        <h4 style='margin: 0 0 5px 0;'>✅ Robust CR2 Validation Complete</h4>
        <p style='margin: 0; font-size: 14px;'>This table explicitly compares Co-Meta's native linear algebra against R's <code>clubSandwich</code> package. It proves that Co-Meta correctly builds the <b>CR2 covariance matrices</b> and calculates exact <b>Satterthwaite Degrees of Freedom</b> natively in Python. The last rows check both omnibus statistics Step 9 reports: the asymptotic <b>Q<sub>M</sub></b> chi-square and the small-sample <b>HTZ F</b> (<code>Wald_test(test = "HTZ")</code>) that the narrative verdict follows &mdash; R builds the constraint matrix itself, so this checks the hypothesis as well as the arithmetic.</p>
    </div>
    """))

    # Styler.applymap was renamed to Styler.map in pandas 2.1 and removed in
    # 3.0; support both so the cell runs on any Colab image.
    _sty = df_comp.style
    _elementwise = _sty.map if hasattr(_sty, 'map') else _sty.applymap
    display(_elementwise(color_status, subset=['Diff'])
            .set_properties(**{'text-align': 'center'}))

    # --- 5. Pseudo-R² Validation (Cell 9 population-mismatch fix) ---
    r2_rows = [
        {'Component': 'tau² (null, intercept-only)', 'Co-Meta (Python)': py_tau2_null, 'metafor (R, free)': r_tau2_null},
        {'Component': 'sigma² (null, intercept-only)', 'Co-Meta (Python)': py_sigma2_null, 'metafor (R, free)': r_sigma2_null},
        {'Component': 'tau² (full, with subgroup)', 'Co-Meta (Python)': py_tau2, 'metafor (R, free)': r_tau2_full},
        {'Component': 'sigma² (full, with subgroup)', 'Co-Meta (Python)': py_sigma2, 'metafor (R, free)': r_sigma2_full},
        {'Component': 'Pseudo-R² % (recalculated here)', 'Co-Meta (Python)': py_r2_recalc, 'metafor (R, free)': r_r2},
        {'Component': 'Pseudo-R² % (saved by Cell 9)', 'Co-Meta (Python)': py_r2_saved, 'metafor (R, free)': r_r2},
    ]

    row_diffs = []
    row_tols = []
    for row in r2_rows:
        is_pct = row['Component'].startswith('Pseudo-R')
        tol = 0.5 if is_pct else 1e-3  # looser tolerance for the derived % quantity
        d = abs(row['Co-Meta (Python)'] - row['metafor (R, free)'])
        row_diffs.append(d)
        row_tols.append(tol)

    row_colors = [
        'background-color: #d4edda' if d < tol else 'background-color: #fff3cd'
        for d, tol in zip(row_diffs, row_tols)
    ]

    df_r2 = pd.DataFrame(r2_rows)
    df_r2['Diff'] = row_diffs
    _v3_checks = pd.concat([_v3_checks, df_r2.rename(columns={'Component': 'Parameter'}).assign(
        status=['PASS' if d < tol else 'FAIL' for d, tol in zip(row_diffs, row_tols)])], ignore_index=True)

    for col in ['Co-Meta (Python)', 'metafor (R, free)']:
        df_r2[col] = df_r2[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "N/A")
    df_r2['Diff'] = df_r2['Diff'].apply(lambda x: f"{x:.2e}" if pd.notna(x) else "N/A")

    display(HTML(f"""
    <div style='padding: 15px; background-color: #d1ecf1; border-left: 5px solid #17a2b8; color: #0c5460; border-radius: 4px; font-family: sans-serif; margin: 20px 0 15px 0;'>
        <h4 style='margin: 0 0 5px 0;'>🧪 Pseudo-R² Validation (Cell 9 population-mismatch fix)</h4>
        <p style='margin: 0; font-size: 14px;'>Checks the fix applied to <code>SubgroupAnalyzer.analyze_all_subgroups</code>: the null
        (intercept-only) model must be refit on the <b>same</b> filtered subgroup subset as the full
        (moderator) model, not borrowed from the global Overall Meta-Analysis. Co-Meta's null/full
        variance components are compared against R's independently-estimated (<code>metafor</code>, free REML,
        not pinned to Python's values) null/full variance components, along with the resulting Pseudo-R².
        The last two rows compare the value recalculated in this cell against the value actually saved by
        Cell 9's last run (<code>ANALYSIS_CONFIG['subgroup_results']['R_squared']</code>) -- they should
        match unless the underlying data changed since Cell 9 was last run.</p>
        <p style='margin: 8px 0 0 0; font-size: 13px;'>Sampling-covariance blocks fitted by BOTH engines:
        <b>{py_est.get('model_type', 'unknown')}</b>
        (shared-control off-diagonals present: <b>{'Yes' if has_shared else 'No'}</b>).
        R receives <code>block_diag()</code> of these exact blocks, so Sigma is identical by construction.</p>
        <p style='margin: 8px 0 0 0; font-size: 13px;'>2-level model (σ² = 0, one effect per study) used --
        null model: <b>{'Yes ⚠️' if py_null_regularized else 'No ✅'}</b>,
        full model: <b>{'Yes ⚠️' if py_full_regularized else 'No ✅'}</b>.
        "Yes" is expected only when no study in this subset has more than one effect size (sigma^2
        not identifiable); otherwise it should read "No".</p>
    </div>
    """))

    display(
        df_r2.style
             .apply(lambda col: row_colors, axis=0, subset=['Diff'])
             .set_properties(**{'text-align': 'center'})
    )
    if 'record_validation' in globals():
        record_validation(3, "Subgroup Analysis CR2", _v3_checks)

except Exception as e:
    # A crash is a failed validation, not 'not run'
    if 'record_validation' in globals() and not validation_prerequisite_missing(e):
        record_validation(3, 'Subgroup Analysis CR2',
                          [('validation ran to completion', 'FAIL')],
                          notes=f"{type(e).__name__}: {e}")
    clear_output()
    display(HTML(f"<div style='padding: 15px; background-color: #f8d7da; border-left: 5px solid #dc3545; color: #721c24; border-radius: 4px;'><b>❌ Error:</b> {e}</div>"))
    import traceback
    traceback.print_exc()

In [ ]:
#@title 🧪 Validation 4: Meta-Regression, Multiple Moderators (Python vs metafor + clubSandwich)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

import pandas as pd
import numpy as np
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import rpy2.robjects.numpy2ri as numpy2ri
from scipy.linalg import block_diag
from IPython.display import display, HTML, clear_output
import logging
import warnings

logging.getLogger('rpy2.rinterface_lib.callbacks').setLevel(logging.ERROR)
pandas2ri.activate()
numpy2ri.activate()

display(HTML("<div style='padding: 15px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; border-radius: 4px; font-family: sans-serif;'>⏳ <b>Running Multi-Factor Meta-Regression Cross-Validation with R (metafor)...</b> Please wait.</div>"))

try:
    # =============================================================
    # 1. CHECK DEPENDENCIES & GET PYTHON RESULTS
    # =============================================================
    if 'ANALYSIS_CONFIG' not in globals() or 'meta_regression_RVE_results' not in ANALYSIS_CONFIG:
        raise ValueError("ANALYSIS_CONFIG or Meta-Regression results not found. Please run earlier cells first.")

    py_res = ANALYSIS_CONFIG['meta_regression_RVE_results']

    py_betas = np.asarray(py_res['betas_all'], dtype=float).ravel()
    py_ses   = np.asarray(py_res['std_errors_all'], dtype=float).ravel()
    py_pvals = np.asarray(py_res['p_values_all'], dtype=float).ravel()
    term_names = list(py_res['term_names'])

    py_tau2 = float(py_res.get('tau_sq', np.nan))
    py_sigma2 = float(py_res.get('sigma_sq', np.nan))
    py_model_type = py_res.get('model_type', 'Unknown')
    is_aggregated = ("Aggregated" in py_model_type) or ("2-Level" in py_model_type)
    py_fitted = np.asarray(py_res.get('fitted', []), dtype=float).ravel()

    # =============================================================
    # 2. PARSE TERM NAMES  ->  TYPE TRUTH (FIX #1, #3)
    #    Do NOT infer categorical-vs-continuous from dtype. The Python
    #    engine may dummy-code a *numeric* low-cardinality column. The only
    #    reliable source of truth is term_names: 'factor[level]' == dummy.
    # =============================================================
    def parse_term(name):
        s = str(name)
        if s == 'Intercept':
            return ('intercept', None, None)
        if '[' in s and s.endswith(']'):
            factor = s.split('[', 1)[0]
            level = s.split('[', 1)[1][:-1]
            return ('categorical', factor, level)
        base = s.replace(' (centered)', '')
        return ('continuous', base, None)

    source_mods = []                 # ordered unique source moderator names
    mod_is_categorical = {}          # base name -> bool
    factor_dummy_levels = {}         # factor -> set(levels present as dummies)
    for name in term_names[1:]:      # skip intercept
        kind, base, level = parse_term(name)
        if base not in source_mods:
            source_mods.append(base)
            mod_is_categorical[base] = False
        if kind == 'categorical':
            mod_is_categorical[base] = True
            factor_dummy_levels.setdefault(base, set()).add(level)

    # =============================================================
    # 3. GET EXACT DATA USED BY PYTHON MODEL
    # =============================================================
    if 'reg_df' not in py_res:
        raise ValueError("Regression dataframe not found in results.")

    df_clean = py_res['reg_df'].copy()
    effect_col = str(ANALYSIS_CONFIG['effect_col'])
    var_col = str(ANALYSIS_CONFIG['var_col'])
    vcv_dict = ANALYSIS_CONFIG.get('vcv_matrices', {})

    missing = [c for c in source_mods if c not in df_clean.columns]
    if missing:
        raise KeyError(f"Validation failed: could not find raw moderator columns: {missing}. "
                       f"Columns available: {list(df_clean.columns)}")

    # Attach Python fitted BEFORE any sort so it stays row-aligned (3-level only;
    # 2-level fitted are study-aggregated and won't match n_obs rows).
    can_check_fitted = (not is_aggregated) and (py_fitted.size == len(df_clean))
    if can_check_fitted:
        df_clean['_py_fitted'] = py_fitted

    # Safe column names to dodge rpy2/pandas name mangling.
    safe_mod_cols = [f"mod_{i}" for i in range(len(source_mods))]
    safe_of = dict(zip(source_mods, safe_mod_cols))
    df_clean = df_clean.rename(columns=safe_of)

    # Coding decided by term_names, NOT dtype.
    cat_safe_cols, cat_refs = [], []
    for base in source_mods:
        safe = safe_of[base]
        if mod_is_categorical[base]:
            # Categorical: stringify, do NOT center; infer R reference level.
            df_clean[safe] = df_clean[safe].astype(str)
            data_levels = set(df_clean[safe].dropna().unique())
            ref = data_levels - factor_dummy_levels.get(base, set())
            if len(ref) == 1:
                ref_lvl = next(iter(ref))
            else:
                # Pathological (shouldn't happen): fall back to alphabetical first.
                ref_lvl = sorted(data_levels)[0]
                warnings.warn(f"Could not uniquely infer reference level for '{base}' "
                              f"(candidates: {sorted(ref)}); using alphabetical first.")
            cat_safe_cols.append(safe)
            cat_refs.append(ref_lvl)
        else:
            # Continuous: center to match Python's centered design exactly.
            df_clean[safe] = df_clean[safe].astype(np.float64)
            df_clean[safe] = df_clean[safe] - df_clean[safe].mean()

    if 'attach_vcv_row_pos' in globals():
        df_clean = attach_vcv_row_pos(df_clean)   # same V slicing as the engine
    df_clean = df_clean.sort_values(['id'], kind='mergesort').reset_index(drop=True)
    df_clean['_row_id'] = np.arange(1, len(df_clean) + 1)
    n_obs = len(df_clean)
    n_studies = df_clean['id'].nunique()

    # =============================================================
    # 4. BUILD VCV MATRIX (3-Level only)
    # =============================================================
    use_vcv_matrix = False
    has_off_diag = False
    V_full = np.diag(df_clean[var_col].values.astype(np.float64))
    if not is_aggregated:
        vcv_blocks = []
        for study_id, group in df_clean.groupby('id', sort=False):
            k = len(group)
            vi = group[var_col].values.astype(np.float64)
            V_i, _src = cluster_vcv(vcv_dict, study_id, group, var_col)
            vcv_blocks.append(V_i)
        try:
            V_full = block_diag(*vcv_blocks)
            use_vcv_matrix = True
            has_off_diag = any(not np.allclose(m, np.diag(np.diag(m))) for m in vcv_blocks)
        except Exception:
            V_full = np.diag(df_clean[var_col].values.astype(np.float64))

    use_t_dist = (ANALYSIS_CONFIG.get('global_settings', {}).get('dist_type') == 't')

    # =============================================================
    # 5. RUN R ANALYSIS
    # =============================================================
    ro.globalenv['df_r'] = df_clean
    ro.globalenv['V_matrix'] = V_full
    ro.globalenv['eff_col'] = effect_col
    ro.globalenv['var_col'] = var_col
    ro.globalenv['safe_mod_cols'] = ro.StrVector(safe_mod_cols)
    ro.globalenv['cat_cols'] = ro.StrVector(cat_safe_cols)
    ro.globalenv['cat_refs'] = ro.StrVector([str(r) for r in cat_refs])
    ro.globalenv['use_vcv'] = use_vcv_matrix and has_off_diag
    ro.globalenv['use_t_dist'] = use_t_dist
    ro.globalenv['is_aggregated'] = is_aggregated

    r_script = """
    suppressPackageStartupMessages(library(metafor))
    suppressPackageStartupMessages(library(clubSandwich))
    dat <- df_r
    dat$row_id <- dat$`_row_id`
    dat$study_id <- as.factor(dat$id)

    # Relevel categorical factors to Python's reference (FIX #3): guarantees
    # R drops the SAME dummy column Python dropped, even for custom references.
    if (length(cat_cols) > 0) {
        for (j in seq_along(cat_cols)) {
            col <- cat_cols[j]
            dat[[col]] <- relevel(as.factor(as.character(dat[[col]])), ref = cat_refs[j])
        }
    }

    form_str <- paste("~", paste(safe_mod_cols, collapse=" + "))
    mods_form <- as.formula(form_str)
    if (use_vcv) { V_input <- V_matrix } else { V_input <- dat[[var_col]] }
    test_type <- ifelse(use_t_dist, "t", "z")

    tryCatch({
        if (is_aggregated) {
            # FIX #2/#4: run BOTH the KH (test="knha") and the un-truncated
            # standard (test="z") fits, so Python's truncated-KH SEs can be
            # compared against the correct reference for either regime.
            res_kh <- rma(yi = dat[[eff_col]], vi = dat[[var_col]], mods = mods_form,
                          data = dat, method = "REML", test = "knha")
            res_z  <- rma(yi = dat[[eff_col]], vi = dat[[var_col]], mods = mods_form,
                          data = dat, method = "REML", test = "z")
            list(
                status = "success", level = "2",
                betas = as.numeric(res_kh$beta),
                ses_kh = as.numeric(res_kh$se),   pvals_kh = as.numeric(res_kh$pval),
                ses_z  = as.numeric(res_z$se),    pvals_z  = as.numeric(res_z$pval),
                tau2 = as.numeric(res_kh$tau2), sigma2 = 0.0,
                coef_names = rownames(res_kh$beta)
            )
        } else {
            res <- rma.mv(yi = dat[[eff_col]], V = V_input, mods = mods_form,
                          random = ~ 1 | study_id/row_id, data = dat, method = "REML",
                          control = list(optimizer = "nlminb"))
            res_rob <- robust(res, cluster = dat$study_id, clubSandwich = TRUE, test = test_type)
            list(
                status = "success", level = "3",
                betas = as.numeric(res_rob$beta),
                ses = as.numeric(res_rob$se),
                pvals = as.numeric(res_rob$pval),
                tau2 = as.numeric(res$sigma2[1]),
                sigma2 = as.numeric(res$sigma2[2]),
                coef_names = rownames(res_rob$beta),
                fitted = as.numeric(fitted(res))   # marginal Xβ, for fitted-value check
            )
        }
    }, error = function(e) { list(status = "error", msg = conditionMessage(e)) })
    """
    r_res = ro.r(r_script)

    def rkeys(obj):
        try: return list(obj.names)
        except Exception: return []
    rk = rkeys(r_res)

    if 'status' in rk and r_res.rx2('status')[0] == 'error':
        raise ValueError(f"R Error: {r_res.rx2('msg')[0]}")

    # =============================================================
    # 6. ALIGN BY NAME (fixes positional misalignment)
    # =============================================================
    def safe_extract(r_obj, key, default=np.nan):
        try: return float(r_obj.rx2(key)[0])
        except Exception: return default

    r_betas = np.asarray(r_res.rx2('betas'), dtype=float).ravel()
    r_coef_names = [str(x) for x in r_res.rx2('coef_names')]
    r_tau2 = safe_extract(r_res, 'tau2')
    r_sigma2 = safe_extract(r_res, 'sigma2')

    def expected_r_name(name):
        kind, base, level = parse_term(name)
        if kind == 'intercept':
            return 'intrcpt'
        safe = safe_of[base]
        return safe if kind == 'continuous' else f"{safe}{level}"

    # Fail loudly on a genuine structural mismatch instead of silently
    # truncating with min(len(...)).
    if len(r_betas) != len(py_betas):
        raise ValueError(
            f"Coefficient count mismatch: Python has {len(py_betas)} terms, "
            f"R has {len(r_betas)}. Python terms: {term_names}. R terms: {r_coef_names}."
        )

    r_idx_of = {n: i for i, n in enumerate(r_coef_names)}
    aligned = []
    unmatched = []
    for name in term_names:
        ern = expected_r_name(name)
        if ern in r_idx_of:
            aligned.append(r_idx_of[ern])
        else:
            unmatched.append((name, ern))
    if unmatched:
        raise ValueError(
            "Could not align these Python terms to R coefficients "
            f"(parsed -> expected R name): {unmatched}. R coef names: {r_coef_names}."
        )
    aligned = np.array(aligned, dtype=int)
    r_betas = r_betas[aligned]

    # Pull SE / p-value arrays per path, then realign.
    regime_note = ""
    if is_aggregated:
        ses_kh = np.asarray(r_res.rx2('ses_kh'), dtype=float).ravel()[aligned]
        pv_kh  = np.asarray(r_res.rx2('pvals_kh'), dtype=float).ravel()[aligned]
        ses_z  = np.asarray(r_res.rx2('ses_z'),  dtype=float).ravel()[aligned]
        pv_z   = np.asarray(r_res.rx2('pvals_z'), dtype=float).ravel()[aligned]
        # Detect which regime Python's truncated-KH SEs fall in. This is
        # diagnostic only — both reference columns are shown in the table.
        d_kh = float(np.nansum(np.abs(py_ses - ses_kh)))
        d_z  = float(np.nansum(np.abs(py_ses - ses_z)))
        if d_z < d_kh:
            regime_note = ("Python's SEs match metafor's <b>un-truncated standard (z)</b> fit, "
                           "i.e. the reduced χ² &lt; 1 and the KH scale was <b>truncated to 1.0</b>. "
                           "Compare the SE/P rows against the <code>metafor (R, no-KH)</code> column.")
        else:
            regime_note = ("Python's SEs match metafor's <b>Knapp-Hartung</b> fit "
                           "(reduced χ² ≥ 1, truncation inactive). Compare against "
                           "the <code>metafor (R, KHA)</code> column.")
        r_ses_main, r_pv_main = ses_kh, pv_kh
        r_ses_alt,  r_pv_alt  = ses_z, pv_z
    else:
        r_ses_main = np.asarray(r_res.rx2('ses'), dtype=float).ravel()[aligned]
        r_pv_main  = np.asarray(r_res.rx2('pvals'), dtype=float).ravel()[aligned]
        r_ses_alt = r_pv_alt = None

    # =============================================================
    # 7. BUILD COMPARISON TABLE
    # =============================================================
    rows = []
    rows.append({'Parameter': 'Tau² (Between)', 'py': py_tau2, 'r': r_tau2, 'r_alt': np.nan})
    if not is_aggregated:
        rows.append({'Parameter': 'Sigma² (Within)', 'py': py_sigma2, 'r': r_sigma2, 'r_alt': np.nan})

    for i, name in enumerate(term_names):
        rows.append({'Parameter': f'β: {name}', 'py': py_betas[i], 'r': r_betas[i], 'r_alt': np.nan})
        rows.append({'Parameter': f'SE: {name}', 'py': py_ses[i], 'r': r_ses_main[i],
                     'r_alt': (r_ses_alt[i] if r_ses_alt is not None else np.nan)})
        rows.append({'Parameter': f'P-val: {name}', 'py': py_pvals[i], 'r': r_pv_main[i],
                     'r_alt': (r_pv_alt[i] if r_pv_alt is not None else np.nan)})

    df_comp = pd.DataFrame(rows)
    df_comp['Diff'] = np.abs(df_comp['py'] - df_comp['r'])

    main_label = "metafor (R, KHA)" if is_aggregated else "metafor (R, CR2)"
    df_comp = df_comp.rename(columns={'py': 'Co-Meta (Python)', 'r': main_label})
    show_alt = is_aggregated
    if show_alt:
        df_comp = df_comp.rename(columns={'r_alt': 'metafor (R, no-KH)'})
        df_comp['Diff (no-KH)'] = np.abs(df_comp['Co-Meta (Python)'] - df_comp['metafor (R, no-KH)'])
        col_order = ['Parameter', 'Co-Meta (Python)', main_label, 'Diff',
                     'metafor (R, no-KH)', 'Diff (no-KH)']
    else:
        df_comp = df_comp.drop(columns=['r_alt'])
        col_order = ['Parameter', 'Co-Meta (Python)', main_label, 'Diff']
    df_comp = df_comp[col_order]
    # No tolerance is applied to this table (SEs depend on the KH scale), so its
    # rows are recorded as INFO; the fitted-value check below is pass/fail.
    _v4_checks = df_comp.assign(status='INFO')

    fmt_val = lambda x: f"{x:.4f}" if pd.notna(x) else "N/A"
    fmt_dif = lambda x: f"{x:.2e}" if pd.notna(x) else "N/A"
    for c in df_comp.columns:
        if c == 'Parameter': continue
        df_comp[c] = df_comp[c].apply(fmt_dif if c.startswith('Diff') else fmt_val)

    # Optional fitted-value cross-check (3-level): catches the centered-
    # intercept / raw-x fitted-value bug, which the β/SE table cannot see.
    fitted_html = ""
    if can_check_fitted and 'fitted' in rkeys(r_res):
        r_fitted = np.asarray(r_res.rx2('fitted'), dtype=float).ravel()
        py_fit_sorted = df_clean['_py_fitted'].values.astype(float)
        if r_fitted.size == py_fit_sorted.size:
            max_abs = float(np.nanmax(np.abs(py_fit_sorted - r_fitted)))
            ok = max_abs < 1e-4
            _v4_checks = pd.concat([_v4_checks, pd.DataFrame([{
                'Parameter': 'Fitted values (marginal Xβ)', 'Diff': max_abs,
                'status': 'PASS' if ok else 'FAIL'}])], ignore_index=True)
            fitted_html = f"""
            <div style='background-color: {'#d4edda' if ok else '#f8d7da'}; padding: 12px;
                        border-radius: 5px; border-left: 4px solid {'#28a745' if ok else '#dc3545'};
                        margin-top: 10px; font-family: sans-serif; font-size: 13px;
                        color: {'#155724' if ok else '#721c24'};'>
                <b>{'✓' if ok else '❌'} Fitted-value check (marginal Xβ):</b>
                max |Python − R| = {max_abs:.2e} over {r_fitted.size} effects.
                {'' if ok else ' A large gap here typically means the intercept was estimated on the centered design but fitted values were reconstructed with the <b>raw</b> moderator.'}
            </div>"""

    clear_output()
    display(HTML(f"""
    <div style='padding: 15px; background-color: #d4edda; border-left: 5px solid #28a745; color: #155724; border-radius: 4px; font-family: sans-serif; margin-bottom: 15px;'>
        <h4 style='margin: 0 0 5px 0;'>✅ Multi-Factor Meta-Regression Cross-Validation Complete</h4>
        <p style='margin: 0; font-size: 14px;'>Validating <b>{len(source_mods)} moderator(s)</b>
        ({sum(mod_is_categorical.values())} categorical) — n={n_obs}, k={n_studies}.
        Distribution: <b>{'t' if use_t_dist else 'z'}</b>.</p>
    </div>
    """))
    display(df_comp.style.set_properties(**{'text-align': 'center'}))

    if is_aggregated:
        display(HTML(f"""
        <div style='background-color: #e7f3ff; padding: 12px; border-radius: 5px; border-left: 4px solid #007bff; margin-top: 15px; font-family: sans-serif; font-size: 13px; color: #004085;'>
            <b>ℹ️ 2-Level note (truncated KH).</b> Point estimates (β, τ²) are scale-invariant and should match to ~1e-6.
            Standard errors depend on the KH scale: Python <b>floors it at 1.0</b>, metafor's <code>knha</code> does not.
            {regime_note}<br>
            {('<b>⚠ Distribution mismatch:</b> dist_type = "z" but Python applies KH-scaled SEs; '
              'p-values may diverge by design — judge SE rows, not P rows.') if not use_t_dist else ''}
        </div>
        """))
    else:
        display(HTML(f"""
        <div style='background-color: #e7f3ff; padding: 12px; border-radius: 5px; border-left: 4px solid #007bff; margin-top: 15px; font-family: sans-serif; font-size: 13px; color: #004085;'>
            <b>ℹ️ 3-Level note.</b> metafor uses CR2 + Satterthwaite df via <code>clubSandwich</code>.
            This check only passes if Python's Cell 9.5 engine also uses <b>CR2</b> (not CR0/CR1) and
            <b>Satterthwaite</b> df. If SE/P rows disagree, confirm the small-sample correction and df
            method in the engine before concluding metafor is the discrepancy.
            {('<b>VCV:</b> off-diagonal covariances supplied to both.' if (use_vcv_matrix and has_off_diag) else '<b>VCV:</b> diagonal (independent sampling errors) assumed.')}
        </div>
        """))

    if fitted_html:
        display(HTML(fitted_html))

    display(HTML("""
    <div style='background-color: #fff3cd; padding: 12px; border-radius: 5px; border-left: 4px solid #ffc107; margin-top: 10px; font-family: sans-serif; font-size: 12px; color: #856404;'>
        <b>Not validated here:</b> omnibus F / QM, pseudo-R², and per-term df. The omnibus in particular
        (homebrew truncated-KH F rescaling) deserves its own check against <code>anova()</code> / the fit's QM.
    </div>
    """))
    if 'record_validation' in globals():
        record_validation(4, "Meta-Regression, Multiple Moderators", _v4_checks,
                          notes='Coefficient table is descriptive (INFO); only the fitted-value check has a tolerance.')

except Exception as e:
    # A crash is a failed validation, not 'not run'
    if 'record_validation' in globals() and not validation_prerequisite_missing(e):
        record_validation(4, 'Meta-Regression, Multiple Moderators',
                          [('validation ran to completion', 'FAIL')],
                          notes=f"{type(e).__name__}: {e}")
    clear_output()
    import traceback as _tb
    display(HTML(f"""
    <div style='padding: 15px; background-color: #f8d7da; border-left: 5px solid #dc3545; color: #721c24; border-radius: 4px; font-family: sans-serif;'>
        <h4 style='margin: 0 0 5px 0;'>❌ Validation Error</h4>
        <p style='margin: 0;'>{e}</p>
        <pre style='margin-top: 10px; font-size: 11px; white-space: pre-wrap;'>{_tb.format_exc()}</pre>
    </div>
    """))

In [ ]:
#@title 🧪 Validation 5: Meta-Regression Monte Carlo (Python vs metafor + clubSandwich)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Simulates three-level data with a within-study moderator and correlated
# sampling errors (full V per study), then fits every replicate twice:
#     Python : _run_three_level_reml_regression_v2 (REML + CR2/Satterthwaite)
#     R      : rma.mv(yi, V, mods = ~ mod_x, random = ~ 1 | study/row, method = "REML")
#              clubSandwich::coef_test(vcov = "CR2", test = "Satterthwaite")
#
# Two kinds of check:
#   1. Paired agreement, every replicate: intercept (at mean moderator, as
#      CoMeta centres continuous moderators) and slope, slope CR2 SE,
#      Satterthwaite df, tau² and sigma² (tolerances in TOL_MR).
#   2. Operating characteristics of the Python CR2 interval for the slope:
#      coverage within COVERAGE_BAND (± 2 Monte Carlo SE). In the null
#      scenario (slope = 0) this is the Type I error rate (1 - coverage).
#
# Scenarios: standard, null slope, high heterogeneity, few studies (k = 12).
# The user's ANALYSIS_CONFIG is restored afterwards.
# Requires: Cells 1-7 and the Validation R setup cell.

import time
import warnings
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from scipy.linalg import block_diag
from IPython.display import display, HTML

MR_N_SIMS = 200          # replicates per scenario (more = narrower Monte Carlo band)
TOL_MR = {'beta': 1e-4, 'se_rel': 1e-3, 'df_rel': 1e-2, 'tau2': 1e-4, 'sigma2': 1e-4}
COVERAGE_BAND = (0.93, 0.97)

MR_SCENARIOS = [
    {'name': 'Standard',              'n_studies': 30, 'intercept': 0.5, 'slope': 0.5, 'tau_sq': 0.3, 'sigma_sq': 0.1},
    {'name': 'Null slope (Type I)',   'n_studies': 30, 'intercept': 0.5, 'slope': 0.0, 'tau_sq': 0.3, 'sigma_sq': 0.1},
    {'name': 'High heterogeneity',    'n_studies': 30, 'intercept': 0.5, 'slope': 0.5, 'tau_sq': 0.8, 'sigma_sq': 0.1},
    {'name': 'Few studies (k = 12)',  'n_studies': 12, 'intercept': 0.5, 'slope': 0.5, 'tau_sq': 0.3, 'sigma_sq': 0.1},
]


def _mr_simulate(sc, rho=0.5, seed=None):
    """One dataset: 1-3 effects per study, moderator varies within studies,
    sampling errors correlated rho within a study."""
    rng = np.random.default_rng(seed)
    rows, vcv = [], {}
    for i in range(sc['n_studies']):
        sid = f"s{i + 1:03d}"
        k = int(rng.choice([1, 2, 3]))
        x = rng.uniform(0, 5, k)
        vi = rng.uniform(0.05, 0.15, k)
        V = rho * np.sqrt(np.outer(vi, vi))
        np.fill_diagonal(V, vi)
        vcv[sid] = V
        e = rng.multivariate_normal(np.zeros(k), V + sc['sigma_sq'] * np.eye(k))
        y = sc['intercept'] + sc['slope'] * x + rng.normal(0, np.sqrt(sc['tau_sq'])) + e
        rows += [{'id': sid, 'yi': y[j], 'vi': vi[j], 'mod_x': x[j]} for j in range(k)]
    return pd.DataFrame(rows), vcv


def _mr_python(df, vcv):
    g = globals()
    saved = g.get('ANALYSIS_CONFIG')
    try:
        g['ANALYSIS_CONFIG'] = {'vcv_matrices': vcv, 'global_settings': {'alpha': 0.05}}
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            res, _, _ = _run_three_level_reml_regression_v2(df, 'mod_x', 'yi', 'vi')
    finally:
        if saved is None:
            g.pop('ANALYSIS_CONFIG', None)
        else:
            g['ANALYSIS_CONFIG'] = saved
    if not res:
        return None
    return {'b0': res['betas'][0], 'b1': res['betas'][1], 'se1': res['se_betas'][1],
            'df1': np.asarray(res['df'])[1], 'tau2': res['tau_sq'], 'sigma2': res['sigma_sq'],
            'lo': res['ci_lower'][1], 'hi': res['ci_upper'][1]}


def _mr_r(df, vcv):
    blocks = [vcv[s] for s in pd.unique(df['id'])]
    ro.globalenv['yi_v'] = ro.FloatVector(df['yi'].values)
    ro.globalenv['x_v'] = ro.FloatVector(df['mod_x'].values)
    ro.globalenv['cl_v'] = ro.StrVector(df['id'].astype(str).tolist())
    V = block_diag(*blocks)
    ro.globalenv['V_m'] = ro.r.matrix(ro.FloatVector(V.ravel(order='F')), nrow=V.shape[0])
    out = ro.r("""
    suppressPackageStartupMessages({ library(metafor); library(clubSandwich) })
    d <- data.frame(yi = as.numeric(yi_v), x = as.numeric(x_v), cl = factor(as.character(cl_v)))
    d$x <- d$x - mean(d$x)   # CoMeta centres continuous moderators (intercept = effect at mean x)
    d$row <- factor(seq_len(nrow(d)))
    tryCatch({
        f <- rma.mv(yi, V_m, mods = ~ x, random = ~ 1 | cl/row, data = d, method = "REML",
                    control = list(rel.tol = 1e-10, iter.max = 1000, eval.max = 2000))
        ct <- coef_test(f, vcov = "CR2", cluster = d$cl, test = "Satterthwaite")
        c(as.numeric(f$b), ct$SE[2], ct$df_Satt[2], f$sigma2)
    }, error = function(e) rep(NA_real_, 6))
    """)
    v = np.asarray(out, float)
    if not np.all(np.isfinite(v)):
        return None
    return dict(zip(['b0', 'b1', 'se1', 'df1', 'tau2', 'sigma2'], v))


def _v3c_main():
    """Main block, run in a function so its working variables do not leak
    into the notebook's shared namespace."""
    if '_run_three_level_reml_regression_v2' not in globals():
        display(HTML("<div style='padding:12px; background:#fff3cd; border-radius:4px; font-family:sans-serif;'>"
                     "⚠️ Run Cell 1 and the Validation R setup cell first.</div>"))
        return None
    t0 = time.time()
    rows = []
    for si, sc in enumerate(MR_SCENARIOS):
        d = {k: [] for k in ('b', 'se', 'df', 'tau2', 'sigma2')}
        covered, n_py_fail, n_r_fail = [], 0, 0
        for i in range(MR_N_SIMS):
            df, vcv = _mr_simulate(sc, seed=10_000 * si + i)
            py = _mr_python(df, vcv)
            if py is None:
                n_py_fail += 1
                continue
            covered.append(py['lo'] <= sc['slope'] <= py['hi'])
            r = _mr_r(df, vcv)
            if r is None:
                n_r_fail += 1
                continue
            d['b'].append(max(abs(py['b0'] - r['b0']), abs(py['b1'] - r['b1'])))
            d['se'].append(abs(py['se1'] - r['se1']) / r['se1'])
            d['df'].append(abs(py['df1'] - r['df1']) / r['df1'])
            d['tau2'].append(abs(py['tau2'] - r['tau2']))
            d['sigma2'].append(abs(py['sigma2'] - r['sigma2']))
        mx = {k: (max(v) if v else np.nan) for k, v in d.items()}
        cov = float(np.mean(covered)) if covered else np.nan
        cov_se = float(np.sqrt(cov * (1 - cov) / len(covered))) if covered else np.nan
        agree = (n_py_fail == 0 and len(d['b']) > 0 and mx['b'] < TOL_MR['beta'] and mx['se'] < TOL_MR['se_rel']
                 and mx['df'] < TOL_MR['df_rel'] and mx['tau2'] < TOL_MR['tau2'] and mx['sigma2'] < TOL_MR['sigma2'])
        cov_ok = COVERAGE_BAND[0] - 2 * cov_se <= cov <= COVERAGE_BAND[1] + 2 * cov_se
        rows.append({'scenario': sc['name'], 'true slope': sc['slope'], 'fits (Py/R)':
                     f"{MR_N_SIMS - n_py_fail}/{MR_N_SIMS - n_py_fail - n_r_fail}",
                     'max |Δ| β': mx['b'], 'max rel Δ SE': mx['se'], 'max rel Δ df': mx['df'],
                     'max |Δ| τ²': mx['tau2'], 'max |Δ| σ²': mx['sigma2'],
                     'agreement': 'PASS' if agree else 'FAIL',
                     'CR2 coverage': f"{cov * 100:.1f}% ± {cov_se * 100:.1f}",
                     'coverage': 'PASS' if cov_ok else 'FAIL'})

    res_df = pd.DataFrame(rows)
    n_checks = 2 * len(res_df)
    n_fail = int((res_df['agreement'] != 'PASS').sum() + (res_df['coverage'] != 'PASS').sum())
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
        f"<b>{icon} Meta-regression Monte Carlo vs metafor + clubSandwich: {n_checks - n_fail}/{n_checks} checks passed.</b><br>"
        f"<small>{MR_N_SIMS} replicates per scenario, {(time.time() - t0) / 60:.1f} min. Paired tolerances: {TOL_MR}. "
        f"Coverage of the 95% CR2 interval for the slope must lie in "
        f"[{COVERAGE_BAND[0]:.0%}, {COVERAGE_BAND[1]:.0%}] ± 2 Monte Carlo SE; in the null scenario "
        f"1 − coverage is the Type I error rate.</small></div>"))
    display(res_df.style.format({c: '{:.1e}' for c in ('max |Δ| β', 'max rel Δ SE', 'max rel Δ df',
                                                         'max |Δ| τ²', 'max |Δ| σ²')}, na_rep=''))
    return res_df


res_df = _v3c_main()
if res_df is not None and 'record_validation' in globals():
    record_validation(5, "Meta-Regression Monte Carlo", pd.concat(
        [res_df.assign(check=c, status=res_df[c]) for c in ('agreement', 'coverage')], ignore_index=True))


In [ ]:
#@title 🧪 Validation 6: Spline Meta-Regression — Full Pipeline (Python vs metafor + clubSandwich)
# Reference-only validation: it fits the spline pipeline on a fixed simulated
# curved-truth (sine) dataset with diagonal V and compares GLS, CR2, joint REML
# and the HTZ omnibus with metafor + clubSandwich. The spline on YOUR data is
# checked by Validation 10; this synthetic case runs only when the reference
# datasets are switched on (VALIDATION_INCLUDE_REFERENCE).
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

_V6_RUN = reference_only_validation(6, "Spline Meta-Regression — Full Pipeline")

import numpy as np
import pandas as pd
import patsy
import warnings
import logging
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import rpy2.robjects.numpy2ri as numpy2ri
from IPython.display import display, HTML, clear_output

logging.getLogger('rpy2.rinterface_lib.callbacks').setLevel(logging.ERROR)
pandas2ri.activate()
numpy2ri.activate()

if _V6_RUN:
    display(HTML("<div style='padding:15px;background:#e2e3e5;border-left:4px solid #6c757d;"
                 "color:#383d41;border-radius:4px;font-family:sans-serif;'>"
                 "⏳ <b>Running Python &amp; metafor spline models…</b></div>"))


def _badge(kind):
    c = {'PASS': ('#d4edda', '#155724', '✅ PASS'),
         'FAIL': ('#f8d7da', '#721c24', '❌ FAIL'),
         'INFO': ('#d1ecf1', '#0c5460', 'ℹ️ INFO'),
         'WARN': ('#fff3cd', '#856404', '⚠️ CHECK')}[kind]
    return (f"<span style='background:{c[0]};color:{c[1]};padding:2px 8px;"
            f"border-radius:10px;font-size:12px;font-weight:600;'>{c[2]}</span>")


def _v4a_main():
    """Main block, run in a function so its working variables do not leak
    into the notebook's shared namespace. The user's ANALYSIS_CONFIG is
    restored in the finally clause, also when a step fails."""
    global ANALYSIS_CONFIG
    if 'ANALYSIS_CONFIG' not in globals():
        ANALYSIS_CONFIG = {}
    original_config = None
    rows = []
    try:
        # ----------------------------------------------------------------
        # 1. DATA — curved (sine) truth: the spline model is needed to fit it,
        #    so this is where joint REML under the spline matters. n studies x 2 esz.
        # ----------------------------------------------------------------
        def generate_sine_data(n=80, tau2=0.2, sigma2=0.1, seed=42):
            rng = np.random.default_rng(seed)
            x = np.sort(rng.uniform(-3, 3, n))
            true_y = np.sin(x) + 0.5
            n_studies = n // 2
            ids = np.repeat(np.arange(n_studies), 2)
            u = np.repeat(rng.normal(0, np.sqrt(tau2), n_studies), 2)
            e = rng.normal(0, np.sqrt(sigma2), n)
            df = pd.DataFrame({'id': ids, 'y': true_y + u + e,
                               'vi': rng.uniform(0.05, 0.15, n), 'x': x})
            df['row_id'] = np.arange(len(df))
            return df

        df_sim = generate_sine_data()

        # Safe config override (create if absent; never KeyError) -----------
        original_config = dict(ANALYSIS_CONFIG)
        ANALYSIS_CONFIG['analysis_data'] = df_sim
        ANALYSIS_CONFIG['vcv_matrices'] = {}        # empty -> diagonal V

        # ----------------------------------------------------------------
        # 2. PYTHON SPLINE — capture (do NOT suppress) the function's own
        #    warnings; a validation harness must surface them.
        # ----------------------------------------------------------------
        with warnings.catch_warnings(record=True) as wlist:
            warnings.simplefilter('always')
            py_res, err = _run_robust_spline_analysis(df_sim, 'x', 'y', 'vi', df_spline=3)
            captured = [str(w.message) for w in wlist]
        if err:
            raise ValueError(f"Python function returned an error: {err}")

        reg = py_res['reg_df'].reset_index(drop=True)        # canonical order (sorted by id)
        betas_py = np.asarray(py_res['betas'], float)
        fitted_py = np.asarray(py_res['fitted'], float)
        se_py = np.asarray(py_res['se_betas'], float)
        p = len(betas_py)

        # ----------------------------------------------------------------
        # 3. Rebuild the EXACT internal basis (raw x, reg order) and prove
        #    it matches the function before trusting any comparison.
        #    cr() knots are deterministic from the data, so rebuilding on
        #    the same raw values in the same order reproduces X exactly.
        # ----------------------------------------------------------------
        X = np.asarray(patsy.dmatrix(py_res['formula'],
                                     {"x": reg['x'].values.astype(float)},
                                     return_type='matrix'))
        consistency = float(np.max(np.abs(X @ betas_py - fitted_py)))
        if consistency > 1e-8:
            raise ValueError(
                f"Rebuilt basis does NOT match the function's internal design "
                f"(max |X·β − fitted| = {consistency:.2e}). The validation basis is "
                f"out of sync — e.g. standardisation or a knot/order mismatch. "
                f"Fix this before interpreting any result below."
            )

        # ----------------------------------------------------------------
        # 4. METAFOR — fit twice: FIXED components (isolates the GLS / CR2
        #    algebra) and FREE REML (checks the joint variance estimation).
        #    Component order for ~1|id/row_id is [outer=id (between, tau^2),
        #    inner=row_id (within, sigma^2)] -> c(tau2, sigma2).
        # ----------------------------------------------------------------
        ro.globalenv['df_r'] = reg[['id', 'row_id', 'y', 'vi', 'x']]
        ro.globalenv['X_py'] = X
        ro.globalenv['fixed_tau2'] = float(py_res['tau_sq'])
        ro.globalenv['fixed_sigma2'] = float(py_res['sigma_sq'])

        r_out = ro.r("""
        suppressPackageStartupMessages({ library(metafor); library(clubSandwich) })
        dat <- df_r
        dat$id     <- factor(dat$id)
        dat$row_id <- factor(dat$row_id)
        p <- ncol(X_py)

        # ~ 0 + X_py : X_py already contains patsy's intercept column; do not add another
        res_fix  <- rma.mv(yi=y, V=vi, mods=~0+X_py, random=~1|id/row_id,
                           data=dat, method="REML", sigma2=c(fixed_tau2, fixed_sigma2))
        res_free <- rma.mv(yi=y, V=vi, mods=~0+X_py, random=~1|id/row_id,
                           data=dat, method="REML")

        se_cr2 <- tryCatch(as.numeric(robust(res_fix, cluster=dat$id, clubSandwich=TRUE)$se),
                           error=function(e) rep(NA_real_, p))
        wt <- tryCatch(clubSandwich::Wald_test(res_fix,
                           constraints=clubSandwich::constrain_zero(2:p),
                           vcov="CR2", cluster=dat$id, test="HTZ"),
                       error=function(e) data.frame(Fstat=NA, df_num=NA, df_denom=NA, p_val=NA))

        list(fitted   = as.numeric(fitted(res_fix)),
             betas    = as.numeric(coef(res_fix)),
             se_cr2   = se_cr2,
             tau2_free= as.numeric(res_free$sigma2[1]),
             sig2_free= as.numeric(res_free$sigma2[2]),
             omni_F   = as.numeric(wt$Fstat[1]),
             omni_dfn = as.numeric(wt$df_num[1]),
             omni_dfd = as.numeric(wt$df_denom[1]),
             omni_p   = as.numeric(wt$p_val[1]))
        """)

        g = lambda k: np.asarray(r_out.rx2(k))
        fitted_r, betas_r, se_cr2_r = g('fitted'), g('betas'), g('se_cr2')
        tau2_free, sig2_free = float(g('tau2_free')[0]), float(g('sig2_free')[0])
        omni_F_r, omni_dfd_r, omni_p_r = (float(g('omni_F')[0]),
                                          float(g('omni_dfd')[0]),
                                          float(g('omni_p')[0]))


        # ----------------------------------------------------------------
        # 5. COMPARISONS
        # ----------------------------------------------------------------
        rows = []

        d = float(np.max(np.abs(fitted_py - fitted_r)))
        rows.append(("GLS fitted values (fixed components)",
                     f"max |Δ| = {d:.2e}", 'PASS' if d < 1e-6 else 'FAIL'))

        d = float(np.max(np.abs(betas_py - betas_r)))
        rows.append(("Coefficients β (fixed components)",
                     f"max |Δ| = {d:.2e}", 'PASS' if d < 1e-6 else 'FAIL'))

        # Headline check: robust SEs vs clubSandwich CR2
        if np.isnan(se_cr2_r).any():
            rows.append(("Robust CR2 standard errors (vs clubSandwich)",
                         "clubSandwich call failed in R", 'WARN'))
        elif py_res.get('inference_se_method') != 'CR2-CRVE':
            rows.append(("Robust CR2 standard errors (vs clubSandwich)",
                         f"Python used '{py_res.get('inference_se_method')}', not CR2 "
                         f"— not comparable", 'WARN'))
        else:
            d = float(np.max(np.abs(se_py - se_cr2_r)))
            rel = float(np.max(np.abs(se_py - se_cr2_r) /
                               np.maximum(np.abs(se_cr2_r), 1e-12)))
            rows.append(("Robust CR2 standard errors (vs clubSandwich)",
                         f"max |Δ| = {d:.2e} &nbsp; (max rel = {rel:.2e})",
                         'PASS' if rel < 1e-3 else 'FAIL'))

        # Joint REML of tau² and sigma² under the spline model vs metafor (free fit)
        d_vc = max(abs(py_res['tau_sq'] - tau2_free), abs(py_res['sigma_sq'] - sig2_free))
        rows.append(("Variance components: Python joint REML vs metafor (free REML)",
                     f"τ²: {py_res['tau_sq']:.6f} vs {tau2_free:.6f} &nbsp;|&nbsp; "
                     f"σ²: {py_res['sigma_sq']:.6f} vs {sig2_free:.6f} &nbsp; (max |Δ| = {d_vc:.1e})",
                     'PASS' if d_vc < 1e-4 else 'FAIL'))

        # Omnibus — Python AHZ vs clubSandwich HTZ
        py_F = py_res['omnibus_F']
        py_dfd = py_res['omnibus_df_denom']
        py_p = py_res['omnibus_p']
        py_method = py_res.get('omnibus_p_method', '')

        if "AHZ" not in py_method:
            rows.append(("Omnibus Check: Method Used",
                         f"Expected AHZ, but ran fallback: '{py_method}'", 'WARN'))

        d_om = max(abs(py_F - omni_F_r) / max(abs(omni_F_r), 1e-12),
                   abs(py_dfd - omni_dfd_r) / max(abs(omni_dfd_r), 1e-12))
        rows.append(("Omnibus: Python AHZ vs clubSandwich HTZ",
                     f"F: {py_F:.4f} vs {omni_F_r:.4f} &nbsp;|&nbsp; "
                     f"df_den: {py_dfd:.3f} vs {omni_dfd_r:.3f} &nbsp;|&nbsp; "
                     f"p: {py_p:.4g} vs {omni_p_r:.4g} &nbsp; (max rel Δ F/df = {d_om:.1e})",
                     'PASS' if (d_om < 1e-4 and abs(py_p - omni_p_r) < 1e-4) else 'FAIL'))

        # ----------------------------------------------------------------
        # 6. RENDER
        # ----------------------------------------------------------------
        clear_output()
        any_fail = any(s == 'FAIL' for *_, s in rows)
        hbg, htx, htext = (('#d4edda', '#155724', '✅ Core numerics match metafor')
                           if not any_fail else
                           ('#f8d7da', '#721c24', '❌ Discrepancy detected'))

        n_studies = int(df_sim['id'].nunique())
        html = "<div style='font-family:sans-serif;'>"
        html += (f"<div style='padding:14px;background:{hbg};color:{htx};border-radius:6px;"
                 f"margin-bottom:12px;'><b style='font-size:15px;'>{htext}</b><br>"
                 f"<span style='font-size:13px;'>Curved-truth (sine) data · n={len(df_sim)} "
                 f"effects · {n_studies} studies · df_spline=3 · basis self-consistency "
                 f"{consistency:.1e}</span></div>")
        html += ("<table style='border-collapse:collapse;width:100%;font-size:13px;'>"
                 "<tr style='background:#f1f3f5;'>"
                 "<th style='text-align:left;padding:8px;'>Check</th>"
                 "<th style='text-align:left;padding:8px;'>Result</th>"
                 "<th style='padding:8px;'>Status</th></tr>")
        for name, res, st in rows:
            html += (f"<tr style='border-bottom:1px solid #eee;'>"
                     f"<td style='padding:8px;'>{name}</td>"
                     f"<td style='padding:8px;font-family:monospace;'>{res}</td>"
                     f"<td style='padding:8px;text-align:center;'>{_badge(st)}</td></tr>")
        html += "</table>"

        if captured:
            html += ("<div style='margin-top:10px;padding:8px;background:#fff8e1;"
                     "border-radius:4px;font-size:12px;'><b>Function warnings captured "
                     "during the run:</b><br>" + "<br>".join(captured) + "</div>")
        html += ("<div style='margin-top:10px;font-size:12px;color:#555;'>"
                 "Fixed-component rows test the GLS and CR2 algebra; the variance-"
                 "component and omnibus rows test joint REML and the HTZ test "
                 "against metafor's free REML fit and clubSandwich.</div></div>")
        display(HTML(html))

    except Exception:
        import traceback
        clear_output()
        display(HTML(
            "<div style='padding:15px;background:#f8d7da;border-left:5px solid #dc3545;"
            "color:#721c24;border-radius:4px;font-family:sans-serif;'>"
            "<h4 style='margin:0 0 6px 0;'>❌ Validation Error</h4>"
            f"<pre style='white-space:pre-wrap;margin:0;font-size:12px;'>"
            f"{traceback.format_exc()}</pre></div>"))
    finally:
        if original_config is not None:
            ANALYSIS_CONFIG.clear()
            ANALYSIS_CONFIG.update(original_config)
    return pd.DataFrame(rows, columns=['check', 'result', 'status']) if rows else None


res_df = _v4a_main() if _V6_RUN else None
if res_df is not None and 'record_validation' in globals():
    record_validation(6, "Spline Meta-Regression — Full Pipeline", res_df)



In [ ]:
#@title 🧪 Validation 7: HTZ Wald Test Helper on Your Data — q = 1 Self-Consistency + clubSandwich
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Checks the q-general HTZ Wald helper of _compute_robust_var_betas
# (with_wald_helper=True), which Steps 9 and 12 use for their omnibus tests
# and the subgroup contrast df, on the model design of YOUR analysis:
#   - your Step 9 subgroup model (the fitted unified model: rows, sampling V
#     with your shared-control covariances, design, tau², sigma²), and
#   - your Step 12 meta-regression (the fit's stacked design and V blocks);
#   - if neither was run: an intercept + one moderator of your data
#     (categorical if available, else continuous), fitted by REML.
# Check 1 (Python only): htz_fn(e_i') reproduces the per-coefficient
#   Satterthwaite df to 1e-10; for the fits of Steps 9 / 12, the helper's df
#   also reproduces the df the step reported (subgroup contrast df; Step 12
#   coefficient df).
# Check 2: HTZ F, denominator df and p vs clubSandwich::Wald_test(test =
#   "HTZ") on rma.mv(y, V, mods = ~ 0 + X, random = ~ 1 | study/effect) with
#   the variance components fixed at Python's, for the omnibus (all
#   non-intercept terms), two terms (q = 2) and a non-axis contrast (q = 1).
# Reference (VALIDATION_INCLUDE_REFERENCE): the fixed simulated sine dataset
# with the 3-df spline model (diagonal V), as before.

import numpy as np
import pandas as pd
import patsy
import warnings
import logging
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import rpy2.robjects.numpy2ri as numpy2ri
from scipy.linalg import block_diag
from IPython.display import display, HTML, clear_output

logging.getLogger('rpy2.rinterface_lib.callbacks').setLevel(logging.ERROR)
pandas2ri.activate()
numpy2ri.activate()

display(HTML("<div style='padding:15px;background:#e2e3e5;border-left:4px solid #6c757d;"
             "color:#383d41;border-radius:4px;font-family:sans-serif;'>"
             "⏳ <b>Validating HTZ helper (Check 1 + Check 2)…</b></div>"))

_V7_TITLE = "HTZ Wald Test Helper on Your Data — q = 1 Self-Consistency + clubSandwich"
_V7_NOT_MODERATORS = {'id', 'yi', 'vi', '_vcv_row_pos', 'row_id', 'shared_group_id', 'w_fixed', 'w_random',
                      'ci_width', 'xe', 'sde', 'ne', 'xc', 'sdc', 'nc', 'events_e', 'nonevents_e', 'events_c',
                      'nonevents_c', 'n_total', 'se', 'variance', 'sp', 'sp_squared', 'df', 'cohen_d',
                      'hedges_j', 'cv_e', 'cv_c'}


def _badge(kind):
    c = {'PASS': ('#d4edda', '#155724', '✅ PASS'),
         'FAIL': ('#f8d7da', '#721c24', '❌ FAIL'),
         'INFO': ('#d1ecf1', '#0c5460', 'ℹ️ INFO'),
         'WARN': ('#fff3cd', '#856404', '⚠️ CHECK')}[kind]
    return (f"<span style='background:{c[0]};color:{c[1]};padding:2px 8px;"
            f"border-radius:10px;font-size:12px;font-weight:600;'>{c[2]}</span>")


# ---------------------------------------------------------------------------
# DESIGNS: each is the (y, V, X) blocks per study of a fitted model + its
# variance components; 'expected_df' optionally lists (label, contrast row,
# df the analysis step reported) to check against the helper.
# ---------------------------------------------------------------------------
def _v7_split_stacked(fit_in):
    """Per-study blocks from a Step 12 fit's stacked inputs."""
    X = np.asarray(fit_in['X_stacked'], float)
    y = np.asarray(fit_in['y_stacked'], float).ravel()
    ids = [str(s) for s in fit_in['row_ids']]
    V_used = [np.atleast_2d(np.asarray(m, float)) for m in fit_in['vcv_used']]
    order = list(dict.fromkeys(ids))
    sizes = [V.shape[0] for V in V_used]
    if sum(sizes) != len(y) or len(order) != len(V_used):
        raise ValueError("the Step 12 fit's stacked rows and V blocks do not line up")
    y_b, X_b, pos = [], [], 0
    for k in sizes:                       # blocks are consecutive in fit order
        y_b.append(y[pos:pos + k]); X_b.append(X[pos:pos + k]); pos += k
    return y_b, V_used, X_b, order


def _v7_user_designs():
    u = validation_user_data()
    cfg, df, vcv = u['cfg'], u['df'], u['vcv']
    designs, notes = [], []

    # Step 9: the unified subgroup model
    sg = cfg.get('subgroup_results') or {}
    mf = sg.get('model_fit') or {}
    fit = mf.get('fit')
    if fit is not None:
        s = fit.spec
        L = np.asarray(mf['L'], float)
        res = sg.get('results_df')
        exp = []
        if isinstance(res, pd.DataFrame) and 'df_inference' in res.columns:
            r_ = res.set_index(res['group'].astype(str))
            for i, lv in enumerate(mf['levels']):
                if str(lv) in r_.index and 'AHZ' in str(r_.loc[str(lv), 'df_inference_method']):
                    exp.append((f"Step 9 df of group '{lv}'", L[i], float(r_.loc[str(lv), 'df_inference'])))
        mods = str(sg.get('moderator1')) + (f" × {sg.get('moderator2')}" if sg.get('moderator2') else '')
        designs.append({'name': f"your Step 9 subgroup model ({mods})", 'y': list(s.y_blocks),
                        'V': list(s.V_blocks), 'X': list(s.X_blocks), 'betas': np.asarray(fit.betas, float),
                        'tau2': float(fit.tau2), 'sigma2': float(fit.sigma2), 'terms': list(s.term_names or []),
                        'expected': exp, 'df_floor': 1.0,
                        'omnibus': (sg.get('F_omnibus'), sg.get('df_F_den'))})
    elif sg:
        notes.append("Step 9 used separate per-group fits (fallback), so its design was not checked.")

    # Step 12: the meta-regression fit
    last = globals().get('LAST_REGRESSION_RESULT')
    if last is not None and cfg.get('meta_regression_RVE_results') is not None:
        fit_in = getattr(last, 'fit_inputs', None) or {}
        if fit_in.get('X_stacked') is not None and fit_in.get('vcv_used') is not None:
            y_b, V_b, X_b, _ = _v7_split_stacked(fit_in)
            b = np.asarray(last.betas, float)
            exp = [(f"Step 12 df of {t}", np.eye(b.size)[i], float(np.asarray(last.df_resid, float)[i]))
                   for i, t in enumerate(last.term_names)]
            ov = (getattr(last, 'omnibus', None) or {}).get('overall') or {}
            designs.append({'name': f"your Step 12 meta-regression ({' + '.join(last.moderator_cols or [])})",
                            'y': y_b, 'V': V_b, 'X': X_b, 'betas': b, 'tau2': float(last.tau_sq),
                            'sigma2': float(last.sigma_sq), 'terms': list(last.term_names), 'expected': exp,
                            'df_floor': 0.0, 'omnibus': (ov.get('F'), ov.get('df_denom'))})
        else:
            notes.append("The Step 12 fit is the two-level aggregated model (no CR2 / HTZ), so it was not checked.")

    if not designs:
        # Fallback: intercept + one moderator of the user's data, fitted by REML
        excl = set(_V7_NOT_MODERATORS) | {cfg.get('effect_col'), cfg.get('var_col'), cfg.get('se_col')}
        cats, conts = [], []
        for c in df.columns:
            low = str(c).lower()
            if c in excl or 'imputed' in low or 'ci_' in low or 'shared_group' in low or 'lnrr' in low \
                    or 'ratio' in low or 'fold' in low or 'percent' in low:
                continue
            n_lev = int(df[c].nunique(dropna=True))
            if pd.api.types.is_numeric_dtype(df[c]):
                if n_lev >= 3:
                    conts.append(c)
            elif 2 <= n_lev <= max(2, len(df) // 2):
                cats.append(c)
        mod = None
        for c in cats:                        # categorical with >= 2 levels of >= 2 studies each
            d0 = df[df[c].notna()]
            ok = [lv for lv, g in d0.groupby(d0[c].astype(str)) if g['id'].nunique() >= 2]
            if len(ok) >= 2:
                mod, kind, lv_ok = c, 'categorical', ok
                break
        if mod is None and conts:
            mod, kind = conts[0], 'continuous'
        if mod is None:
            raise ValidationPrerequisiteError(
                "No moderator in your data to build a model design: add a moderator and run Steps 2-7 "
                "(or run Step 9 / Step 12) first.")
        d0 = df[df[mod].notna() & df['yi'].notna() & df['vi'].notna()].copy()
        if kind == 'categorical':
            d0 = d0[d0[mod].astype(str).isin(lv_ok)]
            levels = sorted(d0[mod].astype(str).unique())
            Xd = np.column_stack([np.ones(len(d0))] + [(d0[mod].astype(str) == lv).to_numpy(float)
                                                       for lv in levels[1:]])
            terms = ['Intercept'] + [f"{mod}[{lv}]" for lv in levels[1:]]
        else:
            x = pd.to_numeric(d0[mod], errors='coerce').to_numpy(float)
            Xd = np.column_stack([np.ones(len(d0)), x - x.mean()])
            terms = ['Intercept', f"{mod} (centred)"]
        d0['_row'] = np.arange(len(d0))
        studies = list(pd.unique(d0['id']))
        y_b, V_b, X_b = [], [], []
        for s_ in studies:
            g = d0[d0['id'] == s_]
            y_b.append(g['yi'].to_numpy(float))
            V_b.append(cluster_vcv(vcv, s_, g, 'vi')[0] if vcv else np.diag(g['vi'].to_numpy(float)))
            X_b.append(Xd[g['_row'].to_numpy()])
        f = fit_blocks(y_b, V_b, X_b, cluster_ids=studies)
        designs.append({'name': f"your data: intercept + {mod} ({kind}; REML fit)", 'y': y_b, 'V': V_b,
                        'X': X_b, 'betas': np.asarray(f.betas, float), 'tau2': float(f.tau2),
                        'sigma2': float(f.sigma2), 'terms': terms, 'expected': [], 'df_floor': 0.0,
                        'omnibus': (None, None)})
        notes.append(f"Steps 9 and 12 were not run: the helper was checked on an intercept + '{mod}' model "
                     f"fitted to your data with your sampling V.")
    return designs, notes


def _v7_reference_design():
    """The fixed simulated sine dataset with the 3-df spline (the original case)."""
    def generate_sine_data(n=80, tau2=0.2, sigma2=0.1, seed=42):
        rng = np.random.default_rng(seed)
        x = np.sort(rng.uniform(-3, 3, n))
        true_y = np.sin(x) + 0.5
        n_studies = n // 2
        ids = np.repeat(np.arange(n_studies), 2)
        u = np.repeat(rng.normal(0, np.sqrt(tau2), n_studies), 2)
        e = rng.normal(0, np.sqrt(sigma2), n)
        df = pd.DataFrame({'id': ids, 'y': true_y + u + e,
                           'vi': rng.uniform(0.05, 0.15, n), 'x': x})
        df['row_id'] = np.arange(len(df))
        return df

    df_sim = generate_sine_data()
    original_config = dict(ANALYSIS_CONFIG)
    try:
        ANALYSIS_CONFIG['analysis_data'] = df_sim
        ANALYSIS_CONFIG['vcv_matrices'] = {}
        with warnings.catch_warnings(record=True):
            warnings.simplefilter('always')
            py_res, err = _run_robust_spline_analysis(df_sim, 'x', 'y', 'vi', df_spline=3)
    finally:
        ANALYSIS_CONFIG.clear()
        ANALYSIS_CONFIG.update(original_config)
    if err:
        raise ValueError(f"Function returned error: {err}")
    if py_res.get('inference_se_method') != 'CR2-CRVE':
        raise ValueError(f"Function did not use CR2 inference (inference_se_method="
                         f"{py_res.get('inference_se_method')}); helper validation requires CR2.")
    reg = py_res['reg_df'].reset_index(drop=True)
    betas = np.asarray(py_res['betas'], float)
    X_full = np.asarray(patsy.dmatrix(py_res['formula'], {"x": reg['x'].values.astype(float)},
                                      return_type='matrix'))
    consistency = float(np.max(np.abs(X_full @ betas - np.asarray(py_res['fitted'], float))))
    if consistency > 1e-8:
        raise ValueError(f"Rebuilt basis does not match the function (max |Δ| = {consistency:.2e}). "
                         f"Cannot trust helper validation.")
    y_b, V_b, X_b, idx = [], [], [], 0
    for sid, grp in reg.groupby('id', sort=False):
        k = len(grp)
        y_b.append(grp['y'].values.astype(float))
        V_b.append(np.diag(grp['vi'].values.astype(float)))
        X_b.append(X_full[idx: idx + k])
        idx += k
    df_stored = np.asarray(py_res['dfs'], float)
    return {'name': 'reference: simulated sine data, 3-df spline (diagonal V)', 'y': y_b, 'V': V_b, 'X': X_b,
            'betas': betas, 'tau2': float(py_res['tau_sq']), 'sigma2': float(py_res['sigma_sq']),
            'terms': [f"β[{i}]" for i in range(betas.size)],
            'expected': [(f"spline df of β[{i}]", np.eye(betas.size)[i], df_stored[i]) for i in range(betas.size)],
            'df_floor': 0.0, 'omnibus': (py_res.get('omnibus_F'), py_res.get('omnibus_df_denom'))}


# ---------------------------------------------------------------------------
# CHECKS on one design
# ---------------------------------------------------------------------------
def _v7_check(des):
    p = int(des['betas'].size)
    _, df_vec, htz_fn = _compute_robust_var_betas(des['betas'], des['y'], des['V'], des['X'],
                                                  des['tau2'], des['sigma2'], with_wald_helper=True)
    df_vec = np.asarray(df_vec, float)
    I = np.eye(p)
    terms = des['terms'] if len(des['terms']) == p else [f"β[{i}]" for i in range(p)]
    check1 = []
    for i in range(p):
        out = htz_fn(I[i:i + 1])
        check1.append((f"htz_fn(e_{i}') df = df_vec[{i}] ({terms[i]})", out['df_denom'], df_vec[i],
                       abs(out['df_denom'] - df_vec[i])))
    for lab, c, df_rep in des['expected']:
        out = htz_fn(np.atleast_2d(c))
        dfh = max(des['df_floor'], float(out['df_denom'])) if des['df_floor'] else float(out['df_denom'])
        check1.append((f"helper df = {lab}", dfh, df_rep, abs(dfh - df_rep) / max(1.0, abs(df_rep))))
    check1_pass = all(d < 1e-10 for *_, d in check1)

    # R fit with the variance components fixed at Python's
    sizes = [len(y) for y in des['y']]
    ro.globalenv['y_v'] = ro.FloatVector(np.concatenate(des['y']))
    X = np.vstack(des['X'])
    ro.globalenv['X_py'] = ro.r['matrix'](ro.FloatVector(X.ravel(order='F')), nrow=X.shape[0], ncol=X.shape[1])
    V = block_diag(*des['V'])
    ro.globalenv['V_py'] = ro.r['matrix'](ro.FloatVector(V.ravel(order='F')), nrow=V.shape[0], ncol=V.shape[1])
    ro.globalenv['cl_v'] = ro.IntVector(np.repeat(np.arange(len(sizes)), sizes).tolist())
    ro.globalenv['fixed_tau2'], ro.globalenv['fixed_sigma2'] = des['tau2'], des['sigma2']
    ro.r("""
    suppressPackageStartupMessages({ library(metafor); library(clubSandwich) })
    dat <- data.frame(y = as.numeric(y_v), cl = factor(as.integer(cl_v)))
    dat$es <- factor(seq_len(nrow(dat)))
    res_fix <- rma.mv(yi = y, V = V_py, mods = ~ 0 + X_py, random = ~ 1 | cl/es, data = dat,
                      method = "REML", sigma2 = c(fixed_tau2, fixed_sigma2))
    """)
    cases = []
    if p >= 2:
        cases.append(('Omnibus (all non-intercept terms)', I[1:]))
    if p >= 3:
        cases.append((f"Two terms ({terms[1]}, {terms[2]})", I[1:3]))
        cases.append((f"Non-axis contrast ({terms[1]} − {terms[2]})", (I[1] - I[2])[None, :]))
    elif p == 2:
        cases.append((f"Non-axis contrast ({terms[0]} + {terms[1]})", (I[0] + I[1])[None, :]))
    check2 = []
    if check1_pass:
        for case_name, C in cases:
            py_out = htz_fn(C)
            ro.globalenv['C_test'] = ro.r['matrix'](ro.FloatVector(np.asarray(C, float).ravel(order='F')),
                                                    nrow=C.shape[0], ncol=C.shape[1])
            r_out = ro.r("""
            wt <- tryCatch(clubSandwich::Wald_test(res_fix, constraints = C_test, vcov = "CR2",
                                                   cluster = dat$cl, test = "HTZ"),
                           error = function(e) data.frame(Fstat = NA, df_num = NA, df_denom = NA, p_val = NA))
            c(as.numeric(wt$Fstat[1]), as.numeric(wt$df_num[1]), as.numeric(wt$df_denom[1]), as.numeric(wt$p_val[1]))
            """)
            r_F, r_dfn, r_dfd, r_p = [float(v) for v in np.asarray(r_out)]
            d_dfd = abs(py_out['df_denom'] - r_dfd) / max(abs(r_dfd), 1e-12)
            d_F = abs(py_out['F'] - r_F) / max(abs(r_F), 1e-12)
            status = 'PASS' if (d_dfd < 1e-6 and d_F < 1e-6 and abs(py_out['p_val'] - r_p) < 1e-8) else 'FAIL'
            check2.append(dict(case=case_name, q=C.shape[0], py_F=py_out['F'], r_F=r_F, dF=d_F,
                               py_dfd=py_out['df_denom'], r_dfd=r_dfd, d_dfd=d_dfd,
                               py_p=py_out['p_val'], r_p=r_p, status=status))
        # The omnibus the analysis step reported must be the helper's
        F_rep, dfd_rep = des['omnibus']
        if F_rep is not None and dfd_rep is not None and p >= 2 and np.isfinite(float(F_rep)):
            py_out = htz_fn(I[1:])
            d = max(abs(py_out['F'] - float(F_rep)), abs(py_out['df_denom'] - float(dfd_rep)))
            check2.append(dict(case='Reported omnibus = helper omnibus', q=p - 1, py_F=py_out['F'], r_F=float(F_rep),
                               dF=d, py_dfd=py_out['df_denom'], r_dfd=float(dfd_rep), d_dfd=d,
                               py_p=py_out['p_val'], r_p=np.nan, status='PASS' if d < 1e-8 else 'FAIL'))
    return {'name': des['name'], 'p': p, 'clusters': len(sizes), 'n': int(sum(sizes)),
            'check1': check1, 'check1_pass': check1_pass, 'check2': check2}


def _v7_render(results, notes):
    overall_pass = all(r['check1_pass'] and all(c['status'] == 'PASS' for c in r['check2']) for r in results)
    hbg, htx, htext = (('#d4edda', '#155724', '✅ HTZ helper validated')
                       if overall_pass else ('#f8d7da', '#721c24', '❌ Discrepancy detected'))
    html = "<div style='font-family:sans-serif;'>"
    html += (f"<div style='padding:14px;background:{hbg};color:{htx};border-radius:6px;margin-bottom:12px;'>"
             f"<b style='font-size:15px;'>{htext}</b><br><span style='font-size:13px;'>"
             + " · ".join(f"{r['name']}: p={r['p']}, {r['clusters']} clusters, {r['n']} rows" for r in results)
             + "</span>" + "".join(f"<br><span style='font-size:12px;'>ℹ️ {n}</span>" for n in notes) + "</div>")
    for r in results:
        html += f"<h4 style='margin:16px 0 6px 0;'>{r['name']}</h4>"
        html += "<div style='font-size:13px;margin:4px 0;'><b>Check 1 — q=1 self-consistency (Python only)</b></div>"
        html += ("<table style='border-collapse:collapse;width:100%;font-size:13px;'>"
                 "<tr style='background:#f1f3f5;'><th style='text-align:left;padding:8px;'>Check</th>"
                 "<th style='text-align:left;padding:8px;'>helper df</th>"
                 "<th style='text-align:left;padding:8px;'>expected df</th>"
                 "<th style='text-align:left;padding:8px;'>|Δ|</th><th style='padding:8px;'>Status</th></tr>")
        for (lab, h_dfd, v_dfd, d) in r['check1']:
            st = 'PASS' if d < 1e-10 else 'FAIL'
            html += (f"<tr style='border-bottom:1px solid #eee;'><td style='padding:8px;'>{lab}</td>"
                     f"<td style='padding:8px;font-family:monospace;'>{h_dfd:.10f}</td>"
                     f"<td style='padding:8px;font-family:monospace;'>{v_dfd:.10f}</td>"
                     f"<td style='padding:8px;font-family:monospace;'>{d:.2e}</td>"
                     f"<td style='padding:8px;text-align:center;'>{_badge(st)}</td></tr>")
        html += "</table>"
        if not r['check1_pass']:
            html += ("<div style='margin-top:10px;padding:10px;background:#f8d7da;color:#721c24;border-radius:4px;"
                     "font-size:13px;'><b>Stopping before Check 2.</b> If q=1 doesn't reduce to the existing df_vec, "
                     "the q-general formula is wrong and comparing it to clubSandwich would just confuse the "
                     "diagnosis. Inspect the helper's trace_P / sum_P_sq constructions against the q=1 "
                     "expressions in Pass 3.</div>")
            continue
        html += "<div style='font-size:13px;margin:10px 0 4px 0;'><b>Check 2 — q-general vs clubSandwich</b></div>"
        html += ("<table style='border-collapse:collapse;width:100%;font-size:13px;'>"
                 "<tr style='background:#f1f3f5;'><th style='text-align:left;padding:8px;'>Test constraint</th>"
                 "<th style='padding:8px;'>q</th><th style='text-align:left;padding:8px;'>df_denom (py vs R)</th>"
                 "<th style='text-align:left;padding:8px;'>F (py vs R)</th>"
                 "<th style='text-align:left;padding:8px;'>p (py vs R)</th><th style='padding:8px;'>Status</th></tr>")
        for c in r['check2']:
            html += (f"<tr style='border-bottom:1px solid #eee;'><td style='padding:8px;'>{c['case']}</td>"
                     f"<td style='padding:8px;text-align:center;'>{c['q']}</td>"
                     f"<td style='padding:8px;font-family:monospace;'>{c['py_dfd']:.4f} / {c['r_dfd']:.4f} "
                     f"<span style='color:#888;'>(rel {c['d_dfd']:.1e})</span></td>"
                     f"<td style='padding:8px;font-family:monospace;'>{c['py_F']:.3f} / {c['r_F']:.3f} "
                     f"<span style='color:#888;'>(rel {c['dF']:.1e})</span></td>"
                     f"<td style='padding:8px;font-family:monospace;'>{c['py_p']:.4g} / {c['r_p']:.4g}</td>"
                     f"<td style='padding:8px;text-align:center;'>{_badge(c['status'])}</td></tr>")
        html += "</table>"
    html += ("<div style='margin-top:12px;font-size:12px;color:#555;'>Reading guide: Check 1 PASS means the "
             "helper's q-general code reduces correctly to the existing q=1 path (machine epsilon) and gives the "
             "df your analysis reported. Check 2 PASS means the HTZ F statistic, its denominator df and the "
             "p-value all match clubSandwich's Wald_test(test = \"HTZ\") on your model design, and that the "
             "omnibus test the step reported is the helper's.</div></div>")
    clear_output()
    display(HTML(html))


try:
    if 'ANALYSIS_CONFIG' not in globals():
        ANALYSIS_CONFIG = {}
    notes = []
    try:
        designs, notes = _v7_user_designs()
    except ValidationPrerequisiteError:
        if not VALIDATION_INCLUDE_REFERENCE:
            raise
        designs, notes = [], ["No user analysis in this session: only the reference design was checked."]
    if VALIDATION_INCLUDE_REFERENCE:
        designs.append(_v7_reference_design())
    results = [_v7_check(d) for d in designs]
    _v7_render(results, notes)
    checks = []
    for r in results:
        checks += [(f"{r['name']} — Check 1: {lab}", 'PASS' if d < 1e-10 else 'FAIL') for (lab, _, _, d) in r['check1']]
        checks += [(f"{r['name']} — Check 2: {c['case']}", c['status']) for c in r['check2']]
    checks += [(f"notice: {n}", 'INFO') for n in notes]
    if 'record_validation' in globals():
        record_validation(7, _V7_TITLE, checks, notes=' '.join(notes))

except ValidationPrerequisiteError:
    clear_output()
    raise
except Exception:
    # A crash is a failed validation, not 'not run'
    import sys as _sys
    _exc = _sys.exc_info()[1]
    if 'record_validation' in globals() and not validation_prerequisite_missing(_exc):
        record_validation(7, _V7_TITLE, [('validation ran to completion', 'FAIL')],
                          notes=f"{type(_exc).__name__}: {_exc}")
    import traceback
    clear_output()
    display(HTML(
        "<div style='padding:15px;background:#f8d7da;border-left:5px solid #dc3545;"
        "color:#721c24;border-radius:4px;font-family:sans-serif;'>"
        "<h4 style='margin:0 0 6px 0;'>❌ Validation Error</h4>"
        f"<pre style='white-space:pre-wrap;margin:0;font-size:12px;'>"
        f"{traceback.format_exc()}</pre></div>"))


In [ ]:
#@title 🧪 Validation 8: Spline Edge Cases — Unbalanced Clusters & Correlated VCV
# Reference-only validation: synthetic edge cases (unbalanced clusters of 1-5
# effects, dense correlated VCV blocks, rho = 0.6) for the spline pipeline vs
# metafor + clubSandwich. The spline on YOUR data is checked by Validation 10;
# this synthetic case runs only when the reference datasets are switched on
# (VALIDATION_INCLUDE_REFERENCE).
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

_V8_RUN = reference_only_validation(8, "Spline Edge Cases — Unbalanced Clusters & Correlated VCV")

import numpy as np
import pandas as pd
import patsy
import warnings
import logging
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import rpy2.robjects.numpy2ri as numpy2ri
from IPython.display import display, HTML, clear_output

logging.getLogger('rpy2.rinterface_lib.callbacks').setLevel(logging.ERROR)
pandas2ri.activate()
numpy2ri.activate()

if _V8_RUN:
    display(HTML("<div style='padding:15px;background:#e2e3e5;border-left:4px solid #6c757d;"
                 "color:#383d41;border-radius:4px;font-family:sans-serif;'>"
                 "⏳ <b>Running Edge Case Tests (Unbalanced & Correlated VCV)…</b></div>"))

def _badge(kind):
    c = {'PASS': ('#d4edda', '#155724', '✅ PASS'),
         'FAIL': ('#f8d7da', '#721c24', '❌ FAIL'),
         'INFO': ('#d1ecf1', '#0c5460', 'ℹ️ INFO'),
         'WARN': ('#fff3cd', '#856404', '⚠️ CHECK')}[kind]
    return (f"<span style='background:{c[0]};color:{c[1]};padding:2px 8px;"
            f"border-radius:10px;font-size:12px;font-weight:600;'>{c[2]}</span>")

def _v4b_edge_main():
    """Main block, run in a function so its working variables do not leak
    into the notebook's shared namespace. The user's ANALYSIS_CONFIG is
    restored in the finally clause, also when a step fails."""
    global ANALYSIS_CONFIG
    if 'ANALYSIS_CONFIG' not in globals():
        ANALYSIS_CONFIG = {}
    original_config = None
    rows = []
    try:
        # ----------------------------------------------------------------
        # 1. EDGE CASE DATA: Unbalanced cluster sizes + Correlated VCV blocks
        # ----------------------------------------------------------------
        def generate_edge_case_data(seed=101):
            rng = np.random.default_rng(seed)
            # Unbalanced clusters: 30 studies, 1 to 5 effect sizes each
            cluster_sizes = rng.integers(1, 6, size=30)
            n = np.sum(cluster_sizes)
            ids = np.repeat(np.arange(30), cluster_sizes)

            # Curved truth (cosine this time)
            x = np.sort(rng.uniform(-2, 2, n))
            true_y = np.cos(x) * 0.8

            tau2, sigma2 = 0.15, 0.05
            u = np.repeat(rng.normal(0, np.sqrt(tau2), 30), cluster_sizes)
            e = rng.normal(0, np.sqrt(sigma2), n)

            vi = rng.uniform(0.04, 0.12, n)
            df = pd.DataFrame({'id': ids, 'y': true_y + u + e, 'vi': vi, 'x': x})
            df['row_id'] = np.arange(len(df))

            # Build dense block-diagonal VCV (rho = 0.6 correlation within studies)
            vcv_dict = {}
            V_full = np.zeros((n, n))
            rho = 0.6
            idx = 0

            for st_id, size in zip(range(30), cluster_sizes):
                v_block = np.diag(vi[idx:idx+size])
                # Inject correlation for multi-effect studies
                for i in range(size):
                    for j in range(size):
                        if i != j:
                            v_block[i, j] = rho * np.sqrt(v_block[i, i] * v_block[j, j])
                vcv_dict[st_id] = v_block
                V_full[idx:idx+size, idx:idx+size] = v_block
                idx += size

            return df, vcv_dict, V_full

        df_sim, vcv_dict, V_full = generate_edge_case_data()

        # Configure Python to use the dense VCV dictionary
        original_config = dict(ANALYSIS_CONFIG)
        ANALYSIS_CONFIG['analysis_data'] = df_sim
        ANALYSIS_CONFIG['vcv_matrices'] = vcv_dict

        # ----------------------------------------------------------------
        # 2. PYTHON SPLINE RUN
        # ----------------------------------------------------------------
        with warnings.catch_warnings(record=True) as wlist:
            warnings.simplefilter('always')
            py_res, err = _run_robust_spline_analysis(df_sim, 'x', 'y', 'vi', df_spline=3)
            captured = [str(w.message) for w in wlist]
        if err:
            raise ValueError(f"Python function returned an error: {err}")

        reg = py_res['reg_df'].reset_index(drop=True)
        betas_py = np.asarray(py_res['betas'], float)
        fitted_py = np.asarray(py_res['fitted'], float)
        se_py = np.asarray(py_res['se_betas'], float)
        p = len(betas_py)

        X = np.asarray(patsy.dmatrix(py_res['formula'],
                                     {"x": reg['x'].values.astype(float)},
                                     return_type='matrix'))

        consistency = float(np.max(np.abs(X @ betas_py - fitted_py)))
        if consistency > 1e-8:
            raise ValueError(f"Basis sync failed (max |Δ| = {consistency:.2e}).")

        # ----------------------------------------------------------------
        # 3. METAFOR RUN (Providing the full dense V matrix)
        # ----------------------------------------------------------------
        ro.globalenv['df_r'] = reg[['id', 'row_id', 'y', 'vi', 'x']]
        ro.globalenv['X_py'] = X
        ro.globalenv['V_r'] = V_full
        ro.globalenv['fixed_tau2'] = float(py_res['tau_sq'])
        ro.globalenv['fixed_sigma2'] = float(py_res['sigma_sq'])

        r_out = ro.r("""
        suppressPackageStartupMessages({ library(metafor); library(clubSandwich) })
        dat <- df_r
        dat$id     <- factor(dat$id)
        dat$row_id <- factor(dat$row_id)
        p <- ncol(X_py)

        # Use the dense V_r matrix directly in rma.mv
        res_fix  <- rma.mv(yi=y, V=V_r, mods=~0+X_py, random=~1|id/row_id,
                           data=dat, method="REML", sigma2=c(fixed_tau2, fixed_sigma2))
        res_free <- rma.mv(yi=y, V=V_r, mods=~0+X_py, random=~1|id/row_id,
                           data=dat, method="REML")

        se_cr2 <- tryCatch(as.numeric(robust(res_fix, cluster=dat$id, clubSandwich=TRUE)$se),
                           error=function(e) rep(NA_real_, p))
        wt <- tryCatch(clubSandwich::Wald_test(res_fix,
                           constraints=clubSandwich::constrain_zero(2:p),
                           vcov="CR2", cluster=dat$id, test="HTZ"),
                       error=function(e) data.frame(Fstat=NA, df_num=NA, df_denom=NA, p_val=NA))

        list(fitted   = as.numeric(fitted(res_fix)),
             betas    = as.numeric(coef(res_fix)),
             se_cr2   = se_cr2,
             tau2_free= as.numeric(res_free$sigma2[1]),
             sig2_free= as.numeric(res_free$sigma2[2]),
             omni_F   = as.numeric(wt$Fstat[1]),
             omni_dfd = as.numeric(wt$df_denom[1]),
             omni_p   = as.numeric(wt$p_val[1]))
        """)

        g = lambda k: np.asarray(r_out.rx2(k))
        fitted_r, betas_r, se_cr2_r = g('fitted'), g('betas'), g('se_cr2')
        tau2_free, sig2_free = float(g('tau2_free')[0]), float(g('sig2_free')[0])
        omni_F_r, omni_dfd_r, omni_p_r = (float(g('omni_F')[0]),
                                          float(g('omni_dfd')[0]),
                                          float(g('omni_p')[0]))


        # ----------------------------------------------------------------
        # 4. COMPARISONS
        # ----------------------------------------------------------------
        rows = []

        d = float(np.max(np.abs(fitted_py - fitted_r)))
        rows.append(("GLS fitted values (Dense VCV)",
                     f"max |Δ| = {d:.2e}", 'PASS' if d < 1e-6 else 'FAIL'))

        d = float(np.max(np.abs(betas_py - betas_r)))
        rows.append(("Coefficients β (Dense VCV)",
                     f"max |Δ| = {d:.2e}", 'PASS' if d < 1e-6 else 'FAIL'))

        if np.isnan(se_cr2_r).any():
            rows.append(("Robust CR2 standard errors", "R failed", 'WARN'))
        elif py_res.get('inference_se_method') != 'CR2-CRVE':
            rows.append(("Robust CR2 standard errors", "Python used Naive GLS", 'WARN'))
        else:
            d = float(np.max(np.abs(se_py - se_cr2_r)))
            rel = float(np.max(np.abs(se_py - se_cr2_r) / np.maximum(np.abs(se_cr2_r), 1e-12)))
            rows.append(("Robust CR2 standard errors",
                         f"max |Δ| = {d:.2e} &nbsp; (max rel = {rel:.2e})",
                         'PASS' if rel < 1e-3 else 'FAIL'))

        d_vc = max(abs(py_res['tau_sq'] - tau2_free), abs(py_res['sigma_sq'] - sig2_free))
        rows.append(("Variance components: Python joint REML vs metafor (free REML)",
                     f"τ²: {py_res['tau_sq']:.6f} vs {tau2_free:.6f} &nbsp;|&nbsp; "
                     f"σ²: {py_res['sigma_sq']:.6f} vs {sig2_free:.6f} &nbsp; (max |Δ| = {d_vc:.1e})",
                     'PASS' if d_vc < 1e-4 else 'FAIL'))

        py_F = py_res['omnibus_F']
        py_dfd = py_res['omnibus_df_denom']
        py_p = py_res['omnibus_p']
        py_method = py_res.get('omnibus_p_method', '')

        if "AHZ" not in py_method:
            rows.append(("Omnibus Check: Method Used", f"Expected AHZ, but ran fallback: '{py_method}'", 'WARN'))

        d_om = max(abs(py_F - omni_F_r) / max(abs(omni_F_r), 1e-12),
                   abs(py_dfd - omni_dfd_r) / max(abs(omni_dfd_r), 1e-12))
        rows.append(("Omnibus: Python AHZ vs clubSandwich HTZ",
                     f"F: {py_F:.4f} vs {omni_F_r:.4f} &nbsp;|&nbsp; "
                     f"df_den: {py_dfd:.3f} vs {omni_dfd_r:.3f} &nbsp;|&nbsp; "
                     f"p: {py_p:.4g} vs {omni_p_r:.4g} &nbsp; (max rel Δ F/df = {d_om:.1e})",
                     'PASS' if (d_om < 1e-4 and abs(py_p - omni_p_r) < 1e-4) else 'FAIL'))

        # ----------------------------------------------------------------
        # 5. RENDER
        # ----------------------------------------------------------------
        clear_output()
        any_fail = any(s == 'FAIL' for *_, s in rows)
        hbg, htx, htext = (('#d4edda', '#155724', '✅ Edge cases match metafor + clubSandwich')
                           if not any_fail else
                           ('#f8d7da', '#721c24', '❌ Discrepancy detected in Edge Cases'))

        n_studies = int(df_sim['id'].nunique())
        html = f"<div style='font-family:sans-serif;'>"
        html += (f"<div style='padding:14px;background:{hbg};color:{htx};border-radius:6px;"
                 f"margin-bottom:12px;'><b style='font-size:15px;'>{htext}</b><br>"
                 f"<span style='font-size:13px;'>Unbalanced data · Correlated VCV (ρ=0.6) · n={len(df_sim)} "
                 f"effects · {n_studies} studies · df_spline=3</span></div>")
        html += ("<table style='border-collapse:collapse;width:100%;font-size:13px;'>"
                 "<tr style='background:#f1f3f5;'>"
                 "<th style='text-align:left;padding:8px;'>Check</th>"
                 "<th style='text-align:left;padding:8px;'>Result</th>"
                 "<th style='padding:8px;'>Status</th></tr>")
        for name, res, st in rows:
            html += (f"<tr style='border-bottom:1px solid #eee;'>"
                     f"<td style='padding:8px;'>{name}</td>"
                     f"<td style='padding:8px;font-family:monospace;'>{res}</td>"
                     f"<td style='padding:8px;text-align:center;'>{_badge(st)}</td></tr>")
        html += "</table>"

        if captured:
            html += ("<div style='margin-top:10px;padding:8px;background:#fff8e1;"
                     "border-radius:4px;font-size:12px;'><b>Warnings Captured:</b><br>" +
                     "<br>".join(captured) + "</div>")
        html += "</div>"
        display(HTML(html))

    except Exception:
        import traceback
        clear_output()
        display(HTML(
            "<div style='padding:15px;background:#f8d7da;border-left:5px solid #dc3545;"
            "color:#721c24;border-radius:4px;font-family:sans-serif;'>"
            "<h4 style='margin:0 0 6px 0;'>❌ Validation Error</h4>"
            f"<pre style='white-space:pre-wrap;margin:0;font-size:12px;'>"
            f"{traceback.format_exc()}</pre></div>"))
    finally:
        if original_config is not None:
            ANALYSIS_CONFIG.clear()
            ANALYSIS_CONFIG.update(original_config)
    return pd.DataFrame(rows, columns=['check', 'result', 'status']) if rows else None


res_df = _v4b_edge_main() if _V8_RUN else None
if res_df is not None and 'record_validation' in globals():
    record_validation(8, "Spline Edge Cases — Unbalanced Clusters & Correlated VCV", res_df)



In [ ]:
#@title 🧪 Validation 9: Spline Curve Recovery — Monte Carlo (bias and band coverage)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Simulates three-level data around a known non-linear truth,
#     y = sin(x) + 0.5 + u_study + e_within + e_sampling(vi),
# fits the spline engine (_run_robust_spline_analysis, df_spline = 5) and
# evaluates the curve and pointwise confidence band exactly as the spline
# plot draws them (spline_prediction_band, Cell 1) on an interior grid.
#
# Checks per scenario:
#   1. every replicate fits;
#   2. mean pointwise coverage of the 95% band in SPL_COVERAGE_BAND
#      (wider than a single-interval band: a spline only approximates
#      sin(x), so small approximation bias remains);
#   3. no grid point where |mean bias| exceeds SPL_MAX_BIAS by more than
#      3 Monte Carlo SE (so the check does not depend on SPL_N_SIMS);
#   4. mean tau² within SPL_TAU2_REL of the truth (REML).
# No R comparison here: Validations 6, 8 and 10 compare the spline fit
# with metafor + clubSandwich directly.
# Requires: Cell 1.

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

SPL_N_SIMS = 100
SPL_COVERAGE_BAND = (0.90, 0.98)
SPL_MAX_BIAS = 0.05   # beyond 3 Monte Carlo SE, at any grid point
SPL_TAU2_REL = 0.25
SPL_SCENARIOS = [
    {'name': 'Few studies (k = 12)', 'n_studies': 12, 'tau_sq': 0.2, 'sigma_sq': 0.1},
    {'name': 'Moderate (k = 30)',    'n_studies': 30, 'tau_sq': 0.2, 'sigma_sq': 0.1},
    {'name': 'Large (k = 60)',       'n_studies': 60, 'tau_sq': 0.2, 'sigma_sq': 0.1},
]
SPL_GRID = np.linspace(-2.7, 2.7, 41)   # interior of the moderator range [-3, 3]


def _spl_simulate(sc, seed, per_study=2):
    rng = np.random.default_rng(seed)
    k, n = sc['n_studies'], sc['n_studies'] * per_study
    x = rng.uniform(-3, 3, n)
    vi = rng.uniform(0.05, 0.15, n)
    y = (np.sin(x) + 0.5 + np.repeat(rng.normal(0, np.sqrt(sc['tau_sq']), k), per_study)
         + rng.normal(0, np.sqrt(sc['sigma_sq']), n) + rng.normal(0, np.sqrt(vi)))
    return pd.DataFrame({'id': np.repeat([f"s{i:03d}" for i in range(k)], per_study),
                         'y': y, 'vi': vi, 'x': x})


def _v9_main():
    """Main block, run in a function so its working variables do not leak
    into the notebook's shared namespace."""
    if 'spline_prediction_band' not in globals():
        display(HTML("<div style='padding:12px; background:#fff3cd; border-radius:4px; font-family:sans-serif;'>"
                     "⚠️ Run Cell 1 first.</div>"))
        return None
    t0 = time.time()
    truth = np.sin(SPL_GRID) + 0.5
    rows, curves = [], {}
    for si, sc in enumerate(SPL_SCENARIOS):
        cov, yhat, tau2, n_fail = [], [], [], 0
        for i in range(SPL_N_SIMS):
            df = _spl_simulate(sc, seed=1000 * si + i)
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                est, err = _run_robust_spline_analysis(df, 'x', 'y', 'vi', df_spline=5)
            if err or est is None:
                n_fail += 1
                continue
            est['moderator_col'] = 'x'
            y_pred, _, lo, hi, _ = spline_prediction_band(est, SPL_GRID, alpha=0.05, dist_type='t')
            cov.append(np.mean((lo <= truth) & (truth <= hi)))
            yhat.append(y_pred)
            tau2.append(est['tau_sq'])
        curves[sc['name']] = np.mean(yhat, axis=0) if yhat else None
        mean_cov = float(np.mean(cov)) if cov else np.nan
        bias = np.mean(yhat, axis=0) - truth if yhat else np.full(len(truth), np.nan)
        bias_mcse = np.std(yhat, axis=0, ddof=1) / np.sqrt(len(yhat)) if len(yhat) > 1 else np.full(len(truth), np.nan)
        max_bias = float(np.max(np.abs(bias)))
        bias_excess = float(np.max(np.abs(bias) - 3 * bias_mcse))
        tau2_rel = float(np.mean(tau2) / sc['tau_sq'] - 1) if tau2 else np.nan
        checks = {'fits': n_fail == 0,
                  'coverage': SPL_COVERAGE_BAND[0] <= mean_cov <= SPL_COVERAGE_BAND[1],
                  'bias': bias_excess < SPL_MAX_BIAS,
                  'tau2': abs(tau2_rel) < SPL_TAU2_REL}
        rows.append({'scenario': sc['name'], 'fits': f"{SPL_N_SIMS - n_fail}/{SPL_N_SIMS}",
                     'mean pointwise coverage': mean_cov, 'max |mean bias|': max_bias,
                     'max (|bias| − 3 MC SE)': bias_excess,
                     'mean τ² (true 0.2)': float(np.mean(tau2)) if tau2 else np.nan,
                     'failed checks': ', '.join(k for k, ok in checks.items() if not ok),
                     'status': 'PASS' if all(checks.values()) else 'FAIL'})

    res_df = pd.DataFrame(rows)
    n_fail = int((res_df['status'] != 'PASS').sum())
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
        f"<b>{icon} Spline curve recovery: {len(res_df) - n_fail}/{len(res_df)} scenarios passed.</b><br>"
        f"<small>{SPL_N_SIMS} replicates per scenario, {(time.time() - t0) / 60:.1f} min. Criteria: all fits succeed; "
        f"mean pointwise coverage of the 95% band in [{SPL_COVERAGE_BAND[0]:.0%}, {SPL_COVERAGE_BAND[1]:.0%}]; "
        f"|mean bias| < {SPL_MAX_BIAS} + 3 MC SE at every grid point; mean τ² within ±{SPL_TAU2_REL:.0%} of the truth.</small></div>"))
    display(res_df.style.format({'mean pointwise coverage': '{:.3f}', 'max |mean bias|': '{:.3f}',
                                 'max (|bias| − 3 MC SE)': '{:.3f}', 'mean τ² (true 0.2)': '{:.3f}'}, na_rep=''))

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(SPL_GRID, truth, 'k', linewidth=3, label='True curve: sin(x) + 0.5')
    for (name, c), ls in zip(curves.items(), ('--', '-.', ':')):
        if c is not None:
            ax.plot(SPL_GRID, c, ls, linewidth=2, label=f'Mean estimate, {name}')
    ax.set_xlabel('Moderator'); ax.set_ylabel('Effect size')
    ax.set_title('Spline curve recovery (mean over replicates)')
    ax.legend(); ax.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout(); plt.show()
    return res_df


res_df = _v9_main()
if res_df is not None and 'record_validation' in globals():
    record_validation(9, "Spline Curve Recovery — Monte Carlo", res_df)


In [ ]:
#@title 🧪 Validation 10: Spline Model on Your Data (Python vs metafor + clubSandwich)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

import pandas as pd
import numpy as np
import patsy
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import rpy2.robjects.numpy2ri as numpy2ri
from scipy.linalg import block_diag
from IPython.display import display, HTML, clear_output

# Activate R conversions
pandas2ri.activate()
numpy2ri.activate()

display(HTML("<div style='padding: 15px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; border-radius: 4px; font-family: sans-serif;'>⏳ <b>Running Spline Cross-Validation with R (metafor)...</b> Please wait.</div>"))

try:
    # --- 1. Dependencies Check ---
    if 'ANALYSIS_CONFIG' not in globals():
        raise ValueError("Global configuration not found. Please run the Setup cells.")

    if 'spline_model_results' not in ANALYSIS_CONFIG:
        raise ValueError("Spline results not found. Please run the Spline Meta-Regression cell first.")

    py_res = ANALYSIS_CONFIG['spline_model_results']

    # --- 2. Data Preparation ---
    # Use the exact dataframe that the Python spline model used (already sorted and cleaned)
    df_py = py_res['reg_df'].reset_index(drop=True)
    effect_col = ANALYSIS_CONFIG.get('effect_col', 'yi')
    var_col = ANALYSIS_CONFIG.get('var_col', 'vi')
    mod_col = py_res['moderator_col']

    # --- 3. Rebuild the Exact Design Matrix (X) ---
    formula = py_res['formula']
    X_py = np.asarray(patsy.dmatrix(
        formula,
        {"x": df_py[mod_col].values.astype(float)},
        return_type='matrix'
    ))

    # --- 4. Matrix Injection (Shared Controls) ---
    use_vcv_matrix = False
    if 'vcv_matrices' in ANALYSIS_CONFIG and ANALYSIS_CONFIG['vcv_matrices']:
        vcv_dict = ANALYSIS_CONFIG['vcv_matrices']
        matrix_list = []
        unique_ids = df_py['id'].unique()
        missing_ids = [uid for uid in unique_ids if uid not in vcv_dict and str(uid) not in vcv_dict]

        if not missing_ids:
            for uid in unique_ids:
                mat = vcv_dict.get(uid, vcv_dict.get(str(uid)))
                matrix_list.append(mat)
            V_full = block_diag(*matrix_list)
            ro.globalenv['V_full'] = V_full
            use_vcv_matrix = True

    # Check for multi-level (if studies have multiple rows)
    is_multilevel = len(df_py['id'].unique()) < len(df_py)

    # Pass Data & Settings to R
    ro.globalenv['df_python'] = df_py[['id', effect_col, var_col]]
    ro.globalenv['X_py'] = X_py
    ro.globalenv['eff_col'] = effect_col
    ro.globalenv['var_col'] = var_col
    ro.globalenv['use_matrix'] = use_vcv_matrix
    ro.globalenv['is_multilevel'] = is_multilevel

    # --- 5. Run R (metafor + clubSandwich) ---
    r_script = """
    suppressPackageStartupMessages({ library(metafor); library(clubSandwich) })
    dat <- df_python
    dat$row_id <- 1:nrow(dat)
    dat$study_id <- as.factor(dat$id)
    p <- ncol(X_py)

    if (use_matrix) {
        V_input <- V_full
    } else {
        V_input <- dat[[var_col]]
    }

    if (is_multilevel) {
        res <- tryCatch({
            rma.mv(yi = dat[[eff_col]], V = V_input, mods = ~ 0 + X_py,
                   random = ~ 1 | study_id/row_id, data = dat, method = "REML")
        }, error = function(e) NULL)

        tau2 <- ifelse(is.null(res), NA, res$sigma2[1])
        sigma2 <- ifelse(is.null(res), NA, res$sigma2[2])
    } else {
        res <- tryCatch({
            rma.mv(yi = dat[[eff_col]], V = V_input, mods = ~ 0 + X_py,
                   random = ~ 1 | study_id, data = dat, method = "REML")
        }, error = function(e) NULL)

        tau2 <- ifelse(is.null(res), NA, res$sigma2[1])
        sigma2 <- 0
    }

    if (!is.null(res)) {
        cr2_test <- tryCatch(coef_test(res, vcov = "CR2", cluster = dat$study_id, test = "Satterthwaite"),
                             error = function(e) data.frame(SE=rep(NA, p)))

        wt <- tryCatch(clubSandwich::Wald_test(res, constraints = clubSandwich::constrain_zero(2:p),
                                               vcov = "CR2", cluster = dat$study_id, test = "HTZ"),
                       error = function(e) data.frame(Fstat=NA, df_num=NA, df_denom=NA, p_val=NA))

        list(
            betas = as.numeric(res$b),
            se_cr2 = as.numeric(cr2_test$SE),
            tau2 = tau2,
            sigma2 = sigma2,
            omni_F = as.numeric(wt$Fstat[1]),
            omni_dfd = as.numeric(wt$df_denom[1]),
            omni_p = as.numeric(wt$p_val[1])
        )
    } else {
        list(betas=NA, se_cr2=NA, tau2=NA, sigma2=NA, omni_F=NA, omni_dfd=NA, omni_p=NA)
    }
    """

    r_out = ro.r(r_script)

    if np.any(pd.isna(r_out.rx2('tau2'))):
        raise ValueError("R (metafor) failed to fit the model or converge.")

    # --- 6. Extract & Compare ---
    r_betas = np.asarray(r_out.rx2('betas'))
    r_se_cr2 = np.asarray(r_out.rx2('se_cr2'))

    py_betas = np.asarray(py_res['betas'])
    py_se_cr2 = np.asarray(py_res['se_betas'])
    py_method = py_res.get('omnibus_p_method', 'Unknown Method')

    max_beta_diff = np.max(np.abs(py_betas - r_betas))
    max_se_diff = np.max(np.abs(py_se_cr2 - r_se_cr2)) if not np.isnan(r_se_cr2).any() else np.nan

    comparison_data = [
        {'Parameter': 'Max Coefficient Difference (|Δβ|)', 'Co-Meta (Python)': f"{max_beta_diff:.2e}", 'metafor (R)': "Baseline"},
        {'Parameter': 'Max Standard Error Difference (|ΔSE|)', 'Co-Meta (Python)': f"{max_se_diff:.2e}", 'metafor (R)': "Baseline"},
        {'Parameter': 'Tau² (Between-Study Variance)', 'Co-Meta (Python)': f"{py_res['tau_sq']:.5f}", 'metafor (R)': f"{r_out.rx2('tau2')[0]:.5f}"}
    ]

    if is_multilevel:
        comparison_data.append(
            {'Parameter': 'Sigma² (Within-Study Variance)', 'Co-Meta (Python)': f"{py_res['sigma_sq']:.5f}", 'metafor (R)': f"{r_out.rx2('sigma2')[0]:.5f}"}
        )

    comparison_data.extend([
        {'Parameter': 'Omnibus Method Used', 'Co-Meta (Python)': py_method, 'metafor (R)': "HTZ (clubSandwich)"},
        {'Parameter': 'Omnibus F-Statistic', 'Co-Meta (Python)': f"{py_res['omnibus_F']:.3f}", 'metafor (R)': f"{r_out.rx2('omni_F')[0]:.3f}"},
        {'Parameter': 'Omnibus Denominator df', 'Co-Meta (Python)': f"{py_res['omnibus_df_denom']:.1f}", 'metafor (R)': f"{r_out.rx2('omni_dfd')[0]:.1f}"},
        {'Parameter': 'Omnibus p-value', 'Co-Meta (Python)': f"{py_res['omnibus_p']:.5f}", 'metafor (R)': f"{r_out.rx2('omni_p')[0]:.5f}"}
    ])

    # --- 7. Pass / fail (R estimates everything itself: free REML) ---
    TOL_SPL = {'beta': 1e-5, 'se_rel': 1e-3, 'var': 1e-5, 'F_rel': 1e-4, 'df_rel': 1e-3, 'p': 1e-4}
    _r = lambda k: float(np.asarray(r_out.rx2(k))[0])
    _rel = lambda a, b: abs(a - b) / max(abs(b), 1e-12)
    checks = {
        'coefficients': max_beta_diff < TOL_SPL['beta'],
        'CR2 SE': bool(np.max(np.abs(py_se_cr2 - r_se_cr2) / np.abs(r_se_cr2)) < TOL_SPL['se_rel']),
        'τ²': abs(py_res['tau_sq'] - _r('tau2')) < TOL_SPL['var'],
        'σ²': (abs(py_res['sigma_sq'] - _r('sigma2')) < TOL_SPL['var']) if is_multilevel else True,
        'omnibus F': _rel(py_res['omnibus_F'], _r('omni_F')) < TOL_SPL['F_rel'],
        'omnibus df': _rel(py_res['omnibus_df_denom'], _r('omni_dfd')) < TOL_SPL['df_rel'],
        'omnibus p': abs(py_res['omnibus_p'] - _r('omni_p')) < TOL_SPL['p'],
    }
    status_of = {'Max Coefficient Difference (|Δβ|)': 'coefficients', 'Max Standard Error Difference (|ΔSE|)': 'CR2 SE',
                 'Tau² (Between-Study Variance)': 'τ²', 'Sigma² (Within-Study Variance)': 'σ²',
                 'Omnibus F-Statistic': 'omnibus F', 'Omnibus Denominator df': 'omnibus df', 'Omnibus p-value': 'omnibus p'}
    for row in comparison_data:
        k = status_of.get(row['Parameter'])
        row['Status'] = ('PASS' if checks[k] else 'FAIL') if k else ''
    df_comp = pd.DataFrame(comparison_data)
    spline_validation_pass = all(checks.values())
    if 'record_validation' in globals():
        record_validation(10, "Spline Model on Your Data", [(k, 'PASS' if v else 'FAIL') for k, v in checks.items()],
                          notes=f"Python method: {py_method}")

    clear_output()
    _ok = spline_validation_pass
    display(HTML(f"""
    <div style='padding: 15px; background-color: {"#d4edda" if _ok else "#f8d7da"}; border-left: 5px solid {"#28a745" if _ok else "#dc3545"}; color: {"#155724" if _ok else "#721c24"}; border-radius: 4px; font-family: sans-serif; margin-bottom: 15px;'>
        <h4 style='margin: 0 0 5px 0;'>{"✅ Spline engine matches metafor + clubSandwich on your data" if _ok else "❌ Spline discrepancy: " + ", ".join(k for k, v in checks.items() if not v)}</h4>
        <p style='margin: 0; font-size: 14px;'>Your last spline fit refitted in R (free REML, same rows and sampling V). Tolerances: {TOL_SPL}.</p>
    </div>
    """))

    if "AHZ" not in py_method:
        display(HTML(f"""
        <div style='background-color: #fff3cd; padding: 12px; border-radius: 5px; border-left: 4px solid #ffc107; margin-bottom: 15px; font-family: sans-serif; font-size: 13px; color: #856404;'>
            <b>⚠️ Fallback Method Detected:</b> Python was unable to use the AHZ method (likely due to sparse data or singular matrices) and fell back to: <i>{py_method}</i>. Discrepancies with R are expected here.
        </div>
        """))

    # Display the clean table
    display(df_comp.style.set_properties(**{'text-align': 'center'}))


except Exception as e:
    # A crash is a failed validation, not 'not run'
    if 'record_validation' in globals() and not validation_prerequisite_missing(e):
        record_validation(10, 'Spline Model on Your Data',
                          [('validation ran to completion', 'FAIL')],
                          notes=f"{type(e).__name__}: {e}")
    import traceback
    clear_output()
    display(HTML(f"""
    <div style='padding: 15px; background-color: #f8d7da; border-left: 5px solid #dc3545; color: #721c24; border-radius: 4px; font-family: sans-serif;'>
        <h4 style='margin: 0 0 5px 0;'>❌ Validation Error</h4>
        <pre style='white-space:pre-wrap;margin:0;font-size:12px;'>{traceback.format_exc()}</pre>
    </div>
    """))

In [ ]:
#@title 🧪 Validation 11: Automated Unit Tests (ipytest)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

import sys
import subprocess
from IPython.display import display, HTML, clear_output

display(HTML("<div style='padding: 15px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; border-radius: 4px; font-family: sans-serif;'>⏳ <b>Running Automated Software Unit Tests...</b> Please wait.</div>"))

# 1. Install ipytest quietly
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pytest", "ipytest"])

import ipytest
import pytest
import json
import numpy as np
import pandas as pd

# Configure ipytest to work seamlessly inside Colab
ipytest.autoconfig()

# =============================================================================
# 2. DEFINE TESTS (These run directly in the notebook's memory)
# =============================================================================

def test_hedges_g_exact_correction():
    """Verify the exact Gamma correction factor used in Cell 6"""
    # For df=10, the exact J correction factor is approx 0.9227
    j_val = _hedges_j(10)
    assert np.isclose(j_val, 0.92274, atol=0.0001)

    # For df=2, J is approx 0.5641
    j_val_small = _hedges_j(2)
    assert np.isclose(j_val_small, 0.56418, atol=0.0001)

def test_dl_estimator_edge_cases():
    """Test DerSimonian-Laird estimator handles zero heterogeneity"""
    df_zero = pd.DataFrame({'yi': [0.5, 0.5, 0.5], 'vi': [0.1, 0.1, 0.1]})

    tau2_zero = calculate_tau_squared_DL(df_zero, 'yi', 'vi')
    assert tau2_zero == 0.0

def test_variance_weights():
    """Test inverse variance weighting math logic"""
    variances = np.array([0.1, 0.2, 0.5])
    tau2 = 0.1
    weights = 1 / (variances + tau2)

    assert np.isclose(weights[0], 5.0)  # 1 / (0.1 + 0.1)
    assert np.isclose(weights[1], 3.333, atol=0.001) # 1 / (0.2 + 0.1)

def test_reproducibility_json_stripping():
    """Test that the JSON exporter correctly strips non-serializable objects"""
    mock_config = {
        'alpha': 0.05,
        'clean_dataframe': pd.DataFrame({'a': [1,2]}), # Should be embedded as CSV
        'complex_object': set([1, 2, 3]) # Should be skipped/converted
    }

    safe_json = export_config_for_reproducibility(mock_config)

    assert safe_json['_reproducibility']['is_reproducing'] == True
    assert isinstance(safe_json['embedded_data'], str)
    assert 'a' in safe_json['embedded_data']

def test_user_control_column_overrides_detection():
    """A 'control_id' column replaces the automatic matching: identical control
    statistics with different labels are NOT shared, and vice versa."""
    df = pd.DataFrame({'id': ['A', 'A', 'A', 'B', 'B'],
                       'nc': [10, 10, 10, 8, 8], 'xc': [1.0, 1.0, 1.0, 2.0, 2.5],
                       'sdc': [.5, .5, .5, .4, .4],
                       'Control_ID': ['c1', 'c1', 'c2', 'k', 'k']})
    out, n = detect_shared_controls(df)
    assert n == 4
    assert out.loc[0, 'shared_group_id'] == out.loc[1, 'shared_group_id'] is not None
    assert out.loc[2, 'shared_group_id'] is None
    assert out.loc[3, 'shared_group_id'] == out.loc[4, 'shared_group_id'] is not None
    out2, n2 = detect_shared_controls(df.drop(columns='Control_ID'))   # automatic
    assert n2 == 3 and out2.loc[3, 'shared_group_id'] is None

def test_detect_shared_controls():
    """Verify the algorithm detects identical control groups within a study"""
    # Create mock data where Treatment A and B share Control C
    df = pd.DataFrame({
        'id': ['Study1', 'Study1', 'Study2'],
        'nc': [10, 10, 15],
        'xc': [5.5, 5.5, 8.0],
        'sdc': [1.1, 1.1, 2.0]
    })

    df_processed, shared_count = detect_shared_controls(df)

    # Study 1 rows should have a shared_group_id, Study 2 should be None
    assert shared_count == 2
    assert pd.notna(df_processed.loc[0, 'shared_group_id'])
    assert df_processed.loc[0, 'shared_group_id'] == df_processed.loc[1, 'shared_group_id']
    assert pd.isna(df_processed.loc[2, 'shared_group_id'])

def test_vcv_matrix_shared_controls():
    """Verify covariance is calculated for shared controls (Gleser & Olkin)"""
    # Mock data with a shared control, already flagged by detection algorithm
    df = pd.DataFrame({
        'id': ['Study1', 'Study1'],
        'shared_group_id': ['grp1', 'grp1'],
        'yi': [0.5, 0.8],     # Effect sizes
        'vi': [0.1, 0.15],    # Variances
        'ne': [20, 20],       # Exp sample sizes
        'nc': [20, 20]        # Shared control sample size
    })

    # Build VCV matrix for Cohen's d
    vcv_dict = build_vcv_matrices(df, effect_type='cohen_d', var_col_name='vi',
                                  yi_col='yi', nt_col='ne', nc_col='nc')

    matrix = vcv_dict['Study1']

    # Assert it created a 2x2 matrix
    assert matrix.shape == (2, 2)
    # Assert diagonal matches the variances (0.1, 0.15)
    assert np.isclose(matrix[0, 0], 0.1)
    assert np.isclose(matrix[1, 1], 0.15)
    # CRITICAL: Assert off-diagonal covariance is positive (shared control)
    assert matrix[0, 1] > 0.0
    assert matrix[0, 1] == matrix[1, 0] # Must be symmetric

def test_impute_sd_median_cv():
    """Verify missing standard deviations are imputed correctly"""
    # Data: 2 valid rows with CV = 0.2 (SD / Mean = 2/10 and 4/20)
    # 1 row missing SD with Mean = 30. It should impute SD = 30 * 0.2 = 6.0
    df = pd.DataFrame({
        'sde': [2.0, 4.0, np.nan],
        'xe': [10.0, 20.0, 30.0],
        'ne': [10, 10, 10]
    })

    imputed_sde = impute_sd_median_cv(df, 'sde', 'xe')

    assert pd.notna(imputed_sde.iloc[2])
    assert np.isclose(imputed_sde.iloc[2], 6.0)

# =============================================================================
#  ADDITIONAL UNIT TESTS
#  All inputs are tiny and hand-verified; no notebook globals are referenced.
# =============================================================================

# ---------- τ² estimators ----------------------------------------------------

def test_dl_estimator_known_value():
    """DL on yi=[0.1,0.2,0.3,0.4], vi=0.01: hand-calc gives τ²=(Q-3)/C=2/300≈0.00667."""
    df = pd.DataFrame({'yi': [0.1, 0.2, 0.3, 0.4],
                       'vi': [0.01, 0.01, 0.01, 0.01]})
    # FE weights w=100, μ_FE=0.25, Q=100*(0.0225+0.0025+0.0025+0.0225)=5,
    # C = 400 - 40000/400 = 300, τ² = (5-3)/300.
    tau2 = calculate_tau_squared_DL(df, 'yi', 'vi')
    assert np.isclose(tau2, 2.0 / 300.0, atol=1e-8)

def test_reml_estimator_zero_heterogeneity():
    """REML on perfectly homogeneous data should converge to τ²≈0."""
    df = pd.DataFrame({'yi': [0.4, 0.4, 0.4, 0.4],
                       'vi': [0.05, 0.05, 0.05, 0.05]})
    tau2 = calculate_tau_squared_REML(df, 'yi', 'vi')
    assert tau2 >= 0.0
    assert np.isclose(tau2, 0.0, atol=1e-4)

def test_reml_estimator_positive_heterogeneity():
    """REML must return a strictly positive τ² when effects clearly differ."""
    df = pd.DataFrame({'yi': [-1.0, -0.5, 0.0, 0.5, 1.0],
                       'vi': [0.01, 0.01, 0.01, 0.01, 0.01]})
    tau2 = calculate_tau_squared_REML(df, 'yi', 'vi')
    assert tau2 > 0.1                       # spread >> sampling variance
    assert tau2 < 2.0                       # sane upper bound

def test_ml_underestimates_reml():
    """Classical result: ML τ² ≤ REML τ² on the same data."""
    df = pd.DataFrame({'yi': [-1.0, -0.5, 0.0, 0.5, 1.0],
                       'vi': [0.05, 0.05, 0.05, 0.05, 0.05]})
    tau2_ml   = calculate_tau_squared_ML(df, 'yi', 'vi')
    tau2_reml = calculate_tau_squared_REML(df, 'yi', 'vi')
    assert tau2_ml <= tau2_reml + 1e-6

def test_pm_estimator_solves_q_equals_k_minus_1():
    """Paule–Mandel: Q(τ²) should land at k-1 at the solution."""
    df = pd.DataFrame({'yi': [-0.4, -0.1, 0.2, 0.5],
                       'vi': [0.02, 0.02, 0.02, 0.02]})
    tau2 = calculate_tau_squared_PM(df, 'yi', 'vi')
    w   = 1.0 / (df['vi'].values + tau2)
    mu  = (w * df['yi'].values).sum() / w.sum()
    Q   = (w * (df['yi'].values - mu) ** 2).sum()
    assert np.isclose(Q, len(df) - 1, atol=1e-3)

def test_sj_two_studies_matches_metafor():
    """SJ with k=2 (metafor computes it; no DL fallback)."""
    df = pd.DataFrame({'yi': [0.1, 0.6], 'vi': [0.04, 0.09]})
    assert np.isclose(calculate_tau_squared_SJ(df, 'yi', 'vi'), 0.0612745098, rtol=1e-8)

def test_sj_falls_back_with_two_studies():
    """SJ with k=2 identical estimates: τ²=0 (tau0² = 0)."""
    df = pd.DataFrame({'yi': [0.3, 0.3], 'vi': [0.1, 0.1]})
    tau2 = calculate_tau_squared_SJ(df, 'yi', 'vi')
    assert np.isclose(tau2, 0.0, atol=1e-12)

def test_pm_matches_metafor():
    """PM must reproduce metafor::rma(method = "PM")$tau2 (metafor 5.2.1)."""
    df = pd.DataFrame({'yi': [-0.30, 0.05, 0.12, 0.48, 0.71, 0.20],
                       'vi': [0.04, 0.02, 0.09, 0.05, 0.03, 0.06]})
    tau2 = calculate_tau_squared_PM(df, 'yi', 'vi')
    assert np.isclose(tau2, 0.0870504610, rtol=1e-8)

def test_sj_matches_metafor():
    """SJ must reproduce metafor::rma(method = "SJ")$tau2 (metafor 5.2.1)."""
    df = pd.DataFrame({'yi': [-0.30, 0.05, 0.12, 0.48, 0.71, 0.20],
                       'vi': [0.04, 0.02, 0.09, 0.05, 0.03, 0.06]})
    tau2 = calculate_tau_squared_SJ(df, 'yi', 'vi')
    assert np.isclose(tau2, 0.0911164759, rtol=1e-8)

def test_calculate_tau_squared_dispatcher_selects_method():
    """Dispatcher must return the same value as the underlying estimator and
    correctly populate the info dict."""
    df = pd.DataFrame({'yi': [0.1, 0.2, 0.3, 0.4],
                       'vi': [0.01, 0.01, 0.01, 0.01]})
    tau2_disp, info = calculate_tau_squared(df, 'yi', 'vi', method='DL')
    tau2_direct = calculate_tau_squared_DL(df, 'yi', 'vi')
    assert np.isclose(tau2_disp, tau2_direct)
    assert info['method'] == 'DL'
    assert np.isclose(info['tau'], np.sqrt(tau2_direct))


# ---------- Pooled estimate, CI, I² -----------------------------------------

def test_re_pooled_zero_heterogeneity():
    """With identical effects: pooled = effect, I² = 0, CI symmetric in z·SE."""
    df = pd.DataFrame({'yi': [0.5, 0.5, 0.5], 'vi': [0.1, 0.1, 0.1]})
    mu, se, lo, hi, I2 = calculate_re_pooled(df, tau_squared=0.0,
                                             effect_col='yi', var_col='vi')
    assert np.isclose(mu, 0.5)
    assert np.isclose(se, np.sqrt(1.0 / 30.0), atol=1e-8)
    assert np.isclose(I2, 0.0, atol=1e-8)
    assert np.isclose(lo, 0.5 - 1.959964 * se, atol=1e-3)
    assert np.isclose(hi, 0.5 + 1.959964 * se, atol=1e-3)

def test_re_pooled_high_heterogeneity_yields_large_I2():
    """Highly heterogeneous data should produce I² > 75%."""
    df = pd.DataFrame({'yi': [-1.0, -0.5, 0.0, 0.5, 1.0],
                       'vi': [0.001, 0.001, 0.001, 0.001, 0.001]})
    _, _, _, _, I2 = calculate_re_pooled(df, tau_squared=0.5,
                                         effect_col='yi', var_col='vi')
    assert I2 > 75.0


# ---------- Knapp–Hartung ----------------------------------------------------

def test_knapp_hartung_basic_structure():
    """KH adjustment returns finite values and a CI that brackets the mean."""
    yi = np.array([0.2, 0.4, 0.1, 0.3, 0.5])
    vi = np.array([0.04, 0.05, 0.03, 0.04, 0.06])
    tau_sq = 0.02
    w = 1.0 / (vi + tau_sq)
    mu = (w * yi).sum() / w.sum()
    out = calculate_knapp_hartung_ci(yi, vi, tau_sq, mu, alpha=0.05)
    assert out is not None
    assert out['ci_lower'] < mu < out['ci_upper']
    assert out['se_KH'] > 0
    assert out['df'] == len(yi) - 1
    assert 0.0 <= out['p_value'] <= 1.0

def test_knapp_hartung_returns_none_for_k_equals_one():
    """With a single study (df=0) KH cannot be computed."""
    out = calculate_knapp_hartung_ci(np.array([0.5]), np.array([0.1]),
                                     tau_sq=0.0, pooled_effect=0.5)
    assert out is None


# ---------- Hedges' g pooled SD path -----------------------------------------

def test_calculate_hedges_g_python_simple():
    """ne=nc=10, sde=sdc=2, xe-xc=2 ⇒ d=1, g=1·J(18)≈0.9579 (positive, <1)."""
    df = pd.DataFrame({'ne': [10], 'nc': [10],
                       'sde': [2.0], 'sdc': [2.0],
                       'xe': [12.0], 'xc': [10.0]})
    g, vg = calculate_hedges_g_python(df)
    j_18 = _hedges_j(18)
    assert np.isclose(g.iloc[0], 1.0 * j_18, atol=1e-6)
    assert vg.iloc[0] > 0
    # variance formula (metafor escalc "SMD"): 20/100 + g²/40
    expected_vg = 0.2 + (g.iloc[0] ** 2) / 40.0
    assert np.isclose(vg.iloc[0], expected_vg, atol=1e-8)

def test_calculate_hedges_g_python_zero_difference():
    """Equal means ⇒ d=g=0 and vg = (ne+nc)/(ne·nc)."""
    df = pd.DataFrame({'ne': [15], 'nc': [15],
                       'sde': [1.5], 'sdc': [1.5],
                       'xe': [4.0], 'xc': [4.0]})
    g, vg = calculate_hedges_g_python(df)
    j_28 = _hedges_j(28)
    assert np.isclose(g.iloc[0], 0.0, atol=1e-12)
    assert np.isclose(vg.iloc[0], 30.0 / 225.0, atol=1e-10)


# ---------- Shared fitting adapters (Cell 1b, section 8) ---------------------

def test_fit_blocks_reduces_to_fixed_effects():
    """With tau² = sigma² = 0 held fixed, the engine must reproduce the
    inverse-variance fixed-effects pooled mean and its variance."""
    y_all   = [np.array([0.5]), np.array([0.7]), np.array([0.3])]
    vcv_all = [np.array([[0.1]]), np.array([[0.1]]), np.array([[0.1]])]
    out = fit_estimates(fit_blocks(y_all, vcv_all, tau2=0.0, sigma2=0.0))
    # FE: w=10 each, mu=(0.5+0.7+0.3)/3 = 0.5, var=1/30
    assert np.isclose(out['mu'], 0.5, atol=1e-10)
    assert np.isclose(out['var_mu'], 1.0 / 30.0, atol=1e-10)
    assert np.isfinite(out['log_lik_reml'])

def test_fit_blocks_with_shared_control_block():
    """A 2x2 dense VCV with off-diagonal covariance must invert and yield mu
    between the two effect sizes."""
    V = np.array([[0.05, 0.01],
                  [0.01, 0.05]])           # positive-definite
    out = fit_estimates(fit_blocks([np.array([0.4, 0.6])], [V], tau2=0.0, sigma2=0.0))
    assert 0.4 <= out['mu'] <= 0.6
    assert out['var_mu'] > 0

def test_fit_blocks_two_level_without_replication():
    """sigma² is fixed at 0 when no cluster has more than one effect size
    (tau² and sigma² are not separately identified), estimated otherwise."""
    rng = np.random.default_rng(0)
    one = [np.array([v]) for v in rng.normal(0.3, 0.4, 12)]
    fit1 = fit_blocks(one, [np.array([[0.05]])] * 12)
    assert fit1.sigma2_fixed == 0.0 and fit1.sigma2 == 0.0 and fit1.tau2 > 0
    many = [rng.normal(0.3, 0.4, 2) for _ in range(12)]
    fit2 = fit_blocks(many, [np.diag([0.05, 0.05])] * 12)
    assert fit2.sigma2_fixed is None and fit2.tau2_fixed is None

def test_fit_blocks_falls_back_to_diagonal_V():
    """A sampling V that is not positive definite (and stays so at every
    variance-component value) is replaced by the fallback, and the fit says so."""
    bad = np.array([[0.05, 0.2], [0.2, 0.05]])      # eigenvalues 0.25, -0.15
    y = [np.array([0.2, 0.5]), np.array([0.1, 0.4]), np.array([0.3, 0.6])]
    fit = fit_blocks(y, [bad] * 3, V_fallback_blocks=[np.diag([0.05, 0.05])] * 3)
    assert fit.v_fallback and fit.notes

def test_profile_ci_bounds_sit_on_the_threshold():
    """At each finite profile-CI bound the profile log-likelihood is
    chi2_{0.95}(1) / 2 below the maximum (as metafor's confint)."""
    from scipy.stats import chi2
    rng = np.random.default_rng(3)
    y = [0.3 + rng.normal(0, 0.4) + rng.normal(0, 0.25, 3) for _ in range(15)]
    fit = fit_blocks(y, [np.diag([0.04, 0.05, 0.06])] * 15)
    lo, hi = profile_ci(fit, 'tau2', 0.05)
    drop = 0.5 * chi2.ppf(0.95, 1)
    for b in (lo, hi):
        if 0 < b < np.inf:
            assert np.isclose(fit.loglik - fit_mv(fit.spec, tau2=b).loglik, drop, atol=1e-6)
    assert lo <= fit.tau2 <= hi


# ---------- Imputation helpers ----------------------------------------------

def test_impute_sd_median_cv_leaves_zero_mean_missing():
    """A zero mean gives no SD from a CV: the SD stays missing (row dropped and logged later)."""
    df = pd.DataFrame({'sde': [2.0, np.nan], 'xe': [10.0, 0.0]})
    out = impute_sd_median_cv(df, 'sde', 'xe')
    assert np.isclose(out.iloc[0], 2.0) and pd.isna(out.iloc[1])

def test_pooled_cv_df_weights_and_small_sample_correction():
    """Pooled CV: CV* = CV (1 + 1/(4n)), pooled in CV² space with weights n - 1."""
    df = pd.DataFrame({'sde': [2.0, 6.0, np.nan], 'xe': [10.0, 20.0, 5.0], 'ne': [11, 21, 8]})
    cv = np.array([0.2, 0.3]) * (1 + 1 / (4 * np.array([11.0, 21.0])))
    expected = np.sqrt(np.sum(np.array([10.0, 20.0]) * cv ** 2) / 30.0)
    assert np.isclose(calculate_pooled_cv(df, 'sde', 'xe', 'ne'), expected)

def test_pooled_cv_none_without_usable_rows():
    """No row with SD > 0, mean != 0 and n > 1: refuse to impute (None), never invent a value."""
    df = pd.DataFrame({'sde': [np.nan, 0.0], 'xe': [10.0, 5.0], 'ne': [10, 10]})
    assert calculate_pooled_cv(df, 'sde', 'xe', 'ne') is None

def test_zero_sd_is_treated_as_missing():
    """A zero SD becomes NaN (handled as missing), never the largest weight."""
    out = zeros_to_nan(pd.Series([0.0, 0.5, 2.0]))
    assert pd.isna(out.iloc[0]) and np.isclose(out.iloc[1], 0.5) and np.isclose(out.iloc[2], 2.0)


def test_build_vcv_lnRR_shared_control():
    """For lnRR with shared control: cov = sdc² / (nc · xc²)."""
    df = pd.DataFrame({
        'id': ['S', 'S'],
        'shared_group_id': ['g', 'g'],
        'yi':  [0.2, 0.3],
        'vi':  [0.04, 0.05],
        'ne':  [20, 25],
        'nc':  [20, 20],
        'xc':  [5.0, 5.0],
        'sdc': [1.0, 1.0],
    })
    vcv = build_vcv_matrices(df, effect_type='lnRR', var_col_name='vi',
                             yi_col='yi', nt_col='ne', nc_col='nc',
                             xc_col='xc', sdc_col='sdc')['S']
    expected_cov = (1.0 ** 2) / (20.0 * 5.0 ** 2)   # = 0.002
    assert np.isclose(vcv[0, 0], 0.04)
    assert np.isclose(vcv[1, 1], 0.05)
    assert np.isclose(vcv[0, 1], expected_cov)
    assert np.isclose(vcv[0, 1], vcv[1, 0])         # symmetry

def test_build_vcv_log_or_shared_control():
    """log-OR shared control: cov = 1/c_c + 1/d_c."""
    df = pd.DataFrame({
        'id': ['S', 'S'],
        'shared_group_id': ['g', 'g'],
        'yi':  [0.5, -0.2],
        'vi':  [0.30, 0.40],
        'ne':  [30, 30],
        'nc':  [40, 40],
        'events_c':    [10.0, 10.0],
        'nonevents_c': [20.0, 20.0],
    })
    vcv = build_vcv_matrices(df, effect_type='log_or', var_col_name='vi',
                             yi_col='yi', nt_col='ne', nc_col='nc',
                             events_c_col='events_c',
                             nonevents_c_col='nonevents_c')['S']
    assert np.isclose(vcv[0, 1], 1.0/10.0 + 1.0/20.0)   # 0.15
    assert np.isclose(vcv[1, 0], vcv[0, 1])

def test_build_vcv_log_rr_shared_control():
    """log-RR shared control: cov = 1/c_c − 1/(c_c+d_c)."""
    df = pd.DataFrame({
        'id': ['S', 'S'],
        'shared_group_id': ['g', 'g'],
        'yi':  [0.1, 0.4],
        'vi':  [0.20, 0.25],
        'ne':  [30, 30],
        'nc':  [50, 50],
        'events_c':    [10.0, 10.0],
        'nonevents_c': [40.0, 40.0],
    })
    vcv = build_vcv_matrices(df, effect_type='log_rr', var_col_name='vi',
                             yi_col='yi', nt_col='ne', nc_col='nc',
                             events_c_col='events_c',
                             nonevents_c_col='nonevents_c')['S']
    expected_cov = 1.0/10.0 - 1.0/50.0       # 0.08
    assert np.isclose(vcv[0, 1], expected_cov)

def test_build_vcv_falls_back_to_diagonal_without_shared_groups():
    """No shared_group_id ⇒ purely diagonal VCV = diag(vi)."""
    df = pd.DataFrame({
        'id': ['S', 'S'],
        'yi':  [0.2, 0.3],
        'vi':  [0.04, 0.05],
        'ne':  [20, 25],
        'nc':  [20, 20],
        'xc':  [5.0, 5.0],
        'sdc': [1.0, 1.0],
    })
    vcv = build_vcv_matrices(df, effect_type='lnRR', var_col_name='vi',
                             nt_col='ne', nc_col='nc',
                             xc_col='xc', sdc_col='sdc')['S']
    assert np.allclose(vcv, np.diag([0.04, 0.05]))


# ---------- PET-PEESE decision rule -----------------------------------------

def test_petpeese_uses_peese_when_pet_intercept_significant():
    """PET intercept significant in the expected direction => PEESE (Stanley & Doucouliagos 2014)."""
    d = PETPEESEDecisionMaker.make_decision(
        pet_intercept_p=0.02, pet_slope_p=0.60, expected_sign=1.0,
        pet_intercept=0.10,  pet_intercept_se=0.05,  pet_intercept_ci=(0.00, 0.20),
        peese_intercept=0.30, peese_intercept_se=0.06, peese_intercept_ci=(0.18, 0.42),
        p_threshold=0.10,
    )
    assert d.effect_detected is True and d.bias_detected is False   # the slope does not decide
    assert d.recommended_model == 'PEESE'
    assert np.isclose(d.recommended_estimate, 0.30)

def test_petpeese_uses_pet_when_pet_intercept_not_significant():
    """PET intercept not significant => PET, even with a significant slope."""
    d = PETPEESEDecisionMaker.make_decision(
        pet_intercept_p=0.45, pet_slope_p=0.001, expected_sign=1.0,
        pet_intercept=0.05,  pet_intercept_se=0.07,  pet_intercept_ci=(-0.09, 0.19),
        peese_intercept=0.27, peese_intercept_se=0.05, peese_intercept_ci=(0.17, 0.37),
        p_threshold=0.10,
    )
    assert d.effect_detected is False and d.bias_detected is True
    assert d.recommended_model == 'PET'
    assert np.isclose(d.recommended_estimate, 0.05)

def test_petpeese_wrong_direction_uses_pet():
    """Significant PET intercept with the opposite sign to the uncorrected effect => PET."""
    d = PETPEESEDecisionMaker.make_decision(
        pet_intercept_p=0.01, expected_sign=1.0,
        pet_intercept=-0.20, pet_intercept_se=0.07, pet_intercept_ci=(-0.34, -0.06),
        peese_intercept=0.05, peese_intercept_se=0.05, peese_intercept_ci=(-0.05, 0.15),
    )
    assert d.recommended_model == 'PET' and d.effect_detected is False


# ---------- Unified engine (Cell 1b): nesting, row identity, refusals -------

def _engine_sim(seed, M=12):
    rng = np.random.default_rng(seed)
    rows, vcv = [], {}
    for i in range(M):
        k = int(rng.integers(1, 5))
        V = np.full((k, k), rng.uniform(.01, .06)) + np.diag(rng.uniform(.02, .1, k))
        y = 0.3 + rng.normal(0, .2) + rng.normal(0, .15, k) + rng.multivariate_normal(np.zeros(k), V)
        x = rng.normal(size=k)
        vcv[f"S{i:02d}"] = V
        rows += [dict(id=f"S{i:02d}", yi=y[j], vi=V[j, j], x=x[j]) for j in range(k)]
    df = pd.DataFrame(rows)
    return df, vcv

@pytest.mark.parametrize("seed", range(8))
def test_engine_nested_aic_bound(seed):
    """Reduced variance structures on the SAME rows and V can never beat the
    full three-level model by more than 2 AIC units."""
    df, vcv = _engine_sim(seed)
    spec = build_fit_spec(df, 'yi', 'vi', vcv_dict=vcv, vcv_reference_df=df)
    full = fit_mv(spec)
    for kw in ({'tau2': 0.0}, {'sigma2': 0.0}):
        c = compare_fits(full, fit_mv(spec, **kw))
        assert c.nested and c.n_boundary == 1
        assert c.delta_aic >= -2 - 1e-6
        assert not c.warnings

def test_engine_refuses_different_rows():
    df, vcv = _engine_sim(1)
    a = fit_mv(build_fit_spec(df, 'yi', 'vi', vcv_dict=vcv, vcv_reference_df=df))
    b = fit_mv(build_fit_spec(df.iloc[2:], 'yi', 'vi', vcv_dict=vcv, vcv_reference_df=df), sigma2=0.0)
    with pytest.raises(ComparisonError):
        compare_fits(a, b)

def test_engine_refuses_diagonal_vs_full_V():
    df, vcv = _engine_sim(2)
    a = fit_mv(build_fit_spec(df, 'yi', 'vi', vcv_dict=vcv, vcv_reference_df=df))
    b = fit_mv(build_fit_spec(df, 'yi', 'vi', vcv_dict={}, vcv_reference_df=df), tau2=0.0)
    with pytest.raises(ComparisonError):
        compare_fits(a, b)

def test_engine_partial_cluster_keeps_shared_covariance():
    """Dropping one row of a shared-control cluster must slice V by row
    identity, not fall back to a diagonal V."""
    df, vcv = _engine_sim(3)
    big = df.groupby('id').size().idxmax()
    rows_big = df.index[df['id'] == big]
    df2 = df.copy()
    df2.loc[rows_big[0], 'x'] = np.nan
    spec = build_fit_spec(df2, 'yi', 'vi', complete_case_cols=['x'],
                          vcv_dict=vcv, vcv_reference_df=df)
    keep = list(range(1, len(rows_big)))
    j = spec.cluster_ids.index(big)
    assert np.allclose(spec.V_blocks[j], vcv[big][np.ix_(keep, keep)])
    assert spec.accounting.n_used_effects == len(df) - 1
    assert spec.accounting.dropped[0]['reason'] == "missing value in 'x'"

def test_engine_pseudo_r2_uses_matching_null():
    df, vcv = _engine_sim(4)
    X = pd.DataFrame({'intrcpt': 1.0, 'x': df['x']}, index=df.index)
    m = fit_mv(build_fit_spec(df, 'yi', 'vi', X=X, vcv_dict=vcv, vcv_reference_df=df))
    r = pseudo_r2(m)
    assert r['null_fit'].spec.data_fingerprint == m.spec.data_fingerprint
    assert r['null_fit'].spec.n_coef == 1 and r['R2'] >= 0.0

def test_engine_chibar_mixture():
    assert np.isclose(chibar_pvalue(2.7055, 0, 1), 0.05, atol=1e-3)


# ---------- Overall analysis: row alignment + nested model selection --------

def test_three_level_engine_matches_unified_engine():
    """Regression test for the unstable sort: the legacy three-level engine
    must pair each effect size with its own sampling (co)variances, so its
    REML optimum equals the unified engine's on the same rows and V."""
    rng = np.random.default_rng(11)
    rows, vcv = [], {}
    for i in range(25):
        k = int(rng.integers(2, 5))
        V = np.full((k, k), 0.02) + np.diag(rng.uniform(.01, .2, k))
        y = 0.3 + rng.normal(0, .2) + rng.normal(0, .15, k) + rng.multivariate_normal(np.zeros(k), V)
        vcv[f"S{i:02d}"] = V
        rows += [dict(id=f"S{i:02d}", yi=y[j], vi=V[j, j]) for j in range(k)]
    df = pd.DataFrame(rows)
    legacy = ThreeLevelEngine().fit(df, 'yi', 'vi', vcv)
    unified = fit_mv(build_fit_spec(df, 'yi', 'vi', vcv_dict=vcv, vcv_reference_df=df))
    assert np.isclose(legacy['tau2'], unified.tau2, rtol=1e-3, atol=1e-6)
    assert np.isclose(legacy['sigma2'], unified.sigma2, rtol=1e-3, atol=1e-6)
    assert np.isclose(legacy['mu'], unified.betas[0], atol=1e-6)

def test_overall_selection_rule():
    """The design decides: 3-level whenever it can be fitted, whatever the LRT says."""
    sel = OverallEngine._select_model
    vc_sig = {'tests': {'tau2': {'lrt': 5.0, 'p_value': 0.0127}}}
    vc_ns = {'tests': {'tau2': {'lrt': 0.1, 'p_value': 0.38}}}
    for vc in (vc_sig, vc_ns, {'error': 'x'}, None):
        assert sel(None, THREE_LEVEL_CHOICE, True, vc, 0.05)[0] == '3-Level'
    assert sel(None, THREE_LEVEL_CHOICE, False, None, 0.05)[0] == '2-Level'   # not estimable
    assert sel(None, TWO_LEVEL_CHOICE, True, vc_sig, 0.05)[0] == '2-Level'    # sensitivity analysis
    assert sel(None, None, True, None, 0.05)[0] == '3-Level'                  # default


# ---------- Meta-regression pseudo-R² (matching intercept-only null) -----------

class _FakeRegDM:
    def __init__(self, df, vcv):
        self._config = {'vcv_matrices': vcv}
        self.analysis_data = df

def _r2_case(seed, one_per_study=False):
    rng = np.random.default_rng(seed)
    rows, vcv = [], {}
    for i in range(24):
        k = 1 if one_per_study else int(rng.integers(1, 5))
        xs = rng.normal(size=k)
        V = np.full((k, k), 0.02) + np.diag(rng.uniform(.02, .08, k))
        y = 0.2 + 0.3 * (i % 2) + rng.normal(0, .15) + 0.2 * xs + rng.normal(0, .2, k) \
            + rng.multivariate_normal(np.zeros(k), V)
        vcv[f"S{i:02d}"] = V
        rows += [dict(id=f"S{i:02d}", yi=y[j], vi=V[j, j], x=xs[j], g=i % 2) for j in range(k)]
    df = pd.DataFrame(rows)
    X = np.column_stack([np.ones(len(df)), df['x'], df['g']])
    design = {'X': X, 'term_names': ['Intercept', 'x', 'g']}
    cfg = type('C', (), {'effect_col': 'yi', 'var_col': 'vi'})()
    eng = type('E', (), {'data_manager': _FakeRegDM(df, vcv)})()
    return df, vcv, X, design, cfg, eng

@pytest.mark.parametrize("seed", range(4))
def test_metareg_pseudo_r2_matches_matching_null(seed):
    df, vcv, X, design, cfg, eng = _r2_case(seed)
    d = RegressionEngine._compute_pseudo_r2(eng, df, cfg, design, three_level=True)
    spec = build_fit_spec(df, 'yi', 'vi', X=pd.DataFrame(X, index=df.index),
                          vcv_dict=vcv, vcv_reference_df=df)
    m, n = fit_mv(spec), fit_mv(spec.intercept_only())
    raw = 1 - (m.tau2 + m.sigma2) / (n.tau2 + n.sigma2)
    assert np.isclose(d['R2_raw'], raw, atol=1e-6)
    assert np.isclose(d['R2'], 100 * max(0.0, raw), atol=1e-4)
    assert np.isclose(d['tau2_null'], n.tau2, atol=1e-6) and np.isclose(d['sigma2_null'], n.sigma2, atol=1e-6)
    assert 0.0 <= d['R2_tau2'] <= 100.0 and 0.0 <= d['R2_sigma2'] <= 100.0
    assert d['k_effects'] == len(df)

def test_metareg_pseudo_r2_two_level_structure():
    df, vcv, X, design, cfg, eng = _r2_case(9, one_per_study=True)
    d = RegressionEngine._compute_pseudo_r2(eng, df, cfg, design, three_level=False)
    assert not d['three_level'] and d['sigma2_null'] == 0.0 and np.isnan(d['R2_sigma2'])

def test_metareg_pseudo_r2_reports_failure_not_zero():
    df, vcv, X, design, cfg, eng = _r2_case(1)
    bad = {'X': np.ones((len(df), 2)), 'term_names': ['Intercept', 'dup']}   # rank-deficient
    d = RegressionEngine._compute_pseudo_r2(eng, df, cfg, bad, three_level=True)
    assert np.isnan(d['R2']) and d['error']


# ---------- Subgroup pseudo-R² (matching null, no global fallback) --------------

def test_fit_spec_from_blocks_matches_build_fit_spec():
    df, vcv = _engine_sim(5)
    a = build_fit_spec(df, 'yi', 'vi', vcv_dict=vcv, vcv_reference_df=df)
    b = fit_spec_from_blocks(a.y_blocks, a.V_blocks, a.X_blocks)
    fa, fb = fit_mv(a), fit_mv(b)
    assert np.isclose(fa.loglik, fb.loglik) and np.isclose(fa.tau2, fb.tau2) and np.isclose(fa.sigma2, fb.sigma2)

def test_subgroup_partition_uses_r2_detail():
    df, vcv = _engine_sim(6)
    s0 = build_fit_spec(df, 'yi', 'vi', vcv_dict=vcv, vcv_reference_df=df)
    X = [np.column_stack([np.ones(len(y)), np.full(len(y), float(j % 2))])
         for j, y in enumerate(s0.y_blocks)]
    spec = fit_spec_from_blocks(s0.y_blocks, s0.V_blocks, X, term_names=['Intercept', 'g'])
    d = pseudo_r2_report(spec, three_level=True)
    het = SubgroupAnalyzer._partition_from_r2_detail(
        SubgroupAnalyzer.__new__(SubgroupAnalyzer), d,
        {'dfs': np.array([10.0, 10.0])}, 4.0, 1, 30.0, 40)
    same = lambda a, b: np.isclose(a, b, equal_nan=True)
    assert same(het['R_squared'], d['R2']) and same(het['R_squared_tau'], d['R2_tau2'])
    assert same(het['R_squared_sigma'], d['R2_sigma2']) and het['R_squared_detail'] is d
    assert het['tau_squared_null'] == d['tau2_null'] and het['QM'] == 4.0

def test_subgroup_r2_not_computable_is_nan_not_zero():
    het = SubgroupAnalyzer._partition_from_r2_detail(
        SubgroupAnalyzer.__new__(SubgroupAnalyzer), {'R2': float('nan'), 'error': 'x'},
        {'dfs': np.array([10.0, 10.0])}, 4.0, 1, 30.0, 40)
    assert np.isnan(het['R_squared'])

def test_subgroup_legacy_fallback_labels_q_ratio():
    het = SubgroupAnalyzer._partition_heterogeneity_legacy(
        SubgroupAnalyzer.__new__(SubgroupAnalyzer),
        qt_overall=100.0, q_within_sum=60.0, n_groups=3, k_overall=50)
    assert np.isnan(het['R_squared']) and np.isclose(het['Q_ratio_explained'], 40.0)
    assert 'not a pseudo-R' in het['R_squared_detail']['error']


# ---------- Row accounting + identity-based V slicing (Step 4) ----------------

def test_select_rows_records_reasons_in_order():
    df = pd.DataFrame({'id': ['a', 'a', 'b', 'b', None, 'c'],
                       'yi': [0.1, np.nan, 0.2, 0.3, 0.1, 0.4],
                       'vi': [0.1, 0.1, -1.0, 0.1, 0.1, 0.1],
                       'm':  ['x', 'y', 'x', '', 'x', 'z']}, index=[10, 11, 12, 13, 14, 15])
    used, acc = select_rows(df, 'yi', 'vi', 'id', complete_case_cols=['m'])
    assert list(used.index) == [10, 15]
    assert [(d['reason'].split(' (')[0], d['n_effects']) for d in acc.dropped] == [
        ('missing cluster id', 1), ('missing/non-finite effect size', 1),
        ('missing/non-positive variance', 1), ("missing value in 'm'", 1)]
    assert acc.n_used_effects == 2 and acc.n_input_clusters == 3 and acc.n_used_clusters == 2
    assert acc.dropped[3]['clusters_lost'] == ['b']
    d = acc.to_dict(); json.dumps(d)
    assert RowAccounting.from_dict(d).n_used_effects == 2

def test_vcv_row_positions_rejects_reset_index_labels():
    ref = pd.DataFrame({'id': ['a', 'a', 'b', 'b'], 'vi': [.1, .2, .3, .4]}, index=[5, 6, 7, 8])
    sub = ref.loc[[6, 8]]
    assert list(vcv_row_positions(sub, ref)) == [1, 1]
    shuffled = ref.loc[[8, 6]].reset_index(drop=True)          # labels 0,1 not in ref
    assert vcv_row_positions(shuffled, ref) is None
    ref2 = ref.reset_index(drop=True)                         # labels 0..3 exist ...
    wrong = ref2.loc[[2, 3]].copy(); wrong.index = [0, 1]      # ... but point at other rows
    assert vcv_row_positions(wrong, ref2) is None

def test_cluster_vcv_slices_partial_study():
    V = np.array([[.10, .02, .02], [.02, .20, .02], [.02, .02, .30]])
    ref = pd.DataFrame({'id': ['s'] * 3, 'vi': np.diag(V)}, index=[1, 2, 3])
    g = attach_vcv_row_pos(ref.loc[[1, 3]], ref).reset_index(drop=True)
    Vs, src = cluster_vcv({'s': V}, 's', g, 'vi')
    assert src == 'full' and np.allclose(Vs, V[np.ix_([0, 2], [0, 2])])

def test_regression_engine_keeps_shared_covariance_on_partial_studies(monkeypatch):
    """Rows dropped inside shared-control studies: the meta-regression engine
    must slice V by row identity (it used to silently use a diagonal V), so its
    REML optimum equals the unified engine's on the same rows."""
    df, vcv = _engine_sim(7)
    big = df.groupby('id').filter(lambda g: len(g) >= 3).groupby('id').head(1).index[:3]
    df.loc[big, 'x'] = np.nan
    monkeypatch.setitem(globals(), 'ANALYSIS_CONFIG', {'vcv_matrices': vcv, 'analysis_data': df})
    reg = df.dropna(subset=['x'])
    est = _run_three_level_reml_regression_v2(reg, 'x', 'yi', 'vi')[0]
    X = pd.DataFrame({'intrcpt': 1.0, 'x': reg['x'] - reg['x'].mean()}, index=reg.index)
    f = fit_mv(build_fit_spec(reg, 'yi', 'vi', X=X, vcv_dict=vcv, vcv_reference_df=df))
    assert np.isclose(est['tau_sq'], f.tau2, rtol=1e-3, atol=1e-6)
    assert np.isclose(est['sigma_sq'], f.sigma2, rtol=1e-3, atol=1e-6)

def test_common_row_set_applied_in_step6(monkeypatch):
    monkeypatch.setitem(globals(), 'ANALYSIS_CONFIG', {'complete_case_moderators': ['m']})
    df = pd.DataFrame({'id': ['a', 'a', 'b'], 'm': ['x', None, ' ']})
    removed, log = [], []
    out = _apply_common_row_set(df, removed, log)
    assert list(out.index) == [0]
    assert globals()['ANALYSIS_CONFIG']['complete_case_log']['n_dropped'] == 2
    assert set(pd.concat(removed)['Reason']) == {"Common row set: missing value in 'm'"}


# ---------- Spline vs linear: ML comparison on the same rows (Step 5) -----------

def _spline_case(seed, curved):
    rng = np.random.default_rng(seed)
    rows, vcv = [], {}
    for i in range(35):
        k = int(rng.integers(1, 5))
        xs = rng.uniform(0, 10, size=k)
        V = np.full((k, k), 0.02) + np.diag(rng.uniform(.02, .06, k))
        f = 0.5 * np.sin(xs / 2.0) if curved else 0.05 * xs
        y = 0.1 + f + rng.normal(0, .12) + rng.normal(0, .1, k) + rng.multivariate_normal(np.zeros(k), V)
        vcv[f"S{i:02d}"] = V
        rows += [dict(id=f"S{i:02d}", yi=y[j], vi=V[j, j], x=xs[j]) for j in range(k)]
    return pd.DataFrame(rows), vcv

@pytest.mark.parametrize("curved", [False, True])
def test_spline_vs_linear_is_ml_nested_comparison(curved, monkeypatch):
    df, vcv = _spline_case(1, curved)
    monkeypatch.setitem(globals(), 'ANALYSIS_CONFIG',
                        {'vcv_matrices': vcv, 'analysis_data': df, 'global_settings': {'alpha': 0.05}})
    model = SplineMetaRegressionModel.__new__(SplineMetaRegressionModel)
    mc = model._compare_linear_spline(df, 'x', 'yi', 'vi', 4)
    assert mc['method'] == 'ML' and mc['nested'] and mc['df'] == 3
    # AIC counts the variance components: 2 + 2 and 5 + 2 parameters
    assert mc['linear']['n_params'] == 4 and mc['spline']['n_params'] == 7
    assert np.isclose(mc['lrt'], 2 * (mc['spline']['loglik'] - mc['linear']['loglik']))
    assert np.isclose(mc['delta_aic'], mc['linear']['aic'] - mc['spline']['aic'])
    assert mc['nonlinear'] == curved


# --- Proportion of meaningful effects (Cell 1b, section 9; Step 9b) ---

def test_meaningful_parametric_share_basics():
    mu, t2 = 0.3, 0.04
    assert np.isclose(prop_beyond(mu, t2, mu, 'above'), 0.5)
    assert np.isclose(prop_beyond(mu, t2, 0.1, 'above') + prop_beyond(mu, t2, 0.1, 'below'), 1.0)
    # no heterogeneity: every true effect equals mu
    assert prop_beyond(mu, 0.0, 0.1, 'above') == 1.0 and prop_beyond(mu, 0.0, 0.1, 'below') == 0.0
    assert np.isnan(prop_beyond_se(mu, 0.0, 0.1, 0.05, 0.01))


def test_calibrated_effects_shrink_toward_model_mean():
    y, v = np.array([1.0, -1.0, 0.5]), np.array([0.1, 0.1, 0.4])
    assert np.allclose(calibrated_effects(y, v, np.zeros(3), 0.1), y * np.sqrt(0.1 / (0.1 + v)))
    assert np.allclose(calibrated_effects(y, v, np.full(3, 0.2), 0.0), 0.2)


def test_bca_without_bias_or_acceleration_is_the_percentile_interval():
    boot = np.random.default_rng(3).normal(0.0, 1.0, 4001)
    est = float(np.median(boot))                      # z0 = 0
    lo, hi, how = bca_interval(est, boot, None, 0.05)  # a = 0
    assert how == 'BCa'
    assert np.allclose([lo, hi], _norm_inter(boot, [0.025, 0.975]))
    lo, hi, how = bca_interval(10.0, boot, None, 0.05)  # every replicate below the estimate
    assert how == 'percentile'


def test_meaningful_proportions_subgroups_and_bootstrap():
    rng = np.random.default_rng(9)
    y_b, V_b, X_b = [], [], []
    for j in range(24):
        g = j % 2
        k = int(rng.integers(1, 4))
        v = rng.uniform(0.01, 0.03, k)
        y_b.append(0.1 + 0.3 * g + rng.normal(0, 0.15) + rng.normal(0, 0.1, k) + rng.normal(0, np.sqrt(v)))
        V_b.append(np.diag(v))
        X_b.append(np.tile([1.0, float(g)], (k, 1)))
    fit = fit_blocks(y_b, V_b, X_b, two_level=False)

    def refit(yy, VV, XX):
        try:
            f = fit_blocks(yy, VV, XX, two_level=False)
        except (ValueError, RuntimeError, np.linalg.LinAlgError):
            return None
        return f.betas, f.tau2, f.sigma2

    model = MeaningfulModel(y_b, V_b, X_b, fit.betas, fit.tau2, fit.sigma2,
                            {'A': np.array([1.0, 0.0]), 'B': np.array([1.0, 1.0])},
                            {'A': 0.05, 'B': 0.05}, var_components_vcov(fit), refit)
    assert all((lab != '').all() for lab in model.row_targets(X_b))
    res = meaningful_proportions(model, (-0.1, 0.2), n_boot=60, seed=1)
    assert list(res['target']) == ['A', 'A', 'B', 'B']
    assert np.allclose(res['p_increase'] + res['p_decrease'] + res['p_negligible'], 1.0)
    par = res[res['method'] == 'parametric']
    t2 = fit.tau2 + fit.sigma2
    assert np.allclose(par['p_increase'], [prop_beyond(fit.betas[0], t2, 0.2, 'above'),
                                           prop_beyond(fit.betas.sum(), t2, 0.2, 'above')])
    assert (res['n_boot'] > 0).all()
    assert ((res['increase_lo_boot'] <= res['p_increase'] + 1e-12)
            & (res['p_increase'] <= res['increase_hi_boot'] + 1e-12)).all()
    with pytest.raises(ValueError):
        meaningful_proportions(model, (0.2, -0.1))


def test_full_v_likelihood_matches_dense_algebra_and_warm_start():
    """The batched full-V likelihood equals the dense formula on block_diag(Sigma),
    and a warm-started fit reaches the same optimum as the default multi-start."""
    from scipy.linalg import block_diag
    rng = np.random.default_rng(11)
    y_b, V_b, X_b = [], [], []
    for j in range(15):
        k = int(rng.integers(1, 5))
        V = np.diag(rng.uniform(0.01, 0.05, k)) + 0.004 * (np.ones((k, k)) - np.eye(k))
        x = rng.normal(size=k)
        y_b.append(0.2 + 0.1 * x + rng.normal(0, 0.2) + rng.normal(0, 0.1, k))
        V_b.append(V)
        X_b.append(np.column_stack([np.ones(k), x]))
    spec = fit_spec_from_blocks(y_b, V_b, X_b)
    t2, s2 = 0.03, 0.02
    Sig = block_diag(*[V + s2 * np.eye(len(V)) + t2 * np.ones((len(V), len(V))) for V in V_b])
    y, X = np.concatenate(y_b), np.vstack(X_b)
    W = np.linalg.inv(Sig)
    XtWX = X.T @ W @ X
    beta = np.linalg.solve(XtWX, X.T @ W @ y)
    r = y - X @ beta
    N, p = len(y), X.shape[1]
    ll = -0.5 * ((N - p) * np.log(2 * np.pi) + np.linalg.slogdet(Sig)[1] + np.linalg.slogdet(XtWX)[1]
                 + r @ W @ r) + 0.5 * np.linalg.slogdet(X.T @ X)[1]
    assert np.isclose(_mv_loglik(spec, t2, s2, 'REML', True), ll, rtol=1e-12, atol=1e-10)
    cold = fit_blocks(y_b, V_b, X_b, two_level=False)
    warm = fit_blocks(y_b, V_b, X_b, two_level=False, starts=[(cold.tau2 * 1.5 + 0.01, cold.sigma2 * 0.5)])
    assert np.isclose(cold.loglik, warm.loglik, atol=1e-8)
    assert np.allclose(cold.betas, warm.betas, atol=1e-6)


# =============================================================================
# 3. EXECUTE & DISPLAY
# =============================================================================

# Clear the "Initializing" banner and pip install text
clear_output()

# Run the tests first (full verbosity), then report the ACTUAL outcome
exit_code = ipytest.run("-vv", "--color=yes")
if 'record_validation' in globals():
    record_validation(11, "Automated Unit Tests (ipytest)",
                      [('pytest exit code 0', 'PASS' if exit_code == 0 else 'FAIL')],
                      notes=f"pytest exit code {int(exit_code)}")

if exit_code == 0:
    display(HTML("""
    <div style='padding: 15px; background-color: #d4edda; border-left: 5px solid #28a745; color: #155724; border-radius: 4px; font-family: sans-serif; margin-top: 15px;'>
        <h4 style='margin: 0 0 5px 0;'>✅ All software unit tests passed</h4>
        <p style='margin: 0; font-size: 14px;'>Core mathematical functions, edge-case handling and serialization logic behave as specified (log above).</p>
    </div>
    """))
else:
    display(HTML(f"""
    <div style='padding: 15px; background-color: #f8d7da; border-left: 5px solid #dc3545; color: #721c24; border-radius: 4px; font-family: sans-serif; margin-top: 15px;'>
        <h4 style='margin: 0 0 5px 0;'>❌ Software unit tests FAILED (pytest exit code {int(exit_code)})</h4>
        <p style='margin: 0; font-size: 14px;'>One or more tests failed; see the log above.</p>
    </div>
    """))


In [ ]:
#@title 🧪 Validation 12: Exploratory Meta-Forest (Python vs R metaforest) — descriptive comparison
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Not a pass/fail validation: CoMeta (scikit-learn, weighted splits) and R's
# metaforest (ranger, weighted bootstrap) are different algorithms with
# different random number streams, so importance scores cannot agree to a
# tolerance. The cell reports both rankings and their Spearman correlation.
import pandas as pd
import numpy as np
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import rpy2.robjects.numpy2ri as numpy2ri
from IPython.display import display, HTML, clear_output

# Activate R conversions
pandas2ri.activate()
numpy2ri.activate()

display(HTML("<div style='padding: 15px; background-color: #e2e3e5; border-left: 4px solid #6c757d; color: #383d41; border-radius: 4px; font-family: sans-serif;'>⏳ <b>Running Cross-Validation with R (metaforest)...</b> Please wait. This requires training 500 trees in R.</div>"))

try:
    # --- 1. Dependencies Check ---
    if 'ANALYSIS_CONFIG' not in globals() or 'meta_forest_results' not in ANALYSIS_CONFIG:
        raise ValueError("Meta-Forest results not found. Please run Step 16 (Exploratory Meta-Forest) first.")

    py_res = ANALYSIS_CONFIG['meta_forest_results']
    df_py = ANALYSIS_CONFIG['analysis_data'].copy()

    effect_col = ANALYSIS_CONFIG['effect_col']
    var_col = ANALYSIS_CONFIG['var_col']

    # Get the features that were evaluated in Python
    py_imp_df = py_res['importance_df'].copy()
    features = py_imp_df['Feature'].tolist()

    # --- 2. Install/Load metaforest in R ---
    r_install_script = """
    suppressPackageStartupMessages(library(utils))
    if (!requireNamespace("ranger", quietly = TRUE)) {
        install.packages("ranger", quiet=TRUE)   # repos: the CRAN snapshot set in the R setup cell
    }
    if (!requireNamespace("metaforest", quietly = TRUE)) {
        install.packages("metaforest", quiet=TRUE)
    }
    suppressPackageStartupMessages(library(ranger))
    suppressPackageStartupMessages(library(metaforest))
    """
    ro.r(r_install_script)

    # --- 3. Data Preparation for R ---
    # R's metaforest doesn't have our SimpleImputer pipeline built-in,
    # so we pass complete cases for the selected columns to ensure a fair comparison.
    cols_to_keep = [effect_col, var_col] + features
    df_r = df_py[cols_to_keep].dropna().copy()

    # Pass Data & Settings to R
    ro.globalenv['df_python'] = df_r
    ro.globalenv['eff_col'] = effect_col
    ro.globalenv['var_col'] = var_col
    ro.globalenv['features'] = ro.StrVector(features)

    # --- 4. Run R's MetaForest ---
    r_script = """
    set.seed(42) # reproducible in R; R and Python random streams cannot be matched

    # Dynamically build the formula: yi ~ feat1 + feat2
    form_str <- paste(eff_col, "~", paste(features, collapse = " + "))
    form <- as.formula(form_str)

    # Run R's metaforest
    # It automatically estimates tau2 using REML and applies random-effects weights
    mf_model <- MetaForest(form, data = df_python, vi = var_col, num.trees = 500)

    # Extract Variable Importance
    imp_scores <- mf_model$forest$variable.importance

    # Return as DataFrame
    data.frame(
        Feature = names(imp_scores),
        R_Importance = as.numeric(imp_scores)
    )
    """

    r_out = ro.r(r_script)
    r_imp_df = ro.conversion.rpy2py(r_out)

    # --- 5. Format & Merge Comparison ---
    # Calculate Ranks (1 is best)
    py_imp_df['Python_Rank'] = py_imp_df['Importance'].rank(ascending=False, method='min').astype(int)
    r_imp_df['R_Rank'] = r_imp_df['R_Importance'].rank(ascending=False, method='min').astype(int)

    # Merge
    comp_df = pd.merge(py_imp_df, r_imp_df, on='Feature')

    # Reorder and format columns
    comp_df = comp_df[['Feature', 'Python_Rank', 'R_Rank', 'Importance', 'R_Importance']]
    comp_df = comp_df.rename(columns={
        'Importance': 'Python Score (Weighted MSE Drop)',
        'R_Importance': 'R Score (Permutation Imp.)'
    })
    comp_df = comp_df.sort_values('Python_Rank')
    rho_s = comp_df['Python_Rank'].corr(comp_df['R_Rank'], method='spearman') if len(comp_df) > 2 else float('nan')

    # --- 6. Render Output ---
    clear_output()
    display(HTML("""
    <div style='padding: 15px; background-color: #e2e3e5; border-left: 5px solid #6c757d; color: #383d41; border-radius: 4px; font-family: sans-serif; margin-bottom: 15px;'>
        <h4 style='margin: 0 0 5px 0;'>ℹ️ Descriptive comparison (no pass/fail): Meta-Forest importance rankings</h4>
        <p style='margin: 0; font-size: 14px;'>Co-Meta (Python) rankings vs the R <code>metaforest</code> package.
        Spearman rank correlation: <b>{rho_s:.2f}</b> ({n_feat} moderators).</p>
    </div>
    """.format(rho_s=rho_s, n_feat=len(comp_df))))

    # Display the clean table
    display(comp_df.style.set_properties(**{'text-align': 'center'})\
            .format({'Python Score (Weighted MSE Drop)': '{:.4f}', 'R Score (Permutation Imp.)': '{:.4f}'})\
            .background_gradient(subset=['Python_Rank', 'R_Rank'], cmap='Blues_r'))

    # Architectural Note regarding Algorithm differences
    display(HTML("""
    <div style='background-color: #e8f4f8; padding: 15px; border-radius: 5px; border-left: 4px solid #17a2b8; margin-top: 20px; font-family: sans-serif; font-size: 13px; color: #0c5460;'>
        <h4 style='margin: 0 0 8px 0; color: #0c5460;'>💡 Architectural Note: Why don't the raw scores match?</h4>
        <p style='margin: 0 0 8px 0;'>While both packages aim to achieve a Random-Effects Meta-Forest, their underlying mathematical machinery differs significantly:</p>
        <ul style='margin-top: 0; margin-bottom: 8px;'>
            <li><b>R (<code>metaforest</code> / <code>ranger</code>):</b> Applies sample precision via <b>Weighted Bootstrapping</b>. Highly precise studies have a higher probability of being drawn into the trees. <i>(Risk: Rare/small studies may be entirely excluded from the ensemble).</i></li>
            <li><b>Python (<code>Co-Meta</code> / <code>scikit-learn</code>):</b> Applies sample precision via <b>Weighted Splits</b>. It uses uniform bootstrapping (preserving all studies) but mathematically forces the MSE loss function to prioritize predicting the high-precision studies accurately during node splitting.</li>
        </ul>
        <p style='margin: 0;'>Because the algorithms evaluate loss differently, the raw importance scores are on different scales. The ordinal rankings are expected to be broadly similar, especially for clearly dominant moderators; swaps among moderators of similar importance, and differences between runs, are normal for random forests. Meta-Forest output is exploratory in Co-Meta and is not used for inference.</p>
    </div>
    """))
    if 'record_validation' in globals():
        record_validation(12, "Exploratory Meta-Forest (descriptive)", comp_df.assign(status='INFO'),
                          notes=f"Descriptive only (no pass/fail). Spearman rank correlation: {rho_s:.3f}")

except Exception as e:
    # A crash is a failed validation, not 'not run'
    if 'record_validation' in globals() and not validation_prerequisite_missing(e):
        record_validation(12, 'Exploratory Meta-Forest (descriptive)',
                          [('validation ran to completion', 'FAIL')],
                          notes=f"{type(e).__name__}: {e}")
    clear_output()
    display(HTML(f"""
    <div style='padding: 15px; background-color: #f8d7da; border-left: 5px solid #dc3545; color: #721c24; border-radius: 4px; font-family: sans-serif;'>
        <h4 style='margin: 0 0 5px 0;'>❌ Validation Error</h4>
        <p style='margin: 0;'>{e}</p>
        <p style='margin-top: 10px; font-size: 12px;'><i>Note: Ensure you have run Step 16 to generate the Python Meta-Forest results before running this validation.</i></p>
    </div>
    """))

In [ ]:
#@title 🧪 Validation 13: Two-Way Factorial Decomposition (Python vs clubSandwich HTZ)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# Validates the factor-A / factor-B / A x B decomposition that Step 9 reports
# for a two-way subgroup analysis, against R's metafor + clubSandwich.
#
# Two independent things are checked, and the table separates them so a
# failure points at one or the other:
#
#   Python  vs  R (Python's C)   -> the AHZ small-sample Wald engine.
#       Identical contrast matrix handed to clubSandwich's
#       Wald_test(test = "HTZ"), on a model whose Sigma is pinned to
#       Python's tau^2/sigma^2. Any gap is in the test machinery.
#
#   R (Python's C)  vs  R (R's own C)  -> the contrast CONSTRUCTION.
#       R rebuilds the interaction and Type II main-effect spaces from its
#       own model.matrix() calls and its own QR/SVD, with no input from
#       Python beyond the cell labels. Two constraint matrices spanning the
#       same row space must give the same Wald test, so a gap here means the
#       two engines disagree about WHICH hypothesis is being tested.
#
# When the observed cells are an incomplete rectangle, both the primary
# (Type II) decomposition and the complete-subgrid sensitivity refit are
# validated, each against its own R fit.
# =============================================================================
import pandas as pd
import numpy as np
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import rpy2.robjects.numpy2ri as numpy2ri
from scipy.linalg import block_diag
from IPython.display import display, HTML, clear_output
import logging

logging.getLogger('rpy2.rinterface_lib.callbacks').setLevel(logging.ERROR)
pandas2ri.activate()
numpy2ri.activate()

display(HTML(
    "<div style='padding: 15px; background-color: #e2e3e5; border-left: 4px solid "
    "#6c757d; color: #383d41; border-radius: 4px; font-family: sans-serif;'>"
    "⏳ <b>Validating the two-way factorial decomposition against "
    "metafor + clubSandwich...</b> Please wait.</div>"
))

LABEL_SEP = ' | '
TERM_KEYS = ('main_a', 'main_b', 'interaction')

R_SCRIPT = """
suppressPackageStartupMessages(library(metafor))
suppressPackageStartupMessages(library(clubSandwich))

tryCatch({
    dat <- df_r
    dat$row_id   <- 1:nrow(dat)
    dat$study_id <- as.factor(dat$id)

    # Level ORDER comes from Python so the contrast columns of res$b line up
    # one-for-one with Python's feature_names (locale collation must not get
    # a vote). relevel() then puts the reference first, exactly as Step 9's
    # treatment coding does.
    dat$subgroup <- factor(as.character(dat$subgroup_label), levels = py_levels)
    if (any(is.na(dat$subgroup))) {
        stop("subgroup_label contains levels absent from Python's level list.")
    }
    dat$subgroup <- relevel(dat$subgroup, ref = ref_level)

    is_multilevel <- length(unique(dat$study_id)) < nrow(dat)
    random_form <- if (is_multilevel) ~ 1 | study_id/row_id else ~ 1 | study_id
    fixed_vars  <- if (is_multilevel) c(py_tau2, py_sigma2) else py_tau2

    # Sigma pinned to Python's variance components: this cell tests the
    # hypothesis machinery, not REML estimation (Validation 3 covers that).
    res <- suppressWarnings(rma.mv(
        yi = dat[[eff_col]], V = V_matrix, mods = ~ subgroup,
        random = random_form, data = dat, method = "REML", test = "z",
        sigma2 = fixed_vars))

    V_cr2 <- vcovCR(res, type = "CR2")

    # ---- R's OWN contrast construction -------------------------------
    # Built from R's model matrices and R's own SVD/QR. The only thing
    # taken from Python is the list of cell labels.
    pos  <- regexpr(label_sep, py_levels, fixed = TRUE)   # FIRST occurrence
    if (any(pos < 1)) stop("A cell label does not contain the two-way separator.")
    a_of <- substr(py_levels, 1, pos - 1)
    b_of <- substr(py_levels, pos + nchar(label_sep), nchar(py_levels))

    cellinfo <- data.frame(stringsAsFactors = FALSE,
                           subgroup = factor(py_levels, levels = py_levels))
    cellinfo$subgroup <- relevel(cellinfo$subgroup, ref = ref_level)
    cellinfo$FA <- factor(a_of)
    cellinfo$FB <- factor(b_of)

    # One row per cell, in py_levels order. L maps beta -> cell means, and
    # its columns follow res$b because both come from the same relevel'd
    # factor. Which level each factor uses as ITS baseline is irrelevant
    # below: every space is defined by a column span, not by a basis.
    L_R     <- model.matrix(~ subgroup, data = cellinfo)
    M_add   <- model.matrix(~ FA + FB,  data = cellinfo)
    M_a     <- model.matrix(~ FA,       data = cellinfo)
    M_b     <- model.matrix(~ FB,       data = cellinfo)

    onb <- function(M, tol = 1e-10) {
        if (is.null(M) || ncol(M) == 0) return(matrix(0, nrow(M), 0))
        s <- svd(M)
        if (length(s$d) == 0) return(matrix(0, nrow(M), 0))
        keep <- s$d > tol * max(1, s$d[1])
        if (!any(keep)) return(matrix(0, nrow(M), 0))
        s$u[, keep, drop = FALSE]
    }
    compl <- function(big, small) {
        Qb <- onb(big);   if (ncol(Qb) == 0) return(Qb)
        Qs <- onb(small); if (ncol(Qs) == 0) return(Qb)
        onb(Qb - Qs %*% (t(Qs) %*% Qb))
    }

    n_cells <- nrow(L_R)
    K_R <- list(main_a      = compl(M_add, M_b),            # A adjusted for B
                main_b      = compl(M_add, M_a),            # B adjusted for A
                interaction = compl(diag(n_cells), M_add))  # non-additivity

    run_wald <- function(C) {
        if (is.null(C) || length(C) == 0) return(c(NA, NA, NA, NA))
        C <- as.matrix(C)
        if (nrow(C) == 0) return(c(NA, NA, NA, NA))
        wt <- as.data.frame(Wald_test(res, constraints = C, vcov = V_cr2,
                                      test = "HTZ"))
        nm <- names(wt)
        fstat <- as.numeric(wt[[if ("Fstat" %in% nm) "Fstat" else "F"]])
        dnum  <- if ("df_num"   %in% nm) as.numeric(wt[["df_num"]]) else nrow(C)
        dden  <- if ("df_denom" %in% nm) as.numeric(wt[["df_denom"]])
                 else as.numeric(wt[["df"]])
        c(fstat, dnum, dden, as.numeric(wt[["p_val"]]))
    }

    C_py <- list(main_a = C_main_a, main_b = C_main_b, interaction = C_inter)
    out_py <- list(); out_r <- list()
    for (k in c("main_a", "main_b", "interaction")) {
        out_py[[k]] <- run_wald(C_py[[k]])
        out_r[[k]]  <- run_wald(t(K_R[[k]]) %*% L_R)
    }

    list(ok = TRUE,
         coef_names = rownames(res$b),
         n_cells    = n_cells,
         q_r        = as.numeric(sapply(K_R, ncol)),
         py_main_a = out_py[["main_a"]], py_main_b = out_py[["main_b"]],
         py_inter  = out_py[["interaction"]],
         r_main_a  = out_r[["main_a"]],  r_main_b  = out_r[["main_b"]],
         r_inter   = out_r[["interaction"]])

}, error = function(e) list(ok = FALSE, msg = as.character(e)))
"""


def _run_r_block(df_py, V_py, levels_py, ref_py, tau2, sigma2, C_by_term,
                 effect_col):
    """Hand one fit to R and get both constraint-matrix variants back."""
    ro.globalenv['df_r']      = df_py
    ro.globalenv['V_matrix']  = V_py
    ro.globalenv['eff_col']   = effect_col
    ro.globalenv['py_tau2']   = float(tau2)
    ro.globalenv['py_sigma2'] = float(sigma2)
    ro.globalenv['ref_level'] = str(ref_py)
    ro.globalenv['py_levels'] = ro.StrVector([str(lv) for lv in levels_py])
    ro.globalenv['label_sep'] = LABEL_SEP
    for name, key in (('C_main_a', 'main_a'), ('C_main_b', 'main_b'),
                      ('C_inter', 'interaction')):
        ro.globalenv[name] = np.atleast_2d(
            np.asarray(C_by_term[key], dtype=np.float64)
        )

    out = ro.r(R_SCRIPT)
    if not out.rx2('ok')[0]:
        raise Exception(f"R error: {out.rx2('msg')[0]}")

    r_names = [str(n) for n in out.rx2('coef_names')]
    expected = ['intrcpt'] + [f'subgroup{lv}' for lv in levels_py
                              if str(lv) != str(ref_py)]
    if r_names != expected:
        raise Exception(
            "R's coefficient order does not match Python's design matrix, so "
            f"the contrast columns would not line up. R: {r_names} | "
            f"expected: {expected}"
        )
    grab = lambda key: [float(v) for v in out.rx2(key)]
    return {
        'py_C': {'main_a': grab('py_main_a'), 'main_b': grab('py_main_b'),
                 'interaction': grab('py_inter')},
        'r_C':  {'main_a': grab('r_main_a'),  'main_b': grab('r_main_b'),
                 'interaction': grab('r_inter')},
        'q_r':  [int(q) for q in out.rx2('q_r')],
    }


try:
    if 'ANALYSIS_CONFIG' not in globals():
        raise Exception("ANALYSIS_CONFIG not found. Run the pipeline first.")
    _sc = ANALYSIS_CONFIG.get('subgroup_config', {})
    if _sc.get('analysis_type') != 'two_way' or not _sc.get('moderator2'):
        raise Exception(
            "This validation applies only to a TWO-WAY subgroup analysis. "
            "Configure two moderators in Step 8 and re-run Step 9."
        )

    # --- 1. The live pipeline output -- exactly what Step 9 publishes ----
    dm = SubgroupDataManager(ANALYSIS_CONFIG)
    engine = SubgroupAnalysisEngine(dm)
    _results_df, het_live = engine.analyze_all_subgroups()
    live = het_live.get('factorial_tests') or {}
    if not live.get('available'):
        raise Exception(
            "Step 9 produced no factorial decomposition: "
            f"{live.get('reason', 'unknown reason')}"
        )

    # --- 2. Rebuild the same fit locally to recover L, Sigma and C -------
    bundle = dm.get_subgroup_data()

    def _fit_cells(cells=None):
        """Reproduce Step 9's construction, optionally on a subset of cells.

        cells=None reuses the bundle verbatim, so the 'primary' block is the
        engine's own fit rather than a look-alike. A subset mirrors
        _refit_on_cells(): stable sort, treatment coding, most-studies
        reference.
        """
        if cells is None:
            sub, X = bundle.df, bundle.X
            levels = list(bundle.subgroup_levels)
            feats, ref = list(bundle.feature_names), bundle.reference_level
        else:
            sub = bundle.df.loc[bundle.df['subgroup_label'].isin(cells)]
            sub = sub.sort_values('id', kind='stable').reset_index(drop=True)
            levels = sorted(sub['subgroup_label'].unique().tolist())
            spl = sub.groupby('subgroup_label')['id'].nunique()
            opl = sub['subgroup_label'].value_counts()
            pinned = bundle.reference_level
            ref = pinned if pinned in levels else min(
                levels, key=lambda lv: (-int(spl.get(lv, 0)),
                                        -int(opl.get(lv, 0)), lv))
            dummies = pd.get_dummies(sub['subgroup_label'], prefix='subgroup',
                                     prefix_sep='[', drop_first=False)
            dummies.columns = [f"{c}]" for c in dummies.columns]
            dummies = dummies.drop(columns=[f"subgroup[{ref}]"])
            feats = ['(intercept)'] + list(dummies.columns)
            X = np.column_stack([np.ones(len(sub), dtype=np.float64),
                                 dummies.to_numpy(dtype=np.float64)])

        y_all, vcv_full, vcv_diag, X_blocks, has_shared = (
            dm.prepare_three_level_data(sub, X)
        )
        order = np.argsort(sub['id'].to_numpy(), kind='stable')
        sub = sub.iloc[order].reset_index(drop=True)
        X = X[order, :]

        est = engine.analyzer.run_three_level_analysis(
            y_all=y_all, vcv_full=vcv_full, vcv_diag=vcv_diag,
            X_blocks=X_blocks, has_shared_controls=has_shared,
            n_total=int(len(sub)), m_studies=int(sub['id'].nunique()))
        if est is None or est.get('htz_wald') is None:
            raise Exception("The Python fit produced no AHZ machinery to test.")

        # Sigma for R must be the blocks Python actually fitted, not an
        # assumed list (see Validation 3).
        V = block_diag(*[np.atleast_2d(np.asarray(m, dtype=np.float64))
                         for m in (est.get('vcv_used') or vcv_full)]
                       ).astype(np.float64)

        y_flat = np.concatenate([np.asarray(v, dtype=np.float64).ravel()
                                 for v in y_all])
        X_flat = np.vstack([np.asarray(b, dtype=np.float64) for b in X_blocks])
        problems = []
        if V.shape != (len(sub), len(sub)):
            problems.append(f"V is {V.shape}, expected ({len(sub)}, {len(sub)})")
        if not np.allclose(y_flat, sub[dm.effect_col].to_numpy(float),
                           rtol=1e-10, atol=1e-12):
            problems.append("effect sizes are not row-aligned with df_r")
        if X_flat.shape != X.shape or not np.allclose(X_flat, X, rtol=1e-10,
                                                      atol=1e-12):
            problems.append("design-matrix rows are not aligned with df_r")
        if not np.allclose(np.diag(V), sub[dm.var_col].to_numpy(float),
                           rtol=1e-8, atol=1e-12):
            problems.append("diag(V) disagrees with the sampling-variance column")
        if problems:
            raise Exception("Python/R handoff is misaligned: "
                            + "; ".join(problems))

        L, _ = _cell_mean_contrasts(levels, feats, ref)
        bases = _factorial_contrast_bases(levels, sep=LABEL_SEP)
        if bases is None:
            raise Exception("Cell labels are not of the form 'levelA | levelB'.")
        C = {k: (bases[k].T @ L) for k in TERM_KEYS}
        py = {k: _factorial_wald_tests(bases, L, est['htz_wald'],
                                       _sc['moderator1'], _sc['moderator2'])[k]
              for k in TERM_KEYS}
        return {'df': sub, 'V': V, 'levels': levels, 'ref': ref, 'est': est,
                'C': C, 'py': py, 'q': {k: int(bases[k].shape[1])
                                        for k in TERM_KEYS}}

    def _compare(fit, stored_tests, title, note):
        """Run R on this fit and build the side-by-side table."""
        r_out = _run_r_block(
            fit['df'], fit['V'], fit['levels'], fit['ref'],
            fit['est']['tau_sq'], fit['est']['sigma_sq'], fit['C'],
            dm.effect_col)

        rows, worst = [], 0.0
        for key in TERM_KEYS:
            py_e = fit['py'][key]
            if not py_e.get('estimable'):
                rows.append({'Term': py_e['term'], 'Quantity': '(not estimable)',
                             'Co-Meta (Python)': np.nan,
                             'R · Python contrasts': np.nan,
                             'R · own contrasts': np.nan})
                continue
            # The engine's live output must match this local rebuild, or the
            # table would be validating something the user never sees.
            stored = (stored_tests or {}).get(key, {})
            if stored.get('estimable') and not np.isclose(
                    stored['F'], py_e['F'], rtol=1e-8, atol=1e-10):
                raise Exception(
                    f"Rebuilt fit disagrees with Step 9's stored result for "
                    f"'{py_e['term']}' (F {py_e['F']:.6f} vs "
                    f"{stored['F']:.6f}); the comparison below would be "
                    "meaningless."
                )
            pyv = [py_e['F'], float(py_e['df_num']), py_e['df_denom'],
                   py_e['p_value']]
            for label, idx in (('F statistic', 0), ('df numerator', 1),
                               ('df denominator', 2), ('p value', 3)):
                a, b, c = pyv[idx], r_out['py_C'][key][idx], r_out['r_C'][key][idx]
                rows.append({'Term': py_e['term'], 'Quantity': label,
                             'Co-Meta (Python)': a,
                             'R · Python contrasts': b,
                             'R · own contrasts': c})
                for other in (b, c):
                    if np.isfinite(a) and np.isfinite(other):
                        worst = max(worst, abs(a - other)
                                    / max(1.0, abs(a)))
        df_cmp = pd.DataFrame(rows)
        df_cmp['Max rel. diff'] = np.nan
        for i, row in df_cmp.iterrows():
            vals = [row['Co-Meta (Python)'], row['R · Python contrasts'],
                    row['R · own contrasts']]
            if all(pd.notna(v) for v in vals):
                df_cmp.at[i, 'Max rel. diff'] = max(
                    abs(vals[0] - vals[1]), abs(vals[0] - vals[2])
                ) / max(1.0, abs(vals[0]))

        ok = worst < 1e-6
        banner_bg, banner_bd, banner_fg, icon, verdict = (
            ('#d4edda', '#28a745', '#155724', '✅', 'agree to numerical precision')
            if ok else
            ('#fff3cd', '#ffc107', '#856404', '⚠️', 'DIFFER — investigate')
        )
        display(HTML(f"""
        <div style='padding: 15px; background-color: {banner_bg}; border-left: 5px
             solid {banner_bd}; color: {banner_fg}; border-radius: 4px;
             font-family: sans-serif; margin: 20px 0 10px 0;'>
            <h4 style='margin: 0 0 5px 0;'>{icon} {title}</h4>
            <p style='margin: 0; font-size: 14px;'>{note}
            Largest relative difference: <b>{worst:.2e}</b> — the three
            columns {verdict}.</p>
        </div>"""))

        shown = df_cmp.copy()
        for col in ('Co-Meta (Python)', 'R · Python contrasts',
                    'R · own contrasts'):
            shown[col] = shown[col].apply(
                lambda x: f"{x:.6f}" if pd.notna(x) else "N/A")
        shown['Max rel. diff'] = shown['Max rel. diff'].apply(
            lambda x: f"{x:.2e}" if pd.notna(x) else "N/A")

        def _shade(val):
            if val in ('', 'N/A'):
                return ''
            try:
                return ('background-color: #d4edda' if float(val) < 1e-6
                        else 'background-color: #fff3cd')
            except ValueError:
                return ''
        # Styler.applymap was renamed to Styler.map in pandas 2.1 and removed
        # in 3.0; support both so the cell runs on any Colab image.
        _sty = shown.style
        _elementwise = _sty.map if hasattr(_sty, 'map') else _sty.applymap
        _styled = (_elementwise(_shade, subset=['Max rel. diff'])
                   .set_properties(**{'text-align': 'center'}))
        # The row index carries no information here; hide it on whichever
        # pandas is installed.
        try:
            _styled = _styled.hide(axis='index')
        except (AttributeError, TypeError):
            try:
                _styled = _styled.hide_index()
            except AttributeError:
                pass
        display(_styled)
        _v13_checks.append({'check': title, 'max rel. diff': worst, 'status': 'PASS' if ok else 'FAIL'})
        return ok

    clear_output()
    all_ok = True
    _v13_checks = []

    primary_fit = _fit_cells(None)
    kind = ('Type III — the cross-classification is complete'
            if live.get('is_complete_grid') else
            'Type II — the cross-classification is incomplete '
            f"({live.get('n_cells_observed')} of "
            f"{live.get('n_cells_full_grid')} cells observed)")
    all_ok &= _compare(
        primary_fit, live.get('primary'),
        'Primary decomposition — main effects and interaction',
        f"{kind}. Sigma is pinned to Co-Meta's tau^2/sigma^2 in R, so this "
        "isolates the hypothesis machinery from variance estimation.")

    sens = live.get('sensitivity')
    if sens and sens.get('tests'):
        sens_fit = _fit_cells(sens['cells'])
        all_ok &= _compare(
            sens_fit, sens['tests'],
            'Sensitivity refit — largest complete subgrid',
            f"Refit on {len(sens['cells'])} fully-crossed cells "
            f"({sens['k_obs']} effect sizes from {sens['n_studies']} studies; "
            f"{sens['k_obs_dropped']} effect sizes dropped). ")
    elif not live.get('is_complete_grid'):
        display(HTML(
            "<div style='padding: 12px; background-color: #e2e3e5; border-left: "
            "4px solid #6c757d; color: #383d41; border-radius: 4px; font-family: "
            "sans-serif;'>ℹ️ No complete-subgrid sensitivity analysis was "
            f"produced: {live.get('sensitivity_skipped_reason', 'not applicable')}"
            "</div>"))

    display(HTML(f"""
    <div style='padding: 15px; background-color: #f8f9fa; border-left: 5px solid
         #6c757d; color: #383d41; border-radius: 4px; font-family: sans-serif;
         margin-top: 20px;'>
        <p style='margin: 0; font-size: 13px;'><b>How to read this.</b>
        <i>Co-Meta (Python)</i> vs <i>R · Python contrasts</i> tests the
        approximate Hotelling–Zhang implementation: the same constraint matrix
        goes to <code>clubSandwich::Wald_test(test = "HTZ")</code>.
        <i>R · Python contrasts</i> vs <i>R · own contrasts</i> tests the
        contrast construction: R rebuilds the interaction and Type II spaces
        from its own <code>model.matrix()</code> and SVD, and two constraint
        matrices spanning the same row space must yield the same test. A gap
        in the first pair points at the test engine; a gap in the second
        points at which hypothesis is being tested.</p>
    </div>"""))
    if 'record_validation' in globals():
        record_validation(13, "Two-Way Factorial Decomposition", pd.DataFrame(_v13_checks))

except Exception as e:
    # A crash is a failed validation, not 'not run'
    if 'record_validation' in globals() and not validation_prerequisite_missing(e):
        record_validation(13, 'Two-Way Factorial Decomposition',
                          [('validation ran to completion', 'FAIL')],
                          notes=f"{type(e).__name__}: {e}")
    clear_output()
    display(HTML(
        "<div style='padding: 15px; background-color: #f8d7da; border-left: 5px "
        "solid #dc3545; color: #721c24; border-radius: 4px;'>"
        f"<b>❌ Error:</b> {e}</div>"))
    import traceback
    traceback.print_exc()


In [ ]:
#@title 🧪 Validation 14: Meta-Regression Omnibus & Predicted Effects (Python vs R)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

# =============================================================================
# Covers the two parts of Step 12 that had no cross-check:
#
#   OMNIBUS   the AHZ small-sample F for each categorical moderator and for
#             all moderator terms jointly. Validation 4 (MULTI) says in its
#             own banner that the omnibus is "not validated here"; this is
#             that check.
#   PREDICT   the effect predicted at given moderator values, its CR2
#             standard error, its contrast-specific Satterthwaite DF, and
#             the resulting confidence and prediction intervals.
#
# As in Validation 13, two independent things are separated so a failure
# points at one of them:
#
#   Python  vs  R · Co-Meta design   -> the test/prediction ARITHMETIC. R is
#       handed the exact design matrix and sampling-VCV blocks the engine
#       fitted, with tau^2 and sigma^2 pinned to Python's values.
#   R · Co-Meta design  vs  R · own formula  -> the DESIGN and the hypothesis.
#       R rebuilds the model from the raw moderator columns with its own
#       formula and groups the columns with attr(, "assign"), taking nothing
#       from Python. R may pick different reference levels, but a per-factor
#       omnibus tests that factor's whole column space and is invariant to
#       them, so the F values must still agree.
#
# NOTE on predict.rma: metafor's predict() uses the MODEL-BASED covariance
# (res$vb); it has no vcov argument. Its point prediction is therefore the
# right reference for the estimate, but its standard error is not comparable
# to Co-Meta's CR2 one. The CR2 SE is instead taken from clubSandwich two
# ways -- directly from vcovCR, and implied by Wald_test's F at q = 1 --
# both of which are clubSandwich's own output.
# =============================================================================
import pandas as pd
import numpy as np
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import rpy2.robjects.numpy2ri as numpy2ri
from scipy.linalg import block_diag
from scipy.stats import t as _t_dist
from IPython.display import display, HTML, clear_output
import logging

logging.getLogger('rpy2.rinterface_lib.callbacks').setLevel(logging.ERROR)
pandas2ri.activate()
numpy2ri.activate()

display(HTML(
    "<div style='padding: 15px; background-color: #e2e3e5; border-left: 4px solid "
    "#6c757d; color: #383d41; border-radius: 4px; font-family: sans-serif;'>"
    "⏳ <b>Validating the meta-regression omnibus and predicted effects "
    "against metafor + clubSandwich...</b> Please wait.</div>"))

R_SCRIPT = """
suppressPackageStartupMessages(library(metafor))
suppressPackageStartupMessages(library(clubSandwich))

tryCatch({
    dat <- df_r
    dat$row_id   <- 1:nrow(dat)
    dat$study_id <- as.factor(dat$.sid)

    is_multilevel <- length(unique(dat$study_id)) < nrow(dat)
    random_form <- if (is_multilevel) ~ 1 | study_id/row_id else ~ 1 | study_id
    fixed_vars  <- if (is_multilevel) c(py_tau2, py_sigma2) else py_tau2

    # ---- Fit 1: Co-Meta's own design, Sigma pinned ------------------
    Xmods <- X_py[, -1, drop = FALSE]
    res <- suppressWarnings(rma.mv(
        yi = dat$.yi, V = V_matrix, mods = Xmods, random = random_form,
        data = dat, method = "REML", test = "z", sigma2 = fixed_vars))
    V_cr2 <- vcovCR(res, type = "CR2")

    htz <- function(obj, vc, C) {
        C <- as.matrix(C)
        if (nrow(C) == 0) return(c(NA, NA, NA, NA))
        wt <- as.data.frame(Wald_test(obj, constraints = C, vcov = vc,
                                      test = "HTZ"))
        nm <- names(wt)
        c(as.numeric(wt[[if ("Fstat" %in% nm) "Fstat" else "F"]]),
          if ("df_num"   %in% nm) as.numeric(wt[["df_num"]])   else nrow(C),
          if ("df_denom" %in% nm) as.numeric(wt[["df_denom"]]) else as.numeric(wt[["df"]]),
          as.numeric(wt[["p_val"]]))
    }
    sel <- function(p_tot, idx) {        # rows of I_p selecting those columns
        C <- matrix(0, nrow = length(idx), ncol = p_tot)
        for (i in seq_along(idx)) C[i, idx[i]] <- 1
        C
    }

    p_tot <- length(res$b)
    # grp[j] = factor index of non-intercept column j (0 = its own term)
    omni_py <- list()
    for (g in sort(unique(grp_ids))) {
        idx <- which(grp_ids == g) + 1L     # +1: skip the intercept
        omni_py[[as.character(g)]] <- htz(res, V_cr2, sel(p_tot, idx))
    }
    overall_py <- htz(res, V_cr2, sel(p_tot, 2:p_tot))

    # ---- Fit 2: R's own formula from the raw moderators -------------
    for (i in seq_along(mod_names)) {
        if (is_cat[i] == 1) dat[[mod_names[i]]] <- as.factor(dat[[mod_names[i]]])
        else                dat[[mod_names[i]]] <- as.numeric(dat[[mod_names[i]]])
    }
    frm <- as.formula(paste("~", paste(mod_names, collapse = " + ")))
    MM  <- model.matrix(frm, data = dat)
    assign_vec <- attr(MM, "assign")        # R's own column -> term mapping
    res_r <- suppressWarnings(rma.mv(
        yi = dat$.yi, V = V_matrix, mods = MM[, -1, drop = FALSE],
        random = random_form, data = dat, method = "REML", test = "z",
        sigma2 = fixed_vars))
    V_cr2_r <- vcovCR(res_r, type = "CR2")
    p_r <- length(res_r$b)

    omni_r <- list()
    for (g in seq_along(mod_names)) {
        idx <- which(assign_vec == g)       # already 1-based incl. intercept
        omni_r[[as.character(g)]] <- htz(res_r, V_cr2_r, sel(p_r, idx))
    }
    overall_r <- htz(res_r, V_cr2_r, sel(p_r, 2:p_r))

    # ---- Predicted effects on Co-Meta's design ----------------------
    # pred from predict.rma (point prediction; vcov-independent).
    # SE from clubSandwich's CR2 matrix, and again from the Wald F, so the
    # robust SE never comes from metafor's model-based one.
    newm <- X0[, -1, drop = FALSE]
    pr   <- predict(res, newmods = newm)
    n0   <- nrow(X0)
    pred_est <- as.numeric(pr$pred)
    pred_se  <- numeric(n0); pred_se_f <- numeric(n0)
    pred_df  <- numeric(n0)
    for (i in seq_len(n0)) {
        x0 <- matrix(X0[i, ], nrow = 1)
        pred_se[i] <- sqrt(as.numeric(x0 %*% V_cr2 %*% t(x0)))
        w <- htz(res, V_cr2, x0)
        pred_df[i] <- w[3]
        pred_se_f[i] <- if (is.finite(w[1]) && w[1] > 0)
                            abs(as.numeric(x0 %*% res$b)) / sqrt(w[1]) else NA
    }
    crit    <- qt(1 - (1 - ci_level/100)/2, df = pred_df)
    se_pi   <- sqrt(pred_se^2 + py_tau2 + py_sigma2)
    list(ok = TRUE,
         n_coef = p_tot, n_coef_r = p_r,
         omni_py = unlist(omni_py), omni_r = unlist(omni_r),
         overall_py = overall_py, overall_r = overall_r,
         pred_est = pred_est, pred_se = pred_se, pred_se_f = pred_se_f,
         pred_df = pred_df,
         ci_lo = pred_est - crit * pred_se, ci_hi = pred_est + crit * pred_se,
         pi_lo = pred_est - crit * se_pi,   pi_hi = pred_est + crit * se_pi)

}, error = function(e) list(ok = FALSE, msg = as.character(e)))
"""

try:
    res_obj = globals().get('LAST_REGRESSION_RESULT')
    if res_obj is None:
        raise Exception(
            "No meta-regression in memory. Run the Meta-Regression cell first."
        )
    fit_in = getattr(res_obj, 'fit_inputs', None) or {}
    col_meta = getattr(res_obj, 'col_meta', None)
    term_groups = getattr(res_obj, 'term_groups', None)
    mod_cols = list(getattr(res_obj, 'moderator_cols', None) or [])
    cat_cols = set(getattr(res_obj, 'categorical_cols', None) or [])
    if not (fit_in.get('X_stacked') is not None and col_meta and term_groups
            and mod_cols):
        raise Exception(
            "This result predates the validation plumbing (no design/fit "
            "metadata). Re-run the Meta-Regression cell."
        )
    if getattr(res_obj, 'htz_wald', None) is None:
        raise Exception(
            "The fit carries no AHZ machinery (robust covariance unavailable), "
            "so there is nothing of this kind to validate."
        )

    X_py = np.asarray(fit_in['X_stacked'], dtype=np.float64)
    y_py = np.asarray(fit_in['y_stacked'], dtype=np.float64)
    sid_py = list(fit_in['row_ids'])
    vcv_used = fit_in.get('vcv_used')
    if vcv_used is None:
        raise Exception("The fit did not record which sampling-VCV blocks it used.")
    V_matrix = block_diag(*[np.atleast_2d(np.asarray(m, dtype=np.float64))
                            for m in vcv_used]).astype(np.float64)

    # Rebuild the engine's row order from the saved regression frame, then
    # prove it matches: same y, same design, same sampling variances.
    # Step 12 stores its results under 'meta_regression_RVE_results'
    saved = (ANALYSIS_CONFIG.get('meta_regression_RVE_results')
             or ANALYSIS_CONFIG.get('regression_results') or {})
    reg_df = saved.get('reg_df')
    eff_col = saved.get('effect_col')
    var_col = ANALYSIS_CONFIG.get('var_col')
    if reg_df is None or not eff_col:
        raise Exception("ANALYSIS_CONFIG has no saved regression frame to align against.")
    ordered = reg_df.sort_values(['id'], kind='mergesort').reset_index(drop=True)

    problems = []
    if len(ordered) != X_py.shape[0]:
        problems.append(f"{len(ordered)} saved rows vs {X_py.shape[0]} fitted rows")
    elif not np.allclose(ordered[eff_col].to_numpy(float), y_py, rtol=1e-10, atol=1e-12):
        problems.append("the saved frame's effect sizes are not in the engine's row order")
    if [str(s) for s in ordered['id']] != [str(s) for s in sid_py]:
        problems.append("study ids are not in the engine's row order")
    if V_matrix.shape != (X_py.shape[0], X_py.shape[0]):
        problems.append(f"V is {V_matrix.shape}, expected "
                        f"({X_py.shape[0]}, {X_py.shape[0]})")
    elif var_col and var_col in ordered.columns and not np.allclose(
            np.diag(V_matrix), ordered[var_col].to_numpy(float),
            rtol=1e-8, atol=1e-12):
        n_bad = int(np.sum(~np.isclose(np.diag(V_matrix),
                                       ordered[var_col].to_numpy(float),
                                       rtol=1e-8, atol=1e-12)))
        problems.append(
            f"diag(V) disagrees with '{var_col}' on {n_bad} of {len(ordered)} "
            "rows -- the cached per-study sampling VCVs do not describe these rows")
    if problems:
        raise Exception("Python/R handoff is misaligned: " + "; ".join(problems))

    betas = np.asarray(res_obj.betas, dtype=np.float64).ravel()
    V_py = np.asarray(res_obj.var_betas_robust, dtype=np.float64)
    p_tot = betas.size

    # Per-moderator column groups, as 1-based indices among NON-intercept cols
    ordered_mods = [m for m in mod_cols if m in term_groups]
    grp_ids = np.zeros(p_tot - 1, dtype=np.int32)
    for gi, m in enumerate(ordered_mods, start=1):
        for c in term_groups[m]:
            grp_ids[c - 1] = gi

    # Python side: omnibus per moderator and overall, via the live closure
    htz_py = res_obj.htz_wald
    def _py_htz(idx):
        C = np.zeros((len(idx), p_tot))
        for r, c in enumerate(idx):
            C[r, c] = 1.0
        out = htz_py(C, test='AHZ')
        return [float(out['F']), float(out['df_num']),
                float(out['df_denom']), float(out['p_val'])]

    py_omni = {m: _py_htz(term_groups[m]) for m in ordered_mods}
    py_overall = _py_htz(list(range(1, p_tot)))

    # Prediction grid and Python's own predictions (production code path)
    grid = default_prediction_grid(res_obj)
    ci_level = float(100.0 * (1.0 - ANALYSIS_CONFIG.get(
        'global_settings', {}).get('alpha', 0.05)))
    py_pred = predict_from_fit(res_obj, grid, ci_level=ci_level)
    X0 = np.vstack([_resolve_prediction_row(col_meta, g, res_obj.term_names)[0]
                    for g in grid])

    # ---- hand everything to R ---------------------------------------
    df_r = ordered[[c for c in mod_cols if c in ordered.columns]].copy()
    df_r['.yi'] = y_py
    df_r['.sid'] = [str(s) for s in sid_py]
    ro.globalenv['df_r'] = df_r
    ro.globalenv['V_matrix'] = V_matrix
    ro.globalenv['X_py'] = X_py
    ro.globalenv['X0'] = X0
    ro.globalenv['py_tau2'] = float(res_obj.tau_sq)
    ro.globalenv['py_sigma2'] = float(res_obj.sigma_sq)
    ro.globalenv['grp_ids'] = ro.IntVector([int(g) for g in grp_ids])
    ro.globalenv['mod_names'] = ro.StrVector(ordered_mods)
    ro.globalenv['is_cat'] = ro.IntVector(
        [1 if m in cat_cols else 0 for m in ordered_mods])
    ro.globalenv['ci_level'] = ci_level

    r_out = ro.r(R_SCRIPT)
    if not r_out.rx2('ok')[0]:
        raise Exception(f"R error: {r_out.rx2('msg')[0]}")

    n_coef_r = int(r_out.rx2('n_coef_r')[0])
    if int(r_out.rx2('n_coef')[0]) != p_tot:
        raise Exception("R's model on Co-Meta's design has a different number "
                        "of coefficients than Co-Meta's fit.")

    def _grab4(key, offset=0):
        v = [float(x) for x in r_out.rx2(key)]
        return v[offset:offset + 4]

    clear_output()

    # ---------------- OMNIBUS TABLE ----------------------------------
    rows, worst_arith, worst_design = [], 0.0, 0.0
    labels = ('F statistic', 'df numerator', 'df denominator', 'p value')
    blocks = [(m, py_omni[m], _grab4('omni_py', 4 * i), _grab4('omni_r', 4 * i))
              for i, m in enumerate(ordered_mods)]
    blocks.append(('ALL moderator terms', py_overall,
                   [float(x) for x in r_out.rx2('overall_py')],
                   [float(x) for x in r_out.rx2('overall_r')]))
    for name, a, b, c in blocks:
        for j, lab in enumerate(labels):
            rows.append({'Term': name, 'Quantity': lab,
                         'Co-Meta (Python)': a[j],
                         'R · Co-Meta design': b[j],
                         'R · own formula': c[j]})
            if np.isfinite(a[j]) and np.isfinite(b[j]):
                worst_arith = max(worst_arith, abs(a[j] - b[j]) / max(1.0, abs(a[j])))
            if np.isfinite(b[j]) and np.isfinite(c[j]):
                worst_design = max(worst_design, abs(b[j] - c[j]) / max(1.0, abs(b[j])))
    df_om = pd.DataFrame(rows)

    _v14_checks = []

    def _render(df_tbl, num_cols, title, note, worsts):
        shown = df_tbl.copy()
        for col in num_cols:
            shown[col] = shown[col].apply(
                lambda x: f"{x:.6f}" if pd.notna(x) else "N/A")
        ok = all(w < 1e-6 for _, w in worsts)
        _v14_checks.extend({'check': f"{title}: {lab}", 'max rel. diff': w,
                            'status': 'PASS' if w < 1e-6 else 'FAIL'} for lab, w in worsts)
        bg, bd, fg, icon = (('#d4edda', '#28a745', '#155724', '✅') if ok
                            else ('#fff3cd', '#ffc107', '#856404', '⚠️'))
        wtxt = " &nbsp;|&nbsp; ".join(f"{lab}: <b>{w:.2e}</b>" for lab, w in worsts)
        display(HTML(f"""
        <div style='padding:15px;background-color:{bg};border-left:5px solid {bd};
             color:{fg};border-radius:4px;font-family:sans-serif;margin:20px 0 10px 0;'>
          <h4 style='margin:0 0 5px 0;'>{icon} {title}</h4>
          <p style='margin:0;font-size:14px;'>{note} Largest relative
          difference &mdash; {wtxt}.</p></div>"""))
        sty = shown.style
        elementwise = sty.map if hasattr(sty, 'map') else sty.applymap
        styled = sty.set_properties(**{'text-align': 'center'})
        try:
            styled = styled.hide(axis='index')
        except (AttributeError, TypeError):
            pass
        display(styled)

    _render(df_om, ['Co-Meta (Python)', 'R · Co-Meta design', 'R · own formula'],
            'Omnibus tests — per moderator and all terms jointly',
            "Sigma is pinned to Co-Meta's tau^2/sigma^2 in both R fits, so this "
            "isolates the hypothesis machinery from variance estimation. R's own "
            "formula may choose different reference levels; a per-factor omnibus "
            "is invariant to that, so the F values must still agree.",
            [('arithmetic', worst_arith), ('design', worst_design)])

    # ---------------- PREDICTION TABLE -------------------------------
    r_est = [float(x) for x in r_out.rx2('pred_est')]
    r_se = [float(x) for x in r_out.rx2('pred_se')]
    r_se_f = [float(x) for x in r_out.rx2('pred_se_f')]
    r_df = [float(x) for x in r_out.rx2('pred_df')]
    r_ci_lo = [float(x) for x in r_out.rx2('ci_lo')]
    r_ci_hi = [float(x) for x in r_out.rx2('ci_hi')]
    r_pi_lo = [float(x) for x in r_out.rx2('pi_lo')]
    r_pi_hi = [float(x) for x in r_out.rx2('pi_hi')]

    mod_disp = [c for c in py_pred.columns
                if c not in ('estimate', 'se', 'df', 'ci_lower', 'ci_upper',
                             'pi_lower', 'pi_upper', 'df_method', 'notes')]
    prows, worst_pred, worst_se2 = [], 0.0, 0.0
    for i in range(len(py_pred)):
        tag = ", ".join(
            f"{c}={py_pred.iloc[i][c]:.4g}" if isinstance(py_pred.iloc[i][c], float)
            else f"{c}={py_pred.iloc[i][c]}" for c in mod_disp)
        for lab, pv, rv in (
            ('estimate (predict.rma)', py_pred.iloc[i]['estimate'], r_est[i]),
            ('SE, CR2 (vcovCR)',       py_pred.iloc[i]['se'],       r_se[i]),
            ('df (Wald_test HTZ)',     py_pred.iloc[i]['df'],       r_df[i]),
            ('CI lower',               py_pred.iloc[i]['ci_lower'], r_ci_lo[i]),
            ('CI upper',               py_pred.iloc[i]['ci_upper'], r_ci_hi[i]),
            ('PI lower',               py_pred.iloc[i]['pi_lower'], r_pi_lo[i]),
            ('PI upper',               py_pred.iloc[i]['pi_upper'], r_pi_hi[i]),
        ):
            prows.append({'At': tag, 'Quantity': lab,
                          'Co-Meta (Python)': pv, 'metafor / clubSandwich (R)': rv})
            if np.isfinite(pv) and np.isfinite(rv):
                worst_pred = max(worst_pred, abs(pv - rv) / max(1.0, abs(pv)))
        if np.isfinite(r_se[i]) and np.isfinite(r_se_f[i]):
            worst_se2 = max(worst_se2,
                            abs(r_se[i] - r_se_f[i]) / max(1.0, abs(r_se[i])))

    _render(pd.DataFrame(prows),
            ['Co-Meta (Python)', 'metafor / clubSandwich (R)'],
            'Predicted effects at the default grid',
            "Point predictions come from <code>metafor::predict.rma</code>; the "
            "robust SE and DF come from <code>clubSandwich</code> "
            "(<code>vcovCR</code> and <code>Wald_test(test=\"HTZ\")</code>), because "
            "predict.rma reports a model-based SE, not a CR2 one. CI and PI bounds "
            "are rebuilt in R from R's own estimate, SE and DF.",
            [('prediction', worst_pred),
             ('CR2 SE, two clubSandwich routes', worst_se2)])

    display(HTML(f"""
    <div style='padding:15px;background-color:#f8f9fa;border-left:5px solid #6c757d;
         color:#383d41;border-radius:4px;font-family:sans-serif;margin-top:20px;'>
      <p style='margin:0;font-size:13px;'><b>How to read this.</b> In the omnibus
      table, a gap in the <i>arithmetic</i> figure points at the AHZ
      implementation; a gap in the <i>design</i> figure means R and Co-Meta
      disagree about which hypothesis is being tested, not about how to test it.
      In the prediction table, the second figure compares two routes to the same
      CR2 standard error inside clubSandwich &mdash; straight from
      <code>vcovCR</code>, and implied by the Wald <i>F</i> at q&nbsp;=&nbsp;1 &mdash;
      so a gap there is internal to R and means the comparison itself is
      unsound. Model fitted on {X_py.shape[0]} effect sizes
      ({len(set(map(str, sid_py)))} studies), {p_tot} coefficients; R's own
      formula produced {n_coef_r}.</p></div>"""))
    if 'record_validation' in globals():
        record_validation(14, "Meta-Regression Omnibus & Predicted Effects", pd.DataFrame(_v14_checks))

except Exception as e:
    # A crash is a failed validation, not 'not run'
    if 'record_validation' in globals() and not validation_prerequisite_missing(e):
        record_validation(14, 'Meta-Regression Omnibus & Predicted Effects',
                          [('validation ran to completion', 'FAIL')],
                          notes=f"{type(e).__name__}: {e}")
    clear_output()
    display(HTML(
        "<div style='padding:15px;background-color:#f8d7da;border-left:5px solid "
        f"#dc3545;color:#721c24;border-radius:4px;'><b>❌ Error:</b> {e}</div>"))
    import traceback
    traceback.print_exc()


In [ ]:
#@title 🧪 Validation 15: Unified Engine (fit_mv) vs metafor rma.mv — nested variance structures on your data
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# YOUR DATA (default). Builds the FitSpec exactly as Step 7's nested
# variance-structure comparison does (the rows Step 7 uses, your effect and
# variance columns, your sampling VCV with the shared-control covariances)
# and fits, with the unified engine and with metafor on the same rows:
#     full     : rma.mv(yi, V, random = ~ 1 | id/es_id)              sigma2 = c(NA, NA)
#     between  : ... sigma2 = c(NA, 0)   <->  fit_mv(spec, sigma2 = 0)
#     within   : ... sigma2 = c(0, NA)   <->  fit_mv(spec, tau2 = 0)
# under REML and ML; compared: logLik, AIC, tau2, sigma2 and beta, plus the
# nesting guarantee AIC(reduced) - AIC(full) >= -2. The V sent to R is your
# VCV sliced independently of the engine (validation_block_V), and the cell
# checks that the FitSpec's V is that same matrix. When Step 7 has run, the
# REML fits it stored (variance_structure_comparison) are checked against R
# too. With one effect size per study the full model is not identifiable
# (tau2 and sigma2 cannot be separated): only the two reduced structures are
# checked, and the full one is listed as INFO.
#
# REFERENCE DATASETS (VALIDATION_INCLUDE_REFERENCE = True): three simulated
# shared-control datasets (fixed seeds) with 1-4 effects per study.
# Requires: Cell 1b (engine), Steps 2-6 (and 7 for the stored comparison)
# and the Validation R setup cell.

import numpy as np
import pandas as pd
from scipy.linalg import block_diag
import rpy2.robjects as ro
from IPython.display import display, HTML

# R objects are built explicitly instead of through the numpy converter:
# depending on the rpy2 version, the converter hands a 1-D numpy array to R
# as an *array* (a vector with a dim attribute), which rma.mv rejects for yi.


def _rvec(a):
    return ro.FloatVector(np.asarray(a, dtype=float).ravel())


def _rmat(M):
    M = np.asarray(M, dtype=float)
    return ro.r['matrix'](ro.FloatVector(M.ravel(order='F')), nrow=M.shape[0], ncol=M.shape[1])

TOL = {'loglik': 1e-4, 'aic': 1e-3, 'var': 1e-4, 'beta': 1e-4, 'V': 1e-12}   # var, beta: scaled (see _sdiff)

def _sdiff(a, b):
    """|a - b| on the scale of the value: absolute below 1, relative above,
    so the same tolerance works for lnRR (~0.1) and mean differences (~50)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.nanmax(np.abs(a - b) / np.maximum(1.0, np.abs(b))))

STRUCTURES = {               # name: (fit_mv kwargs, metafor sigma2)
    'full':    ({},              'c(NA, NA)'),
    'between': ({'sigma2': 0.0}, 'c(NA, 0)'),
    'within':  ({'tau2': 0.0},   'c(0, NA)'),
}


def _simulated(seed, M=20):
    rng = np.random.default_rng(seed)
    rows, vcv = [], {}
    for i in range(M):
        k = int(rng.integers(1, 5))
        V = np.full((k, k), rng.uniform(.01, .06)) + np.diag(rng.uniform(.02, .1, k))
        y = (0.3 + rng.normal(0, .2) + rng.normal(0, .15, k)
             + rng.multivariate_normal(np.zeros(k), V))
        vcv[f"S{i:02d}"] = V
        rows += [dict(id=f"S{i:02d}", yi=y[j], vi=V[j, j]) for j in range(k)]
    df = pd.DataFrame(rows)
    return df, vcv


def _r_fit(y, V, cl, sigma2_r, method):
    """rma.mv on the rows (y, V, cluster) sent from the validation side."""
    ro.globalenv['y_v'] = _rvec(y)
    ro.globalenv['V_m'] = _rmat(V)
    ro.globalenv['cl_v'] = ro.StrVector([str(c) for c in cl])
    out = ro.r(f"""
    suppressPackageStartupMessages(library(metafor))
    tryCatch({{
        y_v <- as.numeric(y_v)
        d <- data.frame(cl = factor(as.character(cl_v), levels = unique(as.character(cl_v))),
                        es = factor(seq_along(y_v)))
        r <- suppressWarnings(rma.mv(yi = y_v, V = V_m, random = ~ 1 | cl/es, data = d,
                    method = "{method}", sigma2 = {sigma2_r},
                    control = list(rel.tol = 1e-10, iter.max = 1000)))
        list(ok = TRUE, ll = as.numeric(logLik(r)), aic = AIC(r),
             s2 = as.numeric(r$sigma2), b = as.numeric(r$b))
    }}, error = function(e) list(ok = FALSE, msg = conditionMessage(e)))
    """)
    if not bool(out.rx2('ok')[0]):
        raise RuntimeError(str(out.rx2('msg')[0]))
    s2 = np.asarray(out.rx2('s2'))
    return {'loglik': float(out.rx2('ll')[0]), 'aic': float(out.rx2('aic')[0]),
            'tau2': float(s2[0]), 'sigma2': float(s2[1]),
            'beta': np.asarray(out.rx2('b'), dtype=float)}


def _rows_and_V(spec, df, vcv):
    """The spec's rows taken from `df` with V sliced by validation_block_V,
    plus the max |difference| between that V and the FitSpec's V."""
    d, V = validation_block_V(df.loc[list(spec.row_index)], vcv)
    V_spec = block_diag(*spec.V_blocks)
    pos = {lab: i for i, lab in enumerate(spec.row_index)}
    P = [pos[lab] for lab in d.index]
    dV = float(np.max(np.abs(V_spec[np.ix_(P, P)] - V))) if len(P) else 0.0
    return d, V, dV


def validate_dataset(name, spec, df, vcv, stored=None):
    rows = []
    d, V, dV = _rows_and_V(spec, df, vcv)
    n_off = int(np.count_nonzero(np.abs(V - np.diag(np.diag(V))) > 0) // 2)
    rows.append({'dataset': name, 'method': '—', 'structure': 'FitSpec V = sampling VCV (sliced independently)',
                 'max |Δ|': dV, 'status': 'PASS' if dV < TOL['V'] else 'FAIL',
                 'note': f"{spec.n_effects} effects, {spec.n_clusters} studies, {n_off} covariances"})
    identifiable = spec.n_effects > spec.n_clusters
    y, cl = d['yi'].to_numpy(dtype=float), d['id'].astype(str).tolist()
    r_reml = {}
    for method in ('REML', 'ML'):
        fits = {}
        for sname, (kw, s2r) in STRUCTURES.items():
            if sname == 'full' and not identifiable:
                rows.append({'dataset': name, 'method': method, 'structure': sname, 'status': 'INFO',
                             'note': 'one effect size per study: tau2 and sigma2 are not separately '
                                     'identifiable, so the full model is not compared'})
                continue
            py = fit_mv(spec, method=method, **kw)
            fits[sname] = py
            try:
                rr = _r_fit(y, V, cl, s2r, method)
            except Exception as e:
                rows.append({'dataset': name, 'method': method, 'structure': sname,
                             'status': f'R error: {e}'})
                continue
            if method == 'REML':
                r_reml[sname] = rr
            dd = {
                'loglik': abs(py.loglik - rr['loglik']),
                'aic': abs(py.aic - rr['aic']),
                'tau2': _sdiff(py.tau2, rr['tau2']),
                'sigma2': _sdiff(py.sigma2, rr['sigma2']),
                'beta': _sdiff(py.betas, rr['beta']),
            }
            ok = (dd['loglik'] < TOL['loglik'] and dd['aic'] < TOL['aic']
                  and dd['tau2'] < TOL['var'] and dd['sigma2'] < TOL['var']
                  and dd['beta'] < TOL['beta'])
            rows.append({'dataset': name, 'method': method, 'structure': sname,
                         'logLik (Py)': py.loglik, 'logLik (R)': rr['loglik'],
                         'AIC (Py)': py.aic, 'AIC (R)': rr['aic'],
                         'tau2 (Py)': py.tau2, 'tau2 (R)': rr['tau2'],
                         'sigma2 (Py)': py.sigma2, 'sigma2 (R)': rr['sigma2'],
                         'max |Δ|': max(dd.values()), 'status': 'PASS' if ok else 'FAIL'})
        if 'full' in fits:
            for red in ('between', 'within'):
                c = compare_fits(fits['full'], fits[red])
                ok = c.delta_aic >= -2 - 1e-6 and not c.warnings
                rows.append({'dataset': name, 'method': method,
                             'structure': f'ΔAIC({red} − full) ≥ −2',
                             'AIC (Py)': c.delta_aic, 'max |Δ|': np.nan,
                             'status': ('PASS' if ok else 'FAIL') +
                                       f' (LRT={c.lrt:.3f}, p̄={c.p_value:.4f})'})
    # The REML fits Step 7 stored (what the user's report shows)
    if stored is not None:
        for sname, st in stored.items():
            rr = r_reml.get(sname)
            if rr is None:
                continue
            dd = {'loglik': abs(st['loglik'] - rr['loglik']), 'aic': abs(st['aic'] - rr['aic']),
                  'tau2': _sdiff(st['tau2'], rr['tau2']), 'sigma2': _sdiff(st['sigma2'], rr['sigma2']),
                  'beta': _sdiff(st['mu'], rr['beta'][0])}
            ok = (dd['loglik'] < TOL['loglik'] and dd['aic'] < TOL['aic'] and dd['tau2'] < TOL['var']
                  and dd['sigma2'] < TOL['var'] and dd['beta'] < TOL['beta'])
            rows.append({'dataset': name, 'method': 'REML', 'structure': f'{sname} (stored by Step 7)',
                         'logLik (Py)': st['loglik'], 'logLik (R)': rr['loglik'],
                         'AIC (Py)': st['aic'], 'AIC (R)': rr['aic'],
                         'tau2 (Py)': st['tau2'], 'tau2 (R)': rr['tau2'],
                         'sigma2 (Py)': st['sigma2'], 'sigma2 (R)': rr['sigma2'],
                         'max |Δ|': max(dd.values()), 'status': 'PASS' if ok else 'FAIL'})
    return rows


def _user_dataset():
    """Your data as Step 7's nested comparison builds them: (spec, df, vcv, stored fits)."""
    u = validation_user_data()
    cfg = u['cfg']
    if 'OverallDataManager' in globals():
        used = OverallDataManager(cfg).prepare_data()     # the rows Step 7 uses
    else:
        used = cfg['analysis_data']
    spec = build_fit_spec(used, u['effect_col'], u['var_col'], vcv_dict=cfg.get('vcv_matrices') or {},
                          vcv_reference_df=cfg['analysis_data'], label='overall')
    vc = (cfg.get('overall_results') or {}).get('variance_structure_comparison') or {}
    stored = vc.get('fits') if isinstance(vc, dict) and 'error' not in vc else None
    return spec, u['df'], u['vcv'], stored


def _reference_datasets():
    out = []
    for seed in range(3):
        df_s, vcv_s = _simulated(seed)
        out.append((f'simulated #{seed}', build_fit_spec(df_s, 'yi', 'vi', vcv_dict=vcv_s, vcv_reference_df=df_s),
                    df_s, vcv_s, None))
    return out


def _v15_main():
    datasets, notices = [], []
    try:
        spec, df_u, vcv_u, stored = _user_dataset()
        datasets.append(('your data', spec, df_u, vcv_u, stored))
        if stored is None:
            ran7 = bool((validation_user_data()['cfg'].get('overall_results') or {}))
            notices.append("Step 7 did not store a nested comparison (it is computed only when the three-level "
                           "model is fitted), so only the refits are checked." if ran7 else
                           "Step 7 results not found in this session: the stored nested comparison is not "
                           "checked (run Step 7 to include it).")
    except ValidationPrerequisiteError:
        if not VALIDATION_INCLUDE_REFERENCE:
            raise
        notices.append("No effect sizes in this session: only the reference datasets are checked.")
    if VALIDATION_INCLUDE_REFERENCE:
        datasets += _reference_datasets()

    results = []
    for name, spec, df, vcv, stored in datasets:
        results += validate_dataset(name, spec, df, vcv, stored)

    res_df = pd.DataFrame(results)
    st = res_df['status'].astype(str)
    n_info = int((st == 'INFO').sum())
    n_fail = int((~st.str.startswith('PASS') & (st != 'INFO')).sum())
    n_graded = len(res_df) - n_info
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; "
        f"font-family:sans-serif;'><b>{icon} Unified engine vs metafor: "
        f"{n_graded - n_fail}/{n_graded} checks passed"
        f"{f' ({n_info} INFO)' if n_info else ''}.</b><br>"
        f"<small>Data: {', '.join(d[0] for d in datasets)}. Tolerances: {TOL} (τ², σ², β on the scale max(1, |value|)). Same rows and the same V "
        f"(your shared-control VCV) passed to both engines.</small></div>"))
    for n in notices:
        display(HTML(f"<div style='padding:8px; background:#e2e3e5; border-radius:4px; font-family:sans-serif;"
                     f" margin-top:6px;'>ℹ️ {n}</div>"))
    for name, spec, *_ in datasets:
        display(HTML(f"<p><b>{name}</b>: {spec.describe()}</p>" + spec.accounting.to_html()))
    display(res_df.style.format(precision=5, na_rep=''))
    return res_df, n_fail, notices


res_df, _n_fail15, _notes15 = _v15_main()
if 'record_validation' in globals():
    record_validation(15, "Unified Engine (fit_mv) vs metafor rma.mv", res_df, notes=' '.join(_notes15),
                      n_fail=_n_fail15)


In [ ]:
#@title 🧪 Validation 16: Egger's Test on your data (Python vs metafor + clubSandwich)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Checks the multilevel Egger test of the Publication Bias cell
# (PublicationBiasModel.run_eggers_test) against R. Both sides fit
#     yi = b0 + b1 * SE_i,   random = ~ 1 | study / effect,   REML,
# with the same sampling (co)variances, and the asymmetry test is the
# SLOPE b1 with CR2 standard errors and Satterthwaite df:
#     rma.mv(yi ~ sei, V = V, random = ~ 1 | cl/es, method = "REML")
#     clubSandwich::coef_test(fit, vcov = "CR2", cluster = cl, test = "Satterthwaite")
# Compared: b0, b1, model-based SE(b1), tau², sigma², CR2 SE(b1), df and p.
#
# Your data: the effect sizes of your analysis (Steps 2-6) with your
# sampling VCV, i.e. the shared-control covariances of Step 6 (the same V
# block by block in Python and in R, sliced by row identity). The Egger test
# always uses CR2 inference, as the Publication Bias cell does.
# Reference datasets (switch VALIDATION_INCLUDE_REFERENCE in the Validation
# R setup cell): metadat::dat.konstantopoulos2011 (11 districts / 56
# schools), two fixed simulated datasets (with single-effect studies, with
# and without small-study suppression) and one with shared controls (full
# sampling V, each study's effects correlated 0.5).
# Requires: Cells 1/1b, Steps 2-6, the Publication Bias cell and the
# Validation R setup cell.

import numpy as np
import pandas as pd
import rpy2.robjects as ro
from IPython.display import display, HTML


def _rvec(a):
    return ro.FloatVector(np.asarray(a, dtype=float).ravel())


TOL = {'beta': 1e-4, 'var': 1e-4, 'se_rel': 1e-3, 'df_rel': 1e-3, 'p': 1e-3}

def _pubbias_reference_datasets(need_year=False):
    """Fixed reference datasets of Validations 16-19 (only built when
    VALIDATION_INCLUDE_REFERENCE is on): [(name, DataFrame[id, yi, vi(, year)], vcv)].
    Defined identically in Validations 16, 17, 18 and 19, so each cell runs on its own."""
    out = []
    ro.r("suppressPackageStartupMessages(library(metadat))")
    kd = {c: np.asarray(ro.r(f"as.numeric(dat.konstantopoulos2011${c})")) for c in
          ('district', 'year', 'yi', 'vi')}
    out.append(('metadat: konstantopoulos2011', pd.DataFrame(
        {'id': kd['district'].astype(int).astype(str), 'yi': kd['yi'], 'vi': kd['vi'],
         'year': kd['year'].astype(int)}), {}))

    def sim(seed, n_studies, suppress):
        rng = np.random.default_rng(seed)
        rows = []
        for i in range(n_studies):
            u, year = rng.normal(0, 0.25), int(rng.integers(1990, 2021))
            for _ in range(int(rng.integers(1, 5))):        # 1–4 effects (singletons included)
                v = rng.uniform(0.005, 0.25)
                y = 0.3 + u + rng.normal(0, 0.15) + rng.normal(0, np.sqrt(v))
                rows.append(dict(id=f"S{i:02d}", yi=y, vi=v, year=year))
        df = pd.DataFrame(rows)
        if suppress:   # drop small, low-effect results -> funnel asymmetry
            df = df[~((df['yi'] < 0.3) & (df['vi'] > 0.06))]
        return df.reset_index(drop=True)

    out.append(('simulated: no bias', sim(101, 30, False), {}))
    out.append(('simulated: small-study suppression', sim(202, 45, True), {}))
    if need_year:
        return out

    # studies whose 1-4 effects share a control: sampling errors correlated
    # 0.5 within a study (full V), with small-study suppression; no year
    rng = np.random.default_rng(303)
    rows, vcv = [], {}
    for i in range(30):
        sid, k = f"C{i:02d}", int(rng.integers(1, 5))
        v = rng.uniform(0.005, 0.25, k)
        V = 0.5 * np.sqrt(np.outer(v, v)); np.fill_diagonal(V, v)
        y = 0.3 + rng.normal(0, 0.25) + rng.multivariate_normal(np.zeros(k), V + 0.02 * np.eye(k))
        keep = ~((y < 0.3) & (v > 0.06))                      # small-study suppression
        if keep.sum() == 0:
            continue
        vcv[sid] = V[np.ix_(keep, keep)]
        rows += [dict(id=sid, yi=y[j], vi=v[j]) for j in np.flatnonzero(keep)]
    out.append(('simulated: shared controls (full V)', pd.DataFrame(rows), vcv))
    return out


def _clean_rows(df):
    """Rows with a usable effect size and a positive variance (the
    analysis cells drop the others); '_vcv_row_pos' is kept, so the
    shared-control VCV is still sliced by row identity."""
    ok = np.isfinite(df['yi']) & np.isfinite(df['vi']) & (df['vi'] > 0)
    return df[ok]


def _V_to_r(name, V):
    ro.globalenv[name] = ro.r.matrix(ro.FloatVector(np.asarray(V, dtype=float).ravel(order='F')),
                                     nrow=V.shape[0])


def _show_notes(notes):
    if notes:
        display(HTML("<div style='padding:10px; background:#fff3cd; color:#856404; border-radius:4px; "
                     "font-family:sans-serif; margin-bottom:6px;'>ℹ️ " + "<br>ℹ️ ".join(notes) + "</div>"))


def _py_egger(df, vcv):
    m = PublicationBiasModel()
    m.analysis_config = {'vcv_matrices': vcv or {}}
    m.analysis_data = df.assign(se=np.sqrt(df['vi']))
    m.effect_col, m.var_col, m.se_col = 'yi', 'vi', 'se'
    res = m.run_eggers_test()
    est = res['estimates']
    return {'b0': res['intercept'], 'b1': res['slope'],
            'se_model': float(np.sqrt(np.diag(est['var_betas']))[1]),
            'tau2': float(est['tau_sq']), 'sigma2': float(est['sigma_sq']),
            'se_cr2': res['se'], 'df': res['df'], 'p': res['p_value'],
            'robust': res.get('used_robust'), 'sampling_v': est.get('sampling_v', 'diagonal')}   # None: older engine without the flag


def _r_egger(df, vcv):
    d, V = validation_block_V(df, vcv)          # same rows, same V as Python
    ro.globalenv['yi_v'] = _rvec(d['yi'])
    ro.globalenv['vi_v'] = _rvec(d['vi'])
    _V_to_r('V_m', V)
    ro.globalenv['cl_v'] = ro.StrVector(d['id'].astype(str).tolist())
    out = ro.r("""
    suppressPackageStartupMessages({library(metafor); library(clubSandwich)})
    tryCatch({
        d <- data.frame(yi = as.numeric(yi_v), vi = as.numeric(vi_v),
                        cl = factor(as.character(cl_v)))
        d$sei <- sqrt(d$vi); d$es <- factor(seq_len(nrow(d)))
        f <- rma.mv(yi ~ sei, V = V_m, random = ~ 1 | cl/es, data = d,
                    method = "REML", test = "t",
                    control = list(rel.tol = 1e-10, iter.max = 1000, eval.max = 2000))
        ct <- coef_test(f, vcov = "CR2", cluster = d$cl, test = "Satterthwaite")
        list(ok = TRUE, b = as.numeric(f$b), se = as.numeric(f$se),
             s2 = as.numeric(f$sigma2), se_cr2 = as.numeric(ct$SE[2]),
             df = as.numeric(ct$df_Satt[2]), p = as.numeric(ct$p_Satt[2]))
    }, error = function(e) list(ok = FALSE, msg = conditionMessage(e)))
    """)
    if not bool(out.rx2('ok')[0]):
        raise RuntimeError(str(out.rx2('msg')[0]))
    b, s2 = np.asarray(out.rx2('b')), np.asarray(out.rx2('s2'))
    return {'b0': b[0], 'b1': b[1], 'se_model': float(np.asarray(out.rx2('se'))[1]),
            'tau2': float(s2[0]), 'sigma2': float(s2[1]),
            'se_cr2': float(out.rx2('se_cr2')[0]), 'df': float(out.rx2('df')[0]),
            'p': float(out.rx2('p')[0])}


def _rel(a, b):
    return abs(a - b) / max(abs(b), 1e-12)


def _v8_main():
    """Main block, run in a function so its working variables do not leak
    into the notebook's shared namespace."""
    if 'PublicationBiasModel' not in globals():
        raise ValidationPrerequisiteError("PublicationBiasModel not found: run the Publication Bias "
                                          "(Egger's & Trim-Fill) cell first.")
    rows = []
    for name, df, vcv in validation_sources(_pubbias_reference_datasets):
        df = _clean_rows(df)
        try:
            py, rr = _py_egger(df, vcv), _r_egger(df, vcv)
        except Exception as e:
            rows.append({'dataset': name, 'status': f'ERROR: {e}'})
            continue
        d = {'b0': abs(py['b0'] - rr['b0']), 'b1': abs(py['b1'] - rr['b1']),
             'tau2': abs(py['tau2'] - rr['tau2']), 'sigma2': abs(py['sigma2'] - rr['sigma2']),
             'SE model (rel)': _rel(py['se_model'], rr['se_model']),
             'SE CR2 (rel)': _rel(py['se_cr2'], rr['se_cr2']),
             'df (rel)': _rel(py['df'], rr['df']), 'p': abs(py['p'] - rr['p'])}
        ok = (d['b0'] < TOL['beta'] and d['b1'] < TOL['beta'] and d['tau2'] < TOL['var']
              and d['sigma2'] < TOL['var'] and d['SE model (rel)'] < TOL['se_rel']
              and d['SE CR2 (rel)'] < TOL['se_rel'] and d['df (rel)'] < TOL['df_rel']
              and d['p'] < TOL['p'] and py['robust'] is not False)
        rows.append({'dataset': name, 'k': len(df), 'studies': df['id'].nunique(),
                     'sampling V': py['sampling_v'],
                     'b1 (Py)': py['b1'], 'b1 (R)': rr['b1'],
                     'SE CR2 (Py)': py['se_cr2'], 'SE CR2 (R)': rr['se_cr2'],
                     'df (Py)': py['df'], 'df (R)': rr['df'],
                     'p (Py)': py['p'], 'p (R)': rr['p'],
                     'max |Δ| est.': max(d['b0'], d['b1'], d['tau2'], d['sigma2']),
                     'max rel Δ SE/df': max(d['SE model (rel)'], d['SE CR2 (rel)'], d['df (rel)']),
                     'status': 'PASS' if ok else ('FAIL (CR2 not used)' if py['robust'] is False else 'FAIL')})

    res_df = pd.DataFrame(rows)
    n_fail = int((res_df['status'] != 'PASS').sum())
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
        f"<b>{icon} Egger's test vs metafor + clubSandwich: {len(res_df) - n_fail}/{len(res_df)} datasets passed.</b><br>"
        f"<small>Slope on SE, CR2 + Satterthwaite, same sampling V (shared controls) in Python and R. "
        f"Tolerances: {TOL}. "
        f"R: metafor {ro.r('as.character(packageVersion(\"metafor\"))')[0]}, "
        f"clubSandwich {ro.r('as.character(packageVersion(\"clubSandwich\"))')[0]}.</small></div>"))
    display(res_df.style.format(precision=5, na_rep=''))
    return res_df


res_df = _v8_main()
if res_df is not None and 'record_validation' in globals():
    record_validation(16, "Egger's Test", res_df)


In [ ]:
#@title 🧪 Validation 17: Trim-and-Fill on your data (Python vs metafor + clubSandwich)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Checks PublicationBiasModel.run_trimfill_analysis against R in two parts.
#
# 1. Trimming and filling (Duval & Tweedie L0 and R0, DerSimonian-Laird centring,
#    effect-size level, as implemented in Python):
#        trimfill(rma(yi, vi, method = "DL"), side = side, estimator = "L0" | "R0")
#    Compared for side = "left", "right" and "auto" (both programs choose the
#    side from the sign of the slope of the DL meta-regression yi ~ sqrt(vi)):
#    the side, k0 and the imputed effect sizes. (The Publication Bias cell
#    itself runs L0 with side "auto".)
#
# 2. Adjusted estimate: Python refits the filled data with the three-level
#    model (observed studies with their sampling V, full when they share a
#    control; each imputed effect as its own pseudo-study) and CR2 inference:
#        rma.mv(yi, V, random = ~ 1 | cl/es, method = "REML")
#        clubSandwich::coef_test(fit, vcov = "CR2", cluster = cl, test = "Satterthwaite")
#        clubSandwich::conf_int(fit, vcov = "CR2", level = 1 - alpha)
#    on exactly the same filled data. Compared: estimate, CR2 SE and CI.
#
# Your data: your effect sizes with your sampling VCV (shared-control
# covariances of Step 6, the same V in Python and R) and the CI level of
# Step 7 (alpha). Reference datasets (switch VALIDATION_INCLUDE_REFERENCE):
# as in Validation 16, at the 95% level.
# Requires: Cells 1/1b, Steps 2-6, the Publication Bias cell and the
# Validation R setup cell (this cell does not need Validation 16).

import numpy as np
import pandas as pd
import rpy2.robjects as ro
from scipy.linalg import block_diag
from IPython.display import display, HTML


def _rvec(a):
    return ro.FloatVector(np.asarray(a, dtype=float).ravel())


TOL_TF = {'filled_yi': 1e-6, 'estimate': 1e-4, 'se_rel': 1e-3, 'ci': 1e-3}
_DUMMY_ORIGINAL = {'three_level_results': {'pooled_effect': np.nan, 'se': np.nan,
                                           'ci_lower': np.nan, 'ci_upper': np.nan}}

def _pubbias_reference_datasets(need_year=False):
    """Fixed reference datasets of Validations 16-19 (only built when
    VALIDATION_INCLUDE_REFERENCE is on): [(name, DataFrame[id, yi, vi(, year)], vcv)].
    Defined identically in Validations 16, 17, 18 and 19, so each cell runs on its own."""
    out = []
    ro.r("suppressPackageStartupMessages(library(metadat))")
    kd = {c: np.asarray(ro.r(f"as.numeric(dat.konstantopoulos2011${c})")) for c in
          ('district', 'year', 'yi', 'vi')}
    out.append(('metadat: konstantopoulos2011', pd.DataFrame(
        {'id': kd['district'].astype(int).astype(str), 'yi': kd['yi'], 'vi': kd['vi'],
         'year': kd['year'].astype(int)}), {}))

    def sim(seed, n_studies, suppress):
        rng = np.random.default_rng(seed)
        rows = []
        for i in range(n_studies):
            u, year = rng.normal(0, 0.25), int(rng.integers(1990, 2021))
            for _ in range(int(rng.integers(1, 5))):        # 1–4 effects (singletons included)
                v = rng.uniform(0.005, 0.25)
                y = 0.3 + u + rng.normal(0, 0.15) + rng.normal(0, np.sqrt(v))
                rows.append(dict(id=f"S{i:02d}", yi=y, vi=v, year=year))
        df = pd.DataFrame(rows)
        if suppress:   # drop small, low-effect results -> funnel asymmetry
            df = df[~((df['yi'] < 0.3) & (df['vi'] > 0.06))]
        return df.reset_index(drop=True)

    out.append(('simulated: no bias', sim(101, 30, False), {}))
    out.append(('simulated: small-study suppression', sim(202, 45, True), {}))
    if need_year:
        return out

    # studies whose 1-4 effects share a control: sampling errors correlated
    # 0.5 within a study (full V), with small-study suppression; no year
    rng = np.random.default_rng(303)
    rows, vcv = [], {}
    for i in range(30):
        sid, k = f"C{i:02d}", int(rng.integers(1, 5))
        v = rng.uniform(0.005, 0.25, k)
        V = 0.5 * np.sqrt(np.outer(v, v)); np.fill_diagonal(V, v)
        y = 0.3 + rng.normal(0, 0.25) + rng.multivariate_normal(np.zeros(k), V + 0.02 * np.eye(k))
        keep = ~((y < 0.3) & (v > 0.06))                      # small-study suppression
        if keep.sum() == 0:
            continue
        vcv[sid] = V[np.ix_(keep, keep)]
        rows += [dict(id=sid, yi=y[j], vi=v[j]) for j in np.flatnonzero(keep)]
    out.append(('simulated: shared controls (full V)', pd.DataFrame(rows), vcv))
    return out


def _clean_rows(df):
    """Rows with a usable effect size and a positive variance (the
    analysis cells drop the others); '_vcv_row_pos' is kept, so the
    shared-control VCV is still sliced by row identity."""
    ok = np.isfinite(df['yi']) & np.isfinite(df['vi']) & (df['vi'] > 0)
    return df[ok]


def _V_to_r(name, V):
    ro.globalenv[name] = ro.r.matrix(ro.FloatVector(np.asarray(V, dtype=float).ravel(order='F')),
                                     nrow=V.shape[0])


def _show_notes(notes):
    if notes:
        display(HTML("<div style='padding:10px; background:#fff3cd; color:#856404; border-radius:4px; "
                     "font-family:sans-serif; margin-bottom:6px;'>ℹ️ " + "<br>ℹ️ ".join(notes) + "</div>"))


def _py_trimfill(df, side, estimator, vcv, alpha):
    m = PublicationBiasModel()
    m.analysis_config = dict(_DUMMY_ORIGINAL, vcv_matrices=vcv or {}, global_settings={'alpha': alpha})
    m.analysis_data = df.assign(se=np.sqrt(df['vi']))
    m.effect_col, m.var_col, m.se_col = 'yi', 'vi', 'se'
    return m.run_trimfill_analysis(estimator=estimator, side=side)


def _r_trimfill(df, side, estimator='L0'):
    ro.globalenv['yi_v'] = _rvec(df['yi'])
    ro.globalenv['vi_v'] = _rvec(df['vi'])
    side_arg = '' if side == 'auto' else f', side = "{side}"'
    out = ro.r(f"""
    suppressPackageStartupMessages(library(metafor))
    tryCatch({{
        f  <- rma(as.numeric(yi_v), as.numeric(vi_v), method = "DL")
        tf <- trimfill(f, estimator = "{estimator}"{side_arg})
        list(ok = TRUE, k0 = as.integer(tf$k0), side = as.character(tf$side),
             filled = sort(as.numeric(tf$yi[tf$fill])))
    }}, error = function(e) list(ok = FALSE, msg = conditionMessage(e)))
    """)
    if not bool(out.rx2('ok')[0]):
        raise RuntimeError(str(out.rx2('msg')[0]))
    return {'k0': int(out.rx2('k0')[0]), 'side': str(out.rx2('side')[0]),
            'filled': np.asarray(out.rx2('filled'), dtype=float)}


def _r_refit(df, py, vcv, alpha):
    """Three-level CR2 fit of the SAME filled data that Python analysed: the
    observed studies' sampling V (full with shared controls, sliced by row
    identity) plus a diagonal entry for every imputed study."""
    k0 = int(py['k0'])
    d, V_obs = validation_block_V(df, vcv)
    v_fill = np.asarray(py['vi_filled'], dtype=float)
    yi = np.concatenate([d['yi'].values, np.asarray(py['yi_filled'], dtype=float)])
    vi = np.concatenate([d['vi'].values, v_fill])
    cl = d['id'].astype(str).tolist() + [f"Imputed_TF_{i}" for i in range(k0)]
    ro.globalenv['yi_v'], ro.globalenv['vi_v'] = _rvec(yi), _rvec(vi)
    ro.globalenv['cl_v'] = ro.StrVector(cl)
    _V_to_r('V_m', block_diag(V_obs, np.diag(v_fill)) if k0 else V_obs)
    out = ro.r(f"""
    suppressPackageStartupMessages({{library(metafor); library(clubSandwich)}})
    tryCatch({{
        d <- data.frame(yi = as.numeric(yi_v), vi = as.numeric(vi_v),
                        cl = factor(as.character(cl_v)))
        d$es <- factor(seq_len(nrow(d)))
        f  <- rma.mv(yi, V_m, random = ~ 1 | cl/es, data = d, method = "REML",
                     control = list(rel.tol = 1e-10, iter.max = 1000, eval.max = 2000))
        ct <- coef_test(f, vcov = "CR2", cluster = d$cl, test = "Satterthwaite")
        ci <- conf_int(f, vcov = "CR2", cluster = d$cl, level = {1 - alpha!r})
        list(ok = TRUE, b = as.numeric(f$b), se = as.numeric(ct$SE[1]),
             lo = as.numeric(ci$CI_L[1]), hi = as.numeric(ci$CI_U[1]))
    }}, error = function(e) list(ok = FALSE, msg = conditionMessage(e)))
    """)
    if not bool(out.rx2('ok')[0]):
        raise RuntimeError(str(out.rx2('msg')[0]))
    return {k: float(out.rx2(k)[0]) for k in ('b', 'se', 'lo', 'hi')}


def _v9_main():
    """Main block, run in a function so its working variables do not leak
    into the notebook's shared namespace."""
    if 'PublicationBiasModel' not in globals():
        raise ValidationPrerequisiteError("PublicationBiasModel not found: run the Publication Bias "
                                          "(Egger's & Trim-Fill) cell first.")
    try:
        user_alpha = float(validation_user_data()['settings'].get('alpha', 0.05))
    except ValidationPrerequisiteError:
        user_alpha = 0.05
    rows = []
    for name, df, vcv in validation_sources(_pubbias_reference_datasets):
        df = _clean_rows(df)
        alpha = user_alpha if name == 'your data' else 0.05
        for estimator, side in [(e, s) for e in ('L0', 'R0') for s in ('left', 'right', 'auto')]:
            try:
                py = _py_trimfill(df, side, estimator, vcv, alpha)
                rr = _r_trimfill(df, side, estimator)
            except Exception as e:
                rows.append({'dataset': name, 'estimator': estimator, 'side': side, 'status': f'ERROR: {e}'})
                continue
            fy = np.sort(np.asarray(py['yi_filled'], dtype=float))
            d_fill = float(np.max(np.abs(fy - rr['filled']))) if py['k0'] == rr['k0'] and py['k0'] > 0 else 0.0
            ok = py['side'] == rr['side'] and py['k0'] == rr['k0'] and d_fill < TOL_TF['filled_yi']
            row = {'dataset': name, 'estimator': estimator,
                   'side': side if side != 'auto' else f"auto → Py {py['side']} / R {rr['side']}",
                   'CI level': f"{100 * (1 - alpha):g}%",
                   'k0 (Py)': py['k0'], 'k0 (R)': rr['k0'], 'max |Δ| filled yi': d_fill}
            try:
                ref = _r_refit(df, py, vcv, alpha)
                d_est = abs(py['pooled_filled'] - ref['b'])
                d_se = abs(py['se_filled'] - ref['se']) / ref['se']
                d_ci = max(abs(py['ci_lower_filled'] - ref['lo']), abs(py['ci_upper_filled'] - ref['hi']))
                ok = (ok and py.get('adjusted_model') == '3-Level' and d_est < TOL_TF['estimate']
                      and d_se < TOL_TF['se_rel'] and d_ci < TOL_TF['ci'])
                row.update({'adjusted (Py)': py['pooled_filled'], 'adjusted (R)': ref['b'],
                            'SE CR2 (Py)': py['se_filled'], 'SE CR2 (R)': ref['se'],
                            'max |Δ| estimate': d_est, 'max rel Δ SE': d_se, 'max |Δ| CI': d_ci})
            except Exception as e:
                ok = False
                row['note'] = f'refit: {e}'
            row['status'] = 'PASS' if ok else 'FAIL'
            rows.append(row)

    res_df = pd.DataFrame(rows)
    n_fail = int((res_df['status'] != 'PASS').sum())
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
        f"<b>{icon} Trim-and-fill vs metafor + clubSandwich: {len(res_df) - n_fail}/{len(res_df)} checks passed.</b><br>"
        f"<small>L0 and R0 estimators, DL centring, side left/right/auto; adjusted estimate = 3-level REML + CR2 "
        f"on the same filled data and the same sampling V (shared controls). Tolerances: {TOL_TF}.</small></div>"))
    display(res_df.style.format(precision=5, na_rep=''))
    return res_df


res_df = _v9_main()
if res_df is not None and 'record_validation' in globals():
    record_validation(17, "Trim-and-Fill", res_df)


In [ ]:
#@title 🧪 Validation 18: Cumulative Meta-Analysis on your data (Python vs metafor)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Checks CumulativeEngine.run_analysis (Cumulative Meta-Analysis cell)
# against R in three parts.
#
# 1. Aggregation to study level ('study' mode): inverse-variance mean per
#    study with v = 1 / sum(1/v_ij), i.e.
#        aggregate(escalc(measure = "GEN", yi, vi), cluster = id, struct = "ID")
#
# 2. Cumulative random-effects models ('study' and 'obs' modes), REML tau²,
#    Wald CI (z, or t with k - 1 df), studies added in the order Python used:
#        cumul(rma(yi, vi, method = "REML", test = "z" | "t", level = 1 - alpha))
#    Compared at every step k >= 2: estimate, SE, CI, tau² and the Q-based
#    I² (from the Q statistic cumul() returns), ascending and descending.
#    These two modes use the sampling variances only (no VCV), as in the cell.
#
# 3. Multilevel mode ('multilevel', the cell's default): studies added one
#    at a time; at each step R refits, on all effect sizes of the studies so
#    far and with their sampling V (shared-control covariances, sliced by row
#    identity, the same V Python uses),
#        rma.mv(yi, V, random = ~ 1 | study/es)   (rma() while no study has
#                                                   more than one effect size)
#        coef_test(vcov = "CR2", test = "Satterthwaite")   when CR2 is on
#    (CR2 used when its df >= 4, else model-based SE with studies - 1 df,
#    the rule CoMeta applies at early steps; CR2 off: model-based SE)
#    Compared at every step: estimate, SE, df (t only), CI, tau², sigma².
#
# Python also has to put the studies in a valid year order; tied years keep
# their order in the data (stable sort).
#
# Your data: your effect sizes, year column and sampling VCV, with the
# settings of Step 7 the cell uses: alpha (CI level), dist_type (z or t) and
# CR2 on/off; all three modes, ascending and descending. Without a 'year'
# column the validation is 'not run' (unless the reference datasets are on).
# Reference datasets (switch VALIDATION_INCLUDE_REFERENCE): the datasets of
# Validation 16 that have a year (konstantopoulos2011 and two simulated
# sets; 'study'/'obs' with both z and t, multilevel with t and CR2, 95%)
# and a simulated dataset with shared-control covariances and a year.
# Requires: Cells 1/1b, Steps 2-7, the Cumulative Meta-Analysis cell and the
# Validation R setup cell (this cell does not need Validation 16).

import numpy as np
import pandas as pd
import rpy2.robjects as ro
from IPython.display import display, HTML


def _rvec(a):
    return ro.FloatVector(np.asarray(a, dtype=float).ravel())


TOL_CUM = {'aggregate': 1e-10, 'estimate': 1e-6, 'se_rel': 1e-5, 'tau2': 1e-6, 'I2': 1e-4}
TOL_ML = {'estimate': 1e-5, 'se_rel': 1e-4, 'df_rel': 1e-3, 'var': 1e-5}
_REF_SETTINGS = {'alpha': 0.05, 'dist_type': 't', 'use_kh': True}

def _pubbias_reference_datasets(need_year=False):
    """Fixed reference datasets of Validations 16-19 (only built when
    VALIDATION_INCLUDE_REFERENCE is on): [(name, DataFrame[id, yi, vi(, year)], vcv)].
    Defined identically in Validations 16, 17, 18 and 19, so each cell runs on its own."""
    out = []
    ro.r("suppressPackageStartupMessages(library(metadat))")
    kd = {c: np.asarray(ro.r(f"as.numeric(dat.konstantopoulos2011${c})")) for c in
          ('district', 'year', 'yi', 'vi')}
    out.append(('metadat: konstantopoulos2011', pd.DataFrame(
        {'id': kd['district'].astype(int).astype(str), 'yi': kd['yi'], 'vi': kd['vi'],
         'year': kd['year'].astype(int)}), {}))

    def sim(seed, n_studies, suppress):
        rng = np.random.default_rng(seed)
        rows = []
        for i in range(n_studies):
            u, year = rng.normal(0, 0.25), int(rng.integers(1990, 2021))
            for _ in range(int(rng.integers(1, 5))):        # 1–4 effects (singletons included)
                v = rng.uniform(0.005, 0.25)
                y = 0.3 + u + rng.normal(0, 0.15) + rng.normal(0, np.sqrt(v))
                rows.append(dict(id=f"S{i:02d}", yi=y, vi=v, year=year))
        df = pd.DataFrame(rows)
        if suppress:   # drop small, low-effect results -> funnel asymmetry
            df = df[~((df['yi'] < 0.3) & (df['vi'] > 0.06))]
        return df.reset_index(drop=True)

    out.append(('simulated: no bias', sim(101, 30, False), {}))
    out.append(('simulated: small-study suppression', sim(202, 45, True), {}))
    if need_year:
        return out

    # studies whose 1-4 effects share a control: sampling errors correlated
    # 0.5 within a study (full V), with small-study suppression; no year
    rng = np.random.default_rng(303)
    rows, vcv = [], {}
    for i in range(30):
        sid, k = f"C{i:02d}", int(rng.integers(1, 5))
        v = rng.uniform(0.005, 0.25, k)
        V = 0.5 * np.sqrt(np.outer(v, v)); np.fill_diagonal(V, v)
        y = 0.3 + rng.normal(0, 0.25) + rng.multivariate_normal(np.zeros(k), V + 0.02 * np.eye(k))
        keep = ~((y < 0.3) & (v > 0.06))                      # small-study suppression
        if keep.sum() == 0:
            continue
        vcv[sid] = V[np.ix_(keep, keep)]
        rows += [dict(id=sid, yi=y[j], vi=v[j]) for j in np.flatnonzero(keep)]
    out.append(('simulated: shared controls (full V)', pd.DataFrame(rows), vcv))
    return out


def _clean_rows(df):
    """Rows with a usable effect size and a positive variance (the
    analysis cells drop the others); '_vcv_row_pos' is kept, so the
    shared-control VCV is still sliced by row identity."""
    ok = np.isfinite(df['yi']) & np.isfinite(df['vi']) & (df['vi'] > 0)
    return df[ok]


def _V_to_r(name, V):
    ro.globalenv[name] = ro.r.matrix(ro.FloatVector(np.asarray(V, dtype=float).ravel(order='F')),
                                     nrow=V.shape[0])


def _show_notes(notes):
    if notes:
        display(HTML("<div style='padding:10px; background:#fff3cd; color:#856404; border-radius:4px; "
                     "font-family:sans-serif; margin-bottom:6px;'>ℹ️ " + "<br>ℹ️ ".join(notes) + "</div>"))


def _shared_vcv_dataset(seed=18, n_studies=20):
    """Simulated nested data with shared-control covariances and a year."""
    rng = np.random.default_rng(seed)
    rows, vcv = [], {}
    for i in range(n_studies):
        k = int(rng.integers(1, 5))
        V = np.full((k, k), rng.uniform(.01, .04)) + np.diag(rng.uniform(.02, .1, k))
        y = 0.25 + rng.normal(0, .2) + rng.normal(0, .1, k) + rng.multivariate_normal(np.zeros(k), V)
        sid = f"S{i:02d}"
        vcv[sid] = V
        year = int(rng.integers(1995, 2021))
        rows += [dict(id=sid, yi=y[j], vi=V[j, j], year=year) for j in range(k)]
    return pd.DataFrame(rows), vcv


def _cum_reference():
    return (_pubbias_reference_datasets(need_year=True)
            + [('simulated: shared-control VCV',) + _shared_vcv_dataset()])


def _cfg_for(df, vcv, settings):
    return {'analysis_data': df, 'effect_col': 'yi', 'var_col': 'vi', 'vcv_matrices': vcv or {},
            'global_settings': dict(settings)}


def _py_run(cfg, aggregation, order):
    """run_analysis as the cell calls it, plus the prepared rows (same
    preparation and stable sort as run_analysis) so R gets the same sequence."""
    dm = CumulativeDataManager(cfg)
    res = CumulativeEngine(dm).run_analysis(sort_order=order, aggregation_method=aggregation)
    if res is None:
        raise RuntimeError("run_analysis returned None")
    prep = dm.prepare_data(aggregation)
    prep = prep.assign(id=prep['id'].astype(str), yi=prep[dm.effect_col], vi=prep[dm.var_col])
    return res.steps.reset_index(drop=True), prep


def _r_multilevel(prep, vcv, ids_in_order, settings):
    """R refit at each step k >= 2 on the rows of the first k studies, with
    their sampling V sliced by row identity."""
    alpha, use_cr2 = float(settings.get('alpha', 0.05)), bool(settings.get('use_kh', True))
    crit = (f"qt(1 - {alpha!r} / 2, dfi)" if settings.get('dist_type', 'norm') == 't'
            else f"qnorm(1 - {alpha!r} / 2)")
    out = []
    for k in range(2, len(ids_in_order) + 1):
        d, V = validation_block_V(prep[prep['id'].isin(ids_in_order[:k])], vcv)
        ro.globalenv['yi_v'] = _rvec(d['yi'])
        ro.globalenv['cl_v'] = ro.StrVector(d['id'].astype(str).tolist())
        _V_to_r('V_m', V)
        r = ro.r(f"""
        suppressPackageStartupMessages({{library(metafor); library(clubSandwich)}})
        cl <- factor(as.character(cl_v), levels = unique(as.character(cl_v)))
        es <- factor(seq_along(yi_v))
        ctl <- list(rel.tol = 1e-12, iter.max = 2000, eval.max = 4000)
        if (any(table(cl) > 1)) {{
            f <- tryCatch(rma.mv(as.numeric(yi_v), V_m, random = ~ 1 | cl/es, method = "REML", control = ctl),
                          error = function(e) tryCatch(
                              rma.mv(as.numeric(yi_v), V_m, random = ~ 1 | cl/es, method = "REML",
                                     control = list(optimizer = "optim", optmethod = "BFGS", reltol = 1e-12)),
                              error = function(e2) rma.mv(as.numeric(yi_v), V_m, random = ~ 1 | cl/es,
                                                          method = "REML")))
            s2 <- f$sigma2
        }} else {{
            f <- rma(as.numeric(yi_v), diag(V_m), method = "REML",
                     control = list(threshold = 1e-12, maxiter = 5000))
            s2 <- c(f$tau2, 0)
        }}
        # Same rule as CoMeta: CR2 (when on) only when its Satterthwaite
        # df >= 4, otherwise model-based SE with studies - 1 df
        se <- f$se; dfi <- nlevels(cl) - 1
        if ({'TRUE' if use_cr2 else 'FALSE'}) {{
            ct <- coef_test(f, vcov = "CR2", cluster = cl, test = "Satterthwaite")
            if (is.finite(ct$df_Satt) && ct$df_Satt >= 4) {{ se <- ct$SE; dfi <- ct$df_Satt }}
        }}
        h <- {crit} * se
        c(as.numeric(f$b), se, dfi, as.numeric(f$b) - h, as.numeric(f$b) + h, s2[1], s2[2])
        """)
        out.append(np.asarray(r, dtype=float))
    return pd.DataFrame(out, columns=['b', 'se', 'df', 'lo', 'hi', 'tau2', 'sigma2'])


def _r_aggregate(df):
    ro.globalenv['yi_v'], ro.globalenv['vi_v'] = _rvec(df['yi']), _rvec(df['vi'])
    ro.globalenv['cl_v'] = ro.StrVector(df['id'].astype(str).tolist())
    out = ro.r("""
    suppressPackageStartupMessages(library(metafor))
    d <- escalc(measure = "GEN", yi = as.numeric(yi_v), vi = as.numeric(vi_v),
                data = data.frame(cl = as.character(cl_v)))
    a <- aggregate(d, cluster = cl, struct = "ID")
    list(cl = as.character(a$cl), yi = as.numeric(a$yi), vi = as.numeric(a$vi))
    """)
    return pd.DataFrame({'id': list(out.rx2('cl')), 'yi_R': np.asarray(out.rx2('yi')),
                         'vi_R': np.asarray(out.rx2('vi'))})


def _r_cumul(ordered, dist, alpha):
    ro.globalenv['yi_v'], ro.globalenv['vi_v'] = _rvec(ordered['yi']), _rvec(ordered['vi'])
    test = 't' if dist == 't' else 'z'
    out = ro.r(f"""
    suppressPackageStartupMessages(library(metafor))
    tryCatch({{
        f  <- rma(as.numeric(yi_v), as.numeric(vi_v), method = "REML", test = "{test}",
                  level = {100 * (1 - alpha)!r}, control = list(threshold = 1e-12, maxiter = 5000))
        cm <- cumul(f)
        list(ok = TRUE, b = as.numeric(cm$estimate), se = as.numeric(cm$se),
             lo = as.numeric(cm$ci.lb), hi = as.numeric(cm$ci.ub),
             tau2 = as.numeric(cm$tau2), QE = as.numeric(cm$Q))
    }}, error = function(e) list(ok = FALSE, msg = conditionMessage(e)))
    """)
    if not bool(out.rx2('ok')[0]):
        raise RuntimeError(str(out.rx2('msg')[0]))
    r = pd.DataFrame({k: np.asarray(out.rx2(k), dtype=float) for k in ('b', 'se', 'lo', 'hi', 'tau2', 'QE')})
    k = np.arange(1, len(r) + 1)
    r['I2'] = np.where(r['QE'] > 0, np.maximum(0, (r['QE'] - (k - 1)) / r['QE'] * 100), 0.0)
    return r.iloc[1:].reset_index(drop=True)          # Python starts at k = 2


def _v10_main():
    """Main block, run in a function so its working variables do not leak
    into the notebook's shared namespace."""
    if 'CumulativeEngine' not in globals():
        raise ValidationPrerequisiteError("CumulativeEngine not found: run the Cumulative Meta-Analysis cell first.")
    notes, u = [], None
    try:
        u = validation_user_data(need_year=True)
    except ValidationPrerequisiteError as e:
        if not VALIDATION_INCLUDE_REFERENCE:
            raise
        notes.append(f"Your data were not checked: {e} Only the reference datasets were run.")
    sources = ([('your data', u['df'], u['vcv'])] if u is not None else [])
    if VALIDATION_INCLUDE_REFERENCE:
        sources += _cum_reference()

    rows = []
    for name, df, vcv in sources:
        is_user = u is not None and name == 'your data'
        if is_user:
            gs = u['settings']
            settings = {'alpha': float(gs.get('alpha', 0.05)), 'dist_type': gs.get('dist_type', 'norm'),
                        'use_kh': bool(gs.get('use_kh', True))}
            # the cell's code path: the engine on your ANALYSIS_CONFIG, with your
            # effect-size columns (u['df'] adds only yi/vi copies and the VCV row identity)
            base_cfg = dict(u['cfg'], analysis_data=df, global_settings=dict(gs, **settings))
            dists = [settings['dist_type']]
            n_no_year = int(pd.to_numeric(df['year'], errors='coerce').isna().sum())
            if n_no_year:
                notes.append(f"{n_no_year} of your effect sizes have no year: 'study'/'obs' modes drop them, "
                             f"the multilevel mode gives them their study's year (as the cell does).")
        else:
            settings = dict(_REF_SETTINGS)
            df = df.assign(id=df['id'].astype(str))
            vcv = {str(k): v for k, v in (vcv or {}).items()}
            if vcv:
                df = attach_vcv_row_pos(df, reference_df=df)
            base_cfg = _cfg_for(df, vcv, settings)
            dists = ['norm', 't']
        alpha = settings['alpha']
        label_set = (f"α = {alpha:g}, {'t' if settings['dist_type'] == 't' else 'z'}, "
                     f"CR2 {'on' if settings['use_kh'] else 'off'}")

        # --- 1. study-level aggregation ---
        try:
            prep = CumulativeDataManager(base_cfg).prepare_data('study')
            prep = prep.assign(id=prep['id'].astype(str))
            ecol = base_cfg['effect_col']; vcol = base_cfg['var_col']
            src = df.assign(year=pd.to_numeric(df['year'], errors='coerce')).dropna(subset=['year'])
            src = src[np.isfinite(src['yi']) & (src['vi'] > 0)]
            m = prep.merge(_r_aggregate(src), on='id')
            d_agg = float(max(np.max(np.abs(m[ecol] - m['yi_R'])), np.max(np.abs(m[vcol] - m['vi_R']))))
            ok = len(m) == len(prep) and d_agg < TOL_CUM['aggregate']
            rows.append({'dataset': name, 'check': 'aggregation to study level',
                         'steps': len(m), 'max |Δ|': d_agg, 'status': 'PASS' if ok else 'FAIL'})
        except Exception as e:
            rows.append({'dataset': name, 'check': 'aggregation to study level', 'status': f'ERROR: {e}'})

        # --- 2. cumulative models ('study' and 'obs') ---
        for aggregation in ('study', 'obs'):
            for order in ('ascending', 'descending'):
                for dist in dists:
                    label = f"cumulative ({aggregation}, {order}, {'t' if dist == 't' else 'z'}, α = {alpha:g})"
                    try:
                        steps, prep = _py_run(dict(base_cfg, global_settings=dict(
                            base_cfg['global_settings'], dist_type=dist)), aggregation, order)
                        ordered = prep.sort_values('year', ascending=(order == 'ascending'),
                                                   kind='mergesort').reset_index(drop=True)
                        rr = _r_cumul(ordered, dist, alpha)
                    except Exception as e:
                        rows.append({'dataset': name, 'check': label, 'status': f'ERROR: {e}'})
                        continue
                    yrs = ordered['year'].values
                    sorted_ok = bool(np.all(np.diff(yrs) >= 0) if order == 'ascending' else np.all(np.diff(yrs) <= 0))
                    n_tied = int(ordered['year'].duplicated(keep=False).sum())
                    d = {'estimate': np.max(np.abs(steps['pooled_effect'] - rr['b'])),
                         'ci': max(np.max(np.abs(steps['ci_lower'] - rr['lo'])),
                                   np.max(np.abs(steps['ci_upper'] - rr['hi']))),
                         'se_rel': np.max(np.abs(steps['se'] - rr['se']) / rr['se']),
                         'tau2': np.max(np.abs(steps['tau2'] - rr['tau2'])),
                         'I2': np.max(np.abs(steps['I2'] - rr['I2']))}
                    ok = (len(steps) == len(rr) and sorted_ok
                          and d['estimate'] < TOL_CUM['estimate'] and d['ci'] < TOL_CUM['estimate']
                          and d['se_rel'] < TOL_CUM['se_rel'] and d['tau2'] < TOL_CUM['tau2']
                          and d['I2'] < TOL_CUM['I2'])
                    rows.append({'dataset': name, 'check': label, 'steps': len(steps),
                                 'final μ (Py)': steps['pooled_effect'].iloc[-1], 'final μ (R)': rr['b'].iloc[-1],
                                 'final τ² (Py)': steps['tau2'].iloc[-1], 'final τ² (R)': rr['tau2'].iloc[-1],
                                 'max |Δ|': max(d['estimate'], d['ci'], d['tau2']),
                                 'max rel Δ SE': d['se_rel'], 'max |Δ| I² (%)': d['I2'],
                                 'tied years': n_tied,
                                 'status': ('PASS' if ok else 'FAIL') + ('' if sorted_ok else ' (not in year order)')})

        # --- 3. multilevel mode (the cell's default) ---
        for order in ('ascending', 'descending'):
            label = f"cumulative (multilevel, {order}, {label_set})"
            try:
                steps, prep = _py_run(base_cfg, 'multilevel', order)
                ids_in_order = (prep.drop_duplicates('id')
                                    .sort_values('study_year', ascending=(order == 'ascending'), kind='mergesort')
                                    ['id'].tolist())
                rr = _r_multilevel(prep, vcv, ids_in_order, settings)
            except Exception as e:
                rows.append({'dataset': name, 'check': label, 'status': f'ERROR: {e}'})
                continue
            same_order = steps['study_added'].astype(str).tolist() == ids_in_order[1:]
            d = {'estimate': np.max(np.abs(steps['pooled_effect'] - rr['b'])),
                 'ci': max(np.max(np.abs(steps['ci_lower'] - rr['lo'])),
                           np.max(np.abs(steps['ci_upper'] - rr['hi']))),
                 'se_rel': np.max(np.abs(steps['se'] - rr['se']) / rr['se']),
                 'df_rel': (np.max(np.abs(steps['df'] - rr['df']) / rr['df'])
                            if settings['dist_type'] == 't' else 0.0),
                 'var': max(np.max(np.abs(steps['tau2'] - rr['tau2'])),
                            np.max(np.abs(steps['sigma2'] - rr['sigma2'])))}
            n_cr2 = int((steps['inference'] == 'CR2').sum()) if 'inference' in steps.columns else np.nan
            ok = (len(steps) == len(rr) and same_order
                  and d['estimate'] < TOL_ML['estimate'] and d['ci'] < TOL_ML['estimate'] * 10
                  and d['se_rel'] < TOL_ML['se_rel'] and d['df_rel'] < TOL_ML['df_rel']
                  and d['var'] < TOL_ML['var'])
            rows.append({'dataset': name, 'check': label, 'steps': len(steps),
                         'steps with CR2': n_cr2,
                         'final μ (Py)': steps['pooled_effect'].iloc[-1], 'final μ (R)': rr['b'].iloc[-1],
                         'final τ² (Py)': steps['tau2'].iloc[-1], 'final τ² (R)': rr['tau2'].iloc[-1],
                         'max |Δ|': max(d['estimate'], d['ci'], d['var']),
                         'max rel Δ SE': d['se_rel'], 'max rel Δ df': d['df_rel'],
                         'status': ('PASS' if ok else 'FAIL') + ('' if same_order else ' (study order differs)')})

    _show_notes(notes)
    res_df = pd.DataFrame(rows)
    n_fail = int((res_df['status'] != 'PASS').sum())
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
        f"<b>{icon} Cumulative meta-analysis vs metafor: {len(res_df) - n_fail}/{len(res_df)} checks passed.</b><br>"
        f"<small>aggregate(struct = 'ID') and cumul(rma(method = 'REML')) on the same study order; "
        f"multilevel mode: rma.mv (same sampling V) + clubSandwich CR2 refitted at every step "
        f"(tolerances {TOL_ML}). Tolerances: {TOL_CUM}. Tied years keep data order (stable sort).</small></div>"))
    display(res_df.style.format(precision=6, na_rep=''))
    res_df.attrs['notes'] = ' '.join(notes)
    return res_df


res_df = _v10_main()
if res_df is not None and 'record_validation' in globals():
    record_validation(18, "Cumulative Meta-Analysis", res_df, notes=res_df.attrs.get('notes', ''))


In [ ]:
#@title 🧪 Validation 19: PET-PEESE on your data (Python vs metafor + clubSandwich)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Runs the PET-PEESE cell's engine (PETPEESEEngine.run_analysis) end to end
# and refits both models in R on the same rows and the same sampling V:
#     PET  : rma.mv(yi ~ sei, V = V, random = ~ 1 | cl/es, method = "REML")
#     PEESE: rma.mv(yi ~ vi,  V = V, random = ~ 1 | cl/es, method = "REML")
#     clubSandwich::coef_test(fit, vcov = "CR2", cluster = cl, test = "Satterthwaite")
# Compared: intercept (the effect at SE = 0 / variance = 0), slope, CR2 SE,
# p and CI (1 - alpha, Satterthwaite df) of each, and the conditional choice
# (Stanley & Doucouliagos, 2014): PEESE when the PET intercept is significant
# (two-tailed p < threshold) with the sign of the uncorrected estimate,
# otherwise PET.
#
# Your data: the engine runs on your ANALYSIS_CONFIG exactly as the PET-PEESE
# cell does: your effect sizes, your sampling VCV (shared-control
# covariances of Step 6; R gets the same V, sliced by row identity), the CI
# level of Step 7, the uncorrected estimate reported in Step 7 (its sign is
# the expected direction) and the p threshold of your last PET-PEESE run
# (0.10 if it has not been run). PET-PEESE always uses CR2 inference.
# Reference datasets (switch VALIDATION_INCLUDE_REFERENCE): as in
# Validation 16, at the 95% level, threshold 0.10, the uncorrected estimate
# taken from R's three-level fit with the same V.
# Requires: Cells 1/1b, Steps 2-7, the PET-PEESE cell and the Validation R
# setup cell (this cell does not need Validation 16).

import numpy as np
import pandas as pd
import rpy2.robjects as ro
from IPython.display import display, HTML


def _rvec(a):
    return ro.FloatVector(np.asarray(a, dtype=float).ravel())


TOL_PP = {'beta': 1e-4, 'se_rel': 1e-3, 'p': 1e-3, 'ci': 1e-3}
P_THRESHOLD = 0.10

def _pubbias_reference_datasets(need_year=False):
    """Fixed reference datasets of Validations 16-19 (only built when
    VALIDATION_INCLUDE_REFERENCE is on): [(name, DataFrame[id, yi, vi(, year)], vcv)].
    Defined identically in Validations 16, 17, 18 and 19, so each cell runs on its own."""
    out = []
    ro.r("suppressPackageStartupMessages(library(metadat))")
    kd = {c: np.asarray(ro.r(f"as.numeric(dat.konstantopoulos2011${c})")) for c in
          ('district', 'year', 'yi', 'vi')}
    out.append(('metadat: konstantopoulos2011', pd.DataFrame(
        {'id': kd['district'].astype(int).astype(str), 'yi': kd['yi'], 'vi': kd['vi'],
         'year': kd['year'].astype(int)}), {}))

    def sim(seed, n_studies, suppress):
        rng = np.random.default_rng(seed)
        rows = []
        for i in range(n_studies):
            u, year = rng.normal(0, 0.25), int(rng.integers(1990, 2021))
            for _ in range(int(rng.integers(1, 5))):        # 1–4 effects (singletons included)
                v = rng.uniform(0.005, 0.25)
                y = 0.3 + u + rng.normal(0, 0.15) + rng.normal(0, np.sqrt(v))
                rows.append(dict(id=f"S{i:02d}", yi=y, vi=v, year=year))
        df = pd.DataFrame(rows)
        if suppress:   # drop small, low-effect results -> funnel asymmetry
            df = df[~((df['yi'] < 0.3) & (df['vi'] > 0.06))]
        return df.reset_index(drop=True)

    out.append(('simulated: no bias', sim(101, 30, False), {}))
    out.append(('simulated: small-study suppression', sim(202, 45, True), {}))
    if need_year:
        return out

    # studies whose 1-4 effects share a control: sampling errors correlated
    # 0.5 within a study (full V), with small-study suppression; no year
    rng = np.random.default_rng(303)
    rows, vcv = [], {}
    for i in range(30):
        sid, k = f"C{i:02d}", int(rng.integers(1, 5))
        v = rng.uniform(0.005, 0.25, k)
        V = 0.5 * np.sqrt(np.outer(v, v)); np.fill_diagonal(V, v)
        y = 0.3 + rng.normal(0, 0.25) + rng.multivariate_normal(np.zeros(k), V + 0.02 * np.eye(k))
        keep = ~((y < 0.3) & (v > 0.06))                      # small-study suppression
        if keep.sum() == 0:
            continue
        vcv[sid] = V[np.ix_(keep, keep)]
        rows += [dict(id=sid, yi=y[j], vi=v[j]) for j in np.flatnonzero(keep)]
    out.append(('simulated: shared controls (full V)', pd.DataFrame(rows), vcv))
    return out


def _clean_rows(df):
    """Rows with a usable effect size and a positive variance (the
    analysis cells drop the others); '_vcv_row_pos' is kept, so the
    shared-control VCV is still sliced by row identity."""
    ok = np.isfinite(df['yi']) & np.isfinite(df['vi']) & (df['vi'] > 0)
    return df[ok]


def _V_to_r(name, V):
    ro.globalenv[name] = ro.r.matrix(ro.FloatVector(np.asarray(V, dtype=float).ravel(order='F')),
                                     nrow=V.shape[0])


def _show_notes(notes):
    if notes:
        display(HTML("<div style='padding:10px; background:#fff3cd; color:#856404; border-radius:4px; "
                     "font-family:sans-serif; margin-bottom:6px;'>ℹ️ " + "<br>ℹ️ ".join(notes) + "</div>"))


def _r_petpeese(df, vcv, alpha):
    """b0, b1, se0, se1, p0, p1, ci_lo0, ci_hi0 of PET and PEESE, and the
    uncorrected three-level estimate, all with the sampling V of `vcv`."""
    d, V = validation_block_V(df, vcv)
    ro.globalenv['yi_v'], ro.globalenv['vi_v'] = _rvec(d['yi']), _rvec(d['vi'])
    ro.globalenv['cl_v'] = ro.StrVector(d['id'].astype(str).tolist())
    _V_to_r('V_m', V)
    out = ro.r(f"""
    suppressPackageStartupMessages({{library(metafor); library(clubSandwich)}})
    tryCatch({{
        d <- data.frame(yi = as.numeric(yi_v), vi = as.numeric(vi_v), cl = factor(as.character(cl_v)))
        d$sei <- sqrt(d$vi); d$es <- factor(seq_len(nrow(d)))
        ctl <- list(rel.tol = 1e-10, iter.max = 1000, eval.max = 2000)
        fit <- function(f) {{
            m  <- rma.mv(f, V = V_m, random = ~ 1 | cl/es, data = d, method = "REML", control = ctl)
            ct <- coef_test(m, vcov = "CR2", cluster = d$cl, test = "Satterthwaite")
            h  <- qt(1 - {alpha!r} / 2, ct$df_Satt[1]) * ct$SE[1]
            c(as.numeric(m$b), ct$SE, ct$p_Satt, m$b[1] - h, m$b[1] + h)
        }}
        naive <- rma.mv(yi, V_m, random = ~ 1 | cl/es, data = d, method = "REML", control = ctl)
        list(ok = TRUE, pet = fit(yi ~ sei), peese = fit(yi ~ vi), naive = as.numeric(naive$b))
    }}, error = function(e) list(ok = FALSE, msg = conditionMessage(e)))
    """)
    if not bool(out.rx2('ok')[0]):
        raise RuntimeError(str(out.rx2('msg')[0]))
    return (np.asarray(out.rx2('pet')), np.asarray(out.rx2('peese')), float(out.rx2('naive')[0]))


def _py_petpeese_reference(df, vcv, naive_mu, p_threshold):
    cfg = {'analysis_data': df, 'effect_col': 'yi', 'var_col': 'vi', 'vcv_matrices': vcv or {},
           'overall_results': {'reported': {'model': '3-Level', 'mu': naive_mu, 'se': np.nan,
                                            'ci_lower': np.nan, 'ci_upper': np.nan}}}
    globals()['ANALYSIS_CONFIG'] = cfg          # the regression engine reads vcv_matrices from here
    return PETPEESEEngine(PETPEESEDataManager(cfg)).run_analysis(p_threshold=p_threshold)


def _v11_main():
    """Main block, run in a function so its working variables do not leak
    into the notebook's shared namespace."""
    if 'PETPEESEEngine' not in globals():
        raise ValidationPrerequisiteError("PETPEESEEngine not found: run the PET-PEESE cell first.")
    notes = []
    try:
        u = validation_user_data()
    except ValidationPrerequisiteError:
        if not VALIDATION_INCLUDE_REFERENCE:
            raise
        u = None
    _saved_cfg = globals().get('ANALYSIS_CONFIG')
    rows = []
    try:
        for name, df, vcv in validation_sources(_pubbias_reference_datasets):
            df = _clean_rows(df)       # the rows PETPEESEDataManager.prepare_data keeps
            is_user = u is not None and name == 'your data'
            try:
                if is_user:
                    alpha = float(u['settings'].get('alpha', 0.05))
                    p_thr = float(((u['cfg'].get('pet_peese_results') or {}).get('decision') or {})
                                  .get('p_threshold', P_THRESHOLD))
                    if not ((u['cfg'].get('overall_results') or {}).get('reported') or {}):
                        notes.append("Step 7 (Overall Meta-Analysis) results not found: the expected "
                                     "direction of the PET-PEESE choice cannot be taken from your analysis.")
                    pet_r, peese_r, naive_r = _r_petpeese(df, vcv, alpha)
                    # the PET-PEESE cell's code path: the engine on your ANALYSIS_CONFIG
                    res = PETPEESEEngine(PETPEESEDataManager(_saved_cfg)).run_analysis(p_threshold=p_thr)
                else:
                    alpha, p_thr = 0.05, P_THRESHOLD
                    pet_r, peese_r, naive_r = _r_petpeese(df, vcv, alpha)
                    res = _py_petpeese_reference(df, vcv, naive_r, p_thr)
                if res is None:
                    raise RuntimeError("PETPEESEEngine.run_analysis returned None")
            except Exception as e:
                rows.append({'dataset': name, 'status': f'ERROR: {e}'})
                continue
            finally:
                if _saved_cfg is not None:
                    globals()['ANALYSIS_CONFIG'] = _saved_cfg
            naive_used = res.naive.estimate if res.naive.estimate is not None else np.nan
            d = {
                'PET b0': abs(res.pet.intercept - pet_r[0]), 'PET b1': abs(res.pet.slope - pet_r[1]),
                'PEESE b0': abs(res.peese.intercept - peese_r[0]), 'PEESE b1': abs(res.peese.slope - peese_r[1]),
            }
            d_se = max(abs(res.pet.intercept_se - pet_r[2]) / pet_r[2], abs(res.peese.intercept_se - peese_r[2]) / peese_r[2],
                       abs(res.pet.slope_se - pet_r[3]) / pet_r[3], abs(res.peese.slope_se - peese_r[3]) / peese_r[3])
            d_p = max(abs(getattr(res.pet, 'intercept_p', np.nan) - pet_r[4]), abs(res.pet.slope_p - pet_r[5]),
                      abs(res.peese.slope_p - peese_r[5]))
            d_ci = max(abs(res.pet.intercept_ci[0] - pet_r[6]), abs(res.pet.intercept_ci[1] - pet_r[7]),
                       abs(res.peese.intercept_ci[0] - peese_r[6]), abs(res.peese.intercept_ci[1] - peese_r[7]))
            r_choice = 'PEESE' if (pet_r[4] < p_thr and np.sign(pet_r[0]) == np.sign(naive_used)) else 'PET'
            ok = (max(d.values()) < TOL_PP['beta'] and d_se < TOL_PP['se_rel'] and bool(d_p < TOL_PP['p'])
                  and bool(d_ci < TOL_PP['ci']) and res.decision.recommended_model == r_choice)
            rows.append({'dataset': name, 'k': res.n_obs, 'studies': res.n_studies,
                         'sampling V': res.pet.estimates.get('model_type', ''),
                         'threshold': p_thr, 'CI level': f"{100 * (1 - alpha):g}%",
                         'uncorrected (used)': naive_used, 'uncorrected (R 3-level)': naive_r,
                         'PET b0 (Py)': res.pet.intercept, 'PET b0 (R)': pet_r[0],
                         'PET b0 p (Py)': getattr(res.pet, 'intercept_p', np.nan), 'PET b0 p (R)': pet_r[4],
                         'PEESE b0 (Py)': res.peese.intercept, 'PEESE b0 (R)': peese_r[0],
                         'choice (Py)': res.decision.recommended_model, 'choice (R)': r_choice,
                         'max |Δ| coef': max(d.values()), 'max rel Δ SE': d_se, 'max |Δ| p': d_p,
                         'max |Δ| CI': d_ci, 'status': 'PASS' if ok else 'FAIL'})
    finally:
        if _saved_cfg is not None:
            globals()['ANALYSIS_CONFIG'] = _saved_cfg   # leave the user's analysis untouched

    _show_notes(notes)
    res_df = pd.DataFrame(rows)
    n_fail = int((res_df['status'] != 'PASS').sum())
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
        f"<b>{icon} PET-PEESE vs metafor + clubSandwich: {len(res_df) - n_fail}/{len(res_df)} datasets passed.</b><br>"
        f"<small>Intercepts at SE = 0 / variance = 0; CR2 + Satterthwaite; same sampling V (shared controls) in "
        f"Python and R; choice by the PET intercept (p &lt; threshold, expected sign). "
        f"Tolerances: {TOL_PP}.</small></div>"))
    display(res_df.style.format(precision=5, na_rep=''))
    res_df.attrs['notes'] = ' '.join(notes)
    return res_df


res_df = _v11_main()
if res_df is not None and 'record_validation' in globals():
    record_validation(19, "PET-PEESE", res_df, notes=res_df.attrs.get('notes', ''))


In [ ]:
#@title 🧪 Validation 20: Effect Sizes on your data (Python vs metafor::escalc)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# YOUR DATA (default). Recomputes every effect size Step 6 stored for you
# (ANALYSIS_CONFIG['analysis_data']) with metafor::escalc, from your raw
# columns and your settings (effect-size type, small-sample bias correction):
#     lnRR  -> "ROM", correct = <your setting>     Hedges' g -> "SMD"
#     Cohen's d -> "SMD", correct = FALSE          MD -> "MD"
#     lnCVR -> "CVR", correct = <your setting>
#     log OR -> "OR"   log RR -> "RR"   RD -> "RD"   logit p -> "PLO"
# and compares yi, vi and the 95% CI columns row by row.
#   * SDs: the SDs Step 6 actually used (sde_imputed / sdc_imputed), so
#     imputed SDs are reproduced as used; the number of rows with an imputed
#     (or replaced) SD is reported.
#   * 2x2 counts: escalc gets your ORIGINAL counts (Step 5 data) with its
#     default continuity correction (add = 1/2, to = "only0"), which checks
#     CoMeta's correction too; double-zero studies dropped for OR / RR are
#     not in your effect sizes (logged in Step 6). If the Step 5 data cannot
#     be matched, the stored (already corrected) counts are used with add = 0.
#   * Pre-calculated effect sizes: there is nothing to recompute (INFO).
#   * metafor < 4.8 has no second-order lnRR / lnCVR correction: such rows
#     are INFO (cannot be checked) rather than failed.
#
# REFERENCE DATASETS (VALIDATION_INCLUDE_REFERENCE = True): the notebook's
# own pipeline -- Data Cleaning (run_processing) and Effect Size Calculation
# (run_calculation) on a private configuration -- for every effect-size type
# (lnRR and lnCVR with the correction on and off) on metadat's
# dat.curtis1998, dat.normand1999, dat.bcg, dat.nielweise2007,
# dat.pritz1997 and small constructed sets with zero means and zero cells;
# for OR and RR, double-zero studies must be dropped and logged. Your
# ANALYSIS_CONFIG, data_filtered and Effect Size tabs are untouched.
# Requires: Steps 2-6 (your data) and the Validation R setup cell.

import numpy as np
import pandas as pd
import rpy2.robjects as ro
from IPython.display import display, HTML

TOL_ES = {'yi': 1e-10, 'vi': 1e-10}

_ESCALC = {   # label: (escalc call, data kind), or (CoMeta type, call, kind, bias correction)
    'lnRR (bias-corrected)': ('lnRR', 'measure = "ROM", correct = TRUE, m1i = xe, sd1i = sde, n1i = ne, m2i = xc, sd2i = sdc, n2i = nc', 'cont', True),
    'lnRR (no correction)':  ('lnRR', 'measure = "ROM", correct = FALSE, m1i = xe, sd1i = sde, n1i = ne, m2i = xc, sd2i = sdc, n2i = nc', 'cont', False),
    'hedges_g':   ('measure = "SMD", m1i = xe, sd1i = sde, n1i = ne, m2i = xc, sd2i = sdc, n2i = nc', 'cont'),
    'cohen_d':    ('measure = "SMD", correct = FALSE, m1i = xe, sd1i = sde, n1i = ne, m2i = xc, sd2i = sdc, n2i = nc', 'cont'),
    'MD':         ('measure = "MD", m1i = xe, sd1i = sde, n1i = ne, m2i = xc, sd2i = sdc, n2i = nc', 'cont'),
    'lnCVR (bias-corrected)': ('lnCVR', 'measure = "CVR", correct = TRUE, m1i = xe, sd1i = sde, n1i = ne, m2i = xc, sd2i = sdc, n2i = nc', 'cont', True),
    'lnCVR (no correction)':  ('lnCVR', 'measure = "CVR", correct = FALSE, m1i = xe, sd1i = sde, n1i = ne, m2i = xc, sd2i = sdc, n2i = nc', 'cont', False),
    'log_or':     ('measure = "OR", ai = events_e, bi = nonevents_e, ci = events_c, di = nonevents_c', 'bin'),
    'log_rr':     ('measure = "RR", ai = events_e, bi = nonevents_e, ci = events_c, di = nonevents_c', 'bin'),
    'risk_diff':  ('measure = "RD", ai = events_e, bi = nonevents_e, ci = events_c, di = nonevents_c', 'bin'),
    'proportion': ('measure = "PLO", xi = events_e, mi = nonevents_e', 'prop'),
}


def _r_df(name, mapping):
    ro.r("suppressPackageStartupMessages(library(metadat))")
    return pd.DataFrame({new: np.asarray(ro.r(f"as.numeric({name}${old})")) for new, old in mapping.items()})


def _es_datasets():
    cont = {'xe': 'm1i', 'sde': 'sd1i', 'ne': 'n1i', 'xc': 'm2i', 'sdc': 'sd2i', 'nc': 'n2i'}
    out = {'cont': [('dat.curtis1998', _r_df('dat.curtis1998', cont)),
                    ('dat.normand1999', _r_df('dat.normand1999', cont)),
                    ('constructed: zero mean, fractional n', pd.DataFrame(
                        {'xe': [0.0, 5.2, 7.1], 'sde': [1.0, 1.3, 2.2], 'ne': [10, 12.5, 8],
                         'xc': [4.0, 4.4, 6.0], 'sdc': [1.0, 1.1, 2.0], 'nc': [10, 9.5, 8]}))]}
    b = _r_df('dat.bcg', {'events_e': 'tpos', 'nonevents_e': 'tneg', 'events_c': 'cpos', 'nonevents_c': 'cneg'})
    nw = _r_df('dat.nielweise2007', {'events_e': 'ai', 'n1': 'n1i', 'events_c': 'ci', 'n2': 'n2i'})
    nw = pd.DataFrame({'events_e': nw.events_e, 'nonevents_e': nw.n1 - nw.events_e,
                       'events_c': nw.events_c, 'nonevents_c': nw.n2 - nw.events_c})
    zc = pd.DataFrame({'events_e': [0, 0, 3, 10], 'nonevents_e': [10, 10, 7, 0],
                       'events_c': [2, 0, 5, 4], 'nonevents_c': [8, 10, 5, 6]})
    out['bin'] = [('dat.bcg', b), ('dat.nielweise2007', nw), ('constructed: zero cells', zc)]
    pr = _r_df('dat.pritz1997', {'events_e': 'xi', 'n': 'ni'})
    pr = pd.DataFrame({'events_e': pr.events_e, 'nonevents_e': pr.n - pr.events_e})
    out['prop'] = [('dat.pritz1997', pr),
                   ('constructed: 0 and all events', pd.DataFrame({'events_e': [0, 4, 12], 'nonevents_e': [12, 8, 0]}))]
    return out


def _metafor_has_second_order_correction():
    """True when escalc applies the second-order ROM/CVR corrections (metafor >= 4.8)."""
    return bool(ro.r("""suppressPackageStartupMessages(library(metafor))
        a <- escalc("ROM", m1i = 5, sd1i = 3, n1i = 5, m2i = 4, sd2i = 2, n2i = 5, correct = TRUE)$yi
        b <- escalc("ROM", m1i = 5, sd1i = 3, n1i = 5, m2i = 4, sd2i = 2, n2i = 5, correct = FALSE)$yi
        abs(a - b) > 1e-12""")[0])


def _cometa_pipeline(raw, es_type, bias_correction=True):
    """Run Cells 5 + 6 on `raw` with a private configuration; returns
    (analysis_data, removed_records). Restores the user's state."""
    g = globals()
    saved = {k: g.get(k) for k in ('ANALYSIS_CONFIG', 'data_filtered', '_populate_tabs')}
    captured = {}
    cfg = {'effect_size_type': es_type, 'bias_correction': bias_correction,
           'sd_missing_strategy': 'drop', 'sd_zero_strategy': 'drop',
           'prefilter_col': 'None', 'prefilter_values': [],
           'es_config': {'effect_col': 'yi_py', 'var_col': 'vi_py', 'se_col': 'sei_py',
                         'ci_lower_col': 'lo_py', 'ci_upper_col': 'hi_py',
                         'effect_label': es_type, 'effect_label_short': es_type}}
    try:
        g['ANALYSIS_CONFIG'] = cfg
        g['_populate_tabs'] = lambda df, *a, **k: captured.update(removed=a[8])
        ok, res = run_processing(raw.assign(id=[f"row{i:03d}" for i in range(len(raw))]))
        if not ok:
            raise RuntimeError(res.get('error', 'run_processing failed') if isinstance(res, dict) else res)
        run_calculation()
        if 'analysis_data' not in cfg:
            raise RuntimeError("run_calculation did not produce analysis_data")
        removed = pd.concat(captured['removed']) if captured.get('removed') else pd.DataFrame(columns=['id', 'Reason'])
        return cfg['analysis_data'].copy(), removed
    finally:
        for k, v in saved.items():
            if v is None:
                g.pop(k, None)
            else:
                g[k] = v


def _escalc(raw, call):
    for c in raw.columns:
        ro.globalenv[c] = ro.FloatVector(raw[c].astype(float).values)
    args = ', '.join(f"{c} = as.numeric({c})" for c in raw.columns)
    out = ro.r(f"""suppressPackageStartupMessages(library(metafor))
        d <- data.frame({args})
        e <- suppressWarnings(escalc({call}, data = d))
        list(yi = as.numeric(e$yi), vi = as.numeric(e$vi))""")
    return np.asarray(out.rx2('yi')), np.asarray(out.rx2('vi'))


_USER_MEASURE = {   # CoMeta type: (escalc measure, data kind)
    'lnRR': ('ROM', 'cont'), 'hedges_g': ('SMD', 'cont'), 'cohen_d': ('SMD', 'cont'), 'MD': ('MD', 'cont'),
    'lnCVR': ('CVR', 'cont'), 'log_or': ('OR', 'bin'), 'log_rr': ('RR', 'bin'), 'risk_diff': ('RD', 'bin'),
    'proportion': ('PLO', 'prop'),
}
_BIN = ['events_e', 'nonevents_e', 'events_c', 'nonevents_c']


def _escalc_full(raw, call):
    """escalc yi, vi and the 95% CI (summary.escalc) for the columns of `raw`."""
    for c in raw.columns:
        ro.globalenv[c] = ro.FloatVector(raw[c].astype(float).values)
    args = ', '.join(f"{c} = as.numeric({c})" for c in raw.columns)
    out = ro.r(f"""suppressPackageStartupMessages(library(metafor))
        d <- data.frame({args})
        e <- suppressWarnings(escalc({call}, data = d))
        s <- summary(e)
        list(yi = as.numeric(e$yi), vi = as.numeric(e$vi), lo = as.numeric(s$ci.lb), hi = as.numeric(s$ci.ub))""")
    return {k: np.asarray(out.rx2(k), dtype=float) for k in ('yi', 'vi', 'lo', 'hi')}


def _original_counts(ad, cols):
    """The Step 5 counts of your rows (before Step 6's continuity correction),
    when data_filtered can be matched to analysis_data by row label; else None."""
    src = globals().get('data_filtered')
    if not isinstance(src, pd.DataFrame) or not set(cols).issubset(src.columns) or not ad.index.isin(src.index).all():
        return None
    orig = src.loc[ad.index, cols].apply(pd.to_numeric, errors='coerce').astype(float)
    stored = ad[cols].astype(float)
    if 'id' in src.columns and (src.loc[ad.index, 'id'].astype(str).values != ad['id'].astype(str).values).any():
        return None
    corrected = (orig == 0).any(axis=1)
    same = np.isclose(stored.values, orig.values).all(axis=1)
    plus = np.isclose(stored.values, orig.values + 0.5).all(axis=1)
    if not np.all(np.where(corrected, plus, same)):
        return None
    return orig


def _user_rows(u, new_mf):
    cfg, ad = u['cfg'], u['df']
    es = u['effect_size_type']
    if cfg.get('data_type') == 'pre_calculated' or es not in _USER_MEASURE:
        return [{'effect size': es, 'dataset': 'your data', 'status': 'INFO',
                 'notes': 'pre-calculated effect sizes: Step 6 did not compute them, so there is nothing '
                          'to recompute with escalc (turn on the reference datasets to check the formulas)'}], \
               ["Your effect sizes are pre-calculated: nothing to recompute."]
    measure, kind = _USER_MEASURE[es]
    bias = bool(cfg.get('bias_correction', True))
    label = es + ((' (bias-corrected)' if bias else ' (no correction)') if es in ('lnRR', 'lnCVR') else '')
    if not new_mf and es in ('lnRR', 'lnCVR') and (bias or es == 'lnCVR'):
        return [{'effect size': label, 'dataset': 'your data', 'status': 'INFO',
                 'notes': 'not checkable: this metafor predates the second-order correction (4.8); '
                          'install a current metafor'}], []
    rows, notices = [], []
    info = []
    if kind == 'cont':
        sde = 'sde_imputed' if 'sde_imputed' in ad.columns else 'sde'
        sdc = 'sdc_imputed' if 'sdc_imputed' in ad.columns else 'sdc'
        raw = pd.DataFrame({'m1i': ad['xe'], 'sd1i': ad[sde], 'n1i': ad['ne'],
                            'm2i': ad['xc'], 'sd2i': ad[sdc], 'n2i': ad['nc']}).astype(float)
        correct = {'lnRR': bias, 'lnCVR': bias, 'cohen_d': False}.get(es, True)
        call = (f'measure = "{measure}", m1i = m1i, sd1i = sd1i, n1i = n1i, m2i = m2i, sd2i = sd2i, n2i = n2i'
                + ('' if es == 'MD' else f', correct = {"TRUE" if correct else "FALSE"}'))
        if sde != 'sde':
            imp = int(((ad['sde'].isna() | ~np.isclose(ad['sde'].astype(float), ad[sde].astype(float)))
                       | (ad['sdc'].isna() | ~np.isclose(ad['sdc'].astype(float), ad[sdc].astype(float)))).sum())
            strategy = cfg.get('sd_missing_strategy', 'drop')
            info.append(f"{imp} row(s) with an imputed or replaced SD ({strategy}); compared with the SDs Step 6 used")
    elif kind == 'bin':
        orig = _original_counts(ad, _BIN)
        if orig is not None:
            raw = orig.rename(columns=dict(zip(_BIN, ['ai', 'bi', 'ci', 'di'])))
            call = f'measure = "{measure}", ai = ai, bi = bi, ci = ci, di = di'
            n_cc = int((orig == 0).any(axis=1).sum())
            info.append(f"{n_cc} row(s) with a zero cell: continuity correction +1/2 checked against escalc "
                        f"(add = 1/2, to = \"only0\")")
        else:
            raw = ad[_BIN].astype(float).rename(columns=dict(zip(_BIN, ['ai', 'bi', 'ci', 'di'])))
            call = f'measure = "{measure}", ai = ai, bi = bi, ci = ci, di = di, add = 0'
            info.append("Step 5 counts not matched: stored (already corrected) counts used with add = 0; "
                        "the continuity correction itself is not checked")
            notices.append(info[-1] + ".")
    else:
        raw = ad[['events_e', 'nonevents_e']].astype(float).rename(columns={'events_e': 'xi', 'nonevents_e': 'mi'})
        call = f'measure = "{measure}", xi = xi, mi = mi'
        n_cc = int(((raw['xi'] == 0) | (raw['mi'] == 0)).sum())
        info.append(f"{n_cc} row(s) with 0 or all events (+1/2 to both cells, as escalc)")
    r = _escalc_full(raw.reset_index(drop=True), call)
    yi, vi = ad['yi'].to_numpy(dtype=float), ad['vi'].to_numpy(dtype=float)
    na = ~(np.isfinite(r['yi']) & np.isfinite(r['vi']))
    d_yi = float(np.max(np.abs(yi[~na] - r['yi'][~na]))) if (~na).any() else 0.0
    d_vi = float(np.max(np.abs(vi[~na] - r['vi'][~na]))) if (~na).any() else 0.0
    ok = not na.any() and d_yi < TOL_ES['yi'] and d_vi < TOL_ES['vi']
    notes = [f"escalc({call.split(',')[0]}{', correct = ' + ('TRUE' if correct else 'FALSE') if kind == 'cont' and es != 'MD' else ''})"]
    if na.any():
        notes.append(f"{int(na.sum())} of your rows are undefined in escalc (NA) but KEPT by CoMeta")
    rows.append({'effect size': label, 'dataset': 'your data', 'rows compared': int((~na).sum()),
                 'max |Δ| yi': d_yi, 'max |Δ| vi': d_vi, 'notes': '; '.join(notes),
                 'status': 'PASS' if ok else 'FAIL'})
    # the stored 95% CI columns (normal approximation, as summary.escalc)
    lo_c, hi_c = cfg.get('ci_lower_col'), cfg.get('ci_upper_col')
    if lo_c in ad.columns and hi_c in ad.columns:
        d_ci = float(np.max(np.abs(np.r_[ad[lo_c].to_numpy(dtype=float)[~na] - r['lo'][~na],
                                         ad[hi_c].to_numpy(dtype=float)[~na] - r['hi'][~na]])))
        rows.append({'effect size': label, 'dataset': 'your data', 'rows compared': int((~na).sum()),
                     'max |Δ| CI': d_ci, 'notes': f'95% CI columns ({lo_c}, {hi_c}) vs summary(escalc)',
                     'status': 'PASS' if d_ci < 1e-8 else 'FAIL'})
    for t in info:
        rows.append({'effect size': label, 'dataset': 'your data', 'notes': t, 'status': 'INFO'})
    return rows, notices


def _reference_rows(new_mf):
    rows = []
    data = _es_datasets()
    for label, spec in _ESCALC.items():
        # entries are (call, kind) or (CoMeta type, call, kind, bias correction)
        es_type, call, kind, bias_corr = spec if len(spec) == 4 else (label, *spec, True)
        # Old metafor: no second-order correction, and correct = FALSE is
        # ignored for CVR, so these combinations have no R counterpart.
        uncheckable = (not new_mf and es_type in ('lnRR', 'lnCVR')
                       and (bias_corr or es_type == 'lnCVR'))
        for name, raw in data[kind]:
            if uncheckable:
                rows.append({'effect size': label, 'dataset': name,
                             'notes': 'not checkable: this metafor predates the second-order '
                                      'correction (4.8); install a current metafor',
                             'status': 'INFO'})
                continue
            try:
                py, removed = _cometa_pipeline(raw, es_type, bias_corr)
                r_yi, r_vi = _escalc(raw, call)
            except Exception as e:
                rows.append({'effect size': label, 'dataset': name, 'status': f'ERROR: {e}'})
                continue
            row_ids = [f"row{i:03d}" for i in range(len(raw))]
            r = pd.DataFrame({'id': row_ids, 'yi_R': r_yi, 'vi_R': r_vi})
            double_zero = ((raw['events_e'] == 0) & (raw['events_c'] == 0)).values if kind == 'bin' else np.zeros(len(raw), bool)
            notes = []
            if es_type in ('log_or', 'log_rr') and double_zero.any():
                dz_ids = set(np.array(row_ids)[double_zero])
                logged = set(removed['id']) if 'id' in removed.columns else set()
                if not dz_ids <= logged:
                    notes.append('double-zero study NOT logged')
                notes.append(f'{int(double_zero.sum())} double-zero dropped+logged (by design)')
                r = r[~np.isin(r['id'], list(dz_ids))]
            # rows R cannot compute (NA) must be excluded and logged by CoMeta too
            r_na = r[~np.isfinite(r['yi_R']) | ~np.isfinite(r['vi_R'])]
            r = r[np.isfinite(r['yi_R']) & np.isfinite(r['vi_R'])]
            m = r.merge(py[['id', 'yi_py', 'vi_py']], on='id', how='left')
            missing = int(m['yi_py'].isna().sum())
            extra_kept = int(py['id'].isin(r_na['id']).sum())
            d_yi = float(np.nanmax(np.abs(m['yi_py'] - m['yi_R']))) if len(m) else 0.0
            d_vi = float(np.nanmax(np.abs(m['vi_py'] - m['vi_R']))) if len(m) else 0.0
            ok = (missing == 0 and extra_kept == 0 and d_yi < TOL_ES['yi'] and d_vi < TOL_ES['vi']
                  and 'double-zero study NOT logged' not in notes)
            if len(r_na):
                notes.append(f'{len(r_na)} rows undefined in escalc (NA): '
                             + ('excluded by CoMeta too' if extra_kept == 0 else 'KEPT by CoMeta'))
            if missing:
                notes.append(f'{missing} rows computed by escalc but dropped by CoMeta')
            rows.append({'effect size': label, 'dataset': name, 'rows compared': len(m),
                         'max |Δ| yi': d_yi, 'max |Δ| vi': d_vi, 'notes': '; '.join(notes),
                         'status': 'PASS' if ok else 'FAIL'})
    return rows


def _v20_main():
    """Main block, run in a function so its working variables do not leak
    into the notebook's shared namespace."""
    if 'run_calculation' not in globals() or 'run_processing' not in globals():
        raise ValidationPrerequisiteError("Effect Size Calculation not found: run Steps 5-6 (Data Cleaning and "
                                          "Effect Size Calculation) first.")
    new_mf = _metafor_has_second_order_correction()
    rows, notices = [], []
    try:
        u = validation_user_data()
        ur, notices = _user_rows(u, new_mf)
        rows += ur
    except ValidationPrerequisiteError:
        if not VALIDATION_INCLUDE_REFERENCE:
            raise
        notices.append("No effect sizes in this session: only the reference datasets are checked.")
    if VALIDATION_INCLUDE_REFERENCE:
        rows += _reference_rows(new_mf)

    res_df = pd.DataFrame(rows)
    n_info = int((res_df['status'] == 'INFO').sum())
    n_graded = len(res_df) - n_info
    n_fail = int(((res_df['status'] != 'PASS') & (res_df['status'] != 'INFO')).sum())
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
        f"<b>{icon} Effect sizes vs metafor::escalc: {n_graded - n_fail}/{n_graded} checks passed"
        f"{f' ({n_info} INFO)' if n_info else ''}.</b><br>"
        f"<small>Your Step 6 effect sizes, row by row"
        f"{'; reference datasets through the notebook pipeline' if VALIDATION_INCLUDE_REFERENCE else '; reference datasets switched off'}"
        f". Tolerances: {TOL_ES}. metafor {ro.r('as.character(packageVersion(\"metafor\"))')[0]}.</small></div>"))
    for n in notices:
        display(HTML(f"<div style='padding:8px; background:#e2e3e5; border-radius:4px; font-family:sans-serif;"
                     f" margin-top:6px;'>ℹ️ {n}</div>"))
    display(res_df.style.format({c: '{:.1e}' for c in ('max |Δ| yi', 'max |Δ| vi', 'max |Δ| CI') if c in res_df.columns},
                                na_rep=''))
    return res_df, notices


res_df, _notes20 = _v20_main()
if res_df is not None and 'record_validation' in globals():
    record_validation(20, "Effect Sizes (escalc)", res_df, notes=' '.join(_notes20))


In [ ]:
#@title 🧪 Validation 21: Overall Model on your data — Heterogeneity, I², PI, Knapp-Hartung (Python vs metafor)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# YOUR DATA (default). Re-runs the Overall Meta-Analysis engine
# (OverallEngine.run_analysis) with the settings Step 7 used (CI level,
# t / z inference, CR2 / Knapp-Hartung on or off, tau² estimator of the
# 2-level model, model choice), checks that it reproduces the results Step 7
# stored, and refits everything in R on the same rows with the same sampling
# V (your shared-control VCV):
#   Fixed effect    rma(method = "FE")               -> mu, SE, CI, Q, p(Q), I² = (Q - df)/Q
#   2-level         rma(method = <your tau² method>) -> mu, tau², PI; with CR2 on,
#                   coef_test(CR2, cluster = study)  -> SE, Satterthwaite df, CI, p
#                   (CR2 off: model SE, df = studies - 1)
#   3-level         rma.mv(V, random = ~ 1 | study/effect) + coef_test(CR2) (when Step 7 fitted it)
#                                                    -> mu, tau², sigma², SE, df, CI, p,
#                                                       profile CIs (confint), PI, I² split
#   Knapp-Hartung   calculate_knapp_hartung_ci vs rma(test = "knha") and, with
#                   the max(1, Q/df) truncation, rma(test = "adhoc"), at your CI level
# Conventions (computed in R from metafor's own estimates, as CoMeta reports
# them): multilevel I² uses the Higgins-Thompson typical sampling variance;
# PI = mu ± crit · sqrt(SE_model² + tau² [+ sigma²]) with crit = t(studies - 1)
# for t inference and the normal quantile for z inference (2- and 3-level).
# The 2-level point estimate uses diagonal weights 1 / (v_i + tau²) while its
# CR2 working model uses the full V: with shared controls and CR2 on there is
# no exact clubSandwich counterpart, so its SE / df / CI are then shown as
# INFO (with rma.mv(V, sigma2 = tau²) + CR2 for comparison).
#
# REFERENCE DATASETS (VALIDATION_INCLUDE_REFERENCE = True), with the default
# settings (95%, t, CR2 on, REML): dat.konstantopoulos2011, dat.curtis1998
# (lnRR, effects nested in papers) and a simulated shared-control dataset
# (full V). Your state is restored afterwards.
# Requires: Steps 2-7, the Validation R setup cell.

import numpy as np
import pandas as pd
import rpy2.robjects as ro
from scipy.linalg import block_diag
from IPython.display import display, HTML

# 'var': variance components (REML optimiser tolerance), 'I2' in percentage points
TOL_OV = {'est': 1e-5, 'var': 1e-4, 'se_rel': 1e-4, 'df_rel': 1e-3, 'ci': 1e-4, 'Q': 1e-6, 'I2': 1e-3, 'prof': 5e-4, 'p': 1e-5}

def _sdiff(a, b):
    """|a - b| on the scale of the value: absolute below 1, relative above,
    so the same tolerance works for lnRR (~0.1) and mean differences (~50)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.nanmax(np.abs(a - b) / np.maximum(1.0, np.abs(b))))

_DEFAULT_SETTINGS = {'alpha': 0.05, 'dist_type': 't', 'use_kh': True, 'tau_method': 'REML',
                     'model_choice': THREE_LEVEL_CHOICE}


def _rv(a):
    return ro.FloatVector(np.asarray(a, dtype=float).ravel())


def _rm(M):
    M = np.asarray(M, dtype=float)
    return ro.r['matrix'](ro.FloatVector(M.ravel(order='F')), nrow=M.shape[0], ncol=M.shape[1])


def _ov_datasets():
    """Fixed reference datasets [(name, df, vcv)] (also used by Validation 22)."""
    ro.r("suppressPackageStartupMessages({library(metafor); library(metadat)})")
    k = {c: np.asarray(ro.r(f"as.numeric(dat.konstantopoulos2011${c})")) for c in ('district', 'yi', 'vi')}
    out = [('dat.konstantopoulos2011', pd.DataFrame({'id': k['district'].astype(int).astype(str), 'yi': k['yi'], 'vi': k['vi']}), {})]
    cu = ro.r("""e <- escalc("ROM", m1i = m1i, sd1i = sd1i, n1i = n1i, m2i = m2i, sd2i = sd2i, n2i = n2i,
                         data = dat.curtis1998); list(id = as.character(e$paper), yi = as.numeric(e$yi), vi = as.numeric(e$vi))""")
    out.append(('dat.curtis1998 (lnRR)', pd.DataFrame({'id': list(cu.rx2('id')), 'yi': np.asarray(cu.rx2('yi')),
                                                      'vi': np.asarray(cu.rx2('vi'))}), {}))
    rng = np.random.default_rng(48)
    rows, vcv = [], {}
    for i in range(20):
        kk = int(rng.integers(1, 5))
        V = np.full((kk, kk), rng.uniform(.01, .06)) + np.diag(rng.uniform(.02, .1, kk))
        y = 0.3 + rng.normal(0, .2) + rng.normal(0, .15, kk) + rng.multivariate_normal(np.zeros(kk), V)
        vcv[f"S{i:02d}"] = V
        rows += [dict(id=f"S{i:02d}", yi=y[j], vi=V[j, j]) for j in range(kk)]
    out.append(('simulated: shared controls (full V)', pd.DataFrame(rows), vcv))
    return out


def _py_overall_reference(df, vcv, s):
    """The engine on a reference dataset, on a private configuration."""
    g = globals()
    saved = g.get('ANALYSIS_CONFIG')
    cfg = {'analysis_data': df, 'effect_col': 'yi', 'var_col': 'vi', 'vcv_matrices': vcv,
           'es_config': {'effect_label': 'es', 'effect_label_short': 'es'}, 'global_settings': {'alpha': s['alpha']}}
    try:
        g['ANALYSIS_CONFIG'] = cfg
        dm = OverallDataManager(cfg)
        res = OverallEngine(dm).run_analysis(**s)
        used = dm.prepare_data()
        return res, used
    finally:
        if saved is not None:
            g['ANALYSIS_CONFIG'] = saved


def _r_overall(d, V, s, three):
    """FE, 2-level, 3-level and Knapp-Hartung refits in R on rows d (id, yi, vi) with sampling V."""
    shared = bool(np.any(np.abs(V - np.diag(np.diag(V))) > 0))
    ro.globalenv['yi_v'], ro.globalenv['vi_v'] = _rv(d['yi']), _rv(d['vi'])
    ro.globalenv['cl_v'], ro.globalenv['V_m'] = ro.StrVector(d['id'].astype(str).tolist()), _rm(V)
    out = ro.r(f"""
    suppressPackageStartupMessages({{library(metafor); library(clubSandwich)}})
    tryCatch({{
        a <- {s['alpha']}; dist <- "{s['dist_type']}"; kh <- {'TRUE' if s['use_kh'] else 'FALSE'}
        tm <- "{s['tau_method']}"; three <- {'TRUE' if three else 'FALSE'}; shared <- {'TRUE' if shared else 'FALSE'}
        d <- data.frame(yi = as.numeric(yi_v), vi = as.numeric(vi_v), cl = factor(as.character(cl_v),
                        levels = unique(as.character(cl_v))))
        d$es <- factor(seq_len(nrow(d))); m <- nlevels(d$cl); k <- nrow(d); q <- 1 - a / 2
        crit <- function(df) if (dist == "t") qt(q, max(1, df)) else qnorm(q)
        pv <- function(z, df) if (dist == "t") 2 * pt(-abs(z), max(1, df)) else 2 * pnorm(-abs(z))
        fe <- rma(yi, vi, data = d, method = "FE")
        fe_o <- c(fe$b, fe$se, fe$b + c(-1, 1) * crit(k - 1) * fe$se, fe$QE, fe$QEp,
                  max(0, (fe$QE - (k - 1)) / fe$QE) * 100)
        f2 <- rma(yi, vi, data = d, method = tm, control = list(threshold = 1e-12, maxiter = 5000))
        if (kh) {{ c2 <- coef_test(f2, vcov = "CR2", cluster = d$cl, test = "Satterthwaite")
                   se2 <- c2$SE; df2 <- c2$df_Satt }} else {{ se2 <- f2$se; df2 <- m - 1 }}
        pic2 <- if (dist == "t") qt(q, max(1, m - 1)) else qnorm(q)
        two <- c(f2$b, f2$tau2, se2, df2, f2$b + c(-1, 1) * crit(df2) * se2, pv(f2$b / se2, df2),
                 f2$b + c(-1, 1) * pic2 * sqrt(f2$se^2 + f2$tau2))
        alt2 <- c(NA, NA, NA)
        if (shared && kh) {{
            g2 <- rma.mv(yi, V_m, random = ~ 1 | es, sigma2 = f2$tau2, data = d)
            cg <- coef_test(g2, vcov = "CR2", cluster = d$cl, test = "Satterthwaite")
            alt2 <- c(g2$b, cg$SE, cg$df_Satt)
        }}
        th <- NA
        if (three) {{
            f3 <- rma.mv(yi, V_m, random = ~ 1 | cl/es, data = d, method = "REML",
                         control = list(rel.tol = 1e-10, iter.max = 1000, eval.max = 2000))
            if (kh) {{ c3 <- coef_test(f3, vcov = "CR2", cluster = d$cl, test = "Satterthwaite")
                       se3 <- c3$SE; df3 <- c3$df_Satt }} else {{ se3 <- f3$se; df3 <- m - 1 }}
            # profile CIs on a default-control fit: strict optimiser settings are
            # inherited by confint()'s profile refits and make them fail
            f3d <- rma.mv(yi, V_m, random = ~ 1 | cl/es, data = d, method = "REML")
            p1 <- confint(f3d, sigma2 = 1, level = 1 - a); p2 <- confint(f3d, sigma2 = 2, level = 1 - a)
            w <- 1 / d$vi; vt <- (k - 1) * sum(w) / (sum(w)^2 - sum(w^2))
            th <- c(f3$b, f3$sigma2, se3, df3, f3$b + c(-1, 1) * crit(df3) * se3,
                    p1$random[1, 2:3], p2$random[1, 2:3],
                    f3$b + c(-1, 1) * pic2 * sqrt(f3$se^2 + sum(f3$sigma2)),
                    100 * f3$sigma2 / (sum(f3$sigma2) + vt), pv(f3$b / se3, df3))
        }}
        fk <- rma(yi, vi, data = d, method = tm, test = "knha", level = 100 * (1 - a),
                  control = list(threshold = 1e-12, maxiter = 5000))
        fa <- rma(yi, vi, data = d, method = tm, test = "adhoc", level = 100 * (1 - a),
                  control = list(threshold = 1e-12, maxiter = 5000))
        list(ok = TRUE, fe = fe_o, two = two, alt2 = alt2, three = th,
             kh = c(fk$b, fk$tau2, fk$ci.lb, fk$ci.ub, fa$ci.lb, fa$ci.ub))
    }}, error = function(e) list(ok = FALSE, msg = conditionMessage(e)))
    """)
    if not bool(out.rx2('ok')[0]):
        raise RuntimeError(str(out.rx2('msg')[0]))
    res = {k: np.asarray(out.rx2(k), dtype=float) for k in ('fe', 'two', 'alt2', 'three', 'kh')}
    res['shared'] = shared
    return res


def _rel(a, b):
    return abs(a - b) / max(abs(b), 1e-12)


def _compare(name, py, d, rr, s):
    """Check rows for one dataset: engine result `py` vs R refits `rr`."""
    rows = []
    fe = rr['fe']
    dd = {'mu': _sdiff(py.mu_fixed, fe[0]), 'se': _rel(py.se_fixed, fe[1]),
          'ci': _sdiff([py.ci_lower_fixed, py.ci_upper_fixed], [fe[2], fe[3]]),
          'Q': abs(py.Q - fe[4]), 'pQ': abs(py.p_Q - fe[5]), 'I2': abs(py.I2 - fe[6])}
    ok = (dd['mu'] < TOL_OV['est'] and dd['se'] < TOL_OV['se_rel'] and dd['ci'] < TOL_OV['ci']
          and dd['Q'] < TOL_OV['Q'] and dd['pQ'] < TOL_OV['Q'] and dd['I2'] < TOL_OV['I2'])
    rows.append({'dataset': name, 'check': 'fixed effect: μ, SE, CI, Q, p(Q), I²', 'Py': f"Q={py.Q:.4f}, I²={py.I2:.2f}%",
                 'R': f"Q={fe[4]:.4f}, I²={fe[6]:.2f}%", 'max |Δ|': max(dd.values()), 'status': 'PASS' if ok else 'FAIL'})
    r2 = rr['two']
    tm = s['tau_method']
    dd = {'mu': _sdiff(py.mu_random, r2[0]), 'tau2': _sdiff(py.tau_squared, r2[1]),
          'pi': _sdiff([py.pi_lower_random, py.pi_upper_random], [r2[7], r2[8]])}
    ok = dd['mu'] < TOL_OV['est'] and dd['tau2'] < TOL_OV['var'] and dd['pi'] < TOL_OV['ci']
    rows.append({'dataset': name, 'check': f'2-level ({tm}): μ, τ², prediction interval',
                 'Py': f"μ={py.mu_random:.5f}, τ²={py.tau_squared:.5f}", 'R': f"μ={r2[0]:.5f}, τ²={r2[1]:.5f}",
                 'max |Δ|': max(dd.values()), 'status': 'PASS' if ok else 'FAIL'})
    inf = 'CR2 SE / Satterthwaite df' if s['use_kh'] else 'model SE, df = studies − 1'
    dd = {'se': _rel(py.se_random, r2[2]), 'df': _rel(py.df_infer, r2[3]),
          'ci': _sdiff([py.ci_lower_random, py.ci_upper_random], [r2[4], r2[5]]),
          'p': abs(py.p_value_random - r2[6])}
    ok = (dd['se'] < TOL_OV['se_rel'] and dd['df'] < TOL_OV['df_rel'] and dd['ci'] < TOL_OV['ci']
          and dd['p'] < TOL_OV['p'] and (py.used_robust == bool(s['use_kh'])))
    if rr['shared'] and s['use_kh']:
        a2 = rr['alt2']
        rows.append({'dataset': name, 'check': f'2-level: {inf}, CI, p (shared-control V)',
                     'Py': f"SE={py.se_random:.5f}, df={py.df_infer:.3f}",
                     'R': f"rma CR2: SE={r2[2]:.5f}, df={r2[3]:.3f}; rma.mv(V, σ²=τ²) CR2: μ={a2[0]:.5f}, "
                          f"SE={a2[1]:.5f}, df={a2[2]:.3f}",
                     'max |Δ|': max(dd.values()), 'status': 'INFO',
                     'note': 'diagonal-weight estimate with a full-V CR2 working model: no exact R counterpart'})
    else:
        rows.append({'dataset': name, 'check': f'2-level: {inf}, CI, p',
                     'Py': f"SE={py.se_random:.5f}, df={py.df_infer:.3f}", 'R': f"SE={r2[2]:.5f}, df={r2[3]:.3f}",
                     'max |Δ|': max(dd.values()), 'status': 'PASS' if ok else 'FAIL'})
    r3 = rr['three']
    if py.has_3level and r3.size > 1:
        dd = {'mu': _sdiff(py.mu_3level, r3[0]), 'tau2': _sdiff(py.tau_squared_3level, r3[1]),
              'sigma2': _sdiff(py.sigma_squared_3level, r3[2]), 'se': _rel(py.se_3level, r3[3]),
              'df': _rel(py.df_infer_3level, r3[4]),
              'ci': _sdiff([py.ci_lower_3level, py.ci_upper_3level], [r3[5], r3[6]]),
              'p': abs(py.p_value_3level - r3[15])}
        ok = all(v < TOL_OV[k] for k, v in (('est', dd['mu']), ('var', dd['tau2']), ('var', dd['sigma2']),
                                             ('se_rel', dd['se']), ('df_rel', dd['df']), ('ci', dd['ci']),
                                             ('p', dd['p']))) \
            and py.used_robust_3level == bool(s['use_kh'])
        rows.append({'dataset': name, 'check': f'3-level: μ, τ², σ², {inf}, CI, p',
                     'Py': f"μ={py.mu_3level:.5f}, τ²={py.tau_squared_3level:.5f}, σ²={py.sigma_squared_3level:.5f}",
                     'R': f"μ={r3[0]:.5f}, τ²={r3[1]:.5f}, σ²={r3[2]:.5f}", 'max |Δ|': max(dd.values()),
                     'status': 'PASS' if ok else 'FAIL'})
        prof = [py.tau2_ci_lower_3level, py.tau2_ci_upper_3level, py.sigma2_ci_lower_3level, py.sigma2_ci_upper_3level]
        dp = _sdiff(prof, r3[7:11])
        rows.append({'dataset': name, 'check': f'3-level: {100 * (1 - s["alpha"]):g}% profile CIs of τ² and σ²',
                     'Py': ', '.join(f"{v:.4f}" for v in prof), 'R': ', '.join(f"{v:.4f}" for v in r3[7:11]),
                     'max |Δ|': dp, 'status': 'PASS' if dp < TOL_OV['prof'] else 'FAIL'})
        dpi = _sdiff([py.pi_lower_3level, py.pi_upper_3level], [r3[11], r3[12]])
        di2 = max(abs(py.icc_l3 - r3[13]), abs(py.icc_l2 - r3[14]))
        rows.append({'dataset': name, 'check': '3-level: prediction interval, I² split',
                     'Py': f"PI=({py.pi_lower_3level:.4f}, {py.pi_upper_3level:.4f}), I²={py.icc_l3:.2f}/{py.icc_l2:.2f}%",
                     'R': f"PI=({r3[11]:.4f}, {r3[12]:.4f}), I²={r3[13]:.2f}/{r3[14]:.2f}%", 'max |Δ|': max(dpi, di2),
                     'status': 'PASS' if (dpi < TOL_OV['ci'] and di2 < TOL_OV['I2']) else 'FAIL'})
    else:
        rows.append({'dataset': name, 'check': '3-level model', 'status': 'INFO',
                     'note': 'not fitted by Step 7 (needs at least two studies with several effect sizes)'})
    kh_r = rr['kh']
    out = []
    for modified, lo, hi in ((False, kh_r[2], kh_r[3]), (True, kh_r[4], kh_r[5])):
        kh = calculate_knapp_hartung_ci(d['yi'].to_numpy(dtype=float), d['vi'].to_numpy(dtype=float),
                                        kh_r[1], kh_r[0], alpha=s['alpha'], modified=modified)
        out.append(_sdiff([kh['ci_lower'], kh['ci_upper']], [lo, hi]))
    rows.append({'dataset': name, 'check': f'Knapp-Hartung CI ({tm}; knha / adhoc)', 'Py': '', 'R': '',
                 'max |Δ|': max(out), 'status': 'PASS' if max(out) < TOL_OV['ci'] else 'FAIL'})
    return rows


def _stored_check(py, cfg):
    """The re-run reproduces the results Step 7 stored (what your report shows)."""
    st = cfg.get('overall_results') or {}
    rep = st.get('reported') or {}
    use3 = rep.get('model') == '3-Level'
    pairs = [(st.get('pooled_effect_fixed'), py.mu_fixed), (st.get('pooled_effect_random'), py.mu_random),
             (st.get('tau_squared'), py.tau_squared), (st.get('Qt'), py.Q), (st.get('I_squared'), py.I2),
             (rep.get('mu'), py.mu_3level if use3 else py.mu_random),
             (rep.get('se'), py.se_3level if use3 else py.se_random),
             (rep.get('ci_lower'), py.ci_lower_3level if use3 else py.ci_lower_random),
             (rep.get('ci_upper'), py.ci_upper_3level if use3 else py.ci_upper_random),
             (rep.get('pi_lower'), py.pi_lower_3level if use3 else py.pi_lower_random),
             (rep.get('pi_upper'), py.pi_upper_3level if use3 else py.pi_upper_random)]
    diffs = [abs(float(a) - float(b)) for a, b in pairs if a is not None and b is not None]
    dmax = max(diffs) if diffs else np.nan
    same = bool(diffs) and dmax < 1e-10 and rep.get('model') == ("3-Level" if (py.best_model == "3-Level" and py.has_3level) else "2-Level")
    return {'dataset': 'your data', 'check': f"re-run reproduces your Step 7 results (reported: {rep.get('model', '?')} model)",
            'Py': '', 'R': '', 'max |Δ|': dmax, 'status': 'PASS' if same else 'INFO',
            'note': '' if same else 'Step 7 results differ from a re-run with the current code and data: re-run Step 7'}


def _user_rows():
    u = validation_user_data()
    cfg = u['cfg']
    if not cfg.get('overall_results'):
        raise ValidationPrerequisiteError("Overall results not found: run Step 7 (Overall Meta-Analysis) first.")
    s = {**_DEFAULT_SETTINGS, **{k: v for k, v in (u['settings'] or {}).items() if k in _DEFAULT_SETTINGS}}
    dm = OverallDataManager(cfg)
    py = OverallEngine(dm).run_analysis(**s)
    if py is None:
        raise RuntimeError("OverallEngine.run_analysis returned None on your data")
    used = dm.prepare_data()                       # the rows the engine used (VCV row identity attached)
    used['id'] = used['id'].astype(str)
    used['yi'], used['vi'] = used[u['effect_col']].astype(float), used[u['var_col']].astype(float)
    d, V = validation_block_V(used, u['vcv'])
    rr = _r_overall(d, V, s, py.has_3level)
    desc = (f"{100 * (1 - s['alpha']):g}% CI, {'t' if s['dist_type'] == 't' else 'z'} inference, "
            f"CR2/KH {'on' if s['use_kh'] else 'off'}, 2-level τ²: {s['tau_method']}, "
            f"{'two-level' if s['model_choice'] == TWO_LEVEL_CHOICE else 'three-level'} model chosen; "
            f"{len(d)} effects, {d['id'].nunique()} studies, "
            f"{int(np.count_nonzero(V - np.diag(np.diag(V))) // 2)} sampling covariances")
    return [_stored_check(py, cfg)] + _compare('your data', py, d, rr, s), desc


def _reference_rows():
    rows = []
    s = dict(_DEFAULT_SETTINGS)
    for name, df, vcv in _ov_datasets():
        try:
            py, used = _py_overall_reference(df, vcv, s)
            used['id'] = used['id'].astype(str)
            d, V = validation_block_V(used, vcv)
            rr = _r_overall(d, V, s, py.has_3level)
            rows += _compare(name, py, d, rr, s)
        except Exception as e:
            rows.append({'dataset': name, 'check': '—', 'status': f'ERROR: {e}'})
    return rows


def _v21_main():
    """Main block, run in a function so its working variables do not leak
    into the notebook's shared namespace."""
    if 'OverallEngine' not in globals():
        raise ValidationPrerequisiteError("Overall Meta-Analysis cell not found: run Step 7 first.")
    rows, notices = [], []
    try:
        ur, desc = _user_rows()
        rows += ur
        notices.append(f"Your Step 7 settings: {desc}.")
    except ValidationPrerequisiteError:
        if not VALIDATION_INCLUDE_REFERENCE:
            raise
        notices.append("No Step 7 results in this session: only the reference datasets are checked.")
    if VALIDATION_INCLUDE_REFERENCE:
        rows += _reference_rows()

    res_df = pd.DataFrame(rows)
    st = res_df['status'].astype(str)
    n_info = int((st == 'INFO').sum())
    n_fail = int(((st != 'PASS') & (st != 'INFO')).sum())
    n_graded = len(res_df) - n_info
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
        f"<b>{icon} Overall model vs metafor + clubSandwich: {n_graded - n_fail}/{n_graded} checks passed"
        f"{f' ({n_info} INFO)' if n_info else ''}.</b><br>"
        f"<small>Tolerances: {TOL_OV}; estimates, variances and CI limits are compared on the scale max(1, |value|).</small></div>"))
    for n in notices:
        display(HTML(f"<div style='padding:8px; background:#e2e3e5; border-radius:4px; font-family:sans-serif;"
                     f" margin-top:6px;'>ℹ️ {n}</div>"))
    display(res_df.style.format({'max |Δ|': '{:.1e}'}, na_rep=''))
    return res_df, notices


res_df, _notes21 = _v21_main()
if res_df is not None and 'record_validation' in globals():
    record_validation(21, "Overall Model — Heterogeneity, I², PI, Knapp-Hartung", res_df, notes=' '.join(_notes21))


In [ ]:
#@title 🧪 Validation 22: Leave-One-Out and Baujat on your data (Python vs metafor + clubSandwich)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# YOUR DATA (default).
# 1. Leave-one-out: runs the Leave-One-Out engine (LOOEngine.run_analysis,
#    Step 21) on your analysis -- your rows, your sampling VCV with the
#    shared-control covariances, your CI level -- and refits the full model
#    and each reduced dataset in R on the same rows with the same V:
#        rma.mv(yi, V, random = ~ 1 | study/effect, method = "REML")
#        clubSandwich::coef_test(fit, vcov = "CR2", cluster = study, test = "Satterthwaite")
#    (a refit in which no study has more than one effect size: rma(method =
#    "REML") + CR2, as the engine). CI = mu ± t(Satterthwaite df) · SE.
#    Compared for the full model and every left-out study: mu, CR2 SE, CI,
#    tau², sigma². If Step 21 stored results in this session, the cell also
#    checks that they equal the re-run.
# 2. Baujat (compute_baujat_data, Step 23): x = each study's contribution to
#    Q of the fixed-effect model vs metafor::baujat(rma(method = "FE"))
#    summed over the study's effect sizes; y = (mu - mu_(-i))^2 / SE(mu_(-i))^2
#    vs the R refits above (CR2 SE of each refit).
#
# REFERENCE DATASETS (VALIDATION_INCLUDE_REFERENCE = True), at 95%:
# dat.konstantopoulos2011, dat.curtis1998 (lnRR), a simulated shared-control
# dataset (full V) and dat.bcg (log RR, one effect per study); these use
# the dataset helper of Validation 21. Your state is restored afterwards.
# Requires: Steps 2-6 (7 for your CI level), the Leave-One-Out (Step 21)
# and Baujat (Step 23) cells and the Validation R setup cell.

import numpy as np
import pandas as pd
import rpy2.robjects as ro
from IPython.display import display, HTML

TOL_LOO = {'est': 5e-5, 'se_rel': 1e-4, 'ci': 1e-4, 'q': 1e-8, 'infl_rel': 1e-3}

def _sdiff(a, b):
    """|a - b| on the scale of the value: absolute below 1, relative above,
    so the same tolerance works for lnRR (~0.1) and mean differences (~50)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.nanmax(np.abs(a - b) / np.maximum(1.0, np.abs(b))))



def _rv22(a):
    return ro.FloatVector(np.asarray(a, dtype=float).ravel())


def _rm22(M):
    M = np.asarray(M, dtype=float)
    return ro.r['matrix'](ro.FloatVector(M.ravel(order='F')), nrow=M.shape[0], ncol=M.shape[1])


def _loo_datasets():
    out = list(_ov_datasets())
    ro.r("suppressPackageStartupMessages({library(metafor); library(metadat)})")
    b = ro.r("""e <- escalc("RR", ai = tpos, bi = tneg, ci = cpos, di = cneg, data = dat.bcg)
                list(id = as.character(e$trial), yi = as.numeric(e$yi), vi = as.numeric(e$vi))""")
    out.append(('dat.bcg (log RR, one effect per study)',
                pd.DataFrame({'id': list(b.rx2('id')), 'yi': np.asarray(b.rx2('yi')), 'vi': np.asarray(b.rx2('vi'))}), {}))
    return out


def _py_loo_reference(df, vcv, alpha):
    g = globals()
    saved = g.get('ANALYSIS_CONFIG')
    cfg = {'analysis_data': df, 'effect_col': 'yi', 'var_col': 'vi', 'vcv_matrices': vcv,
           'es_config': {'effect_label': 'es', 'effect_label_short': 'es'},
           'global_settings': {'alpha': alpha, 'dist_type': 't'}}
    try:
        g['ANALYSIS_CONFIG'] = cfg
        return LOOEngine(LOODataManager(cfg)).run_analysis()
    finally:
        if saved is not None:
            g['ANALYSIS_CONFIG'] = saved


def _r_loo(d, V, alpha):
    """Full fit (first row) and one fit per left-out study: mu, SE(CR2), lo, hi, tau2, sigma2,
    plus each study's Q contribution from metafor::baujat."""
    ro.globalenv['yi_v'], ro.globalenv['vi_v'] = _rv22(d['yi']), _rv22(d['vi'])
    ro.globalenv['cl_v'], ro.globalenv['V_m'] = ro.StrVector(d['id'].astype(str).tolist()), _rm22(V)
    out = ro.r(f"""
    suppressPackageStartupMessages({{library(metafor); library(clubSandwich)}})
    a <- {alpha}
    d <- data.frame(yi = as.numeric(yi_v), vi = as.numeric(vi_v), cl = factor(as.character(cl_v),
                    levels = unique(as.character(cl_v))))
    d$es <- seq_len(nrow(d))
    fit <- function(keep) {{
        dd <- d[keep, ]; dd$cl <- droplevels(dd$cl); dd$es <- factor(dd$es); V <- V_m[keep, keep, drop = FALSE]
        if (any(table(dd$cl) > 1)) {{
            f <- rma.mv(yi, V, random = ~ 1 | cl/es, data = dd, method = "REML",
                        control = list(rel.tol = 1e-10, iter.max = 1000, eval.max = 2000))
            s2 <- f$sigma2
        }} else {{
            f <- rma(yi, vi, data = dd, method = "REML", control = list(threshold = 1e-12, maxiter = 5000))
            s2 <- c(f$tau2, 0)
        }}
        ct <- coef_test(f, vcov = "CR2", cluster = dd$cl, test = "Satterthwaite")
        h <- qt(1 - a / 2, ct$df_Satt) * ct$SE
        c(as.numeric(f$b), ct$SE, f$b - h, f$b + h, s2)
    }}
    res <- rbind(fit(rep(TRUE, nrow(d))), t(sapply(levels(d$cl), function(s) fit(d$cl != s))))
    fe <- rma(yi, vi, data = d, method = "FE")
    pdf(NULL); bj <- baujat(fe, progbar = FALSE); invisible(dev.off())   # coordinates only, no plot file
    list(res = res, ids = levels(d$cl), qx = as.numeric(tapply(bj$x, d$cl, sum)))
    """)
    res = np.asarray(out.rx2('res'))
    tab = pd.DataFrame(res, columns=['mu', 'se', 'lo', 'hi', 'tau2', 'sigma2'])
    tab.insert(0, 'excluded_study', ['(none)'] + list(out.rx2('ids')))
    return tab, pd.DataFrame({'unit_removed': list(out.rx2('ids')), 'q_R': np.asarray(out.rx2('qx'))})


def _compare_loo(name, res, d, V, raw, ecol, vcol, alpha):
    rows = []
    r_tab, r_q = _r_loo(d, V, alpha)
    o = res.original
    py_tab = pd.concat([pd.DataFrame([{'excluded_study': '(none)', 'mu': o.mu, 'se': o.se, 'ci_lower': o.ci_lower,
                                       'ci_upper': o.ci_upper, 'tau2': o.tau2, 'sigma2': o.sigma2}]),
                        res.iterations], ignore_index=True)
    py_tab['excluded_study'] = py_tab['excluded_study'].astype(str)
    m = r_tab.merge(py_tab, on='excluded_study', how='left', suffixes=('_R', ''))
    n_missing = int(m['mu'].isna().sum())
    d_est = _sdiff(np.r_[m['mu'], m['tau2'], m['sigma2']], np.r_[m['mu_R'], m['tau2_R'], m['sigma2_R']])
    d_se = float(np.nanmax(np.abs(m['se'] - m['se_R']) / m['se_R']))
    d_ci = _sdiff(np.r_[m['ci_lower'], m['ci_upper']], np.r_[m['lo'], m['hi']])
    ok = n_missing == 0 and d_est < TOL_LOO['est'] and d_se < TOL_LOO['se_rel'] and d_ci < TOL_LOO['ci']
    n_cov = int(np.count_nonzero(V - np.diag(np.diag(V))) // 2)
    rows.append({'dataset': name,
                 'check': f'leave-one-out: full + {len(m) - 1} refits (μ, SE, {100 * (1 - alpha):g}% CI, τ², σ²)',
                 'max |Δ| est.': d_est, 'max rel Δ SE': d_se, 'max |Δ| CI': d_ci,
                 'note': (f'{n_missing} refits missing in Python; ' if n_missing else '')
                         + f'{len(d)} effects, {n_cov} sampling covariances',
                 'status': 'PASS' if ok else 'FAIL'})
    # Baujat coordinates
    loo_in = res.iterations.rename(columns={'excluded_study': 'unit'})
    bj, _ = compute_baujat_data(raw, loo_in, ecol, vcol, o.mu)
    bj = bj.rename(columns={'unit': 'unit_removed'})
    y_R = r_tab.iloc[1:].assign(infl_R=lambda x: (x['mu'] - r_tab['mu'].iloc[0]) ** 2 / x['se'] ** 2)
    bm = bj.merge(r_q, on='unit_removed').merge(
        y_R[['excluded_study', 'infl_R']].rename(columns={'excluded_study': 'unit_removed'}), on='unit_removed')
    dq = float(np.max(np.abs(bm['q_contrib'] - bm['q_R']))) if len(bm) else np.nan
    dy = float(np.max(np.abs(bm['influence'] - bm['infl_R']) / np.maximum(1.0, bm['infl_R']))) if len(bm) else np.nan
    ok = len(bm) == len(r_q) and dq < TOL_LOO['q'] and dy < TOL_LOO['infl_rel']
    rows.append({'dataset': name, 'check': f'Baujat: Q contribution (x) and (Δμ)²/SE² (y), {len(bm)} studies',
                 'max |Δ| est.': dq, 'max rel Δ SE': dy, 'status': 'PASS' if ok else 'FAIL'})
    return rows


def _stored_loo_check(res, cfg):
    st = cfg.get('loo_results')
    if not isinstance(st, dict) or not isinstance(st.get('data'), pd.DataFrame):
        return None
    a = st['data'][['excluded_study', 'mu']].assign(excluded_study=lambda x: x['excluded_study'].astype(str))
    b = res.iterations[['excluded_study', 'mu']].assign(excluded_study=lambda x: x['excluded_study'].astype(str))
    m = a.merge(b, on='excluded_study', how='outer', suffixes=('_stored', '_rerun'))
    dmax = float(np.nanmax(np.abs(np.r_[m['mu_stored'] - m['mu_rerun'], [st.get('orig_mu', np.nan) - res.original.mu]])))
    same = bool(m[['mu_stored', 'mu_rerun']].notna().all().all()) and dmax < 1e-10
    return {'dataset': 'your data', 'check': 're-run reproduces your Step 21 results', 'max |Δ| est.': dmax,
            'status': 'PASS' if same else 'INFO',
            'note': '' if same else 'Step 21 results differ from a re-run with the current code and data: re-run Step 21'}


def _user_rows():
    u = validation_user_data()
    cfg = u['cfg']
    alpha = float((u['settings'] or {}).get('alpha', 0.05))
    dm = LOODataManager(cfg)
    if u['df']['id'].nunique() < 3:
        raise ValidationPrerequisiteError("Leave-one-out applies only to data with at least 3 studies.")
    res = LOOEngine(dm).run_analysis()
    if res is None:
        raise RuntimeError("LOOEngine.run_analysis returned None on your data")
    # the rows the engine uses (prepare_data drops missing / non-positive variances)
    d0 = u['df'].dropna(subset=['yi', 'vi', 'id'])
    d0 = d0[d0['vi'] > 0]
    d, V = validation_block_V(d0, u['vcv'])
    rows = _compare_loo('your data', res, d, V, cfg['analysis_data'], u['effect_col'], u['var_col'], alpha)
    st = _stored_loo_check(res, cfg)
    return ([st] if st else []) + rows, alpha, st is None


def _reference_rows():
    rows = []
    for name, df, vcv in _loo_datasets():
        try:
            res = _py_loo_reference(df, vcv, 0.05)
            if res is None:
                raise RuntimeError("LOOEngine.run_analysis returned None")
            d, V = validation_block_V(df.assign(id=df['id'].astype(str)), vcv)
            rows += _compare_loo(name, res, d, V, df, 'yi', 'vi', 0.05)
        except Exception as e:
            rows.append({'dataset': name, 'check': '—', 'status': f'ERROR: {e}'})
    return rows


def _v22_main():
    """Main block, run in a function so its working variables do not leak
    into the notebook's shared namespace."""
    if 'LOOEngine' not in globals():
        raise ValidationPrerequisiteError("Leave-One-Out cell not found: run Step 21 (Leave-One-Out: Execution) first.")
    if 'compute_baujat_data' not in globals():
        raise ValidationPrerequisiteError("Baujat cell not found: run Step 23 (Baujat Plot) first.")
    rows, notices = [], []
    try:
        ur, alpha, no_stored = _user_rows()
        rows += ur
        notices.append(f"Your data at your CI level ({100 * (1 - alpha):g}%)."
                       + (" Step 21 results not in this session: the stored leave-one-out table is not compared."
                          if no_stored else ""))
    except ValidationPrerequisiteError:
        if not VALIDATION_INCLUDE_REFERENCE:
            raise
        notices.append("Your data cannot be used (no effect sizes, or fewer than 3 studies): only the "
                       "reference datasets are checked.")
    if VALIDATION_INCLUDE_REFERENCE:
        if '_ov_datasets' in globals():
            rows += _reference_rows()
        else:
            notices.append("Reference datasets need the dataset helper of Validation 21: run Validation 21 first.")
            rows.append({'dataset': 'reference', 'check': 'reference datasets', 'status': 'INFO',
                         'note': 'run Validation 21 first (dataset helper)'})

    res_df = pd.DataFrame(rows)
    st = res_df['status'].astype(str)
    n_info = int((st == 'INFO').sum())
    n_fail = int(((st != 'PASS') & (st != 'INFO')).sum())
    n_graded = len(res_df) - n_info
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
        f"<b>{icon} Leave-one-out and Baujat vs metafor + clubSandwich: {n_graded - n_fail}/{n_graded} checks passed"
        f"{f' ({n_info} INFO)' if n_info else ''}.</b><br>"
        f"<small>Tolerances: {TOL_LOO}; estimates and CI limits are compared on the scale max(1, |value|).</small></div>"))
    for n in notices:
        display(HTML(f"<div style='padding:8px; background:#e2e3e5; border-radius:4px; font-family:sans-serif;"
                     f" margin-top:6px;'>ℹ️ {n}</div>"))
    display(res_df.style.format({c: '{:.1e}' for c in ('max |Δ| est.', 'max rel Δ SE', 'max |Δ| CI')
                                 if c in res_df.columns}, na_rep=''))
    return res_df, notices


res_df, _notes22 = _v22_main()
if res_df is not None and 'record_validation' in globals():
    record_validation(22, "Leave-One-Out and Baujat", res_df, notes=' '.join(_notes22))


In [ ]:
#@title 🧪 Validation 23: Subgroups and Meta-Regression on Your Data (Python vs metafor + clubSandwich)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Runs the Subgroup engine (SubgroupAnalysisEngine.analyze_all_subgroups) and
# the Meta-Regression engine (RegressionEngine.run_meta_regression) end to end
# on YOUR analysis and refits the same models in R. R estimates everything
# itself (no values pinned to Python's), on the same rows and the same
# sampling V (your shared-control covariances included):
#   subgroups   rma.mv(yi ~ 0 + group, V, random = ~ 1 | study/effect)
#               -> each group's mean, CR2 SE, Satterthwaite df, CI, p;
#               rma.mv(yi ~ group) -> tau², sigma²; Wald_test(HTZ) of the
#               between-group differences -> F, df, p; model-based QM of
#               rma.mv (test = "z") -> QM
#   meta-reg.   rma.mv(yi ~ x_centred + factor(f), V, random = ~ 1 | study/effect)
#               -> coefficients, CR2 SE, df, CI, p, tau², sigma²; Wald_test(HTZ)
#               per factor and for all moderators; pseudo-R² against the
#               intercept-only REML fit on the same rows. With one effect per
#               study the engine's two-level path is checked against
#               rma(yi, vi, mods, test = "adhoc") (truncated Knapp-Hartung).
# Your data: the subgroup configuration of Step 8 (moderator(s), groups kept
# by your min-studies / min-effects thresholds, single or two-way) and the
# moderators, types and reference levels of your Step 12 meta-regression,
# with your Step 7 alpha (CI level) and t / normal choice. When Step 8 or
# Step 12 was not run, every suitable categorical moderator (subgroups, with
# Step 8's default thresholds) and every categorical / continuous moderator
# (single-moderator meta-regressions) in your data is checked instead. When
# Step 9 / Step 12 results are in memory, they must also equal a fresh run
# (the numbers you report are the ones validated).
# Reference datasets (VALIDATION_INCLUDE_REFERENCE): dat.curtis1998 (lnRR;
# method, functional group, duration), dat.konstantopoulos2011 (year) and a
# simulated shared-control dataset with a study-level group and a continuous
# moderator. The user's state is never modified. Requires: Cells 1-12
# (subgroup) and the meta-regression cell, and the Validation R setup cell.

import numpy as np
import pandas as pd
import rpy2.robjects as ro
from IPython.display import display, HTML

TOL_MOD = {'est': 1e-5, 'se_rel': 1e-4, 'df_rel': 1e-3, 'p': 1e-4, 'F_rel': 1e-4, 'R2': 1e-3}

# Columns that are never moderators (pipeline keys, raw data, derived effect sizes)
_V23_NOT_MODERATORS = {'id', 'yi', 'vi', '_vcv_row_pos', 'row_id', 'shared_group_id', 'w_fixed', 'w_random',
                       'ci_width', 'xe', 'sde', 'ne', 'xc', 'sdc', 'nc', 'events_e', 'nonevents_e', 'events_c',
                       'nonevents_c', 'n_total', 'se', 'variance', 'sp', 'sp_squared', 'df', 'cohen_d',
                       'hedges_j', 'cv_e', 'cv_c', 'subgroup_label'}


def _v23_rm(M):
    M = np.asarray(M, dtype=float)
    return ro.r['matrix'](ro.FloatVector(M.ravel(order='F')), nrow=M.shape[0], ncol=M.shape[1])


def _v23_reference():
    """Fixed reference datasets (only built when the switch is on)."""
    ro.r("suppressPackageStartupMessages({library(metafor); library(metadat)})")
    cu = ro.r("""e <- escalc("ROM", m1i = m1i, sd1i = sd1i, n1i = n1i, m2i = m2i, sd2i = sd2i, n2i = n2i,
                         data = dat.curtis1998)
                 list(id = as.character(e$paper), yi = as.numeric(e$yi), vi = as.numeric(e$vi),
                      method = as.character(e$method), fungrp = as.character(e$fungrp), time = as.numeric(e$time))""")
    curtis = pd.DataFrame({k: (list(cu.rx2(k)) if k in ('id', 'method', 'fungrp') else np.asarray(cu.rx2(k)))
                           for k in ('id', 'yi', 'vi', 'method', 'fungrp', 'time')})
    kd = {c: np.asarray(ro.r(f"as.numeric(dat.konstantopoulos2011${c})")) for c in ('district', 'yi', 'vi', 'year')}
    konst = pd.DataFrame({'id': kd['district'].astype(int).astype(str), 'yi': kd['yi'], 'vi': kd['vi'], 'year': kd['year']})
    rng = np.random.default_rng(15)
    rows, vcv = [], {}
    for i in range(24):
        kk = int(rng.integers(1, 5)); grp = ['A', 'B', 'C'][i % 3]; x = rng.uniform(0, 10)
        V = np.full((kk, kk), rng.uniform(.01, .05)) + np.diag(rng.uniform(.02, .1, kk))
        mu = {'A': 0.1, 'B': 0.4, 'C': 0.25}[grp] + 0.03 * x
        y = mu + rng.normal(0, .15) + rng.normal(0, .1, kk) + rng.multivariate_normal(np.zeros(kk), V)
        vcv[f"S{i:02d}"] = V
        rows += [dict(id=f"S{i:02d}", yi=y[j], vi=V[j, j], grp=grp, x=x) for j in range(kk)]
    sim = pd.DataFrame(rows)

    def sub(name, df, vcv, mod, levels):
        cfg = _v23_base_cfg(df, vcv)
        cfg['subgroup_config'] = {'analysis_type': 'single', 'moderator1': mod, 'moderator2': None,
                                  'valid_groups_list': list(levels), 'reference_level': None}
        return {'kind': 'subgroups', 'name': name, 'cfg': cfg, 'df': df, 'vcv': vcv, 'stored': None}

    def reg(name, df, vcv, mods, cats):
        return {'kind': 'meta-regression', 'name': name, 'cfg': _v23_base_cfg(df, vcv), 'df': df, 'vcv': vcv,
                'mods': list(mods), 'cats': list(cats), 'refs': {}, 'stored': None}

    return [sub('reference: dat.curtis1998, method', curtis, {}, 'method', ['GC', 'GH', 'OTC']),
            sub('reference: dat.curtis1998, functional group', curtis, {}, 'fungrp', ['ANGIO', 'GYMNO']),
            sub('reference: simulated group (shared V)', sim, vcv, 'grp', ['A', 'B', 'C']),
            reg('reference: dat.curtis1998, time + method', curtis, {}, ['time', 'method'], ['method']),
            reg('reference: dat.konstantopoulos2011, year', konst, {}, ['year'], []),
            reg('reference: simulated x + group (shared V)', sim, vcv, ['x', 'grp'], ['grp'])]


def _v23_base_cfg(df, vcv):
    return {'analysis_data': df, 'effect_col': 'yi', 'var_col': 'vi', 'vcv_matrices': vcv,
            'es_config': {'effect_label': 'es', 'effect_label_short': 'es', 'has_fold_change': False},
            'global_settings': {'alpha': 0.05, 'dist_type': 't'},
            'overall_results': {'tau_squared': 0.0, 'Qt': np.nan, 'I_squared': np.nan, 'reported': None}}


def _v23_with_config(cfg, fn):
    """Run fn with ANALYSIS_CONFIG temporarily pointing at cfg (a copy, so the
    user's own dict is never written to)."""
    g = globals()
    saved = g.get('ANALYSIS_CONFIG')
    try:
        g['ANALYSIS_CONFIG'] = cfg
        return fn()
    finally:
        if saved is not None:
            g['ANALYSIS_CONFIG'] = saved


def _v23_candidates(df, cfg):
    """(categorical, continuous) moderator candidates in the user's data."""
    excl = set(_V23_NOT_MODERATORS)
    for k in ('effect_col', 'var_col', 'se_col'):
        if cfg.get(k):
            excl.add(cfg[k])
    if 'RegressionDataManager' in globals():
        try:
            excl |= set(RegressionDataManager(cfg)._build_exclusion_set())
        except Exception:
            pass
    cats, conts = [], []
    for c in df.columns:
        low = str(c).lower()
        if c in excl or 'imputed' in low or 'ci_' in low or 'shared_group' in low:
            continue
        s = df[c]
        n_lev = int(s.nunique(dropna=True))
        if n_lev < 2 or int(s.notna().sum()) < 3:
            continue
        if pd.api.types.is_numeric_dtype(s):
            if n_lev >= 3:
                conts.append(c)
        elif n_lev <= max(2, len(df) // 2):
            cats.append(c)
    return cats, conts


def _v23_user_jobs(u):
    """Subgroup and meta-regression jobs on the user's analysis."""
    cfg, df, vcv = u['cfg'], u['df'], u['vcv']
    gs = dict(u['settings'] or {})
    jobs, notes = [], []
    cats, conts = _v23_candidates(df, cfg)

    def user_cfg():
        c = dict(cfg)                   # shallow copy: the engines never write to the user's dict
        c['global_settings'] = gs
        return c

    # ---- subgroups: Step 8 configuration, else every categorical moderator ----
    sc = cfg.get('subgroup_config')
    if sc and sc.get('moderator1') in df.columns:
        c = user_cfg()
        c['subgroup_config'] = dict(sc)
        mods = sc['moderator1'] + (f" × {sc['moderator2']}" if sc.get('moderator2') else '')
        stored = cfg.get('subgroup_results')
        if stored and (stored.get('moderator1'), stored.get('moderator2')) != (sc.get('moderator1'), sc.get('moderator2')):
            stored = None
        jobs.append({'kind': 'subgroups', 'name': f"your data: {mods} (Step 8, {sc.get('analysis_type', 'single')})",
                     'cfg': c, 'df': df, 'vcv': vcv, 'stored': stored})
    else:
        notes.append("Step 8 (subgroup configuration) was not run: every categorical moderator was checked "
                     "with Step 8's default thresholds (≥ 3 studies and ≥ 5 effect sizes per group).")
        for m in cats:
            lab = df[m].astype(str).str.strip()
            ok = [g for g in pd.unique(lab[df[m].notna()])
                  if df.loc[lab == g, 'id'].nunique() >= 3 and (lab == g).sum() >= 5]
            if len(ok) < 2:
                continue
            c = user_cfg()
            c['subgroup_config'] = {'analysis_type': 'single', 'moderator1': m, 'moderator2': None,
                                    'valid_groups_list': list(ok), 'reference_level': None,
                                    'min_papers': 3, 'min_obs': 5}
            jobs.append({'kind': 'subgroups', 'name': f"your data: {m} (all groups with ≥ 3 studies, ≥ 5 effects)",
                         'cfg': c, 'df': df, 'vcv': vcv, 'stored': None})

    # ---- meta-regression: Step 12 model, else each moderator on its own ----
    rr = cfg.get('meta_regression_RVE_results')
    last = globals().get('LAST_REGRESSION_RESULT')
    mods = [str(m).replace(' (centered)', '') for m in (rr or {}).get('moderator_cols') or []]
    if rr and last is not None and getattr(last, 'moderator_cols', None) \
            and list(getattr(last, 'term_names', None) or []) == list(rr.get('term_names') or []):
        mods = list(last.moderator_cols)        # the exact Step 12 selection
    if rr and mods and all(m in df.columns for m in mods):
        terms = [str(t) for t in rr.get('term_names') or []]
        rcats = [m for m in mods if any(t.startswith(f"{m}[") for t in terms)]
        refs = {}
        if last is not None and list(getattr(last, 'moderator_cols', None) or []) == mods:
            refs = dict(getattr(last, 'reference_levels', None) or {})
        else:
            clean = df.dropna(subset=mods)
            for m in rcats:
                nonref = {t[len(m) + 1:-1] for t in terms if t.startswith(f"{m}[")}
                rest = sorted(set(clean[m].astype(str)) - nonref)
                if len(rest) == 1:
                    refs[m] = rest[0]
        jobs.append({'kind': 'meta-regression', 'name': f"your data: {' + '.join(mods)} (Step 12)",
                     'cfg': user_cfg(), 'df': df, 'vcv': vcv, 'mods': mods, 'cats': rcats, 'refs': refs,
                     'stored': rr})
    else:
        notes.append("Step 12 (meta-regression) was not run: a single-moderator meta-regression was checked "
                     "for every categorical and continuous moderator in your data.")
        for m in conts + cats:
            jobs.append({'kind': 'meta-regression', 'name': f"your data: {m} (single moderator)",
                         'cfg': user_cfg(), 'df': df, 'vcv': vcv, 'mods': [m],
                         'cats': [m] if m in cats else [], 'refs': {}, 'stored': None})
    if not jobs:
        notes.append("No categorical or continuous moderator suitable for a subgroup analysis or a "
                     "meta-regression was found in your data.")
    return jobs, notes


def _v23_r_frame(d, V, cols):
    """Send rows d (grouped by study, matching V) to R as data frame `d`."""
    for name, vals in cols.items():
        ro.globalenv[f'c_{name}'] = vals
    ro.globalenv['V_m'] = _v23_rm(V)
    ro.globalenv['id_v'] = ro.StrVector(d['id'].astype(str).tolist())
    ro.globalenv['yi_v'] = ro.FloatVector(d['yi'].astype(float).values)
    ro.globalenv['vi_v'] = ro.FloatVector(d['vi'].astype(float).values)
    extra = ''.join(f", {n} = c_{n}" for n in cols)
    ro.r(f"""suppressPackageStartupMessages({{library(metafor); library(clubSandwich)}})
             d <- data.frame(id = as.character(id_v), yi = as.numeric(yi_v), vi = as.numeric(vi_v){extra},
                             stringsAsFactors = FALSE)
             d$cl <- factor(d$id, levels = unique(d$id)); d$es <- factor(seq_len(nrow(d)))
             ctl <- list(rel.tol = 1e-10, iter.max = 1000, eval.max = 2000)""")


def _v23_subgroup_rows(job):
    """Rows the subgroup engine uses (Step 8 groups, labels as Cell 9 builds them)."""
    sc = job['cfg']['subgroup_config']
    df = job['df'].copy()
    m1, m2 = sc['moderator1'], sc.get('moderator2')
    if sc.get('analysis_type', 'single') == 'two_way' and m2:
        lab = df[m1].astype(str).str.strip() + ' | ' + df[m2].astype(str).str.strip()
        valid = {f"{str(a).strip()} | {str(b).strip()}" for a, b in sc['valid_groups_list']}
    else:
        lab = df[m1].astype(str).str.strip()
        valid = {str(v).strip() for v in sc['valid_groups_list']}
    df['g'] = lab
    df = df[~df['g'].isin(['', 'nan', 'None', 'NaN']) & df['g'].isin(valid)
            & df['yi'].notna() & df['vi'].notna() & (df['vi'] > 0)]
    cnt = df['g'].value_counts()
    return df[df['g'].isin(cnt.index[cnt >= 2])]


def _v23_r_subgroup(job, rows, levels, alpha, dist):
    d, V = validation_block_V(rows, job['vcv'])
    _v23_r_frame(d, V, {'g': ro.StrVector(d['g'].astype(str).tolist())})
    ro.globalenv['lev_v'] = ro.StrVector([str(l) for l in levels])
    single = bool((d.groupby('id').size() == 1).all())
    rnd = '~ 1 | cl' if single else '~ 1 | cl/es'
    out = ro.r(f"""
        d$g <- factor(d$g, levels = as.character(lev_v)); a <- {alpha}
        fm <- rma.mv(yi ~ 0 + g, V_m, random = {rnd}, data = d, method = "REML", control = ctl)
        ct <- coef_test(fm, vcov = "CR2", cluster = d$cl, test = "Satterthwaite")
        fd <- rma.mv(yi ~ g, V_m, random = {rnd}, data = d, method = "REML", control = ctl)
        q <- nlevels(d$g) - 1; C <- cbind(0, diag(q))
        w <- Wald_test(fd, constraints = C, vcov = "CR2", cluster = d$cl, test = "HTZ")
        dfl <- pmax(1, ct$df_Satt)
        crit <- if ("{dist}" == "t") qt(1 - a / 2, dfl) else qnorm(1 - a / 2)
        pv <- if ("{dist}" == "t") 2 * pt(-abs(ct$beta / ct$SE), dfl) else 2 * pnorm(-abs(ct$beta / ct$SE))
        list(mu = as.numeric(fm$b), se = ct$SE, df = ct$df_Satt, lo = as.numeric(fm$b) - crit * ct$SE,
             hi = as.numeric(fm$b) + crit * ct$SE, p = pv, s2 = if ({'TRUE' if single else 'FALSE'}) c(fd$sigma2, 0) else fd$sigma2,
             F = c(w$Fstat, w$df_num, w$df_denom, w$p_val), QM = fd$QM, k = as.numeric(table(d$g)))""")
    return {k: np.asarray(out.rx2(k), dtype=float) for k in ('mu', 'se', 'df', 'lo', 'hi', 'p', 's2', 'F', 'QM', 'k')}


def _v23_r_regression(job, rows, res, alpha, dist):
    d, V = validation_block_V(rows, job['vcv'])
    refs = dict(getattr(res, 'reference_levels', None) or {})
    cols, terms, lev_order = {}, [], {}
    for i, m in enumerate(job['mods']):
        nm = f"m{i + 1}"
        if m in job['cats']:
            vals = d[m].astype(str)
            levs = sorted(vals.unique().tolist())
            ref = refs.get(m, levs[0])
            lev_order[nm] = [ref] + [l for l in levs if l != ref]
            cols[nm] = ro.StrVector(vals.tolist())
            terms.append(nm)
        else:
            cols[nm] = ro.FloatVector(pd.to_numeric(d[m], errors='coerce').astype(float).values)
            terms.append(f"{nm}_c")
    _v23_r_frame(d, V, cols)
    for nm, levs in lev_order.items():
        ro.globalenv['lev_v'] = ro.StrVector(levs)
        ro.r(f"d${nm} <- factor(d${nm}, levels = as.character(lev_v))")
    for i, m in enumerate(job['mods']):
        if m not in job['cats']:
            ro.r(f"d$m{i + 1}_c <- d$m{i + 1} - mean(d$m{i + 1})")
    form = '~ ' + ' + '.join(terms)
    two_level = bool((d.groupby('id').size() == 1).all())
    catn = [f"m{i + 1}" for i, m in enumerate(job['mods']) if m in job['cats']]
    if two_level:
        # Cell 12's two-level path: REML tau², truncated Knapp-Hartung (test = "adhoc"), df = k - p
        tests = ''.join(f"; w_{n} <- {{ix <- grep('^{n}', rownames(f$b)); anova(f, btt = ix)}}" for n in catn)
        per = ', '.join(f"{n} = c(w_{n}$QM, w_{n}$QMdf, w_{n}$QMp)" for n in catn)
        out = ro.r(f"""
            a <- {alpha}
            f <- rma(yi, vi, mods = {form}, data = d, method = "REML", test = "adhoc",
                     control = list(threshold = 1e-12, maxiter = 5000))
            f0 <- rma(yi, vi, data = d, method = "REML", control = list(threshold = 1e-12, maxiter = 5000)){tests}
            p <- length(coef(f)); dfr <- rep(f$k - p, p)
            crit <- if ("{dist}" == "t") qt(1 - a / 2, dfr) else qnorm(1 - a / 2)
            pv <- if ("{dist}" == "t") 2 * pt(-abs(f$zval), dfr) else 2 * pnorm(-abs(f$zval))
            list(b = as.numeric(f$b), se = f$se, df = dfr, p = pv, lo = as.numeric(f$b) - crit * f$se,
                 hi = as.numeric(f$b) + crit * f$se, s2 = c(f$tau2, 0),
                 R2 = max(0, (f0$tau2 - f$tau2) / f0$tau2) * 100,
                 overall = if (p > 2) c(f$QM, f$QMdf, f$QMp) else rep(NA_real_, 4)
                 {', ' + per if per else ''})""")
    else:
        tests = ''.join(f"""; w_{n} <- {{ix <- grep('^{n}', names(coef(f))); Wald_test(f, constraints = constrain_zero(ix),
                           vcov = "CR2", cluster = d$cl, test = "HTZ")}}""" for n in catn)
        per = ', '.join(f"{n} = c(w_{n}$Fstat, w_{n}$df_num, w_{n}$df_denom, w_{n}$p_val)" for n in catn)
        out = ro.r(f"""
            a <- {alpha}
            f <- rma.mv(yi {form}, V_m, random = ~ 1 | cl/es, data = d, method = "REML", control = ctl)
            f0 <- rma.mv(yi, V_m, random = ~ 1 | cl/es, data = d, method = "REML", control = ctl)
            ct <- coef_test(f, vcov = "CR2", cluster = d$cl, test = "Satterthwaite"){tests}
            p <- length(coef(f))
            crit <- if ("{dist}" == "t") qt(1 - a / 2, ct$df_Satt) else qnorm(1 - a / 2)
            pv <- if ("{dist}" == "t") ct$p_Satt else 2 * pnorm(-abs(ct$beta / ct$SE))
            wa <- if (p > 2) Wald_test(f, constraints = constrain_zero(2:p), vcov = "CR2", cluster = d$cl, test = "HTZ") else NULL
            list(b = as.numeric(f$b), se = ct$SE, df = ct$df_Satt, p = pv, lo = as.numeric(f$b) - crit * ct$SE,
                 hi = as.numeric(f$b) + crit * ct$SE, s2 = f$sigma2,
                 R2 = max(0, (sum(f0$sigma2) - sum(f$sigma2)) / sum(f0$sigma2)) * 100,
                 overall = if (is.null(wa)) rep(NA_real_, 4) else c(wa$Fstat, wa$df_num, wa$df_denom, wa$p_val)
                 {', ' + per if per else ''})""")
    r = {k: np.asarray(out.rx2(k), dtype=float) for k in ('b', 'se', 'df', 'p', 'lo', 'hi', 's2', 'R2', 'overall')}
    r['per'] = {m: np.asarray(out.rx2(f"m{i + 1}"), dtype=float)
                for i, m in enumerate(job['mods']) if m in job['cats']}
    r['two_level'] = two_level
    return r


def _relm(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(np.max(np.abs(a - b) / np.maximum(np.abs(b), 1e-12)))


def _absm(a, b):
    return float(np.max(np.abs(np.asarray(a, float) - np.asarray(b, float))))


def _v23_worst(d, tol):
    return max(d, key=lambda k: (d[k] / tol.get(k, TOL_MOD['F_rel'])) if np.isfinite(d[k]) else np.inf)


def _v23_run_subgroups(job):
    cfg = job['cfg']
    gs = cfg.get('global_settings') or {}
    alpha, dist = float(gs.get('alpha', 0.05)), str(gs.get('dist_type', 't'))
    res_tab, het = _v23_with_config(cfg, lambda: SubgroupAnalysisEngine(SubgroupDataManager(cfg)).analyze_all_subgroups())
    if het.get('fallback_used'):
        raise RuntimeError("the unified subgroup model did not converge (per-group fallback fits used)")
    levels = [str(g) for g in res_tab['group']]
    rows = _v23_subgroup_rows(job)
    rr = _v23_r_subgroup(job, rows, levels, alpha, dist)
    t = res_tab.set_index(res_tab['group'].astype(str)).loc[levels]
    d = {'rows per group': _absm(t['k'], rr['k']),
         'means': _absm(t['pooled_effect_re'], rr['mu']), 'SE (rel)': _relm(t['pooled_se_re'], rr['se']),
         'df (rel)': _relm(t['df_inference'], np.maximum(1, rr['df'])),
         'CI': max(_absm(t['ci_lower_re'], rr['lo']), _absm(t['ci_upper_re'], rr['hi'])),
         'p': _absm(t['p_value_re'], rr['p']),
         'τ², σ²': _absm([t['tau_squared'].iloc[0], t['sigma_squared'].iloc[0]], rr['s2']),
         'HTZ F (rel)': _relm([het['F_omnibus'], het['df_F_num'], het['df_F_den']], rr['F'][:3]),
         'HTZ p': abs(het['p_value_F_omnibus'] - rr['F'][3]), 'QM (rel)': _relm([het['QM']], rr['QM'])}
    tol = {'rows per group': 0.5, 'means': TOL_MOD['est'], 'SE (rel)': TOL_MOD['se_rel'], 'df (rel)': TOL_MOD['df_rel'],
           'CI': TOL_MOD['est'] * 10, 'p': TOL_MOD['p'], 'τ², σ²': TOL_MOD['est'], 'HTZ F (rel)': TOL_MOD['F_rel'],
           'HTZ p': TOL_MOD['p'], 'QM (rel)': TOL_MOD['F_rel']}
    ok = all(v < tol[k] for k, v in d.items())
    worst = _v23_worst(d, tol)
    out = [{'analysis': 'subgroups', 'dataset': job['name'], 'k': int(len(rows)),
            'F (Py)': het['F_omnibus'], 'F (R)': rr['F'][0], 'p F (Py)': het['p_value_F_omnibus'], 'p F (R)': rr['F'][3],
            'worst check': f"{worst}: {d[worst]:.1e}", 'status': 'PASS' if ok else 'FAIL',
            'note': f"{len(levels)} groups, {100 * (1 - alpha):g}% CI, {dist}"}]
    st = job.get('stored')
    if st is not None and isinstance(st.get('results_df'), pd.DataFrame):
        s = st['results_df'].set_index(st['results_df']['group'].astype(str))
        same = set(s.index) == set(levels)
        dd = (max(_absm(s.loc[levels, 'pooled_effect_re'], t['pooled_effect_re']),
                  _absm(s.loc[levels, 'pooled_se_re'], t['pooled_se_re']),
                  _absm(s.loc[levels, 'p_value_re'], t['p_value_re']),
                  abs(float(st.get('F_omnibus', np.nan)) - float(het['F_omnibus'])))
              if same else np.inf)
        out.append({'analysis': 'subgroups', 'dataset': job['name'] + ' — Step 9 results in memory',
                    'worst check': f"stored vs fresh run: {dd:.1e}", 'status': 'PASS' if dd < 1e-8 else 'FAIL',
                    'note': '' if dd < 1e-8 else "Step 9 results differ from a fresh run of the engine: if you "
                                                  "changed the data, Step 7 or Step 8 after Step 9, re-run Step 9."})
    return out


def _v23_run_regression(job):
    cfg = job['cfg']
    gs = cfg.get('global_settings') or {}
    alpha, dist = float(gs.get('alpha', 0.05)), str(gs.get('dist_type', 't'))
    res = _v23_with_config(cfg, lambda: RegressionEngine(RegressionDataManager(cfg)).run_meta_regression(
        moderator_cols=list(job['mods']), categorical_cols=list(job['cats']), reference_levels=dict(job['refs'])))
    if res is None:
        raise RuntimeError("run_meta_regression returned None")
    res, reg_df = res if isinstance(res, tuple) else (res, None)
    src = job['df']
    if reg_df is not None and 'id' in reg_df.columns and reg_df.index.isin(src.index).all() \
            and (src.loc[reg_df.index, 'id'].astype(str).values == reg_df['id'].astype(str).values).all():
        rows = src.loc[reg_df.index]            # the rows the engine used (3-level path)
    else:
        # two-level path: one effect per study, aggregated frame = the cleaned rows
        rows = src.dropna(subset=list(job['mods']) + ['yi', 'vi'])
        rows = rows[rows['vi'] > 0]
    rr = _v23_r_regression(job, rows, res, alpha, dist)
    cis = np.asarray(res.cis, dtype=float)
    d = {'coefficients': _absm(res.betas, rr['b']), 'SE (rel)': _relm(res.se_betas, rr['se']),
         'df (rel)': _relm(res.df_resid, rr['df']), 'p': _absm(res.p_values, rr['p']),
         'CI': max(_absm(cis[:, 0], rr['lo']), _absm(cis[:, 1], rr['hi'])),
         'τ², σ²': _absm([res.tau_sq, res.sigma_sq], rr['s2']), 'pseudo-R²': abs(res.r_squared - rr['R2'][0])}
    om = res.omnibus or {}
    if rr['two_level']:
        # rma(test = "adhoc") gives F, df_num, p (df_denom = k - p)
        if np.isfinite(rr['overall'][0]):
            ov = om.get('overall') or {}
            d['F all (rel)'] = _relm([ov.get('F', np.nan), ov.get('df_num', np.nan)], rr['overall'][:2])
        for m in job['cats']:
            if len([t for t in res.term_names if str(t).startswith(f"{m}[")]) > 1:
                pf = (om.get('per_factor') or {}).get(m) or {}
                d[f'F {m} (rel)'] = _relm([pf.get('F', np.nan), pf.get('df_num', np.nan)], rr['per'][m][:2])
        r_over = (rr['overall'][0], rr['overall'][3])
    else:
        if np.isfinite(rr['overall'][0]):
            ov = om.get('overall') or {}
            d['HTZ all (rel)'] = _relm([ov.get('F', np.nan), ov.get('df_num', np.nan), ov.get('df_denom', np.nan)],
                                       rr['overall'][:3])
        for m in job['cats']:
            pf = (om.get('per_factor') or {}).get(m) or {}
            d[f'HTZ {m} (rel)'] = _relm([pf.get('F', np.nan), pf.get('df_num', np.nan), pf.get('df_denom', np.nan)],
                                        rr['per'][m][:3])
        r_over = (rr['overall'][0], rr['overall'][3])
    tol = {'coefficients': TOL_MOD['est'], 'SE (rel)': TOL_MOD['se_rel'], 'df (rel)': TOL_MOD['df_rel'],
           'p': TOL_MOD['p'], 'CI': TOL_MOD['est'] * 10, 'τ², σ²': TOL_MOD['est'], 'pseudo-R²': TOL_MOD['R2']}
    ok = all((v < tol.get(k, TOL_MOD['F_rel'])) for k, v in d.items())
    worst = _v23_worst(d, tol)
    out = [{'analysis': 'meta-regression', 'dataset': job['name'], 'k': int(len(rows)),
            'F (Py)': (om.get('overall') or {}).get('F', np.nan), 'F (R)': r_over[0],
            'p F (Py)': (om.get('overall') or {}).get('p_val', np.nan), 'p F (R)': r_over[1],
            'worst check': f"{worst}: {d[worst]:.1e}", 'status': 'PASS' if ok else 'FAIL',
            'note': (f"{'two-level (one effect per study), rma test = adhoc' if rr['two_level'] else 'three-level, CR2'}, "
                     f"{100 * (1 - alpha):g}% CI, {dist}")}]
    st = job.get('stored')
    if st is not None and st.get('betas_all') is not None:
        b_st = np.asarray(st['betas_all'], float)
        same = b_st.size == np.asarray(res.betas).size and list(st.get('term_names') or []) == list(res.term_names)
        dd = (max(_absm(b_st, res.betas), _absm(st['std_errors_all'], res.se_betas),
                  _absm(st['p_values_all'], res.p_values)) if same else np.inf)
        out.append({'analysis': 'meta-regression', 'dataset': job['name'] + ' — Step 12 results in memory',
                    'worst check': f"stored vs fresh run: {dd:.1e}", 'status': 'PASS' if dd < 1e-8 else 'FAIL',
                    'note': '' if dd < 1e-8 else "Step 12 results differ from a fresh run of the engine: if you "
                                                  "changed the data or Step 7 settings after Step 12, re-run Step 12."})
    return out


def _v15_main():
    """Main block, run in a function so its working variables do not leak
    into the notebook's shared namespace."""
    if 'SubgroupAnalysisEngine' not in globals() or 'RegressionEngine' not in globals():
        raise ValidationPrerequisiteError(
            "The subgroup and meta-regression engines are not defined: run the Subgroup Analysis (Steps 8-9) "
            "and Meta-Regression (Step 12) cells first.")
    jobs, notes = [], []
    try:
        u = validation_user_data()
        jobs, notes = _v23_user_jobs(u)
    except ValidationPrerequisiteError:
        if not VALIDATION_INCLUDE_REFERENCE:
            raise
        notes.append("No user analysis in this session: only the reference datasets were checked.")
    if VALIDATION_INCLUDE_REFERENCE:
        jobs += _v23_reference()
    if not jobs:
        raise ValidationPrerequisiteError(
            "No moderator in your data supports a subgroup analysis or a meta-regression: add moderators and "
            "run Steps 2-7 (and Steps 8-9 / 12) first.")
    rows = []
    for job in jobs:
        try:
            rows += (_v23_run_subgroups(job) if job['kind'] == 'subgroups' else _v23_run_regression(job))
        except Exception as e:
            rows.append({'analysis': job['kind'], 'dataset': job['name'], 'status': 'FAIL',
                         'worst check': f"ERROR: {type(e).__name__}: {e}"})
    for n in notes:
        rows.append({'analysis': 'notice', 'dataset': 'your data', 'status': 'INFO', 'note': n})
    res_df = pd.DataFrame(rows)
    n_info = int((res_df['status'] == 'INFO').sum())
    n_fail = int((~res_df['status'].isin(['PASS', 'INFO'])).sum())
    n_chk = len(res_df) - n_info
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    note_html = ''.join(f"<br>ℹ️ {n}" for n in notes)
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
        f"<b>{icon} Subgroups and meta-regression vs metafor + clubSandwich: {n_chk - n_fail}/{n_chk} analyses passed.</b><br>"
        f"<small>Your data, sampling V (shared controls included) and Step 7 settings"
        f"{'' if not VALIDATION_INCLUDE_REFERENCE else ', plus the fixed reference datasets'}. R estimates the "
        f"variance components itself. 'worst check' is the comparison closest to (or beyond) its tolerance. "
        f"Tolerances: {TOL_MOD}.</small>{note_html}</div>"))
    display(res_df.style.format(precision=5, na_rep=''))
    return res_df


res_df = _v15_main()
if res_df is not None and 'record_validation' in globals():
    record_validation(23, "Subgroups and Meta-Regression on Your Data", res_df)


In [ ]:
#@title 🧪 Validation 24: Shared-Control Covariance Matrices on your data (Python vs metafor::vcalc and Monte Carlo)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Checks the sampling variance-covariance matrices that Step 6 (Effect Size
# Calculation) built for YOUR data when several treatments share one control
# group (shared_group_id from Data Cleaning), exactly as the analyses use
# them: the block-diagonal V for all your rows, sliced from
# ANALYSIS_CONFIG['vcv_matrices'] by row identity (validation_block_V).
#
# YOUR DATA (default), for your effect-size type:
#   * diagonal = your sampling-variance column;
#   * lnRR, MD, lnCVR, log OR, log RR, RD: each variance is a treatment part
#     plus a control part, v_i = A_i + C, and effects sharing the control
#     have covariance C. metafor::vcalc reproduces this exactly with
#         vcalc(vi, cluster = study, grp1 = treatment, grp2 = control,
#               w1 = 1 / A_i, w2 = 1 / C)
#     (parts computed from the SDs Step 6 used, i.e. the imputed SDs, and
#     from the continuity-corrected counts). Compared: every element of V,
#     plus v_i = A_i + C. Groups whose rows report different control data
#     (e.g. a continuity correction applied to one arm only) use the first
#     row's control data by design; they are listed as INFO, not compared.
#   * Hedges' g / Cohen's d: vcalc can only approximate the covariance, so V
#     is compared with the analytic Gleser-Olkin covariance
#     1/nc + y_i y_j nc / (2 N_i N_j) (computed here independently);
#     vcalc's default approximation (w = n) is shown as INFO.
#   * Without shared controls (or for proportions / pre-calculated data) the
#     matrix must be diagonal; this is checked and the absence of shared
#     controls is reported as INFO.
#
# REFERENCE DATASETS (VALIDATION_INCLUDE_REFERENCE = True), through the
# notebook's own pipeline on a private configuration (your analysis is
# untouched):
# A. a constructed dataset (6 studies, 3 with 2-4 treatments sharing a
#    control) for lnRR, MD, log OR, log RR and RD vs metafor::vcalc;
# B. Monte Carlo check of the formulas, for every type (including Hedges' g,
#    Cohen's d and lnCVR): summary data of one control and three treatments
#    (n = 280-320 per group) simulated many times; the empirical covariance
#    matrix of the effect sizes is compared with CoMeta's matrix at the true
#    parameters (for g and d with equal SDs, the assumption of their variance
#    formula); for g and d, vcalc's default approximation is shown (INFO).
#
# Requires: Steps 2-6 (your data), the Validation R setup cell.

import numpy as np
import pandas as pd
import rpy2.robjects as ro
from IPython.display import display, HTML

TOL_VCV = {'vcalc': 1e-10, 'mc_rel': 0.05}
_MC_REPS = 40000
_COMP = {   # treatment / control variance parts for the vcalc reference (column names of the pipeline data)
    'lnRR':      ('sde**2 / (ne * xe**2)', 'sdc**2 / (nc * xc**2)'),
    'MD':        ('sde**2 / ne', 'sdc**2 / nc'),
    'log_or':    ('1/events_e + 1/nonevents_e', '1/events_c + 1/nonevents_c'),
    'log_rr':    ('1/events_e - 1/(events_e + nonevents_e)', '1/events_c - 1/(events_c + nonevents_c)'),
    'risk_diff': ('(events_e/(events_e+nonevents_e)) * (1 - events_e/(events_e+nonevents_e)) / (events_e+nonevents_e)',
                  '(events_c/(events_c+nonevents_c)) * (1 - events_c/(events_c+nonevents_c)) / (events_c+nonevents_c)'),
}


def _pipeline_vcv(raw, es_type):
    """Run Cells 5 + 6 on `raw` (keeps its 'id'); returns (analysis_data, vcv dict)."""
    g = globals()
    saved = {k: g.get(k) for k in ('ANALYSIS_CONFIG', 'data_filtered', '_populate_tabs')}
    cfg = {'effect_size_type': es_type, 'sd_missing_strategy': 'drop', 'sd_zero_strategy': 'drop',
           'prefilter_col': 'None', 'prefilter_values': [],
           'es_config': {'effect_col': 'yi_py', 'var_col': 'vi_py', 'se_col': 'sei_py',
                         'ci_lower_col': 'lo_py', 'ci_upper_col': 'hi_py',
                         'effect_label': es_type, 'effect_label_short': es_type}}
    try:
        g['ANALYSIS_CONFIG'] = cfg
        g['_populate_tabs'] = lambda *a, **k: None
        ok, res = run_processing(raw)
        if not ok:
            raise RuntimeError(res.get('error', 'run_processing failed') if isinstance(res, dict) else res)
        run_calculation()
        return cfg['analysis_data'].copy(), cfg['vcv_matrices']
    finally:
        for k, v in saved.items():
            if v is None:
                g.pop(k, None)
            else:
                g[k] = v


def _shared_dataset(kind, seed=16):
    """6 studies: 3 with 2-4 treatments sharing one control, 3 independent."""
    rng = np.random.default_rng(seed)
    rows = []
    for s in range(6):
        k = int(rng.integers(2, 5)) if s < 3 else 1
        if kind == 'cont':
            xc, sdc, nc = rng.uniform(8, 12), rng.uniform(1.5, 3), int(rng.integers(15, 40))
            for _ in range(k):
                rows.append(dict(id=f"S{s}", xe=xc * rng.uniform(0.7, 1.5), sde=rng.uniform(1.5, 3.5),
                                 ne=int(rng.integers(15, 40)), xc=xc, sdc=sdc, nc=nc))
        else:
            c, nc = int(rng.integers(5, 20)), int(rng.integers(40, 80))
            for _ in range(k):
                ne = int(rng.integers(40, 80)); a = int(rng.integers(5, ne - 5))
                rows.append(dict(id=f"S{s}", events_e=a, nonevents_e=ne - a, events_c=c, nonevents_c=nc - c))
    return pd.DataFrame(rows)


def _vcalc_check(es_type):
    kind = 'bin' if es_type in ('log_or', 'log_rr', 'risk_diff') else 'cont'
    raw = _shared_dataset(kind)
    py, vcv = _pipeline_vcv(raw, es_type)
    py = py.reset_index(drop=True)
    A = py.eval(_COMP[es_type][0]).values
    C = py.eval(_COMP[es_type][1]).values
    trt = [f"T{i}" for i in range(len(py))]
    ctl = [f"C_{s}" if pd.notna(g) else f"own_{i}" for i, (s, g) in enumerate(zip(py['id'], py['shared_group_id']))]
    ro.globalenv['vi_v'] = ro.FloatVector(py['vi_py'].values)
    ro.globalenv['st_v'], ro.globalenv['g1_v'], ro.globalenv['g2_v'] = (ro.StrVector(py['id'].astype(str).tolist()),
                                                                     ro.StrVector(trt), ro.StrVector(ctl))
    ro.globalenv['w1_v'], ro.globalenv['w2_v'] = ro.FloatVector(1.0 / A), ro.FloatVector(1.0 / C)
    V_R = np.asarray(ro.r("""suppressPackageStartupMessages(library(metafor))
        d <- data.frame(vi = as.numeric(vi_v), st = as.character(st_v), g1 = as.character(g1_v), g2 = as.character(g2_v))
        as.matrix(vcalc(vi, cluster = st, grp1 = g1, grp2 = g2, w1 = as.numeric(w1_v), w2 = as.numeric(w2_v),
                        data = d, sparse = FALSE))"""))
    worst = 0.0
    for s in pd.unique(py['id']):
        idx = np.where(py['id'].values == s)[0]
        worst = max(worst, float(np.max(np.abs(np.asarray(vcv[s]) - V_R[np.ix_(idx, idx)]))))
    n_shared = int(py['shared_group_id'].notna().sum())
    return worst, n_shared, len(py)


def _mc_check(es_type, seed=2016):
    rng = np.random.default_rng(seed)
    R = _MC_REPS
    if es_type in ('log_or', 'log_rr', 'risk_diff'):
        nc, pc = 300, 0.30
        ne, pe = np.array([280, 300, 320]), np.array([0.45, 0.20, 0.35])
        truth = pd.DataFrame({'id': 'S', 'events_e': pe * ne, 'nonevents_e': (1 - pe) * ne,
                              'events_c': pc * nc, 'nonevents_c': (1 - pc) * nc})
        c = rng.binomial(nc, pc, R)[:, None].astype(float)
        a = rng.binomial(ne, pe, (R, 3)).astype(float)
        b, d = ne - a, nc - c
        if es_type == 'log_or':
            y = np.log(a * d / (b * c))
        elif es_type == 'log_rr':
            y = np.log((a / ne) / (c / nc))
        else:
            y = a / ne - c / nc
    else:
        xc, sdc, nc = 10.0, 2.0, 300
        xe, ne = np.array([12.0, 9.0, 14.0]), np.array([280, 300, 320])
        # The SMD variance and covariance formulas assume equal SDs in all groups
        sde = np.full(3, sdc) if es_type in ('hedges_g', 'cohen_d') else np.array([2.5, 2.0, 3.0])
        truth = pd.DataFrame({'id': 'S', 'xe': xe, 'sde': sde, 'ne': ne, 'xc': xc, 'sdc': sdc, 'nc': nc})
        mc = rng.normal(xc, sdc / np.sqrt(nc), R)[:, None]
        sc = sdc * np.sqrt(rng.chisquare(nc - 1, R) / (nc - 1))[:, None]
        me = rng.normal(xe, sde / np.sqrt(ne), (R, 3))
        se = sde * np.sqrt(rng.chisquare(ne - 1, (R, 3)) / (ne - 1))
        if es_type == 'lnRR':
            y = np.log(me / mc)
        elif es_type == 'MD':
            y = me - mc
        elif es_type == 'lnCVR':
            y = (np.log(se / me) + 1 / (2 * (ne - 1))) - (np.log(sc / mc) + 1 / (2 * (nc - 1)))
        else:
            sp = np.sqrt(((ne - 1) * se ** 2 + (nc - 1) * sc ** 2) / (ne + nc - 2))
            y = (me - mc) / sp
            if es_type == 'hedges_g':
                y = y * np.array([_hedges_j(v) for v in ne + nc - 2])
    emp = np.cov(y, rowvar=False)
    _, vcv = _pipeline_vcv(truth, es_type)
    V = np.asarray(vcv['S'])
    iu = np.triu_indices(3, 1)
    rel_cov = float(np.max(np.abs(V[iu] - emp[iu]) / np.abs(emp[iu])))
    rel_var = float(np.max(np.abs(np.diag(V) - np.diag(emp)) / np.diag(emp)))
    info = ''
    if es_type in ('hedges_g', 'cohen_d'):
        # vcalc's default approximation (w = n): correlation sqrt(n_i n_j / ((n_i + nc)(n_j + nc)))
        r = np.sqrt(np.outer(ne, ne) / np.outer(ne + nc, ne + nc))
        Vv = r * np.sqrt(np.outer(np.diag(V), np.diag(V)))
        info = f"vcalc default (w = n) max rel. error of covariances: {np.max(np.abs(Vv[iu] - emp[iu]) / np.abs(emp[iu])):.1%}"
    return rel_cov, rel_var, info


_VCALC_TYPES = ('lnRR', 'MD', 'lnCVR', 'log_or', 'log_rr', 'risk_diff')


def _user_parts(d, es_type):
    """Treatment and control variance parts (A, C) of each row, from the
    columns Step 6 used; None when the columns are not in the data."""
    sde = 'sde_imputed' if 'sde_imputed' in d.columns else 'sde'
    sdc = 'sdc_imputed' if 'sdc_imputed' in d.columns else 'sdc'
    f = lambda c: pd.to_numeric(d[c], errors='coerce').to_numpy(dtype=float)
    try:
        if es_type == 'lnRR':
            return f(sde) ** 2 / (f('ne') * f('xe') ** 2), f(sdc) ** 2 / (f('nc') * f('xc') ** 2)
        if es_type == 'MD':
            return f(sde) ** 2 / f('ne'), f(sdc) ** 2 / f('nc')
        if es_type == 'lnCVR':
            return (f(sde) ** 2 / (f('ne') * f('xe') ** 2) + 1 / (2 * (f('ne') - 1)),
                    f(sdc) ** 2 / (f('nc') * f('xc') ** 2) + 1 / (2 * (f('nc') - 1)))
        a, b, c, e = f('events_e'), f('nonevents_e'), f('events_c'), f('nonevents_c')
        if es_type == 'log_or':
            return 1 / a + 1 / b, 1 / c + 1 / e
        if es_type == 'log_rr':
            return 1 / a - 1 / (a + b), 1 / c - 1 / (c + e)
        if es_type == 'risk_diff':
            pe, pc = a / (a + b), c / (c + e)
            return pe * (1 - pe) / (a + b), pc * (1 - pc) / (c + e)
    except KeyError:
        return None
    return None


def _vcalc_matrix(d, A, C, ctl):
    ro.globalenv['vi_v'] = ro.FloatVector(d['vi'].to_numpy(dtype=float))
    ro.globalenv['st_v'] = ro.StrVector(d['id'].astype(str).tolist())
    ro.globalenv['g1_v'] = ro.StrVector([f"T{i}" for i in range(len(d))])
    ro.globalenv['g2_v'] = ro.StrVector(ctl)
    ro.globalenv['w1_v'], ro.globalenv['w2_v'] = ro.FloatVector(1.0 / A), ro.FloatVector(1.0 / C)
    return np.asarray(ro.r("""suppressPackageStartupMessages(library(metafor))
        d <- data.frame(vi = as.numeric(vi_v), st = factor(as.character(st_v), levels = unique(as.character(st_v))),
                        g1 = as.character(g1_v), g2 = as.character(g2_v))
        as.matrix(vcalc(vi, cluster = st, grp1 = g1, grp2 = g2, w1 = as.numeric(w1_v), w2 = as.numeric(w2_v),
                        data = d, sparse = FALSE))"""))


def _user_vcv_rows(u):
    es = u['effect_size_type']
    d, V = validation_block_V(u['df'], u['vcv'])
    d = d.reset_index(drop=True)
    n = len(d)
    rows, notices = [], []
    off = V - np.diag(np.diag(V))
    dd = float(np.max(np.abs(np.diag(V) - d['vi'].to_numpy(dtype=float))))
    rows.append({'effect size': es, 'check': 'diagonal = your sampling variances',
                 'result': f"{n} effects, max |Δ| {dd:.1e}", 'status': 'PASS' if dd < TOL_VCV['vcalc'] else 'FAIL'})
    # shared-control groups (study + shared_group_id, at least 2 rows)
    sg = d['shared_group_id'] if 'shared_group_id' in d.columns else pd.Series([None] * n)
    key = np.array([f"{s}|{g}" if pd.notna(g) and g is not None else None for s, g in zip(d['id'], sg)], dtype=object)
    sizes = pd.Series([k for k in key if k is not None]).value_counts()
    groups = [k for k, c in sizes.items() if c >= 2]
    in_grp = np.isin(key, groups)
    same = np.equal.outer(key, key) & np.outer(in_grp, in_grp) & ~np.eye(n, dtype=bool)
    n_nonzero_off = int(np.count_nonzero(off) // 2)
    if not groups:
        notices.append("Your data have no shared-control groups (every effect size has its own control, "
                       "or the data are pre-calculated): the sampling VCV is diagonal and there are no "
                       "covariances to compare with vcalc. Turn on the reference datasets to check the "
                       "covariance formulas.")
        rows.append({'effect size': es, 'check': 'no shared controls: VCV is diagonal',
                     'result': f"{n_nonzero_off} non-zero off-diagonal elements",
                     'status': 'PASS' if n_nonzero_off == 0 else 'FAIL'})
        rows.append({'effect size': es, 'check': 'shared-control groups in your data', 'result': 'none', 'status': 'INFO'})
        return rows, notices
    rows.append({'effect size': es, 'check': 'shared-control groups in your data',
                 'result': f"{len(groups)} group(s), {int(in_grp.sum())} effects in "
                           f"{len({g.split('|')[0] for g in groups})} studies; {n_nonzero_off} covariances in V",
                 'status': 'INFO'})
    # covariances outside the shared groups must be zero
    stray = float(np.max(np.abs(off[~same]))) if (~same).any() else 0.0
    rows.append({'effect size': es, 'check': 'zero covariance outside shared-control groups',
                 'result': f"max |V_ij| {stray:.1e}", 'status': 'PASS' if stray == 0 else 'FAIL'})
    parts = _user_parts(d, es) if es in _VCALC_TYPES else None
    if es in _VCALC_TYPES and parts is not None and np.all(np.isfinite(parts[0][in_grp])) \
            and np.all(np.isfinite(parts[1][in_grp])):
        A, C = parts
        A = np.where(np.isfinite(A) & (A > 0), A, 1.0)
        C = np.where(np.isfinite(C) & (C > 0), C, 1.0)
        # groups whose rows report different control parts: first row's data by design
        bad = [g for g in groups if np.ptp(C[key == g]) > 1e-12 * np.max(C[key == g])]
        ok_mask = same & ~np.isin(key, bad)[:, None]
        ctl = [k if (k is not None and k in groups) else f"own_{i}" for i, k in enumerate(key)]
        V_R = _vcalc_matrix(d, A, C, ctl)
        dv = float(np.max(np.abs(V[ok_mask] - V_R[ok_mask]))) if ok_mask.any() else 0.0
        rows.append({'effect size': es, 'check': 'covariances vs metafor::vcalc (w1 = 1/A, w2 = 1/C)',
                     'result': f"{int(ok_mask.sum() // 2)} covariances compared, max |Δ| {dv:.1e}",
                     'status': 'PASS' if (dv < TOL_VCV['vcalc'] and ok_mask.any()) else
                               ('INFO' if not ok_mask.any() else 'FAIL')})
        r = in_grp & ~np.isin(key, bad)
        dsum = float(np.max(np.abs(d['vi'].to_numpy(dtype=float)[r] - (A + C)[r]) / d['vi'].to_numpy(dtype=float)[r])) if r.any() else 0.0
        rows.append({'effect size': es, 'check': 'variance = treatment part + control part (rows in groups)',
                     'result': f"max rel. |Δ| {dsum:.1e}", 'status': 'PASS' if dsum < 1e-10 else 'FAIL'})
        if bad:
            bm = same & np.isin(key, bad)[:, None]
            dvb = float(np.max(np.abs(V[bm] - V_R[bm])))
            rows.append({'effect size': es, 'check': 'groups with differing control data (not compared)',
                         'result': f"{len(bad)} group(s) ({', '.join(b.split('|')[0] for b in bad[:5])}): the rows "
                                   f"report different control data, e.g. a continuity correction on one arm; "
                                   f"CoMeta uses the first row's control (max |Δ| to vcalc {dvb:.1e})",
                         'status': 'INFO'})
    elif es in ('hedges_g', 'cohen_d') and {'ne', 'nc'}.issubset(d.columns):
        y = d['yi'].to_numpy(dtype=float)
        ne, nc = d['ne'].to_numpy(dtype=float), d['nc'].to_numpy(dtype=float)
        Ni = ne + nc
        E = 1.0 / nc[:, None] + np.outer(y, y) * nc[:, None] / (2.0 * np.outer(Ni, Ni))
        dv = float(np.max(np.abs(V[same] - E[same])))
        rows.append({'effect size': es, 'check': 'covariances vs analytic 1/nc + y_i y_j nc / (2 N_i N_j)',
                     'result': f"{int(same.sum() // 2)} covariances compared, max |Δ| {dv:.1e}",
                     'status': 'PASS' if dv < TOL_VCV['vcalc'] else 'FAIL'})
        # vcalc's default approximation for comparison
        ro.globalenv['vi_v'] = ro.FloatVector(d['vi'].to_numpy(dtype=float))
        ro.globalenv['st_v'] = ro.StrVector(d['id'].astype(str).tolist())
        ro.globalenv['g1_v'] = ro.StrVector([f"T{i}" for i in range(n)])
        ro.globalenv['g2_v'] = ro.StrVector([k if (k is not None and k in groups) else f"own_{i}" for i, k in enumerate(key)])
        ro.globalenv['w1_v'], ro.globalenv['w2_v'] = ro.FloatVector(ne), ro.FloatVector(nc)
        V_d = np.asarray(ro.r("""d <- data.frame(vi = as.numeric(vi_v), st = factor(as.character(st_v), levels = unique(as.character(st_v))),
                                                g1 = as.character(g1_v), g2 = as.character(g2_v))
            as.matrix(vcalc(vi, cluster = st, grp1 = g1, grp2 = g2, w1 = as.numeric(w1_v), w2 = as.numeric(w2_v),
                            data = d, sparse = FALSE))"""))
        rel = float(np.max(np.abs(V_d[same] - V[same]) / np.abs(V[same])))
        rows.append({'effect size': es, 'check': 'vcalc default approximation (w = n), for comparison',
                     'result': f"max rel. difference to CoMeta's covariances {rel:.1%}", 'status': 'INFO'})
    else:
        notices.append(f"Effect-size type '{es}' with the columns in your data: CoMeta does not compute "
                       f"shared-control covariances here (the VCV must be diagonal).")
        rows.append({'effect size': es, 'check': 'VCV is diagonal (no covariance formula for this type/data)',
                     'result': f"{n_nonzero_off} non-zero off-diagonal elements",
                     'status': 'PASS' if n_nonzero_off == 0 else 'FAIL'})
    return rows, notices


def _reference_rows():
    rows = []
    for es_type in ('lnRR', 'MD', 'log_or', 'log_rr', 'risk_diff'):
        try:
            worst, n_shared, n = _vcalc_check(es_type)
            rows.append({'data': 'reference', 'effect size': es_type, 'check': 'A. exact vs metafor::vcalc',
                         'result': f"{n} effects, {n_shared} in shared-control groups; max |Δ| element {worst:.1e}",
                         'status': 'PASS' if (worst < TOL_VCV['vcalc'] and n_shared > 0) else 'FAIL'})
        except Exception as e:
            rows.append({'data': 'reference', 'effect size': es_type, 'check': 'A. exact vs metafor::vcalc',
                         'status': f'ERROR: {e}'})
    for es_type in ('lnRR', 'MD', 'hedges_g', 'cohen_d', 'lnCVR', 'log_or', 'log_rr', 'risk_diff'):
        try:
            rel_cov, rel_var, info = _mc_check(es_type)
            ok = rel_cov < TOL_VCV['mc_rel'] and rel_var < TOL_VCV['mc_rel']
            rows.append({'data': 'reference', 'effect size': es_type, 'check': f'B. Monte Carlo ({_MC_REPS} replicates)',
                         'result': f"max rel. error: covariances {rel_cov:.1%}, variances {rel_var:.1%}"
                                   + (f" | INFO {info}" if info else ''),
                         'status': 'PASS' if ok else 'FAIL'})
        except Exception as e:
            rows.append({'data': 'reference', 'effect size': es_type, 'check': 'B. Monte Carlo', 'status': f'ERROR: {e}'})
    return rows


def _v24_main():
    """Main block, run in a function so its working variables do not leak
    into the notebook's shared namespace."""
    if 'run_calculation' not in globals() or 'build_vcv_matrices' not in globals():
        raise ValidationPrerequisiteError("Effect Size Calculation not found: run Steps 5-6 (Data Cleaning and "
                                          "Effect Size Calculation) first.")
    rows, notices = [], []
    try:
        u = validation_user_data()
        ur, notices = _user_vcv_rows(u)
        rows += [{'data': 'your data', **r} for r in ur]
    except ValidationPrerequisiteError:
        if not VALIDATION_INCLUDE_REFERENCE:
            raise
        notices.append("No effect sizes in this session: only the reference datasets are checked.")
    if VALIDATION_INCLUDE_REFERENCE:
        rows += _reference_rows()

    res_df = pd.DataFrame(rows)
    st = res_df['status'].astype(str)
    n_info = int((st == 'INFO').sum())
    n_fail = int(((st != 'PASS') & (st != 'INFO')).sum())
    n_graded = len(res_df) - n_info
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
        f"<b>{icon} Shared-control covariance matrices: {n_graded - n_fail}/{n_graded} checks passed"
        f"{f' ({n_info} INFO)' if n_info else ''}.</b><br>"
        f"<small>Your data: the VCV the analyses use vs vcalc / the analytic covariance (tolerance "
        f"{TOL_VCV['vcalc']:g})"
        + (f"; reference A: exact agreement with vcalc; B: analytic vs empirical covariance (tolerance "
           f"{TOL_VCV['mc_rel']:.0%}, n = 280-320 per group, fixed seed)" if VALIDATION_INCLUDE_REFERENCE else
           "; reference datasets switched off")
        + ".</small></div>"))
    for n in notices:
        display(HTML(f"<div style='padding:8px; background:#e2e3e5; border-radius:4px; font-family:sans-serif;"
                     f" margin-top:6px;'>ℹ️ {n}</div>"))
    display(res_df)
    return res_df, notices


res_df, _notes24 = _v24_main()
if res_df is not None and 'record_validation' in globals():
    record_validation(24, "Shared-Control Covariance Matrices", res_df, notes=' '.join(_notes24))


In [ ]:
#@title 🧪 Validation 25: Proportion of Meaningful Effects on Your Data (Python vs metafor + MetaUtility + boot)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Checks the engine of Step 9b (Cell 1b, section 9) on YOUR analysis: the
# overall model of Step 7 (rebuilt as Step 9b does: same rows, sampling V with
# your shared-control covariances, 2- or 3-level as in Step 7) and, if Step 9
# was run, the unified subgroup model; with your Step 9b thresholds, level
# (individual effects / study means) and alpha. If Step 9b was not run, its
# default threshold (±10% on ratio scales, ±0.2 SD for standardized mean
# differences, ±0.1 otherwise) is used on the overall model, and a notice says so.
# 1. Covariance of the variance components (var_components_vcov) vs
#    metafor::rma.mv(..., cvvc = "varcor")$vvc (observed information).
# 2. Parametric shares above / below your thresholds and their delta-method
#    SE and CI (meaningful_proportions) vs MetaUtility::prop_stronger(
#    estimate.method = ci.method = "parametric"), fed with metafor's mean,
#    the SE Step 9b uses (the reported Step 7 / Step 9 SE), T² and SE(T²)
#    from vvc, per target (overall, each subgroup).
# 3. Calibrated estimates and their shares vs the calibrated estimates
#    computed in R from metafor's fit (fitted + sqrt(T² / (T² + v)) ·
#    residual) and, with one effect per study, vs MetaUtility::calib_ests
#    and prop_stronger(estimate.method = "calibrated").
# 4. BCa interval (bca_interval) vs boot::boot.ci(type = "bca") on the SAME
#    bootstrap replicates of your overall model (studies resampled, model
#    refitted; parametric share of meaningful increases) and the same
#    leave-one-study-out jackknife values (passed as L).
# 5. When Step 9b results are in memory, they must equal a fresh run.
# Reference datasets (VALIDATION_INCLUDE_REFERENCE): dat.konstantopoulos2011,
# dat.curtis1998 lnRR, simulated shared controls with full V, and dat.bcg
# (log RR, one effect per study), at thresholds mu ± T/2.
# Requires: Cells 1-7, the Validation R setup cell; MetaUtility and boot in R.

import numpy as np
import pandas as pd
import rpy2.robjects as ro
from scipy.linalg import block_diag
from scipy.stats import norm as _v25_norm
from IPython.display import display, HTML

TOL_ME = {'vvc_rel': 2e-3, 'share': 1e-5, 'se': 1e-5, 'calib': 1e-6, 'bca': 1e-8}
_V25_BOOT = 200            # bootstrap replicates for the BCa check on your data (reference: 400)


def _rv(a):
    return ro.FloatVector(np.asarray(a, dtype=float).ravel())


def _rm(M):
    M = np.asarray(M, dtype=float)
    return ro.r['matrix'](ro.FloatVector(M.ravel(order='F')), nrow=M.shape[0], ncol=M.shape[1])


def _v25_has_pkg():
    return int(np.asarray(ro.r("as.integer(suppressWarnings(requireNamespace('MetaUtility', quietly = TRUE)))")).ravel()[0]) == 1


def _v25_have_metautility():
    ok = _v25_has_pkg()
    if not ok:
        try:
            ro.r("install.packages('MetaUtility', quiet = TRUE)")   # CRAN snapshot from the R setup cell
            ok = _v25_has_pkg()
        except Exception:
            ok = False
    if not ok:
        raise RuntimeError("The R package MetaUtility is not installed and could not be installed: run "
                           "install.packages('MetaUtility') in R (or re-run the Validation R setup cell).")


# ---------------------------------------------------------------------------
# YOUR MODELS, rebuilt exactly as Step 9b builds them
# ---------------------------------------------------------------------------
def _v25_overall_model(cfg):
    rep = (cfg.get('overall_results') or {}).get('reported')
    df0 = cfg.get('analysis_data')
    if not rep or df0 is None:
        raise ValidationPrerequisiteError("No overall model in this session: run Step 7 (Overall Meta-Analysis) first.")
    eff, var = cfg.get('effect_col'), cfg.get('var_col')
    df, _acc = select_rows(df0, eff, var, 'id')
    df = attach_vcv_row_pos(df, reference_df=df0)
    df = df.sort_values('id', kind='mergesort').reset_index(drop=True)
    groups = list(df.groupby('id', sort=False))
    y_b = [g[eff].to_numpy(dtype=np.float64) for _, g in groups]
    X_b = [np.ones((len(y), 1)) for y in y_b]
    gs = cfg.get('global_settings') or {}
    se_mu = float(rep.get('se', np.nan))
    if rep.get('model') == '3-Level':
        V_b = [cluster_vcv(cfg.get('vcv_matrices'), sid, g, var)[0] for sid, g in groups]
        fit = fit_blocks(y_b, V_b, two_level=False, cluster_ids=[sid for sid, _ in groups])

        def refit(yy, VV, XX):
            try:
                f = fit_blocks(yy, VV, XX, two_level=False, starts=[(fit.tau2, fit.sigma2)])
            except (ValueError, RuntimeError, np.linalg.LinAlgError):
                return None
            return f.betas, f.tau2, f.sigma2

        model = MeaningfulModel(y_b, V_b, X_b, fit.betas, fit.tau2, fit.sigma2, {'Overall': np.array([1.0])},
                                {'Overall': se_mu}, var_components_vcov(fit), refit, 'three-level (Step 7)')
        return model, {'label': 'overall three-level model (Step 7)', 'random': 'cl/es', 'method': 'REML',
                       'mu_rep': float(rep.get('mu', np.nan))}
    method = str(gs.get('tau_method', 'REML')).upper()
    if method not in ('REML', 'ML'):
        raise ValidationPrerequisiteError(
            f"Step 7 used the two-level model with tau² by {method}; Step 9b then has no delta-method CI and "
            f"MetaUtility's parametric check needs a likelihood fit: run Step 7 with REML or ML first to check it.")
    V_b = [np.diag(g[var].to_numpy(dtype=np.float64)) for _, g in groups]

    def pooled(yy, VV):
        y = np.concatenate(yy)
        v = np.concatenate([np.diag(V) for V in VV])
        t2 = max(0.0, tau2_univariate(y, v, method))
        w = 1.0 / (v + t2)
        return np.array([float(np.sum(w * y) / np.sum(w))]), t2

    def refit(yy, VV, XX):
        try:
            b, t2 = pooled(yy, VV)
        except (ValueError, RuntimeError, np.linalg.LinAlgError, ZeroDivisionError):
            return None
        return b, t2, 0.0

    betas, t2 = pooled(y_b, V_b)
    rows_y = [np.array([y]) for y in np.concatenate(y_b)]
    rows_V = [np.array([[v]]) for V in V_b for v in np.diag(V)]
    vc = var_components_vcov(fit_blocks(rows_y, rows_V, two_level=True, method=method))
    model = MeaningfulModel(y_b, V_b, X_b, betas, t2, 0.0, {'Overall': np.array([1.0])}, {'Overall': se_mu},
                            vc, refit, f'two-level, τ² by {method} (Step 7)')
    return model, {'label': f'overall two-level model, τ² by {method} (Step 7)', 'random': 'es', 'method': method,
                   'mu_rep': float(rep.get('mu', np.nan))}


def _v25_subgroup_model(cfg):
    sg = cfg.get('subgroup_results')
    if not sg:
        return None, None
    mf = sg.get('model_fit') or {}
    fit = mf.get('fit')
    if fit is None:
        return None, "The Step 9 results come from separate per-group fits (fallback): subgroups not checked."
    res = sg['results_df']
    levels, L = list(mf['levels']), np.asarray(mf['L'])
    se = {str(r['group']): float(r['pooled_se_re']) for _, r in res.iterrows()}
    two_level = fit.sigma2_fixed is not None and fit.sigma2_fixed == 0

    def refit(yy, VV, XX):
        try:
            f = fit_blocks(yy, VV, XX, two_level=two_level, starts=[(fit.tau2, fit.sigma2)])
        except (ValueError, RuntimeError, np.linalg.LinAlgError):
            return None
        return f.betas, f.tau2, f.sigma2

    s = fit.spec
    model = MeaningfulModel(list(s.y_blocks), list(s.V_blocks), list(s.X_blocks), fit.betas, fit.tau2, fit.sigma2,
                            {str(lv): L[i] for i, lv in enumerate(levels)},
                            {str(lv): se.get(str(lv), np.nan) for lv in levels},
                            var_components_vcov(fit), refit, 'subgroup model (Step 9)')
    mods = sg.get('moderator1') + (f" × {sg.get('moderator2')}" if sg.get('moderator2') else '')
    return model, {'label': f"subgroup model by {mods} (Step 9)", 'random': 'cl' if two_level else 'cl/es',
                   'method': 'REML', 'mu_rep': np.nan}


def _v25_default_thresholds(cfg):
    es = cfg.get('es_config') or {}
    es_type = str(cfg.get('effect_size_type') or '')
    null = es_null_value()
    if es.get('has_fold_change'):
        return null + float(np.log1p(-0.10)), null + float(np.log1p(0.10)), "±10% (ratio scale)"
    if es_type in ('hedges_g', 'cohen_d'):
        return null - 0.2, null + 0.2, "±0.2 SD"
    return null - 0.1, null + 0.1, "±0.1"


# ---------------------------------------------------------------------------
# R side
# ---------------------------------------------------------------------------
def _v25_r_model(model, info, q_lo, q_hi, level, alpha):
    """metafor fit (+ vvc) on the model's rows and V; MetaUtility parametric
    shares per target with the SE Step 9b uses; calibrated estimates from
    metafor's fit; MetaUtility calib_ests when every study has one effect."""
    sizes = [len(y) for y in model.y_blocks]
    X = np.vstack(model.X_blocks)
    ro.globalenv['y_v'] = _rv(np.concatenate(model.y_blocks))
    ro.globalenv['vi_v'] = _rv(np.concatenate([np.diag(V) for V in model.V_blocks]))
    ro.globalenv['V_m'], ro.globalenv['X_m'] = _rm(block_diag(*model.V_blocks)), _rm(X)
    ro.globalenv['cl_v'] = ro.IntVector(np.repeat(np.arange(len(sizes)), sizes).tolist())
    labels = list(model.targets)
    C = np.vstack([np.asarray(model.targets[k], float) for k in labels])
    ro.globalenv['C_m'] = _rm(C)
    ro.globalenv['seM_v'] = _rv([model.se_mu.get(k, np.nan) for k in labels])
    one_each = all(k == 1 for k in sizes)
    out = ro.r(f"""
    suppressPackageStartupMessages({{library(metafor); library(MetaUtility)}})
    d <- data.frame(yi = as.numeric(y_v), vi = as.numeric(vi_v), cl = factor(as.integer(cl_v)))
    d$es <- factor(seq_len(nrow(d)))
    f <- rma.mv(yi, V_m, mods = ~ 0 + X_m, random = ~ 1 | {info['random']}, data = d,
                method = "{info['method']}", cvvc = "varcor")
    s2 <- f$sigma2; vvc <- as.matrix(f$vvc)
    g <- if ("{level}" == "study" && length(s2) == 2) c(1, 0) else rep(1, length(s2))
    T2 <- sum(g * s2); seT2 <- sqrt(max(0, as.numeric(t(g) %*% vvc %*% g)))
    M <- as.numeric(C_m %*% f$b)
    one <- function(m, se, q, tl) {{
        r <- unlist(suppressMessages(suppressWarnings(prop_stronger(q = q, M = m, t2 = T2,
             se.M = se, se.t2 = seT2, ci.level = {1 - alpha}, tail = tl, estimate.method = "parametric",
             ci.method = "parametric", bootstrap = "never")))[c("est", "se", "lo", "hi")])
        # MetaUtility withholds the delta-method SE when the share is < 0.15 or > 0.85;
        # then use its formula (prop_stronger's own code) on metafor's estimates
        fb <- 0
        if (!is.finite(r[2]) && T2 > 0) {{
            z <- (q - m) / sqrt(T2)
            r[2] <- sqrt(se^2 / T2 + seT2^2 * (q - m)^2 / (4 * T2^3)) * dnorm(z)
            r[3] <- max(0, r[1] - qnorm(1 - {alpha} / 2) * r[2]); r[4] <- min(1, r[1] + qnorm(1 - {alpha} / 2) * r[2])
            fb <- 1
        }}
        c(r, fb)
    }}
    up <- t(sapply(seq_along(M), function(i) one(M[i], seM_v[i], {q_hi}, "above")))
    dn <- t(sapply(seq_along(M), function(i) one(M[i], seM_v[i], {q_lo}, "below")))
    fitted <- as.numeric(X_m %*% f$b)
    cal <- fitted + sqrt(T2 / (T2 + d$vi)) * (d$yi - fitted)
    mu_ <- NA; cal_mu <- NA; pc <- NA
    if ({'TRUE' if one_each else 'FALSE'} && ncol(X_m) == 1) {{
        cal_mu <- calib_ests(d$yi, sqrt(d$vi), method = "{info['method']}")
        pc <- c(suppressMessages(suppressWarnings(prop_stronger(q = {q_hi}, tail = "above",
                    estimate.method = "calibrated", ci.method = "calibrated", calib.est.method = "{info['method']}",
                    dat = d, R = 50)))$est,
                suppressMessages(suppressWarnings(prop_stronger(q = {q_lo}, tail = "below",
                    estimate.method = "calibrated", ci.method = "calibrated", calib.est.method = "{info['method']}",
                    dat = d, R = 50)))$est)
    }}
    list(M = M, s2 = s2, vvc = vvc, T2 = T2, up = up, dn = dn, cal = cal, cal_mu = cal_mu, pc = pc)
    """)
    r = {k: np.asarray(out.rx2(k), dtype=float) for k in ('M', 's2', 'vvc', 'T2', 'up', 'dn', 'cal', 'cal_mu', 'pc')}
    r['up'], r['dn'] = r['up'].reshape(len(labels), 5), r['dn'].reshape(len(labels), 5)
    r['labels'], r['one_each'] = labels, one_each
    return r


def _r_bca(t0, tb, jack, alpha):
    """boot::boot.ci(type = 'bca') on given replicates, acceleration from jackknife values L."""
    ro.globalenv['t0_v'], ro.globalenv['tb_v'] = ro.FloatVector([t0]), _rv(tb)
    L = (len(jack) - 1) * (np.mean(jack) - np.asarray(jack))
    ro.globalenv['L_v'] = _rv(L)
    return np.asarray(ro.r(f"""
    suppressPackageStartupMessages(library(boot))
    n <- length(L_v)
    bo <- structure(list(t0 = as.numeric(t0_v), t = matrix(as.numeric(tb_v), ncol = 1), R = length(tb_v),
                         data = matrix(0, n, 1), statistic = function(d, i) 0, sim = "ordinary", stype = "i",
                         call = quote(boot(data = d, statistic = s, R = r)), strata = rep(1, n),
                         weights = rep(1 / n, n)), class = "boot")
    as.numeric(boot.ci(bo, conf = {1 - alpha}, type = "bca", L = as.numeric(L_v))$bca[4:5])
    """))


# ---------------------------------------------------------------------------
# CHECKS on one model (user's or reference)
# ---------------------------------------------------------------------------
def _v25_check_model(name, model, info, q_lo, q_hi, level, alpha, seed, stored=None, do_bca=True,
                     n_boot=_V25_BOOT):
    rows = []
    tab = meaningful_proportions(model, (q_lo, q_hi), level=level, alpha=alpha, n_boot=0)
    r = _v25_r_model(model, info, q_lo, q_hi, level, alpha)
    multi = len(r['s2']) == 2
    # 1. vvc
    vc = np.asarray(model.vc_vcov, float)
    vvc_p = vc if multi else vc[:1, :1]
    d_v = float(np.max(np.abs(vvc_p - r['vvc'])) / max(np.max(np.abs(r['vvc'])), 1e-300))
    rows.append({'dataset': name, 'check': 'covariance of the variance components (vvc)', 'max |Δ|': d_v,
                 'status': 'PASS' if d_v < TOL_ME['vvc_rel'] else 'FAIL', 'note': 'relative to max |vvc|'})
    # 2. parametric shares, delta SE and CI per target
    par = tab[tab['method'] == 'parametric'].set_index('target')
    d_p = d_s = d_c = d_m = 0.0
    n_fb = int(r['up'][:, 4].sum() + r['dn'][:, 4].sum())
    for i, lab in enumerate(r['labels']):
        row = par.loc[lab]
        d_m = max(d_m, abs(row['mu'] - r['M'][i]))
        for key, rr in (('increase', r['up'][i]), ('decrease', r['dn'][i])):
            d_p = max(d_p, abs(row[f'p_{key}'] - rr[0]))
            if np.isfinite(rr[1]):
                d_s = max(d_s, abs(row[f'se_{key}_delta'] - rr[1]))
                d_c = max(d_c, abs(row[f'{key}_lo_delta'] - rr[2]), abs(row[f'{key}_hi_delta'] - rr[3]))
            elif np.isfinite(row[f'se_{key}_delta']):
                d_s = np.inf
    n_t = len(r['labels'])
    rows.append({'dataset': name, 'check': f'parametric shares, delta-method SE and CI ({n_t} target'
                                           f'{"s" if n_t > 1 else ""} × 2 thresholds)',
                 'max |Δ|': max(d_p, d_s, d_c),
                 'status': 'PASS' if (d_p < TOL_ME['share'] and d_s < TOL_ME['se'] and d_c < TOL_ME['se'])
                 else 'FAIL', 'note': f'thresholds {q_lo:.4g} / {q_hi:.4g}, {level} level; |Δ mean| = {d_m:.1e}'
                                      + (f'; {n_fb} share(s) < 0.15 or > 0.85, where MetaUtility gives no SE: SE '
                                         f'from its formula on metafor\'s estimates' if n_fb else '')})
    # 3. calibrated estimates and their shares (individual effect sizes only)
    if level == 'effect':
        t2, _ = true_effect_variance(model.tau2, model.sigma2, level)
        y = np.concatenate(model.y_blocks)
        v = np.concatenate([np.diag(V) for V in model.V_blocks])
        fitted = np.vstack(model.X_blocks) @ np.asarray(model.betas)
        cal_p = calibrated_effects(y, v, fitted, t2)
        d_c = float(np.max(np.abs(cal_p - r['cal'])))
        rt = np.concatenate(model.row_targets(model.X_blocks))
        cal = tab[tab['method'] == 'calibrated'].set_index('target')
        d_pc = 0.0
        for lab in r['labels']:
            sel = rt == lab
            d_pc = max(d_pc, abs(cal.loc[lab, 'p_increase'] - np.mean(r['cal'][sel] > q_hi)),
                       abs(cal.loc[lab, 'p_decrease'] - np.mean(r['cal'][sel] < q_lo)))
        note = 'vs the calibrated estimates computed from metafor\'s fit'
        if r['one_each'] and np.all(np.isfinite(r['cal_mu'])) and r['cal_mu'].size == cal_p.size:
            d_c = max(d_c, float(np.max(np.abs(cal_p - r['cal_mu']))))
            d_pc = max(d_pc, abs(cal.loc['Overall', 'p_increase'] - r['pc'][0]),
                       abs(cal.loc['Overall', 'p_decrease'] - r['pc'][1]))
            note += ' and MetaUtility calib_ests / prop_stronger (one effect per study)'
        rows.append({'dataset': name, 'check': 'calibrated estimates and their shares', 'max |Δ|': max(d_c, d_pc),
                     'status': 'PASS' if d_c < TOL_ME['calib'] and d_pc < TOL_ME['calib'] else 'FAIL', 'note': note})
    # 4. BCa vs boot::boot.ci on the same replicates (parametric share of meaningful increases, first target)
    if do_bca:
        lab0 = r['labels'][0]
        c0 = np.asarray(model.targets[lab0], float)
        y_b, V_b, X_b = model.y_blocks, model.V_blocks, model.X_blocks
        M_ = len(y_b)
        rng = np.random.default_rng(seed)

        def share(idx):
            f = model.refit([y_b[i] for i in idx], [V_b[i] for i in idx], [X_b[i] for i in idx])
            if f is None:
                return np.nan
            b, tt, ss = f
            t2_, _ = true_effect_variance(tt, ss, level)
            return prop_beyond(float(c0 @ np.asarray(b)), t2_, q_hi, 'above')

        est = float(par.loc[lab0, 'p_increase'])
        tb = np.array([share(rng.integers(0, M_, M_)) for _ in range(n_boot)])
        tb = tb[np.isfinite(tb)]
        jk = np.array([share(np.delete(np.arange(M_), j)) for j in range(M_)])
        lo_p, hi_p, how = bca_interval(est, tb, jk, alpha)
        if how == 'BCa':
            lo_r, hi_r = _r_bca(est, tb, jk, alpha)
            d_b = float(max(abs(lo_p - lo_r), abs(hi_p - hi_r)))
            rows.append({'dataset': name, 'check': f'BCa CI vs boot.ci, same {tb.size} cluster resamples ({lab0})',
                         'max |Δ|': d_b, 'status': 'PASS' if d_b < TOL_ME['bca'] else 'FAIL',
                         'note': f'[{lo_p:.4f}, {hi_p:.4f}] vs [{lo_r:.4f}, {hi_r:.4f}]'})
        else:
            rows.append({'dataset': name, 'check': f'BCa CI vs boot.ci ({lab0})', 'status': 'INFO',
                         'note': f'no BCa interval for this share ({how}, estimate {est:.3g}): not compared'})
    # 5. Step 9b results in memory = fresh run
    if stored is not None:
        st = stored[stored['target'].astype(str).isin(r['labels'])].set_index(['target', 'method'])
        fr = tab.set_index(['target', 'method'])
        cols = [c for c in ('mu', 'T2', 'p_increase', 'p_decrease', 'p_negligible', 'se_increase_delta',
                            'se_decrease_delta', 'increase_lo_delta', 'increase_hi_delta', 'decrease_lo_delta',
                            'decrease_hi_delta') if c in st.columns and c in fr.columns]
        same = set(st.index) == set(fr.index)
        dd = (float(np.nanmax(np.abs(st.loc[fr.index, cols].to_numpy(float) - fr[cols].to_numpy(float))))
              if same and cols else np.inf)
        rows.append({'dataset': name, 'check': 'Step 9b results in memory = fresh run', 'max |Δ|': dd,
                     'status': 'PASS' if dd < 1e-10 else 'FAIL',
                     'note': '' if dd < 1e-10 else 'Step 9b results differ from a fresh run: if you changed the data, '
                                                   'Step 7 or Step 9 after Step 9b, re-run Step 9b.'})
    return rows


# ---------------------------------------------------------------------------
# REFERENCE datasets (only when the switch is on)
# ---------------------------------------------------------------------------
def _v25_reference_models():
    ro.r("suppressPackageStartupMessages({library(metafor); library(metadat)})")
    k = {c: np.asarray(ro.r(f"as.numeric(dat.konstantopoulos2011${c})")) for c in ('district', 'yi', 'vi')}
    out = [('reference: dat.konstantopoulos2011',
            pd.DataFrame({'id': k['district'].astype(int).astype(str), 'yi': k['yi'], 'vi': k['vi']}), {})]
    cu = ro.r("""e <- escalc("ROM", m1i = m1i, sd1i = sd1i, n1i = n1i, m2i = m2i, sd2i = sd2i, n2i = n2i,
                         data = dat.curtis1998); list(id = as.character(e$paper), yi = as.numeric(e$yi), vi = as.numeric(e$vi))""")
    out.append(('reference: dat.curtis1998 (lnRR)', pd.DataFrame({'id': list(cu.rx2('id')), 'yi': np.asarray(cu.rx2('yi')),
                                                                 'vi': np.asarray(cu.rx2('vi'))}), {}))
    rng = np.random.default_rng(48)
    rows, vcv = [], {}
    for i in range(20):
        kk = int(rng.integers(1, 5))
        V = np.full((kk, kk), rng.uniform(.01, .06)) + np.diag(rng.uniform(.02, .1, kk))
        y = 0.3 + rng.normal(0, .2) + rng.normal(0, .15, kk) + rng.multivariate_normal(np.zeros(kk), V)
        vcv[f"S{i:02d}"] = V
        rows += [dict(id=f"S{i:02d}", yi=y[j], vi=V[j, j]) for j in range(kk)]
    out.append(('reference: simulated shared controls (full V)', pd.DataFrame(rows), vcv))
    b = ro.r("""e <- escalc("RR", ai = tpos, bi = tneg, ci = cpos, di = cneg, data = dat.bcg)
                list(id = as.character(e$trial), yi = as.numeric(e$yi), vi = as.numeric(e$vi))""")
    out.append(('reference: dat.bcg (log RR, one effect per study)',
                pd.DataFrame({'id': list(b.rx2('id')), 'yi': np.asarray(b.rx2('yi')), 'vi': np.asarray(b.rx2('vi'))}), {}))
    models = []
    for name, df, vcv in out:
        ids = list(pd.unique(df['id']))
        y_b = [df.loc[df['id'] == s, 'yi'].to_numpy(dtype=float) for s in ids]
        V_b = [np.asarray(vcv[s]) if s in vcv else np.diag(df.loc[df['id'] == s, 'vi'].to_numpy(dtype=float))
               for s in ids]
        X_b = [np.ones((len(y), 1)) for y in y_b]
        multi = any(len(y) > 1 for y in y_b)
        fit = fit_blocks(y_b, V_b, two_level=not multi)

        def refit(yy, VV, XX, multi=multi):
            try:
                f = fit_blocks(yy, VV, two_level=not multi)
            except (ValueError, RuntimeError, np.linalg.LinAlgError):
                return None
            return f.betas, f.tau2, f.sigma2

        se_mu = float(np.sqrt(fit.cov_beta[0, 0]))
        m = MeaningfulModel(y_b, V_b, X_b, fit.betas, fit.tau2, fit.sigma2, {'Overall': np.array([1.0])},
                            {'Overall': se_mu}, var_components_vcov(fit), refit, name)
        T = np.sqrt(fit.tau2 + fit.sigma2)
        mu = float(fit.betas[0])
        models.append((name, m, {'random': 'cl/es' if multi else 'cl', 'method': 'REML'}, mu - T / 2, mu + T / 2))
    return models


def _v25_main():
    for fn in ('var_components_vcov', 'meaningful_proportions', 'MeaningfulModel', 'bca_interval'):
        if fn not in globals():
            raise ValidationPrerequisiteError(f"{fn} is not defined: run Cells 1 and 1b first.")
    _v25_have_metautility()
    rows, notes = [], []
    user_ok = True
    try:
        u = validation_user_data()
        cfg = u['cfg']
        overall, oinfo = _v25_overall_model(cfg)
    except ValidationPrerequisiteError:
        if not VALIDATION_INCLUDE_REFERENCE:
            raise
        user_ok = False
        notes.append("No overall model (Step 7) in this session: only the reference datasets were checked.")
    if user_ok:
        gs = u['settings'] or {}
        mr = cfg.get('meaningful_results')
        stored = None
        if mr and isinstance(mr.get('settings'), dict) and np.isfinite(mr['settings'].get('q_lo', np.nan)):
            st = mr['settings']
            q_lo, q_hi = float(st['q_lo']), float(st['q_hi'])
            level = str(st.get('level', 'effect'))
            alpha = float(st.get('alpha', gs.get('alpha', 0.05)))
            seed = int(st.get('seed', 2024))
            stored = mr.get('table') if isinstance(mr.get('table'), pd.DataFrame) else None
            src = f"your Step 9b thresholds ({st.get('inc_short', q_hi)} / {st.get('dec_short', q_lo)}), {level} level"
            sub, sinfo = _v25_subgroup_model(cfg)
            if stored is not None and sub is not None and not set(sub.targets) <= set(stored['target'].astype(str)):
                notes.append("Step 9 was re-run after Step 9b: the subgroup shares were checked, but not compared "
                             "with the Step 9b results in memory.")
        else:
            q_lo, q_hi, how = _v25_default_thresholds(cfg)
            level, alpha, seed = 'effect', float(gs.get('alpha', 0.05)), 2024
            sub, sinfo = _v25_subgroup_model(cfg)
            src = f"Step 9b default threshold {how}, individual effect sizes"
            notes.append(f"Step 9b was not run: its default threshold ({how}: {q_lo:.4g} / {q_hi:.4g}) was used "
                         f"on your overall model{' and subgroups' if sub is not None else ''}. Run Step 9b to "
                         f"check your own thresholds.")
        if oinfo['random'] == 'es' and level == 'study':
            level = 'effect'
        rows += _v25_check_model(f"your data: {oinfo['label']}", overall, oinfo, q_lo, q_hi, level, alpha, seed,
                                 stored=stored)
        if np.isfinite(oinfo['mu_rep']):
            d_mu = abs(float(overall.betas[0]) - oinfo['mu_rep'])
            rows.append({'dataset': f"your data: {oinfo['label']}", 'check': 'rebuilt model mean = Step 7 mean',
                         'max |Δ|': d_mu, 'status': 'PASS' if d_mu < 1e-6 * max(1.0, abs(oinfo['mu_rep'])) else 'FAIL',
                         'note': '' if d_mu < 1e-6 else 're-run Step 7 (its results are stale)'})
        if sub is not None:
            st_sub = stored if stored is not None and set(sub.targets) <= set(stored['target'].astype(str)) else None
            rows += _v25_check_model(f"your data: {sinfo['label']}", sub, sinfo, q_lo, q_hi, level, alpha, seed,
                                     stored=st_sub, do_bca=False)
        elif sinfo:
            notes.append(sinfo)
        notes.insert(0, f"Settings checked: {src}, {100 * (1 - alpha):g}% CIs.")
    if VALIDATION_INCLUDE_REFERENCE:
        for name, m, info, q_lo, q_hi in _v25_reference_models():
            try:
                rows += _v25_check_model(name, m, info, q_lo, q_hi, 'effect', 0.05, 25, n_boot=400)
            except Exception as e:
                rows.append({'dataset': name, 'check': '—', 'status': 'FAIL', 'note': f'ERROR: {e}'})
    for n in notes:
        rows.append({'dataset': 'your data', 'check': 'notice', 'status': 'INFO', 'note': n})

    res_df = pd.DataFrame(rows)
    n_info = int((res_df['status'] == 'INFO').sum())
    n_fail = int((~res_df['status'].isin(['PASS', 'INFO'])).sum())
    n_chk = len(res_df) - n_info
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
        f"<b>{icon} Proportion of meaningful effects vs metafor + MetaUtility + boot: "
        f"{n_chk - n_fail}/{n_chk} checks passed.</b><br><small>Tolerances: {TOL_ME}.</small>"
        + ''.join(f"<br>ℹ️ {n}" for n in notes) + "</div>"))
    display(res_df.style.format({'max |Δ|': '{:.1e}'}, na_rep=''))
    return res_df


res_df = _v25_main()
if res_df is not None and 'record_validation' in globals():
    record_validation(25, "Proportion of Meaningful Effects on Your Data", res_df)


In [ ]:
#@title 🧪 Validation 26: Time-Lag Test on your data (Python vs metafor + clubSandwich)
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Checks fit_timelag_model (Step 19b) against R. For each dataset and each
# precision term (none, SE, sqrt(1/n~) when group sizes are available) R
# fits, on the same rows and the same sampling V,
#     rma.mv(yi, V, mods = ~ year_c [+ precision], random = ~ 1 | study/es)
#     coef_test(vcov = "CR2", test = "Satterthwaite")      (CR2 on)
#     model-based SE, df = studies - coefficients          (CR2 off)
# and every coefficient's estimate, SE, df, p-value and CI (1 - alpha) are
# compared.
#
# Your data: the Time-Lag cell's call on your ANALYSIS_CONFIG: your effect
# sizes and year column, your sampling VCV (shared-control covariances of
# Step 6; R gets the same V, sliced by row identity for the rows kept), and
# the settings of Step 7 the cell uses (CR2 on/off, alpha). Effect sizes
# without a year are dropped by Python and R alike. Without a 'year' column
# the validation is 'not run' (unless the reference datasets are on).
# Reference datasets (switch VALIDATION_INCLUDE_REFERENCE in the Validation
# R setup cell): dat.konstantopoulos2011 (SE and year only) and a simulated
# nested dataset with shared-control covariances, group sizes and a few
# effect sizes without a year inside shared-control studies (Python must
# drop them and slice the VCV accordingly); CR2 on, 95%.
# Requires: Steps 2-7, the Time-Lag Test cell (19b) and the Validation R setup cell.

import numpy as np
import pandas as pd
import rpy2.robjects as ro
from IPython.display import display, HTML

TOL_TL = {'est': 1e-5, 'se_rel': 1e-4, 'df_rel': 1e-3, 'p': 1e-4, 'ci': 1e-4}


def _tl_reference():
    """Fixed reference datasets: [(name, df, vcv)]; only built when the switch is on."""
    out = []
    ro.r("suppressPackageStartupMessages(library(metadat))")
    kd = {c: np.asarray(ro.r(f"as.numeric(dat.konstantopoulos2011${c})")) for c in ('district', 'year', 'yi', 'vi')}
    out.append(('metadat: konstantopoulos2011',
                pd.DataFrame({'id': kd['district'].astype(int).astype(str), 'yi': kd['yi'], 'vi': kd['vi'],
                              'year': kd['year']}), {}))
    rng = np.random.default_rng(26)
    rows, vcv = [], {}
    for i in range(26):
        k = int(rng.integers(1, 5)); year = int(rng.integers(1990, 2022))
        V = np.full((k, k), rng.uniform(.01, .03)) + np.diag(rng.uniform(.02, .1, k))
        y = 0.5 - 0.015 * (year - 2005) + rng.normal(0, .15) + rng.normal(0, .1, k) \
            + rng.multivariate_normal(np.zeros(k), V)
        sid = f"S{i:02d}"; vcv[sid] = V
        ne, nc = rng.integers(5, 40, k), rng.integers(5, 40, k)
        rows += [dict(id=sid, yi=y[j], vi=V[j, j], year=year, ne=ne[j], nc=nc[j]) for j in range(k)]
    sim = pd.DataFrame(rows)
    # a few effect sizes without a year, inside studies with shared controls:
    # Python must drop them and slice the shared-control VCV accordingly
    multi = sim[sim.duplicated('id', keep=False)].index
    sim.loc[multi[[1, 5, 9]], 'year'] = np.nan
    sim = attach_vcv_row_pos(sim, reference_df=sim)     # row identity for the R side's V
    out.append(('simulated: shared-control VCV, missing years', sim, vcv))
    return out


def _tl_r(res, rows_df, vcv, use_cr2, alpha):
    """R fit on exactly the rows Python used, with their sampling V
    (validation_block_V: same row identity, shared-control covariances)."""
    d, V = validation_block_V(rows_df, vcv)
    ro.globalenv['yi_v'] = ro.FloatVector(d['yi'].to_numpy(dtype=float))
    ro.globalenv['yr_v'] = ro.FloatVector((pd.to_numeric(d['year']) - res['year_mean']).to_numpy(dtype=float))
    ro.globalenv['cl_v'] = ro.StrVector(d['id'].astype(str).tolist())
    ro.globalenv['V_m'] = ro.r.matrix(ro.FloatVector(V.ravel(order='F')), nrow=V.shape[0])
    mods = "~ yr_v"
    if res['precision'] != 'none':
        ro.globalenv['pr_v'] = ro.FloatVector(timelag_precision(d, res['precision'], 'vi'))
        mods = "~ yr_v + pr_v"
    out = ro.r(f"""
    suppressPackageStartupMessages({{library(metafor); library(clubSandwich)}})
    cl <- factor(as.character(cl_v), levels = unique(as.character(cl_v)))
    es <- factor(seq_along(yi_v))
    fitf <- function(ctl) rma.mv(as.numeric(yi_v), V_m, mods = {mods}, random = ~ 1 | cl/es,
                                 method = "REML", control = ctl)
    f <- tryCatch(fitf(list(rel.tol = 1e-10, iter.max = 1000, eval.max = 2000)),
                  error = function(e) fitf(list(optimizer = "optim", optmethod = "BFGS", reltol = 1e-12)))
    if ({'TRUE' if use_cr2 else 'FALSE'}) {{
        ct <- coef_test(f, vcov = "CR2", cluster = cl, test = "Satterthwaite")
        b <- as.numeric(f$b); se <- ct$SE; dfs <- ct$df_Satt; p <- ct$p_Satt
    }} else {{
        b <- as.numeric(f$b); se <- as.numeric(f$se)
        dfs <- rep(max(1, nlevels(cl) - length(b)), length(b))
        p <- 2 * pt(abs(b / se), dfs, lower.tail = FALSE)
    }}
    h <- qt(1 - {alpha!r} / 2, dfs) * se
    cbind(b, se, dfs, p, b - h, b + h)
    """)
    return np.asarray(out, dtype=float)


def _v26_main():
    if 'fit_timelag_model' not in globals():
        raise ValidationPrerequisiteError("Time-lag test not found: run the Time-Lag Test cell (19b) first.")
    notes, u = [], None
    try:
        u = validation_user_data(need_year=True)
    except ValidationPrerequisiteError as e:
        if not VALIDATION_INCLUDE_REFERENCE:
            raise
        notes.append(f"Your data were not checked: {e} Only the reference datasets were run.")
    sources = validation_sources(_tl_reference, need_year=True) if u is not None else _tl_reference()

    rows = []
    for name, df, vcv in sources:
        is_user = u is not None and name == 'your data'
        if is_user:
            # the Time-Lag cell's call (TimeLagModel.run) on your ANALYSIS_CONFIG
            cfg, gs = u['cfg'], u['settings']
            use_cr2, alpha = bool(gs.get('use_kh', True)), float(gs.get('alpha', 0.05))
            data, ecol, vcol, ref_df = cfg['analysis_data'], cfg['effect_col'], cfg['var_col'], cfg['analysis_data']
        else:
            use_cr2, alpha = True, 0.05
            data, ecol, vcol, ref_df = df, 'yi', 'vi', None
        precs = ['none', 'se'] + (['ess'] if {'ne', 'nc'} <= set(df.columns) else [])
        if is_user and 'ess' not in precs:
            notes.append("Your data have no group sizes (ne, nc): the sqrt(1/ñ) precision term was not checked.")
        for prec in precs:
            label = f"time-lag (precision: {prec}, CR2 {'on' if use_cr2 else 'off'}, α = {alpha:g})"
            try:
                res = fit_timelag_model(data, ecol, vcol, vcv, precision=prec, use_cr2=use_cr2, alpha=alpha,
                                        reference_df=ref_df)
                r = _tl_r(res, df.loc[res['data']['row'].to_numpy()], vcv, use_cr2, alpha)
            except Exception as e:
                rows.append({'dataset': name, 'check': label, 'status': f'ERROR: {e}'})
                continue
            t = res['table']
            d_est = float(np.max(np.abs(t['estimate'] - r[:, 0])))
            d_se = float(np.max(np.abs(t['se'] - r[:, 1]) / r[:, 1]))
            d_df = float(np.max(np.abs(t['df'] - r[:, 2]) / r[:, 2]))
            d_p = float(np.max(np.abs(t['p_value'] - r[:, 3])))
            d_ci = float(max(np.max(np.abs(t['ci_lower'] - r[:, 4])), np.max(np.abs(t['ci_upper'] - r[:, 5]))))
            ok = (d_est < TOL_TL['est'] and d_se < TOL_TL['se_rel'] and d_df < TOL_TL['df_rel']
                  and d_p < TOL_TL['p'] and d_ci < TOL_TL['ci'] and res['used_cr2'] == use_cr2)
            yr = t.set_index('term').loc['year']
            rows.append({'dataset': name, 'check': label, 'k': res['n_obs'], 'studies': res['n_studies'],
                         'dropped': res['n_dropped'], 'sampling V': res['sampling_v'],
                         'year slope (Py)': yr['estimate'], 'year slope (R)': r[1, 0],
                         'p year (Py)': yr['p_value'], 'p year (R)': r[1, 3],
                         'max |Δ| est': d_est, 'max rel Δ SE': d_se, 'max rel Δ df': d_df, 'max |Δ| p': d_p,
                         'max |Δ| CI': d_ci,
                         'status': 'PASS' if ok else ('FAIL (CR2 setting not applied)'
                                                      if res['used_cr2'] != use_cr2 else 'FAIL')})
    if notes:
        display(HTML("<div style='padding:10px; background:#fff3cd; color:#856404; border-radius:4px; "
                     "font-family:sans-serif; margin-bottom:6px;'>ℹ️ " + "<br>ℹ️ ".join(notes) + "</div>"))
    res_df = pd.DataFrame(rows)
    n_fail = int((res_df['status'] != 'PASS').sum())
    colour, bg, icon = (('#155724', '#d4edda', '✅') if n_fail == 0 else ('#721c24', '#f8d7da', '❌'))
    display(HTML(
        f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
        f"<b>{icon} Time-lag test vs metafor + clubSandwich: {len(res_df) - n_fail}/{len(res_df)} checks passed.</b>"
        f"<br><small>rma.mv(mods = ~ year_c [+ precision], random = ~ 1 | study/es) with coef_test(CR2, "
        f"Satterthwaite) or model-based SE (CR2 off), same rows and sampling V. Tolerances: {TOL_TL}.</small></div>"))
    display(res_df.style.format(precision=6, na_rep=''))
    res_df.attrs['notes'] = ' '.join(notes)
    return res_df


res_df = _v26_main()
if res_df is not None and 'record_validation' in globals():
    record_validation(26, "Time-Lag Test", res_df, notes=res_df.attrs.get('notes', ''))


In [ ]:
#@title 🧪 Validation Summary & Export
# Requires Cells 1 and 1b in this session (they define the shared functions).
if not globals().get('COMETA_SETUP_DONE'):
    raise RuntimeError("CoMeta setup not found in this session: run Cell 1 (Environment Setup) and Cell 1b "
                       "(Unified Model Engine) first. This happens after a runtime restart, e.g. when Colab "
                       "asks to restart after installing packages.")

#
# Collects the outcome every validation cell recorded in VALIDATION_RESULTS
# (registry defined in "Validation: Imports & R Setup"), shows one summary
# table and writes cometa_validation_results.json with the results and the
# Python / R software versions, so the validation record can be archived
# alongside the paper.
import json
import os
import platform
import sys
import importlib
from datetime import datetime as _dt, timezone as _tz   # aliased: other cells use `import datetime`
import pandas as pd
import rpy2.robjects as ro
from IPython.display import display, HTML

VALIDATION_FILE = 'cometa_validation_results.json'
N_VALIDATIONS = 26

registry = globals().get('VALIDATION_RESULTS', {})

# --- 1. Summary table ---
_summary_rows = []
for n in sorted(set(range(1, N_VALIDATIONS + 1)) | set(registry)):
    v = registry.get(n)
    if v is None:
        _summary_rows.append({'#': n, 'Validation': '—', 'Checks passed': '', 'Status': 'not run', 'Run at (UTC)': ''})
        continue
    graded = v['n_checks'] - v['n_info']
    if v.get('skipped'):
        _summary_rows.append({'#': n, 'Validation': v['title'], 'Checks passed': '',
                              'Status': 'skipped', 'Run at (UTC)': v['timestamp']})
        continue
    status = 'INFO' if v['passed'] is None else ('PASS' if v['passed'] else 'FAIL')
    _summary_rows.append({'#': n, 'Validation': v['title'],
                 'Checks passed': f"{v['n_pass']}/{graded}" + (f" (+{v['n_info']} info)" if v['n_info'] else ''),
                 'Status': status, 'Run at (UTC)': v['timestamp']})
summary_df = pd.DataFrame(_summary_rows)

n_skipped = int((summary_df['Status'] == 'skipped').sum())
n_run = int((~summary_df['Status'].isin(['not run', 'skipped'])).sum())
n_failed = int((summary_df['Status'] == 'FAIL').sum())
# Green only when every validation ran and none failed; amber when some
# were not run in this session; red when any failed (including crashes).
if n_failed > 0:
    colour, bg, icon = ('#721c24', '#f8d7da', '❌')
elif n_run + n_skipped < len(summary_df):
    colour, bg, icon = ('#856404', '#fff3cd', '⚠️')
else:
    colour, bg, icon = ('#155724', '#d4edda', '✅')
display(HTML(
    f"<div style='padding:12px; background:{bg}; color:{colour}; border-radius:4px; font-family:sans-serif;'>"
    f"<b>{icon} Validation summary: {n_run}/{len(summary_df)} validations run on "
    f"{'your data and the reference datasets' if globals().get('VALIDATION_INCLUDE_REFERENCE') else 'your data'}, "
    f"{n_failed} with failures" + (f", {n_skipped} skipped (reference datasets off)" if n_skipped else "") + ".</b><br>"
    f"<small>'not run': the analysis step it checks has not been run in this session. 'skipped': it uses "
    f"only fixed reference datasets, which are switched off. INFO marks descriptive comparisons without "
    f"a pass/fail criterion.</small></div>"))


def _status_colour(val):
    return {'PASS': 'background-color: #d4edda', 'FAIL': 'background-color: #f8d7da',
            'INFO': 'background-color: #d1ecf1', 'not run': 'color: #888'}.get(val, '')


_sty = summary_df.style
_elementwise = _sty.map if hasattr(_sty, 'map') else _sty.applymap
display(_elementwise(_status_colour, subset=['Status']).hide(axis='index'))

# --- 2. Software versions ---
py_versions = {'python': sys.version.split()[0], 'platform': platform.platform()}
for mod in ('numpy', 'scipy', 'pandas', 'statsmodels', 'patsy', 'rpy2', 'sklearn'):
    try:
        py_versions[mod] = importlib.import_module(mod).__version__
    except Exception:
        py_versions[mod] = None

r_versions = {}
for pkg in ('metafor', 'clubSandwich', 'metadat', 'MetaUtility', 'metaforest', 'ranger', 'boot'):
    try:
        r_versions[pkg] = str(ro.r(f'as.character(packageVersion("{pkg}"))')[0])
    except Exception:
        r_versions[pkg] = None
try:
    r_versions['R'] = str(ro.r('R.version.string')[0])
    r_session_info = str(ro.r("paste(capture.output(sessionInfo()), collapse='\\n')")[0])
except Exception as e:
    r_session_info = f'sessionInfo() unavailable: {e}'

# --- 3. JSON export ---
export = {
    'generated_utc': _dt.now(_tz.utc).isoformat(timespec='seconds'),
    'n_validations': N_VALIDATIONS,
    'n_run': n_run,
    'n_failed': n_failed,
    'not_run': [int(n) for n in summary_df.loc[summary_df['Status'] == 'not run', '#']],
    'python_versions': py_versions,
    'r_versions': r_versions,
    'r_cran_snapshot': {'date': globals().get('R_SNAPSHOT_DATE'), 'repos': globals().get('R_SNAPSHOT_REPO')},
    'r_session_info': r_session_info,
    'validations': {str(n): registry[n] for n in sorted(registry)},
}
try:
    with open(VALIDATION_FILE, 'w', encoding='utf-8') as f:
        json.dump(export, f, indent=2, ensure_ascii=False, default=str)
    display(HTML(f"<p style='font-size:13px;color:#383d41;font-family:sans-serif;'>📁 Validation record written to "
                 f"<code>{os.path.abspath(VALIDATION_FILE)}</code> (commit it alongside the paper).</p>"))
    if globals().get('IS_COLAB', False):
        try:
            from google.colab import files
            files.download(VALIDATION_FILE)
        except Exception:
            display(HTML("<p style='font-size:12px;color:#856404;'>Download it from the Colab file browser (left sidebar).</p>"))
except Exception as e:
    display(HTML(f"<p style='color:#856404'>Could not write {VALIDATION_FILE}: {e}</p>"))